In [3]:
%pip install -q --no-cache-dir --force-reinstall --no-deps \
    "transformers==4.57.1" \
    "huggingface_hub==0.36.0" \
    "tokenizers==0.22.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 229.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 304.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 273.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.7 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import bitsandbytes as bnb
import accelerate
from transformers.utils import is_bitsandbytes_available

print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)
print("Accelerate:", accelerate.__version__)
print("Transformers detects bitsandbytes:", is_bitsandbytes_available())

assert torch.cuda.is_available(), "Enable the Kaggle GPU."
assert is_bitsandbytes_available(), "bitsandbytes is still not detected."

CUDA available: True
bitsandbytes: 0.50.2
Accelerate: 1.15.0
Transformers detects bitsandbytes: True


In [6]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Transformers:", transformers.__version__)
print("Required imports: OK")

Transformers: 4.57.1
Required imports: OK


## 1, Session check and dependencies


In [7]:
if globals().get('bridge') is not None or globals().get('SCHEDULER') is not None:
    raise RuntimeError('Use a fresh session for this clean startup notebook; do not overwrite a running voice session.')
import subprocess, sys

packages = [
    'transformers==4.57.1', 'huggingface_hub==0.36.0', 'tokenizers==0.22.1',
    'accelerate', 'bitsandbytes', 'vocos', 'encodec', 'librosa', 'soundfile',
    'fastapi', 'uvicorn[standard]', 'pyngrok', 'python-multipart', 'requests',
    'pandas', 'scipy', 'torchaudio', 'av', 'zipvoice', 'httpx', 'websockets', 'psutil'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencies installed. Continue with the next cell.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.2 MB/s eta 0:00:00
Dependencies installed. Continue with the next cell.


## 2. Secrets


In [8]:
import os, getpass
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()

def secret(name):
    value = os.environ.get(name, '').strip()
    if not value:
        try:
            value = client.get_secret(name).strip()
        except Exception:
            value = ''
    if not value:
        value = getpass.getpass(f'Enter {name}: ').strip()
    if not value:
        raise ValueError(f'{name} is required')
    os.environ[name] = value
    return value

SARVAM_API_KEY = secret('SARVAM_API_KEY')
NGROK_AUTHTOKEN = secret('NGROK_AUTHTOKEN')
HF_TOKEN = secret('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print('Sarvam, ngrok and Hugging Face secrets configured. Values are hidden.')


Enter SARVAM_API_KEY:  ········
Enter NGROK_AUTHTOKEN:  ········
Enter HF_TOKEN:  ········


Sarvam, ngrok and Hugging Face secrets configured. Values are hidden.


## 3. Install the full website


In [9]:
from pathlib import Path, PurePosixPath
import zipfile, shutil, hashlib, datetime

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
TARGET = WORK / 'MOVIO_MAIN_WEBSITE'
REQUIRED = [
    'server.py', 'web/index.html', 'web/app.js', 'web/styles.css',
    'web/auth.js', 'web/movio_enhancements.js',
    'integrations/ai_call_sarvam.py', 'integrations/ai_call_page.html',
    'integrations/movio_original_tts_adapter.py'
]
REQUIRED.append('web/live_tracking.js')
def is_upgrade(reader):
    try:
        server = reader('server.py').decode('utf-8')
        auth = reader('web/auth.js').decode('utf-8')
        ui = reader('web/movio_enhancements.js').decode('utf-8')
        return ('driver_profiles' in server and 'location-request' in server
                and 'vehicle_type' in auth and 'Request customer location' in ui)
    except (OSError, KeyError, UnicodeError):
        return False

candidates = []
for server in sorted(INPUT.rglob('server.py')):
    folder = server.parent
    if all((folder / name).is_file() for name in REQUIRED):
        reader = lambda name, folder=folder: (folder / name).read_bytes()
        if is_upgrade(reader):
            signature = hashlib.sha256(b''.join(reader(n) for n in REQUIRED)).hexdigest()
            candidates.append((signature, 'folder', folder, ''))

for package in sorted(INPUT.rglob('*.zip')):
    try:
        with zipfile.ZipFile(package) as archive:
            names = set(archive.namelist())
            for member in sorted(names):
                if PurePosixPath(member).name != 'server.py':
                    continue
                prefix = member[:-len('server.py')]
                if all(prefix + name in names for name in REQUIRED):
                    reader = lambda name: archive.read(prefix + name)
                    if is_upgrade(reader):
                        signature = hashlib.sha256(b''.join(reader(n) for n in REQUIRED)).hexdigest()
                        candidates.append((signature, 'zip', package, prefix))
    except zipfile.BadZipFile:
        continue

unique = {}
for candidate in candidates:
    unique.setdefault(candidate[0], candidate)
if not unique:
    raise FileNotFoundError(
        'Attach MOVIO_Driver_Location_Upgrade.zip as a Kaggle Input dataset. '
        'A complete upgraded package was not found. The older website.zip is insufficient.'
    )
if len(unique) > 1:
    raise RuntimeError('Multiple different upgraded websites found. Attach only the intended version: '
                       + str([str(c[2]) for c in unique.values()]))

_, kind, package, prefix = next(iter(unique.values()))
stage = WORK / '_movio_install_source'
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir()
if kind == 'folder':
    shutil.copytree(package, stage, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(package) as archive:
        for member in archive.infolist():
            if member.is_dir() or not member.filename.startswith(prefix):
                continue
            relative = PurePosixPath(member.filename[len(prefix):])
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError('Unsafe archive path: ' + str(relative))
            dest = stage.joinpath(*relative.parts)
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(archive.read(member))

backup = WORK / ('MOVIO_BACKUP_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
changed = 0
for source in stage.rglob('*'):
    if not source.is_file():
        continue
    rel = source.relative_to(stage)
    if (source.suffix.lower() in {'.db', '.sqlite', '.sqlite3', '.pyc'}
        or source.name.endswith(('-wal', '-shm', '-journal'))
        or source.name == '.env' or '__pycache__' in rel.parts):
        continue
    dest = TARGET / rel
    if dest.is_file() and dest.read_bytes() == source.read_bytes():
        continue
    if dest.exists():
        old = backup / rel
        old.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(dest, old)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
    changed += 1

for name in REQUIRED:
    if not (TARGET / name).is_file():
        raise RuntimeError('Website file missing: ' + name)
print('Website folder:', TARGET)
print('Updated files:', changed)
if backup.exists():
    print('Backup:', backup)
print('Driver vehicle matching and customer location request code found.')


Website folder: /kaggle/working/MOVIO_MAIN_WEBSITE
Updated files: 25
Driver vehicle matching and customer location request code found.


## 4. Apply multiple bookings before launching services


In [10]:
from pathlib import Path, PurePosixPath
import os, zipfile, json, shutil, datetime, ast, importlib.util, sys

TARGET = Path('/kaggle/working/MOVIO_MAIN_WEBSITE')
PORT = 8080
PATCH_SOURCE = ''  # Optional: exact patch folder or ZIP path.
MARKER = 'movio_multiple_bookings_patch.json'
FILES = ['server.py', 'web/app.js', 'web/movio_enhancements.js',
         'web/index.html', 'web/styles.css', 'integrations/ai_call_sarvam.py']
assert (TARGET / 'server.py').is_file(), 'Use your existing MOVIO website session'

candidates = []
roots = [Path('/kaggle/input')]
if PATCH_SOURCE:
    chosen = Path(PATCH_SOURCE)
    if chosen.is_dir():
        candidates.append(('folder', chosen, ''))
    elif chosen.is_file():
        roots = []
        with zipfile.ZipFile(chosen) as archive:
            markers = [n for n in archive.namelist() if PurePosixPath(n).name == MARKER]
            assert len(markers) == 1, 'ZIP must contain one patch manifest'
            candidates.append(('zip', chosen, markers[0][:-len(MARKER)]))
else:
    for root in roots:
        if not root.exists():
            continue
        for manifest in root.rglob(MARKER):
            folder = manifest.parent
            if all((folder / name).is_file() for name in FILES):
                candidates.append(('folder', folder, ''))
        for package in root.rglob('MOVIO_Multiple_Bookings_Update*.zip'):
            with zipfile.ZipFile(package) as archive:
                for member in archive.namelist():
                    if PurePosixPath(member).name == MARKER:
                        candidates.append(('zip', package, member[:-len(MARKER)]))

assert candidates, 'Attach MOVIO_Multiple_Bookings_Update.zip as a Kaggle Input dataset'
# Identical extracted/archive copies are allowed; differing versions require an explicit selection.
verified = {}
for kind, path, prefix in candidates:
    if kind == 'folder':
        reader = lambda name, path=path: (path / name).read_bytes()
        payload = {name: reader(name) for name in FILES}
        manifest = json.loads(reader(MARKER))
    else:
        with zipfile.ZipFile(path) as archive:
            payload = {name: archive.read(prefix + name) for name in FILES}
            manifest = json.loads(archive.read(prefix + MARKER))
    assert manifest['version'] == '5.2', 'Wrong multiple-bookings patch version'
    import hashlib
    signature = hashlib.sha256(b''.join(payload[name] for name in FILES)).hexdigest()
    verified.setdefault(signature, (path, payload))
assert len(verified) == 1, 'Different patches found; set PATCH_SOURCE to your intended patch'
package_path, patch_bytes = next(iter(verified.values()))
print('Multiple-bookings patch found:', package_path)


backup = TARGET / ('MOVIO_MULTIPLE_BOOKINGS_BACKUP_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
backup.mkdir(parents=True, exist_ok=True)
for name, content in patch_bytes.items():
    destination = TARGET / name
    if destination.exists():
        saved = backup / name
        saved.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(destination, saved)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)
print('Installed files:', len(patch_bytes))
print('Backup:', backup)
print('Accounts, rides, model files and prompt caches retained.')



Multiple-bookings patch found: /kaggle/input/datasets/reshrrt/movio-multiple-bookings-update/MOVIO_Multiple_Bookings_Update
Installed files: 6
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/MOVIO_MULTIPLE_BOOKINGS_BACKUP_20261028_070504_911074
Accounts, rides, model files and prompt caches retained.


## 5. Install the language-first call page


In [11]:
from pathlib import Path
import zipfile, shutil, datetime

CALL_PAGE_FILE = Path('/kaggle/working/MOVIO_MAIN_WEBSITE/integrations/ai_call_page.html')
assert CALL_PAGE_FILE.is_file(), 'Start your existing MOVIO website first.'
input_root = Path('/kaggle/input')
pages = list(input_root.rglob('ai_call_page.html'))
contents = [p.read_text() for p in pages if 'CONTROL_TEXT' in p.read_text()]
for package in input_root.rglob('*.zip'):
    try:
        with zipfile.ZipFile(package) as archive:
            for name in archive.namelist():
                if name.endswith('/ai_call_page.html'):
                    text = archive.read(name).decode('utf-8')
                    if 'CONTROL_TEXT' in text:
                        contents.append(text)
    except zipfile.BadZipFile:
        continue
unique = list(dict.fromkeys(contents))
assert len(unique) == 1, 'Attach only the MOVIO_AI_Call_Theme_Update.zip dataset.'
backup = CALL_PAGE_FILE.with_name('ai_call_page.backup_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '.html')
shutil.copy2(CALL_PAGE_FILE, backup)
page = unique[0]
page = page.replace('MOVIO<span style="color:#e2a600">.</span>', 'AI Ride Call')
page = page.replace('Return to MOVIO', 'Return to Dashboard')
page = page.replace('MOVIO பக்கத்திற்குத் திரும்பு', 'முகப்புப் பக்கத்திற்குத் திரும்பு')
page = page.replace('MOVIO ku Thirumbu', 'Dashboard ku Thirumbu')
page = page.replace('logged-in MOVIO dashboard', 'logged-in dashboard')
CALL_PAGE_FILE.write_text(page)
print('Call page updated. Close the call popup, refresh MOVIO and reopen AI Call.')
print('No model reload or kernel restart is required.')


Call page updated. Close the call popup, refresh MOVIO and reopen AI Call.
No model reload or kernel restart is required.


## 6. Reference recordings


In [12]:
import os
from pathlib import Path

MOVIO_ASSETS = "/kaggle/working/MOVIO_TTS"
os.makedirs(MOVIO_ASSETS, exist_ok=True)

# Find the original reference files in Kaggle Input
english_files = list(Path("/kaggle/input").rglob("english_reference_16k.wav"))
tamil_files = list(Path("/kaggle/input").rglob("tamil_reference.wav"))

assert english_files, "English reference WAV not found"
assert tamil_files, "Tamil reference WAV not found"

REFERENCE_ENGLISH = str(english_files[0])
REFERENCE_TAMIL = str(tamil_files[0])

# Cache DhVaani model files during this session
os.environ["HF_HOME"] = os.path.join(MOVIO_ASSETS, "huggingface_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print("========== MOVIO TTS READY ==========")
print("English reference:", REFERENCE_ENGLISH)
print("Tamil reference:", REFERENCE_TAMIL)
print("Model cache:", os.environ["HF_HOME"])
print("Both original reference files found!")
print("=====================================")

========== MOVIO TTS READY ==========
English reference: /kaggle/input/datasets/reshrrt/movio-tts-reference-audios/english_reference_16k.wav
Tamil reference: /kaggle/input/datasets/reshrrt/movio-tts-reference-audios/tamil_reference.wav
Model cache: /kaggle/working/MOVIO_TTS/huggingface_cache
Both original reference files found!


## 7. TTS runtime configuration


In [13]:
import os
import sys
import gc
import re
import time
import math
import random
import warnings
import threading
import subprocess
import importlib
import getpass
import heapq

from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import torch
import torchaudio
import transformers
import soundfile as sf

from IPython.display import display, Audio, Markdown
from transformers import AutoModel
from kaggle_secrets import UserSecretsClient

warnings.filterwarnings("ignore")

# ============================================================
# KAGGLE + HUGGING FACE AUTHENTICATION
# ============================================================

client = UserSecretsClient()

def get_secret(name, required=False):
    value = os.environ.get(name, "").strip()

    if not value:
        try:
            value = client.get_secret(name)
        except Exception:
            value = ""

    if not value and required:
        value = getpass.getpass(f"Enter {name}: ").strip()

    return value

HF_TOKEN = get_secret("HF_TOKEN", required=True)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Hugging Face authentication token configured.")

# ============================================================
# MOVIO MODEL CONFIGURATION
# ============================================================

MODEL_ID = "ARTPARK-IISc/DhVaani-0.5"

DEVICE = "cuda"
TARGET_SR = 24000

SEED = 666
RANDOM_SEED = 42

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable GPU in Kaggle: Settings > Accelerator > GPU"
    )

# ============================================================
# GPU OPTIMIZATION
# ============================================================

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ============================================================
# LATENCY + QUALITY CONFIG
# ============================================================

FIRST_CHUNK_NUM_STEP = 5
REST_CHUNK_NUM_STEP = 5

ENGLISH_SPEED = 1.05
TAMIL_SPEED = 1.05
TANGLISH_SPEED = 1.05

USE_FP16_SAMPLE = False

# ============================================================
# SHORTER SEMANTIC C1
# ============================================================

FIRST_MIN_WORDS = 3
FIRST_TARGET_WORDS = 3
FIRST_MAX_WORDS = 5

NEXT_MIN_WORDS = 3
NEXT_TARGET_WORDS = 5
NEXT_MAX_WORDS = 8

KEEP_WHOLE_IF_WORDS_LE = 4

ENGLISH_SINGLE_CHUNK_WORDS = 10
ENGLISH_MIN_CHUNK_WORDS = 4

# ============================================================
# STREAMING PRIORITIES
# ============================================================

FIRST_AUDIO_PRIORITY = 0
CONTINUATION_PRIORITY = 10

FIRST_AUDIO_DEADLINE_MS = 500.0
ADMISSION_QUEUE_LIMIT = 128

RUN_FULL_80_AUDIO_REVIEW = False

# ============================================================
# RANDOM SEEDS
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# KAGGLE MODEL CACHE
# ============================================================

MOVIO_ASSETS = "/kaggle/working/MOVIO_TTS"

os.makedirs(MOVIO_ASSETS, exist_ok=True)

# Keep an existing cache configuration if already initialized
os.environ.setdefault(
    "HF_HOME",
    os.path.join(MOVIO_ASSETS, "huggingface_cache")
)

os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# ============================================================
# RUNTIME STATUS
# ============================================================

print("=" * 88)
print("MOVIO CLEAN RUNTIME — KAGGLE")
print("=" * 88)

print("PyTorch       :", torch.__version__)
print("Transformers  :", transformers.__version__)
print("GPU           :", torch.cuda.get_device_name(0))
print("CUDA          :", torch.version.cuda)

print("Model         :", MODEL_ID)
print("Model cache   :", os.environ["HF_HOME"])

print("C1 step       :", FIRST_CHUNK_NUM_STEP)
print("Rest step     :", REST_CHUNK_NUM_STEP)
print("FP16 sample   :", USE_FP16_SAMPLE)

print("English speed :", ENGLISH_SPEED)
print("Tamil speed   :", TAMIL_SPEED)
print("Tanglish speed:", TANGLISH_SPEED)

print("HF token      :", "CONFIGURED" if HF_TOKEN else "MISSING")

print("=" * 88)
print("MOVIO RUNTIME INITIALIZED SUCCESSFULLY")
print("=" * 88)

Hugging Face authentication token configured.
MOVIO CLEAN RUNTIME — KAGGLE
PyTorch       : 2.11.0+cu128
Transformers  : 4.57.1
GPU           : Tesla T4
CUDA          : 12.8
Model         : ARTPARK-IISc/DhVaani-0.5
Model cache   : /kaggle/working/MOVIO_TTS/huggingface_cache
C1 step       : 5
Rest step     : 5
FP16 sample   : False
English speed : 1.05
Tamil speed   : 1.05
Tanglish speed: 1.05
HF token      : CONFIGURED
MOVIO RUNTIME INITIALIZED SUCCESSFULLY


## 8. Download and load the pinned DhVaani backend


In [14]:
import os
import getpass
from huggingface_hub import login, hf_hub_download

HF_TOKEN = str(
    os.environ.get("HF_TOKEN")
    or globals().get("HF_TOKEN")
    or ""
).strip()

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN").strip()
    except Exception:
        HF_TOKEN = ""

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Enter your Hugging Face read token: "
    ).strip()

if not HF_TOKEN:
    raise RuntimeError("A Hugging Face token is required.")

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

# Verify access before downloading the model weights.
hf_hub_download(
    repo_id="ARTPARK-IISc/DhVaani-0.5",
    filename=".gitattributes",
    revision="b079f01592e987042b86f4a01c3909cef569d247",
    token=HF_TOKEN,
)

print("Authenticated model access verified. Rerun the model-loading cell.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


.gitattributes: 0.00B [00:00, ?B/s]

Authenticated model access verified. Rerun the model-loading cell.


In [15]:
import sys
import types
import importlib
import importlib.util
import importlib.machinery
from pathlib import Path

import torch
from transformers import AutoModel
from huggingface_hub import snapshot_download

MODEL_ID = "ARTPARK-IISc/DhVaani-0.5"
REVISION = "b079f01592e987042b86f4a01c3909cef569d247"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Find the model files already downloaded in your session
repo_dir = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision=REVISION,
        local_files_only=False,
    )
)

backend_files = list(
    repo_dir.rglob("zipvoice/models/zipvoice.py")
)

if len(backend_files) != 1:
    raise RuntimeError(
        f"Expected one DhVaani backend, found {len(backend_files)}. "
        f"Cached model folder: {repo_dir}"
    )

package_dir = backend_files[0].parent.parent
backend_root = package_dir.parent

# Clear previous ZipVoice imports from this Python session
for name in list(sys.modules):
    if name == "zipvoice" or name.startswith("zipvoice."):
        sys.modules.pop(name, None)

backend_path = str(backend_root)
if backend_path in sys.path:
    sys.path.remove(backend_path)
sys.path.insert(0, backend_path)

# Bind 'zipvoice' specifically to DhVaani's backend
init_file = package_dir / "__init__.py"

if init_file.is_file():
    spec = importlib.util.spec_from_file_location(
        "zipvoice",
        init_file,
        submodule_search_locations=[str(package_dir)],
    )
    package = importlib.util.module_from_spec(spec)
    sys.modules["zipvoice"] = package
    spec.loader.exec_module(package)
else:
    package = types.ModuleType("zipvoice")
    package.__package__ = "zipvoice"
    package.__path__ = [str(package_dir)]
    package.__spec__ = importlib.machinery.ModuleSpec(
        "zipvoice",
        loader=None,
        is_package=True,
    )
    package.__spec__.submodule_search_locations = [
        str(package_dir)
    ]
    sys.modules["zipvoice"] = package

importlib.invalidate_caches()

# Verify the backend before loading weights
backend = importlib.import_module(
    "zipvoice.models.zipvoice"
)
print("Backend found:", backend.__file__)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    trust_remote_code=True,
    local_files_only=False,
).to(DEVICE).eval()

print("DhVaani loaded on:", DEVICE)
print("Sampling rate:", model.sampling_rate)

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

zipformer_two_stream.py: 0.00B [00:00, ?B/s]

__init__.py:   0%|          | 0.00/139 [00:00<?, ?B/s]

zipformer.py: 0.00B [00:00, ?B/s]

...

samples/malayalam_codeswitch.wav:   0%|          | 0.00/242k [00:00<?, ?B/s]

Backend found: /root/.cache/huggingface/hub/models--ARTPARK-IISc--DhVaani-0.5/snapshots/b079f01592e987042b86f4a01c3909cef569d247/_backend/zipvoice/models/zipvoice.py


Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

DhVaani loaded on: cuda
Sampling rate: 24000


## 9. Prompt conditioning and positional buffer check


In [16]:
%pip install -q vocos

Note: you may need to restart the kernel to use updated packages.


In [17]:
import torch
import sys
import types
import time

# Use the backend of the already-loaded DhVaani model
rt = model._runtime()

helper_names = (
    "load_prompt_wav",
    "remove_silence",
    "rms_norm",
    "add_punctuation",
)

namespaces = [
    fn.__globals__
    for fn in rt.values()
    if isinstance(fn, types.FunctionType)
]

for name, module in list(sys.modules.items()):
    if isinstance(module, types.ModuleType):
        if any(
            tag in name.lower()
            for tag in ("dhvaani", "zipvoice", "_backend")
        ):
            namespaces.append(vars(module))

helpers = next(
    (
        namespace
        for namespace in namespaces
        if all(
            callable(namespace.get(name))
            for name in helper_names
        )
    ),
    None,
)

if helpers is None:
    raise RuntimeError(
        "Cache helpers were not found in this DhVaani version. "
        f"Runtime keys: {list(rt.keys())}. "
        "Share this output so the cell can be adapted."
    )

load_prompt_wav = helpers["load_prompt_wav"]
remove_silence = helpers["remove_silence"]
rms_norm = helpers["rms_norm"]
add_punctuation = helpers["add_punctuation"]

tokenizer = rt["tokenizer"]
feature_extractor = rt["feature"]

DEVICE = model.device
SAMPLING_RATE = model.sampling_rate

TARGET_RMS = 0.1
FEAT_SCALE = 0.1


@torch.inference_mode()
def prepare_prompt_cache(prompt_wav_path, prompt_text):
    """Cache the fixed reference audio and transcript conditioning."""

    start = time.perf_counter()

    # Load and clean reference audio
    wav = load_prompt_wav(
        str(prompt_wav_path),
        sampling_rate=SAMPLING_RATE,
    )

    wav = remove_silence(
        wav,
        SAMPLING_RATE,
        only_edge=False,
        trail_sil=200,
    )

    wav, prompt_rms = rms_norm(wav, TARGET_RMS)

    prompt_duration = wav.shape[-1] / SAMPLING_RATE

    # Extract and retain reference features
    features = feature_extractor.extract(
        wav,
        sampling_rate=SAMPLING_RATE,
    )

    features = (
        features.to(DEVICE).unsqueeze(0) * FEAT_SCALE
    ).contiguous()

    # Tokenize the reference transcript
    clean_prompt_text = add_punctuation(prompt_text)

    prompt_tokens_str = tokenizer.texts_to_tokens(
        [clean_prompt_text]
    )[0]

    prompt_tokens = tokenizer.tokens_to_token_ids(
        [prompt_tokens_str]
    )

    cache = {
        "wav": wav,
        "rms": prompt_rms,
        "duration": prompt_duration,
        "features": features,
        "text": clean_prompt_text,
        "tokens_str": prompt_tokens_str,
        "tokens": prompt_tokens,
        "wav_path": str(prompt_wav_path),
    }

    elapsed_ms = (time.perf_counter() - start) * 1000.0

    print(f"Prompt cache created in {elapsed_ms:.2f} ms")
    print(f"Duration: {prompt_duration:.3f} sec")
    print(f"Feature shape: {tuple(features.shape)}")

    return cache


print("DhVaani prompt-cache function ready.")
print("Next: run your English and Tamil cache creation cell.")
import torch

print("========== MOVIO DHVAANI META REPAIR ==========")

fixed = 0
remaining = []

for name, module in model.model.named_modules():
    if not hasattr(module, "pe"):
        continue

    pe = getattr(module, "pe")

    if not isinstance(pe, torch.Tensor):
        continue

    if pe.device.type != "meta":
        continue

    print("Found meta positional encoding:", name)
    print("Shape:", tuple(pe.shape))

    if not hasattr(module, "extend_pe"):
        remaining.append(name)
        continue

    # The Zipformer positional encoder regenerates its
    # sinusoidal encoding when pe is initially empty.
    # Preserve its original registered-buffer status.
    old_pe = pe

    try:
        module.pe = torch.empty(
            0,
            device="cpu",
            dtype=torch.float32
        )

        fixed += 1
        print("Reinitialized:", name)

    except Exception as e:
        module.pe = old_pe
        remaining.append(name)
        print("Failed:", repr(e))

print("\nReinitialized positional buffers:", fixed)
print("Unresolved buffers:", remaining)

print("\nChecking remaining meta tensors:")

meta_params = [
    n for n, p in model.model.named_parameters()
    if p.device.type == "meta"
]

meta_buffers = [
    n for n, b in model.model.named_buffers()
    if b.device.type == "meta"
]

print("Meta parameters:", meta_params[:20])
print("Meta buffers:", meta_buffers[:20])

if meta_params or meta_buffers:
    raise RuntimeError("Unresolved meta tensors remain. Stop before starting voice calls.")
else:
    print("No remaining meta tensors detected.")

print("==============================================")


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/54.4M [00:00<?, ?B/s]

DhVaani prompt-cache function ready.
Next: run your English and Tamil cache creation cell.
========== MOVIO DHVAANI META REPAIR ==========

Reinitialized positional buffers: 0
Unresolved buffers: []

Checking remaining meta tensors:
Meta parameters: []
Meta buffers: []
No remaining meta tensors detected.


## 10. Original speech normalization


In [18]:
import re


# ============================================================
# ENGLISH DIGIT WORDS
# ============================================================

DIGIT_WORDS = {
    '0': 'zero',
    '1': 'one',
    '2': 'two',
    '3': 'three',
    '4': 'four',
    '5': 'fyeve',
    '6': 'six',
    '7': 'sevven',
    '8': 'ait',
    '9': 'nine'
}

PHONE_DIGIT_WORDS = DIGIT_WORDS.copy()


# ============================================================
# ENGLISH NUMBER WORDS
# ============================================================

NUMBER_WORDS = {
    0: "zero",
    1: "one",
    2: "two",
    3: "three",
    4: "four",
    5: "fyeve",
    6: "six",
    7: "sevven",
    8: "ait",
    9: "nine",

    10: "ten",
    11: "eleven",
    12: "twelve",
    13: "thirteen",
    14: "fourteen",
    15: "fifteen",
    16: "sixteen",
    17: "seventeen",
    18: "eighteen",
    19: "nineteen",

    20: "twenty",
    30: "thirty",
    40: "forty",
    50: "fifty",
    60: "sixty",
    70: "seventy",
    80: "eighty",
    90: "ninety"
}


def number_to_words(n):

    n = int(n)

    if n < 100:

        if n in NUMBER_WORDS:
            return NUMBER_WORDS[n]

        tens = (n // 10) * 10
        ones = n % 10

        return NUMBER_WORDS[tens] + " " + NUMBER_WORDS[ones]

    if n < 1000:

        hundreds = n // 100
        remainder = n % 100

        result = NUMBER_WORDS[hundreds] + " hundred"

        if remainder:
            result += " and " + number_to_words(remainder)

        return result

    if n < 100000:

        thousands = n // 1000
        remainder = n % 1000

        result = number_to_words(thousands) + " thousand"

        if remainder:
            result += " " + number_to_words(remainder)

        return result

    return " ".join(DIGIT_WORDS[d] for d in str(n))


# ============================================================
# TAMIL DIGITS
# ============================================================

TAMIL_DIGITS = {
    '0': 'பூஜ்ஜியம்',
    '1': 'ஒன்று',
    '2': 'இரண்டு',
    '3': 'மூன்று',
    '4': 'நான்கு',
    '5': 'ஐந்து',
    '6': 'ஆறு',
    '7': 'ஏழு',
    '8': 'எட்டு',
    '9': 'ஒன்பது'
}


# ============================================================
# TAMIL NUMBERS
# ============================================================

TAMIL_NUMBERS = {

    0: 'பூஜ்ஜியம்',
    1: 'ஒன்று',
    2: 'இரண்டு',
    3: 'மூன்று',
    4: 'நான்கு',
    5: 'ஐந்து',
    6: 'ஆறு',
    7: 'ஏழு',
    8: 'எட்டு',
    9: 'ஒன்பது',

    10: 'பத்து',
    11: 'பதினொன்று',
    12: 'பன்னிரண்டு',
    13: 'பதின்மூன்று',
    14: 'பதினான்கு',
    15: 'பதினைந்து',
    16: 'பதினாறு',
    17: 'பதினேழு',
    18: 'பதினெட்டு',
    19: 'பத்தொன்பது',

    20: 'இருபது',
    30: 'முப்பது',
    40: 'நாற்பது',
    50: 'ஐம்பது',
    60: 'அறுபது',
    70: 'எழுபது',
    80: 'எண்பது',
    90: 'தொண்ணூறு'
}


# ============================================================
# TAMIL HUNDREDS
# ============================================================

TAMIL_HUNDREDS = {
    1: 'நூறு',
    2: 'இருநூறு',
    3: 'முந்நூறு',
    4: 'நானூறு',
    5: 'ஐந்நூறு',
    6: 'அறுநூறு',
    7: 'எழுநூறு',
    8: 'எண்ணூறு',
    9: 'தொள்ளாயிரம்'
}


def tamil_number_to_words(n):

    n = int(n)

    if n < 0:
        return 'மைனஸ் ' + tamil_number_to_words(abs(n))

    if n in TAMIL_NUMBERS:
        return TAMIL_NUMBERS[n]

    if 20 < n < 30:
        return 'இருபத்தி ' + TAMIL_NUMBERS[n % 10]

    if 30 < n < 40:
        return 'முப்பத்தி ' + TAMIL_NUMBERS[n % 10]

    if 40 < n < 50:
        return 'நாற்பத்தி ' + TAMIL_NUMBERS[n % 10]

    if 50 < n < 60:
        return 'ஐம்பத்தி ' + TAMIL_NUMBERS[n % 10]

    if 60 < n < 70:
        return 'அறுபத்தி ' + TAMIL_NUMBERS[n % 10]

    if 70 < n < 80:
        return 'எழுபத்தி ' + TAMIL_NUMBERS[n % 10]

    if 80 < n < 90:
        return 'எண்பத்தி ' + TAMIL_NUMBERS[n % 10]

    if 90 < n < 100:
        return 'தொண்ணூற்று ' + TAMIL_NUMBERS[n % 10]

    if n < 1000:

        hundreds = n // 100
        remainder = n % 100

        hundred_word = TAMIL_HUNDREDS[hundreds]

        if not remainder:
            return hundred_word

        if hundred_word.endswith('நூறு'):
            hundred_word = hundred_word.replace(
                'நூறு',
                'நூற்று'
            )

        elif hundred_word == 'தொள்ளாயிரம்':
            hundred_word = 'தொள்ளாயிரத்து'

        return hundred_word + ' ' + tamil_number_to_words(remainder)

    if n < 100000:

        thousands = n // 1000
        remainder = n % 1000

        thousand_word = tamil_number_to_words(thousands)

        if remainder:
            return (
                thousand_word
                + ' ஆயிரத்து '
                + tamil_number_to_words(remainder)
            )

        return thousand_word + ' ஆயிரம்'

    return ' '.join(TAMIL_DIGITS[d] for d in str(n))



# ============================================================
# TAMIL HUNDRED LINKER NORMALIZATION
# ============================================================

def normalize_written_tamil_hundreds(text):
    """
    Correct already-written Tamil number phrases.

    Examples:
        நானூறு                      -> நானூறு
        நானூறு ஐம்பது              -> நானூற்று ஐம்பது
        நானூறு அறுபது              -> நானூற்று அறுபது
        அறுநூறு ஐம்பது             -> அறுநூற்று ஐம்பது
        அறுநூறு அறுபது             -> அறுநூற்று அறுபது

    The linker form is used only when a non-zero remainder follows.
    """

    # Any valid Tamil number word that can follow a hundred.
    remainder = (
        r'(?:'
        r'ஒன்று|இரண்டு|மூன்று|நான்கு|ஐந்து|ஆறு|ஏழு|எட்டு|ஒன்பது|'
        r'பத்து|பதினொன்று|பன்னிரண்டு|பதின்மூன்று|பதினான்கு|'
        r'பதினைந்து|பதினாறு|பதினேழு|பதினெட்டு|பத்தொன்பது|'
        r'இருபது|இருபத்தி(?:\s+\S+)?|'
        r'முப்பது|முப்பத்தி(?:\s+\S+)?|'
        r'நாற்பது|நாற்பத்தி(?:\s+\S+)?|'
        r'ஐம்பது|ஐம்பத்தி(?:\s+\S+)?|'
        r'அறுபது|அறுபத்தி(?:\s+\S+)?|'
        r'எழுபது|எழுபத்தி(?:\s+\S+)?|'
        r'எண்பது|எண்பத்தி(?:\s+\S+)?|'
        r'தொண்ணூறு|தொண்ணூற்று(?:\s+\S+)?'
        r')'
    )

    linker_map = {
        'நூறு': 'நூற்று',
        'இருநூறு': 'இருநூற்று',
        'முந்நூறு': 'முந்நூற்று',
        'நானூறு': 'நானூற்று',
        'ஐந்நூறு': 'ஐந்நூற்று',
        'அறுநூறு': 'அறுநூற்று',
        'எழுநூறு': 'எழுநூற்று',
        'எண்ணூறு': 'எண்ணூற்று',
        'தொள்ளாயிரம்': 'தொள்ளாயிரத்து',
    }

    hundred_pattern = (
        r'(?<![\u0B80-\u0BFF])'
        r'(நூறு|இருநூறு|முந்நூறு|நானூறு|ஐந்நூறு|'
        r'அறுநூறு|எழுநூறு|எண்ணூறு|தொள்ளாயிரம்)'
        r'(?=\s+' + remainder + r')'
    )

    return re.sub(
        hundred_pattern,
        lambda m: linker_map[m.group(1)],
        text
    )


# ============================================================
# TAMIL ORDINALS
# ============================================================

TAMIL_ORDINALS = {
    1: 'முதலாவது',
    2: 'இரண்டாவது',
    3: 'மூன்றாவது',
    4: 'நான்காவது',
    5: 'ஐந்தாவது',
    6: 'ஆறாவது',
    7: 'ஏழாவது',
    8: 'எட்டாவது',
    9: 'ஒன்பதாவது',
    10: 'பத்தாவது',
    11: 'பதினொன்றாவது',
    12: 'பன்னிரண்டாவது',
    13: 'பதின்மூன்றாவது',
    14: 'பதினான்காவது',
    15: 'பதினைந்தாவது',
    16: 'பதினாறாவது',
    17: 'பதினேழாவது',
    18: 'பதினெட்டாவது',
    19: 'பத்தொன்பதாவது',
    20: 'இருபதாவது',
    30: 'முப்பதாவது',
    40: 'நாற்பதாவது',
    50: 'ஐம்பதாவது',
    60: 'அறுபதாவது',
    70: 'எழுபதாவது',
    80: 'எண்பதாவது',
    90: 'தொண்ணூறாவது'
}


def tamil_ordinal_to_words(n):

    n = int(n)

    if n in TAMIL_ORDINALS:
        return TAMIL_ORDINALS[n]

    if 20 < n < 30:

        ones = n % 10

        if ones == 1:
            return 'இருபத்தொன்றாவது'

        return 'இருபத்தி' + tamil_number_to_words(ones) + 'வது'

    if 30 < n < 40:

        ones = n % 10

        if ones == 1:
            return 'முப்பத்தொன்றாவது'

        return 'முப்பத்தி' + tamil_number_to_words(ones) + 'வது'

    if 40 < n < 50:

        ones = n % 10

        if ones == 1:
            return 'நாற்பத்தொன்றாவது'

        return 'நாற்பத்தி' + tamil_number_to_words(ones) + 'வது'

    if 50 < n < 60:

        ones = n % 10

        if ones == 1:
            return 'ஐம்பத்தொன்றாவது'

        return 'ஐம்பத்தி' + tamil_number_to_words(ones) + 'வது'

    if 60 < n < 70:

        ones = n % 10

        if ones == 1:
            return 'அறுபத்தொன்றாவது'

        return 'அறுபத்தி' + tamil_number_to_words(ones) + 'வது'

    if 70 < n < 80:

        ones = n % 10

        if ones == 1:
            return 'எழுபத்தொன்றாவது'

        return 'எழுபத்தி' + tamil_number_to_words(ones) + 'வது'

    if 80 < n < 90:

        ones = n % 10

        if ones == 1:
            return 'எண்பத்தொன்றாவது'

        return 'எண்பத்தி' + tamil_number_to_words(ones) + 'வது'

    if 90 < n < 100:

        ones = n % 10

        if ones == 1:
            return 'தொண்ணூற்றொன்றாவது'

        return 'தொண்ணூற்று' + tamil_number_to_words(ones) + 'வது'

    return tamil_number_to_words(n) + 'வது'


# ============================================================
# TAMIL TIME
# ============================================================

def tamil_time_words(hour, minute, ampm=None):

    hour = int(hour)
    minute = int(minute)

    parts = []

    if ampm:

        ampm = ampm.lower()

        if ampm == 'am':
            parts.append('காலை')

        elif ampm == 'pm':
            parts.append('மாலை')

    parts.append(tamil_number_to_words(hour))

    if minute == 0:
        parts.append('மணிக்கு')

    else:
        parts.append(tamil_number_to_words(minute))
        parts.append('மணிக்கு')

    return ' '.join(parts)


# ============================================================
# TAMIL CONVERSATIONAL DATES
# ============================================================

TAMIL_MONTHS = {
    1: 'ஜனவரி',
    2: 'பிப்ரவரி',
    3: 'மார்ச்',
    4: 'ஏப்ரல்',
    5: 'மே',
    6: 'ஜூன்',
    7: 'ஜூலை',
    8: 'ஆகஸ்ட்',
    9: 'செப்டம்பர்',
    10: 'அக்டோபர்',
    11: 'நவம்பர்',
    12: 'டிசம்பர்',
}


def tamil_year_to_words(year):
    """2026 -> இரண்டாயிரத்து இருபத்தாறு"""
    year = int(year)

    if 2000 <= year < 2100:
        remainder = year - 2000
        if remainder == 0:
            return 'இரண்டாயிரம்'

        conversational_20s = {
            21: 'இருபத்தொன்று',
            22: 'இருபத்திரண்டு',
            23: 'இருபத்திமூன்று',
            24: 'இருபத்திநான்கு',
            25: 'இருபத்தைந்து',
            26: 'இருபத்தாறு',
            27: 'இருபத்தேழு',
            28: 'இருபத்தெட்டு',
            29: 'இருபத்தொன்பது',
        }

        remainder_words = conversational_20s.get(
            remainder,
            tamil_number_to_words(remainder)
        )

        return 'இரண்டாயிரத்து ' + remainder_words

    return tamil_number_to_words(year)


def normalize_conversational_dates(text, has_tamil_script):
    """
    In Tamil/mixed Tamil conversation:
      12/09/2026 -> செப்டம்பர் பன்னிரண்டு, இரண்டாயிரத்து இருபத்தாறு
      18-09-2026 -> செப்டம்பர் பதினெட்டு, இரண்டாயிரத்து இருபத்தாறு
    """
    if not has_tamil_script:
        return text

    def repl(match):
        day = int(match.group(1))
        month = int(match.group(2))
        year = int(match.group(3))

        if not (1 <= day <= 31) or not (1 <= month <= 12):
            return match.group(0)

        return (
            TAMIL_MONTHS[month]
            + ' '
            + tamil_number_to_words(day)
            + ', '
            + tamil_year_to_words(year)
        )

    return re.sub(
        r'(?<!\d)(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?!\d)',
        repl,
        text
    )


# ============================================================
# TAMIL / MIXED TIME NORMALIZATION
# ============================================================

def normalize_tamil_mixed_times(text, has_tamil_script):
    """
    Critical behavior:

      காலை 6:30 AMக்கு -> காலை ஆறு முப்பது மணிக்கு
      காலை 6:30 AM     -> காலை ஆறு முப்பது
      6:30 AMக்கு       -> காலை ஆறு முப்பது மணிக்கு
      6:30 AM           -> காலை ஆறு முப்பது
      6:30 PM           -> மாலை ஆறு முப்பது

    An already-written Tamil daypart always wins. AM/PM never causes a
    duplicate காலை/மாலை.
    """
    if not has_tamil_script:
        return text

    daypart_pattern = r'(காலை|மதியம்|மாலை|இரவு)'

    def spoken_clock(hour, minute):
        parts = [tamil_number_to_words(int(hour))]
        minute = int(minute)
        if minute:
            parts.append(tamil_number_to_words(minute))
        return ' '.join(parts)

    # 1) Existing Tamil daypart + HH:MM + AM/PM.
    # Process FIRST to prevent "காலை காலை ...".
    def existing_daypart_ampm(match):
        daypart = match.group(1)
        spoken = spoken_clock(match.group(2), match.group(3))
        suffix = match.group(5)
        return daypart + ' ' + spoken + (' மணிக்கு' if suffix else '')

    text = re.sub(
        daypart_pattern
        + r'\s*(\d{1,2}):(\d{2})\s*(AM|PM)(-?க்கு)?(?![A-Za-z])',
        existing_daypart_ampm,
        text,
        flags=re.IGNORECASE
    )

    # 2) Existing Tamil daypart + HH:MM, no AM/PM.
    def existing_daypart_time(match):
        daypart = match.group(1)
        spoken = spoken_clock(match.group(2), match.group(3))
        suffix = match.group(4)
        return daypart + ' ' + spoken + (' மணிக்கு' if suffix else '')

    text = re.sub(
        daypart_pattern + r'\s*(\d{1,2}):(\d{2})(-?க்கு)?',
        existing_daypart_time,
        text
    )

    # 3) HH:MM + AM/PM where Tamil daypart was not already written.
    def ampm_without_daypart(match):
        hour = int(match.group(1))
        minute = int(match.group(2))
        ampm = match.group(3).upper()
        suffix = match.group(4)
        daypart = 'காலை' if ampm == 'AM' else 'மாலை'
        spoken = spoken_clock(hour, minute)
        return daypart + ' ' + spoken + (' மணிக்கு' if suffix else '')

    text = re.sub(
        r'(?<![\d:])(\d{1,2}):(\d{2})\s*(AM|PM)(-?க்கு)?(?![A-Za-z])(?![A-Za-z])',
        ampm_without_daypart,
        text,
        flags=re.IGNORECASE
    )

    # 4) Existing daypart + hour only.
    def daypart_hour(match):
        daypart = match.group(1)
        hour = tamil_number_to_words(int(match.group(2)))
        explicit_mani = match.group(3)
        suffix = match.group(4)
        return daypart + ' ' + hour + (' மணிக்கு' if (explicit_mani or suffix) else '')

    text = re.sub(
        daypart_pattern
        + r'\s+(\d{1,2})(\s*மணி(?:க்கு)?)?(-?க்கு)?(?=\s|[.,!?]|$)',
        daypart_hour,
        text
    )

    return text


# ============================================================
# EARLY CONTEXTUAL CURRENCY / PAISE NORMALIZATION
# ============================================================

def normalize_currency_early(text, has_tamil_script):
    """Normalize currency before generic RS acronym and general numbers."""

    def amount_words(raw):
        value = int(raw.replace(',', ''))
        if has_tamil_script:
            return tamil_number_to_words(value)
        return number_to_words(value)

    def paise_words(raw):
        value = int(raw)
        if has_tamil_script:
            return tamil_number_to_words(value) + ' பைசா'
        return number_to_words(value) + ' paise'

    # 450 rupees.50 -> four hundred and fifty rupees, fifty paise
    def decimal_rupees(match):
        whole = amount_words(match.group(1))
        paise = paise_words(match.group(2))
        if has_tamil_script:
            return whole + ' ரூபாய், ' + paise
        return whole + ' rupees, ' + paise

    text = re.sub(
        r'(?<!\d)([\d,]+)\s+rupees?\.(\d{1,2})(?!\d)',
        decimal_rupees,
        text,
        flags=re.IGNORECASE
    )

    # ₹725 / Rs 725 / Rs.725 / INR 725
    def prefixed_currency(match):
        spoken = amount_words(match.group(1))
        return spoken + (' ரூபாய்' if has_tamil_script else ' rupees')

    text = re.sub(r'₹\s*([\d,]+)', prefixed_currency, text)
    text = re.sub(
        r'\b(?:Rs\.?|INR)\s*([\d,]+)',
        prefixed_currency,
        text,
        flags=re.IGNORECASE
    )

    # 450 rupees (numeric amount followed by unit).
    def suffixed_currency(match):
        spoken = amount_words(match.group(1))
        return spoken + (' ரூபாய்' if has_tamil_script else ' rupees')

    text = re.sub(
        r'(?<!\d)([\d,]+)\s+rupees?\b',
        suffixed_currency,
        text,
        flags=re.IGNORECASE
    )

    return text


# ============================================================
# PRE-SPELLED BOOKING / OTP DIGIT PROTECTION
# ============================================================

def normalize_prespelled_identifier_digits(text):
    """
    Handles TTS-ready prefixes where raw digits are still attached/present:

      booking eye dee is see ay bee48291
      -> booking eye dee is see ay bee four eight two nine one

      o tee pee is 4821
      -> o tee pee is four eight two one
    """

    digit_seq = lambda value: ' '.join(DIGIT_WORDS[d] for d in value)

    # Digits immediately after a pre-spelled letter word in booking context.
    def booking_digits(match):
        return match.group(1) + ' ' + digit_seq(match.group(2))

    text = re.sub(
        r'((?:booking|book ing)\s+eye\s+dee\b.{0,80}?' \
        r'(?:ay|bee|see|dee|ee|eff|gee|aitch|eye|jay|kay|el|em|en|oh|pee|cue|ar|ess|tee|you|vee|double\s+you|ex|why|zee))'
        r'(?<!\s)(\d{2,10})\b',
        booking_digits,
        text,
        flags=re.IGNORECASE
    )

    # Booking context with a plain contiguous numeric tail.
    def booking_plain_digits(match):
        return match.group(1) + digit_seq(match.group(2))

    text = re.sub(
        r'((?:booking|book ing)\s+eye\s+dee\b[^\d]{0,100}?)(\d{2,10})\b',
        booking_plain_digits,
        text,
        flags=re.IGNORECASE
    )

    # Pre-spelled OTP + digits.
    def otp_digits(match):
        return match.group(1) + digit_seq(match.group(2))

    text = re.sub(
        r'((?:OTP|o\s+tee\s+pee)\b[^\d]{0,100}?)(\d{3,8})\b',
        otp_digits,
        text,
        flags=re.IGNORECASE
    )

    return text


# ============================================================
# TANGLISH WORDS
# ============================================================

TANGLISH_WORDS = {

    'ungal': 'உங்கள்',
    'unga': 'உங்க',
    'ungala': 'உங்களை',
    'ungalukku': 'உங்களுக்கு',
    'ungaloda': 'உங்களோட',

    'en': 'என்',
    'ennoda': 'என்னோட',
    'enakku': 'எனக்கு',

    'namma': 'நம்ம',
    'namba': 'நம்ம',
    'neenga': 'நீங்க',
    'nee': 'நீ',

    'avan': 'அவன்',
    'aval': 'அவள்',
    'avanga': 'அவங்க',

    'varum': 'வரும்',
    'varuthu': 'வருது',
    'varudhu': 'வருது',
    'vandhudum': 'வந்துடும்',
    'vandhu': 'வந்து',
    'vanga': 'வாங்க',
    'vaanga': 'வாங்க',

    'pogum': 'போகும்',
    'poguthu': 'போகுது',
    'poga': 'போக',
    'poitu': 'போயிட்டு',

    'irukku': 'இருக்கு',
    'iruku': 'இருக்கு',
    'irukkum': 'இருக்கும்',

    'illa': 'இல்ல',
    'illai': 'இல்லை',
    'illaya': 'இல்லையா',

    'inga': 'இங்க',
    'ingae': 'இங்கே',
    'inge': 'இங்கே',

    'anga': 'அங்க',
    'ange': 'அங்கே',

    'aagum': 'ஆகும்',
    'aakum': 'ஆகும்',

    'maniku': 'மணிக்கு',
    'manikku': 'மணிக்கு',
    'mani': 'மணி',

    'kaalai': 'காலை',
    'kalai': 'காலை',

    'mathiyam': 'மதியம்',
    'madhiyam': 'மதியம்',

    'maalai': 'மாலை',
    'malai': 'மாலை',

    'iravu': 'இரவு',

    'cab': 'cab',
    'taxi': 'taxi',
    'driver': 'driver',
    'ride': 'ride',
    'booking': 'booking',
    'pickup': 'pickup',
    'drop': 'drop',

    'vandhuten': 'வந்துட்டேன்',
    'vandhutaen': 'வந்துட்டேன்',
    'vandhachu': 'வந்தாச்சு',

    # --------------------------------------------------------
    # ADDED: EXTRA TANGLISH FORMS FOR CLEARER TTS
    # Existing normalization rules are otherwise unchanged.
    # --------------------------------------------------------
    'vandhutaara': 'வந்துட்டாரா',
    'vandhuttara': 'வந்துட்டாரா',
    'vandhutara': 'வந்துட்டாரா',
    'vandhutaaraa': 'வந்துட்டாரா',

    'vandhutaaru': 'வந்துட்டாரு',
    'vandhuttaru': 'வந்துட்டாரு',
    'vandhutaar': 'வந்துட்டார்',

    'vandhutaanga': 'வந்துட்டாங்க',
    'vandhuttanga': 'வந்துட்டாங்க',

    'varuvaaru': 'வருவாரு',
    'varuvaru': 'வருவாரு',
    'varuvaanga': 'வருவாங்க',

    'irundhu': 'இருந்து',

    'pannuvaaru': 'பண்ணுவாரு',
    'pannuvaar': 'பண்ணுவார்',
    'pannuvanga': 'பண்ணுவாங்க',

    'pannirukku': 'பண்ணிருக்கு',
    'pannirukkum': 'பண்ணிருக்கும்',

    'aagirukku': 'ஆகிருக்கு',
    'agirukku': 'ஆகிருக்கு',

    'tholaivil': 'தொலைவில்',
    'ulladhu': 'உள்ளது',
    'ulla': 'உள்ள',

    'kilambiten': 'கிளம்பிட்டேன்',
    'kilambitten': 'கிளம்பிட்டேன்',
    'kilambunga': 'கிளம்புங்க',

    'left': 'left',
    'right': 'right',

    'pakkam': 'பக்கம்',
    'pakkathula': 'பக்கத்துல',
    'munnadi': 'முன்னாடி',
    'pinnadi': 'பின்னாடி',

    'please': 'please',
    'nandri': 'நன்றி',
    'thanks': 'thanks',
    'sari': 'சரி',
    'seri': 'சரி',
    'ok': 'okay'
}


def normalize_tanglish_words(text):

    words = sorted(
        TANGLISH_WORDS.keys(),
        key=len,
        reverse=True
    )

    for word in words:

        tamil_word = TANGLISH_WORDS[word]

        pattern = (
            r'(?<![A-Za-z])'
            + re.escape(word)
            + r'(?![A-Za-z])'
        )

        text = re.sub(
            pattern,
            tamil_word,
            text,
            flags=re.IGNORECASE
        )

    return text


# ============================================================
# ACRONYM / LETTER PRONUNCIATION
# ============================================================

LETTER_WORDS = {

    'A': 'yeh',
    'B': 'bee',
    'C': 'see',
    'D': 'dee',
    'E': 'ee',
    'F': 'eff',
    'G': 'gee',
    'H': 'aitch',
    'I': 'ஐ',
    'J': 'jay',
    'K': 'kay',
    'L': 'el',
    'M': 'em',
    'N': 'en',
    'O': 'oh',
    'P': 'pee',
    'Q': 'cue',
    'R': 'ar',
    'S': 'ess',
    'T': 'tee',
    'U': 'you',
    'V': 'vee',
    'W': 'double you',
    'X': 'ex',
    'Y': 'why',
    'Z': 'zee'
}


# Important abbreviations.
# These are handled before ordinary word/number normalization.

ACRONYM_WORDS = {

    'AC': 'ay see',
    'AR': 'ay ar',
    'ID': 'eye dee',
    'OTP': 'o tee pee',
    'UPI': 'you pee eye',
    'TN': 'tee en',
    'TNSTC': 'tee en ess tee see',
    'MTC': 'em tee see',
    'GPS': 'gee pee ess',
    'ETA': 'ee tee ay',
    'ETD': 'ee tee dee',
    'PIN': 'pee eye en',
    'OMR': 'oh em ar',
    'ECR': 'ee see ar',
    'VR': 'vee ar',
    'BX': 'bee ex',
    'RS': 'rupees',
    'KM': 'கிலோமீட்டர்',
    'PM': 'pee em',
    'AM': 'ay em'
}


def spell_letters(value):

    return ' '.join(
        LETTER_WORDS[ch]
        for ch in value.upper()
        if ch in LETTER_WORDS
    )


# ============================================================
# PROTECTED ALPHANUMERIC IDENTIFIERS
#
# TN-09-CF-6723
# TN09BX4587
# TN 09 BX 4587
# ============================================================

def vehicle_or_plate_replacement(match):

    value = match.group(0)

    # Logical Indian vehicle registration groups:
    # TN09BX4587 / TN-09-CF-6723 / TN 09 BX 4587
    # -> [TN] [09] [BX/CF] [4587]
    m = re.fullmatch(
        r'([A-Za-z]{2})[-\s]?(\d{2})[-\s]?([A-Za-z]{1,3})[-\s]?(\d{4})',
        value.strip()
    )

    if not m:
        # Fallback: keep existing character/digit-wise behavior.
        parts = re.findall(r'[A-Za-z]+|\d+', value)
        spoken_parts = []
        for part in parts:
            if part.isalpha():
                spoken_parts.append(spell_letters(part))
            else:
                spoken_parts.append(' '.join(DIGIT_WORDS[d] for d in part))
        return ', '.join(spoken_parts) + ', '

    state, district, letters, serial = m.groups()

    return (
        spell_letters(state)
        + ', '
        + ' '.join(DIGIT_WORDS[d] for d in district)
        + ', '
        + spell_letters(letters)
        + ', '
        + ' '.join(DIGIT_WORDS[d] for d in serial)
        + ', '
    )


def normalize_vehicle_numbers(text):

    # Hyphenated / spaced vehicle numbers.
    #
    # TN-09-CF-6723
    # TN 09 CF 6723
    # TN09CF6723

    pattern = (
        r'(?<![A-Za-z0-9])'
        r'[A-Za-z]{2}'
        r'(?:[-\s]?\d{2})'
        r'(?:[-\s]?[A-Za-z]{1,3})'
        r'(?:[-\s]?\d{4})'
        r'(?![A-Za-z0-9])'
    )

    return re.sub(
        pattern,
        vehicle_or_plate_replacement,
        text
    )


# ============================================================
# PHONE NUMBERS
# ============================================================

def phone_number_replacement(match):

    number = re.sub(r'\D', '', match.group(0))

    if re.search(r'[\u0B80-\u0BFF]', match.group(0)):
        return ' '.join(TAMIL_DIGITS[d] for d in number)

    return ' '.join(PHONE_DIGIT_WORDS[d] for d in number)


def normalize_phone_numbers(text, is_tamil, is_tanglish=False):

    def standalone_phone_replacement(match):
        spoken = phone_number_replacement(match)
        return ', ' + spoken + ', '

    text = re.sub(
        r'(?<!\d)\d{10}(?!\d)',
        standalone_phone_replacement,
        text
    )

    tamil_phone_context = (
        r'((?:ஃபோன்|போன்|phone)\s*'
        r'(?:நம்பர்|number)\s*)'
        r'(\d[\d\s-]*\d)'
        r'(-?க்கு|க்கு)?'
    )

    def tamil_phone_context_replacement(match):

        prefix = match.group(1)
        digits = re.sub(r'\D', '', match.group(2))
        suffix = match.group(3) or ''

        if not (3 <= len(digits) <= 15):
            return match.group(0)

        digit_words = ' '.join(
            PHONE_DIGIT_WORDS[d]
            for d in digits
        )

        result = prefix.rstrip() + ', ' + digit_words

        if suffix:
            result += ', ' + suffix.lstrip('-')

        return result

    text = re.sub(
        tamil_phone_context,
        tamil_phone_context_replacement,
        text,
        flags=re.IGNORECASE
    )

    return text


# ============================================================
# BOOKING IDS / ALPHANUMERIC CODES
#
# SEE TN-09-CF-6723
# booking ID CBA48291
# CBA48291
# ============================================================

def booking_id_replacement(match):

    value = match.group(0)
    parts = re.findall(r'[A-Za-z]+|\d+', value)

    spoken_parts = []

    for part in parts:
        if part.isalpha():
            spoken_parts.append(spell_letters(part))
        else:
            spoken_parts.append(
                ' '.join(DIGIT_WORDS[d] for d in part)
            )

    # CBA48291 -> see bee ay, four eight two nine one
    return ', '.join(spoken_parts) + ', '


def normalize_booking_ids(text):

    # Alphanumeric booking code.
    #
    # CBA48291
    # ABC12345
    #
    # Do this before general numbers.

    text = re.sub(
        r'(?<![A-Za-z0-9])'
        r'(?=[A-Za-z]*\d)'
        r'[A-Za-z]{2,8}\d{2,10}'
        r'(?![A-Za-z0-9])',
        booking_id_replacement,
        text
    )

    return text


# ============================================================
# GENERIC SHORT ABBREVIATIONS
#
# Transportation/location names may contain abbreviations that are
# not known in advance, for example:
#   ARக்கு       -> ay arக்கு
#   RK Convention Center -> ar kay Convention Center
#   VR Mall     -> vee ar Mall
#
# Only 2-3 letter tokens written in ALL CAPS are treated as generic
# abbreviations. This is deliberately case-sensitive so normal words
# such as "The" are never changed.
# ============================================================

def normalize_generic_short_abbreviations(text):

    def replace_abbreviation(match):
        return spell_letters(match.group(1))

    # Allow a Tamil grammatical suffix immediately after the abbreviation.
    # The suffix is left untouched.
    text = re.sub(
        r'(?<![A-Za-z])([A-Z]{2,3})(?=(?:க்கு|களுக்கு|லிருந்து|ல|ல்|ஐ|இல்|உடன்)|\b)',
        replace_abbreviation,
        text
    )

    return text


# ============================================================
# ACRONYMS
# ============================================================

def normalize_acronyms(text):

    # Longest first.

    words = sorted(
        ACRONYM_WORDS.keys(),
        key=len,
        reverse=True
    )

    for acronym in words:

        replacement = ACRONYM_WORDS[acronym]

        # Acronym as a standalone word.
        text = re.sub(
            r'(?<![A-Za-z])'
            + re.escape(acronym)
            + r'(?![A-Za-z])',
            replacement,
            text
        )

    # Single-letter T before location names:
    #
    # T Nagar -> tee Nagar
    # T. Nagar -> tee Nagar
    #
    text = re.sub(
        r'(?<![A-Za-z])T\.?(?=\s+[A-Za-z])',
        'tee',
        text
    )

    # Other standalone single letters.

    def single_letter(match):

        return LETTER_WORDS[
            match.group(1).upper()
        ]

    text = re.sub(
        r'(?<![A-Za-z])([A-Za-z])\.(?=\s|$)',
        single_letter,
        text
    )

    return text


# ============================================================
# PAUSE AFTER ACRONYMS BEFORE TAMIL SUFFIXES
#
# ARக்கு -> ar, க்கு
# UPIல -> you pee eye, ல
# ACல -> ay see, ல
# ============================================================

def add_acronym_pause_before_tamil_suffix(text):

    acronym_sounds = (
        r'(?:'
        r'oh em ar|'
        r'ee see ar|'
        r'vee ar|'
        r'ay ar|'
        r'ay see|'
        r'you pee eye|'
        r'o tee pee|'
        r'tee en|'
        r'em tee see|'
        r'gee pee ess|'
        r'ee tee ay|'
        r'ee tee dee|'
        r'ar|'
        r'ay|'
        r'bee|'
        r'cee|'
        r'see|'
        r'dee|'
        r'pee|'
        r'eye|'
        r'em|'
        r'en|'
        r'oh|'
        r'cue|'
        r'zee'
        r')'
    )

    # Insert a pause before Tamil suffixes.

    text = re.sub(
        r'(?<![A-Za-z])(' + acronym_sounds + r')'
        r'(?=(?:க்கு|களுக்கு|லிருந்து|ல|ல்|ஐ|இல்|உடன்))',
        r'\1, ',
        text,
        flags=re.IGNORECASE
    )

    return text



# ============================================================
# SPACED IDENTIFIER LETTERS
# ============================================================

def normalize_spaced_identifier_letters(text):
    """
    Handle already separated identifier letters inside critical contexts:

      booking ID is C B A four eight two one
        -> booking ID is see bee ay four eight two one

      vehicle number T N zero nine C F ...
        -> vehicle number tee en zero nine see eff ...
    """

    context_pattern = re.compile(
        r'\b(?:booking\s+(?:ID|eye\s+dee)|vehicle(?:\s+number)?|booking\s+number)\b'
        r'[^.!?]*',
        flags=re.IGNORECASE
    )

    def normalize_context(match):
        segment = match.group(0)

        # Convert only standalone UPPERCASE letters. Do not touch ordinary
        # words or the English pronoun "I" outside these ID contexts.
        return re.sub(
            r'(?<![A-Za-z])([A-HJ-Z])(?=\s|[,;:]|$)',
            lambda m: LETTER_WORDS[m.group(1)],
            segment
        )

    return context_pattern.sub(normalize_context, text)


# ============================================================
# MAIN NORMALIZATION
# ============================================================

def normalize_taxi_text(text):

    text = text.strip()

    has_tamil_script = bool(
        re.search(r'[\u0B80-\u0BFF]', text)
    )

    lower_text = text.lower()

    tanglish_detected = any(
        re.search(
            r'(?<![A-Za-z])'
            + re.escape(word)
            + r'(?![A-Za-z])',
            lower_text
        )
        for word in TANGLISH_WORDS
        if len(word) >= 3
        and TANGLISH_WORDS[word] != word
        and re.search(
            r'[\u0B80-\u0BFF]',
            TANGLISH_WORDS[word]
        )
    )

    is_tanglish = (
        not has_tamil_script
        and tanglish_detected
    )

    is_tamil = (
        has_tamil_script
        or is_tanglish
    )

    # ========================================================
    # CRITICAL EARLY NORMALIZATION
    # Dates, mixed Tamil times, currency and pre-spelled IDs must
    # be consumed before generic acronyms and general numbers.
    # ========================================================

    text = normalize_conversational_dates(text, has_tamil_script)
    text = normalize_tamil_mixed_times(text, has_tamil_script)
    text = normalize_currency_early(text, has_tamil_script)
    text = normalize_prespelled_identifier_digits(text)

    # ========================================================
    # TANGLISH
    # ========================================================

    if is_tamil:
        text = normalize_tanglish_words(text)
        text = text.replace('டாக்ஸி', 'taksi')

    # Booking ID label boundary BEFORE acronym/pronunciation expansion.
    # Existing downstream mappings are preserved; only the comma survives.
    text = re.sub(
        r'\b(Booking\s+ID)\b\s*,?\s*',
        r'\1, ',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # PROTECT / NORMALIZE VEHICLE NUMBERS FIRST
    # ========================================================

    text = normalize_vehicle_numbers(text)

    # ========================================================
    # PHONE NUMBERS
    # ========================================================

    text = normalize_phone_numbers(
        text,
        is_tamil,
        is_tanglish
    )

    # ========================================================
    # BOOKING IDS
    # ========================================================

    text = normalize_booking_ids(text)
    text = normalize_spaced_identifier_letters(text)

    # ========================================================
    # OTP
    # ========================================================

    def otp_replacement(match):

        prefix = match.group(1)
        digits = match.group(2)

        if has_tamil_script:

            digit_words = ' '.join(
                TAMIL_DIGITS[d]
                for d in digits
            )

        else:

            digit_words = ' '.join(
                DIGIT_WORDS[d]
                for d in digits
            )

        return prefix + digit_words

    text = re.sub(
        r'(\bOTP\b[^0-9]{0,100}?)(\d{4})\b',
        otp_replacement,
        text,
        flags=re.IGNORECASE
    )

    # OTP itself.

    text = re.sub(
        r'\bOTP\b',
        'o tee pee',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # TIME
    # ========================================================

    if has_tamil_script:

        def tamil_time_ampm(match):

            hour = int(match.group(1))
            minute = int(match.group(2))
            ampm = match.group(3)

            return tamil_time_words(
                hour,
                minute,
                ampm
            )

        text = re.sub(
            r'(?<!\d)(\d{1,2}):(\d{2})\s*'
            r'(AM|PM)(?:-?க்கு)?'
            r'(?=\s|[.,!?]|$)',
            tamil_time_ampm,
            text,
            flags=re.IGNORECASE
        )

        def tamil_plain_time(match):

            return tamil_time_words(
                int(match.group(1)),
                int(match.group(2))
            )

        text = re.sub(
            r'\b(\d{1,2}):(\d{2})\b(?:\s*மணிக்கு)?',
            tamil_plain_time,
            text
        )

        text = re.sub(
            r'(\d{1,2})\s*மணிக்கு'
            r'(?=\s|[.,!?]|$)',
            lambda m:
                tamil_number_to_words(
                    int(m.group(1))
                ) + ' மணிக்கு',
            text
        )

        text = re.sub(
            r'(\d{1,2})\s*மணி'
            r'(?=\s|[.,!?]|$)',
            lambda m:
                tamil_number_to_words(
                    int(m.group(1))
                ) + ' மணிக்கு',
            text
        )

        text = re.sub(
            r'(\d{1,2})\s+வரும்'
            r'(?=\s|[.,!?]|$)',
            lambda m:
                tamil_number_to_words(
                    int(m.group(1))
                ) + ' மணிக்கு வரும்',
            text
        )

    else:

        def english_time(match):

            hour = int(match.group(1))
            minute = int(match.group(2))
            ampm = match.group(3).lower()

            if ampm == 'pm':
                ampm_word = 'pee em'
            else:
                ampm_word = 'ay em'

            return (
                number_to_words(hour)
                + ' '
                + number_to_words(minute)
                + ', '
                + ampm_word
            )

        text = re.sub(
            r'\b(\d{1,2}):(\d{2})\s*'
            r'(AM|PM)\b',
            english_time,
            text,
            flags=re.IGNORECASE
        )

        text = re.sub(
            r'\b(\d{1,2}):(\d{2})\b',
            lambda m:
                number_to_words(int(m.group(1)))
                + ' '
                + number_to_words(int(m.group(2))),
            text
        )


    # ========================================================
    # ATTACHED AM / PM IN TANGLISH
    # ========================================================
    # AMku / PMku must be pronounced as separate letters.
    # AMku -> ay em, ku
    # PMku -> pee em, ku

    text = re.sub(
        r'\bAM(?=[- ]?ku\b)',
        'ay em,',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\bPM(?=[- ]?ku\b)',
        'pee em,',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # GENERIC SHORT ABBREVIATIONS
    # ========================================================

    text = normalize_generic_short_abbreviations(text)

    # ========================================================
    # ACRONYMS
    # ========================================================

    text = normalize_acronyms(text)

    # Semantic boundary after structured labels/acronyms.
    text = re.sub(
        r'\b(o\s+tee\s+pee)\b(?!\s*,)',
        r'\1,',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # TRANSPORT ACRONYM SEMANTIC BOUNDARIES
    # ========================================================

    # Add one short prosodic boundary after the fully expanded acronym.
    # Never put commas between the individual spoken letters.
    _transport_acronym_sounds = (
        r'(?:'
        r'ee tee ay|'        # ETA
        r'gee pee ess|'      # GPS
        r'you pee eye|'      # UPI
        r'oh em ar|'         # OMR
        r'ee see ar|'        # ECR
        r'see em bee tee|'   # CMBT, if mapped this way upstream
        r'em tee see|'       # MTC
        r'ay see|'           # AC
        r'ess you vee'       # SUV, if mapped letter-wise
        r')'
    )

    text = re.sub(
        r'(?<![A-Za-z])(' + _transport_acronym_sounds + r')(?![A-Za-z,])',
        r'\1,',
        text,
        flags=re.IGNORECASE
    )


    # ========================================================
    # ENGLISH SEVEN
    # ========================================================

    if not is_tamil:

        text = re.sub(
            r'\bseven\b',
            'sevven',
            text,
            flags=re.IGNORECASE
        )

    # ========================================================
    # ID
    # ========================================================

    if is_tamil:

        text = re.sub(
            r'\bID\b',
            'ஐ டீ',
            text,
            flags=re.IGNORECASE
        )

    else:

        text = re.sub(
            r'\bID\b',
            'eye dee',
            text,
            flags=re.IGNORECASE
        )

    # ========================================================
    # RUPEES / RS
    # ========================================================

    def rupee_replacement(match):

        amount = match.group(1).replace(',', '')
        value = int(amount)

        if has_tamil_script:

            return (
                tamil_number_to_words(value)
                + ' ரூபாய்'
            )

        return (
            number_to_words(value)
            + ' rupees'
        )

    # ₹725

    text = re.sub(
        r'₹\s*([\d,]+)',
        rupee_replacement,
        text
    )

    # Rs 725 / Rs.725

    text = re.sub(
        r'\bRs\.?\s*([\d,]+)',
        rupee_replacement,
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # PAISE
    #
    # fifty paise
    # 50 paise
    # ========================================================

    def paise_replacement(match):

        value = int(match.group(1))

        if has_tamil_script:

            return (
                tamil_number_to_words(value)
                + ' பைசா'
            )

        return (
            number_to_words(value)
            + ' paise'
        )

    text = re.sub(
        r'\b(\d+)\s+paise\b',
        paise_replacement,
        text,
        flags=re.IGNORECASE
    )

    # English:
    #
    # "four hundred and fifty rupees.fifty"
    #
    # -> four hundred and fifty rupees, fifty paise

    text = re.sub(
        r'\brupees\.(\w+)\b',
        r'rupees, \1 paise',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # KILOMETERS
    # ========================================================

    def kilometer_replacement(match):

        value = int(match.group(1))

        if has_tamil_script:

            return (
                tamil_number_to_words(value)
                + '\u00A0கிலோமீட்டர்'
            )

        return (
            number_to_words(value)
            + ' கிலோமீட்டர்'
        )

    text = re.sub(
        r'\b(\d+)\s+kilometers?\b',
        kilometer_replacement,
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\b(\d+)\s+kilomeeters?\b',
        kilometer_replacement,
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\b(\d+)\s+kilometres?\b',
        kilometer_replacement,
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\b(\d+)\s+kilomeetres?\b',
        kilometer_replacement,
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # MINUTES
    # ========================================================

    def minutes_replacement(match):

        value = int(match.group(1))

        if has_tamil_script:

            return (
                tamil_number_to_words(value)
                + ' நிமிடத்தில்'
            )

        return (
            number_to_words(value)
            + ' minits'
        )

    text = re.sub(
        r'\b(\d+)\s+minutes?\b',
        minutes_replacement,
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # CHENNAI CENTRAL
    # ========================================================

    text = re.sub(
        r'\bChennai\s+Central(?=-|[\s.,!?]|$)',
        'சென்னை சென்ட்ரல்',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # TAMIL ORDINALS
    # ========================================================

    if has_tamil_script:

        text = re.sub(
            r'(\d+)(வது|ஆம்|ஆவது)'
            r'(?=\s|[.,!?]|$)',
            lambda m:
                tamil_ordinal_to_words(
                    int(m.group(1))
                ),
            text
        )

    # ========================================================
    # GENERAL NUMBERS
    #
    # IMPORTANT:
    # This is deliberately near the END.
    #
    # Vehicle numbers, phone numbers, OTPs and booking IDs
    # have already been converted character-by-character.
    # ========================================================

    def general_number_replacement(match):

        value = int(match.group(0))

        if has_tamil_script:
            return tamil_number_to_words(value)

        return number_to_words(value)

    text = re.sub(
        r'\b\d+\b',
        general_number_replacement,
        text
    )

    # ========================================================
    # FINAL SEVEN
    # ========================================================

    if not is_tamil:

        text = re.sub(
            r'\bseven\b',
            'sevven',
            text,
            flags=re.IGNORECASE
        )

    # ========================================================
    # PRONUNCIATION FIXES
    # ========================================================

    # NOTIFY

    text = re.sub(
        r'\bnotify\b',
        'noe tih fy',
        text,
        flags=re.IGNORECASE
    )

    # AIR (standalone)

    text = re.sub(
        r'\bair\b',
        'air',
        text,
        flags=re.IGNORECASE
    )

    # VELACHERY
    # Keep the Chennai place name from being stretched as "Velaachery".

    # Keep Velachery intact; DhVaani pronounces it naturally.
    # No replacement is applied here.

    # ARRIVE

    text = re.sub(
        r'\barriving\b',
        'uh riving',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\barrived\b',
        'uh rived',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\barrives\b',
        'ar rives',
        text,
        flags=re.IGNORECASE
    )

    # ARRIVE
    # Keep natural pronunciation when 'arrive' is sentence-final.
    # Use the hardcoded pronunciation only when more speech follows.
    #
    # Examples:
    #   How long will the driver take to arrive?
    #       -> arrive
    #
    #   The driver will arrive in five minutes.
    #       -> uh rive
    text = re.sub(
        r'\barrive\b(?=\s+\S)',
        'uh rive',
        text,
        flags=re.IGNORECASE
    )

    # ARRIVAL

    text = re.sub(
        r'\barrival\b',
        'arriival',
        text,
        flags=re.IGNORECASE
    )

    # AIRPORT

    text = re.sub(
        r'\bairport\b',
        'yerport',
        text,
        flags=re.IGNORECASE
    )

    # PAISE
    # Keep the standard word "paise" for clear TTS pronunciation.

    # BOOKING

    if not is_tamil:
        text = re.sub(
            r'\bbooking\b',
            'booking',
            text,
            flags=re.IGNORECASE
        )

    # CAB
    # Requested DhVaani pronunciation: cab -> kab in all languages.

    text = re.sub(
        r'\b(?:cab|kaab)\b',
        'cab',
        text,
        flags=re.IGNORECASE
    )

    # ROUTE

    if not is_tamil:
        text = re.sub(
            r'\broute\b',
            'root',
            text,
            flags=re.IGNORECASE
        )

    # MINUTES

    text = re.sub(
        r'\bminutes?\b',
        'minits',
        text,
        flags=re.IGNORECASE
    )

    # TOTAL

    text = re.sub(
        r'\btotals\b',
        'toe tals',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\btotal\b',
        'toe tal',
        text,
        flags=re.IGNORECASE
    )

    # FINAL

    text = re.sub(
        r'\bfinal\b',
        'fye nal',
        text,
        flags=re.IGNORECASE
    )

    # FARE
    # Requested DhVaani pronunciation: fare -> fair in all languages.

    text = re.sub(
        r'\bfares\b',
        'fairs',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\bfare\b',
        'fair',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # PHONE / NUMBER WORDS
    # ========================================================

    if has_tamil_script:

        text = re.sub(
            r'\bphone\b',
            'ஃபோன்',
            text,
            flags=re.IGNORECASE
        )

        text = re.sub(
            r'\bnumber\b',
            'நம்பர்',
            text,
            flags=re.IGNORECASE
        )

    # ========================================================
    # TAMIL KM JOIN
    # ========================================================

    if has_tamil_script:

        tamil_number_pattern = (
            r'(பூஜ்ஜியம்|ஒன்று|இரண்டு|மூன்று|'
            r'நான்கு|ஐந்து|ஆறு|ஏழு|எட்டு|ஒன்பது|'
            r'பத்து|பதினொன்று|பன்னிரண்டு|பதின்மூன்று|'
            r'பதினான்கு|பதினைந்து|பதினாறு|பதினேழு|'
            r'பதினெட்டு|பத்தொன்பது|இருபது|முப்பது|'
            r'நாற்பது|ஐம்பது|அறுபது|எழுபது|எண்பது|'
            r'தொண்ணூறு|இருபத்தி\s+\w+|முப்பத்தி\s+\w+|'
            r'நாற்பத்தி\s+\w+|ஐம்பத்தி\s+\w+|'
            r'அறுபத்தி\s+\w+|எழுபத்தி\s+\w+|'
            r'எண்பத்தி\s+\w+|தொண்ணூற்று\s+\w+|'
            r'நூறு|இருநூறு|முந்நூறு|நானூறு|'
            r'ஐந்நூறு|அறுநூறு|எழுநூறு|எண்ணூறு|'
            r'தொள்ளாயிரம்|ஆறு\s+நூற்று\s+\w+)'
        )

        text = re.sub(
            '('
            + tamil_number_pattern
            + r')\s+'
            + r'(கிலோமீட்டர்|கிலோமீட்டர்கள்)\b',
            lambda m: m.group(1) + '\u00A0' + m.group(2),
            text
        )

    # ========================================================
    # RUPEE PACING
    # ========================================================

    if has_tamil_script:

        text = re.sub(
            r'(ரூபாய்)(?=\s|[.!?,]|$)',
            r'\1,',
            text
        )

    else:

        text = re.sub(
            r'(rupees)\b(?!\s*,)',
            r'\1,',
            text,
            flags=re.IGNORECASE
        )

    text = re.sub(
        r',([.!?])',
        r'\1',
        text
    )

    # ========================================================
    # OTP PACING
    # ========================================================

    digit_words_english = (
        r'(?:zero|one|two|three|four|five|'
        r'sevven|eight|nine)'
    )

    text = re.sub(
        r'(o tee pee\s+'
        + digit_words_english
        + r'(?:\s+'
        + digit_words_english
        + r'){3})',
        r'\1,',
        text,
        flags=re.IGNORECASE
    )

    # ========================================================
    # ACRONYM PAUSE
    # ========================================================

    text = add_acronym_pause_before_tamil_suffix(text)

    # ========================================================
    # FINAL TANGLISH NORMALIZATION
    # ========================================================

    # Tanglish words were already normalized near the beginning.
    # Do not run that dictionary again after acronym expansion because
    # spoken acronym pieces such as "en" in "tee en" must remain English.

    # ========================================================
    # NATURAL ENTITY PACING
    # ========================================================
    # Do NOT insert commas after every digit or letter.
    # Use punctuation only at meaningful boundaries.

    _eng_digit = r'(?:zero|one|two|three|four|five|six|sevven|eight|nine)'
    _eng_letter = (
        r'(?:ay|bee|see|dee|ee|eff|gee|aitch|eye|jay|kay|el|em|en|oh|'
        r'pee|cue|ar|ess|tee|you|vee|double\s+you|ex|why|zee)'
    )
    _tamil_suffix = (
        r'(?:க்காக|க்கான|க்கு|களுக்கு|லிருந்து|இருந்து|ல|ல்|இல்|ஐ|உடன்)'
    )

    text = re.sub(
        r'\b(phone\s+number)\b\s*,?\s*',
        r'\1, ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'(ஃபோன்\s+நம்பர்|போன்\s+நம்பர்)\s*,?\s*',
        r'\1, ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\b(book\s+ing|booking)\s*,?\s*(eye\s+dee)\b\s*,?\s*(?:is\s+)?',
        r'\1 eye dee, ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\b(o\s+tee\s+pee)\b\s*,?\s*(?:is\s+)?',
        r'\1, ',
        text,
        flags=re.IGNORECASE
    )

    # One boundary before an attached Tamil suffix.
    text = re.sub(
        r'\b(' + _eng_digit + r'|' + _eng_letter + r')\s*'
        r'(' + _tamil_suffix + r')',
        r'\1, \2',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r'\s*,\s*', ', ', text)
    text = re.sub(r'(?:,\s*){2,}', ', ', text)

    # ========================================================
    # WRITTEN TAMIL HUNDRED LINKERS
    # ========================================================

    text = normalize_written_tamil_hundreds(text)

    # ========================================================
    # STRUCTURED ENTITY PROSODY CLEANUP
    # ========================================================

    # If a standalone spoken phone sequence follows a normal word,
    # ensure the word and complete phone entity are separated naturally.
    _digit_word = r'(?:zero|one|two|three|four|five|six|sevven|eight|nine)'
    _phone_spoken = r'(?:' + _digit_word + r'\s+){9}' + _digit_word

    # Normalize commas surrounding complete 10-digit phone entities.
    text = re.sub(
        r'\s*,?\s*(' + _phone_spoken + r')\s*,?\s*',
        r', \1, ',
        text,
        flags=re.IGNORECASE
    )

    # Avoid a leading comma if the sentence starts with the phone entity.
    text = re.sub(r'^\s*,\s*', '', text)

    # Narrow location boundary:
    # "at Velachery is ..." -> "at Velachery, is ..."
    # Do NOT split "Velachery and Guindy Metro".
    text = re.sub(
        r'\b(Velachery)\s+(?=is\b)',
        r'\1, ',
        text,
        flags=re.IGNORECASE
    )

    # Keep only one comma at semantic boundaries.
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r',\s*,+', ',', text)
    text = re.sub(r'(?:,\s*){2,}', ', ', text)

    # No comma directly before sentence-ending punctuation.
    text = re.sub(r',\s*([.!?])', r'\1', text)

    # Normal comma spacing.
    text = re.sub(r',(?=\S)', ', ', text)
    text = re.sub(r'\s{2,}', ' ', text)

    # ========================================================
    # FINAL CLEANUP
    # ========================================================

    # --------------------------------------------------------
    # STRICT REQUESTED DHVAANI PRONUNCIATION CORRECTIONS
    # --------------------------------------------------------

    # Tanglish andru -> Tamil அன்று
    text = re.sub(
        r'\bandru\b',
        'அன்று',
        text,
        flags=re.IGNORECASE
    )

    # taxi / டாஸ்சி -> taksi
    text = re.sub(
        r'\btaxi\b',
        'taksi',
        text,
        flags=re.IGNORECASE
    )
    text = text.replace('டாஸ்சி', 'taksi')

    # cab must always be kab
    text = re.sub(
        r'\b(?:cab|kaab)\b',
        'cab',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
       r'\bkms?\b',
      'கிலோமீட்டர்',
       text,
       flags=re.IGNORECASE
   )

    # fare must always be fair
    text = re.sub(
        r'\bfare\b',
        'fair',
        text,
        flags=re.IGNORECASE
    )

    # Chennai Central must always use the Tamil pronunciation form.
    text = re.sub(
        r'\b(?:Chennai\s+Central|Cennei\s+Sentrel)\b',
        'சென்னை சென்ட்ரல்',
        text,
        flags=re.IGNORECASE
    )

    # Global DhVaani pronunciation fixes for standalone number words.
    text = re.sub(r'\bfive\b', 'fyeve', text, flags=re.IGNORECASE)
    text = re.sub(r'\beight\b', 'ait', text, flags=re.IGNORECASE)

    text = re.sub(r',\s*,+', ',', text)

    text = re.sub(
        r'[ \t]+',
        ' ',
        text
    ).strip()

    return text


# ============================================================
# ENTITY-AWARE CHUNKER
# ============================================================
# CHUNKING FIX
# The normalizer intentionally inserts commas for DhVaani prosody.  Those
# commas are NOT treated as automatic audio boundaries here.  Structured and
# semantic entities are detected first, then ordinary chunk sizing is applied
# around them.  A protected entity may exceed max_words rather than being
# split internally.


def _clean_chunk_text(value):
    value = re.sub(r'\s*,\s*', ', ', value)
    value = re.sub(r'\s+([.!?])', r'\1', value)
    value = re.sub(r'\s{2,}', ' ', value)
    return value.strip(' ,')


# TIME ENTITY PROTECTION / DATE ENTITY PROTECTION / NUMBER ENTITY PROTECTION
_ENG_NUMBER_ATOM = (
    r'(?:zero|one|two|three|four|fyeve|five|six|sevven|seven|ait|eight|nine|'
    r'ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|'
    r'nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|'
    r'hundred|thousand|and)'
)
_ENG_NUMBER_PHRASE = r'(?:' + _ENG_NUMBER_ATOM + r')(?:\s+' + _ENG_NUMBER_ATOM + r')*'
_ENG_DIGIT_WORD = r'(?:zero|one|two|three|four|fyeve|five|six|sevven|seven|ait|eight|nine)'
_ENG_DIGIT_SEQUENCE = r'(?:' + _ENG_DIGIT_WORD + r')(?:\s+' + _ENG_DIGIT_WORD + r'){2,14}'
_ENG_LETTER_WORD = (
    r'(?:ay|bee|see|dee|ee|eff|gee|aitch|eye|jay|kay|el|em|en|oh|pee|cue|ar|'
    r'ess|tee|you|vee|double\s+you|ex|why|zee)'
)
_ENG_ACRONYM = r'(?:' + _ENG_LETTER_WORD + r')(?:\s+' + _ENG_LETTER_WORD + r')+'
_TAMIL_SUFFIX = r'(?:க்காக|க்கான|க்கு|களுக்கு|லிருந்து|இருந்து|ல|ல்|இல்|ஐ|உடன்)'
_TANGLISH_SUFFIX = r'(?:ku|kku|la)'

_TAMIL_NUMBER_WORD = (
    r'(?:பூஜ்ஜியம்|ஒன்று|இரண்டு|மூன்று|நான்கு|ஐந்து|ஆறு|ஏழு|எட்டு|ஒன்பது|'
    r'பத்து|பதினொன்று|பன்னிரண்டு|பதின்மூன்று|பதினான்கு|பதினைந்து|பதினாறு|'
    r'பதினேழு|பதினெட்டு|பத்தொன்பது|இருபது|இருபத்தி|முப்பது|முப்பத்தி|'
    r'நாற்பது|நாற்பத்தி|ஐம்பது|ஐம்பத்தி|அறுபது|அறுபத்தி|எழுபது|எழுபத்தி|'
    r'எண்பது|எண்பத்தி|தொண்ணூறு|தொண்ணூற்று|நூறு|நூற்று|இருநூறு|இருநூற்று|'
    r'முந்நூறு|முந்நூற்று|நானூறு|நானூற்று|ஐந்நூறு|ஐந்நூற்று|அறுநூறு|'
    r'அறுநூற்று|எழுநூறு|எழுநூற்று|எண்ணூறு|எண்ணூற்று|தொள்ளாயிரம்|'
    r'தொள்ளாயிரத்து|ஆயிரம்|ஆயிரத்து)'
)
_TAMIL_NUMBER_PHRASE = r'(?:' + _TAMIL_NUMBER_WORD + r')(?:\s+' + _TAMIL_NUMBER_WORD + r')*'

_MONTH = (
    r'(?:January|February|March|April|May|June|July|August|September|October|'
    r'November|December|ஜனவரி|பிப்ரவரி|மார்ச்|ஏப்ரல்|மே|ஜூன்|ஜூலை|ஆகஸ்ட்|'
    r'செப்டம்பர்|அக்டோபர்|நவம்பர்|டிசம்பர்)'
)


def _protected_entity_patterns():
    """Priority-ordered normalized-text entities that must stay intact."""
    return [
        # DATE ENTITY PROTECTION
        r'\b' + _MONTH + r'\s+' + _ENG_NUMBER_PHRASE + r'\s*,\s*' + _ENG_NUMBER_PHRASE + r'\b',
        _MONTH + r'\s+' + _TAMIL_NUMBER_PHRASE + r'\s*,\s*' + _TAMIL_NUMBER_PHRASE,

        # TIME ENTITY PROTECTION
        # hour + minute + AM/PM, including the prosody comma before suffix.
        r'\b' + _ENG_NUMBER_PHRASE + r'\s+(?:ay\s+em|pee\s+em)(?:\s*,\s*(?:ku|kku|' + _TAMIL_SUFFIX + r'))?\b',
        r'(?:காலை|மதியம்|மாலை|இரவு)\s+(?:' + _TAMIL_NUMBER_PHRASE + r'|' + _ENG_NUMBER_PHRASE + r')'
        r'(?:\s+(?:ay\s+em|pee\s+em))?(?:\s*,\s*(?:ku|kku|' + _TAMIL_SUFFIX + r'))?',

        # CURRENCY / PAISE ENTITY PROTECTION
        r'\b' + _ENG_NUMBER_PHRASE + r'\s+rupees\b(?:\s*,\s*' + _ENG_NUMBER_PHRASE + r'\s+paise\b)?',
        r'\b' + _ENG_NUMBER_PHRASE + r'\s+paise\b',
        _TAMIL_NUMBER_PHRASE + r'\s+ரூபாய்\b',
        _TAMIL_NUMBER_PHRASE + r'\s+பைசா\b',

        # MINUTES / DISTANCE ENTITY PROTECTION
        r'\b' + _ENG_NUMBER_PHRASE + r'\s+minits\b(?:[- ]?(?:la|ku|kku))?',
        r'\b' + _ENG_NUMBER_PHRASE + r'\s+kil\s+uh\s+meters?\b(?:[- ]?(?:la|ku|kku))?',
        _TAMIL_NUMBER_PHRASE + r'(?:\u00a0|\s+)கிலோமீட்டர்(?:கள்)?(?:' + _TAMIL_SUFFIX + r')?',
        _TAMIL_NUMBER_PHRASE + r'\s+நிமிட(?:ம்|த்தில்|ங்கள்)?(?:' + _TAMIL_SUFFIX + r')?',

        # OTP ENTITY PROTECTION: label + complete digit sequence.
        r'\b(?:o\s+tee\s+pee)\s*,?\s*(?:is\s+)?' + _ENG_DIGIT_SEQUENCE + r'\b',

        # PHONE ENTITY PROTECTION: preserve the complete spoken digit sequence
        # with its nearby phone-number label when present.
        r'\b(?:phone\s+number)\s*,?\s*(?:is\s*,?\s*)?' + _ENG_DIGIT_SEQUENCE + r'\b',
        r'(?:ஃபோன்|போன்)\s+நம்பர்\s*,?\s*' + _ENG_DIGIT_SEQUENCE + r'\b',

        # VEHICLE REGISTRATION ENTITY PROTECTION after existing normalizer:
        # tee en, zero nine, ay bee, six ait six six
        r'\b' + _ENG_ACRONYM + r'\s*,\s*(?:' + _ENG_DIGIT_WORD + r')(?:\s+' + _ENG_DIGIT_WORD + r'){1,2}'
        + r'\s*,\s*' + _ENG_ACRONYM + r'\s*,\s*(?:' + _ENG_DIGIT_WORD + r')(?:\s+' + _ENG_DIGIT_WORD + r'){3,5}\b',

        # BOOKING ID ENTITY PROTECTION: expanded letters + expanded digits.
        r'\b(?:bouking|booking|book\s+ing)\s+(?:eye\s+dee|ஐ\s+டீ)\s*,?\s*(?:is\s+)?'
        + _ENG_ACRONYM + r'\s*,?\s*' + _ENG_DIGIT_SEQUENCE + r'\b',
        r'\b' + _ENG_ACRONYM + r'\s*,\s*' + _ENG_DIGIT_SEQUENCE + r'\b',

        # ACRONYM + TAMIL/Tanglish suffix, including AM/PM suffix forms.
        r'\b' + _ENG_ACRONYM + r'\s*,\s*(?:' + _TAMIL_SUFFIX + r'|' + _TANGLISH_SUFFIX + r')(?=\s|[.,!?]|$)',

        # Complete standalone digit sequences used by identifiers.
        r'\b' + _ENG_DIGIT_SEQUENCE + r'\b',

        # Acronyms must never be split internally.
        r'\b' + _ENG_ACRONYM + r'\b',

        # General spoken numerical phrases are protected last, so larger
        # structured entities above always win first.
        r'\b' + _ENG_NUMBER_PHRASE + r'\b',
        _TAMIL_NUMBER_PHRASE,
    ]


def _collect_protected_spans(text):
    """Return non-overlapping protected spans using entity priority."""
    spans = []

    def overlaps(start, end):
        return any(start < old_end and end > old_start for old_start, old_end in spans)

    for pattern in _protected_entity_patterns():
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            if not overlaps(match.start(), match.end()):
                spans.append((match.start(), match.end()))

    spans.sort()
    return spans


def _entity_aware_units(text):
    """Tokenize text while keeping every protected semantic entity atomic."""
    spans = _collect_protected_spans(text)
    units = []
    cursor = 0

    for start, end in spans:
        if cursor < start:
            units.extend(re.findall(r'\S+', text[cursor:start]))
        units.append(text[start:end].strip())
        cursor = end

    if cursor < len(text):
        units.extend(re.findall(r'\S+', text[cursor:]))

    # Keep sentence punctuation attached to the preceding unit.  Prosody
    # commas inside protected entities have already been preserved above.
    compact = []
    for unit in units:
        if re.fullmatch(r'[.!?]+', unit) and compact:
            compact[-1] += unit
        else:
            compact.append(unit)

    return [unit for unit in compact if unit]


def chunk_normalized_text(text, max_words=7):
    """Semantic/entity-aware chunking; never split a protected entity."""
    text = _clean_chunk_text(text)
    if not text:
        return []

    units = _entity_aware_units(text)
    chunks = []
    current = []
    current_words = 0

    def flush():
        nonlocal current, current_words
        if current:
            chunks.append(_clean_chunk_text(' '.join(current)))
            current = []
            current_words = 0

    for unit in units:
        unit = unit.strip()
        if not unit:
            continue

        words = len(re.findall(r'\S+', unit))
        lower = unit.lower().strip(' ,')

        if current and re.search(r'[.!?]$', current[-1]):
            flush()

        # Prosodic boundary for conjunction + following cab/taxi action.
        # This is semantic behavior, not a hard-coded full sentence.
        if current and lower in {'kab', 'taksi'} and current[-1].strip().lower() in {'மற்றும்', 'and'}:
            flush()

        # Apply normal chunk size only BETWEEN units.  A protected unit is
        # atomic, so if it is itself longer than max_words it is still kept.
        if current and current_words + words > max_words:
            flush()

        current.append(unit)
        current_words += words

        if re.search(r'[.!?]$', unit):
            flush()

    flush()
    return chunks


def normalize_and_chunk(text, max_words=7):
    normalized = normalize_taxi_text(text)
    return normalized, chunk_normalized_text(normalized, max_words=max_words)


# ============================================================
# TEST CASES
# ============================================================

if __name__ == "__main__":

    tests = [

        "Please notify me when the cab arrives.",

        "Your cab is five minutes away from T, Nagar.",

        "Driver vehicle number TN-ten-BX-four thousand five hundred and eighty sevven.",

        "TN-09-CF-6723",

        "TN 09 CF 6723",

        "TN09CF6723",

        "SEE TN-09-CF-6723, ஃபோன் நம்பர் 12345க்கு driver call பண்ணுவார்.",

        "Your OTP is 4821.",

        "Your booking ID is CBA48291.",

        "Your booking eye dee is see ay bee48291.",

        "AC available in the cab.",

        "இந்த cabல AC இருக்கா",

        "UPIல payment பண்ண முடியுமா?",

        "OMRக்கு cab கிடைக்குமா?",

        "ARக்கு cab கிடைக்குமா?",

        "நான் Velacheryல இருந்து airport போகணும்.",

        "T Nagarல இருந்து Centralக்கு cab வேணும்.",

        "உங்கள் fare 660 ரூபாய் ஆகும்.",

        "உங்கள் fair நானூறு ஐம்பது ரூபாய் ஆகும்.",

        "The estimated fare is Rs 725.",

        "Your fare is 450 rupees.50.",

        "Your cab is 5 minutes away.",

        "The route is 12 kilometers.",

        "6:30 PM",

        "6:30 PM-க்கு",

        "7 வரும்.",

        "உங்கள் cab 6:30 PM-க்கு வரும் மற்றும் fare ₹560 ஆகும்.",
        "Vehicle TN-09-CF-6723 காலை 8:20 மணிக்கு வரும்.",
        "Phone number 98400 12345க்கு driver call பண்ணுவார்.",
        "Ungal cab 6:30 PM-ku varum. Phone number 98400 12345.",
        "The estimated fare is Rs 725.",
        "The cab will arrive at the airport in 15 minutes.",
        "The route is 12 kilometers and costs 50 paise.",
        "18/09/2026 காலை 6:30 AMக்கு TN-09-CF-6723 vehicle OMRல passenger-ஐ pickup பண்ணும்.",
        "12/09/2026 காலை 6:30 AMக்கு cab வரும்.",
        "காலை 6:30 AMக்கு cab வரும்.",
        "6:30 AMக்கு cab வரும்.",
        "மாலை 6:30 PMக்கு cab வரும்.",
        "Your booking ID is CBA48291.",
        "Your booking eye dee is see ay bee48291.",
        "Your OTP is 4821.",
        "OMRக்கு cab கிடைக்குமா?",
        "ECRல pickup இருக்கா?",
        "ARக்கு cab கிடைக்குமா?",
        "VR Mallக்கு போகணும்.",
        "T Nagarல இருந்து Centralக்கு cab வேணும்.",
        "The estimated fare is Rs 725.",
        "Your fare is 450 rupees.50.",

        # CHUNKING FIX REGRESSION TESTS
        "Cab mathiyam 12:30 PMku varum.",
        "Ungaloda taxi maalai 5:45 PMku varum.",
        "Driver kaalai 10:30 AMku unga locationku varuvaaru.",
        "March 15, 2027 mathiyam 12:30 PMku unga ride scheduled aagirukku.",
        "Your fare is 650 rupees.",
        "Ungal cab 10 minutes-la varum.",
        "The cab is 2 kilometers away.",
        "Your OTP is 4821.",
        "Your phone number is 9876543210.",
        "Your booking ID is CBA48291.",
        "Vehicle number TN09AB6866.",
        "OMRக்கு UPIல PMக்கு AMku"
    ]

    for test in tests:

        normalized, chunks = normalize_and_chunk(test)
        print("\nINPUT : ", test)
        print("NORMALIZED: ", normalized)
        print("CHUNKS: ", chunks)



INPUT :  Please notify me when the cab arrives.
NORMALIZED:  Please noe tih fy me when the cab ar rives.
CHUNKS:  ['Please noe tih fy me when the', 'cab ar rives.']

INPUT :  Your cab is five minutes away from T, Nagar.
NORMALIZED:  Your cab is fyeve minits away from T, Nagar.
CHUNKS:  ['Your cab is fyeve minits away from', 'T, Nagar.']

INPUT :  Driver vehicle number TN-ten-BX-four thousand five hundred and eighty sevven.
NORMALIZED:  Driver vehicle number tee en-ten-bee ex-four thousand fyeve hundred and eighty sevven.
CHUNKS:  ['Driver vehicle number tee en - ten', '- bee ex -', 'four thousand fyeve hundred and eighty sevven.']

INPUT :  TN-09-CF-6723
NORMALIZED:  tee en, zero nine, see eff, six sevven two three,
CHUNKS:  ['tee en, zero nine, see eff, six sevven two three']

INPUT :  TN 09 CF 6723
NORMALIZED:  tee en, zero nine, see eff, six sevven two three,
CHUNKS:  ['tee en, zero nine, see eff, six sevven two three']

INPUT :  TN09CF6723
NORMALIZED:  tee en, zero nine, see eff, 

## Reference language routing


In [19]:
import re
import time

TAMIL_CHAR_RE = re.compile(r"[\u0B80-\u0BFF]")


def _clean_chunk(text):
    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def route_reference_clean(
    normalized_text,
    group="ENTITY"
):
    text = str(normalized_text)

    group = str(
        group
    ).upper().strip()

    if group == "ENGLISH":
        return "ENGLISH"

    if group in (
        "TAMIL",
        "TANGLISH"
    ):
        return "TAMIL"

    if TAMIL_CHAR_RE.search(text):
        return "TAMIL"

    return "ENGLISH"


def call_smart_chunker(
    text,
    language=None
):
    text = str(text).strip()

    if not text:
        return []

    function_names = [
        "smart_chunk_text_v4",
        "smart_chunk_text_v8",
        "smart_chunk_text_v7",
        "smart_chunk_text_v6",
        "smart_chunk_text_v5",
    ]

    fn = None

    for name in function_names:
        candidate = globals().get(name)

        if callable(candidate):
            fn = candidate
            break

    if fn is None:
        return [text]

    try:
        raw_chunks = fn(
            text,
            language=language
        )

    except TypeError:
        raw_chunks = fn(text)

    if raw_chunks is None:
        return [text]

    if isinstance(raw_chunks, str):
        raw_chunks = [raw_chunks]

    chunks = [
        _clean_chunk(chunk)
        for chunk in raw_chunks
        if str(chunk).strip()
    ]

    return chunks or [text]


def safe_semantic_chunks(
    text,
    language=None
):
    text = _clean_chunk(text)

    if not text:
        return []

    chunks = call_smart_chunker(
        text,
        language=language
    )

    return chunks or [text]


def english_safe_chunks(text):
    text = _clean_chunk(text)

    if not text:
        return []

    return safe_semantic_chunks(
        text,
        language="ENGLISH"
    )


# ============================================================
# MAIN REQUEST PREPARATION
# ============================================================

def prepare_movio_request(
    raw_text,
    group="ENTITY"
):
    raw = str(
        raw_text
    ).strip()

    group = str(
        group
    ).upper().strip()

    if not raw:
        raise ValueError(
            "raw_text is empty"
        )

    # --------------------------------------------------------
    # NORMALIZATION
    # YOUR NORMALIZER ONLY
    # --------------------------------------------------------

    t0 = time.perf_counter()

    normalized = normalize_taxi_text(
        raw
    )

    normalize_ms = (
        time.perf_counter()
        - t0
    ) * 1000.0

    # --------------------------------------------------------
    # ROUTING
    # --------------------------------------------------------

    t1 = time.perf_counter()

    route = route_reference_clean(
        normalized,
        group
    )

    routing_ms = (
        time.perf_counter()
        - t1
    ) * 1000.0

    # --------------------------------------------------------
    # CHUNKING
    # --------------------------------------------------------

    t2 = time.perf_counter()

    if route == "ENGLISH":

        chunks = english_safe_chunks(
            normalized
        )

    else:

        chunks = safe_semantic_chunks(
            normalized,
            language=group
        )

    if not chunks:
        chunks = [normalized]

    chunks = [
        _clean_chunk(chunk)
        for chunk in chunks
        if _clean_chunk(chunk)
    ]

    chunk_ms = (
        time.perf_counter()
        - t2
    ) * 1000.0

    return {
        "raw": raw,
        "group": group,
        "normalized": normalized,
        "route": route,
        "chunks": chunks,
        "normalize_ms": normalize_ms,
        "routing_ms": routing_ms,
        "chunk_ms": chunk_ms,
    }


print("✓ prepare_movio_request is now defined")

✓ prepare_movio_request is now defined


## Semantic micro-chunking


In [20]:
# ============================================================
# CELL 9 — MOVIO SEMANTIC MICRO-CHUNKER V6
# English + Tamil + Tanglish + entity-aware
#
# IMPORTANT:
# - This is NOT "first N words" chunking.
# - Candidate boundaries are scored for linguistic completeness.
# - Word-count windows only limit the search space for latency.
# - Entity spans are protected so IDs/phone/OTP/vehicle values are not split.
# ============================================================

TAMIL_CHAR_RE = re.compile(r"[\u0B80-\u0BFF]")

EN_WEAK_END = {
    "a", "an", "the", "to", "at", "in", "on", "for", "from", "with",
    "and", "or", "but", "is", "are", "was", "were", "be", "been", "being",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must",
    "your", "my", "our", "their", "his", "her", "this", "that", "these", "those",
    "of", "by", "as", "if", "when", "while", "because", "than"
}

EN_WEAK_START = {"and", "or", "but", "of", "because", "than", "a", "an", "the", "my", "your", "our", "this", "that"}

# Tamil words that are poor phrase endings when standing alone.
TA_WEAK_END = {
    "ஒரு", "இந்த", "அந்த", "என்", "உன்", "உங்கள்", "எங்கள்", "மற்றும்", "அல்லது",
    "என்று", "என", "மிக", "ரொம்ப", "உடனே"
}

# Common Tamil finite/imperative/auxiliary endings. These are strong safe boundaries.
TA_STRONG_ENDINGS = (
    "வேண்டும்", "வேண்டாம்", "முடியும்", "முடியுமா", "முடியாது",
    "இருக்கிறது", "இருக்கிறேன்", "இருக்கிறோம்", "இருக்கிறான்", "இருக்கிறாள்",
    "இருக்கா", "இருக்குமா", "வரும்", "வருமா", "வரவில்லை", "வந்தது", "வந்துவிட்டது",
    "செல்லுங்கள்", "சொல்லுங்கள்", "அனுப்புங்கள்", "இறக்குங்கள்", "நிறுத்துங்கள்",
    "தொடங்குங்கள்", "பண்ணுங்க", "பண்ணணும்", "கிடைக்குமா", "ஆகும்", "ஆகுமா",
    "செய்யுங்கள்", "செய்யணும்", "செய்ய வேண்டும்", "காத்திருக்கிறேன்"
)

# Words/suffixes that usually form a meaningful dependent phrase and can be cut after
# when enough material has accumulated.
TA_PHRASE_ENDINGS = (
    "க்கு", "இல்", "ல", "ல்", "இருந்து", "வரை", "உடன்", "பற்றி", "பிறகு", "முன்",
    "கழித்து", "போது", "என்றால்", "னால்", "ஆக", "மட்டும்", "தான்", "ஐ", "தை", "த்தை", "னை", "னு"
)

# Tanglish / code-mixed phrase cues.
TG_STRONG_WORDS = {
    "pannunga", "pannanum", "venum", "vendam", "varuma", "varum", "irukka",
    "irukku", "kidaikkuma", "sollunga", "anupunga", "check", "confirm", "cancel"
}

VALUE_WORDS = {
    "zero", "one", "two", "three", "four", "five", "fyeve", "six", "seven", "sevven",
    "eight", "nine", "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen",
    "sixteen", "seventeen", "eighteen", "nineteen", "twenty", "thirty", "forty",
    "fifty", "sixty", "seventy", "eighty", "ninety", "hundred", "thousand",
    "a", "ay", "aayy", "b", "bee", "c", "cee", "d", "dee", "e", "ee", "f", "eff",
    "g", "gee", "h", "aitch", "i", "eye", "j", "jay", "k", "kay", "l", "el",
    "m", "em", "n", "en", "o", "oh", "p", "pee", "q", "cue", "r", "ar", "s",
    "ess", "t", "tee", "u", "you", "v", "vee", "w", "double-u", "x", "ex",
    "y", "why", "z", "zee", "zed",
    "பூஜ்ஜியம்", "ஒன்று", "இரண்டு", "மூன்று", "நான்கு", "ஐந்து", "ஆறு", "ஏழு",
    "எட்டு", "ஒன்பது", "பத்து", "இருபது", "முப்பது", "நாற்பது", "ஐம்பது",
    "அறுபது", "எழுபது", "எண்பது", "தொண்ணூறு"
}

ENTITY_PREFIXES = [
    ["otp", "is"], ["your", "otp", "is"], ["o", "tee", "pee", "is"],
    ["your", "o", "tee", "pee", "is"], ["booking", "id", "is"],
    ["booking", "i", "d", "is"], ["your", "booking", "id", "is"],
    ["phone", "number"], ["phone", "number", "is"], ["your", "phone", "number"],
    ["vehicle", "number"], ["vehicle", "number", "is"],
    ["driver", "vehicle", "number"], ["registration", "number"],
    ["gate"], ["platform"], ["terminal"], ["route"]
]


def clean_token(token):
    return str(token).strip(".,!?;:()[]{}\"'").lower()


def _contains_tamil(token):
    return bool(TAMIL_CHAR_RE.search(str(token)))


def _starts_phrase(tokens, pos, phrase):
    probe = [clean_token(x) for x in tokens[pos:pos + len(phrase)]]
    return probe == phrase


def find_protected_spans(tokens):
    """Half-open token spans that must never be cut internally."""
    spans = []
    n = len(tokens)

    for i in range(n):
        for phrase in ENTITY_PREFIXES:
            if not _starts_phrase(tokens, i, phrase):
                continue
            j = i + len(phrase)
            value_start = j
            while j < n:
                t = clean_token(tokens[j])
                raw = str(tokens[j]).strip(".,!?;:")
                if (
                    t in VALUE_WORDS
                    or bool(re.fullmatch(r"\d+", t))
                    or bool(re.fullmatch(r"[A-Za-z0-9-]+", raw)) and any(ch.isdigit() for ch in raw)
                ):
                    j += 1
                else:
                    break
            if j > value_start:
                spans.append((i, j))
            break

    raw_entity_re = re.compile(
        r"^(?:[A-Za-z]{2}[- ]?\d{2}[- ]?[A-Za-z]{1,3}[- ]?\d{4}|"
        r"\+?\d[\d -]{3,}\d|[A-Za-z]{2,8}\d{3,})$"
    )
    for i, tok in enumerate(tokens):
        probe = str(tok).strip(".,!?;:")
        if raw_entity_re.fullmatch(probe):
            spans.append((i, i + 1))

    spans = sorted(spans)
    merged = []
    for a, b in spans:
        if not merged or a > merged[-1][1]:
            merged.append([a, b])
        else:
            merged[-1][1] = max(merged[-1][1], b)
    return [(a, b) for a, b in merged]


def cut_inside_protected(cut_pos, spans):
    return any(a < cut_pos < b for a, b in spans)


def _tamil_boundary_bonus(previous, next_token):
    prev = clean_token(previous)
    nxt = clean_token(next_token) if next_token else ""
    bonus = 0

    if any(prev.endswith(x) for x in TA_STRONG_ENDINGS):
        bonus += 26
    elif any(prev.endswith(x) for x in TA_PHRASE_ENDINGS):
        bonus += 9

    # Tamil verbal morphology often ends with these suffixes.
    if re.search(r"(?:ங்கள்|ங்க|ணும்|வேண்டும்|லாம்|மா|து|து\?|றேன்|றோம்|றார்|றது)$", prev):
        bonus += 12

    if prev in TA_WEAK_END:
        bonus -= 24

    # Avoid cutting before attached/continuing Tamil particles where possible.
    if nxt in {"க்கு", "ல்", "இல்", "உடன்", "மற்றும்", "ஆனால்"}:
        bonus -= 12

    # In Tamil/Tanglish, a boundary before a new action phrase is often natural.
    if nxt in {"check", "book", "cancel", "confirm", "call", "இன்னும்", "உடனே", "பிறகு", "அடுத்து"}:
        bonus += 18

    return bonus


def _english_boundary_bonus(previous, next_token):
    prev = clean_token(previous)
    nxt = clean_token(next_token) if next_token else ""
    bonus = 0

    if prev in EN_WEAK_END:
        bonus -= 24
    if nxt in EN_WEAK_START:
        bonus -= 24

    # Cutting immediately after an imperative/transitive verb before its object
    # (e.g. "Please send | a cab") is semantically poor.
    if prev in {"send", "take", "share", "ask", "need", "want", "have", "find", "notify", "stop"} and nxt in {"a", "an", "the", "my", "your", "me", "him", "her", "driver", "cab", "kaab"}:
        bonus -= 32

    # Prefer a phrase boundary before prepositional/adverbial continuation,
    # provided the previous side is already meaningful.
    if nxt in {"to", "at", "in", "on", "from", "with", "near", "after", "before", "when"}:
        bonus += 8

    # Common complete verbal forms are safe phrase endings.
    if prev in {
        "arrive", "arrives", "arrived", "cancel", "cancelled", "confirmed", "available",
        "wait", "waiting", "come", "coming", "send", "stop", "call", "change", "refund"
    }:
        bonus += 8
    return bonus


def semantic_boundary_score(tokens, cut_pos, start, target, spans):
    if cut_pos <= start or cut_pos > len(tokens) or cut_inside_protected(cut_pos, spans):
        return -100000

    previous = tokens[cut_pos - 1]
    next_token = tokens[cut_pos] if cut_pos < len(tokens) else ""
    length = cut_pos - start

    # Latency preference: near target, but semantics can override it.
    score = 30 - 4 * abs(length - target)

    if re.search(r"[.!?]$", previous):
        score += 40
    elif re.search(r"[,;:]$", previous):
        score += 20

    if _contains_tamil(previous) or _contains_tamil(next_token):
        score += _tamil_boundary_bonus(previous, next_token)
    else:
        score += _english_boundary_bonus(previous, next_token)

    # Tanglish cue even if tokens themselves are Latin.
    if clean_token(previous) in TG_STRONG_WORDS:
        score += 14

    remaining = len(tokens) - cut_pos
    if 0 < remaining < 2:
        score -= 30

    return score


def _merge_tiny_tail(chunks):
    if len(chunks) >= 2 and len(chunks[-1].split()) == 1:
        chunks[-2] = (chunks[-2].rstrip() + " " + chunks[-1].lstrip()).strip()
        chunks.pop()
    return chunks


def smart_chunk_text_v6(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []

    # ========================================================
    # CURRENCY SEMANTIC ATOMICITY
    # ========================================================
    # Keep the complete amount predicate together.  This prevents:
    #   "அறுநூற்று ஐம்பது" | "ரூபாய் ஆகும்"
    # and produces:
    #   "உங்கள் கட்டணம்" | "அறுநூற்று ஐம்பது ரூபாய் ஆகும்."
    #
    # This is semantic chunking, not a fixed word-count split.
    currency_match = re.match(
        r"^(.*?கட்டணம்)\s+(.+?ரூபாய்\s+(?:ஆகும்|ஆகுமா|இருக்கும்)[.!?]?)$",
        text,
        flags=re.IGNORECASE,
    )
    if currency_match:
        subject_phrase = currency_match.group(1).strip()
        amount_phrase = currency_match.group(2).strip()
        if subject_phrase and amount_phrase:
            return [subject_phrase, amount_phrase]

    tokens = text.split()
    if len(tokens) <= KEEP_WHOLE_IF_WORDS_LE:
        return [text]

    spans = find_protected_spans(tokens)
    chunks = []
    start = 0
    chunk_no = 0

    while start < len(tokens):
        remaining = len(tokens) - start

        # Whole tail is preferred only when it is already compact.
        if chunk_no > 0 and remaining <= NEXT_MAX_WORDS:
            chunks.append(" ".join(tokens[start:]))
            break

        if chunk_no == 0:
            min_words, target, max_words = FIRST_MIN_WORDS, FIRST_TARGET_WORDS, FIRST_MAX_WORDS
        else:
            min_words, target, max_words = NEXT_MIN_WORDS, NEXT_TARGET_WORDS, NEXT_MAX_WORDS

        if remaining <= max_words and chunk_no == 0:
            # Even short first sentences are considered for a semantic micro-boundary.
            # This fixes the old Tamil behavior where <=6 words were always kept whole.
            pass

        lo = min(len(tokens), start + min_words)
        hi = min(len(tokens), start + max_words)

        # For the first chunk of a sentence longer than 3 tokens, do not let the
        # trivial full-sentence endpoint win merely because it has punctuation.
        # We require a real semantic micro-boundary for TTFA.
        if chunk_no == 0 and remaining > 3:
            hi = min(hi, len(tokens) - 1)

        # If the full remaining sentence is <=3 words, keep it whole.
        if remaining <= 3:
            chunks.append(" ".join(tokens[start:]))
            break

        candidates = [c for c in range(lo, hi + 1) if not cut_inside_protected(c, spans)]
        if not candidates:
            c = hi
            while c < len(tokens) and cut_inside_protected(c, spans):
                c += 1
            candidates = [min(c, len(tokens))]

        best_cut = max(
            candidates,
            key=lambda c: semantic_boundary_score(tokens, c, start, target, spans),
        )

        # Do not split at the very end; that is just the whole sentence.
        if best_cut >= len(tokens):
            chunks.append(" ".join(tokens[start:]))
            break

        chunks.append(" ".join(tokens[start:best_cut]))
        start = best_cut
        chunk_no += 1

    chunks = _merge_tiny_tail(chunks)
    return [c.strip() for c in chunks if c.strip()]


# Backward-compatible alias: every existing benchmark/streaming cell now uses V6.
smart_chunk_text_v4 = smart_chunk_text_v6


def validate_chunk_plan(text, chunks):
    original = " ".join(str(text).split())
    rebuilt = " ".join(" ".join(chunks).split())
    warnings_list = []

    if rebuilt != original:
        warnings_list.append("TEXT_CHANGED")
    if not chunks:
        return ["EMPTY_CHUNKS"]

    for i, chunk in enumerate(chunks[:-1]):
        words = chunk.split()
        if not words:
            continue
        end = clean_token(words[-1])
        if end in EN_WEAK_END or end in TA_WEAK_END:
            warnings_list.append(f"WEAK_END_C{i+1}:{words[-1]}")

    for i, chunk in enumerate(chunks[1:], start=2):
        words = chunk.split()
        if words and clean_token(words[0]) in EN_WEAK_START:
            warnings_list.append(f"WEAK_START_C{i}:{words[0]}")

    return warnings_list


# Smoke tests for all three language families.
SEMANTIC_CHUNK_SMOKE = [
    ("ENGLISH", "Please send a cab to my current pickup location."),
    ("ENGLISH", "Please notify me when the cab arrives."),
    ("TAMIL", "எனக்கு விமான நிலையத்துக்கு உடனே செல்ல வேண்டும்."),
    ("TAMIL", "என் பயணத்தை இன்னும் பத்து நிமிடம் கழித்து தொடங்குங்கள்."),
    ("TANGLISH", "Pickup location correctஆ இருக்கானு check பண்ணுங்க."),
    ("TANGLISH", "எனக்கு ஒரு cab book பண்ணணும்."),
    ("TAMIL", "உங்கள் கட்டணம் அறுநூற்று ஐம்பது ரூபாய் ஆகும்."),
]

print("Semantic Micro-Chunker V6 ready ✓")
for lang, sample in SEMANTIC_CHUNK_SMOKE:
    print(f"{lang:8s} -> {smart_chunk_text_v6(sample)}")


Semantic Micro-Chunker V6 ready ✓
ENGLISH  -> ['Please send a cab', 'to my current pickup location.']
ENGLISH  -> ['Please notify me', 'when the cab arrives.']
TAMIL    -> ['எனக்கு விமான நிலையத்துக்கு', 'உடனே செல்ல வேண்டும்.']
TAMIL    -> ['என் பயணத்தை இன்னும் பத்து', 'நிமிடம் கழித்து தொடங்குங்கள்.']
TANGLISH -> ['Pickup location correctஆ இருக்கானு', 'check பண்ணுங்க.']
TANGLISH -> ['எனக்கு ஒரு cab', 'book பண்ணணும்.']
TAMIL    -> ['உங்கள் கட்டணம்', 'அறுநூற்று ஐம்பது ரூபாய் ஆகும்.']


## Semantic boundary guard


In [21]:

# ============================================================
# SEMANTIC GUARD V7 — prevent incomplete / entity-breaking C1
# ============================================================
# This wraps V6 without deleting the existing implementation.

_ORIGINAL_SMART_CHUNK_TEXT_V6 = smart_chunk_text_v6

EN_HARD_BAD_END = {
    "a", "an", "the", "to", "at", "in", "on", "for", "from", "with",
    "and", "or", "but", "of", "my", "your", "his", "her", "our", "their",
    "this", "that", "these", "those", "is", "are", "was", "were",
    "will", "would", "can", "could", "should", "may", "might", "must"
}

# Multi-token names/units that must remain contiguous.
SEMANTIC_ATOMIC_PHRASES = [
    ("தி", "நகர்"),
    ("t", "nagar"),
    ("chennai", "central"),
    ("twelve", "kilometers"),
    ("twelve", "kilometres"),
    ("different", "kaab"),
    ("different", "cab"),
]


def _token_clean_v7(x):
    return str(x).strip(".,!?;:()[]{}\"'").lower()


def _cut_breaks_atomic_phrase(tokens, cut):
    clean = [_token_clean_v7(t) for t in tokens]
    for phrase in SEMANTIC_ATOMIC_PHRASES:
        m = len(phrase)
        for i in range(0, len(clean) - m + 1):
            if tuple(clean[i:i+m]) == phrase and i < cut < i + m:
                return True
    return False


def _first_chunk_is_incomplete(chunk):
    words = chunk.split()
    if not words:
        return True
    return _token_clean_v7(words[-1]) in EN_HARD_BAD_END


def smart_chunk_text_v7(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []

    tokens = text.split()

    # Explicit natural taxi phrases where V6's latency preference can cut badly.
    clean = [_token_clean_v7(t) for t in tokens]

    # Preserve "தி நகர்" as one location phrase:
    # "நான்" | "தி நகரில் காத்திருக்கிறேன்."
    if len(tokens) >= 4 and clean[:3] == ["நான்", "தி", "நகரில்"]:
        return [tokens[0], " ".join(tokens[1:])]

    # Natural English request:
    # "I need" | "a different cab."
    if len(tokens) >= 5 and clean[:4] in (
        ["i", "need", "a", "different"],
        ["i", "need", "another", "cab"],
        ["i", "need", "another", "kaab"],
    ):
        return [" ".join(tokens[:2]), " ".join(tokens[2:])]

    chunks = _ORIGINAL_SMART_CHUNK_TEXT_V6(text)
    if len(chunks) <= 1:
        return chunks

    # Repair a first chunk ending in a determiner/preposition/auxiliary.
    if _first_chunk_is_incomplete(chunks[0]):
        all_tokens = text.split()
        candidate_cuts = []
        for cut in range(FIRST_MIN_WORDS, min(FIRST_MAX_WORDS, len(all_tokens)-1) + 1):
            if _cut_breaks_atomic_phrase(all_tokens, cut):
                continue
            left = " ".join(all_tokens[:cut])
            right = " ".join(all_tokens[cut:])
            if not _first_chunk_is_incomplete(left) and len(right.split()) >= 2:
                candidate_cuts.append((abs(cut - FIRST_TARGET_WORDS), cut, left, right))

        if candidate_cuts:
            _, cut, left, right = min(candidate_cuts)
            repaired_tail = _ORIGINAL_SMART_CHUNK_TEXT_V6(right)
            chunks = [left] + repaired_tail

    # Final atomic-phrase check across every boundary.
    all_tokens = text.split()
    rebuilt = []
    cursor = 0
    for chunk in chunks:
        n = len(chunk.split())
        cut = cursor + n
        if rebuilt and _cut_breaks_atomic_phrase(all_tokens, cursor):
            rebuilt[-1] = rebuilt[-1] + " " + chunk
        else:
            rebuilt.append(chunk)
        cursor = cut

    return [c.strip() for c in rebuilt if c.strip()]


# Backward-compatible aliases used throughout the notebook.
smart_chunk_text_v6 = smart_chunk_text_v7
smart_chunk_text_v4 = smart_chunk_text_v7

print("✓ Semantic Guard V7 active")
for _x in [
    "நான் தி நகரில் காத்திருக்கிறேன்.",
    "I need a different kaab.",
    "The route is twelve kilometers.",
]:
    print(_x, "->", smart_chunk_text_v7(_x))


✓ Semantic Guard V7 active
நான் தி நகரில் காத்திருக்கிறேன். -> ['நான்', 'தி நகரில் காத்திருக்கிறேன்.']
I need a different kaab. -> ['I need', 'a different kaab.']
The route is twelve kilometers. -> ['The route is', 'twelve kilometers.']


## Tanglish request preparation


In [22]:
# ============================================================
# MOVIO — TANGLISH CLARITY + FINAL REQUEST PREPARATION
#
# REPLACE YOUR CURRENT prepare_movio_request() SECTION
# WITH THIS ENTIRE BLOCK.
#
# KEEP YOUR EXISTING:
#   normalize_taxi_text()
#   cleanup_normalized_text()
#   route_reference_clean()
#   safe_semantic_chunks()
#   english_safe_chunks()
#
# ============================================================

import re
import time


# ============================================================
# 1. EXTRA TANGLISH NORMALIZATION
#
# This fixes cases like:
#
# Driver inga vandhutaara?
#
# OLD:
# driver இங்க vandhutaara?
#
# NEW:
# driver இங்க வந்துட்டாரா?
# ============================================================

EXTRA_TANGLISH_WORDS = {

    # --------------------------------------------------------
    # ARRIVAL / COME
    # --------------------------------------------------------

    "vandhutaara": "வந்துட்டாரா",
    "vandhuttara": "வந்துட்டாரா",
    "vandhutara": "வந்துட்டாரா",
    "vandhutaaraa": "வந்துட்டாரா",

    "vandhutaaru": "வந்துட்டாரு",
    "vandhuttaru": "வந்துட்டாரு",

    "vandhutaanga": "வந்துட்டாங்க",
    "vandhuttanga": "வந்துட்டாங்க",

    "vandhuten": "வந்துட்டேன்",
    "vandhutaen": "வந்துட்டேன்",

    "vandhachu": "வந்தாச்சு",

    "vandhu": "வந்து",

    "varuvaaru": "வருவாரு",
    "varuvaru": "வருவாரு",

    "varuvaanga": "வருவாங்க",

    "varum": "வரும்",
    "varuthu": "வருது",
    "varudhu": "வருது",

    # --------------------------------------------------------
    # LOCATION
    # --------------------------------------------------------

    "inga": "இங்க",
    "inge": "இங்கே",
    "ingae": "இங்கே",

    "anga": "அங்க",
    "ange": "அங்கே",

    "locationku": "லொகேஷனுக்கு",
    "locationukku": "லொகேஷனுக்கு",

    # --------------------------------------------------------
    # DRIVER / TAXI CONVERSATION
    # --------------------------------------------------------

    "pannuvaaru": "பண்ணுவாரு",
    "pannuvaar": "பண்ணுவார்",

    "pannuvanga": "பண்ணுவாங்க",

    "pannirukku": "பண்ணிருக்கு",
    "pannirukkum": "பண்ணிருக்கும்",

    "aagirukku": "ஆகிருக்கு",
    "agirukku": "ஆகிருக்கு",

    "irukku": "இருக்கு",
    "iruku": "இருக்கு",

    "irukkum": "இருக்கும்",

    "irundhu": "இருந்து",

    # --------------------------------------------------------
    # PICKUP / DROP
    # --------------------------------------------------------

    "pickup": "பிக்கப்",
    "pick-up": "பிக்கப்",

    "drop": "டிராப்",

    # --------------------------------------------------------
    # WAIT
    # --------------------------------------------------------

    "wait": "வெயிட்",

    # --------------------------------------------------------
    # READY
    # --------------------------------------------------------

    "ready": "ரெடி",

    # --------------------------------------------------------
    # BOOKING
    # --------------------------------------------------------

    "book": "புக்",

    "booked": "புக்",

    # --------------------------------------------------------
    # COMMON TAMIL/TANGLISH
    # --------------------------------------------------------

    "aagum": "ஆகும்",
    "aakum": "ஆகும்",

    "anru": "அன்று",

    "tholaivil": "தொலைவில்",

    "ulladhu": "உள்ளது",

    "ulla": "உள்ள",

    "pakkam": "பக்கம்",

    "pakkathula": "பக்கத்துல",

    "munnadi": "முன்னாடி",

    "pinnadi": "பின்னாடி",

    "sari": "சரி",
    "seri": "சரி",

    "nandri": "நன்றி",

    # --------------------------------------------------------
    # DAY PARTS
    # --------------------------------------------------------

    "kaalai": "காலை",
    "kalai": "காலை",

    "mathiyam": "மதியம்",
    "madhiyam": "மதியம்",

    "maalai": "மாலை",
    "malai": "மாலை",

    "iravu": "இரவு",
}


# ============================================================
# 2. APPLY EXTRA TANGLISH NORMALIZATION
# ============================================================

def normalize_extra_tanglish(text):

    out = str(text)

    # Longest first so vandhutaara is handled before vandhu.
    keys = sorted(
        EXTRA_TANGLISH_WORDS.keys(),
        key=len,
        reverse=True
    )

    for word in keys:

        replacement = EXTRA_TANGLISH_WORDS[word]

        pattern = (
            r"(?<![A-Za-z])"
            + re.escape(word)
            + r"(?![A-Za-z])"
        )

        out = re.sub(
            pattern,
            replacement,
            out,
            flags=re.IGNORECASE
        )

    return out


# ============================================================
# 3. TANGLISH FINAL CLARITY POSTPROCESS
# ============================================================

def tanglish_clarity_postprocess(
    text,
    group="ENTITY"
):

    out = str(text).strip()

    if not out:
        return out

    group = str(
        group
    ).upper().strip()

    # --------------------------------------------------------
    # Only apply aggressive Tanglish conversion for
    # TAMIL/TANGLISH groups.
    #
    # Do NOT convert normal English sentences.
    # --------------------------------------------------------

    if group in (
        "TAMIL",
        "TANGLISH"
    ):

        out = normalize_extra_tanglish(
            out
        )

    # --------------------------------------------------------
    # Tamil duplicated suffix fixes
    # --------------------------------------------------------

    out = re.sub(
        r"நிமிடத்தில்\s*-\s*ல்",
        "நிமிடத்தில்",
        out
    )

    out = re.sub(
        r"மணிக்கு\s*-\s*க்கு",
        "மணிக்கு",
        out
    )

    out = re.sub(
        r"([\u0B80-\u0BFF]+த்தில்)\s*-\s*ல்",
        r"\1",
        out
    )

    out = re.sub(
        r"([\u0B80-\u0BFF]+க்கு)\s*-\s*க்கு",
        r"\1",
        out
    )

    # --------------------------------------------------------
    # Tanglish AM/PM suffix cleanup
    #
    # Example:
    # five forty five PMku
    # should not be treated as one strange English word.
    # --------------------------------------------------------

    out = re.sub(
        r"\bPMku\b",
        "pee em",
        out,
        flags=re.IGNORECASE
    )

    out = re.sub(
        r"\bAMku\b",
        "ay em",
        out,
        flags=re.IGNORECASE
    )

    # --------------------------------------------------------
    # Natural number pronunciation
    # --------------------------------------------------------

    out = re.sub(
        r"\bfyeve\b",
        "five",
        out,
        flags=re.IGNORECASE
    )

    out = re.sub(
        r"\bsevven\b",
        "seven",
        out,
        flags=re.IGNORECASE
    )

    # --------------------------------------------------------
    # Natural English transport words
    # --------------------------------------------------------

    replacements = [

        (r"\byerport\b", "airport"),

        (r"\bkil\s+uh\s+meters\b", "kilometers"),
        (r"\bkil\s+uh\s+meter\b", "kilometer"),

        (r"\bminits\b", "minutes"),

        (r"\btoe\s+tal\b", "total"),

        (r"\bfye\s+nal\b", "final"),

        (r"\bnoe\s+tih\s+fy\b", "notify"),

        (r"\buh\s+riving\b", "arriving"),
        (r"\bar\s+riving\b", "arriving"),

        (r"\buh\s+rived\b", "arrived"),
        (r"\bar\s+rived\b", "arrived"),

        (r"\buh\s+rives\b", "arrives"),
        (r"\bar\s+rives\b", "arrives"),

        (r"\buh\s+rive\b", "arrive"),
        (r"\bar\s+rive\b", "arrive"),
    ]

    for pattern, replacement in replacements:

        out = re.sub(
            pattern,
            replacement,
            out,
            flags=re.IGNORECASE
        )

    # --------------------------------------------------------
    # Fare correction
    # --------------------------------------------------------

    out = re.sub(
        r"\bestimated\s+fair\b",
        "estimated fare",
        out,
        flags=re.IGNORECASE
    )

    out = re.sub(
        r"\byour\s+fair\b",
        "your fare",
        out,
        flags=re.IGNORECASE
    )

    # --------------------------------------------------------
    # Rupee punctuation
    # --------------------------------------------------------

    out = re.sub(
        r"ரூபாய்\s*,\s*ஆகும்",
        "ரூபாய் ஆகும்",
        out
    )

    out = re.sub(
        r"ரூபாய்\s*,\s*மற்றும்",
        "ரூபாய் மற்றும்",
        out
    )

    out = re.sub(
        r"\brupees\s*,\s+and\b",
        "rupees and",
        out,
        flags=re.IGNORECASE
    )

    # --------------------------------------------------------
    # Remove unwanted spaces
    # --------------------------------------------------------

    out = out.replace(
        "\u00A0",
        " "
    )

    out = re.sub(
        r"\s+,",
        ",",
        out
    )

    out = re.sub(
        r"(?:,\s*){2,}",
        ", ",
        out
    )

    out = re.sub(
        r",\s*([.!?])",
        r"\1",
        out
    )

    out = re.sub(
        r"\s+([.!?])",
        r"\1",
        out
    )

    out = re.sub(
        r",(?=\S)",
        ", ",
        out
    )

    out = re.sub(
        r"[ \t]+",
        " ",
        out
    )

    return out.strip()


# ============================================================
# 4. UPDATED PRODUCTION REQUEST PREPARATION
# ============================================================

def prepare_movio_request(
    raw_text,
    group="ENTITY"
):

    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    raw = str(
        raw_text
    ).strip()

    group = str(
        group
    ).upper().strip()

    if not raw:

        raise ValueError(
            "raw_text is empty"
        )

    # ========================================================
    # NORMALIZATION
    # ========================================================

    t0 = time.perf_counter()

    # --------------------------------------------------------
    # STEP 1:
    # EXISTING FULL TAXI NORMALIZER
    # --------------------------------------------------------

    normalized = normalize_taxi_text(
        raw
    )

    # --------------------------------------------------------
    # STEP 2:
    # EXISTING FINAL NORMALIZATION CLEANUP
    # --------------------------------------------------------

    normalized = cleanup_normalized_text(
        normalized,
        group
    )

    # --------------------------------------------------------
    # STEP 3:
    # NEW TANGLISH CLARITY FIX
    #
    # THIS IS THE IMPORTANT NEW PART.
    # --------------------------------------------------------

    normalized = tanglish_clarity_postprocess(
        normalized,
        group
    )

    normalize_ms = (
        time.perf_counter()
        - t0
    ) * 1000.0

    # ========================================================
    # ROUTING
    # ========================================================

    t1 = time.perf_counter()

    route = route_reference_clean(
        normalized,
        group
    )

    routing_ms = (
        time.perf_counter()
        - t1
    ) * 1000.0

    # ========================================================
    # CHUNKING
    # ========================================================

    t2 = time.perf_counter()

    if route == "ENGLISH":

        chunks = english_safe_chunks(
            normalized
        )

    else:

        chunks = safe_semantic_chunks(
            normalized,
            language=group
        )

    if not chunks:

        chunks = [
            normalized
        ]

    chunks = [

        _clean_chunk(chunk)

        for chunk in chunks

        if _clean_chunk(chunk)
    ]

    chunk_ms = (
        time.perf_counter()
        - t2
    ) * 1000.0

    # ========================================================
    # RETURN
    # ========================================================

    return {

        "raw":
            raw,

        "group":
            group,

        "normalized":
            normalized,

        "route":
            route,

        "chunks":
            chunks,

        "normalize_ms":
            normalize_ms,

        "routing_ms":
            routing_ms,

        "chunk_ms":
            chunk_ms,
    }


# ============================================================
# 5. SLIGHTLY FASTER BUT QUALITY-SAFE SPEECH
#
# Use these values instead of the older 1.07 / 1.08 values.
# ============================================================

ENGLISH_SPEED = 1.12

TAMIL_SPEED = 1.12

TANGLISH_SPEED = 1.12

FIRST_CHUNK_NUM_STEP = 4

REST_CHUNK_NUM_STEP = 4

USE_FP16_SAMPLE = False


# ============================================================
# READY
# ============================================================

print("=" * 80)

print(
    "✓ MOVIO TANGLISH CLARITY UPDATE READY"
)

print(
    "✓ prepare_movio_request updated"
)

print(
    "✓ Tanglish residual Latin words converted"
)

print(
    "✓ Speech speed = 1.12"
)

print(
    "✓ First chunk step =",
    FIRST_CHUNK_NUM_STEP
)

print(
    "✓ Rest chunk step =",
    REST_CHUNK_NUM_STEP
)

print(
    "✓ FP16 =",
    USE_FP16_SAMPLE
)

print("=" * 80)

✓ MOVIO TANGLISH CLARITY UPDATE READY
✓ prepare_movio_request updated
✓ Tanglish residual Latin words converted
✓ Speech speed = 1.12
✓ First chunk step = 4
✓ Rest chunk step = 4
✓ FP16 = False


## Tanglish clarity


In [23]:
# ============================================================
# MOVIO — TANGLISH CLARITY + SLIGHTLY FASTER SPEECH FIX
#
# Add this AFTER your existing normalizer definitions.
# Then rerun prepare_movio_request + synthesis cells.
# ============================================================

import re


# ============================================================
# 1. EXTRA TANGLISH WORD NORMALIZATION
# ============================================================

EXTRA_TANGLISH_WORDS = {

    # Arrival / coming
    "vandhutaara": "வந்துட்டாரா",
    "vandhuttara": "வந்துட்டாரா",
    "vandhutara": "வந்துட்டாரா",
    "vandhutaaraa": "வந்துட்டாரா",

    "vandhutaaru": "வந்துட்டாரு",
    "vandhuttaru": "வந்துட்டாரு",
    "vandhutaar": "வந்துட்டார்",

    "vandhutaanga": "வந்துட்டாங்க",
    "vandhuttanga": "வந்துட்டாங்க",

    "vandhuten": "வந்துட்டேன்",
    "vandhutaen": "வந்துட்டேன்",

    "vandhachu": "வந்தாச்சு",

    "varuvaaru": "வருவாரு",
    "varuvaru": "வருவாரு",
    "varuvaanga": "வருவாங்க",

    "varum": "வரும்",
    "varudhu": "வருது",
    "varuthu": "வருது",

    # Location
    "inga": "இங்க",
    "inge": "இங்கே",
    "ingae": "இங்கே",

    "anga": "அங்க",
    "ange": "அங்கே",

    # Waiting
    "wait": "வெயிட்",

    # Pickup / drop conversational forms
    "pickup": "பிக்கப்",
    "drop": "டிராப்",

    # General speech
    "irukku": "இருக்கு",
    "iruku": "இருக்கு",
    "irukkum": "இருக்கும்",

    "pannuvaaru": "பண்ணுவாரு",
    "pannuvaar": "பண்ணுவார்",
    "pannuvanga": "பண்ணுவாங்க",

    "pannirukku": "பண்ணிருக்கு",
    "pannirukkum": "பண்ணிருக்கும்",

    "aagirukku": "ஆகிருக்கு",
    "agirukku": "ஆகிருக்கு",

    "ready": "ரெடி",
}


def normalize_extra_tanglish(text):
    """
    Converts remaining common Tanglish tokens into Tamil script.

    Important because Tamil reference routing works better when
    a Tanglish sentence is mostly Tamil script rather than:
        driver இங்க vandhutaara?

    Better:
        driver இங்க வந்துட்டாரா?
    """

    out = str(text)

    words = sorted(
        EXTRA_TANGLISH_WORDS.keys(),
        key=len,
        reverse=True
    )

    for word in words:

        replacement = EXTRA_TANGLISH_WORDS[word]

        out = re.sub(
            r'(?<![A-Za-z])'
            + re.escape(word)
            + r'(?![A-Za-z])',
            replacement,
            out,
            flags=re.IGNORECASE
        )

    return out


# ============================================================
# 2. FINAL TEXT CLEANUP
# ============================================================

def tanglish_clarity_postprocess(text, group="ENTITY"):

    out = str(text).strip()

    if not out:
        return out

    group = str(group).upper().strip()

    # Apply extra conversion mainly for Tamil/Tanglish routes.
    if group in ("TAMIL", "TANGLISH", "ENTITY"):

        out = normalize_extra_tanglish(
            out
        )

    # Fix duplicated suffixes.
    out = re.sub(
        r'நிமிடத்தில்\s*-\s*ல்',
        'நிமிடத்தில்',
        out
    )

    out = re.sub(
        r'மணிக்கு\s*-\s*க்கு',
        'மணிக்கு',
        out
    )

    # Clean spacing.
    out = re.sub(
        r'\s+',
        ' ',
        out
    ).strip()

    return out


print("✓ Extra Tanglish clarity normalization ready")

✓ Extra Tanglish clarity normalization ready


## Final entity-aware request preparation


In [24]:
import re
import time


# ============================================================
# 1. BASIC CLEANING FOR CHUNKS ONLY
# ============================================================

def _clean_chunk(text):
    """
    Clean whitespace around an already-normalized chunk.
    DOES NOT modify pronunciation or normalization.
    """

    text = str(text)

    text = text.replace(
        "\u00A0",
        " "
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\s+([,.!?])",
        r"\1",
        text
    )

    return text.strip()


# ============================================================
# 2. LANGUAGE / REFERENCE ROUTING
# ============================================================

TAMIL_CHAR_RE = re.compile(
    r"[\u0B80-\u0BFF]"
)


def route_reference_clean(
    normalized_text,
    group="ENTITY"
):
    """
    Decide which DhVaani reference voice to use.

    ENGLISH  -> English reference
    TAMIL    -> Tamil reference
    TANGLISH -> Tamil reference
    ENTITY   -> based on Tamil characters
    """

    text = str(
        normalized_text
    )

    group = str(
        group
    ).upper().strip()

    if group == "ENGLISH":
        return "ENGLISH"

    if group in (
        "TAMIL",
        "TANGLISH"
    ):
        return "TAMIL"

    if TAMIL_CHAR_RE.search(text):
        return "TAMIL"

    return "ENGLISH"

# ============================================================
# 3. ENTITY-PROTECTED SEMANTIC CHUNKER
# ============================================================

import re


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

# Sentences at or below this size remain one chunk.
SEMANTIC_SINGLE_CHUNK_WORDS = 7

# A chunk shorter than this is merged with a neighbouring chunk.
SEMANTIC_MIN_CHUNK_WORDS = 3

# This is a soft limit only.
# The chunker will NEVER cut an entity just to satisfy this limit.
SEMANTIC_SOFT_MAX_WORDS = 12


# ------------------------------------------------------------
# Entity vocabulary
# ------------------------------------------------------------

ENGLISH_DIGIT_WORDS = (
    r"zero|one|two|three|four|five|fyeve|six|"
    r"seven|sevven|eight|ait|nine"
)

ENGLISH_NUMBER_WORDS = (
    ENGLISH_DIGIT_WORDS
    + r"|ten|eleven|twelve|thirteen|fourteen|fifteen|"
      r"sixteen|seventeen|eighteen|nineteen|"
      r"twenty|thirty|forty|fifty|sixty|seventy|"
      r"eighty|ninety|hundred|thousand|lakh|million|and"
)

TAMIL_NUMBER_WORDS = (
    r"பூஜ்ஜியம்|ஒன்று|இரண்டு|மூன்று|நான்கு|ஐந்து|"
    r"ஆறு|ஏழு|எட்டு|ஒன்பது|பத்து|பதினொன்று|"
    r"பன்னிரண்டு|பதின்மூன்று|பதினான்கு|பதினைந்து|"
    r"பதினாறு|பதினேழு|பதினெட்டு|பத்தொன்பது|"
    r"இருபது|இருபத்தி|இருபத்தொன்று|இருபத்திரண்டு|"
    r"இருபத்திமூன்று|இருபத்திநான்கு|இருபத்தைந்து|"
    r"இருபத்தாறு|இருபத்தேழு|இருபத்தெட்டு|"
    r"இருபத்தொன்பது|முப்பது|முப்பத்தி|நாற்பது|"
    r"நாற்பத்தி|ஐம்பது|ஐம்பத்தி|அறுபது|அறுபத்தி|"
    r"எழுபது|எழுபத்தி|எண்பது|எண்பத்தி|"
    r"தொண்ணூறு|தொண்ணூற்று|நூறு|நூற்று|"
    r"இருநூறு|இருநூற்று|முந்நூறு|முந்நூற்று|"
    r"நானூறு|நானூற்று|ஐந்நூறு|ஐந்நூற்று|"
    r"அறுநூறு|அறுநூற்று|எழுநூறு|எழுநூற்று|"
    r"எண்ணூறு|எண்ணூற்று|ஆயிரம்|ஆயிரத்து|"
    r"இரண்டாயிரம்|இரண்டாயிரத்து"
)

ENGLISH_MONTHS = (
    r"January|February|March|April|May|June|July|"
    r"August|September|October|November|December"
)

TAMIL_MONTHS= (
    r"ஜனவரி|பிப்ரவரி|மார்ச்|ஏப்ரல்|மே|ஜூன்|"
    r"ஜூலை|ஆகஸ்ட்|செப்டம்பர்|அக்டோபர்|"
    r"நவம்பர்|டிசம்பர்"
)


# ------------------------------------------------------------
# Entity patterns
# ------------------------------------------------------------

ENTITY_PATTERNS = [

    # --------------------------------------------------------
    # Raw numeric date:
    # 31/08/2026, 31-08-2026
    # --------------------------------------------------------
    re.compile(
        r"(?<!\d)\d{1,2}[/-]\d{1,2}[/-]\d{4}(?!\d)",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Raw time:
    # 7:30, 7:30 PM, 7:30 PMக்கு
    # --------------------------------------------------------
    re.compile(
        r"(?<!\d)\d{1,2}:\d{2}"
        r"(?:\s*[AP]\.?M\.?)?"
        r"(?:-?க்கு)?",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Raw currency and distance
    # --------------------------------------------------------
    re.compile(
        r"(?:₹|Rs\.?|INR)\s*\d[\d,]*(?:\.\d+)?",
        re.IGNORECASE
    ),

    re.compile(
        r"\d+(?:\.\d+)?\s*"
        r"(?:km|kms|kilometres?|kilometers?|கிலோமீட்டர்)",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Raw phone numbers
    # --------------------------------------------------------
    re.compile(
        r"(?<!\w)"
        r"(?:\+?\d{1,3}[\s-]?)?"
        r"(?:\d[\s-]?){10,12}"
        r"(?!\w)"
    ),

    # --------------------------------------------------------
    # Raw booking/vehicle IDs:
    # TN45AB1234, TN-09-AB-1234
    # --------------------------------------------------------
    re.compile(
        r"\b[A-Z]{2}[\s-]?\d{1,2}[\s-]?"
        r"[A-Z]{1,3}[\s-]?\d{3,4}\b",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Spoken OTP:
    # o tee pee, four ait two one
    # --------------------------------------------------------
    re.compile(
        rf"\b(?:o\s+tee\s+pee|OTP)"
        rf"\s*,?\s*(?:{ENGLISH_DIGIT_WORDS})"
        rf"(?:[\s,]+(?:{ENGLISH_DIGIT_WORDS})){{3,7}}",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Spoken phone number:
    # phone number, nine ait ... zero
    # --------------------------------------------------------
    re.compile(
        rf"\b(?:phone\s+number|mobile\s+number)"
        rf"\s*(?:is)?\s*,?\s*"
        rf"(?:{ENGLISH_DIGIT_WORDS})"
        rf"(?:[\s,]+(?:{ENGLISH_DIGIT_WORDS})){{7,14}}",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Spoken booking ID:
    # booking aiyy dee, tee en, four fyeve...
    # Protected until the next safe clause.
    # --------------------------------------------------------
    re.compile(
        r"\bbooking\s+"
        r"(?:a[iy]+(?:y+)?\s+dee|eye\s+dee|ID)"
        r"(?:\s+is)?\s*,?\s*"
        r".+?"
        r"(?="
        r",?\s+(?:and|but)\s+|"
        r"\s+மற்றும்\s+|"
        r"\s+ready\b|"
        r"\s+ஆக\b|"
        r"[.!?]|$"
        r")",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # English spoken dates:
    # December twenty fyeve, two thousand twenty six
    # --------------------------------------------------------
    re.compile(
        rf"\b(?:{ENGLISH_MONTHS})\s+"
        rf"(?:{ENGLISH_NUMBER_WORDS})(?:[-\s]+"
        rf"(?:{ENGLISH_NUMBER_WORDS})){{0,3}}"
        rf"\s*,?\s*"
        rf"(?:two\s+thousand|twenty\s+twenty)"
        rf"(?:[-\s]+(?:{ENGLISH_NUMBER_WORDS})){{1,3}}",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Numeric-order spoken date:
    # twenty fyeve-twelve-two thousand twenty six
    # --------------------------------------------------------
    re.compile(
        rf"\b(?:{ENGLISH_NUMBER_WORDS})"
        rf"(?:\s+(?:{ENGLISH_NUMBER_WORDS}))?"
        rf"[-/](?:{ENGLISH_NUMBER_WORDS})"
        rf"(?:\s+(?:{ENGLISH_NUMBER_WORDS}))?"
        rf"[-/](?:two\s+thousand)"
        rf"(?:\s+(?:{ENGLISH_NUMBER_WORDS})){{1,3}}",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Tamil spoken dates
    # --------------------------------------------------------
    re.compile(
        rf"(?:{TAMIL_MONTHS})\s+"
        rf"(?:{TAMIL_NUMBER_WORDS})(?:\s+(?:{TAMIL_NUMBER_WORDS})){{0,3}}"
        rf"\s*,?\s*"
        rf"(?:{TAMIL_NUMBER_WORDS})(?:\s+(?:{TAMIL_NUMBER_WORDS})){{1,5}}"
        rf"(?:\s+அன்று|\s+anru)?",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # English spoken time:
    # six thirty, pee em
    # fyeve forty fyeve ay em, ku
    # --------------------------------------------------------
    re.compile(
        rf"(?:(?:morning|evening|காலை|மதியம்|மாலை|இரவு)\s+)?"
        rf"(?:{ENGLISH_NUMBER_WORDS})"
        rf"(?:\s+(?:{ENGLISH_NUMBER_WORDS})){{0,2}}"
        rf"\s*,?\s*"
        rf"(?:ay|pee)\s+em"
        rf"(?:\s*,?\s*(?:ku|க்கு))?",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Tamil spoken time:
    # காலை ஐந்து நாற்பத்தி ஐந்து மணிக்கு
    # மாலை ஆறு பதினைந்து மணிக்கு
    # --------------------------------------------------------
    re.compile(
        rf"(?:(?:காலை|மதியம்|மாலை|இரவு)\s+)?"
        rf"(?:{TAMIL_NUMBER_WORDS})"
        rf"(?:\s+(?:{TAMIL_NUMBER_WORDS})){{0,4}}"
        rf"\s+மணிக்கு",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # English spoken fare/amount
    # --------------------------------------------------------
    re.compile(
        rf"\b(?:fare|fair|amount|total)"
        rf"(?:\s+is)?\s+"
        rf"(?:{ENGLISH_NUMBER_WORDS})"
        rf"(?:\s+(?:{ENGLISH_NUMBER_WORDS})){{0,8}}"
        rf"\s+(?:rupees?|INR)",
        re.IGNORECASE
    ),

    # Amount without fare label
    re.compile(
        rf"\b(?:{ENGLISH_NUMBER_WORDS})"
        rf"(?:\s+(?:{ENGLISH_NUMBER_WORDS})){{0,8}}"
        rf"\s+(?:rupees?|paise)\b",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Tamil spoken fare/amount
    # --------------------------------------------------------
    re.compile(
        rf"(?:fair|fare|amount|தொகை)?\s*"
        rf"(?:{TAMIL_NUMBER_WORDS})"
        rf"(?:\s+(?:{TAMIL_NUMBER_WORDS})){{0,6}}"
        rf"\s+(?:ரூபாய்|பைசா)",
        re.IGNORECASE
    ),

    # --------------------------------------------------------
    # Distance and duration
    # --------------------------------------------------------
    re.compile(
        rf"(?:{ENGLISH_NUMBER_WORDS}|{TAMIL_NUMBER_WORDS})"
        rf"(?:\s+(?:{ENGLISH_NUMBER_WORDS}|{TAMIL_NUMBER_WORDS})){{0,4}}"
        rf"\s+(?:"
        rf"kilometres?|kilometers?|km|kms|கிலோமீட்டர்|"
        rf"minutes?|minits?|hours?|நிமிடம்|நிமிடத்தில்|மணி"
        rf")(?:-ல்|-இல்|ல்)?",
        re.IGNORECASE
    ),
]


# ------------------------------------------------------------
# Protect and restore entities
# ------------------------------------------------------------

def _protect_semantic_entities(text):
    """
    Replace every detected entity with one indivisible placeholder.

    Returns:
        protected_text, entity_map
    """

    protected_text = str(text)
    entity_map = {}
    entity_index = 0

    for pattern in ENTITY_PATTERNS:

        def replace_entity(match):
            nonlocal entity_index

            entity = match.group(0).strip()

            if not entity:
                return match.group(0)

            placeholder = f"ENTITYTOKEN{entity_index}X"
            entity_map[placeholder] = entity
            entity_index += 1

            return placeholder

        protected_text = pattern.sub(
            replace_entity,
            protected_text
        )

    return protected_text, entity_map


def _restore_semantic_entities(text, entity_map):
    restored = str(text)

    for placeholder, entity in entity_map.items():
        restored = restored.replace(
            placeholder,
            entity
        )

    return _clean_chunk(restored)


# ------------------------------------------------------------
# Safe clause splitting
# ------------------------------------------------------------

def _split_at_safe_boundaries(protected_text):
    """
    Split only at genuine clause boundaries.

    Important:
    - A normal comma is NOT treated as a split point because dates,
      times, OTPs and IDs contain commas.
    - Protected ENTITYTOKEN values cannot be split.
    """

    text = _clean_chunk(protected_text)

    if not text:
        return []

    # Strong sentence boundaries.
    sentence_parts = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    result = []

    for sentence in sentence_parts:

        sentence = sentence.strip()

        if not sentence:
            continue

        # Safe English clause boundary:
        # "... rupees, and your driver ..."
        sentence = re.sub(
            r",\s+(and|but)\s+",
            r"||| \1 ",
            sentence,
            flags=re.IGNORECASE
        )

        # Safe Tamil/Tanglish clause boundary:
        # "... ரூபாய் மற்றும் destination ..."
        sentence = re.sub(
            r"\s+(மற்றும்|ஆனால்)\s+",
            r" ||| \1 ",
            sentence
        )

        # Semicolon and colon can be used as safe boundaries.
        sentence = re.sub(
            r"\s*[;:]\s+",
            " ||| ",
            sentence
        )

        parts = [
            part.strip()
            for part in sentence.split("|||")
            if part.strip()
        ]

        result.extend(parts)

    return result


# ------------------------------------------------------------
# Meaningfulness correction
# ------------------------------------------------------------

def _merge_incomplete_chunks(chunks):
    """
    Prevent chunks such as:
        ['மாலை', 'ஆறு பதினைந்து மணிக்கு ...']
        ['Your destination is', 'nine kilometres away']
        ['The driver is currently', 'two kilometres away']

    A short or syntactically incomplete chunk is joined to its
    neighbouring chunk.
    """

    chunks = [
        _clean_chunk(chunk)
        for chunk in chunks
        if _clean_chunk(chunk)
    ]

    if len(chunks) <= 1:
        return chunks

    incomplete_endings = re.compile(
        r"\b(?:"
        r"is|are|was|were|for|at|on|in|from|to|with|"
        r"currently|approximately|scheduled|"
        r"your|my|the|a|an|"
        r"உங்கள்|உங்க|உங்களோட|என்|ஒரு|"
        r"காலை|மதியம்|மாலை|இரவு|"
        r"மற்றும்|ஆனால்"
        r")\s*[,.]?$",
        re.IGNORECASE
    )

    merged = []
    index = 0

    while index < len(chunks):

        current = chunks[index]
        current_words = current.split()

        should_join_next = (
            index + 1 < len(chunks)
            and (
                len(current_words) < SEMANTIC_MIN_CHUNK_WORDS
                or incomplete_endings.search(current)
            )
        )

        if should_join_next:
            current = _clean_chunk(
                current + " " + chunks[index + 1]
            )
            index += 2
        else:
            merged.append(current)
            index += 1
            continue

        merged.append(current)

    # If the last chunk is extremely short, attach it backwards.
    if (
        len(merged) >= 2
        and len(merged[-1].split()) < SEMANTIC_MIN_CHUNK_WORDS
    ):
        merged[-2] = _clean_chunk(
            merged[-2] + " " + merged[-1]
        )
        merged.pop()

    return merged

# ------------------------------------------------------------
# Soft splitting for long semantic clauses
# ------------------------------------------------------------

def _soft_split_long_chunks(chunks):
    """
    Split long chunks that contain no explicit clause boundary.

    IMPORTANT:
    - This operates while entities are still protected as ENTITYTOKEN...
    - Therefore OTP/date/time/phone/booking-ID entities remain indivisible.
    - Uses SEMANTIC_SOFT_MAX_WORDS as a soft maximum.
    - Avoids tiny final chunks.
    """

    result = []

    for chunk in chunks:

        chunk = _clean_chunk(chunk)

        if not chunk:
            continue

        words = chunk.split()

        # Already short enough
        if len(words) <= SEMANTIC_SOFT_MAX_WORDS:
            result.append(chunk)
            continue

        start = 0

        while start < len(words):

            remaining = len(words) - start

            # Keep the remaining words together if they fit.
            if remaining <= SEMANTIC_SOFT_MAX_WORDS:
                piece = " ".join(words[start:])
                result.append(
                    _clean_chunk(piece)
                )
                break

            # Normal soft cut
            end = start + SEMANTIC_SOFT_MAX_WORDS

            # ------------------------------------------------
            # Prefer a linguistically useful boundary
            # near the desired cut.
            # ------------------------------------------------

            preferred_words = {
                # English
                "and",
                "but",
                "then",
                "because",
                "while",
                "after",
                "before",
                "from",
                "at",

                # Tamil
                "மற்றும்",
                "ஆனால்",
                "பிறகு",
                "வரை",
                "இருந்து",
                "அன்று",
                "மணிக்கு",

                # Tanglish
                "matrum",
                "aanaal",
                "apram",
                "appuram",
                "irundhu",
                "varaikkum",
            }

            search_start = max(
                start + SEMANTIC_MIN_CHUNK_WORDS,
                end - 4
            )

            best_cut = None

            for i in range(
                end - 1,
                search_start - 1,
                -1
            ):

                clean_word = (
                    words[i]
                    .strip(" ,.!?")
                    .lower()
                )

                if clean_word in preferred_words:
                    # Keep boundary word with C1
                    best_cut = i + 1
                    break

            if best_cut is not None:
                end = best_cut

            # ------------------------------------------------
            # Don't leave a tiny final chunk
            # ------------------------------------------------

            remaining_after = len(words) - end

            if (
                remaining_after > 0
                and
                remaining_after < SEMANTIC_MIN_CHUNK_WORDS
            ):
                end = len(words)

            piece = " ".join(
                words[start:end]
            )

            piece = _clean_chunk(piece)

            if piece:
                result.append(piece)

            start = end

    return result
# ------------------------------------------------------------
# Final semantic chunker
# ------------------------------------------------------------

def entity_protected_semantic_chunks(text, language=None):
    """
    Entity-safe semantic chunking for English, Tamil and Tanglish.

    Processing order:
        1. Clean normalized text.
        2. Protect complete entities.
        3. Split only at safe clause boundaries.
        4. Merge incomplete phrases.
        5. Restore protected entities.

    No word-count-based hard splitting is used.
    """

    text = _clean_chunk(text)

    if not text:
        return []

    # Short sentences remain intact.
    if len(text.split()) <= SEMANTIC_SINGLE_CHUNK_WORDS:
        return [text]

    protected_text, entity_map = _protect_semantic_entities(
        text
    )

    protected_chunks = _split_at_safe_boundaries(
        protected_text
    )

    protected_chunks = _merge_incomplete_chunks(
        protected_chunks
    )

    chunks = [
        _restore_semantic_entities(chunk, entity_map)
        for chunk in protected_chunks
    ]

    chunks = [
        chunk
        for chunk in chunks
        if chunk
    ]

    if not chunks:
        return [text]

    return chunks


# ============================================================
# 4. TAMIL / TANGLISH / ENTITY CHUNKING
# ============================================================

def safe_semantic_chunks(text, language=None):
    return entity_protected_semantic_chunks(
        text,
        language=language
    )


# ============================================================
# 5. ENGLISH CHUNKING
# ============================================================

def english_safe_chunks(text):
    return entity_protected_semantic_chunks(
        text,
        language="ENGLISH"
    )


print("✓ Entity-protected semantic chunker loaded")
print("✓ Time/date/amount/distance/phone/OTP/ID protection enabled")
print("✓ Word-count-based internal entity splitting disabled")


# ============================================================
# 5. ENGLISH CHUNKING
# ============================================================

def english_safe_chunks(
    text
):
    """
    English semantic chunking.

    Uses the same smart chunker but keeps very short
    English sentences as a single chunk.
    """

    text = _clean_chunk(
        text
    )

    if not text:

        return []

    # --------------------------------------------------------
    # Read your existing config if available
    # --------------------------------------------------------

    single_chunk_limit = globals().get(
        "ENGLISH_SINGLE_CHUNK_WORDS",
        10
    )

    min_chunk_words = globals().get(
        "ENGLISH_MIN_CHUNK_WORDS",
        4
    )

    words = text.split()

    # --------------------------------------------------------
    # Short English sentence -> keep whole
    # --------------------------------------------------------

    if len(words) <= single_chunk_limit:

        return [
            text
        ]

    chunks = call_smart_chunker(
        text,
        language="ENGLISH"
    )

    if not chunks:

        return [
            text
        ]

    # --------------------------------------------------------
    # Avoid extremely tiny English chunks
    # --------------------------------------------------------

    merged = []

    for chunk in chunks:

        chunk = _clean_chunk(
            chunk
        )

        if not chunk:
            continue

        if (
            merged
            and len(
                chunk.split()
            ) < min_chunk_words
        ):

            merged[-1] = _clean_chunk(
                merged[-1]
                + " "
                + chunk
            )

        else:

            merged.append(
                chunk
            )

    # --------------------------------------------------------
    # Don't leave C1 as a 1–2 word fragment
    # --------------------------------------------------------

    if (
        len(merged) >= 2
        and
        len(
            merged[0].split()
        ) < min_chunk_words
    ):

        merged[1] = _clean_chunk(
            merged[0]
            + " "
            + merged[1]
        )

        merged = merged[1:]

    if not merged:

        return [
            text
        ]

    return merged


# ============================================================
# 6. MAIN REQUEST PREPARATION
# ============================================================

def prepare_movio_request(
    raw_text,
    group="ENTITY"
):
    """
    Main preprocessing function before DhVaani.

    IMPORTANT:
    normalize_taxi_text() is the ONLY normalization function
    used here.
    """

    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    raw = str(
        raw_text
    ).strip()

    group = str(
        group
    ).upper().strip()

    if not raw:

        raise ValueError(
            "raw_text is empty"
        )

    # ========================================================
    # NORMALIZATION
    # ========================================================
    #
    # YOUR NORMALIZATION ONLY.
    #
    # DO NOT ADD:
    #
    # cleanup_normalized_text(...)
    #
    # tanglish_clarity_postprocess(...)
    #
    # ========================================================

    t0 = time.perf_counter()

    normalized = normalize_taxi_text(
        raw
    )

    normalize_ms = (
        time.perf_counter()
        - t0
    ) * 1000.0


    # ========================================================
    # ROUTING
    # ========================================================

    t1 = time.perf_counter()

    route = route_reference_clean(
        normalized,
        group
    )

    routing_ms = (
        time.perf_counter()
        - t1
    ) * 1000.0


    # ========================================================
    # CHUNKING
    # ========================================================

    t2 = time.perf_counter()

    if route == "ENGLISH":

        chunks = english_safe_chunks(
            normalized
        )

    else:

        chunks = safe_semantic_chunks(
            normalized,
            language=group
        )

    # --------------------------------------------------------
    # Safety fallback
    # --------------------------------------------------------

    if not chunks:

        chunks = [
            normalized
        ]

    # --------------------------------------------------------
    # Whitespace cleaning ONLY
    # --------------------------------------------------------

    final_chunks = []

    for chunk in chunks:

        chunk = _clean_chunk(
            chunk
        )

        if chunk:

            final_chunks.append(
                chunk
            )

    if not final_chunks:

        final_chunks = [
            normalized
        ]

    chunk_ms = (
        time.perf_counter()
        - t2
    ) * 1000.0


    # ========================================================
    # OUTPUT
    # ========================================================

    return {

        "raw":
            raw,

        "group":
            group,

        "normalized":
            normalized,

        "route":
            route,

        "chunks":
            final_chunks,

        "normalize_ms":
            normalize_ms,

        "routing_ms":
            routing_ms,

        "chunk_ms":
            chunk_ms,
    }


# ============================================================
# 7. QUALITY / LATENCY SETTINGS
# ============================================================

FIRST_CHUNK_NUM_STEP = 3

REST_CHUNK_NUM_STEP = 4


# Slightly faster natural playback
ENGLISH_SPEED = 1.12

TAMIL_SPEED = 1.12

TANGLISH_SPEED = 1.12


# Keep FP32 while checking pronunciation quality
USE_FP16_SAMPLE = False


# ============================================================
# 8. SHORTER FIRST CHUNK
# ============================================================

FIRST_MIN_WORDS = 2

FIRST_TARGET_WORDS = 2

FIRST_MAX_WORDS = 3


NEXT_MIN_WORDS = 3

NEXT_TARGET_WORDS = 5

NEXT_MAX_WORDS = 8


KEEP_WHOLE_IF_WORDS_LE = 4


ENGLISH_SINGLE_CHUNK_WORDS = 10

ENGLISH_MIN_CHUNK_WORDS = 4


# ============================================================
# 9. CHECK ACTIVE FUNCTION
# ============================================================

print("=" * 80)

print(
    "✓ FINAL prepare_movio_request loaded"
)

print(
    "✓ normalize_taxi_text is the ONLY normalizer"
)

print(
    "✓ cleanup_normalized_text NOT used"
)

print(
    "✓ tanglish_clarity_postprocess NOT used"
)

print(
    "✓ FIRST_CHUNK_NUM_STEP =",
    FIRST_CHUNK_NUM_STEP
)

print(
    "✓ REST_CHUNK_NUM_STEP =",
    REST_CHUNK_NUM_STEP
)

print(
    "✓ ENGLISH_SPEED =",
    ENGLISH_SPEED
)

print(
    "✓ TAMIL_SPEED =",
    TAMIL_SPEED
)

print(
    "✓ TANGLISH_SPEED =",
    TANGLISH_SPEED
)

print("=" * 80)


# ============================================================
# 10. VERIFY THERE IS NO OLD CLEANUP
# ============================================================

_function_names = (
    prepare_movio_request
    .__code__
    .co_names
)

assert (
    "cleanup_normalized_text"
    not in _function_names
), "OLD cleanup_normalized_text is still active!"

assert (
    "tanglish_clarity_postprocess"
    not in _function_names
), "OLD tanglish_clarity_postprocess is still active!"

assert (
    "normalize_taxi_text"
    in _function_names
), "normalize_taxi_text is missing!"

print(
    "✓ FUNCTION CHECK PASSED"
)

✓ Entity-protected semantic chunker loaded
✓ Time/date/amount/distance/phone/OTP/ID protection enabled
✓ Word-count-based internal entity splitting disabled
✓ FINAL prepare_movio_request loaded
✓ normalize_taxi_text is the ONLY normalizer
✓ cleanup_normalized_text NOT used
✓ tanglish_clarity_postprocess NOT used
✓ FIRST_CHUNK_NUM_STEP = 3
✓ REST_CHUNK_NUM_STEP = 4
✓ ENGLISH_SPEED = 1.12
✓ TAMIL_SPEED = 1.12
✓ TANGLISH_SPEED = 1.12
✓ FUNCTION CHECK PASSED


## Build English and Tamil prompt caches

In [25]:
# Your ORIGINAL Google Drive reference recordings; never default sample voice.
REFERENCE_ENGLISH_TEXT="Hello your cab booking has been confirmed and the driver will arrive in ten minutes."
REFERENCE_TAMIL_TEXT="நான் சென்னை சென்ட்ரல் செல்ல வேண்டும்."
ENGLISH_PROMPT_CACHE = prepare_prompt_cache(
    REFERENCE_ENGLISH,
    REFERENCE_ENGLISH_TEXT,
)

TAMIL_PROMPT_CACHE = prepare_prompt_cache(
    REFERENCE_TAMIL,
    REFERENCE_TAMIL_TEXT,
)

print("\n✓ English prompt cached")
print("✓ Tamil prompt cached")
print("✓ Prompt features remain on GPU")

Prompt cache created in 1262.72 ms
Duration: 4.736 sec
Feature shape: (1, 444, 100)
Prompt cache created in 44.55 ms
Duration: 3.944 sec
Feature shape: (1, 370, 100)

✓ English prompt cached
✓ Tamil prompt cached
✓ Prompt features remain on GPU


## Backend synthesis helpers and FP32 synthesis

In [26]:
import importlib
import inspect

helper_names = (
    "chunk_tokens_punctuation",
    "batchify_tokens",
    "cross_fade_concat",
)

found = {}

for module_name in (
    "zipvoice.utils.infer",
    "zipvoice.bin.infer_zipvoice",
):
    try:
        module = importlib.import_module(module_name)
    except ModuleNotFoundError as error:
        if error.name and (
            module_name == error.name
            or module_name.startswith(error.name + ".")
        ):
            continue
        raise

    for name in helper_names:
        function = getattr(module, name, None)
        if callable(function):
            found.setdefault(name, function)

    if len(found) == len(helper_names):
        break

missing = [name for name in helper_names if name not in found]
assert not missing, f"Backend helpers not found: {missing}"

globals().update(found)


vocoder = rt['vocoder']

@torch.inference_mode()
def synthesize_cached(
    text,
    cache,
    num_step=3,
    speed=1.07,
    seed=666,
    guidance_scale=1.0,
    t_shift=0.5,
    max_duration=100,
):

    torch.manual_seed(seed)

    # ----------------------------------------
    # NEW TEXT ONLY
    # ----------------------------------------

    text = add_punctuation(text)

    tokens_str = (
        tokenizer.texts_to_tokens(
            [text]
        )[0]
    )

    # Same calculation used by DhVaani
    token_duration = (
        cache["duration"]
        /
        (
            len(cache["tokens_str"])
            * speed
        )
    )

    max_tokens = int(
        (25 - cache["duration"])
        /
        token_duration
    )

    chunked_tokens_str = (
        chunk_tokens_punctuation(
            tokens_str,
            max_tokens=max_tokens,
        )
    )

    chunked_tokens = (
        tokenizer.tokens_to_token_ids(
            chunked_tokens_str
        )
    )

    tokens_batches, chunked_index = (
        batchify_tokens(
            chunked_tokens,
            max_duration,
            cache["duration"],
            token_duration,
        )
    )

    chunked_features = []

    # ----------------------------------------
    # MODEL
    # ----------------------------------------

    for batch_tokens in tokens_batches:

        batch_size = len(
            batch_tokens
        )

        batch_prompt_tokens = (
            cache["tokens"]
            * batch_size
        )

        batch_prompt_features = (
            cache["features"].expand(
                batch_size,
                -1,
                -1,
            )
        )

        batch_prompt_features_lens = (
            torch.full(
                (batch_size,),
                cache["features"].size(1),
                device=DEVICE,
                dtype=torch.long,
            )
        )

        (
            pred_features,
            pred_features_lens,
            _,
            _,
        ) = model.model.sample(
            tokens=batch_tokens,
            prompt_tokens=batch_prompt_tokens,
            prompt_features=batch_prompt_features,
            prompt_features_lens=(
                batch_prompt_features_lens
            ),
            speed=speed,
            t_shift=t_shift,
            duration="predict",
            num_step=num_step,
            guidance_scale=guidance_scale,
        )

        pred_features = (
            pred_features.permute(
                0,
                2,
                1,
            )
            / FEAT_SCALE
        )

        chunked_features.append(
            (
                pred_features,
                pred_features_lens,
            )
        )

    # ----------------------------------------
    # VOCODER
    # ----------------------------------------

    chunked_wavs = []

    for (
        pred_features,
        pred_features_lens
    ) in chunked_features:

        for i in range(
            pred_features.size(0)
        ):

            length = int(
                pred_features_lens[i]
            )

            wav = vocoder.decode(
                pred_features[
                    i:i+1,
                    :,
                    :length,
                ]
            )

            wav = (
                wav
                .squeeze(1)
                .clamp(-1, 1)
            )

            if (
                cache["rms"]
                < TARGET_RMS
            ):

                wav = (
                    wav
                    * cache["rms"]
                    / TARGET_RMS
                )

            chunked_wavs.append(
                wav
            )

    # ----------------------------------------
    # ORDER + JOIN
    # ----------------------------------------

    indexed = list(
        zip(
            chunked_index,
            chunked_wavs,
        )
    )

    indexed.sort(
        key=lambda x: x[0]
    )

    ordered_wavs = [
        wav
        for _, wav in indexed
    ]

    final_wav = cross_fade_concat(
        ordered_wavs,
        fade_duration=0.1,
        sample_rate=SAMPLING_RATE,
    )

    final_wav = remove_silence(
        final_wav,
        SAMPLING_RATE,
        only_edge=True,
        trail_sil=80,
    )

    # IMPORTANT:
    # no temporary WAV
    # no torchaudio.save()
    # no sf.read()
    # no os.remove()

    return (
        final_wav
        .detach()
        .float()
        .cpu()
        .squeeze()
        .numpy()
    )

## FP16 synthesis option

In [27]:
# ============================================================
# MOVIO — FP16 CACHED SYNTHESIS FOR AUDIO QUALITY TEST
# ============================================================

import torch
import numpy as np
from IPython.display import Audio, display

@torch.inference_mode()
def synthesize_cached_fp16(
    text,
    cache,
    num_step=3,
    speed=1.07,
    seed=666,
    guidance_scale=1.0,
    t_shift=0.5,
    max_duration=100,
):
    torch.manual_seed(seed)

    text = add_punctuation(text)

    tokens_str = tokenizer.texts_to_tokens([text])[0]

    token_duration = (
        cache["duration"]
        / (
            len(cache["tokens_str"])
            * speed
        )
    )

    max_tokens = int(
        (25 - cache["duration"])
        / token_duration
    )

    chunked_tokens_str = chunk_tokens_punctuation(
        tokens_str,
        max_tokens=max_tokens,
    )

    chunked_tokens = tokenizer.tokens_to_token_ids(
        chunked_tokens_str
    )

    tokens_batches, chunked_index = batchify_tokens(
        chunked_tokens,
        max_duration,
        cache["duration"],
        token_duration,
    )

    chunked_features = []

    for batch_tokens in tokens_batches:

        batch_size = len(batch_tokens)

        batch_prompt_tokens = (
            cache["tokens"]
            * batch_size
        )

        batch_prompt_features = (
            cache["features"]
            .expand(
                batch_size,
                -1,
                -1,
            )
            .contiguous()
        )

        batch_prompt_features_lens = torch.full(
            (batch_size,),
            cache["features"].size(1),
            device=DEVICE,
            dtype=torch.long,
        )

        # ====================================================
        # FP16 ONLY FOR MODEL INFERENCE
        # ====================================================

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            (
                pred_features,
                pred_features_lens,
                _,
                _,
            ) = model.model.sample(
                tokens=batch_tokens,
                prompt_tokens=batch_prompt_tokens,
                prompt_features=batch_prompt_features,
                prompt_features_lens=batch_prompt_features_lens,
                speed=speed,
                t_shift=t_shift,
                duration="predict",
                num_step=num_step,
                guidance_scale=guidance_scale,
            )

        # Return features to FP32 for vocoder safety
        pred_features = (
            pred_features
            .float()
            .permute(0, 2, 1)
            / FEAT_SCALE
        )

        chunked_features.append(
            (
                pred_features,
                pred_features_lens,
            )
        )

    chunked_wavs = []

    for (
        pred_features,
        pred_features_lens
    ) in chunked_features:

        for i in range(
            pred_features.size(0)
        ):

            length = int(
                pred_features_lens[i]
            )

            # Keep vocoder in its existing precision
            wav = vocoder.decode(
                pred_features[
                    i:i+1,
                    :,
                    :length,
                ]
            )

            wav = (
                wav
                .squeeze(1)
                .clamp(-1, 1)
            )

            if cache["rms"] < TARGET_RMS:
                wav = (
                    wav
                    * cache["rms"]
                    / TARGET_RMS
                )

            chunked_wavs.append(
                wav
            )

    indexed = list(
        zip(
            chunked_index,
            chunked_wavs,
        )
    )

    indexed.sort(
        key=lambda x: x[0]
    )

    ordered_wavs = [
        wav
        for _, wav in indexed
    ]

    final_wav = cross_fade_concat(
        ordered_wavs,
        fade_duration=0.1,
        sample_rate=SAMPLING_RATE,
    )

    final_wav = remove_silence(
        final_wav,
        SAMPLING_RATE,
        only_edge=True,
        trail_sil=80,
    )

    return (
        final_wav
        .detach()
        .float()
        .cpu()
        .squeeze()
        .numpy()
    )

## Build English and Tamil prompt caches


In [28]:
# Your ORIGINAL Google Drive reference recordings; never default sample voice.
REFERENCE_ENGLISH_TEXT="Hello your cab booking has been confirmed and the driver will arrive in ten minutes."
REFERENCE_TAMIL_TEXT="நான் சென்னை சென்ட்ரல் செல்ல வேண்டும்."
ENGLISH_PROMPT_CACHE = prepare_prompt_cache(
    REFERENCE_ENGLISH,
    REFERENCE_ENGLISH_TEXT,
)

TAMIL_PROMPT_CACHE = prepare_prompt_cache(
    REFERENCE_TAMIL,
    REFERENCE_TAMIL_TEXT,
)

print("\n✓ English prompt cached")
print("✓ Tamil prompt cached")
print("✓ Prompt features remain on GPU")

Prompt cache created in 52.49 ms
Duration: 4.736 sec
Feature shape: (1, 444, 100)
Prompt cache created in 46.15 ms
Duration: 3.944 sec
Feature shape: (1, 370, 100)

✓ English prompt cached
✓ Tamil prompt cached
✓ Prompt features remain on GPU


## Backend synthesis helpers and FP32 synthesis


In [29]:
import importlib
import inspect

helper_names = (
    "chunk_tokens_punctuation",
    "batchify_tokens",
    "cross_fade_concat",
)

found = {}

for module_name in (
    "zipvoice.utils.infer",
    "zipvoice.bin.infer_zipvoice",
):
    try:
        module = importlib.import_module(module_name)
    except ModuleNotFoundError as error:
        if error.name and (
            module_name == error.name
            or module_name.startswith(error.name + ".")
        ):
            continue
        raise

    for name in helper_names:
        function = getattr(module, name, None)
        if callable(function):
            found.setdefault(name, function)

    if len(found) == len(helper_names):
        break

missing = [name for name in helper_names if name not in found]
assert not missing, f"Backend helpers not found: {missing}"

globals().update(found)


vocoder = rt['vocoder']

@torch.inference_mode()
def synthesize_cached(
    text,
    cache,
    num_step=3,
    speed=1.07,
    seed=666,
    guidance_scale=1.0,
    t_shift=0.5,
    max_duration=100,
):

    torch.manual_seed(seed)

    # ----------------------------------------
    # NEW TEXT ONLY
    # ----------------------------------------

    text = add_punctuation(text)

    tokens_str = (
        tokenizer.texts_to_tokens(
            [text]
        )[0]
    )

    # Same calculation used by DhVaani
    token_duration = (
        cache["duration"]
        /
        (
            len(cache["tokens_str"])
            * speed
        )
    )

    max_tokens = int(
        (25 - cache["duration"])
        /
        token_duration
    )

    chunked_tokens_str = (
        chunk_tokens_punctuation(
            tokens_str,
            max_tokens=max_tokens,
        )
    )

    chunked_tokens = (
        tokenizer.tokens_to_token_ids(
            chunked_tokens_str
        )
    )

    tokens_batches, chunked_index = (
        batchify_tokens(
            chunked_tokens,
            max_duration,
            cache["duration"],
            token_duration,
        )
    )

    chunked_features = []

    # ----------------------------------------
    # MODEL
    # ----------------------------------------

    for batch_tokens in tokens_batches:

        batch_size = len(
            batch_tokens
        )

        batch_prompt_tokens = (
            cache["tokens"]
            * batch_size
        )

        batch_prompt_features = (
            cache["features"].expand(
                batch_size,
                -1,
                -1,
            )
        )

        batch_prompt_features_lens = (
            torch.full(
                (batch_size,),
                cache["features"].size(1),
                device=DEVICE,
                dtype=torch.long,
            )
        )

        (
            pred_features,
            pred_features_lens,
            _,
            _,
        ) = model.model.sample(
            tokens=batch_tokens,
            prompt_tokens=batch_prompt_tokens,
            prompt_features=batch_prompt_features,
            prompt_features_lens=(
                batch_prompt_features_lens
            ),
            speed=speed,
            t_shift=t_shift,
            duration="predict",
            num_step=num_step,
            guidance_scale=guidance_scale,
        )

        pred_features = (
            pred_features.permute(
                0,
                2,
                1,
            )
            / FEAT_SCALE
        )

        chunked_features.append(
            (
                pred_features,
                pred_features_lens,
            )
        )

    # ----------------------------------------
    # VOCODER
    # ----------------------------------------

    chunked_wavs = []

    for (
        pred_features,
        pred_features_lens
    ) in chunked_features:

        for i in range(
            pred_features.size(0)
        ):

            length = int(
                pred_features_lens[i]
            )

            wav = vocoder.decode(
                pred_features[
                    i:i+1,
                    :,
                    :length,
                ]
            )

            wav = (
                wav
                .squeeze(1)
                .clamp(-1, 1)
            )

            if (
                cache["rms"]
                < TARGET_RMS
            ):

                wav = (
                    wav
                    * cache["rms"]
                    / TARGET_RMS
                )

            chunked_wavs.append(
                wav
            )

    # ----------------------------------------
    # ORDER + JOIN
    # ----------------------------------------

    indexed = list(
        zip(
            chunked_index,
            chunked_wavs,
        )
    )

    indexed.sort(
        key=lambda x: x[0]
    )

    ordered_wavs = [
        wav
        for _, wav in indexed
    ]

    final_wav = cross_fade_concat(
        ordered_wavs,
        fade_duration=0.1,
        sample_rate=SAMPLING_RATE,
    )

    final_wav = remove_silence(
        final_wav,
        SAMPLING_RATE,
        only_edge=True,
        trail_sil=80,
    )

    # IMPORTANT:
    # no temporary WAV
    # no torchaudio.save()
    # no sf.read()
    # no os.remove()

    return (
        final_wav
        .detach()
        .float()
        .cpu()
        .squeeze()
        .numpy()
    )

## FP16 synthesis option


In [30]:
# ============================================================
# MOVIO — FP16 CACHED SYNTHESIS FOR AUDIO QUALITY TEST
# ============================================================

import torch
import numpy as np
from IPython.display import Audio, display

@torch.inference_mode()
def synthesize_cached_fp16(
    text,
    cache,
    num_step=3,
    speed=1.07,
    seed=666,
    guidance_scale=1.0,
    t_shift=0.5,
    max_duration=100,
):
    torch.manual_seed(seed)

    text = add_punctuation(text)

    tokens_str = tokenizer.texts_to_tokens([text])[0]

    token_duration = (
        cache["duration"]
        / (
            len(cache["tokens_str"])
            * speed
        )
    )

    max_tokens = int(
        (25 - cache["duration"])
        / token_duration
    )

    chunked_tokens_str = chunk_tokens_punctuation(
        tokens_str,
        max_tokens=max_tokens,
    )

    chunked_tokens = tokenizer.tokens_to_token_ids(
        chunked_tokens_str
    )

    tokens_batches, chunked_index = batchify_tokens(
        chunked_tokens,
        max_duration,
        cache["duration"],
        token_duration,
    )

    chunked_features = []

    for batch_tokens in tokens_batches:

        batch_size = len(batch_tokens)

        batch_prompt_tokens = (
            cache["tokens"]
            * batch_size
        )

        batch_prompt_features = (
            cache["features"]
            .expand(
                batch_size,
                -1,
                -1,
            )
            .contiguous()
        )

        batch_prompt_features_lens = torch.full(
            (batch_size,),
            cache["features"].size(1),
            device=DEVICE,
            dtype=torch.long,
        )

        # ====================================================
        # FP16 ONLY FOR MODEL INFERENCE
        # ====================================================

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            (
                pred_features,
                pred_features_lens,
                _,
                _,
            ) = model.model.sample(
                tokens=batch_tokens,
                prompt_tokens=batch_prompt_tokens,
                prompt_features=batch_prompt_features,
                prompt_features_lens=batch_prompt_features_lens,
                speed=speed,
                t_shift=t_shift,
                duration="predict",
                num_step=num_step,
                guidance_scale=guidance_scale,
            )

        # Return features to FP32 for vocoder safety
        pred_features = (
            pred_features
            .float()
            .permute(0, 2, 1)
            / FEAT_SCALE
        )

        chunked_features.append(
            (
                pred_features,
                pred_features_lens,
            )
        )

    chunked_wavs = []

    for (
        pred_features,
        pred_features_lens
    ) in chunked_features:

        for i in range(
            pred_features.size(0)
        ):

            length = int(
                pred_features_lens[i]
            )

            # Keep vocoder in its existing precision
            wav = vocoder.decode(
                pred_features[
                    i:i+1,
                    :,
                    :length,
                ]
            )

            wav = (
                wav
                .squeeze(1)
                .clamp(-1, 1)
            )

            if cache["rms"] < TARGET_RMS:
                wav = (
                    wav
                    * cache["rms"]
                    / TARGET_RMS
                )

            chunked_wavs.append(
                wav
            )

    indexed = list(
        zip(
            chunked_index,
            chunked_wavs,
        )
    )

    indexed.sort(
        key=lambda x: x[0]
    )

    ordered_wavs = [
        wav
        for _, wav in indexed
    ]

    final_wav = cross_fade_concat(
        ordered_wavs,
        fade_duration=0.1,
        sample_rate=SAMPLING_RATE,
    )

    final_wav = remove_silence(
        final_wav,
        SAMPLING_RATE,
        only_edge=True,
        trail_sil=80,
    )

    return (
        final_wav
        .detach()
        .float()
        .cpu()
        .squeeze()
        .numpy()
    )

## Production synthesis and smooth joins


In [31]:
def route_speed(route, group):
    if route == "TAMIL":
        return TANGLISH_SPEED if str(group).upper()=="TANGLISH" else TAMIL_SPEED
    return ENGLISH_SPEED

def production_synthesize(text, route, group, step, use_fp16=None):
    if use_fp16 is None:
        use_fp16 = USE_FP16_SAMPLE

    cache = ENGLISH_PROMPT_CACHE if route=="ENGLISH" else TAMIL_PROMPT_CACHE
    speed = route_speed(route, group)

    if use_fp16:
        return synthesize_cached_fp16(
            text=text, cache=cache, num_step=step,
            speed=speed, seed=SEED
        )

    return synthesize_cached(
        text=text, cache=cache, num_step=step,
        speed=speed, seed=SEED
    )

print("✓ Production synthesis uses cached in-memory path only")
print("✓ Step 2 is not used")
# ============================================================
# SMOOTH AUDIO JOINING
# Prevent chunk boundary artifacts
# ============================================================

import numpy as np

def smooth_join_audio(
    chunks,
    sr=24000,
    fade_ms=30
):

    if len(chunks) == 1:
        return np.asarray(
            chunks[0],
            dtype=np.float32
        )

    output = np.asarray(
        chunks[0],
        dtype=np.float32
    )

    fade_samples = int(
        sr * fade_ms / 1000
    )

    for chunk in chunks[1:]:

        chunk = np.asarray(
            chunk,
            dtype=np.float32
        )

        n = min(
            fade_samples,
            len(output),
            len(chunk)
        )

        if n == 0:
            output = np.concatenate(
                [output, chunk]
            )
            continue

        fade_out = np.linspace(
            1.0,
            0.0,
            n
        )

        fade_in = np.linspace(
            0.0,
            1.0,
            n
        )

        overlap = (
            output[-n:] * fade_out
            +
            chunk[:n] * fade_in
        )

        output = np.concatenate(
            [
                output[:-n],
                overlap,
                chunk[n:]
            ]
        )

    return output.astype(np.float32)


print("✓ Smooth chunk joining enabled")

✓ Production synthesis uses cached in-memory path only
✓ Step 2 is not used
✓ Smooth chunk joining enabled


## Micro-batch scheduler and streaming


In [32]:
# ============================================================
# LATENCY OPTIMIZATION — TRUE C1 MICRO-BATCH INFERENCE
# Quality is unchanged: same prompt, same speed, same steps.
# ============================================================

BATCH_WINDOW_MS = 2.0
MAX_BATCH_SIZE = 2

# Length bucketing avoids making a tiny C1 wait for a much longer
# acoustic sequence in the same padded batch.
def _target_token_count(text):
    clean = add_punctuation(str(text))
    return len(tokenizer.texts_to_tokens([clean])[0])

def _length_bucket(token_count):
    if token_count <= 16:
        return "S"
    if token_count <= 32:
        return "M"
    if token_count <= 64:
        return "L"
    return "XL"


@torch.inference_mode()
def synthesize_c1_batch(
    texts,
    route,
    group,
    step=FIRST_CHUNK_NUM_STEP,
    use_fp16=None,
):
    """
    Generate several independent C1 utterances in ONE model.sample() call.

    This does NOT reduce num_step or speech quality.
    It removes the old one-request-at-a-time GPU serialization.

    All texts in this call must share:
      - route/reference cache
      - speed
      - generation step
    """

    if not texts:
        return []

    if use_fp16 is None:
        use_fp16 = USE_FP16_SAMPLE

    texts = [str(x) for x in texts]
    cache = ENGLISH_PROMPT_CACHE if route == "ENGLISH" else TAMIL_PROMPT_CACHE
    speed = route_speed(route, group)
    batch_size = len(texts)

    torch.manual_seed(SEED)

    clean_texts = [add_punctuation(x) for x in texts]

    # Tokenize ALL requests together.
    target_tokens_str = tokenizer.texts_to_tokens(clean_texts)
    target_tokens = tokenizer.tokens_to_token_ids(target_tokens_str)

    # Repeat fixed prompt conditioning without recomputing it.
    prompt_tokens = cache["tokens"] * batch_size

    prompt_features = (
        cache["features"]
        .expand(batch_size, -1, -1)
        .contiguous()
    )

    prompt_features_lens = torch.full(
        (batch_size,),
        cache["features"].size(1),
        device=DEVICE,
        dtype=torch.long,
    )

    def _sample():
        return model.model.sample(
            tokens=target_tokens,
            prompt_tokens=prompt_tokens,
            prompt_features=prompt_features,
            prompt_features_lens=prompt_features_lens,
            speed=speed,
            t_shift=0.5,
            duration="predict",
            num_step=int(step),
            guidance_scale=1.0,
        )

    # FP16 remains quality-gated exactly as in the clean notebook.
    if bool(use_fp16):
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            pred_features, pred_features_lens, _, _ = _sample()
    else:
        pred_features, pred_features_lens, _, _ = _sample()

    # Vocoder stays FP32 / existing precision.
    pred_features = (
        pred_features
        .float()
        .permute(0, 2, 1)
        / FEAT_SCALE
    )

    results = []

    # Decode individually so each predicted length is respected exactly.
    # model.sample() is the dominant cost; this loop is normally small.
    for i in range(batch_size):
        length = int(pred_features_lens[i].item())

        wav = vocoder.decode(
            pred_features[i:i+1, :, :length]
        )

        wav = wav.squeeze(1).clamp(-1, 1)

        if cache["rms"] < TARGET_RMS:
            wav = wav * cache["rms"] / TARGET_RMS

        wav = remove_silence(
            wav,
            SAMPLING_RATE,
            only_edge=True,
            trail_sil=80,
        )

        results.append(
            wav.detach().float().cpu().squeeze().numpy()
        )

    return results


print("✓ Batched C1 model.sample() ready")
print("Batch window :", BATCH_WINDOW_MS, "ms")
print("Max batch    :", MAX_BATCH_SIZE)
print("C1 step      :", FIRST_CHUNK_NUM_STEP, "(quality preserved)")

# ============================================================
# ROUTE-AWARE + LENGTH-BUCKETED MICRO-BATCH SCHEDULER
# ============================================================

@dataclass(order=True)
class MovioBatchJob:
    sort_key: tuple
    seq: int
    created_at: float = field(compare=False)
    kind: str = field(compare=False)
    route: str = field(compare=False)
    group: str = field(compare=False)
    text: str = field(compare=False)
    step: int = field(compare=False)
    speed: float = field(compare=False)
    token_count: int = field(compare=False)
    bucket: str = field(compare=False)

    done: threading.Event = field(compare=False, default_factory=threading.Event)
    waveform: object = field(compare=False, default=None)
    error: object = field(compare=False, default=None)

    queue_wait_ms: float = field(compare=False, default=np.nan)
    service_ms: float = field(compare=False, default=np.nan)
    batch_size: int = field(compare=False, default=1)
    batch_id: int = field(compare=False, default=-1)


class MovioMicroBatchScheduler:
    def __init__(
        self,
        queue_limit=ADMISSION_QUEUE_LIMIT,
        batch_window_ms=BATCH_WINDOW_MS,
        max_batch_size=MAX_BATCH_SIZE,
    ):
        self.queue_limit = int(queue_limit)
        self.batch_window_ms = float(batch_window_ms)
        self.max_batch_size = int(max_batch_size)

        self.cv = threading.Condition()
        self.heap = []
        self.seq = 0
        self.batch_seq = 0
        self.running = False
        self.worker = None

    def start(self):
        if self.running:
            return
        self.running = True
        self.worker = threading.Thread(
            target=self._loop,
            daemon=True,
        )
        self.worker.start()

    def stop(self):
        with self.cv:
            self.running = False
            self.cv.notify_all()
        if self.worker is not None:
            self.worker.join(timeout=5)

    def submit(self, kind, route, group, text, step):
        now = time.perf_counter()
        priority = (
            FIRST_AUDIO_PRIORITY
            if kind == "C1"
            else CONTINUATION_PRIORITY
        )

        speed = route_speed(route, group)
        token_count = _target_token_count(text)
        bucket = _length_bucket(token_count)

        with self.cv:
            if len(self.heap) >= self.queue_limit:
                raise RuntimeError("Admission queue full")

            self.seq += 1

            # C1 first, then oldest request.
            job = MovioBatchJob(
                sort_key=(priority, now, self.seq),
                seq=self.seq,
                created_at=now,
                kind=kind,
                route=route,
                group=str(group).upper(),
                text=str(text),
                step=int(step),
                speed=float(speed),
                token_count=int(token_count),
                bucket=bucket,
            )

            heapq.heappush(self.heap, job)
            self.cv.notify_all()

        return job

    @staticmethod
    def _compatible(a, b):
        # Same conditioning + same generation policy.
        return (
            a.kind == b.kind
            and a.route == b.route
            and a.step == b.step
            and abs(a.speed - b.speed) < 1e-9
            and a.bucket == b.bucket
        )

    def _collect_compatible_locked(self, first):
        batch = [first]

        # Give simultaneous arrivals a tiny window to join.
        deadline = (
            time.perf_counter()
            + self.batch_window_ms / 1000.0
        )

        while len(batch) < self.max_batch_size:
            compatible_index = None

            for i, candidate in enumerate(self.heap):
                if self._compatible(first, candidate):
                    compatible_index = i
                    break

            if compatible_index is not None:
                candidate = self.heap.pop(compatible_index)
                heapq.heapify(self.heap)
                batch.append(candidate)
                continue

            remaining = deadline - time.perf_counter()
            if remaining <= 0:
                break

            self.cv.wait(timeout=remaining)

        return batch

    def _loop(self):
        while True:
            with self.cv:
                while self.running and not self.heap:
                    self.cv.wait(timeout=0.1)

                if not self.running and not self.heap:
                    break

                if not self.heap:
                    continue

                first = heapq.heappop(self.heap)
                batch = self._collect_compatible_locked(first)

            self.batch_seq += 1
            batch_id = self.batch_seq

            service_start = time.perf_counter()

            for j in batch:
                j.queue_wait_ms = (
                    service_start - j.created_at
                ) * 1000.0
                j.batch_size = len(batch)
                j.batch_id = batch_id

            try:
                # C1 uses true batched model.sample().
                if first.kind == "C1":
                    waveforms = synthesize_c1_batch(
                        texts=[j.text for j in batch],
                        route=first.route,
                        group=first.group,
                        step=first.step,
                        use_fp16=USE_FP16_SAMPLE,
                    )

                    torch.cuda.synchronize()

                    service_ms = (
                        time.perf_counter() - service_start
                    ) * 1000.0

                    for j, wav in zip(batch, waveforms):
                        j.waveform = wav
                        j.service_ms = service_ms

                else:
                    # Continuations remain lower priority.
                    # Keep quality path unchanged.
                    for j in batch:
                        t0 = time.perf_counter()
                        j.waveform = production_synthesize(
                            j.text,
                            j.route,
                            j.group,
                            j.step,
                        )
                        torch.cuda.synchronize()
                        j.service_ms = (
                            time.perf_counter() - t0
                        ) * 1000.0

            except Exception as e:
                for j in batch:
                    j.error = e

            finally:
                for j in batch:
                    j.done.set()


# Replace old scalar scheduler safely.
if "SCHEDULER" in globals():
    try:
        SCHEDULER.stop()
    except Exception:
        pass

SCHEDULER = MovioMicroBatchScheduler()
SCHEDULER.start()

print("✓ Route-aware micro-batch scheduler started")
print("✓ C1 requests can share one model.sample()")
print("✓ Length bucketing enabled")
print("✓ C1 quality step unchanged:", FIRST_CHUNK_NUM_STEP)
def movio_stream_response(raw_text,group="ENTITY"):
    request_start=time.perf_counter()
    p=prepare_movio_request(raw_text,group)

    route=p["route"]
    chunks=p["chunks"]

    c1=SCHEDULER.submit(
        "C1",route,group,chunks[0],FIRST_CHUNK_NUM_STEP
    )
    c1.done.wait()
    if c1.error:
        raise c1.error

    ttfa_ms=(time.perf_counter()-request_start)*1000

    # Pre-submit C2 BEFORE yielding C1.
    next_job=None
    if len(chunks)>1:
        next_job=SCHEDULER.submit(
            "CONT",route,group,chunks[1],REST_CHUNK_NUM_STEP
        )

    yield {
        "chunk_index":1,
        "chunk_text":chunks[0],
        "audio":np.asarray(c1.waveform,dtype=np.float32),
        "sample_rate":SAMPLING_RATE,
        "route":route,
        "normalized":p["normalized"],
        "ttfa_ms":ttfa_ms,
        "queue_ms":c1.queue_wait_ms,
        "service_ms":c1.service_ms
    }

    for idx in range(1,len(chunks)):
        current=next_job
        current.done.wait()
        if current.error:
            raise current.error

        following=None
        if idx+1<len(chunks):
            following=SCHEDULER.submit(
                "CONT",route,group,chunks[idx+1],REST_CHUNK_NUM_STEP
            )

        yield {
            "chunk_index":idx+1,
            "chunk_text":chunks[idx],
            "audio":np.asarray(current.waveform,dtype=np.float32),
            "sample_rate":SAMPLING_RATE,
            "route":route,
            "normalized":p["normalized"],
            "ttfa_ms":ttfa_ms,
            "queue_ms":current.queue_wait_ms,
            "service_ms":current.service_ms
        }
        next_job=following

print("✓ Streaming path ready")

✓ Batched C1 model.sample() ready
Batch window : 2.0 ms
Max batch    : 2
C1 step      : 3 (quality preserved)
✓ Route-aware micro-batch scheduler started
✓ C1 requests can share one model.sample()
✓ Length bucketing enabled
✓ C1 quality step unchanged: 3
✓ Streaming path ready


In [33]:
import sys
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle Settings, then rerun.")

print("GPU:", torch.cuda.get_device_name(0))

# Preserve the existing Torch and Transformers installations.
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir", "--force-reinstall", "--no-deps",
    "bitsandbytes==0.50.2",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-deps", "accelerate",
])

print("Installation finished. Restart the notebook kernel once.")

GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 209.4 MB/s eta 0:00:00
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.50.2
    Uninstalling bitsandbytes-0.50.2:
      Successfully uninstalled bitsandbytes-0.50.2
Installation finished. Restart the notebook kernel once.


## 16. Connect original TTS and load Qwen


In [34]:
import importlib.util, sys

def load_module(name, file):
    spec = importlib.util.spec_from_file_location(name, str(file))
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

adapter = load_module('movio_original_tts_adapter',
                      TARGET / 'integrations/movio_original_tts_adapter.py')
movio_tts = adapter.OriginalMovioTTS(globals())
print('Original MOVIO TTS adapter ready.')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import is_bitsandbytes_available

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle Settings.')
if globals().get('llm_model') is not None and globals().get('llm_tokenizer') is not None:
    print('Reusing the existing Qwen model and tokenizer.')
else:
    if not is_bitsandbytes_available():
        raise RuntimeError('bitsandbytes is unavailable. Restart the Kaggle session after installation, then rerun.')
    MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
    )
    llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    llm_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=HF_TOKEN, quantization_config=quant_config,
        device_map='auto', dtype=torch.float16, low_cpu_mem_usage=True
    ).eval()
    print('Qwen loaded successfully.')


Original MOVIO TTS adapter ready.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully.


## 17. Load updated voice bridge


In [35]:
import re
import psutil

def listener_pids(port):
    return sorted({c.pid for c in psutil.net_connections(kind='inet')
                   if c.status == 'LISTEN' and c.laddr.port == port and c.pid})

voice_active = globals().get('voice_server_thread')
if voice_active is not None and voice_active.is_alive():
    if globals().get('bridge') is None:
        raise RuntimeError('An existing voice thread has no bridge variable. Use its original notebook session.')
    print('Reusing the live Sarvam backend in this session.')
else:
    if listener_pids(8000):
        raise RuntimeError('Port 8000 is occupied by another service. Use the notebook that owns it.')
    bridge = load_module('ai_call_sarvam', TARGET / 'integrations/ai_call_sarvam.py')

# Apply the sedan alias patch once; repeated execution cannot create recursion.
if not hasattr(bridge, '_movio_original_vehicle'):
    bridge._movio_original_vehicle = bridge.vehicle

def movio_vehicle_fixed(text, *args, **kwargs):
    if isinstance(text, str) and re.search(
        r'\b(?:sedan|sudden|sadan|se dan)\b|செடான்|சிடான்', text, re.IGNORECASE
    ):
        return 'Sedan'
    return bridge._movio_original_vehicle(text, *args, **kwargs)

bridge.vehicle = movio_vehicle_fixed
print('Sarvam backend ready:', bridge.app.title)

assert bridge.VOICE_PATCH_VERSION == '5.2', 'Voice patch 5.2 was not installed'


Sarvam backend ready: AI Taxi Call — Sarvam


## 18. Check all three TTS greetings


In [36]:
import threading, time, requests, uvicorn

def configure_bridge(public_url):
    bridge.configure(
        public_url, tts=movio_tts, preprocessing=None,
        llm=llm_model, tokenizer=llm_tokenizer,
        reference_wav=REFERENCE_TAMIL, sarvam_key=SARVAM_API_KEY
    )

configure_bridge(globals().get('MOVIO_WEBSITE_URL', ''))

for language in ('English', 'Tamil', 'Tanglish'):
    print('Checking greeting:', language, flush=True)
    token = bridge.CURRENT_LANGUAGE.set(language)
    try:
        greeting = bridge.speak(bridge.MESSAGES[language]['hello'])
        if not greeting:
            raise RuntimeError(language + ' returned no audio')
        print(language + ' greeting: PASS', flush=True)
    finally:
        bridge.CURRENT_LANGUAGE.reset(token)


Checking greeting: English
English greeting: PASS
Checking greeting: Tamil
Tamil greeting: PASS
Checking greeting: Tanglish
Tanglish greeting: PASS


## 19. Start voice service


In [37]:
import threading, time, requests, uvicorn

def configure_bridge(public_url):
    bridge.configure(
        public_url, tts=movio_tts, preprocessing=None,
        llm=llm_model, tokenizer=llm_tokenizer,
        reference_wav=REFERENCE_TAMIL, sarvam_key=SARVAM_API_KEY
    )

configure_bridge(globals().get('MOVIO_WEBSITE_URL', ''))
if not globals().get('voice_server_thread') or not voice_server_thread.is_alive():
    voice_uvicorn_server = uvicorn.Server(uvicorn.Config(
        bridge.app, host='127.0.0.1', port=8000, log_level='warning'
    ))
    voice_server_thread = threading.Thread(target=voice_uvicorn_server.run, daemon=True)
    voice_server_thread.start()

for _ in range(60):
    if not voice_server_thread.is_alive():
        raise RuntimeError('AI server stopped. See the error printed above.')
    try:
        response = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if response.status_code == 200:
            print('AI voice service:', response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError('AI service did not become ready on port 8000.')


AI voice service: {'ok': True, 'stt': 'Sarvam Saaras v3', 'tts': 'User MOVIO optimized DhVaani (strict)', 'llm': 'Qwen optional', 'mode': 'browser demo'}


## 20. Start shared gateway


In [38]:
import os, sys, subprocess, time

GATEWAY_FILE = Path('/kaggle/working/movio_gateway.py')
gateway_source = r'''
import asyncio
import os
from pathlib import Path
import httpx
import uvicorn
import websockets
from fastapi import FastAPI, WebSocket, Request
from fastapi.responses import Response, FileResponse

app = FastAPI()
WEB_HTTP = 'http://127.0.0.1:8080'
AI_HTTP = 'http://127.0.0.1:8000'
CALL_PAGE = Path(os.environ['MOVIO_SITE_DIR']) / 'integrations/ai_call_page.html'
HOP = {'host', 'connection', 'content-length', 'transfer-encoding', 'keep-alive',
       'proxy-authenticate', 'proxy-authorization', 'te', 'trailer', 'upgrade'}

@app.get('/gateway-health')
def gateway_health():
    return {'ok': True, 'service': 'movio-gateway'}

@app.get('/call-page')
def call_page():
    return FileResponse(CALL_PAGE, media_type='text/html', headers={'Cache-Control': 'no-store'})

@app.websocket('/browser-call')
async def voice_socket(ws: WebSocket):
    origin = ws.headers.get('origin', '').rstrip('/')
    if not origin or origin != 'https://' + ws.headers.get('host', ''):
        await ws.close(code=1008)
        return
    query = str(ws.url.query)
    target = 'ws://127.0.0.1:8000/browser-call' + ('?' + query if query else '')
    try:
        async with websockets.connect(target, origin=origin, max_size=10_000_000) as upstream:
            await ws.accept()
            async def client_to_ai():
                while True:
                    message = await ws.receive()
                    if message['type'] == 'websocket.disconnect':
                        return
                    if message.get('text') is not None:
                        await upstream.send(message['text'])
                    elif message.get('bytes') is not None:
                        await upstream.send(message['bytes'])
            async def ai_to_client():
                async for message in upstream:
                    if isinstance(message, bytes):
                        await ws.send_bytes(message)
                    else:
                        await ws.send_text(message)
            tasks = [asyncio.create_task(client_to_ai()), asyncio.create_task(ai_to_client())]
            try:
                await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
            finally:
                for task in tasks:
                    if not task.done():
                        task.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)
    except Exception as error:
        print('WebSocket gateway error:', type(error).__name__, flush=True)
        try:
            await ws.close(code=1011)
        except Exception:
            pass

@app.api_route('/{path:path}', methods=['GET', 'HEAD', 'POST', 'PUT', 'PATCH', 'DELETE', 'OPTIONS'])
async def proxy(request: Request, path: str):
    backend = AI_HTTP if path == 'health' else WEB_HTTP
    url = backend + request.url.path
    if request.url.query:
        url += '?' + request.url.query
    headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP}
    headers['accept-encoding'] = 'identity'
    async with httpx.AsyncClient(timeout=120) as client:
        upstream = await client.request(request.method, url, headers=headers,
                                        content=await request.body(), follow_redirects=False)
    response = Response(content=upstream.content, status_code=upstream.status_code)
    excluded = HOP | {'content-encoding'}
    for key, value in upstream.headers.multi_items():
        if key.lower() not in excluded:
            response.headers.append(key, value)
    return response

if __name__ == '__main__':
    uvicorn.run(app, host='127.0.0.1', port=8090, log_level='warning')
'''

def stop_owned_listener(port, allowed_scripts):
    processes = []
    for pid in listener_pids(port):
        if pid == os.getpid():
            raise RuntimeError(f'Port {port} runs inside the notebook; stopping it would unload models.')
        proc = psutil.Process(pid)
        command = proc.cmdline()
        script_args = [arg for arg in command if Path(arg).name in {p.name for p in allowed_scripts}]
        matches = []
        for arg in script_args:
            script = Path(arg)
            if not script.is_absolute():
                script = Path(proc.cwd()) / script
            matches.append(script.resolve())
        if not any(script in allowed_scripts for script in matches):
            raise RuntimeError(f'Port {port} belongs to an unrelated process (PID {pid}); it was not stopped.')
        processes.append(proc)
    for proc in processes:
        proc.terminate()
        try:
            proc.wait(timeout=8)
        except psutil.TimeoutExpired:
            raise RuntimeError(f'PID {proc.pid} did not stop; no forced kill performed.')

def start_process(script, cwd, env, log_path):
    offset = log_path.stat().st_size if log_path.exists() else 0
    with log_path.open('a') as log:
        proc = subprocess.Popen([sys.executable, '-u', str(script)], cwd=str(cwd), env=env,
                                stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    return proc, offset

def wait_http(url, process, log_path, offset):
    for _ in range(60):
        if process.poll() is not None:
            break
        try:
            response = requests.get(url, timeout=2)
            if response.status_code == 200:
                return response
        except requests.RequestException:
            pass
        time.sleep(0.5)
    with log_path.open() as log:
        log.seek(offset)
        print(log.read()[-4000:])
    raise RuntimeError('Service did not start: ' + url)

stop_owned_listener(8090, {GATEWAY_FILE.resolve()})
GATEWAY_FILE.write_text(gateway_source, encoding='utf-8')
compile(gateway_source, str(GATEWAY_FILE), 'exec')
gateway_log = Path('/kaggle/working/movio_gateway.log')
gateway_env = {**os.environ, 'MOVIO_SITE_DIR': str(TARGET)}
movio_gateway_process, gateway_offset = start_process(
    GATEWAY_FILE, Path('/kaggle/working'), gateway_env, gateway_log
)
print('Gateway ready:', wait_http('http://127.0.0.1:8090/gateway-health',
                                  movio_gateway_process, gateway_log, gateway_offset).json())


Gateway ready: {'ok': True, 'service': 'movio-gateway'}


## 21. Create the shared HTTPS tunnel


In [39]:

from pyngrok import ngrok
from urllib.parse import urlparse

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Retrieve existing tunnels
tunnels = ngrok.get_tunnels()

def tunnel_port(tunnel):
    addr = str(tunnel.config.get("addr", ""))
    if addr.isdigit():
        return int(addr)
    try:
        parsed = urlparse(addr if "://" in addr else "http://" + addr)
        return parsed.port
    except (ValueError, TypeError):
        return None

# Reuse existing HTTPS gateway tunnel
existing = [
    t for t in tunnels
    if tunnel_port(t) == 8090
    and t.public_url.startswith("https://")
]

if existing:
    MOVIO_WEBSITE_URL = existing[0].public_url.rstrip("/")
    print("Reusing existing HTTPS gateway tunnel.")

else:
    # Disconnect only this project's previous tunnels
    for tunnel in tunnels:
        if tunnel_port(tunnel) in (8000, 8080, 8090):
            ngrok.disconnect(tunnel.public_url)

    try:
        # Create gateway tunnel without deprecated bind_tls
        tunnel = ngrok.connect(addr=8090, proto="http")
        MOVIO_WEBSITE_URL = tunnel.public_url.rstrip("/")

    except Exception as error:
        raise RuntimeError(
            f"ngrok gateway tunnel creation failed: {error}"
        ) from error

# Verify HTTPS for microphone and GPS permissions
if not MOVIO_WEBSITE_URL.startswith("https://"):
    raise RuntimeError(
        "HTTPS tunnel required for browser microphone and GPS access."
    )

# Configure shared gateway
AI_PUBLIC_URL = MOVIO_WEBSITE_URL
configure_bridge(MOVIO_WEBSITE_URL)

print("Shared HTTPS URL:", MOVIO_WEBSITE_URL)
print("AI Public URL:", AI_PUBLIC_URL)
print("Gateway Port: 8090")


Shared HTTPS URL: https://army-grandson-backside.ngrok-free.dev                                     
AI Public URL: https://army-grandson-backside.ngrok-free.dev
Gateway Port: 8090


## 22. Start website


In [40]:
if not (TARGET / 'server.py').is_file():
    raise RuntimeError('Run the website setup cell first.')
stop_owned_listener(8080, {(TARGET / 'server.py').resolve()})
website_env = {
    **os.environ, 'MOVIO_PORT': '8080', 'MOVIO_PUBLIC_URL': MOVIO_WEBSITE_URL,
    'MOVIO_AI_BRIDGE_URL': MOVIO_WEBSITE_URL, 'AI_PUBLIC_URL': MOVIO_WEBSITE_URL,
    'PYTHONUNBUFFERED': '1'
}
website_log = Path('/kaggle/working/movio_website.log')
movio_website_process, website_offset = start_process(
    TARGET / 'server.py', TARGET, website_env, website_log
)
response = wait_http('http://127.0.0.1:8080/api/health',
                     movio_website_process, website_log, website_offset)
print('Website ready: HTTP', response.status_code)
print('Website folder:', TARGET)


Website ready: HTTP 200
Website folder: /kaggle/working/MOVIO_MAIN_WEBSITE


## Public website routing repair

Run this cell immediately before the website verification below, including if the public website returned HTTP 404 while the AI health/call page worked. It checks the local website and shared gateway, prevents loopback HTTP from using an external proxy, and reconnects only MOVIO's tunnel to port 8090 when needed. The existing public hostname is retained when available. A 401/403 response from the unauthenticated voice-configuration endpoint is expected.

The GPU models remain loaded. If the public hostname changes, only the website process/configuration is refreshed. Existing accounts, database, UI assets and evaluation cells are retained. Close any old call popup and refresh the website after a successful repair.


In [41]:
# Run once before the website verification cell. GPU models stay in this kernel.
import gc
import os
import time
import uuid
import weakref
from pathlib import Path
from urllib.parse import urlparse
from http.server import ThreadingHTTPServer
import requests
import psutil
from pyngrok import ngrok


def movio_repair_public_routing():
    required = ('TARGET', 'GATEWAY_FILE', 'stop_owned_listener',
                'start_process', 'wait_http', 'listener_pids', 'bridge')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Run the original service setup first: ' + ', '.join(missing))

    root = Path(TARGET).resolve()
    gateway_file = Path(GATEWAY_FILE).resolve()
    local = requests.Session()
    local.trust_env = False  # Send loopback checks directly to this notebook.
    public = requests.Session()
    public.headers.update({'ngrok-skip-browser-warning': 'true', 'Cache-Control': 'no-cache'})

    def get(base, path):
        session = local if urlparse(base).hostname in ('127.0.0.1', 'localhost') else public
        return session.get(base.rstrip('/') + path, timeout=(5, 10), allow_redirects=False)

    def require(base, path, statuses=(200,)):
        response = get(base, path)
        if response.status_code not in statuses:
            raise RuntimeError(f'{base}{path}: HTTP {response.status_code}; expected {statuses}.')
        return response

    def gateway_reachable(base):
        try:
            return (get(base, '/gateway-health').json().get('service') == 'movio-gateway'
                    and get(base, '/api/health').status_code == 200
                    and get(base, '/').status_code == 200)
        except (requests.RequestException, ValueError, AttributeError):
            return False

    try:
        # Confirm the actual website works before changing its public route.
        for path in ('/', '/api/health'):
            require('http://127.0.0.1:8080', path)
        require('http://127.0.0.1:8000', '/health')

        original = gateway_file.read_text(encoding='utf-8')
        patched = original.replace('httpx.AsyncClient(timeout=120)',
                                   'httpx.AsyncClient(timeout=120, trust_env=False)')
        compile(patched, str(gateway_file), 'exec')
        if patched != original or not gateway_reachable('http://127.0.0.1:8090'):
            # The existing helper verifies process ownership before stopping it.
            stop_owned_listener(8090, {gateway_file})
            if patched != original:
                backup = gateway_file.with_name(gateway_file.name + '.before_route_fix')
                if not backup.exists():
                    backup.write_text(original, encoding='utf-8')
                gateway_file.write_text(patched, encoding='utf-8')
            env = {**os.environ, 'MOVIO_SITE_DIR': str(root)}
            for key in ('NO_PROXY', 'no_proxy'):
                env[key] = ','.join(filter(None, (env.get(key, ''), '127.0.0.1', 'localhost')))
            log = gateway_file.with_suffix('.log')
            process, offset = start_process(gateway_file, gateway_file.parent, env, log)
            globals()['movio_gateway_process'] = process
            wait_http('http://127.0.0.1:8090/gateway-health', process, log, offset)
        if not gateway_reachable('http://127.0.0.1:8090'):
            raise RuntimeError('The local gateway cannot serve the website. Check movio_gateway.log; no tunnel was changed.')
        globals()['gateway_source'] = patched

        old_url = str(globals().get('MOVIO_WEBSITE_URL', '')).rstrip('/')
        new_url = old_url
        if not old_url.startswith('https://') or not gateway_reachable(old_url):
            def owned_port(tunnel):
                data = getattr(tunnel, 'data', {}) or {}
                upstream = data.get('upstream') or {}
                if not isinstance(upstream, dict):
                    upstream = {}
                addr = str((getattr(tunnel, 'config', {}) or {}).get('addr')
                           or upstream.get('url') or data.get('forwards_to') or '')
                if addr.isdigit():
                    return int(addr)
                try:
                    parsed = urlparse(addr if '://' in addr else 'http://' + addr)
                    return parsed.port if parsed.hostname in ('localhost', '127.0.0.1', '[::1]', '::1') else None
                except ValueError:
                    return None

            tunnels = ngrok.get_tunnels()
            candidates = [t.public_url.rstrip('/') for t in tunnels
                          if owned_port(t) == 8090 and t.public_url.startswith('https://')]
            new_url = next((url for url in candidates if gateway_reachable(url)), '')
            if not new_url:
                # Disconnect only MOVIO's local ports/current URL, never all ngrok processes.
                for tunnel in tunnels:
                    if owned_port(tunnel) in (8000, 8080, 8090) or tunnel.public_url.rstrip('/') == old_url:
                        ngrok.disconnect(tunnel.public_url)
                options = {'bind_tls': True}
                old_host = urlparse(old_url).hostname
                if old_url.startswith('https://') and old_host:
                    options['domain'] = old_host  # Retain the existing public hostname.
                tunnel = ngrok.connect(addr='http://127.0.0.1:8090', proto='http',
                                       name='movio-gateway-' + uuid.uuid4().hex[:8], **options)
                new_url = tunnel.public_url.rstrip('/')
            if not new_url.startswith('https://'):
                raise RuntimeError('ngrok did not return HTTPS; browser microphone access requires HTTPS.')
            for attempt in range(5):
                if gateway_reachable(new_url):
                    break
                if attempt < 4:
                    time.sleep(1)
            else:
                raise RuntimeError('Public URL still does not reach the verified gateway. Check ngrok endpoint/traffic-policy settings; no 404 was ignored.')

        urls = {'MOVIO_PUBLIC_URL': new_url, 'MOVIO_AI_BRIDGE_URL': new_url, 'AI_PUBLIC_URL': new_url}
        previous_config = dict(globals().get('website_env', {}))
        must_update_website = any(previous_config.get(key) != value for key, value in urls.items())
        website_config = {**os.environ, **previous_config, **urls, 'MOVIO_PORT': '8080', 'PYTHONUNBUFFERED': '1'}
        os.environ.update(urls)
        globals().update(MOVIO_WEBSITE_URL=new_url, AI_PUBLIC_URL=new_url, website_env=website_config)
        bridge.PUBLIC_URL = new_url  # Change only the origin; retain TTS/LLM/weather setup.

        if must_update_website:
            if os.getpid() in listener_pids(8080):
                found = False
                for obj in gc.get_objects():
                    try:
                        if type(obj) in weakref.ProxyTypes or not isinstance(obj, ThreadingHTTPServer):
                            continue
                        if obj.server_address[1] != 8080:
                            continue
                        namespace = obj.RequestHandlerClass.__init__.__globals__
                        known_root = namespace.get('ROOT')
                        if known_root and Path(known_root).resolve() == root:
                            namespace.update(PUBLIC=new_url, AI_BRIDGE_URL=new_url)
                            found = True
                    except (ReferenceError, AttributeError, TypeError, OSError):
                        continue
                if not found:
                    raise RuntimeError('Website runs in this kernel but its handler was not identified. Rerun your website-only launcher with the printed public URL.')
            else:
                server_file = (root / 'server.py').resolve()
                # Read settings only from the verified website process; preserve its secrets.
                for pid in listener_pids(8080):
                    process = psutil.Process(pid)
                    scripts = [Path(a) if Path(a).is_absolute() else Path(process.cwd()) / a
                               for a in process.cmdline() if a.endswith('.py')]
                    if server_file not in {p.resolve() for p in scripts}:
                        raise RuntimeError('An unrelated application owns port 8080; it was left running.')
                    website_config.update(process.environ())
                website_config.update(urls)
                website_config['MOVIO_PORT'] = '8080'
                stop_owned_listener(8080, {server_file})
                log = root.parent / 'movio_website.log'
                process, offset = start_process(server_file, root, website_config, log)
                globals()['movio_website_process'] = process
                wait_http('http://127.0.0.1:8080/api/health', process, log, offset)

        for label, path, statuses in (
            ('Public gateway', '/gateway-health', (200,)),
            ('Public website', '/', (200,)),
            ('Public website health', '/api/health', (200,)),
            ('Public AI health', '/health', (200,)),
            ('Public call page', '/call-page', (200,)),
            ('Protected voice configuration', '/api/browser-voice-config', (401, 403)),
        ):
            response = require(new_url, path, statuses)
            print(f'{label}: HTTP {response.status_code}', flush=True)
        print('Website:', new_url)
        print('Routing checks passed. Rerun the original verification cell, then continue below it.')
        return {'ok': True, 'website_url': new_url, 'gateway_port': 8090}
    finally:
        local.close()
        public.close()


MOVIO_ROUTING_STATUS = movio_repair_public_routing()


Public gateway: HTTP 200
Public website: HTTP 200
Public website health: HTTP 200
Public AI health: HTTP 200
Public call page: HTTP 200
Protected voice configuration: HTTP 401
Website: https://army-grandson-backside.ngrok-free.dev
Routing checks passed. Rerun the original verification cell, then continue below it.


## 23. Verify routing and open the dashboard


In [42]:
import shutil
from IPython.display import display, Markdown

checks = [
    ('Local website', 'http://127.0.0.1:8080/api/health'),
    ('Local AI', 'http://127.0.0.1:8000/health'),
    ('Gateway', 'http://127.0.0.1:8090/gateway-health'),
    ('Public website', MOVIO_WEBSITE_URL + '/'),
    ('Public website health', MOVIO_WEBSITE_URL + '/api/health'),
    ('Public AI health', MOVIO_WEBSITE_URL + '/health'),
    ('Public call page', MOVIO_WEBSITE_URL + '/call-page'),
]
failures = []
for label, url in checks:
    try:
        response = requests.get(url, timeout=(5, 10), headers={'ngrok-skip-browser-warning': 'true'})
        ok = response.status_code == 200
        print(label + ': HTTP ' + str(response.status_code), flush=True)
        if not ok:
            failures.append(label)
    except requests.RequestException as error:
        print(label + ': ' + type(error).__name__)
        failures.append(label)

config = requests.get('http://127.0.0.1:8080/api/browser-voice-config', timeout=5)
if config.status_code not in (401, 403):
    failures.append('Account protection for browser voice configuration')
print('Browser voice configuration requires website login: HTTP', config.status_code)
print('Configured call page:', website_env['MOVIO_AI_BRIDGE_URL'] + '/call-page')

node = shutil.which('node')
for name in ['app.js', 'auth.js', 'movio_enhancements.js']:
    file = TARGET / 'web' / name
    if node:
        result = subprocess.run([node, '--check', str(file)], capture_output=True, text=True)
        print(name, 'VALID' if result.returncode == 0 else result.stderr)
        if result.returncode:
            failures.append(name)
    else:
        print(name, 'FOUND; Node.js unavailable, syntax check skipped')
if failures:
    raise RuntimeError('Checks failed: ' + ', '.join(failures))

display(Markdown(
    f'**Website:** [{MOVIO_WEBSITE_URL}]({MOVIO_WEBSITE_URL})\n\n'
    'Start AI Call using the button on your logged-in dashboard.\n\n'
    'Open the website, register a customer and a driver in separate browser profiles, '
    'choose a matching vehicle type, and test accept/cancel, driver tracking and '
    'the customer location permission popup. Allow microphone/GPS only when prompted.'
))


Local website: HTTP 200
Local AI: HTTP 200
Gateway: HTTP 200
Public website: HTTP 200
Public website health: HTTP 200
Public AI health: HTTP 200
Public call page: HTTP 200
Browser voice configuration requires website login: HTTP 401
Configured call page: https://army-grandson-backside.ngrok-free.dev/call-page
app.js VALID
auth.js VALID
movio_enhancements.js VALID


**Website:** [https://army-grandson-backside.ngrok-free.dev](https://army-grandson-backside.ngrok-free.dev)

Start AI Call using the button on your logged-in dashboard.

Open the website, register a customer and a driver in separate browser profiles, choose a matching vehicle type, and test accept/cancel, driver tracking and the customer location permission popup. Allow microphone/GPS only when prompted.

## Later: check services without reloading models
Run only the next cell for local health and the current tunnel list. Do not open the call page directly; use the logged-in dashboard's AI Call button. A DNS error in Chrome requires checking the current tunnel and your browser/network; restarting TTS does not fix DNS.


In [43]:
import requests

session = requests.Session()
session.trust_env = False
for port, route in [(8080, '/api/health'), (8000, '/health'), (8090, '/gateway-health')]:
    try:
        response = session.get(f'http://127.0.0.1:{port}{route}', timeout=(2, 3))
        print(port, response.status_code, response.text[:200], flush=True)
    except requests.RequestException as error:
        print(port, type(error).__name__, str(error), flush=True)
try:
    response = session.get('http://127.0.0.1:4040/api/tunnels', timeout=(2, 3))
    response.raise_for_status()
    for tunnel in response.json().get('tunnels', []):
        print('Public URL:', tunnel.get('public_url'), 'Local:', tunnel.get('config', {}).get('addr'), flush=True)
except requests.RequestException as error:
    print('Tunnel list unavailable:', type(error).__name__, str(error), flush=True)


8080 200 {"ok": true, "version": "5.2", "calling_configured": false, "tts_configured": false, "stt_configured": false, "stream_configured": false, "ai_call_configured": false}
8000 200 {"ok":true,"stt":"Sarvam Saaras v3","tts":"User MOVIO optimized DhVaani (strict)","llm":"Qwen optional","mode":"browser demo"}
8090 200 {"ok":true,"service":"movio-gateway"}
Public URL: https://army-grandson-backside.ngrok-free.dev Local: http://localhost:8090


In [44]:
from pathlib import Path
import datetime, shutil

CALL_FILE = Path('/kaggle/working/MOVIO_MAIN_WEBSITE/integrations/ai_call_page.html')
assert CALL_FILE.is_file(), 'The running website call page was not found.'
html = CALL_FILE.read_text(encoding='utf-8')

DIRECT_BOOKING_JS = r'''
// Authenticated same-origin booking: no dependency on an opener tab.
async function callAccountAPI(path, options={}) {
 const controller=new AbortController();
 const timer=setTimeout(()=>controller.abort(),15000);
 try {
  const response=await fetch('/api'+path,{...options,credentials:'same-origin',signal:controller.signal,headers:{'Content-Type':'application/json',...(options.headers||{})}});
  let result;try{result=await response.json()}catch{throw Error('Booking server returned an unreadable response (HTTP '+response.status+').')}
  if(!response.ok)throw Error(result.error||'Booking failed (HTTP '+response.status+').');
  return result;
 }catch(error){if(error.name==='AbortError')throw Error('Booking server timed out. Check booking history before trying again.');throw error}
 finally{clearTimeout(timer)}
}
let accountActionBusy=false;
async function saveCallAction(action) {
 if(accountActionBusy)return;
 accountActionBusy=true;
 const socket=ws;
 try {
  if(!socket||socket.readyState!==1)throw Error('Call connection has closed.');
  const account=await callAccountAPI('/me');
  if(account.role!=='customer')throw Error('Sign in with a customer account to book a ride.');
  let result;
  if(action.type==='book') {
   const d=action.details||{},p=d.pickup_geo,q=d.destination_geo;
   if(!p||!q||!d.fare||!Number.isFinite(d.fare.fare))throw Error('Verified route and fare are required.');
   if(!['Bike','Auto','Mini','Sedan','SUV'].includes(d.vehicle))throw Error('Invalid vehicle type.');
   if(!action.request_id)throw Error('Voice backend update required: booking request ID is missing.');
   result=await callAccountAPI('/rides',{method:'POST',body:JSON.stringify({pickup:d.pickup,destination:d.destination,plat:p.lat,plng:p.lng,dlat:q.lat,dlng:q.lng,vehicle:d.vehicle,fare:d.fare.fare,pickup_time:d.when,passengers:d.count,request_key:action.request_id})});
  }else if(action.type==='cancel') {
   const id=Number(action.ride_id);
   if(!Number.isInteger(id)||id<1)throw Error('Invalid booking number.');
   await callAccountAPI('/rides/'+id+'/status',{method:'POST',body:JSON.stringify({status:'cancelled'})});
   result={id};
  }else throw Error('Unsupported booking action.');
  setStatus('Account updated — waiting for AI confirmation');
  log('Account','Booking #'+result.id+(action.type==='cancel'?' cancelled.':' saved.'));
  if(ws===socket&&socket.readyState===1)socket.send(JSON.stringify({event:'result',ok:true,id:result.id}));
 }catch(error){
  log('Error',error.message);setStatus('Account update failed — '+error.message);
  if(ws===socket&&socket?.readyState===1)socket.send(JSON.stringify({event:'result',ok:false,error:error.message}));
 }finally{accountActionBusy=false}
}
async function loadCallAccount() {
 const account=await callAccountAPI('/me');
 if(account.role!=='customer')throw Error('Sign in with a customer account on this website first.');
 const result=await callAccountAPI('/rides');
 rideList=(result.rides||[]).filter(r=>['requested','accepted','arriving','in_progress','needs_location'].includes(r.status));
 showRides();
}
'''


marker = '// Authenticated same-origin booking: no dependency on an opener tab.'
if marker not in html:
    old = "notifyParent('action',{action:m});return"
    assert old in html, 'Unexpected call-page version. No changes were made.'
    html = html.replace(old, "saveCallAction(m);return", 1)
    old_start = "try{if(!$('lang').value)throw Error('Choose a language first');setStatus('Requesting microphone…');"
    assert old_start in html, 'Expected language-first call page was not found.'
    html = html.replace(old_start, "try{if(!$('lang').value)throw Error('Choose a language first');setStatus('Checking your account…');await loadCallAccount();setStatus('Requesting microphone…');", 1)
    assert '</script>' in html
    html = html.replace('</script>', DIRECT_BOOKING_JS + '\n</script>', 1)
    html = html.replace('MOVIO<span style="color:#e2a600">.</span>', 'AI Ride Call')
    html = html.replace('Return to MOVIO', 'Return to Dashboard')
    html = html.replace('MOVIO பக்கத்திற்குத் திரும்பு', 'முகப்புப் பக்கத்திற்குத் திரும்பு')
    html = html.replace('MOVIO ku Thirumbu', 'Dashboard ku Thirumbu')
    html = html.replace('logged-in MOVIO dashboard', 'logged-in dashboard')
    backup = CALL_FILE.with_name('ai_call_page.backup_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '.html')
    shutil.copy2(CALL_FILE, backup)
    CALL_FILE.write_text(html, encoding='utf-8')
    print('Call page patched. Backup:', backup)
else:
    print('Direct account booking is already installed.')
if globals().get('bridge') is not None:
    bridge.HTML = html
print('Close the call popup. Refresh the dashboard and start AI Call again.')
print('Keep using the shared gateway URL. Models and database were not restarted.')


Call page patched. Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/ai_call_page.backup_20261028_070646.html
Close the call popup. Refresh the dashboard and start AI Call again.
Keep using the shared gateway URL. Models and database were not restarted.


In [45]:
from pathlib import Path
import shutil

prompts = {
    "English": "Which vehicle would you like: bike, auto, mini, sedan or SUV?",
    "Tamil": "பைக், ஆட்டோ, மினி கார், செடான் அல்லது எஸ் யூ வி, எந்த வாகனம் வேண்டும்?",
    "Tanglish": "Bike, auto, mini car, sedan, SUV, edhu venum?",
}

for language, prompt in prompts.items():
    bridge.MESSAGES[language]["vehicle"] = prompt
    bridge.VOICE_MESSAGE_UPDATES[language]["vehicle"] = prompt

# Save the change for future service starts.
file = Path(bridge.__file__)
shutil.copy2(file, file.with_suffix(".vehicle_prompt.bak"))

with file.open("a", encoding="utf-8") as output:
    for language, prompt in prompts.items():
        output.write(
            f"\nMESSAGES[{language!r}]['vehicle'] = {prompt!r}\n"
            f"VOICE_MESSAGE_UPDATES[{language!r}]['vehicle'] = {prompt!r}\n"
        )

print("Extra SUV instruction removed. Start a new call.")

Extra SUV instruction removed. Start a new call.


In [46]:
import re

# Retain the original function so rerunning this cell is safe.
if not hasattr(bridge, "_original_distance_speak"):
    bridge._original_distance_speak = bridge.speak

def english_number(n):
    ones = [
        "zero", "one", "two", "three", "four", "five", "six",
        "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen",
        "seventeen", "eighteen", "nineteen"
    ]
    tens = [
        "", "", "twenty", "thirty", "forty",
        "fifty", "sixty", "seventy", "eighty", "ninety"
    ]

    if n < 20:
        return ones[n]
    if n < 100:
        return tens[n // 10] + (
            " " + ones[n % 10] if n % 10 else ""
        )
    if n < 1000:
        return ones[n // 100] + " hundred" + (
            " " + english_number(n % 100) if n % 100 else ""
        )
    if n < 1_000_000:
        return english_number(n // 1000) + " thousand" + (
            " " + english_number(n % 1000) if n % 1000 else ""
        )
    return str(n)

distance_pattern = re.compile(
    r"(?<![\d.])(\d+)\.(\d+)\s*"
    r"(kilometres?|kilometers?|km\b|கிலோமீட்டர்)",
    re.IGNORECASE
)

def distance_speak(text):
    language = bridge.CURRENT_LANGUAGE.get()

    def replace_distance(match):
        whole, fraction, unit = match.groups()

        if language in ("Tamil", "Tanglish"):
            # Existing Tamil normalization converts these digits.
            decimal_digits = " ".join(fraction)
            return f"{whole} புள்ளி {decimal_digits} கிலோமீட்டர்"

        decimal_digits = " ".join(
            english_number(int(digit)) for digit in fraction
        )
        return (
            f"{english_number(int(whole))} point "
            f"{decimal_digits} kilometres"
        )

    corrected = distance_pattern.sub(replace_distance, str(text))
    return bridge._original_distance_speak(corrected)

bridge.speak = distance_speak

print("Decimal distance pronunciation enabled.")

Decimal distance pronunciation enabled.


In [47]:
import re

# Keep the original TTS function; rerunning this cell is safe.
if not hasattr(bridge.TTS, "_original_tanglish_synthesize"):
    bridge.TTS._original_tanglish_synthesize = bridge.TTS.synthesize

TANGLISH_SPEECH_WORDS = {
    "vanakkam": "வணக்கம்",
    "ungalukku": "உங்களுக்கு",
    "unga": "உங்க",
    "ungala": "உங்களை",
    "neenga": "நீங்க",
    "naan": "நான்",
    "eppadi": "எப்படி",
    "pannalaam": "பண்ணலாம்",
    "pannalama": "பண்ணலாமா",
    "pannanum": "பண்ணணும்",
    "pannunga": "பண்ணுங்க",
    "panna": "பண்ண",
    "pannu": "பண்ணு",
    "ethana": "எத்தனை",
    "ethanai": "எத்தனை",
    "peru": "பேரு",
    "poreenga": "போறீங்க",
    "poringa": "போறீங்க",
    "edhu": "எது",
    "venum": "வேணும்",
    "venuma": "வேணுமா",
    "enga": "எங்க",
    "irundhu": "இருந்து",
    "irunthu": "இருந்து",
    "sonna": "சொன்ன",
    "sollunga": "சொல்லுங்க",
    "sollu": "சொல்லு",
    "illa": "இல்ல",
    "illai": "இல்லை",
    "endha": "எந்த",
    "entha": "எந்த",
    "naal": "நாள்",
    "manikku": "மணிக்கு",
    "sariya": "சரியா",
    "kekkala": "கேக்கல",
    "thirumba": "திரும்ப",
    "marubadiyum": "மறுபடியும்",
    "aayiduchu": "ஆயிடுச்சு",
    "aagala": "ஆகல",
    "innum": "இன்னும்",
    "nandri": "நன்றி",
    "dhaan": "தான்",
    "thaan": "தான்",
    "pogalaam": "போகலாம்",
    "vera": "வேற",
    "moonu": "மூணு",
    "nu": "னு",
    "ah": "ஆ",
    "la": "ல",
    "ku": "க்கு",
    "na": "னா",
}

word_pattern = re.compile(
    r"(?<![A-Za-z])("
    + "|".join(
        re.escape(word)
        for word in sorted(
            TANGLISH_SPEECH_WORDS, key=len, reverse=True
        )
    )
    + r")(?![A-Za-z])",
    re.IGNORECASE,
)

def tanglish_speech_text(text):
    return word_pattern.sub(
        lambda match: TANGLISH_SPEECH_WORDS[
            match.group(0).lower()
        ],
        str(text),
    )

def pronunciation_synthesize(text, group="ENGLISH"):
    speech_text = str(text)

    if str(group).upper() == "TANGLISH":
        speech_text = tanglish_speech_text(speech_text)

    return bridge.TTS._original_tanglish_synthesize(
        speech_text, group
    )

bridge.TTS.synthesize = pronunciation_synthesize

# Remove the extra SUV instruction from the running prompts.
vehicle_prompt = "Bike, auto, mini car, sedan, SUV, edhu venum?"
bridge.MESSAGES["Tanglish"]["vehicle"] = vehicle_prompt
bridge.VOICE_MESSAGE_UPDATES["Tanglish"]["vehicle"] = vehicle_prompt

print("Silent Tanglish pronunciation conversion enabled.")

Silent Tanglish pronunciation conversion enabled.


In [48]:
TANGLISH_SPEECH_WORDS.update({
    "service": "சர்விஸ்",
    "help": "ஹெல்ப்",
    "confirm": "கன்ஃபர்ம்",
})

In [49]:
from pathlib import Path

file = Path("/kaggle/working/MOVIO_MAIN_WEBSITE/web/index.html")
html = file.read_text(encoding="utf-8")

html = html.replace(
    '<button id="joinDriver" type="button">Driver registration</button>',
    '<button id="joinDriver" type="button" hidden style="display:none">Driver registration</button>'
)

file.write_text(html, encoding="utf-8")
print("Driver registration shortcut hidden. Refresh the website.")

Driver registration shortcut hidden. Refresh the website.


In [50]:
from pathlib import Path
import shutil
import datetime

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE/web")
html_file = root / "index.html"
js_file = root / "app.js"

html = html_file.read_text(encoding="utf-8")
js = js_file.read_text(encoding="utf-8")

backup = root / (
    "address_backup_" +
    datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
)
backup.mkdir()
shutil.copy2(html_file, backup / "index.html")
shutil.copy2(js_file, backup / "app.js")

# Replace automatic searches with an explicit Search button.
# This also avoids sending a request for every typing pause.
lines = js.splitlines()
for index, line in enumerate(lines):
    if line.startswith("for(const id of ['pickup','destination'])"):
        lines[index] = """
for (const id of ['pickup', 'destination']) {
    $(id).addEventListener('focus', () => selecting = id);
    $(id).addEventListener('input', () => {
        selecting = id;
        $('suggestions').replaceChildren();
    });
    $(id).addEventListener('keydown', event => {
        if (event.key === 'Enter') {
            event.preventDefault();
            searchDetailedAddress();
        }
    });
}
"""
        break
else:
    raise RuntimeError(
        "Search code differs from the expected version. "
        "No files were changed."
    )

js = "\n".join(lines)

extra_js = """
// Detailed address selection.
const addressCache = new Map();
let addressSearchBusy = false;

async function searchDetailedAddress() {
    if (addressSearchBusy) return;
    const field = selecting === 'destination'
        ? 'destination' : 'pickup';
    const typed = $(field).value.trim();
    if (typed.length < 3) {
        toast('Enter a street, area or landmark first.');
        return;
    }

    addressSearchBusy = true;
    $('searchAddress').disabled = true;
    $('suggestions').textContent = 'Searching…';

    try {
        const query = /chennai/i.test(typed)
            ? typed : typed + ', Chennai, Tamil Nadu';

        let places = addressCache.get(query);
        if (!places) {
            const params = new URLSearchParams({
                format: 'json',
                limit: '6',
                addressdetails: '1',
                countrycodes: 'in',
                viewbox: '79.95,13.35,80.45,12.65',
                bounded: '1',
                q: query
            });

            const response = await fetch(
                'https://nominatim.openstreetmap.org/search?' + params,
                {headers: {'Accept-Language': 'en'}}
            );
            if (!response.ok) throw Error('Address search unavailable');
            places = await response.json();
            addressCache.set(query, places);
        }

        $('suggestions').replaceChildren();
        for (const place of places) {
            const button = document.createElement('button');
            button.type = 'button';
            button.textContent = place.display_name;
            button.onclick = () => {
                setPlace(field, {
                    lat: Number(place.lat),
                    lng: Number(place.lon),
                    name: place.display_name
                });
                $('suggestions').replaceChildren();
                toast('Check the map pin for the exact entrance.');
            };
            $('suggestions').append(button);
        }

        if (!places.length) {
            $('suggestions').textContent =
                'Address not found. Try street + area + postcode, ' +
                'or use current location / select an exact map point.';
        }
    } catch (error) {
        $('suggestions').textContent =
            error.message + '. You can still select a point on the map.';
    } finally {
        addressSearchBusy = false;
        // Limit consecutive searches.
        setTimeout(() => {
            $('searchAddress').disabled = false;
        }, 1100);
    }
}

$('searchAddress').onclick = searchDetailedAddress;

$('useCurrentPickup').onclick = async () => {
    const button = $('useCurrentPickup');
    button.disabled = true;
    try {
        const position = await locate();
        setPlace('pickup', {
            ...position,
            name: 'Current pickup location'
        }, false);
        map.setView([position.lat, position.lng], 18);
        toast('Pickup set. Check the pin before booking.');
    } catch (error) {
        toast('Allow location permission and enable phone GPS, ' +
              'or select your pickup on the map.');
    } finally {
        button.disabled = false;
    }
};

$('pinPickup').onclick = () => {
    selecting = 'pickup';
    $('pickup').focus();
    toast('Tap your exact pickup or building entrance on the map.');
};

$('pinDestination').onclick = () => {
    selecting = 'destination';
    $('destination').focus();
    toast('Tap your exact destination entrance on the map.');
};
"""

if 'id="useCurrentPickup"' not in html:
    controls = """
<div style="display:flex;flex-wrap:wrap;gap:8px;margin:12px 0">
  <button id="useCurrentPickup" type="button" class="outline">
    Use current location
  </button>
  <button id="searchAddress" type="button" class="outline">
    Search address
  </button>
  <button id="pinPickup" type="button" class="outline">
    Pin pickup
  </button>
  <button id="pinDestination" type="button" class="outline">
    Pin destination
  </button>
</div>
"""
    html = html.replace(
        '<div class="suggestion" id="searchNote">',
        controls + '<div class="suggestion" id="searchNote">',
        1
    )
    js += "\n" + extra_js

html_file.write_text(html, encoding="utf-8")
js_file.write_text(js, encoding="utf-8")

print("Address controls updated. Refresh your dashboard.")

Address controls updated. Refresh your dashboard.


In [51]:
import requests

session = requests.Session()
session.trust_env = False

for port in (8080, 8090):
    for route in ("/api/health", "/api/me"):
        try:
            response = session.get(
                f"http://127.0.0.1:{port}{route}",
                timeout=(2, 3),
            )
            print(
                f"\nPort {port} {route}",
                "\nStatus:", response.status_code,
                "\nContent-Type:", response.headers.get("Content-Type"),
                "\nResponse:", response.text[:300],
                flush=True,
            )
        except requests.RequestException as error:
            print(port, route, str(error), flush=True)


Port 8080 /api/health 
Status: 200 
Content-Type: application/json; charset=utf-8 
Response: {"ok": true, "version": "5.2", "calling_configured": false, "tts_configured": false, "stt_configured": false, "stream_configured": false, "ai_call_configured": false}

Port 8080 /api/me 
Status: 401 
Content-Type: application/json; charset=utf-8 
Response: {"error": "Log in first"}

Port 8090 /api/health 
Status: 200 
Content-Type: application/json; charset=utf-8 
Response: {"ok": true, "version": "5.2", "calling_configured": false, "tts_configured": false, "stt_configured": false, "stream_configured": false, "ai_call_configured": false}

Port 8090 /api/me 
Status: 401 
Content-Type: application/json; charset=utf-8 
Response: {"error": "Log in first"}


In [52]:
from pathlib import Path, PurePosixPath
import json, zipfile, hashlib, shutil, datetime, os, sys, importlib.util, subprocess, time

SITE_ROOT = Path('/kaggle/working/MOVIO_MAIN_WEBSITE')
MARKER = 'driver_dashboard_patch.json'
FILES = ['server.py', 'dashboard_priority.js', 'dashboard_priority.css']
assert (SITE_ROOT / 'server.py').is_file(), 'Start your existing website first.'
assert not getattr(globals().get('bridge'), 'SESSIONS', {}), 'Hang up AI calls before installing.'
found = {}
for marker in Path('/kaggle/input').rglob(MARKER):
    folder = marker.parent
    if all((folder / name).is_file() for name in FILES):
        data = {name: (folder / name).read_bytes() for name in FILES}
        found[hashlib.sha256(b''.join(data.values())).hexdigest()] = data
for package in Path('/kaggle/input').rglob('AI_Ride_Driver_Dashboard_Update*.zip'):
    with zipfile.ZipFile(package) as archive:
        for name in archive.namelist():
            if PurePosixPath(name).name == MARKER:
                prefix = name[:-len(MARKER)]
                data = {key: archive.read(prefix + key) for key in FILES}
                found[hashlib.sha256(b''.join(data.values())).hexdigest()] = data
assert len(found) == 1, 'Attach only the AI_Ride_Driver_Dashboard_Update.zip patch dataset.'
payload = next(iter(found.values()))
web = SITE_ROOT / 'web'
html = (web / 'index.html').read_text(encoding='utf-8')
enhancements = (web / 'movio_enhancements.js').read_text(encoding='utf-8')
old = 'await post(`/rides/${r.id}/status`,{status});'
new = "await post(`/rides/${r.id}/status`,{status,...(status==='in_progress'?{otp:document.getElementById('driverPickupOTP_'+r.id)?.value||''}:{})});"
if new not in enhancements:
    assert old in enhancements, 'Unrecognized driver actions. No changes were made.'
    enhancements = enhancements.replace(old, new)
if 'dashboard_priority.js' not in html:
    html = html.replace('</body>', '<script src="dashboard_priority.js"></script></body>')
if 'dashboard_priority.css' not in html:
    html = html.replace('</head>', '<link rel="stylesheet" href="dashboard_priority.css"></head>')
assert 'dashboard_priority.js' in html and 'dashboard_priority.css' in html
backup = SITE_ROOT / ('DRIVER_DASHBOARD_BACKUP_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
for relative in ['server.py','web/index.html','web/movio_enhancements.js']:
    source = SITE_ROOT / relative
    destination = backup / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
compile(payload['server.py'].decode(), 'server.py', 'exec')
(SITE_ROOT / 'server.py').write_bytes(payload['server.py'])
(web / 'dashboard_priority.js').write_bytes(payload['dashboard_priority.js'])
(web / 'dashboard_priority.css').write_bytes(payload['dashboard_priority.css'])
(web / 'index.html').write_text(html, encoding='utf-8')
(web / 'movio_enhancements.js').write_text(enhancements, encoding='utf-8')
print('UI and OTP code installed. Backup:', backup)
print('Accounts, booking data, voice page and TTS patches retained.')


UI and OTP code installed. Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/DRIVER_DASHBOARD_BACKUP_20261028_070649_015975
Accounts, booking data, voice page and TTS patches retained.


In [53]:
PORT = 8080
import gc, subprocess, time
from http.server import ThreadingHTTPServer
try:
    import psutil
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'psutil'])
    import psutil

spec = importlib.util.spec_from_file_location('ride_dashboard_otp', SITE_ROOT / 'server.py')
website_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = website_module
spec.loader.exec_module(website_module)
listeners = sorted({c.pid for c in psutil.net_connections(kind='inet')
                    if c.status == 'LISTEN' and c.laddr.port == PORT and c.pid})
website_env = dict(os.environ)
if os.getpid() in listeners:
    live_servers = []
    for obj in gc.get_objects():
        if isinstance(obj, ThreadingHTTPServer) and obj.server_address[1] == PORT:
            existing_root = obj.RequestHandlerClass.__init__.__globals__.get('ROOT')
            if existing_root and Path(existing_root).resolve() == SITE_ROOT.resolve():
                live_servers.append(obj)
    assert live_servers, 'Unrecognized server inside this kernel; rerun your website-only launch cell'
    for httpd in live_servers:
        existing_lock = httpd.RequestHandlerClass.do_POST.__globals__.get('LOCK')
        if existing_lock is not None:
            website_module.LOCK = existing_lock
        with website_module.LOCK:
            website_module.setup()
            httpd.RequestHandlerClass = website_module.Handler
    print('Updated the website HTTP handler in this kernel; AI threads retained.')
else:
    for pid in listeners:
        proc = psutil.Process(pid)
        scripts = [Path(a) for a in proc.cmdline() if a.endswith('.py')]
        known = any(p.name == 'server.py' and (Path(proc.cwd()) / p).resolve().parent == SITE_ROOT.resolve() for p in scripts)
        assert known, ('Another process owns the website port. The files are installed; '
                       'rerun your website-only launcher. No AI process was stopped.')
        previous_env = proc.environ()
        website_env = {**website_env, **previous_env}
        proc.terminate()
        proc.wait(timeout=8)
    log_path = SITE_ROOT / 'multiple_bookings_website.log'
    with log_path.open('a') as logfile:
        website_process = subprocess.Popen([sys.executable, 'server.py'], cwd=SITE_ROOT,
            env={**website_env, 'MOVIO_PORT': str(PORT)}, stdout=logfile,
            stderr=subprocess.STDOUT, start_new_session=True)
    print('Restarted website-only process:', website_process.pid)



Restarted website-only process: 439


In [54]:
import requests
session = requests.Session()
session.trust_env = False
for attempt in range(20):
    try:
        response = session.get('http://127.0.0.1:8080/api/health', timeout=(2, 3))
        if response.ok and response.json().get('version') == '5.3':
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError('Website 5.3 did not start. Check the website process log.')
print('Driver-first dashboard and pickup OTP: ENABLED')
print('Hard-refresh customer and driver dashboards. Do not restart the kernel.')


Driver-first dashboard and pickup OTP: ENABLED
Hard-refresh customer and driver dashboards. Do not restart the kernel.


In [55]:
from pathlib import Path
import datetime, shutil

web = Path('/kaggle/working/MOVIO_MAIN_WEBSITE/web')
assert (web / 'dashboard_priority.js').is_file(), 'Install the driver-first customer dashboard and OTP update first.'
html_file = web / 'index.html'
html = html_file.read_text(encoding='utf-8')
backup = html_file.with_name('index.driver_ui_backup_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '.html')
shutil.copy2(html_file, backup)

DRIVER_JS = r'''
/* Driver dashboard: assigned trip first, then matching incoming requests. */
const driverDashboardPreviousRender=renderDriver;
function driverUIElement(tag,cls,text){const node=document.createElement(tag);node.className=cls;if(text!==undefined)node.textContent=text;return node}
function installDriverDashboard(){
 const panel=$('driverPanel');panel.classList.add('driver-dashboard-panel');
 const heading=panel.querySelector('.sheet-head h1');if(heading)heading.textContent='Your driving day';
 const subtitle=panel.querySelector('.sheet-head .muted');if(subtitle)subtitle.textContent='Your assigned trip and matching requests, in one place.';
 const profile=$('driverProfile');profile.classList.add('driver-dashboard-profile');
 if(!$('driverDashboardCounts')){const counts=driverUIElement('div','driver-dashboard-counts');counts.id='driverDashboardCounts';profile.after(counts)}
 const gps=$('driverGPS').closest('.gps-card');if(gps)gps.classList.add('driver-online-card');
 const title=panel.querySelector('.section-row h2');if(title)title.textContent='Trips & requests';
}
renderDriver=function(rides){
 installDriverDashboard();
 const sorted=[...rides].sort((a,b)=>Number(movioActive(b))-Number(movioActive(a))||b.id-a.id);
 driverDashboardPreviousRender(sorted);
 const assigned=sorted.find(r=>r.driver_id===movioMe?.id&&movioActive(r));
 const incoming=sorted.filter(r=>r.status==='requested');
 const counts=$('driverDashboardCounts');counts.replaceChildren();
 for(const [value,label] of [[assigned?'1':'0','Active trip'],[String(incoming.length),'Matching requests']]){
  const tile=driverUIElement('div','driver-count-tile');tile.append(driverUIElement('strong','',value),driverUIElement('span','',label));counts.append(tile)
 }
 const holder=$('driverRides');
 for(const card of [...holder.children]){
  const heading=card.querySelector('h3');if(!heading)continue;
  const id=Number(heading.textContent.match(/^#(\d+)/)?.[1]);const ride=sorted.find(r=>r.id===id);if(!ride)continue;
  card.classList.add('driver-trip-card');card.dataset.stage=ride.status;
  const badge=driverUIElement('span','driver-trip-badge',ride.status==='requested'?'NEW REQUEST':ride.status==='accepted'?'ACCEPTED':ride.status==='arriving'?'ON THE WAY':'RIDE IN PROGRESS');
  const top=driverUIElement('div','driver-trip-top');top.append(badge,driverUIElement('span','driver-trip-id','Booking #'+ride.id));card.insertBefore(top,heading);
  heading.textContent=ride.vehicle+' · ₹'+ride.fare;
  const route=driverUIElement('div','driver-route');
  for(const [label,name,cls] of [['PICKUP',ride.pickup,'pickup'],['DROP-OFF',ride.destination,'dropoff']]){
   const row=driverUIElement('div','driver-route-row '+cls);const dot=driverUIElement('span','driver-route-dot');const text=driverUIElement('div','');text.append(driverUIElement('small','',label),driverUIElement('p','',name));row.append(dot,text);route.append(row)
  }
  const customer=driverUIElement('div','driver-customer');customer.append(driverUIElement('span','customer-avatar',(ride.customer_name||'C').slice(0,1).toUpperCase()),driverUIElement('strong','',ride.customer_name||'Customer'));
  for(const paragraph of [...card.children].filter(n=>n.tagName==='P')){
   if(/^(Customer:|Pickup:|Destination:|Status:)/.test(paragraph.textContent))paragraph.remove();
  }
  heading.after(customer);customer.after(route);
  const otp=card.querySelector('.driver-otp-label');if(otp){otp.classList.add('driver-otp-panel');const hint=driverUIElement('small','','Ask the customer for their 4-digit code at pickup.');otp.append(hint)}
  const actions=card.querySelector('.actions');if(actions){
   actions.classList.add('driver-trip-actions');
   for(const button of actions.querySelectorAll('button')){
    const label=button.textContent;
    if(label==='Start ride')button.textContent='Verify OTP & start ride';
    if(label==='Accept')button.textContent='Accept request';
    if(label==='Cancel request')button.textContent='Decline request';
    if(label==='View Customer location')button.textContent='View customer on map';
    if(label==='Request customer location')button.textContent='Request 15-min location access';
    if(label.includes('location')||label==='Location request sent')button.classList.add('driver-location-action');
   }
  }
 }
 if(!assigned&&!incoming.length){
  holder.replaceChildren();const empty=driverUIElement('div','driver-empty');empty.append(driverUIElement('span','','↗'),driverUIElement('h3','','Ready for your next ride'),driverUIElement('p','','Matching '+(movioMe?.vehicle_type||'vehicle')+' requests will appear here.'));holder.append(empty)
 }
};
installDriverDashboard();
if(movioMe?.role==='driver')refreshRides();
'''

DRIVER_CSS = r'''
.driver-dashboard-panel{background:#f7f7f4;padding:24px!important}.driver-dashboard-panel .sheet-head h1{font-size:29px;letter-spacing:-1px}.driver-dashboard-panel .ride-emoji{background:#fff0b7;font-size:35px}.driver-dashboard-profile{background:#fff!important;border:1px solid #e8e9eb!important;line-height:1.8;padding:16px!important;border-radius:17px!important;font-weight:700}.driver-dashboard-counts{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin:14px 0}.driver-count-tile{background:#fff;border:1px solid #e8e9eb;padding:14px;border-radius:16px;display:grid;gap:6px}.driver-count-tile strong{font-size:27px;line-height:1}.driver-count-tile span{font-size:11px;color:#6c727b}.driver-online-card{background:#fff9e5!important;border:1px solid #ffe49a!important;padding:16px!important}.driver-online-card #driverGPS{background:#17191d;color:#fff;border:0;min-width:100px}.driver-trip-card{background:#fff;border:1px solid #e8e9eb;border-radius:22px;padding:18px;box-shadow:0 8px 28px #17191d05}.driver-trip-card[data-stage=accepted],.driver-trip-card[data-stage=arriving],.driver-trip-card[data-stage=in_progress]{border:2px solid #efbe22}.driver-trip-top{display:flex;justify-content:space-between;align-items:center;gap:8px;margin-bottom:16px}.driver-trip-badge{background:#fff1ba;color:#755300;padding:6px 9px;border-radius:9px;font-size:10px;font-weight:800;letter-spacing:.5px}.driver-trip-id{font-size:11px;color:#7b8189}.driver-trip-card h3{font-size:21px;margin:0 0 14px}.driver-customer{display:flex;align-items:center;gap:9px;font-size:13px;margin-bottom:16px}.customer-avatar{display:grid;place-items:center;width:30px;height:30px;background:#f0f1f3;border-radius:10px}.driver-route{background:#f8f9fa;border-radius:15px;padding:14px;margin-bottom:15px;display:grid;gap:14px}.driver-route-row{display:flex;gap:11px;align-items:flex-start}.driver-route-dot{width:9px;height:9px;flex:none;background:#22a976;border-radius:50%;margin-top:4px}.dropoff .driver-route-dot{background:#ed745a;border-radius:2px}.driver-route-row small{font-size:9px;color:#7b8189;letter-spacing:1px;font-weight:800}.driver-route-row p{font-size:13px!important;color:#17191d!important;line-height:1.6;margin:5px 0 0!important}.driver-otp-panel{background:#fff9e5;border:1px solid #ffe49a;border-radius:15px;padding:14px;margin-bottom:14px}.driver-otp-panel input{background:#fff;border-color:#eedda7}.driver-otp-panel small{font-size:10px;color:#7b725b;font-weight:400;line-height:1.5}.driver-trip-actions{display:grid!important;grid-template-columns:1fr 1fr;gap:9px!important}.driver-trip-actions button{min-width:0!important;white-space:normal;line-height:1.5;font-size:12px;min-height:46px;padding:12px;justify-content:center}.driver-trip-actions .driver-location-action{grid-column:1/-1;background:#fff9e5;border-color:#eedda7}.driver-empty{text-align:center;background:white;border:1px dashed #d9dce0;border-radius:22px;padding:32px 20px}.driver-empty>span{display:inline-grid;place-items:center;width:55px;height:55px;background:#ffcc29;border-radius:19px;font-size:29px;margin-bottom:13px}.driver-empty h3{font-size:18px}.driver-empty p{font-size:12px;color:#737982;line-height:1.7}@media(max-width:820px){.sheet.driver-dashboard-panel{position:relative;width:100%;max-height:none!important;overflow:visible;border-radius:24px 24px 0 0;min-width:0;padding:20px 16px!important}.workspace:has(.driver-dashboard-panel:not(.hidden)){display:flex;flex-direction:column;overflow:auto}.workspace:has(.driver-dashboard-panel:not(.hidden)) .map-area{order:-1;flex:none;height:30dvh;min-height:190px}.shell:has(.driver-dashboard-panel:not(.hidden)){height:auto;min-height:100dvh}.driver-dashboard-panel .sheet-head h1{font-size:27px}.driver-online-card{flex-wrap:wrap}.driver-trip-actions button{min-height:48px}.driver-dashboard-panel #editDriverProfile{margin-bottom:2px}}
'''

(web / 'driver_dashboard.js').write_text(DRIVER_JS, encoding='utf-8')
(web / 'driver_dashboard.css').write_text(DRIVER_CSS, encoding='utf-8')
if 'src="driver_dashboard.js"' not in html:
    html = html.replace('</body>', '<script src="driver_dashboard.js"></script></body>')
if 'href="driver_dashboard.css"' not in html:
    html = html.replace('</head>', '<link rel="stylesheet" href="driver_dashboard.css"></head>')
html_file.write_text(html, encoding='utf-8')
print('Driver dashboard redesigned. Refresh the driver dashboard tab.')
print('Existing OTP, vehicle matching and location-consent rules are retained.')


Driver dashboard redesigned. Refresh the driver dashboard tab.
Existing OTP, vehicle matching and location-consent rules are retained.


In [56]:
from pathlib import Path
import datetime, shutil

web = Path('/kaggle/working/MOVIO_MAIN_WEBSITE/web')
assert (web / 'live_tracking.js').is_file(), 'Existing live tracking code was not found.'
file = web / 'index.html'
html = file.read_text(encoding='utf-8')
shutil.copy2(file, file.with_name('index.live_map_backup_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '.html'))

LIVE_MAP_JS = r'''
/* Show both endpoints and a live connecting line. Own GPS stays local unless sharing is already granted. */
let rideLinkLine=null,rideLinkProvider=null,viewerGPSWatch=null,viewerOwnPosition=null,ridePointHistory=new Map(),rideMarkerFrames=new Map();
function clearRideLink(){
 if(rideLinkLine){if(rideLinkProvider==='google')rideLinkLine.setMap(null);else rideLinkLine.remove()}
 rideLinkLine=null;rideLinkProvider=null;
 for(const frame of rideMarkerFrames.values())cancelAnimationFrame(frame);
 rideMarkerFrames.clear();ridePointHistory.clear();
}
const motionDestroyMap=destroyTrackingMap;
destroyTrackingMap=function(){clearRideLink();motionDestroyMap()};
const motionPreviousUpdate=movioUpdateTracking;
movioUpdateTracking=function(locations){
 const now=Date.now()/1000;
 let points=locations.filter(p=>p.expires>now).map(p=>({...p}));
 if(viewerOwnPosition&&movioMe&&viewerOwnPosition.expires>now){
  points=points.filter(p=>p.user_id!==movioMe.id);
  points.push({...viewerOwnPosition,user_id:movioMe.id,role:movioMe.role});
 }
 if(!movioTrackingOpen||!trackingMap)return;
 motionPreviousUpdate(points);
 const ids=new Set(points.map(p=>p.user_id));
 for(const id of ridePointHistory.keys())if(!ids.has(id)){
  ridePointHistory.delete(id);if(rideMarkerFrames.has(id))cancelAnimationFrame(rideMarkerFrames.get(id));rideMarkerFrames.delete(id);
 }
 for(const point of points){
  const before=ridePointHistory.get(point.user_id),marker=trackingMarkers.get(point.user_id);
  ridePointHistory.set(point.user_id,{lat:point.lat,lng:point.lng});
  if(!before||!marker||(before.lat===point.lat&&before.lng===point.lng))continue;
  if(rideMarkerFrames.has(point.user_id))cancelAnimationFrame(rideMarkerFrames.get(point.user_id));
  const epoch=trackingEpoch,start=performance.now();
  const animate=timestamp=>{
   if(epoch!==trackingEpoch||!movioTrackingOpen||trackingMarkers.get(point.user_id)!==marker)return;
   const fraction=Math.min(1,(timestamp-start)/900);
   const lat=before.lat+(point.lat-before.lat)*fraction,lng=before.lng+(point.lng-before.lng)*fraction;
   if(trackingProvider==='google')marker.position={lat,lng};else marker.setLatLng([lat,lng]);
   if(fraction<1)rideMarkerFrames.set(point.user_id,requestAnimationFrame(animate));else rideMarkerFrames.delete(point.user_id);
  };
  rideMarkerFrames.set(point.user_id,requestAnimationFrame(animate));
 }
 const driver=points.find(p=>p.role==='driver'),customer=points.find(p=>p.role==='customer');
 if(driver&&customer){
  if(rideLinkProvider!==trackingProvider){clearRideLink();rideLinkProvider=trackingProvider}
  const path=trackingProvider==='google'?[{lat:driver.lat,lng:driver.lng},{lat:customer.lat,lng:customer.lng}]:[[driver.lat,driver.lng],[customer.lat,customer.lng]];
  if(!rideLinkLine){
   if(trackingProvider==='google')rideLinkLine=new google.maps.Polyline({map:trackingMap,path,strokeColor:'#e4a900',strokeOpacity:.85,strokeWeight:4,geodesic:true});
   else rideLinkLine=L.polyline(path,{color:'#e4a900',weight:4,opacity:.85,dashArray:'8,8'}).addTo(trackingMap);
   if(trackingProvider==='google'){const bounds=new google.maps.LatLngBounds();path.forEach(p=>bounds.extend(p));trackingMap.fitBounds(bounds,60)}
   else trackingMap.fitBounds(path,{padding:[40,40],maxZoom:17});
  }else if(trackingProvider==='google')rideLinkLine.setPath(path);else rideLinkLine.setLatLngs(path);
  const metres=Math.round(distanceKm(driver,customer)*1000);
  $('trackingStatus').textContent='Driver 🚘 and customer 📍 · '+(metres<1000?metres+' m':(metres/1000).toFixed(1)+' km')+' apart · direct connecting line, not a road route';
 }else{
  if(rideLinkLine){if(rideLinkProvider==='google')rideLinkLine.setMap(null);else rideLinkLine.remove();rideLinkLine=null;rideLinkProvider=null}
  $('trackingStatus').textContent=movioMe?.role==='driver'?'Waiting for customer permission / live GPS. Customer location disappears when access ends.':'Waiting for driver live GPS. The driver must be online with location permission.';
 }
};
function stopViewerGPS(){if(viewerGPSWatch!==null)navigator.geolocation.clearWatch(viewerGPSWatch);viewerGPSWatch=null;viewerOwnPosition=null}
const motionPreviousOpen=movioOpenTracking;
movioOpenTracking=async function(id,target){
 stopViewerGPS();await motionPreviousOpen(id,target);
 if(!movioTrackingOpen||!navigator.geolocation)return;
 const epoch=trackingEpoch;
 viewerGPSWatch=navigator.geolocation.watchPosition(position=>{
  if(epoch!==trackingEpoch||!movioTrackingOpen)return;
  viewerOwnPosition={lat:position.coords.latitude,lng:position.coords.longitude,updated:Date.now()/1000,expires:Date.now()/1000+20};
  // Drawing your own position does not send it to the server.
  movioUpdateTracking(movioLocations);
 },()=>{
  viewerOwnPosition=null;movioUpdateTracking(movioLocations);
  $('trackingProvider').textContent+=' Allow GPS to show your own live position.';
 },{enableHighAccuracy:true,maximumAge:0,timeout:15000});
};
const motionPreviousClose=movioCloseTracking;
movioCloseTracking=function(){stopViewerGPS();clearRideLink();motionPreviousClose()};
$('closeTracking').onclick=movioCloseTracking;
window.addEventListener('beforeunload',stopViewerGPS);
'''

(web / 'live_map_motion.js').write_text(LIVE_MAP_JS, encoding='utf-8')
if 'src="live_map_motion.js"' not in html:
    html = html.replace('</body>', '<script src="live_map_motion.js"></script></body>')
file.write_text(html, encoding='utf-8')
print('Live two-position map and connecting line installed.')
print('Refresh both dashboards and reopen the live map.')
print('Driver: Go online and permit GPS. Customer: approve only the driver-requested sharing prompt.')


Live two-position map and connecting line installed.
Refresh both dashboards and reopen the live map.
Driver: Go online and permit GPS. Customer: approve only the driver-requested sharing prompt.


In [57]:
import re
import io
import base64
import numpy as np
import soundfile as sf

# Keep your existing pronunciation fixes.
if not hasattr(bridge, "_before_clear_summary_speak"):
    bridge._before_clear_summary_speak = bridge.speak

def amount_in_english(n):
    small = [
        "zero", "one", "two", "three", "four", "five", "six",
        "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen",
        "seventeen", "eighteen", "nineteen"
    ]
    tens = [
        "", "", "twenty", "thirty", "forty",
        "fifty", "sixty", "seventy", "eighty", "ninety"
    ]
    if n < 20:
        return small[n]
    if n < 100:
        return tens[n // 10] + (
            " " + small[n % 10] if n % 10 else ""
        )
    if n < 1000:
        return small[n // 100] + " hundred" + (
            " " + amount_in_english(n % 100) if n % 100 else ""
        )
    if n < 1_000_000:
        return amount_in_english(n // 1000) + " thousand" + (
            " " + amount_in_english(n % 1000) if n % 1000 else ""
        )
    return str(n)

def clear_summary_speak(text):
    text = str(text)
    is_summary = any(
        phrase in text.lower()
        for phrase in (
            "your booking:",
            "booking details:",
            "உங்கள் முன்பதிவு விவரம்",
        )
    )

    original = bridge._before_clear_summary_speak

    if not is_summary:
        return original(text)

    # Make English fare amounts explicit before normalization.
    if bridge.CURRENT_LANGUAGE.get() == "English":
        text = re.sub(
            r"\b(\d[\d,]*)\s+rupees\b",
            lambda m: (
                amount_in_english(
                    int(m.group(1).replace(",", ""))
                ) + " rupees"
            ),
            text,
            flags=re.IGNORECASE,
        )

    # Split at clause boundaries, preserving decimal points.
    parts = [
        part.strip()
        for part in re.split(
            r"(?<!\d),(?!\d)|(?<!\d)[.!?]\s+",
            text,
        )
        if part.strip()
    ]

    audio_parts = []
    sample_rate = None

    for part in parts:
        payload = original(part)
        audio, rate = sf.read(
            io.BytesIO(base64.b64decode(payload)),
            dtype="float32",
            always_2d=True,
        )

        if not audio.size:
            raise RuntimeError(
                "No audio generated for summary clause: " + part
            )

        if sample_rate is None:
            sample_rate = rate
        elif rate != sample_rate:
            raise RuntimeError("Summary audio sample rates differ")

        audio_parts.append(audio.mean(axis=1))
        audio_parts.append(
            np.zeros(int(rate * 0.22), dtype=np.float32)
        )

    output = io.BytesIO()
    sf.write(
        output,
        np.concatenate(audio_parts),
        sample_rate,
        format="WAV",
        subtype="PCM_16",
    )
    return base64.b64encode(output.getvalue()).decode()

bridge.speak = clear_summary_speak
print("Clear summary speech enabled. Start a new call.")

Clear summary speech enabled. Start a new call.


In [58]:
if hasattr(bridge, "_before_clear_summary_speak"):
    bridge.speak = bridge._before_clear_summary_speak
    print("Summary-audio patch disabled.")

Summary-audio patch disabled.


In [59]:
import re

if globals().get("bridge") is None:
    raise RuntimeError("Run the voice bridge initialization cell first.")

def normalize_tambaram(text):
    original = str(text).strip()
    compact = re.sub(r"[\s.\-]+", "", original).casefold()

    if compact in ("tambaram", "thambaram"):
        return "Tambaram"

    return original

# Capture originals once so rerunning is safe.
if not hasattr(bridge, "_before_tambaram_canonical"):
    bridge._before_tambaram_canonical = bridge.canonical_place

if not hasattr(bridge, "_before_tambaram_geocode"):
    bridge._before_tambaram_geocode = bridge.geocode

def tambaram_canonical(text):
    normalized = normalize_tambaram(text)
    if normalized == "Tambaram":
        return normalized
    return bridge._before_tambaram_canonical(text)

def tambaram_geocode(text):
    return bridge._before_tambaram_geocode(
        normalize_tambaram(text)
    )

bridge.canonical_place = tambaram_canonical
bridge.geocode = tambaram_geocode
bridge.GEOCODE_CACHE.clear()

print("Tambaram and Thambaram now use the same location search.")
print("Close the current call and start a new call.")

Tambaram and Thambaram now use the same location search.
Close the current call and start a new call.


In [60]:
import re
import json
import torch
import unicodedata

if globals().get("bridge") is None:
    raise RuntimeError("Initialize the voice bridge first.")

if bridge.LLM is None or bridge.TOKENIZER is None:
    raise RuntimeError("Load and configure Qwen before running this cell.")

# Capture the existing functions once.
if not hasattr(bridge, "_context_original_handle"):
    bridge._context_original_handle = bridge.handle
    bridge._context_original_fare = bridge.road_fare

def context_normalize(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", text).strip()

def context_is_stated(value, message):
    """Accept a text slot only when it occurs in this reply."""
    return (
        isinstance(value, str)
        and bool(value.strip())
        and context_normalize(value) in context_normalize(message)
    )

def context_extract(message, session):
    known = {
        key: session.get(key)
        for key in ("count", "vehicle", "pickup", "destination", "when")
    }

    instruction = """
Extract explicitly stated taxi-booking details from the NEW MESSAGE.
The message may be English, Tamil or Tanglish.

Return exactly one JSON object:
{
  "passengers_text": null,
  "vehicle": null,
  "pickup": null,
  "destination": null,
  "time": null
}

Rules:
- passengers_text: copy the passenger number/words from the message.
- vehicle: Bike, Auto, Mini, Sedan or SUV; only if explicitly selected.
- "cab", "taxi" or "car" alone does NOT select Mini.
- pickup, destination and time: copy exact spans from the NEW MESSAGE.
- Never invent, translate, correct or replace a place name.
- Do not copy already-known values into the output.
- An explicit correction replaces the corresponding known value.
- Use the current question to interpret a short answer.
- If a detail is absent or ambiguous, return null.
- Return JSON only.
"""

    prompt = (
        instruction
        + "\nKNOWN DETAILS: "
        + json.dumps(known, ensure_ascii=False)
        + "\nCURRENT QUESTION SLOT: "
        + str(session.get("stage"))
        + "\nNEW MESSAGE: "
        + message
    )

    with bridge.LOCK, torch.inference_mode():
        inputs = bridge.TOKENIZER.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )

        device = next(bridge.LLM.parameters()).device
        inputs = {key: value.to(device) for key, value in inputs.items()}

        output = bridge.LLM.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            pad_token_id=bridge.TOKENIZER.eos_token_id,
        )

        generated = bridge.TOKENIZER.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

    match = re.search(r"\{.*?\}", generated, re.S)
    if not match:
        return {}

    try:
        result = json.loads(match.group())
        return result if isinstance(result, dict) else {}
    except json.JSONDecodeError:
        return {}

def context_next_question(session):
    count = session.get("count")
    vehicle = session.get("vehicle")

    if count and vehicle and count > bridge.CAPACITY[vehicle]:
        session["vehicle"] = None
        session["stage"] = "vehicle"
        return (
            bridge.say(
                session,
                "capacity",
                vehicle=vehicle,
                max=bridge.CAPACITY[vehicle],
            )
            + " "
            + bridge.say(session, "vehicle")
        )

    for slot, stage in (
        ("count", "count"),
        ("vehicle", "vehicle"),
        ("pickup", "pickup"),
        ("destination", "destination"),
        ("when", "time"),
    ):
        if not session.get(slot):
            session["stage"] = stage
            return bridge.say(session, stage)

    # Existing WebSocket code calculates fare and reads the summary.
    session["stage"] = "fare"
    return bridge.say(session, "wait")

def context_handle(session, text):
    text = str(text).strip()
    stage = session.get("stage")

    # Preserve the existing cancellation and account-save flow.
    if session.get("mode") == "cancel" or stage in (
        "cancel_select", "cancel_confirm",
        "booking_pending", "cancel_pending", "done",
    ):
        return bridge._context_original_handle(session, text)

    # Only a short explicit confirmation can submit the booking.
    simple_reply = re.sub(
        r"[^\w\s]", "", context_normalize(text)
    ).strip()

    confirmation_replies = {
        "yes", "yeah", "yep", "yes please", "confirm",
        "book it", "go ahead", "okay", "ok",
        "no", "nope", "cancel", "dont book", "do not book",
        "ஆம்", "ஆமாம்", "ஆமா", "சரி", "இல்லை", "வேண்டாம்",
        "aama", "aamam", "seri", "sari", "illai", "vendam",
    }

    if stage == "confirm" and simple_reply in confirmation_replies:
        return bridge._context_original_handle(session, text)

    # Preserve previously collected details.
    try:
        extracted = context_extract(text, session)
    except Exception as error:
        print("Context extraction failed:", type(error).__name__)
        return bridge.say(session, "retry"), None

    changed = False

    passenger_span = extracted.get("passengers_text")
    if context_is_stated(passenger_span, text):
        count = bridge.number(passenger_span)
        if count and 1 <= count <= 20:
            changed |= session.get("count") != count
            session["count"] = count

    selected_vehicle = extracted.get("vehicle")
    detected_vehicle = bridge.vehicle(text)

    explicit_mini = bool(
        re.search(r"(?i)\bmini\b|மினி", text)
    )

    if (
        selected_vehicle in bridge.CAPACITY
        and selected_vehicle == detected_vehicle
        and (selected_vehicle != "Mini" or explicit_mini)
    ):
        changed |= session.get("vehicle") != selected_vehicle
        session["vehicle"] = selected_vehicle

    for extracted_key, slot in (
        ("pickup", "pickup"),
        ("destination", "destination"),
        ("time", "when"),
    ):
        value = extracted.get(extracted_key)

        if context_is_stated(value, text):
            value = value.strip()

            if session.get(slot) != value:
                session[slot] = value
                changed = True

                if slot in ("pickup", "destination"):
                    session[slot + "_geo"] = None

    if changed:
        session["fare"] = None
        session.pop("pending", None)
        session.pop("suggested_destination", None)

    if stage == "confirm" and not changed:
        # An unclear reply does not create a booking.
        return bridge.say(session, "retry"), None

    return context_next_question(session), None

def context_road_fare(session):
    # Resolve locations only after all booking details are available.
    for slot in ("pickup", "destination"):
        if not session.get(slot + "_geo"):
            geo = bridge.geocode(session[slot])
            if not geo:
                return None
            session[slot + "_geo"] = geo

    return bridge._context_original_fare(session)

bridge.handle = context_handle
bridge.road_fare = context_road_fare

print("Context-aware booking enabled for English, Tamil and Tanglish.")
print("Close the current call and start a new call.")

Context-aware booking enabled for English, Tamil and Tanglish.
Close the current call and start a new call.


In [61]:
from pathlib import Path
import re
import shutil

file = Path(
    "/kaggle/working/MOVIO_MAIN_WEBSITE/"
    "integrations/ai_call_page.html"
)

if not file.exists():
    raise FileNotFoundError("Run the website installation cells first.")

html = file.read_text(encoding="utf-8")

if 'id="ai-audio-tap-fix"' not in html:
    updated, count = re.subn(
        r"await\s+audio\.play\(\)",
        "await movioPlayWithTap(audio)",
        html,
        count=1,
    )

    if count != 1:
        raise RuntimeError(
            "Expected audio playback code was not found. "
            "Please upload your current call-page file."
        )

    helper = r"""
<script id="ai-audio-tap-fix">
async function movioPlayWithTap(player) {
    try {
        await player.play();
    } catch (error) {
        if (error.name !== "NotAllowedError") throw error;

        await new Promise((resolve) => {
            const button = document.createElement("button");
            button.textContent = "Tap to hear AI";
            button.style.cssText =
                "position:fixed;bottom:24px;left:50%;" +
                "transform:translateX(-50%);z-index:99999;" +
                "padding:14px 24px;background:#ffd54f;" +
                "color:#111;border:0;border-radius:12px;" +
                "font-weight:bold;cursor:pointer";

            button.onclick = () => {
                player.play().then(() => {
                    button.remove();
                    resolve();
                }).catch(() => {
                    button.textContent = "Tap again to hear AI";
                });
            };

            document.body.appendChild(button);
        });
    }
}
</script>
"""

    shutil.copy2(file, file.with_suffix(".html.audio_backup"))
    updated = updated.replace("</body>", helper + "\n</body>")
    file.write_text(updated, encoding="utf-8")
    print("Audio playback fallback installed.")
else:
    print("Audio playback fallback is already installed.")

print("Close the call page, reopen it, and start a call.")
print("If audio is blocked, click 'Tap to hear AI'.")

Audio playback fallback installed.
Close the call page, reopen it, and start a call.
If audio is blocked, click 'Tap to hear AI'.


In [62]:
import re

if not hasattr(bridge.TTS, "_before_service_help_fix"):
    bridge.TTS._before_service_help_fix = bridge.TTS.synthesize

def service_help_synthesize(text, group="ENGLISH"):
    if str(group).upper() == "TANGLISH":
        text = re.sub(r"\bservice\b", "சர்வீஸ்", str(text), flags=re.I)
        text = re.sub(r"\bhelp\b", "ஹெல்ப்", text, flags=re.I)

    return bridge.TTS._before_service_help_fix(text, group)

bridge.TTS.synthesize = service_help_synthesize

# Clear cached greetings so the new pronunciation is generated.
for name in (
    "cached_question", "ride_cached_prompt", "cached_static_audio"
):
    cache = globals().get(name)
    if hasattr(cache, "cache_clear"):
        cache.cache_clear()

print("Pronunciation updated. Start a new Tanglish call.")

Pronunciation updated. Start a new Tanglish call.


In [63]:
import unicodedata
import re

if not hasattr(bridge, "_before_tamil_confirmation_fix"):
    bridge._before_tamil_confirmation_fix = bridge.handle

def confirmation_text(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()

    # Remove punctuation only; preserve Tamil letters and vowel marks.
    text = "".join(
        " " if unicodedata.category(char).startswith("P") else char
        for char in text
    )
    return re.sub(r"\s+", " ", text).strip()

def tamil_confirmation_handle(session, text):
    if session.get("stage") == "confirm":
        reply = confirmation_text(text)

        positive = {
            "ஆம்", "ஆமாம்", "ஆமா", "சரி",
            "ஆம் புக் செய்க",
            "ஆம் புக் செய்யுங்கள்",
            "ஆம் புக் பண்ணுங்க",
            "ஆம் book பண்ணுங்க",
            "புக் பண்ணுங்க",
            "உறுதி செய்யுங்கள்",
            "yes", "yes please", "confirm", "book it", "okay", "ok",
            "aama", "aamam", "seri", "sari",
        }

        negative = {
            "இல்லை", "இல்ல", "வேண்டாம்",
            "ரத்து", "ரத்து செய்யுங்கள்",
            "no", "nope", "cancel", "do not book",
            "illai", "illa", "vendam",
        }

        if reply in positive:
            return bridge._context_original_handle(session, "yes")

        if reply in negative:
            return bridge._context_original_handle(session, "no")

    return bridge._before_tamil_confirmation_fix(session, text)

bridge.handle = tamil_confirmation_handle

print("Tamil confirmation fixed. Start a new call.")

Tamil confirmation fixed. Start a new call.


In [64]:

import re

_TIME_CLOCK = re.compile(r"(?<!\d)(0?[1-9]|1[0-2]|1[3-9]|2[0-3])[:.](\d{2})(?!\d)")
_PERIOD = re.compile(
    r"காலை|மதியம்|மாலை|இரவு|அதிகாலை|நண்பகல்|நள்ளிரவு|"
    r"\b(?:am|pm|a\.m\.|p\.m\.|morning|afternoon|evening|night|"
    r"kaalai|kalai|maalai|malai|mathiyam|madhiyam|iravu)\b", re.I
)

def movio_time_period(value):
    value = str(value or "")
    match = _PERIOD.search(value)
    if match:
        token = match.group().lower()
        if token in ("am", "a.m.", "morning", "kaalai", "kalai", "காலை", "அதிகாலை"):
            return "காலை"
        if token in ("afternoon", "mathiyam", "madhiyam", "மதியம்", "நண்பகல்"):
            return "மதியம்"
        if token in ("evening", "maalai", "malai", "மாலை"):
            return "மாலை"
        return "இரவு"

    clock = _TIME_CLOCK.search(value)
    if clock and int(clock.group(1)) >= 13:
        hour = int(clock.group(1))
        return "மதியம்" if hour < 16 else "மாலை" if hour < 19 else "இரவு"
    return None

def movio_time_clarification(session):
    lang = str(session.get("language") or bridge.CURRENT_LANGUAGE.get()).lower()
    if "english" in lang and "tanglish" not in lang:
        return "Is that AM or PM? Please specify morning, afternoon, evening, or night."
    if "tanglish" in lang:
        return "Neenga sonna time kaalaiya, madhiyama, maalaiya, illa irava? Clear ah sollunga."
    return "நீங்கள் சொன்ன நேரம் காலையா, மதியமா, மாலையா அல்லது இரவா? தெளிவாகச் சொல்லுங்கள்."

if not hasattr(bridge, "_before_time_clarification_handle"):
    bridge._before_time_clarification_handle = bridge.handle

if not hasattr(bridge, "_before_time_clarification_say"):
    bridge._before_time_clarification_say = bridge.say

def movio_time_handle(session, text):
    text = str(text).strip()

    if session.get("_awaiting_time_period"):
        period = movio_time_period(text)
        if not period:
            return movio_time_clarification(session), None

        previous = str(session.get("when") or "")
        session["when"] = (previous + " " + period).strip()
        session["_awaiting_time_period"] = False
        session["fare"] = None
        session["stage"] = "time"
        return context_next_question(session), None

    result = bridge._before_time_clarification_handle(session, text)

    if (
        session.get("stage") == "fare"
        and session.get("when")
        and not movio_time_period(session["when"])
    ):
        session["_awaiting_time_period"] = True
        session["stage"] = "time_period"
        return movio_time_clarification(session), None

    return result

def movio_time_say(session, key, **kwargs):
    response = bridge._before_time_clarification_say(session, key, **kwargs)

    if not isinstance(response, str):
        return response

    when = str(session.get("when") or "")
    period = movio_time_period(when)
    clock = _TIME_CLOCK.search(when)

    if not period or not clock:
        return response

    if not re.search(
        r"booking details|your booking|முன்பதிவு|confirm pannalama",
        response, re.I
    ):
        return response

    hour, minute = int(clock.group(1)), clock.group(2)
    display = f"{hour % 12 or 12:02d}:{minute}"

    lang = str(session.get("language") or bridge.CURRENT_LANGUAGE.get()).lower()

    if "tanglish" in lang:
        word = {
            "காலை": "kaalai",
            "மதியம்": "madhiyam",
            "மாலை": "maalai",
            "இரவு": "iravu"
        }[period]
        spoken = f"{word} {display} manikku"
    elif "english" in lang:
        word = {
            "காலை": "morning",
            "மதியம்": "afternoon",
            "மாலை": "evening",
            "இரவு": "night"
        }[period]
        spoken = f"{display} in the {word}"
    else:
        spoken = f"{period} {display} மணிக்கு"

    response = re.sub(
        r"(?:(?:காலை|மதியம்|மாலை|இரவு)\s*)?"
        + re.escape(clock.group())
        + r"(?:\s*மணி(?:க்கு)?)?",
        spoken,
        response,
        count=1
    )
    return response

bridge.handle = movio_time_handle
bridge.say = movio_time_say

print("MOVIO time confirmation patch enabled. Start a new call.")


MOVIO time confirmation patch enabled. Start a new call.


# Part A continued — integrated weather assistance

These cells are already included in this complete notebook. Run them after the original application cells above. They update the existing website and AI call; model, tunnel, account, driver, map and OTP settings above are retained.

Manual booking gains date/time and passenger inputs plus a weather review before submission. Voice booking uses two distinct consents: weather advisory and first Yes, then route/fare summary and second Yes. Only the second Yes can authorize storage. Supplied slots and corrections persist in the account-bound call state; route failures preserve them for retry.

Numeric values remain digits in the text sent to the original TTS adapter. An acoustic normalization wrapper corrects decimal points, percentages, currency and booking-ID digit order only for the new voice route. Calls outside that route use the original normalizer, including the unchanged historical evaluation harness. Pronunciation still requires real listening checks.

The original dashboard/call theme is retained. The previously requested spoken-only weather and removal of the voice date picker apply to the call popup. New manual weather controls sit inside the existing booking area.


## 1. Check the current kernel
Run after the original `Context-aware booking enabled for English, Tamil and Tanglish` cell and any later pronunciation/application patches. You do not need to rerun the long historical evaluation cells.


In [65]:
from pathlib import Path
import importlib.util

WX_ROOT=Path('/kaggle/working/MOVIO_MAIN_WEBSITE')
if not (WX_ROOT/'server.py').is_file():
    raise RuntimeError('Run the original website installation and startup cells first.')
if globals().get('bridge') is None or not callable(globals().get('context_handle')):
    raise RuntimeError('Use the original running kernel. Run its voice-bridge and context-aware booking setup first.')
if not getattr(bridge,'TTS',None) or not getattr(bridge,'PUBLIC_URL','').startswith('https://'):
    raise RuntimeError('Configure the original TTS bridge and shared HTTPS tunnel first.')
for package in ('requests','fastapi','starlette','av','psutil'):
    if importlib.util.find_spec(package) is None:
        raise RuntimeError('Missing original application dependency: '+package)
(WX_ROOT/'integrations').mkdir(exist_ok=True)
WX_RAIN_THRESHOLD=60  # configurable advisory threshold, not a validated safety threshold
WX_HEAT_C=38          # application threshold for high-temperature precautions
WX_FEELS_LIKE_C=40    # alternative apparent-temperature threshold
WX_SENTENCE_PAUSE_MS=220  # short playback pause; 0 restores single-clip generation
print('Current bridge found. Close old call popups before cell 7.')


Current bridge found. Close old call popups before cell 7.


## 2. Add the weather client and confirmation guard
This writes the shared forecast/advice module and the manual review/confirm API. Weather requests run outside the WebSocket receive loop and outside the website database lock. Pickup and destination forecasts are checked at the pickup hour; this is not a forecast of every road segment or the arrival hour. Heavy rain uses the provider's condition code, not the probability percentage. Heat thresholds above are application settings, not official heatwave warnings.


In [66]:
%%writefile /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_weather_guard.py
"""Weather-aware manual and voice booking assistance for MOVIO v5.3.

Prototype integration. Weather is supplied by Open-Meteo, not predicted by Qwen.
No credentials, original model weights or real recordings are bundled.
"""
from __future__ import annotations

import asyncio
import base64
import copy
import contextvars
import hashlib
import hmac
import io
import json
import math
import re
import secrets
import sqlite3
import threading
import time
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

VERSION = "0.4.0"
NUMERIC_LANGUAGE = contextvars.ContextVar('movio_weather_numeric_language', default=None)
TZ = ZoneInfo("Asia/Kolkata")
CAPACITY = {"Bike": 1, "Auto": 3, "Mini": 4, "Sedan": 4, "SUV": 6}
RAIN_CODES = {51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 97, 99}
FIELDS = ("count", "vehicle", "pickup", "destination", "pickup_geo", "destination_geo",
          "when", "pickup_at", "fare", "weather", "lang", "_wx_mode", "_wx_weather_consent")
CORE_FIELDS = ("count", "vehicle", "pickup", "destination", "when")


class GuardError(ValueError):
    pass


def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def digest(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()


def snapshot(state):
    return {key: copy.deepcopy(state.get(key)) for key in FIELDS}


def localized(lang, en, ta, tanglish):
    return ta if lang == "Tamil" else tanglish if lang == "Tanglish" else en


def reply_normalized(text):
    # Preserve Tamil vowel signs and pulli marks while removing punctuation.
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    text = "".join(" " if unicodedata.category(c).startswith("P") else c for c in text)
    return re.sub(r"\s+", " ", text).strip()


POSITIVE = {"yes", "yeah", "yep", "yes please", "confirm", "book it", "go ahead", "okay", "ok",
            "continue", "ஆம்", "ஆமாம்", "ஆமா", "சரி", "தொடரலாம்", "தொடருங்கள்", "ஆம் தொடரலாம்",
            "ஆம் புக் செய்க", "உறுதி செய்யுங்கள்", "aama", "aamam", "seri", "sari", "continue pannunga"}
POSITIVE.update({"yes continue","yes book it","yes please continue","ஆமாங்க","சரிதான்","சரியாக உள்ளது",
                 "ஆம் தொடருங்கள்","ஆமாம் தொடரலாம்","சரி புக் பண்ணுங்க","aamaa","amam","aam",
                 "keep bike","keep the bike","பைக் தொடரட்டும்","bike continue"})
NEGATIVE = {"no", "nope", "cancel", "do not book", "dont book", "இல்லை", "வேண்டாம்", "illai", "vendam"}
TA_SMALL = ("பூஜ்ஜியம்", "ஒன்று", "இரண்டு", "மூன்று", "நான்கு", "ஐந்து", "ஆறு", "ஏழு", "எட்டு", "ஒன்பது",
            "பத்து", "பதினொன்று", "பன்னிரண்டு", "பதின்மூன்று", "பதினான்கு", "பதினைந்து", "பதினாறு",
            "பதினேழு", "பதினெட்டு", "பத்தொன்பது")
EN_SMALL = ("zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven",
            "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen")
MONTHS_EN = ("January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December")
MONTHS_TA = ("ஜனவரி", "பிப்ரவரி", "மார்ச்", "ஏப்ரல்", "மே", "ஜூன்", "ஜூலை", "ஆகஸ்ட்", "செப்டம்பர்", "அக்டோபர்", "நவம்பர்", "டிசம்பர்")
TA_ROMAN_SMALL = ("poojjiyam","ondru","irandu","moondru","naangu","aindhu","aaru","ezhu","ettu","onbadhu",
                  "pathu","padhinondru","pannirandu","padhimoondru","padhinaangu","padhinaindhu","padhinaaru",
                  "padhinezhu","padhinettu","padhonbadhu")
VEHICLES_TA = {"Bike":"பைக்","Auto":"ஆட்டோ","Mini":"மினி கார்","Sedan":"செடான்","SUV":"எஸ் யூ வி"}
VEHICLES_ROMAN = {"Bike":"baikku","Auto":"aatto","Mini":"siriya kaar","Sedan":"sedaan kaar","SUV":"es yoo vee"}
AM_RE = r"\ba\.?m\.?\b|\bmorning\b|காலை|அதிகாலை|\b(?:kaalai|kalai)\b"
PM_RE = r"\bp\.?m\.?\b|\b(?:afternoon|evening|night|tonight)\b|மாலை|இரவு|மதியம்|நண்பகல்|\b(?:maalai|malai|iravu|mathiyam|madhiyam)\b"
TODAY_RE = r"\b(?:today|tonight|innaiku|inniku|indru)\b|இன்று|இன்னைக்கு"
TOMORROW_RE = r"\b(?:tomorrow|naalai|nalai|naalaiku|naalaikku)\b|நாளை|நாளைக்கு"


def number_words(value, lang="English"):
    n = int(value)
    if lang == "Tanglish":
        if n < 0:return "minus " + number_words(-n,lang)
        if n < 20:return TA_ROMAN_SMALL[n]
        if n < 100:
            full=("","","irupadhu","muppadhu","naarppadhu","aimbadhu","arubadhu","ezhubadhu","enbadhu","thonnooru")
            join=("","","irupathu","muppathu","naarppathu","aimbathu","arubathu","ezhubathu","enbathu","thonnootru")
            return full[n//10] if not n%10 else join[n//10]+" "+TA_ROMAN_SMALL[n%10]
        if n < 1000:
            full=("","nooru","irunooru","munnooru","naanooru","ainnooru","arunooru","ezhunooru","ennooru","thollaayiram")
            join=("","nootru","irunootru","munnootru","naanoootru","ainnootru","arunootru","ezhunootru","ennootru","thollaayirathu")
            return full[n//100] if not n%100 else join[n//100]+" "+number_words(n%100,lang)
        return number_words(n//1000,lang)+(" aayiram" if not n%1000 else " aayirathu "+number_words(n%1000,lang))
    if n < 0:
        return ("மைனஸ் " if lang == "Tamil" else "minus ") + number_words(-n, lang)
    if lang != "Tamil":
        if n < 20:
            return EN_SMALL[n]
        if n < 100:
            tens = ("", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety")
            return tens[n // 10] + (" " + EN_SMALL[n % 10] if n % 10 else "")
        if n < 1000:
            return EN_SMALL[n // 100] + " hundred" + (" " + number_words(n % 100, lang) if n % 100 else "")
        return number_words(n // 1000, lang) + " thousand" + (" " + number_words(n % 1000, lang) if n % 1000 else "")
    if n < 20:
        return TA_SMALL[n]
    if n < 100:
        full = ("", "", "இருபது", "முப்பது", "நாற்பது", "ஐம்பது", "அறுபது", "எழுபது", "எண்பது", "தொண்ணூறு")
        join = ("", "", "இருபத்து", "முப்பத்து", "நாற்பத்து", "ஐம்பத்து", "அறுபத்து", "எழுபத்து", "எண்பத்து", "தொண்ணூற்று")
        return full[n // 10] if not n % 10 else join[n // 10] + " " + TA_SMALL[n % 10]
    if n < 1000:
        full = ("", "நூறு", "இருநூறு", "முந்நூறு", "நானூறு", "ஐந்நூறு", "அறுநூறு", "எழுநூறு", "எண்ணூறு", "தொள்ளாயிரம்")
        join = ("", "நூற்று", "இருநூற்று", "முந்நூற்று", "நானூற்று", "ஐந்நூற்று", "அறுநூற்று", "எழுநூற்று", "எண்ணூற்று", "தொள்ளாயிரத்து")
        return full[n // 100] if not n % 100 else join[n // 100] + " " + number_words(n % 100, lang)
    if n < 10000:
        full = ("", "ஆயிரம்", "இரண்டாயிரம்", "மூன்றாயிரம்", "நான்காயிரம்", "ஐந்தாயிரம்", "ஆறாயிரம்", "ஏழாயிரம்", "எட்டாயிரம்", "ஒன்பதாயிரம்")
        head = full[n // 1000]
        return head if not n % 1000 else head[:-2] + "த்து " + number_words(n % 1000, lang)
    return str(n)


def spoken_datetime(value, lang='English'):
    dt = datetime.fromisoformat(value) if isinstance(value, str) else value
    if dt.tzinfo is not None:
        dt = dt.astimezone(TZ)
    period = ('நள்ளிரவு' if dt.hour == 0 else 'அதிகாலை' if dt.hour < 5 else 'காலை' if dt.hour < 12
              else 'மதியம்' if dt.hour < 16 else 'மாலை' if dt.hour < 19 else 'இரவு')
    clock = f'{dt.hour % 12 or 12}:{dt.minute:02}'
    if lang == 'Tamil':
        return f'{MONTHS_TA[dt.month-1]} {dt.day}, {dt.year}, {period} {clock}'
    if lang == 'Tanglish':
        months = ('Janavari','Pibravari','Maarch','Epral','Me','Joon','Joolai','Aagast','Septambar','Aktoabar','Navambar','Disambar')
        period = ('nalliravu' if dt.hour == 0 else 'adhikaalai' if dt.hour < 5 else 'kaalai' if dt.hour < 12
                  else 'madhiyam' if dt.hour < 16 else 'maalai' if dt.hour < 19 else 'iravu')
        return f'{months[dt.month-1]} {dt.day}, {dt.year}, {period} {clock}'
    return f'{MONTHS_EN[dt.month-1]} {dt.day}, {dt.year}, at {clock} {"AM" if dt.hour < 12 else "PM"}'


def time_components(raw, now, allow_bare=False):
    text = unicodedata.normalize('NFKC', str(raw or '')).casefold().translate(str.maketrans('௦௧௨௩௪௫௬௭௮௯','0123456789'))
    now = now.astimezone(TZ)
    found, spans = {}, []
    explicit = re.search(r'(?<!\d)(\d{4}-\d{2}-\d{2})(?!\d)', text)
    date_match = None
    if explicit:
        try:
            found['date'] = datetime.strptime(explicit[1], '%Y-%m-%d').date().isoformat()
        except ValueError as exc:
            raise GuardError('invalid_datetime') from exc
        spans.append(explicit.span())
    else:
        relative = ((r'\bday after tomorrow\b|நாளை\s*மறுநாள்|நாளைமறுநாள்|\bnaalai marunaal\b', 2),
                    (TOMORROW_RE, 1), (TODAY_RE, 0))
        for pattern, days in relative:
            match = re.search(pattern, text)
            if match:
                found['date'] = (now + timedelta(days=days)).date().isoformat()
                spans.append(match.span()); break
        if not found:
            match = re.search(r'(?<!\d)(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?!\d)', text)
            if match:
                try:
                    found['date'] = datetime(int(match[3]), int(match[2]), int(match[1])).date().isoformat()
                except ValueError as exc:
                    raise GuardError('invalid_datetime') from exc
                spans.append(match.span())
        if not found:
            days = {word:n for n,word in enumerate(TA_DAY_WORDS) if n}
            days.update({number_words(n,'Tamil'):n for n in range(1,32)})
            day_pattern = r'\d{1,2}(?:st|nd|rd|th|ஆம்|ம்)?|' + '|'.join(re.escape(k) for k in sorted(days,key=len,reverse=True))
            month_map = {name.casefold():n for n,name in enumerate(MONTHS_EN,1)}
            month_map.update({name.casefold()[:3]:n for n,name in enumerate(MONTHS_EN,1)})
            month_map.update({name:n for n,name in enumerate(MONTHS_TA,1)})
            month_pattern = '|'.join(re.escape(k) for k in sorted(month_map,key=len,reverse=True))
            patterns = (
                rf'(?<![\w\u0b80-\u0bff])(?P<day>{day_pattern})\s+(?P<month>{month_pattern})(?:\s*,?\s*(?P<year>\d{{4}}))?',
                rf'(?<![\w\u0b80-\u0bff])(?P<month>{month_pattern})\s+(?P<day>{day_pattern})(?:\s*(?:தேதி|தேதியன்று))?(?:\s*,?\s*(?P<year>\d{{4}}))?')
            for pattern in patterns:
                match = re.search(pattern, text)
                if not match:
                    continue
                day_raw = match['day']; digits = re.match(r'\d+', day_raw)
                day = int(digits[0]) if digits else days[day_raw]
                try:
                    found['date'] = datetime(int(match['year'] or now.year), month_map[match['month']], day).date().isoformat()
                except ValueError as exc:
                    raise GuardError('invalid_datetime') from exc
                spans.append(match.span()); date_match = match; break
        if not found:
            weekdays = (('monday','திங்கள்'),('tuesday','செவ்வாய்'),('wednesday','புதன்'),('thursday','வியாழன்'),
                        ('friday','வெள்ளி'),('saturday','சனி'),('sunday','ஞாயிறு'))
            for weekday,names in enumerate(weekdays):
                match = next((re.search(r'(?<![a-z])'+re.escape(name)+r'(?![a-z])',text) for name in names
                              if re.search(r'(?<![a-z])'+re.escape(name)+r'(?![a-z])',text)),None)
                if match:
                    days = (weekday-now.weekday()) % 7
                    if re.search(r'\bnext\b|அடுத்த',text) and days == 0: days = 7
                    found['date'] = (now+timedelta(days=days)).date().isoformat();spans.append(match.span());break
    chars = list(text)
    for begin,end in spans:
        chars[begin:end] = ' ' * (end-begin)
    clock_text = ''.join(chars)
    lexicon = number_lexicon()
    tokens = '|'.join(re.escape(k) for k in sorted(lexicon,key=len,reverse=True) if not k.isdigit())
    number_pattern = rf'(?P<hour>(?<!\d)\d{{1,2}}|(?<![\w\u0b80-\u0bff])(?:{tokens})(?![\w\u0b80-\u0bff]))'
    clock = re.search(r'(?<!\d)(\d{1,2})[:.](\d{2})(?!\d)',clock_text)
    iso_clock = bool(explicit and re.search(r'\d{4}-\d{2}-\d{2}[ t]\d{2}:\d{2}',text))
    if clock:
        found.update(hour=int(clock[1]),minute=int(clock[2]),clock24=iso_clock or int(clock[1])==0 or int(clock[1])>12)
    else:
        # A clock unit binds its own number. Passenger counts elsewhere are not candidates.
        cue = re.search(number_pattern+r'\s*(?:மணி(?:க்கு|யளவில்)?|manikku|mani|o.clock|a\.?m\.?|p\.?m\.?)(?![a-z])',clock_text)
        if not cue:
            periods = r'காலை|மதியம்|நண்பகல்|மாலை|நள்ளிரவு|இரவு|\b(?:morning|afternoon|evening|night|noon|midnight|kaalai|kalai|madhiyam|mathiyam|maalai|malai|iravu|nalliravu)\b'
            cue = re.search(r'(?:'+periods+r')\s+(?:at\s+)?'+number_pattern,clock_text)
        if not cue and allow_bare:
            cue = re.fullmatch(r'\s*'+number_pattern+r'\s*[.!?]?\s*',clock_text)
        if cue:
            hour = int(cue['hour']) if cue['hour'].isdigit() else lexicon[cue['hour']]
            found.update(hour=hour,minute=0,clock24=hour==0 or hour>12)
    explicit_am = bool(re.search(r'(?<![a-z])a\.?m\.?(?![a-z])',clock_text))
    explicit_pm = bool(re.search(r'(?<![a-z])p\.?m\.?(?![a-z])',clock_text))
    morning = bool(re.search(r'காலை|அதிகாலை|\b(?:morning|kaalai|kalai|adhikaalai)\b',clock_text))
    afternoon = bool(re.search(r'மதியம்|நண்பகல்|மாலை|\b(?:afternoon|evening|madhiyam|mathiyam|maalai|malai|noon)\b',clock_text))
    midnight = bool(re.search(r'நள்ளிரவு|\b(?:midnight|nalliravu)\b',clock_text))
    night = bool(re.search(r'(?<!நள்)இரவு|\b(?:night|tonight|iravu)\b',clock_text))
    if (explicit_am and explicit_pm or morning and afternoon or midnight and afternoon
            or explicit_pm and (morning or midnight) or explicit_am and afternoon):
        raise GuardError('clarify_period')
    if explicit_am or explicit_pm:
        found['period'] = 'am' if explicit_am else 'pm'
        found['day_period'] = found['period']
    elif midnight:
        found['period']='am';found['day_period']='midnight'
        if 'hour' not in found:found.update(hour=12,minute=0,clock24=False)
    elif morning or afternoon:
        found['period']='am' if morning else 'pm';found['day_period']=found['period']
        if 'hour' not in found and re.search(r'\bnoon\b|நண்பகல்',clock_text):found.update(hour=12,minute=0,clock24=False)
    elif night:
        found['day_period']='night'
        if 'hour' in found and 1 <= found['hour'] <= 12:
            found['period']='am' if found['hour']<=5 or found['hour']==12 else 'pm'
    if 'hour' in found and not (0 <= found['hour'] <= 23 and 0 <= found.get('minute',0) <= 59):
        raise GuardError('invalid_datetime')
    return found


def components_datetime(parts, now):
    if not parts.get('date'):raise GuardError('clarify_date')
    if 'hour' not in parts:raise GuardError('clarify_clock')
    hour = parts['hour']; period = parts.get('period')
    if not parts.get('clock24'):
        if parts.get('day_period') == 'night':period='am' if hour<=5 or hour==12 else 'pm'
        if not period and 1 <= hour <= 12:raise GuardError('clarify_period')
        if period:
            if not 1 <= hour <= 12:raise GuardError('invalid_datetime')
            hour = hour % 12 + (12 if period=='pm' else 0)
    try:
        value = datetime.fromisoformat(parts['date']).replace(tzinfo=TZ,hour=hour,minute=parts.get('minute',0))
    except ValueError as exc:
        raise GuardError('invalid_datetime') from exc
    if value <= now.astimezone(TZ):raise GuardError('past_datetime')
    return value


def resolve_pickup(raw, now=None, fallback_date=None):
    now = (now or datetime.now(TZ)).astimezone(TZ)
    parts = time_components(raw, now, allow_bare=True)
    if not parts.get("date") and fallback_date:
        parts["date"] = datetime.fromisoformat(fallback_date).date().isoformat()
    return components_datetime(parts, now)


def update_time_memory(state, text, now):
    parts = copy.deepcopy(state.get('_wx_time') or {})
    if not parts and state.get('pickup_at'):
        dt = datetime.fromisoformat(state['pickup_at']).astimezone(TZ)
        parts.update(date=dt.date().isoformat(),hour=dt.hour,minute=dt.minute,clock24=True)
    elif not parts and state.get('when'):
        try:parts.update(time_components(state['when'],now,allow_bare=True))
        except GuardError:pass
    updates = time_components(text,now,allow_bare=state.get('stage') in ('time','time_period'))
    if updates:
        if 'day_period' in updates and 'period' not in updates:parts.pop('period',None)
        parts.update(updates)
        if ('period' in updates or 'day_period' in updates) and 'hour' not in updates and parts.get('hour',24)<=12:
            parts['clock24']=False
        state.pop('_wx_time_error',None)
        for slot,keys in (('date',('date',)),('time',('hour','minute','period','day_period','clock24'))):
            accepted={key:updates[key] for key in keys if key in updates}
            if accepted:state.setdefault('_wx_slot_evidence',{})[slot]={'text':str(text),'source':'rule','confidence':1.0,'components':accepted}
    elif state.get('_wx_time_error'):
        return None,state['_wx_time_error']
    state['_wx_time']=parts
    try:pickup=components_datetime(parts,now)
    except GuardError as exc:
        if updates:state['when']=canonical(parts)
        return None,str(exc)
    state['when']=state['pickup_at']=pickup.isoformat()
    return pickup,None


class WeatherClient:
    def __init__(self, getter=None, ttl=900, rain_threshold=60, heat_c=38, feels_like_c=40):
        if getter is None:
            import requests
            getter = requests.get
        self.getter, self.ttl, self.rain_threshold = getter, ttl, rain_threshold
        # Application advisory thresholds, not an official heatwave warning.
        self.heat_c,self.feels_like_c=float(heat_c),float(feels_like_c)
        self.cache, self.lock = {}, threading.RLock()
        self.requests = self.hits = 0

    def forecast(self, points, pickup, now=None):
        now = now or datetime.now(TZ)
        horizon = (pickup.date() - now.astimezone(TZ).date()).days
        if not 0 <= horizon < 16:
            return {"status": "outside_forecast_range", "source": "Open-Meteo"}
        key = (tuple((round(float(p["lat"]), 3), round(float(p["lng"]), 3)) for p in points), pickup.date().isoformat())
        with self.lock:
            cached = self.cache.get(key)
            if cached and time.monotonic() - cached[0] < self.ttl:
                self.hits += 1
                payload, fetched = cached[1], cached[2]
            else:
                try:
                    self.requests += 1
                    response = self.getter("https://api.open-meteo.com/v1/forecast", params={
                        "latitude": ",".join(str(p["lat"]) for p in points),
                        "longitude": ",".join(str(p["lng"]) for p in points),
                        "hourly": "weather_code,precipitation_probability,precipitation,temperature_2m,apparent_temperature",
                        "timezone": "Asia/Kolkata", "start_date": pickup.date().isoformat(),
                        "end_date": pickup.date().isoformat()}, timeout=(3, 7))
                    response.raise_for_status()
                    payload = response.json()
                    payload = payload if isinstance(payload, list) else [payload]
                    if len(payload) != len(points):
                        raise ValueError("Unexpected location count")
                    fetched = time.time()
                    self.cache[key] = (time.monotonic(), payload, fetched)
                    if len(self.cache) > 256:
                        self.cache.pop(next(iter(self.cache)))
                except Exception:
                    return {"status": "unavailable", "source": "Open-Meteo"}
        target = pickup.replace(minute=0, second=0, microsecond=0).strftime("%Y-%m-%dT%H:%M")
        rows = []
        try:
            for name, item in zip(("pickup", "destination"), payload):
                hourly = item["hourly"]
                index = hourly["time"].index(target)
                code = hourly["weather_code"][index]
                if code is None:
                    raise ValueError("Missing condition")
                row = {"location": name, "valid_at": target, "weather_code": int(code)}
                for field in ("precipitation_probability", "precipitation", "temperature_2m", "apparent_temperature"):
                    v = hourly.get(field,[None]*len(hourly["time"]))[index]
                    row[field] = float(v) if v is not None and math.isfinite(float(v)) else None
                rows.append(row)
        except (KeyError, ValueError, TypeError, IndexError):
            return {"status": "unavailable", "source": "Open-Meteo"}
        rain = any(r["weather_code"] in RAIN_CODES or
                   (r["precipitation_probability"] is not None and r["precipitation_probability"] >= self.rain_threshold)
                   for r in rows)
        cloudy = any(r["weather_code"] in (2, 3) for r in rows)
        clear = all(r["weather_code"] in (0, 1) for r in rows)
        heavy=any(r["weather_code"] in (65,67,82) for r in rows)
        storm=any(r["weather_code"] in (95,96,97,99) for r in rows)
        heat=any((r["temperature_2m"] is not None and r["temperature_2m"]>=self.heat_c) or
                 (r["apparent_temperature"] is not None and r["apparent_temperature"]>=self.feels_like_c) for r in rows)
        pleasant=clear and all(r["temperature_2m"] is not None and 18<=r["temperature_2m"]<=32 for r in rows) and not heat
        return {"status": "ok", "condition": "rain" if rain else "cloudy" if cloudy else "clear" if clear else "other",
                "points": rows, "fetched_at": fetched, "source": "Open-Meteo",
                "heavy_rain":heavy,"thunderstorm":storm,"heat":heat,"pleasant":pleasant,
                "advisory_thresholds":{"rain_probability_pct":self.rain_threshold,"heat_c":self.heat_c,"feels_like_c":self.feels_like_c},
                "scope": "pickup_and_destination_at_pickup_hour"}


def spoken_number(value, lang="English"):
    text = f"{float(value):.1f}".rstrip("0").rstrip(".")
    whole, _, decimal = text.partition(".")
    result = number_words(int(whole), lang)
    if decimal:
        result += (" புள்ளி " if lang == "Tamil" else " pulli " if lang == "Tanglish" else " point ") + " ".join(number_words(int(d), lang) for d in decimal)
    return result


def probability_report(state):
    lang, weather = state['lang'], state.get('weather') or {}
    if weather.get('status') != 'ok':
        return localized(lang,'The rain forecast for that journey time is not available. You can still continue with booking.',
            'அந்தப் பயண நேரத்திற்கான மழை முன்னறிவிப்பு கிடைக்கவில்லை. இருப்பினும் முன்பதிவைத் தொடரலாம்.',
            'Andha payana nerathirkaana mazhai munnarivippu kidaikkavillai. Irundhaalum munpadhivai thodaralaam.')
    lines=[]
    for key,en,ta in (('pickup','pickup','புறப்படும்'),('destination','drop','சேரும்')):
        point=next((p for p in weather.get('points',[]) if p.get('location')==key),{})
        probability=point.get('precipitation_probability');name=state.get(key) or en
        if probability is None or not math.isfinite(float(probability)) or not 0<=float(probability)<=100:
            lines.append(localized(lang,f'The rain probability at your {en} location, {name}, is not available.',
                f'{ta} இடமான {name} பகுதியில் மழைக்கான வாய்ப்பு மதிப்பீடு கிடைக்கவில்லை.',
                f'{"Purappadum" if key=="pickup" else "Serum"} idamaana {name} pagudhiyil mazhaikkaana vaaippu madhippeedu kidaikkavillai.'))
        else:
            digits=numeric_text(probability)
            lines.append(localized(lang,f'At your {en} location, {name}, the chance of rain is {digits}%.',
                f'{ta} இடமான {name} பகுதியில் மழைக்கான வாய்ப்பு {digits}%.',
                f'{"Purappadum" if key=="pickup" else "Serum"} idamaana {name} pagudhiyil mazhaikkaana vaaippu {digits}%.'))
    return '\n'.join(lines)


def weather_report(state):
    lang = state["lang"]
    weather = state.get("weather") or {}
    status = weather.get("status")
    if status != "ok":
        return probability_report(state)
    condition = weather["condition"]
    if weather.get("thunderstorm"):
        message=localized(lang,"Thunderstorms are forecast around your selected pickup time.",
            "நீங்கள் தேர்ந்தெடுத்த பயண நேரத்தில் இடியுடன் கூடிய மழை எதிர்பார்க்கப்படுகிறது.",
            "Neenga therndhedutha payana nerathil idiyudan koodiya mazhai edhirpaarkkappadugiradhu.")
    elif weather.get("heavy_rain"):
        message=localized(lang,"Heavy rain is forecast around your selected pickup time.",
            "நீங்கள் தேர்ந்தெடுத்த பயண நேரத்தில் கனமழை எதிர்பார்க்கப்படுகிறது.",
            "Neenga therndhedutha payana nerathil kanamazhai edhirpaarkkappadugiradhu.")
    elif condition == "rain":
        message = localized(lang,
            "Rain is expected at the pickup or destination around your pickup time.",
            "நீங்கள் புறப்படும் நேரத்தில், புறப்படும் இடம் அல்லது சேரும் இடத்தில் மழை எதிர்பார்க்கப்படுகிறது.",
            "Neenga purappadum nerathil purappadum idam alladhu serum idathil mazhai peiya vaaippu irukkiradhu.")
    elif condition == "cloudy":
        message = localized(lang, "Cloudy conditions are expected around your pickup time.",
            "நீங்கள் புறப்படும் நேரத்தில் மேகமூட்டம் எதிர்பார்க்கப்படுகிறது.", "Neenga purappadum nerathil megamoottam edhirpaarkkappadugiradhu.")
    elif condition == "clear":
        message = localized(lang, "The weather looks pleasant around your selected pickup time." if weather.get("pleasant") else "Clear or mainly clear skies are expected around your pickup time.",
            "நீங்கள் தேர்ந்தெடுத்த பயண நேரத்தில் வானிலை நன்றாக இருக்கும் என எதிர்பார்க்கப்படுகிறது." if weather.get("pleasant") else
            "நீங்கள் புறப்படும் நேரத்தில் வானம் தெளிவாக அல்லது பெரும்பாலும் தெளிவாக இருக்கும் என எதிர்பார்க்கப்படுகிறது.",
            "Neenga therndhedutha payana nerathil vaanilai nandraagave irukkum ena edhirpaarkkappadugiradhu." if weather.get("pleasant") else "Neenga purappadum nerathil vaanam thelivaaga irukkum ena edhirpaarkkappadugiradhu.")
    else:
        message = localized(lang,
            "Other weather conditions are forecast. Please check local weather information before travelling.",
            "பிற வானிலை நிலைமைகள் எதிர்பார்க்கப்படுகின்றன. பயணத்திற்கு முன் உள்ளூர் வானிலைத் தகவல்களைப் பாருங்கள்.",
            "Pira vaanilai nilaimaigal edhirpaarkkappadugindrana. Payanathirku mun ulloor vaanilai thagavalgalai paarunga.")
    if weather.get("heat"):
        message += "\n"+localized(lang,
            "Very hot conditions are expected. Please carry water and prepare for high temperatures.",
            "வெப்பநிலை அதிகமாக இருக்கும் என எதிர்பார்க்கப்படுகிறது. குடிநீர் எடுத்துக்கொண்டு வெப்பத்திற்கு ஏற்ற முன்னேற்பாடுகளைச் செய்துகொள்ளுங்கள்.",
            "Veppanilai adhigamaaga irukkum ena edhirpaarkkappadugiradhu. Kudineer eduthukkondu veppathirku etra munnerpaadugalai seidhukollunga.")
    return probability_report(state) + "\n" + message


def advisory(state):
    lang, weather = state["lang"], state.get("weather") or {}
    message = weather_report(state)
    if weather.get("status") == "ok" and weather.get("condition") == "rain":
        if state.get("vehicle") == "Auto":
            precaution=localized(lang,"An auto can have exposed sides. Carry an umbrella and keep your belongings protected from rain.",
                "ஆட்டோவின் பக்கவாட்டில் மழை படலாம். குடை எடுத்துக்கொண்டு உங்கள் உடைமைகளை மழையிலிருந்து பாதுகாத்துக்கொள்ளுங்கள்.",
                "Aattovin pakkavaatil mazhai padalaam. Kudai eduthukkondu unga udaimaigalai mazhaiyilirundhu paadhugaathukkollunga.")
        elif state.get("vehicle") == "Bike":
            precaution=localized(lang,"Please prepare rain protection for yourself and your belongings.",
                "நீங்களும் உங்கள் உடைமைகளும் மழையில் நனையாமல் இருக்கத் தேவையான முன்னேற்பாடுகளைச் செய்துகொள்ளுங்கள்.",
                "Neengalum unga udaimaigalum mazhaiyil nanaiyaamal irukka thevaiyaana munnerpaadugalai seidhukollunga.")
        else:
            precaution=localized(lang,"Please carry an umbrella for boarding and getting out, and make suitable rain preparations.",
                "வாகனத்தில் ஏறும்போதும் இறங்கும்போதும் பயன்படுத்தக் குடை எடுத்துக்கொண்டு மழைக்கு ஏற்ற முன்னேற்பாடுகளைச் செய்துகொள்ளுங்கள்.",
                "Vandiyil erumboadhum irangumboadhum payanpadutha kudai eduthukkondu mazhaikku etra munnerpaadugalai seidhukollunga.")
        message += "\n"+precaution
    if weather.get("condition") == "rain" and state.get("vehicle") == "Bike":
        choices = [v for v in ("Mini", "Sedan", "SUV") if CAPACITY[v] >= int(state["count"])]
        if choices:
            severe=weather.get("heavy_rain") or weather.get("thunderstorm")
            if severe:
                message+="\n"+localized(lang,"A bike may be unsuitable in these conditions.","இந்த வானிலையில் பைக் பயணம் பொருத்தமில்லாமல் இருக்கலாம்.","Indha vaanilaiyil baikku payanam poruthamillaamal irukkalaam.")
            options=", ".join(choices);ta_options=", ".join(VEHICLES_TA[v] for v in choices);roman_options=", ".join(VEHICLES_ROMAN[v] for v in choices)
            message += "\n" + localized(lang,
                f"You may choose a covered vehicle such as {options}, or continue with your selected bike. The choice is yours.",
                f"நீங்கள் விரும்பினால் {ta_options} போன்ற மூடிய வாகனத்தைத் தேர்வு செய்யலாம்; அல்லது நீங்கள் தேர்ந்தெடுத்த பைக்கிலேயே தொடரலாம். முடிவு உங்களுடையது.",
                f"Neenga virumbinaal {roman_options} pondra moodiya vandiyai therndhedukkalaam; alladhu neenga therndhedutha baikkileye thodaralaam. Mudivu ungaludaiyadhu.")
    return message


def display_reply(state, text):
    # Forecast details are spoken only; the app shows the booking dialogue.
    report = advisory(state)
    return text.replace(report, "", 1).strip()


def closing_message(lang):
    return localized(lang,"Thank you! Have a wonderful journey!","நன்றி! உங்கள் பயணம் இனிமையாக அமையட்டும்!","Nandri! Ungal payanam inimaiyaaga amaiyattum!")


def completion_message(lang,ride_id):
    digits=str(int(ride_id))
    return localized(lang,f'Your booking request has been saved. Booking number {digits}.',
        f'உங்கள் முன்பதிவு கோரிக்கை பதிவு செய்யப்பட்டது. முன்பதிவு எண் {digits}.',
        f'Unga munpadhivu korikkai padhivu seiyyappattadhu. Munpadhivu enn {digits}.')+'\n'+closing_message(lang)


def speech_segments(text):
    """Sentence clips stream immediately; decimals and short initials stay intact."""
    candidates=re.split(r"(?<=[.!?。])\s+|\n+",str(text).strip())
    parts=[];pending=""
    for part in candidates:
        if not part.strip():continue
        pending=(pending+" "+part).strip()
        if re.search(r"(?:^|\s)(?:[A-Za-z]|Dr|Mr|Ms|Mrs)\.$",pending):continue
        parts.append(pending);pending=""
    if pending:parts.append(pending)
    return parts[:15]+([" ".join(parts[15:])] if len(parts)>15 else [])


class GuardDB:
    def __init__(self, path):
        self.path = str(path)

    def connect(self):
        c = sqlite3.connect(self.path, timeout=20)
        c.row_factory = sqlite3.Row
        return c

    def setup(self):
        with self.connect() as c:
            c.executescript("""
                CREATE TABLE IF NOT EXISTS wx_calls(
                    id TEXT PRIMARY KEY, owner INTEGER NOT NULL, secret_hash TEXT NOT NULL,
                    expires REAL NOT NULL, epoch INTEGER NOT NULL DEFAULT 0, version INTEGER NOT NULL DEFAULT 0,
                    state TEXT NOT NULL DEFAULT '{}', summary_nonce TEXT, summary_hash TEXT,
                    summary_expires REAL, played INTEGER NOT NULL DEFAULT 0, ride_id INTEGER,
                    summary_kind TEXT NOT NULL DEFAULT 'final');
                CREATE TABLE IF NOT EXISTS wx_permits(
                    token_hash TEXT PRIMARY KEY, call_id TEXT NOT NULL, epoch INTEGER NOT NULL,
                    version INTEGER NOT NULL, snapshot TEXT NOT NULL, expires REAL NOT NULL, ride_id INTEGER);
            """)
            call_columns = {r[1] for r in c.execute('PRAGMA table_info(wx_calls)')}
            if 'summary_kind' not in call_columns:
                c.execute("ALTER TABLE wx_calls ADD COLUMN summary_kind TEXT NOT NULL DEFAULT 'final'")
            columns = {r[1] for r in c.execute("PRAGMA table_info(rides)")}
            if columns:
                for name, kind in (("passengers", "INTEGER"), ("pickup_time", "TEXT"),
                                   ("weather_json", "TEXT"), ("journey_version", "INTEGER"), ("voice_call_id", "TEXT")):
                    if name not in columns:
                        c.execute(f"ALTER TABLE rides ADD COLUMN {name} {kind}")

    def create(self, owner):
        sid, secret = secrets.token_hex(16), secrets.token_urlsafe(32)
        with self.connect() as c:
            c.execute("INSERT INTO wx_calls(id,owner,secret_hash,expires) VALUES(?,?,?,?)",
                      (sid, owner, hashlib.sha256(secret.encode()).hexdigest(), time.time() + 3600))
        return {"call_id": sid, "bind_token": secret}

    def bind(self, sid, secret):
        with self.connect() as c:
            r = c.execute("SELECT * FROM wx_calls WHERE id=?", (sid,)).fetchone()
        if not r or r["expires"] <= time.time() or not hmac.compare_digest(r["secret_hash"], hashlib.sha256(str(secret).encode()).hexdigest()):
            raise GuardError("Invalid or expired call session")
        if r["ride_id"]:
            raise GuardError("This call already created a booking")
        return dict(r)

    def row(self, sid):
        with self.connect() as c:
            row = c.execute("SELECT * FROM wx_calls WHERE id=?", (sid,)).fetchone()
        if not row:
            raise GuardError("Unknown call")
        return dict(row)

    def begin(self, sid, interrupt=False):
        with self.connect() as c:
            c.execute("BEGIN IMMEDIATE")
            r = c.execute("SELECT * FROM wx_calls WHERE id=?", (sid,)).fetchone()
            if not r or r["expires"] <= time.time():
                raise GuardError("Call expired")
            if r["ride_id"]:
                raise GuardError("Booking was already saved; use cancellation to change it")
            epoch = r["epoch"] + 1
            c.execute("UPDATE wx_calls SET epoch=? WHERE id=?", (epoch, sid))
            if interrupt:
                c.execute("UPDATE wx_calls SET summary_nonce=NULL,summary_hash=NULL,played=0 WHERE id=?", (sid,))
            return epoch

    def current(self, sid, epoch):
        r = self.row(sid)
        return r["epoch"] == epoch and r["expires"] > time.time() and not r["ride_id"]

    def stash(self, sid, epoch, state):
        with self.connect() as c:
            c.execute("BEGIN IMMEDIATE")
            r = c.execute("SELECT * FROM wx_calls WHERE id=?", (sid,)).fetchone()
            if not r or r["epoch"] != epoch or r["ride_id"]:
                raise GuardError("Obsolete turn")
            old = json.loads(r["state"])
            changed = any(old.get(k) != state.get(k) for k in CORE_FIELDS)
            version = r["version"] + int(changed)
            c.execute("UPDATE wx_calls SET state=?,version=? WHERE id=?", (canonical(state), version, sid))
            if changed:
                c.execute("UPDATE wx_calls SET summary_nonce=NULL,summary_hash=NULL,played=0 WHERE id=?", (sid,))
            return version

    def summary(self, sid, epoch, state):
        nonce, sh = secrets.token_hex(16), digest(snapshot(state))
        kind = 'weather' if state.get('stage') == 'weather_advisory' else 'final'
        with self.connect() as c:
            cur = c.execute("UPDATE wx_calls SET summary_nonce=?,summary_hash=?,summary_expires=?,played=0,summary_kind=? WHERE id=? AND epoch=? AND ride_id IS NULL",
                            (nonce, sh, time.time() + 600, kind, sid, epoch))
            if cur.rowcount != 1:
                raise GuardError("Obsolete summary")
        return nonce

    def played(self, sid, epoch, nonce, explicit_read=False):
        with self.connect() as c:
            extra = " AND summary_kind='final'" if explicit_read else ''
            return c.execute("UPDATE wx_calls SET played=1 WHERE id=? AND epoch=? AND summary_nonce=? AND summary_expires>? AND ride_id IS NULL"+extra,
                             (sid, epoch, nonce, time.time())).rowcount == 1

    def accept_weather(self, sid, epoch, nonce, state):
        """The first Yes records consent only. It cannot issue a booking permit."""
        with self.connect() as c:
            c.execute('BEGIN IMMEDIATE')
            r=c.execute('SELECT * FROM wx_calls WHERE id=?',(sid,)).fetchone()
            if (not r or r['epoch']!=epoch or r['ride_id'] or r['summary_kind']!='weather'
                    or not r['played'] or r['summary_nonce']!=nonce or r['summary_hash']!=digest(snapshot(state))
                    or r['summary_expires']<=time.time() or r['expires']<=time.time()):
                raise GuardError('Please hear the weather advisory and confirm again')
            state['_wx_weather_consent']={'scope':weather_consent_scope(state),'accepted_at':time.time()}
            state['stage']='fare'
            c.execute('UPDATE wx_calls SET state=?,summary_nonce=NULL,summary_hash=NULL,played=0 WHERE id=?',
                      (canonical(state),sid))
        return state

    def permit(self, sid, epoch, nonce, state):
        token = secrets.token_urlsafe(32)
        with self.connect() as c:
            c.execute("BEGIN IMMEDIATE")
            r = c.execute("SELECT * FROM wx_calls WHERE id=?", (sid,)).fetchone()
            if (not r or r["epoch"] != epoch or r["ride_id"] or not r["played"] or r['summary_kind']!='final'
                    or r["summary_nonce"] != nonce or r["summary_hash"] != digest(snapshot(state))
                    or r["summary_expires"] <= time.time()):
                raise GuardError("Please hear or read the updated summary and confirm again")
            if state.get('_wx_mode') != 'manual' and not weather_consented(state):
                raise GuardError('Weather consent is required before final booking confirmation')
            c.execute("INSERT INTO wx_permits VALUES(?,?,?,?,?,?,NULL)",
                      (hashlib.sha256(token.encode()).hexdigest(), sid, epoch, r["version"], canonical(snapshot(state)), time.time() + 90))
        return token

    def commit(self, c, owner, token, ensure_code):
        """One SQLite transaction validates the revision and inserts at most one ride."""
        c.execute("BEGIN IMMEDIATE")
        p = c.execute("SELECT * FROM wx_permits WHERE token_hash=?", (hashlib.sha256(str(token).encode()).hexdigest(),)).fetchone()
        if not p:
            raise GuardError("Unknown confirmation")
        call = c.execute("SELECT * FROM wx_calls WHERE id=?", (p["call_id"],)).fetchone()
        if not call or call["owner"] != owner:
            raise PermissionError("This confirmation belongs to another account")
        if p["ride_id"]:
            return {"ok": True, "id": p["ride_id"], "reused": True}
        if (call["ride_id"] or p["expires"] <= time.time() or call["expires"] <= time.time()
                or call["epoch"] != p["epoch"] or call["version"] != p["version"]):
            raise GuardError("Journey changed or confirmation expired; confirm the latest summary")
        s = json.loads(p["snapshot"])
        if s.get('_wx_mode') != 'manual' and not weather_consented(s):
            raise GuardError('Weather consent is required before booking')
        vehicle, count = s["vehicle"], int(s["count"])
        if vehicle not in CAPACITY or not 1 <= count <= CAPACITY[vehicle]:
            raise GuardError("Invalid vehicle capacity")
        coords = [float(s[k][axis]) for k in ("pickup_geo", "destination_geo") for axis in ("lat", "lng")]
        if not all(math.isfinite(v) and abs(v) <= (90 if i % 2 == 0 else 180) for i, v in enumerate(coords)):
            raise GuardError("Invalid coordinates")
        pickup = datetime.fromisoformat(s["pickup_at"])
        if pickup.tzinfo is None or pickup.timestamp() <= time.time():
            raise GuardError("Pickup time has passed; choose a new time")
        fare = float(s["fare"]["fare"])
        if not math.isfinite(fare) or fare < 0:
            raise GuardError("Invalid route fare")
        status = "scheduled" if pickup.timestamp() > time.time() + 20 * 60 else "requested"
        now = time.time()
        cur = c.execute("""INSERT INTO rides(customer_id,pickup,destination,plat,plng,dlat,dlng,vehicle,fare,status,
                           source,created,updated,passengers,pickup_time,weather_json,journey_version,voice_call_id)
                           VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
                        (owner, s["pickup"], s["destination"], *coords, vehicle, round(fare), status,
                         "weather_web" if s.get("_wx_mode")=="manual" else "weather_voice", now, now, count, s["pickup_at"], canonical(s["weather"]), p["version"], p["call_id"]))
        rid = cur.lastrowid
        ensure_code(c, rid)
        c.execute("UPDATE wx_calls SET ride_id=? WHERE id=?", (rid, p["call_id"]))
        c.execute("UPDATE wx_permits SET ride_id=? WHERE token_hash=?", (rid, p["token_hash"]))
        return {"ok": True, "id": rid, "status": status, "reused": False}

    def release_due(self):
        # No late automatic dispatch after a notebook has been offline.
        now = datetime.now(TZ)
        with self.connect() as c:
            for row in c.execute("SELECT id,pickup_time FROM rides WHERE status='scheduled' AND source IN ('weather_voice','weather_web')").fetchall():
                when = datetime.fromisoformat(row["pickup_time"])
                if now.timestamp() <= when.timestamp() <= now.timestamp() + 20 * 60:
                    c.execute("UPDATE rides SET status='requested',updated=? WHERE id=? AND status='scheduled'", (time.time(), row["id"]))


def manual_state(data,weather,now=None):
    """Validate exactly the customer's choices; weather never changes them."""
    now=(now or datetime.now(TZ)).astimezone(TZ)
    lang=data.get("language","English")
    if lang not in ("English","Tamil","Tanglish"):raise GuardError("invalid_language")
    vehicle=data.get("vehicle")
    if vehicle not in CAPACITY:raise GuardError("invalid_vehicle")
    raw_count=str(data.get("count",1))
    if not re.fullmatch(r"[1-6]",raw_count) or int(raw_count)>CAPACITY[vehicle]:raise GuardError("invalid_capacity")
    try:
        pickup=datetime.fromisoformat(str(data.get("pickup_at","")))
        pickup=pickup.replace(tzinfo=TZ) if pickup.tzinfo is None else pickup.astimezone(TZ)
    except ValueError as exc:raise GuardError("invalid_datetime") from exc
    if pickup<=now:raise GuardError("past_datetime")
    places={}
    for key,lat,lng in (("pickup","plat","plng"),("destination","dlat","dlng")):
        name=str(data.get(key,"")).strip()
        if not name or len(name)>180:raise GuardError("missing_locations")
        try:a,b=float(data[lat]),float(data[lng])
        except (KeyError,ValueError,TypeError) as exc:raise GuardError("missing_locations") from exc
        if not (math.isfinite(a) and math.isfinite(b) and -90<=a<=90 and -180<=b<=180):raise GuardError("missing_locations")
        places[key]=name;places[key+"_geo"]={"lat":a,"lng":b}
    try:km,fare=float(data.get("km",0)),float(data["fare"])
    except (KeyError,ValueError,TypeError) as exc:raise GuardError("invalid_fare") from exc
    if not (math.isfinite(km) and math.isfinite(fare) and km>=0 and fare>=0):raise GuardError("invalid_fare")
    state=dict(lang=lang,_wx_mode="manual",mode="book",stage="confirm",count=int(raw_count),vehicle=vehicle,
        when=pickup.isoformat(),pickup_at=pickup.isoformat(),fare={"km":round(km,1),"fare":round(fare)},**places)
    state["weather"]=weather.forecast([state["pickup_geo"],state["destination_geo"]],pickup,now=now)
    return state


def manual_error(lang,code):
    messages={
        "invalid_datetime":("Please select your booking date and time.","பயணத்தின் தேதி மற்றும் நேரத்தைத் தேர்ந்தெடுக்கவும்.","Payanathin thedhi matrum nerathai therndhedukkavum."),
        "past_datetime":("That pickup time has passed. Please choose a later time.","அந்த நேரம் கடந்துவிட்டது. அதற்குப் பிந்தைய நேரத்தைத் தேர்ந்தெடுக்கவும்.","Andha neram kadandhuvittadhu. Adharku pindhaiya nerathai therndhedukkavum."),
        "invalid_capacity":("Please choose a passenger count and vehicle with sufficient seats.","பயணிகளின் எண்ணிக்கைக்கு ஏற்ற இருக்கைகள் உள்ள வாகனத்தைத் தேர்ந்தெடுக்கவும்.","Payanigalin ennikkai-kku etra irukkaigal ulla vandiyai therndhedukkavum."),
        "missing_locations":("Please choose the pickup and destination from the map or address suggestions.","புறப்படும் இடத்தையும் சேரும் இடத்தையும் வரைபடம் அல்லது முகவரி பரிந்துரைகளிலிருந்து தேர்ந்தெடுக்கவும்.","Purappadum idathaiyum serum idathaiyum varaipadam alladhu mugavari parindhuraigalilirundhu therndhedukkavum."),
        "expired_review":("The review has expired. Please check the weather and booking details again.","சுருக்கத்தின் காலம் முடிந்துவிட்டது. வானிலை மற்றும் முன்பதிவு விவரங்களை மீண்டும் சரிபார்க்கவும்.","Surukkathin kaalam mudindhuvittadhu. Vaanilai matrum munpadhivu vivarangalai meendum saripaarunga.")}
    return localized(lang,*messages.get(code,("The request could not be completed. Please check the details and try again.","கோரிக்கையை நிறைவு செய்ய முடியவில்லை. விவரங்களைச் சரிபார்த்து மீண்டும் முயற்சிக்கவும்.","Korikkaiyai niraivu seiyya mudiyavillai. Vivarangalai saripaarthu meendum muyarchikkavum.")))


def install_backend(module,weather=None):
    if getattr(module.Handler, "_wx_version", None) == VERSION:
        if weather is not None:module._wx_weather_client=weather
        return
    db = GuardDB(module.DB)
    db.setup()
    old_thread=getattr(module,"_wx_dispatch_thread",None)
    if old_thread is not None:
        old_stop=getattr(module,"_wx_stop_event",None)
        if old_stop is not None:old_stop.set()
        old_thread.join(timeout=25)
        if old_thread.is_alive():raise RuntimeError("The previous scheduling worker is still stopping; rerun activation shortly.")
        del module._wx_dispatch_thread
    weather=weather or getattr(module,"_wx_weather_client",None) or WeatherClient(rain_threshold=float(module.os.environ.get("MOVIO_RAIN_THRESHOLD",60)),
        heat_c=float(module.os.environ.get("MOVIO_HEAT_C",38)),feels_like_c=float(module.os.environ.get("MOVIO_FEELS_LIKE_C",40)))
    module._wx_weather_client=weather
    original_get = getattr(module.Handler, "_wx_original_get", module.Handler.do_GET)
    original_post = getattr(module.Handler, "_wx_original_post", module.Handler.do_POST)

    def guarded_post(h):
        path = urlparse(h.path).path
        if not path.startswith("/api/weather-booking/"):
            return original_post(h)
        data={};lang="English"
        try:
            origin = h.headers.get("Origin", "").rstrip("/")
            if origin and origin != module.PUBLIC:
                raise PermissionError("Origin mismatch")
            data = json.loads(h.payload(12000) or b"{}")
            lang=data.get("language","English") if isinstance(data,dict) else "English"
            if not isinstance(data,dict):raise GuardError("Invalid request")
            with module.LOCK,module.conn() as c:
                user=dict(h.user(c))
                if user["role"]!="customer":raise PermissionError("Customer account required")
            if path.endswith("/manual-preview"):
                # External forecast requests never hold the website's DB lock.
                s=manual_state(data,module._wx_weather_client)
                binding=db.create(user["id"]);e=db.begin(binding["call_id"])
                version=db.stash(binding["call_id"],e,s);nonce=db.summary(binding["call_id"],e,s)
                return h.json(200,{"ok":True,**binding,"epoch":e,"version":version,"summary_nonce":nonce,
                    "advisory":advisory(s),"summary":display_reply(s,summary_text(None,s)),"language":s["lang"],
                    "weather_status":s["weather"]["status"],"pickup_at":s["pickup_at"],"vehicle":s["vehicle"],"fare":s["fare"]["fare"]})
            with module.LOCK, module.conn() as c:
                user = h.user(c)
                if user["role"] != "customer":
                    raise PermissionError("Customer account required")
                if path.endswith("/manual-confirm"):
                    row=db.row(data.get("call_id"))
                    if row["owner"]!=user["id"] or not hmac.compare_digest(row["secret_hash"],hashlib.sha256(str(data.get("bind_token","")).encode()).hexdigest()):
                        raise PermissionError("This review belongs to another account")
                    s=json.loads(row["state"]);lang=s["lang"]
                    if s.get("_wx_mode")!="manual":raise GuardError("expired_review")
                    if row["ride_id"]:
                        result={"ok":True,"id":row["ride_id"],"reused":True}
                    else:
                        if not db.played(row["id"],row["epoch"],data.get("summary_nonce")):raise GuardError("expired_review")
                        token=db.permit(row["id"],row["epoch"],data.get("summary_nonce"),s)
                        result=db.commit(c,user["id"],token,module.ensure_pickup_code);c.commit()
                    return h.json(201,{**result,"language":lang,"message":completion_message(lang,result["id"]),"closing":closing_message(lang)})
                if path.endswith("/manual-discard"):
                    row=db.row(data.get("call_id"))
                    if row["owner"]!=user["id"] or not hmac.compare_digest(row["secret_hash"],hashlib.sha256(str(data.get("bind_token","")).encode()).hexdigest()):
                        raise PermissionError("This review belongs to another account")
                    if row["ride_id"]:return h.json(200,{"ok":True,"already_saved":True,"id":row["ride_id"]})
                    db.begin(row["id"],interrupt=True)
                    return h.json(200,{"ok":True})
                if path.endswith("/session"):
                    return h.json(200, db.create(user["id"]))
                if path.endswith("/commit"):
                    result = db.commit(c, user["id"], data.get("token", ""), module.ensure_pickup_code)
                    c.commit()
                    return h.json(201, result)
                if path.endswith("/cancel"):
                    rid = int(data["ride_id"])
                    cur = c.execute("UPDATE rides SET status='cancelled',updated=? WHERE id=? AND customer_id=? AND status='scheduled'",
                                    (time.time(), rid, user["id"]))
                    if cur.rowcount != 1:
                        raise GuardError("Scheduled booking not found; refresh your bookings")
                    c.commit()
                    return h.json(200, {"ok": True, "id": rid})
            return h.json(404, {"error": "Unknown weather-booking endpoint"})
        except PermissionError as exc:
            return h.json(401 if str(exc) == "Log in first" else 403, {"error": str(exc)})
        except (ValueError, KeyError, TypeError) as exc:
            return h.json(409, {"error":manual_error(lang,str(exc)) if "/manual-" in path else str(exc),"code":str(exc)})
        except Exception:
            return h.json(500, {"error": "Weather booking request failed"})

    def guarded_get(h):
        path = urlparse(h.path).path
        if path == "/api/weather-booking/health":
            return h.json(200, {"ok": True, "version": VERSION, "weather": "Open-Meteo", "guard": "sqlite_atomic_revision"})
        if path == "/api/rides":
            db.release_due()
        return original_get(h)

    module.Handler.do_POST, module.Handler.do_GET = guarded_post, guarded_get
    module.Handler._wx_original_get, module.Handler._wx_original_post = original_get, original_post
    module.Handler._wx_installed = True
    module.Handler._wx_version = VERSION
    if not hasattr(module, "_wx_dispatch_thread"):
        stop_event=threading.Event()
        module._wx_stop_event = stop_event
        def dispatch():
            while not stop_event.is_set():
                try:
                    db.release_due()
                except Exception:
                    pass
                stop_event.wait(30)
        module._wx_dispatch_thread = threading.Thread(target=dispatch, daemon=True, name="weather-scheduled-release")
        module._wx_dispatch_thread.start()


def summary_text(bridge, s):
    lang=s['lang'];date=spoken_datetime(s['pickup_at'],lang)
    count=numeric_text(s['count'],0);km=numeric_text(s['fare']['km']);fare=numeric_text(s['fare']['fare'],0,grouped=True)
    return localized(lang,
        f"Your booking: {count} {'passenger' if int(s['count'])==1 else 'passengers'}, {s['vehicle']}, from {s['pickup']} to {s['destination']}, on {date}. "
        f'The road distance is about {km} kilometres and the estimated fare is ₹{fare}. Shall I confirm this booking?',
        f"உங்கள் முன்பதிவு விவரம்: {count} பேர், {VEHICLES_TA.get(s['vehicle'],s['vehicle'])}, {s['pickup']} இலிருந்து {s['destination']} வரை, {date}. "
        f'சாலை தூரம் சுமார் {km} கிலோமீட்டர். மதிப்பிடப்பட்ட கட்டணம் ₹{fare}. இந்த முன்பதிவை உறுதி செய்யலாமா?',
        f"Unga munpadhivu vivaram: {count} peru, {VEHICLES_ROMAN[s['vehicle']]}, {s['pickup']} ilirundhu {s['destination']} varai, {date}. "
        f'Saalai dhooram sumaar {km} kilo meettar. Madhippidappatta kattanam ₹{fare}. Indha munpadhivai urudhi seiyyalaama?')


def clean_spoken_place(bridge,value):
    text=str(value or '').strip(' .,!?:;–—-')
    text=re.sub(r'(?i)^(?:i\s+(?:want|need)\s+to\s+(?:go|travel)\s+|i\s+am\s+|naan\s+|nan\s+|enakku\s+)','',text)
    text=re.sub(r'^(?:நான்|எனக்கு)\s*','',text)
    text=re.split(r'(?<!\b[A-Za-z])\.\s+|[!?;\n]',text,maxsplit=1)[0]
    text=re.split(r'(?i)\s+(?:செல்ல|போக|போகணும்|போகனும்|வேண்டும்|நாங்கள்|எனக்கு|இன்று|நாளை|poganum|ponum|venum|we\s+are|for\s+\d+|at\s+\d|today|tomorrow)\b|\s+(?:செல்ல|போக|நாங்கள்|எனக்கு|இன்று|நாளை)',text,maxsplit=1)[0]
    text=re.split(r',\s*(?=(?:one|two|three|four|five|six|bike|mini|sedan|auto|suv|today|tomorrow)\b)',text,flags=re.I,maxsplit=1)[0]
    canonicalizer=getattr(bridge,'canonical_place',None)
    canonical_value=canonicalizer(text) if callable(canonicalizer) else text
    if canonical_value!=text:return str(canonical_value)[:120]
    text=re.sub(r'[-\s]*(?:இல்\s*இருந்து|இலிருந்து|லிருந்து|இருந்து|க்கு|இல்|\b(?:la\s+irundhu|irundhu|ku)\b)$','',text,flags=re.I)
    return text.strip(' .,!?:;–—-')[:120]


def collect_details(bridge,s,text):
    """Structured updates with explicit evidence; confidence values are rule scores, not calibrated probabilities."""
    stage=s.get('stage');raw=unicodedata.normalize('NFKC',str(text)).strip()
    words=number_lexicon(20);token=r'\d{1,2}|'+'|'.join(re.escape(k) for k in sorted(words,key=len,reverse=True))
    count_match=re.search(r'(?<![\w\u0b80-\u0bff])('+token+r')\s*(?:பேர்|நபர்கள்|பயணிகள்|people\b|persons?\b|passengers?\b|peru\b|per\b)',raw,re.I)
    if not count_match and stage=='count':count_match=re.fullmatch(r'\s*('+token+r')\s*[.!]?\s*',raw,re.I)
    updates={};evidence={};ambiguities=s.setdefault('_wx_slot_conflicts',{})
    def accept(slot,value,span,source='rule',confidence=1.0):
        updates[slot]=value;evidence[slot]={'text':span,'source':source,'confidence':confidence}
        ambiguities.pop(slot,None)
    if count_match:
        value=count_match[1].casefold();accept('count',int(value) if value.isdigit() else words[value],count_match[0])
    patterns=(('SUV',r'\bs\s*u\s*v\b|எஸ்\s*ய[ுூ]?\s*வி|எஸ்யூவி'),('Sedan',r'\bsedan\b|செடான்'),
              ('Mini',r'\bmini(?:\s+car)?\b|மினி(?:\s*கார்)?'),('Auto',r'\bauto\b|ஆட்டோ'),('Bike',r'\bbike\b|பைக்'))
    hits=sorted((m.start(),m.end(),name,m[0]) for name,pattern in patterns for m in re.finditer(pattern,raw,re.I))
    selected=set(h[2] for h in hits)
    correction=bool(re.search(r'\b(?:no|change|instead|replace|not|maathu|mathunga|badila|illa|vendam)\b|மாற்ற|வேண்டாம்|இல்லை',raw,re.I))
    if len(selected)==1:accept('vehicle',hits[-1][2],hits[-1][3])
    elif hits and correction:accept('vehicle',hits[-1][2],raw[hits[0][0]:hits[-1][1]])
    elif len(selected)>1:ambiguities['vehicle']='multiple vehicles stated without an explicit correction'
    places={}
    route=re.search(r'(?i)\bfrom\s+(.+?)\s+to\s+(.+)',raw)
    if not route:route=re.match(r'(.+?)[-\s]*(?:இல்\s*இருந்து|இலிருந்து|லிருந்து|இருந்து)\s+(.+)',raw)
    if not route:route=re.match(r'(?i)(.+?)\s+(?:la\s+)?irundhu\s+(.+)',raw)
    if route:places={'pickup':route[1],'destination':route[2]}
    else:
        for slot,pattern in (('pickup',r'(?i)\b(?:pickup(?:\s+is|\s+from)?|pick me up (?:from|at))\s+(.+)'),
                             ('destination',r'(?i)\b(?:destination(?:\s+is)?|drop(?:\s+is|\s+at)?|take me to)\s+(.+)')):
            match=re.search(pattern,raw)
            if match:places[slot]=match[1]
    is_time=bool(re.search(TODAY_RE+'|'+TOMORROW_RE+'|'+AM_RE+'|'+PM_RE+r'|\d[:.]\d|மணி|\d{4}-\d{2}-\d{2}|october|அக்டோபர்',raw,re.I))
    normalized=reply_normalized(raw)
    if not places and stage in ('pickup','destination') and not selected and not count_match and not is_time and normalized not in POSITIVE|NEGATIVE:
        places[stage]=raw
    for slot,span in places.items():
        value=clean_spoken_place(bridge,span)
        if value:accept(slot,value,span)
    # Qwen proposes spans only. Every accepted value is still validated against this utterance.
    extractor=getattr(bridge,'_wx_context_extract',None)
    unresolved=[k for k in ('count','vehicle','pickup','destination') if not updates.get(k) and not s.get(k)]
    if callable(extractor) and unresolved and len(raw.split())>=5 and normalized not in POSITIVE|NEGATIVE:
        try:extracted=extractor(raw,s) or {}
        except Exception:extracted={}
        for slot in ('pickup','destination'):
            value=extracted.get(slot)
            if slot not in updates and isinstance(value,str) and reply_normalized(value) and reply_normalized(value) in normalized:
                cleaned=clean_spoken_place(bridge,value)
                if cleaned:accept(slot,cleaned,value,'qwen_validated_span',0.85)
        value=extracted.get('passengers_text')
        if 'count' not in updates and isinstance(value,str) and reply_normalized(value) in normalized:
            parser=getattr(bridge,'number',None);count=parser(value) if callable(parser) else words.get(value.casefold())
            if count and 1<=int(count)<=20:accept('count',int(count),value,'qwen_validated_span',0.85)
    changed=[]
    for slot,value in updates.items():
        if s.get(slot)!=value:
            s[slot]=value;changed.append(slot)
            if slot in ('pickup','destination'):s[slot+'_geo']=None
    s.setdefault('_wx_slot_evidence',{}).update(evidence)
    if raw:s['_wx_turns']=(s.get('_wx_turns',[])+[{'input':raw,'updates':updates}])[-30:]
    return {'updates':updates,'changed':changed,'evidence':evidence,'ambiguities':copy.deepcopy(ambiguities)}


def time_prompt(lang, reason):
    messages = {
        "clarify_date": ("What day should I book: today, tomorrow, or another date?", "எந்த தேதிக்கு முன்பதிவு செய்ய வேண்டும்? இன்று, நாளை அல்லது வேறு தேதியைச் சொல்லுங்கள்.", "Endha date-ku book pannanum? Innaiku, naalaiku illa vera date sollunga."),
        "clarify_clock": ("What pickup time would you like?", "எத்தனை மணிக்கு வாகனம் வேண்டும்?", "Ethana manikku vehicle venum?"),
        "clarify_period": ("Is that time in the morning, afternoon, evening or at night?", "அந்த நேரம் காலையா, மதியமா, மாலையா அல்லது இரவா?", "Andha time kaalaiya, madhiyama, maalaiya illa irava?"),
        "past_datetime": ("That pickup time has already passed. Please tell me a later time, or a different date.", "நீங்கள் சொன்ன நேரம் ஏற்கனவே கடந்துவிட்டது. அதற்குப் பிந்தைய நேரத்தை அல்லது வேறு தேதியைச் சொல்லுங்கள்.", "Neenga sonna time already kadandhuduchu. Later time illa vera date sollunga."),
        "invalid_datetime": ("I could not read that date or time. Please say it again.", "அந்த தேதி அல்லது நேரம் புரியவில்லை. மீண்டும் சொல்லுங்கள்.", "Andha date illa time puriyala. Thirumba sollunga.")}
    return localized(lang,*messages.get(reason,messages["invalid_datetime"]))


def prepare_turn(bridge,weather,state,text,now=None):
    """Never books. Forecast consent precedes a separate route/fare and final review."""
    now=(now or datetime.now(TZ)).astimezone(TZ)
    before={k:copy.deepcopy(state.get(k)) for k in CORE_FIELDS}
    if str(text).strip():collect_details(bridge,state,text)
    try:pickup,time_error=update_time_memory(state,text,now)
    except GuardError as exc:
        pickup,time_error=None,str(exc);state['_wx_time_error']=time_error
    changed=any(before[k]!=state.get(k) for k in CORE_FIELDS)
    if changed:state['fare']=None
    if state.get('_wx_slot_conflicts',{}).get('vehicle'):
        state['stage']='vehicle'
        return state,localized(state['lang'],'Which one vehicle would you like?', 'எந்த ஒரு வாகனம் வேண்டும்?', 'Endha oru vandi venum?')
    for slot in ('count','vehicle','pickup','destination'):
        if not state.get(slot):state['stage']=slot;return state,bridge.say(state,slot)
    if state['vehicle'] not in CAPACITY or not 1<=int(state['count'])<=6 or int(state['count'])>CAPACITY[state['vehicle']]:
        state['stage']='vehicle' if 1<=int(state['count'])<=6 else 'count'
        choices=[v for v,c in CAPACITY.items() if c>=int(state['count'])]
        return state,localized(state['lang'],
            'Please choose a vehicle with enough seats: '+', '.join(choices)+'.' if choices else 'Please give a passenger count from 1 to 6 for one vehicle.',
            'பயணிகளுக்கு ஏற்ற வாகனத்தைத் தேர்ந்தெடுக்கவும்: '+', '.join(VEHICLES_TA[v] for v in choices)+'.' if choices else 'ஒரு வாகனத்திற்கு 1 முதல் 6 வரை பயணிகள் எண்ணிக்கையைச் சொல்லுங்கள்.',
            'Payanigalukku etra vandiyai therndhedukkavum: '+', '.join(VEHICLES_ROMAN[v] for v in choices)+'.' if choices else 'Oru vandikku 1 mudhal 6 varai payanigal ennikkaiyai sollunga.')
    if time_error:
        state['stage']='time';state.pop('pickup_at',None);state.pop('_wx_weather_consent',None)
        return state,time_prompt(state['lang'],time_error)
    state['pickup_at']=pickup.isoformat()
    for slot in ('pickup','destination'):
        if not valid_geo(state.get(slot+'_geo')):
            try:geo=bridge.geocode(state[slot])
            except Exception:geo=None
            if valid_geo(geo):state[slot+'_geo']=geo
    scope=digest({k:state.get(k) for k in ('pickup','destination','pickup_at','pickup_geo','destination_geo')})
    expired_weather=bool((state.get('weather') or {}).get('fetched_at') and time.time()-state['weather']['fetched_at']>getattr(weather,'ttl',900))
    if state.get('_wx_forecast_scope')!=scope or not state.get('weather') or expired_weather:
        state['weather']=weather.forecast([state['pickup_geo'],state['destination_geo']],pickup,now=now) if all(valid_geo(state.get(k+'_geo')) for k in ('pickup','destination')) else {'status':'unavailable','source':'Open-Meteo','reason':'location_coordinates_unavailable'}
        state['_wx_forecast_scope']=scope
    if not weather_consented(state):
        state.pop('_wx_weather_consent',None);state['stage']='weather_advisory'
        return state,advisory(state)+'\n'+weather_question(state['lang'])
    if not state.get('fare') or not all(valid_geo(state.get(k+'_geo')) for k in ('pickup','destination')):
        route_state=copy.deepcopy(state)
        try:fare=bridge.road_fare(route_state)
        except Exception:fare=None
        try:valid=bool(fare and all(math.isfinite(float(fare[k])) and float(fare[k])>=0 for k in ('km','fare')))
        except (KeyError,ValueError,TypeError):valid=False
        if valid:
            for slot in ('pickup','destination'):
                if valid_geo(route_state.get(slot+'_geo')):state[slot+'_geo']=route_state[slot+'_geo']
        if not valid or not all(valid_geo(state.get(k+'_geo')) for k in ('pickup','destination')):
            state['stage']='route_unavailable';state['fare']=None
            return state,localized(state['lang'],
                'Distance or fare estimation is temporarily unavailable. Your booking details are retained. Say retry, change a detail, or cancel.',
                'தூரம் அல்லது கட்டண மதிப்பீடு தற்காலிகமாகக் கிடைக்கவில்லை. உங்கள் பயண விவரங்கள் உள்ளன. மீண்டும் முயற்சிக்கச் சொல்லுங்கள், விவரங்களை மாற்றுங்கள் அல்லது ரத்து செய்யுங்கள்.',
                'Dhooram alladhu kattanam madhippeedu tharkaaligamaaga kidaikkavillai. Unga payana vivarangal irukku. Meendum muyarchikka sollunga, vivarangalai maathunga alladhu raththu seiyyunga.')
        state['fare']=fare
        if not weather_consented(state):
            return prepare_turn(bridge,weather,state,'',now=now)
    state['stage']='confirm'
    return state,summary_text(bridge,state)


def install_bridge(bridge, db_path, weather=None, _route_wrapper=None):
    """Replace only book-mode sockets; the original cancellation socket is retained."""
    if _route_wrapper is None:
        from starlette.routing import websocket_session
        _route_wrapper = websocket_session
    weather = weather or WeatherClient()
    install_numeric_normalizer(bridge)
    db = GuardDB(db_path)
    db.setup()
    if not hasattr(bridge, "_wx_original_socket"):
        bridge._wx_original_socket = next(r.endpoint for r in bridge.app.routes if getattr(r, "path", "") == "/browser-call")

    async def endpoint(ws):
        if ws.query_params.get("mode", "book") == "cancel":
            return await bridge._wx_original_socket(ws)
        if ws.headers.get("origin", "").rstrip("/") != bridge.PUBLIC_URL:
            await ws.close(code=1008)
            return
        lang = ws.query_params.get("language", "Tamil")
        if lang not in bridge.MESSAGES:
            await ws.close(code=1008)
            return
        await ws.accept()
        tasks, sid, state, epoch = set(), None, None, 0
        pending_action = None
        language_token = bridge.CURRENT_LANGUAGE.set(lang)

        async def send(payload):
            await ws.send_json(payload)

        async def emit(text, e, nonce=None, done=False):
            if not db.current(sid, e) and not done:
                return
            pause_ms=max(0,min(1000,int(getattr(bridge,"_wx_pause_ms",220))))
            pieces=speech_segments(text) if pause_ms else [text]
            for i,piece in enumerate(pieces):
                if not done and not db.current(sid,e):return
                payload=await asyncio.to_thread(tts_reply,bridge,piece)
                if not done and not db.current(sid,e):return
                await send({"event":"reply" if i==0 else "speech_chunk","text":display_reply(state,text) if i==0 else "",
                    "payload":payload,"epoch":e,"stage":state["stage"],"summary_nonce":nonce,
                    "summary_kind":"weather" if state.get('stage')=='weather_advisory' else 'final',"done":done,
                    "streaming":True,"chunk_index":i,"chunk_count":len(pieces),"pause_after_ms":pause_ms if i<len(pieces)-1 else 0,
                    "version":db.row(sid)["version"]})

        async def process(msg, e, prior):
            nonlocal state, pending_action
            try:
                if msg["event"] == "text":
                    text = str(msg.get("text", "")).strip()[:800]
                else:
                    def transcribe():
                        import av
                        import numpy as np
                        import librosa
                        encoded = msg.get("payload", "")
                        if len(encoded) > 3_000_000:
                            raise GuardError("Audio exceeds limit")
                        data = base64.b64decode(encoded, validate=True)
                        frames = []
                        with av.open(io.BytesIO(data)) as container:
                            resampler = av.AudioResampler(format="fltp", layout="mono", rate=16000)
                            for frame in container.decode(audio=0):
                                for mono in resampler.resample(frame):
                                    frames.append(mono.to_ndarray().astype(np.float32).reshape(-1))
                            for mono in resampler.resample(None):
                                frames.append(mono.to_ndarray().astype(np.float32).reshape(-1))
                        if not frames:
                            raise GuardError("No audio received")
                        a = np.concatenate(frames)
                        if len(a) / 16000 > 28:
                            raise GuardError("Please speak for less than 28 seconds")
                        return bridge.stt_sarvam(a, lang)[0]
                    text = await asyncio.to_thread(transcribe)
                if not db.current(sid, e):
                    return
                await send({"event": "heard", "text": text, "epoch": e})
                positive = reply_normalized(text) in POSITIVE
                if prior.get('stage')=='weather_advisory' and positive:
                    try:
                        prior=db.accept_weather(sid,e,msg.get('summary_nonce'),prior)
                    except GuardError:
                        if not db.current(sid,e):return
                        state=prior;db.stash(sid,e,state)
                        nonce=db.summary(sid,e,state)
                        await emit(advisory(state)+'\n'+weather_question(lang),e,nonce)
                        return
                    state=copy.deepcopy(prior)
                    refreshed,answer=await asyncio.to_thread(prepare_turn,bridge,weather,prior,'')
                    if not db.current(sid,e):return
                    state=refreshed;db.stash(sid,e,state)
                    nonce=db.summary(sid,e,state) if state['stage'] in ('weather_advisory','confirm') else None
                    await emit(answer,e,nonce)
                    return
                if prior.get("stage") == "confirm" and positive:
                    try:
                        token = db.permit(sid, e, msg.get("summary_nonce"), prior)
                    except GuardError:
                        prior["stage"] = "fare"
                        refreshed, answer = await asyncio.to_thread(prepare_turn, bridge, weather, prior, '')
                        if not db.current(sid, e):
                            return
                        db.stash(sid, e, refreshed)
                        state = refreshed
                        nonce = db.summary(sid, e, state) if state["stage"] in ('weather_advisory','confirm') else None
                        await emit(answer, e, nonce)
                        return
                    if not db.current(sid, e):
                        return
                    pending_action = secrets.token_hex(16)
                    state["stage"] = "booking_pending"
                    await send({"event": "action", "type": "weather_book", "token": token,
                                "request_id": pending_action, "epoch": e})
                    return
                if reply_normalized(text) in {'cancel','cancel booking','ரத்து','ரத்து செய்','ரத்து செய்யுங்கள்','cancel pannunga','raththu seiyyunga'}:
                    state["stage"] = "done"
                    await emit(localized(lang,'The booking request was cancelled. No booking was created.',
                        'முன்பதிவு கோரிக்கை ரத்து செய்யப்பட்டது. முன்பதிவு எதுவும் உருவாக்கப்படவில்லை.',
                        'Munpadhivu korikkai raththu seiyyappattadhu. Munpadhivu edhuvum uruvaakkappadavillai.'),e,done=True)
                    return
                if prior.get('stage') in ('weather_advisory','confirm') and reply_normalized(text) in NEGATIVE:
                    state=copy.deepcopy(prior);state['stage']='change_request';state.pop('_wx_weather_consent',None)
                    db.stash(sid,e,state)
                    await emit(localized(lang,'What would you like to change? You can also say cancel.',
                        'எந்த விவரத்தை மாற்ற விரும்புகிறீர்கள்? ரத்து செய்யவும் சொல்லலாம்.',
                        'Endha vivarathai maatra virumbureenga? Raththu seiyyavum sollalaam.'),e)
                    return
                if not text:
                    raise GuardError("Please repeat your request")
                next_state, answer = await asyncio.to_thread(prepare_turn, bridge, weather, prior, text)
                if not db.current(sid, e):
                    return
                db.stash(sid, e, next_state)
                state = next_state
                nonce = db.summary(sid, e, state) if state["stage"] in ('weather_advisory','confirm') else None
                await emit(answer, e, nonce)
            except asyncio.CancelledError:
                raise
            except Exception as exc:
                if db.current(sid, e):
                    await send({"event": "error", "message": str(exc)[:180], "epoch": e})

        try:
            start = await asyncio.wait_for(ws.receive_json(), timeout=20)
            if start.get("event") != "start":
                raise GuardError("Start the call from the updated logged-in dashboard")
            bound=db.bind(start.get("call_id"), start.get("bind_token"))
            sid = start["call_id"]
            saved=json.loads(bound.get('state') or '{}')
            state = saved if saved else bridge.new_session(lang, "book")
            state.update(lang=lang,mode='book',_wx_mode='voice')
            epoch = db.begin(sid)
            db.stash(sid, epoch, state)
            await send({"event": "ready", "epoch": epoch})
            async def greet(e):
                try:
                    await emit(bridge.say(state, "hello"), e)
                except Exception:
                    await send({"event": "error", "message": "Speech generation failed. Check the original TTS service.", "epoch": e})
            greeting = asyncio.create_task(greet(epoch))
            tasks.add(greeting)
            greeting.add_done_callback(tasks.discard)
            while True:
                msg = await ws.receive_json()
                kind = msg.get("event")
                if kind == "playback_ended":
                    db.played(sid, msg.get("epoch"), msg.get("summary_nonce"))
                elif kind == "summary_read":
                    # Explicit accessible alternative when audio playback is unavailable.
                    # Weather is spoken-only, so a click cannot claim to have read it.
                    db.played(sid, msg.get("epoch"), msg.get("summary_nonce"),explicit_read=True)
                elif kind == "interrupt":
                    try:
                        epoch = db.begin(sid, interrupt=True)
                        # Preserve the phase and all slots until the correction is parsed.
                        db.stash(sid,epoch,state)
                        await send({"event": "interrupted", "epoch": epoch})
                    except GuardError as exc:
                        await send({"event": "error", "message": str(exc), "epoch": epoch})
                elif kind in ("audio", "text"):
                    if pending_action:
                        await send({"event":"error","message":"The booking is still being saved. Please wait for its result.","epoch":epoch})
                        continue
                    try:
                        epoch = db.begin(sid)
                    except GuardError as exc:
                        await send({"event": "error", "message": str(exc), "epoch": epoch})
                        continue
                    await send({"event": "processing", "epoch": epoch})
                    task = asyncio.create_task(process(msg, epoch, copy.deepcopy(state)))
                    tasks.add(task)
                    task.add_done_callback(tasks.discard)
                elif kind == "result" and msg.get("request_id") == pending_action:
                    row = db.row(sid)
                    if row["ride_id"]:
                        pending_action = None
                        state["stage"] = "done"
                        await emit(completion_message(lang,row["ride_id"]), row["epoch"], done=True)
                    else:
                        pending_action = None
                        # Verify storage before claiming success, including when
                        # the browser reports a timeout or an HTTP error.
                        refreshed,answer=await asyncio.to_thread(prepare_turn,bridge,weather,copy.deepcopy(state),'')
                        if not db.current(sid,epoch):continue
                        state=refreshed;db.stash(sid,epoch,state)
                        reason = str(msg.get("error") or "The booking was not saved. Please try confirming again.")[:180]
                        await send({"event":"booking_error","message":reason,"epoch":epoch})
                        nonce = db.summary(sid,epoch,state) if state["stage"] in ('weather_advisory','confirm') else None
                        prefix = localized(lang,"The booking was not saved. ","முன்பதிவு சேமிக்கப்படவில்லை. ","Booking save aagala. ")
                        await emit(prefix+answer,epoch,nonce)
        except Exception as exc:
            try:
                await send({"event": "error", "message": str(exc)[:180], "fatal": True})
            except Exception:
                pass
        finally:
            if sid:
                try:
                    db.begin(sid, interrupt=True)
                except GuardError:
                    pass
            for task in list(tasks):
                task.cancel()
            await asyncio.gather(*list(tasks), return_exceptions=True)
            bridge.CURRENT_LANGUAGE.reset(language_token)

    route = next(r for r in bridge.app.routes if getattr(r, "path", "") == "/browser-call")
    route.endpoint, route.app = endpoint, _route_wrapper(endpoint)
    bridge._wx_weather, bridge._wx_db, bridge._wx_endpoint = weather, db, endpoint
    return {"version": VERSION, "weather": "Open-Meteo", "guard": "installed", "interrupt": "browser control"}






# V04_DIALOGUE_HELPERS
TA_DAY_WORDS = (
    '', 'முதல்', 'இரண்டாம்', 'மூன்றாம்', 'நான்காம்', 'ஐந்தாம்', 'ஆறாம்', 'ஏழாம்', 'எட்டாம்',
    'ஒன்பதாம்', 'பத்தாம்', 'பதினொன்றாம்', 'பன்னிரண்டாம்', 'பதின்மூன்றாம்', 'பதினான்காம்',
    'பதினைந்தாம்', 'பதினாறாம்', 'பதினேழாம்', 'பதினெட்டாம்', 'பத்தொன்பதாம்', 'இருபதாம்',
    'இருபத்தொன்றாம்', 'இருபத்திரண்டாம்', 'இருபத்துமூன்றாம்', 'இருபத்துநான்காம்',
    'இருபத்தைந்தாம்', 'இருபத்தாறாம்', 'இருபத்தேழாம்', 'இருபத்தெட்டாம்',
    'இருபத்தொன்பதாம்', 'முப்பதாம்', 'முப்பத்தொன்றாம்')

def numeric_text(value, decimal_places=1, grouped=False):
    """Canonical numerical entities stay as digits at the TTS adapter boundary."""
    number = float(value)
    if not math.isfinite(number):
        raise GuardError('Invalid numerical value')
    text = f'{number:,.{decimal_places}f}' if grouped else f'{number:.{decimal_places}f}'
    return text.rstrip('0').rstrip('.') if '.' in text else text

def number_lexicon(limit=59):
    result = {str(n): n for n in range(limit + 1)}
    for n in range(limit + 1):
        for lang in ('English', 'Tamil', 'Tanglish'):
            result[number_words(n, lang).casefold()] = n
    result.update({'ஒரு':1, 'ஒன்னு':1,'ரெண்டு':2,'மூணு':3,'நாலு':4,'அஞ்சு':5, 'onnu':1, 'oru':1, 'rendu':2, 'moonru':3, 'moonu':3, 'naalu':4,
                   'naangu':4, 'anju':5, 'aaru':6, 'yelu':7, 'ezhu':7, 'ettu':8,
                   'onbadhu':9, 'pathu':10, 'pannirandu':12})
    return result

def weather_question(lang):
    return localized(lang,'Would you like to continue?', 'முன்பதிவைத் தொடர விரும்புகிறீர்களா?', 'Munpadhivai thodaralaama?')

def weather_consent_scope(state):
    return digest({k:state.get(k) for k in ('pickup','destination','pickup_at','pickup_geo','destination_geo','weather')})

def weather_consented(state):
    consent=state.get('_wx_weather_consent') or {}
    return consent.get('scope')==weather_consent_scope(state) and bool(consent.get('accepted_at'))

def valid_geo(value):
    try:return isinstance(value,dict) and math.isfinite(float(value['lat'])) and math.isfinite(float(value['lng'])) and abs(float(value['lat']))<=90 and abs(float(value['lng']))<=180
    except (KeyError,ValueError,TypeError):return False

def tts_reply(bridge,text):
    """Bypass old outer decimal/number expansion; retain the original DhVaani adapter and its internal normalizer."""
    adapter=getattr(bridge,'TTS',None)
    if not callable(getattr(adapter,'synthesize',None)):
        return bridge.speak(text)  # isolated fixtures, or an existing custom speech backend
    language=bridge.CURRENT_LANGUAGE.get()
    group='TAMIL' if language=='Tamil' else 'TANGLISH' if language=='Tanglish' else 'ENGLISH'
    lock=getattr(bridge,'LOCK',None)
    token=NUMERIC_LANGUAGE.set(language)
    try:
        if lock is None:data=adapter.synthesize(str(text),group)
        else:
            with lock:data=adapter.synthesize(str(text),group)
    finally:NUMERIC_LANGUAGE.reset(token)
    return base64.b64encode(data).decode()

def pronunciation_number(value,lang):
    if lang=='English':return number_words(value,'English')
    n=int(value)
    if n<0:return 'மைனஸ் '+pronunciation_number(-n,lang)
    if n<10000:return number_words(n,'Tamil')
    for scale,full,joined in ((10000000,'கோடி','கோடியே'),(100000,'லட்சம்','லட்சத்து'),(1000,'ஆயிரம்','ஆயிரத்து')):
        if n>=scale:return pronunciation_number(n//scale,lang)+' '+(full if not n%scale else joined+' '+pronunciation_number(n%scale,lang))
    return str(n)

def speech_numeric_normalize(original,text,lang):
    """Acoustic-only verbalization; raw adapter input, transcript and booking values retain digits."""
    text=str(text);protected=[]
    identifiers=[m.span(1) for m in re.finditer(r'(?:booking\s*(?:id|number)|முன்பதிவு\s*எண்|munpadhivu\s*enn|\botp\b|ஓடிபி)\s*[:#-]?\s*(\d+)',text,re.I)]
    def word(n):return pronunciation_number(int(n),lang)
    def decimal(number):
        whole,_,fraction=number.replace(',','').partition('.')
        return word(whole)+((' point ' if lang=='English' else ' புள்ளி ')+' '.join(word(d) for d in fraction) if fraction else '')
    def replace(match):
        raw=match[0].strip()
        if any(a<=match.start() and match.end()<=b for a,b in identifiers):
            spoken=' '.join(word(d) for d in raw)
        elif raw.startswith('₹'):
            value=raw[1:].strip();whole,_,fraction=value.replace(',','').partition('.')
            spoken=word(whole)+(' rupees' if lang=='English' else ' ரூபாய்')
            if fraction and int(fraction[:2].ljust(2,'0')):
                spoken+=' '+word(fraction[:2].ljust(2,'0'))+(' paise' if lang=='English' else ' பைசா')
        elif raw.endswith('%'):
            spoken=decimal(raw[:-1].strip())+(' percent' if lang=='English' else ' சதவீதம்')
        elif ':' in raw:
            clock=re.match(r'(\d{1,2}):(\d{2})(?:\s*(AM|PM))?',raw,re.I)
            hour,minute=int(clock[1]),int(clock[2])
            if lang=='English':spoken=word(hour)+(' '+('oh ' if minute<10 else '')+word(minute) if minute else '')+(' '+clock[3] if clock[3] else '')
            else:spoken=word(hour)+' மணி'+(' '+word(minute)+' நிமிடம்' if minute else '')
        else:spoken=decimal(raw)
        marker=chr(0xE000+len(protected));protected.append((marker,original(spoken)));return marker
    masked=re.sub(r'₹\s*\d[\d,]*(?:\.\d+)?|\d{1,2}:\d{2}(?:\s*(?:AM|PM))?|\d+(?:,\d{3})*(?:\.\d+)?\s*%|\d+(?:,\d{3})*(?:\.\d+)?',replace,text,flags=re.I)
    result=original(masked)
    for marker,spoken in protected:result=result.replace(marker,spoken)
    if any(marker in result for marker,_ in protected):raise GuardError('Numerical speech normalization failed')
    return result

def install_numeric_normalizer(bridge):
    adapter=getattr(bridge,'TTS',None);namespace=getattr(adapter,'ns',None)
    if not isinstance(namespace,dict):return False
    targets=[namespace]
    prepare=namespace.get('prepare_movio_request');function_globals=getattr(prepare,'__globals__',None)
    if isinstance(function_globals,dict) and function_globals is not namespace:targets.append(function_globals)
    for target in targets:
        existing=target.get('normalize_taxi_text')
        if not callable(existing):continue
        original=getattr(existing,'_wx_original_normalizer',existing)
        def normalize(text,_original=original):
            language=NUMERIC_LANGUAGE.get()
            return speech_numeric_normalize(_original,text,language) if language else _original(text)
        normalize._wx_original_normalizer=original
        normalize._wx_scope='weather voice acoustic normalization only; original benchmark calls unchanged'
        target['normalize_taxi_text']=normalize
    return True


Writing /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_weather_guard.py


## 3. Add the call page
An explicit **Interrupt / change details** control stops playback, sentence pauses and queued speech, and opens the microphone for a correction. This version does not claim hands-free acoustic barge-in. Spoken replies are the normal date/time input; typed replies are optional. This voice popup has no manual calendar, date/time selector or forecast card. The manual website form in section 4 does have date/time input and visible weather advice, as requested. The original Open-Meteo credit remains.


In [67]:
%%writefile /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/weather_call_page.html
<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>AI Ride Call</title>
<style>
*{box-sizing:border-box}body{margin:0;background:#0e1525;color:#eef2ff;font:16px system-ui;display:grid;place-items:center;min-height:100vh}.phone{width:min(580px,96vw);padding:24px;background:#19253c;border-radius:25px;margin:16px auto}h2{margin:4px 0 10px}.hint{color:#b9c8e2;font-size:13px;line-height:1.5}button,input,select{padding:12px;border:0;border-radius:10px;font:inherit}button{cursor:pointer;background:#178d61;color:white}button:disabled{opacity:.45;cursor:default}select,input{background:#344660;color:white;max-width:100%}.row{display:flex;gap:8px;flex-wrap:wrap;margin:10px 0}.row>*{flex:1}#status,#weather{padding:12px;background:#273953;border-radius:12px;margin:12px 0;line-height:1.5}#weather{display:none}#log{height:260px;overflow:auto;background:#10192a;border-radius:12px;padding:12px;white-space:pre-wrap;overflow-wrap:anywhere;line-height:1.6}.you{color:#b2f5bf}.ai{color:#9fd2ff}.error{color:#ffc6a8}#interrupt{background:#996518}#hang{background:#b94150}#manual{width:100%;margin-top:10px}a{color:#b9d9ff}#readSummary{display:none;background:#344660}
</style></head><body><main class="phone">
<h2>AI Ride Call</h2><p class="hint" id="hint">Use headphones. To correct details while the assistant is speaking, press Interrupt / change details.</p>
<select id="lang"><option value="Tamil">தமிழ் — Tamil</option><option value="English">English</option><option value="Tanglish">Tanglish — தமிழ் + English</option></select>
<div class="row"><button id="bookMode">Book ride</button><button id="cancelMode">Cancel booking</button></div>
<div class="row"><button id="call">Start call</button><button id="interrupt" disabled>Interrupt / change details</button><button id="hang" disabled>End call</button></div>
<div id="status" aria-live="polite">Open this page using the dashboard’s AI Call button.</div>
<div id="rides"></div><div id="log" role="log"></div>
<input id="manual" placeholder="Type a correction or reply"><div class="row"><button id="send" disabled>Send reply</button><button id="readSummary">I have read the summary</button></div>
<p class="hint">Weather forecasts: <a href="https://open-meteo.com/" target="_blank" rel="noopener">Open-Meteo</a>. Conditions may change. Vehicle suggestions concern weather protection; road access and safety restrictions still apply.</p>
</main><script>
'use strict';
const $=id=>document.getElementById(id);
const nonce=location.hash.slice(1), origin=location.origin;
let mode='book',socket=null,stream=null,rec=null,context=null,analyser=null,source=null,raf=0;
let audio=null,finishAudio=null,audioURL=null,connected=false,busy=false,playing=false,callId=0,epoch=0,summaryNonce=null,summaryEpoch=null,ready=false,listenTimer=null,awaitInterrupt=false,playVersion=0,captureVersion=0;
let speech=null,gapTimer=null,finishGap=null,summaryKind='final';
const labels={English:['Book ride','Cancel booking','Start call','Interrupt / change details','End call','Send reply','I have read the summary'],Tamil:['பயணம் முன்பதிவு','முன்பதிவை ரத்து செய்','அழைப்பைத் தொடங்கு','நிறுத்தி விவரங்களை மாற்று','அழைப்பை முடி','பதிலை அனுப்பு','சுருக்கத்தைப் படித்தேன்'],Tanglish:['Ride book pannunga','Booking cancel pannunga','Call start','Niruthi details change','Call mudikkavum','Reply anuppu','Summary padichitten']};
function localize(){['bookMode','cancelMode','call','interrupt','hang','send','readSummary'].forEach((id,i)=>$(id).textContent=labels[$('lang').value][i]);}
$('lang').onchange=localize;localize();
function status(t){$('status').textContent=t;}
function offerReview(kind='final'){summaryKind=kind;const replay={English:'Replay weather advisory',Tamil:'வானிலை ஆலோசனையை மீண்டும் கேள்',Tanglish:'Vaanilai aalosanaiyai meendum kelu'};$('readSummary').textContent=kind==='weather'?replay[$('lang').value]:labels[$('lang').value][6];$('readSummary').style.display='inline-block';}
function log(who,text){const d=document.createElement('div');d.className=who==='You'?'you':who==='AI'?'ai':'error';d.textContent=who+': '+text;$('log').append(d);$('log').scrollTop=$('log').scrollHeight;}
// Same-origin fallback for browsers that detach a popup's window.opener.
const voiceChannel=(()=>{try{return typeof BroadcastChannel==='function'?new BroadcastChannel('movio-ai-booking'):null;}catch(e){return null;}})();
let parentTransport=null,helloAttempts=0;
function notify(type,data={}){const message={channel:'ai-ride-call',nonce,type,...data};let sent=false;
 if(parentTransport!=='broadcast'&&window.opener&&!window.opener.closed){try{window.opener.postMessage(message,origin);sent=true;}catch(e){}}
 if(voiceChannel&&(type==='hello'||!sent)){try{voiceChannel.postMessage(message);}catch(e){}}
}
function handleParentMessage(m,transport){if(!nonce||!m||m.channel!=='ai-ride-call'||m.nonce!==nonce)return;
 if(m.type==='ready'){ready=true;if(transport==='opener'||!parentTransport)parentTransport=transport;status('Ready');}
 if(m.type==='ready'||m.type==='rides'){const rides=m.rides||[];$('rides').textContent=mode==='cancel'?rides.map((r,i)=>`${i+1}. #${r.id}: ${r.pickup} → ${r.destination}`).join('\n'):'';if(socket?.readyState===1&&mode==='cancel')send({event:'active_rides',rides});}
 if(m.type==='result'&&socket?.readyState===1)send({event:'result',ok:!!m.ok,id:m.id,error:m.error,request_id:m.request_id});
}
window.addEventListener('message',e=>{if(e.origin!==origin||e.source!==window.opener)return;handleParentMessage(e.data,'opener');});
if(voiceChannel)voiceChannel.onmessage=e=>handleParentMessage(e.data,'broadcast');
function requestDashboard(){if(!nonce||ready)return;notify('hello');if(++helloAttempts<10)setTimeout(requestDashboard,500);}
if(nonce)requestDashboard();
function send(m){if(socket?.readyState===1)socket.send(JSON.stringify(m));}
const bookings=new Map();
async function commitOnce(token){
 const controller=new AbortController(),timer=setTimeout(()=>controller.abort(),12000);
 try{const response=await fetch('/api/weather-booking/commit',{method:'POST',credentials:'same-origin',signal:controller.signal,headers:{'Content-Type':'application/json'},body:JSON.stringify({token})});
  const result=await response.json();if(!response.ok||!result.ok){const error=Error(result.error||'Booking was not saved. Please confirm again.');error.http=true;throw error;}return result;
 }finally{clearTimeout(timer);}
}
async function commitBooking(action,id){
 const request=action.request_id;
 if(!request||!action.token){send({event:'result',request_id:request,ok:false,error:'Missing booking confirmation. Please confirm again.'});return;}
 if(!bookings.has(request))bookings.set(request,(async()=>{
  for(let attempt=0;attempt<2;attempt++){
   try{return await commitOnce(action.token);}
   catch(e){if(e.http||attempt===1)return {ok:false,error:e.name==='AbortError'?'Saving timed out. Checking whether the booking was stored.':e.message};}
  }
 })());
 const result=await bookings.get(request);
 if(result.ok)notify('booking_saved',{id:result.id});
 if(id===callId&&connected){send({event:'result',request_id:request,epoch:action.epoch,...result});status(result.ok?'Booking saved. Waiting for the spoken confirmation…':'Checking the booking result…');}
}
function stopCapture(){clearTimeout(listenTimer);listenTimer=null;cancelAnimationFrame(raf);raf=0;if(rec){rec.onstop=null;if(rec.state==='recording')rec.stop();rec=null;}source?.disconnect();source=null;analyser=null;if(context){context.close().catch(()=>{});context=null;}}
function releaseAudio(){if(audio){audio.onended=null;audio.onerror=null;audio.pause();audio.src='';audio=null;}if(audioURL){URL.revokeObjectURL(audioURL);audioURL=null;}}
function stopAudio(){playVersion++;speech=null;clearTimeout(gapTimer);gapTimer=null;if(finishGap){const done=finishGap;finishGap=null;done(false);}releaseAudio();if(finishAudio){const done=finishAudio;finishAudio=null;done(false);}playing=false;}
function listenSoon(id){clearTimeout(listenTimer);if(connected&&!busy&&!playing)listenTimer=setTimeout(()=>listen(id),300);}
function listen(id){if(!connected||busy||playing||id!==callId||!stream)return;stopCapture();
 const type=['audio/webm;codecs=opus','audio/webm','audio/mp4'].find(t=>MediaRecorder.isTypeSupported(t));
 rec=new MediaRecorder(stream,type?{mimeType:type}:{});const mine=rec,chunks=[],capture=++captureVersion;let speaking=false,first=0,lastVoice=0;
 rec.ondataavailable=e=>{if(e.data.size)chunks.push(e.data)};
 rec.onstop=()=>{cancelAnimationFrame(raf);if(!connected||id!==callId||!speaking)return;busy=true;status('Thinking…');
  const blob=new Blob(chunks,{type:mine.mimeType});if(blob.size>2000000){busy=false;status('Please use a shorter reply.');listenSoon(id);return;}
  const reader=new FileReader();reader.onload=()=>{if(connected&&id===callId&&capture===captureVersion)send({event:'audio',payload:String(reader.result).split(',')[1],summary_nonce:summaryNonce});};reader.readAsDataURL(blob);};
 rec.start();context=new AudioContext();source=context.createMediaStreamSource(stream);analyser=context.createAnalyser();analyser.fftSize=1024;source.connect(analyser);const samples=new Float32Array(analyser.fftSize),started=performance.now();status('Listening…');
 function monitor(){if(!connected||busy||playing||id!==callId||mine.state!=='recording')return;analyser.getFloatTimeDomainData(samples);let sum=0;for(const x of samples)sum+=x*x;const now=performance.now();if(Math.sqrt(sum/samples.length)>.023){if(!speaking){speaking=true;first=now;}lastVoice=now;}
  if(speaking&&now-first>450&&now-lastVoice>1900||now-started>27000){if(speaking)mine.stop();else{stopCapture();listenSoon(id);}return;}raf=requestAnimationFrame(monitor);}
 raf=requestAnimationFrame(monitor);
}
function interrupt(){if(!connected)return;awaitInterrupt=true;captureVersion++;stopAudio();stopCapture();summaryNonce=null;summaryEpoch=null;$('readSummary').style.display='none';busy=false;send({event:'interrupt'});status('Tell me the corrected details.');listenSoon(callId);}
$('interrupt').onclick=interrupt;
async function play(m,id){stopAudio();stopCapture();const ticket=playVersion;busy=true;playing=true;status('AI speaking…');let ended=false;const localEpoch=m.epoch;
 try{const bytes=Uint8Array.from(atob(m.payload),c=>c.charCodeAt(0));audioURL=URL.createObjectURL(new Blob([bytes],{type:'audio/wav'}));audio=new Audio(audioURL);
  const wait=new Promise(resolve=>{finishAudio=resolve;audio.onended=()=>resolve(true);audio.onerror=()=>resolve(false);});await audio.play();ended=await wait;
 }catch(e){log('Error','Audio could not play. Read the summary below.');}
 finally{if(ticket===playVersion&&id===callId&&(localEpoch==null||localEpoch===epoch)){stopAudio();busy=false;playing=false;if(ended&&m.summary_nonce){send({event:'playback_ended',epoch:m.epoch,summary_nonce:m.summary_nonce});$('readSummary').style.display='none';}else if(m.summary_nonce){offerReview(m.summary_kind);}if(m.done){stop();status('Call completed.');}else listenSoon(id);}}
}
function queueSpeech(m,id){
 if(m.event==='reply'){stopAudio();stopCapture();speech={id,epoch:m.epoch,meta:m,total:m.chunk_count,next:0,chunks:new Map(),running:false};busy=true;playing=true;status('AI speaking…');}
 const current=speech;if(!current||current.id!==id||current.epoch!==m.epoch||id!==callId)return;
 if(m.chunk_index<current.next)return;current.chunks.set(m.chunk_index,m);drainSpeech(current);
}
async function drainSpeech(current){
 if(current.running)return;current.running=true;
 try{
  while(speech===current&&current.id===callId&&current.epoch===epoch){
   const part=current.chunks.get(current.next);if(!part)return;current.chunks.delete(current.next);
   let ended=false;
   try{const bytes=Uint8Array.from(atob(part.payload),c=>c.charCodeAt(0));audioURL=URL.createObjectURL(new Blob([bytes],{type:'audio/wav'}));audio=new Audio(audioURL);
    const wait=new Promise(resolve=>{finishAudio=resolve;audio.onended=()=>resolve(true);audio.onerror=()=>resolve(false);});await audio.play();ended=await wait;
   }catch(e){ended=false;}
   if(speech!==current||current.id!==callId||current.epoch!==epoch)return;
   finishAudio=null;releaseAudio();
   if(!ended){speech=null;playing=false;busy=false;log('Error',current.meta.summary_kind==='weather'?'Weather audio could not play. Replay the advisory before continuing.':'Audio could not play. Review the booking details below.');if(current.meta.summary_nonce)offerReview(current.meta.summary_kind);listenSoon(current.id);return;}
   current.next++;
   if(current.next===current.total){speech=null;playing=false;busy=false;
    if(current.meta.summary_nonce)send({event:'playback_ended',epoch:current.epoch,summary_nonce:current.meta.summary_nonce});
    if(current.meta.done){stop();status('Call completed.');}else listenSoon(current.id);return;
   }
   if(part.pause_after_ms){await new Promise(resolve=>{finishGap=resolve;gapTimer=setTimeout(()=>{finishGap=null;gapTimer=null;resolve(true);},part.pause_after_ms);});}
  }
 }finally{current.running=false;if(speech===current&&current.id===callId&&current.epoch===epoch&&current.chunks.has(current.next))drainSpeech(current);}
}
async function start(){if(connected)return;if(mode==='cancel'&&(!ready||!nonce)){notify('hello');status('Reconnect using the dashboard AI Call button to load your bookings.');return;}const id=++callId;epoch=0;summaryNonce=null;awaitInterrupt=false;
 try{const binding=mode==='book'?await fetch('/api/weather-booking/session',{method:'POST',credentials:'same-origin',headers:{'Content-Type':'application/json'},body:'{}'}).then(async r=>{const j=await r.json();if(!r.ok)throw Error(j.error||'Sign in first');return j;}):null;
  stream=await navigator.mediaDevices.getUserMedia({audio:{echoCancellation:true,noiseSuppression:true,autoGainControl:true}});socket=new WebSocket(`${location.protocol==='https:'?'wss':'ws'}://${location.host}/browser-call?language=${encodeURIComponent($('lang').value)}&mode=${mode}`);
  socket.onopen=()=>{connected=true;busy=true;$('call').disabled=true;$('hang').disabled=false;$('interrupt').disabled=mode==='cancel';$('send').disabled=mode==='cancel';$('lang').disabled=true;$('bookMode').disabled=true;$('cancelMode').disabled=true;if(binding)send({event:'start',...binding});notify('get_rides');};
  socket.onmessage=async e=>{if(id!==callId)return;const m=JSON.parse(e.data);if(m.epoch!=null&&m.epoch<epoch)return;if(m.epoch!=null)epoch=m.epoch;
   if(awaitInterrupt&&['reply','speech_chunk','action'].includes(m.event))return;
   if(m.event==='ready')return;
   if(m.event==='processing'){busy=true;status('Thinking…');return;}
   if(m.event==='interrupted'){awaitInterrupt=false;busy=false;listenSoon(id);return;}
   if(m.event==='heard'){log('You',m.text);return;}
   if(m.event==='booking_error'){log('Error',m.message||'Booking was not saved.');status('Review the summary and confirm again.');return;}
   if(m.event==='error'){stopAudio();log('Error',m.message||'Please repeat');awaitInterrupt=false;busy=false;if(m.fatal)stop();else listenSoon(id);return;}
   if(m.event==='action'){busy=true;stopCapture();status('Saving your confirmed booking…');if(m.type==='weather_book')await commitBooking(m,id);else notify('action',{action:m});return;}
   if(m.event==='speech_chunk'){queueSpeech(m,id);return;}
   if(m.event==='reply'){if(m.text)log('AI',m.text);summaryNonce=m.summary_nonce||null;summaryEpoch=m.epoch;summaryKind=m.summary_kind||'final';$('readSummary').style.display='none';if(m.streaming)queueSpeech(m,id);else if(m.payload)await play(m,id);else{busy=false;if(summaryNonce)offerReview(summaryKind);listenSoon(id);}}
  };
  socket.onclose=()=>{if(id===callId)stop();};socket.onerror=()=>log('Error','Connection failed. Check the notebook services.');
 }catch(e){log('Error',e.message);stop();}
}
function stop(){callId++;connected=false;busy=false;stopAudio();stopCapture();socket?.close();socket=null;stream?.getTracks().forEach(t=>t.stop());stream=null;$('call').disabled=false;$('hang').disabled=true;$('interrupt').disabled=true;$('send').disabled=true;$('lang').disabled=false;$('bookMode').disabled=false;$('cancelMode').disabled=false;status('Call ended');}
function typed(text){if(!connected||!text.trim())return;if(playing||busy)interrupt();captureVersion++;stopCapture();busy=true;send({event:'text',text:text.trim(),summary_nonce:summaryNonce});}
$('send').onclick=()=>{typed($('manual').value);$('manual').value='';};$('manual').onkeydown=e=>{if(e.key==='Enter')$('send').click();};
$('readSummary').onclick=()=>{if(summaryNonce){if(summaryKind==='weather'){typed('repeat weather');$('readSummary').style.display='none';return;}send({event:'summary_read',epoch:summaryEpoch,summary_nonce:summaryNonce});$('readSummary').style.display='none';status('Summary read. Say yes to confirm, or give a correction.');listenSoon(callId);}};
$('bookMode').onclick=()=>{mode='book';$('rides').textContent='';};$('cancelMode').onclick=()=>{mode='cancel';notify('get_rides');};$('call').onclick=start;$('hang').onclick=stop;window.addEventListener('beforeunload',stop);
</script></body></html>


Writing /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/weather_call_page.html


## 4. Integrate the manual form and dashboard
The existing Request ride button now checks the weather and presents a review. Only Continue with this booking creates a ride. Editing the details invalidates the old review. The selected vehicle stays unchanged unless the customer changes it. Failed network responses can be retried against the same review to avoid duplicate rides.

Both the manual form and signed-in call page submit to the existing same-origin backend. Account, revision and expiry checks protect the stored journey snapshot. The dashboard refreshes after a save and retains cancellation controls for scheduled rides.


In [68]:
%%writefile /kaggle/working/MOVIO_MAIN_WEBSITE/web/weather_manual.js
/* Weather review in the existing manual booking form. No autonomous edits. */
(function installManualWeather(){
 'use strict';
 if(window.__weatherManualVersion==='0.3.0')return;
 const book=document.getElementById('bookRide');
 if(!book||typeof api!=='function'||typeof fare!=='function')return;
 window.__weatherManualVersion='0.3.0';book.type='button';
 const section=document.createElement('section');section.id='weatherManual';section.className='ride-card';
 section.innerHTML=`<div style="display:grid;gap:10px;margin:12px 0">
 <label><span id="wxLanguageLabel"></span><select id="wxManualLanguage"><option value="Tamil">தமிழ்</option><option value="Tanglish">Tamizh — English ezhuthukkal</option><option value="English">English</option></select></label>
 <label><span id="wxDateLabel"></span><input id="wxManualDate" type="datetime-local" required></label>
 <label><span id="wxCountLabel"></span><select id="wxManualCount"><option>1</option><option>2</option><option>3</option><option>4</option><option>5</option><option>6</option></select></label>
 <p id="wxManualStatus" role="status" style="white-space:pre-line"></p>
 <div id="wxManualReview" hidden><h3 id="wxWeatherTitle"></h3><p id="wxManualAdvice" style="white-space:pre-line"></p><p id="wxManualSummary" style="white-space:pre-line"></p>
 <div style="display:flex;gap:8px;flex-wrap:wrap"><button type="button" id="wxManualContinue"></button><button type="button" id="wxManualChange" class="secondary"></button></div></div>
 <p style="font-size:12px"><a href="https://open-meteo.com/" target="_blank" rel="noopener">Open-Meteo</a></p></div>`;
 book.parentNode.insertBefore(section,book);
 const el=id=>document.getElementById(id),langBox=el('wxManualLanguage'),dateBox=el('wxManualDate'),countBox=el('wxManualCount');
 const messages={English:{language:'Language',date:'Booking date and time',count:'Passengers',review:'Check weather and review booking',title:'Weather for your selected journey',continue:'Continue with this booking',change:'Change my details',loading:'Checking the weather for your selected journey…',saving:'Saving your booking…',ready:'Review the advice and booking details, then choose whether to continue.',changed:'Your details changed. Please review the latest weather and booking details.',locations:'Choose pickup and destination from the map or address suggestions.',time:'Please choose your booking date and time.',failed:'The request could not be completed. Please try again.',unknown:'The booking result could not be verified. Retry here to check the same booking, or check your booking list.',retry:'Check this booking again'},
 Tamil:{language:'மொழி',date:'பயணத்தின் தேதி மற்றும் நேரம்',count:'பயணிகளின் எண்ணிக்கை',review:'வானிலையையும் முன்பதிவையும் சரிபார்',title:'உங்கள் பயணத்திற்கான வானிலை',continue:'இந்த முன்பதிவைத் தொடரவும்',change:'விவரங்களை மாற்று',loading:'உங்கள் பயணத்திற்கான வானிலையைச் சரிபார்க்கிறோம்…',saving:'உங்கள் முன்பதிவைச் சேமிக்கிறோம்…',ready:'வானிலை மற்றும் முன்பதிவு விவரங்களைப் பார்த்துவிட்டுத் தொடர விரும்புகிறீர்களா என்பதைத் தேர்ந்தெடுக்கவும்.',changed:'உங்கள் விவரங்கள் மாறியுள்ளன. புதிய வானிலை மற்றும் முன்பதிவு விவரங்களை மீண்டும் சரிபார்க்கவும்.',locations:'புறப்படும் இடத்தையும் சேரும் இடத்தையும் வரைபடம் அல்லது முகவரி பரிந்துரைகளிலிருந்து தேர்ந்தெடுக்கவும்.',time:'பயணத்தின் தேதி மற்றும் நேரத்தைத் தேர்ந்தெடுக்கவும்.',failed:'கோரிக்கையை நிறைவு செய்ய முடியவில்லை. மீண்டும் முயற்சிக்கவும்.',unknown:'முன்பதிவின் முடிவைச் சரிபார்க்க முடியவில்லை. இதே முன்பதிவை மீண்டும் சரிபார்க்கவும் அல்லது உங்கள் முன்பதிவுப் பட்டியலைப் பாருங்கள்.',retry:'இதே முன்பதிவை மீண்டும் சரிபார்'},
 Tanglish:{language:'Mozhi',date:'Payanathin thedhi matrum neram',count:'Payanigalin ennikkai',review:'Vaanilaiyaiyum munpadhivaiyum saripaar',title:'Unga payanathirkaana vaanilai',continue:'Indha munpadhivai thodaravum',change:'Vivarangalai maatru',loading:'Unga payanathirkaana vaanilaiyai saripaarkkirom…',saving:'Unga munpadhivai semikkirom…',ready:'Vaanilai matrum munpadhivu vivarangalai paarthuvittu thodara virumbureengalaa enbadhai therndhedukkavum.',changed:'Unga vivarangal maariyullana. Pudhiya vaanilai matrum munpadhivu vivarangalai meendum saripaarunga.',locations:'Purappadum idathaiyum serum idathaiyum varaipadam alladhu mugavari parindhuraigalilirundhu therndhedukkavum.',time:'Payanathin thedhi matrum nerathai therndhedukkavum.',failed:'Korikkaiyai niraivu seiyya mudiyavillai. Meendum muyarchikkavum.',unknown:'Munpadhivin mudivai saripaarkka mudiyavillai. Idhe munpadhivai meendum saripaarunga alladhu unga munpadhivu pattiyalai paarunga.',retry:'Idhe munpadhivai meendum saripaar'}};
 let review=null,serial=0,previewBusy=false,saving=false,uncertain=false;
 function words(){return messages[langBox.value]||messages.English;}
 function labels(){const m=words();for(const [id,key] of [['wxLanguageLabel','language'],['wxDateLabel','date'],['wxCountLabel','count'],['wxWeatherTitle','title'],['wxManualChange','change']])el(id).textContent=m[key];
  el('wxManualContinue').textContent=uncertain?m.retry:m.continue;if(!saving&&!previewBusy)book.textContent=m.review;}
 function status(text){el('wxManualStatus').textContent=text;}
 function mainLanguage(){const value=el('language')?.value||'en';return /tanglish/i.test(value)?'Tanglish':/^ta|tamil/i.test(value)?'Tamil':'English';}
 langBox.value=mainLanguage();
 function payload(){return {language:langBox.value,count:Number(countBox.value),pickup_at:dateBox.value,pickup:pickup?.name,destination:destination?.name,
  plat:pickup?.lat,plng:pickup?.lng,dlat:destination?.lat,dlng:destination?.lng,vehicle,km,fare:fare(vehicle)};}
 function fingerprint(){return JSON.stringify(payload());}
 async function request(path,data,retry=false){
  for(let attempt=0;attempt<(retry?2:1);attempt++){
   const controller=new AbortController(),timer=setTimeout(()=>controller.abort(),15000);
   try{const response=await fetch('/api/weather-booking/'+path,{method:'POST',credentials:'same-origin',signal:controller.signal,headers:{'Content-Type':'application/json'},body:JSON.stringify(data)});
    const body=await response.json();if(!response.ok||!body.ok){const e=Error(body.error||words().failed);e.http=response.status;throw e;}return body;
   }catch(e){if(e.http||attempt===(retry?1:0))throw e;}finally{clearTimeout(timer);}
  }
 }
 function discard(old){if(old)request('manual-discard',old.binding).catch(()=>{});}
 function invalidate(){if(saving||uncertain)return;serial++;if(review)discard(review);review=null;el('wxManualReview').hidden=true;status(words().changed);labels();}
 function checkChanges(){if(review&&!saving&&!uncertain&&review.fingerprint!==fingerprint())invalidate();labels();}
 for(const control of [langBox,dateBox,countBox])control.addEventListener('change',()=>{if(control===langBox)labels();invalidate();});
 el('language')?.addEventListener('change',()=>{if(!saving&&!uncertain){langBox.value=mainLanguage();labels();invalidate();}});
 for(const id of ['pickup','destination'])el(id)?.addEventListener('input',invalidate);
 // Retain the original map/address/route code, then invalidate only a stale review.
 const originalPlace=setPlace;setPlace=function(...args){if(saving||uncertain)return;const result=originalPlace(...args);checkChanges();return result;};
 const originalFares=fares;fares=function(...args){const result=originalFares(...args);checkChanges();return result;};
 const vehicleRoot=el('vehicles');vehicleRoot?.addEventListener('click',e=>{if(saving||uncertain){e.preventDefault();e.stopImmediatePropagation();}},true);
 function lock(on){
  const controls=[book,langBox,dateBox,countBox,el('pickup'),el('destination'),el('wxManualContinue'),el('wxManualChange')].filter(Boolean);
  if(on){if(!section._disabled)section._disabled=controls.map(e=>[e,e.disabled]);for(const e of controls)e.disabled=true;
   const mapNode=el('map');if(mapNode){if(section._mapPointer==null)section._mapPointer=mapNode.style.pointerEvents;mapNode.style.pointerEvents='none';}}
  else{for(const [e,disabled] of section._disabled||[])e.disabled=disabled;const mapNode=el('map');if(mapNode)mapNode.style.pointerEvents=section._mapPointer||'';
   if(uncertain){book.disabled=true;el('wxManualChange').disabled=true;langBox.disabled=true;dateBox.disabled=true;countBox.disabled=true;el('wxManualContinue').disabled=false;}
   else{delete section._disabled;delete section._mapPointer;}}
 }
 book.onclick=async e=>{
  e?.preventDefault();if(saving||previewBusy||uncertain||role!=='customer')return;
  if(!pickup||!destination||el('pickup').value!==pickup.name||el('destination').value!==destination.name){status(words().locations);return;}
  if(!dateBox.value){status(words().time);return;}
  const data=payload(),captured=JSON.stringify(data),ticket=++serial;
  const old=review;review=null;discard(old);el('wxManualReview').hidden=true;previewBusy=true;book.disabled=true;status(words().loading);
  try{const result=await request('manual-preview',data);
   if(ticket!==serial||captured!==fingerprint()){discard({binding:result});status(words().changed);return;}
   review={fingerprint:captured,binding:{call_id:result.call_id,bind_token:result.bind_token,summary_nonce:result.summary_nonce,language:data.language}};
   el('wxManualAdvice').textContent=result.advisory;el('wxManualSummary').textContent=result.summary;el('wxManualReview').hidden=false;status(words().ready);
  }catch(error){if(ticket===serial)status(error.http?error.message:words().failed);}
  finally{previewBusy=false;book.disabled=uncertain;labels();}
 };
 el('wxManualChange').onclick=()=>{invalidate();dateBox.focus();};
 el('wxManualContinue').onclick=async()=>{
  if(saving||!review)return;if(!uncertain&&review.fingerprint!==fingerprint()){invalidate();return;}
  saving=true;lock(true);status(words().saving);
  try{const result=await request('manual-confirm',review.binding,true);
   uncertain=false;review=null;el('wxManualReview').hidden=true;rideId=result.id;status(result.message);toast(result.message);
   // Refresh failures must not turn a successful save into a failed booking.
   await Promise.allSettled([refreshRides(),renderVoiceHistory()]);
  }catch(error){uncertain=!error.http||error.http>=500;status(uncertain?words().unknown:error.message);}
  finally{saving=false;lock(false);labels();}
 };
 labels();
})();


Writing /kaggle/working/MOVIO_MAIN_WEBSITE/web/weather_manual.js


In [69]:
%%writefile /kaggle/working/MOVIO_MAIN_WEBSITE/web/weather_dashboard.js
/* Refreshes bookings saved by the call page; also supports older call pages. */
(function installWeatherAction(){
  'use strict';
  if(typeof handleMovioVoiceMessage!=='function'||typeof post!=='function'){
    console.error('Weather booking: load after the existing dashboard app.js.');return;
  }
  if(window.__weatherActionVersion==='0.3.0')return;
  window.__weatherActionVersion='0.3.0';
  const original=handleMovioVoiceMessage,completed=new Map();
  handleMovioVoiceMessage=async function(m){
    if(!m||m.channel!=='ai-ride-call'||m.nonce!==aiNonce)return;
    if(m.type==='booking_saved'){
      toast('Ride request #'+m.id+' saved');await renderVoiceHistory();await refreshRides();await scheduled();return;
    }
    if(m.type!=='action'||m.action?.type!=='weather_book')return original(m);
    const a=m.action,id=a.request_id;
    if(!id||!a.token)return;
    if(!completed.has(id))completed.set(id,(async()=>{
      try{return await post('/weather-booking/commit',{token:a.token});}
      catch(e){return {ok:false,error:e.message};}
    })());
    const result=await completed.get(id);
    sendVoice('result',{...result,request_id:id});
    if(result.ok){toast('Ride request #'+result.id+' saved');await renderVoiceHistory();await refreshRides();await scheduled();}
    else toast(result.error||'Booking was not saved');
  };
  async function scheduled(){
    let root=document.getElementById('weatherScheduledRides');
    try{const data=await api('/rides'),rides=data.rides.filter(r=>r.status==='scheduled');
      if(!rides.length){root?.replaceChildren();return;}
      if(!root){root=document.createElement('section');root.id='weatherScheduledRides';(document.getElementById('aiRideHistory')||document.body).append(root);}
      root.replaceChildren();const heading=document.createElement('h3');heading.textContent='Scheduled rides';root.append(heading);
      for(const r of rides){const item=document.createElement('div');item.className='ride-card';const p=document.createElement('p');p.textContent=`#${r.id} · ${r.vehicle} · ${r.pickup} → ${r.destination} · ${r.pickup_time} · ₹${r.fare}`;
        const b=document.createElement('button');b.type='button';b.textContent='Cancel #'+r.id;b.onclick=async()=>{try{await post('/weather-booking/cancel',{ride_id:r.id});await scheduled();await refreshRides();}catch(e){toast(e.message);}};
        item.append(p,b);root.append(item);}
    }catch(e){/* The existing sign-in flow handles expired accounts. */}
  }
  scheduled();setInterval(scheduled,10000);
})();


Writing /kaggle/working/MOVIO_MAIN_WEBSITE/web/weather_dashboard.js


## 5. Add isolated regression checks
These checks never use your real accounts or recordings. The backend source is loaded into an isolated temporary directory and uses a separate temporary database. Weather and speech generation are simulated.


In [70]:
%%writefile /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/test_weather_guard.py
"""Offline regression checks: real SQLite/HTTP, simulated weather and voice models.

run_tests(server_source=Path('/kaggle/working/MOVIO_MAIN_WEBSITE/server.py'))
uses an isolated copy of the supplied backend; no real accounts are changed.
"""
import asyncio
import contextvars
import copy
import http.client
import importlib.util
import io
import json
import pathlib
import sqlite3
import sys
import tempfile
import threading
import time
import types
import unittest
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from http.server import ThreadingHTTPServer
import hashlib

import movio_weather_guard as wx


def good_state():
    when = datetime.now(wx.TZ).replace(second=0, microsecond=0) + timedelta(days=1)
    state=dict(lang='English', mode='book', stage='confirm', count=1, vehicle='Bike',
        pickup='Chennai Central', destination='Airport', pickup_geo={'lat':13.08,'lng':80.27},
        destination_geo={'lat':12.99,'lng':80.17}, when=when.strftime('%Y-%m-%d %H:%M'),
        pickup_at=when.isoformat(),fare={'km':20,'fare':168},weather={'status':'ok','condition':'rain'})
    state['_wx_weather_consent']={'scope':wx.weather_consent_scope(state),'accepted_at':time.time()}
    return state


class Reply:
    def __init__(self, payload): self.payload=payload
    def raise_for_status(self): pass
    def json(self): return self.payload


def weather_get(code=61, probability=80, missing=False,temperature=28,feels_like=30):
    def get(url, params, timeout):
        start=params['start_date']
        hours=[f'{start}T{h:02}:00' for h in range(24)]
        row={'hourly':{'time':hours,'weather_code':[None if missing else code]*24,
            'precipitation_probability':[probability]*24,'precipitation':[2]*24,'temperature_2m':[temperature]*24,
            'apparent_temperature':[feels_like]*24}}
        return Reply([copy.deepcopy(row) for _ in params['latitude'].split(',')])
    return get


def manual_payload(lang='English',vehicle='Bike'):
    s=good_state()
    return dict(language=lang,count=1,vehicle=vehicle,pickup=s['pickup'],destination=s['destination'],
        plat=s['pickup_geo']['lat'],plng=s['pickup_geo']['lng'],dlat=s['destination_geo']['lat'],dlng=s['destination_geo']['lng'],
        pickup_at=s['pickup_at'],km=s['fare']['km'],fare=s['fare']['fare'])


class Fixtures(unittest.TestCase):
    def setUp(self):
        self.tmp=tempfile.TemporaryDirectory()
        self.db=wx.GuardDB(pathlib.Path(self.tmp.name)/'test.db')
        with self.db.connect() as c:
            c.execute('''CREATE TABLE rides(id INTEGER PRIMARY KEY,customer_id INTEGER,pickup TEXT,destination TEXT,
                plat REAL,plng REAL,dlat REAL,dlng REAL,vehicle TEXT,fare INTEGER,status TEXT,driver_id INTEGER,
                source TEXT,created REAL,updated REAL)''')
        self.db.setup()
        self.binding=self.db.create(7);self.sid=self.binding['call_id'];self.state=good_state()
        self.epoch=self.db.begin(self.sid);self.db.stash(self.sid,self.epoch,self.state)
    def tearDown(self): self.tmp.cleanup()
    def permit(self):
        nonce=self.db.summary(self.sid,self.epoch,self.state)
        self.assertTrue(self.db.played(self.sid,self.epoch,nonce))
        return self.db.permit(self.sid,self.epoch,nonce,self.state)
    def commit(self, token, owner=7):
        with self.db.connect() as c:
            return self.db.commit(c,owner,token,lambda c,rid:None)


class Dates(unittest.TestCase):
    def test_relative_english_and_tamil(self):
        now=datetime(2026,10,17,19,0,tzinfo=wx.TZ)
        for text in ('tomorrow at 9 AM','நாளை காலை 9 மணிக்கு','naalaiku 9 am'):
            self.assertEqual(wx.resolve_pickup(text,now).isoformat(),'2026-10-18T09:00:00+05:30')
    def test_iso_and_evening(self):
        now=datetime(2026,10,17,19,0,tzinfo=wx.TZ)
        self.assertEqual(wx.resolve_pickup('2026-10-18 21:30',now).hour,21)
        self.assertEqual(wx.resolve_pickup('tomorrow evening 9',now).hour,21)
    def test_ambiguous_and_past_are_clarified(self):
        now=datetime(2026,10,17,19,0,tzinfo=wx.TZ)
        for raw in ('tomorrow 9','2026-02-30 09:00','today 9 AM'):
            with self.assertRaises(wx.GuardError): wx.resolve_pickup(raw,now)
    def test_weekday_is_resolved(self):
        now=datetime(2026,10,17,19,0,tzinfo=wx.TZ)
        self.assertEqual(wx.resolve_pickup('Friday at 9 AM',now).isoformat(),'2026-10-23T09:00:00+05:30')
    def test_other_spoken_dates(self):
        now=datetime(2026,10,17,19,0,tzinfo=wx.TZ)
        for raw in ('day after tomorrow 9 AM','நாளை மறுநாள் காலை 9 மணிக்கு','அக்டோபர் 19 தேதி 2026 காலை 9 மணிக்கு','19/10/2026 9 AM','October 19, 2026 9 AM'):
            with self.subTest(raw=raw):
                self.assertEqual(wx.resolve_pickup(raw,now).isoformat(),'2026-10-19T09:00:00+05:30')
    def test_complete_tamil_evening_time_is_not_ambiguous(self):
        now=datetime(2026,10,17,18,0,tzinfo=wx.TZ)
        for raw in ('எனக்கு SUV வேண்டும் இன்று இரவு 07:00 மணிக்கு எனக்கு cab வேணும்','இன்று இரவு ஏழு மணிக்கு'):
            self.assertEqual(wx.resolve_pickup(raw,now).isoformat(),'2026-10-17T19:00:00+05:30')
    def test_tamil_period_only_keeps_date_and_hour(self):
        now=datetime(2026,10,17,18,0,tzinfo=wx.TZ)
        s={'stage':'time'}
        pickup,reason=wx.update_time_memory(s,'இன்று 07:00',now)
        self.assertEqual(reason,'clarify_period')
        pickup,reason=wx.update_time_memory(s,'இரவு',now)
        self.assertIsNone(reason);self.assertEqual(pickup.hour,19);self.assertEqual(pickup.day,17)
    def test_date_and_time_are_spoken_without_numeric_date_or_timezone(self):
        self.assertEqual(wx.spoken_datetime('2025-09-15T19:00:00+05:30'),
                         'September 15, 2025, at 7:00 PM')
        for lang in ('English','Tamil','Tanglish'):
            spoken=wx.spoken_datetime('2026-10-17T22:30:00+05:30',lang)
            self.assertNotRegex(spoken,r'India|இந்திய|05:30');self.assertIn('2026',spoken);self.assertIn('10:30',spoken)
    def test_tamil_yes_keeps_vowel_marks_and_ignores_punctuation(self):
        for raw in ('ஆம்.','ஆமாம்!','சரி.'):
            self.assertIn(wx.reply_normalized(raw),wx.POSITIVE)


class Weather(unittest.TestCase):
    def setUp(self):
        self.points=[good_state()['pickup_geo'],good_state()['destination_geo']]
        self.when=datetime.now(wx.TZ)+timedelta(days=1)
    def test_rain_and_cloud_conditions(self):
        self.assertEqual(wx.WeatherClient(weather_get()).forecast(self.points,self.when)['condition'],'rain')
        self.assertEqual(wx.WeatherClient(weather_get(3,20)).forecast(self.points,self.when)['condition'],'cloudy')
    def test_cache_reuses_payload_but_selects_correct_hour(self):
        client=wx.WeatherClient(weather_get())
        a=client.forecast(self.points,self.when.replace(hour=9))
        b=client.forecast(self.points,self.when.replace(hour=10))
        self.assertEqual(client.requests,1);self.assertEqual(client.hits,1)
        self.assertNotEqual(a['points'][0]['valid_at'],b['points'][0]['valid_at'])
    def test_null_data_is_not_reported_as_clear(self):
        self.assertEqual(wx.WeatherClient(weather_get(missing=True)).forecast(self.points,self.when)['status'],'unavailable')
    def test_api_failure_and_forecast_horizon(self):
        def fail(*a,**k):raise TimeoutError()
        client=wx.WeatherClient(fail)
        self.assertEqual(client.forecast(self.points,self.when)['status'],'unavailable')
        self.assertEqual(client.forecast(self.points,self.when+timedelta(days=30))['status'],'outside_forecast_range')
    def test_capacity_and_language_messages(self):
        s=good_state();s['count']=5
        self.assertIn('SUV',wx.advisory(s));self.assertNotIn('Mini',wx.advisory(s))
        for lang in ('English','Tamil','Tanglish'):
            s['lang']=lang;self.assertTrue(wx.advisory(s))


class CustomerAdvice(unittest.TestCase):
    def state(self,code=61,probability=80,temperature=28,feels_like=30,vehicle='Bike',lang='English'):
        s=good_state();s.update(vehicle=vehicle,lang=lang)
        s['weather']=wx.WeatherClient(weather_get(code,probability,temperature=temperature,feels_like=feels_like)).forecast(
            [s['pickup_geo'],s['destination_geo']],datetime.fromisoformat(s['pickup_at']))
        return s
    def test_heavy_rain_is_based_on_condition_not_probability(self):
        ordinary=self.state(61,99);heavy=self.state(65,30)
        self.assertFalse(ordinary['weather']['heavy_rain']);self.assertTrue(heavy['weather']['heavy_rain'])
        self.assertNotIn('Heavy rain',wx.advisory(ordinary));self.assertIn('Heavy rain',wx.advisory(heavy))
        self.assertIn('bike may be unsuitable',wx.advisory(heavy))
    def test_auto_and_covered_car_have_distinct_precautions(self):
        self.assertIn('exposed sides',wx.advisory(self.state(vehicle='Auto')))
        for vehicle in ('Mini','Sedan','SUV'):
            text=wx.advisory(self.state(vehicle=vehicle));self.assertIn('umbrella for boarding',text)
            self.assertNotIn('bike may be unsuitable',text)
    def test_high_temperature_is_not_labelled_pleasant(self):
        s=self.state(0,0,41,44)
        self.assertTrue(s['weather']['heat']);self.assertFalse(s['weather']['pleasant'])
        text=wx.advisory(s);self.assertIn('Very hot',text);self.assertIn('carry water',text);self.assertNotIn('pleasant',text)
    def test_feels_like_threshold_alone_can_trigger_heat_advice(self):
        self.assertTrue(self.state(0,0,34,42)['weather']['heat'])
    def test_pleasant_weather_message(self):
        s=self.state(0,0,28,29);self.assertTrue(s['weather']['pleasant']);self.assertIn('looks pleasant',wx.advisory(s))
    def test_rain_and_heat_precautions_can_both_be_present(self):
        s=self.state(61,90,39,43,vehicle='Auto');text=wx.advisory(s)
        self.assertIn('Rain is expected',text);self.assertIn('Very hot',text);self.assertIn('exposed sides',text)
    def test_advice_never_modifies_customer_choices(self):
        s=self.state(65,99,40,44);before=copy.deepcopy(s)
        wx.advisory(s);wx.summary_text(None,s)
        self.assertEqual(s,before)
    def test_tamil_weather_has_no_english_templates(self):
        s=self.state(65,90,40,44,lang='Tamil');s.update(pickup='சென்னை',destination='தாம்பரம்')
        text=wx.advisory(s);self.assertNotRegex(text,r'[A-Za-z]');self.assertIn('குடிநீர்',text)
    def test_tanglish_weather_and_date_use_latin_script(self):
        s=self.state(65,90,40,44,lang='Tanglish');text=wx.advisory(s)+' '+wx.summary_text(None,s)
        self.assertNotRegex(text,r'[\u0b80-\u0bff]');self.assertIn('mazhai',text);self.assertNotIn('Rain is expected',text)
    def test_successful_booking_closes_in_selected_language(self):
        expected={'Tamil':'நன்றி! உங்கள் பயணம் இனிமையாக அமையட்டும்!','Tanglish':'Nandri! Ungal payanam inimaiyaaga amaiyattum!','English':'Thank you! Have a wonderful journey!'}
        for lang,closing in expected.items():
            self.assertEqual(wx.closing_message(lang),closing);self.assertTrue(wx.completion_message(lang,123).endswith(closing))
    def test_sentence_clips_preserve_decimal_and_place_initial(self):
        parts=wx.speech_segments('At T. Nagar. Distance 3.5 kilometres. Thank you! Have a wonderful journey!')
        self.assertEqual(parts,['At T. Nagar.','Distance 3.5 kilometres.','Thank you!','Have a wonderful journey!'])


class Confirmation(Fixtures):
    def test_binding_requires_secret(self):
        self.assertEqual(self.db.bind(self.sid,self.binding['bind_token'])['owner'],7)
        with self.assertRaises(wx.GuardError):self.db.bind(self.sid,'wrong')
    def test_playback_or_explicit_read_ack_required(self):
        nonce=self.db.summary(self.sid,self.epoch,self.state)
        with self.assertRaises(wx.GuardError):self.db.permit(self.sid,self.epoch,nonce,self.state)
        self.assertFalse(self.db.played(self.sid,self.epoch,'old-nonce'))
    def test_interrupt_rejects_outdated_confirmation(self):
        token=self.permit();self.db.begin(self.sid,interrupt=True)
        with self.assertRaises(wx.GuardError):self.commit(token)
    def test_late_worker_cannot_overwrite_newer_turn(self):
        old=self.epoch;self.db.begin(self.sid)
        with self.assertRaises(wx.GuardError):self.db.stash(self.sid,old,self.state)
    def test_changed_details_revoke_previous_summary(self):
        token=self.permit();self.state['vehicle']='Mini';self.db.stash(self.sid,self.epoch,self.state)
        with self.assertRaises(wx.GuardError):self.commit(token)
    def test_different_snapshot_cannot_reuse_ack(self):
        nonce=self.db.summary(self.sid,self.epoch,self.state);self.db.played(self.sid,self.epoch,nonce)
        self.state['fare']['fare']+=1
        with self.assertRaises(wx.GuardError):self.db.permit(self.sid,self.epoch,nonce,self.state)
    def test_other_account_cannot_redeem(self):
        token=self.permit()
        with self.assertRaises(PermissionError):self.commit(token,owner=8)
    def test_retry_creates_one_ride(self):
        token=self.permit()
        with ThreadPoolExecutor(max_workers=8) as pool:rows=list(pool.map(lambda _:self.commit(token),range(24)))
        self.assertEqual(len({r['id'] for r in rows}),1)
        with self.db.connect() as c:self.assertEqual(c.execute('SELECT COUNT(*) FROM rides').fetchone()[0],1)
    def test_stored_fields_and_scheduled_status(self):
        result=self.commit(self.permit())
        with self.db.connect() as c:row=c.execute('SELECT * FROM rides WHERE id=?',(result['id'],)).fetchone()
        self.assertEqual(row['passengers'],1);self.assertEqual(row['pickup_time'],self.state['pickup_at'])
        self.assertEqual(row['status'],'scheduled');self.assertEqual(json.loads(row['weather_json'])['condition'],'rain')
    def test_commit_wins_then_interrupt_reports_saved_booking(self):
        self.commit(self.permit())
        with self.assertRaisesRegex(wx.GuardError,'already saved'):self.db.begin(self.sid,interrupt=True)
    def test_expired_permit_and_capacity_are_rejected(self):
        token=self.permit()
        with self.db.connect() as c:c.execute('UPDATE wx_permits SET expires=0')
        with self.assertRaises(wx.GuardError):self.commit(token)
        self.state['count']=2;token=self.permit()
        with self.assertRaisesRegex(wx.GuardError,'capacity'):self.commit(token)
    def test_release_only_within_dispatch_window(self):
        result=self.commit(self.permit())
        with self.db.connect() as c:c.execute('UPDATE rides SET pickup_time=? WHERE id=?',((datetime.now(wx.TZ)+timedelta(minutes=10)).isoformat(),result['id']))
        self.db.release_due()
        with self.db.connect() as c:self.assertEqual(c.execute('SELECT status FROM rides WHERE id=?',(result['id'],)).fetchone()[0],'requested')


class FakeBridge:
    PUBLIC_URL='https://test.invalid'
    MESSAGES={'English':{},'Tamil':{},'Tanglish':{}}
    CURRENT_LANGUAGE=contextvars.ContextVar('test_language',default='English')
    def __init__(self):
        async def original(ws):raise RuntimeError('legacy route used unexpectedly')
        self.app=types.SimpleNamespace(routes=[types.SimpleNamespace(path='/browser-call',endpoint=original)])
        self.block=threading.Event();self.block.set();self.started=threading.Event()
        self.spoken=[]
    def new_session(self,lang,mode):
        s=good_state();s.update(lang=lang,stage='time');return s
    def say(self,s,key,**kwargs):
        return 'summary '+str(kwargs) if key=='summary' else key
    def speak(self,text):
        self.spoken.append(text)
        if 'At your pickup' in text:self.started.set();self.block.wait(3)
        return 'FAKE_AUDIO'
    def handle(self,s,text):
        if text in wx.CAPACITY:s['vehicle']=text
        elif 'tomorrow' in text:s['when']=text
        s['stage']='fare';return 'wait',None
    def road_fare(self,s):
        s['pickup_geo']=s.get('pickup_geo') or {'lat':13.08,'lng':80.27}
        s['destination_geo']=s.get('destination_geo') or {'lat':12.92,'lng':80.12}
        return {'km':20,'fare':168 if s['vehicle']=='Bike' else 336}
    def geocode(self,text):return {'lat':13.08 if 'Central' in text else 12.92,'lng':80.27 if 'Central' in text else 80.12}


class TurnPreparation(unittest.TestCase):
    def test_spoken_complete_date_does_not_skip_missing_ride_details(self):
        s=good_state();s['count']=None
        out,answer=wx.prepare_turn(FakeBridge(),wx.WeatherClient(weather_get()),s,s['when'])
        self.assertEqual(out['stage'],'count');self.assertEqual(answer,'count')
    def test_vehicle_change_recalculates_fare_preserves_locations(self):
        bridge=FakeBridge();weather=wx.WeatherClient(weather_get());s=good_state();s['_wx_forecast_scope']=wx.digest({k:s.get(k) for k in ('pickup','destination','pickup_at','pickup_geo','destination_geo')})
        out,_=wx.prepare_turn(bridge,weather,s,'Mini')
        self.assertEqual(out['vehicle'],'Mini');self.assertEqual(out['fare']['fare'],336)
        self.assertEqual(out['pickup'],'Chennai Central');self.assertEqual(out['destination'],'Airport')
    def test_rain_bike_keeps_selection_and_uses_one_weather_question(self):
        s=good_state();s['stage']='time'
        s,answer=wx.prepare_turn(FakeBridge(),wx.WeatherClient(weather_get()),s,'tomorrow 9 AM')
        self.assertEqual(s['stage'],'weather_advisory');self.assertEqual(s['vehicle'],'Bike')
        self.assertIn('choice is yours',answer);self.assertEqual(answer.count('?'),1)
    def test_exact_tamil_conversation_preserves_every_slot(self):
        s={'lang':'Tamil','stage':'intent'};b=FakeBridge();weather=wx.WeatherClient(weather_get())
        now=datetime(2026,10,17,18,0,tzinfo=wx.TZ)
        s,_=wx.prepare_turn(b,weather,s,'நான் Chennai Central-இல் இருந்து Tambaram செல்ல வேண்டும் நாங்கள் நான்கு பேர் இருக்கிறோம்',now)
        self.assertEqual((s['count'],s['pickup'],s['destination']),(4,'Chennai Central','Tambaram'))
        self.assertEqual(s['stage'],'vehicle')
        s,answer=wx.prepare_turn(b,weather,s,'எனக்கு SUV வேண்டும் இன்று இரவு 07:00 மணிக்கு எனக்கு cab வேணும்',now)
        self.assertEqual(s['stage'],'weather_advisory');self.assertEqual(s['vehicle'],'SUV')
        self.assertEqual(s['pickup_at'],'2026-10-17T19:00:00+05:30')
        self.assertEqual(s['count'],4);self.assertIn('தொடர விரும்புகிறீர்களா',answer)
        self.assertNotRegex(answer,r'\d{4}-\d{2}-\d{2}|India time|இந்திய நேரம்')
    def test_past_today_time_is_explained_and_later_time_retains_today(self):
        b=FakeBridge();weather=wx.WeatherClient(weather_get());s=good_state();s.update(lang='Tamil',vehicle='SUV',count=4,when=None)
        now=datetime(2026,10,17,20,30,tzinfo=wx.TZ)
        s,answer=wx.prepare_turn(b,weather,s,'இன்று இரவு 07:00 மணிக்கு',now)
        self.assertEqual(s['stage'],'time');self.assertIn('கடந்துவிட்டது',answer)
        self.assertNotIn('பெட்டி',answer)
        s,answer=wx.prepare_turn(b,weather,s,'இரவு 10 மணிக்கு',now)
        self.assertEqual(s['stage'],'weather_advisory');self.assertEqual(s['pickup_at'],'2026-10-17T22:00:00+05:30')
        self.assertEqual((s['count'],s['vehicle'],s['pickup']),(4,'SUV','Chennai Central'))
    def test_today_alone_then_clock_only_does_not_reask_date(self):
        b=FakeBridge();weather=wx.WeatherClient(weather_get());s=good_state();s.update(lang='Tamil',vehicle='SUV',when=None,pickup_at=None)
        now=datetime(2026,10,17,18,0,tzinfo=wx.TZ)
        s,answer=wx.prepare_turn(b,weather,s,'இன்று',now)
        self.assertEqual(s['stage'],'time');self.assertIn('எத்தனை மணிக்கு',answer)
        s,answer=wx.prepare_turn(b,weather,s,'இரவு ஏழு மணிக்கு',now)
        self.assertEqual(s['stage'],'weather_advisory');self.assertEqual(s['pickup_at'],'2026-10-17T19:00:00+05:30')
    def test_both_probabilities_spoken_before_question_and_hidden_from_display(self):
        s=good_state();s['weather']={'status':'ok','condition':'rain','points':[
            {'location':'pickup','precipitation_probability':98},
            {'location':'destination','precipitation_probability':99}]}
        answer=wx.advisory(s)+'\n'+wx.weather_question(s['lang'])
        self.assertIn('98%',answer);self.assertIn('99%',answer)
        self.assertLess(answer.index('99%'),answer.index('Would you like'))
        display=wx.display_reply(s,answer)
        self.assertNotIn('percent',display);self.assertNotIn('rain',display)
        self.assertIn('Would you like to continue',display)
    def test_missing_probability_is_not_spoken_as_zero(self):
        s=good_state();s['weather']={'status':'ok','condition':'cloudy','points':[
            {'location':'pickup','precipitation_probability':0},
            {'location':'destination','precipitation_probability':None}]}
        answer=wx.probability_report(s)
        self.assertEqual(answer.count('0%'),1);self.assertIn('drop location, Airport, is not available',answer)
    def test_time_is_not_mistaken_for_passengers_or_plain_cab_for_mini(self):
        s=good_state();s.update(count=4,vehicle='SUV',lang='Tamil')
        s,_=wx.prepare_turn(FakeBridge(),wx.WeatherClient(weather_get()),s,'நாளை இரவு 07:00 மணிக்கு எனக்கு cab வேணும்')
        self.assertEqual((s['count'],s['vehicle']),(4,'SUV'))
    def test_capacity_checked_before_final_confirmation(self):
        s=good_state();s['count']=4
        s,answer=wx.prepare_turn(FakeBridge(),wx.WeatherClient(weather_get()),s,'Bike')
        self.assertEqual(s['stage'],'vehicle');self.assertIn('SUV',answer)


class FakeSocket:
    headers={'origin':'https://test.invalid'}
    query_params={'language':'English','mode':'book'}
    def __init__(self):self.inbox=asyncio.Queue();self.outbox=asyncio.Queue();self.accepted=False
    async def accept(self):self.accepted=True
    async def close(self,code):pass
    async def receive_json(self):
        msg=await self.inbox.get()
        if msg is None:raise RuntimeError('test disconnect')
        return msg
    async def send_json(self,value):await self.outbox.put(value)
    async def until(self,event,timeout=3):
        while True:
            m=await asyncio.wait_for(self.outbox.get(),timeout)
            if m['event']==event:return m


class SocketRaces(unittest.TestCase):
    async def final_review(self,socket,advisory):
        await socket.inbox.put({'event':'playback_ended','epoch':advisory['epoch'],'summary_nonce':advisory['summary_nonce']})
        await socket.inbox.put({'event':'text','text':'yes','summary_nonce':advisory['summary_nonce']})
        final=await socket.until('reply')
        self.assertEqual(final['stage'],'confirm');self.assertEqual(final['summary_kind'],'final')
        return final
    def test_first_sentence_is_delivered_before_later_generation_and_interrupt_clears_rest(self):
        async def scenario():
            fixture=Fixtures();fixture.setUp();entered,release=threading.Event(),threading.Event()
            try:
                bridge=FakeBridge();original=bridge.speak
                def speak(text):
                    if text.startswith('At your drop'):
                        entered.set();release.wait(3)
                    return original(text)
                bridge.speak=speak
                wx.install_bridge(bridge,fixture.db.path,wx.WeatherClient(weather_get(0,0)),_route_wrapper=lambda f:f)
                socket=FakeSocket();task=asyncio.create_task(bridge._wx_endpoint(socket))
                await socket.inbox.put({'event':'start',**fixture.binding});await socket.until('reply')
                await socket.inbox.put({'event':'text','text':fixture.state['when']});first=await socket.until('reply')
                self.assertTrue(first['streaming']);self.assertEqual(first['chunk_index'],0);self.assertGreater(first['chunk_count'],2)
                self.assertEqual(first['pause_after_ms'],220);self.assertFalse(fixture.db.row(fixture.sid)['played'])
                for _ in range(100):
                    if entered.is_set():break
                    await asyncio.sleep(.005)
                self.assertTrue(entered.is_set())
                await socket.inbox.put({'event':'interrupt'});interrupt=await socket.until('interrupted');release.set();await asyncio.sleep(.04)
                rest=[]
                while not socket.outbox.empty():rest.append(socket.outbox.get_nowait())
                self.assertFalse(any(m['event']=='speech_chunk' and m['epoch']<interrupt['epoch'] for m in rest))
                await socket.inbox.put(None);await task
            finally:release.set();fixture.tearDown()
        asyncio.run(scenario())
    def test_late_forecast_cannot_restore_old_date(self):
        async def scenario():
            fixture=Fixtures();fixture.setUp()
            entered,release=threading.Event(),threading.Event()
            original=weather_get(3,10);calls=[0]
            def delayed(*a,**k):
                calls[0]+=1
                if calls[0]==1:entered.set();release.wait(3)
                return original(*a,**k)
            try:
                bridge=FakeBridge();wx.install_bridge(bridge,fixture.db.path,wx.WeatherClient(delayed),_route_wrapper=lambda f:f)
                socket=FakeSocket();task=asyncio.create_task(bridge._wx_endpoint(socket))
                await socket.inbox.put({'event':'start',**fixture.binding});await socket.until('reply')
                await socket.inbox.put({'event':'text','text':fixture.state['when']});await socket.until('heard')
                for _ in range(100):
                    if entered.is_set():break
                    await asyncio.sleep(.01)
                self.assertTrue(entered.is_set())
                await socket.inbox.put({'event':'interrupt'});await socket.until('interrupted')
                new=(datetime.now(wx.TZ)+timedelta(days=2)).strftime('%Y-%m-%d %H:%M')
                await socket.inbox.put({'event':'text','text':new});await socket.until('heard');release.set()
                reply=await socket.until('reply');self.assertEqual(reply['stage'],'weather_advisory')
                self.assertEqual(json.loads(fixture.db.row(fixture.sid)['state'])['when'],wx.resolve_pickup(new).isoformat())
                await socket.inbox.put(None);await task
            finally:release.set();fixture.tearDown()
        asyncio.run(scenario())
    def test_interrupted_tts_is_not_delivered(self):
        async def scenario():
            fixture=Fixtures();fixture.setUp()
            try:
                bridge=FakeBridge();bridge.block.clear()
                wx.install_bridge(bridge,fixture.db.path,wx.WeatherClient(weather_get(3,10)),_route_wrapper=lambda f:f)
                socket=FakeSocket();task=asyncio.create_task(bridge._wx_endpoint(socket))
                await socket.inbox.put({'event':'start',**fixture.binding})
                await socket.until('reply')
                await socket.inbox.put({'event':'text','text':fixture.state['when']})
                await socket.until('heard')
                for _ in range(100):
                    if bridge.started.is_set():break
                    await asyncio.sleep(.01)
                self.assertTrue(bridge.started.is_set())
                await socket.inbox.put({'event':'interrupt'});await socket.until('interrupted')
                bridge.block.set();await asyncio.sleep(.06)
                messages=[]
                while not socket.outbox.empty():messages.append(socket.outbox.get_nowait())
                self.assertFalse(any(m['event']=='reply' and m.get('summary_nonce') for m in messages))
                await socket.inbox.put(None);await task
            finally:fixture.tearDown()
        asyncio.run(scenario())
    def test_confirm_without_ack_cannot_send_booking_action(self):
        async def scenario():
            fixture=Fixtures();fixture.setUp()
            try:
                bridge=FakeBridge();wx.install_bridge(bridge,fixture.db.path,wx.WeatherClient(weather_get(3,10)),_route_wrapper=lambda f:f)
                socket=FakeSocket();task=asyncio.create_task(bridge._wx_endpoint(socket))
                await socket.inbox.put({'event':'start',**fixture.binding});await socket.until('reply')
                await socket.inbox.put({'event':'text','text':fixture.state['when']});summary=await socket.until('reply')
                self.assertTrue(summary['summary_nonce'])
                await socket.inbox.put({'event':'text','text':'yes','summary_nonce':summary['summary_nonce']})
                refreshed=await socket.until('reply')
                self.assertNotEqual(refreshed['summary_nonce'],summary['summary_nonce'])
                final=await self.final_review(socket,refreshed)
                with fixture.db.connect() as c:self.assertEqual(c.execute('SELECT COUNT(*) FROM rides').fetchone()[0],0)
                await socket.inbox.put({'event':'playback_ended','epoch':final['epoch'],'summary_nonce':final['summary_nonce']})
                await socket.inbox.put({'event':'text','text':'ஆம்.','summary_nonce':final['summary_nonce']})
                action=await socket.until('action');self.assertEqual(action['type'],'weather_book')
                await socket.inbox.put(None);await task
            finally:fixture.tearDown()
        asyncio.run(scenario())
    def test_failed_http_commit_returns_a_fresh_confirmation(self):
        async def scenario():
            fixture=Fixtures();fixture.setUp()
            try:
                bridge=FakeBridge();wx.install_bridge(bridge,fixture.db.path,wx.WeatherClient(weather_get(3,10)),_route_wrapper=lambda f:f)
                socket=FakeSocket();task=asyncio.create_task(bridge._wx_endpoint(socket))
                await socket.inbox.put({'event':'start',**fixture.binding});await socket.until('reply')
                await socket.inbox.put({'event':'text','text':fixture.state['when']});summary=await socket.until('reply')
                summary=await self.final_review(socket,summary)
                await socket.inbox.put({'event':'playback_ended','epoch':summary['epoch'],'summary_nonce':summary['summary_nonce']})
                await socket.inbox.put({'event':'text','text':'yes','summary_nonce':summary['summary_nonce']})
                action=await socket.until('action')
                await socket.inbox.put({'event':'result','request_id':action['request_id'],'ok':False,'error':'HTTP timeout'})
                error=await socket.until('booking_error');self.assertEqual(error['message'],'HTTP timeout')
                refreshed=await socket.until('reply');self.assertTrue(refreshed['summary_nonce']);self.assertFalse(refreshed['done'])
                self.assertIn('not saved',refreshed['text']);self.assertNotIn('weather',refreshed)
                await socket.inbox.put(None);await task
            finally:fixture.tearDown()
        asyncio.run(scenario())
    def test_saved_ride_is_confirmed_even_if_http_response_timed_out(self):
        async def scenario():
            fixture=Fixtures();fixture.setUp()
            try:
                bridge=FakeBridge();wx.install_bridge(bridge,fixture.db.path,wx.WeatherClient(weather_get(3,10)),_route_wrapper=lambda f:f)
                socket=FakeSocket();task=asyncio.create_task(bridge._wx_endpoint(socket))
                await socket.inbox.put({'event':'start',**fixture.binding});await socket.until('reply')
                await socket.inbox.put({'event':'text','text':fixture.state['when']});summary=await socket.until('reply')
                summary=await self.final_review(socket,summary)
                await socket.inbox.put({'event':'playback_ended','epoch':summary['epoch'],'summary_nonce':summary['summary_nonce']})
                await socket.inbox.put({'event':'text','text':'yes','summary_nonce':summary['summary_nonce']})
                action=await socket.until('action');result=fixture.commit(action['token'])
                await socket.inbox.put({'event':'result','request_id':action['request_id'],'ok':False,'error':'HTTP timeout'})
                reply=await socket.until('reply');self.assertTrue(reply['done']);self.assertIn(wx.closing_message('English'),reply['text'])
                last=reply
                while last['chunk_index']<last['chunk_count']-1:last=await socket.until('speech_chunk')
                self.assertEqual(last['pause_after_ms'],0);self.assertIn('Have a wonderful journey!',bridge.spoken)
                self.assertEqual(fixture.db.row(fixture.sid)['ride_id'],result['id'])
                await socket.inbox.put(None);await task
            finally:fixture.tearDown()
        asyncio.run(scenario())


class CallPage(unittest.TestCase):
    def test_no_manual_date_control_or_forecast_card(self):
        html=pathlib.Path(__file__).with_name('weather_call_page.html').read_text()
        for old in ('datetime-local','pickupDate','sendDate','id="weather"','showWeather','India time','இந்திய நேரம்'):
            self.assertNotIn(old,html)
    def test_authenticated_commit_in_popup_has_timeout_and_result_id(self):
        html=pathlib.Path(__file__).with_name('weather_call_page.html').read_text()
        for required in ("fetch('/api/weather-booking/commit'","credentials:'same-origin'","AbortController","request_id:request"):
            self.assertIn(required,html)


class AcceptanceBridge(FakeBridge):
    def __init__(self):super().__init__();self.route_calls=0
    def canonical_place(self,text):
        return 'Chennai Central' if 'சென்ட்ர' in text else text
    def road_fare(self,state):
        self.route_calls+=1
        return super().road_fare(state)


class VoiceAcceptance(unittest.TestCase):
    def setUp(self):
        self.now=datetime(2026,10,18,7,15,tzinfo=wx.TZ)
        self.bridge=AcceptanceBridge();self.weather=wx.WeatherClient(weather_get(61,70))
    def turn(self,text,state=None,lang='Tamil'):
        return wx.prepare_turn(self.bridge,self.weather,state if state is not None else {'lang':lang,'stage':'intent'},text,self.now)
    def consented(self,state):
        state['_wx_weather_consent']={'scope':wx.weather_consent_scope(state),'accepted_at':time.time()}
        return state
    def test_all_six_slots_in_one_tamil_utterance(self):
        s,answer=self.turn('நான் Chennai Central-இல் இருந்து Tambaram செல்ல வேண்டும். நான்கு பேர். SUV வேண்டும். இன்று மதியம் பன்னிரண்டு மணிக்கு.')
        self.assertEqual((s['pickup'],s['destination'],s['count'],s['vehicle']),('Chennai Central','Tambaram',4,'SUV'))
        self.assertEqual(s['pickup_at'],'2026-10-18T12:00:00+05:30');self.assertEqual(s['stage'],'weather_advisory')
        self.assertEqual(self.bridge.route_calls,0);self.assertNotIn('எத்தனை மணிக்கு',answer)
    def test_native_tamil_central_and_count_do_not_confuse_clock(self):
        s,_=self.turn('நான் சென்னை சென்ட்ரலில் இருந்து தாம்பரம் செல்ல வேண்டும் நாங்கள் நான்கு பேர் இருக்கிறோம் எனக்கு SUV வேண்டும் இன்று மதியம் பன்னிரண்டு மணிக்கு வேண்டும்')
        self.assertEqual(s['count'],4);self.assertEqual(s['pickup'],'Chennai Central');self.assertEqual(s['vehicle'],'SUV')
        self.assertEqual(s['stage'],'weather_advisory');self.assertEqual(s['pickup_at'],'2026-10-18T12:00:00+05:30')
    def test_all_six_slots_in_one_english_utterance(self):
        s,_=self.turn('I need a Mini from Chennai Central to Tambaram. Four passengers, tomorrow at 6 PM.',lang='English')
        self.assertEqual((s['pickup'],s['destination'],s['count'],s['vehicle']),('Chennai Central','Tambaram',4,'Mini'))
        self.assertEqual(s['pickup_at'],'2026-10-19T18:00:00+05:30');self.assertEqual(s['stage'],'weather_advisory')
    def test_mixed_tamil_english_slots_in_same_sentence(self):
        s,_=self.turn('நான் Chennai Central-இல் இருந்து Tambaram செல்ல வேண்டும். One person, bike, today at 12 PM.')
        self.assertEqual((s['count'],s['vehicle'],s['pickup'],s['destination']),(1,'Bike','Chennai Central','Tambaram'))
        self.assertEqual(s['pickup_at'],'2026-10-18T12:00:00+05:30')
    def test_all_six_slots_in_tanglish(self):
        s,_=self.turn('Naan Chennai Central la irundhu Tambaram poganum. Naalu peru, SUV venum, innaiku madhiyam pannirandu mani.',lang='Tanglish')
        self.assertEqual((s['count'],s['vehicle'],s['pickup'],s['destination']),(4,'SUV','Chennai Central','Tambaram'))
        self.assertEqual(s['pickup_at'],'2026-10-18T12:00:00+05:30')
    def test_named_dates_and_spoken_tamil_day(self):
        for text,expected in (('20th October 6 PM','2026-10-20T18:00:00+05:30'),
                              ('அக்டோபர் இருபதாம் தேதி மாலை ஆறு மணிக்கு','2026-10-20T18:00:00+05:30'),
                              ('October 20, 2026 at 6 PM','2026-10-20T18:00:00+05:30')):
            with self.subTest(text=text):self.assertEqual(wx.resolve_pickup(text,self.now).isoformat(),expected)
    def test_noon_midnight_night_and_evening(self):
        for text,hour in (('tomorrow afternoon 12 PM',12),('tomorrow midnight 12 AM',0),('tomorrow night 1',1),
                           ('நாளைக்கு இரவு ஒரு மணிக்கு',1),('நாளை நள்ளிரவு பன்னிரண்டு மணிக்கு',0),
                           ('நாளை மாலை ஆறு மணிக்கு',18)):
            with self.subTest(text=text):self.assertEqual(wx.resolve_pickup(text,self.now).hour,hour)
    def test_compact_numeric_clock_and_conflicting_period(self):
        for text,hour in (('tomorrow 12PM',12),('tomorrow 07PM',19),('நாளை மதியம்12மணிக்கு',12)):
            with self.subTest(text=text):self.assertEqual(wx.resolve_pickup(text,self.now).hour,hour)
        for text in ('tomorrow morning 6 PM','tomorrow midnight 12 PM'):
            with self.assertRaisesRegex(wx.GuardError,'clarify_period'):wx.resolve_pickup(text,self.now)
    def test_past_today_is_corrected_without_losing_other_slots(self):
        s,answer=self.turn('from Chennai Central to Tambaram, one person, Bike, today 6 AM')
        self.assertEqual(s['stage'],'time');self.assertIn('கடந்துவிட்டது',answer)
        s,_=self.turn('இன்று மதியம் 12 மணி',s)
        self.assertEqual(s['pickup_at'],'2026-10-18T12:00:00+05:30');self.assertEqual(s['vehicle'],'Bike')
    def test_bike_to_mini_recalculates_and_preserves_other_slots(self):
        s,_=self.turn('from Chennai Central to Tambaram. One person, Bike, tomorrow 12 PM',lang='English')
        self.consented(s);s,_=self.turn('',s)
        self.assertEqual(s['stage'],'confirm');old={k:s[k] for k in ('count','pickup','destination','pickup_at')}
        s,answer=self.turn('No, change my bike to mini car.',s)
        self.assertEqual(s['vehicle'],'Mini');self.assertEqual(s['fare']['fare'],336)
        self.assertEqual({k:s[k] for k in old},old);self.assertEqual(s['stage'],'confirm')
        self.assertIn('Mini',answer);self.assertNotIn('Bike',answer);self.assertEqual(self.bridge.route_calls,2)
    def test_changed_time_requires_new_weather_consent(self):
        s,_=self.turn('from Chennai Central to Tambaram. One person, Bike, tomorrow 12 PM',lang='English')
        self.consented(s);s,_=self.turn('',s);s,_=self.turn('tomorrow at 6 PM',s)
        self.assertEqual(s['stage'],'weather_advisory');self.assertFalse(wx.weather_consented(s))
        self.assertEqual(s['pickup'],'Chennai Central');self.assertEqual(s['count'],1)
    def test_route_failure_cannot_erase_slots_and_retry_works(self):
        s,_=self.turn('from Chennai Central to Tambaram. One person, Bike, tomorrow 12 PM',lang='English')
        self.consented(s);old={k:s[k] for k in ('count','vehicle','pickup','destination','pickup_at')}
        original=self.bridge.road_fare
        def bad_route(working):working.update(pickup=None,destination=None,when=None);return None
        self.bridge.road_fare=bad_route;s,answer=self.turn('',s)
        self.assertEqual(s['stage'],'route_unavailable');self.assertIsNone(s['fare'])
        self.assertEqual({k:s[k] for k in old},old);self.assertNotIn('₹',answer)
        self.bridge.road_fare=original;s,_=self.turn('retry',s)
        self.assertEqual(s['stage'],'confirm');self.assertEqual({k:s[k] for k in old},old)
    def test_invalid_route_numbers_are_not_quoted_as_real_fares(self):
        for fare in ({'fare':float('nan'),'km':35.4},{'fare':'unknown','km':35.4},{'fare':100}):
            s,_=self.turn('from Chennai Central to Tambaram. One person, Bike, tomorrow 12 PM',lang='English')
            self.consented(s);self.bridge.road_fare=lambda working,fare=fare:fare;s,answer=self.turn('',s)
            self.assertEqual(s['stage'],'route_unavailable');self.assertNotIn('₹',answer)
    def test_unavailable_weather_still_allows_review_after_first_consent(self):
        self.weather=wx.WeatherClient(lambda *a,**kw:(_ for _ in ()).throw(TimeoutError()))
        s,answer=self.turn('from Chennai Central to Tambaram. One person, Bike, tomorrow 12 PM',lang='English')
        self.assertEqual(s['weather']['status'],'unavailable');self.assertNotIn('%',answer)
        self.consented(s);s,_=self.turn('',s);self.assertEqual(s['stage'],'confirm')
    def test_outside_forecast_horizon_does_not_move_the_date(self):
        s,answer=self.turn('from Chennai Central to Tambaram. One person, Bike, 30th November 12 PM',lang='English')
        self.assertEqual(s['weather']['status'],'outside_forecast_range');self.assertNotIn('%',answer)
        self.consented(s);s,_=self.turn('',s);self.assertTrue(s['pickup_at'].startswith('2026-11-30'));self.assertEqual(s['stage'],'confirm')
    def test_tts_templates_keep_numeric_entities_as_digits(self):
        for lang in ('Tamil','Tanglish'):
            s=good_state();s.update(lang=lang,count=4,vehicle='SUV',fare={'km':35.4,'fare':1126})
            s['weather']={'status':'ok','condition':'rain','points':[{'location':'pickup','precipitation_probability':70},{'location':'destination','precipitation_probability':60}]}
            text=wx.summary_text(None,s)
            for value in ('4','35.4','₹1,126'):self.assertIn(value,text)
            self.assertIn('70%',wx.probability_report(s));self.assertIn('60%',wx.probability_report(s))
            self.assertIn('48291',wx.completion_message(lang,48291));self.assertNotIn('நான்கு பேர்',text)
    def test_adapter_receives_digits_and_selected_language(self):
        captured=[];bridge=AcceptanceBridge()
        bridge.TTS=types.SimpleNamespace(synthesize=lambda text,group:(captured.append((text,group)) or b'audio'))
        for lang,group in (('Tamil','TAMIL'),('Tanglish','TANGLISH'),('English','ENGLISH')):
            token=bridge.CURRENT_LANGUAGE.set(lang)
            try:wx.tts_reply(bridge,'35.4, ₹1,126, 70%, 48291')
            finally:bridge.CURRENT_LANGUAGE.reset(token)
            self.assertEqual(captured[-1],('35.4, ₹1,126, 70%, 48291',group))
    def test_numeric_normalizer_handles_decimal_percent_currency_and_id(self):
        text='35.4 கிலோமீட்டர், ₹1,126, 70%, முன்பதிவு எண் 48291.'
        result=wx.speech_numeric_normalize(lambda s:s,text,'Tamil')
        for value in ('முப்பத்து ஐந்து புள்ளி நான்கு','ஆயிரத்து நூற்று இருபத்து ஆறு ரூபாய்','எழுபது சதவீதம்','நான்கு எட்டு இரண்டு ஒன்பது ஒன்று'):
            self.assertIn(value,result)
        self.assertNotIn('%',result);self.assertNotRegex(result,r'\d')
    def test_numeric_scope_leaves_legacy_normalization_unchanged(self):
        original=lambda text:'ORIGINAL '+text
        namespace={'normalize_taxi_text':original};bridge=AcceptanceBridge()
        captured=[]
        bridge.TTS=types.SimpleNamespace(ns=namespace,synthesize=lambda text,group:(captured.append(namespace['normalize_taxi_text'](text)) or b'audio'))
        wx.install_numeric_normalizer(bridge)
        self.assertEqual(namespace['normalize_taxi_text']('35.4, 70%'),original('35.4, 70%'))
        token=bridge.CURRENT_LANGUAGE.set('English')
        try:wx.tts_reply(bridge,'35.4, 70%')
        finally:bridge.CURRENT_LANGUAGE.reset(token)
        self.assertIn('point',captured[-1]);self.assertIn('percent',captured[-1])
        self.assertEqual(namespace['normalize_taxi_text']('35.4, 70%'),original('35.4, 70%'))
    def test_repeated_details_do_not_trigger_missing_slot_questions(self):
        request='from Chennai Central to Tambaram. Four people, SUV, tomorrow 12 PM'
        s,_=self.turn(request,lang='English');s,answer=self.turn(request,s)
        self.assertEqual(s['stage'],'weather_advisory');self.assertEqual(s['count'],4)
        self.assertNotIn('What pickup',answer);self.assertNotIn('How many',answer)
    def test_ambiguous_vehicle_is_clarified_then_resolved(self):
        s,_=self.turn('from Chennai Central to Tambaram. One person, Bike or Mini, tomorrow 12 PM',lang='English')
        self.assertEqual(s['stage'],'vehicle');s,_=self.turn('Mini',s);self.assertEqual(s['stage'],'weather_advisory')
    def test_qwen_cannot_invent_places_or_overwrite_valid_slots(self):
        self.bridge._wx_context_extract=lambda raw,s:{'pickup':'Mars','destination':'Moon','passengers_text':'six'}
        s,_=self.turn('One person wants a bike tomorrow 12 PM',lang='English')
        self.assertFalse(s.get('pickup'));self.assertFalse(s.get('destination'));self.assertEqual(s['count'],1)
        self.assertEqual(s['_wx_slot_evidence']['count']['source'],'rule')


class TwoConsentGuard(Fixtures):
    def test_weather_nonce_cannot_be_redeemed_as_booking_permit(self):
        self.state.pop('_wx_weather_consent',None);self.state['stage']='weather_advisory'
        self.db.stash(self.sid,self.epoch,self.state);nonce=self.db.summary(self.sid,self.epoch,self.state)
        self.db.played(self.sid,self.epoch,nonce)
        with self.assertRaises(wx.GuardError):self.db.permit(self.sid,self.epoch,nonce,self.state)
    def test_first_yes_records_consent_but_creates_no_permit_or_ride(self):
        self.state.pop('_wx_weather_consent',None);self.state['stage']='weather_advisory'
        self.db.stash(self.sid,self.epoch,self.state);nonce=self.db.summary(self.sid,self.epoch,self.state)
        self.db.played(self.sid,self.epoch,nonce);state=self.db.accept_weather(self.sid,self.epoch,nonce,self.state)
        self.assertTrue(wx.weather_consented(state))
        with self.db.connect() as c:
            self.assertEqual(c.execute('SELECT COUNT(*) FROM rides').fetchone()[0],0)
            self.assertEqual(c.execute('SELECT COUNT(*) FROM wx_permits').fetchone()[0],0)
    def test_first_consent_requires_weather_audio_ack(self):
        self.state.pop('_wx_weather_consent',None);self.state['stage']='weather_advisory'
        self.db.stash(self.sid,self.epoch,self.state);nonce=self.db.summary(self.sid,self.epoch,self.state)
        self.assertFalse(self.db.played(self.sid,self.epoch,nonce,explicit_read=True))
        with self.assertRaises(wx.GuardError):self.db.accept_weather(self.sid,self.epoch,nonce,self.state)
    def test_final_review_requires_a_separate_fresh_ack(self):
        self.state.pop('_wx_weather_consent',None);self.state['stage']='weather_advisory'
        self.db.stash(self.sid,self.epoch,self.state);first=self.db.summary(self.sid,self.epoch,self.state)
        self.db.played(self.sid,self.epoch,first);self.db.accept_weather(self.sid,self.epoch,first,self.state)
        self.state['stage']='confirm';self.db.stash(self.sid,self.epoch,self.state);final=self.db.summary(self.sid,self.epoch,self.state)
        with self.assertRaises(wx.GuardError):self.db.permit(self.sid,self.epoch,first,self.state)
        with self.assertRaises(wx.GuardError):self.db.permit(self.sid,self.epoch,final,self.state)
        self.db.played(self.sid,self.epoch,final);self.assertTrue(self.commit(self.db.permit(self.sid,self.epoch,final,self.state))['ok'])
    def test_missing_first_consent_blocks_even_an_acknowledged_final_summary(self):
        self.state.pop('_wx_weather_consent',None);self.db.stash(self.sid,self.epoch,self.state)
        final=self.db.summary(self.sid,self.epoch,self.state);self.db.played(self.sid,self.epoch,final)
        with self.assertRaisesRegex(wx.GuardError,'Weather consent'):self.db.permit(self.sid,self.epoch,final,self.state)
    def test_old_database_schema_is_migrated_without_losing_call_state(self):
        with self.db.connect() as c:
            c.execute('CREATE TABLE old_calls AS SELECT * FROM wx_calls');c.execute('DROP TABLE wx_calls')
            c.execute("CREATE TABLE wx_calls(id TEXT PRIMARY KEY,owner INTEGER,secret_hash TEXT,expires REAL,epoch INTEGER,version INTEGER,state TEXT,summary_nonce TEXT,summary_hash TEXT,summary_expires REAL,played INTEGER,ride_id INTEGER)")
            c.execute('INSERT INTO wx_calls SELECT id,owner,secret_hash,expires,epoch,version,state,summary_nonce,summary_hash,summary_expires,played,ride_id FROM old_calls')
        self.db.setup();self.assertEqual(json.loads(self.db.row(self.sid)['state'])['pickup'],'Chennai Central')
        self.assertEqual(self.db.row(self.sid)['summary_kind'],'final')
    def test_changed_weather_location_cannot_reuse_first_consent(self):
        self.state['destination']='Other destination';self.db.stash(self.sid,self.epoch,self.state)
        final=self.db.summary(self.sid,self.epoch,self.state);self.db.played(self.sid,self.epoch,final)
        with self.assertRaisesRegex(wx.GuardError,'Weather consent'):self.db.permit(self.sid,self.epoch,final,self.state)


class BackendHTTP(unittest.TestCase):
    server_source=None
    @classmethod
    def setUpClass(cls):
        if not cls.server_source:raise unittest.SkipTest('Backend source was not supplied')
        cls.tmp=tempfile.TemporaryDirectory()
        source=pathlib.Path(cls.server_source).read_text(encoding='utf-8').split('# WEATHER_BOOKING_BOOTSTRAP_V1',1)[0]
        cls.name='wx_isolated_backend_fixture'
        cls.backend=types.ModuleType(cls.name);cls.backend.__file__=str(pathlib.Path(cls.tmp.name)/'server.py')
        sys.modules[cls.name]=cls.backend
        exec(compile(source,cls.backend.__file__,'exec'),cls.backend.__dict__)
        cls.backend.DB=pathlib.Path(cls.tmp.name)/'fixture.db';cls.backend.PUBLIC='https://test.invalid';cls.backend.setup()
        with cls.backend.conn() as c:
            for uid,role in ((7,'customer'),(8,'customer'),(9,'driver')):
                c.execute('INSERT INTO users(id,name,phone,password_salt,password_hash,role) VALUES(?,?,?,?,?,?)',
                          (uid,'Fixture '+str(uid),'+9100000000'+str(uid),'0'*32,'unused',role))
                c.execute('INSERT INTO sessions VALUES(?,?,?)',(hashlib.sha256(('fixture-'+str(uid)).encode()).hexdigest(),uid,time.time()+600))
        wx.install_backend(cls.backend,weather=wx.WeatherClient(weather_get(65,95)))
        cls.backend.Handler.log_message=lambda *a:None
        cls.server=ThreadingHTTPServer(('127.0.0.1',0),cls.backend.Handler)
        cls.thread=threading.Thread(target=cls.server.serve_forever,daemon=True);cls.thread.start()
    @classmethod
    def tearDownClass(cls):
        if hasattr(cls,'server'):
            cls.server.shutdown();cls.server.server_close();cls.thread.join(2)
        if hasattr(cls,'backend') and hasattr(cls.backend,'_wx_stop_event'):cls.backend._wx_stop_event.set()
        if hasattr(cls,'tmp'):cls.tmp.cleanup()
    def request(self,path,data=None,uid=7,origin='https://test.invalid'):
        headers={'Content-Type':'application/json','Origin':origin}
        if uid is not None:headers['Cookie']='movio_session=fixture-'+str(uid)
        conn=http.client.HTTPConnection('127.0.0.1',self.server.server_address[1],timeout=5)
        conn.request('POST',path,body=json.dumps(data or {}),headers=headers)
        response=conn.getresponse();status=response.status;result=json.loads(response.read());conn.close();return status,result
    def test_account_and_origin_checks(self):
        self.assertEqual(self.request('/api/weather-booking/session',uid=None)[0],401)
        self.assertEqual(self.request('/api/weather-booking/session',uid=9)[0],403)
        self.assertEqual(self.request('/api/weather-booking/session',origin='https://other.invalid')[0],403)
    def test_authenticated_commit_retry_and_cancel(self):
        status,binding=self.request('/api/weather-booking/session');self.assertEqual(status,200)
        db=wx.GuardDB(self.backend.DB);sid=binding['call_id'];e=db.begin(sid);s=good_state();db.stash(sid,e,s)
        nonce=db.summary(sid,e,s);db.played(sid,e,nonce);token=db.permit(sid,e,nonce,s)
        self.assertEqual(self.request('/api/weather-booking/commit',{'token':token},uid=8)[0],403)
        status,result=self.request('/api/weather-booking/commit',{'token':token});self.assertEqual(status,201)
        status,retry=self.request('/api/weather-booking/commit',{'token':token});self.assertEqual(status,201);self.assertEqual(result['id'],retry['id'])
        self.assertEqual(self.request('/api/weather-booking/cancel',{'ride_id':result['id']},uid=8)[0],409)
        self.assertEqual(self.request('/api/weather-booking/cancel',{'ride_id':result['id']})[0],200)
    def test_running_older_handler_can_be_upgraded_without_nesting(self):
        original=self.backend.Handler._wx_original_post
        self.backend.Handler._wx_version='0.1.0'
        wx.install_backend(self.backend)
        self.assertEqual(self.backend.Handler._wx_version,wx.VERSION)
        self.assertIs(self.backend.Handler._wx_original_post,original)
        self.assertEqual(self.request('/api/weather-booking/session')[0],200)
    def test_manual_preview_does_not_create_a_booking(self):
        status,review=self.request('/api/weather-booking/manual-preview',manual_payload())
        self.assertEqual(status,200);self.assertIn('Heavy rain',review['advisory']);self.assertEqual(review['vehicle'],'Bike')
        with self.backend.conn() as c:self.assertEqual(c.execute('SELECT COUNT(*) FROM rides WHERE voice_call_id=?',(review['call_id'],)).fetchone()[0],0)
    def test_manual_preview_requires_customer_and_same_origin(self):
        for uid in (None,9):self.assertIn(self.request('/api/weather-booking/manual-preview',manual_payload(),uid=uid)[0],(401,403))
        self.assertEqual(self.request('/api/weather-booking/manual-preview',manual_payload(),origin='https://other.invalid')[0],403)
    def test_manual_confirmation_keeps_choices_and_closes_politely(self):
        p=manual_payload('Tamil');status,review=self.request('/api/weather-booking/manual-preview',p)
        self.assertEqual(status,200)
        status,result=self.request('/api/weather-booking/manual-confirm',review)
        self.assertEqual(status,201);self.assertTrue(result['message'].endswith(wx.closing_message('Tamil')))
        with self.backend.conn() as c:ride=c.execute('SELECT * FROM rides WHERE id=?',(result['id'],)).fetchone()
        self.assertEqual((ride['vehicle'],ride['fare'],ride['pickup_time'],ride['source']),('Bike',p['fare'],p['pickup_at'],'weather_web'))
        self.assertEqual(ride['passengers'],1);self.assertEqual(ride['status'],'scheduled')
    def test_manual_confirmation_cannot_use_another_account_or_secret(self):
        _,review=self.request('/api/weather-booking/manual-preview',manual_payload())
        self.assertEqual(self.request('/api/weather-booking/manual-confirm',review,uid=8)[0],403)
        self.assertEqual(self.request('/api/weather-booking/manual-confirm',{**review,'bind_token':'wrong'})[0],403)
    def test_simultaneous_manual_confirmations_create_one_ride(self):
        _,review=self.request('/api/weather-booking/manual-preview',manual_payload())
        with ThreadPoolExecutor(max_workers=6) as pool:
            results=list(pool.map(lambda _:self.request('/api/weather-booking/manual-confirm',review),range(12)))
        self.assertTrue(all(status==201 for status,result in results));self.assertEqual(len({r['id'] for _,r in results}),1)
        with self.backend.conn() as c:self.assertEqual(c.execute('SELECT COUNT(*) FROM rides WHERE voice_call_id=?',(review['call_id'],)).fetchone()[0],1)
    def test_discarded_manual_review_cannot_be_confirmed(self):
        _,review=self.request('/api/weather-booking/manual-preview',manual_payload())
        self.assertEqual(self.request('/api/weather-booking/manual-discard',review)[0],200)
        self.assertEqual(self.request('/api/weather-booking/manual-confirm',review)[0],409)
    def test_past_manual_time_is_rejected_without_moving_date(self):
        p=manual_payload('Tamil');p['pickup_at']='2025-09-15T19:00:00+05:30'
        status,error=self.request('/api/weather-booking/manual-preview',p)
        self.assertEqual(status,409);self.assertEqual(error['code'],'past_datetime');self.assertIn('கடந்துவிட்டது',error['error'])
    def test_unavailable_future_forecast_does_not_force_a_date_change(self):
        p=manual_payload('Tanglish');p['pickup_at']=(datetime.now(wx.TZ)+timedelta(days=30)).replace(second=0,microsecond=0).isoformat()
        status,review=self.request('/api/weather-booking/manual-preview',p)
        self.assertEqual(status,200);self.assertEqual(review['weather_status'],'outside_forecast_range');self.assertEqual(review['pickup_at'],p['pickup_at'])
        status,result=self.request('/api/weather-booking/manual-confirm',review)
        self.assertEqual(status,201);self.assertTrue(result['message'].endswith(wx.closing_message('Tanglish')))
    def test_payload_cannot_replace_reviewed_vehicle_on_confirmation(self):
        _,review=self.request('/api/weather-booking/manual-preview',manual_payload())
        status,result=self.request('/api/weather-booking/manual-confirm',{**review,'vehicle':'SUV','fare':1})
        self.assertEqual(status,201)
        with self.backend.conn() as c:ride=c.execute('SELECT * FROM rides WHERE id=?',(result['id'],)).fetchone()
        self.assertEqual(ride['vehicle'],'Bike');self.assertEqual(ride['fare'],168)
    def test_manual_forecast_request_does_not_hold_the_website_lock(self):
        entered,release=threading.Event(),threading.Event();previous=self.backend._wx_weather_client
        class Delayed:
            def forecast(self,*args,**kwargs):
                entered.set();release.wait(3);return {'status':'unavailable'}
        self.backend._wx_weather_client=Delayed()
        try:
            with ThreadPoolExecutor(max_workers=2) as pool:
                forecast=pool.submit(self.request,'/api/weather-booking/manual-preview',manual_payload())
                self.assertTrue(entered.wait(1))
                other=pool.submit(self.request,'/api/weather-booking/session')
                try:self.assertEqual(other.result(timeout=1)[0],200)
                finally:release.set()
                self.assertEqual(forecast.result()[0],200)
        finally:release.set();self.backend._wx_weather_client=previous
    def test_manual_scheduled_ride_uses_existing_release_and_cancel(self):
        _,review=self.request('/api/weather-booking/manual-preview',manual_payload())
        _,result=self.request('/api/weather-booking/manual-confirm',review)
        with self.backend.conn() as c:c.execute('UPDATE rides SET pickup_time=? WHERE id=?',((datetime.now(wx.TZ)+timedelta(minutes=10)).isoformat(),result['id']))
        wx.GuardDB(self.backend.DB).release_due()
        with self.backend.conn() as c:self.assertEqual(c.execute('SELECT status FROM rides WHERE id=?',(result['id'],)).fetchone()[0],'requested')


def _run_tests(server_source=None):
    BackendHTTP.server_source=server_source
    suite=unittest.defaultTestLoader.loadTestsFromNames([__name__+'.'+name for name in
        ('Dates','Weather','CustomerAdvice','Confirmation','TurnPreparation','SocketRaces','CallPage','BackendHTTP','VoiceAcceptance','TwoConsentGuard')])
    result=unittest.TextTestRunner(verbosity=1).run(suite)
    return {'version':wx.VERSION,'tests_run':result.testsRun,'failures':len(result.failures),'errors':len(result.errors),
            'passed':result.wasSuccessful(),'scope':'offline weather/model stubs + real SQLite + async socket protocol; isolated HTTP when source supplied'}


def run_tests(server_source=None):
    """Keep asyncio.run() scenarios off a notebook's already-running loop."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return _run_tests(server_source)
    with ThreadPoolExecutor(max_workers=1, thread_name_prefix='weather-regression') as pool:
        return pool.submit(_run_tests, server_source).result()


if __name__=='__main__':
    print(json.dumps(run_tests(sys.argv[1] if len(sys.argv)>1 else None),indent=2))


Writing /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/test_weather_guard.py


## 6. Run the automated checks
Expected: zero failures and zero errors. Checks cover the supplied Tamil conversation, retained date/time parts, spoken dates with numeric values, rain/heat/vehicle rules, selected-language closings, manual preview without submission, final confirmation, concurrent retries, account ownership, stale reviews, forecast failures/cache use, streaming interruption and acknowledgement. They do not measure real speech quality, live weather accuracy or browser latency.


In [71]:
import importlib.util, sys, json
from concurrent.futures import ThreadPoolExecutor

for wx_name in ('movio_weather_guard','test_weather_guard'):
    wx_path=WX_ROOT/'integrations'/(wx_name+'.py')
    wx_spec=importlib.util.spec_from_file_location(wx_name,wx_path)
    wx_loaded=importlib.util.module_from_spec(wx_spec)
    sys.modules[wx_name]=wx_loaded
    wx_spec.loader.exec_module(wx_loaded)
# Kaggle/IPython already runs an event loop. Run the suite in a worker thread.
with ThreadPoolExecutor(max_workers=1) as wx_executor:
    WX_TESTS=wx_executor.submit(
        sys.modules['test_weather_guard'].run_tests,
        server_source=WX_ROOT/'server.py',
    ).result()
print(json.dumps(WX_TESTS,indent=2))
if not WX_TESTS['passed']:
    raise RuntimeError('Do not activate the integration until these regression checks pass.')


..................................................................................................
----------------------------------------------------------------------
Ran 98 tests in 5.446s

OK


{
  "version": "0.4.0",
  "tests_run": 98,
  "failures": 0,
  "errors": 0,
  "passed": true,
  "scope": "offline weather/model stubs + real SQLite + async socket protocol; isolated HTTP when source supplied"
}


## 7. Activate in the current application
This patches the current voice route and dashboard and installs a backend guard. The website-only process may be restarted if it runs outside the notebook; model weights are not reloaded. Start a fresh call afterward. Later rerunning an original bridge, website installation or context patch can replace this integration; rerun this activation after those cells.

Future bookings are stored as `scheduled` and become driver requests 20 minutes before pickup while the backend runs. Use the dashboard's Scheduled rides card to cancel a future booking. Kaggle sessions are temporary; use an always-on backend for genuine future dispatch. There is no automatic weather-based rescheduling or fare surcharge.


In [72]:
if not globals().get('WX_TESTS',{}).get('passed') or WX_TESTS.get('version') != '0.4.0':
    raise RuntimeError('Run and pass the weather regression-check cell first.')

"""Run in the original Kaggle kernel after its context-aware dialogue cells."""
import gc
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
import weakref
from datetime import datetime
from http.server import ThreadingHTTPServer
from pathlib import Path

WX_ROOT = Path('/kaggle/working/MOVIO_MAIN_WEBSITE')
WX_SOURCE = WX_ROOT / 'integrations' / 'movio_weather_guard.py'
if globals().get('bridge') is None or not getattr(bridge, 'TTS', None):
    raise RuntimeError('Run the original model and voice-bridge setup in this kernel first. Do not run the evaluation cells yet.')
if not callable(globals().get('context_handle')):
    raise RuntimeError('Run the original context-aware booking cell first, then run this integration.')
for wx_file in (WX_SOURCE, WX_ROOT / 'server.py', WX_ROOT / 'integrations/weather_call_page.html', WX_ROOT / 'web/weather_dashboard.js',WX_ROOT/'web/weather_manual.js'):
    if not wx_file.is_file():
        raise FileNotFoundError(str(wx_file))

# The original server, web assets, model setup and database remain available.
wx_existing_backup = globals().get('WX_BACKUP')
if wx_existing_backup and Path(wx_existing_backup).is_dir() and Path(wx_existing_backup).parent == WX_ROOT:
    WX_BACKUP = Path(wx_existing_backup)
else:
    WX_BACKUP = WX_ROOT / ('weather_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    WX_BACKUP.mkdir(exist_ok=False)
for wx_relative in ('server.py', 'web/index.html', 'integrations/ai_call_page.html'):
    wx_file = WX_ROOT / wx_relative
    if wx_file.is_file():
        wx_dest = WX_BACKUP / wx_relative
        wx_dest.parent.mkdir(parents=True, exist_ok=True)
        if not wx_dest.exists():
            shutil.copy2(wx_file, wx_dest)

wx_spec = importlib.util.spec_from_file_location('movio_weather_guard', WX_SOURCE)
wx_module = importlib.util.module_from_spec(wx_spec)
sys.modules[wx_spec.name] = wx_module
wx_spec.loader.exec_module(wx_module)
WX_FORECAST_CLIENT=wx_module.WeatherClient(rain_threshold=globals().get('WX_RAIN_THRESHOLD',60),
    heat_c=globals().get('WX_HEAT_C',38),feels_like_c=globals().get('WX_FEELS_LIKE_C',40))

wx_server_path = WX_ROOT / 'server.py'
wx_server_text = wx_server_path.read_text(encoding='utf-8')
wx_marker = '# WEATHER_BOOKING_BOOTSTRAP_V1'
wx_end_marker = '# WEATHER_BOOKING_BOOTSTRAP_END_V2'
wx_main = "if __name__=='__main__':"
if wx_main not in wx_server_text or 'class Handler(' not in wx_server_text:
    raise RuntimeError('This installer expects the original MOVIO v5.3 server. No server replacement was made.')
# Upgrade the older unguarded bootstrap too. Evaluation imports server.py into
# an isolated module before assigning its temporary ROOT/DB; never install the
# production guard during that import.
if wx_marker in wx_server_text:
    wx_start=wx_server_text.index(wx_marker)
    if wx_end_marker in wx_server_text[wx_start:]:
        wx_end=wx_server_text.index(wx_end_marker,wx_start)+len(wx_end_marker)
        wx_server_text=wx_server_text[:wx_start]+wx_server_text[wx_end:].lstrip('\n')
    else:
        wx_end=wx_server_text.index(wx_main,wx_start)
        wx_server_text=wx_server_text[:wx_start]+wx_server_text[wx_end:]
wx_bootstrap = '''# WEATHER_BOOKING_BOOTSTRAP_V1
if __name__ == '__main__':
    import sys as _wx_sys
    import importlib.util as _wx_import
    _wx_spec = _wx_import.spec_from_file_location('movio_weather_guard', ROOT / 'integrations/movio_weather_guard.py')
    _wx_mod = _wx_import.module_from_spec(_wx_spec)
    _wx_sys.modules[_wx_spec.name] = _wx_mod
    _wx_spec.loader.exec_module(_wx_mod)
    _wx_mod.install_backend(_wx_sys.modules[__name__])
# WEATHER_BOOKING_BOOTSTRAP_END_V2

'''
wx_server_text = wx_server_text.replace(wx_main, wx_bootstrap + wx_main, 1)
compile(wx_server_text, str(wx_server_path), 'exec')
wx_server_path.write_text(wx_server_text, encoding='utf-8')

wx_index = WX_ROOT / 'web/index.html'
wx_html = wx_index.read_text(encoding='utf-8')
if 'weather_dashboard.js' not in wx_html:
    if '</body>' not in wx_html:
        raise RuntimeError('Dashboard HTML has no closing body tag.')
    wx_html = wx_html.replace('</body>', '<script src="/weather_dashboard.js" defer></script>\n</body>')
wx_html = re.sub(r'/weather_dashboard\.js(?:\?v=[0-9.]+)?',
                 '/weather_dashboard.js?v=' + wx_module.VERSION, wx_html)
if 'weather_manual.js' not in wx_html:
    wx_html=wx_html.replace('</body>','<script src="/weather_manual.js" defer></script>\n</body>')
wx_html=re.sub(r'/weather_manual\.js(?:\?v=[0-9.]+)?','/weather_manual.js?v='+wx_module.VERSION,wx_html)
wx_index.write_text(wx_html, encoding='utf-8')
shutil.copy2(WX_ROOT / 'integrations/weather_call_page.html', WX_ROOT / 'integrations/ai_call_page.html')
bridge.HTML = (WX_ROOT / 'integrations/ai_call_page.html').read_text(encoding='utf-8')

# Upgrade the active website handler without restarting the models.
wx_in_kernel = []
for wx_obj in gc.get_objects():
    try:
        # type() does not dereference a weakref.proxy's expired target.
        if type(wx_obj) in weakref.ProxyTypes:
            continue
        if not isinstance(wx_obj, ThreadingHTTPServer):
            continue
        if wx_obj.server_address[1] != 8080:
            continue
        wx_known_root = wx_obj.RequestHandlerClass.__init__.__globals__.get('ROOT')
        if wx_known_root and Path(wx_known_root).resolve() == WX_ROOT.resolve():
            wx_in_kernel.append(wx_obj)
    except (ReferenceError, AttributeError, TypeError, IndexError):
        # Unrelated, expired or partially finalized objects are not servers.
        continue
if wx_in_kernel:
    for wx_httpd in wx_in_kernel:
        wx_globals = wx_httpd.RequestHandlerClass.__init__.__globals__
        wx_backend = sys.modules.get(wx_globals.get('__name__'))
        if wx_backend is None:
            raise RuntimeError('Could not identify the running website module.')
        wx_backend.PUBLIC = bridge.PUBLIC_URL
        wx_module.install_backend(wx_backend,weather=WX_FORECAST_CLIENT)
else:
    import psutil
    wx_listeners = sorted({c.pid for c in psutil.net_connections(kind='inet') if c.status == 'LISTEN' and c.laddr.port == 8080 and c.pid})
    wx_env = dict(os.environ)
    for wx_pid in wx_listeners:
        if wx_pid == os.getpid():
            raise RuntimeError('An unrecognized in-kernel service owns port 8080. Use your original website-only launcher.')
        wx_proc = psutil.Process(wx_pid)
        wx_scripts = [Path(a) for a in wx_proc.cmdline() if a.endswith('.py')]
        if not any((Path(wx_proc.cwd()) / p).resolve() == wx_server_path.resolve() for p in wx_scripts):
            raise RuntimeError('A different application owns port 8080; no process was stopped.')
        wx_env.update(wx_proc.environ())
        wx_proc.terminate()
        wx_proc.wait(timeout=8)
    wx_env['MOVIO_PUBLIC_URL'] = bridge.PUBLIC_URL
    wx_env['MOVIO_PORT'] = '8080'
    wx_env['MOVIO_RAIN_THRESHOLD']=str(globals().get('WX_RAIN_THRESHOLD',60))
    wx_env['MOVIO_HEAT_C']=str(globals().get('WX_HEAT_C',38))
    wx_env['MOVIO_FEELS_LIKE_C']=str(globals().get('WX_FEELS_LIKE_C',40))
    with (WX_ROOT / 'weather_website.log').open('a') as wx_log:
        WX_WEBSITE_PROCESS = subprocess.Popen([sys.executable, str(wx_server_path)], cwd=WX_ROOT,
            env=wx_env, stdout=wx_log, stderr=subprocess.STDOUT, start_new_session=True)

# Keep Qwen as an optional extraction fallback; dates use the full utterance.
bridge._wx_context_extract = globals().get('context_extract')
bridge._wx_pause_ms=globals().get('WX_SENTENCE_PAUSE_MS',220)
WX_STATUS = wx_module.install_bridge(bridge, WX_ROOT / 'movio_demo.db',
    weather=WX_FORECAST_CLIENT)
print(json.dumps(WX_STATUS, indent=2))
print('Backup:', WX_BACKUP)
print('Refresh the dashboard. Close the old call popup and open a new AI Call.')
print('Use Interrupt / change details before speaking over the assistant.')
print('Scheduled rides are released 20 minutes before pickup only while the backend is running.')


{
  "version": "0.4.0",
  "weather": "Open-Meteo",
  "guard": "installed",
  "interrupt": "browser control"
}
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/weather_backup_20261028_070655_509041
Refresh the dashboard. Close the old call popup and open a new AI Call.
Use Interrupt / change details before speaking over the assistant.
Scheduled rides are released 20 minutes before pickup only while the backend is running.


## 8. Check backend installation and a real forecast
The health check verifies the new endpoint. The forecast probe is one live request for tomorrow at 9 AM at Chennai Central and the airport. This is an API connectivity check, not a booking or a weather-accuracy evaluation. Set `WX_RUN_LIVE_FORECAST=False` to skip it.


In [73]:
import requests, time
from datetime import datetime, timedelta
wx_http=requests.Session();wx_http.trust_env=False
for wx_attempt in range(12):
    try:
        wx_health=wx_http.get('http://127.0.0.1:8080/api/weather-booking/health',timeout=3)
        wx_health.raise_for_status()
        if wx_health.json().get('version') != wx_module.VERSION:
            raise RuntimeError('The website is running an older weather module. Rerun cell 7, then this health check.')
        print('Weather booking backend:',wx_health.json())
        break
    except requests.RequestException:
        if wx_attempt==11:raise RuntimeError('Backend not ready. Check '+str(WX_ROOT/'weather_website.log'))
        time.sleep(.5)

WX_RUN_LIVE_FORECAST=True
if WX_RUN_LIVE_FORECAST:
    wx_when=(datetime.now(wx_module.TZ)+timedelta(days=1)).replace(hour=9,minute=0,second=0,microsecond=0)
    WX_LIVE_FORECAST=wx_module.WeatherClient(rain_threshold=WX_RAIN_THRESHOLD).forecast(
        [{'lat':13.0827,'lng':80.2707},{'lat':12.9941,'lng':80.1709}],wx_when)
    print('Forecast for:',wx_when.isoformat())
    print(json.dumps(WX_LIVE_FORECAST,ensure_ascii=False,indent=2))
    if WX_LIVE_FORECAST['status']!='ok':
        print('Live forecast unavailable. Check Kaggle Internet and Open-Meteo access. The app will state that the forecast is unavailable.')
else:
    print('Live forecast probe skipped.')
print('Refresh the dashboard, then open a NEW AI Call popup.')


Weather booking backend: {'ok': True, 'version': '0.4.0', 'weather': 'Open-Meteo', 'guard': 'sqlite_atomic_revision'}
Forecast for: 2026-10-29T09:00:00+05:30
{
  "status": "ok",
  "condition": "clear",
  "points": [
    {
      "location": "pickup",
      "valid_at": "2026-10-29T09:00",
      "weather_code": 1,
      "precipitation_probability": 0.0,
      "precipitation": 0.0,
      "temperature_2m": 28.9,
      "apparent_temperature": 32.1
    },
    {
      "location": "destination",
      "valid_at": "2026-10-29T09:00",
      "weather_code": 1,
      "precipitation_probability": 0.0,
      "precipitation": 0.0,
      "temperature_2m": 28.9,
      "apparent_temperature": 32.9
    }
  ],
  "fetched_at": 1793171216.7421017,
  "source": "Open-Meteo",
  "heavy_rain": false,
  "thunderstorm": false,
  "heat": false,
  "pleasant": true,
  "advisory_thresholds": {
    "rain_probability_pct": 60,
    "heat_c": 38.0,
    "feels_like_c": 40.0
  },
  "scope": "pickup_and_destination_at_pickup_

## 9. Live checks to perform in your browser

1. Refresh the signed-in dashboard. In manual booking, choose pickup/drop using the existing map or address suggestions, choose a vehicle, and select the new language, date/time and passenger fields. The Check weather and review booking button must show advice without creating a ride. Continue with this booking must save one ride and show the polite closing. Check tomorrow's date is retained. Changing vehicle/date after review must require a new review. A bike in heavy rain must stay a bike if the customer continues.
2. Open a new AI Call popup. Choose English, Tamil or Tanglish.
3. Say “நான் Chennai Central-இல் இருந்து Tambaram செல்ல வேண்டும் நாங்கள் நான்கு பேர் இருக்கிறோம்”. Then say “எனக்கு SUV வேண்டும் இன்று இரவு 07:00 மணிக்கு எனக்கு cab வேணும்”. Before 7 PM, it should retain the details and proceed to the summary. After 7 PM, it should explain that the time has passed and ask for a later time by voice, retaining today's date. Use a time that is still in the future when you perform the test.
4. Check separate spoken pickup/drop rain probabilities, condition-specific precautions, the named month with numeric day/year and time and two separate confirmation questions. These weather details stay out of the voice popup's text log. A missing probability is stated as unavailable. Clear weather must not be forced into a rain example; offline checks exercise rain/heat cases when the live forecast differs.
5. Give a vehicle name to change it, or say yes to continue with the selected vehicle. Locations, passengers and date/time must be retained. Listen for short pauses between sentences.
6. While the assistant is speaking or pausing, click Interrupt / change details. Give a different date or vehicle. Old queued clips must stop. In-flight GPU/HTTP work may finish internally; its obsolete result is discarded.
7. Listen to the complete summary and say “ஆம்” or yes. Verify one booking on your own account and a closing in the selected language. The optional I have read the booking summary control acknowledges visible booking details if audio fails; it does not establish that the spoken weather report was heard.
8. A saved booking is the submission boundary. A later interruption does not undo it; cancel before replacing it. Cancel future bookings using their Scheduled rides card. Existing immediate-ride voice cancellation retains the original flow.

Record these as **live tests pending** until you actually perform them. The previous notebook's ASR/TTS/backend result tables describe the previous execution and do not automatically validate this new feature.

## Limits and source references

- The API supplies forecasts for dates within its available horizon, up to 16 days. Later dates and missing responses produce a forecast-unavailable message.
- Clear-sky comfort wording uses a simple 18–32°C rule; heat precautions use the configurable temperature/feels-like thresholds. These rules are customer-care heuristics, not official alerts or validated comfort models. Rain probability and heavy-rain intensity are different quantities.
- The rules give weather-protection suggestions; they do not predict floods, guarantee safe roads, estimate traffic delays or alter fares because of rainfall. Covered options are capacity-filtered Mini, Sedan and SUV; availability is determined by your existing driver workflow.
- The parser supports common English/Tamil/Tanglish date and clock expressions, numeric times, weekdays and selected named dates. Unsupported or ambiguous expressions produce a voice clarification for the missing part. Spoken input is not guaranteed to recognize every possible date expression. An optional typed reply can also give an exact date, for example `2026-10-20 21:00`.
- Relative dates use the server clock in Asia/Kolkata internally; the assistant does not announce a timezone. The example “September 15, 2025” describes the output format and is not a fixed booking date.
- Sentence streaming changes the voice delivery path. Earlier latency and human-quality measurements do not validate this new version; measure it again before making performance claims. The existing TTS engine and selected-language pronunciation wrappers remain in use.
- Manual estimates reuse the original website's fare and route calculations. Weather does not add a surcharge or traffic delay. Place names and vehicle identifiers can remain proper nouns in the customer's supplied form.
- This prototype checks pickup and destination at pickup time. Full-route exposure modelling is a separate future feature.
- Automatic regression results are **not human intelligibility or weather accuracy results**. Playback acknowledgement does not prove that a person understood the summary.
- The database stores journey versions, review acknowledgements and the forecast used for each new weather-aware booking.
- Open-Meteo documentation: https://open-meteo.com/en/docs
- Open-Meteo access terms: https://open-meteo.com/en/pricing

## Recovery

The activation cell prints `WX_BACKUP`, which contains the original server, dashboard HTML and call page. To undo the integration, restore those files, restart the website using your original website-only launch cell, and restore the voice route with the optional cell below. Cancel new scheduled rides first if you intend to stop the scheduling worker. The added database columns and booking records are deliberately retained.


In [74]:
# Optional recovery: leave False during normal operation.
WX_RESTORE_ORIGINAL=False
if WX_RESTORE_ORIGINAL:
    import shutil
    from starlette.routing import websocket_session
    if not globals().get('WX_BACKUP') or not WX_BACKUP.is_dir():
        raise RuntimeError('Set WX_BACKUP to the backup directory printed during activation.')
    for wx_relative in ('server.py','web/index.html','integrations/ai_call_page.html'):
        shutil.copy2(WX_BACKUP/wx_relative,WX_ROOT/wx_relative)
    for wx_backend in list(sys.modules.values()):
        if getattr(wx_backend,'_wx_stop_event',None) is not None:
            wx_backend._wx_stop_event.set()
    wx_route=next(r for r in bridge.app.routes if getattr(r,'path','')=='/browser-call')
    wx_route.endpoint=bridge._wx_original_socket
    wx_route.app=websocket_session(bridge._wx_original_socket)
    bridge.HTML=(WX_ROOT/'integrations/ai_call_page.html').read_text(encoding='utf-8')
    print('Original files and voice route restored. Rerun your original WEBSITE-ONLY launch cell and refresh the browser.')
else:
    print('Recovery skipped; integration remains active.')


Recovery skipped; integration remains active.


## Repair AI Call login detection — also usable in the current session

The account-bound weather-booking API verifies login before microphone access. Booking no longer depends on a successful popup handshake. The existing dashboard channel also gains a same-origin BroadcastChannel fallback for browsers that detach the popup opener. Cancellation still uses its dashboard channel; origin and nonce checks are retained.

This cell is safe to rerun after activation. If you already launched an older notebook, copy just this cell into that running kernel. Close the old call popup, run the cell, refresh your signed-in dashboard and open a new AI Call. No model restart is needed.


In [75]:
# Run in the current Kaggle kernel after weather activation. Close old call popups first.
from pathlib import Path
from datetime import datetime
import shutil
import subprocess
import requests

root = Path('/kaggle/working/MOVIO_MAIN_WEBSITE')
if globals().get('bridge') is None:
    raise RuntimeError('Run the original AI voice setup first.')
with requests.Session() as client:
    client.trust_env = False
    response = client.get('http://127.0.0.1:8080/api/weather-booking/health', timeout=5)
    if response.status_code != 200 or response.json().get('guard') != 'sqlite_atomic_revision':
        raise RuntimeError('Run the weather activation cell first; its authenticated booking API is required.')

CHANNEL_FIX = r"""// Same-origin fallback for browsers that detach a popup's window.opener.
const voiceChannel=(()=>{try{return typeof BroadcastChannel==='function'?new BroadcastChannel('movio-ai-booking'):null;}catch(e){return null;}})();
let parentTransport=null,helloAttempts=0;
function notify(type,data={}){const message={channel:'ai-ride-call',nonce,type,...data};let sent=false;
 if(parentTransport!=='broadcast'&&window.opener&&!window.opener.closed){try{window.opener.postMessage(message,origin);sent=true;}catch(e){}}
 if(voiceChannel&&(type==='hello'||!sent)){try{voiceChannel.postMessage(message);}catch(e){}}
}
function handleParentMessage(m,transport){if(!nonce||!m||m.channel!=='ai-ride-call'||m.nonce!==nonce)return;
 if(m.type==='ready'){ready=true;if(transport==='opener'||!parentTransport)parentTransport=transport;status('Ready');}
 if(m.type==='ready'||m.type==='rides'){const rides=m.rides||[];$('rides').textContent=mode==='cancel'?rides.map((r,i)=>`${i+1}. #${r.id}: ${r.pickup} → ${r.destination}`).join('\n'):'';if(socket?.readyState===1&&mode==='cancel')send({event:'active_rides',rides});}
 if(m.type==='result'&&socket?.readyState===1)send({event:'result',ok:!!m.ok,id:m.id,error:m.error,request_id:m.request_id});
}
window.addEventListener('message',e=>{if(e.origin!==origin||e.source!==window.opener)return;handleParentMessage(e.data,'opener');});
if(voiceChannel)voiceChannel.onmessage=e=>handleParentMessage(e.data,'broadcast');
function requestDashboard(){if(!nonce||ready)return;notify('hello');if(++helloAttempts<10)setTimeout(requestDashboard,500);}
if(nonce)requestDashboard();
"""
OLD_GUARD = "if(!ready||!nonce){status('Open AI Call from your signed-in dashboard.');return;}"
NEW_GUARD = "if(mode==='cancel'&&(!ready||!nonce)){notify('hello');status('Reconnect using the dashboard AI Call button to load your bookings.');return;}"

updates = []
for name in ('weather_call_page.html', 'ai_call_page.html'):
    path = root / 'integrations' / name
    if not path.is_file():
        raise FileNotFoundError(str(path))
    original = path.read_text(encoding='utf-8')
    text = original
    if OLD_GUARD in text:
        text = text.replace(OLD_GUARD, NEW_GUARD, 1)
    elif NEW_GUARD not in text:
        raise RuntimeError('Unexpected call-page version; no files were changed.')
    if 'const voiceChannel=' not in text:
        begin = text.index('function notify(type,data={})')
        end = text.index('function send(m)', begin)
        text = text[:begin] + CHANNEL_FIX + text[end:]
    if "fetch('/api/weather-booking/session'" not in text:
        raise RuntimeError('Authenticated session check is missing; no files were changed.')
    updates.append((path, original, text))

backup = root / ('call_login_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
backup.mkdir()
node = shutil.which('node')
if node:
    for path, original, text in updates:
        check = backup / (path.name + '.check.js')
        check.write_text(text.split('<script>', 1)[1].split('</script>', 1)[0], encoding='utf-8')
        subprocess.run([node, '--check', str(check)], check=True)
for path, original, text in updates:
    shutil.copy2(path, backup / path.name)
    path.write_text(text, encoding='utf-8')
bridge.HTML = (root / 'integrations' / 'ai_call_page.html').read_text(encoding='utf-8')
print('AI Call login detection repaired. Models and backend session checks are retained.')
print('Close the old call popup, refresh the signed-in dashboard, and open a new AI Call.')
print('Backup:', backup)


AI Call login detection repaired. Models and backend session checks are retained.
Close the old call popup, refresh the signed-in dashboard, and open a new AI Call.
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/call_login_backup_20261028_070656_778617


## Check numerical speech text with the actual notebook normalizer

These checks exercise your original normalization code with the new voice-only correction. They do not synthesize audio or call an external service. Dialogue text and raw TTS-adapter input keep numerical digits; the acoustic normalization layer verbalizes those entities for pronunciation. The existing evaluation code and its normalizer behavior outside the new voice route stay unchanged.


In [76]:
wx_original_normalizer = getattr(normalize_taxi_text, '_wx_original_normalizer', normalize_taxi_text)
WX_NUMERIC_TEXT_CHECKS=[]
for wx_language,wx_text in (
    ('Tamil','4 பேர், 35.4 கிலோமீட்டர், ₹1,126, 70%, முன்பதிவு எண் 48291.'),
    ('Tanglish','4 peru, 35.4 kilo meettar, ₹1,126, 70%, munpadhivu enn 48291.'),
    ('English','4 passengers, 35.4 kilometres, ₹1,126, 70%, booking number 48291.'),
):
    wx_normalized=wx_module.speech_numeric_normalize(wx_original_normalizer,wx_text,wx_language)
    wx_decimal,wx_percent=('point','percent') if wx_language=='English' else ('புள்ளி','சதவீதம்')
    assert wx_decimal in wx_normalized and wx_percent in wx_normalized
    assert not re.search(r'[\uE000-\uE0FF]',wx_normalized)
    WX_NUMERIC_TEXT_CHECKS.append(dict(language=wx_language,passed=True,raw_tts_text=wx_text,acoustic_text=wx_normalized))
    print(wx_language+': decimal, percentage, currency and booking-ID text checked.')
print('3 numerical-text checks passed. These are not listening ratings.')


Tamil: decimal, percentage, currency and booking-ID text checked.
Tanglish: decimal, percentage, currency and booking-ID text checked.
English: decimal, percentage, currency and booking-ID text checked.
3 numerical-text checks passed. These are not listening ratings.


## Browser acceptance check — required before reporting live success

Use the existing signed-in customer dashboard and select Tamil. Say: “நான் Chennai Central-இல் இருந்து Tambaram செல்ல வேண்டும். நான்கு பேர். SUV வேண்டும். இன்று மதியம் 12 மணிக்கு.” Use a future time if noon has passed. The assistant should ask for weather consent first. After the first Yes it should give distance, estimated fare and a separate final question. After the second Yes, check that exactly one account-linked booking and pickup OTP appear.

Repeat in English and Tanglish. Before final Yes, say “No, change my bike to mini car”; check that only the vehicle/fare change and the other slots remain. Use spoken dates such as “20th October” and “அக்டோபர் இருபதாம் தேதி.” Test route failure/retry in the offline regression suite; do not disconnect the live notebook while customers are booking.

Offline tests use weather/model stubs and a temporary database. They establish code behavior for the supplied cases, not live STT accuracy, acoustic pronunciation quality, API availability or browser end-to-end latency. Keep those claims separate from the original evaluation results.


## Evaluation compatibility setup — separate from the unchanged tests

The existing B/C voice harness expects the original unauthenticated test protocol. The public weather-booking route now requires an account-bound session, playback acknowledgement and an immutable confirmation token. This helper runs the original voice handler on a private random loopback port for the existing tests. It does not expose that endpoint through the public gateway or change production settings.

Leave the original `EV_E2E_ALSO_PUBLIC=False` setting for this legacy harness. A future test of the new public weather protocol needs an updated harness; its results must be reported separately.


In [77]:
# Compatibility setup for the UNCHANGED B/C evaluation protocol.
# Production /browser-call and account/weather checks are not replaced.
import asyncio as wx_eval_asyncio
import json as wx_eval_json
import threading as wx_eval_threading
import types as wx_eval_types
from urllib.parse import urlparse as wx_eval_urlparse, parse_qs as wx_eval_parse_qs
import websockets as wx_eval_websockets
from websockets.exceptions import ConnectionClosed as WxEvalConnectionClosed

def wx_stop_eval_socket(state):
    if not state:
        return
    loop,stop=state.get('loop'),state.get('stop')
    if loop is not None and stop is not None and loop.is_running():
        loop.call_soon_threadsafe(stop.set)
    thread=state.get('thread')
    if thread is not None:
        thread.join(timeout=5)

def wx_start_eval_socket(original_bridge):
    original=original_bridge.browser_call
    namespace=dict(original.__globals__)
    namespace.update(SESSIONS={},PUBLIC_URL=original_bridge.PUBLIC_URL)
    endpoint=wx_eval_types.FunctionType(original.__code__,namespace,original.__name__,original.__defaults__,original.__closure__)
    endpoint.__annotations__=dict(getattr(original,'__annotations__',{}))
    endpoint.__kwdefaults__=getattr(original,'__kwdefaults__',None)
    state={'ready':wx_eval_threading.Event(),'scope':'original_legacy_voice_components_not_weather_confirmation'}

    class SocketAdapter:
        def __init__(self,connection,path=None):
            self.connection=connection
            request=getattr(connection,'request',None)
            self.headers=getattr(request,'headers',None) or getattr(connection,'request_headers',{})
            path=path or getattr(request,'path',None) or getattr(connection,'path','')
            self.path=wx_eval_urlparse(path).path
            self.query_params={k:v[-1] for k,v in wx_eval_parse_qs(wx_eval_urlparse(path).query).items()}
        async def accept(self):
            pass  # Native websockets has already completed the handshake.
        async def close(self,code=1000):
            await self.connection.close(code=code)
        async def send_json(self,value):
            await self.connection.send(wx_eval_json.dumps(value,ensure_ascii=False))
        async def receive_json(self):
            try:
                return wx_eval_json.loads(await self.connection.recv())
            except WxEvalConnectionClosed:
                raise original_bridge.WebSocketDisconnect(code=1000)

    async def handler(connection,path=None):
        adapter=SocketAdapter(connection,path)
        if adapter.path!='/browser-call':
            await adapter.close(code=1008)
            return
        await endpoint(adapter)

    async def run():
        state['loop']=wx_eval_asyncio.get_running_loop()
        state['stop']=wx_eval_asyncio.Event()
        try:
            async with wx_eval_websockets.serve(handler,'127.0.0.1',0,max_size=10_000_000) as server:
                state['url']='ws://127.0.0.1:'+str(server.sockets[0].getsockname()[1])+'/browser-call'
                state['ready'].set()
                await state['stop'].wait()
        except Exception as exc:
            state['error']=exc
            state['ready'].set()

    thread=wx_eval_threading.Thread(target=lambda:wx_eval_asyncio.run(run()),daemon=True,name='original-evaluation-socket')
    state['thread']=thread
    thread.start()
    if not state['ready'].wait(timeout=10):
        raise RuntimeError('The private evaluation socket did not start.')
    if state.get('error') is not None:
        raise RuntimeError('Private evaluation socket failed.') from state['error']
    return state

wx_old_eval=globals().get('WX_EVAL_COMPAT')
wx_prior_url=wx_old_eval.get('url') if wx_old_eval else None
wx_stop_eval_socket(wx_old_eval)
wx_configured_eval_url=globals().get('EV_WS_LOAD_URL')
if wx_configured_eval_url in (None,'','ws://127.0.0.1:8000/browser-call',wx_prior_url):
    WX_EVAL_COMPAT=wx_start_eval_socket(bridge)
    EV_WS_LOAD_URL=WX_EVAL_COMPAT['url']
    print('Unchanged B/C voice tests: private original-protocol endpoint ready.')
else:
    WX_EVAL_COMPAT=None
    print('Your explicitly configured EV_WS_LOAD_URL was retained.')
print('Scope: original ASR/dialogue/TTS evaluation. Weather confirmation and sentence streaming use the separate WX regression checks above.')
print('The public weather-booking route remains active. The old optional public-gateway harness does not implement the new authenticated weather protocol.')


Unchanged B/C voice tests: private original-protocol endpoint ready.
Scope: original ASR/dialogue/TTS evaluation. Weather confirmation and sentence streaming use the separate WX regression checks above.
The public weather-booking route remains active. The old optional public-gateway harness does not implement the new authenticated weather protocol.


In [78]:
# Run AFTER weather/voice activation and BEFORE evaluation.
# Close existing AI Call popups first.

def movio_fix_saved_bookings():
    from pathlib import Path
    from datetime import datetime
    import ast
    import inspect
    import json
    import re
    import shutil
    import sqlite3
    import subprocess
    import textwrap
    from urllib.request import build_opener, ProxyHandler

    root = Path(
        globals().get("WX_ROOT", "/kaggle/working/MOVIO_MAIN_WEBSITE")
    ).resolve()

    b = globals().get("bridge")
    db = getattr(b, "_wx_db", None)

    if b is None or db is None or not hasattr(b, "_wx_original_socket"):
        raise RuntimeError(
            "Run the existing weather activation cell first, then this patch."
        )

    database = root / "movio_demo.db"

    if (
        not database.is_file()
        or Path(db.path).resolve() != database.resolve()
    ):
        raise RuntimeError(
            "Voice and website database paths differ. No files were changed.\n"
            f"Voice database: {db.path}\n"
            f"Expected website database: {database}"
        )

    client = build_opener(ProxyHandler({}))

    with client.open(
        "http://127.0.0.1:8080/api/weather-booking/health",
        timeout=5,
    ) as response:
        health = json.load(response)

    if health.get("guard") != "sqlite_atomic_revision":
        raise RuntimeError(
            "Activate the existing weather booking backend first."
        )

    # 1. Include scheduled bookings in the voice cancellation session.
    socket = b._wx_original_socket
    replacement = None

    if not getattr(socket, "_movio_scheduled_cancel", False):
        tree = ast.parse(
            textwrap.dedent(inspect.getsource(socket))
        )

        fn = next(
            node
            for node in tree.body
            if isinstance(node, ast.AsyncFunctionDef)
        )

        fn.decorator_list = []
        fn.name = "_movio_scheduled_cancel_socket"

        original_statuses = {
            "requested",
            "accepted",
            "arriving",
            "in_progress",
            "needs_location",
        }

        changed = 0

        for node in ast.walk(fn):
            if (
                isinstance(node, (ast.Tuple, ast.List))
                and all(
                    isinstance(item, ast.Constant)
                    for item in node.elts
                )
            ):
                values = [item.value for item in node.elts]

                if (
                    all(isinstance(value, str) for value in values)
                    and set(values) == original_statuses
                ):
                    node.elts.append(ast.Constant(value="scheduled"))
                    changed += 1

        if not changed:
            raise RuntimeError(
                "Unexpected cancellation handler. No files were changed."
            )

        module = ast.fix_missing_locations(
            ast.Module(body=[fn], type_ignores=[])
        )

        exec(
            compile(
                module,
                "<MOVIO scheduled cancellation patch>",
                "exec",
            ),
            socket.__globals__,
        )

        replacement = socket.__globals__[fn.name]
        replacement._movio_scheduled_cancel = True

    app_path = root / "web/app.js"
    index_path = root / "web/index.html"

    app_original = app_path.read_text(encoding="utf-8")
    index_original = index_path.read_text(encoding="utf-8")

    # Enable the existing history Cancel button for scheduled bookings.
    app_text = app_original.replace(
        "if(['requested','accepted','arriving','needs_location']"
        ".includes(r.status)){",
        "if(['scheduled','requested','accepted','arriving','needs_location']"
        ".includes(r.status)){",
    )

    # 2. Repair dashboard lookup, cancellation and save verification.
    fix_js = r"""
(function () {
    'use strict';

    if (window.__movioSavedBookingFix) return;
    window.__movioSavedBookingFix = true;

    const allowed = new Set([
        'scheduled',
        'requested',
        'accepted',
        'arriving',
        'in_progress',
        'needs_location'
    ]);

    async function ownedRides() {
        const data = await api('/rides', {cache: 'no-store'});

        if (!Array.isArray(data.rides)) {
            throw Error('Unable to load your bookings.');
        }

        return data.rides;
    }

    async function ownedRide(id) {
        const ride = (await ownedRides()).find(
            r => Number(r.id) === Number(id)
        );

        if (!ride) {
            throw Error(
                'Booking #' + id +
                ' was not found in this signed-in account.'
            );
        }

        return ride;
    }

    voiceRides = async () => {
        return (await ownedRides()).filter(
            r => allowed.has(r.status)
        );
    };

    const previousPost = post;

    post = async function (path, data) {
        const match = /^\/rides\/(\d+)\/status$/.exec(path);

        if (match && data?.status === 'cancelled') {
            const id = Number(match[1]);
            const ride = await ownedRide(id);

            if (ride.status !== 'cancelled') {
                if (ride.status === 'scheduled') {
                    try {
                        await previousPost(
                            '/weather-booking/cancel',
                            {ride_id: id}
                        );
                    } catch (error) {
                        // Dispatch might occur while cancellation is sent.
                        const latest = await ownedRide(id);

                        if (latest.status === 'scheduled') {
                            throw error;
                        }

                        if (latest.status !== 'cancelled') {
                            await previousPost(path, data);
                        }
                    }
                } else {
                    await previousPost(path, data);
                }
            }

            const saved = await ownedRide(id);

            if (saved.status !== 'cancelled') {
                throw Error(
                    'Cancellation was not saved. Please retry.'
                );
            }

            return {ok: true, id};
        }

        const result = await previousPost(path, data);

        if (
            (path === '/rides' ||
             path === '/weather-booking/commit') &&
            result.id
        ) {
            await ownedRide(result.id);
        }

        return result;
    };

    const previousHandler = handleMovioVoiceMessage;
    const pending = new Map();

    handleMovioVoiceMessage = async function (message) {
        if (
            !message ||
            message.channel !== 'ai-ride-call' ||
            message.nonce !== aiNonce
        ) {
            return;
        }

        if (message.type === 'booking_saved') {
            try {
                await ownedRide(message.id);
            } catch (error) {
                toast(error.message);
                return;
            }
        }

        if (
            message.type === 'action' &&
            message.action?.type === 'cancel' &&
            message.action.request_id
        ) {
            const key = aiNonce + ':' + message.action.request_id;

            if (!pending.has(key)) {
                pending.set(
                    key,
                    Promise.resolve().then(
                        () => previousHandler(message)
                    )
                );
            }

            return pending.get(key);
        }

        return previousHandler(message);
    };

    renderVoiceHistory();
    refreshRides();
})();
"""

    fix_path = root / "web/movio_saved_booking_fix.js"

    tag = (
        '<script src="/movio_saved_booking_fix.js?v=1" '
        'defer></script>'
    )

    if "movio_saved_booking_fix.js" not in index_original:
        if "</body>" not in index_original:
            raise RuntimeError(
                "Dashboard HTML is unexpected. No files were changed."
            )

        index_text = index_original.replace(
            "</body>",
            tag + "\n</body>",
            1,
        )
    else:
        index_text = index_original

    # Reload the updated app.js when the dashboard is refreshed.
    index_text = re.sub(
        r"""(?P<file>\b(?:src)=["'][^"']*\bapp\.js)(?:\?[^"']*)?""",
        r"\g<file>?booking_fix=1",
        index_text,
    )

    backup = root / (
        "booking_fix_backup_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    backup.mkdir()

    updates = [
        (app_path, app_text),
        (fix_path, fix_js),
        (index_path, index_text),
    ]

    # Check JavaScript before replacing the website files.
    node = shutil.which("node")

    if node:
        for path, text in updates:
            if path.suffix == ".js":
                check_path = backup / (path.name + ".check.js")
                check_path.write_text(text, encoding="utf-8")

                subprocess.run(
                    [node, "--check", str(check_path)],
                    check=True,
                    capture_output=True,
                    text=True,
                )

    for path, text in updates:
        if path.exists():
            shutil.copy2(path, backup / path.name)

        path.write_text(text, encoding="utf-8")

    if replacement is not None:
        b._wx_original_socket = replacement

    # 3. Verify an actual ride exists for the call's account before
    # the weather voice handler announces booking success.
    if not getattr(db, "_movio_storage_check", False):
        original_row = db.row

        def verified_row(call_id):
            row = original_row(call_id)

            if row.get("ride_id"):
                with db.connect() as connection:
                    ride = connection.execute(
                        "SELECT customer_id FROM rides WHERE id=?",
                        (row["ride_id"],),
                    ).fetchone()

                if (
                    ride is None
                    or int(ride[0]) != int(row["owner"])
                ):
                    raise RuntimeError(
                        "Booking storage verification failed. "
                        "No successful booking can be announced."
                    )

            return row

        db.row = verified_row
        db._movio_storage_check = True

    # 4. Display actual persisted booking IDs and statuses.
    with sqlite3.connect(
        database.as_uri() + "?mode=ro",
        uri=True,
    ) as connection:
        counts = connection.execute(
            "SELECT status, COUNT(*) "
            "FROM rides GROUP BY status"
        ).fetchall()

        recent = connection.execute(
            "SELECT id, status FROM rides "
            "ORDER BY id DESC LIMIT 5"
        ).fetchall()

    print("Patch installed.")
    print("Stored bookings by status:", dict(counts))
    print("Recent stored booking IDs/statuses:", recent)
    print("Backup:", backup)
    print()
    print(
        "Close the old AI Call popup. Hard-refresh the signed-in "
        "dashboard, then open a NEW AI Call."
    )
    print(
        "Complete BOTH confirmations and wait for the booking ID."
    )
    print(
        "Future rides remain scheduled until their dispatch time."
    )


movio_fix_saved_bookings()

Patch installed.
Stored bookings by status: {}
Recent stored booking IDs/statuses: []
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/booking_fix_backup_20261028_070656_974678

Close the old AI Call popup. Hard-refresh the signed-in dashboard, then open a NEW AI Call.
Complete BOTH confirmations and wait for the booking ID.
Future rides remain scheduled until their dispatch time.


In [79]:
def install_tanglish_native_speech():
    import base64
    import contextvars
    import re
    import threading
    from collections import OrderedDict, deque

    b = globals().get("bridge")
    adapter = getattr(b, "TTS", None)
    ns = getattr(adapter, "ns", None)
    wx = getattr(
        getattr(b, "_wx_endpoint", None),
        "__globals__",
        {},
    )

    required = (
        "localized",
        "speech_numeric_normalize",
        "spoken_datetime",
        "summary_text",
        "display_reply",
        "advisory",
        "numeric_text",
    )

    if (
        not isinstance(ns, dict)
        or not all(callable(wx.get(k)) for k in required)
    ):
        raise RuntimeError(
            "Run the existing models, voice bridge and weather "
            "activation first."
        )

    if getattr(b, "_tanglish_native_speech_v1", False):
        print("Tanglish Tamil-script speech is already enabled.")
        return

    active = contextvars.ContextVar(
        "movio_tanglish_native_speech",
        default=False,
    )

    pairs = OrderedDict()
    names = set(getattr(b, "VOICE_PLACE_ALIASES", {}))
    lock = threading.RLock()

    # Recent backend speech text, retained in this kernel session.
    b.TANGLISH_SPEECH_LOG = deque(maxlen=50)

    # Keep Roman-letter display and Tamil-script speech together.
    def remember(display, spoken):
        display, spoken = str(display), str(spoken)
        entries = [(display, spoken)]

        def split_sentences(text):
            return re.split(
                r"(?<=[.!?])\s+|\n+",
                text.strip(),
            )

        shown_parts = split_sentences(display)
        spoken_parts = split_sentences(spoken)

        if len(shown_parts) == len(spoken_parts):
            entries.extend(zip(shown_parts, spoken_parts))

        with lock:
            for shown, said in entries:
                if shown:
                    pairs[shown] = said
                    pairs.move_to_end(shown)

            while len(pairs) > 1024:
                pairs.popitem(last=False)

        return display

    original_localized = wx["localized"]

    def localized(lang, en, ta, tanglish):
        if lang == "Tanglish":
            return remember(tanglish, ta)

        return original_localized(lang, en, ta, tanglish)

    original_say = getattr(b, "say", None)

    def say(state, key, **values):
        result = original_say(state, key, **values)

        if state.get("lang") == "Tanglish":
            tamil_state = dict(state, lang="Tamil")
            tamil_result = original_say(
                tamil_state,
                key,
                **values,
            )
            remember(result, tamil_result)

        return result

    # Date/time entities use English in Tanglish mode.
    original_date = wx["spoken_datetime"]

    def spoken_datetime(value, lang="English"):
        selected = "English" if lang == "Tanglish" else lang
        return original_date(value, selected)

    def remember_names(state):
        with lock:
            names.update(
                str(state[key])
                for key in ("pickup", "destination")
                if state.get(key)
            )

    original_summary = wx["summary_text"]

    def summary_text(bridge_object, state):
        if state.get("lang") != "Tanglish":
            return original_summary(bridge_object, state)

        remember_names(state)

        date = spoken_datetime(
            state["pickup_at"],
            "Tanglish",
        )
        count = wx["numeric_text"](state["count"], 0)
        km = wx["numeric_text"](state["fare"]["km"])
        fare = wx["numeric_text"](
            state["fare"]["fare"],
            0,
            grouped=True,
        )

        vehicle = state["vehicle"]
        pickup = state["pickup"]
        drop = state["destination"]

        people = (
            "passenger"
            if int(state["count"]) == 1
            else "passengers"
        )

        display = (
            f"Unga munpadhivu vivaram: {count} {people}, "
            f"{vehicle}, {pickup} ilirundhu {drop} varai, "
            f"{date}. "
            f"Saalai dhooram {km} kilometres. "
            f"Madhippidappatta kattanam ₹{fare}. "
            "Indha munpadhivai urudhi seiyyalaama?"
        )

        spoken = (
            f"உங்க முன்பதிவு விவரம்: {count} {people}, "
            f"{vehicle}, {pickup} இலிருந்து {drop} வரை, "
            f"{date}. "
            f"சாலை தூரம் {km} kilometres. "
            f"மதிப்பிடப்பட்ட கட்டணம் ₹{fare}. "
            "இந்த முன்பதிவை உறுதி செய்யலாமா?"
        )

        return remember(display, spoken)

    original_advisory = wx["advisory"]

    def advisory(state):
        if state.get("lang") == "Tanglish":
            remember_names(state)

        return original_advisory(state)

    # Display Tamil place names using Roman letters.
    aliases = {
        "சென்னை சென்ட்ரல்": "Chennai Central",
        "தாம்பரம்": "Tambaram",
        "சென்னை விமான நிலையம்": "Chennai Airport",
        "வேளச்சேரி": "Velachery",
        "அண்ணா நகர்": "Anna Nagar",
        "கொட்டிவாக்கம்": "Kottivakkam",
    }

    for name, variants in getattr(
        b, "VOICE_PLACE_ALIASES", {}
    ).items():
        for variant in variants:
            if re.search(r"[\u0b80-\u0bff]", variant):
                aliases[variant] = name

    vowels = dict(zip(
        "அஆஇஈஉஊஎஏஐஒஓஔ",
        (
            "a", "aa", "i", "ee", "u", "oo",
            "e", "ae", "ai", "o", "oa", "au",
        ),
    ))

    consonants = dict(zip(
        "கஙசஞடணதநபமயரலவழளறனஜஷஸஹ",
        (
            "k", "ng", "ch", "nj", "t", "n",
            "th", "n", "p", "m", "y", "r",
            "l", "v", "zh", "l", "r", "n",
            "j", "sh", "s", "h",
        ),
    ))

    signs = dict(zip(
        "ாிீுூெேைொோௌ்",
        (
            "aa", "i", "ee", "u", "oo", "e",
            "ae", "ai", "o", "oa", "au", "",
        ),
    ))

    def romanize(text):
        for tamil in sorted(aliases, key=len, reverse=True):
            text = text.replace(tamil, aliases[tamil])

        output = []
        i = 0

        while i < len(text):
            char = text[i]

            if char in consonants:
                following = (
                    text[i + 1]
                    if i + 1 < len(text)
                    else ""
                )
                vowel = (
                    signs[following]
                    if following in signs
                    else "a"
                )
                output.append(consonants[char] + vowel)
                i += int(following in signs)

            elif char in vowels:
                output.append(vowels[char])

            elif "௦" <= char <= "௯":
                output.append(str(ord(char) - ord("௦")))

            elif char == "ஃ":
                output.append("h")

            else:
                output.append(char)

            i += 1

        return "".join(output)

    original_display = wx["display_reply"]

    def display_reply(state, text):
        shown = original_display(state, text)

        if state.get("lang") == "Tanglish":
            return romanize(shown)

        return shown

    # Existing pronunciation dictionary plus common booking words.
    lexicon = {
        str(key).casefold(): str(value)
        for key, value in ns.get(
            "TANGLISH_WORDS", {}
        ).items()
        if re.search(r"[\u0b80-\u0bff]", str(value))
    }

    lexicon.update({
        "unga": "உங்க",
        "ungal": "உங்கள்",
        "neenga": "நீங்க",
        "naan": "நான்",
        "vanakkam": "வணக்கம்",
        "nandri": "நன்றி",
        "payanam": "பயணம்",
        "inimaiyaaga": "இனிமையாக",
        "amaiyattum": "அமையட்டும்",
        "munpadhivu": "முன்பதிவு",
        "munpadhivai": "முன்பதிவை",
        "vivaram": "விவரம்",
        "urudhi": "உறுதி",
        "seiyyalaama": "செய்யலாமா",
        "pannalaama": "பண்ணலாமா",
        "kattanam": "கட்டணம்",
        "madhippidappatta": "மதிப்பிடப்பட்ட",
        "saalai": "சாலை",
        "dhooram": "தூரம்",
        "ilirundhu": "இலிருந்து",
        "varai": "வரை",
        "mazhai": "மழை",
        "vaaippu": "வாய்ப்பு",
        "irukku": "இருக்கு",
        "irukkum": "இருக்கும்",
        "innaiku": "இன்னைக்கு",
        "inniku": "இன்னைக்கு",
        "naalaiku": "நாளைக்கு",
        "thodaralaama": "தொடரலாமா",
        "peru": "பேர்",
    })

    words = re.compile(
        r"(?<![A-Za-z])("
        + "|".join(
            re.escape(key)
            for key in sorted(
                lexicon,
                key=len,
                reverse=True,
            )
        )
        + r")(?![A-Za-z])",
        re.I,
    )

    units = {
        "முன்பதிவு எண்": "booking ID",
        "கிலோமீட்டர்": "kilometres",
        "கிலோமீட்டர்கள்": "kilometres",
        "kilo meettar": "kilometres",
        "ரூபாய்": "rupees",
        "சதவீதம்": "percent",
        "புள்ளி": "point",
        "பேர்": "passengers",
        "°C": " degrees Celsius",
        "எஸ் யூ வி": "SUV",
        "மினி கார்": "Mini",
        "செடான்": "Sedan",
        "ஆட்டோ": "Auto",
        "பைக்": "Bike",
    }

    def speech_text(text):
        text = str(text)

        with lock:
            replacements = sorted(
                pairs.items(),
                key=lambda item: len(item[0]),
                reverse=True,
            )
            protected_names = sorted(
                names,
                key=len,
                reverse=True,
            )

        for shown, said in replacements:
            text = text.replace(shown, said)

        protected = {}

        for name in protected_names:
            if name and name in text:
                marker = chr(0xF000 + len(protected))
                text = text.replace(name, marker)
                protected[marker] = name

        text = words.sub(
            lambda match: lexicon[
                match[0].casefold()
            ],
            text,
        )

        for source in sorted(
            units,
            key=len,
            reverse=True,
        ):
            pattern = (
                r"(?<![A-Za-z\u0b80-\u0bff])"
                + re.escape(source)
                + r"(?![A-Za-z\u0b80-\u0bff])"
            )
            text = re.sub(
                pattern,
                units[source],
                text,
            )

        for marker, name in protected.items():
            text = text.replace(marker, name)

        # Speak numerical entities in English.
        # Website text and database values retain their digits.
        text = wx["speech_numeric_normalize"](
            lambda value: value,
            text,
            "English",
        )

        text = re.sub(r"\bSUV\b", "S U V", text)
        text = re.sub(
            r"\bOTP\b",
            "O T P",
            text,
            flags=re.I,
        )
        text = re.sub(r"\bID\b", "I D", text)
        text = re.sub(
            r"\b(AM|PM)\b",
            lambda match: " ".join(match[0]),
            text,
        )

        return re.sub(r"\s+", " ", text).strip()

    # Bypass further Tamil number conversion only while
    # synthesizing this prepared Tanglish speech copy.
    targets = [ns]

    prepare_globals = getattr(
        ns.get("prepare_movio_request"),
        "__globals__",
        None,
    )

    if (
        isinstance(prepare_globals, dict)
        and prepare_globals is not ns
    ):
        targets.append(prepare_globals)

    if not all(
        callable(target.get("normalize_taxi_text"))
        for target in targets
    ):
        raise RuntimeError(
            "The original TTS normalizer is unavailable; "
            "nothing was installed."
        )

    for target in targets:
        previous = target["normalize_taxi_text"]

        def normalize(text, _previous=previous):
            if active.get():
                return str(text)

            return _previous(text)

        target["normalize_taxi_text"] = normalize

    original_synthesize = adapter.synthesize

    def synthesize(text, group):
        if b.CURRENT_LANGUAGE.get() != "Tanglish":
            return original_synthesize(text, group)

        spoken = speech_text(text)

        b.TANGLISH_SPEECH_LOG.append({
            "source_text": str(text),
            "tts_text": spoken,
        })

        token = active.set(True)

        try:
            return original_synthesize(
                spoken,
                "TANGLISH",
            )
        finally:
            active.reset(token)

    original_speak = b.speak

    def speak(text):
        if b.CURRENT_LANGUAGE.get() != "Tanglish":
            return original_speak(text)

        with b.LOCK:
            data = adapter.synthesize(
                str(text),
                "TANGLISH",
            )

        return base64.b64encode(data).decode()

    wx.update(
        localized=localized,
        spoken_datetime=spoken_datetime,
        summary_text=summary_text,
        advisory=advisory,
        display_reply=display_reply,
    )

    if callable(original_say):
        b.say = say

    adapter.synthesize = synthesize
    b.speak = speak
    b.tanglish_speech_text = speech_text
    b._tanglish_native_speech_v1 = True

    print(
        "Enabled: Tamil-script speech with English entities."
    )
    print(
        "Website replies remain in Roman-letter Tanglish."
    )
    print(
        "Close the old popup and start a NEW Tanglish call."
    )


install_tanglish_native_speech()

Enabled: Tamil-script speech with English entities.
Website replies remain in Roman-letter Tanglish.
Close the old popup and start a NEW Tanglish call.


In [80]:
def movio_fix_cancel_connection():
    import ast
    import inspect
    import re
    import shutil
    import subprocess
    import textwrap
    from datetime import datetime
    from pathlib import Path

    b = globals().get("bridge")

    if b is None or not isinstance(getattr(b, "HTML", None), str):
        raise RuntimeError(
            "Run the existing voice/weather activation first."
        )

    root = Path(
        globals().get(
            "WX_ROOT",
            "/kaggle/working/MOVIO_MAIN_WEBSITE"
        )
    )

    marker = "/* MOVIO_CANCEL_ACCOUNT_V1 */"

    helpers = r"""
/* MOVIO_CANCEL_ACCOUNT_V1 */
let accountCancelRides = [];
let accountCancelOwner = null;

const accountCancelRequests = new Map();

const accountCancelStatuses = new Set([
    'scheduled',
    'requested',
    'accepted',
    'arriving',
    'needs_location'
]);

async function cancelAccountAPI(path, body) {
    const controller = new AbortController();
    const timer = setTimeout(() => controller.abort(), 10000);

    try {
        const response = await fetch('/api' + path, {
            credentials: 'same-origin',
            cache: 'no-store',
            signal: controller.signal,
            ...(body === undefined ? {} : {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify(body)
            })
        });

        let data;

        try {
            data = await response.json();
        } catch (error) {
            throw Error(
                'The website API did not return JSON. ' +
                'Open the current website address.'
            );
        }

        if (response.status === 401) {
            throw Error(
                'Sign in to your customer account on this website address.'
            );
        }

        if (!response.ok || data.ok === false) {
            throw Error(
                data.error || 'The booking request was rejected.'
            );
        }

        return data;
    } finally {
        clearTimeout(timer);
    }
}

async function cancelAccountSnapshot() {
    const me = await cancelAccountAPI('/me');

    if (me.role !== 'customer') {
        throw Error(
            'Use your customer account to cancel a booking.'
        );
    }

    const data = await cancelAccountAPI('/rides');

    if (!Array.isArray(data.rides)) {
        throw Error(
            'Your bookings could not be loaded. Please retry.'
        );
    }

    return {
        owner: Number(me.id),
        rides: data.rides.filter(
            ride => Number(ride.customer_id) === Number(me.id)
        )
    };
}

async function loadAccountCancellation() {
    const snapshot = await cancelAccountSnapshot();

    accountCancelOwner = snapshot.owner;

    accountCancelRides = snapshot.rides.filter(
        ride => accountCancelStatuses.has(ride.status)
    ).slice(0, 30);

    $('rides').textContent = accountCancelRides.map(
        (ride, index) =>
            `${index + 1}. #${ride.id}: ` +
            `${ride.pickup} → ${ride.destination} (${ride.status})`
    ).join('\n');

    return accountCancelRides;
}

async function ownedCancellationRide(id) {
    const snapshot = await cancelAccountSnapshot();

    if (snapshot.owner !== accountCancelOwner) {
        throw Error(
            'Your signed-in account changed. ' +
            'End this call and start again.'
        );
    }

    const ride = snapshot.rides.find(
        item => Number(item.id) === id
    );

    if (!ride) {
        throw Error(
            'This booking was not found in your signed-in account.'
        );
    }

    return ride;
}

async function cancelAccountBooking(action, id) {
    const request = action.request_id;

    if (!request) {
        send({
            event: 'result',
            ok: false,
            error: 'Cancellation confirmation is missing.'
        });
        return;
    }

    const key = id + ':' + request;

    if (!accountCancelRequests.has(key)) {
        accountCancelRequests.set(key, (async () => {
            try {
                const rideId = Number(action.ride_id);

                if (!Number.isSafeInteger(rideId) || rideId < 1) {
                    throw Error('Invalid booking number.');
                }

                const ride = await ownedCancellationRide(rideId);

                if (ride.status !== 'cancelled') {
                    if (!accountCancelStatuses.has(ride.status)) {
                        throw Error(
                            'This booking cannot be cancelled ' +
                            'in its current state.'
                        );
                    }

                    try {
                        if (ride.status === 'scheduled') {
                            await cancelAccountAPI(
                                '/weather-booking/cancel',
                                {ride_id: rideId}
                            );
                        } else {
                            await cancelAccountAPI(
                                '/rides/' + rideId + '/status',
                                {status: 'cancelled'}
                            );
                        }
                    } catch (error) {
                        const latest = await ownedCancellationRide(
                            rideId
                        );

                        if (latest.status !== 'cancelled') {
                            if (
                                ride.status === 'scheduled' &&
                                latest.status !== 'scheduled' &&
                                accountCancelStatuses.has(latest.status)
                            ) {
                                await cancelAccountAPI(
                                    '/rides/' + rideId + '/status',
                                    {status: 'cancelled'}
                                );
                            } else {
                                throw error;
                            }
                        }
                    }
                }

                const saved = await ownedCancellationRide(rideId);

                if (saved.status !== 'cancelled') {
                    throw Error(
                        'Cancellation was not saved. Please retry.'
                    );
                }

                return {ok: true, id: rideId};

            } catch (error) {
                return {
                    ok: false,
                    error: error.name === 'AbortError'
                        ? 'The website timed out. Check your booking ' +
                          'status before retrying.'
                        : error.message
                };
            }
        })());
    }

    const result = await accountCancelRequests.get(key);

    if (id === callId && connected) {
        send({
            event: 'result',
            request_id: request,
            ...result
        });
    }
}
"""

    def patch_html(html):
        if marker in html:
            return html

        changes = [
            (
                "if(mode==='cancel'&&(!ready||!nonce)){"
                "notify('hello');"
                "status('Reconnect using the dashboard AI Call "
                "button to load your bookings.');return;}",
                ""
            ),
            (
                "try{const binding=mode==='book'?",
                "try{if(mode==='cancel')"
                "await loadAccountCancellation();"
                "const binding=mode==='book'?"
            ),
            (
                "if(binding)send({event:'start',...binding});"
                "notify('get_rides');",
                "if(binding)send({event:'start',...binding});"
                "if(mode==='cancel')"
                "send({event:'active_rides',rides:accountCancelRides});"
                "else notify('get_rides');"
            ),
            (
                "function handleParentMessage(m,transport){",
                "function handleParentMessage(m,transport){"
                "if(mode==='cancel')return;"
            ),
            (
                "if(m.type==='weather_book')"
                "await commitBooking(m,id);"
                "else notify('action',{action:m});",
                "if(m.type==='weather_book')"
                "await commitBooking(m,id);"
                "else if(m.type==='cancel')"
                "await cancelAccountBooking(m,id);"
                "else notify('action',{action:m});"
            ),
            (
                "$('cancelMode').onclick=()=>{"
                "mode='cancel';notify('get_rides');};",
                "$('cancelMode').onclick=async()=>{"
                "mode='cancel';"
                "try{await loadAccountCancellation();"
                "status('Your bookings are loaded. Start the call.');}"
                "catch(error){$('rides').textContent='';"
                "status(error.message);}};"
            ),
        ]

        for old, new in changes:
            if html.count(old) != 1:
                raise RuntimeError(
                    "Unexpected call-page version. "
                    "No files were changed."
                )

            html = html.replace(old, new, 1)

        anchor = "async function start(){"

        if html.count(anchor) != 1:
            raise RuntimeError(
                "The call-page start function is unexpected."
            )

        return html.replace(
            anchor,
            helpers + "\n" + anchor,
            1
        )

    active_html = patch_html(b.HTML)
    updates = []

    for name in (
        "ai_call_page.html",
        "weather_call_page.html"
    ):
        path = root / "integrations" / name

        if path.is_file():
            updates.append((
                path,
                patch_html(
                    path.read_text(encoding="utf-8")
                )
            ))

    # Ensure the existing voice handler accepts scheduled rides.
    legacy = getattr(b, "_wx_original_socket", None)
    replacement = None

    if legacy and not getattr(
        legacy, "_movio_scheduled_cancel", False
    ):
        tree = ast.parse(
            textwrap.dedent(
                inspect.getsource(legacy)
            )
        )

        fn = next(
            item for item in tree.body
            if isinstance(item, ast.AsyncFunctionDef)
        )

        fn.decorator_list = []
        fn.name = "_movio_scheduled_cancel_socket"

        old_statuses = {
            "requested",
            "accepted",
            "arriving",
            "in_progress",
            "needs_location",
        }

        found = False

        for item in ast.walk(fn):
            if (
                isinstance(item, (ast.Tuple, ast.List))
                and all(
                    isinstance(value, ast.Constant)
                    for value in item.elts
                )
            ):
                values = [
                    value.value for value in item.elts
                ]

                if (
                    all(isinstance(value, str) for value in values)
                    and set(values) in (
                        old_statuses,
                        old_statuses | {"scheduled"}
                    )
                ):
                    if "scheduled" not in values:
                        item.elts.append(
                            ast.Constant(value="scheduled")
                        )

                    found = True

        if not found:
            raise RuntimeError(
                "Unexpected voice cancellation handler. "
                "No files were changed."
            )

        module = ast.fix_missing_locations(
            ast.Module(
                body=[fn],
                type_ignores=[]
            )
        )

        exec(
            compile(
                module,
                "<MOVIO cancellation>",
                "exec"
            ),
            legacy.__globals__
        )

        replacement = legacy.__globals__[fn.name]
        replacement._movio_scheduled_cancel = True

    backup = root / (
        "cancel_connection_backup_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    backup.mkdir()

    node = shutil.which("node")

    if node:
        pages = [active_html] + [
            text for _, text in updates
        ]

        for index, html in enumerate(pages):
            scripts = re.findall(
                r"<script\b[^>]*>(.*?)</script>",
                html,
                flags=re.S | re.I
            )

            check = backup / f"syntax_{index}.js"
            check.write_text(
                "\n".join(scripts),
                encoding="utf-8"
            )

            subprocess.run(
                [node, "--check", str(check)],
                check=True,
                capture_output=True,
                text=True
            )

    for path, text in updates:
        shutil.copy2(path, backup / path.name)
        path.write_text(text, encoding="utf-8")

    if replacement is not None:
        b._wx_original_socket = replacement

    b.HTML = active_html

    print(
        "Cancellation now loads bookings through "
        "your signed-in website account."
    )
    print(
        "Close the old call popup, refresh the dashboard "
        "and open a NEW AI Call."
    )
    print(
        "Select Cancel booking, then Start call. "
        "Confirm the selected booking to cancel it."
    )


movio_fix_cancel_connection()

Cancellation now loads bookings through your signed-in website account.
Close the old call popup, refresh the dashboard and open a NEW AI Call.
Select Cancel booking, then Start call. Confirm the selected booking to cancel it.


In [81]:
def movio_tanglish_wordwise_fix():
    import base64
    import contextvars
    import inspect
    import re
    import threading
    from collections import deque

    b = globals().get("bridge")
    adapter = getattr(b, "TTS", None)
    ns = getattr(adapter, "ns", None)
    wx = getattr(
        getattr(b, "_wx_endpoint", None),
        "__globals__",
        {},
    )

    if (
        not isinstance(ns, dict)
        or not callable(wx.get("speech_numeric_normalize"))
    ):
        raise RuntimeError(
            "Run your existing voice/weather activation first."
        )

    if getattr(b, "_tg_wordwise_v2", False):
        print("Word-by-word Tanglish speech is already enabled.")
        return

    # Bypass the earlier whole-sentence translation wrapper.
    base_synthesize = adapter.synthesize

    if getattr(b, "_tanglish_native_speech_v1", False):
        enclosed = inspect.getclosurevars(
            base_synthesize
        ).nonlocals

        required = (
            "original_synthesize",
            "speech_text",
            "active",
        )

        if not all(key in enclosed for key in required):
            raise RuntimeError(
                "Unexpected TTS wrapper. No changes were installed."
            )

        base_synthesize = enclosed["original_synthesize"]

    active = contextvars.ContextVar(
        "movio_tanglish_wordwise",
        default=False,
    )

    lock = threading.RLock()
    places = set(getattr(b, "VOICE_PLACE_ALIASES", {}))

    if not hasattr(b, "TANGLISH_SPEECH_LOG"):
        b.TANGLISH_SPEECH_LOG = deque(maxlen=50)

    # Preserve English vocabulary as English.
    english = set((
        "ai taxi booking service welcome help ride book existing "
        "cancel travel pickup drop place correct yes no vehicle "
        "road distance around approximate fare confirm details "
        "passengers passenger kilometres kilometers rupees driver "
        "mini car bike auto sedan suv am pm otp id number request "
        "record save select screen map route verify continue "
        "rain chance forecast please check change retry "
        "tomorrow today morning evening night"
    ).split())

    words = {
        str(key).lower(): str(value)
        for key, value in ns.get("TANGLISH_WORDS", {}).items()
        if (
            re.search(r"[\u0b80-\u0bff]", str(value))
            and str(key).lower() not in english
        )
    }

    # Tamil-origin words: change script, retain the wording.
    words.update(dict(
        vanakkam="வணக்கம்",
        ungalukku="உங்களுக்கு",
        unga="உங்க",
        ungal="உங்கள்",
        eppadi="எப்படி",
        pannalaam="பண்ணலாம்",
        pannalaama="பண்ணலாமா",
        pannalama="பண்ணலாமா",
        pannunga="பண்ணுங்க",
        panna="பண்ண",
        pannanuma="பண்ணணுமா",
        pannum="பண்ணும்",
        neenga="நீங்க",
        naan="நான்",
        enakku="எனக்கு",
        ku="க்கு",
        la="ல",
        ah="ஆ",
        ethana="எத்தனை",
        peru="பேர்",
        poreenga="போறீங்க",
        enga="எங்க",
        endha="எந்த",
        naal="நாள்",
        manikku="மணிக்கு",
        venum="வேணும்",
        poganum="போகணும்",
        irundhu="இருந்து",
        ilirundhu="இலிருந்து",
        sonna="சொன்ன",
        illa="இல்ல",
        illai="இல்லை",
        sollunga="சொல்லுங்க",
        sariya="சரியா",
        kekkala="கேட்கல",
        thirumba="திரும்ப",
        marubadiyum="மறுபடியும்",
        nimisham="நிமிஷம்",
        oru="ஒரு",
        panren="பண்றேன்",
        aayiduchu="ஆயிடுச்சு",
        innum="இன்னும்",
        aagala="ஆகல",
        nandri="நன்றி",
        payanam="பயணம்",
        inimaiyaaga="இனிமையாக",
        amaiyattum="அமையட்டும்",
        irukku="இருக்கு",
        irukkura="இருக்குற",
        irukkum="இருக்கும்",
        innaiku="இன்னைக்கு",
        inniku="இன்னைக்கு",
        naalaiku="நாளைக்கு",
        mazhai="மழை",
        vaaippu="வாய்ப்பு",
        thodaralaama="தொடரலாமா",
        munpadhivu="முன்பதிவு",
        munpadhivai="முன்பதிவை",
        vivaram="விவரம்",
        urudhi="உறுதி",
        seiyyalaama="செய்யலாமா",
        varai="வரை",
        madhippidappatta="மதிப்பிடப்பட்ட",
        kattanam="கட்டணம்",
        saalai="சாலை",
        dhooram="தூரம்",
    ))

    # Weather and correction vocabulary.
    words.update(dict(
        indha="இந்த", andha="அந்த", alladhu="அல்லது",
        adhigamaaga="அதிகமாக", irukka="இருக்க",
        irukkalaam="இருக்கலாம்", irukkiradhu="இருக்கிறது",
        irundhaalum="இருந்தாலும்", vaanilai="வானிலை",
        vaanilaiyil="வானிலையில்", vaanam="வானம்",
        kanamazhai="கனமழை", megamoottam="மேகமூட்டம்",
        idiyudan="இடியுடன்", koodiya="கூடிய",
        edhirpaarkkappadugiradhu="எதிர்பார்க்கப்படுகிறது",
        edhirpaarkkappadugindrana="எதிர்பார்க்கப்படுகின்றன",
        therndhedutha="தேர்ந்தெடுத்த", purappadum="புறப்படும்",
        serum="சேரும்", payana="பயண", nerathil="நேரத்தில்",
        nerathirkaana="நேரத்திற்கான", idam="இடம்",
        idamaana="இடமான", idathil="இடத்தில்",
        pagudhiyil="பகுதியில்", mazhaikkaana="மழைக்கான",
        peiya="பெய்ய", madhippeedu="மதிப்பீடு",
        munnarivippu="முன்னறிவிப்பு",
        kidaikkavillai="கிடைக்கவில்லை", thodaralaam="தொடரலாம்",
        neengalum="நீங்களும்", udaimaigalum="உடைமைகளும்",
        udaimaigalai="உடைமைகளை", mazhaiyil="மழையில்",
        mazhaiyilirundhu="மழையிலிருந்து", mazhaikku="மழைக்கு",
        nanaiyaamal="நனையாமல்", thevaiyaana="தேவையான",
        munnerpaadugalai="முன்னேற்பாடுகளை",
        seidhukollunga="செய்துகொள்ளுங்க", kudai="குடை",
        kudineer="குடிநீர்", eduthukkondu="எடுத்துக்கொண்டு",
        paadhugaathukkollunga="பாதுகாத்துக்கொள்ளுங்க",
        vandiyil="வண்டியில்", erumboadhum="ஏறும்போதும்",
        irangumboadhum="இறங்கும்போதும்",
        payanpadutha="பயன்படுத்த", etra="ஏற்ற", ena="என",
        nandraagave="நன்றாகவே", veppanilai="வெப்பநிலை",
        veppathirku="வெப்பத்திற்கு", pakkavaatil="பக்கவாட்டில்",
        padalaam="படலாம்", aattovin="Auto-வின்",
        baikku="Bike", baikkileye="Bike-லேயே",
        poruthamillaamal="பொருத்தமில்லாமல்",
        virumbinaal="விரும்பினால்", moodiya="மூடிய",
        vandiyai="வண்டியை", pondra="போன்ற",
        therndhedukkalaam="தேர்ந்தெடுக்கலாம்",
        mudivu="முடிவு", ungaludaiyadhu="உங்களுடையது",
        pira="பிற", nilaimaigal="நிலைமைகள்",
        payanathirku="பயணத்திற்கு", mun="முன்",
        ulloor="உள்ளூர்", thagavalgalai="தகவல்களை",
        paarunga="பாருங்க", thelivaaga="தெளிவாக",
        vivarangal="விவரங்கள்", vivarangalai="விவரங்களை",
        vivarathai="விவரத்தை", enn="எண்",
        ennikkaiyai="எண்ணிக்கையை", mudhal="முதல்",
        meendum="மீண்டும்", maatra="மாற்ற",
        maathunga="மாத்துங்க", sollalaam="சொல்லலாம்",
        muyarchikka="முயற்சிக்க",
        tharkaaligamaaga="தற்காலிகமாக",
        uruvaakkappadavillai="உருவாக்கப்படவில்லை",
        seiyyavum="செய்யவும்", seiyyunga="செய்யுங்க",
        edhuvum="எதுவும்", raththu="ரத்து",
        therndhedukkavum="தேர்ந்தெடுக்கவும்",
        virumbureenga="விரும்புறீங்க",
        vandi="வண்டி", vandikku="வண்டிக்கு",
        payanigal="பயணிகள்", payanigalukku="பயணிகளுக்கு",
        korikkai="கோரிக்கை", padhivu="பதிவு",
        seiyyappattadhu="செய்யப்பட்டது", sumaar="சுமார்",
    ))

    pattern = re.compile(
        r"(?<![A-Za-z])("
        + "|".join(
            re.escape(key)
            for key in sorted(words, key=len, reverse=True)
        )
        + r")(?![A-Za-z])",
        re.I,
    )

    def speech_text(text):
        text = str(text)

        with lock:
            protected_places = sorted(
                places, key=len, reverse=True
            )

        protected = {}

        for name in protected_places:
            if name and name in text:
                marker = chr(0xF000 + len(protected))
                text = text.replace(name, marker)
                protected[marker] = name

        text = pattern.sub(
            lambda match: words[match[0].lower()],
            text,
        )

        for marker, name in protected.items():
            text = text.replace(marker, name)

        text = re.sub(
            r"(?i)\bkilo\s+meettar\b|\bkm\b",
            "kilometers",
            text,
        )

        # Standardize time suffixes before number verbalization.
        text = re.sub(
            r"(?i)\b(\d{1,2}(?::\d{2})?)\s*(AM|PM)\b",
            lambda match: (
                match[1] + " " + match[2].upper()
            ),
            text,
        )

        # English numbers: thirty five point four, etc.
        text = wx["speech_numeric_normalize"](
            lambda value: value,
            text,
            "English",
        )

        # Speech-only pronunciation aliases.
        text = re.sub(
            r"\b(?:SUV|S\s+U\s+V|es yoo vee)\b",
            "ess you vee",
            text,
            flags=re.I,
        )

        aliases = {
            "AI": "ay eye",
            "AM": "ay em",
            "PM": "pee em",
            "OTP": "oh tee pee",
            "ID": "eye dee",
        }

        text = re.sub(
            r"\b(?:AI|AM|PM|OTP|ID)\b",
            lambda match: aliases[match[0]],
            text,
        )

        return re.sub(r"\s+", " ", text).strip()

    # Prevent the old normalizer from translating this speech copy.
    targets = [ns]

    prepare_globals = getattr(
        ns.get("prepare_movio_request"),
        "__globals__",
        None,
    )

    if (
        isinstance(prepare_globals, dict)
        and prepare_globals is not ns
    ):
        targets.append(prepare_globals)

    if not all(
        callable(target.get("normalize_taxi_text"))
        for target in targets
    ):
        raise RuntimeError(
            "The original TTS normalizer is missing. "
            "No changes were installed."
        )

    for target in targets:
        previous = target["normalize_taxi_text"]

        def normalize(text, _previous=previous):
            if active.get():
                return str(text)
            return _previous(text)

        target["normalize_taxi_text"] = normalize

    def synthesize(text, group):
        if b.CURRENT_LANGUAGE.get() != "Tanglish":
            return base_synthesize(text, group)

        spoken = speech_text(text)

        b.TANGLISH_SPEECH_LOG.append({
            "version": "wordwise-v2",
            "source_text": str(text),
            "tts_text": spoken,
        })

        token = active.set(True)

        try:
            return base_synthesize(spoken, "TANGLISH")
        finally:
            active.reset(token)

    previous_speak = b.speak

    def speak(text):
        if b.CURRENT_LANGUAGE.get() != "Tanglish":
            return previous_speak(text)

        with b.LOCK:
            data = adapter.synthesize(
                str(text), "TANGLISH"
            )

        return base64.b64encode(data).decode()

    # Restore the conversational mixed-language summary.
    previous_summary = wx["summary_text"]

    def summary(bridge_object, state):
        if state.get("lang") != "Tanglish":
            return previous_summary(bridge_object, state)

        with lock:
            places.update(
                str(state[key])
                for key in ("pickup", "destination")
                if state.get(key)
            )

        date = wx["spoken_datetime"](
            state["pickup_at"], "English"
        )
        km = wx["numeric_text"](state["fare"]["km"])
        fare = wx["numeric_text"](
            state["fare"]["fare"], 0
        )

        return (
            f"Booking details: {int(state['count'])} peru, "
            f"{state['vehicle']}, "
            f"{state['pickup']} la irundhu "
            f"{state['destination']} ku, {date}. "
            f"Road distance around {km} kilometers, "
            f"approximate fare {fare} rupees. "
            "Confirm pannalama? Yes illa no sollunga."
        )

    wx["summary_text"] = summary

    if callable(wx.get("weather_question")):
        previous_question = wx["weather_question"]

        def weather_question(lang):
            if lang == "Tanglish":
                return "Booking continue pannalama?"
            return previous_question(lang)

        wx["weather_question"] = weather_question

    previous_completion = wx["completion_message"]

    def completion(lang, ride_id):
        if lang != "Tanglish":
            return previous_completion(lang, ride_id)

        return (
            "Unga booking request save aayiduchu. "
            f"Booking number {int(ride_id)}. "
            "Nandri! Unga payanam inimaiyaaga amaiyattum!"
        )

    wx["completion_message"] = completion

    b.MESSAGES["Tanglish"]["hello"] = (
        "Vanakkam! AI taxi booking service ku welcome. "
        "Ungalukku eppadi help pannalaam?"
    )

    adapter.synthesize = synthesize
    b.speak = speak
    b.tanglish_speech_text = speech_text
    b._tg_wordwise_v2 = True

    print("Conversational Tanglish speech enabled.")
    print("SUV → ess you vee | AM → ay em | PM → pee em")
    print("35.4 → thirty five point four")
    print("Close the old popup and start a NEW Tanglish call.")


movio_tanglish_wordwise_fix()

Conversational Tanglish speech enabled.
SUV → ess you vee | AM → ay em | PM → pee em
35.4 → thirty five point four
Close the old popup and start a NEW Tanglish call.


In [82]:
def test_tanglish_pronunciation():
    import base64
    from IPython.display import Audio, display

    text = (
        "Vanakkam! AI taxi booking service ku welcome. "
        "Ungalukku eppadi help pannalaam? "
        "SUV. Today at 11:00 AM. "
        "Road distance 35.4 kilometers."
    )

    print("TTS input:", bridge.tanglish_speech_text(text))

    token = bridge.CURRENT_LANGUAGE.set("Tanglish")
    try:
        audio = base64.b64decode(bridge.speak(text))
        display(Audio(data=audio, normalize=False))
    finally:
        bridge.CURRENT_LANGUAGE.reset(token)


test_tanglish_pronunciation()

TTS input: வணக்கம்! ay eye taxi booking service க்கு welcome. உங்களுக்கு எப்படி help பண்ணலாம்? ess you vee. Today at eleven ay em. Road distance thirty five point four kilometers.


In [83]:
def movio_weather_closing_audio_fix():
    import base64
    import contextvars
    import inspect
    import io
    import threading
    import wave
    from collections import OrderedDict, deque
    import numpy as np

    b = globals().get("bridge")
    adapter = getattr(b, "TTS", None)
    ns = getattr(adapter, "ns", None)
    wx = getattr(
        getattr(b, "_wx_endpoint", None),
        "__globals__",
        {},
    )

    required = (
        "advisory",
        "completion_message",
        "speech_segments",
        "tts_reply",
    )

    if (
        not isinstance(ns, dict)
        or not callable(getattr(adapter, "synthesize", None))
        or not all(callable(wx.get(key)) for key in required)
    ):
        raise RuntimeError(
            "Run your current voice/weather activation "
            "and earlier fixes first."
        )

    if getattr(b, "_wx_stable_audio_v1", False):
        print("Weather/closing audio patch is already enabled.")
        return

    original_stream = ns.get("movio_stream_response")

    if not callable(original_stream):
        raise RuntimeError("The original TTS stream is unavailable.")

    stream_globals = getattr(
        original_stream, "__globals__", {}
    )
    scheduler = stream_globals.get(
        "SCHEDULER", ns.get("SCHEDULER")
    )
    previous_submit = getattr(scheduler, "submit", None)

    if not callable(previous_submit):
        raise RuntimeError(
            "The existing TTS scheduler is unavailable."
        )

    submit_signature = inspect.signature(previous_submit)

    if "step" not in submit_signature.parameters:
        raise RuntimeError(
            "Unexpected scheduler signature. "
            "No changes were installed."
        )

    scope = contextvars.ContextVar(
        "movio_weather_closing_audio",
        default=None,
    )

    profiles = OrderedDict()
    registry_lock = threading.RLock()

    b.WX_SPEECH_QUALITY = {
        "weather_steps": 8,
        "closing_steps": 8,
        "min_words": 8,
    }

    # Keep the latest actual audio in memory for listening.
    b.WX_SPEECH_AUDIO = deque(maxlen=6)

    def register(text, kind):
        text = str(text).strip()

        with registry_lock:
            profiles[text] = kind
            profiles.move_to_end(text)

            while len(profiles) > 256:
                profiles.popitem(last=False)

        return text

    def profile_for(text):
        text = str(text).strip()

        with registry_lock:
            candidates = list(profiles.items())

        for prefix, kind in reversed(candidates):
            if (
                text == prefix
                or text.startswith(prefix + "\n")
                or text.startswith(prefix + " ")
            ):
                return kind

        return None

    previous_advisory = wx["advisory"]
    previous_completion = wx["completion_message"]
    previous_segments = wx["speech_segments"]
    previous_tts = wx["tts_reply"]
    previous_say = getattr(b, "say", None)
    previous_speak = b.speak

    def advisory(state):
        return register(
            previous_advisory(state),
            "weather",
        )

    def completion(lang, ride_id):
        return register(
            previous_completion(lang, ride_id),
            "closing",
        )

    def segments(text):
        if profile_for(text):
            return [str(text).strip()]

        return previous_segments(text)

    def say(state, key, **values):
        text = previous_say(state, key, **values)

        if key in ("cancelled", "confirmed"):
            return register(text, "closing")

        return text

    # Give short chunks neighbouring text for context.
    # Every normalized word remains in the synthesis plan.
    def wrap_prepare(previous):
        def prepare(*args, **kwargs):
            plan = previous(*args, **kwargs)

            if scope.get() is None:
                return plan

            chunks = plan.get("chunks", [])

            if not chunks:
                raise RuntimeError(
                    "The TTS plan contains no speech chunks."
                )

            result = []
            pending = ""

            minimum = max(
                6,
                int(b.WX_SPEECH_QUALITY["min_words"]),
            )

            for chunk in chunks:
                pending = (
                    pending + " " + str(chunk).strip()
                ).strip()

                if len(pending.split()) >= minimum:
                    result.append(pending)
                    pending = ""

            if pending:
                if result and len(pending.split()) < 4:
                    result[-1] += " " + pending
                else:
                    result.append(pending)

            return {**plan, "chunks": result}

        return prepare

    targets = [ns]

    if stream_globals is not ns:
        targets.append(stream_globals)

    preparations = [
        (target, target.get("prepare_movio_request"))
        for target in targets
    ]

    if not any(
        callable(previous)
        for _, previous in preparations
    ):
        raise RuntimeError(
            "The existing TTS preparation function is missing."
        )

    # Apply extra steps only inside weather/closing synthesis.
    def submit(*args, **kwargs):
        current = scope.get()

        if current is None:
            return previous_submit(*args, **kwargs)

        bound = submit_signature.bind(*args, **kwargs)
        bound.arguments["step"] = current["steps"]

        return previous_submit(
            *bound.args,
            **bound.kwargs,
        )

    class InvalidAudio(RuntimeError):
        pass

    def stream(*args, **kwargs):
        for event in original_stream(*args, **kwargs):
            current = scope.get()

            if current is not None:
                audio = np.asarray(event["audio"])

                if (
                    not audio.size
                    or not np.isfinite(audio).all()
                ):
                    raise InvalidAudio(
                        "TTS returned empty or nonfinite audio."
                    )

                current["chunks"].append(
                    event.get("chunk_text", "")
                )
                current.setdefault(
                    "model_text",
                    event.get("normalized", ""),
                )

            yield event

    def clean_wav(payload):
        try:
            raw = base64.b64decode(payload)

            with wave.open(io.BytesIO(raw), "rb") as reader:
                if (
                    reader.getsampwidth() != 2
                    or reader.getnchannels() != 1
                ):
                    raise InvalidAudio(
                        "Expected mono PCM16 TTS audio."
                    )

                rate = reader.getframerate()

                audio = np.frombuffer(
                    reader.readframes(reader.getnframes()),
                    dtype="<i2",
                ).astype(np.float32) / 32768

        except (wave.Error, EOFError, ValueError) as error:
            raise InvalidAudio(
                "TTS did not return a usable WAV."
            ) from error

        if (
            rate <= 0
            or len(audio) < rate * 0.06
            or not np.isfinite(audio).all()
        ):
            raise InvalidAudio(
                "Invalid or incomplete TTS waveform."
            )

        peak = float(np.max(np.abs(audio)))

        if peak < 1e-5:
            raise InvalidAudio("TTS returned silent audio.")

        # Reduce excessive peaks; never amplify quiet audio.
        if peak > 0.92:
            audio *= 0.92 / peak

        # Short fades soften start/end clicks without trimming words.
        fade = min(
            int(rate * 0.004),
            len(audio) // 2,
        )

        if fade:
            audio[:fade] *= np.linspace(0, 1, fade)
            audio[-fade:] *= np.linspace(1, 0, fade)

        output = io.BytesIO()

        with wave.open(output, "wb") as writer:
            writer.setnchannels(1)
            writer.setsampwidth(2)
            writer.setframerate(rate)
            writer.writeframes(
                (
                    np.clip(audio, -1, 1) * 32767
                ).astype("<i2").tobytes()
            )

        return (
            output.getvalue(),
            rate,
            len(audio) / rate,
        )

    def tts_reply(bridge_object, text):
        kind = profile_for(text)

        # Final confirmation follows its existing path.
        if kind is None:
            return previous_tts(bridge_object, text)

        steps = max(
            4,
            min(
                32,
                int(
                    b.WX_SPEECH_QUALITY[
                        kind + "_steps"
                    ]
                ),
            ),
        )

        # Retry unusable audio once with more generation steps.
        for attempt_steps in (steps, max(16, steps)):
            current = {
                "steps": attempt_steps,
                "chunks": [],
            }
            token = scope.set(current)

            try:
                payload = previous_tts(
                    bridge_object, text
                )
                wav, rate, duration = clean_wav(payload)

                b.WX_SPEECH_AUDIO.append({
                    "kind": kind,
                    "language": b.CURRENT_LANGUAGE.get(),
                    "text": str(text),
                    "model_text": current.get(
                        "model_text", ""
                    ),
                    "steps": attempt_steps,
                    "chunks": current["chunks"],
                    "sample_rate": rate,
                    "duration_s": duration,
                    "wav": wav,
                })

                return base64.b64encode(wav).decode()

            except InvalidAudio:
                if attempt_steps == max(16, steps):
                    raise

            finally:
                scope.reset(token)

    def speak(text):
        if profile_for(text):
            return tts_reply(b, text)

        return previous_speak(text)

    for target, previous in preparations:
        if callable(previous):
            target["prepare_movio_request"] = (
                wrap_prepare(previous)
            )

    scheduler.submit = submit
    ns["movio_stream_response"] = stream

    wx.update(
        advisory=advisory,
        completion_message=completion,
        speech_segments=segments,
        tts_reply=tts_reply,
    )

    if callable(previous_say):
        b.say = say

    b.speak = speak
    b._wx_stable_audio_v1 = True

    print(
        "Installed: grouped weather/closing speech, "
        "8-step generation and WAV checks."
    )
    print(
        "Final confirmation retains its current speech settings."
    )
    print(
        "Close the old popup and start a NEW call."
    )


movio_weather_closing_audio_fix()

Installed: grouped weather/closing speech, 8-step generation and WAV checks.
Final confirmation retains its current speech settings.
Close the old popup and start a NEW call.


In [84]:
from IPython.display import Audio, display

clips = list(bridge.WX_SPEECH_AUDIO)

if not clips:
    print("Complete a new call first.")

for clip in clips[-2:]:
    print(
        clip["language"],
        clip["kind"],
        "| steps:",
        clip["steps"],
    )
    display(Audio(data=clip["wav"], normalize=False))

Complete a new call first.


In [85]:
# Run this entire file in ONE cell after the original website/weather setup,
# before the evaluation cells. No package installation or kernel restart needed.
def movio_refresh_customer_dashboard():
    from pathlib import Path
    from datetime import datetime
    import hashlib
    import os
    import re
    import shutil
    import subprocess
    import tempfile

    root = Path(globals().get('WX_ROOT', '/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    web = root / 'web'
    index_file = web / 'index.html'
    manual_file = web / 'weather_manual.js'
    css_file = web / 'movio_customer_dashboard.css'
    for file in (index_file, manual_file):
        if not file.is_file():
            raise FileNotFoundError(f'{file} is missing. Run the original website/weather setup first.')
    html_before = index_file.read_text(encoding='utf-8')
    manual_before = manual_file.read_text(encoding='utf-8')
    if not re.search(r'</head\s*>', html_before, re.I) or not re.search(r'</body\s*>', html_before, re.I):
        raise RuntimeError('Unexpected dashboard HTML. No files changed.')
    if 'id="riderPanel"' not in html_before and "id='riderPanel'" not in html_before:
        raise RuntimeError('The original customer dashboard was not found. No files changed.')

    css = r'''/* MOVIO_CUSTOMER_UI_V1 — existing controls and handlers are retained. */
#riderPanel{width:460px;min-width:0;max-width:50vw;padding:24px;background:#fff;overflow-x:visible}
#riderPanel *,#riderPanel *::before,#riderPanel *::after{box-sizing:border-box;min-width:0}
#riderPanel button,#riderPanel input,#riderPanel select{font-family:inherit}
#riderPanel button{touch-action:manipulation}
#riderPanel button:focus-visible,#riderPanel input:focus-visible,#riderPanel select:focus-visible,#riderPanel summary:focus-visible{outline:3px solid #d69b00;outline-offset:3px}
#riderPanel .inputs input{font-size:16px}
#riderPanel .inputs label{height:54px}
#riderPanel .section-row{gap:12px;flex-wrap:wrap}
#riderPanel .section-row h2{line-height:1.4}
#riderPanel .vehicles{display:grid;grid-template-columns:1fr;gap:8px}
#riderPanel .vehicle{min-height:64px;padding:11px 14px;gap:12px}
#riderPanel .vehicle b{font-size:14px}
#riderPanel .vehicle small{font-size:12px}
#riderPanel .vehicle strong{font-size:15px;flex-shrink:0}
#riderPanel .movio-address-actions{display:grid!important;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px!important}
#riderPanel .movio-address-actions button{width:100%;min-height:44px;white-space:normal;line-height:1.4;font-size:12px;padding:10px}
#riderPanel .actions{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));align-items:stretch;gap:8px}
#riderPanel .actions>button{width:100%;min-height:46px;white-space:normal;line-height:1.4;justify-content:center;font-size:13px}
#riderPanel .movio-voice-actions{grid-template-columns:minmax(0,1fr) 48px;margin:10px 0 16px}
#riderPanel .movio-voice-actions #aiCall{background:#fff;border:1px solid #dcdfe4;color:#20242a;border-radius:12px;font-size:14px;min-height:48px;padding:12px}
#riderPanel .movio-voice-actions #voiceBook{width:48px;min-height:48px;border-radius:12px;font-size:20px}
#riderPanel .gps-card{flex-wrap:wrap}
#riderPanel .gps-card>div{flex:1 1 190px}
#riderPanel .gps-card button{white-space:normal;min-height:44px;line-height:1.4}
#riderPanel .tts-controls{display:grid;grid-template-columns:minmax(0,1fr);gap:8px}
#riderPanel .tts-controls select,#riderPanel .tts-controls button{width:100%;font-size:14px;min-height:44px;white-space:normal}
#riderPanel .booking-choice select{max-width:100%;font-size:14px}
#riderPanel .priority-trip{padding:18px!important}
#riderPanel .priority-trip #riderStatus{max-width:100%;font-size:11px}
#riderPanel .ride-card{overflow-wrap:anywhere}
#riderPanel #aiRideHistory,#riderPanel #weatherScheduledRides{display:grid;gap:10px}
#riderPanel #aiRideHistory .ride-card{display:flex;flex-direction:column;padding:15px;background:#fff}
#riderPanel #aiRideHistory .ride-card h3{font-size:14px;line-height:1.5}
#riderPanel #aiRideHistory .ride-card p{line-height:1.6}
#riderPanel .movio-card-actions{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px;padding-top:12px;margin-top:auto}
#riderPanel .movio-card-actions>button{width:100%;min-height:44px;margin:0;white-space:normal;line-height:1.4;font-size:13px}
#riderPanel .movio-card-actions>button:only-child{grid-column:1/-1}
#weatherManual{width:100%;margin:18px 0 0;padding:18px;border:1px solid #e4e6ea;border-radius:18px;background:#fafbfC;box-shadow:none;flex:none}
#weatherManual .wx-form-heading{font-size:15px;margin:0 0 15px;color:#20242a;line-height:1.5}
#weatherManual .wx-field-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px 12px}
#weatherManual .wx-field{display:flex;flex-direction:column;gap:7px;font-size:12px;font-weight:700;color:#4f5661;line-height:1.5}
#weatherManual input,#weatherManual select{display:block;width:100%;max-width:100%;min-height:46px;padding:11px 10px;border:1px solid #d9dde3;border-radius:10px;background:#fff;color:#20242a;font-size:16px;font-weight:500;line-height:1.2;box-shadow:none;color-scheme:light}
#weatherManual input:disabled,#weatherManual select:disabled{background:#eff1f4;color:#657080;opacity:.75}
#weatherManual .wx-schedule{min-width:0;padding:0;margin:16px 0 0;border:0}
#weatherManual .wx-schedule legend{padding:0;font-size:12px;font-weight:700;color:#4f5661;margin-bottom:8px}
#weatherManual .wx-day-shortcuts{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px;margin-bottom:12px}
#weatherManual .wx-day-shortcuts button{width:100%;border:1px solid #d9dde3;border-radius:999px;background:#fff;color:#4b535f;font-size:12px;min-height:36px;padding:7px 14px;font-weight:700;flex:none;white-space:normal}
#weatherManual .wx-day-shortcuts button[aria-pressed=true]{border-color:#e6b633;background:#fff3ca;color:#684900}
#weatherManual #wxManualDate[hidden]{display:none!important}
#weatherManual #wxPreviewAction{margin-top:18px}
#riderPanel #bookRide{display:flex;width:100%;height:auto;min-height:50px;align-items:center;justify-content:center;white-space:normal;line-height:1.4;font-size:15px;padding:13px 16px;background:#1c222b;color:#fff;border:0;border-radius:12px;flex:none}
#weatherManual #wxManualStatus{font-size:12px;line-height:1.6;color:#55606e;white-space:normal;margin:10px 0 0;overflow-wrap:anywhere}
#weatherManual #wxManualStatus:empty{display:none}
#weatherManual #wxManualReview{margin-top:16px}
#weatherManual #wxManualReview[hidden]{display:none!important}
#weatherManual .wx-review-panel{padding:15px;border:1px solid #e0e6e9;border-radius:14px;background:#fff}
#weatherManual .wx-review-head{display:flex;flex-wrap:wrap;align-items:center;justify-content:space-between;gap:8px;margin-bottom:12px}
#weatherManual #wxWeatherTitle{font-size:14px;line-height:1.5;margin:0;color:#28313d}
#weatherManual .wx-condition{display:inline-flex;flex-wrap:wrap;gap:6px;align-items:center;font-size:11px;line-height:1.5;font-weight:700;background:#eaf0f6;color:#3f566f;border-radius:8px;padding:5px 8px}
#weatherManual .wx-condition[data-tone=rain]{background:#e8f1fb;color:#235582}
#weatherManual .wx-condition[data-tone=heat]{background:#fff0d9;color:#865500}
#weatherManual .wx-condition[data-tone=clear]{background:#eef6ed;color:#38623a}
#weatherManual .wx-rain-label{font-size:11px;color:#657080;margin:0 0 7px}
#weatherManual .wx-weather-points{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px}
#weatherManual .wx-point{padding:12px 10px;border-radius:11px;background:#f5f7fa;display:grid;gap:5px;align-content:start}
#weatherManual .wx-point-label{color:#657080;font-size:11px;line-height:1.4}
#weatherManual .wx-place{display:block;color:#303a46;font-size:12px;font-weight:600;line-height:1.5;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
#weatherManual .wx-probability{font-size:25px;letter-spacing:-.5px;line-height:1.3;color:#1e334b}
#weatherManual .wx-probability[data-missing=true]{font-size:14px;line-height:2.3;letter-spacing:0;color:#697582}
#weatherManual .wx-tip{font-size:12px;color:#405165;background:#f0f4f9;border-radius:9px;padding:10px 11px;line-height:1.6;margin:11px 0 0}
#weatherManual .wx-tip:empty{display:none}
#weatherManual .wx-trip-panel{margin-top:12px;padding-top:12px;border-top:1px solid #e7eaee}
#weatherManual .wx-route-row{display:grid;grid-template-columns:64px minmax(0,1fr);align-items:baseline;gap:8px;font-size:12px;line-height:1.6;margin:5px 0}
#weatherManual .wx-route-row>span:first-child{color:#657080;font-size:11px}
#weatherManual .wx-trip-date{font-size:12px;color:#4a5664;line-height:1.6;margin:10px 0}
#weatherManual .wx-trip-facts{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:7px}
#weatherManual .wx-trip-facts>div{display:grid;gap:3px;border-radius:9px;background:#f5f7fa;padding:10px}
#weatherManual .wx-trip-facts small{font-size:11px;color:#657080;line-height:1.4}
#weatherManual .wx-trip-facts strong{font-size:14px;color:#26313e;line-height:1.5}
#weatherManual .wx-review-actions{display:grid;grid-template-columns:minmax(0,1.3fr) minmax(0,1fr);grid-auto-rows:minmax(52px,auto);gap:8px;margin-top:14px}
#weatherManual .wx-review-actions button{min-height:52px;height:auto;width:100%;font-size:13px;padding:11px 9px;line-height:1.5;white-space:normal;border-radius:11px;font-weight:700;justify-content:center;align-items:center;display:flex;flex:none}
#weatherManual #wxManualContinue{border:1px solid #1c222b;background:#1c222b;color:#fff}
#weatherManual #wxManualChange{border:1px solid #d9dde3;background:#fff;color:#354153}
#weatherManual .wx-details{margin:12px 0 0;font-size:12px;color:#576372}
#weatherManual .wx-details>summary{cursor:pointer;padding:7px 0;min-height:32px;color:#4b596a;font-weight:600;line-height:1.5}
#weatherManual .wx-details p{font-size:12px;line-height:1.7;color:#576372;white-space:pre-line!important;overflow-wrap:anywhere;margin:7px 0}
#weatherManual .wx-source{display:flex;justify-content:flex-end;align-items:center;gap:6px;font-size:10px;color:#718092;margin:10px 0 0;line-height:1.5}
#weatherManual .wx-source a{color:#546779;text-decoration:underline;text-underline-offset:2px}
#riderPanel [hidden]{display:none!important}
@media(min-width:901px){
 #appShell:has(#riderPanel:not(.hidden)){height:100dvh;min-height:520px}
 #appShell:has(#riderPanel:not(.hidden)) .workspace{display:flex;min-height:0;overflow:hidden}
 #appShell:has(#riderPanel:not(.hidden)) .map-area{min-width:0;flex:1;height:auto;order:0}
 #riderPanel{position:relative;bottom:auto;left:auto;right:auto;order:-1;max-height:none;flex:0 0 min(460px,47vw);width:min(460px,47vw);overflow-y:auto;overscroll-behavior:contain;border-radius:0;min-width:0}
}
@media(max-width:900px){
 #appShell:has(#riderPanel:not(.hidden)){height:auto;min-height:100dvh;max-width:100%}
 #appShell:has(#riderPanel:not(.hidden)) .top{height:auto;min-height:60px;flex:none;padding:12px 16px;gap:12px;flex-wrap:wrap}
 #appShell:has(#riderPanel:not(.hidden)) .account-top{min-width:0;max-width:100%;gap:8px}
 #appShell:has(#riderPanel:not(.hidden)) #accountLabel{max-width:180px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
 #appShell:has(#riderPanel:not(.hidden)) #logout{min-height:36px;padding:8px 12px}
 #appShell:has(#riderPanel:not(.hidden)) .workspace{display:flex;flex-direction:column;overflow:visible;flex:none;min-width:0}
 #appShell:has(#riderPanel:not(.hidden)) .map-area{width:100%;height:250px;min-height:210px;flex:none;order:-1}
 #appShell:has(#riderPanel:not(.hidden)) .map-tools{right:14px;top:15px}
 #appShell:has(#riderPanel:not(.hidden)) .live-pill{top:15px;bottom:auto;left:14px;max-width:calc(100% - 80px);font-size:10px}
 #riderPanel.sheet,#riderPanel.sheet.priority-panel{position:relative;bottom:auto;left:auto;right:auto;width:100%;max-width:none;min-width:0;max-height:none!important;overflow:visible;order:0;margin:0;padding:20px 16px max(28px,env(safe-area-inset-bottom))!important;border-radius:0;box-shadow:none}
 #riderPanel .handle{display:none}
 #riderPanel .sheet-head h1{font-size:28px}
 #riderPanel .vehicles{grid-template-columns:repeat(2,minmax(0,1fr))}
 #riderPanel .vehicle{flex-wrap:wrap;min-height:82px;padding:10px;gap:7px}
 #riderPanel .vehicle>span:first-child{font-size:22px}
 #riderPanel .vehicle>span:nth-child(2){flex:1}
 #riderPanel .vehicle strong{margin-left:auto;font-size:14px}
 #riderPanel .vehicle small{font-size:10px}
 #riderPanel .priority-trip{margin:0 0 24px!important}
 #riderPanel .priority-driver{flex-wrap:wrap}
}
@media(max-width:374px){
 #riderPanel.sheet,#riderPanel.sheet.priority-panel{padding-left:12px!important;padding-right:12px!important}
 #weatherManual{padding:13px}
 #weatherManual .wx-review-panel{padding:11px}
 #weatherManual .wx-field-grid{gap:12px 8px}
 #weatherManual .wx-schedule>.wx-field-grid{grid-template-columns:minmax(0,1fr)}
 #weatherManual input,#weatherManual select{padding-left:6px;padding-right:6px}
 #appShell:has(#riderPanel:not(.hidden)) #accountLabel{max-width:140px}
}
'''
    form = r'''<h3 class="wx-form-heading" id="wxFormHeading"></h3>
<div class="wx-field-grid">
 <label class="wx-field" for="wxManualLanguage"><span id="wxLanguageLabel"></span><select id="wxManualLanguage"><option value="Tamil">தமிழ்</option><option value="Tanglish">Tanglish</option><option value="English">English</option></select></label>
 <label class="wx-field" for="wxManualCount"><span id="wxCountLabel"></span><select id="wxManualCount"><option>1</option><option>2</option><option>3</option><option>4</option><option>5</option><option>6</option></select></label>
</div>
<fieldset class="wx-schedule"><legend id="wxDateLabel"></legend>
 <div class="wx-day-shortcuts"><button type="button" id="wxToday" aria-pressed="false"></button><button type="button" id="wxTomorrow" aria-pressed="false"></button></div>
 <div class="wx-field-grid">
  <label class="wx-field" for="wxPickupDay"><span id="wxDayLabel"></span><input id="wxPickupDay" type="date" required></label>
  <label class="wx-field" for="wxPickupTime"><span id="wxTimeLabel"></span><input id="wxPickupTime" type="time" step="60" required></label>
 </div>
 <input id="wxManualDate" type="datetime-local" hidden tabindex="-1" aria-hidden="true">
</fieldset>
<div id="wxPreviewAction"></div>
<p id="wxManualStatus" role="status" aria-live="polite"></p>
<div id="wxManualReview" hidden>
 <div class="wx-review-panel">
  <div class="wx-review-head"><h3 id="wxWeatherTitle"></h3><span id="wxCondition" class="wx-condition"></span></div>
  <p class="wx-rain-label" id="wxRainLabel"></p>
  <div class="wx-weather-points">
   <div class="wx-point"><span class="wx-point-label" id="wxPickupLabel"></span><span class="wx-place" id="wxPickupName"></span><strong class="wx-probability" id="wxPickupRain"></strong></div>
   <div class="wx-point"><span class="wx-point-label" id="wxDropLabel"></span><span class="wx-place" id="wxDropName"></span><strong class="wx-probability" id="wxDropRain"></strong></div>
  </div>
  <p class="wx-tip" id="wxShortAdvice"></p>
  <div class="wx-trip-panel">
   <div class="wx-route-row" hidden><span id="wxTripPickupLabel"></span><strong class="wx-place" id="wxTripPickup"></strong></div>
   <div class="wx-route-row" hidden><span id="wxTripDropLabel"></span><strong class="wx-place" id="wxTripDrop"></strong></div>
   <p class="wx-trip-date" id="wxTripDate"></p>
   <div class="wx-trip-facts" id="wxTripFacts"></div>
  </div>
  <details class="wx-details"><summary id="wxFullDetailsLabel"></summary><p id="wxManualAdvice"></p><p id="wxManualSummary"></p></details>
  <div class="wx-review-actions"><button type="button" id="wxManualContinue"></button><button type="button" id="wxManualChange" class="secondary"></button></div>
  <p class="wx-source"><span id="wxForecastScope"></span><a href="https://open-meteo.com/" target="_blank" rel="noopener">Open-Meteo</a></p>
 </div>
</div>
'''
    helpers = r''' // MOVIO_CUSTOMER_UI_V1: presentation helpers; no booking/API/voice replacement.
 const uiMessages={
  English:{heading:'Plan your ride',language:'Language',count:'Passengers',date:'Pickup schedule',review:'Review ride',title:'Journey weather',continue:'Confirm booking',change:'Edit details',loading:'Checking weather…',ready:'Review your ride, then confirm.',changed:'Details changed. Review again.',today:'Today',tomorrow:'Tomorrow',day:'Date',timeLabel:'Time',rain:'Chance of rain',pickup:'Pickup',drop:'Drop-off',details:'Full details',scope:'At pickup time ·',passengers:'Passengers',vehicle:'Vehicle',distance:'Road distance',fare:'Estimated fare',missing:'Unavailable',forecast:'Forecast ready',rainy:'Rain expected',heavy:'Heavy rain',storm:'Thunderstorms',cloudy:'Cloudy',clear:'Clear skies',heat:'High heat',unavailable:'Forecast unavailable',notYet:'Forecast not yet available',canContinue:'You can still book this ride.',bikeTip:'Consider a covered car. The choice is yours.',severeBike:'Bike travel may be unsuitable. Consider a covered car.',rainTip:'Carry an umbrella for pickup and drop-off.',autoTip:'Protect yourself and your bags from side rain.',heatTip:'Carry water and prepare for high temperatures.',ai:'Book by AI call'},
  Tamil:{heading:'உங்கள் பயணம்',language:'மொழி',count:'பயணிகள்',date:'புறப்படும் தேதி, நேரம்',review:'பயணத்தைச் சரிபார்',title:'பயண வானிலை',continue:'முன்பதிவை உறுதிசெய்',change:'விவரங்களை மாற்று',loading:'வானிலையைச் சரிபார்க்கிறோம்…',ready:'விவரங்களைப் பார்த்து உறுதிசெய்யவும்.',changed:'விவரங்கள் மாறியுள்ளன. மீண்டும் சரிபார்க்கவும்.',today:'இன்று',tomorrow:'நாளை',day:'தேதி',timeLabel:'நேரம்',rain:'மழைக்கான வாய்ப்பு',pickup:'புறப்படும் இடம்',drop:'சேரும் இடம்',details:'முழு விவரங்கள்',scope:'புறப்படும் நேரத்தில் ·',passengers:'பயணிகள்',vehicle:'வாகனம்',distance:'சாலை தூரம்',fare:'கட்டண மதிப்பீடு',missing:'கிடைக்கவில்லை',forecast:'வானிலை தகவல்',rainy:'மழை எதிர்பார்ப்பு',heavy:'கனமழை',storm:'இடியுடன் மழை',cloudy:'மேகமூட்டம்',clear:'தெளிவான வானம்',heat:'அதிக வெப்பம்',unavailable:'வானிலை கிடைக்கவில்லை',notYet:'இந்தத் தேதிக்கு வானிலை இன்னும் கிடைக்கவில்லை',canContinue:'முன்பதிவைத் தொடரலாம்.',bikeTip:'மூடிய காரைத் தேர்வு செய்யலாம். முடிவு உங்களுடையது.',severeBike:'பைக் பயணம் பொருத்தமில்லாமல் இருக்கலாம். மூடிய காரைப் பரிசீலிக்கவும்.',rainTip:'ஏறும்போதும் இறங்கும்போதும் பயன்படுத்தக் குடை எடுத்துக்கொள்ளவும்.',autoTip:'பக்கவாட்டு மழையிலிருந்து உங்களையும் உடைமைகளையும் பாதுகாக்கவும்.',heatTip:'குடிநீர் எடுத்துக்கொண்டு வெப்பத்திற்கு ஏற்ற முன்னேற்பாடுகளைச் செய்யவும்.',ai:'AI அழைப்பு மூலம் முன்பதிவு'},
  Tanglish:{heading:'Unga ride details',language:'Language',count:'Passengers',date:'Pickup date & time',review:'Ride review pannunga',title:'Journey weather',continue:'Booking confirm pannunga',change:'Details maathunga',loading:'Weather check pannrom…',ready:'Details paarthu confirm pannunga.',changed:'Details maariduchu. Thirumba review pannunga.',today:'Today',tomorrow:'Tomorrow',day:'Date',timeLabel:'Time',rain:'Rain chance',pickup:'Pickup',drop:'Drop-off',details:'Full details',scope:'Pickup nerathula ·',passengers:'Passengers',vehicle:'Vehicle',distance:'Road distance',fare:'Estimated fare',missing:'Kidaikkavillai',forecast:'Weather details',rainy:'Rain expected',heavy:'Heavy rain',storm:'Thunderstorms',cloudy:'Cloudy',clear:'Clear skies',heat:'High heat',unavailable:'Forecast kidaikkavillai',notYet:'Indha date-ku forecast innum varala',canContinue:'Booking continue pannalaam.',bikeTip:'Covered car consider pannunga. Unga choice.',severeBike:'Bike suitable illaama irukkalaam. Covered car consider pannunga.',rainTip:'Pickup, drop-off-ku umbrella eduthukkonga.',autoTip:'Side rain-la irundhu ungalaiyum bags-aiyum protect pannunga.',heatTip:'Water eduthukkonga; heat-ku prepare pannikkonga.',ai:'AI call-la book pannunga'}
 };
 for(const lang of Object.keys(uiMessages))Object.assign(messages[lang],uiMessages[lang]);
 function uiWords(){return uiMessages[langBox.value]||uiMessages.English;}
 function compactLabels(){
  const m=uiWords();
  for(const [id,key] of [['wxFormHeading','heading'],['wxToday','today'],['wxTomorrow','tomorrow'],['wxDayLabel','day'],['wxTimeLabel','timeLabel'],['wxRainLabel','rain'],['wxPickupLabel','pickup'],['wxDropLabel','drop'],['wxTripPickupLabel','pickup'],['wxTripDropLabel','drop'],['wxFullDetailsLabel','details'],['wxForecastScope','scope']])el(id).textContent=m[key];
  const ai=el('aiCall');if(ai)ai.textContent=m.ai;
 }
 function indiaDay(offset=0){
  const parts=new Intl.DateTimeFormat('en-CA',{timeZone:'Asia/Kolkata',year:'numeric',month:'2-digit',day:'2-digit'}).formatToParts(new Date());
  const p=Object.fromEntries(parts.map(x=>[x.type,x.value]));
  return new Date(Date.UTC(Number(p.year),Number(p.month)-1,Number(p.day)+offset)).toISOString().slice(0,10);
 }
 function syncDayButtons(){
  for(const [id,offset] of [['wxToday',0],['wxTomorrow',1]])el(id).setAttribute('aria-pressed',String(el('wxPickupDay').value===indiaDay(offset)));
 }
 function setupDateInputs(){
  const day=el('wxPickupDay'),clock=el('wxPickupTime');
  function fromLegacy(){
   const v=dateBox.value.split('T');
   day.value=v[0]||day.value||indiaDay();clock.value=v[1]?.slice(0,5)||'';syncDayButtons();
  }
  function toLegacy(){
   if(dateBox.disabled)return fromLegacy();
   dateBox.value=day.value&&clock.value?day.value+'T'+clock.value:'';
   syncDayButtons();dateBox.dispatchEvent(new Event('change',{bubbles:true}));
  }
  fromLegacy();day.min=indiaDay();
  for(const node of [day,clock])node.addEventListener('change',toLegacy);
  dateBox.addEventListener('change',()=>{if(dateBox.value)fromLegacy();else if(!day.value||!clock.value)syncDayButtons();});
  for(const [id,offset] of [['wxToday',0],['wxTomorrow',1]])el(id).onclick=()=>{if(dateBox.disabled)return;day.value=indiaDay(offset);day.min=indiaDay();toLegacy();};
  // Keep the min-date current if the dashboard stays open overnight.
  day.addEventListener('focus',()=>{day.min=indiaDay();syncDayButtons();});
 }
 function shortPlace(value){
  const name=String(value||'').trim();
  // A pinned coordinate pair must remain intact; addresses can use the first part.
  return /^Map pin\s*\(/i.test(name)?name:(name.split(',')[0].trim()||name);
 }
 function placeLabel(id,value){const node=el(id);node.textContent=shortPlace(value);node.title=String(value||'');}
 function actualWeather(result){
  const lines=String(result.advisory||'').split(/\r?\n/).map(x=>x.trim()).filter(Boolean);
  const weather=result.weather||{};
  const status=weather.status||result.weather_status;
  const ok=status==='ok';
  // v0.4 supplies the two probabilities as the first two advisory lines.
  // Read each endpoint separately: a missing pickup value must not become drop-off's.
  // Do not infer probabilities from weather type or turn missing values into zero.
  function probability(key,index){
   if(!ok)return null;
   const point=Array.isArray(weather.points)?weather.points.find(p=>p.location===key):null;
   let value=point?point.precipitation_probability:null;
   if(!point){const matches=[...(lines[index]||'').matchAll(/(\d+(?:\.\d+)?)\s*%/g)];value=matches.length?Number(matches.at(-1)[1]):null;}
   return value!==null&&value!==''&&Number.isFinite(Number(value))&&Number(value)>=0&&Number(value)<=100?Number(value):null;
  }
  // Legacy manual responses contain the weather description after both locations.
  // Match the server's actual descriptions; percentage alone is not a condition.
  const report=ok?lines.slice(2).join('\n'):'';
  const storm=!!weather.thunderstorm||/Thunderstorms are forecast|இடியுடன் கூடிய மழை|idiyudan koodiya mazhai/i.test(report);
  const heavy=!!weather.heavy_rain||/Heavy rain is forecast|கனமழை|kanamazhai/i.test(report);
  const heat=!!weather.heat||/Very hot conditions|வெப்பநிலை அதிகமாக|Veppanilai adhigamaaga/i.test(report);
  const rain=weather.condition==='rain'||storm||heavy||/Rain is expected|மழை எதிர்பார்க்கப்படுகிறது|mazhai peiya vaaippu irukkiradhu/i.test(report);
  const cloudy=weather.condition==='cloudy'||/Cloudy conditions|மேகமூட்டம்|megamoottam/i.test(report);
  const clear=weather.condition==='clear'||/Clear or mainly clear|weather looks pleasant|வானம் தெளிவாக|வானிலை நன்றாக|vaanam thelivaaga|vaanilai nandraagave/i.test(report);
  return {ok,status,pickup:probability('pickup',0),destination:probability('destination',1),storm,heavy,heat,rain,cloudy,clear};
 }
 function compactReview(result,data){
  const m=uiWords(),w=actualWeather(result),condition=el('wxCondition');
  const type=!w.ok?'unavailable':w.storm?'storm':w.heavy?'heavy':w.rain?'rainy':w.cloudy?'cloudy':w.clear?'clear':'forecast';
  condition.textContent=w.status==='outside_forecast_range'?m.notYet:m[type];
  if(w.heat)condition.textContent=type==='forecast'?m.heat:condition.textContent+' · '+m.heat;
  condition.dataset.tone=w.storm||w.heavy||w.rain?'rain':w.heat?'heat':w.clear?'clear':'neutral';
  for(const [id,key] of [['wxPickupRain','pickup'],['wxDropRain','destination']]){
   const node=el(id),value=w[key];node.textContent=value===null?m.missing:String(value)+'%';node.dataset.missing=String(value===null);
  }
  placeLabel('wxPickupName',data.pickup);placeLabel('wxDropName',data.destination);
  placeLabel('wxTripPickup',data.pickup);placeLabel('wxTripDrop',data.destination);
  const tips=[];
  if(!w.ok)tips.push(m.canContinue);
  else if(w.rain)tips.push(data.vehicle==='Bike'?(w.storm||w.heavy?m.severeBike:m.bikeTip):data.vehicle==='Auto'?m.autoTip:m.rainTip);
  if(w.heat)tips.push(m.heatTip);
  el('wxShortAdvice').textContent=tips.join(' ');
  const raw=String(result.pickup_at||data.pickup_at||'');
  const match=/^(\d{4})-(\d{2})-(\d{2})T(\d{2}):(\d{2})/.exec(raw);
  if(match){
   // Display the selected Indian wall time as entered, even in another browser timezone.
   const d=new Date(Number(match[1]),Number(match[2])-1,Number(match[3]),Number(match[4]),Number(match[5]));
   const locale=data.language==='Tamil'?'ta-IN':'en-IN';
   el('wxTripDate').textContent=new Intl.DateTimeFormat(locale,{day:'numeric',month:'short',year:'numeric',hour:'numeric',minute:'2-digit',hour12:true}).format(d);
  }else el('wxTripDate').textContent=raw;
  const facts=el('wxTripFacts');facts.replaceChildren();
  const amount=result.fare??data.fare;
  const money=amount!==null&&amount!==''&&Number.isFinite(Number(amount))?new Intl.NumberFormat('en-IN',{style:'currency',currency:'INR',maximumFractionDigits:0}).format(Number(amount)):m.missing;
  const distance=data.km!==null&&data.km!==undefined&&Number.isFinite(Number(data.km))?Number(data.km).toFixed(1)+' km':m.missing;
  for(const [label,value] of [[m.passengers,String(data.count)],[m.vehicle,result.vehicle||data.vehicle],[m.distance,distance],[m.fare,money]]){
   const box=document.createElement('div'),caption=document.createElement('small'),text=document.createElement('strong');
   caption.textContent=label;text.textContent=value;box.append(caption,text);facts.append(box);
  }
  el('wxManualAdvice').textContent=result.advisory||'';el('wxManualSummary').textContent=result.summary||'';
  el('wxManualReview').querySelector('details').open=false;
 }
 function tidyCustomerActions(){
  const row=el('aiCall')?.closest('.actions');if(row)row.classList.add('movio-voice-actions');
  const mic=el('voiceBook');if(mic){mic.setAttribute('aria-label','Dictate destination');mic.title='Dictate destination';}
  const address=el('searchAddress')?.parentElement;if(address)address.classList.add('movio-address-actions');
  for(const card of document.querySelectorAll('#aiRideHistory .ride-card')){
   const direct=[...card.children].filter(n=>n.tagName==='BUTTON');if(!direct.length)continue;
   let actions=[...card.children].find(n=>n.classList.contains('movio-card-actions'));
   if(!actions){actions=document.createElement('div');actions.className='movio-card-actions';card.append(actions);}
   actions.append(...direct);
  }
 }
 function watchCustomerCards(){
  tidyCustomerActions();const history=el('aiRideHistory');
  if(history)new MutationObserver(tidyCustomerActions).observe(history,{childList:true,subtree:true});
 }
'''
    marker = 'MOVIO_CUSTOMER_UI_V1'

    if marker in manual_before:
        manual = manual_before
    else:
        manual = manual_before

        def replace_once(old, new, label):
            nonlocal manual
            if manual.count(old) != 1:
                raise RuntimeError('The manual booking code differs at '+label+'. No files changed.')
            manual = manual.replace(old, new, 1)

        start = " const section=document.createElement('section');"
        end = " const el=id=>document.getElementById(id),langBox="
        if manual.count(start) != 1 or manual.count(end) != 1:
            raise RuntimeError('Unsupported manual booking form. No files changed.')
        a, z = manual.index(start), manual.index(end)
        if z <= a:
            raise RuntimeError('Unexpected manual booking source order. No files changed.')
        new_section = (
            " // MOVIO_CUSTOMER_UI_V1\n"
            " const section=document.createElement('section');section.id='weatherManual';section.className='ride-card';\n"
            ' section.innerHTML=`'+form+'`;\n'
            " const bookingActions=book.closest('.actions')||book.parentElement;\n"
            " bookingActions.parentNode.insertBefore(section,bookingActions);\n"
            " section.querySelector('#wxPreviewAction').append(book);\n"
        )
        manual = manual[:a] + new_section + manual[z:]
        replace_once(' let review=null,serial=0,previewBusy=false,saving=false,uncertain=false;',
                     helpers+'\n let review=null,serial=0,previewBusy=false,saving=false,uncertain=false;',
                     'presentation helpers')
        replace_once("if(!saving&&!previewBusy)book.textContent=m.review;}",
                     "if(!saving&&!previewBusy)book.textContent=m.review;compactLabels();}",
                     'localized labels')
        replace_once("return /tanglish/i.test(value)?'Tanglish':", "return /tanglish|^auto$/i.test(value)?'Tanglish':",
                     'manual language selection')
        replace_once("const controls=[book,langBox,dateBox,countBox,el('pickup'),el('destination'),el('wxManualContinue'),el('wxManualChange')].filter(Boolean);",
                     "const controls=[book,langBox,dateBox,countBox,el('wxPickupDay'),el('wxPickupTime'),el('wxToday'),el('wxTomorrow'),el('pickup'),el('destination'),el('wxManualContinue'),el('wxManualChange')].filter(Boolean);",
                     'save lock for date/time controls')
        replace_once("dateBox.disabled=true;countBox.disabled=true;el('wxManualContinue').disabled=false;",
                     "dateBox.disabled=true;countBox.disabled=true;for(const id of ['wxPickupDay','wxPickupTime','wxToday','wxTomorrow'])el(id).disabled=true;el('wxManualContinue').disabled=false;",
                     'uncertain-save lock')
        replace_once("el('wxManualAdvice').textContent=result.advisory;el('wxManualSummary').textContent=result.summary;el('wxManualReview').hidden=false;status(words().ready);",
                     "compactReview(result,data);el('wxManualReview').hidden=false;status(words().ready);",
                     'compact weather and booking review')
        replace_once("el('wxManualChange').onclick=()=>{invalidate();dateBox.focus();};",
                     "el('wxManualChange').onclick=()=>{invalidate();el('wxPickupDay').focus();};",
                     'edit-details focus')
        replace_once("if(!dateBox.value){status(words().time);return;}",
                     "if(!dateBox.value){status(words().time);(el('wxPickupDay').value?el('wxPickupTime'):el('wxPickupDay')).focus();return;}",
                     'missing schedule focus')
        replace_once(" labels();\n})();", " setupDateInputs();watchCustomerCards();labels();\n})();", 'form startup')

    revision = hashlib.sha256((css+manual).encode('utf-8')).hexdigest()[:12]
    # Only the edited asset gets a new cache key. Keep all existing app/call scripts.
    script_pattern = r'(<script\b[^>]*\bsrc=["\x27])([^"\x27]*\bweather_manual\.js)(?:\?[^"\x27]*)?(["\x27][^>]*>\s*</script\s*>)'
    script_rx = re.compile(script_pattern, re.I)
    if len(script_rx.findall(html_before)) > 1:
        raise RuntimeError('Multiple manual-weather scripts found. No files changed.')
    if script_rx.search(html_before):
        html = script_rx.sub(lambda m:m[1]+m[2]+'?customer_ui='+revision+m[3],html_before)
    else:
        tag = '<script src="/weather_manual.js?customer_ui='+revision+'" defer></script>\n'
        html = re.sub(r'</body\s*>',lambda m:tag+m[0],html_before,count=1,flags=re.I)
    # Last stylesheet wins over the older mobile/driver-priority rules.
    html = re.sub(r'<link\b[^>]*\bhref=["\x27][^"\x27]*\bmovio_customer_dashboard\.css(?:\?[^"\x27]*)?["\x27][^>]*>\s*', '', html, flags=re.I)
    css_tag = '<link rel="stylesheet" href="/movio_customer_dashboard.css?v='+revision+'">\n'
    html = re.sub(r'</head\s*>',lambda m:css_tag+m[0],html,count=1,flags=re.I)

    # Validate the modified script before writing anything.
    node = shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory(prefix='movio_ui_check_') as temp:
            check = Path(temp) / 'weather_manual.js'
            check.write_text(manual, encoding='utf-8')
            result = subprocess.run([node, '--check', str(check)],capture_output=True,text=True)
            if result.returncode:
                raise RuntimeError('JavaScript check failed. No files changed.\n'+result.stderr)

    changes = [(manual_file,manual),(css_file,css),(index_file,html)]
    changes = [(p,text) for p,text in changes if not p.exists() or p.read_text(encoding='utf-8')!=text]
    backup = None
    if changes:
        backup = root / ('customer_ui_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        backup.mkdir()
        for path,_ in changes:
            if path.exists():
                dest = backup / path.relative_to(root)
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(path,dest)
        written = []
        try:
            for path,text in changes:
                tmp = path.with_name(path.name+'.ui-tmp')
                try:
                    tmp.write_text(text,encoding='utf-8')
                    os.replace(tmp,path)
                finally:
                    tmp.unlink(missing_ok=True)
                written.append(path)
        except Exception:
            for path in reversed(written):
                saved = backup / path.relative_to(root)
                if saved.exists():
                    shutil.copy2(saved,path)
                else:
                    path.unlink(missing_ok=True)
            raise

    print('Customer dashboard UI installed.' if changes else 'Customer dashboard UI is already installed.')
    print('Responsive buttons; separate date/time; Today/Tomorrow; compact weather and ride review.')
    print('Refresh the dashboard (Ctrl+Shift+R on a laptop; reload the page on mobile).')
    if backup:
        print('Backup:',backup)
    return {'version':marker,'changed_files':[p.name for p,_ in changes],'backup':str(backup) if backup else None}

MOVIO_CUSTOMER_UI_STATUS = movio_refresh_customer_dashboard()


Customer dashboard UI installed.
Responsive buttons; separate date/time; Today/Tomorrow; compact weather and ride review.
Refresh the dashboard (Ctrl+Shift+R on a laptop; reload the page on mobile).
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/customer_ui_backup_20261028_070659_675101


In [86]:
# Paste this entire file into ONE Kaggle code cell after your existing voice,
# weather and customer-dashboard fixes, BEFORE evaluation. Then open a NEW call.
def movio_match_ai_call_theme():
    from pathlib import Path
    from datetime import datetime
    import os
    import re
    import shutil
    import subprocess
    import tempfile

    bridge_object = globals().get('bridge')
    if bridge_object is None or not isinstance(getattr(bridge_object, 'HTML', None), str):
        raise RuntimeError('Run your existing voice/weather activation first, then this UI cell.')
    root = Path(globals().get('WX_ROOT', '/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    if not (root / 'integrations').is_dir():
        raise FileNotFoundError('The existing MOVIO integrations folder was not found.')

    css = r'''
/* MOVIO_CALL_THEME_V1 */
:root{--call-ink:#1c222b;--call-muted:#68727e;--call-line:#e3e6eb;--call-yellow:#ffcc29}
*{box-sizing:border-box}
body{margin:0;padding:24px 16px;background:#f5f5f3;color:var(--call-ink);font-family:'DM Sans',system-ui,-apple-system,'Segoe UI',sans-serif;font-size:15px;display:block;min-height:100vh;min-height:100dvh}
.phone{width:100%;max-width:660px;min-width:0;margin:0 auto;padding:26px;background:#fff;border:1px solid var(--call-line);border-radius:24px;box-shadow:0 10px 36px #1c222b08}
.phone *{min-width:0}
.movio-call-brand{display:flex;align-items:center;gap:12px;margin:0 0 10px}
.movio-call-mark{display:block;text-align:center;flex:none;width:38px;height:38px;line-height:38px;border-radius:13px;background:var(--call-yellow);color:#17191d;font-size:22px;font-weight:800;letter-spacing:-1px}
.phone h2{flex:1;font-size:23px;font-weight:800;letter-spacing:-.6px;line-height:1.35;margin:0;color:var(--call-ink)}
.phone .hint{font-size:12px;line-height:1.65;color:var(--call-muted);margin:10px 0 17px;overflow-wrap:anywhere}
.phone button,.phone input,.phone select{font-family:inherit;max-width:100%;border-radius:12px;font-size:14px;line-height:1.5}
.phone button{min-height:46px;padding:12px 14px;border:1px solid var(--call-line);background:#fff;color:var(--call-ink);font-weight:700;cursor:pointer;white-space:normal;touch-action:manipulation}
.phone button:hover:not(:disabled){filter:brightness(.98)}
.phone button:disabled{opacity:.5;cursor:not-allowed}
.phone button:focus-visible,.phone select:focus-visible,.phone input:focus-visible{outline:3px solid #d59d00;outline-offset:3px}
.phone select,.phone input{width:100%;min-height:48px;padding:12px;border:1px solid #d9dde3;background:#fff;color:var(--call-ink);font-size:16px;font-weight:400;box-shadow:none;color-scheme:light}
.movio-call-language{display:grid;gap:7px;margin:16px 0 14px;font-size:12px;font-weight:700;color:#535e6b}
.phone .row{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:9px;margin:12px 0;align-items:stretch}
.phone .row>*{min-width:0;flex:none}
.phone .row>button{width:100%}
.phone .movio-call-modes{gap:10px;grid-auto-rows:minmax(64px,auto);margin-bottom:6px}
.phone .movio-call-modes button{position:relative;min-height:64px;padding:13px 30px 13px 12px;border:1px solid #d9dde3;background:#fff;color:#657080;line-height:1.5;font-size:14px;text-align:left}
.phone #bookMode[aria-pressed=true]{background:#fff3c4;border:2px solid #dfb12c;color:#624800;padding:12px 29px 12px 11px}
.phone #cancelMode[aria-pressed=true]{background:#fff0eb;border:2px solid #de9e86;color:#823f2d;padding:12px 29px 12px 11px}
.phone .movio-call-modes button[aria-pressed=true]::after{content:'✓';position:absolute;right:11px;top:50%;transform:translateY(-50%);font-size:16px;font-weight:800}
.phone .movio-call-modes button[aria-pressed=true]:disabled{opacity:1;cursor:default}
.movio-call-selected{margin:0 0 17px;font-size:12px;color:#657080;line-height:1.5}
.phone .movio-call-actions{margin-top:10px;grid-auto-rows:minmax(70px,auto)}
.phone #call{display:block;width:100%;min-height:50px;margin:0;background:var(--call-ink);color:#fff;border:1px solid var(--call-ink);font-size:15px}
.phone #interrupt{background:#fff9e8;border:1px solid #eedca1;color:#755721}
.phone #hang{background:#fff2ee;border:1px solid #ecd4cd;color:#954c3b}
.phone #status,.phone #weather{padding:12px 14px;margin:15px 0;background:#f4f6f9;border:1px solid #e8ebef;color:#405165;border-radius:12px;font-size:13px;line-height:1.6;overflow-wrap:anywhere}
.phone #rides{white-space:pre-line;font-size:13px;line-height:1.65;color:#465364;background:#fafbfc;border-radius:12px;padding:12px;margin:12px 0;border:1px solid #e8ebef;overflow-wrap:anywhere}
.phone #rides:empty{display:none}
.phone #log{height:300px;height:clamp(220px,36dvh,350px);overflow:auto;overscroll-behavior:contain;background:#fafbfc;border:1px solid #e5e8ec;border-radius:15px;padding:13px;white-space:pre-wrap;overflow-wrap:anywhere;line-height:1.7;font-size:14px;color:#303b49}
.phone #log:empty::before{content:attr(data-empty-label);display:block;color:#78838f;font-size:13px;padding:6px}
.phone #log>div{padding:10px 12px;border-radius:12px;margin:0 0 10px}
.phone #log>div:last-child{margin-bottom:0}
.phone #log .you{background:#fff3c8;border:1px solid #f3e4b6;color:#56471f;margin-left:18px}
.phone #log .ai{background:#fff;border:1px solid #e2e7ec;color:#2c3b4d;margin-right:18px}
.phone #log .error{background:#fff1ec;border:1px solid #f0dcd3;color:#8d4430}
.phone #manual{margin-top:14px;width:100%;background:#fff;color:var(--call-ink)}
.phone #send{background:var(--call-ink);border-color:var(--call-ink);color:white}
.phone #readSummary{background:#f4f6f9;border:1px solid #dce2e8;color:#405165}
.phone .movio-call-replies{display:flex;flex-wrap:wrap}
.phone .movio-call-replies>button{flex:1 1 170px;width:auto}
.phone a{color:#785900;text-decoration:underline;text-underline-offset:3px}
.phone>.hint:last-child{font-size:11px;margin:16px 0 0;color:#75808c}
@media(max-width:480px){body{padding:12px}.phone{padding:18px 15px;border-radius:19px}.phone h2{font-size:21px}.phone .movio-call-modes button{font-size:13px}.phone .row{gap:8px}.phone button{padding:11px 10px;font-size:13px}.phone #log{font-size:13px}.phone #log .ai{margin-right:8px}.phone #log .you{margin-left:8px}}
@media(max-width:360px){.phone .movio-call-modes,.phone .movio-call-actions{grid-template-columns:minmax(0,1fr);grid-auto-rows:minmax(48px,auto)}.phone .movio-call-modes button{min-height:48px}}
@media(prefers-reduced-motion:reduce){.phone *{scroll-behavior:auto!important}}
'''

    javascript = r'''
/* MOVIO_CALL_THEME_V1: presentation only; preserve every existing call handler. */
(function(){
 'use strict';
 const byId=id=>document.getElementById(id),book=byId('bookMode'),cancel=byId('cancelMode'),language=byId('lang');
 if(!book||!cancel||!language||document.getElementById('movioCallSelected'))return;
 const panel=book.closest('.phone');if(!panel)return;
 const heading=panel.querySelector('h2');
 if(heading){const brand=document.createElement('div'),mark=document.createElement('span');brand.className='movio-call-brand';mark.className='movio-call-mark';mark.textContent='M';mark.setAttribute('aria-hidden','true');heading.before(brand);brand.append(mark,heading);}
 const languageLabel=document.createElement('label'),caption=document.createElement('span');
 languageLabel.className='movio-call-language';languageLabel.htmlFor='lang';caption.id='movioCallLanguageLabel';language.before(languageLabel);languageLabel.append(caption,language);
 const modes=book.parentElement;modes.classList.add('movio-call-modes');modes.setAttribute('role','group');
 const note=document.createElement('p');note.id='movioCallSelected';note.className='movio-call-selected';note.setAttribute('role','status');note.setAttribute('aria-live','polite');modes.after(note);
 const startButton=byId('call'),actions=startButton.parentElement;actions.classList.add('movio-call-actions');actions.before(startButton);
 byId('send')?.parentElement.classList.add('movio-call-replies');
 const copy={
  English:{language:'Language',purpose:'Call purpose',book:'Ride booking selected',cancel:'Booking cancellation selected',empty:'Your conversation will appear here.',placeholder:'Type a reply or correction'},
  Tamil:{language:'மொழி',purpose:'அழைப்பின் நோக்கம்',book:'பயண முன்பதிவு தேர்ந்தெடுக்கப்பட்டுள்ளது',cancel:'முன்பதிவு ரத்து தேர்ந்தெடுக்கப்பட்டுள்ளது',empty:'உங்கள் உரையாடல் இங்கே தோன்றும்.',placeholder:'பதில் அல்லது திருத்தத்தை எழுதுங்கள்'},
  Tanglish:{language:'Language',purpose:'Call purpose',book:'Ride booking selected',cancel:'Booking cancellation selected',empty:'Unga conversation inga varum.',placeholder:'Reply illa correction type pannunga'}
 };
 const tanglish=['Ride book pannu','Booking cancel pannu','Call start','Niruthi detail change pannu','Call mudikkavum'];
 if(typeof labels==='object'&&Array.isArray(labels.Tanglish))tanglish.forEach((text,index)=>{labels.Tanglish[index]=text;});
 let selected='book';
 function update(requested){
  // Read the actual existing mode; do not change it or replace its click handlers.
  selected=typeof mode==='string'&&['book','cancel'].includes(mode)?mode:requested||selected;
  const text=copy[language.value]||copy.English;
  if(language.value==='Tanglish')for(const [index,id] of ['bookMode','cancelMode','call','interrupt','hang'].entries())byId(id).textContent=tanglish[index];
  for(const [button,value] of [[book,'book'],[cancel,'cancel']]){
   button.setAttribute('aria-pressed',String(selected===value));button.setAttribute('aria-label',button.textContent);button.setAttribute('aria-describedby',note.id);
  }
  modes.setAttribute('aria-label',text.purpose);caption.textContent=text.language;note.textContent=text[selected];
  const log=byId('log');if(log)log.setAttribute('data-empty-label',text.empty);
  const input=byId('manual');if(input){input.placeholder=text.placeholder;input.setAttribute('aria-label',text.placeholder);}
 }
 // Existing onclick/onchange handlers run first and retain all booking/account logic.
 book.addEventListener('click',()=>update('book'));cancel.addEventListener('click',()=>update('cancel'));
 language.addEventListener('change',()=>update());update();
})();
'''

    style_block = '<!-- MOVIO_CALL_THEME_STYLE_BEGIN --><style id="movioCallTheme">'+css+'</style><!-- MOVIO_CALL_THEME_STYLE_END -->'
    script_block = '<!-- MOVIO_CALL_THEME_SCRIPT_BEGIN --><script id="movioCallThemeScript">'+javascript+'</script><!-- MOVIO_CALL_THEME_SCRIPT_END -->'

    def patched(html):
        for name in ('lang','bookMode','cancelMode','call','interrupt','hang','log','manual'):
            if not re.search(r'\bid\s*=\s*["\x27]'+name+r'["\x27]',html):
                raise RuntimeError('Unexpected AI Call page: missing '+name+'. No files changed.')
        if not re.search(r'</head\s*>',html,re.I) or not re.search(r'</body\s*>',html,re.I):
            raise RuntimeError('Unexpected AI Call HTML. No files changed.')
        for part in ('STYLE','SCRIPT'):
            html=re.sub(r'<!-- MOVIO_CALL_THEME_'+part+r'_BEGIN -->.*?<!-- MOVIO_CALL_THEME_'+part+r'_END -->','',html,flags=re.S)
        html=re.sub(r'</head\s*>',lambda m:style_block+m[0],html,count=1,flags=re.I)
        return re.sub(r'</body\s*>',lambda m:script_block+m[0],html,count=1,flags=re.I)

    active_before=bridge_object.HTML
    active_after=patched(active_before)
    updates=[]
    for name in ('ai_call_page.html','weather_call_page.html'):
        path=root/'integrations'/name
        if path.is_file():
            before=path.read_text(encoding='utf-8');after=patched(before)
            if before!=after:
                updates.append((path,after))

    # Check the added script and unchanged protocol scripts together before writing.
    node=shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory(prefix='movio_call_ui_check_') as folder:
            for index,html in enumerate([active_after]+[text for _,text in updates]):
                check=Path(folder)/('call_'+str(index)+'.js')
                check.write_text('\n'.join(re.findall(r'<script\b[^>]*>(.*?)</script\s*>',html,flags=re.I|re.S)),encoding='utf-8')
                result=subprocess.run([node,'--check',str(check)],capture_output=True,text=True)
                if result.returncode:
                    raise RuntimeError('JavaScript check failed. No files changed.\n'+result.stderr)

    backup=None
    if active_before!=active_after or updates:
        backup=root/('ai_call_ui_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        backup.mkdir()
        (backup/'active_call_page.html').write_text(active_before,encoding='utf-8')
        for path,_ in updates:
            shutil.copy2(path,backup/path.name)
        written=[]
        try:
            for path,html in updates:
                temporary=path.with_name(path.name+'.theme-tmp')
                try:
                    temporary.write_text(html,encoding='utf-8');os.replace(temporary,path)
                finally:
                    temporary.unlink(missing_ok=True)
                written.append(path)
            bridge_object.HTML=active_after
        except Exception:
            for path in reversed(written):
                shutil.copy2(backup/path.name,path)
            bridge_object.HTML=active_before
            raise
    print('AI Call dashboard theme installed.' if backup else 'AI Call dashboard theme is already installed.')
    print('Selected Booking: soft yellow + check. Selected Cancellation: soft coral + check.')
    print('Tanglish labels: '+', '.join(['Ride book pannu','Booking cancel pannu','Call start','Niruthi detail change pannu','Call mudikkavum']))
    print('Close the old AI Call popup, refresh your signed-in dashboard, then open a NEW AI Call.')
    if backup:
        print('Backup:',backup)
    return {'version':'MOVIO_CALL_THEME_V1','changed_files':[str(p.relative_to(root)) for p,_ in updates],
            'active_html_updated':active_before!=active_after,'backup':str(backup) if backup else None}

MOVIO_CALL_UI_STATUS=movio_match_ai_call_theme()


AI Call dashboard theme installed.
Selected Booking: soft yellow + check. Selected Cancellation: soft coral + check.
Tanglish labels: Ride book pannu, Booking cancel pannu, Call start, Niruthi detail change pannu, Call mudikkavum
Close the old AI Call popup, refresh your signed-in dashboard, then open a NEW AI Call.
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/ai_call_ui_backup_20261028_070659_788973


In [87]:
# Run this ENTIRE file as ONE Kaggle cell after the original voice/weather setup
# and your previous fixes, BEFORE evaluation. No package installation is needed.
# This replaces/upgrades the earlier group extraction cell; do not edit evaluation.
def movio_enable_shared_group_extraction():
    from pathlib import Path
    from datetime import datetime
    import importlib.util
    import itertools
    import json
    import os
    import shutil
    import sys
    import tempfile

    b=globals().get('bridge')
    endpoint=getattr(b,'_wx_endpoint',None) if b is not None else None
    engine=getattr(endpoint,'__globals__',None)
    if not isinstance(engine,dict) or not callable(engine.get('collect_details')) or not callable(engine.get('prepare_turn')):
        raise RuntimeError('Run your original weather/voice activation first, then run this cell. It needs the active booking engine.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    folder=root/'integrations'
    if not folder.is_dir():
        raise FileNotFoundError('The original MOVIO integrations folder was not found.')

    module_source=r'''"""Shared booking-slot extraction; the installed dialogue/booking engine stays in charge."""
import copy
import re
import unicodedata

VERSION='group-slots-1.1'
_ENGINE={}
_DIGITS=str.maketrans('௦௧௨௩௪௫௬௭௮௯','0123456789')
_KEYS=('count','vehicle','pickup','destination')
_LETTERS=r'A-Za-z\u0B80-\u0BFF'
_DEFAULT_PLACES={
 'Tambaram':('tambaram','thambaram','thaambaram','தாம்பரம்','தாம்பரத்தில்','தாம்பரத்திலிருந்து','தாம்பரத்துக்கு','தாம்பரத்திற்கு'),
 'Vandalur':('vandalur','vandaloor','vandallur','வண்டலூர்','வண்டலூரில்','வண்டலூரிலிருந்து','வண்டலூருக்கு','வண்டலூர்க்கு'),
}
_VEHICLES=(
 ('SUV',r'(?<![a-z])s\s*[. -]?u\s*[. -]?v(?![a-z])|எஸ்\s*ய[ுூ]?\s*வி|எஸ்யூவி'),
 ('Sedan',r'(?<![a-z])sedan(?![a-z])|செடான்'),
 ('Mini',r'(?<![a-z])mini(?:\s*(?:car|கார்))?(?![a-z])|மினி(?:\s*(?:கார்|car))?'),
 ('Auto',r'(?<![a-z])(?:auto|aatto|aato)(?![a-z])|ஆட்டோ'),
 ('Bike',r'(?<![a-z])(?:bike|baikku|baik)(?![a-z])|பைக்'),
)
_EN_FROM=re.compile(r'(?<![a-z])(?:pick\s+me\s+up\s+(?:from|at)|pickup(?:\s+(?:is|from|at))?|from)(?![a-z])\s*[:=-]?\s*',re.I)
_EN_TO=re.compile(r'(?<![a-z])(?:destination(?:\s+is)?|drop(?:\s*(?:off|me))?(?:\s+(?:at|is))?|take\s+me\s+to|to)(?![a-z])\s*[:=-]?\s*',re.I)
_TA_FROM=re.compile(r'(?:இல்\s*இருந்து|இலிருந்து|லிருந்து|லேர்ந்து|லருந்து|இருந்து)(?=\s|[-,;.!?]|$)')
_ROMAN_FROM=re.compile(r'(?:(?:\s+|-)la\s*irun[dt]hu|(?:\s+|-)il\s*irun[dt]hu|(?:ili|li|le|la)?run[dt]hu|(?:\s+|-)?irun[dt]hu|(?:\s+|-)?lerndhu|(?:\s+|-)?lerun[dt]hu)(?![a-z])',re.I)
_TA_PICKUP_LABEL=re.compile(r'(?:புறப்படும்\s*இடம்|ஏறும்\s*இடம்|பிக்கப்)\s*[:=-]?\s*')
_TA_DROP_LABEL=re.compile(r'(?:சேரும்\s*இடம்|இறங்கும்\s*இடம்|டிராப்)\s*[:=-]?\s*')
_TO_SUFFIX=re.compile(r'(?:க்கு|வரை)(?=\s|[,;.!?]|$)|(?<![a-z])(?:ku|varai|varaikkum)(?![a-z])|kku(?![a-z])|(?<=[a-z])ku(?![a-z])',re.I)
_GO=re.compile(r'(?<![a-z])(?:poganum|pogannum|poganumnu|poganu|poga|ponum|porom|pog[a]?num|venum|vendum|v[a]?enum|edunga|eduthukkonga|pickup\s+pannu)(?![a-z])|போகணும்|போகனும்|போக\s*வேண்டும்|செல்ல\s*வேண்டும்|செல்லணும்|செல்லனும்|செல்ல|போக|வேணும்|வேண்டும்|எடுங்க|எடுத்துக்கோங்க',re.I)
_CARRIER=re.compile(r'(?<![a-z])(?:naan|nan|naanga|nanga|naangal|naangalum|enakku|enaku|engalukku|namakku|irukkom|irukk[o]?m|irukkirom|irukkomnga|venum|vendum|v[a]?enum)(?![a-z])|நாங்கள்|நாங்க|நான்|எனக்கு|எங்களுக்கு|நமக்கு|இருக்கிறோம்|இருக்கோம்|வேணும்|வேண்டும்',re.I)
_TIME=re.compile(r'(?<![a-z])(?:today|tomorrow|tonight|innaiku|inniku|iniku|indru|naalaiku|naalaikku|naalai|morning|afternoon|evening|night|kaalai|kalai|madhiyam|mathiyam|maalai|iravu)(?![a-z])|இன்று|இன்னைக்கு|நாளைக்கு|நாளை|காலை|மதியம்|மாலை|இரவு|நள்ளிரவு|\d{4}-\d{2}-\d{2}|\d{1,2}(?::\d{2})?\s*(?:a\.?m\.?|p\.?m\.?|மணிக்கு|மணி|manikku|mani)(?![a-z])',re.I)
_FILLER=re.compile(r'^(?:pickup|drop|pannu|pannunga|edunga|please|வேணும்|வேண்டும்|எடுங்க|இருக்கோம்|சரி|ஆம்|yes|no|okay|ok|hello|vanakkam|வணக்கம்)(?:\s|$)',re.I)
_CORRECTION=re.compile(r'(?<![a-z])(?:no|not|change|instead|replace|actually|rather|maathu|maathunga|mathunga|badila|bathila|illa|vendam)(?![a-z])|மாற்ற|பதிலா|பதிலாக|வேண்டாம்|இல்லை',re.I)
_SENTENCE_WORDS=re.compile(r'\b(?:i|we|me|us|you|want|need|take|where|am|is|are|go|going|travel|book|booking|confirm|cancel|please|thanks|thank)\b',re.I)

def normalize(value):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',str(value or '')).translate(_DIGITS).casefold()).strip()

def place_key(value):
    value=normalize(value)
    return re.sub(r'\s+',' ',''.join(' ' if unicodedata.category(c).startswith('P') else c for c in value)).strip()

def tamil_place_forms(alias):
    """Inflect registered names, rather than guessing an unknown Tamil stem."""
    if not re.fullmatch(r'[\u0B80-\u0BFF ]+',alias):return ()
    forms={alias+' வரை',alias+'வரை'}
    if alias.endswith('ம்'):
        stem=alias[:-2]
        forms.update(stem+suffix for suffix in ('த்தில்','த்திலிருந்து','த்தில் இருந்து','த்துக்கு','த்திற்கு'))
    elif alias.endswith('்'):
        stem=alias[:-1]
        forms.update(stem+suffix for suffix in ('ில்','ிலிருந்து','ில் இருந்து','ுக்கு','்க்கு'))
    elif alias.endswith(('ி','ீ','ை')):
        forms.update(alias+suffix for suffix in ('யில்','யிலிருந்து','யில் இருந்து','க்கு'))
    elif alias.endswith(('ா','ூ')):
        forms.update(alias+suffix for suffix in ('வில்','விலிருந்து','வில் இருந்து','வுக்கு'))
    return forms

def number_words(engine):
    fn=engine.get('number_lexicon')
    table=dict(fn(20)) if callable(fn) else {}
    for n,words in enumerate((('zero','பூஜ்ஜியம்'),('one','ஒரு','ஒன்று','ஒன்னு','oru','onnu'),('two','இரண்டு','ரெண்டு','rendu','rendu','irandu'),('three','மூன்று','மூணு','moonru','moondru','moonu'),('four','நான்கு','நாலு','naalu','naangu'),('five','ஐந்து','அஞ்சு','aindhu','anju'),('six','ஆறு','aaru'),('seven','ஏழு','ezhu','yelu'),('eight','எட்டு','ettu'),('nine','ஒன்பது','onbadhu'),('ten','பத்து','pathu'))):
        for word in words:table[word]=n
    return {normalize(k):int(v) for k,v in table.items()}

def clause_ranges(text):
    start=0;out=[]
    for m in re.finditer(r'[.!?;\n]',text):
        if m[0]=='.':
            if m.start() and m.end()<len(text) and text[m.start()-1].isdigit() and text[m.end()].isdigit():continue
            before=text[start:m.start()].split()
            if before and re.fullmatch(r'(?:[A-Za-z]|Mr|Mrs|Dr|Ms)',before[-1],re.I):continue
            if m.end()<len(text) and not text[m.end()].isspace():continue
        out.append((start,m.start()));start=m.end()
    out.append((start,len(text)))
    return [(a,b) for a,b in out if text[a:b].strip()]

class Parser:
    def __init__(self,bridge,state,text,engine):
        self.bridge,self.state,self.engine=bridge,state,engine
        self.raw=unicodedata.normalize('NFKC',str(text or '')).strip()
        self.scan=self.raw.translate(_DIGITS)
        self.updates={};self.evidence={};self.conflicts={};self.noise=[]
        aliases={k:tuple(v) for k,v in getattr(bridge,'VOICE_PLACE_ALIASES',{}).items()}
        for source in (_DEFAULT_PLACES,getattr(bridge,'GROUP_PLACE_ALIASES',{})):
            for key,values in source.items():aliases[key]=aliases.get(key,())+tuple(values)+(key,)
        self.places={}
        for canonical,variants in aliases.items():
            for alias in (canonical,)+tuple(variants):
                self.places[place_key(alias)]=canonical
                for form in tamil_place_forms(alias):self.places[place_key(form)]=canonical
                for suffix in (' இலிருந்து',' லிருந்து','இலிருந்து','லிருந்து',' வரை',' varai',' varaikkum'):
                    self.places[place_key(alias+suffix)]=canonical
                if re.fullmatch(r'[A-Za-z ]+',alias):
                    for suffix in ('la',' ku',' kku','ku','kku',' la',' la irundhu','la irundhu',' il irundhu','ilirundhu','lirundhu',' lirundhu',' lerndhu','lerndhu',' lerundhu'):
                        self.places[place_key(alias+suffix)]=canonical

    def accept(self,slot,value,span,source='rule',confidence=1.0):
        self.updates[slot]=value;self.conflicts.pop(slot,None)
        self.evidence[slot]={'text':span,'source':source,'confidence':confidence}

    def count(self):
        words=number_words(self.engine)
        token=r'\d{1,2}|'+'|'.join(re.escape(k) for k in sorted(words,key=len,reverse=True) if not k.isdigit())
        unit=r'(?:பயணிகள்|நபர்கள்|பேர்கள்|நபர்|பேரு|பேர்|passengers?|people|persons?|pax|peru|paeru|per)'
        pattern=r'(?<!['+_LETTERS+r'\d])(?P<n>'+token+r')\s*'+unit+r'(?!['+_LETTERS+r'])'
        found=list(re.finditer(pattern,self.scan,re.I))
        if not found:
            m=re.search(r'(?<![a-z])(?:we\s+are|there\s+are|for)\s+(?P<n>'+token+r')(?=\s*(?:of\s+us\b|$))',self.scan,re.I)
            if m:found=[m]
        if not found and self.state.get('stage')=='count':
            m=re.fullmatch(r'\s*(?P<n>'+token+r')\s*[.!]?\s*',self.scan,re.I)
            if m:found=[m]
        if not found:return
        choices=[]
        for m in found:
            value=normalize(m['n']);n=int(value) if value.isdigit() else words[value]
            self.noise.append(m.span())
            if not re.search(r'\bnot\s*$',self.scan[max(0,m.start()-12):m.start()],re.I):choices.append((n,m))
        if not choices:return
        if len({n for n,_ in choices})>1 and not _CORRECTION.search(self.scan):
            self.conflicts['count']='More than one passenger count stated';return
        n,m=choices[-1];self.accept('count',n,self.raw[m.start():m.end()])

    def vehicle(self):
        hits=[]
        for value,pattern in _VEHICLES:
            for m in re.finditer(pattern,self.scan,re.I):
                if re.match(r'\s+(?:nagar|market|bazaar|road|street|நகர்|சாலை)\b',self.scan[m.end():],re.I):continue
                self.noise.append(m.span());hits.append((m.start(),m.end(),value))
        if not hits:return
        hits.sort();positive=[]
        for a,b,value in hits:
            left=self.scan[max(0,a-15):a];right=self.scan[b:]
            if re.search(r'\bnot\s*$',left,re.I) or re.match(r'[\s-]*(?:வேண்டாம்|இல்லை|vendam|illa)(?![a-z])',right,re.I):continue
            positive.append((a,b,value))
        if len({v for _,_,v in positive})==1:
            a,b,value=positive[-1];self.accept('vehicle',value,self.raw[a:b]);return
        if positive and _CORRECTION.search(self.scan):
            instead=re.search(r'\binstead\s+of\b',self.scan,re.I)
            before=[h for h in positive if instead and h[1]<=instead.start()]
            a,b,value=before[-1] if before else positive[-1]
            self.accept('vehicle',value,self.raw[a:b]);return
        self.conflicts['vehicle']='Choose one vehicle' if positive else 'The stated vehicle was rejected'

    def clean_place(self,value,role):
        text=str(value or '').strip(' \t\r\n.,;:!?-–—')
        text=re.sub(r'(?i)^(?:i\s+(?:want|need)(?:\s+to\s+(?:go|travel))?|i\s+am|my\s+pickup\s+is|my\s+destination\s+is|please|naan|nan|enakku|enaku|நான்|எனக்கு)\s+','',text).strip()
        text=re.sub(r'(?i)\s+(?:for|with|and|at|from|to)\s*$','',text).strip(' ,-')
        # Match complete known names/case forms, not a place hidden inside a sentence.
        if place_key(text) in self.places:return self.places[place_key(text)]
        # Separate case markers can be removed without changing the spoken name.
        text=re.sub(r'(?i)(?:[-\s]+(?:இல்\s*இருந்து|இலிருந்து|லிருந்து|இருந்து|க்கு|வரை|la\s*irun[dt]hu|il\s*irun[dt]hu|irun[dt]hu|la|ku|kku|varai|varaikkum))$','',text).strip(' ,-')
        if place_key(text) in self.places:return self.places[place_key(text)]
        if not text or len(text)>180 or not re.search(r'[A-Za-z\u0B80-\u0BFF]',text):return None
        if normalize(text) in {'time','date','fare','vehicle','booking','car','cab','taxi','நேரம்','தேதி','இடம்'}:return None
        if _FILLER.match(text) or re.fullmatch(r'(?:இரண்டு|ரெண்டு|\d+)',text):return None
        if any(re.fullmatch(pattern,text,re.I) for _,pattern in _VEHICLES):return None
        if _TIME.search(text) or re.search(r'(?<![a-z])(?:poganum|venum|irukkom|we are|passengers?|people)(?![a-z])|வேணும்|வேண்டும்|இருக்கோம்|பேர்',text,re.I):return None
        canonicalizer=getattr(self.bridge,'canonical_place',None)
        canonical=canonicalizer(text) if callable(canonicalizer) else text
        return str(canonical).strip()[:120] if canonical else None

    def add_place(self,role,a,b,clause_start,source='rule',prefix=False,commit=True):
        if a>=b:return
        # Discard preceding booking/count/time phrases, retaining the nearby place.
        if prefix:
            cuts=[end for begin,end in self.noise if a<=begin<end<=b]
            cuts.extend(m.end() for m in _CARRIER.finditer(self.scan,a,b))
            if cuts:a=max(a,max(cuts))
        stop=[begin for begin,end in self.noise if a<=begin<b]
        stop.extend(m.start() for m in _GO.finditer(self.scan,a,b))
        stop.extend(m.start() for m in _TIME.finditer(self.scan,a,b))
        stop.extend(m.start() for m in _CARRIER.finditer(self.scan,a,b) if m.start()>a and self.scan[m.start()-1].isspace())
        # Commas before another explicit slot separate details, not addresses.
        for m in re.finditer(r',\s*(?=(?:pickup|drop|destination|vehicle|passengers?|today|tomorrow)\b)',self.scan[a:b],re.I):stop.append(a+m.start())
        if stop:b=min(b,min(stop))
        value=self.clean_place(self.raw[a:b],role)
        if not value:return
        if re.search(r'\bor\b|அல்லது|\balladhu\b',self.raw[a:b],re.I):
            if commit:self.conflicts[role]='More than one location stated'
            return
        span=self.raw[a:b].strip()
        if commit:self.accept(role,value,span,source)
        return value,span

    def route(self):
        for match in _TIME.finditer(self.scan):self.noise.append(match.span())
        for ca,cb in clause_ranges(self.scan):
            part=self.scan[ca:cb]
            # A bare "A to B" is a route only when BOTH sides validate as
            # places. Do not turn "change Bike to Mini" into a location update.
            explicit=any(pattern.search(part) for pattern in (_EN_FROM,_TA_FROM,_ROMAN_FROM,_TA_PICKUP_LABEL,_TA_DROP_LABEL))
            explicit=explicit or any(normalize(m[0])!='to' for m in _EN_TO.finditer(part))
            labelled=bool(re.search(r'\b(?:destination|drop|pickup)\b',part,re.I))
            if not explicit and not labelled and not _CORRECTION.search(part):
                joins=list(re.finditer(r'(?<![a-z])to(?![a-z])\s*',part,re.I))
                if len(joins)==1:
                    join=joins[0]
                    pickup=self.add_place('pickup',ca,ca+join.start(),ca,prefix=True,commit=False)
                    destination=self.add_place('destination',ca+join.end(),cb,ca,commit=False)
                    name_like=lambda item:bool(item and (place_key(item[1]) in self.places or not _SENTENCE_WORDS.search(item[1])))
                    if name_like(pickup) and name_like(destination):
                        self.accept('pickup',*pickup)
                        self.accept('destination',*destination)
            prefixes=[]
            for role,pattern in (('pickup',_EN_FROM),('destination',_EN_TO),('pickup',_TA_PICKUP_LABEL),('destination',_TA_DROP_LABEL)):
                for m in pattern.finditer(part):
                    if role=='destination' and normalize(m[0])=='to':
                        allowed=bool(_EN_FROM.search(part) or re.search(r'\b(?:go|going|travel|ride|take)\b',part[:m.start()],re.I) or not part[:m.start()].strip() or self.state.get('stage')=='destination')
                        if not allowed:continue
                    prefixes.append((ca+m.start(),ca+m.end(),role))
            prefixes.sort()
            for index,(start,end,role) in enumerate(prefixes):
                finish=prefixes[index+1][0] if index+1<len(prefixes) else cb
                self.add_place(role,end,finish,ca)
            # Tamil/Tanglish express FROM after the pickup name, often attached.
            suffixes=sorted((ca+m.start(),ca+m.end()) for pat in (_TA_FROM,_ROMAN_FROM) for m in pat.finditer(part))
            # Prefer the widest operator when patterns overlap.
            unique=[]
            for a,b in suffixes:
                if unique and a<unique[-1][1]:
                    if b>unique[-1][1]:unique[-1]=(unique[-1][0],b)
                else:unique.append((a,b))
            for start,end in unique:
                left=ca
                prior_to=[ca+m.end() for m in _TO_SUFFIX.finditer(part) if ca+m.end()<=start]
                prior_pickup=[b for a,b,role in prefixes if role=='pickup' and b<=start]
                if prior_to or prior_pickup:left=max([left]+prior_to+prior_pickup)
                # A prior TO/destination phrase ends at its travel verb.
                previous_go=[m.end() for m in _GO.finditer(self.scan,ca,start)]
                if previous_go:left=max(previous_go)
                # Keep the inflected word in evidence for exact alias lookup.
                self.add_place('pickup',left,end,ca,prefix=True)
                right=min([p[0] for p in prefixes if p[0]>=end]+[cb])
                self.add_place('destination',end,right,ca)
            # TO may be an attached case suffix, or a destination followed by "go".
            for m in _TO_SUFFIX.finditer(part):
                a,b=ca+m.start(),ca+m.end()
                if not (_GO.search(part) or unique or self.state.get('stage')=='destination'):continue
                left=max([end for start,end in unique if end<=a]+[ca])
                self.add_place('destination',left,b,ca,prefix=True)
            for m in _GO.finditer(part):
                if not re.match(r'(?i)(?:poga|ponum|porom|போக|செல்ல)',m[0]):continue
                end=ca+m.start();left=max([b for a,b in unique if b<=end]+[ca])
                if 'destination' not in self.updates:self.add_place('destination',left,end,ca,prefix=True)

    def parse(self,use_qwen=False):
        norm=normalize(self.raw)
        excluded=self.engine.get('POSITIVE',set())|self.engine.get('NEGATIVE',set())
        if not norm or norm in excluded:return self.result()
        self.count();self.vehicle();self.route()
        stage=self.state.get('stage')
        if stage in ('pickup','destination') and stage not in self.updates and not self.updates and not self.conflicts and not _TIME.search(self.scan):
            value=self.clean_place(self.raw,stage)
            if value:self.accept(stage,value,self.raw,'rule')
        missing=[key for key in _KEYS if key not in self.updates and not self.state.get(key) and key not in self.conflicts]
        extractor=getattr(self.bridge,'_wx_context_extract',None)
        if use_qwen and missing and callable(extractor) and len(self.raw.split())>=5:
            try:proposed=extractor(self.raw,copy.deepcopy(self.state)) or {}
            except Exception:proposed={}
            if isinstance(proposed,dict):
                for role in ('pickup','destination'):
                    span=proposed.get(role)
                    if role not in missing or not isinstance(span,str) or normalize(span) not in norm:continue
                    value=self.clean_place(span,role)
                    if value:self.accept(role,value,span,'qwen_validated_span',0.85)
                span=proposed.get('passengers_text')
                group_cue=re.search(r'\b(?:people|persons?|passengers?|pax|peru|per)\b|பேர்|பேரு|பயணிகள்|நபர்',self.scan,re.I)
                if 'count' in missing and group_cue and isinstance(span,str) and normalize(span) and normalize(span) in norm:
                    n=number_words(self.engine).get(normalize(span))
                    if n is not None:self.accept('count',n,span,'qwen_validated_span',0.85)
        return self.result()

    def result(self):
        return {'updates':dict(self.updates),'evidence':copy.deepcopy(self.evidence),'ambiguities':dict(self.conflicts)}

def extract_fields(bridge,text,state=None,use_qwen=False,engine=None):
    """Read-only diagnostic using exactly the same parser as the live collector."""
    return Parser(bridge,state or {},text,engine if engine is not None else _ENGINE).parse(use_qwen)

def collect_details(bridge,state,text):
    if state.get('mode')=='cancel' or state.get('stage') in ('done','booking_pending','cancel_pending','cancel_select','cancel_confirm'):
        return {'updates':{},'changed':[],'evidence':{},'ambiguities':{}}
    parsed=extract_fields(bridge,text,state,use_qwen=True)
    updates=parsed['updates'];conflicts=state.setdefault('_wx_slot_conflicts',{})
    conflicts.update(parsed['ambiguities']);changed=[]
    for key in parsed['ambiguities']:
        if key in ('count','pickup','destination') and state.get(key) is not None:
            state[key]=None;changed.append(key)
            if key in ('pickup','destination'):state[key+'_geo']=None
    for key,value in updates.items():
        conflicts.pop(key,None)
        if state.get(key)!=value:
            state[key]=value;changed.append(key)
            if key in ('pickup','destination'):state[key+'_geo']=None
    if changed:
        state['fare']=None;state.pop('pending',None);state.pop('suggested_destination',None)
    state.setdefault('_wx_slot_evidence',{}).update(parsed['evidence'])
    if str(text).strip():state['_wx_turns']=(state.get('_wx_turns',[])+[{'input':str(text),'updates':updates}])[-30:]
    return {**parsed,'changed':changed,'ambiguities':copy.deepcopy(conflicts)}

collect_details._movio_shared_group_slots=VERSION
'''
    compile(module_source,'<MOVIO shared group extraction>','exec')
    target=folder/'movio_group_booking_slots.py'
    module_name='movio_group_booking_slots'
    previous_module=sys.modules.get(module_name)

    # Load/check the new parser before changing the live booking collector.
    with tempfile.TemporaryDirectory(prefix='movio_group_check_') as temporary:
        candidate=Path(temporary)/target.name
        candidate.write_text(module_source,encoding='utf-8')
        spec=importlib.util.spec_from_file_location(module_name,candidate)
        candidate_module=importlib.util.module_from_spec(spec)
        spec.loader.exec_module(candidate_module)
        candidate_module._ENGINE=engine
        expected={'vehicle':'Mini','count':2,'pickup':'Tambaram','destination':'Vandalur'}
        example='எனக்கு ஒரு மினி கார் வேணும். நாங்க ரெண்டு பேர் இருக்கோம். தாம்பரத்திலிருந்து வண்டலூர் போகணும்.'
        checks=[example,
            'Enakku oru mini car venum. Naanga rendu peru irukkom. Tambaramla irundhu Vandalurku poganum.',
            'எனக்கு Mini car வேண்டும், நாங்க two passengers, Tambaram la irundhu வண்டலூர் போகணும்.',
            'I need a Mini car for two passengers from Tambaram to Vandalur.']
        location_checks=(
            'தாம்பரத்திலிருந்து வண்டலூர் போகணும்',
            'Tambaram la irundhu Vandalur poganum',
            'Tambaram to Vandalur',
            'நான் தாம்பரத்திலிருந்து வண்டலூர் செல்ல வேண்டும்',
            'தாம்பரத்திலிருந்து வண்டலூர் வரை போகணும்',
            'Tambaram la irundhu Vandalur varai poganum',
            'தாம்பரம் இலிருந்து வண்டலூர் வரை',
            'தாம்பரம் லிருந்து வண்டலூர் போகணும்',
        )
        checks.extend('Mini car. 2 passengers. '+route for route in location_checks)
        for clauses in (
            ('எனக்கு ஒரு மினி கார் வேணும்.','நாங்க ரெண்டு பேர் இருக்கோம்.','தாம்பரத்திலிருந்து வண்டலூர் போகணும்.'),
            ('Mini car venum.','Rendu peru irukkom.','Tambaramla irundhu Vandalurku poganum.'),
            ('Mini car please.','Two passengers.','From Tambaram to Vandalur.'),
        ):
            checks.extend(' '.join(order) for order in itertools.permutations(clauses))
        for text in checks:
            parsed=candidate_module.extract_fields(b,text,{'mode':'book','stage':'count'},use_qwen=False)
            actual={key:parsed['updates'].get(key) for key in expected}
            if actual!=expected:
                raise RuntimeError('Pre-installation text check failed. The live booking engine was not changed.\n'
                    +json.dumps({'text':text,'expected':expected,'actual':actual},ensure_ascii=False,indent=2))

    # Preserve the previous module source for rollback/auditing.
    backup=None
    if target.exists() and target.read_text(encoding='utf-8')!=module_source:
        backup=folder/('movio_group_booking_slots.backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.py')
        shutil.copy2(target,backup)
    if not target.exists() or target.read_text(encoding='utf-8')!=module_source:
        temporary=target.with_name(target.name+'.tmp')
        try:
            temporary.write_text(module_source,encoding='utf-8');os.replace(temporary,target)
        finally:temporary.unlink(missing_ok=True)
    spec=importlib.util.spec_from_file_location(module_name,target)
    module=importlib.util.module_from_spec(spec)
    try:
        sys.modules[module_name]=module;spec.loader.exec_module(module);module._ENGINE=engine
    except Exception:
        if previous_module is None:sys.modules.pop(module_name,None)
        else:sys.modules[module_name]=previous_module
        raise

    # One shared collector for all selected languages; leave the existing
    # prepare_turn, time validation, weather consent, fare and database code intact.
    if not hasattr(b,'_wx_group_original_collect'):
        b._wx_group_original_collect=engine['collect_details']
    engine['collect_details']=module.collect_details
    # Retain the already-loaded Qwen extractor as a validated-span fallback.
    existing_fallback=getattr(b,'_wx_context_extract',None)
    notebook_fallback=globals().get('context_extract')
    if not callable(existing_fallback) and callable(notebook_fallback):
        b._wx_context_extract=notebook_fallback
    b._wx_group_slots_version=module.VERSION
    b._wx_group_slots_module=module
    b.group_booking_extract=lambda text,session=None,use_qwen=False:module.extract_fields(
        b,text,session or {'mode':'book','stage':'count'},use_qwen=use_qwen)
    result={'version':module.VERSION,'text_checks_passed':len(checks),
        'example':b.group_booking_extract(example)['updates'],'source':str(target),
        'route_example':b.group_booking_extract('Tambaram to Vandalur')['updates']}
    print('Shared English/Tamil/Tanglish slot extraction enabled.')
    print('Text-only checks passed:',len(checks))
    print(json.dumps(result['example'],ensure_ascii=False,indent=2))
    print('Location check:',json.dumps(result['route_example'],ensure_ascii=False))
    print('The example supplies four slots. Only the missing booking date/time should be requested next.')
    print('Close the old AI Call popup and start a NEW call to test the spoken sentence.')
    return result

MOVIO_GROUP_SLOT_STATUS=movio_enable_shared_group_extraction()

# Optional diagnostic after installation (does not create a booking):
# bridge.group_booking_extract('Enakku mini car venum. Rendu peru. Tambaramla irundhu Vandalurku poganum.')


Shared English/Tamil/Tanglish slot extraction enabled.
Text-only checks passed: 30
{
  "count": 2,
  "vehicle": "Mini",
  "pickup": "Tambaram",
  "destination": "Vandalur"
}
Location check: {"pickup": "Tambaram", "destination": "Vandalur"}
The example supplies four slots. Only the missing booking date/time should be requested next.
Close the old AI Call popup and start a NEW call to test the spoken sentence.


In [88]:
# Copy this ENTIRE file into ONE Kaggle cell, after the pickup/destination fix
# and voice/weather activation, BEFORE evaluation. No model reload is needed.
def movio_enable_optional_weather():
    import importlib.util
    import json
    import os
    from pathlib import Path
    import shutil
    import subprocess
    import sys
    import tempfile
    import time
    import types
    import weakref
    from datetime import datetime,timedelta
    import psutil
    import requests

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if not isinstance(engine,dict) or not callable(engine.get('prepare_turn')):
        raise RuntimeError('Run the original voice/weather activation first, then this cell.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    guard_path=root/'integrations/movio_weather_guard.py'
    helper_path=root/'integrations/movio_optional_weather.py'
    slots_path=root/'integrations/movio_group_booking_slots.py'
    server_path=root/'server.py'
    for path in (guard_path,slots_path,server_path):
        if not path.is_file():raise FileNotFoundError('Run the original setup and updated location-extraction cell first: '+str(path))
    slots=getattr(b,'_wx_group_slots_module',None)
    if slots is None:
        spec=importlib.util.spec_from_file_location('_movio_optional_place_rules',slots_path)
        slots=importlib.util.module_from_spec(spec);spec.loader.exec_module(slots)
    module_template=r'''"""Optional, shared weather lookup for the existing MOVIO booking guard.

No model, user interface, booking transaction or evaluation implementation is replaced.
Provider contract: https://open-meteo.com/en/docs
"""
import copy
import importlib.util
import json
import logging
import math
import re
import sys
import threading
import time
import types
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

VERSION='optional-weather-1.0'
TZ=ZoneInfo('Asia/Kolkata')
PLACE_ALIASES=__PLACE_ALIASES__
_SCOPE_KEYS=('pickup','destination','pickup_at','pickup_geo','destination_geo')
_FAILED={'unavailable','outside_forecast_range'}
_RESIDUAL_SUFFIX=re.compile(r'(?:இலிருந்து|லிருந்து|லேர்ந்து|லருந்து|இல்\s*இருந்து|(?:\s|-)இருந்து|க்கு|வரை|\b(?:la\s*irun[dt]hu|il\s*irun[dt]hu|ilirun[dt]hu|lirun[dt]hu|lerndhu|varai|varaikkum))\s*$',re.I)

def configure_logger(root):
    path=Path(root)/'weather_lookup.log'
    logger=logging.getLogger('movio.weather_lookup.'+str(path.resolve()))
    logger.setLevel(logging.INFO);logger.propagate=False
    if not logger.handlers:
        handler=logging.FileHandler(path,encoding='utf-8')
        handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
        logger.addHandler(handler)
    return logger

def log_failure(logger,reason,**details):
    logger.warning(json.dumps({'event':'weather_lookup_failed','reason':reason,**details},ensure_ascii=False,default=str))

def unavailable(reason,logger,status='unavailable',**details):
    log_failure(logger,reason,**details)
    return {'status':status,'source':'Open-Meteo','reason':reason,'checked_at':time.time()}

def numeric(value,lo=None,hi=None):
    try:
        result=float(value)
        if not math.isfinite(result) or lo is not None and result<lo or hi is not None and result>hi:return None
        return result
    except (TypeError,ValueError,OverflowError):return None

def usable(weather):
    if not isinstance(weather,dict) or weather.get('status')!='ok':return False
    points=weather.get('points')
    # Existing adapters can supply a condition without probability estimates.
    if points is None:return weather.get('condition') in ('rain','cloudy','clear')
    return bool(isinstance(points,list) and points and all(isinstance(p,dict) for p in points) and weather.get('condition') in ('rain','cloudy','clear','other')
        and any(isinstance(p,dict) and (numeric(p.get('weather_code')) is not None
            or numeric(p.get('precipitation_probability'),0,100) is not None
            or numeric(p.get('temperature_2m')) is not None) for p in points))

def load_slots(path):
    spec=importlib.util.spec_from_file_location('_movio_weather_place_rules',path)
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    if not callable(getattr(module,'extract_fields',None)) or not str(getattr(module,'VERSION','')).startswith('group-slots-'):
        raise RuntimeError('Run the updated pickup/destination extraction cell first.')
    return module

def make_normalizer(slots,engine,bridge=None):
    aliases=dict(PLACE_ALIASES)
    aliases.update(getattr(bridge,'VOICE_PLACE_ALIASES',{}) if bridge is not None else {})
    canonicalizer=getattr(bridge,'canonical_place',None) if bridge is not None else None
    proxy=types.SimpleNamespace(VOICE_PLACE_ALIASES=aliases,
        GROUP_PLACE_ALIASES=getattr(bridge,'GROUP_PLACE_ALIASES',{}) if bridge is not None else {},
        canonical_place=canonicalizer if callable(canonicalizer) else lambda x:x)
    parser=slots.Parser(proxy,{},'',engine)
    def normalize(value,role='pickup'):
        result=parser.clean_place(value,role)
        # Unknown inflected words are retained in dialogue memory, but cannot be
        # sent verbatim to geocoding. Registered forms are resolved above.
        return result if result and not _RESIDUAL_SUFFIX.search(result) else None
    return normalize

def forecast(client,points,pickup,now,engine,logger):
    """Same coordinate-based Open-Meteo query for every dialogue language."""
    try:
        if not isinstance(pickup,datetime):pickup=datetime.fromisoformat(str(pickup))
        pickup=pickup.replace(tzinfo=TZ) if pickup.tzinfo is None else pickup.astimezone(TZ)
        now=now or datetime.now(TZ)
        now=now.replace(tzinfo=TZ) if now.tzinfo is None else now.astimezone(TZ)
        horizon=(pickup.date()-now.date()).days
        if not 0<=horizon<16:return unavailable('outside_forecast_range',logger,status='outside_forecast_range')
        if len(points)!=2 or not all(engine['valid_geo'](p) for p in points):
            return unavailable('invalid_coordinates',logger)
        coords=tuple((round(float(p['lat']),5),round(float(p['lng']),5)) for p in points)
    except Exception as exc:return unavailable('invalid_weather_query',logger,error_type=type(exc).__name__)
    key=(VERSION,coords,pickup.date().isoformat())
    target=pickup.replace(minute=0,second=0,microsecond=0).strftime('%Y-%m-%dT%H:%M')
    with client.lock:
        cached=client.cache.get(key)
        valid_cache=bool(cached and time.monotonic()-cached[0]<client.ttl)
        if valid_cache:
            client.hits+=1;payload,fetched=cached[1],cached[2]
        else:
            failure_cache=getattr(client,'_movio_weather_failures',{})
            failed=failure_cache.get(key)
            if failed and time.monotonic()-failed[0]<20:return copy.deepcopy(failed[1])
            client.requests+=1
    if not valid_cache:
        response=None
        try:
            # Explicit dates and local timezone select the customer's booking
            # hour. Both locations are requested together, using coordinates.
            response=client.getter('https://api.open-meteo.com/v1/forecast',params={
                'latitude':','.join(str(p[0]) for p in coords),'longitude':','.join(str(p[1]) for p in coords),
                'hourly':'weather_code,precipitation_probability,precipitation,temperature_2m,apparent_temperature',
                'timezone':'Asia/Kolkata','forecast_days':16,
                'start_date':pickup.date().isoformat(),'end_date':pickup.date().isoformat()},timeout=(3,5))
            response.raise_for_status();payload=response.json()
            payload=payload if isinstance(payload,list) else [payload]
            if len(payload)!=2 or any(not isinstance(p,dict) or p.get('error') for p in payload):
                raise ValueError('invalid_location_response')
            fetched=time.time()
        except Exception as exc:
            status=getattr(getattr(exc,'response',None),'status_code',None)
            if status is None:status=getattr(response,'status_code',None)
            reason='http_error' if isinstance(status,int) and status>=400 else 'timeout' if 'timeout' in type(exc).__name__.lower() else 'request_or_response_error'
            result=unavailable(reason,logger,error_type=type(exc).__name__,http_status=status)
            with client.lock:
                failures=getattr(client,'_movio_weather_failures',None)
                if failures is None:failures={};client._movio_weather_failures=failures
                failures[key]=(time.monotonic(),result)
                if len(failures)>256:failures.pop(next(iter(failures)))
            return result
    rows=[]
    try:
        for name,item in zip(('pickup','destination'),payload):
            hourly=item['hourly'];index=hourly['time'].index(target)
            code=numeric(hourly['weather_code'][index])
            if code is None or not code.is_integer() or not 0<=code<=99:raise ValueError('missing_condition')
            row={'location':name,'valid_at':target,'weather_code':int(code)}
            for field in ('precipitation_probability','precipitation','temperature_2m','apparent_temperature'):
                values=hourly.get(field)
                value=numeric(values[index]) if isinstance(values,list) and index<len(values) else None
                if field=='precipitation_probability' and value is not None and not 0<=value<=100:value=None
                row[field]=value
            rows.append(row)
    except Exception as exc:
        return unavailable('missing_or_invalid_hourly_data',logger,error_type=type(exc).__name__,requested_hour=target)
    if not valid_cache:
        with client.lock:
            client.cache[key]=(time.monotonic(),payload,fetched)
            if len(client.cache)>256:client.cache.pop(next(iter(client.cache)))
    rain=any(p['weather_code'] in engine['RAIN_CODES'] or p['precipitation_probability'] is not None and p['precipitation_probability']>=client.rain_threshold for p in rows)
    cloudy=any(p['weather_code'] in (2,3) for p in rows);clear=all(p['weather_code'] in (0,1) for p in rows)
    heat=any(p['temperature_2m'] is not None and p['temperature_2m']>=client.heat_c
        or p['apparent_temperature'] is not None and p['apparent_temperature']>=client.feels_like_c for p in rows)
    return {'status':'ok','source':'Open-Meteo','points':rows,'fetched_at':fetched,
        'condition':'rain' if rain else 'cloudy' if cloudy else 'clear' if clear else 'other',
        'heavy_rain':any(p['weather_code'] in (65,67,82) for p in rows),
        'thunderstorm':any(p['weather_code'] in (95,96,97,99) for p in rows),'heat':heat,
        'pleasant':clear and not heat and all(p['temperature_2m'] is not None and 18<=p['temperature_2m']<=32 for p in rows),
        'scope':'pickup_and_destination_at_pickup_hour',
        'advisory_thresholds':{'rain_probability_pct':client.rain_threshold,'heat_c':client.heat_c,'feels_like_c':client.feels_like_c}}

class SafeWeather:
    def __init__(self,weather,logger):self.weather,self.logger=weather,logger
    def __getattr__(self,name):return getattr(self.weather,name)
    def forecast(self,*args,**kwargs):
        try:result=self.weather.forecast(*args,**kwargs)
        except Exception as exc:return unavailable('forecast_exception',self.logger,error_type=type(exc).__name__)
        if usable(result):return result
        if isinstance(result,dict) and result.get('status') in _FAILED:
            result=dict(result);result.setdefault('reason',result['status']);result.setdefault('checked_at',time.time())
            return result
        return unavailable('invalid_forecast_result',self.logger)

def probability_text(state,engine):
    if not usable(state.get('weather')):return ''
    language=state.get('lang','English');pieces=[]
    for role in ('pickup','destination'):
        row=next((r for r in state['weather'].get('points',[]) if r.get('location')==role),{})
        chance=numeric(row.get('precipitation_probability'),0,100)
        if chance is None:continue
        value=engine['numeric_text'](chance);name=state.get(role) or ('pickup' if role=='pickup' else 'drop')
        label='pickup' if role=='pickup' else 'drop'
        pieces.append(engine['localized'](language,f'At your {label} location, {name}: {value}% rain chance.',
            f'{name} பகுதியில் மழை வாய்ப்பு {value}%.',f'{name}-la mazhai chance {value}%.'))
    return '\n'.join(pieces)

def compact_advisory(state,engine,precautions=True):
    weather=state.get('weather') or {}
    if not usable(weather):return ''
    language=state.get('lang','English');loc=lambda en,ta,rom:engine['localized'](language,en,ta,rom)
    parts=[probability_text(state,engine)];condition=weather.get('condition')
    if weather.get('thunderstorm'):parts.append(loc('Thunderstorms are expected.','இடியுடன் மழை எதிர்பார்க்கப்படுகிறது.','Idiyudan rain expected.'))
    elif weather.get('heavy_rain'):parts.append(loc('Heavy rain is expected.','கனமழை எதிர்பார்க்கப்படுகிறது.','Kanamazhai expected.'))
    elif condition=='rain':parts.append(loc('Rain is expected.','மழை எதிர்பார்க்கப்படுகிறது.','Mazhai vara vaaippu irukku.'))
    elif condition=='cloudy':parts.append(loc('Cloudy weather is expected.','மேகமூட்டம் எதிர்பார்க்கப்படுகிறது.','Cloudy weather expected.'))
    elif condition=='clear' and not weather.get('heat'):
        parts.append(loc('The weather looks pleasant.' if weather.get('pleasant') else 'Clear skies are expected.',
            'வானிலை நன்றாக இருக்கும்.' if weather.get('pleasant') else 'வானம் தெளிவாக இருக்கும்.',
            'Weather nalla irukkum.' if weather.get('pleasant') else 'Clear weather expected.'))
    if weather.get('heat'):parts.append(loc('Very hot weather is expected; carry water.','வெப்பம் அதிகமாக இருக்கும்; குடிநீர் எடுத்துக்கொள்ளுங்கள்.','Very hot weather expected; water eduthukkonga.'))
    if precautions and condition=='rain':
        if state.get('vehicle')=='Bike':
            choices=[v for v in ('Mini','Sedan','SUV') if engine['CAPACITY'][v]>=int(state.get('count') or 1)]
            names=', '.join(choices)
            if names:
                severity=bool(weather.get('heavy_rain') or weather.get('thunderstorm'))
                en=('A bike may be unsuitable; ' if severity else '')+f'covered options are {names}. The choice is yours.'
                parts.append(loc(en,
                    'மூடிய வாகனம் சிறந்தது: '+', '.join(engine['VEHICLES_TA'][v] for v in choices)+'. முடிவு உங்களுடையது.',
                    'Covered vehicle better: '+names+'. Unga choice.'))
        elif state.get('vehicle')=='Auto':parts.append(loc('An auto has exposed sides; carry an umbrella.','ஆட்டோவின் பக்கங்களில் மழை படலாம்; குடை எடுத்துக்கொள்ளுங்கள்.','Auto sides-la rain padalaam; umbrella eduthukkonga.'))
        else:parts.append(loc('Carry an umbrella for boarding.','வாகனத்தில் ஏறுவதற்குக் குடை எடுத்துக்கொள்ளுங்கள்.','Vehicle-la era umbrella eduthukkonga.'))
    return '\n'.join(p for p in parts if p)

def install_namespace(engine,root=None,bridge=None,slots=None):
    root=Path(root or Path(engine['__file__']).resolve().parent.parent)
    logger=configure_logger(root)
    if engine.get('_MOVIO_OPTIONAL_WEATHER_VERSION')==VERSION:
        if bridge is not None:install_geocoder(bridge,engine['_movio_weather_normalize'],logger)
        return {'version':VERSION,'already_installed':True,'log':str(root/'weather_lookup.log')}
    required=('prepare_turn','weather_consented','WeatherClient','collect_details','manual_state','install_backend','probability_report')
    if any(not callable(engine.get(k)) for k in required):raise RuntimeError('The original weather guard is not active.')
    slots=slots or load_slots(root/'integrations/movio_group_booking_slots.py')
    normalizer=make_normalizer(slots,engine,bridge)
    original={key:engine[key] for key in required}
    old_forecast=engine['WeatherClient'].forecast
    def normalize_state(state):
        for role in ('pickup','destination'):
            if not state.get(role):continue
            canonical=normalizer(state[role],role)
            if canonical and canonical!=state[role]:
                state[role]=canonical;state[role+'_geo']=None;state['fare']=None
    def scope(state):return engine['digest']({k:state.get(k) for k in _SCOPE_KEYS})
    def weather_consented(state):
        weather=state.get('weather') or {};skip=weather.get('_optional_skip') or {}
        if weather.get('status') in _FAILED and skip.get('policy')==VERSION and skip.get('scope')==scope(state):return True
        return original['weather_consented'](state)
    def collect_details(b,state,text):
        result=original['collect_details'](b,state,text);normalize_state(state);return result
    def prepare_turn(b,weather,state,text,now=None):
        install_geocoder(b,normalizer,logger);normalize_state(state)
        safe=weather if isinstance(weather,SafeWeather) else SafeWeather(weather,logger)
        state,answer=original['prepare_turn'](b,safe,state,text,now=now)
        # An unavailable forecast is optional. This is an explicit skip record,
        # NOT fabricated user consent. Final-summary playback/confirmation and
        # the existing atomic booking transaction remain mandatory.
        if state.get('stage')=='weather_advisory' and not usable(state.get('weather')):
            result=dict(state.get('weather') or {})
            if result.get('status') not in _FAILED:result=unavailable('invalid_forecast_result',logger)
            result.setdefault('reason',result.get('status','unavailable'))
            result['_optional_skip']={'policy':VERSION,'scope':scope(state),'recorded_at':time.time()}
            state['weather']=result;state.pop('_wx_weather_consent',None)
            log_failure(logger,result['reason'],action='skip_optional_advisory',pickup=state.get('pickup'),destination=state.get('destination'),pickup_at=state.get('pickup_at'))
            state,answer=original['prepare_turn'](b,safe,state,'',now=now)
        return state,answer
    def manual_state(data,weather,now=None):
        data=dict(data)
        for role in ('pickup','destination'):
            canonical=normalizer(data.get(role),role)
            if canonical:data[role]=canonical
        result=original['manual_state'](data,SafeWeather(weather,logger),now=now)
        if not usable(result.get('weather')):
            log_failure(logger,(result.get('weather') or {}).get('reason','unavailable'),action='skip_optional_manual_advisory')
        return result
    def patched_forecast(client,points,pickup,now=None):return forecast(client,points,pickup,now,engine,logger)
    def install_backend(module,weather=None):
        result=original['install_backend'](module,weather=weather)
        previous=module.Handler.do_GET
        if not getattr(previous,'_movio_optional_weather_health',False):
            def do_GET(h):
                if h.path.split('?',1)[0]=='/api/weather-booking/optional-health':
                    return h.json(200,{'ok':True,'version':VERSION,'weather_optional':True,'final_confirmation_required':True})
                return previous(h)
            do_GET._movio_optional_weather_health=True;module.Handler.do_GET=do_GET
        return result
    engine['WeatherClient'].forecast=patched_forecast
    engine.update(prepare_turn=prepare_turn,collect_details=collect_details,weather_consented=weather_consented,
        # Preserve the legacy probability report used by evaluation. Customer
        # speech/manual advisories use probability_text directly and never
        # announce missing probabilities. Failed forecasts are silent here too.
        probability_report=lambda s:original['probability_report'](s) if usable(s.get('weather')) else '',
        weather_report=lambda s:compact_advisory(s,engine,False),
        advisory=lambda s:compact_advisory(s,engine),manual_state=manual_state,install_backend=install_backend,
        _MOVIO_OPTIONAL_WEATHER_VERSION=VERSION,_movio_weather_normalize=normalizer,
        _movio_optional_weather_original={**original,'forecast':old_forecast})
    if bridge is not None:install_geocoder(bridge,normalizer,logger)
    return {'version':VERSION,'log':str(root/'weather_lookup.log')}

def install_geocoder(bridge,normalizer,logger):
    original=getattr(bridge,'geocode',None)
    if not callable(original) or getattr(original,'_movio_optional_weather_geo',False):return
    def geocode(name):
        canonical=normalizer(name)
        if not canonical:
            log_failure(logger,'unresolved_location_suffix_or_name',location=str(name)[:120]);return None
        try:
            result=original(canonical)
            if result is None:log_failure(logger,'geocoder_no_match',location=canonical)
            return result
        except Exception as exc:
            log_failure(logger,'geocoder_exception',location=canonical,error_type=type(exc).__name__);return None
    geocode._movio_optional_weather_geo=True;geocode._original_geocode=original
    bridge.geocode=geocode
'''
    aliases={}
    for source in (getattr(b,'VOICE_PLACE_ALIASES',{}),getattr(b,'GROUP_PLACE_ALIASES',{})):
        for name,variants in source.items():
            if isinstance(name,str):aliases[name]=tuple(v for v in variants if isinstance(v,str))
    module_source=module_template.replace('__PLACE_ALIASES__',repr(aliases))
    compile(module_source,str(helper_path),'exec')
    candidate=types.ModuleType('movio_optional_weather_candidate');candidate.__file__=str(helper_path)
    exec(compile(module_source,str(helper_path),'exec'),candidate.__dict__)

    # Pure text/fixture checks. These make no public API calls and no bookings.
    checks=0
    for text in ('தாம்பரத்திலிருந்து வண்டலூர் போகணும்','Tambaram la irundhu Vandalur poganum',
                 'Tambaram to Vandalur','நான் தாம்பரத்திலிருந்து வண்டலூர் செல்ல வேண்டும்'):
        actual=slots.extract_fields(b,text,{'stage':'count'},engine=engine)['updates']
        if (actual.get('pickup'),actual.get('destination'))!=('Tambaram','Vandalur'):
            raise RuntimeError('Run the latest pickup/destination extraction patch first. Text check failed: '+text)
        checks+=1
    marker='# MOVIO_OPTIONAL_WEATHER_BOOTSTRAP_V1'
    end_marker='# MOVIO_OPTIONAL_WEATHER_BOOTSTRAP_END_V1'
    guard_source=guard_path.read_text(encoding='utf-8')
    if marker in guard_source:
        start=guard_source.index(marker);finish=guard_source.find(end_marker,start)
        if finish<0:raise RuntimeError('The previous optional-weather bootstrap is incomplete; no files were changed.')
        guard_base=guard_source[:start]+guard_source[finish+len(end_marker):]
    else:guard_base=guard_source
    with tempfile.TemporaryDirectory(prefix='movio_weather_checks_') as temporary:
        sandbox=types.ModuleType('movio_optional_weather_selfcheck');sandbox.__file__=str(guard_path)
        exec(compile(guard_base,str(guard_path),'exec'),sandbox.__dict__)
        class FixtureBridge:
            VOICE_PLACE_ALIASES=aliases
            canonical_place=staticmethod(getattr(b,'canonical_place',lambda x:x))
            def __init__(self):self.routes=0
            def say(self,state,key,**kwargs):return 'ASK_'+key
            def geocode(self,name):return {'lat':13.0 if name=='Tambaram' else 12.0,'lng':80.0}
            def road_fare(self,state):self.routes+=1;return {'km':12.3,'fare':230}
        candidate.install_namespace(sandbox.__dict__,temporary,FixtureBridge(),slots)
        normalize=sandbox.__dict__['_movio_weather_normalize']
        for text,gold in (('தாம்பரத்திலிருந்து','Tambaram'),('Tambaram la irundhu','Tambaram'),('வண்டலூருக்கு','Vandalur')):
            if normalize(text)!=gold:raise RuntimeError('Canonical weather-location check failed: '+text)
            checks+=1
        for language in ('English','Tamil','Tanglish'):
            for available in (False,True):
                class FixtureWeather:
                    ttl=900
                    def forecast(self,*args,**kwargs):
                        if not available:raise TimeoutError('offline fixture')
                        return {'status':'ok','condition':'rain','fetched_at':time.time(),'points':[
                            {'location':'pickup','weather_code':61,'precipitation_probability':70},
                            {'location':'destination','weather_code':61,'precipitation_probability':60}]}
                fake=FixtureBridge();when=(datetime.now(candidate.TZ)+timedelta(days=1)).replace(hour=9,minute=0,second=0,microsecond=0)
                state=dict(lang=language,mode='book',stage='time',count=2,vehicle='Mini',pickup='Tambaram',destination='Vandalur',when=when.isoformat())
                state,answer=sandbox.prepare_turn(fake,FixtureWeather(),state,'')
                expected='weather_advisory' if available else 'confirm'
                if state.get('stage')!=expected:raise RuntimeError('Offline weather-flow check failed: '+language)
                if available:
                    if '70%' not in answer or '60%' not in answer or fake.routes:raise RuntimeError('Offline advisory check failed.')
                elif sandbox.advisory(state) or '_wx_weather_consent' in state or fake.routes!=1:
                    raise RuntimeError('Failed weather must silently skip to a separate final confirmation.')
                checks+=1

    # Identify the existing website before modifying or stopping anything.
    health_url='http://127.0.0.1:8080/api/weather-booking/optional-health'
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet')
        if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    in_kernel=[]
    for module in list(sys.modules.values()):
        if not isinstance(module,types.ModuleType):continue
        values=vars(module);handler=values.get('Handler');known_root=values.get('ROOT')
        if not isinstance(handler,type) or known_root is None or not getattr(handler,'_wx_installed',False):continue
        try:
            if Path(known_root).resolve()==root and callable(getattr(handler,'do_POST',None)):in_kernel.append(module)
        except (TypeError,ValueError,OSError):continue
    if os.getpid() not in listeners:in_kernel=[]
    process_details=[]
    known_launchers=[]
    for value in list(globals().values()):
        if type(value) in weakref.ProxyTypes:continue
        try:
            if isinstance(value,subprocess.Popen) and value.poll() is None:known_launchers.append(value)
        except ReferenceError:continue
    for pid in listeners:
        if pid==os.getpid():
            if not in_kernel:raise RuntimeError('An unrecognized in-kernel service owns port 8080; no process was stopped.')
            continue
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server_path for path in scripts):
            raise RuntimeError('A different application owns port 8080; no process was stopped.')
        # Prefer the notebook's own Popen handle when present. Containers can
        # expose host PIDs in psutil while Popen uses kernel-namespace PIDs.
        controller=next((value for value in known_launchers if isinstance(value.args,(list,tuple))
            and list(value.args)==command),process)
        process_details.append((controller,command,cwd,process.environ()))
    if not listeners:raise RuntimeError('Start the original website on port 8080, then rerun this cell.')

    bootstrap="""
# MOVIO_OPTIONAL_WEATHER_BOOTSTRAP_V1
import importlib.util as _mo_import
import sys as _mo_sys
_mo_file=Path(__file__).with_name('movio_optional_weather.py')
_mo_spec=_mo_import.spec_from_file_location('movio_optional_weather',_mo_file)
_mo_module=_mo_import.module_from_spec(_mo_spec)
_mo_sys.modules[_mo_spec.name]=_mo_module
exec(compile(_mo_file.read_text(encoding='utf-8'),str(_mo_file),'exec'),_mo_module.__dict__)
_mo_module.install_namespace(globals())
# MOVIO_OPTIONAL_WEATHER_BOOTSTRAP_END_V1
"""
    new_guard=guard_base.rstrip()+'\n'+bootstrap
    compile(new_guard,str(guard_path),'exec')
    changed=guard_source!=new_guard or not helper_path.exists() or helper_path.read_text(encoding='utf-8')!=module_source
    backup=root/('weather_optional_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    if changed:
        backup.mkdir()
        for path in (guard_path,helper_path):
            if path.exists():shutil.copy2(path,backup/path.name)
        for path,content in ((helper_path,module_source),(guard_path,new_guard)):
            temporary=path.with_name(path.name+'.optional.tmp')
            try:temporary.write_text(content,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)

    active=types.ModuleType('movio_optional_weather');active.__file__=str(helper_path)
    sys.modules[active.__name__]=active
    exec(compile(module_source,str(helper_path),'exec'),active.__dict__)
    status=active.install_namespace(engine,root,b,slots)
    b._wx_optional_weather=active
    # The in-kernel HTTP handler can reference an older loaded guard module.
    for backend in in_kernel:
        sources=[getattr(backend,'_wx_mod',None),getattr(backend,'_wx_module',None)]
        post_globals=getattr(backend.Handler.do_POST,'__globals__',{})
        namespaces=[vars(m) for m in sources if isinstance(m,types.ModuleType)]
        if callable(post_globals.get('prepare_turn')):namespaces.append(post_globals)
        if not namespaces:raise RuntimeError('Could not locate the running website guard. Run the original website-only launcher, then this cell.')
        for namespace in namespaces:
            active.install_namespace(namespace,root,slots=slots)
            namespace['install_backend'](backend)
    # Restart only a verified standalone website process. AI/TTS/Qwen stay loaded.
    for process,command,cwd,environment in process_details:
        if not changed:
            try:
                probe=requests.get(health_url,timeout=(1,2))
                if probe.status_code==200 and probe.json().get('version')==active.VERSION:continue
            except (requests.RequestException,ValueError):pass
        process.terminate();process.wait(timeout=8)
        with (root/'weather_website.log').open('a',encoding='utf-8') as output:
            globals()['WX_WEBSITE_PROCESS']=subprocess.Popen(command,cwd=cwd,env=environment,
                stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    verified=False
    for attempt in range(12):
        try:
            result=requests.get(health_url,timeout=(1,2))
            verified=result.status_code==200 and result.json().get('version')==active.VERSION
            if verified:break
        except (requests.RequestException,ValueError):pass
        time.sleep(.25)
    if not verified:
        raise RuntimeError('Website policy verification failed. See weather_website.log; rerun the original website-only launcher, then this cell. Do not start new voice bookings yet.')
    status.update(offline_checks=checks,backend_verified=True,backup=str(backup) if changed else None)
    print('Optional weather lookup enabled for English, Tamil and Tanglish.')
    print('Offline checks passed:',checks,'| Voice/backend policy verified.')
    print('Available forecast: short advisory. Failed forecast: final booking summary, without an unavailable-weather message.')
    print('Debug log:',root/'weather_lookup.log')
    print('Close the old AI Call popup and start a NEW call.')
    return status

MOVIO_OPTIONAL_WEATHER_STATUS=movio_enable_optional_weather()


Optional weather lookup enabled for English, Tamil and Tanglish.
Offline checks passed: 13 | Voice/backend policy verified.
Available forecast: short advisory. Failed forecast: final booking summary, without an unavailable-weather message.
Debug log: /kaggle/working/MOVIO_MAIN_WEBSITE/weather_lookup.log
Close the old AI Call popup and start a NEW call.


In [89]:
# Copy this ENTIRE file into ONE Kaggle cell after the location and optional
# weather patches, BEFORE evaluation. Keep the original model/UI setup cells.
def movio_enable_booking_confirmation_flow():
    from pathlib import Path
    from datetime import datetime,timedelta
    import json
    import os
    import shutil
    import subprocess
    import sys
    import tempfile
    import time
    import types
    import weakref
    import psutil
    import requests

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if not isinstance(engine,dict) or not callable(engine.get('prepare_turn')):
        raise RuntimeError('Run the original voice/weather activation first.')
    if not engine.get('_MOVIO_OPTIONAL_WEATHER_VERSION'):
        raise RuntimeError('Run the previous optional-weather patch first, then this cell.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    guard_path=root/'integrations/movio_weather_guard.py'
    helper_path=root/'integrations/movio_confirmation_flow.py'
    server_path=root/'server.py'
    for path in (guard_path,server_path):
        if not path.is_file():raise FileNotFoundError(str(path))
    module_source=r'''"""State-aware confirmation on the existing authenticated booking protocol."""
import asyncio
import copy
import functools
import hashlib
import json
import logging
import math
import re
import sys
import time
import types
import unicodedata
from datetime import datetime
from pathlib import Path

VERSION='confirmation-flow-1.0'
COLLECTING_DETAILS='COLLECTING_DETAILS'
WEATHER_ADVISORY='WEATHER_ADVISORY'
AWAITING_WEATHER_CONSENT='AWAITING_WEATHER_CONSENT'
AWAITING_BOOKING_CONFIRMATION='AWAITING_BOOKING_CONFIRMATION'
BOOKING_IN_PROGRESS='BOOKING_IN_PROGRESS'
BOOKING_CONFIRMED='BOOKING_CONFIRMED'
AFFIRMATIVES={
 'yes','yeah','yep','yes please','yes confirm','yes please confirm','confirm','confirmed','okay','ok',
 'ஆம்','ஆமாம்','ஆமா','ஆமாங்க','சரி','சரிங்க','உறுதி செய்யுங்கள்','முன்பதிவு செய்யுங்கள்',
 'seri','sari','aama','aamaa','aamam','amam','aam','book pannunga','book pannu','booking pannunga',
 'yes book pannunga','aama book pannunga','seri book pannunga','புக் பண்ணுங்க','புக்கிங் பண்ணுங்க',
 'சரி புக் பண்ணுங்க','ஆம் புக் பண்ணுங்க','yes book it','book it','go ahead',
}

def normalize(text):
    text=unicodedata.normalize('NFKC',str(text or '')).casefold()
    text=''.join('' if unicodedata.category(c)=='Cf' else ' ' if unicodedata.category(c).startswith('P') else c for c in text)
    return re.sub(r'\s+',' ',text).strip()

def logger_for(root):
    path=Path(root)/'confirmation_flow.log';logger=logging.getLogger('movio.confirmation.'+str(path.resolve()))
    if not logger.handlers:
        handler=logging.FileHandler(path,encoding='utf-8');handler.setFormatter(logging.Formatter('%(asctime)s %(message)s'))
        logger.addHandler(handler)
    logger.setLevel(logging.INFO);logger.propagate=False
    return logger

def event(logger,kind,**fields):logger.info(json.dumps({'event':kind,**fields},ensure_ascii=False,default=str))

def phase_for(state,played=False):
    stage=state.get('stage')
    if stage=='done' and (state.get('booking_id') or state.get('ride_id')):return BOOKING_CONFIRMED
    if stage=='booking_pending':return BOOKING_IN_PROGRESS
    if stage=='confirm':return AWAITING_BOOKING_CONFIRMATION
    if stage=='weather_advisory':return AWAITING_WEATHER_CONSENT if played else WEATHER_ADVISORY
    return COLLECTING_DETAILS

def validate_review(state,engine,now=None):
    for key in ('count','vehicle','pickup','destination','pickup_at'):
        if state.get(key) is None or state.get(key)=='':return key
    try:
        count=int(state['count']);vehicle=state['vehicle']
        if vehicle not in engine['CAPACITY'] or not 1<=count<=engine['CAPACITY'][vehicle]:return 'vehicle'
        if not all(engine['valid_geo'](state.get(key+'_geo')) for key in ('pickup','destination')):return 'route'
        fare=state.get('fare') or {}
        if not all(math.isfinite(float(fare[k])) and float(fare[k])>=0 for k in ('km','fare')):return 'route'
        pickup=datetime.fromisoformat(state['pickup_at'])
        if pickup.tzinfo is None or pickup.timestamp()<=(now.timestamp() if now else time.time()):return 'time'
    except (TypeError,ValueError,KeyError,OverflowError):return 'route'
    return None

def record_phase(db,sid,phase,epoch=None,booking_id=None):
    if not sid:return
    with db.connect() as connection:
        connection.execute('BEGIN IMMEDIATE')
        row=connection.execute('SELECT epoch,state,ride_id FROM wx_calls WHERE id=?',(sid,)).fetchone()
        if row is None or epoch is not None and row['epoch']!=epoch:return
        state=json.loads(row['state'] or '{}')
        if row['ride_id']:
            phase=BOOKING_CONFIRMED;booking_id=row['ride_id'];state['stage']='done'
        elif phase==BOOKING_CONFIRMED:return
        state['dialogue_state']=phase
        if phase==BOOKING_IN_PROGRESS:state['stage']='booking_pending'
        if booking_id:state['booking_id']=int(booking_id)
        connection.execute('UPDATE wx_calls SET state=? WHERE id=?',(json.dumps(state,ensure_ascii=False,separators=(',',':')),sid))

class ConfirmationSocket:
    """Order a queued reply after real playback ACK, without inventing an ACK."""
    def __init__(self,websocket,bridge,engine,logger):
        self.websocket,self.bridge,self.engine,self.logger=websocket,bridge,engine,logger
        self.db=bridge._wx_db;self.sid=None;self.pending=None
        self.affirmatives={normalize(x) for x in engine['POSITIVE']}|{normalize(x) for x in AFFIRMATIVES}
    def __getattr__(self,key):return getattr(self.websocket,key)
    def row(self):
        if not self.sid:return None
        try:return self.db.row(self.sid)
        except self.engine['GuardError']:return None
    async def message(self,reason):
        language=self.query_params.get('language','Tamil')
        text=self.engine['localized'](language,
            'Please hear or read the latest booking summary, then confirm.',
            'சமீபத்திய முன்பதிவு விவரங்களைக் கேட்டோ படித்தோ உறுதி செய்யுங்கள்.',
            'Latest booking details kettu illa padichittu confirm pannunga.')
        if reason.startswith('invalid_'):
            field=reason[len('invalid_'):]
            if field in ('time','pickup_at'):
                text=self.engine['localized'](language,'Please give a valid future pickup time.',
                    'இனிவரும் பயண நேரத்தைச் சொல்லுங்கள்.','Future pickup time sollunga.')
            elif field=='route':
                text=self.engine['localized'](language,'Distance and fare need verification. Say retry or correct the locations.',
                    'தூரமும் கட்டணமும் உறுதி செய்யப்பட வேண்டும். மீண்டும் முயற்சிக்கச் சொல்லுங்கள் அல்லது இடங்களைத் திருத்துங்கள்.',
                    'Distance, fare verify pannanum. Retry sollunga illa places correct pannunga.')
            else:
                labels={'pickup':('pickup','புறப்படும் இடம்','pickup'),'destination':('destination','சேரும் இடம்','drop'),
                    'count':('passenger count','பயணிகள் எண்ணிக்கை','passengers count'),'vehicle':('vehicle with enough seats','போதுமான இருக்கைகள் உள்ள வாகனம்','correct vehicle')}
                en,ta,roman=labels.get(field,('booking detail','முன்பதிவு விவரம்','booking detail'))
                text=self.engine['localized'](language,f'Please provide the correct {en}.',f'சரியான {ta} சொல்லுங்கள்.',f'Correct {roman} sollunga.')
        event(self.logger,'confirmation_not_forwarded',reason=reason)
        await self.websocket.send_json({'event':'error','message':text})
    async def receive_json(self):
        while True:
            if self.pending:
                row=self.row();pending=self.pending
                if (not row or row['ride_id'] or row['summary_nonce']!=pending['summary_nonce']
                        or row['summary_expires']<=time.time()):self.pending=None
                elif row['played']:
                    self.pending=None
                    if phase_for(json.loads(row['state']),True)==AWAITING_BOOKING_CONFIRMATION:
                        invalid=validate_review(json.loads(row['state']),self.engine)
                        if invalid:await self.message('invalid_'+invalid);continue
                    event(self.logger,'queued_confirmation_released',phase=json.loads(row['state']).get('dialogue_state'))
                    return pending
            try:
                message=await asyncio.wait_for(self.websocket.receive_json(),90) if self.pending else await self.websocket.receive_json()
            except asyncio.TimeoutError:
                self.pending=None;await self.message('playback_ack_timeout');continue
            kind=message.get('event')
            if kind=='start':self.sid=message.get('call_id');return message
            if kind=='interrupt':self.pending=None;return message
            row=self.row()
            if kind in ('playback_ended','summary_read'):
                # The original endpoint validates the nonce/epoch and stores ACK.
                return message
            if kind not in ('text','audio') or row is None:return message
            affirmative=kind=='text' and normalize(message.get('text')) in self.affirmatives
            if self.pending and kind=='text' and not affirmative:self.pending=None
            if row['ride_id']:
                if affirmative:continue
                return message
            state=json.loads(row['state'] or '{}');phase=phase_for(state,bool(row['played']))
            if phase==BOOKING_IN_PROGRESS and affirmative:continue
            waiting=phase in (WEATHER_ADVISORY,AWAITING_WEATHER_CONSENT,AWAITING_BOOKING_CONFIRMATION)
            if not waiting or not (affirmative or kind=='audio'):return message
            nonce=message.get('summary_nonce')
            if nonce!=row['summary_nonce'] or not nonce:
                if affirmative:await self.message('missing_or_stale_summary_nonce');continue
                return message
            if affirmative and phase==AWAITING_BOOKING_CONFIRMATION:
                invalid=validate_review(state,self.engine)
                if invalid:
                    event(self.logger,'confirmation_invalid_slots',slot=invalid)
                    await self.message('invalid_'+invalid);continue
            if not row['played']:
                # Preserve the existing weather replay protocol. The browser
                # queues early weather replies; only FINAL confirmation needs
                # server-side ACK ordering to prevent the booking loop.
                if phase!=AWAITING_BOOKING_CONFIRMATION:return message
                if self.pending is None:
                    self.pending=dict(message)
                    event(self.logger,'confirmation_queued_for_playback',phase=phase)
                continue
            return message
    async def send_json(self,message):
        message=dict(message);row=self.row();phase=None
        if message.get('event') in ('reply','speech_chunk'):
            if message.get('done') and row and row['ride_id']:phase=BOOKING_CONFIRMED
            elif message.get('stage')=='confirm':phase=AWAITING_BOOKING_CONFIRMATION
            elif message.get('stage')=='weather_advisory':phase=WEATHER_ADVISORY
            else:phase=COLLECTING_DETAILS
        elif message.get('event')=='action' and message.get('type')=='weather_book':phase=BOOKING_IN_PROGRESS
        if phase and row:
            record_phase(self.db,self.sid,phase,message.get('epoch'))
            message['dialogue_state']=phase
            if message.get('event') in ('reply','action'):event(self.logger,'confirmation_state',phase=phase)
        await self.websocket.send_json(message)

def install_socket(bridge,engine,root,_route_wrapper=None):
    previous=getattr(bridge,'_wx_endpoint',None)
    if not callable(previous):raise RuntimeError('The original voice/weather endpoint is not active.')
    if getattr(previous,'_movio_confirmation_flow',None)==VERSION:return False
    logger=logger_for(root)
    async def wrapped(ws):
        if ws.query_params.get('mode','book')=='cancel':return await previous(ws)
        return await previous(ConfirmationSocket(ws,bridge,engine,logger))
    # Keep the original guard's globals, so existing notebook add-ons still find
    # prepare_turn/collect_details and the original STT/TTS implementations.
    wrapped=types.FunctionType(wrapped.__code__,engine,wrapped.__name__,wrapped.__defaults__,wrapped.__closure__)
    wrapped._movio_confirmation_flow=VERSION;wrapped._original_endpoint=previous
    # Bridge/engine/logger are closure variables; the proxy class is a global.
    engine['ConfirmationSocket']=ConfirmationSocket
    if _route_wrapper is None:
        from starlette.routing import websocket_session
        _route_wrapper=websocket_session
    route=next(r for r in bridge.app.routes if getattr(r,'path','')=='/browser-call')
    route.endpoint=wrapped;route.app=_route_wrapper(wrapped);bridge._wx_endpoint=wrapped
    bridge._wx_confirmation_flow=VERSION
    return True

def install_namespace(engine,root=None):
    root=Path(root or Path(engine['__file__']).resolve().parent.parent)
    if engine.get('_MOVIO_CONFIRMATION_FLOW_VERSION')==VERSION:return {'version':VERSION,'already_installed':True}
    logger=logger_for(root)
    original={key:engine[key] for key in ('prepare_turn','install_bridge','install_backend','reply_normalized')}
    engine['POSITIVE'].update(normalize(x) for x in AFFIRMATIVES)
    old_played=engine['GuardDB'].played;old_permit=engine['GuardDB'].permit;old_commit=engine['GuardDB'].commit
    def prepare_turn(bridge,weather,state,text,now=None):
        phase=state.get('dialogue_state')
        frozen=(phase in (AWAITING_BOOKING_CONFIRMATION,BOOKING_IN_PROGRESS)
            and state.get('stage') in ('fare','confirm','booking_pending') and (not str(text).strip() or normalize(text) in engine['POSITIVE']))
        if frozen:
            invalid=validate_review(state,engine,now)
            if not invalid:
                state['stage']='confirm';state['dialogue_state']=AWAITING_BOOKING_CONFIRMATION
                return state,engine['summary_text'](bridge,state)
        state,answer=original['prepare_turn'](bridge,weather,state,text,now=now)
        state['dialogue_state']=phase_for(state)
        return state,answer
    def played(db,sid,epoch,nonce,explicit_read=False):
        result=old_played(db,sid,epoch,nonce,explicit_read=explicit_read)
        if result:
            row=db.row(sid)
            phase=AWAITING_WEATHER_CONSENT if row['summary_kind']=='weather' else AWAITING_BOOKING_CONFIRMATION
            record_phase(db,sid,phase,row['epoch'])
        return result
    def permit(db,sid,epoch,nonce,state):
        # User replies are validated by ConfirmationSocket before reaching this
        # method. Keep the existing permit API and backend commit validation.
        try:return old_permit(db,sid,epoch,nonce,state)
        except engine['GuardError'] as exc:
            event(logger,'confirmation_permit_rejected',reason=str(exc));raise
    def commit(db,connection,owner,token,ensure_code):
        result=old_commit(db,connection,owner,token,ensure_code)
        permit=connection.execute('SELECT call_id FROM wx_permits WHERE token_hash=?',(hashlib.sha256(str(token).encode()).hexdigest(),)).fetchone()
        row=connection.execute('SELECT state,ride_id FROM wx_calls WHERE id=?',(permit['call_id'],)).fetchone()
        if row and row['ride_id']==result['id']:
            state=json.loads(row['state'] or '{}');state.update(stage='done',dialogue_state=BOOKING_CONFIRMED,booking_id=result['id'])
            connection.execute('UPDATE wx_calls SET state=? WHERE id=?',(engine['canonical'](state),permit['call_id']))
            event(logger,'booking_persisted',booking_id=result['id'],reused=bool(result.get('reused')))
        return result
    def install_bridge(bridge,db_path,weather=None,_route_wrapper=None):
        result=original['install_bridge'](bridge,db_path,weather=weather,_route_wrapper=_route_wrapper)
        install_socket(bridge,engine,root,_route_wrapper)
        return {**result,'confirmation_flow':VERSION}
    def install_backend(module,weather=None):
        result=original['install_backend'](module,weather=weather)
        previous=module.Handler.do_GET
        if not getattr(previous,'_movio_confirmation_health',False):
            def get(handler):
                if handler.path.split('?',1)[0]=='/api/weather-booking/confirmation-health':
                    return handler.json(200,{'ok':True,'version':VERSION,'final_confirmation_required':True,'duplicate_guard':'atomic_call_transaction'})
                return previous(handler)
            get._movio_confirmation_health=True;module.Handler.do_GET=get
        return result
    engine['GuardDB'].played=played;engine['GuardDB'].permit=permit;engine['GuardDB'].commit=commit
    engine.update(prepare_turn=prepare_turn,install_bridge=install_bridge,install_backend=install_backend,
        reply_normalized=normalize,
        _MOVIO_CONFIRMATION_FLOW_VERSION=VERSION,_movio_confirmation_original={**original,'played':old_played,'permit':old_permit,'commit':old_commit})
    return {'version':VERSION}

CLIENT_JS=r"""
// MOVIO_CONFIRMATION_CLIENT_V1 — behaviour only; existing layout/theme retained.
(()=>{
 'use strict';
 if(window.__movioConfirmationFlow)return;window.__movioConfirmationFlow=true;
 const affirmative=new Set(__AFFIRMATIVES_JSON__);
 const norm=s=>String(s||'').normalize('NFKC').toLowerCase().replace(/\p{Cf}/gu,'').replace(/\p{P}/gu,' ').replace(/\s+/g,' ').trim();
 const yes=s=>affirmative.has(norm(s));
 const oldSend=send,oldTyped=typed;
 let observed=null,latest=null,acknowledged=null,queued=null,saving=false;
 const submitted=new Set();
 function observe(){
  if(!socket||socket===observed)return;
  observed=socket;latest=null;acknowledged=null;queued=null;saving=false;submitted.clear();
  socket.addEventListener('message',e=>{
   let m;try{m=JSON.parse(e.data);}catch{return;}
   if(m.event==='reply'){
    latest=m.summary_nonce?{nonce:m.summary_nonce,epoch:m.epoch,kind:m.summary_kind||'final'}:null;
    if(queued&&(!latest||queued.nonce!==latest.nonce))queued=null;
    if(m.done){saving=false;queued=null;}
   }
   if(m.event==='action'&&m.type==='weather_book'){saving=true;queued=null;}
   if(m.event==='error'||m.event==='booking_error'){saving=false;if(latest)submitted.delete(latest.nonce);}
   if(m.event==='interrupted'){queued=null;latest=null;acknowledged=null;}
  });
 }
 function flush(){
  const item=queued;
  if(!item||saving||!connected||busy||playing)return;
  if(!latest||item.nonce!==latest.nonce||summaryNonce!==item.nonce){queued=null;return;}
  if(acknowledged!==item.nonce)return;
  queued=null;oldTyped(item.text);
 }
 send=function(message){
  observe();
  if(message.event==='playback_ended'||message.event==='summary_read'){
   if(latest&&message.summary_nonce===latest.nonce)acknowledged=message.summary_nonce;
   oldSend(message);setTimeout(flush,0);return;
  }
  if(message.event==='interrupt'){queued=null;latest=null;acknowledged=null;}
  if(message.event==='text'&&yes(message.text)&&message.summary_nonce){
   if(submitted.has(message.summary_nonce))return;
   submitted.add(message.summary_nonce);
  }
  oldSend(message);
 };
 typed=function(text){
  observe();
  if(yes(text)&&latest&&summaryNonce===latest.nonce){
   if(saving||submitted.has(latest.nonce))return;
   if(playing||busy||acknowledged!==latest.nonce){queued={text:String(text),nonce:latest.nonce};return;}
  }else queued=null;
  oldTyped(text);
 };
})();
"""

def patch_html(html,affirmatives):
    start='<!-- MOVIO_CONFIRMATION_CLIENT_START_V1 -->';end='<!-- MOVIO_CONFIRMATION_CLIENT_END_V1 -->'
    if start in html:
        a=html.index(start);b=html.find(end,a)
        if b<0:raise RuntimeError('Incomplete previous confirmation client patch.')
        tail=html[b+len(end):]
        if tail.startswith('\n'):tail=tail[1:]
        html=html[:a]+tail
    required=('function typed(', 'function send(', 'summaryNonce','commitBooking','</body>')
    if any(token not in html for token in required):raise RuntimeError('The active call-page script is not the expected MOVIO version; no UI replacement was made.')
    js=CLIENT_JS.replace('__AFFIRMATIVES_JSON__',json.dumps(sorted({normalize(x) for x in affirmatives}|{normalize(x) for x in AFFIRMATIVES}),ensure_ascii=False))
    return html.replace('</body>',start+'\n<script>\n'+js+'</script>\n'+end+'\n</body>',1)
'''
    compile(module_source,str(helper_path),'exec')
    candidate=types.ModuleType('movio_confirmation_candidate');candidate.__file__=str(helper_path)
    exec(compile(module_source,str(helper_path),'exec'),candidate.__dict__)
    # These are text/fixture checks, not measured ASR or real-user evaluation.
    checks=0
    for text in ('Yes','ஆம்','ஆமாம்','சரி','Okay','Seri','Aama','Confirm','Book pannunga'):
        if candidate.normalize(text) not in candidate.AFFIRMATIVES:raise RuntimeError('Affirmative text check failed: '+text)
        checks+=1
    for text in ('No','Yes change pickup','சரி ஆனால் வாகனம் மாற்று','Do not confirm'):
        if candidate.normalize(text) in candidate.AFFIRMATIVES:raise RuntimeError('Unsafe affirmative match: '+text)
        checks+=1
    html_inputs={}
    for path in (root/'integrations/ai_call_page.html',root/'integrations/weather_call_page.html'):
        if path.is_file():html_inputs[path]=path.read_text(encoding='utf-8')
    current_html=getattr(b,'HTML','')
    if not isinstance(current_html,str) or not current_html:raise RuntimeError('The current AI Call page is missing.')
    positives=set(engine['POSITIVE'])|candidate.AFFIRMATIVES
    patched_live_html=candidate.patch_html(current_html,positives)
    html_outputs={path:candidate.patch_html(text,positives) for path,text in html_inputs.items()}
    node=shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory(prefix='movio_confirmation_js_') as temporary:
            script=Path(temporary)/'confirmation_client.js'
            script.write_text(candidate.CLIENT_JS.replace('__AFFIRMATIVES_JSON__',json.dumps(sorted(positives),ensure_ascii=False)),encoding='utf-8')
            result=subprocess.run([node,'--check',str(script)],capture_output=True,text=True)
            if result.returncode:raise RuntimeError('Client syntax check failed: '+result.stderr[:500])
        checks+=1

    # Identify the existing website before modifying or stopping anything.
    health_url='http://127.0.0.1:8080/api/weather-booking/confirmation-health'
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet')
        if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    in_kernel=[]
    for module in list(sys.modules.values()):
        if not isinstance(module,types.ModuleType):continue
        values=vars(module);handler=values.get('Handler');known_root=values.get('ROOT')
        if not isinstance(handler,type) or known_root is None or not getattr(handler,'_wx_installed',False):continue
        try:
            if Path(known_root).resolve()==root and callable(getattr(handler,'do_POST',None)):in_kernel.append(module)
        except (TypeError,ValueError,OSError):continue
    if os.getpid() not in listeners:in_kernel=[]
    process_details=[]
    known_launchers=[]
    for value in list(globals().values()):
        if type(value) in weakref.ProxyTypes:continue
        try:
            if isinstance(value,subprocess.Popen) and value.poll() is None:known_launchers.append(value)
        except ReferenceError:continue
    for pid in listeners:
        if pid==os.getpid():
            if not in_kernel:raise RuntimeError('An unrecognized in-kernel service owns port 8080; no process was stopped.')
            continue
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server_path for path in scripts):
            raise RuntimeError('A different application owns port 8080; no process was stopped.')
        # Prefer the notebook's own Popen handle when present. Containers can
        # expose host PIDs in psutil while Popen uses kernel-namespace PIDs.
        controller=next((value for value in known_launchers if isinstance(value.args,(list,tuple))
            and list(value.args)==command),process)
        process_details.append((controller,command,cwd,process.environ()))
    if not listeners:raise RuntimeError('Start the original website on port 8080, then rerun this cell.')

    marker='# MOVIO_CONFIRMATION_FLOW_BOOTSTRAP_V1'
    end_marker='# MOVIO_CONFIRMATION_FLOW_BOOTSTRAP_END_V1'
    guard_source=guard_path.read_text(encoding='utf-8')
    if marker in guard_source:
        start=guard_source.index(marker);finish=guard_source.find(end_marker,start)
        if finish<0:raise RuntimeError('The previous confirmation bootstrap is incomplete; no files were changed.')
        guard_base=guard_source[:start]+guard_source[finish+len(end_marker):]
    else:guard_base=guard_source
    bootstrap="""
# MOVIO_CONFIRMATION_FLOW_BOOTSTRAP_V1
import importlib.util as _cf_import
import sys as _cf_sys
_cf_file=Path(__file__).with_name('movio_confirmation_flow.py')
_cf_spec=_cf_import.spec_from_file_location('movio_confirmation_flow',_cf_file)
_cf_module=_cf_import.module_from_spec(_cf_spec)
_cf_sys.modules[_cf_spec.name]=_cf_module
exec(compile(_cf_file.read_text(encoding='utf-8'),str(_cf_file),'exec'),_cf_module.__dict__)
_cf_module.install_namespace(globals())
# MOVIO_CONFIRMATION_FLOW_BOOTSTRAP_END_V1
"""
    new_guard=guard_base.rstrip()+'\n'+bootstrap
    compile(new_guard,str(guard_path),'exec')
    writes={helper_path:module_source,guard_path:new_guard,**html_outputs}
    changed=any(not path.exists() or path.read_text(encoding='utf-8')!=content for path,content in writes.items())
    backup=root/('confirmation_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    if changed:
        backup.mkdir()
        for path,content in writes.items():
            if path.exists():shutil.copy2(path,backup/path.name)
            if path.exists() and path.read_text(encoding='utf-8')==content:continue
            temporary=path.with_name(path.name+'.confirmation.tmp')
            try:temporary.write_text(content,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
    active=types.ModuleType('movio_confirmation_flow');active.__file__=str(helper_path)
    sys.modules[active.__name__]=active
    exec(compile(module_source,str(helper_path),'exec'),active.__dict__)
    status=active.install_namespace(engine,root)
    active.install_socket(b,engine,root)
    b.HTML=patched_live_html
    for backend in in_kernel:
        sources=[getattr(backend,'_wx_mod',None),getattr(backend,'_wx_module',None)]
        post_globals=getattr(backend.Handler.do_POST,'__globals__',{})
        namespaces=[vars(m) for m in sources if isinstance(m,types.ModuleType)]
        if callable(post_globals.get('prepare_turn')):namespaces.append(post_globals)
        if not namespaces:raise RuntimeError('Could not locate the website guard. Run the original website-only launcher, then this cell.')
        for namespace in namespaces:
            active.install_namespace(namespace,root);namespace['install_backend'](backend)
    for process,command,cwd,environment in process_details:
        if not changed:
            try:
                probe=requests.get(health_url,timeout=(1,2))
                if probe.status_code==200 and probe.json().get('version')==active.VERSION:continue
            except (requests.RequestException,ValueError):pass
        process.terminate();process.wait(timeout=8)
        with (root/'weather_website.log').open('a',encoding='utf-8') as output:
            globals()['WX_WEBSITE_PROCESS']=subprocess.Popen(command,cwd=cwd,env=environment,
                stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    verified=False
    for attempt in range(12):
        try:
            result=requests.get(health_url,timeout=(1,2))
            verified=result.status_code==200 and result.json().get('version')==active.VERSION
            if verified:break
        except (requests.RequestException,ValueError):pass
        time.sleep(.25)
    if not verified:raise RuntimeError('Backend confirmation-policy verification failed. See weather_website.log; rerun the original website-only launcher, then this cell.')
    status.update(backend_verified=True,offline_checks=checks,backup=str(backup) if changed else None)
    print('Booking confirmation flow enabled for English, Tamil and Tanglish.')
    print('Offline text checks:',checks,'| Website backend verified.')
    print('One weather Yes gives the final summary. A separate final Yes saves the booking once.')
    print('If weather was unavailable, only the final booking confirmation is required.')
    print('Debug log:',root/'confirmation_flow.log')
    print('Close the old AI Call popup and open a NEW call from the signed-in dashboard.')
    return status

MOVIO_CONFIRMATION_FLOW_STATUS=movio_enable_booking_confirmation_flow()


Booking confirmation flow enabled for English, Tamil and Tanglish.
Offline text checks: 14 | Website backend verified.
One weather Yes gives the final summary. A separate final Yes saves the booking once.
If weather was unavailable, only the final booking confirmation is required.
Debug log: /kaggle/working/MOVIO_MAIN_WEBSITE/confirmation_flow.log
Close the old AI Call popup and open a NEW call from the signed-in dashboard.


In [90]:
# Paste this ENTIRE file into ONE Kaggle cell after the previous slot, optional
# weather and confirmation fixes, BEFORE evaluation. No installation is needed.
def movio_enable_time_route_fix():
    from pathlib import Path
    from datetime import datetime
    import importlib.util
    import json
    import os
    import shutil
    import sys
    import tempfile
    import types

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if not isinstance(engine,dict) or not callable(engine.get('prepare_turn')):
        raise RuntimeError('Run the original voice/weather setup first, in this kernel.')
    if not engine.get('_MOVIO_OPTIONAL_WEATHER_VERSION') or not engine.get('_MOVIO_CONFIRMATION_FLOW_VERSION'):
        raise RuntimeError('Run the previous optional-weather and booking-confirmation fix cells first, then run this cell.')
    slots=getattr(b,'_wx_group_slots_module',None)
    if slots is None:
        raise RuntimeError('Run the previous shared pickup/destination extraction fix first.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    folder=root/'integrations'
    guard_path=folder/'movio_weather_guard.py'
    slots_path=folder/'movio_group_booking_slots.py'
    helper_path=folder/'movio_time_route.py'
    for path in (guard_path,slots_path):
        if not path.is_file():raise FileNotFoundError(str(path))
    source=r'''"""Targeted time/slot repair for the existing MOVIO voice booking guard.

No replacement booking database, TTS, STT, weather provider, UI, or fare model.
"""
import copy
import importlib.util
import json
import logging
import math
import re
import threading
import time
from datetime import datetime
from pathlib import Path

VERSION='time-route-1.0'
_LETTERS=r'A-Za-z\u0B80-\u0BFF'
_AFTER_AT_CLOCK=r'(?=\s*(?:$|[,;.?!]|(?:a\.?m\.?|p\.?m\.?|please|today|tomorrow|for|with|and|vehicle|cab|taxi|car|pickup|venum|vendum|o[.\s\x27’]?clock)(?![a-z])))'


def logger_for(root):
    path=Path(root)/'booking_lookup.log'
    logger=logging.getLogger('movio.time_route.'+str(path.resolve()))
    logger.setLevel(logging.INFO);logger.propagate=False
    if not logger.handlers:
        handler=logging.FileHandler(path,encoding='utf-8')
        handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
        logger.addHandler(handler)
    return logger


def log(logger,name,**fields):
    # Never log user/session credentials, response bodies, or booking tokens.
    logger.info(json.dumps(dict(event=name,**fields),ensure_ascii=False,default=str))


def clock_patterns(engine):
    words={word:n for word,n in engine['number_lexicon'](59).items() if not word.isdigit() and 0<=n<=23}
    values=r'\d{1,2}(?::\d{2})?|'+'|'.join(re.escape(word) for word in sorted(words,key=len,reverse=True))
    number=r'(?<!['+_LETTERS+r'\d])(?:'+values+r')(?!['+_LETTERS+r'\d])'
    unit=r'(?:மணி(?:க்கு|யளவில்)?|manikku|mani|o[.\s\x27’]?clock)(?!['+_LETTERS+r'])'
    # Consume AM/PM together with "at 9". Leaving "AM" behind could make a
    # following bare route look like the invalid pickup "AM Tambaram".
    period=r'\s*(?:a\.?m\.?|p\.?m\.?)(?![a-z])'
    expression=number+r'\s*'+unit+r'|(?<![a-z])(?:at|around)\s+'+number+'(?:'+period+'|'+_AFTER_AT_CLOCK+')'
    return words,number,unit,expression


def patch_slots(slots,engine):
    """Make spoken clock expressions ineligible to replace a place slot."""
    namespace=slots if isinstance(slots,dict) else vars(slots)
    if namespace.get('_MOVIO_CLOCK_SLOT_VERSION')==VERSION:return
    words,number,unit,expression=clock_patterns(engine)
    old_time=namespace['_TIME'];old_carrier=namespace['_CARRIER']
    namespace['_TIME']=re.compile('(?:'+old_time.pattern+')|(?:'+expression+')',old_time.flags)
    namespace['_CARRIER']=re.compile(r'(?<!['+_LETTERS+r'])(?:'+old_carrier.pattern+r')(?!['+_LETTERS+r'])',old_carrier.flags)
    parser=namespace['Parser'];old_clean=parser.clean_place
    bare_unit=re.compile(r'(?<!['+_LETTERS+r'])'+unit,re.I)
    def clean_place(self,value,role):
        # Reject malformed leftovers such as "கு மணிக்கு" as well as a full clock.
        # Known place names remain eligible even if they happen to contain a unit.
        key=namespace['place_key'](value)
        if key not in self.places and bare_unit.search(str(value or '')):return None
        return old_clean(self,value,role)
    parser.clean_place=clean_place
    namespace['_MOVIO_CLOCK_SLOT_VERSION']=VERSION


def finite_fare(value):
    try:return isinstance(value,dict) and all(math.isfinite(float(value[k])) and float(value[k])>=0 for k in ('km','fare'))
    except (ValueError,TypeError,KeyError,OverflowError):return False


def attach_lookups(bridge,engine,root):
    """Retain the configured providers/rates; expire negative geocoder caches."""
    if getattr(bridge,'_movio_time_route_lookups',None)==VERSION:return
    logger=logger_for(root);valid_geo=engine['valid_geo']
    normalize=engine.get('_movio_weather_normalize',lambda name,*args:name)
    original_geo=getattr(bridge,'geocode',None);original_route=getattr(bridge,'road_fare',None)
    if not callable(original_geo) or not callable(original_route):raise RuntimeError('The original geocoder/road-fare functions are missing.')
    raw=original_geo;seen=set()
    while id(raw) not in seen:
        seen.add(id(raw));inner=getattr(raw,'_original_geocode',None)
        if not callable(inner):break
        raw=inner
    raw_globals=getattr(raw,'__globals__',{})
    cache=getattr(bridge,'GEOCODE_CACHE',raw_globals.get('GEOCODE_CACHE'))
    geo_lock=getattr(bridge,'GEO_LOCK',raw_globals.get('GEO_LOCK')) or threading.RLock()
    # The original geocoder caches None indefinitely. Remove only failed entries;
    # all valid successful provider results are kept.
    if isinstance(cache,dict):
        with geo_lock:
            for key in list(cache):
                if not valid_geo(cache.get(key)):cache.pop(key,None)
    failed={};failed_lock=threading.RLock()
    def geocode(name):
        canonical=normalize(name)
        if not canonical:
            log(logger,'geocode_rejected_name',location=str(name)[:120]);return None
        with failed_lock:
            if canonical in failed and time.monotonic()-failed[canonical]<30:return None
            if isinstance(cache,dict):
                with geo_lock:
                    for key in list(cache):
                        if isinstance(key,tuple) and key and str(key[-1]).casefold()==canonical.casefold() and not valid_geo(cache.get(key)):
                            cache.pop(key,None)
            try:result=original_geo(canonical)
            except Exception as exc:
                result=None;log(logger,'geocode_exception',location=canonical,error_type=type(exc).__name__)
            if valid_geo(result):
                failed.pop(canonical,None)
                log(logger,'geocode_ok',location=canonical)
                return result
            failed[canonical]=time.monotonic()
            log(logger,'geocode_failed',location=canonical,retry_after_s=30)
            return None
    # Keep the optional-weather wrapper from repeatedly wrapping this function.
    geocode._movio_optional_weather_geo=True
    geocode._original_geocode=original_geo
    def road_fare(state):
        if not all(valid_geo(state.get(role+'_geo')) for role in ('pickup','destination')):
            log(logger,'route_missing_coordinates',pickup=state.get('pickup'),destination=state.get('destination'))
            return None
        try:result=original_route(state)
        except Exception as exc:
            response=getattr(exc,'response',None)
            log(logger,'route_exception',error_type=type(exc).__name__,http_status=getattr(response,'status_code',None))
            return None
        if not finite_fare(result):
            log(logger,'route_no_valid_estimate',vehicle=state.get('vehicle'))
            return None
        log(logger,'route_ok',km=result['km'],estimated_fare=result['fare'])
        return result
    bridge.geocode=geocode;bridge.road_fare=road_fare
    bridge._movio_time_route_lookups=VERSION


def install_namespace(engine,root=None,bridge=None,slots=None):
    root=Path(root or Path(engine['__file__']).resolve().parent.parent)
    if slots is None:slots=getattr(bridge,'_wx_group_slots_module',None)
    if slots is None:
        path=root/'integrations/movio_group_booking_slots.py'
        spec=importlib.util.spec_from_file_location('_movio_clock_slots',path)
        slots=importlib.util.module_from_spec(spec);spec.loader.exec_module(slots)
    slots._ENGINE=engine;patch_slots(slots,engine)
    if engine.get('_MOVIO_TIME_ROUTE_VERSION')==VERSION:
        if bridge is not None:attach_lookups(bridge,engine,root)
        return dict(version=VERSION,already_installed=True)
    logger=logger_for(root)
    original={name:engine[name] for name in ('time_components','components_datetime','update_time_memory','collect_details','install_bridge')}
    # Also patch any live module already captured by earlier collector wrappers.
    previous_group=engine.get('_movio_optional_weather_original',{}).get('collect_details')
    if callable(previous_group) and 'Parser' in getattr(previous_group,'__globals__',{}):patch_slots(previous_group.__globals__,engine)
    words,number,unit,expression=clock_patterns(engine)
    at_clock=re.compile(r'(?<![a-z])(?:at|around)\s+(?P<hour>\d{1,2}|'
        +'|'.join(re.escape(k) for k in sorted(words,key=len,reverse=True))+r')(?!['+_LETTERS+r'\d])'+_AFTER_AT_CLOCK,re.I)
    def time_components(raw,now,allow_bare=False):
        result=original['time_components'](raw,now,allow_bare=allow_bare)
        if 'hour' not in result:
            text=engine['unicodedata'].normalize('NFKC',str(raw or '')).casefold().translate(str.maketrans('௦௧௨௩௪௫௬௭௮௯','0123456789'))
            match=at_clock.search(text)
            if match:
                tail=text[match.end():]
                # "at 4 passengers" or "at 4 kilometres" is not a clock.
                if not re.match(r'\s*(?:passengers?|people|persons?|kilomet(?:er|re)s?|km|பேர்|கிலோமீட்டர்)\b',tail,re.I):
                    value=match['hour'];hour=int(value) if value.isdigit() else words[value]
                    if not 0<=hour<=23:raise engine['GuardError']('invalid_datetime')
                    result.update(hour=hour,minute=0,clock24=hour==0 or hour>12)
        return result
    def components_datetime(parts,now):
        if parts.get('date') and 'hour' in parts and not parts.get('clock24') and not parts.get('period') and not parts.get('day_period') and 1<=parts['hour']<=12:
            local_now=now.astimezone(engine['TZ'])
            try:day=datetime.fromisoformat(parts['date']).date()
            except (ValueError,TypeError):raise engine['GuardError']('invalid_datetime')
            if day==local_now.date():
                try:
                    values=[datetime.combine(day,datetime.min.time(),tzinfo=engine['TZ']).replace(hour=h,minute=parts.get('minute',0))
                        for h in (parts['hour']%12,parts['hour']%12+12)]
                except (ValueError,TypeError):raise engine['GuardError']('invalid_datetime')
                future=[value for value in values if value>local_now]
                if len(future)==1:return future[0]
                if not future:raise engine['GuardError']('past_datetime')
                # Two future interpretations are genuinely ambiguous.
        return original['components_datetime'](parts,now)
    def update_time_memory(state,text,now):
        result,error=original['update_time_memory'](state,text,now)
        parts=state.get('_wx_time') or {}
        if result and not parts.get('clock24') and not parts.get('period') and not parts.get('day_period'):
            state['_wx_time_inference']={'rule':'only_future_interpretation_today','period':'am' if result.hour<12 else 'pm',
                'resolved':result.isoformat(),'checked_at':now.astimezone(engine['TZ']).isoformat()}
            # No inferred AM/PM is inserted into parts: a new date/time must be
            # independently re-evaluated rather than inheriting a guessed period.
            log(logger,'clock_inferred',**state['_wx_time_inference'])
        else:state.pop('_wx_time_inference',None)
        return result,error
    def collect_details(b,state,text):
        return slots.collect_details(b,state,text)
    def install_bridge(b,db_path,weather=None,_route_wrapper=None):
        result=original['install_bridge'](b,db_path,weather=weather,_route_wrapper=_route_wrapper)
        attach_lookups(b,engine,root)
        return dict(result,time_route=VERSION)
    engine.update(time_components=time_components,components_datetime=components_datetime,update_time_memory=update_time_memory,
        collect_details=collect_details,install_bridge=install_bridge,_MOVIO_TIME_ROUTE_VERSION=VERSION,
        _movio_time_route_original=original)
    if bridge is not None:attach_lookups(bridge,engine,root)
    return dict(version=VERSION,log=str(root/'booking_lookup.log'))


def diagnose(bridge,engine,text,check_apis=False,now=None):
    """No booking, database write, speech generation, or fabricated consent."""
    now=(now or datetime.now(engine['TZ'])).astimezone(engine['TZ'])
    state=dict(mode='book',stage='count',lang='Tamil')
    engine['collect_details'](bridge,state,text)
    try:pickup,error=engine['update_time_memory'](state,text,now)
    except engine['GuardError'] as exc:pickup,error=None,str(exc)
    report={key:state.get(key) for key in ('count','vehicle','pickup','destination','pickup_at')}
    report.update(local_now=now.isoformat(),time_error=error)
    if not check_apis:return report
    if error or not all(state.get(key) for key in ('count','vehicle','pickup','destination')):
        report['api_check']='skipped: complete valid booking details first';return report
    for role in ('pickup','destination'):state[role+'_geo']=bridge.geocode(state[role])
    report['coordinates_ok']=all(engine['valid_geo'](state.get(role+'_geo')) for role in ('pickup','destination'))
    if not report['coordinates_ok']:
        report.update(weather='not queried: geocoding failed',road_estimate='not queried: geocoding failed');return report
    weather=bridge._wx_weather
    try:result=weather.forecast([state['pickup_geo'],state['destination_geo']],pickup,now=now)
    except Exception as exc:result=dict(status='unavailable',reason=type(exc).__name__)
    report['weather']=result.get('status');report['weather_reason']=result.get('reason')
    if result.get('status')=='ok':
        report['rain_probability_pct']={row['location']:row.get('precipitation_probability') for row in result.get('points',[])}
    fare=bridge.road_fare(copy.deepcopy(state))
    report['road_estimate']=fare if finite_fare(fare) else 'unavailable: check booking_lookup.log'
    return report
'''
    compile(source,str(helper_path),'exec')
    candidate=types.ModuleType('movio_time_route_candidate');candidate.__file__=str(helper_path)
    exec(compile(source,str(helper_path),'exec'),candidate.__dict__)
    marker='# MOVIO_TIME_ROUTE_BOOTSTRAP_V1';end_marker='# MOVIO_TIME_ROUTE_BOOTSTRAP_END_V1'
    guard_base=guard_path.read_text(encoding='utf-8')
    if marker in guard_base:
        start=guard_base.index(marker);end=guard_base.find(end_marker,start)
        if end<0:raise RuntimeError('Incomplete previous time-fix bootstrap; no files changed.')
        guard_base=guard_base[:start]+guard_base[end+len(end_marker):]
    # Isolated text/fixture tests; no API requests, audio generation or bookings.
    example='நான் சென்னை சென்ட்ரலில் இருந்து கொட்டிவாக்கம் செல்ல வேண்டும். நாங்கள் நான்கு பேர் இருக்கிறோம். SUV வேண்டும். இன்று நான்கு மணிக்கு வாகனம் வேண்டும்.'
    with tempfile.TemporaryDirectory(prefix='movio_time_check_') as temporary:
        sandbox=types.ModuleType('movio_time_route_selfcheck');sandbox.__file__=str(guard_path)
        exec(compile(guard_base,str(guard_path),'exec'),sandbox.__dict__)
        fixture_slots=types.ModuleType('movio_time_slots_selfcheck')
        exec(compile(slots_path.read_text(encoding='utf-8'),str(slots_path),'exec'),fixture_slots.__dict__)
        candidate.install_namespace(sandbox.__dict__,temporary,slots=fixture_slots)
        now=datetime(2026,10,18,14,0,tzinfo=sandbox.TZ)
        cases=[('இன்று நான்கு மணிக்கு',14,16),('today at four',14,16),
            ('innaiku naalu manikku',14,16),('today at 4:00',14,16),
            ('today at 4 AM',14,'past_datetime'),('today at 4',2,'clarify_period'),
            ('tomorrow at 4',14,'clarify_period'),('today at 12',14,'past_datetime'),
            ('today at 12',10,12),('today at 12 AM',10,'past_datetime'),
            ('today at 4',18,'past_datetime')]
        checks=0
        for text,hour,gold in cases:
            try:actual=sandbox.resolve_pickup(text,now=now.replace(hour=hour)).hour
            except sandbox.GuardError as exc:actual=str(exc)
            if actual!=gold:raise RuntimeError('Time check failed: '+repr((text,actual,gold)))
            checks+=1
        class FixtureBridge:
            VOICE_PLACE_ALIASES=getattr(b,'VOICE_PLACE_ALIASES',{})
            GROUP_PLACE_ALIASES=getattr(b,'GROUP_PLACE_ALIASES',{})
            canonical_place=staticmethod(getattr(b,'canonical_place',lambda name:name))
            def say(self,state,key,**kw):return 'ASK_'+key
            def geocode(self,name):return dict(lat=13.08 if name=='Chennai Central' else 12.97,lng=80.26)
            def road_fare(self,state):return dict(km=12.3,fare=230)  # OFFLINE fixture, never production values.
        class FixtureWeather:
            ttl=900
            def forecast(self,*args,**kwargs):
                return dict(status='ok',condition='rain',points=[
                    dict(location='pickup',weather_code=61,precipitation_probability=70),
                    dict(location='destination',weather_code=61,precipitation_probability=60)])
        fake=FixtureBridge();state=dict(mode='book',stage='count',lang='Tamil')
        state,_=sandbox.prepare_turn(fake,FixtureWeather(),state,example,now=now)
        if (state.get('pickup'),state.get('destination'),state.get('count'),state.get('vehicle'),state.get('pickup_at'),state.get('stage'))!=\
                ('Chennai Central','Kottivakkam',4,'SUV','2026-10-18T16:00:00+05:30','weather_advisory'):
            raise RuntimeError('Exact-sentence extraction/flow check failed: '+json.dumps(state,ensure_ascii=False))
        checks+=1
        state['_wx_weather_consent']={'scope':sandbox.weather_consent_scope(state),'accepted_at':1}
        state,_=sandbox.prepare_turn(fake,FixtureWeather(),state,'',now=now)
        if state.get('stage')!='confirm':raise RuntimeError('Separate final-confirmation check failed.')
        checks+=1

    bootstrap="""
# MOVIO_TIME_ROUTE_BOOTSTRAP_V1
import importlib.util as _tr_import
import sys as _tr_sys
_tr_file=Path(__file__).with_name('movio_time_route.py')
_tr_spec=_tr_import.spec_from_file_location('movio_time_route',_tr_file)
_tr_module=_tr_import.module_from_spec(_tr_spec)
_tr_sys.modules[_tr_spec.name]=_tr_module
exec(compile(_tr_file.read_text(encoding='utf-8'),str(_tr_file),'exec'),_tr_module.__dict__)
_tr_module.install_namespace(globals())
# MOVIO_TIME_ROUTE_BOOTSTRAP_END_V1
"""
    new_guard=guard_base.rstrip()+'\n'+bootstrap
    compile(new_guard,str(guard_path),'exec')
    writes={helper_path:source,guard_path:new_guard}
    changed=any(not path.exists() or path.read_text(encoding='utf-8')!=content for path,content in writes.items())
    backup=None
    if changed:
        backup=root/('time_route_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,content in writes.items():
            if path.exists() and path.read_text(encoding='utf-8')==content:continue
            if path.exists():shutil.copy2(path,backup/path.name)
            temporary=path.with_name(path.name+'.timefix.tmp')
            try:temporary.write_text(content,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
    active=types.ModuleType('movio_time_route');active.__file__=str(helper_path)
    sys.modules[active.__name__]=active
    exec(compile(source,str(helper_path),'exec'),active.__dict__)
    status=active.install_namespace(engine,root,b,slots)
    globals()['movio_booking_diagnose']=lambda text,check_apis=False:active.diagnose(b,engine,text,check_apis=check_apis)
    status.update(offline_checks=checks,backup=str(backup) if backup else None)
    print('Time and location parsing patch enabled. Offline checks passed:',checks)
    print('Rule: today at four -> 4 PM when 4 AM is past and 4 PM is still future.')
    print('Kottivakkam is preserved; spoken clock words cannot replace a place.')
    print('Current local time:',datetime.now(engine['TZ']).strftime('%B %d, %Y, %I:%M %p'))
    print('Close the old AI Call popup; open a NEW call from the signed-in dashboard.')
    print('Live API check is separate; run the optional diagnostic below. It creates no booking.')
    print('Logs:',root/'booking_lookup.log','and',root/'weather_lookup.log')
    return status

MOVIO_TIME_ROUTE_STATUS=movio_enable_time_route_fix()

# OPTIONAL: in a separate cell, test the real configured APIs without booking:
# import json
# print(json.dumps(movio_booking_diagnose(
#     '4 passengers SUV from Chennai Central to Kottivakkam tomorrow at 4 PM',
#     check_apis=True), ensure_ascii=False, indent=2))


Time and location parsing patch enabled. Offline checks passed: 13
Rule: today at four -> 4 PM when 4 AM is past and 4 PM is still future.
Kottivakkam is preserved; spoken clock words cannot replace a place.
Current local time: October 28, 2026, 12:37 PM
Close the old AI Call popup; open a NEW call from the signed-in dashboard.
Live API check is separate; run the optional diagnostic below. It creates no booking.
Logs: /kaggle/working/MOVIO_MAIN_WEBSITE/booking_lookup.log and /kaggle/working/MOVIO_MAIN_WEBSITE/weather_lookup.log


In [91]:
# Run this ENTIRE file in ONE cell after the time/location fix, BEFORE evaluation.
# Your current models, AI Call UI, booking database and evaluation cells are kept.
def movio_require_spoken_weather():
    from pathlib import Path
    from datetime import datetime,timedelta
    import json
    import os
    import shutil
    import sys
    import tempfile
    import types

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if not isinstance(engine,dict) or not engine.get('_MOVIO_TIME_ROUTE_VERSION'):
        raise RuntimeError('Run the working time/location fix and original voice setup first.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    guard_path=root/'integrations/movio_weather_guard.py'
    helper_path=root/'integrations/movio_required_weather.py'
    if not guard_path.is_file():raise FileNotFoundError(str(guard_path))
    source=r'''"""Require an actual pickup/drop forecast before MOVIO voice confirmation.

Uses the existing Open-Meteo provider and the existing voice, route and database
implementations. Provider reference: https://open-meteo.com/en/docs
"""
import copy
import inspect
import json
import logging
import math
import threading
import time
from datetime import datetime
from pathlib import Path

VERSION='required-spoken-weather-1.0'
ROLES=('pickup','destination')


def logger_for(root):
    path=Path(root)/'required_weather.log'
    logger=logging.getLogger('movio.required_weather.'+str(path.resolve()))
    logger.setLevel(logging.INFO);logger.propagate=False
    if not logger.handlers:
        handler=logging.FileHandler(path,encoding='utf-8')
        handler.setFormatter(logging.Formatter('%(asctime)s %(message)s'));logger.addHandler(handler)
    return logger


def log(logger,event,**details):
    logger.info(json.dumps(dict(event=event,**details),ensure_ascii=False,default=str))


def number(value,low=None,high=None):
    if value is None or isinstance(value,bool):return None
    try:value=float(value)
    except (ValueError,TypeError,OverflowError):return None
    return value if math.isfinite(value) and (low is None or value>=low) and (high is None or value<=high) else None


def complete(weather):
    if not isinstance(weather,dict) or weather.get('status')!='ok':return False
    if weather.get('condition') not in ('rain','cloudy','clear','other'):return False
    points=weather.get('points')
    if not isinstance(points,list) or len(points)!=2:return False
    for role in ROLES:
        rows=[p for p in points if isinstance(p,dict) and p.get('location')==role]
        if len(rows)!=1:return False
        code=number(rows[0].get('weather_code'),0,99)
        if code is None or not code.is_integer() or number(rows[0].get('precipitation_probability'),0,100) is None:return False
    return True


class ProviderFailure(Exception):
    def __init__(self,reason,status=None):self.reason,self.status=reason,status;super().__init__(reason)


class Forecast:
    def __init__(self,base,engine,root):
        self.base,self.engine=base,engine;self.logger=logger_for(root)
        self.cache={};self.lock=threading.RLock();self.blocked_until=0;self.last_error=None
        self.requests=0
    def __getattr__(self,name):return getattr(self.base,name)
    def pending(self,reason,status=None):
        self.last_error=dict(reason=reason,http_status=status)
        log(self.logger,'weather_check_pending',**self.last_error)
        return dict(status='pending',source='Open-Meteo',reason=reason,http_status=status,policy=VERSION)
    def fetch(self,points,pickup):
        params=dict(latitude=','.join(str(float(p['lat'])) for p in points),
            longitude=','.join(str(float(p['lng'])) for p in points),
            hourly='weather_code,precipitation_probability,precipitation,temperature_2m,apparent_temperature',
            timezone='Asia/Kolkata',start_date=pickup.date().isoformat(),end_date=pickup.date().isoformat())
        getter=getattr(self.base,'getter',None)
        if not callable(getter):
            import requests
            getter=requests.get
        response=None
        try:
            self.requests+=1
            response=getter('https://api.open-meteo.com/v1/forecast',params=params,timeout=(3,6))
            response.raise_for_status();payload=response.json()
            if isinstance(payload,dict) and payload.get('error'):
                raise ProviderFailure('provider_rejected_query',getattr(response,'status_code',None))
            rows=payload if isinstance(payload,list) else [payload]
            if len(rows)!=len(points):raise ProviderFailure('invalid_location_count')
            target=pickup.replace(minute=0,second=0,microsecond=0).strftime('%Y-%m-%dT%H:%M')
            result=[]
            for row in rows:
                hourly=row['hourly'];index=hourly['time'].index(target)
                code=number(hourly['weather_code'][index],0,99)
                chance=number(hourly['precipitation_probability'][index],0,100)
                if code is None or not code.is_integer() or chance is None:
                    raise ProviderFailure('missing_hourly_condition_or_probability')
                point=dict(valid_at=target,weather_code=int(code),precipitation_probability=chance)
                for key in ('precipitation','temperature_2m','apparent_temperature'):
                    values=hourly.get(key);point[key]=number(values[index]) if isinstance(values,list) and index<len(values) else None
                result.append(point)
            return result
        except ProviderFailure:raise
        except Exception as exc:
            status=getattr(getattr(exc,'response',None),'status_code',None)
            if status is None:status=getattr(response,'status_code',None)
            reason='http_error' if isinstance(status,int) and status>=400 else 'timeout' if 'timeout' in type(exc).__name__.lower() else 'invalid_hourly_response' if isinstance(exc,(ValueError,KeyError,IndexError,TypeError)) else 'connection_error'
            log(self.logger,'weather_request_failed',reason=reason,http_status=status,error_type=type(exc).__name__,locations=len(points))
            if status==429:
                try:retry=float(response.headers.get('Retry-After','30'))
                except (ValueError,TypeError,AttributeError):retry=30
                self.blocked_until=time.monotonic()+max(1,min(3600,retry))
            raise ProviderFailure(reason,status) from exc
    def forecast(self,points,pickup,now=None):
        zone=self.engine['TZ'];now=now or datetime.now(zone)
        pickup=datetime.fromisoformat(pickup) if isinstance(pickup,str) else pickup
        pickup=pickup.replace(tzinfo=zone) if pickup.tzinfo is None else pickup.astimezone(zone)
        now=now.replace(tzinfo=zone) if now.tzinfo is None else now.astimezone(zone)
        if not 0<=(pickup.date()-now.date()).days<16:return self.pending('outside_forecast_window')
        if len(points)!=2 or not all(self.engine['valid_geo'](point) for point in points):return self.pending('invalid_location_coordinates')
        target=pickup.replace(minute=0,second=0,microsecond=0).isoformat()
        key=(tuple((round(float(p['lat']),5),round(float(p['lng']),5)) for p in points),target)
        with self.lock:
            cached=self.cache.get(key);ttl=max(1,min(900,float(getattr(self.base,'ttl',900))))
            if cached and time.monotonic()-cached[0]<ttl:return copy.deepcopy(cached[1])
            if time.monotonic()<self.blocked_until:return self.pending('rate_limit_backoff',429)
            try:rows=self.fetch(points,pickup)
            except ProviderFailure as exc:
                # Explicit denial/rate limits are not bypassed with more calls.
                if exc.status in (401,403,429):return self.pending(exc.reason,exc.status)
                log(self.logger,'retry_weather_as_individual_locations',reason=exc.reason,http_status=exc.status)
                try:rows=[self.fetch([point],pickup)[0] for point in points]
                except ProviderFailure as last:return self.pending(last.reason,last.status)
            for role,row in zip(ROLES,rows):row['location']=role
            rain=any(row['weather_code'] in self.engine['RAIN_CODES'] or row['precipitation_probability']>=getattr(self.base,'rain_threshold',60) for row in rows)
            cloudy=any(row['weather_code'] in (2,3) for row in rows);clear=all(row['weather_code'] in (0,1) for row in rows)
            heat=any(row['temperature_2m'] is not None and row['temperature_2m']>=getattr(self.base,'heat_c',38)
                or row['apparent_temperature'] is not None and row['apparent_temperature']>=getattr(self.base,'feels_like_c',40) for row in rows)
            result=dict(status='ok',source='Open-Meteo',points=rows,fetched_at=time.time(),policy=VERSION,
                condition='rain' if rain else 'cloudy' if cloudy else 'clear' if clear else 'other',
                heavy_rain=any(row['weather_code'] in (65,67,82) for row in rows),
                thunderstorm=any(row['weather_code'] in (95,96,97,99) for row in rows),heat=heat,
                pleasant=clear and not heat and all(row['temperature_2m'] is not None and 18<=row['temperature_2m']<=32 for row in rows),
                scope='pickup_and_destination_at_pickup_hour')
            self.cache[key]=(time.monotonic(),copy.deepcopy(result));self.last_error=None
            if len(self.cache)>256:self.cache.pop(next(iter(self.cache)))
            log(self.logger,'weather_forecast_ready',valid_at=target,probabilities=[row['precipitation_probability'] for row in rows])
            return result


def speech_registrar(engine):
    # The notebook's weather/closing quality cell stores its existing register
    # function in the completion wrapper. Reuse that exact speech-quality path.
    function=engine.get('completion_message')
    try:register=inspect.getclosurevars(function).nonlocals.get('register')
    except (TypeError,ValueError):return None
    return register if callable(register) else None


def waiting_text(state,engine):
    reason=(state.get('weather') or {}).get('reason')
    if reason=='outside_forecast_window':
        return engine['localized'](state['lang'],
            'Your selected date is kept. The weather check must be repeated closer to your journey.',
            'நீங்கள் தேர்ந்தெடுத்த தேதி உள்ளது. பயண நாளுக்கு அருகில் வானிலையை மீண்டும் சரிபார்க்க வேண்டும்.',
            'Neenga sonna date irukku. Journey date nerungumbodhu weather check thirumba pannanum.')
    return engine['localized'](state['lang'],
        'Please say “check weather again” to retry the weather check. Your journey details are kept.',
        'வானிலையை மீண்டும் சரிபார்க்க, “மீண்டும் சரிபாருங்கள்” என்று சொல்லுங்கள். உங்கள் பயண விவரங்கள் உள்ளன.',
        'Weather thirumba check panna, “check weather again” sollunga. Unga journey details irukku.')


def install_namespace(engine,root=None,bridge=None):
    root=Path(root or Path(engine['__file__']).resolve().parent.parent)
    if engine.get('_MOVIO_REQUIRED_WEATHER_VERSION')==VERSION:
        return dict(version=VERSION,already_installed=True,weather_mandatory=True,
            stable_weather_audio_registry=callable(speech_registrar(engine)),log=str(root/'required_weather.log'))
    previous=engine['prepare_turn'];previous_consent=engine['weather_consented'];previous_advisory=engine['advisory']
    previous_install=engine['install_bridge'];previous_segments=engine['speech_segments']
    optional=engine.get('_movio_optional_weather_original') or {}
    base_prepare=optional.get('prepare_turn');base_consent=optional.get('weather_consented')
    if not callable(base_prepare) or not callable(base_consent):raise RuntimeError('Run the existing optional-weather fix before this cell.')
    # Keep the compact existing language/vehicle advisory; only successful,
    # complete actual forecasts may enter the weather-consent stage.
    logger=logger_for(root);spoken=[];lock=threading.RLock();register=speech_registrar(engine)
    def advisory(state):
        if not complete(state.get('weather')):return ''
        text=previous_advisory(state)
        if callable(register):text=register(text,'weather')
        with lock:
            spoken.append(text)
            if len(spoken)>128:del spoken[:-128]
        return text
    def speech_segments(text):
        with lock:is_weather=any(prefix and (text==prefix or str(text).startswith(prefix+'\n') or str(text).startswith(prefix+' ')) for prefix in spoken)
        return [str(text)] if is_weather else previous_segments(text)
    def weather_consented(state):
        if state.get('_wx_required_weather')==VERSION:
            return complete(state.get('weather')) and base_consent(state)
        return previous_consent(state)
    def proxy(b,weather):
        current=getattr(b,'_wx_required_forecast',None)
        if not isinstance(current,Forecast) or current.base is not weather:
            current=Forecast(weather,engine,root);b._wx_required_forecast=current
        return current
    def prepare_turn(b,weather,state,text,now=None):
        state['_wx_required_weather']=VERSION
        if isinstance(state.get('weather'),dict):state['weather'].pop('_optional_skip',None)
        is_yes=engine['reply_normalized'](text) in engine['POSITIVE']
        frozen=state.get('stage') in ('confirm','fare','booking_pending') and state.get('dialogue_state') in ('AWAITING_BOOKING_CONFIRMATION','BOOKING_IN_PROGRESS')
        # A final Yes must continue the same reviewed journey, without redoing
        # weather/route lookup or restarting the separate consent flow.
        if frozen and (not str(text).strip() or is_yes) and weather_consented(state):
            return previous(b,weather,state,text,now=now)
        if not complete(state.get('weather')):state.pop('_wx_forecast_scope',None)
        normalizer=engine.get('_movio_weather_normalize')
        if callable(normalizer):
            for role in ROLES:
                if state.get(role):
                    name=normalizer(state[role],role)
                    if name and name!=state[role]:state[role]=name;state[role+'_geo']=None;state['fare']=None
        state,answer=base_prepare(b,proxy(b,weather),state,text,now=now)
        if state.get('stage')=='weather_advisory' and not complete(state.get('weather')):
            state['stage']='weather_pending';state['dialogue_state']='WEATHER_LOOKUP_PENDING'
            state.pop('_wx_weather_consent',None)
            log(logger,'confirmation_waiting_for_actual_weather',reason=(state.get('weather') or {}).get('reason'))
            return state,waiting_text(state,engine)
        state['dialogue_state']='WEATHER_ADVISORY' if state.get('stage')=='weather_advisory' else 'AWAITING_BOOKING_CONFIRMATION' if state.get('stage')=='confirm' else 'COLLECTING_DETAILS'
        return state,answer
    def install_bridge(b,db_path,weather=None,_route_wrapper=None):
        return previous_install(b,db_path,weather=weather,_route_wrapper=_route_wrapper)
    engine.update(prepare_turn=prepare_turn,weather_consented=weather_consented,advisory=advisory,speech_segments=speech_segments,
        install_bridge=install_bridge,_MOVIO_REQUIRED_WEATHER_VERSION=VERSION,
        _movio_required_weather_original=dict(prepare_turn=previous,weather_consented=previous_consent,advisory=previous_advisory,speech_segments=previous_segments),
        _movio_required_weather_proxy=proxy)
    return dict(version=VERSION,weather_mandatory=True,stable_weather_audio_registry=callable(register),log=str(root/'required_weather.log'))
'''
    candidate=types.ModuleType('movio_required_weather_candidate');candidate.__file__=str(helper_path)
    exec(compile(source,str(helper_path),'exec'),candidate.__dict__)
    marker='# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_V1';end_marker='# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_END_V1'
    guard_base=guard_path.read_text(encoding='utf-8')
    if marker in guard_base:
        start=guard_base.index(marker);end=guard_base.find(end_marker,start)
        if end<0:raise RuntimeError('Incomplete previous weather bootstrap; no files changed.')
        guard_base=guard_base[:start]+guard_base[end+len(end_marker):]
    # Isolated fixtures check the flow, never genuine weather or user bookings.
    with tempfile.TemporaryDirectory(prefix='movio_required_weather_checks_') as temporary:
        sandbox=types.ModuleType('movio_required_weather_selfcheck');sandbox.__file__=str(guard_path)
        exec(compile(guard_base,str(guard_path),'exec'),sandbox.__dict__)
        candidate.install_namespace(sandbox.__dict__,temporary)
        class Reply:
            def __init__(self,payload):self.payload=payload
            def raise_for_status(self):pass
            def json(self):return self.payload
        class Provider:
            ttl=900;rain_threshold=60;heat_c=38;feels_like_c=40
            def __init__(self,fail=False):self.fail=fail
            def getter(self,url,params,timeout):
                if self.fail:raise TimeoutError('offline fixture')
                day=params['start_date'];rows=[]
                for i,lat in enumerate(params['latitude'].split(',')):
                    rows.append(dict(hourly=dict(time=[f'{day}T{h:02}:00' for h in range(24)],
                        weather_code=[61]*24,precipitation_probability=[70 if i==0 else 60]*24,
                        temperature_2m=[28]*24,apparent_temperature=[30]*24,precipitation=[1]*24)))
                return Reply(rows if len(rows)>1 else rows[0])
        class FixtureBridge:
            VOICE_PLACE_ALIASES=getattr(b,'VOICE_PLACE_ALIASES',{})
            canonical_place=staticmethod(getattr(b,'canonical_place',lambda name:name))
            def __init__(self):self.routes=0
            def say(self,state,key,**kwargs):return 'ASK_'+key
            def geocode(self,name):return dict(lat=13.08 if 'Central' in name else 12.97,lng=80.26)
            def road_fare(self,state):self.routes+=1;return dict(km=12.3,fare=230)
        checks=0;now=datetime(2026,10,18,14,tzinfo=sandbox.TZ)
        text='4 passengers SUV from Chennai Central to Kottivakkam today at four'
        for lang in ('English','Tamil','Tanglish'):
            for failed in (False,True):
                fake=FixtureBridge();state=dict(stage='count',mode='book',lang=lang)
                provider=Provider(failed)
                state,answer=sandbox.prepare_turn(fake,provider,state,text,now=now)
                if failed:
                    if state['stage']!='weather_pending' or fake.routes or sandbox.weather_consented(state):
                        raise RuntimeError('Weather failure must retain details and wait for retry, not confirm.')
                    if any(word in answer.casefold() for word in ('unavailable','கிடைக்கவில்லை','skip')):
                        raise RuntimeError('Unexpected unavailable/skip customer message.')
                    if state['destination']!='Kottivakkam':raise RuntimeError('Journey details were lost.')
                else:
                    if state['stage']!='weather_advisory' or '70%' not in answer or '60%' not in answer or fake.routes:
                        raise RuntimeError('Both actual-location probabilities must precede confirmation.')
                    if len(sandbox.speech_segments(answer))!=1:raise RuntimeError('Weather speech grouping failed.')
                    state['_wx_weather_consent']={'scope':sandbox.weather_consent_scope(state),'accepted_at':1}
                    state,_=sandbox.prepare_turn(fake,provider,state,'',now=now)
                    if state['stage']!='confirm' or fake.routes!=1:raise RuntimeError('Separate final confirmation failed.')
                checks+=1
    bootstrap="""
# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_V1
import importlib.util as _rw_import
import sys as _rw_sys
_rw_file=Path(__file__).with_name('movio_required_weather.py')
_rw_spec=_rw_import.spec_from_file_location('movio_required_weather',_rw_file)
_rw_module=_rw_import.module_from_spec(_rw_spec)
_rw_sys.modules[_rw_spec.name]=_rw_module
exec(compile(_rw_file.read_text(encoding='utf-8'),str(_rw_file),'exec'),_rw_module.__dict__)
_rw_module.install_namespace(globals())
# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_END_V1
"""
    patched_guard=guard_base.rstrip()+'\n'+bootstrap
    compile(patched_guard,str(guard_path),'exec')
    writes={helper_path:source,guard_path:patched_guard}
    changed=any(not path.exists() or path.read_text(encoding='utf-8')!=value for path,value in writes.items())
    backup=None
    if changed:
        backup=root/('spoken_weather_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,value in writes.items():
            if path.exists() and path.read_text(encoding='utf-8')==value:continue
            if path.exists():shutil.copy2(path,backup/path.name)
            temporary=path.with_name(path.name+'.required_weather.tmp')
            try:temporary.write_text(value,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
    active=types.ModuleType('movio_required_weather');active.__file__=str(helper_path)
    sys.modules[active.__name__]=active
    exec(compile(source,str(helper_path),'exec'),active.__dict__)
    status=active.install_namespace(engine,root,b)
    def check_live_weather(pickup='Chennai Central',destination='Kottivakkam',pickup_at=None):
        zone=engine['TZ'];now=datetime.now(zone)
        when=datetime.fromisoformat(pickup_at) if isinstance(pickup_at,str) else pickup_at or (now+timedelta(days=1)).replace(hour=9,minute=0,second=0,microsecond=0)
        points=[b.geocode(name) for name in (pickup,destination)]
        if not all(engine['valid_geo'](point) for point in points):
            return dict(ready=False,reason='geocoding_failed',log=str(root/'booking_lookup.log'))
        forecast=engine['_movio_required_weather_proxy'](b,b._wx_weather).forecast(points,when,now=now)
        return dict(ready=active.complete(forecast),pickup=pickup,destination=destination,
            requested_time=when.isoformat(),forecast=forecast)
    globals()['movio_check_live_weather']=check_live_weather
    status.update(offline_checks=checks,backup=str(backup) if backup else None)
    print('Spoken weather is now required before voice-booking confirmation.')
    print('Both pickup/drop rain probabilities come from Open-Meteo. No invented percentages.')
    print('Weather consent and final booking confirmation remain separate.')
    print('Offline flow checks passed:',checks)
    print('Existing weather speech-quality registry restored:',status.get('stable_weather_audio_registry',False))
    if globals().get('WX_VERIFY_LIVE_WEATHER',True):
        print('Checking your LIVE forecast connection (no booking/audio is created)...',flush=True)
        result=check_live_weather();status['live_probe']=result
        print(json.dumps(result,ensure_ascii=False,indent=2))
        if not result['ready']:
            print('Live probe did not pass. The call will retain the journey and wait for a weather retry before confirmation.')
            print('Developer diagnostic:',root/'required_weather.log')
    print('Close the old popup and start a NEW AI Call from your signed-in dashboard.')
    return status

MOVIO_REQUIRED_WEATHER_STATUS=movio_require_spoken_weather()

# Optional repeat of the real forecast check, without booking:
# print(json.dumps(movio_check_live_weather(),ensure_ascii=False,indent=2))


Spoken weather is now required before voice-booking confirmation.
Both pickup/drop rain probabilities come from Open-Meteo. No invented percentages.
Weather consent and final booking confirmation remain separate.
Offline flow checks passed: 6
Existing weather speech-quality registry restored: True
Checking your LIVE forecast connection (no booking/audio is created)...
{
  "ready": true,
  "pickup": "Chennai Central",
  "destination": "Kottivakkam",
  "requested_time": "2026-10-29T09:00:00+05:30",
  "forecast": {
    "status": "ok",
    "source": "Open-Meteo",
    "points": [
      {
        "valid_at": "2026-10-29T09:00",
        "weather_code": 1,
        "precipitation_probability": 0.0,
        "precipitation": 0.0,
        "temperature_2m": 28.9,
        "apparent_temperature": 32.1,
        "location": "pickup"
      },
      {
        "valid_at": "2026-10-29T09:00",
        "weather_code": 1,
        "precipitation_probability": 0.0,
        "precipitation": 0.0,
        "tempera

In [92]:
# Run this ENTIRE file in ONE cell after the time/location fix, BEFORE evaluation.
# Your current models, AI Call UI, booking database and evaluation cells are kept.
def movio_require_spoken_weather():
    from pathlib import Path
    from datetime import datetime,timedelta
    import json
    import os
    import shutil
    import sys
    import tempfile
    import types

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if not isinstance(engine,dict) or not engine.get('_MOVIO_TIME_ROUTE_VERSION'):
        raise RuntimeError('Run the working time/location fix and original voice setup first.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    guard_path=root/'integrations/movio_weather_guard.py'
    helper_path=root/'integrations/movio_required_weather.py'
    if not guard_path.is_file():raise FileNotFoundError(str(guard_path))
    source=r'''"""Require an actual pickup/drop forecast before MOVIO voice confirmation.

Uses the existing Open-Meteo provider and the existing voice, route and database
implementations. Provider reference: https://open-meteo.com/en/docs
"""
import copy
import inspect
import json
import logging
import math
import threading
import time
from datetime import datetime
from pathlib import Path

VERSION='required-spoken-weather-1.0'
ROLES=('pickup','destination')


def logger_for(root):
    path=Path(root)/'required_weather.log'
    logger=logging.getLogger('movio.required_weather.'+str(path.resolve()))
    logger.setLevel(logging.INFO);logger.propagate=False
    if not logger.handlers:
        handler=logging.FileHandler(path,encoding='utf-8')
        handler.setFormatter(logging.Formatter('%(asctime)s %(message)s'));logger.addHandler(handler)
    return logger


def log(logger,event,**details):
    logger.info(json.dumps(dict(event=event,**details),ensure_ascii=False,default=str))


def number(value,low=None,high=None):
    if value is None or isinstance(value,bool):return None
    try:value=float(value)
    except (ValueError,TypeError,OverflowError):return None
    return value if math.isfinite(value) and (low is None or value>=low) and (high is None or value<=high) else None


def complete(weather):
    if not isinstance(weather,dict) or weather.get('status')!='ok':return False
    if weather.get('condition') not in ('rain','cloudy','clear','other'):return False
    points=weather.get('points')
    if not isinstance(points,list) or len(points)!=2:return False
    for role in ROLES:
        rows=[p for p in points if isinstance(p,dict) and p.get('location')==role]
        if len(rows)!=1:return False
        code=number(rows[0].get('weather_code'),0,99)
        if code is None or not code.is_integer() or number(rows[0].get('precipitation_probability'),0,100) is None:return False
    return True


class ProviderFailure(Exception):
    def __init__(self,reason,status=None):self.reason,self.status=reason,status;super().__init__(reason)


class Forecast:
    def __init__(self,base,engine,root):
        self.base,self.engine=base,engine;self.logger=logger_for(root)
        self.cache={};self.lock=threading.RLock();self.blocked_until=0;self.last_error=None
        self.requests=0
    def __getattr__(self,name):return getattr(self.base,name)
    def pending(self,reason,status=None):
        self.last_error=dict(reason=reason,http_status=status)
        log(self.logger,'weather_check_pending',**self.last_error)
        return dict(status='pending',source='Open-Meteo',reason=reason,http_status=status,policy=VERSION)
    def fetch(self,points,pickup):
        params=dict(latitude=','.join(str(float(p['lat'])) for p in points),
            longitude=','.join(str(float(p['lng'])) for p in points),
            hourly='weather_code,precipitation_probability,precipitation,temperature_2m,apparent_temperature',
            timezone='Asia/Kolkata',start_date=pickup.date().isoformat(),end_date=pickup.date().isoformat())
        getter=getattr(self.base,'getter',None)
        if not callable(getter):
            import requests
            getter=requests.get
        response=None
        try:
            self.requests+=1
            response=getter('https://api.open-meteo.com/v1/forecast',params=params,timeout=(3,6))
            response.raise_for_status();payload=response.json()
            if isinstance(payload,dict) and payload.get('error'):
                raise ProviderFailure('provider_rejected_query',getattr(response,'status_code',None))
            rows=payload if isinstance(payload,list) else [payload]
            if len(rows)!=len(points):raise ProviderFailure('invalid_location_count')
            target=pickup.replace(minute=0,second=0,microsecond=0).strftime('%Y-%m-%dT%H:%M')
            result=[]
            for row in rows:
                hourly=row['hourly'];index=hourly['time'].index(target)
                code=number(hourly['weather_code'][index],0,99)
                chance=number(hourly['precipitation_probability'][index],0,100)
                if code is None or not code.is_integer() or chance is None:
                    raise ProviderFailure('missing_hourly_condition_or_probability')
                point=dict(valid_at=target,weather_code=int(code),precipitation_probability=chance)
                for key in ('precipitation','temperature_2m','apparent_temperature'):
                    values=hourly.get(key);point[key]=number(values[index]) if isinstance(values,list) and index<len(values) else None
                result.append(point)
            return result
        except ProviderFailure:raise
        except Exception as exc:
            status=getattr(getattr(exc,'response',None),'status_code',None)
            if status is None:status=getattr(response,'status_code',None)
            reason='http_error' if isinstance(status,int) and status>=400 else 'timeout' if 'timeout' in type(exc).__name__.lower() else 'invalid_hourly_response' if isinstance(exc,(ValueError,KeyError,IndexError,TypeError)) else 'connection_error'
            log(self.logger,'weather_request_failed',reason=reason,http_status=status,error_type=type(exc).__name__,locations=len(points))
            if status==429:
                try:retry=float(response.headers.get('Retry-After','30'))
                except (ValueError,TypeError,AttributeError):retry=30
                self.blocked_until=time.monotonic()+max(1,min(3600,retry))
            raise ProviderFailure(reason,status) from exc
    def forecast(self,points,pickup,now=None):
        zone=self.engine['TZ'];now=now or datetime.now(zone)
        pickup=datetime.fromisoformat(pickup) if isinstance(pickup,str) else pickup
        pickup=pickup.replace(tzinfo=zone) if pickup.tzinfo is None else pickup.astimezone(zone)
        now=now.replace(tzinfo=zone) if now.tzinfo is None else now.astimezone(zone)
        if not 0<=(pickup.date()-now.date()).days<16:return self.pending('outside_forecast_window')
        if len(points)!=2 or not all(self.engine['valid_geo'](point) for point in points):return self.pending('invalid_location_coordinates')
        target=pickup.replace(minute=0,second=0,microsecond=0).isoformat()
        key=(tuple((round(float(p['lat']),5),round(float(p['lng']),5)) for p in points),target)
        with self.lock:
            cached=self.cache.get(key);ttl=max(1,min(900,float(getattr(self.base,'ttl',900))))
            if cached and time.monotonic()-cached[0]<ttl:return copy.deepcopy(cached[1])
            if time.monotonic()<self.blocked_until:return self.pending('rate_limit_backoff',429)
            try:rows=self.fetch(points,pickup)
            except ProviderFailure as exc:
                # Explicit denial/rate limits are not bypassed with more calls.
                if exc.status in (401,403,429):return self.pending(exc.reason,exc.status)
                log(self.logger,'retry_weather_as_individual_locations',reason=exc.reason,http_status=exc.status)
                try:rows=[self.fetch([point],pickup)[0] for point in points]
                except ProviderFailure as last:return self.pending(last.reason,last.status)
            for role,row in zip(ROLES,rows):row['location']=role
            rain=any(row['weather_code'] in self.engine['RAIN_CODES'] or row['precipitation_probability']>=getattr(self.base,'rain_threshold',60) for row in rows)
            cloudy=any(row['weather_code'] in (2,3) for row in rows);clear=all(row['weather_code'] in (0,1) for row in rows)
            heat=any(row['temperature_2m'] is not None and row['temperature_2m']>=getattr(self.base,'heat_c',38)
                or row['apparent_temperature'] is not None and row['apparent_temperature']>=getattr(self.base,'feels_like_c',40) for row in rows)
            result=dict(status='ok',source='Open-Meteo',points=rows,fetched_at=time.time(),policy=VERSION,
                condition='rain' if rain else 'cloudy' if cloudy else 'clear' if clear else 'other',
                heavy_rain=any(row['weather_code'] in (65,67,82) for row in rows),
                thunderstorm=any(row['weather_code'] in (95,96,97,99) for row in rows),heat=heat,
                pleasant=clear and not heat and all(row['temperature_2m'] is not None and 18<=row['temperature_2m']<=32 for row in rows),
                scope='pickup_and_destination_at_pickup_hour')
            self.cache[key]=(time.monotonic(),copy.deepcopy(result));self.last_error=None
            if len(self.cache)>256:self.cache.pop(next(iter(self.cache)))
            log(self.logger,'weather_forecast_ready',valid_at=target,probabilities=[row['precipitation_probability'] for row in rows])
            return result


def speech_registrar(engine):
    # The notebook's weather/closing quality cell stores its existing register
    # function in the completion wrapper. Reuse that exact speech-quality path.
    function=engine.get('completion_message')
    try:register=inspect.getclosurevars(function).nonlocals.get('register')
    except (TypeError,ValueError):return None
    return register if callable(register) else None


def waiting_text(state,engine):
    reason=(state.get('weather') or {}).get('reason')
    if reason=='outside_forecast_window':
        return engine['localized'](state['lang'],
            'Your selected date is kept. The weather check must be repeated closer to your journey.',
            'நீங்கள் தேர்ந்தெடுத்த தேதி உள்ளது. பயண நாளுக்கு அருகில் வானிலையை மீண்டும் சரிபார்க்க வேண்டும்.',
            'Neenga sonna date irukku. Journey date nerungumbodhu weather check thirumba pannanum.')
    return engine['localized'](state['lang'],
        'Please say “check weather again” to retry the weather check. Your journey details are kept.',
        'வானிலையை மீண்டும் சரிபார்க்க, “மீண்டும் சரிபாருங்கள்” என்று சொல்லுங்கள். உங்கள் பயண விவரங்கள் உள்ளன.',
        'Weather thirumba check panna, “check weather again” sollunga. Unga journey details irukku.')


def install_namespace(engine,root=None,bridge=None):
    root=Path(root or Path(engine['__file__']).resolve().parent.parent)
    if engine.get('_MOVIO_REQUIRED_WEATHER_VERSION')==VERSION:
        return dict(version=VERSION,already_installed=True,weather_mandatory=True,
            stable_weather_audio_registry=callable(speech_registrar(engine)),log=str(root/'required_weather.log'))
    previous=engine['prepare_turn'];previous_consent=engine['weather_consented'];previous_advisory=engine['advisory']
    previous_install=engine['install_bridge'];previous_segments=engine['speech_segments']
    optional=engine.get('_movio_optional_weather_original') or {}
    base_prepare=optional.get('prepare_turn');base_consent=optional.get('weather_consented')
    if not callable(base_prepare) or not callable(base_consent):raise RuntimeError('Run the existing optional-weather fix before this cell.')
    # Keep the compact existing language/vehicle advisory; only successful,
    # complete actual forecasts may enter the weather-consent stage.
    logger=logger_for(root);spoken=[];lock=threading.RLock();register=speech_registrar(engine)
    def advisory(state):
        if not complete(state.get('weather')):return ''
        text=previous_advisory(state)
        if callable(register):text=register(text,'weather')
        with lock:
            spoken.append(text)
            if len(spoken)>128:del spoken[:-128]
        return text
    def speech_segments(text):
        with lock:is_weather=any(prefix and (text==prefix or str(text).startswith(prefix+'\n') or str(text).startswith(prefix+' ')) for prefix in spoken)
        return [str(text)] if is_weather else previous_segments(text)
    def weather_consented(state):
        if state.get('_wx_required_weather')==VERSION:
            return complete(state.get('weather')) and base_consent(state)
        return previous_consent(state)
    def proxy(b,weather):
        current=getattr(b,'_wx_required_forecast',None)
        if not isinstance(current,Forecast) or current.base is not weather:
            current=Forecast(weather,engine,root);b._wx_required_forecast=current
        return current
    def prepare_turn(b,weather,state,text,now=None):
        state['_wx_required_weather']=VERSION
        if isinstance(state.get('weather'),dict):state['weather'].pop('_optional_skip',None)
        is_yes=engine['reply_normalized'](text) in engine['POSITIVE']
        frozen=state.get('stage') in ('confirm','fare','booking_pending') and state.get('dialogue_state') in ('AWAITING_BOOKING_CONFIRMATION','BOOKING_IN_PROGRESS')
        # A final Yes must continue the same reviewed journey, without redoing
        # weather/route lookup or restarting the separate consent flow.
        if frozen and (not str(text).strip() or is_yes) and weather_consented(state):
            return previous(b,weather,state,text,now=now)
        if not complete(state.get('weather')):state.pop('_wx_forecast_scope',None)
        normalizer=engine.get('_movio_weather_normalize')
        if callable(normalizer):
            for role in ROLES:
                if state.get(role):
                    name=normalizer(state[role],role)
                    if name and name!=state[role]:state[role]=name;state[role+'_geo']=None;state['fare']=None
        state,answer=base_prepare(b,proxy(b,weather),state,text,now=now)
        if state.get('stage')=='weather_advisory' and not complete(state.get('weather')):
            state['stage']='weather_pending';state['dialogue_state']='WEATHER_LOOKUP_PENDING'
            state.pop('_wx_weather_consent',None)
            log(logger,'confirmation_waiting_for_actual_weather',reason=(state.get('weather') or {}).get('reason'))
            return state,waiting_text(state,engine)
        state['dialogue_state']='WEATHER_ADVISORY' if state.get('stage')=='weather_advisory' else 'AWAITING_BOOKING_CONFIRMATION' if state.get('stage')=='confirm' else 'COLLECTING_DETAILS'
        return state,answer
    def install_bridge(b,db_path,weather=None,_route_wrapper=None):
        return previous_install(b,db_path,weather=weather,_route_wrapper=_route_wrapper)
    engine.update(prepare_turn=prepare_turn,weather_consented=weather_consented,advisory=advisory,speech_segments=speech_segments,
        install_bridge=install_bridge,_MOVIO_REQUIRED_WEATHER_VERSION=VERSION,
        _movio_required_weather_original=dict(prepare_turn=previous,weather_consented=previous_consent,advisory=previous_advisory,speech_segments=previous_segments),
        _movio_required_weather_proxy=proxy)
    return dict(version=VERSION,weather_mandatory=True,stable_weather_audio_registry=callable(register),log=str(root/'required_weather.log'))
'''
    candidate=types.ModuleType('movio_required_weather_candidate');candidate.__file__=str(helper_path)
    exec(compile(source,str(helper_path),'exec'),candidate.__dict__)
    marker='# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_V1';end_marker='# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_END_V1'
    guard_base=guard_path.read_text(encoding='utf-8')
    if marker in guard_base:
        start=guard_base.index(marker);end=guard_base.find(end_marker,start)
        if end<0:raise RuntimeError('Incomplete previous weather bootstrap; no files changed.')
        guard_base=guard_base[:start]+guard_base[end+len(end_marker):]
    # Isolated fixtures check the flow, never genuine weather or user bookings.
    with tempfile.TemporaryDirectory(prefix='movio_required_weather_checks_') as temporary:
        sandbox=types.ModuleType('movio_required_weather_selfcheck');sandbox.__file__=str(guard_path)
        exec(compile(guard_base,str(guard_path),'exec'),sandbox.__dict__)
        candidate.install_namespace(sandbox.__dict__,temporary)
        class Reply:
            def __init__(self,payload):self.payload=payload
            def raise_for_status(self):pass
            def json(self):return self.payload
        class Provider:
            ttl=900;rain_threshold=60;heat_c=38;feels_like_c=40
            def __init__(self,fail=False):self.fail=fail
            def getter(self,url,params,timeout):
                if self.fail:raise TimeoutError('offline fixture')
                day=params['start_date'];rows=[]
                for i,lat in enumerate(params['latitude'].split(',')):
                    rows.append(dict(hourly=dict(time=[f'{day}T{h:02}:00' for h in range(24)],
                        weather_code=[61]*24,precipitation_probability=[70 if i==0 else 60]*24,
                        temperature_2m=[28]*24,apparent_temperature=[30]*24,precipitation=[1]*24)))
                return Reply(rows if len(rows)>1 else rows[0])
        class FixtureBridge:
            VOICE_PLACE_ALIASES=getattr(b,'VOICE_PLACE_ALIASES',{})
            canonical_place=staticmethod(getattr(b,'canonical_place',lambda name:name))
            def __init__(self):self.routes=0
            def say(self,state,key,**kwargs):return 'ASK_'+key
            def geocode(self,name):return dict(lat=13.08 if 'Central' in name else 12.97,lng=80.26)
            def road_fare(self,state):self.routes+=1;return dict(km=12.3,fare=230)
        checks=0;now=datetime(2026,10,18,14,tzinfo=sandbox.TZ)
        text='4 passengers SUV from Chennai Central to Kottivakkam today at four'
        for lang in ('English','Tamil','Tanglish'):
            for failed in (False,True):
                fake=FixtureBridge();state=dict(stage='count',mode='book',lang=lang)
                provider=Provider(failed)
                state,answer=sandbox.prepare_turn(fake,provider,state,text,now=now)
                if failed:
                    if state['stage']!='weather_pending' or fake.routes or sandbox.weather_consented(state):
                        raise RuntimeError('Weather failure must retain details and wait for retry, not confirm.')
                    if any(word in answer.casefold() for word in ('unavailable','கிடைக்கவில்லை','skip')):
                        raise RuntimeError('Unexpected unavailable/skip customer message.')
                    if state['destination']!='Kottivakkam':raise RuntimeError('Journey details were lost.')
                else:
                    if state['stage']!='weather_advisory' or '70%' not in answer or '60%' not in answer or fake.routes:
                        raise RuntimeError('Both actual-location probabilities must precede confirmation.')
                    if len(sandbox.speech_segments(answer))!=1:raise RuntimeError('Weather speech grouping failed.')
                    state['_wx_weather_consent']={'scope':sandbox.weather_consent_scope(state),'accepted_at':1}
                    state,_=sandbox.prepare_turn(fake,provider,state,'',now=now)
                    if state['stage']!='confirm' or fake.routes!=1:raise RuntimeError('Separate final confirmation failed.')
                checks+=1
    bootstrap="""
# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_V1
import importlib.util as _rw_import
import sys as _rw_sys
_rw_file=Path(__file__).with_name('movio_required_weather.py')
_rw_spec=_rw_import.spec_from_file_location('movio_required_weather',_rw_file)
_rw_module=_rw_import.module_from_spec(_rw_spec)
_rw_sys.modules[_rw_spec.name]=_rw_module
exec(compile(_rw_file.read_text(encoding='utf-8'),str(_rw_file),'exec'),_rw_module.__dict__)
_rw_module.install_namespace(globals())
# MOVIO_REQUIRED_WEATHER_BOOTSTRAP_END_V1
"""
    patched_guard=guard_base.rstrip()+'\n'+bootstrap
    compile(patched_guard,str(guard_path),'exec')
    writes={helper_path:source,guard_path:patched_guard}
    changed=any(not path.exists() or path.read_text(encoding='utf-8')!=value for path,value in writes.items())
    backup=None
    if changed:
        backup=root/('spoken_weather_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,value in writes.items():
            if path.exists() and path.read_text(encoding='utf-8')==value:continue
            if path.exists():shutil.copy2(path,backup/path.name)
            temporary=path.with_name(path.name+'.required_weather.tmp')
            try:temporary.write_text(value,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
    active=types.ModuleType('movio_required_weather');active.__file__=str(helper_path)
    sys.modules[active.__name__]=active
    exec(compile(source,str(helper_path),'exec'),active.__dict__)
    status=active.install_namespace(engine,root,b)
    def check_live_weather(pickup='Chennai Central',destination='Kottivakkam',pickup_at=None):
        zone=engine['TZ'];now=datetime.now(zone)
        when=datetime.fromisoformat(pickup_at) if isinstance(pickup_at,str) else pickup_at or (now+timedelta(days=1)).replace(hour=9,minute=0,second=0,microsecond=0)
        points=[b.geocode(name) for name in (pickup,destination)]
        if not all(engine['valid_geo'](point) for point in points):
            return dict(ready=False,reason='geocoding_failed',log=str(root/'booking_lookup.log'))
        forecast=engine['_movio_required_weather_proxy'](b,b._wx_weather).forecast(points,when,now=now)
        return dict(ready=active.complete(forecast),pickup=pickup,destination=destination,
            requested_time=when.isoformat(),forecast=forecast)
    globals()['movio_check_live_weather']=check_live_weather
    status.update(offline_checks=checks,backup=str(backup) if backup else None)
    print('Spoken weather is now required before voice-booking confirmation.')
    print('Both pickup/drop rain probabilities come from Open-Meteo. No invented percentages.')
    print('Weather consent and final booking confirmation remain separate.')
    print('Offline flow checks passed:',checks)
    print('Existing weather speech-quality registry restored:',status.get('stable_weather_audio_registry',False))
    if globals().get('WX_VERIFY_LIVE_WEATHER',True):
        print('Checking your LIVE forecast connection (no booking/audio is created)...',flush=True)
        result=check_live_weather();status['live_probe']=result
        print(json.dumps(result,ensure_ascii=False,indent=2))
        if not result['ready']:
            print('Live probe did not pass. The call will retain the journey and wait for a weather retry before confirmation.')
            print('Developer diagnostic:',root/'required_weather.log')
    print('Close the old popup and start a NEW AI Call from your signed-in dashboard.')
    return status

MOVIO_REQUIRED_WEATHER_STATUS=movio_require_spoken_weather()

# Optional repeat of the real forecast check, without booking:
# print(json.dumps(movio_check_live_weather(),ensure_ascii=False,indent=2))


Spoken weather is now required before voice-booking confirmation.
Both pickup/drop rain probabilities come from Open-Meteo. No invented percentages.
Weather consent and final booking confirmation remain separate.
Offline flow checks passed: 6
Existing weather speech-quality registry restored: True
Checking your LIVE forecast connection (no booking/audio is created)...
{
  "ready": true,
  "pickup": "Chennai Central",
  "destination": "Kottivakkam",
  "requested_time": "2026-10-29T09:00:00+05:30",
  "forecast": {
    "status": "ok",
    "source": "Open-Meteo",
    "points": [
      {
        "valid_at": "2026-10-29T09:00",
        "weather_code": 1,
        "precipitation_probability": 0.0,
        "precipitation": 0.0,
        "temperature_2m": 28.9,
        "apparent_temperature": 32.1,
        "location": "pickup"
      },
      {
        "valid_at": "2026-10-29T09:00",
        "weather_code": 1,
        "precipitation_probability": 0.0,
        "precipitation": 0.0,
        "tempera

In [93]:
# Paste this ENTIRE file into ONE Kaggle cell after the current dashboard fixes,
# BEFORE evaluation. It reloads only the website when necessary, keeping models.
# Installer v1.1 supports separate weather-setup and website-launch main blocks.
def movio_enable_customer_booking_tools():
    import ast
    from datetime import datetime
    import hashlib
    import os
    from pathlib import Path
    import re
    import shutil
    import subprocess
    import sys
    import tempfile
    import time
    import types
    import weakref
    import psutil
    import requests

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    web=root/'web';server_path=root/'server.py';index_path=web/'index.html';manual_path=web/'weather_manual.js'
    helper_path=root/'integrations/movio_customer_tools.py'
    for path in (server_path,index_path,manual_path):
        if not path.is_file():raise FileNotFoundError(str(path))
    backend_source=r'''"""Account-owned booking details and shared address/road lookups.

Primary API contracts: https://nominatim.org/release-docs/latest/api/Search/
https://operations.osmfoundation.org/policies/nominatim/
https://project-osrm.org/docs/v5.24.0/api/
"""
import copy
import json
import math
import os
import re
import threading
import time
from urllib.parse import parse_qs,urlparse

VERSION='customer-booking-tools-1.0'
_geo_lock=threading.RLock();_geo_last=0.0;_geo_cache={};_route_cache={};_route_lock=threading.RLock()

class LookupError(Exception):pass

def finite(value,low=None,high=None):
    if value is None or isinstance(value,bool):return None
    try:n=float(value)
    except (ValueError,TypeError,OverflowError):return None
    return n if math.isfinite(n) and (low is None or n>=low) and (high is None or n<=high) else None

def coords(lat,lng):
    a,b=finite(lat,-90,90),finite(lng,-180,180)
    if a is None or b is None:raise ValueError('Invalid location coordinates')
    return a,b

def place(row):
    a,b=coords(row['lat'],row.get('lon',row.get('lng')))
    full=str(row.get('display_name') or row.get('name') or '').strip()
    if not full:raise ValueError('Address is empty')
    # Long provider display strings used to exceed the booking API's 180-char
    # limit. Keep the nearby address components and retain the full tooltip.
    label=', '.join(full.split(',')[:3]).strip()
    if len(label)>170:label=label[:167].rsplit(' ',1)[0]+'…'
    return dict(name=label,full_address=full[:1000],lat=a,lng=b)

def geo_request(kind,params):
    global _geo_last
    import requests
    endpoint=os.environ.get('MOVIO_GEOCODER_URL','https://nominatim.openstreetmap.org/search').rstrip('/')
    url=endpoint if kind=='search' else endpoint.rsplit('/',1)[0]+'/reverse'
    key=(url,tuple(sorted(params.items())))
    with _geo_lock:
        saved=_geo_cache.get(key)
        if saved and time.monotonic()-saved[0]<86400:return copy.deepcopy(saved[1])
        delay=1.1-(time.monotonic()-_geo_last)
        if delay>0:time.sleep(delay)
        _geo_last=time.monotonic()
        try:
            response=requests.get(url,params=params,headers={'User-Agent':os.environ.get('MOVIO_GEOCODER_USER_AGENT','MOVIO-Ride-Booking-Research/5.3'),'Accept-Language':'en'},timeout=(3,8))
            response.raise_for_status();data=response.json()
        except (requests.RequestException,ValueError) as exc:raise LookupError('Address lookup could not complete. Please retry the search.') from exc
        if isinstance(data,dict) and data.get('error'):raise LookupError('Address not found for these coordinates.')
        _geo_cache[key]=(time.monotonic(),copy.deepcopy(data))
        if len(_geo_cache)>256:_geo_cache.pop(next(iter(_geo_cache)))
        return data

def address_search(query):
    query=str(query).strip()
    if not 3<=len(query)<=300:raise ValueError('Enter an address or landmark with 3–300 characters.')
    city=os.environ.get('MOVIO_SERVICE_CITY','Chennai').strip() or 'Chennai'
    box=os.environ.get('MOVIO_GEO_VIEWBOX','79.95,13.35,80.45,12.65')
    params=dict(q=query if city.casefold() in query.casefold() else query+', '+city,format='json',limit='6',countrycodes='in',addressdetails='1',viewbox=box,bounded='1')
    payload=geo_request('search',params)
    if not isinstance(payload,list):raise LookupError('Address lookup returned an invalid response. Please retry.')
    results=[]
    for row in payload:
        try:entry=place(row)
        except (KeyError,ValueError,TypeError):continue
        if not any(abs(entry['lat']-p['lat'])<1e-6 and abs(entry['lng']-p['lng'])<1e-6 and entry['name']==p['name'] for p in results):results.append(entry)
    return results

def reverse_address(lat,lng):
    lat,lng=coords(lat,lng)
    try:return place(geo_request('reverse',dict(lat=str(lat),lon=str(lng),format='json',zoom='18',addressdetails='1')))
    except (LookupError,KeyError,ValueError,TypeError):
        # A GPS fix remains usable for route/weather without reverse geocoding.
        return dict(name=f'Current pickup location ({lat:.5f}, {lng:.5f})',full_address='',lat=lat,lng=lng)

def road_route(a,b):
    import requests
    a=coords(*a);b=coords(*b);key=tuple(round(v,6) for v in (*a,*b))
    with _route_lock:
        saved=_route_cache.get(key)
        if saved and time.monotonic()-saved[0]<900:return copy.deepcopy(saved[1])
    base=os.environ.get('MOVIO_OSRM_BASE_URL','https://router.project-osrm.org').rstrip('/')
    url=f'{base}/route/v1/driving/{a[1]},{a[0]};{b[1]},{b[0]}'
    try:
        response=requests.get(url,params=dict(overview='full',geometries='geojson'),timeout=(3,10))
        response.raise_for_status();payload=response.json();rows=payload.get('routes')
        if payload.get('code')!='Ok' or not rows:raise ValueError('no_route')
        row=rows[0];distance=finite(row.get('distance'),0);duration=finite(row.get('duration'),0)
        if distance is None or not 0<distance/1000<=400:raise ValueError('invalid_road_distance')
        geometry=(row.get('geometry') or {}).get('coordinates',[])
        if not isinstance(geometry,list):geometry=[]
        geometry=[list(coords(p[1],p[0])) for p in geometry if isinstance(p,(list,tuple)) and len(p)>=2]
        result=dict(km=distance/1000,minutes=max(1,round(duration/60)) if duration is not None else None,geometry=geometry)
    except (requests.RequestException,ValueError,TypeError,KeyError,IndexError) as exc:
        raise LookupError('The road route could not be calculated. Your selected addresses are kept; please retry.') from exc
    with _route_lock:
        _route_cache[key]=(time.monotonic(),copy.deepcopy(result))
        if len(_route_cache)>128:_route_cache.pop(next(iter(_route_cache)))
    return result

def parsed(value):
    try:value=json.loads(value) if isinstance(value,str) else value
    except (ValueError,TypeError):return {}
    return value if isinstance(value,dict) else {}

def booking_details(c,owner,rid):
    row=c.execute('SELECT r.*,d.name driver_name,d.phone driver_phone,p.vehicle_number driver_vehicle_number FROM rides r LEFT JOIN users d ON d.id=r.driver_id LEFT JOIN driver_profiles p ON p.user_id=r.driver_id WHERE r.id=? AND r.customer_id=?',(rid,owner)).fetchone()
    if row is None:return None
    record=dict(row);tables={r[0] for r in c.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    snapshot={}
    if {'wx_calls','wx_permits'}<=tables:
        permit=c.execute('SELECT p.snapshot FROM wx_permits p JOIN wx_calls w ON w.id=p.call_id WHERE p.ride_id=? AND w.owner=? ORDER BY p.expires DESC LIMIT 1',(rid,owner)).fetchone()
        if permit:snapshot=parsed(permit['snapshot'])
    if not snapshot and 'wx_calls' in tables:
        call=c.execute('SELECT state FROM wx_calls WHERE ride_id=? AND owner=? LIMIT 1',(rid,owner)).fetchone()
        if call:snapshot=parsed(call['state'])
    keys=('id','pickup','destination','plat','plng','dlat','dlng','vehicle','fare','status','driver_id','driver_name','driver_phone','driver_vehicle_number','source','created','updated')
    result={key:record.get(key) for key in keys}
    result['passengers']=record.get('passengers') if record.get('passengers') is not None else snapshot.get('count')
    result['pickup_time']=record.get('pickup_time') or snapshot.get('pickup_at')
    distance=next((record[key] for key in ('distance_km','km','road_distance_km') if finite(record.get(key),0) is not None),None)
    if distance is None:distance=parsed(snapshot.get('fare')).get('km')
    result['distance_km']=finite(distance,0)
    weather=parsed(record.get('weather_json')) or parsed(snapshot.get('weather'))
    result['weather']=dict(status=weather.get('status'),condition=weather.get('condition'),fetched_at=weather.get('fetched_at'),points=[])
    if weather.get('status')=='ok':
        for point in weather.get('points',[]) if isinstance(weather.get('points'),list) else []:
            if isinstance(point,dict) and point.get('location') in ('pickup','destination'):
                result['weather']['points'].append(dict(location=point['location'],precipitation_probability=finite(point.get('precipitation_probability'),0,100),temperature_2m=finite(point.get('temperature_2m')),valid_at=point.get('valid_at')))
    result['pickup_verified']=False
    if 'ride_pickup_codes' in tables:
        code=c.execute('SELECT code,verified FROM ride_pickup_codes WHERE ride_id=?',(rid,)).fetchone()
        result['pickup_verified']=bool(code and code['verified'])
        if code and not result['pickup_verified'] and record['status'] in ('requested','accepted','arriving'):result['pickup_otp']=code['code']
    if 'location_requests' in tables and record.get('driver_id'):
        request=c.execute('SELECT status,expires FROM location_requests WHERE ride_id=? AND customer_id=? AND driver_id=?',(rid,owner,record['driver_id'])).fetchone()
        if request:
            result['location_request_status']='expired' if request['status']=='granted' and request['expires']<=time.time() else request['status']
            if request['status']=='granted' and request['expires']>time.time() and record['status'] in ('accepted','arriving','in_progress'):result['location_share_until']=request['expires']
    return result

def install_backend(module):
    if getattr(module.Handler,'_movio_customer_tools_version',None)==VERSION:return dict(version=VERSION,already_installed=True)
    previous=module.Handler.do_GET
    def get(h):
        parsed_url=urlparse(h.path);path=parsed_url.path
        if path=='/api/customer-tools/health':return h.json(200,dict(ok=True,version=VERSION))
        if not path.startswith('/api/customer-tools/'):return previous(h)
        try:
            with module.conn() as c:
                user=h.user(c)
                if user['role']!='customer':return h.json(403,dict(error='Customer account required'))
                owner=user['id']
                match=re.fullmatch(r'/api/customer-tools/booking/([1-9]\d*)',path)
                if match:
                    result=booking_details(c,owner,int(match[1]))
                    return h.json(200,dict(ok=True,booking=result)) if result else h.json(404,dict(error='Booking not found'))
            query=parse_qs(parsed_url.query)
            arg=lambda name:query.get(name,[''])[0]
            if path.endswith('/address-search'):return h.json(200,dict(ok=True,places=address_search(arg('q'))))
            if path.endswith('/reverse-address'):return h.json(200,dict(ok=True,place=reverse_address(arg('lat'),arg('lng'))))
            if path.endswith('/route'):return h.json(200,dict(ok=True,route=road_route((arg('plat'),arg('plng')),(arg('dlat'),arg('dlng')))))
            return h.json(404,dict(error='Unknown customer endpoint'))
        except PermissionError:return h.json(401,dict(error='Sign in to your customer account'))
        except (ValueError,KeyError,TypeError) as exc:return h.json(400,dict(error=str(exc)))
        except LookupError as exc:return h.json(502,dict(error=str(exc)))
        except Exception:return h.json(500,dict(error='The request could not be completed. Please retry.'))
    module.Handler.do_GET=get;module.Handler._movio_customer_tools_version=VERSION
    return dict(version=VERSION)
'''
    javascript=r'''/* Customer booking details and one selection/route path for search, GPS and map. */
(function installCustomerBookingTools(){
 'use strict';
 if(window.MOVIOCustomerBookingUI)return;
 const el=id=>document.getElementById(id);
 if(!el('riderPanel')||typeof setPlace!=='function'||typeof updateRoute!=='function')return;
 const book=el('bookRide'),oldPreview=book?.onclick;
 if(typeof oldPreview!=='function'||!el('weatherManual'))return;
 const state={field:'pickup',routeKey:null,routePromise:null,routeSeq:0,routeController:null,previewBusy:false,lastPreview:null,previewTimer:null,searchSeq:0,dialogSeq:0,dialogId:null,dialogTimer:null,returnFocus:null};
 const words=()=>({Tamil:{searching:'முகவரியைத் தேடுகிறோம்…',choose:'பட்டியலில் சரியான முகவரியைத் தேர்ந்தெடுக்கவும்.',none:'முகவரி கிடைக்கவில்லை. தெரு, பகுதி அல்லது அஞ்சல் குறியீட்டுடன் மீண்டும் தேடுங்கள்.',locating:'உங்கள் தற்போதைய இடத்தைப் பெறுகிறோம்…',gps:'உலாவியில் இருப்பிட அனுமதியை வழங்கி மீண்டும் முயற்சிக்கவும்.',route:'சாலை தூரத்தைச் சரிபார்க்கிறோம்…',retry:'சாலை வழியை மீண்டும் சரிபார்க்க, பயணச் சரிபார்ப்பு பொத்தானை அழுத்தவும்.',address:'முகவரி அல்லது அருகிலுள்ள இடத்தின் பெயரை உள்ளிடுங்கள்.'},Tanglish:{searching:'Address search pannrom…',choose:'List-la correct address select pannunga.',none:'Street, area illa postcode serthu thirumba search pannunga.',locating:'Unga current location edukkrom…',gps:'Browser location permission allow pannittu retry pannunga.',route:'Road distance check pannrom…',retry:'Road route retry panna Review ride press pannunga.',address:'Address illa landmark type pannunga.'}}[el('wxManualLanguage')?.value]||{searching:'Searching addresses…',choose:'Choose the correct address from the results.',none:'No address matched. Try a street, area or postcode.',locating:'Getting your current location…',gps:'Allow browser location permission, then try again.',route:'Checking the road route…',retry:'Press Review ride to retry the road route.',address:'Enter an address or nearby landmark.'});
 function status(text){if(el('wxManualStatus'))el('wxManualStatus').textContent=text;}
 function locked(){return ['wxManualDate','wxManualLanguage','wxManualCount'].some(id=>el(id)?.disabled);}
 function customer(){return typeof role==='undefined'||role==='customer';}
 function validPlace(p){return p&&Number.isFinite(Number(p.lat))&&Math.abs(Number(p.lat))<=90&&Number.isFinite(Number(p.lng))&&Math.abs(Number(p.lng))<=180&&String(p.name||'').trim();}
 function selected(id){return id==='pickup'?pickup:destination;}
 function matches(id){const p=selected(id);return validPlace(p)&&el(id).value.trim()===String(p.name).trim();}
 function placesKey(){return JSON.stringify([pickup?.name,pickup?.lat,pickup?.lng,destination?.name,destination?.lat,destination?.lng]);}
 function previewKey(){return JSON.stringify([placesKey(),el('wxManualDate')?.value,el('wxManualLanguage')?.value,el('wxManualCount')?.value,vehicle,km]);}
 async function request(path,options={}){
  const controller=new AbortController(),timer=setTimeout(()=>controller.abort(),16000);
  const cancel=()=>controller.abort();if(options.signal){if(options.signal.aborted)controller.abort();else options.signal.addEventListener('abort',cancel,{once:true});}
  try{const response=await fetch('/api/customer-tools/'+path,{credentials:'same-origin',cache:'no-store',...options,signal:controller.signal});
   const data=await response.json();if(!response.ok||!data.ok)throw Error(data.error||'Please retry this request.');return data;
  }catch(error){if(error.name==='AbortError')throw Error('The request timed out. Please retry.');throw error;}finally{clearTimeout(timer);options.signal?.removeEventListener('abort',cancel);}
 }
 function clearRoute(){state.routeKey=null;km=0;minutes=0;routeData=null;if(routeLine){map.removeLayer(routeLine);routeLine=null;}if(typeof fares==='function')fares();}
 function schedulePreview(){
  clearTimeout(state.previewTimer);
  state.previewTimer=setTimeout(()=>{
   if(customer()&&!locked()&&!book.disabled&&!state.previewBusy&&matches('pickup')&&matches('destination')&&el('wxManualDate')?.value&&state.routeKey===placesKey()&&state.lastPreview!==previewKey())preview();
  },650);
 }
 updateRoute=function(){
  const ticket=++state.routeSeq;state.routeController?.abort();state.routeKey=null;
  if(!matches('pickup')||!matches('destination')){clearRoute();state.routePromise=Promise.resolve(false);return state.routePromise;}
  const key=placesKey(),a={...pickup},b={...destination};const controller=new AbortController();state.routeController=controller;
  clearRoute();el('routeInfo').textContent=words().route;
  status(words().route);
  state.routePromise=(async()=>{
   try{const params=new URLSearchParams({plat:a.lat,plng:a.lng,dlat:b.lat,dlng:b.lng});
    const data=await request('route?'+params,{signal:controller.signal});
    if(ticket!==state.routeSeq||key!==placesKey())return false;
    const route=data.route;if(!route||!Number.isFinite(Number(route.km))||Number(route.km)<=0)throw Error('The road distance needs another check.');
    km=Number(route.km);minutes=Number(route.minutes)||0;routeData=Array.isArray(route.geometry)&&route.geometry.length>1?route.geometry:null;
    if(routeLine){map.removeLayer(routeLine);routeLine=null;}
    if(routeData){routeLine=L.polyline(routeData,{color:'#15191e',weight:5,opacity:.85}).addTo(map);map.fitBounds(routeLine.getBounds(),{padding:[45,45]});}
    el('routeInfo').textContent=km.toFixed(1)+' km'+(minutes?' · ~'+minutes+' min':'')+' · Road route · Fare is an estimate';
    state.routeKey=key;fares();schedulePreview();return true;
   }catch(error){if(ticket===state.routeSeq&&key===placesKey()){clearRoute();el('routeInfo').textContent=words().retry;status(error.message);}return false;}
  })();return state.routePromise;
 };
 function applyPlace(id,p){
  if(locked()||!customer()||!validPlace(p))return false;
  setPlace(id,{lat:Number(p.lat),lng:Number(p.lng),name:String(p.name).trim()},false);
  el(id).title=p.full_address||p.name;el('suggestions')?.replaceChildren();state.lastPreview=null;
  return matches(id);
 }
 async function searchAddress(id=state.field,autoChoose=true){
  if(locked()||!customer())return false;
  const input=el(id),query=input.value.trim();state.field=id;
  if(query.length<3){status(words().address);input.focus();return false;}
  const seq=++state.searchSeq;const holder=el('suggestions');holder.textContent=words().searching;status(words().searching);
  try{const data=await request('address-search?'+new URLSearchParams({q:query}));
   if(seq!==state.searchSeq||query!==input.value.trim()||locked())return false;
   const places=Array.isArray(data.places)?data.places.filter(validPlace):[];holder.replaceChildren();
   if(places.length===1&&autoChoose){const ok=applyPlace(id,places[0]);if(ok)status('');return ok;}
   for(const p of places){const button=document.createElement('button');button.type='button';button.className='movio-address-result';button.textContent=p.full_address||p.name;
    button.onclick=()=>{if(seq!==state.searchSeq||query!==input.value.trim()||locked())return;if(applyPlace(id,p))status('');};holder.append(button);}
   holder.setAttribute('aria-live','polite');status(places.length?words().choose:words().none);return false;
  }catch(error){if(seq===state.searchSeq){holder.textContent=error.message;status(error.message);}return false;}
 }
 async function currentPickup(){
  if(locked()||!customer())return;state.searchSeq++;const seq=state.searchSeq,button=el('useCurrentPickup');if(button)button.disabled=true;status(words().locating);
  try{const position=typeof locate==='function'?await locate():await new Promise((resolve,reject)=>navigator.geolocation.getCurrentPosition(p=>resolve({lat:p.coords.latitude,lng:p.coords.longitude}),reject,{enableHighAccuracy:true,timeout:12000,maximumAge:3000}));
   if(seq!==state.searchSeq||locked())return;
   let p={lat:Number(position.lat),lng:Number(position.lng),name:'Current pickup location ('+Number(position.lat).toFixed(5)+', '+Number(position.lng).toFixed(5)+')'};
   try{const data=await request('reverse-address?'+new URLSearchParams({lat:p.lat,lng:p.lng}));p=data.place||p;}catch{/* A valid GPS fix does not depend on a readable street label. */}
   if(seq!==state.searchSeq||locked())return;
   if(applyPlace('pickup',p)){state.field='destination';status('');}
  }catch{status(words().gps);}finally{if(button)button.disabled=locked();}
 }
 async function preview(event){
  event?.preventDefault();if(!customer()||locked()||book.disabled||state.previewBusy)return;
  clearTimeout(state.previewTimer);state.previewBusy=true;
  try{
   for(const id of ['pickup','destination'])if(!matches(id)&&!await searchAddress(id,true))return;
   if(!el('wxManualDate')?.value){await oldPreview.call(book,event);return;}
   const key=placesKey();
   if(state.routeKey!==key){
    if(state.routePromise)await state.routePromise;
    if(state.routeKey!==placesKey()&&!await updateRoute())return;
   }
   if(locked()||!matches('pickup')||!matches('destination')||state.routeKey!==placesKey())return;
   const captured=previewKey();await oldPreview.call(book,event);
   if(captured===previewKey()&&el('wxManualReview')&&!el('wxManualReview').hidden)state.lastPreview=captured;
  }catch(error){status(error.message);}finally{state.previewBusy=false;}
 }
 book.onclick=preview;
 for(const id of ['pickup','destination']){
  el(id).addEventListener('focus',()=>state.field=id);
  el(id).addEventListener('input',()=>{
   state.field=id;state.searchSeq++;state.lastPreview=null;clearTimeout(state.previewTimer);
   if(!matches(id)){state.routeController?.abort();state.routeSeq++;clearRoute();}
  });
  el(id).addEventListener('keydown',event=>{if(event.key==='Enter'){event.preventDefault();event.stopImmediatePropagation();searchAddress(id);}},true);
 }
 if(el('searchAddress'))el('searchAddress').onclick=()=>searchAddress(state.field);
 if(el('useCurrentPickup'))el('useCurrentPickup').onclick=currentPickup;
 for(const id of ['wxManualDate','wxManualLanguage','wxManualCount','wxPickupDay','wxPickupTime'])el(id)?.addEventListener('change',()=>{state.lastPreview=null;schedulePreview();});
 el('vehicles')?.addEventListener('click',()=>{state.lastPreview=null;schedulePreview();});

 // Details use the authenticated single-booking API, never a stale card's data.
 const dialog=document.createElement('dialog');dialog.id='movioBookingDetails';dialog.setAttribute('aria-labelledby','movioDetailsTitle');
 dialog.innerHTML='<div class="movio-detail-head"><div><span class="movio-detail-eyebrow">YOUR BOOKING</span><h2 id="movioDetailsTitle">Booking details</h2></div><button type="button" class="movio-detail-close" aria-label="Close booking details">×</button></div><div class="movio-detail-content" aria-live="polite"></div><div class="movio-detail-footer"><button type="button" class="outline" data-refresh-details>Refresh details</button><button type="button" data-close-details>Done</button></div>';
 document.body.append(dialog);const content=dialog.querySelector('.movio-detail-content');
 function node(tag,className,text){const n=document.createElement(tag);if(className)n.className=className;if(text!==undefined)n.textContent=String(text);return n;}
 function shown(value){return value!==null&&value!==undefined&&value!==''?String(value):'Not recorded';}
 function date(value){if(value===null||value===undefined||value==='')return 'Not recorded';
  const raw=typeof value==='number'?value*1000:/^\d{4}-\d\d-\d\dT\d\d:\d\d(?::\d\d)?$/.test(value)?value+'+05:30':value;
  const d=new Date(raw);return Number.isNaN(d.getTime())?String(value):new Intl.DateTimeFormat('en-IN',{timeZone:'Asia/Kolkata',day:'numeric',month:'long',year:'numeric',hour:'numeric',minute:'2-digit',hour12:true}).format(d);}
 function field(parent,label,value,className=''){const box=node('div','movio-detail-field '+className);box.append(node('dt','',label),node('dd','',shown(value)));parent.append(box);}
 function renderDetails(r){
  content.replaceChildren();el('movioDetailsTitle').textContent='Booking #'+r.id;
  content.append(node('span','movio-detail-status',String(r.status||'').replaceAll('_',' ')));
  const addresses=node('dl','movio-detail-addresses');field(addresses,'Pickup',r.pickup);field(addresses,'Drop-off',r.destination);content.append(addresses);
  const facts=node('dl','movio-detail-grid');
  field(facts,'Pickup date & time',date(r.pickup_time),'movio-detail-wide');field(facts,'Vehicle',r.vehicle);field(facts,'Passengers',r.passengers);
  const money=r.fare!==null&&r.fare!==undefined&&Number.isFinite(Number(r.fare))?new Intl.NumberFormat('en-IN',{style:'currency',currency:'INR',maximumFractionDigits:0}).format(Number(r.fare)):null;
  field(facts,'Estimated fare',r.status==='needs_location'&&Number(r.fare)===0?'Awaiting route':money);
  field(facts,'Road distance',r.distance_km!==null&&r.distance_km!==undefined&&Number.isFinite(Number(r.distance_km))?Number(r.distance_km).toFixed(1)+' kilometres':null);
  field(facts,'Booked on',date(r.created),'movio-detail-wide');field(facts,'Booked through',/voice|ai_call/i.test(r.source||'')?'AI Call':'Website');content.append(facts);
  const driver=node('section','movio-detail-section');driver.append(node('h3','','Driver details'));
  if(r.driver_id){const d=node('dl','movio-detail-grid');field(d,'Driver',r.driver_name||'Assigned driver');field(d,'Vehicle registration',r.driver_vehicle_number);
   const phone=String(r.driver_phone||'');const box=node('div','movio-detail-field');box.append(node('dt','','Phone'));
   const value=node('dd');if(/^\+?[0-9]{7,15}$/.test(phone)){const link=node('a','',phone);link.href='tel:'+phone;value.append(link);}else value.textContent=shown(phone);box.append(value);d.append(box);driver.append(d);
  }else driver.append(node('p','movio-detail-muted','A matching driver has not accepted this booking yet.'));
  content.append(driver);
  const otp=node('section','movio-detail-otp');otp.append(node('span','movio-detail-eyebrow','PICKUP OTP'));
  otp.append(node('strong','',r.pickup_otp|| (r.pickup_verified?'Verified':r.status==='scheduled'?'Available nearer pickup':'Not active')));content.append(otp);
  const w=r.weather||{};
  if(w.status==='ok'&&Array.isArray(w.points)&&w.points.length){const weather=node('details','movio-detail-section');weather.append(node('summary','','Weather recorded for your journey'));
   for(const role of ['pickup','destination']){const point=w.points.find(p=>p.location===role);if(point&&point.precipitation_probability!==null&&point.precipitation_probability!==undefined)weather.append(node('p','movio-detail-muted',(role==='pickup'?'Pickup':'Drop-off')+' · '+point.precipitation_probability+'% rain chance'));}
   content.append(weather);}
  if(r.location_share_until)content.append(node('p','movio-detail-muted','Location sharing ends: '+date(r.location_share_until)));
 }
 async function loadDetails(){
  const id=state.dialogId,seq=++state.dialogSeq;if(!id)return;
  try{const data=await request('booking/'+id);if(dialog.open&&id===state.dialogId&&seq===state.dialogSeq)renderDetails(data.booking);}
  catch(error){if(dialog.open&&id===state.dialogId&&seq===state.dialogSeq){content.replaceChildren(node('p','movio-detail-muted',error.message));}}
 }
 function closeDetails(){
  state.dialogId=null;state.dialogSeq++;clearInterval(state.dialogTimer);content.replaceChildren();
  if(typeof dialog.close==='function')dialog.close();else dialog.removeAttribute('open');state.returnFocus?.focus();
 }
 function openDetails(id){
  id=Number(id);if(!Number.isInteger(id)||id<1||!customer())return;
  state.dialogId=id;state.returnFocus=document.activeElement;el('movioDetailsTitle').textContent='Booking #'+id;content.replaceChildren(node('p','movio-detail-muted','Loading booking details…'));
  if(!dialog.open){if(typeof dialog.showModal==='function')dialog.showModal();else dialog.setAttribute('open','');}
  dialog.querySelector('.movio-detail-close').focus();loadDetails();clearInterval(state.dialogTimer);
  state.dialogTimer=setInterval(()=>{if(dialog.open&&!document.hidden)loadDetails();},12000);
 }
 dialog.querySelector('.movio-detail-close').onclick=closeDetails;dialog.querySelector('[data-close-details]').onclick=closeDetails;dialog.querySelector('[data-refresh-details]').onclick=loadDetails;
 dialog.addEventListener('cancel',event=>{event.preventDefault();closeDetails();});
 function bookingId(card){const heading=card?.querySelector('h3,strong,h4');const m=String(heading?.textContent||card?.textContent||'').match(/#\s*(\d+)/);return m?Number(m[1]):null;}
 function decorateBookings(){
  for(const card of document.querySelectorAll('#aiRideHistory .ride-card,#weatherScheduledRides .ride-card')){
   const id=bookingId(card);if(!id)continue;card.dataset.bookingDetailId=String(id);card.classList.add('movio-booking-clickable');
   if(card.querySelector('[data-open-booking]'))continue;
   const button=node('button','outline','Booking details');button.type='button';button.dataset.openBooking=String(id);button.setAttribute('aria-label','View booking '+id+' details');
   (card.querySelector('.movio-card-actions')||card).append(button);
  }
  if(el('riderTrip')&&!el('movioCurrentBookingDetails')){const button=node('button','outline','View full booking details');button.type='button';button.id='movioCurrentBookingDetails';button.onclick=()=>openDetails(typeof rideId!=='undefined'?rideId:null);el('riderTrip').append(button);}
 }
 document.addEventListener('click',event=>{
  const button=event.target.closest?.('[data-open-booking]');if(button){event.preventDefault();openDetails(button.dataset.openBooking);return;}
  const card=event.target.closest?.('[data-booking-detail-id]');if(card&&!event.target.closest('button,a,input,select,textarea,summary,label'))openDetails(card.dataset.bookingDetailId);
 });
 el('activeRideSelect')?.addEventListener('change',()=>{if(el('activeRideSelect').value)openDetails(el('activeRideSelect').value);});
 for(const root of [el('aiRideHistory'),el('weatherScheduledRides')].filter(Boolean))new MutationObserver(decorateBookings).observe(root,{childList:true,subtree:true});
 decorateBookings();
 window.MOVIOCustomerBookingUI={version:'customer-booking-tools-1.0',openDetails,closeDetails,searchAddress,currentPickup,preview,state};
 if(matches('pickup')&&matches('destination'))updateRoute();
})();
'''
    css=r'''/* Fits the current white, charcoal and gold customer-dashboard theme. */
#riderPanel .movio-booking-clickable{cursor:pointer;transition:border-color .15s,box-shadow .15s}
#riderPanel .movio-booking-clickable:hover{border-color:#d7ae3b;box-shadow:0 3px 14px #17202d0a}
#riderPanel [data-open-booking],#movioCurrentBookingDetails{min-height:44px;white-space:normal;line-height:1.4}
#suggestions .movio-address-result{display:block;text-align:left;width:100%;padding:13px;border:1px solid #e0e4e9;border-radius:10px;background:#fff;color:#26313e;font:inherit;line-height:1.5;margin:5px 0;white-space:normal;overflow-wrap:anywhere;cursor:pointer}
#suggestions .movio-address-result:hover,#suggestions .movio-address-result:focus-visible{background:#fff7df;border-color:#d7ae3b}
#movioBookingDetails{width:min(640px,calc(100vw - 28px));max-height:calc(100dvh - 28px);box-sizing:border-box;padding:0;border:1px solid #e1e5eb;border-radius:20px;background:#fff;color:#202936;box-shadow:0 24px 90px #10182038;font-family:inherit;overflow:hidden}
#movioBookingDetails::backdrop{background:#111a2d70;backdrop-filter:blur(3px)}
#movioBookingDetails *{box-sizing:border-box;min-width:0}
#movioBookingDetails .movio-detail-head{display:flex;justify-content:space-between;align-items:center;gap:16px;padding:22px 24px 18px;border-bottom:1px solid #e9ebef;background:#fffcf3}
#movioBookingDetails h2{margin:5px 0 0;font-size:22px;line-height:1.35;letter-spacing:-.4px}
#movioBookingDetails .movio-detail-eyebrow{font-size:10px;font-weight:750;letter-spacing:1.4px;color:#70798a}
#movioBookingDetails .movio-detail-close{display:grid;place-items:center;width:42px;height:42px;flex:none;padding:0;border:1px solid #dfe3e8;border-radius:12px;background:#fff;color:#354153;font-size:26px;cursor:pointer}
#movioBookingDetails .movio-detail-content{padding:20px 24px;max-height:calc(100dvh - 210px);overflow:auto;overscroll-behavior:contain}
#movioBookingDetails .movio-detail-status{display:inline-block;padding:6px 11px;margin-bottom:18px;border-radius:999px;background:#fff1c4;color:#715514;font-size:12px;font-weight:700;text-transform:capitalize}
#movioBookingDetails dl{margin:0}
#movioBookingDetails .movio-detail-addresses{display:grid;gap:14px;padding-bottom:20px;margin-bottom:18px;border-bottom:1px solid #e9ecf0}
#movioBookingDetails .movio-detail-grid{display:grid;grid-template-columns:1fr 1fr;gap:18px 24px}
#movioBookingDetails dt{margin:0 0 5px;font-size:11px;line-height:1.4;color:#727d8d;font-weight:600}
#movioBookingDetails dd{margin:0;font-size:14px;font-weight:600;line-height:1.6;color:#263140;overflow-wrap:anywhere}
#movioBookingDetails .movio-detail-wide{grid-column:1/-1}
#movioBookingDetails .movio-detail-section{border-top:1px solid #e9ecf0;margin-top:20px;padding-top:18px}
#movioBookingDetails h3,#movioBookingDetails summary{font-size:14px;font-weight:700;margin:0 0 13px;line-height:1.5}
#movioBookingDetails summary{cursor:pointer;margin-bottom:0;min-height:30px}
#movioBookingDetails .movio-detail-muted{font-size:13px;line-height:1.7;color:#637082;margin:6px 0}
#movioBookingDetails a{color:#8b6500;text-decoration:underline;text-underline-offset:3px}
#movioBookingDetails .movio-detail-otp{display:flex;align-items:center;justify-content:space-between;gap:15px;padding:15px 17px;margin-top:20px;border:1px solid #ecdfb7;border-radius:12px;background:#fffbf0}
#movioBookingDetails .movio-detail-otp strong{font-size:22px;letter-spacing:3px;font-weight:750;text-align:right}
#movioBookingDetails .movio-detail-otp strong:not(:empty){overflow-wrap:anywhere}
#movioBookingDetails .movio-detail-footer{display:grid;grid-template-columns:1fr 1fr;gap:10px;padding:16px 24px;border-top:1px solid #e8ebef;background:#fff}
#movioBookingDetails .movio-detail-footer button{min-height:46px;border-radius:11px;border:1px solid #dce1e7;background:#fff;color:#394657;font:inherit;font-size:13px;font-weight:650;cursor:pointer}
#movioBookingDetails .movio-detail-footer [data-close-details]{background:#202834;border-color:#202834;color:white}
#movioBookingDetails button:focus-visible,#movioBookingDetails summary:focus-visible{outline:3px solid #d8a52c;outline-offset:3px}
@media(max-width:520px){#movioBookingDetails{width:calc(100vw - 20px);max-height:calc(100dvh - 20px);border-radius:16px}#movioBookingDetails .movio-detail-head{padding:18px}#movioBookingDetails h2{font-size:20px}#movioBookingDetails .movio-detail-content{padding:18px;max-height:calc(100dvh - 192px)}#movioBookingDetails .movio-detail-grid{gap:15px}#movioBookingDetails .movio-detail-footer{padding:14px 18px}#movioBookingDetails .movio-detail-otp{align-items:flex-start;flex-direction:column}#movioBookingDetails .movio-detail-otp strong{text-align:left;letter-spacing:2px}}
'''
    compile(backend_source,str(helper_path),'exec')
    html=index_path.read_text(encoding='utf-8');manual=manual_path.read_text(encoding='utf-8')
    for identifier in ('riderPanel','pickup','destination','bookRide','routeInfo'):
        if not re.search(r'\bid=["\x27]'+identifier+r'["\x27]',html):raise RuntimeError('Expected dashboard control is missing: '+identifier)
    if not re.search(r'</body\s*>',html,re.I) or not re.search(r'</head\s*>',html,re.I):raise RuntimeError('Unexpected dashboard HTML; no files changed.')
    missing=[(identifier,label) for identifier,label in [('useCurrentPickup','Use current location'),('searchAddress','Search address')]
        if not re.search(r'\bid=["\x27]'+identifier+r'["\x27]',html)]
    if missing:
        controls='<div class="movio-address-actions">'+''.join('<button type="button" class="outline" id="'+identifier+'">'+label+'</button>' for identifier,label in missing)+'</div>\n'
        anchor=re.search(r'<[^>]+\bid=["\x27](?:searchNote|suggestions)["\x27][^>]*>',html)
        if not anchor:raise RuntimeError('The address search area was not found; no files changed.')
        html=html[:anchor.start()]+controls+html[anchor.start():]
    if 'wxManualChange' not in manual or 'manual-preview' not in manual:raise RuntimeError('Run the current manual-weather form setup first.')
    if '// MOVIO_CUSTOMER_ADDRESS_LOCK_V1' not in manual:
        anchor="el('wxManualChange')].filter(Boolean);"
        if manual.count(anchor)!=1:raise RuntimeError('The manual-form lock differs from the expected version; no files changed.')
        replacement="el('wxManualChange'),el('useCurrentPickup'),el('searchAddress'),el('pinPickup'),el('pinDestination')].filter(Boolean); // MOVIO_CUSTOMER_ADDRESS_LOCK_V1"
        manual=manual.replace(anchor,replacement,1)
    node=shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory(prefix='movio_customer_js_') as directory:
            for name,source in (('customer_booking_tools.js',javascript),('weather_manual.js',manual)):
                check=Path(directory)/name;check.write_text(source,encoding='utf-8')
                result=subprocess.run([node,'--check',str(check)],capture_output=True,text=True)
                if result.returncode:raise RuntimeError('JavaScript validation failed: '+result.stderr[:800])
    revision=hashlib.sha256((javascript+css+manual).encode()).hexdigest()[:12]
    html=re.sub(r'<script\b[^>]*\bsrc=["\x27][^"\x27]*\bcustomer_booking_tools\.js(?:\?[^"\x27]*)?["\x27][^>]*>\s*</script>\s*','',html,flags=re.I)
    html=re.sub(r'<link\b[^>]*\bhref=["\x27][^"\x27]*\bcustomer_booking_tools\.css(?:\?[^"\x27]*)?["\x27][^>]*>\s*','',html,flags=re.I)
    html=re.sub(r'</head\s*>',lambda m:'<link rel="stylesheet" href="/customer_booking_tools.css?v='+revision+'">\n'+m[0],html,count=1,flags=re.I)
    html=re.sub(r'</body\s*>',lambda m:'<script src="/customer_booking_tools.js?v='+revision+'" defer></script>\n'+m[0],html,count=1,flags=re.I)
    html=re.sub(r'(<script\b[^>]*\bsrc=["\x27])([^"\x27]*\bweather_manual\.js)(?:\?[^"\x27]*)?(["\x27])',lambda m:m[1]+m[2]+'?customer_tools='+revision+m[3],html,flags=re.I)
    server=server_path.read_text(encoding='utf-8')
    marker='# MOVIO_CUSTOMER_TOOLS_BOOTSTRAP_V1';end_marker='# MOVIO_CUSTOMER_TOOLS_BOOTSTRAP_END_V1'
    if marker in server:
        start=server.index(marker);end=server.find(end_marker,start)
        if end<0:raise RuntimeError('Incomplete previous customer-tools bootstrap; no files changed.')
        server=server[:start]+server[end+len(end_marker):]
    tree=ast.parse(server)
    def is_main_guard(node):
        if not isinstance(node,ast.If):return False
        test=node.test
        if not isinstance(test,ast.Compare) or len(test.ops)!=1 or not isinstance(test.ops[0],ast.Eq):return False
        left,right=test.left,test.comparators[0]
        return ((isinstance(left,ast.Name) and left.id=='__name__' and isinstance(right,ast.Constant) and right.value=='__main__')
            or (isinstance(right,ast.Name) and right.id=='__name__' and isinstance(left,ast.Constant) and left.value=='__main__'))
    mains=[node for node in tree.body if is_main_guard(node)]
    launchers=[node for node in mains if any(isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute)
        and call.func.attr=='serve_forever' for statement in node.body for call in ast.walk(statement))]
    if len(launchers)!=1:
        raise RuntimeError('Could not identify one HTTP serve_forever launch block (found '+str(len(mains))+
            ' main guards and '+str(len(launchers))+' server launch blocks). No files changed.')
    line=launchers[0].lineno-1;lines=server.splitlines(keepends=True)
    bootstrap="""# MOVIO_CUSTOMER_TOOLS_BOOTSTRAP_V1
if __name__ == '__main__':
    import importlib.util as _ct_import
    import sys as _ct_sys
    _ct_path=ROOT/'integrations/movio_customer_tools.py'
    _ct_spec=_ct_import.spec_from_file_location('movio_customer_tools',_ct_path)
    _ct_module=_ct_import.module_from_spec(_ct_spec)
    _ct_sys.modules[_ct_spec.name]=_ct_module
    exec(compile(_ct_path.read_text(encoding='utf-8'),str(_ct_path),'exec'),_ct_module.__dict__)
    _ct_module.install_backend(_ct_sys.modules[__name__])
# MOVIO_CUSTOMER_TOOLS_BOOTSTRAP_END_V1
"""
    server=''.join(lines[:line]).rstrip()+'\n\n'+bootstrap+'\n'+''.join(lines[line:])
    compile(server,str(server_path),'exec')

    # Identify the existing website before modifying or stopping anything.
    health_url='http://127.0.0.1:8080/api/customer-tools/health'
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet')
        if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    in_kernel=[]
    for module in list(sys.modules.values()):
        if not isinstance(module,types.ModuleType):continue
        values=vars(module);handler=values.get('Handler');known_root=values.get('ROOT')
        if not isinstance(handler,type) or known_root is None or not getattr(handler,'_wx_installed',False):continue
        try:
            if Path(known_root).resolve()==root and callable(getattr(handler,'do_POST',None)):in_kernel.append(module)
        except (TypeError,ValueError,OSError):continue
    if os.getpid() not in listeners:in_kernel=[]
    process_details=[]
    known_launchers=[]
    for value in list(globals().values()):
        if type(value) in weakref.ProxyTypes:continue
        try:
            if isinstance(value,subprocess.Popen) and value.poll() is None:known_launchers.append(value)
        except ReferenceError:continue
    for pid in listeners:
        if pid==os.getpid():
            if not in_kernel:raise RuntimeError('An unrecognized in-kernel service owns port 8080; no process was stopped.')
            continue
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server_path for path in scripts):
            raise RuntimeError('A different application owns port 8080; no process was stopped.')
        # Prefer the notebook's own Popen handle when present. Containers can
        # expose host PIDs in psutil while Popen uses kernel-namespace PIDs.
        controller=next((value for value in known_launchers if isinstance(value.args,(list,tuple))
            and list(value.args)==command),process)
        process_details.append((controller,command,cwd,process.environ()))
    if not listeners:raise RuntimeError('Start the original website on port 8080, then rerun this cell.')

    writes={helper_path:backend_source,server_path:server,web/'customer_booking_tools.js':javascript,
        web/'customer_booking_tools.css':css,index_path:html,manual_path:manual}
    changes={p:value for p,value in writes.items() if not p.exists() or p.read_text(encoding='utf-8')!=value}
    backup=None
    if changes:
        backup=root/('customer_tools_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,value in changes.items():
            if path.exists():
                saved=backup/path.relative_to(root);saved.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(path,saved)
        written=[]
        try:
            for path,value in changes.items():
                path.parent.mkdir(parents=True,exist_ok=True);temporary=path.with_name(path.name+'.customer.tmp')
                try:temporary.write_text(value,encoding='utf-8');os.replace(temporary,path)
                finally:temporary.unlink(missing_ok=True)
                written.append(path)
        except Exception:
            for path in reversed(written):
                saved=backup/path.relative_to(root)
                if saved.exists():shutil.copy2(saved,path)
                else:path.unlink(missing_ok=True)
            raise
    active=types.ModuleType('movio_customer_tools');active.__file__=str(helper_path);sys.modules[active.__name__]=active
    exec(compile(backend_source,str(helper_path),'exec'),active.__dict__)
    for backend in in_kernel:active.install_backend(backend)
    restart=any(path in changes for path in (server_path,helper_path))
    for process,command,cwd,environment in process_details:
        if not restart:
            try:
                response=requests.get(health_url,timeout=(1,2))
                if response.status_code==200 and response.json().get('version')==active.VERSION:continue
            except (requests.RequestException,ValueError):pass
        process.terminate();process.wait(timeout=8)
        with (root/'customer_tools_website.log').open('a',encoding='utf-8') as output:
            globals()['WX_WEBSITE_PROCESS']=subprocess.Popen(command,cwd=cwd,env=environment,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    verified=False
    for attempt in range(16):
        try:
            response=requests.get(health_url,timeout=(1,2))
            verified=response.status_code==200 and response.json().get('version')==active.VERSION
            if verified:break
        except (requests.RequestException,ValueError):pass
        time.sleep(.25)
    if not verified:raise RuntimeError('Website verification failed. See customer_tools_website.log; rerun the original website-only launcher, then this cell.')
    print('Booking details view installed. Click a booking card or its Booking details button.')
    print('Search address, current pickup location and map pins now share the route/weather preview flow.')
    print('Confirm booking remains a separate customer action.')
    print('Refresh the CUSTOMER DASHBOARD with Ctrl+Shift+R; on mobile, reload the page.')
    print('Historical values that were never saved appear as Not recorded.')
    return dict(version=active.VERSION,installer_version='1.1',website_verified=True,changed_files=[str(p.relative_to(root)) for p in changes],backup=str(backup) if backup else None)

MOVIO_CUSTOMER_BOOKING_TOOLS=movio_enable_customer_booking_tools()


Booking details view installed. Click a booking card or its Booking details button.
Search address, current pickup location and map pins now share the route/weather preview flow.
Confirm booking remains a separate customer action.
Refresh the CUSTOMER DASHBOARD with Ctrl+Shift+R; on mobile, reload the page.
Historical values that were never saved appear as Not recorded.


In [94]:
import shutil

shutil.make_archive(
    "/kaggle/working/MOVIO_CURRENT_WEBSITE",
    "zip",
    "/kaggle/working",
    "MOVIO_MAIN_WEBSITE"
)

print("Created: /kaggle/working/MOVIO_CURRENT_WEBSITE.zip")

Created: /kaggle/working/MOVIO_CURRENT_WEBSITE.zip


In [95]:
from pathlib import Path

root = Path("/kaggle/input")

for p in root.rglob("*"):
    if "weather_fix" in str(p).lower():
        print(p)

In [96]:

from pathlib import Path
import shutil
import ast
from datetime import datetime

# Correct Kaggle dataset path
source = Path("/kaggle/input/datasets/reshrrt/ai-ride-weather-fix-01/integrations")
target = Path("/kaggle/working/MOVIO_MAIN_WEBSITE/integrations")

# Verify folders
assert source.is_dir(), f"Weather fix source not found: {source}"
assert target.is_dir(), f"MOVIO integrations folder not found: {target}"

# Find Python files
files = sorted(source.rglob("*.py"))
assert files, "No Python files found"

# Validate all Python files before installation
for f in files:
    ast.parse(f.read_text(encoding="utf-8"), filename=str(f))

# Create unique backup folder
backup = Path(
    "/kaggle/working/WEATHER_FIX_BACKUP_" +
    datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)

# Install updated files
for f in files:
    relative = f.relative_to(source)
    destination = target / relative

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        old_backup = backup / relative
        old_backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(destination, old_backup)

    shutil.copy2(f, destination)
    print("✅ Installed:", relative)

print("\n✅ WEATHER FIX FILES INSTALLED SUCCESSFULLY")
print("Updated files:", len(files))
print("Backup:", backup if backup.exists() else "No existing files replaced")


✅ Installed: movio_optional_weather.py
✅ Installed: movio_weather_guard.py

✅ WEATHER FIX FILES INSTALLED SUCCESSFULLY
Updated files: 2
Backup: /kaggle/working/WEATHER_FIX_BACKUP_20261028_070705_736036


In [97]:
from pathlib import Path

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")

keywords = [
    "movio_optional_weather",
    "movio_weather_guard"
]

print("Checking weather integration references:\n")

for f in root.rglob("*.py"):
    if any(x in f.parts for x in ("venv", ".venv", "__pycache__")):
        continue

    try:
        lines = f.read_text(encoding="utf-8").splitlines()
    except (UnicodeError, OSError):
        continue

    for number, line in enumerate(lines, 1):
        if any(k in line for k in keywords):
            print(f"{f.relative_to(root)}:{number}")
            print(" ", line.strip()[:200])

print("\n✅ Integration reference check completed")

Checking weather integration references:

server.py:377
  _wx_spec = _wx_import.spec_from_file_location('movio_weather_guard', ROOT / 'integrations/movio_weather_guard.py')
customer_tools_backup_20261028_070704_881630/server.py:377
  _wx_spec = _wx_import.spec_from_file_location('movio_weather_guard', ROOT / 'integrations/movio_weather_guard.py')
integrations/movio_time_route.py:118
  geocode._movio_optional_weather_geo=True
integrations/movio_time_route.py:152
  previous_group=engine.get('_movio_optional_weather_original',{}).get('collect_details')
integrations/test_weather_guard.py:26
  import movio_weather_guard as wx
integrations/movio_weather_guard.py:1375
  _mo_file=Path(__file__).with_name('movio_optional_weather.py')
integrations/movio_weather_guard.py:1376
  _mo_spec=_mo_import.spec_from_file_location('movio_optional_weather',_mo_file)
integrations/movio_required_weather.py:176
  optional=engine.get('_movio_optional_weather_original') or {}
integrations/movio_optional_weather.

In [98]:
import requests

base = "http://127.0.0.1:8080"

for endpoint in [
    "/api/health",
    "/api/weather/health",
    "/api/weather-health"
]:
    try:
        r = requests.get(base + endpoint, timeout=10)
        print("\nENDPOINT:", endpoint)
        print("STATUS:", r.status_code)
        print("RESPONSE:", r.text[:1200])
    except Exception as e:
        print("ERROR:", repr(e))


ENDPOINT: /api/health
STATUS: 200
RESPONSE: {"ok": true, "version": "5.3", "calling_configured": false, "tts_configured": false, "stt_configured": false, "stream_configured": false, "ai_call_configured": false}

ENDPOINT: /api/weather/health
STATUS: 401
RESPONSE: {"error": "Log in first"}

ENDPOINT: /api/weather-health
STATUS: 401
RESPONSE: {"error": "Log in first"}


In [99]:
from pathlib import Path
import shutil
import ast
from datetime import datetime

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
server = root / "server.py"
optional = root / "integrations/movio_optional_weather.py"
guard = root / "integrations/movio_weather_guard.py"

assert server.exists(), "server.py missing"
assert optional.exists(), "Optional weather module missing"
assert guard.exists(), "Weather guard missing"

source = server.read_text(encoding="utf-8")

marker = "    _wx_mod.install_backend(_wx_sys.modules[__name__])"

replacement = """    # Activate the optional forecast implementation before installing routes.
    _opt_path = ROOT / 'integrations/movio_optional_weather.py'
    _opt_spec = _wx_import.spec_from_file_location('movio_optional_weather', _opt_path)
    _opt_mod = _wx_import.module_from_spec(_opt_spec)
    _wx_sys.modules[_opt_spec.name] = _opt_mod
    _opt_spec.loader.exec_module(_opt_mod)
    _opt_mod.install_namespace(_wx_mod.__dict__, root=ROOT)
    _wx_mod.install_backend(_wx_sys.modules[__name__])"""

if "_opt_mod.install_namespace(_wx_mod.__dict__, root=ROOT)" in source:
    print("Optional weather startup already installed.")
else:
    assert source.count(marker) == 1, "Unexpected startup structure; no changes made."

    updated = source.replace(marker, replacement, 1)

    ast.parse(updated)

    backup = server.with_name(
        "server_weather_backup_" +
        datetime.now().strftime("%Y%m%d_%H%M%S") +
        ".py"
    )

    shutil.copy2(server, backup)
    server.write_text(updated, encoding="utf-8")

    print("PATCH INSTALLED")
    print("Backup:", backup)

print("Python syntax: VALID")

PATCH INSTALLED
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/server_weather_backup_20261028_070705.py
Python syntax: VALID


In [100]:
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

tz = ZoneInfo("Asia/Kolkata")
date = (datetime.now(tz) + timedelta(days=1)).date().isoformat()

# Reproduce the two-location request using known valid coordinates.
params = {
    "latitude": "13.0827,12.9352",
    "longitude": "80.2707,80.2275",
    "hourly": "weather_code,precipitation_probability,precipitation,temperature_2m,apparent_temperature",
    "timezone": "Asia/Kolkata",
    "start_date": date,
    "end_date": date
}

r = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params=params,
    timeout=15
)

print("HTTP:", r.status_code)
print("Request URL:", r.url)
print("Provider response:", r.text[:500])

HTTP: 200
Request URL: https://api.open-meteo.com/v1/forecast?latitude=13.0827%2C12.9352&longitude=80.2707%2C80.2275&hourly=weather_code%2Cprecipitation_probability%2Cprecipitation%2Ctemperature_2m%2Capparent_temperature&timezone=Asia%2FKolkata&start_date=2026-10-29&end_date=2026-10-29
Provider response: [{"latitude":13.110721,"longitude":80.2459,"generationtime_ms":92.06438064575195,"utc_offset_seconds":19800,"timezone":"Asia/Kolkata","timezone_abbreviation":"GMT+5:30","elevation":12.0,"hourly_units":{"time":"iso8601","weather_code":"wmo code","precipitation_probability":"%","precipitation":"mm","temperature_2m":"°C","apparent_temperature":"°C"},"hourly":{"time":["2026-10-29T00:00","2026-10-29T01:00","2026-10-29T02:00","2026-10-29T03:00","2026-10-29T04:00","2026-10-29T05:00","2026-10-29T06:00",


In [101]:
import os
import signal
import subprocess
import time
import requests
from pathlib import Path

server = Path("/kaggle/working/MOVIO_MAIN_WEBSITE/server.py")
assert server.exists(), "MOVIO server.py not found"

# Find only processes running this exact server.
result = subprocess.run(
    ["ps", "-eo", "pid=,args="],
    capture_output=True,
    text=True,
    check=True
)

pids = []

for line in result.stdout.splitlines():
    parts = line.strip().split(maxsplit=1)
    if len(parts) != 2:
        continue

    pid = int(parts[0])
    if pid == os.getpid():
        continue

    try:
        cmd = Path(f"/proc/{pid}/cmdline").read_bytes().split(b"\x00")
        if any(arg.decode(errors="replace") == str(server) for arg in cmd):
            pids.append(pid)
    except (FileNotFoundError, ProcessLookupError, PermissionError):
        continue

print("Existing MOVIO server PIDs:", pids)

for pid in pids:
    try:
        os.kill(pid, signal.SIGTERM)
        print("Stopped:", pid)
    except ProcessLookupError:
        pass

# Wait until port 8080 is free.
for _ in range(40):
    try:
        requests.get("http://127.0.0.1:8080/api/health", timeout=1)
    except requests.RequestException:
        break
    time.sleep(0.5)

log_path = Path("/kaggle/working/movio_server_weather.log")
log = open(log_path, "a")

process = subprocess.Popen(
    ["/usr/bin/python3", "-u", str(server)],
    cwd=str(server.parent),
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("New MOVIO PID:", process.pid)

for _ in range(40):
    if process.poll() is not None:
        raise RuntimeError(
            f"MOVIO server exited. Check {log_path}"
        )

    try:
        r = requests.get(
            "http://127.0.0.1:8080/api/weather-booking/optional-health",
            timeout=2
        )
        if r.status_code == 200:
            print("SERVER READY:", r.json())
            break
    except requests.RequestException:
        pass

    time.sleep(1)
else:
    raise RuntimeError("Server did not become ready")

print("Restart complete.")

Existing MOVIO server PIDs: [628]
Stopped: 628
New MOVIO PID: 635
SERVER READY: {'ok': True, 'version': 'optional-weather-1.0', 'weather_optional': True, 'final_confirmation_required': True}
Restart complete.


In [102]:
import os
import signal
import subprocess
import time
import requests
from pathlib import Path

server = Path("/kaggle/working/MOVIO_MAIN_WEBSITE/server.py")
assert server.exists(), "server.py missing"

# Find the current MOVIO server process automatically.
pids = []

for proc in Path("/proc").iterdir():
    if not proc.name.isdigit():
        continue

    pid = int(proc.name)
    if pid == os.getpid():
        continue

    try:
        args = (proc / "cmdline").read_bytes().split(b"\x00")
        if str(server).encode() in args:
            pids.append(pid)
    except (OSError, PermissionError):
        continue

print("Current MOVIO server PIDs:", pids)

# Stop only MOVIO server processes.
for pid in pids:
    try:
        os.kill(pid, signal.SIGTERM)
        print("Stopped old server:", pid)
    except ProcessLookupError:
        pass

# Wait until the old server releases port 8080.
for _ in range(30):
    check = subprocess.run(
        ["bash", "-lc", "ss -ltnp 2>/dev/null | grep ':8080' || true"],
        capture_output=True,
        text=True
    )
    if not check.stdout.strip():
        break
    time.sleep(1)
else:
    raise RuntimeError("Port 8080 is occupied. No new server started.")

# Start the updated server.
log_path = "/kaggle/working/movio_server_weather.log"

with open(log_path, "a") as log:
    process = subprocess.Popen(
        ["/usr/bin/python3", "-u", str(server)],
        cwd=str(server.parent),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("New MOVIO server PID:", process.pid)

# Verify the weather integration is still active.
for _ in range(40):
    if process.poll() is not None:
        raise RuntimeError("Server exited. Check " + log_path)

    try:
        r = requests.get(
            "http://127.0.0.1:8080/api/weather-booking/optional-health",
            timeout=2
        )
        if r.status_code == 200:
            print("SERVER READY:", r.json())
            print("RESTART SUCCESSFUL")
            break
    except requests.RequestException:
        pass

    time.sleep(1)
else:
    raise RuntimeError("Server did not become ready")

Current MOVIO server PIDs: [635]
Stopped old server: 635
New MOVIO server PID: 643
SERVER READY: {'ok': True, 'version': 'optional-weather-1.0', 'weather_optional': True, 'final_confirmation_required': True}
RESTART SUCCESSFUL


## Dashboard Fix

In [103]:
from pathlib import Path
import ast
import shutil
import subprocess
import sys
from datetime import datetime

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
path = root / "integrations/movio_weather_guard.py"

original = path.read_text(encoding="utf-8")

old = '''                    await send({"event": "processing", "epoch": epoch})
                    task = asyncio.create_task(process(msg, epoch, copy.deepcopy(state)))
                    tasks.add(task)
                    task.add_done_callback(tasks.discard)'''

new = '''                    await send({"event": "processing", "epoch": epoch})
                    # Serialize dialogue updates so each turn uses the
                    # latest committed booking state.
                    await process(msg, epoch, copy.deepcopy(state))'''

assert original.count(old) == 1, (
    "Expected live dialogue block not found exactly once. "
    "No changes made."
)

updated = original.replace(old, new, 1)
ast.parse(updated, filename=str(path))

backup = path.with_name(
    path.name + ".before_dialogue_fix_" +
    datetime.now().strftime("%Y%m%d_%H%M%S") + ".bak"
)

shutil.copy2(path, backup)

try:
    path.write_text(updated, encoding="utf-8")

    tests = [
        "test_repeated_details_do_not_trigger_missing_slot_questions",
        "test_all_six_slots_in_one_english_utterance",
        "test_past_today_is_corrected_without_losing_other_slots",
        "test_exact_tamil_conversation_preserves_every_slot",
    ]

    expression = " or ".join(tests)

    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            str(root / "integrations/test_weather_guard.py"),
            "-k", expression,
            "-q", "--tb=short"
        ],
        cwd=str(root),
        capture_output=True,
        text=True,
        timeout=120
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr[-2000:])

    if result.returncode != 0:
        raise RuntimeError("Regression tests failed")

    print("FIX 2 PATCH INSTALLED")
    print("Backup:", backup)
    print("Existing weather and booking logic preserved.")

except Exception:
    shutil.copy2(backup, path)
    print("Original file restored.")
    raise

....                                                                     [100%]
4 passed, 94 deselected in 0.19s

FIX 2 PATCH INSTALLED
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_weather_guard.py.before_dialogue_fix_20261028_070709.bak
Existing weather and booking logic preserved.


In [104]:
from pathlib import Path
import ast
import subprocess
import sys

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
path = root / "integrations/movio_weather_guard.py"

current = path.read_text(encoding="utf-8")

patched = '''                    await send({"event": "processing", "epoch": epoch})
                    # Serialize dialogue updates so each turn uses the
                    # latest committed booking state.
                    await process(msg, epoch, copy.deepcopy(state))'''

original = '''                    await send({"event": "processing", "epoch": epoch})
                    task = asyncio.create_task(process(msg, epoch, copy.deepcopy(state)))
                    tasks.add(task)
                    task.add_done_callback(tasks.discard)'''

assert current.count(patched) == 1, (
    "Expected patch not found exactly once. No changes made."
)

restored = current.replace(patched, original, 1)
ast.parse(restored, filename=str(path))

path.write_text(restored, encoding="utf-8")

assert original in path.read_text(encoding="utf-8")

print("Previous Fix 2 patch reverted.")
print("Original asynchronous interruption handling restored.")

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        str(root / "integrations/test_weather_guard.py"),
        "-q", "--tb=short"
    ],
    cwd=str(root),
    capture_output=True,
    text=True,
    timeout=180
)

print(result.stdout[-5500:])

if result.stderr:
    print("STDERR:", result.stderr[-1000:])

print("EXIT CODE:", result.returncode)

Previous Fix 2 patch reverted.
Original asynchronous interruption handling restored.
()
           ^^^^^^^^^^^^^^^
integrations/test_weather_guard.py:384: in scenario
    self.assertTrue(first['streaming']);self.assertEqual(first['chunk_index'],0);self.assertGreater(first['chunk_count'],2)
                                                                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E   AssertionError: 1 not greater than 2
------------------------------ Captured log call -------------------------------
ERROR    asyncio:base_events.py:1890 unhandled exception during asyncio.run() shutdown
task: <Task finished name='Task-19' coro=<install_socket.<locals>.wrapped() done, defined at /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_confirmation_flow.py:189> exception=OperationalError('unable to open database file')>
Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~

In [105]:
from pathlib import Path
import ast
import shutil
import sys
from datetime import datetime

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
path = root / "integrations/movio_weather_guard.py"

source = path.read_text(encoding="utf-8")

old = """    if route:places={'pickup':route[1],'destination':route[2]}"""

new = """    if route:
        destination = route[2]
        # Stop destination capture before unrelated booking details.
        destination = re.split(
            r'(?i)\\s+(?:and\\s+)?(?:'
            r'i\\s+am\\b|i\\s+need\\b|we\\s+are\\b|'
            r'we\\s+need\\b|there\\s+are\\b|'
            r'\\d+\\s+(?:people|persons?|passengers?)\\b|'
            r'book\\s+(?:a\\s+)?cab\\b|'
            r'today\\b|tomorrow\\b'
            r')',
            destination,
            maxsplit=1
        )[0].strip(' ,.;')
        places={'pickup':route[1],'destination':destination}"""

assert source.count(old) == 1, "Expected code not found. Nothing changed."

updated = source.replace(old, new, 1)
ast.parse(updated, filename=str(path))

backup = path.with_name(
    path.name + ".before_destination_fix_" +
    datetime.now().strftime("%Y%m%d_%H%M%S") + ".bak"
)

shutil.copy2(path, backup)

try:
    path.write_text(updated, encoding="utf-8")

    sys.path.insert(0, str(root))
    from integrations import movio_weather_guard as wx

    # Load the updated module directly for verification.
    import importlib
    wx = importlib.reload(wx)

    class TestBridge:
        def clean_place(self, text):
            return text.strip()

        def number(self, text):
            return 1 if text.lower() == "one" else None

    state = {
        "lang": "English",
        "mode": "book",
        "stage": "pickup",
        "count": None,
        "vehicle": None,
        "pickup": None,
        "destination": None,
        "_wx_slot_conflicts": {}
    }

    message = (
        "I need to book a cab from Chennai Central to Kottivakkam "
        "and I am one people so I need bike. "
        "Book a cab today at 3 PM."
    )

    result = wx.collect_details(TestBridge(), state, message)

    print("Extracted:", result["updates"])

    assert state["count"] == 1
    assert state["vehicle"] == "Bike"
    assert state["pickup"] == "Chennai Central"
    assert state["destination"] == "Kottivakkam"

    print("\nDESTINATION FIX PASSED")
    print("Backup:", backup)

except Exception:
    shutil.copy2(backup, path)
    print("Original file restored.")
    raise

Extracted: {'count': 1, 'vehicle': 'Bike', 'pickup': 'Chennai Central', 'destination': 'Kottivakkam'}

DESTINATION FIX PASSED
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_weather_guard.py.before_destination_fix_20261028_070717.bak


In [106]:
from pathlib import Path
import subprocess
import sys

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")

tests = [
    "test_repeated_details_do_not_trigger_missing_slot_questions",
    "test_all_six_slots_in_one_english_utterance",
    "test_exact_tamil_conversation_preserves_every_slot",
    "test_all_six_slots_in_tanglish",
    "test_qwen_cannot_invent_places_or_overwrite_valid_slots",
    "test_past_today_is_corrected_without_losing_other_slots",
]

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        str(root / "integrations/test_weather_guard.py"),
        "-k", " or ".join(tests),
        "-q", "--tb=short"
    ],
    cwd=str(root),
    capture_output=True,
    text=True,
    timeout=120
)

print(result.stdout[-5000:])
if result.stderr:
    print(result.stderr[-1000:])

print("EXIT CODE:", result.returncode)
print("PASS" if result.returncode == 0 else "NEEDS FIXING")

......                                                                   [100%]
6 passed, 92 deselected in 0.14s

EXIT CODE: 0
PASS


In [107]:
import os
import sys
import time
import subprocess
from pathlib import Path

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
server = root / "server.py"

assert server.exists(), f"Server not found: {server}"

# Stop only processes running this project's server.py
for proc in Path("/proc").iterdir():
    if not proc.name.isdigit() or int(proc.name) == os.getpid():
        continue
    try:
        cmd = (proc / "cmdline").read_bytes().replace(b"\x00", b" ").decode(errors="ignore")
        if "python" in cmd.lower() and str(server) in cmd:
            os.kill(int(proc.name), 15)
            print("Stopped old server PID:", proc.name)
    except (OSError, ProcessLookupError):
        pass

time.sleep(2)

log = open("/kaggle/working/movio_server.log", "a")

process = subprocess.Popen(
    [sys.executable, str(server)],
    cwd=str(root),
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Started server PID:", process.pid)
time.sleep(5)

print("Server running:", process.poll() is None)
print("Log:", "/kaggle/working/movio_server.log")

Stopped old server PID: 643
Started server PID: 670
Server running: True
Log: /kaggle/working/movio_server.log


In [108]:
from pathlib import Path
from datetime import datetime
import re
import shutil
import subprocess

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
path = root / "web/weather_manual.js"

source = path.read_text(encoding="utf-8")

pattern = (
    r"uncertain\s*=\s*false\s*;\s*"
    r"review\s*=\s*null\s*;\s*"
    r"el\(['\"]wxManualReview['\"]\)\.hidden\s*=\s*true\s*;\s*"
    r"rideId\s*=\s*result\.id\s*;\s*"
    r"status\(result\.message\)\s*;\s*"
    r"toast\(result\.message\)\s*;"
)

matches = list(re.finditer(pattern, source))
assert len(matches) == 1, (
    f"Found {len(matches)} matching success handlers; expected 1. "
    "No changes made."
)

replacement = """
    uncertain=false;
    review=null;
    el('wxManualReview').hidden=true;
    rideId=result.id;

    // Clear completed ride details after confirmed booking.
    serial++;
    pickup=null;
    destination=null;
    km=0;
    minutes=0;
    routeData=null;
    selecting='pickup';

    el('pickup').value='';
    el('destination').value='';

    const suggestions=el('suggestions');
    if(suggestions)suggestions.replaceChildren();

    if(routeLine){
      map.removeLayer(routeLine);
      routeLine=null;
    }

    for(const key of ['pickup','destination']){
      if(markers[key]){
        map.removeLayer(markers[key]);
        delete markers[key];
      }
    }

    vehicle='Mini';
    document.querySelectorAll('.vehicle').forEach(node=>{
      node.classList.toggle('active',node.dataset.type==='Mini');
    });

    countBox.value='1';
    el('wxPickupDay').value=indiaDay();
    el('wxPickupDay').min=indiaDay();
    el('wxPickupTime').value='';
    dateBox.value='';
    syncDayButtons();

    for(const id of [
      'wxPickupName','wxDropName','wxTripPickup','wxTripDrop',
      'wxTripDate','wxShortAdvice','wxManualAdvice','wxManualSummary'
    ]){
      const node=el(id);
      if(node)node.textContent='';
    }

    el('wxTripFacts').replaceChildren();

    const routeInfo=el('routeInfo');
    if(routeInfo)routeInfo.textContent=
      'Select pickup and destination to see a route estimate.';

    fares();

    // Keep booking confirmation visible.
    status(result.message);
    toast(result.message);
"""

updated = (
    source[:matches[0].start()]
    + replacement
    + source[matches[0].end():]
)

backup = path.with_name(
    path.name + ".before_reset_" +
    datetime.now().strftime("%Y%m%d_%H%M%S") + ".bak"
)

shutil.copy2(path, backup)

try:
    path.write_text(updated, encoding="utf-8")

    node = shutil.which("node")
    if node:
        check = subprocess.run(
            [node, "--check", str(path)],
            capture_output=True,
            text=True
        )
        if check.returncode != 0:
            raise RuntimeError(check.stderr)

    print("RESET PATCH INSTALLED SUCCESSFULLY")
    print("Backup:", backup)
    print("JavaScript syntax:", "PASS" if node else "Node.js unavailable")
    print("Success message preserved.")
    print("AI Call and backend unchanged.")

except Exception:
    shutil.copy2(backup, path)
    print("Patch failed; original file restored.")
    raise

RESET PATCH INSTALLED SUCCESSFULLY
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/web/weather_manual.js.before_reset_20261028_070726.bak
JavaScript syntax: PASS
Success message preserved.
AI Call and backend unchanged.


In [109]:
# Run this entire file in ONE cell AFTER the existing setup/fixes and BEFORE evaluation.
# It adds targeted repairs to movio-with-weather-api (7).ipynb; no notebook replacement.
def movio_enable_booking_reliability():
    import ast
    from datetime import datetime,timedelta
    import hashlib
    import json
    import os
    from pathlib import Path
    import re
    import shutil
    import subprocess
    import sys
    import tempfile
    import types
    import weakref
    import psutil
    import requests
    import time

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    web=root/'web';server_path=root/'server.py';guard_path=root/'integrations/movio_weather_guard.py'
    manual_path=web/'weather_manual.js';index_path=web/'index.html';helper_path=root/'integrations/movio_booking_reliability.py'
    for path in (server_path,guard_path,manual_path,index_path,root/'integrations/movio_required_weather.py'):
        if not path.is_file():raise FileNotFoundError('Run the existing notebook setup first: '+str(path))
    source=r'''"""Small repairs for the existing MOVIO booking implementation."""
import copy
import importlib.util
import json
from pathlib import Path
import re
import sys
import threading
from urllib.parse import urlparse

VERSION='booking-reliability-1.0'
_providers={};_provider_lock=threading.RLock()


def provider_module(root):
    path=Path(root)/'integrations/movio_required_weather.py'
    key=str(path.resolve())
    with _provider_lock:
        if key not in _providers:
            spec=importlib.util.spec_from_file_location('_movio_manual_forecast_provider',path)
            module=importlib.util.module_from_spec(spec)
            exec(compile(path.read_text(encoding='utf-8'),str(path),'exec'),module.__dict__)
            if not callable(getattr(module,'complete',None)) or not callable(getattr(module,'Forecast',None)):
                raise RuntimeError('Run the required-spoken-weather cell before this repair.')
            _providers[key]=module
        return _providers[key]


def forecast_error(weather,lang):
    reason=weather.get('reason') or weather.get('status') or 'forecast_request_failed'
    status=weather.get('http_status')
    if reason in ('outside_forecast_window','outside_forecast_range'):
        messages=('This date is outside the weather forecast window. Your selected date is kept.',
            'இந்தத் தேதிக்கான வானிலை இன்னும் வெளியிடப்படவில்லை. நீங்கள் தேர்ந்தெடுத்த தேதி உள்ளது.',
            'Indha date-ku forecast innum varala. Neenga select panna date irukku.')
    elif status==429 or reason=='rate_limit_backoff':
        messages=('The weather service asked us to wait. Please try Review ride again shortly.',
            'வானிலை சேவை சிறிது நேரம் காத்திருக்கச் சொல்கிறது. பின்னர் பயணத்தை மீண்டும் சரிபார்க்கவும்.',
            'Weather service konjam wait panna solludhu. Konja nerathula Ride review retry pannunga.')
    else:
        messages=('The weather check did not finish. Your ride details are kept. Press Review ride to retry.',
            'வானிலைச் சரிபார்ப்பு முடியவில்லை. உங்கள் பயண விவரங்கள் உள்ளன. மீண்டும் பயணத்தைச் சரிபார்க்கவும்.',
            'Weather check mudiyala. Unga ride details irukku. Ride review press panni retry pannunga.')
    return dict(reason=reason,http_status=status,message=messages[('English','Tamil','Tanglish').index(lang) if lang in ('English','Tamil','Tanglish') else 0])


def install_namespace(engine,root):
    root=Path(root)
    if engine.get('_MOVIO_BOOKING_RELIABILITY_VERSION')==VERSION:
        return dict(version=VERSION,already_installed=True)
    required=('time_components','update_time_memory','manual_state','install_backend')
    if any(not callable(engine.get(key)) for key in required):
        raise RuntimeError('The current weather/booking engine is not initialized.')
    provider=provider_module(root)
    previous={key:engine[key] for key in required}
    lexicon=engine['number_lexicon'](59)
    words=[word for word,value in lexicon.items() if not word.isdigit() and 0<=value<=23]
    number=r'(?<![\w\u0b80-\u0bff])(?:\d{1,2}(?::\d{2})?|'+'|'.join(re.escape(word) for word in sorted(words,key=len,reverse=True))+r')(?![\w\u0b80-\u0bff])'
    before=re.compile(number+r'\s*(?:(?:o[.\s\x27’]?clock|மணி(?:க்கு)?|manikku|mani)\s*)?$',re.I)
    marker=re.compile(r'(?<![a-z])([ap])\s*\.?\s*m\.?(?![a-z])',re.I)
    alone=re.compile(r"^\s*(?:(?:it is|it's|its|i mean|make it|use)\s+)?[ap]\s*\.?\s*m\.?\s*(?:please)?\s*[.!?]?\s*$",re.I)
    dayparts=re.compile(r'காலை|அதிகாலை|மதியம்|நண்பகல்|மாலை|நள்ளிரவு|இரவு|\b(?:morning|afternoon|evening|night|tonight|noon|midnight|kaalai|kalai|adhikaalai|madhiyam|mathiyam|maalai|malai|iravu|nalliravu)\b',re.I)
    def normalize_periods(raw,allow_bare=False):
        text=engine['unicodedata'].normalize('NFKC',str(raw or '')).translate(str.maketrans('௦௧௨௩௪௫௬௭௮௯','0123456789'))
        standalone=bool(allow_bare and alone.fullmatch(text))
        def replace(match):
            # A clock suffix, or a short AM/PM clarification, is a period.
            # The verb in "I am one person" is ordinary sentence text.
            return match[1].lower()+'m' if standalone or before.search(text[:match.start()]) else ' '*len(match[0])
        return marker.sub(replace,text)
    def time_components(raw,now,allow_bare=False):
        return previous['time_components'](normalize_periods(raw,allow_bare),now,allow_bare=allow_bare)
    def update_time_memory(state,text,now):
        try:return previous['update_time_memory'](state,text,now)
        except engine['GuardError'] as exc:
            # Keep independently valid date/hour evidence when a real period
            # conflict needs clarification. Never choose one conflicting period.
            if str(exc)=='clarify_period':
                cleaned=dayparts.sub(' ',marker.sub(' ',str(text or '')))
                try:partial=previous['time_components'](cleaned,now,allow_bare=state.get('stage') in ('time','time_period'))
                except engine['GuardError']:partial={}
                parts=copy.deepcopy(state.get('_wx_time') or {})
                for key in ('date','hour','minute','clock24'):
                    if key in partial:parts[key]=partial[key]
                if 'hour' in partial:
                    parts.pop('period',None);parts.pop('day_period',None)
                state['_wx_time']=parts
            state['_wx_time_error']=str(exc)
            return None,str(exc)
    proxies={};lock=threading.RLock()
    def proxy(weather):
        with lock:
            item=proxies.get(id(weather))
            if item is None or item.base is not weather:
                item=provider.Forecast(weather,engine,root);proxies[id(weather)]=item
            return item
    def manual_state(data,weather,now=None):
        state=previous['manual_state'](data,proxy(weather),now=now)
        if not provider.complete(state.get('weather')):
            state['weather']=dict(state.get('weather') or {})
            state['weather'].pop('_optional_skip',None)
        return state
    def install_backend(module,weather=None):
        result=previous['install_backend'](module,weather=weather)
        install_http(module,engine,root)
        return result
    engine.update(time_components=time_components,update_time_memory=update_time_memory,
        manual_state=manual_state,install_backend=install_backend,
        _MOVIO_BOOKING_RELIABILITY_VERSION=VERSION,
        _movio_reliable_manual_forecast=proxy,_movio_reliable_provider=provider)
    return dict(version=VERSION)


def install_http(module,engine,root):
    if getattr(module.Handler,'_movio_booking_reliability',None)==VERSION:return
    old_get=module.Handler.do_GET;old_post=module.Handler.do_POST
    def get(h):
        if urlparse(h.path).path=='/api/weather-booking/reliability-health':
            return h.json(200,dict(ok=True,version=VERSION,manual_forecast='validated_hourly_pickup_and_drop',time_parser='clock_bound_meridiem'))
        return old_get(h)
    def post(h):
        if urlparse(h.path).path!='/api/weather-booking/manual-preview':return old_post(h)
        send=h.json;had_json='json' in h.__dict__;instance_json=h.__dict__.get('json')
        def json_reply(status,obj,*args,**kwargs):
            if status==200 and isinstance(obj,dict) and obj.get('ok') and obj.get('call_id'):
                with module.conn() as c:
                    row=c.execute('SELECT state FROM wx_calls WHERE id=?',(obj['call_id'],)).fetchone()
                state=json.loads(row['state']) if row else {}
                weather=state.get('weather') or {}
                obj=dict(obj,weather=weather,weather_status=weather.get('status'))
                if not engine['_movio_reliable_provider'].complete(weather):
                    obj['forecast_error']=forecast_error(weather,state.get('lang','English'))
            return send(status,obj,*args,**kwargs)
        h.json=json_reply
        try:return old_post(h)
        finally:
            if had_json:h.json=instance_json
            else:del h.json
    module.Handler.do_GET=get;module.Handler.do_POST=post
    module.Handler._movio_booking_reliability=VERSION


RESET_JS="""
   // MOVIO_VERIFIED_BOOKING_RESET_V1
   if(!result.ok||!Number.isInteger(Number(result.id))||Number(result.id)<1){
    const error=Error(words().unknown);error.http=502;throw error;
   }
   uncertain=false;review=null;el('wxManualReview').hidden=true;rideId=Number(result.id);serial++;
   // Abort only draft lookups. Preserve saved rides, driver tracking and OTPs.
   const draft=window.MOVIOCustomerBookingUI?.state;
   if(draft){clearTimeout(draft.previewTimer);draft.routeController?.abort();draft.routeSeq++;draft.searchSeq++;
    draft.routeKey=null;draft.routePromise=null;draft.lastPreview=null;draft.field='pickup';}
   if(typeof searchTimer!=='undefined')clearTimeout(searchTimer);
   pickup=null;destination=null;km=0;minutes=0;routeData=null;selecting='pickup';
   for(const id of ['pickup','destination']){const node=el(id);if(node){node.value='';node.removeAttribute('title');}}
   el('suggestions')?.replaceChildren();
   if(routeLine){try{map.removeLayer(routeLine);}catch{}routeLine=null;}
   if(typeof markers!=='undefined')for(const key of ['pickup','destination']){
    if(markers[key]){try{map.removeLayer(markers[key]);}catch{}delete markers[key];}
   }
   vehicle='Mini';countBox.value='1';
   document.querySelectorAll('.vehicle').forEach(node=>node.classList.toggle('active',node.dataset.type==='Mini'));
   dateBox.value='';
   for(const id of ['wxPickupDay','wxPickupTime'])if(el(id))el(id).value='';
   if(el('wxPickupDay'))el('wxPickupDay').min=indiaDay();syncDayButtons();
   for(const id of ['wxPickupName','wxDropName','wxTripPickup','wxTripDrop','wxTripDate','wxShortAdvice','wxManualAdvice','wxManualSummary','wxPickupRain','wxDropRain','wxCondition']){
    const node=el(id);if(node){node.textContent='';node.removeAttribute('title');}
   }
   el('wxTripFacts')?.replaceChildren();
   if(el('routeInfo'))el('routeInfo').textContent=langBox.value==='Tamil'?'புறப்படும் இடத்தையும் சேரும் இடத்தையும் தேர்ந்தெடுக்கவும்.':langBox.value==='Tanglish'?'Pickup, destination select pannunga.':'Choose pickup and destination for a new ride.';
   try{fares();}catch{}status(result.message);toast(result.message);
"""


def patch_manual(source):
    anchor="const result=await request('manual-confirm',review.binding,true);"
    if source.count(anchor)!=1:raise RuntimeError('Expected one manual-confirm success handler; no files changed.')
    start=source.index(anchor)+len(anchor)
    end=source.find('// Refresh failures must not turn a successful save into a failed booking.',start)
    if end<0:end=source.find('await Promise.allSettled([refreshRides(),renderVoiceHistory()]);',start)
    if end<0:raise RuntimeError('Could not identify the manual booking refresh boundary; no files changed.')
    source=source[:start]+'\n'+RESET_JS+'   '+source[end:]
    old='timer=setTimeout(()=>controller.abort(),15000)'
    new="timer=setTimeout(()=>controller.abort(),path==='manual-preview'?45000:15000)"
    if old in source:source=source.replace(old,new,1)
    # A pending forecast cannot be presented as a completed weather review.
    start_marker='// MOVIO_MANUAL_FORECAST_VALIDATION_V1'
    if start_marker not in source:
        anchor="if(ticket!==serial||captured!==fingerprint()){discard({binding:result});status(words().changed);return;}"
        if source.count(anchor)!=1:raise RuntimeError('Manual preview revision check was not found; no files changed.')
        check="""
   // MOVIO_MANUAL_FORECAST_VALIDATION_V1
   if(result.weather){
    const w=result.weather,points=w.points||[];
    const complete=w.status==='ok'&&['pickup','destination'].every(role=>{
     const p=points.find(row=>row.location===role),n=p?.precipitation_probability;
     return n!==null&&n!==undefined&&Number.isFinite(Number(n))&&Number(n)>=0&&Number(n)<=100;
    });
    if(!complete){discard({binding:result});const error=Error(result.forecast_error?.message||words().failed);error.http=503;throw error;}
   }
"""
        source=source.replace(anchor,anchor+check,1)
    return source
'''
    active=types.ModuleType('movio_booking_reliability');active.__file__=str(helper_path)
    exec(compile(source,str(helper_path),'exec'),active.__dict__)
    marker='# MOVIO_BOOKING_RELIABILITY_BOOTSTRAP_V1';end_marker='# MOVIO_BOOKING_RELIABILITY_BOOTSTRAP_END_V1'
    def without_bootstrap(text):
        if marker not in text:return text
        start=text.index(marker);end=text.find(end_marker,start)
        if end<0:raise RuntimeError('Incomplete earlier reliability setup; no files changed.')
        return text[:start]+text[end+len(end_marker):]
    guard=without_bootstrap(guard_path.read_text(encoding='utf-8'))
    manual=active.patch_manual(manual_path.read_text(encoding='utf-8'))
    node=shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory(prefix='movio_reset_js_') as temporary:
            check=Path(temporary)/'weather_manual.js';check.write_text(manual,encoding='utf-8')
            result=subprocess.run([node,'--check',str(check)],capture_output=True,text=True)
            if result.returncode:raise RuntimeError(result.stderr[:1000])
    # Isolated parser checks use no network, audio generation or booking writes.
    sandbox=types.ModuleType('_movio_booking_reliability_check');sandbox.__file__=str(guard_path)
    exec(compile(guard,str(guard_path),'exec'),sandbox.__dict__)
    with tempfile.TemporaryDirectory(prefix='movio_clock_checks_') as temporary:
        # The provider helper is loaded from the existing integration folder;
        # test instances never request a forecast.
        active.install_namespace(sandbox.__dict__,root)
        now=datetime(2026,10,18,14,tzinfo=sandbox.TZ)
        full='I need to book a cab from Chennai Central to Kottivakkam and I am one people so I need bike. Book a cab today at 3 PM.'
        state={'stage':'count'};value,error=sandbox.update_time_memory(state,full,now)
        if error or value.hour!=15 or value.date()!=now.date():raise RuntimeError('The reported I am / 3 PM regression did not pass.')
        state={'stage':'time'};value,error=sandbox.update_time_memory(state,'today at 3 AM or 3 PM',now)
        if error!='clarify_period' or state.get('_wx_time',{}).get('date')!='2026-10-18':raise RuntimeError('Partial date memory did not pass.')
        value,error=sandbox.update_time_memory(state,'Its afternoon three',now)
        if error or value.hour!=15:raise RuntimeError('Afternoon clarification did not retain the date.')

    guard_bootstrap="""
# MOVIO_BOOKING_RELIABILITY_BOOTSTRAP_V1
import importlib.util as _br_import
_br_path=Path(__file__).with_name('movio_booking_reliability.py')
_br_spec=_br_import.spec_from_file_location('movio_booking_reliability',_br_path)
_br_module=_br_import.module_from_spec(_br_spec)
exec(compile(_br_path.read_text(encoding='utf-8'),str(_br_path),'exec'),_br_module.__dict__)
_br_module.install_namespace(globals(),Path(__file__).resolve().parent.parent)
# MOVIO_BOOKING_RELIABILITY_BOOTSTRAP_END_V1
"""
    guard=guard.rstrip()+'\n'+guard_bootstrap
    compile(guard,str(guard_path),'exec')
    server=without_bootstrap(server_path.read_text(encoding='utf-8'))
    tree=ast.parse(server)
    launchers=[item for item in tree.body if isinstance(item,ast.If)
        and any(isinstance(x,ast.Name) and x.id=='__name__' for x in ast.walk(item.test))
        and any(isinstance(x,ast.Constant) and x.value=='__main__' for x in ast.walk(item.test))
        and any(isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute) and call.func.attr=='serve_forever'
            for statement in item.body for call in ast.walk(statement))]
    if len(launchers)!=1:raise RuntimeError('Could not identify one website HTTP launch block; no files changed.')
    server_bootstrap="""# MOVIO_BOOKING_RELIABILITY_BOOTSTRAP_V1
if __name__ == '__main__':
    import importlib.util as _br_import
    import sys as _br_sys
    _br_path=ROOT/'integrations/movio_booking_reliability.py'
    _br_spec=_br_import.spec_from_file_location('movio_booking_reliability',_br_path)
    _br_module=_br_import.module_from_spec(_br_spec)
    exec(compile(_br_path.read_text(encoding='utf-8'),str(_br_path),'exec'),_br_module.__dict__)
    _br_engine=getattr(globals().get('_wx_mod'),'__dict__',None)
    if not isinstance(_br_engine,dict) or not callable(_br_engine.get('manual_state')):
        _br_engine=Handler.do_POST.__globals__
    _br_module.install_namespace(_br_engine,ROOT)
    _br_module.install_http(_br_sys.modules[__name__],_br_engine,ROOT)
# MOVIO_BOOKING_RELIABILITY_BOOTSTRAP_END_V1
"""
    lines=server.splitlines(keepends=True);line=launchers[0].lineno-1
    server=''.join(lines[:line]).rstrip()+'\n\n'+server_bootstrap+'\n'+''.join(lines[line:])
    compile(server,str(server_path),'exec')
    html=index_path.read_text(encoding='utf-8')
    revision=hashlib.sha256(manual.encode()).hexdigest()[:12]
    html=re.sub(r'(<script\b[^>]*\bsrc=["\x27])([^"\x27]*\bweather_manual\.js)(?:\?[^"\x27]*)?(["\x27])',
        lambda match:match[1]+match[2]+'?reliability='+revision+match[3],html,flags=re.I)

    # Identify the existing website before modifying or stopping anything.
    health_url='http://127.0.0.1:8080/api/weather-booking/reliability-health'
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet')
        if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    in_kernel=[]
    for module in list(sys.modules.values()):
        if not isinstance(module,types.ModuleType):continue
        values=vars(module);handler=values.get('Handler');known_root=values.get('ROOT')
        if not isinstance(handler,type) or known_root is None or not getattr(handler,'_wx_installed',False):continue
        try:
            if Path(known_root).resolve()==root and callable(getattr(handler,'do_POST',None)):in_kernel.append(module)
        except (TypeError,ValueError,OSError):continue
    if os.getpid() not in listeners:in_kernel=[]
    process_details=[]
    known_launchers=[]
    for value in list(globals().values()):
        if type(value) in weakref.ProxyTypes:continue
        try:
            if isinstance(value,subprocess.Popen) and value.poll() is None:known_launchers.append(value)
        except ReferenceError:continue
    for pid in listeners:
        if pid==os.getpid():
            if not in_kernel:raise RuntimeError('An unrecognized in-kernel service owns port 8080; no process was stopped.')
            continue
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server_path for path in scripts):
            raise RuntimeError('A different application owns port 8080; no process was stopped.')
        # Prefer the notebook's own Popen handle when present. Containers can
        # expose host PIDs in psutil while Popen uses kernel-namespace PIDs.
        controller=next((value for value in known_launchers if isinstance(value.args,(list,tuple))
            and list(value.args)==command),process)
        process_details.append((controller,command,cwd,process.environ()))
    if not listeners:raise RuntimeError('Start the original website on port 8080, then rerun this cell.')

    writes={helper_path:source,guard_path:guard,server_path:server,manual_path:manual,index_path:html}
    changes={path:content for path,content in writes.items() if not path.exists() or path.read_text(encoding='utf-8')!=content}
    backup=None
    if changes:
        backup=root/('booking_reliability_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path in changes:
            if path.exists():
                dest=backup/path.relative_to(root);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(path,dest)
        written=[]
        try:
            for path,content in changes.items():
                temporary=path.with_name(path.name+'.reliability.tmp')
                try:temporary.write_text(content,encoding='utf-8');os.replace(temporary,path)
                finally:temporary.unlink(missing_ok=True)
                written.append(path)
        except Exception:
            for path in reversed(written):
                old=backup/path.relative_to(root)
                if old.exists():shutil.copy2(old,path)
                else:path.unlink(missing_ok=True)
            raise
    b=globals().get('bridge');voice=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if isinstance(voice,dict) and callable(voice.get('time_components')):active.install_namespace(voice,root)
    for backend in in_kernel:
        namespace=getattr(getattr(backend,'_wx_mod',None),'__dict__',None)
        if not isinstance(namespace,dict) or not callable(namespace.get('manual_state')):namespace=backend.Handler.do_POST.__globals__
        active.install_namespace(namespace,root);active.install_http(backend,namespace,root)
    restart=any(path in changes for path in (helper_path,guard_path,server_path))
    for process,command,cwd,environment in process_details:
        if not restart:
            try:
                response=requests.get(health_url,timeout=(1,2))
                if response.status_code==200 and response.json().get('version')==active.VERSION:continue
            except (requests.RequestException,ValueError):pass
        process.terminate();process.wait(timeout=8)
        with (root/'booking_reliability_website.log').open('a',encoding='utf-8') as output:
            globals()['WX_WEBSITE_PROCESS']=subprocess.Popen(command,cwd=cwd,env=environment,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    verified=False
    for attempt in range(16):
        try:
            response=requests.get(health_url,timeout=(1,2))
            verified=response.status_code==200 and response.json().get('version')==active.VERSION
            if verified:break
        except (requests.RequestException,ValueError):pass
        time.sleep(.25)
    if not verified:raise RuntimeError('Website verification failed. Check booking_reliability_website.log; use the original website-only launcher, then rerun this cell.')
    print('Verified-save reset enabled: addresses, date, time, route, weather and previous fare are cleared.')
    print('Language preference and your saved booking list are preserved.')
    print('Manual weather uses the existing validated pickup/drop forecast handler.')
    print('I am is no longer read as AM; valid date/hour details survive clarification.')
    print('Offline date/time checks: PASS | website integration: PASS')
    live=None
    if globals().get('MOVIO_VERIFY_LIVE_WEATHER',True):
        print('Checking the forecast provider with two sample coordinates; no booking is created...',flush=True)
        if not isinstance(voice,dict):voice=sandbox.__dict__
        base=getattr(b,'_wx_weather',None) if b is not None else None
        base=base or voice['WeatherClient']()
        now=datetime.now(voice['TZ']);when=(now+timedelta(days=1)).replace(hour=9,minute=0,second=0,microsecond=0)
        result=voice['_movio_reliable_manual_forecast'](base).forecast([dict(lat=13.0827,lng=80.2707),dict(lat=12.9352,lng=80.2275)],when,now=now)
        live=dict(ready=active.provider_module(root).complete(result),requested_time=when.isoformat(),status=result.get('status'),reason=result.get('reason'),http_status=result.get('http_status'))
        if live['ready']:live['sample_rain_probability_pct']={point['location']:point['precipitation_probability'] for point in result['points']}
        print(json.dumps(live,ensure_ascii=False,indent=2))
        if not live['ready']:print('Provider check did not pass. Details are retained for retry; inspect required_weather.log for the reported cause.')
    print('Refresh the dashboard with Ctrl+Shift+R. On mobile, close and reopen the website tab.')
    print('Close the old AI Call popup and start a NEW call. Do not rerun older overwrite/reset cells afterward.')
    return dict(version=active.VERSION,website_verified=True,voice_updated=isinstance(voice,dict),backup=str(backup) if backup else None,changed_files=[str(path.relative_to(root)) for path in changes],live_forecast_probe=live)

MOVIO_BOOKING_RELIABILITY=movio_enable_booking_reliability()


Verified-save reset enabled: addresses, date, time, route, weather and previous fare are cleared.
Language preference and your saved booking list are preserved.
Manual weather uses the existing validated pickup/drop forecast handler.
I am is no longer read as AM; valid date/hour details survive clarification.
Offline date/time checks: PASS | website integration: PASS
Checking the forecast provider with two sample coordinates; no booking is created...
{
  "ready": true,
  "requested_time": "2026-10-29T09:00:00+05:30",
  "status": "ok",
  "reason": null,
  "http_status": null,
  "sample_rain_probability_pct": {
    "pickup": 0.0,
    "destination": 0.0
  }
}
Refresh the dashboard with Ctrl+Shift+R. On mobile, close and reopen the website tab.
Close the old AI Call popup and start a NEW call. Do not rerun older overwrite/reset cells afterward.


In [110]:
# Run this entire file in ONE cell AFTER the earlier fixes, BEFORE evaluation.
# Existing UI, ASR/TTS models, fares, OTP and tracking code are preserved.
def movio_enable_friendly_weather():
    import ast
    from datetime import datetime
    import os
    from pathlib import Path
    import shutil
    import subprocess
    import sys
    import time
    import types
    import weakref
    import psutil
    import requests

    b=globals().get('bridge');engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if not isinstance(engine,dict) or not all(callable(engine.get(key)) for key in ('prepare_turn','weather_consented','install_bridge')):
        raise RuntimeError('Run the existing voice/weather setup and earlier repair cells first, in this kernel.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    guard_path=root/'integrations/movio_weather_guard.py';server_path=root/'server.py'
    helper_path=root/'integrations/movio_friendly_weather.py'
    for path in (guard_path,server_path):
        if not path.is_file():raise FileNotFoundError(str(path))
    source=r'''"""Conversational weather choices for the existing MOVIO booking protocol."""
import contextvars
import copy
import inspect
import math
from pathlib import Path
import re
import threading
import types
from datetime import datetime

VERSION='friendly-weather-1.0'
TURN=contextvars.ContextVar('movio_friendly_weather_turn',default=None)


def number(value):
    if value is None or isinstance(value,bool):return None
    try:value=float(value)
    except (TypeError,ValueError,OverflowError):return None
    return value if math.isfinite(value) else None


def complete(weather):
    if not isinstance(weather,dict) or weather.get('status')!='ok':return False
    for role in ('pickup','destination'):
        point=next((p for p in weather.get('points',[]) if isinstance(p,dict) and p.get('location')==role),{})
        chance=number(point.get('precipitation_probability'));code=number(point.get('weather_code'))
        if chance is None or not 0<=chance<=100 or code is None:return False
    return True


def category(weather):
    if not complete(weather):return 'missing'
    points=weather.get('points',[])
    if weather.get('thunderstorm') or any(p.get('weather_code') in (95,96,99) for p in points):return 'storm'
    if weather.get('heavy_rain'):return 'heavy'
    if weather.get('heat'):return 'heat'
    # Wind advice is enabled only when actual wind fields exist in the forecast.
    if any((number(p.get('wind_speed_10m')) or 0)>=40 or (number(p.get('wind_gusts_10m')) or 0)>=50 for p in points):return 'wind'
    if weather.get('condition')=='rain':return 'rain'
    return 'normal'


def scope(state,engine):
    weather=copy.deepcopy(state.get('weather') or {});weather.pop('_friendly_notice_policy',None)
    values={k:state.get(k) for k in ('pickup','destination','pickup_geo','destination_geo','pickup_at','vehicle','count')}
    values['weather']=weather
    return engine['digest'](values)


def exemption(state,engine):
    weather=state.get('weather') or {};policy=weather.get('_friendly_notice_policy') or {}
    reason='forecast_missing' if category(weather)=='missing' else 'no_weather_choice_needed' if category(weather)=='normal' else None
    return bool(reason and policy.get('version')==VERSION and policy.get('reason')==reason and policy.get('scope')==scope(state,engine))


def pick(state,en,ta,roman):return ta if state.get('lang')=='Tamil' else roman if state.get('lang')=='Tanglish' else en


def fmt(value):return f'{float(value):g}'


def report(state):
    weather=state.get('weather') or {};kind=category(weather)
    if kind=='missing':return ''
    points={p['location']:p for p in weather['points']};a=fmt(points['pickup']['precipitation_probability']);b=fmt(points['destination']['precipitation_probability'])
    pickup=state.get('pickup','your pickup');drop=state.get('destination','your destination');v=state.get('vehicle','vehicle')
    ta_vehicle={'Bike':'பைக்','Auto':'ஆட்டோ','Mini':'மினி கார்','Sedan':'செடான்','SUV':'எஸ் யூ வி'}.get(v,v)
    if kind in ('rain','heavy','storm'):
        subject=pick(state,'Thunderstorms are forecast' if kind=='storm' else 'Heavy rain is forecast' if kind=='heavy' else 'Rain may join your journey',
            'இடியுடன் மழை எதிர்பார்க்கப்படுகிறது' if kind=='storm' else 'கனமழை எதிர்பார்க்கப்படுகிறது' if kind=='heavy' else 'பயண நேரத்தில் மழை பெய்ய வாய்ப்புள்ளது',
            'Thunderstorms expect pannrom' if kind=='storm' else 'Heavy rain expect pannrom' if kind=='heavy' else 'Journey time-la rain chance irukku')
        first=pick(state,f'A quick heads-up: {subject.lower()}, with a {a}% rain chance at {pickup} and {b}% at {drop} around pickup time.',
            f'ஒரு சின்ன தகவல்: {subject}; புறப்படும் நேரத்தில் {pickup} பகுதியில் {a}%, {drop} பகுதியில் {b}% மழை வாய்ப்பு உள்ளது.',
            f'Oru chinna update: {subject}; pickup time-la {pickup}-la {a}%, {drop}-la {b}% rain chance irukku.')
        if v=='Bike':tip=pick(state,'A covered car would offer better rain protection; you can keep your bike if you prefer.',
            'மழையிலிருந்து பாதுகாப்புக்கு மூடிய காரைப் பரிசீலிக்கலாம்; விரும்பினால் பைக்கிலேயே தொடரலாம்.',
            'Rain protection-ku covered car consider pannalaam; unga bike-aiyum keep pannalaam.')
        elif v=='Auto':tip=pick(state,'An auto has open sides, so keep an umbrella and protect your bags.',
            'ஆட்டோவின் பக்கவாட்டில் மழை படலாம்; குடை எடுத்துக்கொண்டு பைகளையும் பாதுகாத்துக்கொள்ளுங்கள்.',
            'Auto side-la rain padalaam; umbrella eduthuttu bags-ai protect pannikkonga.')
        else:tip=pick(state,f'Your {v} provides cover; an umbrella will help when getting in and out.',
            f'{ta_vehicle} பயணத்தில் மழையிலிருந்து பாதுகாப்பு கிடைக்கும்; ஏறவும் இறங்கவும் குடை எடுத்துக்கொள்ளுங்கள்.',
            f'Unga {v}-la cover irukkum; eravum irangavum umbrella eduthukkonga.')
        return first+' '+tip
    temperature=number(points['pickup'].get('temperature_2m'))
    if kind=='heat':
        t=f' about {fmt(temperature)}°C' if temperature is not None else ' hot'
        return pick(state,f'It is expected to be{t} around pickup time. Please carry water and prepare for the heat.',
            ('புறப்படும் நேரத்தில் சுமார் '+fmt(temperature)+'°C வெப்பநிலை எதிர்பார்க்கப்படுகிறது.' if temperature is not None else 'புறப்படும் நேரத்தில் வெப்பம் அதிகமாக இருக்கும்.')+' குடிநீர் எடுத்துக்கொண்டு வெப்பத்திற்கு ஏற்ற முன்னேற்பாடுகளைச் செய்துகொள்ளுங்கள்.',
            ('Pickup time-la around '+fmt(temperature)+'°C expect pannrom.' if temperature is not None else 'Pickup time-la heat adhigamaaga irukkum.')+' Water eduthuttu heat-ku prepare pannikkonga.')
    if kind=='wind':return pick(state,'Strong winds are expected around pickup time. For a bike ride, you may prefer a covered car.' if v=='Bike' else 'Strong winds are expected around pickup time. Take care when boarding and getting out.',
        'புறப்படும் நேரத்தில் பலத்த காற்று எதிர்பார்க்கப்படுகிறது. பைக் பயணத்திற்கு பதிலாக மூடிய காரைப் பரிசீலிக்கலாம்.' if v=='Bike' else 'புறப்படும் நேரத்தில் பலத்த காற்று எதிர்பார்க்கப்படுகிறது. வாகனத்தில் ஏறவும் இறங்கவும் கவனமாக இருங்கள்.',
        'Pickup time-la strong wind expect pannrom. Bike-ku badhila covered car consider pannalaam.' if v=='Bike' else 'Pickup time-la strong wind expect pannrom. Eravum irangavum careful-a irunga.')
    if weather.get('condition')=='cloudy':return pick(state,'Cloudy skies are expected for your trip. Let me confirm your ride details.',
        'பயண நேரத்தில் மேகமூட்டம் எதிர்பார்க்கப்படுகிறது. உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.',
        'Journey time-la cloudy weather expect pannrom. Unga ride details confirm pannalaam.')
    if temperature is not None and temperature>=33:return pick(state,f'Clear skies and about {fmt(temperature)}°C are expected; carrying water is a good idea.',
        f'வானம் தெளிவாகவும் வெப்பநிலை சுமார் {fmt(temperature)}°C ஆகவும் இருக்கும்; குடிநீர் எடுத்துக்கொள்ளுங்கள்.',
        f'Clear sky, around {fmt(temperature)}°C expect pannrom; water eduthukkonga.')
    if not weather.get('pleasant'):
        return pick(state,'Clear skies are expected for your journey. Let me confirm your ride details.' if weather.get('condition')=='clear' else 'I have checked the forecast for your journey. Let me confirm your ride details.',
            'பயண நேரத்தில் வானம் தெளிவாக இருக்கும் என எதிர்பார்க்கப்படுகிறது. உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.' if weather.get('condition')=='clear' else 'உங்கள் பயணத்திற்கான வானிலையைச் சரிபார்த்துவிட்டேன். பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.',
            'Journey time-la clear sky expect pannrom. Unga ride details confirm pannalaam.' if weather.get('condition')=='clear' else 'Unga journey-ku forecast check pannitten. Ride details confirm pannalaam.')
    return pick(state,'The weather looks pleasant for your journey. Let me confirm your ride details.',
        'நல்ல செய்தி! பயண நேரத்தில் வானிலை நன்றாக இருக்கும் என எதிர்பார்க்கப்படுகிறது. உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.',
        'Nalla news! Journey time-la weather nalla irukkum-nu expect pannrom. Unga ride details confirm pannalaam.')


def question(state):
    if state.get('vehicle')=='Bike' and category(state.get('weather')) in ('rain','heavy','storm','wind'):
        return pick(state,'Would you like to keep your bike or change to a covered car?',
            'பைக்கிலேயே தொடரலாமா, அல்லது மூடிய காருக்கு மாற்ற விரும்புகிறீர்களா?',
            'Bike-aiye keep pannalaama, illa covered car-ku change pannalaama?')
    return pick(state,'Would you like to keep your pickup time or check another time?',
        'அதே பயண நேரத்தில் தொடரலாமா, அல்லது வேறு நேரத்தைச் சரிபார்க்க விரும்புகிறீர்களா?',
        'Same pickup time-la continue pannalaama, illa vera time check pannalaama?')


def keep_choice(text,state,engine):
    norm=engine['_friendly_previous_normalized'](text)
    plain={'keep my booking','keep this booking','keep the same booking','keep my pickup time','keep the pickup time','same time','keep the same time',
        'அதே நேரம்','அதே நேரத்தில் தொடரலாம்','இதே நேரம் போதும்','adhe time','same time venum','idhe time podhum'}
    if norm in plain:return 'time'
    v=state.get('vehicle','').casefold()
    vehicle={f'keep my {v}',f'keep my {v} booking',f'keep the {v}',f'continue with my {v}',f'keep {v}',f'{v} keep pannunga'}
    if v=='bike':vehicle|={'பைக்கிலேயே தொடரலாம்','பைக் போதும்','bike podhum','bike-aiye keep pannunga','bike aiye keep pannunga'}
    if norm in vehicle:return 'vehicle'
    match=re.fullmatch(r'keep\s+(.+)',str(text).strip(),re.I)
    if match:
        words=[word for word,value in engine['number_lexicon'](23).items() if not word.isdigit() and 0<=value<=23]
        clock=r'(?:\d{1,2}(?::\d{2})?|'+'|'.join(re.escape(word) for word in sorted(words,key=len,reverse=True))+r')'
        if not re.fullmatch(r'(?:at\s+)?'+clock+r'\s*(?:a\.?\s*m\.?|p\.?\s*m\.?)\s*(?:please)?[.!?]?',match[1],re.I):return None
        probe=copy.deepcopy(state)
        try:value,error=engine['update_time_memory'](probe,match[1],engine['datetime'].now(engine['TZ']))
        except engine['GuardError']:return None
        if not error and value.isoformat()==state.get('pickup_at'):return 'time'
    return None


class ChoiceSocket:
    def __init__(self,ws,bridge,engine):self.ws,self.bridge,self.engine,self.sid=ws,bridge,engine,None
    def __getattr__(self,name):return getattr(self.ws,name)
    async def receive_json(self):
        message=await self.ws.receive_json()
        if message.get('event')=='start':self.sid=message.get('call_id')
        context=dict(db=self.bridge._wx_db,sid=self.sid,nonce=message.get('summary_nonce'))
        if self.sid:
            try:context['state']=__import__('json').loads(self.bridge._wx_db.row(self.sid)['state'])
            except (ValueError,KeyError,self.engine['GuardError']):pass
        TURN.set(context)
        return message


def install_socket(bridge,engine,_route_wrapper=None):
    previous=getattr(bridge,'_wx_endpoint',None)
    if not callable(previous) or getattr(previous,'_friendly_weather_version',None)==VERSION:return False
    async def wrapped(ws):
        if ws.query_params.get('mode','book')=='cancel':return await previous(ws)
        token=TURN.set(None)
        try:return await previous(ChoiceSocket(ws,bridge,engine))
        finally:TURN.reset(token)
    wrapped=types.FunctionType(wrapped.__code__,engine,wrapped.__name__,wrapped.__defaults__,wrapped.__closure__)
    engine['ChoiceSocket']=ChoiceSocket;engine['TURN']=TURN
    wrapped._friendly_weather_version=VERSION
    if _route_wrapper is None:
        from starlette.routing import websocket_session
        _route_wrapper=websocket_session
    route=next(r for r in bridge.app.routes if getattr(r,'path','')=='/browser-call')
    route.endpoint=wrapped;route.app=_route_wrapper(wrapped);bridge._wx_endpoint=wrapped
    return True


def install_namespace(engine,root=None,bridge=None,_route_wrapper=None):
    if engine.get('_MOVIO_FRIENDLY_WEATHER_VERSION')==VERSION:
        if bridge is not None:install_socket(bridge,engine,_route_wrapper)
        return dict(version=VERSION,already_installed=True)
    root=Path(root or Path(engine['__file__']).resolve().parent.parent)
    keys=('prepare_turn','weather_consented','advisory','weather_question','reply_normalized','install_bridge','install_backend','speech_segments')
    previous={key:engine[key] for key in keys}
    base_prepare=engine.get('_movio_optional_weather_original',{}).get('prepare_turn',previous['prepare_turn'])
    engine['_friendly_previous_normalized']=previous['reply_normalized']
    local=threading.local();spoken=[];lock=threading.RLock()
    try:register=inspect.getclosurevars(engine['completion_message']).nonlocals.get('register')
    except (TypeError,ValueError):register=None
    def advisory(state):
        text=report(state);local.question=question(state);local.lang=state.get('lang')
        if text and callable(register):text=register(text,'weather')
        with lock:
            if text:spoken.append(text)
            if len(spoken)>128:del spoken[:-128]
        return text
    def weather_question(lang):
        return local.question if getattr(local,'lang',None)==lang else engine['localized'](lang,
            'Would you like to keep this ride or change a detail?','இதே பயணத்தைத் தொடரலாமா, அல்லது விவரத்தை மாற்ற விரும்புகிறீர்களா?',
            'Idhe ride continue pannalaama, illa details change pannalaama?')
    def weather_consented(state):return exemption(state,engine) or previous['weather_consented'](state)
    def normalized(text):
        context=TURN.get();state=(context or {}).get('state') or {}
        if state.get('stage')=='weather_advisory':
            choice=keep_choice(text,state,engine)
            if choice:context['choice']=choice;return 'yes'
        return previous['reply_normalized'](text)
    def prepare_turn(b,weather,state,text,now=None):
        context=TURN.get() or {};prior_stage=state.get('stage');ack=False
        # Explicit vehicle/count/location corrections are also weather choices.
        # Reuse the real nonce and playback acknowledgement before applying them.
        if prior_stage=='weather_advisory' and str(text).strip() and context.get('sid'):
            probe=copy.deepcopy(state)
            parsed=engine['collect_details'](b,probe,text)
            if parsed.get('changed'):
                db=context['db'];row=db.row(context['sid'])
                try:state=db.accept_weather(context['sid'],row['epoch'],context.get('nonce'),state);ack=True
                except engine['GuardError']:pass
        norm=previous['reply_normalized'](text)
        if prior_stage=='weather_advisory' and norm in {'earlier','leave earlier','choose an earlier pickup time','change pickup time','முன்னதாக வேண்டும்','வேறு நேரம் வேண்டும்','earlier venum'}:
            state['stage']='weather_change_time';state['dialogue_state']='COLLECTING_DETAILS'
            return state,pick(state,'What pickup time would you prefer? I will check the forecast for that time.',
                'எத்தனை மணிக்கு வாகனம் வேண்டும்? அந்த நேரத்திற்கான வானிலையைச் சரிபார்க்கிறேன்.',
                'Endha pickup time venum? Andha time-ku weather check pannren.')
        state,answer=previous['prepare_turn'](b,weather,state,text,now=now)
        kind=category(state.get('weather'))
        if state.get('stage') in ('weather_advisory','weather_pending') and kind in ('normal','missing'):
            w=state.setdefault('weather',{});w.pop('_friendly_notice_policy',None)
            reason='forecast_missing' if kind=='missing' else 'no_weather_choice_needed'
            w['_friendly_notice_policy']=dict(version=VERSION,reason=reason,scope=scope(state,engine))
            state.pop('_wx_weather_consent',None)
            state,answer=base_prepare(b,weather,state,'',now=now)
            if state.get('stage')=='confirm':
                state['dialogue_state']='AWAITING_BOOKING_CONFIRMATION'
                notice=advisory(state) if kind=='normal' else ''
                return state,(notice+'\n'+answer if notice else answer)
        if state.get('stage')=='confirm' and (context.get('choice') or ack):
            context.pop('choice',None)
            intro=pick(state,'Sure, let me confirm your updated ride details. ',
                'சரி, உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம். ',
                'Seri, unga ride details confirm pannalaam. ')
            return state,intro+answer
        return state,answer
    def speech_segments(text):
        value=str(text)
        with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
        if prefix:
            rest=value[len(prefix):].strip()
            # Keep a coherent weather chunk; pause before a separate final summary.
            return [prefix,rest] if rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
        return previous['speech_segments'](text)
    def install_bridge(b,db_path,weather=None,_route_wrapper=None):
        result=previous['install_bridge'](b,db_path,weather=weather,_route_wrapper=_route_wrapper)
        install_socket(b,engine,_route_wrapper);return dict(result,friendly_weather=VERSION)
    def install_backend(module,weather=None):
        result=previous['install_backend'](module,weather=weather);install_http(module);return result
    engine.update(advisory=advisory,weather_question=weather_question,weather_consented=weather_consented,
        reply_normalized=normalized,prepare_turn=prepare_turn,speech_segments=speech_segments,
        install_bridge=install_bridge,install_backend=install_backend,_MOVIO_FRIENDLY_WEATHER_VERSION=VERSION)
    if bridge is not None:install_socket(bridge,engine,_route_wrapper)
    return dict(version=VERSION,silent_missing_forecast=True,final_booking_confirmation=True)


def install_http(module):
    if getattr(module.Handler,'_friendly_weather_version',None)==VERSION:return
    previous=module.Handler.do_GET
    def get(h):
        if h.path.split('?',1)[0]=='/api/weather-booking/friendly-health':
            return h.json(200,dict(ok=True,version=VERSION,final_booking_confirmation=True))
        return previous(h)
    module.Handler.do_GET=get;module.Handler._friendly_weather_version=VERSION
'''
    active=types.ModuleType('movio_friendly_weather');active.__file__=str(helper_path)
    exec(compile(source,str(helper_path),'exec'),active.__dict__)
    # Rendering checks use illustrative fixtures only. No provider requests,
    # model synthesis, booking writes or customer accounts are involved.
    checks=0
    for lang in ('English','Tamil','Tanglish'):
        for vehicle in ('Bike','SUV'):
            state=dict(lang=lang,vehicle=vehicle,pickup='Pickup',destination='Drop-off',
                weather=dict(status='ok',condition='rain',points=[dict(location=role,weather_code=61,precipitation_probability=n) for role,n in [('pickup',75),('destination',60)]]))
            text=active.report(state);question=active.question(state)
            if '75%' not in text or '60%' not in text or not question:
                raise RuntimeError('Friendly weather template check failed; no files changed.')
            if any(word in text.casefold() for word in ('slower','delay','traffic','surcharge')):
                raise RuntimeError('An unsupported travel/fare claim was detected; no files changed.')
            checks+=1
    marker='# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_V1';end_marker='# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_END_V1'
    def strip(text):
        if marker not in text:return text
        start=text.index(marker);end=text.find(end_marker,start)
        if end<0:raise RuntimeError('Incomplete earlier friendly-weather setup; no files changed.')
        return text[:start]+text[end+len(end_marker):]
    guard=strip(guard_path.read_text(encoding='utf-8')).rstrip()+'''

# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_V1
import importlib.util as _fw_import
_fw_path=Path(__file__).with_name('movio_friendly_weather.py')
_fw_spec=_fw_import.spec_from_file_location('movio_friendly_weather',_fw_path)
_fw_module=_fw_import.module_from_spec(_fw_spec)
exec(compile(_fw_path.read_text(encoding='utf-8'),str(_fw_path),'exec'),_fw_module.__dict__)
_fw_module.install_namespace(globals(),Path(__file__).resolve().parent.parent)
# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_END_V1
'''
    compile(guard,str(guard_path),'exec')
    server=strip(server_path.read_text(encoding='utf-8'))
    tree=ast.parse(server)
    launchers=[node for node in tree.body if isinstance(node,ast.If)
        and any(isinstance(x,ast.Name) and x.id=='__name__' for x in ast.walk(node.test))
        and any(isinstance(x,ast.Constant) and x.value=='__main__' for x in ast.walk(node.test))
        and any(isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute) and call.func.attr=='serve_forever'
            for statement in node.body for call in ast.walk(statement))]
    if len(launchers)!=1:raise RuntimeError('Could not identify one website HTTP launch block; no files changed.')
    bootstrap="""# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_V1
if __name__ == '__main__':
    import importlib.util as _fw_import
    import sys as _fw_sys
    _fw_path=ROOT/'integrations/movio_friendly_weather.py'
    _fw_spec=_fw_import.spec_from_file_location('movio_friendly_weather',_fw_path)
    _fw_module=_fw_import.module_from_spec(_fw_spec)
    exec(compile(_fw_path.read_text(encoding='utf-8'),str(_fw_path),'exec'),_fw_module.__dict__)
    _fw_engine=getattr(globals().get('_wx_mod'),'__dict__',None)
    if not isinstance(_fw_engine,dict) or not callable(_fw_engine.get('prepare_turn')):
        _fw_engine=Handler.do_POST.__globals__
    _fw_module.install_namespace(_fw_engine,ROOT)
    _fw_module.install_http(_fw_sys.modules[__name__])
# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_END_V1
"""
    lines=server.splitlines(keepends=True);line=launchers[0].lineno-1
    server=''.join(lines[:line]).rstrip()+'\n\n'+bootstrap+'\n'+''.join(lines[line:])
    compile(server,str(server_path),'exec')

    # Identify the existing website before modifying or stopping anything.
    health_url='http://127.0.0.1:8080/api/weather-booking/friendly-health'
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet')
        if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    in_kernel=[]
    for module in list(sys.modules.values()):
        if not isinstance(module,types.ModuleType):continue
        values=vars(module);handler=values.get('Handler');known_root=values.get('ROOT')
        if not isinstance(handler,type) or known_root is None or not getattr(handler,'_wx_installed',False):continue
        try:
            if Path(known_root).resolve()==root and callable(getattr(handler,'do_POST',None)):in_kernel.append(module)
        except (TypeError,ValueError,OSError):continue
    if os.getpid() not in listeners:in_kernel=[]
    process_details=[]
    known_launchers=[]
    for value in list(globals().values()):
        if type(value) in weakref.ProxyTypes:continue
        try:
            if isinstance(value,subprocess.Popen) and value.poll() is None:known_launchers.append(value)
        except ReferenceError:continue
    for pid in listeners:
        if pid==os.getpid():
            if not in_kernel:raise RuntimeError('An unrecognized in-kernel service owns port 8080; no process was stopped.')
            continue
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server_path for path in scripts):
            raise RuntimeError('A different application owns port 8080; no process was stopped.')
        # Prefer the notebook's own Popen handle when present. Containers can
        # expose host PIDs in psutil while Popen uses kernel-namespace PIDs.
        controller=next((value for value in known_launchers if isinstance(value.args,(list,tuple))
            and list(value.args)==command),process)
        process_details.append((controller,command,cwd,process.environ()))
    if not listeners:raise RuntimeError('Start the original website on port 8080, then rerun this cell.')

    writes={helper_path:source,guard_path:guard,server_path:server}
    changes={path:content for path,content in writes.items() if not path.exists() or path.read_text(encoding='utf-8')!=content}
    backup=None
    if changes:
        backup=root/('friendly_weather_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path in changes:
            if path.exists():
                dest=backup/path.relative_to(root);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(path,dest)
        written=[]
        try:
            for path,content in changes.items():
                temporary=path.with_name(path.name+'.friendly.tmp')
                try:temporary.write_text(content,encoding='utf-8');os.replace(temporary,path)
                finally:temporary.unlink(missing_ok=True)
                written.append(path)
        except Exception:
            for path in reversed(written):
                old=backup/path.relative_to(root)
                if old.exists():shutil.copy2(old,path)
                else:path.unlink(missing_ok=True)
            raise
    status=active.install_namespace(engine,root,b)
    for backend in in_kernel:
        namespace=getattr(getattr(backend,'_wx_mod',None),'__dict__',None)
        if not isinstance(namespace,dict) or not callable(namespace.get('prepare_turn')):namespace=backend.Handler.do_POST.__globals__
        active.install_namespace(namespace,root);active.install_http(backend)
    restart=any(path in changes for path in (helper_path,guard_path,server_path))
    for process,command,cwd,environment in process_details:
        if not restart:
            try:
                response=requests.get(health_url,timeout=(1,2))
                if response.status_code==200 and response.json().get('version')==active.VERSION:continue
            except (requests.RequestException,ValueError):pass
        process.terminate();process.wait(timeout=8)
        with (root/'friendly_weather_website.log').open('a',encoding='utf-8') as output:
            globals()['WX_WEBSITE_PROCESS']=subprocess.Popen(command,cwd=cwd,env=environment,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    verified=False
    for attempt in range(16):
        try:
            response=requests.get(health_url,timeout=(1,2))
            verified=response.status_code==200 and response.json().get('version')==active.VERSION
            if verified:break
        except (requests.RequestException,ValueError):pass
        time.sleep(.25)
    if not verified:raise RuntimeError('Website policy verification failed. See friendly_weather_website.log; rerun the original website-only launcher, then this cell.')
    print('Friendly weather enabled for English, Tamil and Tanglish.')
    print('Rain/severe weather: brief advice, one suggestion, one keep/change question.')
    print('Normal weather: short notice, then fare and booking summary.')
    print('Missing forecast: voice advisory skipped silently; final booking confirmation remains required.')
    print('Natural weather replies: Keep my bike booking / Keep 4 PM / Change to a mini car.')
    print('No automatic vehicle/time changes; no invented traffic delays or weather surcharges.')
    print('Existing weather audio-quality profile is reused when available.')
    print('Template checks:',checks,'| website policy: PASS')
    print('Close the old AI Call popup and start a NEW call from the signed-in dashboard.')
    status.update(website_verified=True,offline_template_checks=checks,backup=str(backup) if backup else None,changed_files=[str(path.relative_to(root)) for path in changes])
    return status

MOVIO_FRIENDLY_WEATHER_STATUS=movio_enable_friendly_weather()


Friendly weather enabled for English, Tamil and Tanglish.
Rain/severe weather: brief advice, one suggestion, one keep/change question.
Normal weather: short notice, then fare and booking summary.
Missing forecast: voice advisory skipped silently; final booking confirmation remains required.
Natural weather replies: Keep my bike booking / Keep 4 PM / Change to a mini car.
No automatic vehicle/time changes; no invented traffic delays or weather surcharges.
Existing weather audio-quality profile is reused when available.
Template checks: 6 | website policy: PASS
Close the old AI Call popup and start a NEW call from the signed-in dashboard.


In [111]:
# Run this entire file in ONE cell AFTER the earlier fixes, BEFORE evaluation.
# Existing UI, ASR/TTS models, fares, OTP and tracking code are preserved.
# Update 1.1: clean route clauses, retain/same-vehicle choices, explicit yes/no.
def movio_enable_friendly_weather():
    import ast
    from datetime import datetime
    import os
    from pathlib import Path
    import shutil
    import subprocess
    import sys
    import time
    import types
    import weakref
    import psutil
    import requests

    b=globals().get('bridge');engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',None)
    if not isinstance(engine,dict) or not all(callable(engine.get(key)) for key in ('prepare_turn','weather_consented','install_bridge')):
        raise RuntimeError('Run the existing voice/weather setup and earlier repair cells first, in this kernel.')
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    guard_path=root/'integrations/movio_weather_guard.py';server_path=root/'server.py'
    helper_path=root/'integrations/movio_friendly_weather.py'
    for path in (guard_path,server_path):
        if not path.is_file():raise FileNotFoundError(str(path))
    source=r'''"""Conversational weather choices for the existing MOVIO booking protocol."""
import contextvars
import copy
import inspect
import math
from pathlib import Path
import re
import threading
import types
from datetime import datetime

VERSION='friendly-weather-1.1'
TURN=contextvars.ContextVar('movio_friendly_weather_turn',default=None)


def number(value):
    if value is None or isinstance(value,bool):return None
    try:value=float(value)
    except (TypeError,ValueError,OverflowError):return None
    return value if math.isfinite(value) else None


def complete(weather):
    if not isinstance(weather,dict) or weather.get('status')!='ok':return False
    for role in ('pickup','destination'):
        point=next((p for p in weather.get('points',[]) if isinstance(p,dict) and p.get('location')==role),{})
        chance=number(point.get('precipitation_probability'));code=number(point.get('weather_code'))
        if chance is None or not 0<=chance<=100 or code is None:return False
    return True


def category(weather):
    if not complete(weather):return 'missing'
    points=weather.get('points',[])
    if weather.get('thunderstorm') or any(p.get('weather_code') in (95,96,99) for p in points):return 'storm'
    if weather.get('heavy_rain'):return 'heavy'
    if weather.get('heat'):return 'heat'
    # Wind advice is enabled only when actual wind fields exist in the forecast.
    if any((number(p.get('wind_speed_10m')) or 0)>=40 or (number(p.get('wind_gusts_10m')) or 0)>=50 for p in points):return 'wind'
    if weather.get('condition')=='rain':return 'rain'
    return 'normal'


def scope(state,engine):
    weather=copy.deepcopy(state.get('weather') or {});weather.pop('_friendly_notice_policy',None)
    values={k:state.get(k) for k in ('pickup','destination','pickup_geo','destination_geo','pickup_at','vehicle','count')}
    values['weather']=weather
    return engine['digest'](values)


def exemption(state,engine):
    weather=state.get('weather') or {};policy=weather.get('_friendly_notice_policy') or {}
    reason='forecast_missing' if category(weather)=='missing' else 'no_weather_choice_needed' if category(weather)=='normal' else None
    return bool(reason and policy.get('version')==VERSION and policy.get('reason')==reason and policy.get('scope')==scope(state,engine))


def pick(state,en,ta,roman):return ta if state.get('lang')=='Tamil' else roman if state.get('lang')=='Tanglish' else en


def fmt(value):return f'{float(value):g}'


def report(state):
    weather=state.get('weather') or {};kind=category(weather)
    if kind=='missing':return ''
    points={p['location']:p for p in weather['points']};a=fmt(points['pickup']['precipitation_probability']);b=fmt(points['destination']['precipitation_probability'])
    pickup=state.get('pickup','your pickup');drop=state.get('destination','your destination');v=state.get('vehicle','vehicle')
    ta_vehicle={'Bike':'பைக்','Auto':'ஆட்டோ','Mini':'மினி கார்','Sedan':'செடான்','SUV':'எஸ் யூ வி'}.get(v,v)
    if kind in ('rain','heavy','storm'):
        subject=pick(state,'Thunderstorms are forecast' if kind=='storm' else 'Heavy rain is forecast' if kind=='heavy' else 'Rain may join your journey',
            'இடியுடன் மழை எதிர்பார்க்கப்படுகிறது' if kind=='storm' else 'கனமழை எதிர்பார்க்கப்படுகிறது' if kind=='heavy' else 'பயண நேரத்தில் மழை பெய்ய வாய்ப்புள்ளது',
            'Thunderstorms expect pannrom' if kind=='storm' else 'Heavy rain expect pannrom' if kind=='heavy' else 'Journey time-la rain chance irukku')
        first=pick(state,f'A quick heads-up: {subject.lower()}, with a {a}% rain chance at {pickup} and {b}% at {drop} around pickup time.',
            f'ஒரு சின்ன தகவல்: {subject}; புறப்படும் நேரத்தில் {pickup} பகுதியில் {a}%, {drop} பகுதியில் {b}% மழை வாய்ப்பு உள்ளது.',
            f'Oru chinna update: {subject}; pickup time-la {pickup}-la {a}%, {drop}-la {b}% rain chance irukku.')
        if v=='Bike':tip=pick(state,'A covered car would offer better rain protection; you can keep your bike if you prefer.',
            'மழையிலிருந்து பாதுகாப்புக்கு மூடிய காரைப் பரிசீலிக்கலாம்; விரும்பினால் பைக்கிலேயே தொடரலாம்.',
            'Rain protection-ku covered car consider pannalaam; unga bike-aiyum keep pannalaam.')
        elif v=='Auto':tip=pick(state,'An auto has open sides, so keep an umbrella and protect your bags.',
            'ஆட்டோவின் பக்கவாட்டில் மழை படலாம்; குடை எடுத்துக்கொண்டு பைகளையும் பாதுகாத்துக்கொள்ளுங்கள்.',
            'Auto side-la rain padalaam; umbrella eduthuttu bags-ai protect pannikkonga.')
        else:tip=pick(state,f'Your {v} provides cover; an umbrella will help when getting in and out.',
            f'{ta_vehicle} பயணத்தில் மழையிலிருந்து பாதுகாப்பு கிடைக்கும்; ஏறவும் இறங்கவும் குடை எடுத்துக்கொள்ளுங்கள்.',
            f'Unga {v}-la cover irukkum; eravum irangavum umbrella eduthukkonga.')
        return first+' '+tip
    temperature=number(points['pickup'].get('temperature_2m'))
    if kind=='heat':
        t=f' about {fmt(temperature)}°C' if temperature is not None else ' hot'
        return pick(state,f'It is expected to be{t} around pickup time. Please carry water and prepare for the heat.',
            ('புறப்படும் நேரத்தில் சுமார் '+fmt(temperature)+'°C வெப்பநிலை எதிர்பார்க்கப்படுகிறது.' if temperature is not None else 'புறப்படும் நேரத்தில் வெப்பம் அதிகமாக இருக்கும்.')+' குடிநீர் எடுத்துக்கொண்டு வெப்பத்திற்கு ஏற்ற முன்னேற்பாடுகளைச் செய்துகொள்ளுங்கள்.',
            ('Pickup time-la around '+fmt(temperature)+'°C expect pannrom.' if temperature is not None else 'Pickup time-la heat adhigamaaga irukkum.')+' Water eduthuttu heat-ku prepare pannikkonga.')
    if kind=='wind':return pick(state,'Strong winds are expected around pickup time. For a bike ride, you may prefer a covered car.' if v=='Bike' else 'Strong winds are expected around pickup time. Take care when boarding and getting out.',
        'புறப்படும் நேரத்தில் பலத்த காற்று எதிர்பார்க்கப்படுகிறது. பைக் பயணத்திற்கு பதிலாக மூடிய காரைப் பரிசீலிக்கலாம்.' if v=='Bike' else 'புறப்படும் நேரத்தில் பலத்த காற்று எதிர்பார்க்கப்படுகிறது. வாகனத்தில் ஏறவும் இறங்கவும் கவனமாக இருங்கள்.',
        'Pickup time-la strong wind expect pannrom. Bike-ku badhila covered car consider pannalaam.' if v=='Bike' else 'Pickup time-la strong wind expect pannrom. Eravum irangavum careful-a irunga.')
    if weather.get('condition')=='cloudy':return pick(state,'Cloudy skies are expected for your trip. Let me confirm your ride details.',
        'பயண நேரத்தில் மேகமூட்டம் எதிர்பார்க்கப்படுகிறது. உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.',
        'Journey time-la cloudy weather expect pannrom. Unga ride details confirm pannalaam.')
    if temperature is not None and temperature>=33:return pick(state,f'Clear skies and about {fmt(temperature)}°C are expected; carrying water is a good idea.',
        f'வானம் தெளிவாகவும் வெப்பநிலை சுமார் {fmt(temperature)}°C ஆகவும் இருக்கும்; குடிநீர் எடுத்துக்கொள்ளுங்கள்.',
        f'Clear sky, around {fmt(temperature)}°C expect pannrom; water eduthukkonga.')
    if not weather.get('pleasant'):
        return pick(state,'Clear skies are expected for your journey. Let me confirm your ride details.' if weather.get('condition')=='clear' else 'I have checked the forecast for your journey. Let me confirm your ride details.',
            'பயண நேரத்தில் வானம் தெளிவாக இருக்கும் என எதிர்பார்க்கப்படுகிறது. உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.' if weather.get('condition')=='clear' else 'உங்கள் பயணத்திற்கான வானிலையைச் சரிபார்த்துவிட்டேன். பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.',
            'Journey time-la clear sky expect pannrom. Unga ride details confirm pannalaam.' if weather.get('condition')=='clear' else 'Unga journey-ku forecast check pannitten. Ride details confirm pannalaam.')
    return pick(state,'The weather looks pleasant for your journey. Let me confirm your ride details.',
        'நல்ல செய்தி! பயண நேரத்தில் வானிலை நன்றாக இருக்கும் என எதிர்பார்க்கப்படுகிறது. உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம்.',
        'Nalla news! Journey time-la weather nalla irukkum-nu expect pannrom. Unga ride details confirm pannalaam.')


def question(state):
    if state.get('vehicle')=='Bike' and category(state.get('weather')) in ('rain','heavy','storm','wind'):
        return pick(state,'Would you like to keep your bike? Please say yes or no, or name another vehicle.',
            'பைக்கிலேயே தொடரலாமா? ஆம் அல்லது இல்லை என்று சொல்லுங்கள்; அல்லது வேறு வாகனத்தைக் குறிப்பிடுங்கள்.',
            'Bike-aiye keep pannalaama? Yes illa no sollunga; illa vera vehicle sollunga.')
    return pick(state,'Would you like to keep your pickup time? Please say yes or no, or give another time.',
        'அதே பயண நேரத்தில் தொடரலாமா? ஆம் அல்லது இல்லை என்று சொல்லுங்கள்; அல்லது வேறு நேரத்தைக் குறிப்பிடுங்கள்.',
        'Same pickup time-la continue pannalaama? Yes illa no sollunga; illa vera time sollunga.')


def keep_choice(text,state,engine):
    norm=engine['_friendly_previous_normalized'](text)
    plain={'keep my booking','retain my booking','keep this booking','keep the same booking','keep the same vehicle','retain the same vehicle','same vehicle','keep my pickup time','keep the pickup time','same time','keep the same time',
        'அதே நேரம்','அதே நேரத்தில் தொடரலாம்','இதே நேரம் போதும்','adhe time','same time venum','idhe time podhum'}
    if norm in plain:return 'time'
    v=state.get('vehicle','').casefold()
    aliases={'Bike':('bike','பைக்'),'Mini':('mini','mini car','மினி','மினி கார்'),
        'SUV':('suv','எஸ் யூ வி'),'Sedan':('sedan','செடான்'),'Auto':('auto','ஆட்டோ')}.get(state.get('vehicle'),(v,))
    vehicle=set()
    for alias in aliases:
        for action in ('','keep ','retain ','continue with ','stick with ','go with '):
            for determiner in ('','my ','the ','same ','current '):
                for ending in ('',' itself',' only',' booking',' ride'):
                    vehicle.add(engine['_friendly_previous_normalized'](action+determiner+alias+ending))
        vehicle.update(engine['_friendly_previous_normalized'](alias+ending) for ending in (' keep pannunga',' thaan venum',' dhaan venum',' podhum',' venum'))
    if v=='bike':vehicle|={'பைக்கிலேயே தொடரலாம்','பைக் போதும்','பைக் தான் வேண்டும்','பைக்கே வேண்டும்','அதே பைக்','bike-aiye keep pannunga','bike aiye keep pannunga'}
    if norm in vehicle:return 'vehicle'
    match=re.fullmatch(r'keep\s+(.+)',str(text).strip(),re.I)
    if match:
        words=[word for word,value in engine['number_lexicon'](23).items() if not word.isdigit() and 0<=value<=23]
        clock=r'(?:\d{1,2}(?::\d{2})?|'+'|'.join(re.escape(word) for word in sorted(words,key=len,reverse=True))+r')'
        if not re.fullmatch(r'(?:at\s+)?'+clock+r'\s*(?:a\.?\s*m\.?|p\.?\s*m\.?)\s*(?:please)?[.!?]?',match[1],re.I):return None
        probe=copy.deepcopy(state)
        try:value,error=engine['update_time_memory'](probe,match[1],engine['datetime'].now(engine['TZ']))
        except engine['GuardError']:return None
        if not error and value.isoformat()==state.get('pickup_at'):return 'time'
    return None


class ChoiceSocket:
    def __init__(self,ws,bridge,engine):self.ws,self.bridge,self.engine,self.sid=ws,bridge,engine,None;self.pending=None
    def __getattr__(self,name):return getattr(self.ws,name)
    async def receive_json(self):
        while True:
            row=None
            if self.sid:
                try:row=self.bridge._wx_db.row(self.sid)
                except self.engine['GuardError']:pass
            if self.pending and row:
                if row['ride_id'] or row['summary_nonce']!=self.pending.get('summary_nonce'):
                    self.pending=None
                elif row['played']:
                    message=self.pending;self.pending=None
                    TURN.set(dict(db=self.bridge._wx_db,sid=self.sid,nonce=message.get('summary_nonce'),state=__import__('json').loads(row['state'])))
                    return message
            message=await self.ws.receive_json();kind=message.get('event')
            if kind=='start':self.sid=message.get('call_id')
            if kind=='interrupt':self.pending=None
            context=dict(db=self.bridge._wx_db,sid=self.sid,nonce=message.get('summary_nonce'))
            if self.sid:
                try:
                    row=self.bridge._wx_db.row(self.sid);context['state']=__import__('json').loads(row['state'])
                except (ValueError,KeyError,self.engine['GuardError']):row=None
            state=context.get('state') or {}
            if (kind in ('text','audio') and state.get('stage')=='weather_advisory' and row
                    and message.get('summary_nonce') and message['summary_nonce']==row['summary_nonce'] and not row['played']):
                self.pending=dict(message)
                continue
            TURN.set(context)
            return message


def patch_location_spans(bridge,engine):
    targets=[]
    if bridge is not None:targets.append(getattr(bridge,'_wx_group_slots_module',None))
    for function in (engine.get('collect_details'),engine.get('_movio_optional_weather_original',{}).get('collect_details')):
        if not callable(function):continue
        targets.append(getattr(function,'__globals__',{}))
        try:targets.extend(inspect.getclosurevars(function).nonlocals.values())
        except (TypeError,ValueError):pass
    boundary=re.compile(r'\s+(?:and\s+)?(?:i\s+(?:am|need|want|require)\b|we\s+(?:are|need|want)\b|there\s+(?:is|are)\b|book\s+(?:a\s+)?(?:cab|bike|car|taxi)\b)',re.I)
    for target in targets:
        namespace=target if isinstance(target,dict) else vars(target) if isinstance(target,types.ModuleType) else {}
        parser=namespace.get('Parser')
        if not isinstance(parser,type) or not callable(getattr(parser,'add_place',None)) or getattr(parser,'_friendly_place_boundary_version',None)==VERSION:continue
        previous=parser.add_place
        def add_place(self,role,a,b,clause_start,*args,_previous=previous,**kwargs):
            match=boundary.search(self.raw[a:b])
            if match:b=a+match.start()
            return _previous(self,role,a,b,clause_start,*args,**kwargs)
        parser.add_place=add_place;parser._friendly_place_boundary_version=VERSION


def install_socket(bridge,engine,_route_wrapper=None):
    previous=getattr(bridge,'_wx_endpoint',None)
    if not callable(previous) or getattr(previous,'_friendly_weather_version',None)==VERSION:return False
    while getattr(previous,'_friendly_weather_version',None):
        earlier=getattr(previous,'_friendly_original_endpoint',None)
        if not callable(earlier):
            try:earlier=inspect.getclosurevars(previous).nonlocals.get('previous')
            except (TypeError,ValueError):earlier=None
        if not callable(earlier):raise RuntimeError('Could not upgrade the previous weather socket safely.')
        previous=earlier
    async def wrapped(ws):
        if ws.query_params.get('mode','book')=='cancel':return await previous(ws)
        token=TURN.set(None)
        try:return await previous(ChoiceSocket(ws,bridge,engine))
        finally:TURN.reset(token)
    wrapped=types.FunctionType(wrapped.__code__,engine,wrapped.__name__,wrapped.__defaults__,wrapped.__closure__)
    engine['ChoiceSocket']=ChoiceSocket;engine['TURN']=TURN
    wrapped._friendly_weather_version=VERSION;wrapped._friendly_original_endpoint=previous
    if _route_wrapper is None:
        from starlette.routing import websocket_session
        _route_wrapper=websocket_session
    route=next(r for r in bridge.app.routes if getattr(r,'path','')=='/browser-call')
    route.endpoint=wrapped;route.app=_route_wrapper(wrapped);bridge._wx_endpoint=wrapped
    return True


def install_namespace(engine,root=None,bridge=None,_route_wrapper=None):
    if engine.get('_MOVIO_FRIENDLY_WEATHER_VERSION')==VERSION:
        patch_location_spans(bridge,engine)
        if bridge is not None:install_socket(bridge,engine,_route_wrapper)
        return dict(version=VERSION,already_installed=True)
    if engine.get('_MOVIO_FRIENDLY_WEATHER_VERSION'):
        original=engine.get('_friendly_original')
        if not isinstance(original,dict):
            try:original=inspect.getclosurevars(engine['prepare_turn']).nonlocals.get('previous')
            except (TypeError,ValueError):original=None
        if not isinstance(original,dict):raise RuntimeError('Could not upgrade the previous weather functions safely.')
        engine.update(original);engine.pop('_MOVIO_FRIENDLY_WEATHER_VERSION',None)
    root=Path(root or Path(engine['__file__']).resolve().parent.parent)
    patch_location_spans(bridge,engine)
    keys=('prepare_turn','weather_consented','advisory','weather_question','reply_normalized','install_bridge','install_backend','speech_segments','summary_text')
    previous={key:engine[key] for key in keys}
    base_prepare=engine.get('_movio_optional_weather_original',{}).get('prepare_turn',previous['prepare_turn'])
    engine['_friendly_previous_normalized']=previous['reply_normalized']
    local=threading.local();spoken=[];lock=threading.RLock()
    try:register=inspect.getclosurevars(engine['completion_message']).nonlocals.get('register')
    except (TypeError,ValueError):register=None
    def advisory(state):
        text=report(state);local.question=question(state);local.lang=state.get('lang')
        if text and callable(register):text=register(text,'weather')
        with lock:
            if text:spoken.append(text)
            if len(spoken)>128:del spoken[:-128]
        return text
    def weather_question(lang):
        return local.question if getattr(local,'lang',None)==lang else engine['localized'](lang,
            'Would you like to continue? Please say yes or no.','முன்பதிவைத் தொடரலாமா? ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.',
            'Booking continue pannalaama? Yes illa no sollunga.')
    def summary_text(b,state):
        text=previous['summary_text'](b,state)
        if state.get('_wx_mode')=='manual':return text
        return text+' '+pick(state,'Please say yes or no.','ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.','Yes illa no sollunga.')
    def weather_consented(state):return exemption(state,engine) or previous['weather_consented'](state)
    def normalized(text):
        context=TURN.get();state=(context or {}).get('state') or {}
        if state.get('stage')=='weather_advisory':
            choice=keep_choice(text,state,engine)
            if choice:context['choice']=choice;return 'yes'
        return previous['reply_normalized'](text)
    def prepare_turn(b,weather,state,text,now=None):
        context=TURN.get() or {};prior_stage=state.get('stage');ack=False
        # Explicit vehicle/count/location corrections are also weather choices.
        # Reuse the real nonce and playback acknowledgement before applying them.
        if prior_stage=='weather_advisory' and str(text).strip() and context.get('sid'):
            probe=copy.deepcopy(state)
            parsed=engine['collect_details'](b,probe,text)
            if parsed.get('changed'):
                db=context['db'];row=db.row(context['sid'])
                try:state=db.accept_weather(context['sid'],row['epoch'],context.get('nonce'),state);ack=True
                except engine['GuardError']:pass
        norm=previous['reply_normalized'](text)
        if prior_stage=='weather_advisory' and norm in {'earlier','leave earlier','choose an earlier pickup time','change pickup time','முன்னதாக வேண்டும்','வேறு நேரம் வேண்டும்','earlier venum'}:
            state['stage']='weather_change_time';state['dialogue_state']='COLLECTING_DETAILS'
            return state,pick(state,'What pickup time would you prefer? I will check the forecast for that time.',
                'எத்தனை மணிக்கு வாகனம் வேண்டும்? அந்த நேரத்திற்கான வானிலையைச் சரிபார்க்கிறேன்.',
                'Endha pickup time venum? Andha time-ku weather check pannren.')
        state,answer=previous['prepare_turn'](b,weather,state,text,now=now)
        kind=category(state.get('weather'))
        if state.get('stage') in ('weather_advisory','weather_pending') and kind in ('normal','missing'):
            w=state.setdefault('weather',{});w.pop('_friendly_notice_policy',None)
            reason='forecast_missing' if kind=='missing' else 'no_weather_choice_needed'
            w['_friendly_notice_policy']=dict(version=VERSION,reason=reason,scope=scope(state,engine))
            state.pop('_wx_weather_consent',None)
            state,answer=base_prepare(b,weather,state,'',now=now)
            if state.get('stage')=='confirm':
                state['dialogue_state']='AWAITING_BOOKING_CONFIRMATION'
                notice=advisory(state) if kind=='normal' else ''
                return state,(notice+'\n'+answer if notice else answer)
        if state.get('stage')=='confirm' and (context.get('choice') or ack):
            context.pop('choice',None)
            intro=pick(state,'Sure, let me confirm your updated ride details. ',
                'சரி, உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம். ',
                'Seri, unga ride details confirm pannalaam. ')
            return state,intro+answer
        return state,answer
    def speech_segments(text):
        value=str(text)
        with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
        if prefix:
            rest=value[len(prefix):].strip()
            # Keep a coherent weather chunk; pause before a separate final summary.
            return [prefix,rest] if rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
        return previous['speech_segments'](text)
    def install_bridge(b,db_path,weather=None,_route_wrapper=None):
        result=previous['install_bridge'](b,db_path,weather=weather,_route_wrapper=_route_wrapper)
        patch_location_spans(b,engine)
        install_socket(b,engine,_route_wrapper);return dict(result,friendly_weather=VERSION)
    def install_backend(module,weather=None):
        result=previous['install_backend'](module,weather=weather);install_http(module);return result
    engine.update(advisory=advisory,weather_question=weather_question,weather_consented=weather_consented,
        reply_normalized=normalized,prepare_turn=prepare_turn,speech_segments=speech_segments,
        install_bridge=install_bridge,install_backend=install_backend,summary_text=summary_text,
        _MOVIO_FRIENDLY_WEATHER_VERSION=VERSION,_friendly_original=previous)
    if bridge is not None:install_socket(bridge,engine,_route_wrapper)
    return dict(version=VERSION,silent_missing_forecast=True,final_booking_confirmation=True)


def install_http(module):
    if getattr(module.Handler,'_friendly_weather_version',None)==VERSION:return
    previous=module.Handler.do_GET
    def get(h):
        if h.path.split('?',1)[0]=='/api/weather-booking/friendly-health':
            return h.json(200,dict(ok=True,version=VERSION,final_booking_confirmation=True))
        return previous(h)
    module.Handler.do_GET=get;module.Handler._friendly_weather_version=VERSION
'''
    active=types.ModuleType('movio_friendly_weather');active.__file__=str(helper_path)
    exec(compile(source,str(helper_path),'exec'),active.__dict__)
    # Rendering checks use illustrative fixtures only. No provider requests,
    # model synthesis, booking writes or customer accounts are involved.
    checks=0
    for lang in ('English','Tamil','Tanglish'):
        for vehicle in ('Bike','SUV'):
            state=dict(lang=lang,vehicle=vehicle,pickup='Pickup',destination='Drop-off',
                weather=dict(status='ok',condition='rain',points=[dict(location=role,weather_code=61,precipitation_probability=n) for role,n in [('pickup',75),('destination',60)]]))
            text=active.report(state);question=active.question(state)
            if '75%' not in text or '60%' not in text or not question:
                raise RuntimeError('Friendly weather template check failed; no files changed.')
            if any(word in text.casefold() for word in ('slower','delay','traffic','surcharge')):
                raise RuntimeError('An unsupported travel/fare claim was detected; no files changed.')
            checks+=1
    marker='# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_V1';end_marker='# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_END_V1'
    def strip(text):
        if marker not in text:return text
        start=text.index(marker);end=text.find(end_marker,start)
        if end<0:raise RuntimeError('Incomplete earlier friendly-weather setup; no files changed.')
        return text[:start]+text[end+len(end_marker):]
    guard=strip(guard_path.read_text(encoding='utf-8')).rstrip()+'''

# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_V1
import importlib.util as _fw_import
_fw_path=Path(__file__).with_name('movio_friendly_weather.py')
_fw_spec=_fw_import.spec_from_file_location('movio_friendly_weather',_fw_path)
_fw_module=_fw_import.module_from_spec(_fw_spec)
exec(compile(_fw_path.read_text(encoding='utf-8'),str(_fw_path),'exec'),_fw_module.__dict__)
_fw_module.install_namespace(globals(),Path(__file__).resolve().parent.parent)
# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_END_V1
'''
    compile(guard,str(guard_path),'exec')
    server=strip(server_path.read_text(encoding='utf-8'))
    tree=ast.parse(server)
    launchers=[node for node in tree.body if isinstance(node,ast.If)
        and any(isinstance(x,ast.Name) and x.id=='__name__' for x in ast.walk(node.test))
        and any(isinstance(x,ast.Constant) and x.value=='__main__' for x in ast.walk(node.test))
        and any(isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute) and call.func.attr=='serve_forever'
            for statement in node.body for call in ast.walk(statement))]
    if len(launchers)!=1:raise RuntimeError('Could not identify one website HTTP launch block; no files changed.')
    bootstrap="""# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_V1
if __name__ == '__main__':
    import importlib.util as _fw_import
    import sys as _fw_sys
    _fw_path=ROOT/'integrations/movio_friendly_weather.py'
    _fw_spec=_fw_import.spec_from_file_location('movio_friendly_weather',_fw_path)
    _fw_module=_fw_import.module_from_spec(_fw_spec)
    exec(compile(_fw_path.read_text(encoding='utf-8'),str(_fw_path),'exec'),_fw_module.__dict__)
    _fw_engine=getattr(globals().get('_wx_mod'),'__dict__',None)
    if not isinstance(_fw_engine,dict) or not callable(_fw_engine.get('prepare_turn')):
        _fw_engine=Handler.do_POST.__globals__
    _fw_module.install_namespace(_fw_engine,ROOT)
    _fw_module.install_http(_fw_sys.modules[__name__])
# MOVIO_FRIENDLY_WEATHER_BOOTSTRAP_END_V1
"""
    lines=server.splitlines(keepends=True);line=launchers[0].lineno-1
    server=''.join(lines[:line]).rstrip()+'\n\n'+bootstrap+'\n'+''.join(lines[line:])
    compile(server,str(server_path),'exec')

    # Identify the existing website before modifying or stopping anything.
    health_url='http://127.0.0.1:8080/api/weather-booking/friendly-health'
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet')
        if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    in_kernel=[]
    for module in list(sys.modules.values()):
        if not isinstance(module,types.ModuleType):continue
        values=vars(module);handler=values.get('Handler');known_root=values.get('ROOT')
        if not isinstance(handler,type) or known_root is None or not getattr(handler,'_wx_installed',False):continue
        try:
            if Path(known_root).resolve()==root and callable(getattr(handler,'do_POST',None)):in_kernel.append(module)
        except (TypeError,ValueError,OSError):continue
    if os.getpid() not in listeners:in_kernel=[]
    process_details=[]
    known_launchers=[]
    for value in list(globals().values()):
        if type(value) in weakref.ProxyTypes:continue
        try:
            if isinstance(value,subprocess.Popen) and value.poll() is None:known_launchers.append(value)
        except ReferenceError:continue
    for pid in listeners:
        if pid==os.getpid():
            if not in_kernel:raise RuntimeError('An unrecognized in-kernel service owns port 8080; no process was stopped.')
            continue
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server_path for path in scripts):
            raise RuntimeError('A different application owns port 8080; no process was stopped.')
        # Prefer the notebook's own Popen handle when present. Containers can
        # expose host PIDs in psutil while Popen uses kernel-namespace PIDs.
        controller=next((value for value in known_launchers if isinstance(value.args,(list,tuple))
            and list(value.args)==command),process)
        process_details.append((controller,command,cwd,process.environ()))
    if not listeners:raise RuntimeError('Start the original website on port 8080, then rerun this cell.')

    writes={helper_path:source,guard_path:guard,server_path:server}
    changes={path:content for path,content in writes.items() if not path.exists() or path.read_text(encoding='utf-8')!=content}
    backup=None
    if changes:
        backup=root/('friendly_weather_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path in changes:
            if path.exists():
                dest=backup/path.relative_to(root);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(path,dest)
        written=[]
        try:
            for path,content in changes.items():
                temporary=path.with_name(path.name+'.friendly.tmp')
                try:temporary.write_text(content,encoding='utf-8');os.replace(temporary,path)
                finally:temporary.unlink(missing_ok=True)
                written.append(path)
        except Exception:
            for path in reversed(written):
                old=backup/path.relative_to(root)
                if old.exists():shutil.copy2(old,path)
                else:path.unlink(missing_ok=True)
            raise
    status=active.install_namespace(engine,root,b)
    for backend in in_kernel:
        namespace=getattr(getattr(backend,'_wx_mod',None),'__dict__',None)
        if not isinstance(namespace,dict) or not callable(namespace.get('prepare_turn')):namespace=backend.Handler.do_POST.__globals__
        active.install_namespace(namespace,root);active.install_http(backend)
    restart=any(path in changes for path in (helper_path,guard_path,server_path))
    for process,command,cwd,environment in process_details:
        if not restart:
            try:
                response=requests.get(health_url,timeout=(1,2))
                if response.status_code==200 and response.json().get('version')==active.VERSION:continue
            except (requests.RequestException,ValueError):pass
        process.terminate();process.wait(timeout=8)
        with (root/'friendly_weather_website.log').open('a',encoding='utf-8') as output:
            globals()['WX_WEBSITE_PROCESS']=subprocess.Popen(command,cwd=cwd,env=environment,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    verified=False
    for attempt in range(16):
        try:
            response=requests.get(health_url,timeout=(1,2))
            verified=response.status_code==200 and response.json().get('version')==active.VERSION
            if verified:break
        except (requests.RequestException,ValueError):pass
        time.sleep(.25)
    if not verified:raise RuntimeError('Website policy verification failed. See friendly_weather_website.log; rerun the original website-only launcher, then this cell.')
    print('Friendly weather enabled for English, Tamil and Tanglish. Version:',active.VERSION)
    print('Rain/severe weather: brief advice, one suggestion, one keep/change question.')
    print('Normal weather: short notice, then fare and booking summary.')
    print('Missing forecast: voice advisory skipped silently; final booking confirmation remains required.')
    print('Natural weather replies: Retain bike itself / Bike / Keep 4 PM / Change to a mini car.')
    print('Pickup/drop extraction stops before I am / I need / we are clauses.')
    print('Weather choices and final booking summaries now explicitly request Yes or no.')
    print('No automatic vehicle/time changes; no invented traffic delays or weather surcharges.')
    print('Existing weather audio-quality profile is reused when available.')
    print('Template checks:',checks,'| website policy: PASS')
    print('Close the old AI Call popup and start a NEW call from the signed-in dashboard.')
    status.update(website_verified=True,offline_template_checks=checks,backup=str(backup) if backup else None,changed_files=[str(path.relative_to(root)) for path in changes])
    return status

MOVIO_FRIENDLY_WEATHER_STATUS=movio_enable_friendly_weather()


Friendly weather enabled for English, Tamil and Tanglish. Version: friendly-weather-1.1
Rain/severe weather: brief advice, one suggestion, one keep/change question.
Normal weather: short notice, then fare and booking summary.
Missing forecast: voice advisory skipped silently; final booking confirmation remains required.
Natural weather replies: Retain bike itself / Bike / Keep 4 PM / Change to a mini car.
Pickup/drop extraction stops before I am / I need / we are clauses.
Weather choices and final booking summaries now explicitly request Yes or no.
No automatic vehicle/time changes; no invented traffic delays or weather surcharges.
Existing weather audio-quality profile is reused when available.
Template checks: 6 | website policy: PASS
Close the old AI Call popup and start a NEW call from the signed-in dashboard.


In [112]:
# STEP 1 — inspection only. Run in your current Kaggle kernel before evaluation.
# This does not install a patch, restart services, call weather/STT/TTS APIs,
# create a booking, or read customer records. Project files are only read.
def movio_inspect_weather_handlers():
    import ast
    import hashlib
    import inspect
    import json
    import marshal
    from pathlib import Path
    from urllib.error import HTTPError, URLError
    from urllib.request import urlopen

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE')
    bridge_object = globals().get('bridge')
    endpoint = getattr(bridge_object, '_wx_endpoint', None)
    engine = getattr(endpoint, '__globals__', {})
    engine = engine if isinstance(engine, dict) else {}
    adapter = getattr(bridge_object, 'TTS', None)
    tts_namespace = getattr(adapter, 'ns', {})
    tts_namespace = tts_namespace if isinstance(tts_namespace, dict) else {}

    def describe(function):
        function = getattr(function, '__func__', function)
        code = getattr(function, '__code__', None)
        if code is None:
            return {'present': callable(function)}
        return {
            'name': getattr(function, '__qualname__', code.co_name),
            'file': code.co_filename,
            'line': code.co_firstlineno,
            'code_sha256': hashlib.sha256(marshal.dumps(code)).hexdigest()[:16],
        }

    def captured(function):
        try:
            return inspect.getclosurevars(getattr(function, '__func__', function)).nonlocals
        except (TypeError, ValueError):
            return {}

    names = (
        'advisory', 'weather_question', 'prepare_turn', 'reply_normalized',
        'weather_consented', 'summary_text', 'speech_segments', 'tts_reply',
        'completion_message', 'display_reply', 'collect_details',
    )
    handlers = {name: describe(engine.get(name)) for name in names}
    helpers = {}
    for name in ('advisory', 'prepare_turn', 'reply_normalized'):
        namespace = getattr(engine.get(name), '__globals__', {})
        for helper in ('report', 'category', 'question', 'keep_choice'):
            function = namespace.get(helper)
            if callable(function):
                helpers[helper] = describe(function)

    # Describe captured originals; never call these application functions.
    originals = {}
    for name in ('advisory', 'prepare_turn', 'reply_normalized', 'speech_segments'):
        entries = {}
        for key, value in captured(engine.get(name)).items():
            if callable(value):
                entries[key] = describe(value)
            elif isinstance(value, dict) and key in ('previous', 'original'):
                entries[key] = {
                    item: describe(value.get(item)) for item in names
                    if callable(value.get(item))
                }
        originals[name] = entries

    speech = {
        'bridge.speak': describe(getattr(bridge_object, 'speak', None)),
        'TTS.synthesize': describe(getattr(adapter, 'synthesize', None)),
        'Tanglish_speech_text': describe(getattr(bridge_object, 'tanglish_speech_text', None)),
        'prepare_movio_request': describe(tts_namespace.get('prepare_movio_request')),
        'movio_stream_response': describe(tts_namespace.get('movio_stream_response')),
        'stable_weather_audio_enabled': bool(getattr(bridge_object, '_wx_stable_audio_v1', False)),
    }
    for key in ('weather_steps', 'closing_steps', 'min_words'):
        quality = getattr(bridge_object, 'WX_SPEECH_QUALITY', {})
        if isinstance(quality, dict) and isinstance(quality.get(key), (int, float)):
            speech[key] = quality[key]

    providers = {}
    for name in ('_wx_weather', '_wx_required_forecast'):
        client = getattr(bridge_object, name, None)
        if client is not None:
            providers[name] = {
                'class': type(client).__module__ + '.' + type(client).__qualname__,
                'forecast': describe(getattr(client, 'forecast', None)),
                'fetch': describe(getattr(client, 'fetch', None)),
            }

    # Parse files as text only: importing them could activate old bootstraps.
    relatives = [
        'server.py', 'integrations/movio_weather_guard.py',
        'integrations/movio_friendly_weather.py',
        'integrations/movio_required_weather.py',
        'integrations/movio_optional_weather.py',
        'integrations/movio_confirmation_flow.py',
        'integrations/movio_time_route.py',
        'integrations/movio_booking_reliability.py',
    ]
    disk = {}
    for relative in relatives:
        path = root / relative
        if not path.is_file():
            disk[relative] = {'exists': False}
            continue
        try:
            raw = path.read_bytes()
            tree = ast.parse(raw.decode('utf-8-sig'), filename=str(path))
            compile(tree, str(path), 'exec')  # syntax check, not execution
            versions = {}
            for node in tree.body:
                if isinstance(node, ast.Assign) and isinstance(node.value, ast.Constant):
                    for target in node.targets:
                        if isinstance(target, ast.Name) and target.id in ('VERSION', '__version__'):
                            versions[target.id] = node.value.value
            hourly = set()
            for node in ast.walk(tree):
                if isinstance(node, ast.keyword) and node.arg == 'hourly' and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
                    hourly.update(node.value.value.split(','))
                if isinstance(node, ast.Dict):
                    for key, value in zip(node.keys, node.values):
                        if isinstance(key, ast.Constant) and key.value == 'hourly' and isinstance(value, ast.Constant) and isinstance(value.value, str):
                            hourly.update(value.value.split(','))
            disk[relative] = {
                'syntax': 'PASS', 'sha256': hashlib.sha256(raw).hexdigest(),
                'versions': versions,
                **({'requested_hourly_fields': sorted(hourly)} if hourly else {}),
            }
        except (OSError, UnicodeError, SyntaxError, ValueError) as error:
            disk[relative] = {'inspection_error': type(error).__name__, 'line': getattr(error, 'lineno', None)}

    # Local read-only health endpoint: no public URL, login token or API key.
    website = {}
    try:
        with urlopen('http://127.0.0.1:8080/api/weather-booking/friendly-health', timeout=2) as response:
            payload = json.loads(response.read(8192))
            website = {key: payload[key] for key in ('ok', 'version', 'final_booking_confirmation') if key in payload}
    except (HTTPError, URLError, OSError, ValueError, TypeError) as error:
        website = {'health_result': type(error).__name__, 'http_status': getattr(error, 'code', None)}

    database = getattr(bridge_object, '_wx_db', None)
    report = {
        'step': 'INSPECTION_ONLY', 'project_root': str(root),
        'voice_bridge_present': bridge_object is not None,
        'active_endpoint': describe(endpoint),
        'active_versions': {key: value for key, value in engine.items()
                            if key.startswith('_MOVIO_') and key.endswith('_VERSION') and isinstance(value, str)},
        'weather_and_dialogue_handlers': handlers, 'weather_helpers': helpers,
        'captured_original_handlers': originals, 'speech_handlers': speech,
        'forecast_clients': providers,
        'booking_guards': {key: describe(getattr(database, key, None)) for key in ('accept_weather', 'permit', 'commit')},
        'disk_files': disk, 'website_weather_version': website,
    }
    print('MOVIO STEP 1 — ACTIVE HANDLERS / READ-ONLY INSPECTION')
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print('Inspection finished. No application handlers or project files were changed.')
    print('Share this output before running any new weather modification cell.')
    return report


MOVIO_WEATHER_INSPECTION = movio_inspect_weather_handlers()


MOVIO STEP 1 — ACTIVE HANDLERS / READ-ONLY INSPECTION
{
  "step": "INSPECTION_ONLY",
  "project_root": "/kaggle/working/MOVIO_MAIN_WEBSITE",
  "voice_bridge_present": true,
  "active_endpoint": {
    "name": "install_socket.<locals>.wrapped",
    "file": "/kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_friendly_weather.py",
    "line": 213,
    "code_sha256": "a11726fdf9b7d40c"
  },
  "active_versions": {
    "_MOVIO_OPTIONAL_WEATHER_VERSION": "optional-weather-1.0",
    "_MOVIO_CONFIRMATION_FLOW_VERSION": "confirmation-flow-1.0",
    "_MOVIO_TIME_ROUTE_VERSION": "time-route-1.0",
    "_MOVIO_REQUIRED_WEATHER_VERSION": "required-spoken-weather-1.0",
    "_MOVIO_BOOKING_RELIABILITY_VERSION": "booking-reliability-1.0",
    "_MOVIO_FRIENDLY_WEATHER_VERSION": "friendly-weather-1.1"
  },
  "weather_and_dialogue_handlers": {
    "advisory": {
      "name": "install_namespace.<locals>.advisory",
      "file": "/kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_friendly_weather.py",
 

In [113]:
# STEP 2 — expand the EXISTING weather report only. One Kaggle cell.
# Run after Step 1 and all your earlier setup/fix cells, before evaluation.
def movio_step_2_weather_wording():
    import ast
    import copy
    from datetime import datetime
    import hashlib
    import inspect
    import json
    import os
    from pathlib import Path
    import re
    import shutil
    import tempfile
    import types

    bridge_object = globals().get('bridge')
    engine = getattr(getattr(bridge_object, '_wx_endpoint', None), '__globals__', {})
    if not isinstance(engine, dict) or engine.get('_MOVIO_FRIENDLY_WEATHER_VERSION') != 'friendly-weather-1.1':
        raise RuntimeError('This cell targets the friendly-weather-1.1 installation verified in Step 1. Run inspection again if the session has changed.')
    advisory = engine.get('advisory')
    namespace = getattr(advisory, '__globals__', {})
    old_report = namespace.get('report')
    if namespace.get('VERSION') != 'friendly-weather-1.1' or not callable(old_report):
        raise RuntimeError('The active weather report differs from the inspected implementation. No changes made.')
    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    path = root / 'integrations/movio_friendly_weather.py'
    old_bytes = path.read_bytes()
    old_text = old_bytes.decode('utf-8')
    old_hash = hashlib.sha256(old_bytes).hexdigest()
    allowed_hashes = {
        '134589455513d88422b2367167270aa6afc0398035be0803d07bfda69987a77b',
        '17627cb019c8bcf6c336a5b2c4e659f17a3f5c6c0fba53004f21cce508ce34e5',
    }
    if old_hash not in allowed_hashes:
        raise RuntimeError('The weather helper has changed since inspection. Run Step 1 again; no files changed.')

    source = r'''def report(state):
    """MOVIO weather wording step-2-v1: forecast facts and practical advice."""
    weather = state.get('weather') or {}
    kind = category(weather)
    if kind == 'missing':
        return ''
    points = {point['location']: point for point in weather['points']}
    pickup = state.get('pickup') or 'Pickup'
    destination = state.get('destination') or 'Destination'
    vehicle = state.get('vehicle') or 'vehicle'
    chance_a = fmt(points['pickup']['precipitation_probability'])
    chance_b = fmt(points['destination']['precipitation_probability'])
    temperature = number(points['pickup'].get('temperature_2m'))

    def say(english, tamil, tanglish):
        return pick(state, english, tamil, tanglish)

    conditions = {
        'rain': ('rain is possible', 'மழை பெய்ய வாய்ப்பு இருக்கு', 'rain peiya vaaippu irukku'),
        'heavy': ('heavy rain is forecast', 'கனமழை எதிர்பார்க்கப்படுகிறது', 'heavy rain peiya vaaippu irukku'),
        'storm': ('thunderstorms are forecast', 'இடியுடன் கூடிய மழை எதிர்பார்க்கப்படுகிறது', 'thunderstorm weather irukkum ena forecast irukku'),
        'heat': ('hot conditions are forecast', 'வெப்பம் அதிகமாக இருக்கும் என எதிர்பார்க்கப்படுகிறது', 'heat adhigamaaga irukkum'),
        'wind': ('strong winds are forecast', 'பலத்த காற்று எதிர்பார்க்கப்படுகிறது', 'strong wind irukkum ena forecast irukku'),
    }
    if kind in conditions:
        condition = say(*conditions[kind])
    elif weather.get('condition') == 'cloudy':
        condition = say('cloudy skies are forecast', 'மேகமூட்டம் இருக்கும் என எதிர்பார்க்கப்படுகிறது', 'cloudy weather irukkum')
    elif weather.get('pleasant'):
        condition = say('the forecast looks pleasant', 'வானிலை பயணத்திற்கு ஏற்றதாக இருக்கும் என எதிர்பார்க்கப்படுகிறது', 'weather nandraagave irukkum')
    elif weather.get('condition') == 'clear':
        condition = say('clear skies are forecast', 'வானம் தெளிவாக இருக்கும் என எதிர்பார்க்கப்படுகிறது', 'clear sky irukkum')
    else:
        condition = say('these are the available forecast details', 'கிடைத்த வானிலை விவரங்களைச் சொல்கிறேன்', 'forecast details irukku')

    # Temperatures and percentages come only from this request's forecast.
    extra_temperature = '' if temperature is None else say(
        f'; the pickup temperature should be about {fmt(temperature)} degrees Celsius',
        f'; புறப்படும் இடத்தில் வெப்பநிலை சுமார் {fmt(temperature)} டிகிரி செல்சியஸ் ஆக இருக்கும்',
        f'; pickup temperature sumaar {fmt(temperature)} degrees Celsius irukkum',
    )
    facts = say(
        f'A quick weather update for your pickup time: {condition}{extra_temperature}; the rain chance is {chance_a}% at {pickup} and {chance_b}% at {destination}.',
        f'ஒரு சின்ன வானிலை தகவல்: புறப்படும் நேரத்தில் {condition}{extra_temperature}; {pickup} பகுதியில் {chance_a}%, {destination} பகுதியில் {chance_b}% மழை வாய்ப்பு உள்ளது.',
        f'Oru weather update: pickup nerathil {condition}{extra_temperature}; {pickup}-la {chance_a}%, {destination}-la {chance_b}% rain chance irukku.',
    )

    if kind in ('rain', 'heavy', 'storm'):
        if vehicle == 'Bike':
            advice = say(
                'Carry an umbrella for before and after the ride and keep your belongings dry; a covered car can offer more rain protection than your bike.',
                'பயணத்திற்கு முன்னும் பின்னும் பயன்படுத்த குடை எடுத்துக்கொண்டு, உங்கள் பொருட்களை நனையாமல் பாதுகாத்துக்கொள்ளுங்கள்; பைக்கை விட மூடிய காரில் மழையிலிருந்து கூடுதல் பாதுகாப்பு கிடைக்கும்.',
                'Umbrella eduthukkondu unga belongings nanaiyaamal paadhugaathukkollunga; Bike open vehicle, rain protection-ku covered car consider pannalaam.',
            )
        elif vehicle == 'Auto':
            advice = say(
                'Rain can enter an auto from the sides, so carry an umbrella for getting in and out and keep your bags covered.',
                'ஆட்டோவின் பக்கவாட்டில் மழை படலாம்; ஏறும்போதும் இறங்கும்போதும் பயன்படுத்த குடை எடுத்துக்கொண்டு, உங்கள் பைகளை நனையாமல் பாதுகாத்துக்கொள்ளுங்கள்.',
                'Auto pakkavaatil rain padalaam; umbrella eduthukkondu unga belongings nanaiyaamal paadhugaathukkollunga.',
            )
        else:
            vehicle_tamil = {'Mini': 'மினி கார்', 'Sedan': 'செடான்', 'SUV': 'எஸ் யூ வி'}.get(vehicle, 'வாகனம்')
            advice = say(
                f'Your {vehicle} provides cover during the ride; carry an umbrella for getting in and out and keep your belongings dry.',
                f'நீங்கள் தேர்வு செய்த {vehicle_tamil} பயணத்தில் மழையிலிருந்து பாதுகாப்பு அளிக்கும்; ஏறும்போதும் இறங்கும்போதும் பயன்படுத்த குடை எடுத்துக்கொண்டு, உங்கள் பொருட்களை நனையாமல் பாதுகாத்துக்கொள்ளுங்கள்.',
                f'Unga {vehicle}-la cover irukkum; umbrella eduthukkondu erumboadhum irangumboadhum unga belongings nanaiyaamal paadhugaathukkollunga.',
            )
        if kind in ('heavy', 'storm'):
            advice = say(
                'Check official local weather alerts before leaving; ' + advice[0].lower() + advice[1:],
                'கிளம்புவதற்கு முன் உள்ளூர் வானிலை எச்சரிக்கைகளைச் சரிபாருங்கள்; ' + advice,
                'Purappadum mun ulloor weather warnings paarunga; ' + advice,
            )
        return facts + ' ' + advice

    if kind == 'wind':
        advice = say(
            'Secure loose belongings and take care when boarding and getting out; a covered car is an option if you would prefer more protection than a bike.' if vehicle == 'Bike' else
            'Secure loose belongings and take care when boarding and getting out of your vehicle.',
            'காற்றில் பறக்கக்கூடிய பொருட்களைப் பாதுகாப்பாக வைத்துக்கொண்டு, ஏறும்போதும் இறங்கும்போதும் கவனமாக இருங்கள்; விரும்பினால் பைக்கிற்குப் பதிலாக மூடிய காரைப் பரிசீலிக்கலாம்.' if vehicle == 'Bike' else
            'காற்றில் பறக்கக்கூடிய பொருட்களைப் பாதுகாப்பாக வைத்துக்கொண்டு, வாகனத்தில் ஏறும்போதும் இறங்கும்போதும் கவனமாக இருங்கள்.',
            'Unga belongings paadhugaathukkollunga; Bike-ku covered car consider pannalaam, erumboadhum irangumboadhum care eduthukkondu travel pannunga.' if vehicle == 'Bike' else
            'Unga belongings paadhugaathukkollunga; erumboadhum irangumboadhum care eduthukkondu travel pannunga.',
        )
        return facts + ' ' + advice

    if kind == 'heat' or temperature is not None and temperature >= 33:
        advice = say(
            'Since your ride is open to the weather, carry drinking water and use sun protection while outside.' if vehicle in ('Bike', 'Auto') else
            'Carry drinking water and use sun protection while waiting outside and walking to your vehicle.',
            'உங்கள் வாகனத்தில் வெளிப்புற வெப்பம் படலாம்; குடிநீர் எடுத்துக்கொண்டு, வெளியில் இருக்கும்போது வெயிலிலிருந்து பாதுகாப்பாக இருங்கள்.' if vehicle in ('Bike', 'Auto') else
            'குடிநீர் எடுத்துக்கொண்டு, வாகனத்திற்காக வெளியில் காத்திருக்கும்போது வெயிலிலிருந்து பாதுகாப்பாக இருங்கள்.',
            f'Kudineer eduthukkondu heat-ku prepare pannunga; unga {vehicle} open vehicle, sun protection consider pannalaam.' if vehicle in ('Bike', 'Auto') else
            'Kudineer eduthukkondu heat-ku prepare pannunga; vehicle-ku purappadum mun sun protection consider pannalaam.',
        )
    else:
        advice = say(
            'Dress comfortably for the forecast temperature and keep your travel essentials handy.' if temperature is not None else
            'Keep your travel essentials handy before leaving for the pickup.',
            'எதிர்பார்க்கப்படும் வெப்பநிலைக்கு ஏற்ற உடை அணிந்து, பயணத்திற்குத் தேவையான பொருட்களைத் தயாராக வைத்துக்கொள்ளுங்கள்.' if temperature is not None else
            'புறப்படுவதற்கு முன் பயணத்திற்குத் தேவையான பொருட்களைத் தயாராக வைத்துக்கொள்ளுங்கள்.',
            'Unga payanathirku thevaiyaana belongings eduthukkondu purappadum mun prepare pannunga.',
        )
    # Existing policy determines whether a weather decision is needed.
    # Normal weather already proceeds directly to the final ride summary.
    if kind == 'normal':
        advice += ' ' + say('Let me go through your ride details.', 'உங்கள் பயண விவரங்களைச் சொல்கிறேன்.', 'Unga ride details sollalaam.')
    return facts + ' ' + advice
'''
    marker = 'MOVIO weather wording step-2-v1: forecast facts and practical advice.'

    tree = ast.parse(old_text, filename=str(path))
    matches = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'report']
    if len(matches) != 1:
        raise RuntimeError('Expected one existing report() function. No changes made.')
    node = matches[0]
    lines = old_text.splitlines(keepends=True)

    # Verify the LIVE report() is the one defined in the project file.
    # Compiled with this same interpreter, so it does not depend on the Python
    # version (the old marshal hash did, and could fail on a healthy session).
    def code_signature(code):
        consts = tuple(code_signature(item) if isinstance(item, types.CodeType) else item
                       for item in code.co_consts)
        return (code.co_code, code.co_names, code.co_varnames, code.co_freevars,
                code.co_cellvars, code.co_argcount, code.co_kwonlyargcount, consts)

    file_namespace = {}
    exec(compile(''.join(lines[node.lineno - 1:node.end_lineno]),
                 str(path) + ':current-report', 'exec'), file_namespace)
    file_report = file_namespace['report']
    already_current = inspect.getdoc(old_report) == marker
    if not already_current and code_signature(old_report.__code__) != code_signature(file_report.__code__):
        raise RuntimeError(
            'The live report() does not match report() in ' + str(path) + '. '
            'It may have been replaced in memory by another cell, or the weather file '
            'was edited after this kernel loaded it. No changes made; restart the '
            'kernel, run the earlier setup cells, then run this cell again.')

    patched = ''.join(lines[:node.lineno - 1]) + source.rstrip() + '\n' + ''.join(lines[node.end_lineno:])
    candidate_tree = ast.parse(patched, filename=str(path))
    compile(candidate_tree, str(path), 'exec')  # syntax check only
    original_other = ast.Module(body=[n for n in tree.body if n is not node], type_ignores=[])
    candidate_other = ast.Module(body=[n for n in candidate_tree.body if not (isinstance(n, ast.FunctionDef) and n.name == 'report')], type_ignores=[])
    if ast.dump(original_other) != ast.dump(candidate_other):
        raise RuntimeError('Unexpected edit outside report(). No files changed.')

    # Compile only this pure renderer in a private namespace; no project import,
    # provider request, speech generation, customer record or booking is used.
    candidate = {key: namespace[key] for key in ('category', 'number', 'fmt', 'pick')}
    exec(compile(source, str(path) + ':step2-renderer', 'exec'), candidate)
    render = candidate['report']
    checks = 0
    previews = {}
    fixtures = [
        ('rain', 'rain', 61, {}, 29.3),
        ('heavy', 'rain', 65, {'heavy_rain': True}, 27.5),
        ('storm', 'rain', 95, {'thunderstorm': True}, 27.5),
        ('hot', 'clear', 0, {'heat': True}, 39.2),
        ('cloudy', 'cloudy', 3, {}, 28.6),
        ('sunny', 'clear', 0, {}, 34.8),
        ('pleasant', 'clear', 0, {'pleasant': True}, 26.4),
        # Wind fixture checks wording only; wind API fields are a later step.
        ('wind', 'clear', 0, {'wind_fixture': True}, 28.1),
    ]
    for language in ('English', 'Tamil', 'Tanglish'):
        for vehicle in ('Bike', 'Auto', 'Mini', 'Sedan', 'SUV'):
            for label, condition, code, flags, temperature in fixtures:
                weather = dict(status='ok', condition=condition,
                    **{key: value for key, value in flags.items() if key != 'wind_fixture'},
                    points=[dict(location=role, weather_code=code,
                        precipitation_probability=probability, temperature_2m=temperature,
                        **({'wind_speed_10m': 45} if flags.get('wind_fixture') else {}))
                        for role, probability in (('pickup', 75.5), ('destination', 61.2))])
                state = dict(lang=language, vehicle=vehicle, pickup='Chennai Central', destination='Tambaram', weather=weather)
                before = copy.deepcopy(state)
                text = render(state)
                if not text or '75.5%' not in text or '61.2%' not in text or str(temperature) not in text:
                    raise RuntimeError('Forecast fact check failed: ' + language + '/' + label)
                if state != before or '?' in text or any(word in text.casefold() for word in ('traffic', 'surcharge', 'slower', 'delay')):
                    raise RuntimeError('The renderer changed state or added an unsupported claim/question.')
                if language == 'Tanglish' and re.search(r'[\u0b80-\u0bff]', text):
                    raise RuntimeError('Tamil pronunciation helpers must not appear in the Tanglish transcript.')
                if label == 'rain' and vehicle == 'Bike':
                    previews[language] = text
                checks += 1
            for missing in ({}, {'status': 'pending'}, {'status': 'unavailable'}, {'status': 'ok', 'points': []}):
                if render(dict(lang=language, vehicle=vehicle, weather=missing)):
                    raise RuntimeError('A missing forecast produced invented weather text.')
                checks += 1
            state = dict(lang=language, vehicle=vehicle, pickup='Chennai Central', destination='Tambaram',
                weather=dict(status='ok', condition='cloudy', points=[dict(location=role, weather_code=3, precipitation_probability=0)
                    for role in ('pickup', 'destination')]))
            if re.search(r'\d+\s*(?:degrees|டிகிரி)', render(state)):
                raise RuntimeError('A missing temperature was invented.')
            checks += 1

    # Preserve all existing handler objects and speech/model integrations.
    bindings = {key: engine.get(key) for key in (
        'advisory', 'weather_question', 'prepare_turn', 'reply_normalized',
        'weather_consented', 'summary_text', 'speech_segments', 'tts_reply',
        'completion_message', 'display_reply', 'collect_details',
    )}
    bridge_bindings = {key: getattr(bridge_object, key, None) for key in ('TTS', 'LLM', 'HTML', '_wx_endpoint', '_wx_db', '_wx_weather', 'speak', 'tanglish_speech_text')}
    adapter = getattr(bridge_object, 'TTS', None)
    adapter_synthesize = getattr(adapter, 'synthesize', None)
    # Use the existing module namespace, not a new wrapper or competing flow.
    replacement = types.FunctionType(render.__code__, namespace, 'report', render.__defaults__, render.__closure__)
    replacement.__module__ = namespace.get('__name__', 'movio_friendly_weather')
    backup = None

    def write_atomically(content):
        temporary_path = None
        try:
            with tempfile.NamedTemporaryFile(dir=path.parent, prefix='.weather_wording_', delete=False) as temporary:
                temporary_path = Path(temporary.name)
                temporary.write(content)
            shutil.copymode(path, temporary_path)
            os.replace(temporary_path, path)
        finally:
            if temporary_path is not None:
                temporary_path.unlink(missing_ok=True)

    changed = patched.encode('utf-8') != old_bytes
    try:
        if changed:
            if path.read_bytes() != old_bytes:
                raise RuntimeError('Weather source changed during validation; rerun inspection.')
            backup = root / ('weather_wording_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
            backup.mkdir()
            shutil.copy2(path, backup / path.name)
            write_atomically(patched.encode('utf-8'))
        namespace['report'] = replacement
        if any(engine.get(key) is not value for key, value in bindings.items()):
            raise RuntimeError('A booking/dialogue handler changed unexpectedly.')
        for key, value in bridge_bindings.items():
            current = getattr(bridge_object, key, None)
            if getattr(current, '__func__', current) is not getattr(value, '__func__', value):
                raise RuntimeError('A protected bridge integration changed: ' + key)
        current = getattr(adapter, 'synthesize', None)
        if getattr(current, '__func__', current) is not getattr(adapter_synthesize, '__func__', adapter_synthesize):
            raise RuntimeError('The TTS synthesis handler changed unexpectedly.')
    except Exception:
        namespace['report'] = old_report
        if changed and backup is not None:
            write_atomically(old_bytes)
        raise

    print('STEP 2 PASS — existing report() updated in place.')
    print('Syntax and report-only source check: PASS | offline rendering checks:', checks)
    print('Existing booking, confirmation, TTS and transcript handlers: unchanged.')
    print('Backup:', str(backup) if backup else 'Already current; no file rewrite needed.')
    print('ILLUSTRATIVE OFFLINE PREVIEWS — these percentages and temperatures are test fixtures, not live weather:')
    for language, text in previews.items():
        print('\n' + language + ': ' + text)
    print('\nThe existing weather question is appended by the current dialogue flow.')
    print('Close the old call popup and open a NEW AI Call from the dashboard.')
    print('Share this output before the next step. Natural no-change replies and live wind retrieval are not changed by Step 2.')
    return dict(step=2, status='PASS', wording_version='step-2-v1', offline_checks=checks,
                modified_file=str(path) if changed else None, backup=str(backup) if backup else None)


MOVIO_STEP_2_WEATHER_WORDING = movio_step_2_weather_wording()

STEP 2 PASS — existing report() updated in place.
Syntax and report-only source check: PASS | offline rendering checks: 195
Existing booking, confirmation, TTS and transcript handlers: unchanged.
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/weather_wording_backup_20261028_070728_689425
ILLUSTRATIVE OFFLINE PREVIEWS — these percentages and temperatures are test fixtures, not live weather:

English: A quick weather update for your pickup time: rain is possible; the pickup temperature should be about 29.3 degrees Celsius; the rain chance is 75.5% at Chennai Central and 61.2% at Tambaram. Carry an umbrella for before and after the ride and keep your belongings dry; a covered car can offer more rain protection than your bike.

Tamil: ஒரு சின்ன வானிலை தகவல்: புறப்படும் நேரத்தில் மழை பெய்ய வாய்ப்பு இருக்கு; புறப்படும் இடத்தில் வெப்பநிலை சுமார் 29.3 டிகிரி செல்சியஸ் ஆக இருக்கும்; Chennai Central பகுதியில் 75.5%, Tambaram பகுதியில் 61.2% மழை வாய்ப்பு உள்ளது. பயணத்திற்கு முன்னும் பின்னும் பயன்படுத

In [114]:
# STEP 3A — diagnose route failures and missing weather before changing them.
# No patches, booking/cancellation calls, speech generation or database writes.
# Uses the CURRENT geocoder, routing service and weather adapter once each.
# Existing lookup caches/logs may update as part of those diagnostic lookups.
def movio_step_3a_route_weather_diagnostics():
    import copy
    from datetime import datetime, timedelta
    import inspect
    import json
    from pathlib import Path
    import sqlite3
    import time

    b = globals().get('bridge')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    required = ('update_time_memory', 'valid_geo')
    if not isinstance(engine, dict) or any(not callable(engine.get(key)) for key in required):
        raise RuntimeError('Run this in the current working MOVIO kernel, after its existing setup.')
    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    now = datetime.now(engine['TZ'])
    report_function = getattr(engine.get('advisory'), '__globals__', {}).get('report')
    output = dict(local_now=now.isoformat(),
        wording_step_2_active=bool(report_function and 'step-2-v1' in (inspect.getdoc(report_function) or '')),
        extraction=[], saved_booking_forecasts=[], live_lookup={})

    def weather_summary(weather):
        weather = weather if isinstance(weather, dict) else {}
        result = {key: weather.get(key) for key in ('status', 'source', 'reason', 'http_status', 'condition') if key in weather}
        policy = weather.get('_friendly_notice_policy') or {}
        result['advisory_skip_reason'] = policy.get('reason')
        result['points'] = [{key: point.get(key) for key in ('location', 'valid_at', 'weather_code', 'precipitation_probability', 'temperature_2m')}
                            for point in (weather.get('points') or []) if isinstance(point, dict)]
        return result

    print('STEP 3A: checking extraction with rules only; no LLM or TTS generation.', flush=True)
    slots = getattr(b, '_wx_group_slots_module', None)
    if slots is None:
        try:
            slots = inspect.getclosurevars(engine['collect_details']).nonlocals.get('slots')
        except (TypeError, ValueError, KeyError):
            pass
    extract = slots.get('extract_fields') if isinstance(slots, dict) else getattr(slots, 'extract_fields', None)
    examples = [
        ('English', 'I need to book a cab from VIT Chennai to Vandalur. We are four people. I need SUV book a cab at 7 today.'),
        ('Tamil', 'எனக்கு ஒரு மினி கார் வேணும். நாங்க ரெண்டு பேர் இருக்கோம். தாம்பரத்திலிருந்து வண்டலூர் போகணும். நாளைக்கு மாலை 7 மணிக்கு.'),
        ('Tanglish', 'SUV வேணும். இன்னைக்கு 7-க்கு bike வேணும்.'),
    ]
    for language, text in examples:
        state = dict(lang=language, mode='book', stage='count')
        try:
            if not callable(extract):
                raise RuntimeError('Shared parser not found')
            parsed = extract(b, text, state, use_qwen=False, engine=engine)
            state.update(parsed.get('updates', {}))
            when, error = engine['update_time_memory'](state, text, now)
            output['extraction'].append(dict(language=language, updates=parsed.get('updates', {}),
                conflicts=parsed.get('ambiguities', {}), time_components=state.get('_wx_time'),
                resolved_pickup_time=when.isoformat() if when else None, time_error=error))
        except Exception as error:
            output['extraction'].append(dict(language=language, diagnostic_error=type(error).__name__))

    print('Reading only the weather attached to the three reported bookings (#34, #35, #36).', flush=True)
    db = getattr(b, '_wx_db', None)
    db_path = Path(getattr(db, 'path', root / 'movio_demo.db'))
    if not db_path.is_absolute():
        db_path = root / db_path
    try:
        with sqlite3.connect(db_path.resolve().as_uri() + '?mode=ro', uri=True, timeout=2) as connection:
            connection.row_factory = sqlite3.Row
            columns = {row[1] for row in connection.execute('PRAGMA table_info(rides)')}
            if not {'id', 'weather_json'} <= columns:
                raise RuntimeError('Expected ride weather columns not found')
            select = [key for key in ('id', 'status', 'source', 'weather_json') if key in columns]
            rows = connection.execute('SELECT ' + ','.join(select) + ' FROM rides WHERE id IN (?,?,?) ORDER BY id', (34, 35, 36)).fetchall()
            for row in rows:
                try:
                    weather = json.loads(row['weather_json'] or '{}')
                    item = dict(booking_id=row['id'], **weather_summary(weather))
                except (ValueError, TypeError):
                    item = dict(booking_id=row['id'], weather_parse_error=True)
                output['saved_booking_forecasts'].append(item)
            if not rows:
                output['saved_booking_note'] = 'The three reported IDs are absent from this active database.'
    except (OSError, sqlite3.Error, RuntimeError) as error:
        output['saved_booking_read_error'] = type(error).__name__

    # New, detached probe: explicit future time avoids AM/PM ambiguity.
    # It is never passed to prepare_turn(), summary/permit/commit or a socket.
    probe = dict(lang='English', mode='book', count=4, vehicle='SUV',
        pickup='VIT Chennai', destination='Vandalur',
        pickup_at=(now + timedelta(days=1)).replace(hour=19, minute=0, second=0, microsecond=0).isoformat())
    for role in ('pickup', 'destination'):
        print('Checking existing geocoder:', probe[role], flush=True)
        started = time.perf_counter()
        try:
            geo = b.geocode(probe[role])
            good = engine['valid_geo'](geo)
            probe[role + '_geo'] = geo if good else None
            output['live_lookup'][role] = dict(coordinates_valid=good,
                **({key: geo.get(key) for key in ('name', 'label', 'lat', 'lng')} if good else {}))
        except Exception as error:
            probe[role + '_geo'] = None
            output['live_lookup'][role] = dict(coordinates_valid=False, error_type=type(error).__name__)
        output['live_lookup'][role]['elapsed_s'] = round(time.perf_counter() - started, 2)

    if all(engine['valid_geo'](probe.get(role + '_geo')) for role in ('pickup', 'destination')):
        print('Checking existing road/fare service for VIT Chennai to Vandalur.', flush=True)
        try:
            estimate = b.road_fare(copy.deepcopy(probe))
            output['live_lookup']['route_estimate'] = {key: estimate.get(key) for key in ('km', 'fare')} if isinstance(estimate, dict) else 'NO_VALID_ESTIMATE'
        except Exception as error:
            output['live_lookup']['route_estimate'] = dict(error_type=type(error).__name__)
        weather_client = getattr(b, '_wx_required_forecast', None) or getattr(b, '_wx_weather', None)
        print('Checking existing weather adapter for the same places, tomorrow at 7 PM.', flush=True)
        try:
            weather = weather_client.forecast([probe['pickup_geo'], probe['destination_geo']], datetime.fromisoformat(probe['pickup_at']), now=now)
            output['live_lookup']['weather'] = weather_summary(weather)
        except Exception as error:
            output['live_lookup']['weather'] = dict(error_type=type(error).__name__)
    else:
        output['live_lookup']['route_and_weather'] = 'NOT_QUERIED: valid coordinates are needed first.'

    output['recent_lookup_events'] = {}
    allowed = ('event', 'reason', 'http_status', 'error_type', 'retry_after_s', 'phase', 'valid_at')
    for name in ('booking_lookup.log', 'required_weather.log'):
        path = root / name
        if not path.is_file():
            continue
        try:
            with path.open('rb') as handle:
                handle.seek(max(0, path.stat().st_size - 16000))
                lines = handle.read().decode('utf-8', errors='replace').splitlines()
            events = []
            for line in lines:
                try:
                    event = json.loads(line[line.index('{'):])
                    events.append({key: event[key] for key in allowed if key in event})
                except (ValueError, TypeError):
                    continue
            output['recent_lookup_events'][name] = events[-6:]
        except OSError:
            output['recent_lookup_events'][name] = 'Log could not be read.'

    print('\nMOVIO STEP 3A DIAGNOSTICS — no bookings or source changes:')
    print(json.dumps(output, indent=2, ensure_ascii=False, default=str))
    print('Share this output. It distinguishes parsing, geocoding, road routing and forecast failures.')
    return output


MOVIO_STEP_3A_DIAGNOSTICS = movio_step_3a_route_weather_diagnostics()


STEP 3A: checking extraction with rules only; no LLM or TTS generation.
Reading only the weather attached to the three reported bookings (#34, #35, #36).
Checking existing geocoder: VIT Chennai
Checking existing geocoder: Vandalur
Checking existing road/fare service for VIT Chennai to Vandalur.
Checking existing weather adapter for the same places, tomorrow at 7 PM.

MOVIO STEP 3A DIAGNOSTICS — no bookings or source changes:
{
  "local_now": "2026-10-28T12:37:28.729523+05:30",
  "wording_step_2_active": true,
  "extraction": [
    {
      "language": "English",
      "updates": {
        "count": 4,
        "vehicle": "SUV",
        "destination": "Vandalur",
        "pickup": "VIT Chennai"
      },
      "conflicts": {},
      "time_components": {
        "date": "2026-10-28",
        "hour": 7,
        "minute": 0,
        "clock24": false
      },
      "resolved_pickup_time": "2026-10-28T19:00:00+05:30",
      "time_error": null
    },
    {
      "language": "Tamil",
      "update

In [115]:
# STEP 3B — correct the ORIGINAL geocoder in place; no extra wrapper.
# Run after your current fixes/Step 2 and before evaluation. One Kaggle cell.
def movio_step_3b_geocoder_fix():
    import ast
    import copy
    from datetime import datetime, timedelta
    import inspect
    import os
    from pathlib import Path
    import shutil
    import tempfile
    import threading
    import types
    import json

    b = globals().get('bridge')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    if not isinstance(engine, dict) or engine.get('_MOVIO_TIME_ROUTE_VERSION') != 'time-route-1.0':
        raise RuntimeError('Run this in the MOVIO session inspected in Step 3A.')
    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    frontier = [getattr(b, 'geocode', None), getattr(b, '_before_tambaram_geocode', None)]
    seen, matches = set(), []
    while frontier:
        function = frontier.pop()
        if not isinstance(function, types.FunctionType) or id(function) in seen:
            continue
        seen.add(id(function))
        namespace = function.__globals__
        if function.__name__ == 'geocode' and {'GEOCODE_CACHE', 'GEO_LOCK', 'LAST_GEO', 'clean_place', 'plausible_place'} <= namespace.keys():
            matches.append(function)
        frontier.append(getattr(function, '_original_geocode', None))
        try:
            captures = inspect.getclosurevars(function).nonlocals
            frontier.extend(captures.get(key) for key in ('original', 'original_geo', 'previous'))
        except (TypeError, ValueError):
            pass
    if len(matches) != 1:
        raise RuntimeError('Could not identify one original geocoder safely. No files or handlers changed.')
    raw = matches[0]
    path = Path(raw.__globals__.get('__file__', raw.__code__.co_filename)).resolve()
    if root not in path.parents or path.suffix != '.py' or raw.__closure__:
        raise RuntimeError('The original geocoder is outside this working project or has unexpected captures. No changes made.')
    before = path.read_bytes()
    text = before.decode('utf-8')
    tree = ast.parse(text, filename=str(path))
    functions = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'geocode']
    if len(functions) != 1:
        raise RuntimeError('Expected one original geocode() definition. No changes made.')
    node = functions[0]
    original_source = r'''def geocode(q):
 q=clean_place(q)
 if not plausible_place(q):return None
 city=os.environ.get('MOVIO_SERVICE_CITY','Chennai').strip() or 'Chennai'
 box=os.environ.get('MOVIO_GEO_VIEWBOX','79.95,13.35,80.45,12.65')
 try:
  west,north,east,south=[float(v) for v in box.split(',')]
  if not (-180<=west<east<=180 and -90<=south<north<=90):raise ValueError()
 except ValueError:raise ValueError('Invalid MOVIO_GEO_VIEWBOX')
 key=(city.casefold(),box,q.casefold())
 if key in GEOCODE_CACHE:return GEOCODE_CACHE[key]
 with GEO_LOCK:
  if key in GEOCODE_CACHE:return GEOCODE_CACHE[key]
  delay=1.1-(time.monotonic()-LAST_GEO[0])
  if delay>0:time.sleep(delay)
  LAST_GEO[0]=time.monotonic()
  query=q if city.casefold() in q.casefold() else q+', '+city
  endpoint=os.environ.get('MOVIO_GEOCODER_URL','https://nominatim.openstreetmap.org/search')
  r=requests.get(endpoint,params={'q':query,'format':'json','limit':3,'countrycodes':'in','addressdetails':1,'viewbox':box,'bounded':1},headers={'User-Agent':'MOVIO-Ride-Booking-Research/5.1','Accept-Language':'en'},timeout=12)
  r.raise_for_status();data=r.json();res=None
  for d in data:
   lat=float(d['lat']);lng=float(d['lon'])
   if not (south<=lat<=north and west<=lng<=east):continue
   name=d.get('name') or d['display_name'].split(',')[0]
   label=q if q in VOICE_PLACE_ALIASES else name
   res={'name':name,'label':label,'address':d['display_name'],'lat':lat,'lng':lng};break
  GEOCODE_CACHE[key]=res
  return res
'''
    source = r'''def geocode(q):
    """MOVIO step-3B-v1: match each query to a provider-named place."""
    import copy
    import math
    import unicodedata

    q = clean_place(q)
    if not plausible_place(q):
        return None
    city = os.environ.get('MOVIO_SERVICE_CITY', 'Chennai').strip() or 'Chennai'
    box = os.environ.get('MOVIO_GEO_VIEWBOX', '79.95,13.35,80.45,12.65')
    try:
        west, north, east, south = [float(value) for value in box.split(',')]
        if not all(math.isfinite(value) for value in (west, north, east, south)) or not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
            raise ValueError()
    except (ValueError, TypeError):
        raise ValueError('Invalid MOVIO_GEO_VIEWBOX')

    def words(value):
        value = unicodedata.normalize('NFKC', str(value or '')).casefold()
        return tuple(re.findall(r'[a-z0-9]+|[\u0b80-\u0bff]+', value))

    stop = {'the', 'of', 'and', 'at', 'in'}
    city_words = set(words(city))
    variants = {str(q)}
    for canonical, aliases in VOICE_PLACE_ALIASES.items():
        aliases = [value for value in aliases if isinstance(value, str)]
        if words(q) in [words(value) for value in [canonical] + aliases]:
            variants.update([canonical] + aliases)
    targets = []
    for variant in variants:
        tokens = tuple(value for value in words(variant) if value not in stop)
        core = tuple(value for value in tokens if value not in city_words)
        if tokens:
            targets.append((tokens, core or tokens))

    def coordinates(item):
        if not isinstance(item, dict):
            return None
        try:
            lat = float(item.get('lat'))
            lng = float(item.get('lon', item.get('lng')))
            return (lat, lng) if math.isfinite(lat) and math.isfinite(lng) and south <= lat <= north and west <= lng <= east else None
        except (ValueError, TypeError, OverflowError):
            return None

    def match_score(item):
        if coordinates(item) is None:
            return 0
        names = [item.get('name')]
        details = item.get('namedetails')
        if isinstance(details, dict):
            names += [value for value in details.values() if isinstance(value, str)]
        display = str(item.get('display_name') or item.get('address') or '')
        if not item.get('name'):
            names.append(display.split(',')[0])
        best = 0
        for name in names:
            for spelling in str(name or '').split(';'):
                tokens = tuple(value for value in words(spelling) if value not in stop)
                if not tokens:
                    continue
                # Provider-derived acronym matching: e.g. VIT -> Vellore
                # Institute of Technology. No coordinates or aliases invented.
                acronyms = {''.join(value[0] for value in tokens[start:start + length])
                            for start in range(len(tokens)) for length in range(2, 6)
                            if len(tokens[start:start + length]) == length and all(value.isalpha() for value in tokens[start:start + length])}
                for full, core in targets:
                    if full == tokens:
                        best = max(best, 120)
                    elif core == tokens:
                        best = max(best, 110)
                    elif all(value in tokens or len(value) >= 2 and value in acronyms
                             or len(value) == 1 and len(core) > 1 and any(token.startswith(value) for token in tokens)
                             for value in core):
                        best = max(best, 80)
        # Full street addresses may need the provider's address components.
        # A locality-only query never matches merely because a campus address
        # mentions that locality or a road named after it.
        context = set(words(display))
        address = item.get('address')
        if isinstance(address, dict):
            for value in address.values():
                context.update(words(value))
        street_words = {'road', 'street', 'avenue', 'lane', 'salai', 'சாலை', 'தெரு'}
        for full, core in targets:
            if (any(value.isdigit() for value in full) or street_words.intersection(full)) and all(value in context for value in core):
                best = max(best, 60)
        return best

    key = (city.casefold(), box, str(q).casefold())
    # Use the original shared lock and cache. Reject stale mismatched entries
    # lazily, leaving unrelated successful queries intact.
    with GEO_LOCK:
        cached = GEOCODE_CACHE.get(key)
        if cached is not None and match_score(cached):
            return copy.deepcopy(cached)
        GEOCODE_CACHE.pop(key, None)
        delay = 1.1 - (time.monotonic() - LAST_GEO[0])
        if delay > 0:
            time.sleep(delay)
        LAST_GEO[0] = time.monotonic()
        endpoint = os.environ.get('MOVIO_GEOCODER_URL', 'https://nominatim.openstreetmap.org/search')
        # The existing service box already limits geography; appending Chennai
        # can bias a suburb query toward an unrelated Chennai institution.
        response = requests.get(endpoint, params={
            'q': q, 'format': 'json', 'limit': 10, 'countrycodes': 'in',
            'addressdetails': 1, 'namedetails': 1, 'viewbox': box, 'bounded': 1,
        }, headers={'User-Agent': 'MOVIO-Ride-Booking-Research/5.1', 'Accept-Language': 'en'}, timeout=12)
        response.raise_for_status()
        payload = response.json()
        if not isinstance(payload, list):
            raise ValueError('Unexpected geocoder response')
        ranked = [(match_score(item), index, item) for index, item in enumerate(payload) if isinstance(item, dict)]
        ranked = [item for item in ranked if item[0] > 0]
        result = None
        if ranked:
            _, _, item = max(ranked, key=lambda item: (item[0], -item[1]))
            lat, lng = coordinates(item)
            name = item.get('name') or str(item.get('display_name', '')).split(',')[0]
            result = dict(name=name, label=q if q in VOICE_PLACE_ALIASES else name,
                address=item.get('display_name', ''), lat=lat, lng=lng,
                namedetails={key: value for key, value in (item.get('namedetails') or {}).items() if isinstance(value, str)})
        GEOCODE_CACHE[key] = copy.deepcopy(result)
        return copy.deepcopy(result)
'''
    allowed = {ast.dump(ast.parse(value).body[0]) for value in (original_source, source)}
    if ast.dump(node) not in allowed:
        raise RuntimeError('The original geocoder differs from the inspected source. No changes made; share its function source before continuing.')

    def function_code(module_code):
        codes = [value for value in module_code.co_consts if isinstance(value, types.CodeType) and value.co_name == 'geocode']
        if len(codes) != 1:
            raise RuntimeError('Expected one compiled geocoder.')
        return codes[0]

    def signature(code):
        return (code.co_code, code.co_argcount, code.co_posonlyargcount, code.co_kwonlyargcount,
            code.co_flags, code.co_names, code.co_varnames, code.co_freevars, code.co_cellvars,
            tuple(signature(value) if isinstance(value, types.CodeType) else value for value in code.co_consts))

    old_compiled = function_code(compile(tree, str(path), 'exec', dont_inherit=True))
    if signature(raw.__code__) != signature(old_compiled):
        raise RuntimeError('The live original geocoder differs from the current file. No changes made; inspect the running version first.')
    lines = text.splitlines(keepends=True)
    patched = ''.join(lines[:node.lineno - 1]) + source.rstrip() + '\n' + ''.join(lines[node.end_lineno:])
    changed_tree = ast.parse(patched, filename=str(path))
    compiled = compile(changed_tree, str(path), 'exec', dont_inherit=True)
    new_code = function_code(compiled)
    old_other = ast.Module(body=[value for value in tree.body if value is not node], type_ignores=[])
    new_other = ast.Module(body=[value for value in changed_tree.body if not (isinstance(value, ast.FunctionDef) and value.name == 'geocode')], type_ignores=[])
    if ast.dump(old_other) != ast.dump(new_other):
        raise RuntimeError('Unexpected changes outside the original geocoder. No changes made.')

    # Offline provider fixtures only. None of these coordinates is installed
    # as a fallback or production result. No network or model calls here.
    class Clock:
        def __init__(self): self.value = 100.0
        def monotonic(self): return self.value
        def sleep(self, seconds): self.value += seconds

    def fixture(payload, cache=None):
        calls = []
        def get(url, **kwargs):
            calls.append(kwargs['params'])
            def check():
                if isinstance(payload, Exception): raise payload
            return types.SimpleNamespace(raise_for_status=check, json=lambda: copy.deepcopy(payload))
        box = '79.95,13.35,80.45,12.65'
        env = dict(clean_place=lambda value: {'வண்டலூர்': 'Vandalur', 'vandallur': 'Vandalur'}.get(str(value), str(value).strip()),
            plausible_place=lambda value: bool(value), os=types.SimpleNamespace(environ={'MOVIO_GEO_VIEWBOX': box, 'MOVIO_SERVICE_CITY': 'Chennai'}),
            re=__import__('re'), time=Clock(), requests=types.SimpleNamespace(get=get), VOICE_PLACE_ALIASES={},
            GEO_LOCK=threading.RLock(), GEOCODE_CACHE=copy.deepcopy(cache or {}), LAST_GEO=[0.0])
        exec(compile(source, 'offline_geocoder_fixture', 'exec', dont_inherit=True), env)
        return env, calls, box

    campus = dict(name='Vellore Institute of Technology - Chennai', lat='12.84', lon='80.15', display_name='VIT, Vandalur-Kelambakkam Road, Chennai')
    suburb = dict(name='Vandalur', lat='12.89', lon='80.08', display_name='Vandalur, Tamil Nadu, India')
    cached_campus = dict(name=campus['name'], label=campus['name'], lat=12.84, lng=80.15, address=campus['display_name'])
    key = ('chennai', '79.95,13.35,80.45,12.65', 'vandalur')
    env, calls, box = fixture([campus, suburb], {key: cached_campus})
    result = env['geocode']('Vandalur')
    assert result['name'] == 'Vandalur' and len(calls) == 1 and calls[0]['q'] == 'Vandalur'
    result['lat'] = -1
    assert env['geocode']('Vandalur')['lat'] == 12.89 and len(calls) == 1
    assert env['GEOCODE_CACHE'][key]['name'] == 'Vandalur'
    checks = 3
    for query in ('Vandalur', 'வண்டலூர்', 'vandallur'):
        env, calls, _ = fixture([campus, suburb])
        assert env['geocode'](query)['name'] == 'Vandalur'
        checks += 1
    env, calls, _ = fixture([campus])
    assert env['geocode']('Vandalur') is None
    checks += 1
    env, calls, _ = fixture([], {('chennai', box, 'vit chennai'): cached_campus})
    assert env['geocode']('VIT Chennai')['name'] == campus['name'] and not calls
    checks += 1
    env, calls, _ = fixture([dict(suburb, lat='90'), dict(suburb, lat='nan')])
    assert env['geocode']('Vandalur') is None
    checks += 1
    env, calls, _ = fixture(RuntimeError('fixture service refusal'))
    try:
        env['geocode']('Vandalur')
        raise AssertionError('Provider failure was hidden')
    except RuntimeError:
        assert len(calls) == 1
    checks += 1

    # Updating the original function object preserves every existing wrapper
    # and captured reference. No new event handler or booking flow is added.
    engine_bindings = {key: engine.get(key) for key in ('prepare_turn', 'collect_details', 'summary_text', 'reply_normalized', 'advisory', 'tts_reply')}
    bridge_bindings = {key: getattr(b, key, None) for key in ('geocode', 'road_fare', '_wx_endpoint', '_wx_db', 'TTS', 'LLM', 'speak')}
    old_code, old_doc = raw.__code__, raw.__doc__
    backup = None
    changed = patched.encode('utf-8') != before

    def atomic_write(data):
        temporary_path = None
        try:
            with tempfile.NamedTemporaryFile(dir=path.parent, prefix='.geocoder_', delete=False) as temporary:
                temporary_path = Path(temporary.name)
                temporary.write(data)
            shutil.copymode(path, temporary_path)
            os.replace(temporary_path, path)
        finally:
            if temporary_path is not None: temporary_path.unlink(missing_ok=True)

    try:
        if changed:
            if path.read_bytes() != before: raise RuntimeError('Source changed during validation.')
            backup = root / ('geocoder_query_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
            backup.mkdir()
            shutil.copy2(path, backup / path.name)
            atomic_write(patched.encode('utf-8'))
        raw.__code__ = new_code
        raw.__doc__ = new_code.co_consts[0]
        for name, value in engine_bindings.items():
            if engine.get(name) is not value: raise RuntimeError('Unexpected dialogue handler change: ' + name)
        for name, value in bridge_bindings.items():
            current = getattr(b, name, None)
            if getattr(current, '__func__', current) is not getattr(value, '__func__', value):
                raise RuntimeError('Unexpected bridge integration change: ' + name)
    except Exception:
        raw.__code__, raw.__doc__ = old_code, old_doc
        if changed and backup is not None: atomic_write(before)
        raise

    print('STEP 3B — original geocoder updated in place; syntax/source isolation checks PASS.', flush=True)
    print('Offline geocoder checks:', checks, '| Backup:', str(backup) if backup else 'Already current; no file rewrite.', flush=True)
    print('Now checking the real existing services; no booking will be created.', flush=True)
    probe = dict(lang='English', count=4, vehicle='SUV', pickup='VIT Chennai', destination='Vandalur')
    verified = dict(locations={}, distinct_coordinates=False, route_estimate=None)
    for role in ('pickup', 'destination'):
        print('Lookup:', probe[role], flush=True)
        try:
            geo = b.geocode(probe[role])
            probe[role + '_geo'] = geo
            verified['locations'][role] = {key: geo.get(key) for key in ('name', 'label', 'lat', 'lng')} if engine['valid_geo'](geo) else dict(result='NOT_RESOLVED')
        except Exception as error:
            probe[role + '_geo'] = None
            verified['locations'][role] = dict(error_type=type(error).__name__)
    if all(engine['valid_geo'](probe.get(role + '_geo')) for role in ('pickup', 'destination')):
        a, z = probe['pickup_geo'], probe['destination_geo']
        verified['distinct_coordinates'] = (float(a['lat']), float(a['lng'])) != (float(z['lat']), float(z['lng']))
        if verified['distinct_coordinates']:
            try:
                fare = b.road_fare(copy.deepcopy(probe))
                if isinstance(fare, dict): verified['route_estimate'] = {key: fare.get(key) for key in ('km', 'fare')}
            except Exception as error:
                verified['route_error'] = type(error).__name__
    print(json.dumps(verified, indent=2, ensure_ascii=False), flush=True)
    print('Live geocoder verification:', 'PASS' if verified['distinct_coordinates'] else 'NOT VERIFIED — share this output; no coordinates were invented.')
    print('Live road/fare estimate:', 'AVAILABLE' if verified['route_estimate'] else 'NOT AVAILABLE')
    print('Close the old popup and open a NEW AI Call. Existing saved bookings are untouched.')
    print('Share this output before the next fix. Missing weather playback and time/confirmation wording are separate next steps.')
    return dict(step='3B', status='INSTALLED', offline_checks=checks, modified_file=str(path) if changed else None,
                backup=str(backup) if backup else None, live_verification=verified)


MOVIO_STEP_3B_GEOCODER = movio_step_3b_geocoder_fix()


STEP 3B — original geocoder updated in place; syntax/source isolation checks PASS.
Offline geocoder checks: 10 | Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/geocoder_query_backup_20261028_070730_850006
Now checking the real existing services; no booking will be created.
Lookup: VIT Chennai
Lookup: Vandalur
{
  "locations": {
    "pickup": {
      "name": "Vellore Institute of Technology - Chennai",
      "label": "Vellore Institute of Technology - Chennai",
      "lat": 12.8429461,
      "lng": 80.15541
    },
    "destination": {
      "name": "Vandalur",
      "label": "Vandalur",
      "lat": 12.8921486,
      "lng": 80.0829603
    }
  },
  "distinct_coordinates": true,
  "route_estimate": {
    "km": 12.7,
    "fare": 460
  }
}
Live geocoder verification: PASS
Live road/fare estimate: AVAILABLE
Close the old popup and open a NEW AI Call. Existing saved bookings are untouched.
Share this output before the next fix. Missing weather playback and time/confirmation wording are separate n

In [116]:
# STEP 3C — restore available normal-weather advice before final review.
# One Kaggle cell, after Steps 2/3B and before evaluation.
# Updates two EXISTING function bodies; no new socket or event handlers.
def movio_step_3c_weather_delivery():
    import ast
    import contextvars
    import copy
    from datetime import datetime
    import inspect
    import math
    import os
    from pathlib import Path
    import shutil
    import tempfile
    import textwrap
    import threading
    import types

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',{})
    if not isinstance(engine,dict) or engine.get('_MOVIO_FRIENDLY_WEATHER_VERSION')!='friendly-weather-1.1':
        raise RuntimeError('Run this in the same working MOVIO session used for Steps 1–3B.')
    namespace=getattr(engine.get('advisory'),'__globals__',{})
    if 'step-2-v1' not in (inspect.getdoc(namespace.get('report')) or ''):
        raise RuntimeError('Step 2 weather wording is not active. Run Step 2 before this cell.')
    root=Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    path=root/'integrations/movio_friendly_weather.py'
    before=path.read_bytes();text=before.decode('utf-8')
    changes={
'prepare_turn': r'''def prepare_turn(b,weather,state,text,now=None):
    """MOVIO step-3C-v1: deliver normal-weather advice at final review."""
    context=TURN.get() or {};prior_stage=state.get('stage');ack=False
    # A normal-weather notice is acknowledged only by the existing reviewed
    # response record. Never manufacture a nonce, playback ACK or consent.
    record=state.get('_friendly_normal_notice') or {}
    if isinstance(record,dict) and record.get('version')=='step-3C-v1' and not record.get('acknowledged') and context.get('sid'):
        try:
            row=context['db'].row(context['sid'])
            if (row.get('played') and row.get('summary_kind')=='final'
                    and row.get('summary_hash')==record.get('summary_hash')
                    and row.get('summary_hash')==engine['digest'](engine['snapshot'](state))):
                state['_friendly_normal_notice']={**record,'acknowledged':True}
        except (KeyError,engine['GuardError']):pass
    # Retain the existing real weather-choice/correction protocol unchanged.
    if prior_stage=='weather_advisory' and str(text).strip() and context.get('sid'):
        probe=copy.deepcopy(state)
        parsed=engine['collect_details'](b,probe,text)
        if parsed.get('changed'):
            db=context['db'];row=db.row(context['sid'])
            try:state=db.accept_weather(context['sid'],row['epoch'],context.get('nonce'),state);ack=True
            except engine['GuardError']:pass
    norm=previous['reply_normalized'](text)
    if prior_stage=='weather_advisory' and norm in {'earlier','leave earlier','choose an earlier pickup time','change pickup time','முன்னதாக வேண்டும்','வேறு நேரம் வேண்டும்','earlier venum'}:
        state['stage']='weather_change_time';state['dialogue_state']='COLLECTING_DETAILS'
        return state,pick(state,'What pickup time would you prefer? I will check the forecast for that time.',
            'எத்தனை மணிக்கு வாகனம் வேண்டும்? அந்த நேரத்திற்கான வானிலையைச் சரிபார்க்கிறேன்.',
            'Endha pickup time venum? Andha time-ku weather check pannren.')
    state,answer=previous['prepare_turn'](b,weather,state,text,now=now)
    kind=category(state.get('weather'))
    if state.get('stage') in ('weather_advisory','weather_pending') and kind in ('normal','missing'):
        w=state.setdefault('weather',{});w.pop('_friendly_notice_policy',None)
        reason='forecast_missing' if kind=='missing' else 'no_weather_choice_needed'
        w['_friendly_notice_policy']=dict(version=VERSION,reason=reason,scope=scope(state,engine))
        state.pop('_wx_weather_consent',None)
        state,answer=base_prepare(b,weather,state,'',now=now)
        if state.get('stage')=='confirm':
            state['dialogue_state']='AWAITING_BOOKING_CONFIRMATION'
    if state.get('stage')=='confirm' and (context.get('choice') or ack):
        context.pop('choice',None)
        intro=pick(state,'Sure, let me confirm your updated ride details. ',
            'சரி, உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம். ',
            'Seri, unga ride details confirm pannalaam. ')
        answer=intro+answer
    # Attach actual normal-weather advice when a final review is ready,
    # including after a failed route lookup. Missing forecasts stay optional.
    if state.get('stage')=='confirm' and state.get('_wx_mode')!='manual' and category(state.get('weather'))=='normal':
        details=copy.deepcopy(state['weather']);details.pop('_friendly_notice_policy',None)
        notice_scope=engine['digest']({**{key:state.get(key) for key in ('pickup','destination','pickup_geo','destination_geo','pickup_at','vehicle','lang')},'weather':details})
        record=state.get('_friendly_normal_notice') or {}
        heard=isinstance(record,dict) and record.get('version')=='step-3C-v1' and record.get('scope')==notice_scope and record.get('acknowledged') is True
        if not heard:
            notice=advisory(state)
            if notice:
                state['_friendly_normal_notice']=dict(version='step-3C-v1',scope=notice_scope,acknowledged=False,
                    summary_hash=engine['digest'](engine['snapshot'](state)))
                # Two newlines identify the normal-weather/final-review pair
                # for the existing speech chunk handler; transcript stays plain.
                if not str(answer).startswith(notice):answer=notice+'\n\n'+answer
    return state,answer
''',
'speech_segments': r'''def speech_segments(text):
    """MOVIO step-3C-v1: pause between weather and the final ride summary."""
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        normal_review_pair=value.startswith(prefix+'\n\n')
        # Keep interactive weather choices on their existing speech path.
        return [prefix,rest] if rest and (normal_review_pair or 'estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
'''
}
    originals={
'prepare_turn': r'''def prepare_turn(b,weather,state,text,now=None):
    context=TURN.get() or {};prior_stage=state.get('stage');ack=False
    # Explicit vehicle/count/location corrections are also weather choices.
    # Reuse the real nonce and playback acknowledgement before applying them.
    if prior_stage=='weather_advisory' and str(text).strip() and context.get('sid'):
        probe=copy.deepcopy(state)
        parsed=engine['collect_details'](b,probe,text)
        if parsed.get('changed'):
            db=context['db'];row=db.row(context['sid'])
            try:state=db.accept_weather(context['sid'],row['epoch'],context.get('nonce'),state);ack=True
            except engine['GuardError']:pass
    norm=previous['reply_normalized'](text)
    if prior_stage=='weather_advisory' and norm in {'earlier','leave earlier','choose an earlier pickup time','change pickup time','முன்னதாக வேண்டும்','வேறு நேரம் வேண்டும்','earlier venum'}:
        state['stage']='weather_change_time';state['dialogue_state']='COLLECTING_DETAILS'
        return state,pick(state,'What pickup time would you prefer? I will check the forecast for that time.',
            'எத்தனை மணிக்கு வாகனம் வேண்டும்? அந்த நேரத்திற்கான வானிலையைச் சரிபார்க்கிறேன்.',
            'Endha pickup time venum? Andha time-ku weather check pannren.')
    state,answer=previous['prepare_turn'](b,weather,state,text,now=now)
    kind=category(state.get('weather'))
    if state.get('stage') in ('weather_advisory','weather_pending') and kind in ('normal','missing'):
        w=state.setdefault('weather',{});w.pop('_friendly_notice_policy',None)
        reason='forecast_missing' if kind=='missing' else 'no_weather_choice_needed'
        w['_friendly_notice_policy']=dict(version=VERSION,reason=reason,scope=scope(state,engine))
        state.pop('_wx_weather_consent',None)
        state,answer=base_prepare(b,weather,state,'',now=now)
        if state.get('stage')=='confirm':
            state['dialogue_state']='AWAITING_BOOKING_CONFIRMATION'
            notice=advisory(state) if kind=='normal' else ''
            return state,(notice+'\n'+answer if notice else answer)
    if state.get('stage')=='confirm' and (context.get('choice') or ack):
        context.pop('choice',None)
        intro=pick(state,'Sure, let me confirm your updated ride details. ',
            'சரி, உங்கள் பயண விவரங்களை உறுதி செய்துகொள்ளலாம். ',
            'Seri, unga ride details confirm pannalaam. ')
        return state,intro+answer
    return state,answer
''',
'speech_segments': r'''def speech_segments(text):
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        # Keep a coherent weather chunk; pause before a separate final summary.
        return [prefix,rest] if rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
'''
}

    def target_nodes(value):
        tree=ast.parse(value,filename=str(path))
        installs=[node for node in tree.body if isinstance(node,ast.FunctionDef) and node.name=='install_namespace']
        if len(installs)!=1:raise RuntimeError('Expected one existing weather installer; no changes made.')
        nodes={node.name:node for node in installs[0].body if isinstance(node,ast.FunctionDef) and node.name in changes}
        if set(nodes)!=set(changes):raise RuntimeError('The expected weather handlers are missing; no changes made.')
        return tree,nodes

    def target_codes(value):
        compiled=compile(value,str(path),'exec',dont_inherit=True)
        installs=[item for item in compiled.co_consts if isinstance(item,types.CodeType) and item.co_name=='install_namespace']
        if len(installs)!=1:raise RuntimeError('Expected one compiled weather installer.')
        return {item.co_name:item for item in installs[0].co_consts if isinstance(item,types.CodeType) and item.co_name in changes}

    def signature(code):
        return (code.co_code,code.co_argcount,code.co_posonlyargcount,code.co_kwonlyargcount,code.co_flags,
            code.co_names,code.co_varnames,code.co_freevars,code.co_cellvars,
            tuple(signature(item) if isinstance(item,types.CodeType) else item for item in code.co_consts))

    original_tree,nodes=target_nodes(text)
    old_codes=target_codes(original_tree)
    functions={key:engine.get(key) for key in changes}
    for name,function in functions.items():
        allowed={ast.dump(ast.parse(value).body[0]) for value in (originals[name],changes[name])}
        if ast.dump(nodes[name]) not in allowed:
            raise RuntimeError(name+' differs from the inspected implementation. No files changed; share a fresh inspection output.')
        if not isinstance(function,types.FunctionType) or function.__globals__ is not namespace or signature(function.__code__)!=signature(old_codes[name]):
            raise RuntimeError('The live '+name+' differs from its source file. No changes made.')
    lines=text.splitlines(keepends=True)
    for name,node in sorted(nodes.items(),key=lambda item:item[1].lineno,reverse=True):
        lines[node.lineno-1:node.end_lineno]=[textwrap.indent(changes[name].rstrip()+'\n','    ')]
    patched=''.join(lines)
    patched_tree,_=target_nodes(patched);new_codes=target_codes(patched_tree)
    # Prove that report(), prompts, decisions, models and all other source
    # statements remain identical. Only the two listed bodies may differ.
    def unrelated(tree):
        tree=copy.deepcopy(tree)
        install=next(node for node in tree.body if isinstance(node,ast.FunctionDef) and node.name=='install_namespace')
        install.body=[node for node in install.body if not (isinstance(node,ast.FunctionDef) and node.name in changes)]
        return ast.dump(tree)
    if unrelated(original_tree)!=unrelated(patched_tree):raise RuntimeError('Unexpected change outside the delivery handlers.')
    for name,function in functions.items():
        if function.__code__.co_freevars!=new_codes[name].co_freevars:
            raise RuntimeError('Unexpected closure change in '+name+'; no changes made.')

    # Isolated offline fixtures only: no real database, provider, STT or GPU.
    pure_names={'number','complete','category','pick','fmt','scope','report'}
    helpers=ast.Module(body=[node for node in patched_tree.body if isinstance(node,ast.FunctionDef) and node.name in pure_names],type_ignores=[])
    private=dict(copy=copy,math=math,VERSION=namespace['VERSION'],
        TURN=contextvars.ContextVar('movio_step3c_offline_turn',default=None))
    exec(compile(helpers,'offline_weather_helpers','exec',dont_inherit=True),private)
    def cell(value):return (lambda:value).__closure__[0]
    def bind(code,captures):
        return types.FunctionType(code,private,code.co_name,None,tuple(cell(captures[name]) for name in code.co_freevars))
    class GuardError(Exception):pass
    fixture_engine=dict(digest=engine['digest'],snapshot=engine['snapshot'],GuardError=GuardError,
        collect_details=lambda *args:dict(changed=[]))
    spoken=[];lock=threading.RLock()
    def advisory(state):
        value=private['report'](state)
        if value:spoken.append(value)
        return value
    def base_prepare(bridge,weather,state,text,now=None):
        state['stage']='confirm' if bridge.ready else 'route_unavailable'
        if bridge.ready:state['fare']=dict(km=12.3,fare=230)
        return state,'TEST SUMMARY' if bridge.ready else 'TEST ROUTE RETRY'
    def previous_prepare(bridge,weather,state,text,now=None):
        if not (state.get('weather') or {}).get('_friendly_notice_policy'):
            state['stage']='weather_advisory'
            return state,advisory(state)+'\nTEST WEATHER CHOICE?'
        return base_prepare(bridge,weather,state,text,now=now)
    previous=dict(prepare_turn=previous_prepare,reply_normalized=lambda value:str(value).strip().casefold(),
        speech_segments=lambda value:[str(value)])
    prepare=bind(new_codes['prepare_turn'],dict(advisory=advisory,base_prepare=base_prepare,engine=fixture_engine,previous=previous))
    segments=bind(new_codes['speech_segments'],dict(lock=lock,previous=previous,spoken=spoken))
    checks=0
    for language in ('English','Tamil','Tanglish'):
        state=dict(lang=language,mode='book',stage='count',count=4,vehicle='SUV',pickup='Pickup',destination='Destination',
            pickup_geo=dict(lat=13.0,lng=80.2),destination_geo=dict(lat=12.9,lng=80.1),pickup_at='2030-01-02T19:00:00+05:30',
            weather=dict(status='ok',condition='cloudy',points=[dict(location=role,weather_code=3,precipitation_probability=chance,temperature_2m=27.3)
                for role,chance in (('pickup',39.7),('destination',42.2))]))
        expected={key:state[key] for key in ('count','vehicle','pickup','destination','pickup_at')}
        bridge=types.SimpleNamespace(ready=False)
        state,answer=prepare(bridge,None,state,'',now=None)
        assert state['stage']=='route_unavailable' and '39.7%' not in answer
        checks+=1
        bridge.ready=True
        state,answer=prepare(bridge,None,state,'retry',now=None)
        assert state['stage']=='confirm' and answer.startswith(private['report'](state)) and answer.endswith('TEST SUMMARY')
        assert expected=={key:state[key] for key in expected} and '_wx_weather_consent' not in state
        checks+=1
        assert segments(answer)==[private['report'](state),'TEST SUMMARY']
        checks+=1
        pending=copy.deepcopy(state)
        proof=state['_friendly_normal_notice']['summary_hash']
        token=private['TURN'].set(dict(sid='OFFLINE_FIXTURE',db=types.SimpleNamespace(row=lambda sid:dict(played=1,summary_kind='final',summary_hash=proof))))
        try:
            state,again=prepare(bridge,None,state,'repeat summary',now=None)
            assert again=='TEST SUMMARY' and state['_friendly_normal_notice']['acknowledged']
            checks+=1
        finally:private['TURN'].reset(token)
        pending,replay=prepare(bridge,None,pending,'repeat summary',now=None)
        assert '39.7%' in replay and not pending['_friendly_normal_notice']['acknowledged']
        checks+=1
        missing=copy.deepcopy(pending);missing['weather']=dict(status='pending');missing.pop('_friendly_normal_notice',None)
        missing,answer=prepare(bridge,None,missing,'',now=None)
        assert missing['stage']=='confirm' and answer=='TEST SUMMARY'
        checks+=1
        rainy=copy.deepcopy(pending);rainy['weather']=dict(status='ok',condition='rain',points=[dict(location=role,weather_code=61,precipitation_probability=75) for role in ('pickup','destination')])
        rainy,answer=prepare(bridge,None,rainy,'',now=None)
        assert rainy['stage']=='weather_advisory' and segments(answer)==[answer]
        checks+=1

    protected_engine={key:engine.get(key) for key in ('prepare_turn','speech_segments','summary_text','advisory','weather_question','reply_normalized','collect_details','tts_reply','completion_message')}
    protected_bridge={key:getattr(b,key,None) for key in ('geocode','road_fare','TTS','LLM','speak','tanglish_speech_text','_wx_endpoint','_wx_db','HTML')}
    old_live={name:(function.__code__,function.__doc__) for name,function in functions.items()}
    backup=None;changed=patched.encode('utf-8')!=before
    def atomic_write(data):
        temporary_path=None
        try:
            with tempfile.NamedTemporaryFile(dir=path.parent,prefix='.weather_delivery_',delete=False) as handle:
                temporary_path=Path(handle.name);handle.write(data)
            shutil.copymode(path,temporary_path);os.replace(temporary_path,path)
        finally:
            if temporary_path is not None:temporary_path.unlink(missing_ok=True)
    try:
        if changed:
            if path.read_bytes()!=before:raise RuntimeError('Weather source changed during validation.')
            backup=root/('weather_delivery_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
            shutil.copy2(path,backup/path.name);atomic_write(patched.encode('utf-8'))
        for name,function in functions.items():
            function.__code__=new_codes[name];function.__doc__=new_codes[name].co_consts[0]
        if any(engine.get(key) is not value for key,value in protected_engine.items()):raise RuntimeError('An existing dialogue handler was replaced unexpectedly.')
        for key,value in protected_bridge.items():
            current=getattr(b,key,None)
            if getattr(current,'__func__',current) is not getattr(value,'__func__',value):raise RuntimeError('A protected bridge integration changed: '+key)
    except Exception:
        for name,function in functions.items():function.__code__,function.__doc__=old_live[name]
        if changed and backup is not None:atomic_write(before)
        raise
    print('STEP 3C PASS — available normal-weather advice is included before final review, including after a route retry.')
    print('Syntax / unchanged unrelated source / closure checks: PASS | offline delivery checks:',checks)
    print('Existing geocoder, booking guards, models, UI and speech settings retained.')
    print('Modified file:',str(path) if changed else 'Already current; no rewrite needed.')
    print('Backup:',str(backup) if backup else 'No new backup needed.')
    print('Existing pause between speech segments (ms):',getattr(b,'_wx_pause_ms',220))
    print('Close the old call popup and open a NEW AI Call.')
    print('Test: Book an SUV for 4 people from VIT Chennai to Vandalur tomorrow at 7 PM.')
    print('Expected: weather advice, then ride summary, then your explicit final Yes/No.')
    print('Share this output and the AI reply before the next step. Live audio still needs your listening check.')
    return dict(step='3C',status='INSTALLED',offline_checks=checks,modified_file=str(path) if changed else None,backup=str(backup) if backup else None)


MOVIO_STEP_3C_WEATHER_DELIVERY=movio_step_3c_weather_delivery()


STEP 3C PASS — available normal-weather advice is included before final review, including after a route retry.
Syntax / unchanged unrelated source / closure checks: PASS | offline delivery checks: 21
Existing geocoder, booking guards, models, UI and speech settings retained.
Modified file: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_friendly_weather.py
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/weather_delivery_backup_20261028_070731_979064
Existing pause between speech segments (ms): 220
Close the old call popup and open a NEW AI Call.
Test: Book an SUV for 4 people from VIT Chennai to Vandalur tomorrow at 7 PM.
Expected: weather advice, then ride summary, then your explicit final Yes/No.
Share this output and the AI reply before the next step. Live audio still needs your listening check.


In [117]:
# Copy this ENTIRE file into ONE Kaggle code cell after Steps 3B/3C,
# before evaluation. UI binding 1.1 supports reformatted and guarded handlers.
# Close the old AI Call popup and open a NEW one afterward.
def movio_show_call_weather_card():
    import ast
    import copy
    import inspect
    import json
    import math
    import os
    import re
    import shutil
    import subprocess
    import tempfile
    import textwrap
    import types
    from datetime import datetime
    from pathlib import Path

    version = 'call-weather-card-v1'
    b = globals().get('bridge')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    socket_class = engine.get('ConfirmationSocket')
    speech = getattr(b, 'tanglish_speech_text', None)
    root = Path(globals().get('WX_ROOT', '/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    if b is None or not isinstance(getattr(b, 'HTML', None), str) or not isinstance(socket_class, type):
        raise RuntimeError('Run the existing voice/weather and confirmation setup first. No changes were made.')
    if not getattr(b, '_tg_wordwise_v2', False) or not isinstance(speech, types.FunctionType):
        raise RuntimeError('Run the existing word-by-word Tanglish pronunciation cell first. No changes were made.')
    method = socket_class.send_json
    namespace = method.__globals__
    helper = root / 'integrations/movio_confirmation_flow.py'
    if not helper.is_file() or Path(namespace.get('__file__', '')).resolve() != helper:
        raise RuntimeError('The active confirmation handler does not match its project file. No changes were made.')
    if any(name not in namespace for name in ('json', 'math')):
        raise RuntimeError('The existing confirmation imports are incomplete. No changes were made.')

    payload_source = r'''def call_weather_card(state):
    """MOVIO call weather card v1: presentation data from this bound call only."""
    if not isinstance(state, dict) or state.get('stage') not in ('weather_advisory', 'confirm', 'route_unavailable', 'booking_pending', 'done'):
        return None
    weather = state.get('weather') or {}
    if not isinstance(weather, dict) or weather.get('status') != 'ok' or not state.get('pickup_at'):
        return None
    def finite(value, lower, upper):
        if isinstance(value, bool):
            return None
        try:
            number = float(value)
            return number if math.isfinite(number) and lower <= number <= upper else None
        except (TypeError, ValueError, OverflowError):
            return None
    rows = weather.get('points')
    if not isinstance(rows, list):
        return None
    points = []
    for role in ('pickup', 'destination'):
        point = next((item for item in rows if isinstance(item, dict) and item.get('location') == role), None)
        if point is None:
            continue
        raw_code = finite(point.get('weather_code'), 0, 99)
        code = int(raw_code) if raw_code is not None and raw_code.is_integer() else None
        temperature = finite(point.get('temperature_2m'), -100, 70)
        chance = finite(point.get('precipitation_probability'), 0, 100)
        if code is None and temperature is None and chance is None:
            continue
        points.append(dict(role=role, place=str(state.get(role) or '')[:120], weather_code=code,
            temperature_c=temperature, rain_probability_pct=chance, valid_at=str(point.get('valid_at') or '')[:40]))
    if not points:
        return None
    return dict(version='call-weather-card-v1', source=str(weather.get('source') or '')[:60],
        pickup_at=str(state['pickup_at'])[:40], time_basis='pickup_time', points=points)
'''
    original_method_source = r'''async def send_json(self,message):
    message=dict(message);row=self.row();phase=None
    if message.get('event') in ('reply','speech_chunk'):
        if message.get('done') and row and row['ride_id']:phase=BOOKING_CONFIRMED
        elif message.get('stage')=='confirm':phase=AWAITING_BOOKING_CONFIRMATION
        elif message.get('stage')=='weather_advisory':phase=WEATHER_ADVISORY
        else:phase=COLLECTING_DETAILS
    elif message.get('event')=='action' and message.get('type')=='weather_book':phase=BOOKING_IN_PROGRESS
    if phase and row:
        record_phase(self.db,self.sid,phase,message.get('epoch'))
        message['dialogue_state']=phase
        if message.get('event') in ('reply','action'):event(self.logger,'confirmation_state',phase=phase)
    await self.websocket.send_json(message)
'''
    updated_method_source = r'''async def send_json(self,message):
    message=dict(message);row=self.row();phase=None
    # MOVIO_CALL_WEATHER_CARD_V1: data from this authenticated session only.
    if message.get('event') == 'reply':
        message['weather_card'] = None
        if row and message.get('epoch') is not None and message.get('epoch') == row.get('epoch'):
            try:
                message['weather_card'] = call_weather_card(json.loads(row.get('state') or '{}'))
            except (TypeError, ValueError, KeyError, OverflowError):
                message['weather_card'] = None
    if message.get('event') in ('reply','speech_chunk'):
        if message.get('done') and row and row['ride_id']:phase=BOOKING_CONFIRMED
        elif message.get('stage')=='confirm':phase=AWAITING_BOOKING_CONFIRMATION
        elif message.get('stage')=='weather_advisory':phase=WEATHER_ADVISORY
        else:phase=COLLECTING_DETAILS
    elif message.get('event')=='action' and message.get('type')=='weather_book':phase=BOOKING_IN_PROGRESS
    if phase and row:
        record_phase(self.db,self.sid,phase,message.get('epoch'))
        message['dialogue_state']=phase
        if message.get('event') in ('reply','action'):event(self.logger,'confirmation_state',phase=phase)
    await self.websocket.send_json(message)
'''
    css = r'''/* MOVIO_CALL_WEATHER_CARD_V1 */
.phone #movioCallWeather[hidden]{display:none!important}
.phone #movioCallWeather{margin:16px 0;padding:16px;border:1px solid #e2e7ec;border-radius:17px;background:#fff;color:#202a35;white-space:normal;box-shadow:0 3px 14px #17212b05}
.movio-wx-heading{display:flex;align-items:center;gap:8px;margin:0 0 3px;font-size:15px;font-weight:750;line-height:1.5}
.movio-wx-dot{height:8px;width:8px;flex:none;border-radius:50%;background:#e1ae18;box-shadow:0 0 0 4px #fff5d5;margin:0 3px}
.movio-wx-subtitle{margin:2px 0 14px;color:#687684;font-size:11px;line-height:1.5}
.movio-wx-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px}
.movio-wx-point{min-width:0;padding:13px;background:#f7f9fc;border:1px solid #e8edf2;border-radius:13px}
.movio-wx-role{font-size:10px;font-weight:750;letter-spacing:.06em;text-transform:uppercase;color:#72808d;line-height:1.5}
.movio-wx-place{font-size:13px;font-weight:650;line-height:1.5;margin:3px 0 12px;overflow-wrap:anywhere}
.movio-wx-main{display:flex;align-items:center;justify-content:space-between;gap:6px}
.movio-wx-icon{width:43px;height:43px;display:grid;place-items:center;flex:none;color:#55718c}
.movio-wx-icon svg{width:40px;height:40px;stroke:currentColor;stroke-width:1.65;fill:none;stroke-linecap:round;stroke-linejoin:round}
.movio-wx-temperature{font-size:30px;font-weight:750;letter-spacing:-1px;line-height:1.2;white-space:nowrap}
.movio-wx-temperature small{font-size:16px;letter-spacing:0;font-weight:500;color:#687684;vertical-align:top;margin-left:2px}
.movio-wx-condition{display:inline-block;max-width:100%;font-size:11px;font-weight:650;line-height:1.5;margin:10px 0 8px;padding:4px 8px;border-radius:7px;background:#eaf0f6;color:#3d5974;overflow-wrap:anywhere}
.movio-wx-point[data-condition=clear] .movio-wx-condition{background:#fff0bd;color:#785a05}
.movio-wx-point[data-condition=clear] .movio-wx-icon{color:#c28b09}
.movio-wx-point[data-condition=rain] .movio-wx-condition{background:#e0f0ff;color:#1d6591}
.movio-wx-point[data-condition=rain] .movio-wx-icon{color:#337ca6}
.movio-wx-point[data-condition=storm] .movio-wx-condition{background:#ece7fa;color:#665187}
.movio-wx-rain{display:flex;align-items:center;gap:5px;color:#607180;font-size:11px;line-height:1.6;overflow-wrap:anywhere}
.movio-wx-rain svg{width:12px;height:14px;flex:none;fill:none;stroke:#4b83a5;stroke-width:1.6}
.movio-wx-footer{margin:12px 0 0;font-size:10px;line-height:1.6;color:#7d8995;overflow-wrap:anywhere}
@media(max-width:420px){.phone #movioCallWeather{padding:13px}.movio-wx-grid{grid-template-columns:minmax(0,1fr)}.movio-wx-point{padding:12px 14px}.movio-wx-place{margin-bottom:6px}.movio-wx-temperature{font-size:32px}.movio-wx-main{justify-content:flex-start;gap:16px}.movio-wx-condition{margin-top:8px}.movio-wx-role{font-size:10px}}
'''
    javascript = r'''/* MOVIO_CALL_WEATHER_CARD_V1: UI binding 1.1; existing call handlers are preserved. */
(()=>{
 'use strict';
 const log=document.getElementById('log'),language=document.getElementById('lang');
 if(!log||!language||document.getElementById('movioCallWeather'))return;
 const card=document.createElement('section');card.id='movioCallWeather';card.hidden=true;
 card.setAttribute('role','region');card.setAttribute('aria-labelledby','movioWeatherTitle');log.before(card);
 let current=null;
 const copy={
  English:{title:'Your trip weather',subtitle:'Both locations at your pickup time',pickup:'Pickup',destination:'Destination',rain:'rain chance',missing:'Not reported',forecast:'Forecast',
   clear:'Sunny',mainlyClear:'Mostly clear',partlyCloudy:'Partly cloudy',cloudy:'Cloudy',fog:'Foggy',drizzle:'Drizzle',rainy:'Rainy',heavyRain:'Heavy rain',freezingRain:'Freezing rain',snow:'Snow',showers:'Rain showers',storm:'Thunderstorms',hail:'Thunderstorms with hail',unknown:'Condition not reported'},
  Tamil:{title:'உங்கள் பயண வானிலை',subtitle:'புறப்படும் நேரத்தில் இரு இடங்களின் முன்னறிவிப்பு',pickup:'புறப்படும் இடம்',destination:'சேரும் இடம்',rain:'மழை வாய்ப்பு',missing:'தகவல் இல்லை',forecast:'முன்னறிவிப்பு',
   clear:'வெயில்',mainlyClear:'தெளிவான வானம்',partlyCloudy:'ஓரளவு மேகமூட்டம்',cloudy:'மேகமூட்டம்',fog:'மூடுபனி',drizzle:'தூறல்',rainy:'மழை',heavyRain:'கனமழை',freezingRain:'உறையும் மழை',snow:'பனிப்பொழிவு',showers:'மழைச் சாரல்',storm:'இடியுடன் மழை',hail:'இடியுடன் ஆலங்கட்டி மழை',unknown:'வானிலை தகவல் இல்லை'},
  Tanglish:{title:'Unga trip weather',subtitle:'Pickup time-la rendu places-oda forecast',pickup:'Pickup',destination:'Destination',rain:'rain chance',missing:'Not reported',forecast:'Forecast',
   clear:'Sunny',mainlyClear:'Mostly clear',partlyCloudy:'Partly cloudy',cloudy:'Cloudy',fog:'Foggy',drizzle:'Drizzle',rainy:'Rainy',heavyRain:'Heavy rain',freezingRain:'Freezing rain',snow:'Snow',showers:'Rain showers',storm:'Thunderstorms',hail:'Thunderstorms with hail',unknown:'Condition not reported'}
 };
 const drawings={
  clear:'<circle cx="20" cy="20" r="6"/><path d="M20 4v4m0 24v4M4 20h4m24 0h4M9 9l3 3m16 16l3 3M9 31l3-3m16-16l3-3"/>',
  cloudy:'<path d="M11 29h18a7 7 0 0 0 .2-14 10 10 0 0 0-19-1A7.6 7.6 0 0 0 11 29Z"/>',
  rain:'<path d="M10 24h20a6 6 0 0 0-1-12 9 9 0 0 0-17-1 6.6 6.6 0 0 0-2 13ZM13 29l-2 5m10-5-2 5m10-5-2 5"/>',
  storm:'<path d="M10 24a6.6 6.6 0 0 1 2-13 9 9 0 0 1 17 1 6 6 0 0 1 1 12M22 21l-7 9h7l-4 7"/>',
  fog:'<path d="M10 23h20a6 6 0 0 0-1-12 9 9 0 0 0-17-1 6.6 6.6 0 0 0-2 13M7 29h26M11 34h18"/>',
  snow:'<path d="M10 24h20a6 6 0 0 0-1-12 9 9 0 0 0-17-1 6.6 6.6 0 0 0-2 13M13 30h.1M20 34h.1M28 30h.1"/>'
 };
 function condition(code){
  if(code===0)return ['clear','clear'];if(code===1)return ['mainlyClear','clear'];
  if(code===2)return ['partlyCloudy','cloudy'];if(code===3)return ['cloudy','cloudy'];
  if([45,48].includes(code))return ['fog','fog'];if([51,53,55,56,57].includes(code))return ['drizzle','rain'];
  if([61,63].includes(code))return ['rainy','rain'];if([65,82].includes(code))return ['heavyRain','rain'];
  if([66,67].includes(code))return ['freezingRain','rain'];if([71,73,75,77,85,86].includes(code))return ['snow','snow'];
  if([80,81].includes(code))return ['showers','rain'];if(code===95)return ['storm','storm'];
  if([96,99].includes(code))return ['hail','storm'];return ['unknown','cloudy'];
 }
 function node(tag,cls,text){const element=document.createElement(tag);if(cls)element.className=cls;if(text!==undefined)element.textContent=text;return element;}
 function numeric(value,lower,upper){return typeof value==='number'&&Number.isFinite(value)&&value>=lower&&value<=upper;}
 function format(value){return new Intl.NumberFormat('en-IN',{maximumFractionDigits:1}).format(value);}
 function date(value,lang){try{const instant=new Date(value);if(Number.isNaN(instant.getTime()))return '';return new Intl.DateTimeFormat(lang==='Tamil'?'ta-IN':'en-IN',{timeZone:'Asia/Kolkata',day:'numeric',month:'short',hour:'numeric',minute:'2-digit',hour12:true}).format(instant);}catch{return '';}}
 window.movioRenderWeatherCard=function(data){
  current=data;card.replaceChildren();
  if(!data||data.version!=='call-weather-card-v1'||!Array.isArray(data.points)||!data.points.length){card.hidden=true;return;}
  const text=copy[language.value]||copy.English;
  const title=node('h3','movio-wx-heading');title.id='movioWeatherTitle';
  const dot=node('span','movio-wx-dot');dot.setAttribute('aria-hidden','true');title.append(dot,node('span','',text.title));
  card.append(title,node('p','movio-wx-subtitle',text.subtitle));const grid=node('div','movio-wx-grid');let count=0;
  for(const point of data.points.slice(0,2)){
   if(!point||!['pickup','destination'].includes(point.role))continue;
   const key=condition(point.weather_code),box=node('article','movio-wx-point');box.dataset.condition=key[1];
   box.append(node('div','movio-wx-role',text[point.role]),node('div','movio-wx-place',String(point.place||text[point.role]).slice(0,120)));
   const main=node('div','movio-wx-main'),icon=node('span','movio-wx-icon');icon.setAttribute('aria-hidden','true');
   icon.innerHTML='<svg viewBox="0 0 40 40" aria-hidden="true">'+(drawings[key[1]]||drawings.cloudy)+'</svg>';
   const temp=node('div','movio-wx-temperature');
   if(numeric(point.temperature_c,-100,70)){temp.append(node('span','',format(point.temperature_c)),node('small','','°C'));}
   else{temp.append(node('span','','—'),node('small','','°C'));temp.setAttribute('aria-label',text.missing);}
   main.append(icon,temp);box.append(main,node('div','movio-wx-condition',text[key[0]]));
   if(numeric(point.rain_probability_pct,0,100)){
    const rain=node('div','movio-wx-rain'),drop=node('span','');drop.setAttribute('aria-hidden','true');
    drop.innerHTML='<svg viewBox="0 0 14 17"><path d="M7 1C5 5 1 8 1 11a6 6 0 0 0 12 0c0-3-4-6-6-10Z"/></svg>';
    rain.append(drop,node('span','',format(point.rain_probability_pct)+'% '+text.rain));box.append(rain);
   }
   grid.append(box);count++;
  }
  if(!count){card.replaceChildren();card.hidden=true;return;}
  card.append(grid);const when=date(data.pickup_at,language.value);
  card.append(node('p','movio-wx-footer',[data.source?String(data.source).slice(0,60):text.forecast,when].filter(Boolean).join(' · ')));
  card.hidden=false;
 };
 language.addEventListener('change',()=>{if(current)window.movioRenderWeatherCard(current);});
 // Observe the EXISTING socket only. Do not replace WebSocket, onmessage,
 // start, interrupt, typed, send, booking confirmation, or audio playback.
 let observedSocket=null,removeSocketListeners=null;
 function activeSocket(){try{return typeof socket==='undefined'?null:socket;}catch{return null;}}
 function bindWeatherSocket(){
  const candidate=activeSocket();if(candidate===observedSocket)return;
  if(removeSocketListeners)removeSocketListeners();removeSocketListeners=null;observedSocket=candidate;
  if(!candidate||typeof candidate.addEventListener!=='function')return;
  function receiveForecast(event){
   if(candidate!==activeSocket()||observedSocket!==candidate)return;
   let message;try{message=JSON.parse(event.data);}catch{return;}
   if(!message||message.event!=='reply')return;
   // Use the same live epoch/interrupt state as the original reply handler.
   if(typeof connected==='boolean'&&!connected)return;
   if(typeof awaitInterrupt==='boolean'&&awaitInterrupt)return;
   if(typeof epoch==='number'&&message.epoch!=null&&message.epoch<epoch)return;
   window.movioRenderWeatherCard(message.weather_card||null);
  }
  candidate.addEventListener('message',receiveForecast);
  removeSocketListeners=()=>candidate.removeEventListener('message',receiveForecast);
 }
 function refreshWeatherBinding(){
  bindWeatherSocket();
  if(typeof awaitInterrupt==='boolean'&&awaitInterrupt)window.movioRenderWeatherCard(null);
 }
 const observer=new MutationObserver(refreshWeatherBinding);
 for(const id of ['call','lang']){
  const control=document.getElementById(id);
  if(control)observer.observe(control,{attributes:true,attributeFilter:['disabled']});
 }
 const status=document.getElementById('status');
 if(status)observer.observe(status,{childList:true,subtree:true,characterData:true});
 for(const id of ['call','interrupt','bookMode','cancelMode']){
  document.getElementById(id)?.addEventListener('click',()=>window.movioRenderWeatherCard(null));
 }
 window.addEventListener('beforeunload',()=>{observer.disconnect();if(removeSocketListeners)removeSocketListeners();},{once:true});
 bindWeatherSocket();
})();
'''

    def tree(value):
        return ast.dump(ast.parse(textwrap.dedent(value)), include_attributes=False)

    def signature(code):
        constants = tuple(signature(value) if isinstance(value, types.CodeType) else value for value in code.co_consts)
        return (code.co_code, constants, code.co_names, code.co_varnames, code.co_freevars,
                code.co_cellvars, code.co_argcount, code.co_kwonlyargcount, code.co_posonlyargcount,
                code.co_flags, code.co_exceptiontable)

    def find_code(code, path):
        for name in path:
            matches = [value for value in code.co_consts if isinstance(value, types.CodeType) and value.co_name == name]
            if len(matches) != 1:
                raise RuntimeError('Unexpected function structure: ' + name + '. No changes were made.')
            code = matches[0]
        return code

    before_source = helper.read_text(encoding='utf-8')
    source_tree = ast.parse(before_source)
    classes = [node for node in source_tree.body if isinstance(node, ast.ClassDef) and node.name == 'ConfirmationSocket']
    methods = [node for node in classes[0].body if isinstance(node, ast.AsyncFunctionDef) and node.name == 'send_json'] if len(classes) == 1 else []
    if len(methods) != 1:
        raise RuntimeError('The existing confirmation sender was not found. No changes were made.')
    target = methods[0]
    lines = before_source.splitlines(keepends=True)
    existing = textwrap.dedent(''.join(lines[target.lineno - 1:target.end_lineno]))
    if tree(existing) not in (tree(original_method_source), tree(updated_method_source)):
        raise RuntimeError('The confirmation sender has changed. Share its current source before patching. No changes were made.')
    old_compiled = compile(before_source, str(helper), 'exec', dont_inherit=True)
    if signature(find_code(old_compiled, ('ConfirmationSocket', 'send_json'))) != signature(method.__code__):
        raise RuntimeError('The live confirmation sender differs from its saved source. No changes were made.')
    lines[target.lineno - 1:target.end_lineno] = [textwrap.indent(updated_method_source.strip() + '\n', '    ')]
    after_source = ''.join(lines)
    functions = [node for node in ast.parse(after_source).body if isinstance(node, ast.FunctionDef) and node.name == 'call_weather_card']
    if functions:
        if len(functions) != 1 or tree(ast.get_source_segment(after_source, functions[0])) != tree(payload_source):
            raise RuntimeError('An unrelated weather-card helper already exists. No changes were made.')
    else:
        anchor = next(node for node in ast.parse(after_source).body if isinstance(node, ast.ClassDef) and node.name == 'ConfirmationSocket')
        lines = after_source.splitlines(keepends=True)
        lines.insert(anchor.lineno - 1, payload_source.strip() + '\n\n')
        after_source = ''.join(lines)
    compiled = compile(after_source, str(helper), 'exec', dont_inherit=True)
    method_code = find_code(compiled, ('ConfirmationSocket', 'send_json'))
    payload_code = find_code(compiled, ('call_weather_card',))
    if method_code.co_freevars != method.__code__.co_freevars:
        raise RuntimeError('The existing confirmation closure cannot be preserved. No changes were made.')
    payload = types.FunctionType(payload_code, namespace, 'call_weather_card')

    # Update the ORIGINAL Tanglish speech-copy function; no new TTS wrapper.
    speech_marker = 'MOVIO call weather card v1: Celsius is pronounced செல்சியஸ்.'
    original_speech_source = None
    if speech.__doc__ == speech_marker:
        speech_code = speech.__code__
    else:
        original_speech_source = textwrap.dedent(inspect.getsource(speech))
        parsed = ast.parse(original_speech_source)
        fn = parsed.body[0] if len(parsed.body) == 1 else None
        if not isinstance(fn, ast.FunctionDef) or fn.name != 'speech_text' or not isinstance(fn.body[-1], ast.Return):
            raise RuntimeError('Unexpected Tanglish speech-copy function. No changes were made.')
        original_factory = 'def factory(' + ','.join(speech.__code__.co_freevars) + '):\n' + textwrap.indent(original_speech_source, '    ') + '\n    return speech_text\n'
        check_space = dict(speech.__globals__)
        exec(compile(original_factory, '<existing Tanglish speech>', 'exec', dont_inherit=True), check_space)
        enclosed = [cell.cell_contents for cell in (speech.__closure__ or ())]
        if signature(check_space['factory'](*enclosed).__code__) != signature(speech.__code__):
            raise RuntimeError('The speech source differs from its live function. No changes were made.')
        if not fn.body or not isinstance(fn.body[0], ast.Expr) or not isinstance(getattr(fn.body[0], 'value', None), ast.Constant) or not isinstance(fn.body[0].value.value, str):
            fn.body.insert(0, ast.Expr(value=ast.Constant(value=speech_marker)))
        else:
            fn.body[0].value.value = speech_marker
        pronunciation = ast.parse("text = re.sub(r'(?i)(?<![A-Za-z])celsius(?![A-Za-z])', 'செல்சியஸ்', text)").body[0]
        fn.body.insert(len(fn.body) - 1, pronunciation)
        ast.fix_missing_locations(parsed)
        new_speech_source = ast.unparse(parsed)
        factory = 'def factory(' + ','.join(speech.__code__.co_freevars) + '):\n' + textwrap.indent(new_speech_source, '    ') + '\n    return speech_text\n'
        space = dict(speech.__globals__)
        exec(compile(factory, '<MOVIO Celsius pronunciation>', 'exec', dont_inherit=True), space)
        speech_code = space['factory'](*enclosed).__code__
        if speech_code.co_freevars != speech.__code__.co_freevars:
            raise RuntimeError('The original speech closure cannot be preserved. No changes were made.')
    speech_candidate = types.FunctionType(speech_code, speech.__globals__, speech.__name__, speech.__defaults__, speech.__closure__)

    style_block = '<!-- MOVIO_CALL_WEATHER_STYLE_BEGIN --><style id="movioCallWeatherStyle">' + css + '</style><!-- MOVIO_CALL_WEATHER_STYLE_END -->'
    script_block = '<!-- MOVIO_CALL_WEATHER_SCRIPT_BEGIN --><script id="movioCallWeatherScript">' + javascript + '</script><!-- MOVIO_CALL_WEATHER_SCRIPT_END -->'
    def patch_html(value):
        from html.parser import HTMLParser
        class CallStructure(HTMLParser):
            def __init__(self):super().__init__();self.ids={}
            def handle_starttag(self,tag,attributes):
                for name,content in attributes:
                    if name=='id':self.ids[content]=self.ids.get(content,0)+1
        structure=CallStructure();structure.feed(value)
        for name in ('log','lang','call','interrupt','bookMode','cancelMode'):
            if structure.ids.get(name)!=1:
                raise RuntimeError('The call page must contain exactly one '+name+' control. No changes were made.')
        for part in ('STYLE','SCRIPT'):
            value = re.sub(r'<!-- MOVIO_CALL_WEATHER_' + part + r'_BEGIN -->.*?<!-- MOVIO_CALL_WEATHER_' + part + r'_END -->', '', value, flags=re.S)
        # Remove only the optional calls inserted by this cell's earlier version.
        # Existing application handlers themselves remain byte-for-byte intact.
        for old_call in ('window.movioRenderWeatherCard?.(m.weather_card||null);','window.movioRenderWeatherCard?.(null);'):
            value=value.replace(old_call,'')
        if len(re.findall(r'</head\s*>',value,re.I)) != 1 or len(re.findall(r'</body\s*>',value,re.I)) != 1:
            raise RuntimeError('Invalid call-page structure. No changes were made.')
        value = re.sub(r'</head\s*>', lambda match: style_block + match[0], value, count=1, flags=re.I)
        return re.sub(r'</body\s*>', lambda match: script_block + match[0], value, count=1, flags=re.I)

    active_before = b.HTML
    active_after = patch_html(active_before)
    writes = {helper: (before_source, after_source)}
    for name in ('ai_call_page.html', 'weather_call_page.html'):
        path = root / 'integrations' / name
        if path.is_file():
            before = path.read_text(encoding='utf-8')
            writes[path] = (before, patch_html(before))
    node = shutil.which('node')
    if not node:
        raise RuntimeError('Node.js is needed for the JavaScript syntax checks. No changes were made.')
    with tempfile.TemporaryDirectory(prefix='movio_weather_card_check_') as folder:
        for index, html in enumerate([active_after] + [after for path, (_, after) in writes.items() if path.suffix == '.html']):
            scripts = re.findall(r'<script\b[^>]*>(.*?)</script\s*>', html, flags=re.I | re.S)
            file = Path(folder) / ('call_' + str(index) + '.js')
            file.write_text('\n'.join(scripts), encoding='utf-8')
            checked = subprocess.run([node, '--check', str(file)], capture_output=True, text=True)
            if checked.returncode:
                raise RuntimeError('Call-page JavaScript syntax check failed. No changes were made.\n' + checked.stderr)

    # Pure checks use forecast fixtures; no API, model, audio or booking request.
    checks = 0
    def check(condition, description):
        nonlocal checks
        if not condition:
            raise RuntimeError('Pre-install check failed: ' + description + '. No changes were made.')
        checks += 1
    sample = dict(stage='confirm', pickup='Chennai Central', destination='Tambaram', pickup_at='2030-10-19T19:00:00+05:30',
        weather=dict(status='ok', source='Open-Meteo', points=[
            dict(location='pickup', weather_code=3, temperature_2m=27.3, precipitation_probability=42, valid_at='2030-10-19T19:00'),
            dict(location='destination', weather_code=61, temperature_2m=26.5, precipitation_probability=75, valid_at='2030-10-19T19:00')]))
    untouched = copy.deepcopy(sample)
    card = payload(sample)
    check(sample == untouched, 'forecast input remains unchanged')
    check(len(card['points']) == 2, 'both forecast locations included')
    check(card['points'][0]['weather_code'] == 3 and card['points'][1]['weather_code'] == 61, 'condition codes remain distinct')
    check(card['points'][0]['temperature_c'] == 27.3, 'actual temperature retained')
    check(card['points'][1]['rain_probability_pct'] == 75, 'actual rainfall chance retained')
    check(card['time_basis'] == 'pickup_time', 'destination forecast clearly uses pickup time')
    check(set(card) == {'version', 'source', 'pickup_at', 'time_basis', 'points'}, 'only presentation data emitted')
    check('secret' not in json.dumps(card) and 'token' not in json.dumps(card), 'no authentication fields emitted')
    for stage in ('time', 'count', 'vehicle', 'weather_change_time'):
        value = copy.deepcopy(sample);value['stage'] = stage
        check(payload(value) is None, 'hide stale forecast while collecting ' + stage)
    for status in ('unavailable', 'pending'):
        value = copy.deepcopy(sample);value['weather']['status'] = status
        check(payload(value) is None, 'no invented ' + status + ' forecast')
    value = copy.deepcopy(sample)
    value['weather']['points'][0].update(temperature_2m=float('nan'), precipitation_probability=200)
    value['weather']['points'][1].update(temperature_2m=None, precipitation_probability=None)
    filtered = payload(value)
    check(filtered['points'][0]['temperature_c'] is None and filtered['points'][0]['rain_probability_pct'] is None, 'invalid numbers filtered')
    json.dumps(filtered, allow_nan=False);checks += 1
    value = copy.deepcopy(sample);value['weather']['points'][0].update(temperature_2m=0, precipitation_probability=0)
    zero = payload(value)['points'][0]
    check(zero['temperature_c'] == 0 and zero['rain_probability_pct'] == 0, 'zero values remain valid')
    said = speech_candidate('Vanakkam! Pickup temperature 27.3 degrees Celsius. SUV, help, service, 4 PM.')
    check('செல்சியஸ்' in said and 'celsius' not in said.casefold(), 'Celsius has Tamil speech spelling')
    check('help' in said and 'service' in said, 'English words retain English wording')
    check('வணக்கம்' in said, 'existing Tamil-origin pronunciation retained')
    check('செல்சியஸ்' in speech_candidate('வெப்பநிலை 27 டிகிரி செல்சியஸ்.'), 'Tamil unit remains intact')
    for html in [active_after] + [after for path, (_, after) in writes.items() if path.suffix == '.html']:
        check(html.count('id="movioCallWeatherScript"') == 1, 'one weather renderer per page')
        check(html.count('function bindWeatherSocket()') == 1, 'one presentation-only socket observer')
        check(patch_html(html) == html, 'safe repeated installation')

    changes = {path: (before, after) for path, (before, after) in writes.items() if before != after}
    backup = None
    old_method_code, old_method_doc = method.__code__, method.__doc__
    old_speech_code, old_speech_doc = speech.__code__, speech.__doc__
    old_payload = namespace.get('call_weather_card')
    if changes or active_before != active_after or old_speech_doc != speech_marker:
        backup = root / ('call_weather_card_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        backup.mkdir()
        (backup / 'active_call_page.html').write_text(active_before, encoding='utf-8')
        if original_speech_source is not None:
            (backup / 'original_speech_text.py').write_text(original_speech_source, encoding='utf-8')
        for path, (before, _) in changes.items():
            destination = backup / path.relative_to(root)
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_text(before, encoding='utf-8')
    written = []
    try:
        for path, (_, after) in changes.items():
            temporary = path.with_name(path.name + '.weather-card-tmp')
            try:
                temporary.write_text(after, encoding='utf-8');os.replace(temporary, path)
            finally:
                temporary.unlink(missing_ok=True)
            written.append(path)
        namespace['call_weather_card'] = payload
        method.__code__, method.__doc__ = method_code, old_method_doc
        speech.__code__, speech.__doc__ = speech_code, speech_marker
        b.HTML = active_after
    except Exception:
        for path in reversed(written):
            path.write_text(writes[path][0], encoding='utf-8')
        if old_payload is None:namespace.pop('call_weather_card', None)
        else:namespace['call_weather_card'] = old_payload
        method.__code__, method.__doc__ = old_method_code, old_method_doc
        speech.__code__, speech.__doc__ = old_speech_code, old_speech_doc
        b.HTML = active_before
        raise
    print('Call-page weather card installed — UI binding 1.1: condition, temperature in °C, and rain chance for pickup and destination.')
    print('Tanglish Celsius pronunciation: செல்சியஸ். Tamil weather speech already uses செல்சியஸ்; English speech is retained.')
    print('Python / JavaScript syntax checks PASS | offline checks:', checks)
    print('Close the OLD call popup, refresh the signed-in dashboard, then open a NEW AI Call.')
    if backup:print('Backup:', backup)
    print('Test one complete ride request. The weather card should appear when the weather reply starts.')
    return dict(version=version, ui_binding='existing-socket-observer-1.1',offline_checks=checks, changed_files=[str(path.relative_to(root)) for path in changes],
        live_handler_identity_preserved=True, speech_function_identity_preserved=True, backup=str(backup) if backup else None)

MOVIO_CALL_WEATHER_CARD = movio_show_call_weather_card()


Call-page weather card installed — UI binding 1.1: condition, temperature in °C, and rain chance for pickup and destination.
Tanglish Celsius pronunciation: செல்சியஸ். Tamil weather speech already uses செல்சியஸ்; English speech is retained.
Python / JavaScript syntax checks PASS | offline checks: 30
Close the OLD call popup, refresh the signed-in dashboard, then open a NEW AI Call.
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/call_weather_card_backup_20261028_070732_163055
Test one complete ride request. The weather card should appear when the weather reply starts.


In [118]:
# Copy this ENTIRE file into ONE Kaggle cell AFTER your weather/UI fixes,
# BEFORE evaluation. Run it once, then open a NEW AI Call for listening.
def movio_tanglish_pronunciation_and_pauses():
    import ast
    import copy
    import inspect
    import json
    import os
    import re
    import textwrap
    import threading
    import types
    from datetime import datetime
    from pathlib import Path

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',{})
    friendly=getattr(engine.get('advisory'),'__globals__',{})
    speech=getattr(b,'tanglish_speech_text',None)
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    file=root/'integrations/movio_friendly_weather.py'
    speech_file=root/'integrations/movio_tanglish_speech_quality.py'
    if not getattr(b,'_tg_wordwise_v2',False) or not isinstance(speech,types.FunctionType):
        raise RuntimeError('Run the existing word-by-word Tanglish speech setup first. No changes were made.')
    if not file.is_file() or Path(friendly.get('__file__','')).resolve()!=file:
        raise RuntimeError('The active friendly-weather helper does not match its project file. No changes were made.')
    functions={name:engine.get(name) for name in ('summary_text','speech_segments')}
    if any(not isinstance(fn,types.FunctionType) or fn.__globals__ is not friendly for fn in functions.values()):
        raise RuntimeError('The existing summary/speech handlers have a different owner. No changes were made.')
    cellmap=dict(zip(speech.__code__.co_freevars,speech.__closure__ or ()))
    if set(cellmap)!={'lock','pattern','places','re','words','wx'}:
        raise RuntimeError('The Tanglish speech-copy closure is different. No changes were made.')
    values={name:cell.cell_contents for name,cell in cellmap.items()}
    if not isinstance(values['words'],dict) or not isinstance(values['places'],set):
        raise RuntimeError('The existing Tanglish pronunciation dictionary was not found. No changes were made.')
    synthesize=getattr(getattr(b,'TTS',None),'synthesize',None)
    if not callable(synthesize) or inspect.getclosurevars(synthesize).nonlocals.get('speech_text') is not speech:
        raise RuntimeError('The active TTS adapter does not use this speech-copy function. No changes were made.')
    register=inspect.getclosurevars(engine['advisory']).nonlocals.get('register')
    profile_for=inspect.getclosurevars(engine['tts_reply']).nonlocals.get('profile_for')
    if not callable(register) or not callable(profile_for):
        raise RuntimeError('Run the existing stable weather/closing audio setup first. No changes were made.')
    registry=inspect.getclosurevars(register).nonlocals.get('profiles')
    if registry is None or inspect.getclosurevars(profile_for).nonlocals.get('profiles') is not registry:
        raise RuntimeError('The weather audio registry is inconsistent. No changes were made.')

    helper_source=r'''"""Speech-copy helpers for the existing MOVIO Tanglish adapter."""
import re

VERSION = 'tanglish-speech-quality-1.0'

# English words stay English. The three requested pronunciation exceptions
# retain their meaning while using Tamil script in the speech copy only.
ENGLISH_WORDS = set(("""
ai welcome help service ride book cancel cancellation existing travel pickup
drop destination place correct yes no vehicle road distance around approximate
estimated fare confirm confirmation details detail passengers passenger
kilometres kilometers kilometer kilometres rupees rupee driver mini car bike
auto sedan suv am pm otp id number request record save saved select selected
screen map route verify continue rain chance forecast please check change
retry tomorrow today morning evening night weather update temperature degrees
degree cloudy sunny pleasant heavy strong wind windy thunderstorm thunderstorms
umbrella belongings bags care cover covered consider keep same time open
protection prepare sun heat warnings booking taxi celsius point zero oh one
two three four five six seven eight nine ten eleven twelve thirteen fourteen
fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty
seventy eighty ninety hundred thousand lakh lakhs crore crores million billion
and a an the to from for at in on by of with your you our my me it is are will
be can have thank thanks wonderful journey comfortable keep retain switch
january february march april may june july august september october november
december monday tuesday wednesday thursday friday saturday sunday
""").split())

ALIASES = dict(
    taxi='டாக்ஸி', booking='புக்கிங்', celsius='செல்சியஸ்',
    vanakkam='வணக்கம்', nandri='நன்றி', unga='உங்க', ungal='உங்கள்',
    ungalukku='உங்களுக்கு', neenga='நீங்க', eppadi='எப்படி',
    pannalama='பண்ணலாமா', pannalaama='பண்ணலாமா', pannalaam='பண்ணலாம்',
    pannunga='பண்ணுங்க', pannikkonga='பண்ணிக்கோங்க', pannrom='பண்றோம்',
    sollunga='சொல்லுங்க', sollalaam='சொல்லலாம்', illa='இல்ல', illai='இல்லை',
    vera='வேற', veru='வேறு', aiye='ஐயே', aiyum='ஐயும்', um='உம்',
    adhe='அதே', athe='அதே', idhe='இதே', ithae='இதே', indha='இந்த',
    andha='அந்த', naal='நாள்', naalaiku='நாளைக்கு', innikku='இன்னைக்கு',
    innaiku='இன்னைக்கு', neram='நேரம்', nerathil='நேரத்தில்',
    nerathula='நேரத்துல', kelambura='கிளம்புற', kelambumbodhu='கிளம்பும்போது',
    purappadum='புறப்படும்', purappadumbodhu='புறப்படும்போது',
    irukku='இருக்கு', irukkum='இருக்கும்', irukkura='இருக்குற',
    irukkalaam='இருக்கலாம்', irundhu='இருந்து', ilirundhu='இலிருந்து',
    peru='பேர்', ku='க்கு', la='ல', ah='ஆ',
    mazhai='மழை', peiya='பெய்ய', vaaippu='வாய்ப்பு',
    vaanilai='வானிலை', veppanilai='வெப்பநிலை', sumaar='சுமார்',
    adhigamaaga='அதிகமாக', adhigamaa='அதிகமா', konjam='கொஞ்சம்',
    megamoottam='மேகமூட்டம்', kanamazhai='கனமழை',
    kudai='குடை', kudineer='குடிநீர்', eduthuttu='எடுத்துட்டு',
    eduthukkonga='எடுத்துக்கோங்க', eduthukkondu='எடுத்துக்கொண்டு',
    nanaiyaamal='நனையாமல்', nanaiyama='நனையாம',
    paadhugaathukkollunga='பாதுகாத்துக்கொள்ளுங்க',
    paadhugaappa='பாதுகாப்பா', vachukkonga='வச்சுக்கோங்க',
    erumboadhum='ஏறும்போதும்', irangumboadhum='இறங்கும்போதும்',
    eravum='ஏறவும்', irangavum='இறங்கவும்',
    pakkavaatil='பக்கவாட்டில்', padalaam='படலாம்',
    purappadumun='புறப்படும் முன்', mun='முன்', ulloor='உள்ளூர்',
    paarunga='பாருங்க', thevaiyaana='தேவையான', munnerpaadugal='முன்னேற்பாடுகள்',
    payanathirku='பயணத்திற்கு', payanam='பயணம்', inimaiyaaga='இனிமையாக',
    amaiyattum='அமையட்டும்', nandraagave='நன்றாகவே',
)


def make_speech_text(lock, pattern, places, re, words, wx):
    def speech_text(text):
        """MOVIO call weather card v1: Celsius is pronounced செல்சியஸ்."""
        text = str(text)
        with lock:
            names = sorted(places, key=len, reverse=True)
        protected = {}
        for name in names:
            if name and name in text:
                marker = chr(0xF000 + len(protected))
                text = text.replace(name, marker)
                protected[marker] = name
        text = re.sub(r'(?i)\bkilo\s+meettar\b|\bkm\b', 'kilometers', text)
        text = re.sub(r'(?i)\b(\d{1,2}(?::\d{2})?)\s*(AM|PM)\b', lambda match: match[1] + ' ' + match[2].upper(), text)
        # Read identifiers BEFORE booking becomes புக்கிங். This preserves
        # whole OTP/booking identifiers as ordered English spoken digits.
        text = wx['speech_numeric_normalize'](lambda value: value, text, 'English')
        text = pattern.sub(lambda match: words[match[0].lower()], text)
        text = re.sub(r'\b(?:SUV|S\s+U\s+V|es yoo vee)\b', 'ess you vee', text, flags=re.I)
        aliases = {'AI':'ay eye', 'AM':'ay em', 'PM':'pee em', 'OTP':'oh tee pee', 'ID':'eye dee'}
        text = re.sub(r'\b(?:AI|AM|PM|OTP|ID)\b', lambda match: aliases[match[0]], text)
        # Punctuation affects speech only. The displayed response is untouched.
        text = re.sub(r';\s*', '. ', text)
        text = re.sub(r':(?=\s)', ',', text)
        for marker, name in protected.items():
            spoken_name = wx['speech_numeric_normalize'](lambda value: value, name, 'English')
            text = text.replace(marker, spoken_name)
        return re.sub(r'\s+', ' ', text).strip()
    return speech_text


def phrases(text, protected_places=(), minimum_words=8):
    """Complete clauses, retaining decimal values and protected place names."""
    value = str(text).strip()
    if not value:
        return []
    protected = {}
    for name in sorted(set(protected_places), key=len, reverse=True):
        if name and name in value:
            marker = chr(0xF000 + len(protected))
            value = value.replace(name, marker)
            protected[marker] = name
    clauses = re.split(r'(?<=[.!?;])\s+|\n+', value)
    parts, pending = [], ''
    for clause in clauses:
        if not clause.strip():
            continue
        pending = (pending + ' ' + clause.strip()).strip()
        if re.search(r'(?:^|\s)(?:[A-Za-z]|Dr|Mr|Ms|Mrs|St)\.$', pending):
            continue
        if len(pending.split()) >= minimum_words:
            parts.append(pending);pending = ''
    if pending:
        if parts:
            parts[-1] += ' ' + pending
        else:
            parts.append(pending)
    for index, part in enumerate(parts):
        for marker, name in protected.items():
            part = part.replace(marker, name)
        parts[index] = part
    return parts[:11] + ([' '.join(parts[11:])] if len(parts) > 11 else [])
'''
    replacements={
'summary_text': r'''def summary_text(b,state):
    """MOVIO step-3E: exactly one Tanglish Yes/No instruction."""
    text=previous['summary_text'](b,state)
    if state.get('_wx_mode')=='manual':return text
    if state.get('lang')=='Tanglish':
        text=re.sub(r'(?:\s*Yes\s+illa\s+no\s+sollunga\s*[.!?]*)+\s*$', '', str(text), flags=re.I).rstrip()
        return text+' Yes illa no sollunga.'
    return text+' '+pick(state,'Please say yes or no.','ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.','Yes illa no sollunga.')
''',
'speech_segments': r'''def speech_segments(text):
    """MOVIO step-3E: Tanglish sentence pauses with stable weather audio."""
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    runtime=globals().get('_MOVIO_TG_SPEECH_RUNTIME') or {}
    bridge=runtime.get('bridge')
    if bridge is not None and bridge.CURRENT_LANGUAGE.get()=='Tanglish':
        with runtime['lock']:names=tuple(runtime['places'])
        split=lambda part:runtime['phrases'](part,names)
        if prefix:
            rest=value[len(prefix):].strip()
            review_pair=value.startswith(prefix+'\n\n') or bool(rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold() or 'approximate fare' in rest.casefold()))
            weather_parts=split(prefix)
            if rest and not review_pair:
                if weather_parts:weather_parts[-1]+=' '+rest
                else:weather_parts=[value]
            for part in weather_parts:runtime['register'](part,'weather')
            return weather_parts+(split(rest) if rest and review_pair else [])
        if runtime['profile_for'](value)=='closing':return previous['speech_segments'](text)
        return split(value)
    # Retain the existing English/Tamil weather and final-review behaviour.
    if prefix:
        rest=value[len(prefix):].strip()
        normal_review_pair=value.startswith(prefix+'\n\n')
        return [prefix,rest] if rest and (normal_review_pair or 'estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
'''
}
    allowed={
'summary_text': [r'''def summary_text(b,state):
    text=previous['summary_text'](b,state)
    if state.get('_wx_mode')=='manual':return text
    return text+' '+pick(state,'Please say yes or no.','ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.','Yes illa no sollunga.')
'''],
'speech_segments': [r'''def speech_segments(text):
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        # Keep a coherent weather chunk; pause before a separate final summary.
        return [prefix,rest] if rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
''',
r'''def speech_segments(text):
    """MOVIO step-3C-v1: pause between weather and the final ride summary."""
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        normal_review_pair=value.startswith(prefix+'\n\n')
        # Keep interactive weather choices on their existing speech path.
        return [prefix,rest] if rest and (normal_review_pair or 'estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
''']
}
    version='tanglish-speech-quality-1.0'
    helper=types.ModuleType('movio_tanglish_speech_quality');helper.__file__=str(speech_file)
    exec(compile(helper_source,str(speech_file),'exec',dont_inherit=True),helper.__dict__)

    def structural(source):return ast.dump(ast.parse(textwrap.dedent(source)),include_attributes=False)
    def fingerprint(code):
        constants=tuple(fingerprint(value) if isinstance(value,types.CodeType) else value for value in code.co_consts)
        return (code.co_code,constants,code.co_names,code.co_varnames,code.co_freevars,code.co_cellvars,
            code.co_argcount,code.co_kwonlyargcount,code.co_posonlyargcount,code.co_flags,code.co_exceptiontable)
    def code_at(code,names):
        for name in names:
            matches=[value for value in code.co_consts if isinstance(value,types.CodeType) and value.co_name==name]
            if len(matches)!=1:raise RuntimeError('Unexpected existing function structure: '+name+'. No changes were made.')
            code=matches[0]
        return code

    original=file.read_text(encoding='utf-8')
    parsed=ast.parse(original)
    installers=[node for node in parsed.body if isinstance(node,ast.FunctionDef) and node.name=='install_namespace']
    if len(installers)!=1:raise RuntimeError('The existing friendly-weather installer was not found. No changes were made.')
    targets={node.name:node for node in installers[0].body if isinstance(node,ast.FunctionDef) and node.name in replacements}
    if set(targets)!=set(replacements):raise RuntimeError('The original speech handlers were not found. No changes were made.')
    compiled=compile(original,str(file),'exec',dont_inherit=True)
    lines=original.splitlines(keepends=True)
    for name,node in targets.items():
        source=textwrap.dedent(''.join(lines[node.lineno-1:node.end_lineno]))
        if structural(source) not in {structural(value) for value in allowed[name]+[replacements[name]]}:
            raise RuntimeError('The existing '+name+' source has changed. No changes were made.')
        if fingerprint(functions[name].__code__)!=fingerprint(code_at(compiled,('install_namespace',name))):
            raise RuntimeError('The live '+name+' differs from its saved source. No changes were made.')
    for name,node in sorted(targets.items(),key=lambda item:item[1].lineno,reverse=True):
        lines[node.lineno-1:node.end_lineno]=[textwrap.indent(replacements[name].strip()+'\n','    ')]
    updated=''.join(lines)
    candidate=compile(updated,str(file),'exec',dont_inherit=True)
    codes={name:code_at(candidate,('install_namespace',name)) for name in functions}
    if any(codes[name].co_freevars!=fn.__code__.co_freevars for name,fn in functions.items()):
        raise RuntimeError('An existing dialogue closure cannot be preserved. No changes were made.')

    words=dict(values['words'])
    for word in helper.ENGLISH_WORDS:words.pop(word,None)
    words.update(helper.ALIASES)
    pattern=re.compile(r'(?<![A-Za-z])('+'|'.join(re.escape(key) for key in sorted(words,key=len,reverse=True))+r')(?![A-Za-z])',re.I)
    candidate_speech=helper.make_speech_text(values['lock'],pattern,values['places'],values['re'],words,values['wx'])
    if candidate_speech.__code__.co_freevars!=speech.__code__.co_freevars:
        raise RuntimeError('The original speech-copy closure cannot be preserved. No changes were made.')
    summary_candidate=types.FunctionType(codes['summary_text'],friendly,'summary_text',functions['summary_text'].__defaults__,functions['summary_text'].__closure__)

    checks=0
    def check(condition,label):
        nonlocal checks
        if not condition:raise RuntimeError('Pre-install check failed: '+label+'. No changes were made.')
        checks+=1
    spoken=candidate_speech('Vanakkam! AI taxi booking service ku welcome. Ungalukku eppadi help pannalaam?')
    check('டாக்ஸி' in spoken and 'புக்கிங்' in spoken,'requested taxi and booking pronunciation')
    check('வணக்கம்' in spoken and 'உங்களுக்கு' in spoken,'Tamil-origin words use Tamil script')
    check('help' in spoken and 'service' in spoken and 'welcome' in spoken,'English words retain English wording')
    spoken=candidate_speech('Road distance 35.4 kilometers. Approximate fare ₹1,126. Pickup 7:00 PM, temperature 27.3 degrees Celsius, rain chance 70%.')
    for value in ('thirty five point four','one thousand one hundred twenty six','pee em','twenty seven point three','செல்சியஸ்','seventy percent'):
        check(value in spoken,'entity speech: '+value)
    check('four eight two nine one' in candidate_speech('Booking number 48291. OTP 0426.'),'booking ID stays in digit order')
    check('zero four two six' in candidate_speech('Booking number 48291. OTP 0426.'),'OTP leading zero is retained')
    check('ess you vee' in candidate_speech('4 peru, SUV.'),'SUV letter pronunciation retained')
    spoken=candidate_speech('Unga ride details sollalaam; pickup nerathil cloudy weather irukkum; vera time venuma?')
    check('உங்க' in spoken and 'நேரத்தில்' in spoken and 'வேற' in spoken,'weather and confirmation Tamil vocabulary')
    check('cloudy weather' in spoken and ';' not in spoken,'speech-only full stops and English weather terms')
    state=dict(lang='Tanglish',_wx_mode='voice',count=4,vehicle='SUV',pickup='Chennai Central',destination='Tambaram',
        pickup_at='2030-10-19T19:00:00+05:30',fare=dict(km=35.4,fare=1126))
    rendered=summary_candidate(b,state)
    check(len(re.findall(r'yes\s+illa\s+no\s+sollunga',rendered,re.I))==1,'one final Yes/No instruction')
    check(not re.search(r'[\u0b80-\u0bff]',rendered),'no Tamil speech spelling in the Tanglish display summary')
    check('35.4' in rendered and '1126' in rendered.replace(',',''),'display retains raw numerical entities')
    for lang in ('English','Tamil'):
        state['lang']=lang
        check(summary_candidate(b,state)==functions['summary_text'](b,state),lang+' summary remains unchanged')
    state['lang']='Tanglish';state['_wx_mode']='manual'
    check(summary_candidate(b,state)==functions['summary_text'](b,state),'manual summary remains unchanged')
    preview='Oru weather update: pickup nerathil cloudy weather irukkum; pickup temperature sumaar 27.3 degrees Celsius irukkum; Chennai Central-la 42%, Tambaram-la 51% rain chance irukku. Umbrella eduthukkondu unga belongings nanaiyaamal paadhugaathukkollunga. Unga ride details sollalaam.'
    chunks=helper.phrases(preview,('Chennai Central','Tambaram'))
    check(len(chunks)>=2,'weather is split into complete clauses')
    check(re.sub(r'\s+',' ',' '.join(chunks))==re.sub(r'\s+',' ',preview),'weather words and entities are preserved')
    check(all(len(chunk.split())>=8 for chunk in chunks),'short isolated speech fragments are avoided')
    place='St. Thomas Mount'
    check(any(place in part for part in helper.phrases('Pickup is at '+place+'. Your road distance is 35.4 kilometers. Your estimated fare is 1126 rupees.',(place,))),'place names and decimal values stay intact')

    current_pause=getattr(b,'_wx_pause_ms',220)
    pause_ms=max(0,min(1000,int(current_pause)))
    if not pause_ms:pause_ms=220
    runtime=dict(version=version,bridge=b,phrases=helper.phrases,register=register,profile_for=profile_for,
        lock=values['lock'],places=values['places'])
    writes={file:(original,updated),speech_file:(speech_file.read_text(encoding='utf-8') if speech_file.exists() else None,helper_source)}
    changes={path:(before,after) for path,(before,after) in writes.items() if before!=after}
    live_changes=(getattr(speech,'_movio_tanglish_quality',None)!=version or words!=values['words']
        or fingerprint(speech.__code__)!=fingerprint(candidate_speech.__code__) or pause_ms!=current_pause
        or any(fingerprint(fn.__code__)!=fingerprint(codes[name]) for name,fn in functions.items()))
    backup=None
    if changes or live_changes:
        backup=root/('tanglish_speech_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,(before,_) in changes.items():
            if before is not None:
                target=backup/path.relative_to(root);target.parent.mkdir(parents=True,exist_ok=True);target.write_text(before,encoding='utf-8')
        (backup/'original_pronunciation_dictionary.json').write_text(json.dumps(values['words'],ensure_ascii=False,indent=2),encoding='utf-8')
        try:(backup/'original_speech_function.py').write_text(textwrap.dedent(inspect.getsource(speech)),encoding='utf-8')
        except (OSError,TypeError):pass
    old_codes={name:(fn.__code__,fn.__doc__) for name,fn in functions.items()}
    old_speech=(speech.__code__,speech.__doc__,getattr(speech,'_movio_tanglish_quality',None))
    old_runtime=friendly.get('_MOVIO_TG_SPEECH_RUNTIME')
    written=[]
    try:
        for path,(_,after) in changes.items():
            temporary=path.with_name(path.name+'.tg-speech-tmp')
            try:temporary.write_text(after,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
            written.append(path)
        with getattr(b,'LOCK',threading.RLock()),values['lock']:
            friendly['_MOVIO_TG_SPEECH_RUNTIME']=runtime
            for name,fn in functions.items():fn.__code__=codes[name];fn.__doc__=codes[name].co_consts[0]
            cellmap['words'].cell_contents=words;cellmap['pattern'].cell_contents=pattern
            speech.__code__=candidate_speech.__code__;speech.__doc__=candidate_speech.__doc__
            speech._movio_tanglish_quality=version
            b._wx_pause_ms=pause_ms
    except Exception:
        for path in reversed(written):
            if writes[path][0] is None:path.unlink(missing_ok=True)
            else:path.write_text(writes[path][0],encoding='utf-8')
        for name,fn in functions.items():fn.__code__,fn.__doc__=old_codes[name]
        speech.__code__,speech.__doc__=old_speech[:2]
        if old_speech[2] is None:speech.__dict__.pop('_movio_tanglish_quality',None)
        else:speech._movio_tanglish_quality=old_speech[2]
        cellmap['words'].cell_contents=values['words'];cellmap['pattern'].cell_contents=values['pattern']
        if old_runtime is None:friendly.pop('_MOVIO_TG_SPEECH_RUNTIME',None)
        else:friendly['_MOVIO_TG_SPEECH_RUNTIME']=old_runtime
        b._wx_pause_ms=current_pause
        raise
    print('Step 3E installed — Tanglish speech spelling and meaningful sentence pauses.')
    print('taxi → டாக்ஸி | booking → புக்கிங் | Celsius → செல்சியஸ்')
    print('Confirmation: Yes illa no sollunga appears once. Browser transcript stays in Romanized Tanglish.')
    print('Complete weather/confirmation phrases use the existing',pause_ms,'ms playback gap; weather clips retain their stable audio profile.')
    print('Syntax/source checks PASS | offline checks:',checks)
    if backup:print('Backup:',backup)
    print('Open a NEW Tanglish AI Call and listen to the greeting, weather advisory and final confirmation.')
    print('Actual voice quality still requires listening; no TTS audio or real booking was generated by this installer.')
    return dict(version=version,offline_checks=checks,pause_ms=pause_ms,changed_files=[str(path.relative_to(root)) for path in changes],
        backup=str(backup) if backup else None,visible_tanglish='Romanized; duplicate instruction removed',tts='existing DhVaani adapter')

MOVIO_TANGLISH_SPEECH_QUALITY=movio_tanglish_pronunciation_and_pauses()


Step 3E installed — Tanglish speech spelling and meaningful sentence pauses.
taxi → டாக்ஸி | booking → புக்கிங் | Celsius → செல்சியஸ்
Confirmation: Yes illa no sollunga appears once. Browser transcript stays in Romanized Tanglish.
Complete weather/confirmation phrases use the existing 220 ms playback gap; weather clips retain their stable audio profile.
Syntax/source checks PASS | offline checks: 24
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/tanglish_speech_backup_20261028_070732_248998
Open a NEW Tanglish AI Call and listen to the greeting, weather advisory and final confirmation.
Actual voice quality still requires listening; no TTS audio or real booking was generated by this installer.


In [119]:
# Copy this ENTIRE file into ONE Kaggle cell AFTER your weather/UI fixes,
# BEFORE evaluation. Run it once, then open a NEW AI Call for listening.
def movio_tanglish_pronunciation_and_pauses():
    import ast
    import copy
    import inspect
    import json
    import os
    import re
    import textwrap
    import threading
    import types
    from datetime import datetime
    from pathlib import Path

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',{})
    friendly=getattr(engine.get('advisory'),'__globals__',{})
    speech=getattr(b,'tanglish_speech_text',None)
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    file=root/'integrations/movio_friendly_weather.py'
    speech_file=root/'integrations/movio_tanglish_speech_quality.py'
    if not getattr(b,'_tg_wordwise_v2',False) or not isinstance(speech,types.FunctionType):
        raise RuntimeError('Run the existing word-by-word Tanglish speech setup first. No changes were made.')
    if not file.is_file() or Path(friendly.get('__file__','')).resolve()!=file:
        raise RuntimeError('The active friendly-weather helper does not match its project file. No changes were made.')
    functions={name:engine.get(name) for name in ('summary_text','speech_segments')}
    if any(not isinstance(fn,types.FunctionType) or fn.__globals__ is not friendly for fn in functions.values()):
        raise RuntimeError('The existing summary/speech handlers have a different owner. No changes were made.')
    cellmap=dict(zip(speech.__code__.co_freevars,speech.__closure__ or ()))
    if set(cellmap)!={'lock','pattern','places','re','words','wx'}:
        raise RuntimeError('The Tanglish speech-copy closure is different. No changes were made.')
    values={name:cell.cell_contents for name,cell in cellmap.items()}
    if not isinstance(values['words'],dict) or not isinstance(values['places'],set):
        raise RuntimeError('The existing Tanglish pronunciation dictionary was not found. No changes were made.')
    synthesize=getattr(getattr(b,'TTS',None),'synthesize',None)
    if not callable(synthesize) or inspect.getclosurevars(synthesize).nonlocals.get('speech_text') is not speech:
        raise RuntimeError('The active TTS adapter does not use this speech-copy function. No changes were made.')
    register=inspect.getclosurevars(engine['advisory']).nonlocals.get('register')
    profile_for=inspect.getclosurevars(engine['tts_reply']).nonlocals.get('profile_for')
    if not callable(register) or not callable(profile_for):
        raise RuntimeError('Run the existing stable weather/closing audio setup first. No changes were made.')
    registry=inspect.getclosurevars(register).nonlocals.get('profiles')
    if registry is None or inspect.getclosurevars(profile_for).nonlocals.get('profiles') is not registry:
        raise RuntimeError('The weather audio registry is inconsistent. No changes were made.')

    helper_source=r'''"""Speech-copy helpers for the existing MOVIO Tanglish adapter."""
import re

VERSION = 'tanglish-speech-quality-1.3'

# English words stay English. The three requested pronunciation exceptions
# retain their meaning while using Tamil script in the speech copy only.
ENGLISH_WORDS = set(("""
ai welcome help service ride book cancel cancellation existing travel pickup
drop destination place correct yes no vehicle road distance around approximate
estimated fare confirm confirmation details detail passengers passenger
kilometres kilometers kilometer kilometres rupees rupee driver mini car bike
auto sedan suv am pm otp id number request record save saved select selected
screen map route verify continue rain chance forecast please check change
retry tomorrow today morning evening night weather update temperature degrees
degree cloudy sunny pleasant heavy strong wind windy thunderstorm thunderstorms
umbrella belongings bags care cover covered consider keep same time open
protection prepare sun heat warnings booking taxi celsius point zero oh one
two three four five six seven eight nine ten eleven twelve thirteen fourteen
fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty
seventy eighty ninety hundred thousand lakh lakhs crore crores million billion
and a an the to from for at in on by of with your you our my me it is are will
be can have thank thanks wonderful journey comfortable keep retain switch
january february march april may june july august september october november
december monday tuesday wednesday thursday friday saturday sunday
""").split())

ALIASES = dict(
    taxi='டாக்ஸி', booking='புக்கிங்', celsius='செல்சியஸ்',
    temperature='டெம்பரேச்சர்', central='சென்ட்ரல்',
    belongings='பிலாங்கிங்ஸ்', belonging='பிலாங்கிங்',
    iruki='இருக்கு', irukki='இருக்கு', iruku='இருக்கு',
    solluren='சொல்லுறேன்', solluraen='சொல்லுறேன்',
    vanakkam='வணக்கம்', nandri='நன்றி', unga='உங்க', ungal='உங்கள்',
    ungalukku='உங்களுக்கு', neenga='நீங்க', eppadi='எப்படி',
    pannalama='பண்ணலாமா', pannalaama='பண்ணலாமா', pannalaam='பண்ணலாம்',
    pannunga='பண்ணுங்க', pannikkonga='பண்ணிக்கோங்க', pannrom='பண்றோம்',
    sollunga='சொல்லுங்க', sollalaam='சொல்லலாம்', illa='இல்ல', illai='இல்லை',
    vera='வேற', veru='வேறு', aiye='ஐயே', aiyum='ஐயும்', um='உம்',
    adhe='அதே', athe='அதே', idhe='இதே', ithae='இதே', indha='இந்த',
    andha='அந்த', naal='நாள்', naalaiku='நாளைக்கு', innikku='இன்னைக்கு',
    innaiku='இன்னைக்கு', neram='நேரம்', nerathil='நேரத்தில்',
    nerathula='நேரத்துல', kelambura='கிளம்புற', kelambumbodhu='கிளம்பும்போது',
    purappadum='புறப்படும்', purappadumbodhu='புறப்படும்போது',
    irukku='இருக்கு', irukkum='இருக்கும்', irukkura='இருக்குற',
    irukkalaam='இருக்கலாம்', irundhu='இருந்து', ilirundhu='இலிருந்து',
    peru='பேர்', ku='க்கு', la='ல', ah='ஆ',
    mazhai='மழை', peiya='பெய்ய', vaaippu='வாய்ப்பு',
    vaanilai='வானிலை', veppanilai='வெப்பநிலை', sumaar='சுமார்',
    adhigamaaga='அதிகமாக', adhigamaa='அதிகமா', konjam='கொஞ்சம்',
    megamoottam='மேகமூட்டம்', kanamazhai='கனமழை',
    kudai='குடை', kudineer='குடிநீர்', eduthuttu='எடுத்துட்டு',
    eduthukkonga='எடுத்துக்கோங்க', eduthukkondu='எடுத்துக்கொண்டு',
    nanaiyaamal='நனையாமல்', nanaiyama='நனையாம',
    paadhugaathukkollunga='பாதுகாத்துக்கொள்ளுங்க',
    paadhugaappa='பாதுகாப்பா', vachukkonga='வச்சுக்கோங்க',
    erumboadhum='ஏறும்போதும்', irangumboadhum='இறங்கும்போதும்',
    eravum='ஏறவும்', irangavum='இறங்கவும்',
    pakkavaatil='பக்கவாட்டில்', padalaam='படலாம்',
    purappadumun='புறப்படும் முன்', mun='முன்', ulloor='உள்ளூர்',
    paarunga='பாருங்க', thevaiyaana='தேவையான', munnerpaadugal='முன்னேற்பாடுகள்',
    payanathirku='பயணத்திற்கு', payanam='பயணம்', inimaiyaaga='இனிமையாக',
    amaiyattum='அமையட்டும்', nandraagave='நன்றாகவே',
)


def make_speech_text(lock, pattern, places, re, words, wx):
    def speech_text(text):
        """MOVIO call weather card v1: Celsius is pronounced செல்சியஸ்."""
        text = str(text)
        with lock:
            names = sorted(places, key=len, reverse=True)
        protected = {}
        for name in names:
            if name and name in text:
                marker = chr(0xF000 + len(protected))
                text = text.replace(name, marker)
                protected[marker] = name
        text = re.sub(r'(?i)\bkilo\s+meettar\b|\bkm\b', 'kilometers', text)
        text = re.sub(r'(?i)\b(\d{1,2}(?::\d{2})?)\s*(AM|PM)\b', lambda match: match[1] + ' ' + match[2].upper(), text)
        # Read identifiers BEFORE booking becomes புக்கிங். This preserves
        # whole OTP/booking identifiers as ordered English spoken digits.
        text = wx['speech_numeric_normalize'](lambda value: value, text, 'English')
        text = re.sub(r'(?i)\bunga\s+ride\s+details\s+(?:sollalaam|solluren|solluraen|solren|sollren)\b',
                      'உங்க ride details சொல்லுறேன்', text)
        text = pattern.sub(lambda match: words[match[0].lower()], text)
        text = re.sub(r'\b(?:SUV|S\s+U\s+V|es yoo vee)\b', 'ess you vee', text, flags=re.I)
        aliases = {'AI':'ay eye', 'AM':'ay em', 'PM':'pee em', 'OTP':'oh tee pee', 'ID':'eye dee'}
        text = re.sub(r'\b(?:AI|AM|PM|OTP|ID)\b', lambda match: aliases[match[0]], text)
        # Punctuation affects speech only. The displayed response is untouched.
        text = re.sub(r';\s*', '. ', text)
        text = re.sub(r':(?=\s)', ',', text)
        for marker, name in protected.items():
            spoken_name = wx['speech_numeric_normalize'](lambda value: value, name, 'English')
            spoken_name = re.sub(r'(?i)\bcentral\b', 'சென்ட்ரல்', spoken_name)
            text = text.replace(marker, spoken_name)
        return re.sub(r'\s+', ' ', text).strip()
    return speech_text


def phrases(text, protected_places=(), minimum_words=8):
    """Complete clauses, retaining decimal values and protected place names."""
    value = str(text).strip()
    if not value:
        return []
    protected = {}
    for name in sorted(set(protected_places), key=len, reverse=True):
        if name and name in value:
            marker = chr(0xF000 + len(protected))
            value = value.replace(name, marker)
            protected[marker] = name
    clauses = re.split(r'(?<=[.!?;])\s+|\n+', value)
    parts, pending = [], ''
    for clause in clauses:
        if not clause.strip():
            continue
        pending = (pending + ' ' + clause.strip()).strip()
        if re.search(r'(?:^|\s)(?:[A-Za-z]|Dr|Mr|Ms|Mrs|St)\.$', pending):
            continue
        if len(pending.split()) >= minimum_words:
            parts.append(pending);pending = ''
    if pending:
        if parts:
            parts[-1] += ' ' + pending
        else:
            parts.append(pending)
    for index, part in enumerate(parts):
        for marker, name in protected.items():
            part = part.replace(marker, name)
        parts[index] = part
    return parts
'''
    replacements={
'summary_text': r'''def summary_text(b,state):
    """MOVIO step-3E: exactly one Tanglish Yes/No instruction."""
    text=previous['summary_text'](b,state)
    if state.get('_wx_mode')=='manual':return text
    if state.get('lang')=='Tanglish':
        text=re.sub(r'(?:\s*Yes\s+illa\s+no\s+sollunga\s*[.!?]*)+\s*$', '', str(text), flags=re.I).rstrip()
        text=re.sub(r'(?i)\bunga\s+ride\s+details\s+(?:sollalaam|solluren|solluraen|solren|sollren)\b',
                    'Unga ride details solluren', text)
        return text+' Yes illa no sollunga.'
    return text+' '+pick(state,'Please say yes or no.','ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.','Yes illa no sollunga.')
''',
'speech_segments': r'''def speech_segments(text):
    """MOVIO step-3E: Tanglish sentence pauses with stable weather audio."""
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    runtime=globals().get('_MOVIO_TG_SPEECH_RUNTIME') or {}
    bridge=runtime.get('bridge')
    if bridge is not None and bridge.CURRENT_LANGUAGE.get()=='Tanglish':
        with runtime['lock']:names=tuple(runtime['places'])
        split=lambda part:runtime['phrases'](part,names)
        if prefix:
            rest=value[len(prefix):].strip()
            review_pair=value.startswith(prefix+'\n\n') or bool(rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold() or 'approximate fare' in rest.casefold()))
            weather_parts=split(prefix)
            if rest and not review_pair:
                if weather_parts:weather_parts[-1]+=' '+rest
                else:weather_parts=[value]
            for part in weather_parts:runtime['register'](part,'weather')
            return weather_parts+(split(rest) if rest and review_pair else [])
        if runtime['profile_for'](value)=='closing':return previous['speech_segments'](text)
        return split(value)
    # Retain the existing English/Tamil weather and final-review behaviour.
    if prefix:
        rest=value[len(prefix):].strip()
        normal_review_pair=value.startswith(prefix+'\n\n')
        return [prefix,rest] if rest and (normal_review_pair or 'estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
'''
}
    allowed={
'summary_text': [r'''def summary_text(b,state):
    text=previous['summary_text'](b,state)
    if state.get('_wx_mode')=='manual':return text
    return text+' '+pick(state,'Please say yes or no.','ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.','Yes illa no sollunga.')
'''],
'speech_segments': [r'''def speech_segments(text):
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        # Keep a coherent weather chunk; pause before a separate final summary.
        return [prefix,rest] if rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
''',
r'''def speech_segments(text):
    """MOVIO step-3C-v1: pause between weather and the final ride summary."""
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        normal_review_pair=value.startswith(prefix+'\n\n')
        # Keep interactive weather choices on their existing speech path.
        return [prefix,rest] if rest and (normal_review_pair or 'estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
''']
}
    version='tanglish-speech-quality-1.3'
    helper=types.ModuleType('movio_tanglish_speech_quality');helper.__file__=str(speech_file)
    exec(compile(helper_source,str(speech_file),'exec',dont_inherit=True),helper.__dict__)

    def structural(source):return ast.dump(ast.parse(textwrap.dedent(source)),include_attributes=False)
    def fingerprint(code):
        constants=tuple(fingerprint(value) if isinstance(value,types.CodeType) else value for value in code.co_consts)
        return (code.co_code,constants,code.co_names,code.co_varnames,code.co_freevars,code.co_cellvars,
            code.co_argcount,code.co_kwonlyargcount,code.co_posonlyargcount,code.co_flags,code.co_exceptiontable)
    def code_at(code,names):
        for name in names:
            matches=[value for value in code.co_consts if isinstance(value,types.CodeType) and value.co_name==name]
            if len(matches)!=1:raise RuntimeError('Unexpected existing function structure: '+name+'. No changes were made.')
            code=matches[0]
        return code

    original=file.read_text(encoding='utf-8')
    parsed=ast.parse(original)
    installers=[node for node in parsed.body if isinstance(node,ast.FunctionDef) and node.name=='install_namespace']
    if len(installers)!=1:raise RuntimeError('The existing friendly-weather installer was not found. No changes were made.')
    targets={node.name:node for node in installers[0].body if isinstance(node,ast.FunctionDef) and node.name in replacements}
    if set(targets)!=set(replacements):raise RuntimeError('The original speech handlers were not found. No changes were made.')
    compiled=compile(original,str(file),'exec',dont_inherit=True)
    lines=original.splitlines(keepends=True)
    for name,node in targets.items():
        source=textwrap.dedent(''.join(lines[node.lineno-1:node.end_lineno]))
        recognized = structural(source) in {structural(value) for value in allowed[name]+[replacements[name]]}
        if not recognized and name == 'summary_text':
            # Accept the previously installed step-3E handler, including harmless
            # formatting/docstring differences. Still verify its AST behaviour.
            tree = ast.parse(source)
            node0 = tree.body[0]
            normalized = ast.dump(node0, include_attributes=False)
            recognized = (
                isinstance(node0, ast.FunctionDef)
                and node0.name == 'summary_text'
                and len(node0.args.args) == 2
                and [arg.arg for arg in node0.args.args] == ['b', 'state']
                and "previous['summary_text']" in source
                and "state.get('_wx_mode')" in source
                and "state.get('lang')" in source
                and 'Yes illa no sollunga.' in source
                and 're.sub(' in source
                and 'pick(state,' in source
                and len(node0.body) in (5, 6)
            )
        if not recognized:
            raise RuntimeError('The existing '+name+' source has changed. No changes were made.')
        if fingerprint(functions[name].__code__)!=fingerprint(code_at(compiled,('install_namespace',name))):
            raise RuntimeError('The live '+name+' differs from its saved source. No changes were made.')
    for name,node in sorted(targets.items(),key=lambda item:item[1].lineno,reverse=True):
        lines[node.lineno-1:node.end_lineno]=[textwrap.indent(replacements[name].strip()+'\n','    ')]
    updated=''.join(lines)
    candidate=compile(updated,str(file),'exec',dont_inherit=True)
    codes={name:code_at(candidate,('install_namespace',name)) for name in functions}
    if any(codes[name].co_freevars!=fn.__code__.co_freevars for name,fn in functions.items()):
        raise RuntimeError('An existing dialogue closure cannot be preserved. No changes were made.')

    words=dict(values['words'])
    for word in helper.ENGLISH_WORDS:words.pop(word,None)
    words.update(helper.ALIASES)
    pattern=re.compile(r'(?<![A-Za-z])('+'|'.join(re.escape(key) for key in sorted(words,key=len,reverse=True))+r')(?![A-Za-z])',re.I)
    candidate_speech=helper.make_speech_text(values['lock'],pattern,values['places'],values['re'],words,values['wx'])
    if candidate_speech.__code__.co_freevars!=speech.__code__.co_freevars:
        raise RuntimeError('The original speech-copy closure cannot be preserved. No changes were made.')
    summary_candidate=types.FunctionType(codes['summary_text'],friendly,'summary_text',functions['summary_text'].__defaults__,functions['summary_text'].__closure__)

    checks=0
    def check(condition,label):
        nonlocal checks
        if not condition:raise RuntimeError('Pre-install check failed: '+label+'. No changes were made.')
        checks+=1
    spoken=candidate_speech('Vanakkam! AI taxi booking service ku welcome. Ungalukku eppadi help pannalaam?')
    check('டாக்ஸி' in spoken and 'புக்கிங்' in spoken,'requested taxi and booking pronunciation')
    check('வணக்கம்' in spoken and 'உங்களுக்கு' in spoken,'Tamil-origin words use Tamil script')
    check('help' in spoken and 'service' in spoken and 'welcome' in spoken,'English words retain English wording')
    spoken=candidate_speech('Road distance 35.4 kilometers. Approximate fare ₹1,126. Pickup 7:00 PM, temperature 27.3 degrees Celsius, rain chance 70%.')
    for value in ('thirty five point four','one thousand one hundred twenty six','pee em','twenty seven point three','செல்சியஸ்','seventy percent'):
        check(value in spoken,'entity speech: '+value)
    check('four eight two nine one' in candidate_speech('Booking number 48291. OTP 0426.'),'booking ID stays in digit order')
    check('zero four two six' in candidate_speech('Booking number 48291. OTP 0426.'),'OTP leading zero is retained')
    check('ess you vee' in candidate_speech('4 peru, SUV.'),'SUV letter pronunciation retained')
    spoken=candidate_speech('Unga ride details sollalaam; pickup nerathil cloudy weather irukkum; vera time venuma?')
    check('உங்க' in spoken and 'நேரத்தில்' in spoken and 'வேற' in spoken,'weather and confirmation Tamil vocabulary')
    check('cloudy weather' in spoken and ';' not in spoken,'speech-only full stops and English weather terms')
    pronunciation=candidate_speech('Unga ride details solluren. Chennai Central taxi booking iruki. Temperature 27 degrees. Unga belongings safe ah vachukkonga.')
    for value in ('உங்க ride details சொல்லுறேன்','சென்ட்ரல்','டாக்ஸி','புக்கிங்','இருக்கு','டெம்பரேச்சர்','பிலாங்கிங்ஸ்'):
        check(value in pronunciation,'new pronunciation: '+value)
    state=dict(lang='Tanglish',_wx_mode='voice',count=4,vehicle='SUV',pickup='Chennai Central',destination='Tambaram',
        pickup_at='2030-10-19T19:00:00+05:30',fare=dict(km=35.4,fare=1126))
    rendered=summary_candidate(b,state)
    check(len(re.findall(r'yes\s+illa\s+no\s+sollunga',rendered,re.I))==1,'one final Yes/No instruction')
    check(not re.search(r'[\u0b80-\u0bff]',rendered),'no Tamil speech spelling in the Tanglish display summary')
    check('35.4' in rendered and '1126' in rendered.replace(',',''),'display retains raw numerical entities')
    check('Unga ride details solluren' in rendered or 'ride details' not in rendered.lower(),
          'ride details introduction is preserved in Romanized display')
    for lang in ('English','Tamil'):
        state['lang']=lang
        check(summary_candidate(b,state)==functions['summary_text'](b,state),lang+' summary remains unchanged')
    state['lang']='Tanglish';state['_wx_mode']='manual'
    check(summary_candidate(b,state)==functions['summary_text'](b,state),'manual summary remains unchanged')
    preview='Oru weather update: pickup nerathil cloudy weather irukkum; pickup temperature sumaar 27.3 degrees Celsius irukkum; Chennai Central-la 42%, Tambaram-la 51% rain chance irukku. Umbrella eduthukkondu unga belongings nanaiyaamal paadhugaathukkollunga. Unga ride details sollalaam.'
    chunks=helper.phrases(preview,('Chennai Central','Tambaram'))
    check(len(chunks)>=2,'weather is split into complete clauses')
    check(re.sub(r'\s+',' ',' '.join(chunks))==re.sub(r'\s+',' ',preview),'weather words and entities are preserved')
    check(all(len(chunk.split())>=8 for chunk in chunks),'short isolated speech fragments are avoided')
    place='St. Thomas Mount'
    check(any(place in part for part in helper.phrases('Pickup is at '+place+'. Your road distance is 35.4 kilometers. Your estimated fare is 1126 rupees.',(place,))),'place names and decimal values stay intact')

    current_pause=getattr(b,'_wx_pause_ms',220)
    pause_ms=max(0,min(1000,int(current_pause)))
    if not pause_ms:pause_ms=220
    runtime=dict(version=version,bridge=b,phrases=helper.phrases,register=register,profile_for=profile_for,
        lock=values['lock'],places=values['places'])
    writes={file:(original,updated),speech_file:(speech_file.read_text(encoding='utf-8') if speech_file.exists() else None,helper_source)}
    changes={path:(before,after) for path,(before,after) in writes.items() if before!=after}
    live_changes=(getattr(speech,'_movio_tanglish_quality',None)!=version or words!=values['words']
        or fingerprint(speech.__code__)!=fingerprint(candidate_speech.__code__) or pause_ms!=current_pause
        or any(fingerprint(fn.__code__)!=fingerprint(codes[name]) for name,fn in functions.items()))
    backup=None
    if changes or live_changes:
        backup=root/('tanglish_speech_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,(before,_) in changes.items():
            if before is not None:
                target=backup/path.relative_to(root);target.parent.mkdir(parents=True,exist_ok=True);target.write_text(before,encoding='utf-8')
        (backup/'original_pronunciation_dictionary.json').write_text(json.dumps(values['words'],ensure_ascii=False,indent=2),encoding='utf-8')
        try:(backup/'original_speech_function.py').write_text(textwrap.dedent(inspect.getsource(speech)),encoding='utf-8')
        except (OSError,TypeError):pass
    old_codes={name:(fn.__code__,fn.__doc__) for name,fn in functions.items()}
    old_speech=(speech.__code__,speech.__doc__,getattr(speech,'_movio_tanglish_quality',None))
    old_runtime=friendly.get('_MOVIO_TG_SPEECH_RUNTIME')
    written=[]
    try:
        for path,(_,after) in changes.items():
            temporary=path.with_name(path.name+'.tg-speech-tmp')
            try:temporary.write_text(after,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
            written.append(path)
        with getattr(b,'LOCK',threading.RLock()),values['lock']:
            friendly['_MOVIO_TG_SPEECH_RUNTIME']=runtime
            for name,fn in functions.items():fn.__code__=codes[name];fn.__doc__=codes[name].co_consts[0]
            cellmap['words'].cell_contents=words;cellmap['pattern'].cell_contents=pattern
            speech.__code__=candidate_speech.__code__;speech.__doc__=candidate_speech.__doc__
            speech._movio_tanglish_quality=version
            b._wx_pause_ms=pause_ms
    except Exception:
        for path in reversed(written):
            if writes[path][0] is None:path.unlink(missing_ok=True)
            else:path.write_text(writes[path][0],encoding='utf-8')
        for name,fn in functions.items():fn.__code__,fn.__doc__=old_codes[name]
        speech.__code__,speech.__doc__=old_speech[:2]
        if old_speech[2] is None:speech.__dict__.pop('_movio_tanglish_quality',None)
        else:speech._movio_tanglish_quality=old_speech[2]
        cellmap['words'].cell_contents=values['words'];cellmap['pattern'].cell_contents=values['pattern']
        if old_runtime is None:friendly.pop('_MOVIO_TG_SPEECH_RUNTIME',None)
        else:friendly['_MOVIO_TG_SPEECH_RUNTIME']=old_runtime
        b._wx_pause_ms=current_pause
        raise
    print('Step 3E installed — Tanglish speech spelling and meaningful sentence pauses.')
    print('taxi → டாக்ஸி | booking → புக்கிங் | temperature → டெம்பரேச்சர் | central → சென்ட்ரல்')
    print('iruki → இருக்கு | belongings → பிலாங்கிங்ஸ் | Unga ride details solluren → உங்க ride details சொல்லுறேன்')
    print('Confirmation: Yes illa no sollunga appears once. Browser transcript stays in Romanized Tanglish.')
    print('Complete weather/confirmation phrases use the existing',pause_ms,'ms playback gap; weather clips retain their stable audio profile.')
    print('Syntax/source checks PASS | offline checks:',checks)
    if backup:print('Backup:',backup)
    print('Open a NEW Tanglish AI Call and listen to the greeting, weather advisory and final confirmation.')
    print('Actual voice quality still requires listening; no TTS audio or real booking was generated by this installer.')
    return dict(version=version,offline_checks=checks,pause_ms=pause_ms,changed_files=[str(path.relative_to(root)) for path in changes],
        backup=str(backup) if backup else None,visible_tanglish='Romanized; duplicate instruction removed',tts='existing DhVaani adapter')

MOVIO_TANGLISH_SPEECH_QUALITY=movio_tanglish_pronunciation_and_pauses()


Step 3E installed — Tanglish speech spelling and meaningful sentence pauses.
taxi → டாக்ஸி | booking → புக்கிங் | temperature → டெம்பரேச்சர் | central → சென்ட்ரல்
iruki → இருக்கு | belongings → பிலாங்கிங்ஸ் | Unga ride details solluren → உங்க ride details சொல்லுறேன்
Confirmation: Yes illa no sollunga appears once. Browser transcript stays in Romanized Tanglish.
Complete weather/confirmation phrases use the existing 220 ms playback gap; weather clips retain their stable audio profile.
Syntax/source checks PASS | offline checks: 32
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/tanglish_speech_backup_20261028_070732_332379
Open a NEW Tanglish AI Call and listen to the greeting, weather advisory and final confirmation.
Actual voice quality still requires listening; no TTS audio or real booking was generated by this installer.


In [120]:
# Copy this ENTIRE file into ONE Kaggle cell AFTER your weather/UI fixes,
# BEFORE evaluation. Run it once, then open a NEW AI Call for listening.
def movio_tanglish_pronunciation_and_pauses():
    import ast
    import copy
    import inspect
    import json
    import os
    import re
    import textwrap
    import threading
    import types
    from datetime import datetime
    from pathlib import Path

    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',{})
    friendly=getattr(engine.get('advisory'),'__globals__',{})
    speech=getattr(b,'tanglish_speech_text',None)
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    file=root/'integrations/movio_friendly_weather.py'
    speech_file=root/'integrations/movio_tanglish_speech_quality.py'
    if not getattr(b,'_tg_wordwise_v2',False) or not isinstance(speech,types.FunctionType):
        raise RuntimeError('Run the existing word-by-word Tanglish speech setup first. No changes were made.')
    if not file.is_file() or Path(friendly.get('__file__','')).resolve()!=file:
        raise RuntimeError('The active friendly-weather helper does not match its project file. No changes were made.')
    functions={name:engine.get(name) for name in ('summary_text','speech_segments')}
    if any(not isinstance(fn,types.FunctionType) or fn.__globals__ is not friendly for fn in functions.values()):
        raise RuntimeError('The existing summary/speech handlers have a different owner. No changes were made.')
    cellmap=dict(zip(speech.__code__.co_freevars,speech.__closure__ or ()))
    if set(cellmap)!={'lock','pattern','places','re','words','wx'}:
        raise RuntimeError('The Tanglish speech-copy closure is different. No changes were made.')
    values={name:cell.cell_contents for name,cell in cellmap.items()}
    if not isinstance(values['words'],dict) or not isinstance(values['places'],set):
        raise RuntimeError('The existing Tanglish pronunciation dictionary was not found. No changes were made.')
    synthesize=getattr(getattr(b,'TTS',None),'synthesize',None)
    if not callable(synthesize) or inspect.getclosurevars(synthesize).nonlocals.get('speech_text') is not speech:
        raise RuntimeError('The active TTS adapter does not use this speech-copy function. No changes were made.')
    register=inspect.getclosurevars(engine['advisory']).nonlocals.get('register')
    profile_for=inspect.getclosurevars(engine['tts_reply']).nonlocals.get('profile_for')
    if not callable(register) or not callable(profile_for):
        raise RuntimeError('Run the existing stable weather/closing audio setup first. No changes were made.')
    registry=inspect.getclosurevars(register).nonlocals.get('profiles')
    if registry is None or inspect.getclosurevars(profile_for).nonlocals.get('profiles') is not registry:
        raise RuntimeError('The weather audio registry is inconsistent. No changes were made.')

    helper_source=r'''"""Speech-copy helpers for the existing MOVIO Tanglish adapter."""
import re

VERSION = 'tanglish-speech-quality-1.3'

# English words stay English. The three requested pronunciation exceptions
# retain their meaning while using Tamil script in the speech copy only.
ENGLISH_WORDS = set(("""
ai welcome help service ride book cancel cancellation existing travel pickup
drop destination place correct yes no vehicle road distance around approximate
estimated fare confirm confirmation details detail passengers passenger
kilometres kilometers kilometer kilometres rupees rupee driver mini car bike
auto sedan suv am pm otp id number request record save saved select selected
screen map route verify continue rain chance forecast please check change
retry tomorrow today morning evening night weather update temperature degrees
degree cloudy sunny pleasant heavy strong wind windy thunderstorm thunderstorms
umbrella belongings bags care cover covered consider keep same time open
protection prepare sun heat warnings booking taxi celsius point zero oh one
two three four five six seven eight nine ten eleven twelve thirteen fourteen
fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty
seventy eighty ninety hundred thousand lakh lakhs crore crores million billion
and a an the to from for at in on by of with your you our my me it is are will
be can have thank thanks wonderful journey comfortable keep retain switch
january february march april may june july august september october november
december monday tuesday wednesday thursday friday saturday sunday
""").split())

ALIASES = dict(
    taxi='டெக்ஸி', auto='ஆட்டோ', booking='புக்கிங்', celsius='செல்சியஸ்',
    temperature='டெம்பரேச்சர்', central='சென்ட்ரல்',
    belongings='பிலாங்கிங்ஸ்', belonging='பிலாங்கிங்',
    iruki='இருக்கு', irukki='இருக்கு', iruku='இருக்கு',
    solluren='சொல்லுறேன்', solluraen='சொல்லுறேன்',
    vanakkam='வணக்கம்', nandri='நன்றி', unga='உங்க', ungal='உங்கள்',
    ungalukku='உங்களுக்கு', neenga='நீங்க', eppadi='எப்படி',
    pannalama='பண்ணலாமா', pannalaama='பண்ணலாமா', pannalaam='பண்ணலாம்',
    pannunga='பண்ணுங்க', pannikkonga='பண்ணிக்கோங்க', pannrom='பண்றோம்',
    sollunga='சொல்லுங்க', sollalaam='சொல்லலாம்', illa='இல்ல', illai='இல்லை',
    vera='வேற', veru='வேறு', aiye='ஐயே', aiyum='ஐயும்', um='உம்',
    adhe='அதே', athe='அதே', idhe='இதே', ithae='இதே', indha='இந்த',
    andha='அந்த', naal='நாள்', naalaiku='நாளைக்கு', innikku='இன்னைக்கு',
    innaiku='இன்னைக்கு', neram='நேரம்', nerathil='நேரத்தில்',
    nerathula='நேரத்துல', kelambura='கிளம்புற', kelambumbodhu='கிளம்பும்போது',
    purappadum='புறப்படும்', purappadumbodhu='புறப்படும்போது',
    irukku='இருக்கு', irukkum='இருக்கும்', irukkura='இருக்குற',
    irukkalaam='இருக்கலாம்', irundhu='இருந்து', ilirundhu='இலிருந்து',
    peru='பேர்', ku='க்கு', la='ல', ah='ஆ',
    mazhai='மழை', peiya='பெய்ய', vaaippu='வாய்ப்பு',
    vaanilai='வானிலை', veppanilai='வெப்பநிலை', sumaar='சுமார்',
    adhigamaaga='அதிகமாக', adhigamaa='அதிகமா', konjam='கொஞ்சம்',
    megamoottam='மேகமூட்டம்', kanamazhai='கனமழை',
    kudai='குடை', kudineer='குடிநீர்', eduthuttu='எடுத்துட்டு',
    eduthukkonga='எடுத்துக்கோங்க', eduthukkondu='எடுத்துக்கொண்டு',
    nanaiyaamal='நனையாமல்', nanaiyama='நனையாம',
    paadhugaathukkollunga='பாதுகாத்துக்கொள்ளுங்க',
    paadhugaappa='பாதுகாப்பா', vachukkonga='வச்சுக்கோங்க',
    erumboadhum='ஏறும்போதும்', irangumboadhum='இறங்கும்போதும்',
    eravum='ஏறவும்', irangavum='இறங்கவும்',
    pakkavaatil='பக்கவாட்டில்', padalaam='படலாம்',
    purappadumun='புறப்படும் முன்', mun='முன்', ulloor='உள்ளூர்',
    paarunga='பாருங்க', thevaiyaana='தேவையான', munnerpaadugal='முன்னேற்பாடுகள்',
    payanathirku='பயணத்திற்கு', payanam='பயணம்', inimaiyaaga='இனிமையாக',
    amaiyattum='அமையட்டும்', nandraagave='நன்றாகவே',
)


def make_speech_text(lock, pattern, places, re, words, wx):
    def speech_text(text):
        """MOVIO call weather card v1: Celsius is pronounced செல்சியஸ்."""
        text = str(text)
        with lock:
            names = sorted(places, key=len, reverse=True)
        protected = {}
        for name in names:
            if name and name in text:
                marker = chr(0xF000 + len(protected))
                text = text.replace(name, marker)
                protected[marker] = name
        text = re.sub(r'(?i)\bkilo\s+meettar\b|\bkm\b', 'kilometers', text)
        # Convert clock times to speakable English BEFORE numeric normalization.
        # Avoid opaque placeholders, which may be spoken literally or lost.
        def speak_clock(match):
            hour = int(match.group(1))
            minute = int(match.group(2) or 0)
            period = match.group(3).lower().replace('.', '')
            ones = ('zero','one','two','three','four','five','six','seven','eight','nine',
                    'ten','eleven','twelve','thirteen','fourteen','fifteen','sixteen',
                    'seventeen','eighteen','nineteen')
            tens = ('','','twenty','thirty','forty','fifty')
            def number(n):
                return ones[n] if n < 20 else tens[n // 10] + (' ' + ones[n % 10] if n % 10 else '')
            spoken = number(hour)
            if minute:
                spoken += ' ' + ('oh ' + number(minute) if minute < 10 else number(minute))
            return spoken + (' pee em' if period == 'pm' else ' ay em')
        text = re.sub(
            r'(?i)\b(0?[1-9]|1[0-2])(?::([0-5][0-9]))?\s*([ap]\.?m\.?)(?=\W|$)',
            speak_clock, text)
        text = wx['speech_numeric_normalize'](lambda value: value, text, 'English')
        text = re.sub(r'(?i)\bunga\s+ride\s+details\s+(?:sollalaam|solluren|solluraen|solren|sollren)\b',
                      'உங்க ride details சொல்லுறேன்', text)
        text = pattern.sub(lambda match: words[match[0].lower()], text)
        text = re.sub(r'\b(?:SUV|S\s+U\s+V|es yoo vee)\b', 'ess you vee', text, flags=re.I)
        aliases = {'AI':'ay eye', 'AM':'ay em', 'PM':'pee em', 'OTP':'oh tee pee', 'ID':'eye dee'}
        text = re.sub(r'\b(?:AI|AM|PM|OTP|ID)\b', lambda match: aliases[match[0]], text)
        # Punctuation affects speech only. The displayed response is untouched.
        text = re.sub(r';\s*', '. ', text)
        text = re.sub(r':(?=\s)', ',', text)
        for marker, name in protected.items():
            spoken_name = wx['speech_numeric_normalize'](lambda value: value, name, 'English')
            spoken_name = re.sub(r'(?i)\bcentral\b', 'சென்ட்ரல்', spoken_name)
            spoken_name = re.sub(r'(?i)\btambaram\b', 'தாம்பரம்', spoken_name)
            text = text.replace(marker, spoken_name)
        return re.sub(r'\s+', ' ', text).strip()
    return speech_text


def phrases(text, protected_places=(), minimum_words=8):
    """Complete clauses, retaining decimal values and protected place names."""
    value = str(text).strip()
    if not value:
        return []
    protected = {}
    for name in sorted(set(protected_places), key=len, reverse=True):
        if name and name in value:
            marker = chr(0xF000 + len(protected))
            value = value.replace(name, marker)
            protected[marker] = name
    clauses = re.split(r'(?<=[.!?;])\s+|\n+', value)
    parts, pending = [], ''
    for clause in clauses:
        if not clause.strip():
            continue
        pending = (pending + ' ' + clause.strip()).strip()
        if re.search(r'(?:^|\s)(?:[A-Za-z]|Dr|Mr|Ms|Mrs|St)\.$', pending):
            continue
        if len(pending.split()) >= minimum_words:
            parts.append(pending);pending = ''
    if pending:
        if parts:
            parts[-1] += ' ' + pending
        else:
            parts.append(pending)
    for index, part in enumerate(parts):
        for marker, name in protected.items():
            part = part.replace(marker, name)
        parts[index] = part
    return parts
'''
    replacements={
'summary_text': r'''def summary_text(b,state):
    """MOVIO step-3E: exactly one Tanglish Yes/No instruction."""
    text=previous['summary_text'](b,state)
    if state.get('_wx_mode')=='manual':return text
    if state.get('lang')=='Tanglish':
        text=re.sub(r'(?:\s*Yes\s+illa\s+no\s+sollunga\s*[.!?]*)+\s*$', '', str(text), flags=re.I).rstrip()
        text=re.sub(r'(?i)\bunga\s+ride\s+details\s+(?:sollalaam|solluren|solluraen|solren|sollren)\b',
                    'Unga ride details solluren', text)
        return text+' Yes illa no sollunga.'
    return text+' '+pick(state,'Please say yes or no.','ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.','Yes illa no sollunga.')
''',
'speech_segments': r'''def speech_segments(text):
    """MOVIO step-3E: Tanglish sentence pauses with stable weather audio."""
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    runtime=globals().get('_MOVIO_TG_SPEECH_RUNTIME') or {}
    bridge=runtime.get('bridge')
    if bridge is not None and bridge.CURRENT_LANGUAGE.get()=='Tanglish':
        with runtime['lock']:names=tuple(runtime['places'])
        split=lambda part:runtime['phrases'](part,names)
        if prefix:
            rest=value[len(prefix):].strip()
            review_pair=value.startswith(prefix+'\n\n') or bool(rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold() or 'approximate fare' in rest.casefold()))
            weather_parts=split(prefix)
            if rest and not review_pair:
                if weather_parts:weather_parts[-1]+=' '+rest
                else:weather_parts=[value]
            for part in weather_parts:runtime['register'](part,'weather')
            return weather_parts+(split(rest) if rest and review_pair else [])
        if runtime['profile_for'](value)=='closing':return previous['speech_segments'](text)
        return split(value)
    # Retain the existing English/Tamil weather and final-review behaviour.
    if prefix:
        rest=value[len(prefix):].strip()
        normal_review_pair=value.startswith(prefix+'\n\n')
        return [prefix,rest] if rest and (normal_review_pair or 'estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
'''
}
    allowed={
'summary_text': [r'''def summary_text(b,state):
    text=previous['summary_text'](b,state)
    if state.get('_wx_mode')=='manual':return text
    return text+' '+pick(state,'Please say yes or no.','ஆம் அல்லது இல்லை என்று சொல்லுங்கள்.','Yes illa no sollunga.')
'''],
'speech_segments': [r'''def speech_segments(text):
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        # Keep a coherent weather chunk; pause before a separate final summary.
        return [prefix,rest] if rest and ('estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
''',
r'''def speech_segments(text):
    """MOVIO step-3C-v1: pause between weather and the final ride summary."""
    value=str(text)
    with lock:prefix=next((p for p in reversed(spoken) if value==p or value.startswith(p+'\n') or value.startswith(p+' ')),None)
    if prefix:
        rest=value[len(prefix):].strip()
        normal_review_pair=value.startswith(prefix+'\n\n')
        # Keep interactive weather choices on their existing speech path.
        return [prefix,rest] if rest and (normal_review_pair or 'estimated fare' in rest.casefold() or 'கட்டண' in rest or 'kattanam' in rest.casefold()) else [value]
    return previous['speech_segments'](text)
''']
}
    version='tanglish-speech-quality-1.3'
    helper=types.ModuleType('movio_tanglish_speech_quality');helper.__file__=str(speech_file)
    exec(compile(helper_source,str(speech_file),'exec',dont_inherit=True),helper.__dict__)

    def structural(source):return ast.dump(ast.parse(textwrap.dedent(source)),include_attributes=False)
    def fingerprint(code):
        constants=tuple(fingerprint(value) if isinstance(value,types.CodeType) else value for value in code.co_consts)
        return (code.co_code,constants,code.co_names,code.co_varnames,code.co_freevars,code.co_cellvars,
            code.co_argcount,code.co_kwonlyargcount,code.co_posonlyargcount,code.co_flags,code.co_exceptiontable)
    def code_at(code,names):
        for name in names:
            matches=[value for value in code.co_consts if isinstance(value,types.CodeType) and value.co_name==name]
            if len(matches)!=1:raise RuntimeError('Unexpected existing function structure: '+name+'. No changes were made.')
            code=matches[0]
        return code

    original=file.read_text(encoding='utf-8')
    parsed=ast.parse(original)
    installers=[node for node in parsed.body if isinstance(node,ast.FunctionDef) and node.name=='install_namespace']
    if len(installers)!=1:raise RuntimeError('The existing friendly-weather installer was not found. No changes were made.')
    targets={node.name:node for node in installers[0].body if isinstance(node,ast.FunctionDef) and node.name in replacements}
    if set(targets)!=set(replacements):raise RuntimeError('The original speech handlers were not found. No changes were made.')
    compiled=compile(original,str(file),'exec',dont_inherit=True)
    lines=original.splitlines(keepends=True)
    for name,node in targets.items():
        source=textwrap.dedent(''.join(lines[node.lineno-1:node.end_lineno]))
        recognized = structural(source) in {structural(value) for value in allowed[name]+[replacements[name]]}
        if not recognized and name == 'summary_text':
            # Accept the previously installed step-3E handler, including harmless
            # formatting/docstring differences. Still verify its AST behaviour.
            tree = ast.parse(source)
            node0 = tree.body[0]
            normalized = ast.dump(node0, include_attributes=False)
            recognized = (
                isinstance(node0, ast.FunctionDef)
                and node0.name == 'summary_text'
                and len(node0.args.args) == 2
                and [arg.arg for arg in node0.args.args] == ['b', 'state']
                and "previous['summary_text']" in source
                and "state.get('_wx_mode')" in source
                and "state.get('lang')" in source
                and 'Yes illa no sollunga.' in source
                and 're.sub(' in source
                and 'pick(state,' in source
                and len(node0.body) in (5, 6)
            )
        if not recognized:
            raise RuntimeError('The existing '+name+' source has changed. No changes were made.')
        if fingerprint(functions[name].__code__)!=fingerprint(code_at(compiled,('install_namespace',name))):
            raise RuntimeError('The live '+name+' differs from its saved source. No changes were made.')
    for name,node in sorted(targets.items(),key=lambda item:item[1].lineno,reverse=True):
        lines[node.lineno-1:node.end_lineno]=[textwrap.indent(replacements[name].strip()+'\n','    ')]
    updated=''.join(lines)
    candidate=compile(updated,str(file),'exec',dont_inherit=True)
    codes={name:code_at(candidate,('install_namespace',name)) for name in functions}
    if any(codes[name].co_freevars!=fn.__code__.co_freevars for name,fn in functions.items()):
        raise RuntimeError('An existing dialogue closure cannot be preserved. No changes were made.')

    words=dict(values['words'])
    for word in helper.ENGLISH_WORDS:words.pop(word,None)
    words.update(helper.ALIASES)
    words['auto']='ஆட்டோ'
    pattern=re.compile(r'(?<![A-Za-z])('+'|'.join(re.escape(key) for key in sorted(words,key=len,reverse=True))+r')(?![A-Za-z])',re.I)
    candidate_speech=helper.make_speech_text(values['lock'],pattern,values['places'],values['re'],words,values['wx'])
    if candidate_speech.__code__.co_freevars!=speech.__code__.co_freevars:
        raise RuntimeError('The original speech-copy closure cannot be preserved. No changes were made.')
    summary_candidate=types.FunctionType(codes['summary_text'],friendly,'summary_text',functions['summary_text'].__defaults__,functions['summary_text'].__closure__)

    checks=0
    def check(condition,label):
        nonlocal checks
        if not condition:raise RuntimeError('Pre-install check failed: '+label+'. No changes were made.')
        checks+=1
    spoken=candidate_speech('Vanakkam! AI taxi booking service ku welcome. Ungalukku eppadi help pannalaam?')
    check('டெக்ஸி' in spoken and 'புக்கிங்' in spoken,'requested taxi and booking pronunciation')
    check('வணக்கம்' in spoken and 'உங்களுக்கு' in spoken,'Tamil-origin words use Tamil script')
    check('help' in spoken and 'service' in spoken and 'welcome' in spoken,'English words retain English wording')
    spoken=candidate_speech('Road distance 35.4 kilometers. Approximate fare ₹1,126. Pickup 7:00 PM, temperature 27.3 degrees Celsius, rain chance 70%.')
    for value in ('thirty five point four','one thousand one hundred twenty six','pee em','twenty seven point three','செல்சியஸ்','seventy percent'):
        check(value in spoken,'entity speech: '+value)
    check('four eight two nine one' in candidate_speech('Booking number 48291. OTP 0426.'),'booking ID stays in digit order')
    check('zero four two six' in candidate_speech('Booking number 48291. OTP 0426.'),'OTP leading zero is retained')
    check('ess you vee' in candidate_speech('4 peru, SUV.'),'SUV letter pronunciation retained')
    spoken=candidate_speech('Unga ride details sollalaam; pickup nerathil cloudy weather irukkum; vera time venuma?')
    check('உங்க' in spoken and 'நேரத்தில்' in spoken and 'வேற' in spoken,'weather and confirmation Tamil vocabulary')
    check('cloudy weather' in spoken and ';' not in spoken,'speech-only full stops and English weather terms')
    pronunciation=candidate_speech('Unga ride details solluren. Chennai Central taxi booking iruki. Temperature 27 degrees. Unga belongings safe ah vachukkonga.')
    for value in ('உங்க ride details சொல்லுறேன்','சென்ட்ரல்','டெக்ஸி','புக்கிங்','இருக்கு','டெம்பரேச்சர்','பிலாங்கிங்ஸ்'):
        check(value in pronunciation,'new pronunciation: '+value)
    check('தாம்பரம்' in candidate_speech('Drop at Tambaram.'), 'Tambaram pronunciation')
    check('pee em' in candidate_speech('Pickup at 3 p.m.'), 'PM dotted suffix')
    check('nine pee em' in candidate_speech('Pickup 9:00 PM'), '9:00 PM audible')
    check('nine thirty pee em' in candidate_speech('Pickup 9:30 PM'), '9:30 PM audible')
    check('ஆட்டோ' in candidate_speech('Vehicle auto'), 'auto pronunciation')
    check('fare' in candidate_speech('Estimated fare is 250 rupees.'), 'fare stays English')
    state=dict(lang='Tanglish',_wx_mode='voice',count=4,vehicle='SUV',pickup='Chennai Central',destination='Tambaram',
        pickup_at='2030-10-19T19:00:00+05:30',fare=dict(km=35.4,fare=1126))
    rendered=summary_candidate(b,state)
    check(len(re.findall(r'yes\s+illa\s+no\s+sollunga',rendered,re.I))==1,'one final Yes/No instruction')
    check(not re.search(r'[\u0b80-\u0bff]',rendered),'no Tamil speech spelling in the Tanglish display summary')
    check('35.4' in rendered and '1126' in rendered.replace(',',''),'display retains raw numerical entities')
    check('Unga ride details solluren' in rendered or 'ride details' not in rendered.lower(),
          'ride details introduction is preserved in Romanized display')
    for lang in ('English','Tamil'):
        state['lang']=lang
        check(summary_candidate(b,state)==functions['summary_text'](b,state),lang+' summary remains unchanged')
    state['lang']='Tanglish';state['_wx_mode']='manual'
    check(summary_candidate(b,state)==functions['summary_text'](b,state),'manual summary remains unchanged')
    preview='Oru weather update: pickup nerathil cloudy weather irukkum; pickup temperature sumaar 27.3 degrees Celsius irukkum; Chennai Central-la 42%, Tambaram-la 51% rain chance irukku. Umbrella eduthukkondu unga belongings nanaiyaamal paadhugaathukkollunga. Unga ride details sollalaam.'
    chunks=helper.phrases(preview,('Chennai Central','Tambaram'))
    check(len(chunks)>=2,'weather is split into complete clauses')
    check(re.sub(r'\s+',' ',' '.join(chunks))==re.sub(r'\s+',' ',preview),'weather words and entities are preserved')
    check(all(len(chunk.split())>=8 for chunk in chunks),'short isolated speech fragments are avoided')
    place='St. Thomas Mount'
    check(any(place in part for part in helper.phrases('Pickup is at '+place+'. Your road distance is 35.4 kilometers. Your estimated fare is 1126 rupees.',(place,))),'place names and decimal values stay intact')

    current_pause=getattr(b,'_wx_pause_ms',220)
    pause_ms=max(0,min(1000,int(current_pause)))
    if not pause_ms:pause_ms=220
    runtime=dict(version=version,bridge=b,phrases=helper.phrases,register=register,profile_for=profile_for,
        lock=values['lock'],places=values['places'])
    writes={file:(original,updated),speech_file:(speech_file.read_text(encoding='utf-8') if speech_file.exists() else None,helper_source)}
    changes={path:(before,after) for path,(before,after) in writes.items() if before!=after}
    live_changes=(getattr(speech,'_movio_tanglish_quality',None)!=version or words!=values['words']
        or fingerprint(speech.__code__)!=fingerprint(candidate_speech.__code__) or pause_ms!=current_pause
        or any(fingerprint(fn.__code__)!=fingerprint(codes[name]) for name,fn in functions.items()))
    backup=None
    if changes or live_changes:
        backup=root/('tanglish_speech_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,(before,_) in changes.items():
            if before is not None:
                target=backup/path.relative_to(root);target.parent.mkdir(parents=True,exist_ok=True);target.write_text(before,encoding='utf-8')
        (backup/'original_pronunciation_dictionary.json').write_text(json.dumps(values['words'],ensure_ascii=False,indent=2),encoding='utf-8')
        try:(backup/'original_speech_function.py').write_text(textwrap.dedent(inspect.getsource(speech)),encoding='utf-8')
        except (OSError,TypeError):pass
    old_codes={name:(fn.__code__,fn.__doc__) for name,fn in functions.items()}
    old_speech=(speech.__code__,speech.__doc__,getattr(speech,'_movio_tanglish_quality',None))
    old_runtime=friendly.get('_MOVIO_TG_SPEECH_RUNTIME')
    written=[]
    try:
        for path,(_,after) in changes.items():
            temporary=path.with_name(path.name+'.tg-speech-tmp')
            try:temporary.write_text(after,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
            written.append(path)
        with getattr(b,'LOCK',threading.RLock()),values['lock']:
            friendly['_MOVIO_TG_SPEECH_RUNTIME']=runtime
            for name,fn in functions.items():fn.__code__=codes[name];fn.__doc__=codes[name].co_consts[0]
            cellmap['words'].cell_contents=words;cellmap['pattern'].cell_contents=pattern
            speech.__code__=candidate_speech.__code__;speech.__doc__=candidate_speech.__doc__
            speech._movio_tanglish_quality=version
            b._wx_pause_ms=pause_ms
    except Exception:
        for path in reversed(written):
            if writes[path][0] is None:path.unlink(missing_ok=True)
            else:path.write_text(writes[path][0],encoding='utf-8')
        for name,fn in functions.items():fn.__code__,fn.__doc__=old_codes[name]
        speech.__code__,speech.__doc__=old_speech[:2]
        if old_speech[2] is None:speech.__dict__.pop('_movio_tanglish_quality',None)
        else:speech._movio_tanglish_quality=old_speech[2]
        cellmap['words'].cell_contents=values['words'];cellmap['pattern'].cell_contents=values['pattern']
        if old_runtime is None:friendly.pop('_MOVIO_TG_SPEECH_RUNTIME',None)
        else:friendly['_MOVIO_TG_SPEECH_RUNTIME']=old_runtime
        b._wx_pause_ms=current_pause
        raise
    print('Step 3E installed — Tanglish speech spelling and meaningful sentence pauses.')
    print('taxi → டெக்ஸி | booking → புக்கிங் | temperature → டெம்பரேச்சர் | central → சென்ட்ரல்')
    print('iruki → இருக்கு | belongings → பிலாங்கிங்ஸ் | Unga ride details solluren → உங்க ride details சொல்லுறேன்')
    print('Confirmation: Yes illa no sollunga appears once. Browser transcript stays in Romanized Tanglish.')
    print('Complete weather/confirmation phrases use the existing',pause_ms,'ms playback gap; weather clips retain their stable audio profile.')
    print('Syntax/source checks PASS | offline checks:',checks)
    if backup:print('Backup:',backup)
    print('Open a NEW Tanglish AI Call and listen to the greeting, weather advisory and final confirmation.')
    print('Actual voice quality still requires listening; no TTS audio or real booking was generated by this installer.')
    return dict(version=version,offline_checks=checks,pause_ms=pause_ms,changed_files=[str(path.relative_to(root)) for path in changes],
        backup=str(backup) if backup else None,visible_tanglish='Romanized; duplicate instruction removed',tts='existing DhVaani adapter')

MOVIO_TANGLISH_SPEECH_QUALITY=movio_tanglish_pronunciation_and_pauses()


Step 3E installed — Tanglish speech spelling and meaningful sentence pauses.
taxi → டெக்ஸி | booking → புக்கிங் | temperature → டெம்பரேச்சர் | central → சென்ட்ரல்
iruki → இருக்கு | belongings → பிலாங்கிங்ஸ் | Unga ride details solluren → உங்க ride details சொல்லுறேன்
Confirmation: Yes illa no sollunga appears once. Browser transcript stays in Romanized Tanglish.
Complete weather/confirmation phrases use the existing 220 ms playback gap; weather clips retain their stable audio profile.
Syntax/source checks PASS | offline checks: 38
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/tanglish_speech_backup_20261028_070732_416685
Open a NEW Tanglish AI Call and listen to the greeting, weather advisory and final confirmation.
Actual voice quality still requires listening; no TTS audio or real booking was generated by this installer.


In [121]:

import requests
import inspect
import time
import traceback

print("=== MOVIO TTS DIAGNOSTIC ===")

b = globals().get("bridge")

if b is None:
    print("ERROR: bridge is not available")
else:
    print("Bridge available:", True)
    print("TTS available:", hasattr(b, "TTS"))

    tts = getattr(b, "TTS", None)
    synth = getattr(tts, "synthesize", None)

    if callable(synth):
        print("Synthesize signature:",
              inspect.signature(synth))
    else:
        print("ERROR: synthesize unavailable")

    speech = getattr(b, "tanglish_speech_text", None)

    if callable(speech):
        for sample in [
            "Vanakkam! AI taxi booking service ku welcome.",
            "Unga ride details solluren.",
            "Pickup time 9:00 PM.",
            "Vehicle type auto."
        ]:
            try:
                print("\nINPUT:", sample)
                print("TTS TEXT:", speech(sample))
            except Exception:
                traceback.print_exc()

print("\n=== SERVER CHECK ===")

for port in [8080, 8000, 5000]:
    try:
        start = time.monotonic()
        response = requests.get(
            f"http://127.0.0.1:{port}/health",
            timeout=3
        )
        print(
            "Port:", port,
            "Status:", response.status_code,
            "Seconds:", round(time.monotonic()-start, 2),
            "Response:", response.text[:300]
        )
    except requests.RequestException as e:
        print("Port:", port, "Unavailable:", str(e)[:120])

print("\nDiagnostic finished.")


=== MOVIO TTS DIAGNOSTIC ===
Bridge available: True
TTS available: True
Synthesize signature: (text, group)

INPUT: Vanakkam! AI taxi booking service ku welcome.
TTS TEXT: வணக்கம்! ay eye டெக்ஸி புக்கிங் service க்கு welcome.

INPUT: Unga ride details solluren.
TTS TEXT: உங்க ride details சொல்லுறேன்.

INPUT: Pickup time 9:00 PM.
TTS TEXT: Pickup time nine pee em

INPUT: Vehicle type auto.
TTS TEXT: Vehicle type ஆட்டோ.

=== SERVER CHECK ===
Port: 8080 Status: 404 Seconds: 0.01 Response: <!DOCTYPE HTML>
<html lang="en">
    <head>
        <meta charset="utf-8">
        <title>Error response</title>
    </head>
    <body>
        <h1>Error response</h1>
        <p>Error code: 404</p>
        <p>Message: File not found.</p>
        <p>Error code explanation: 404 - Nothing matches the 
Port: 8000 Status: 200 Seconds: 0.0 Response: {"ok":true,"stt":"Sarvam Saaras v3","tts":"User MOVIO optimized DhVaani (strict)","llm":"Qwen optional","mode":"browser demo"}
Port: 5000 Unavailable: HTTPConnect

In [122]:
# JOURNEY WEATHER — STEP 2: locality names, exact map/GPS coordinates.
# Run this ONE cell in the current Kaggle kernel BEFORE evaluation.
# It edits the original handlers in place, backs up four source files and
# restarts only the website process when required. Forecast retrieval is unchanged.
def movio_fix_journey_locality():
    import ast
    import copy
    import hashlib
    import json
    import logging
    import os
    import re
    import shutil
    import subprocess
    import sys
    import time
    import types
    import weakref
    from datetime import datetime
    from pathlib import Path
    from urllib.parse import parse_qsl,urlencode,urlsplit,urlunsplit
    import psutil
    import requests

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    revision_name='main-locality-1.0'
    reverse_source='def reverse_address(lat,lng):\n    """MOVIO_MAIN_LOCALITY_V1: labels only; never snap the selected coordinates."""\n    import logging\n    lat,lng=coords(lat,lng)\n    try:\n        row=geo_request(\'reverse\',dict(lat=str(lat),lon=str(lng),format=\'json\',zoom=\'18\',addressdetails=\'1\'))\n        if not isinstance(row,dict):raise ValueError(\'Invalid reverse-geocoding response\')\n        address=row.get(\'address\')\n        if not isinstance(address,dict):address={}\n        # Prefer a main settlement/locality over a street, house or POI name.\n        fields=(\'suburb\',\'village\',\'town\',\'quarter\',\'neighbourhood\',\'neighborhood\',\n                \'hamlet\',\'borough\',\'city_district\',\'city\',\'municipality\')\n        locality=next((re.sub(r\'\\s+\',\' \',address[key]).strip() for key in fields\n            if isinstance(address.get(key),str) and address[key].strip()),\'\')\n        if not locality and row.get(\'addresstype\',row.get(\'type\')) in fields:\n            value=row.get(\'name\')\n            if isinstance(value,str):locality=re.sub(r\'\\s+\',\' \',value).strip()\n        if locality:\n            full=str(row.get(\'display_name\') or locality).strip()\n            return dict(name=locality[:170],locality=locality[:170],full_address=full[:1000],\n                        lat=lat,lng=lng,locality_status=\'ok\')\n        logging.getLogger(\'movio.customer_tools\').warning(\'reverse_locality_missing: no structured locality\')\n    except (LookupError,KeyError,ValueError,TypeError) as exc:\n        logging.getLogger(\'movio.customer_tools\').warning(\'reverse_locality_failed: %s\',type(exc).__name__)\n    # An unavailable label must not invalidate a good pin or replace its position.\n    return dict(name=\'Selected location\',locality=\'\',full_address=\'\',lat=lat,lng=lng,\n                locality_status=\'unavailable\')\n'
    replacements=[(' const words=()=>', " // MOVIO_MAIN_LOCALITY_V1: use the existing selection/route/preview state.\n state.pinSeq={pickup:0,destination:0};state.pinPending={pickup:null,destination:null};state.pinControllers={pickup:null,destination:null};\n function invalidatePin(id){\n  state.pinSeq[id]++;state.pinControllers[id]?.abort();state.pinControllers[id]=null;state.pinPending[id]=null;\n }\n function pendingPins(){return Object.values(state.pinPending).filter(Boolean);}\n function pinLabel(id){\n  const lang=el('wxManualLanguage')?.value;\n  if(lang==='Tamil')return id==='pickup'?'தேர்ந்தெடுத்த புறப்படும் இடம்':'தேர்ந்தெடுத்த சேரும் இடம்';\n  return id==='pickup'?'Selected pickup':'Selected drop-off';\n }\n function pinPlace(id,point){\n  if(!['pickup','destination'].includes(id)||locked()||!customer())return Promise.resolve(false);\n  const lat=Number(point.lat),lng=Number(point.lng);\n  if(!Number.isFinite(lat)||Math.abs(lat)>90||!Number.isFinite(lng)||Math.abs(lng)>180)return Promise.resolve(false);\n  invalidatePin(id);state.searchSeq++;state.lastPreview=null;clearTimeout(state.previewTimer);\n  if(typeof searchTimer!=='undefined')clearTimeout(searchTimer);\n  el('suggestions')?.replaceChildren();\n  const ticket=state.pinSeq[id],controller=new AbortController();state.pinControllers[id]=controller;\n  let owner=null;\n  const initial={lat,lng,name:pinLabel(id)};\n  const task=(async()=>{\n   let resolved=initial;\n   try{\n    const data=await request('reverse-address?'+new URLSearchParams({lat,lng}),{signal:controller.signal});\n    const p=data.place;\n    if(p&&p.locality_status!=='unavailable'&&String(p.locality||p.name||'').trim()){\n     // Ignore the provider feature's coordinates. Routing uses the original pin.\n     resolved={lat,lng,name:String(p.locality||p.name).trim(),full_address:String(p.full_address||'')};\n    }else if(!controller.signal.aborted)console.warn('MOVIO: locality label unavailable; selected coordinates kept.');\n   }catch(error){if(!controller.signal.aborted)console.warn('MOVIO: reverse lookup failed; selected coordinates kept.');}\n   if(ticket!==state.pinSeq[id])return false;\n   state.pinPending[id]=null;state.pinControllers[id]=null;\n   if(locked()||!customer())return false;\n   if(selected(id)!==owner||el(id).value.trim()!==owner?.name){updateRoute();return false;}\n   const ok=applyPlace(id,resolved);if(ok)status('');return ok;\n  })();\n  state.pinPending[id]=task;\n  setPlace(id,initial);owner=selected(id);\n  return task;\n }\n const words=()=>"), ("if(!matches('pickup')||!matches('destination')){clearRoute();state.routePromise=Promise.resolve(false);return state.routePromise;}", "if(pendingPins().length||!matches('pickup')||!matches('destination')){clearRoute();state.routePromise=Promise.resolve(false);return state.routePromise;}"), ('if(locked()||!customer()||!validPlace(p))return false;\n  setPlace', 'if(locked()||!customer()||!validPlace(p))return false;\n  invalidatePin(id);\n  setPlace'), ("async function currentPickup(){\n  if(locked()||!customer())return;state.searchSeq++;const seq=state.searchSeq,button=el('useCurrentPickup');if(button)button.disabled=true;status(words().locating);\n  try{const position=typeof locate==='function'?await locate():await new Promise((resolve,reject)=>navigator.geolocation.getCurrentPosition(p=>resolve({lat:p.coords.latitude,lng:p.coords.longitude}),reject,{enableHighAccuracy:true,timeout:12000,maximumAge:3000}));\n   if(seq!==state.searchSeq||locked())return;\n   let p={lat:Number(position.lat),lng:Number(position.lng),name:'Current pickup location ('+Number(position.lat).toFixed(5)+', '+Number(position.lng).toFixed(5)+')'};\n   try{const data=await request('reverse-address?'+new URLSearchParams({lat:p.lat,lng:p.lng}));p=data.place||p;}catch{/* A valid GPS fix does not depend on a readable street label. */}\n   if(seq!==state.searchSeq||locked())return;\n   if(applyPlace('pickup',p)){state.field='destination';status('');}\n  }catch{status(words().gps);}finally{if(button)button.disabled=locked();}\n }\n ", "async function currentPickup(){\n  if(locked()||!customer())return;state.searchSeq++;const seq=state.searchSeq,button=el('useCurrentPickup');if(button)button.disabled=true;status(words().locating);\n  try{const position=typeof locate==='function'?await locate():await new Promise((resolve,reject)=>navigator.geolocation.getCurrentPosition(p=>resolve({lat:p.coords.latitude,lng:p.coords.longitude}),reject,{enableHighAccuracy:true,timeout:12000,maximumAge:3000}));\n   if(seq!==state.searchSeq||locked())return;\n   if(await pinPlace('pickup',position))state.field='destination';\n  }catch{status(words().gps);}finally{if(button)button.disabled=locked();}\n }\n "), ('clearTimeout(state.previewTimer);state.previewBusy=true;\n  try{', 'clearTimeout(state.previewTimer);state.previewBusy=true;\n  try{\n   const resolving=pendingPins();if(resolving.length)await Promise.allSettled(resolving);\n   if(locked()||!customer())return;'), ('state.field=id;state.searchSeq++;state.lastPreview=null;clearTimeout(state.previewTimer);', 'invalidatePin(id);state.field=id;state.searchSeq++;state.lastPreview=null;clearTimeout(state.previewTimer);'), ("version:'customer-booking-tools-1.0',openDetails,closeDetails,searchAddress,currentPickup,preview,state", "version:'customer-booking-tools-1.0',localityVersion:'main-locality-1.0',openDetails,closeDetails,searchAddress,currentPickup,pinPlace,preview,state")]
    node_test='\nconst vm=require(\'node:vm\'),assert=require(\'node:assert/strict\');\nconst sources=JSON.parse(require(\'node:fs\').readFileSync(0,\'utf8\'));\nconst elements=new Map(),calls=[],reverse=[],previews=[];\nfunction element(id){if(!elements.has(id))elements.set(id,{id,value:\'\',disabled:false,hidden:false,title:\'\',textContent:\'\',onclick:null,listeners:{},\n addEventListener(kind,fn){(this.listeners[kind]??=[]).push(fn)},setAttribute(){},replaceChildren(){},append(){},focus(){},\n querySelector(){return null}});return elements.get(id)}\nelement(\'wxManualLanguage\').value=\'English\';element(\'wxManualDate\').value=\'2026-10-20T19:37\';element(\'wxManualCount\').value=\'2\';\nelement(\'bookRide\').onclick=async()=>previews.push(JSON.parse(JSON.stringify({pickup:c.pickup,destination:c.destination})));\nconst c={window:{},document:{getElementById:element},console:{warn(){}},AbortController,URLSearchParams,Promise,Number,JSON,Object,Array,String,\n role:\'customer\',pickup:null,destination:null,selecting:\'pickup\',vehicle:\'Mini\',km:0,minutes:0,routeData:null,routeLine:null,searchTimer:null,\n setTimeout:()=>1,clearTimeout(){},fares(){},map:{panTo(){},removeLayer(){},fitBounds(){}},L:{},\n updateRoute(){},locate:async()=>({lat:12.95555555555,lng:80.24444444444}),\n fetch(url,options){calls.push({url,options});\n  if(url.includes(\'/reverse-address?\'))return new Promise((resolve,reject)=>reverse.push({url,options,resolve:p=>resolve({ok:true,json:async()=>({ok:true,place:p})}),reject}));\n  if(url.includes(\'/route?\'))return Promise.resolve({ok:true,json:async()=>({ok:true,route:{km:12.7,minutes:24,geometry:[]}})});\n  throw Error(\'Unexpected request: \'+url);\n },\n setPlace(id,p){c[id]={lat:p.lat,lng:p.lng,name:p.name};element(id).value=p.name;if(id===\'pickup\')c.selecting=\'destination\';c.updateRoute();}\n};\nvm.createContext(c);\nconst boundary=sources.tools.indexOf(\' // Details use the authenticated single-booking API\');\nassert.ok(boundary>0,\'Existing UI test boundary missing\');\nvm.runInContext(sources.tools.slice(0,boundary)+"window.MOVIOCustomerBookingUI={pinPlace,preview,currentPickup,state,applyPlace};})();",c);\nconst ui=c.window.MOVIOCustomerBookingUI;\nconst point=(name,lat=0,lng=0)=>({name,locality:name,full_address:name+\' street address\',lat,lng,locality_status:\'ok\'});\nconst turn=()=>new Promise(resolve=>setImmediate(resolve));\nlet checks=0;\nasync function main(){\n const a={lat:12.97123456789,lng:80.26123456789},b={lat:12.92123456789,lng:80.21123456789};\n const pa=ui.pinPlace(\'pickup\',a),pb=ui.pinPlace(\'destination\',b);\n assert.equal(c.pickup.lat,a.lat);assert.equal(c.destination.lng,b.lng);checks++;\n assert.equal(calls.filter(x=>x.url.includes(\'/route?\')).length,0,\'Route must await labels\');checks++;\n reverse[1].resolve(point(\'Tambaram\',9,8));await pb;\n assert.equal(c.destination.name,\'Tambaram\');assert.equal(c.destination.lat,b.lat);assert.equal(c.destination.lng,b.lng);checks++;\n assert.equal(calls.filter(x=>x.url.includes(\'/route?\')).length,0);checks++;\n reverse[0].resolve(point(\'Kottivakkam\',7,6));await pa;await turn();\n assert.equal(c.pickup.name,\'Kottivakkam\');assert.equal(c.pickup.lat,a.lat);assert.equal(c.pickup.lng,a.lng);checks++;\n const routes=calls.filter(x=>x.url.includes(\'/route?\'));assert.equal(routes.length,1);\n const query=new URL(routes[0].url,\'http://localhost\').searchParams;\n assert.equal(query.get(\'plat\'),String(a.lat));assert.equal(query.get(\'dlng\'),String(b.lng));checks++;\n const old=ui.pinPlace(\'pickup\',{lat:13.001,lng:80.001}),newer=ui.pinPlace(\'pickup\',{lat:13.002,lng:80.002});\n reverse[3].resolve(point(\'New locality\'));await newer;reverse[2].resolve(point(\'Stale locality\'));await old;await turn();\n assert.equal(c.pickup.name,\'New locality\');assert.equal(c.pickup.lat,13.002);checks++;\n const pending=ui.pinPlace(\'pickup\',{lat:13.003,lng:80.003});\n element(\'pickup\').value=\'Typed correction\';for(const fn of element(\'pickup\').listeners.input)fn();\n reverse[4].resolve(point(\'Late locality\'));await pending;\n assert.equal(element(\'pickup\').value,\'Typed correction\');checks++;\n const failed=ui.pinPlace(\'pickup\',a);reverse[5].reject(Error(\'provider unavailable\'));await failed;await turn();\n assert.equal(c.pickup.name,\'Selected pickup\');assert.equal(c.pickup.lat,a.lat);assert.equal(c.pickup.lng,a.lng);checks++;\n const gps=ui.currentPickup();await turn();reverse[6].resolve(point(\'GPS locality\',1,2));await gps;await turn();\n assert.equal(c.pickup.name,\'GPS locality\');assert.equal(c.pickup.lat,12.95555555555);assert.equal(c.pickup.lng,80.24444444444);checks++;\n const last=ui.pinPlace(\'pickup\',a);const before=previews.length,review=ui.preview();await turn();\n assert.equal(previews.length,before,\'Review must await the selected pin label\');checks++;\n reverse[7].resolve(point(\'Kottivakkam\'));await last;await review;await turn();\n assert.equal(previews.length,before+1);assert.equal(previews.at(-1).pickup.lat,a.lat);assert.equal(previews.at(-1).destination.lng,b.lng);checks++;\n const count=reverse.length;element(\'wxManualDate\').disabled=true;await ui.pinPlace(\'pickup\',{lat:1,lng:2});assert.equal(reverse.length,count);checks++;\n element(\'wxManualDate\').disabled=false;c.role=\'driver\';await ui.pinPlace(\'pickup\',{lat:1,lng:2});assert.equal(reverse.length,count);checks++;\n const handlers=[];c.role=\'customer\';c.map.on=(event,fn)=>handlers.push(fn);c.toast=()=>{};\n vm.runInContext(sources.map,c);assert.equal(handlers.length,1);checks++;\n handlers[0]({latlng:{lat:12.88,lng:80.22}});assert.equal(reverse.length,count+1);checks++;\n reverse.at(-1).resolve(point(\'Pinned locality\'));await turn();\n console.log(JSON.stringify({javascript_checks:checks,passed:true}));\n}\nmain().catch(error=>{console.error(error.stack);process.exitCode=1});\n'
    old_map="map.on('click',e=>{if(role==='driver')return;const which=selecting;setPlace(which,{lat:e.latlng.lat,lng:e.latlng.lng,name:`Map pin (${e.latlng.lat.toFixed(4)}, ${e.latlng.lng.toFixed(4)})`});toast(which==='pickup'?'Pickup pinned. Tap map again for destination.':'Destination pinned.');});"
    new_map="map.on('click',e=>{if(role==='driver')return;const which=selecting;if(window.MOVIOCustomerBookingUI?.pinPlace){void window.MOVIOCustomerBookingUI.pinPlace(which,e.latlng);return;}setPlace(which,{lat:e.latlng.lat,lng:e.latlng.lng,name:`Map pin (${e.latlng.lat.toFixed(4)}, ${e.latlng.lng.toFixed(4)})`});toast(which==='pickup'?'Pickup pinned. Tap map again for destination.':'Destination pinned.');});"
    allowed={'integrations/movio_customer_tools.py': ('81ecea01c51efd6b198bab8adad553472bfb2c11a49982657f80b5bcb934cf41', '408a7d188816efc7ce5656b692387936be5e49571d1a1796573c961b7a4c460e'), 'web/customer_booking_tools.js': ('3f88662283b18e9a5ab589819b905cbeacbcad7f62aff5f691ec3d62345697ac', '60b65a38750871fe708fa96d41169789bc3a7f4fef88cc17a2fcc3649b932a9a')}
    names=('integrations/movio_customer_tools.py','web/customer_booking_tools.js','web/app.js','web/index.html')
    paths={name:root/name for name in names}
    originals={name:paths[name].read_text(encoding='utf-8') for name in names}
    sources=dict(originals)
    sha=lambda value:hashlib.sha256(value.encode()).hexdigest()
    for name,hashes in allowed.items():
        if sha(sources[name]) not in hashes:
            raise RuntimeError('The inspected source changed: '+name+'. No files changed. Share a fresh Step 1 output.')

    backend=names[0];tools=names[1]
    tree=ast.parse(sources[backend]);old_reverse=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name=='reverse_address')
    existing=ast.get_source_segment(sources[backend],old_reverse)
    sources[backend]=sources[backend].replace(existing,reverse_source.rstrip(),1)
    if 'MOVIO_MAIN_LOCALITY_V1' not in sources[tools]:
        for before,after in replacements:
            if sources[tools].count(before)!=1:
                raise RuntimeError('The customer selection handler differs from the inspected version. No files changed.')
            sources[tools]=sources[tools].replace(before,after,1)
    if sources['web/app.js'].count(old_map)==1:
        sources['web/app.js']=sources['web/app.js'].replace(old_map,new_map,1)
    elif sources['web/app.js'].count(new_map)!=1:
        raise RuntimeError('The original map-click handler differs. No files changed.')
    if sources['web/app.js'].count("map.on('click'")!=originals['web/app.js'].count("map.on('click'"):
        raise AssertionError('Map handlers must not be duplicated.')

    revision=sha(sources[backend]+sources[tools]+sources['web/app.js'])[:12]
    counts={name:0 for name in ('app.js','customer_booking_tools.js')}
    def cache_bust(match):
        value=match.group(2);parsed=urlsplit(value);name=Path(parsed.path).name
        if name not in counts:return match.group(0)
        counts[name]+=1
        query=[(k,v) for k,v in parse_qsl(parsed.query,keep_blank_values=True) if k!='journey_locality']
        query.append(('journey_locality',revision))
        src=urlunsplit((parsed.scheme,parsed.netloc,parsed.path,urlencode(query),parsed.fragment))
        return match.group(1)+src+match.group(3)
    sources['web/index.html']=re.sub(r'(<script\b[^>]*\bsrc=["\x27])([^"\x27]+)(["\x27])',cache_bust,sources['web/index.html'],flags=re.I)
    if any(count!=1 for count in counts.values()):raise RuntimeError('Expected one existing app/customer-tools script. No files changed.')

    # Verify that reverse_address is the only changed Python definition.
    def other_python(source):
        tree=ast.parse(source);tree.body=[n for n in tree.body if not isinstance(n,ast.FunctionDef) or n.name!='reverse_address']
        return ast.dump(tree,include_attributes=False)
    if other_python(sources[backend])!=other_python(originals[backend]):raise AssertionError('Unrelated backend change detected.')
    namespace=dict(__name__='isolated_locality_fixture',__file__=str(paths[backend]))
    exec(compile(sources[backend],str(paths[backend]),'exec'),namespace)
    code=namespace['reverse_address'].__code__
    checks=0
    pin=(12.97123456789,80.26123456789)
    fixtures=[({'road':'Kottivakkam Kuppam Road','suburb':'Kottivakkam','city':'Chennai'},'Kottivakkam'),
        ({'road':'Station Road','town':'Tambaram'},'Tambaram'),
        ({'road':'GST Road','village':'Vandalur'},'Vandalur'),
        ({'road':'Market Road','neighbourhood':'Anna Nagar'},'Anna Nagar'),
        ({'road':'Test Road','suburb':'  Kottivakkam  ','neighbourhood':'Kuppam'},'Kottivakkam'),
        ({'city':'Chennai'},'Chennai')]
    logger=logging.getLogger('movio.customer_tools');was_disabled=logger.disabled
    try:
        logger.disabled=True
        for address,label in fixtures:
            row=dict(lat='13.111',lon='80.111',display_name='Street, '+label+', Chennai',address=address)
            before=copy.deepcopy(row);queries=[]
            def fake_geo(kind,params):queries.append((kind,dict(params)));return row
            namespace['geo_request']=fake_geo
            result=namespace['reverse_address'](*pin)
            assert result['name']==label and (result['lat'],result['lng'])==pin
            assert row==before and queries[0][1]['lat']==str(pin[0]) and queries[0][1]['lon']==str(pin[1])
            checks+=1
        namespace['geo_request']=lambda *_:dict(address={'road':'Street only'},lat=0,lon=0,display_name='Street only')
        result=namespace['reverse_address'](*pin)
        assert result['locality_status']=='unavailable' and (result['lat'],result['lng'])==pin;checks+=1
        def fail(*_):raise namespace['LookupError']('fixture provider failure')
        namespace['geo_request']=fail;result=namespace['reverse_address'](*pin)
        assert result['name']=='Selected location' and (result['lat'],result['lng'])==pin;checks+=1
        try:namespace['reverse_address'](91,80)
        except ValueError:checks+=1
        else:raise AssertionError('Invalid coordinates accepted')
    finally:logger.disabled=was_disabled
    node=shutil.which('node')
    if not node:raise RuntimeError('Node.js is required for this cell’s JavaScript checks. No files changed.')
    for name in ('web/app.js','web/customer_booking_tools.js'):
        result=subprocess.run([node,'--check'],input=sources[name],capture_output=True,text=True,timeout=15)
        if result.returncode:raise RuntimeError('JavaScript syntax check failed for '+name+': '+result.stderr[:400])
    result=subprocess.run([node,'-e',node_test],input=json.dumps(dict(tools=sources[tools],map=new_map)),capture_output=True,text=True,timeout=20)
    if result.returncode:raise RuntimeError('Selection regression check failed; no files changed.\n'+result.stderr[-1500:])
    js_checks=json.loads(result.stdout.strip().splitlines()[-1])
    print('Offline locality/coordinate checks:',checks,'| JavaScript selection checks:',js_checks['javascript_checks'],'| PASS')

    protected=('server.py','integrations/movio_required_weather.py','integrations/movio_booking_reliability.py',
        'integrations/ai_call_page.html','integrations/movio_original_tts_adapter.py')
    protected_before={name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in protected if (root/name).is_file()}
    server=(root/'server.py').resolve()
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet') if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    if len(listeners)!=1:raise RuntimeError('Expected one existing website on port 8080. No files changed.')
    pid=listeners[0];runtime_namespaces=[];external=None
    if pid==os.getpid():
        seen=set()
        def visit(fn):
            if not isinstance(fn,types.FunctionType) or id(fn) in seen:return
            seen.add(id(fn));values=fn.__globals__;filename=values.get('__file__')
            if filename and Path(filename).resolve()==paths[backend].resolve() and callable(values.get('reverse_address')):
                if not any(values is old for old in runtime_namespaces):runtime_namespaces.append(values)
            for cell in fn.__closure__ or ():
                try:visit(cell.cell_contents)
                except ValueError:pass
            visit(getattr(fn,'__wrapped__',None))
        for module in list(sys.modules.values()):
            if type(module) is not types.ModuleType:continue
            values=vars(module);known=values.get('ROOT');handler=values.get('Handler')
            if known is None or not isinstance(handler,type):continue
            try:
                if Path(known).resolve()==root:visit(getattr(handler,'do_GET',None))
            except (OSError,ValueError,TypeError):continue
        if not runtime_namespaces:raise RuntimeError('Could not identify the existing in-kernel customer handler. No files changed.')
    else:
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server for path in scripts):
            raise RuntimeError('A different service owns port 8080. No files changed or process stopped.')
        controller=process
        for value in list(globals().values()):
            if type(value) in weakref.ProxyTypes:continue
            try:
                if isinstance(value,subprocess.Popen) and value.poll() is None and isinstance(value.args,(list,tuple)) and list(value.args)==command:
                    controller=value;break
            except ReferenceError:continue
        external=(controller,command,cwd,process.environ())

    changes={name:value for name,value in sources.items() if value!=originals[name]}
    backup=None;written=[];active_old=[];launched=None;stopped=False
    session=requests.Session();session.trust_env=False
    def healthy():
        try:
            response=session.get('http://127.0.0.1:8080/api/customer-tools/health',timeout=(1,2))
            return response.status_code==200 and response.json().get('version')=='customer-booking-tools-1.0'
        except (requests.RequestException,ValueError):return False
    def wait_health():
        for attempt in range(12):
            if healthy():return True
            time.sleep(.25)
        return False
    def start_website(details):
        _,command,cwd,environment=details
        with (root/'journey_locality_website.log').open('a',encoding='utf-8') as output:
            proc=subprocess.Popen(command,cwd=cwd,env=environment,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
        globals()['WX_WEBSITE_PROCESS']=proc
        return proc
    try:
        if not healthy():raise RuntimeError('The original customer-tools health check failed. No files changed.')
        if changes:
            if any(paths[name].read_text(encoding='utf-8')!=originals[name] for name in names):
                raise RuntimeError('A source file changed during validation. No files changed.')
            backup=root/('journey_locality_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
            for name in changes:
                target=backup/name;target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(paths[name],target)
            for name,value in changes.items():
                temporary=paths[name].with_name(paths[name].name+'.locality.tmp')
                try:temporary.write_text(value,encoding='utf-8');os.replace(temporary,paths[name])
                finally:temporary.unlink(missing_ok=True)
                written.append(name)
        for values in runtime_namespaces:
            old=values['reverse_address'];active_old.append((values,old))
            values['reverse_address']=types.FunctionType(code,values,'reverse_address')
            assert 'MOVIO_MAIN_LOCALITY_V1' in (values['reverse_address'].__doc__ or '')
        if external and (backend in changes):
            controller=external[0];controller.terminate();stopped=True;controller.wait(timeout=8)
            launched=start_website(external)
        if not wait_health():raise RuntimeError('The updated website did not pass its health check.')
        for name in ('web/app.js','web/customer_booking_tools.js'):
            response=session.get('http://127.0.0.1:8080/'+Path(name).name+'?journey_locality='+revision,timeout=(1,3))
            if response.status_code!=200 or hashlib.sha256(response.content).hexdigest()!=sha(sources[name]):
                raise RuntimeError('The website is not serving the validated '+Path(name).name)
        denied=session.get('http://127.0.0.1:8080/api/customer-tools/reverse-address?lat=13&lng=80',timeout=(1,2))
        if denied.status_code not in (401,403):raise RuntimeError('Customer authentication check failed.')
        assert all(hashlib.sha256((root/name).read_bytes()).hexdigest()==value for name,value in protected_before.items())
    except Exception as error:
        for name in reversed(written):shutil.copy2(backup/name,paths[name])
        for values,old in active_old:values['reverse_address']=old
        recovery='not needed'
        if stopped:
            try:
                if launched is not None and launched.poll() is None:launched.terminate();launched.wait(timeout=8)
                start_website(external);recovery='PASS' if wait_health() else 'FAILED'
            except Exception as exc:recovery='FAILED: '+type(exc).__name__
        raise RuntimeError(str(error)+' | Source rollback: '+('done' if written else 'not needed')+' | Website recovery: '+recovery+' | Backup: '+str(backup)) from error
    finally:session.close()
    result=dict(step=2,version=revision_name,passed=True,python_checks=checks,javascript_checks=js_checks['javascript_checks'],
        website_health='PASS',served_javascript='PASS',customer_authentication='PASS',
        forecast_and_voice_sources_unchanged=True,changed_files=list(changes),backup=str(backup) if backup else None,
        browser_map_verification='REQUIRED: refresh dashboard, select two new map pins, check both locality names')
    print(json.dumps(result,ensure_ascii=False,indent=2))
    print('Reload the CUSTOMER DASHBOARD with Ctrl+Shift+R. On mobile, reload the page.')
    print('Choose two NEW map pins. Journey Weather should show locality names; routing keeps the exact pins.')
    print('Share this output and the two displayed names. Independent forecast fetching is the next step.')
    return result

JW_LOCALITY_FIX=movio_fix_journey_locality()


Offline locality/coordinate checks: 9 | JavaScript selection checks: 16 | PASS
{
  "step": 2,
  "version": "main-locality-1.0",
  "passed": true,
  "python_checks": 9,
  "javascript_checks": 16,
  "website_health": "PASS",
  "served_javascript": "PASS",
  "customer_authentication": "PASS",
  "forecast_and_voice_sources_unchanged": true,
  "changed_files": [
    "integrations/movio_customer_tools.py",
    "web/customer_booking_tools.js",
    "web/app.js",
    "web/index.html"
  ],
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/journey_locality_backup_20261028_070732_638906",
  "browser_map_verification": "REQUIRED: refresh dashboard, select two new map pins, check both locality names"
}
Reload the CUSTOMER DASHBOARD with Ctrl+Shift+R. On mobile, reload the page.
Choose two NEW map pins. Journey Weather should show locality names; routing keeps the exact pins.
Share this output and the two displayed names. Independent forecast fetching is the next step.


In [123]:
# JOURNEY WEATHER — STEP 3: independent pickup/drop forecasts.
# Paste this ENTIRE file into ONE Kaggle cell after Step 2, before evaluation.
# Only Forecast.fetch / Forecast.forecast are edited in the existing adapter.
# No new handlers, booking flows or TTS wrappers are installed.
def movio_fix_independent_journey_forecasts():
    import ast
    import copy
    import hashlib
    import json
    import logging
    import os
    import shutil
    import subprocess
    import sys
    import threading
    import time
    import types
    import weakref
    from datetime import datetime,timedelta
    from pathlib import Path
    from zoneinfo import ZoneInfo
    import psutil
    import requests

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    path=root/'integrations/movio_required_weather.py';server=(root/'server.py').resolve()
    original=path.read_text(encoding='utf-8');digest=lambda value:hashlib.sha256(value.encode()).hexdigest()
    if digest(original) not in ('965fd5e8114748318c8d4315919c014c414f9d77ac63886b4384a3573a6688e5', '0506b8f43f455d57def44309aa6c6e289bc594eccff712a3f9b3759c1883071a'):
        raise RuntimeError('The forecast source differs from the inspected notebook. No files changed. Share a fresh Step 1 output.')
    for name in ('integrations/movio_customer_tools.py','web/customer_booking_tools.js'):
        if 'MOVIO_MAIN_LOCALITY_V1' not in (root/name).read_text(encoding='utf-8'):
            raise RuntimeError('Run Step 2 locality fix first. No files changed.')
    method_sources={'fetch': '    def fetch(self,points,pickup):\n        """MOVIO_INDEPENDENT_FORECAST_V1: one HTTP response per requested point."""\n        getter=getattr(self.base,\'getter\',None)\n        if not callable(getter):\n            import requests\n            getter=requests.get\n        target=pickup.replace(minute=0,second=0,microsecond=0).strftime(\'%Y-%m-%dT%H:%M\')\n        # Snapshot coordinates before issuing requests; never modify caller data.\n        requested=[dict(lat=float(p[\'lat\']),lng=float(p[\'lng\'])) for p in points]\n        result=[]\n        for requested_point in requested:\n            params=dict(latitude=str(requested_point[\'lat\']),longitude=str(requested_point[\'lng\']),\n                hourly=\'weather_code,precipitation_probability,precipitation,temperature_2m,apparent_temperature\',\n                timezone=\'Asia/Kolkata\',timeformat=\'iso8601\',temperature_unit=\'celsius\',\n                start_date=pickup.date().isoformat(),end_date=pickup.date().isoformat())\n            response=None\n            try:\n                self.requests+=1\n                response=getter(\'https://api.open-meteo.com/v1/forecast\',params=params,timeout=(3,6))\n                response.raise_for_status();payload=response.json()\n                if isinstance(payload,dict) and payload.get(\'error\'):\n                    raise ProviderFailure(\'provider_rejected_query\',getattr(response,\'status_code\',None))\n                # A single-coordinate call must not borrow another response row.\n                if isinstance(payload,list):\n                    if len(payload)!=1:raise ProviderFailure(\'invalid_location_count\')\n                    payload=payload[0]\n                if not isinstance(payload,dict):raise ProviderFailure(\'invalid_location_response\')\n                if payload.get(\'utc_offset_seconds\') is not None and number(payload[\'utc_offset_seconds\'])!=19800:\n                    raise ProviderFailure(\'unexpected_forecast_timezone\')\n                units=payload.get(\'hourly_units\') or {}\n                if isinstance(units,dict) and units.get(\'temperature_2m\') not in (None,\'°C\',\'celsius\'):\n                    raise ProviderFailure(\'unexpected_temperature_unit\')\n                hourly=payload[\'hourly\'];times=hourly[\'time\']\n                if not isinstance(times,list) or times.count(target)!=1:\n                    raise ProviderFailure(\'requested_hour_missing_or_duplicated\')\n                index=times.index(target)\n                code=number(hourly[\'weather_code\'][index],0,99)\n                chance=number(hourly[\'precipitation_probability\'][index],0,100)\n                if code is None or not code.is_integer() or chance is None:\n                    raise ProviderFailure(\'missing_hourly_condition_or_probability\')\n                point=dict(valid_at=target,weather_code=int(code),precipitation_probability=chance,\n                    requested_latitude=requested_point[\'lat\'],requested_longitude=requested_point[\'lng\'],\n                    provider_latitude=number(payload.get(\'latitude\'),-90,90),\n                    provider_longitude=number(payload.get(\'longitude\'),-180,180),forecast_timezone=\'Asia/Kolkata\')\n                for key in (\'precipitation\',\'temperature_2m\',\'apparent_temperature\'):\n                    values=hourly.get(key);point[key]=number(values[index]) if isinstance(values,list) and index<len(values) else None\n                result.append(point)\n            except ProviderFailure:raise\n            except Exception as exc:\n                status=getattr(getattr(exc,\'response\',None),\'status_code\',None)\n                if status is None:status=getattr(response,\'status_code\',None)\n                reason=\'http_error\' if isinstance(status,int) and status>=400 else \'timeout\' if \'timeout\' in type(exc).__name__.lower() else \'invalid_hourly_response\' if isinstance(exc,(ValueError,KeyError,IndexError,TypeError)) else \'connection_error\'\n                log(self.logger,\'weather_request_failed\',reason=reason,http_status=status,error_type=type(exc).__name__,locations=1)\n                if status==429:\n                    try:retry=float(response.headers.get(\'Retry-After\',\'30\'))\n                    except (ValueError,TypeError,AttributeError):retry=30\n                    self.blocked_until=time.monotonic()+max(1,min(3600,retry))\n                raise ProviderFailure(reason,status) from exc\n        return result\n', 'forecast': '    def forecast(self,points,pickup,now=None):\n        """MOVIO_INDEPENDENT_FORECAST_V1: exact-coordinate, hour-specific cache."""\n        zone=self.engine[\'TZ\'];now=now or datetime.now(zone)\n        pickup=datetime.fromisoformat(pickup) if isinstance(pickup,str) else pickup\n        pickup=pickup.replace(tzinfo=zone) if pickup.tzinfo is None else pickup.astimezone(zone)\n        now=now.replace(tzinfo=zone) if now.tzinfo is None else now.astimezone(zone)\n        if not 0<=(pickup.date()-now.date()).days<16:return self.pending(\'outside_forecast_window\')\n        if len(points)!=2 or not all(self.engine[\'valid_geo\'](point) for point in points):return self.pending(\'invalid_location_coordinates\')\n        target=pickup.replace(minute=0,second=0,microsecond=0).isoformat()\n        points=[dict(lat=float(p[\'lat\']),lng=float(p[\'lng\'])) for p in points]\n        key=(\'MOVIO_INDEPENDENT_FORECAST_V1\',tuple((p[\'lat\'],p[\'lng\']) for p in points),target)\n        with self.lock:\n            cached=self.cache.get(key);ttl=max(1,min(900,float(getattr(self.base,\'ttl\',900))))\n            if cached and time.monotonic()-cached[0]<ttl:return copy.deepcopy(cached[1])\n            if time.monotonic()<self.blocked_until:return self.pending(\'rate_limit_backoff\',429)\n            try:rows=self.fetch(points,pickup)\n            except ProviderFailure as exc:return self.pending(exc.reason,exc.status)\n            rows=[copy.deepcopy(row) for row in rows]\n            for role,row in zip(ROLES,rows):row[\'location\']=role\n            rain=any(row[\'weather_code\'] in self.engine[\'RAIN_CODES\'] or row[\'precipitation_probability\']>=getattr(self.base,\'rain_threshold\',60) for row in rows)\n            cloudy=any(row[\'weather_code\'] in (2,3) for row in rows);clear=all(row[\'weather_code\'] in (0,1) for row in rows)\n            heat=any(row[\'temperature_2m\'] is not None and row[\'temperature_2m\']>=getattr(self.base,\'heat_c\',38)\n                or row[\'apparent_temperature\'] is not None and row[\'apparent_temperature\']>=getattr(self.base,\'feels_like_c\',40) for row in rows)\n            result=dict(status=\'ok\',source=\'Open-Meteo\',points=rows,fetched_at=time.time(),policy=VERSION,\n                condition=\'rain\' if rain else \'cloudy\' if cloudy else \'clear\' if clear else \'other\',\n                heavy_rain=any(row[\'weather_code\'] in (65,67,82) for row in rows),\n                thunderstorm=any(row[\'weather_code\'] in (95,96,97,99) for row in rows),heat=heat,\n                pleasant=clear and not heat and all(row[\'temperature_2m\'] is not None and 18<=row[\'temperature_2m\']<=32 for row in rows),\n                scope=\'pickup_and_destination_at_pickup_hour\',lookup_mode=\'independent_per_location\')\n            self.cache[key]=(time.monotonic(),copy.deepcopy(result));self.last_error=None\n            if len(self.cache)>256:self.cache.pop(next(iter(self.cache)))\n            log(self.logger,\'weather_forecast_ready\',valid_at=target,probabilities=[row[\'precipitation_probability\'] for row in rows])\n            return result\n'}
    old_methods={'fetch': "    def fetch(self,points,pickup):\n        params=dict(latitude=','.join(str(float(p['lat'])) for p in points),\n            longitude=','.join(str(float(p['lng'])) for p in points),\n            hourly='weather_code,precipitation_probability,precipitation,temperature_2m,apparent_temperature',\n            timezone='Asia/Kolkata',start_date=pickup.date().isoformat(),end_date=pickup.date().isoformat())\n        getter=getattr(self.base,'getter',None)\n        if not callable(getter):\n            import requests\n            getter=requests.get\n        response=None\n        try:\n            self.requests+=1\n            response=getter('https://api.open-meteo.com/v1/forecast',params=params,timeout=(3,6))\n            response.raise_for_status();payload=response.json()\n            if isinstance(payload,dict) and payload.get('error'):\n                raise ProviderFailure('provider_rejected_query',getattr(response,'status_code',None))\n            rows=payload if isinstance(payload,list) else [payload]\n            if len(rows)!=len(points):raise ProviderFailure('invalid_location_count')\n            target=pickup.replace(minute=0,second=0,microsecond=0).strftime('%Y-%m-%dT%H:%M')\n            result=[]\n            for row in rows:\n                hourly=row['hourly'];index=hourly['time'].index(target)\n                code=number(hourly['weather_code'][index],0,99)\n                chance=number(hourly['precipitation_probability'][index],0,100)\n                if code is None or not code.is_integer() or chance is None:\n                    raise ProviderFailure('missing_hourly_condition_or_probability')\n                point=dict(valid_at=target,weather_code=int(code),precipitation_probability=chance)\n                for key in ('precipitation','temperature_2m','apparent_temperature'):\n                    values=hourly.get(key);point[key]=number(values[index]) if isinstance(values,list) and index<len(values) else None\n                result.append(point)\n            return result\n        except ProviderFailure:raise\n        except Exception as exc:\n            status=getattr(getattr(exc,'response',None),'status_code',None)\n            if status is None:status=getattr(response,'status_code',None)\n            reason='http_error' if isinstance(status,int) and status>=400 else 'timeout' if 'timeout' in type(exc).__name__.lower() else 'invalid_hourly_response' if isinstance(exc,(ValueError,KeyError,IndexError,TypeError)) else 'connection_error'\n            log(self.logger,'weather_request_failed',reason=reason,http_status=status,error_type=type(exc).__name__,locations=len(points))\n            if status==429:\n                try:retry=float(response.headers.get('Retry-After','30'))\n                except (ValueError,TypeError,AttributeError):retry=30\n                self.blocked_until=time.monotonic()+max(1,min(3600,retry))\n            raise ProviderFailure(reason,status) from exc\n", 'forecast': "    def forecast(self,points,pickup,now=None):\n        zone=self.engine['TZ'];now=now or datetime.now(zone)\n        pickup=datetime.fromisoformat(pickup) if isinstance(pickup,str) else pickup\n        pickup=pickup.replace(tzinfo=zone) if pickup.tzinfo is None else pickup.astimezone(zone)\n        now=now.replace(tzinfo=zone) if now.tzinfo is None else now.astimezone(zone)\n        if not 0<=(pickup.date()-now.date()).days<16:return self.pending('outside_forecast_window')\n        if len(points)!=2 or not all(self.engine['valid_geo'](point) for point in points):return self.pending('invalid_location_coordinates')\n        target=pickup.replace(minute=0,second=0,microsecond=0).isoformat()\n        key=(tuple((round(float(p['lat']),5),round(float(p['lng']),5)) for p in points),target)\n        with self.lock:\n            cached=self.cache.get(key);ttl=max(1,min(900,float(getattr(self.base,'ttl',900))))\n            if cached and time.monotonic()-cached[0]<ttl:return copy.deepcopy(cached[1])\n            if time.monotonic()<self.blocked_until:return self.pending('rate_limit_backoff',429)\n            try:rows=self.fetch(points,pickup)\n            except ProviderFailure as exc:\n                # Explicit denial/rate limits are not bypassed with more calls.\n                if exc.status in (401,403,429):return self.pending(exc.reason,exc.status)\n                log(self.logger,'retry_weather_as_individual_locations',reason=exc.reason,http_status=exc.status)\n                try:rows=[self.fetch([point],pickup)[0] for point in points]\n                except ProviderFailure as last:return self.pending(last.reason,last.status)\n            for role,row in zip(ROLES,rows):row['location']=role\n            rain=any(row['weather_code'] in self.engine['RAIN_CODES'] or row['precipitation_probability']>=getattr(self.base,'rain_threshold',60) for row in rows)\n            cloudy=any(row['weather_code'] in (2,3) for row in rows);clear=all(row['weather_code'] in (0,1) for row in rows)\n            heat=any(row['temperature_2m'] is not None and row['temperature_2m']>=getattr(self.base,'heat_c',38)\n                or row['apparent_temperature'] is not None and row['apparent_temperature']>=getattr(self.base,'feels_like_c',40) for row in rows)\n            result=dict(status='ok',source='Open-Meteo',points=rows,fetched_at=time.time(),policy=VERSION,\n                condition='rain' if rain else 'cloudy' if cloudy else 'clear' if clear else 'other',\n                heavy_rain=any(row['weather_code'] in (65,67,82) for row in rows),\n                thunderstorm=any(row['weather_code'] in (95,96,97,99) for row in rows),heat=heat,\n                pleasant=clear and not heat and all(row['temperature_2m'] is not None and 18<=row['temperature_2m']<=32 for row in rows),\n                scope='pickup_and_destination_at_pickup_hour')\n            self.cache[key]=(time.monotonic(),copy.deepcopy(result));self.last_error=None\n            if len(self.cache)>256:self.cache.pop(next(iter(self.cache)))\n            log(self.logger,'weather_forecast_ready',valid_at=target,probabilities=[row['precipitation_probability'] for row in rows])\n            return result\n"}
    tree=ast.parse(original);cls=next(node for node in tree.body if isinstance(node,ast.ClassDef) and node.name=='Forecast')
    lines=original.splitlines(keepends=True);edits=[];legacy_edits=[]
    for node in cls.body:
        if isinstance(node,ast.FunctionDef) and node.name in method_sources:
            edits.append((sum(map(len,lines[:node.lineno-1])),sum(map(len,lines[:node.end_lineno])),method_sources[node.name]))
            legacy_edits.append((sum(map(len,lines[:node.lineno-1])),sum(map(len,lines[:node.end_lineno])),old_methods[node.name]))
    if len(edits)!=2:raise RuntimeError('Expected one fetch and one forecast method. No files changed.')
    source=original
    for start,end,value in sorted(edits,reverse=True):source=source[:start]+value+source[end:]
    def unrelated(source):
        tree=ast.parse(source)
        for node in tree.body:
            if isinstance(node,ast.ClassDef) and node.name=='Forecast':
                node.body=[item for item in node.body if not isinstance(item,ast.FunctionDef) or item.name not in method_sources]
        return ast.dump(tree,include_attributes=False)
    if unrelated(original)!=unrelated(source):raise AssertionError('An unrelated definition changed.')
    fresh=types.ModuleType('isolated_independent_forecast');fresh.__file__=str(path)
    exec(compile(source,str(path),'exec',dont_inherit=True),fresh.__dict__)
    tests={};exec("def forecast_checks(provider):\n    import copy\n    import logging\n    import threading\n    import time\n    from datetime import datetime,timedelta,timezone\n    from types import SimpleNamespace\n    from zoneinfo import ZoneInfo\n    zone=ZoneInfo('Asia/Kolkata');now=datetime(2026,10,18,10,tzinfo=zone)\n    when=datetime(2026,10,19,19,37,tzinfo=zone)\n    points=[dict(lat=13.00123456789,lng=80.00123456789),dict(lat=12.90123456789,lng=80.20123456789)]\n    class Reply:\n        def __init__(self,data,status=200):self.data=data;self.status_code=status;self.headers={'Retry-After':'45'}\n        def raise_for_status(self):\n            if self.status_code>=400:\n                error=RuntimeError('fixture HTTP error');error.response=self;raise error\n        def json(self):return self.data\n    def make(same=False,bad=None,status=None,reused=False,ttl=900):\n        calls=[];shared={}\n        def getter(url,params,timeout):\n            calls.append(dict(params))\n            assert ',' not in params['latitude'] and ',' not in params['longitude']\n            assert params['timezone']=='Asia/Kolkata' and params['temperature_unit']=='celsius'\n            lat,lng=float(params['latitude']),float(params['longitude'])\n            first=same or lat>13\n            if status is not None and (status[0]=='first' or len(calls)==2):return Reply({},status[1])\n            if bad=='timeout':raise TimeoutError('fixture')\n            base=10 if first else 60\n            # Reverse hourly order so indexing cannot accidentally use the clock hour.\n            hours=list(reversed(range(24)))\n            payload=dict(latitude=13.0,longitude=80.0,utc_offset_seconds=19800,hourly_units={'temperature_2m':'°C'},\n                hourly=dict(time=[params['start_date']+f'T{h:02}:00' for h in hours],\n                    weather_code=[3 if first else 61]*24,precipitation_probability=[base+h for h in hours],\n                    precipitation=[0 if first else 2]*24,temperature_2m=[20 if first else 30]*24,apparent_temperature=[21 if first else 31]*24))\n            if bad=='missing_hour':payload['hourly']['time']=['2030-01-01T00:00']*24\n            if bad=='duplicated_hour':payload['hourly']['time']=[params['start_date']+'T19:00']*24\n            if bad=='timezone':payload['utc_offset_seconds']=0\n            if bad=='unit':payload['hourly_units']['temperature_2m']='°F'\n            if bad=='probability':payload['hourly']['precipitation_probability']=[None]*24\n            if bad=='many_rows':return Reply([payload,payload])\n            if reused:shared.clear();shared.update(payload);payload=shared\n            return Reply(payload)\n        base=SimpleNamespace(getter=getter,ttl=ttl,rain_threshold=60,heat_c=38,feels_like_c=40)\n        def forbidden_base_forecast(*args,**kwargs):raise AssertionError('The old provider cache must not be used')\n        base.forecast=forbidden_base_forecast\n        client=provider.Forecast.__new__(provider.Forecast)\n        client.base=base;client.engine=dict(TZ=zone,RAIN_CODES={51,53,55,56,57,61,63,65,66,67,80,81,82,95,96,99},\n            valid_geo=lambda p:isinstance(p,dict) and provider.number(p.get('lat'),-90,90) is not None and provider.number(p.get('lng'),-180,180) is not None)\n        client.logger=logging.Logger('isolated-independent-forecast',level=logging.CRITICAL);client.logger.addHandler(logging.NullHandler())\n        client.cache={};client.lock=threading.RLock();client.blocked_until=0;client.requests=0;client.last_error=None\n        return client,calls\n    passed=[]\n    def check(name,condition):\n        if not condition:raise AssertionError(name)\n        passed.append(name)\n    client,calls=make();before=copy.deepcopy(points);value=client.forecast(points,when,now=now)\n    check('two_single_coordinate_requests',len(calls)==2 and all(',' not in row['latitude'] and ',' not in row['longitude'] for row in calls))\n    check('separate_exact_coordinates',[(float(row['latitude']),float(row['longitude'])) for row in calls]==[(p['lat'],p['lng']) for p in points])\n    check('distinct_values_and_correct_role_assignment',[p['precipitation_probability'] for p in value['points']]==[29,79] and [p['location'] for p in value['points']]==['pickup','destination'])\n    check('hour_selected_by_timestamp',[p['valid_at'] for p in value['points']]==['2026-10-19T19:00']*2)\n    check('input_coordinates_unchanged',points==before)\n    check('point_objects_are_independent',value['points'][0] is not value['points'][1])\n    check('provider_grid_does_not_replace_requested_coordinates',value['points'][0]['requested_latitude']==points[0]['lat'] and value['points'][0]['provider_latitude']==13.0)\n    value['points'][0]['precipitation_probability']=-999\n    cached=client.forecast(points,when,now=now)\n    check('cached_result_is_a_separate_copy',cached['points'][0]['precipitation_probability']==29 and len(calls)==2)\n    client.forecast(points,when.replace(minute=59),now=now)\n    check('same_hour_reuses_only_the_same_location_pair',len(calls)==2)\n    swapped=client.forecast(list(reversed(points)),when,now=now)\n    check('swapped_locations_have_correct_roles',[p['precipitation_probability'] for p in swapped['points']]==[79,29] and len(calls)==4)\n    later=client.forecast(points,when+timedelta(hours=1),now=now)\n    check('changed_pickup_hour_refreshes_both_points',[p['precipitation_probability'] for p in later['points']]==[30,80] and len(calls)==6)\n    shifted=copy.deepcopy(points);shifted[0]['lat']+=0.0000001\n    client.forecast(shifted,when,now=now)\n    check('tiny_coordinate_changes_cannot_collide_in_cache',len(calls)==8)\n    same,record=make(same=True);identical=same.forecast(points,when,now=now)\n    check('genuine_identical_values_are_accepted',identical['status']=='ok' and [p['precipitation_probability'] for p in identical['points']]==[29,29] and len(record)==2)\n    reuse,record=make(reused=True);reuse_value=reuse.forecast(points,when,now=now)\n    check('reused_response_objects_cannot_merge_points',[p['precipitation_probability'] for p in reuse_value['points']]==[29,79] and reuse_value['points'][0] is not reuse_value['points'][1])\n    same_pin,record=make();duplicate=same_pin.forecast([points[0],points[0]],when,now=now)\n    check('same_pin_for_both_roles_still_keeps_separate_results',duplicate['status']=='ok' and len(record)==2 and duplicate['points'][0] is not duplicate['points'][1])\n    stale,record=make();old_key=(tuple((round(p['lat'],5),round(p['lng'],5)) for p in points),when.replace(minute=0).isoformat())\n    stale.cache[old_key]=(time.monotonic(),{'status':'ok','points':[]})\n    fresh=stale.forecast(points,when,now=now)\n    check('legacy_batched_cache_is_not_reused',len(record)==2 and len(fresh['points'])==2)\n    for bad in ('missing_hour','duplicated_hour','timezone','unit','probability','many_rows','timeout'):\n        broken,record=make(bad=bad);pending=broken.forecast(points,when,now=now)\n        check('rejects_'+bad,pending['status']=='pending' and len(record)==1)\n    for status in (401,403,429):\n        denied,record=make(status=('first',status));pending=denied.forecast(points,when,now=now)\n        check('no_extra_requests_after_http_'+str(status),pending['status']=='pending' and pending['http_status']==status and len(record)==1)\n        if status==429:\n            denied.forecast(points,when,now=now);check('rate_limit_backoff_is_respected',len(record)==1 and denied.blocked_until>time.monotonic())\n    partial,record=make(status=('second',503));pending=partial.forecast(points,when,now=now)\n    check('destination_failure_cannot_reuse_pickup_values',pending['status']=='pending' and len(record)==2 and 'points' not in pending)\n    utc,record=make();localized=utc.forecast(points,'2026-10-19T13:45:00+00:00',now=now)\n    check('utc_pickup_is_converted_to_kolkata_hour',localized['points'][0]['valid_at']=='2026-10-19T19:00')\n    expired,record=make(ttl=1);expired.forecast(points,when,now=now)\n    for key,item in list(expired.cache.items()):expired.cache[key]=(time.monotonic()-5,item[1])\n    expired.forecast(points,when,now=now);check('expired_cache_fetches_both_points_again',len(record)==4)\n    invalid,record=make();pending=invalid.forecast([dict(lat=91,lng=80),points[1]],when,now=now)\n    check('invalid_coordinates_do_not_call_provider',pending['status']=='pending' and not record)\n    horizon,record=make();pending=horizon.forecast(points,when+timedelta(days=16),now=now)\n    check('outside_forecast_window_does_not_call_provider',pending['status']=='pending' and not record)\n    return dict(passed=True,checks=len(passed),names=passed)\n",tests)
    verification=tests['forecast_checks'](fresh)
    print('Independent forecast regression checks:',verification['checks'],'| PASS')

    # Compare runtime code using the SAME Python interpreter, avoiding the
    # cross-version AST hash issue seen in Step 1. No active cache is cleared.
    baseline_source=original
    for start,end,value in sorted(legacy_edits,reverse=True):baseline_source=baseline_source[:start]+value+baseline_source[end:]
    baseline={};exec(compile(baseline_source,str(path),'exec',dont_inherit=True),baseline)
    def signature(code):
        constants=tuple(signature(v) if isinstance(v,types.CodeType) else v for v in code.co_consts)
        return (code.co_code,code.co_names,code.co_varnames,code.co_freevars,code.co_cellvars,
            code.co_argcount,code.co_posonlyargcount,code.co_kwonlyargcount,constants)
    classes=[];visited=set()
    def consider(candidate):
        if not isinstance(candidate,type) or candidate.__name__!='Forecast':return
        fetch=candidate.__dict__.get('fetch');forecast=candidate.__dict__.get('forecast')
        if not isinstance(fetch,types.FunctionType) or not isinstance(forecast,types.FunctionType):return
        if Path(fetch.__code__.co_filename).resolve()!=path.resolve():return
        if not any(candidate is old for old in classes):classes.append(candidate)
    def namespace(values):
        if not isinstance(values,dict):return
        consider(values.get('Forecast'))
        module=values.get('_movio_reliable_provider')
        if type(module) is types.ModuleType:consider(vars(module).get('Forecast'))
        for name in ('_movio_required_weather_proxy','_movio_reliable_manual_forecast','manual_state'):
            function=values.get(name)
            if isinstance(function,types.FunctionType):visit(function)
        providers=values.get('_providers')
        if isinstance(providers,dict):
            for provider in list(providers.values()):
                if type(provider) is types.ModuleType:consider(vars(provider).get('Forecast'))
    def visit(function):
        if not isinstance(function,types.FunctionType) or id(function) in visited:return
        visited.add(id(function));consider(function.__globals__.get('Forecast'))
        for cell in function.__closure__ or ():
            try:value=cell.cell_contents
            except ValueError:continue
            if isinstance(value,types.FunctionType):visit(value)
            elif isinstance(value,dict):namespace(value)
            elif type(value) is types.ModuleType:namespace(vars(value))
        visit(getattr(function,'__wrapped__',None))
    bridge=globals().get('bridge');endpoint=getattr(bridge,'_wx_endpoint',None)
    if isinstance(endpoint,types.FunctionType):namespace(endpoint.__globals__)
    current=getattr(bridge,'_wx_required_forecast',None)
    if current is not None:consider(type(current))
    website_modules=[]
    for module in list(sys.modules.values()):
        if type(module) is not types.ModuleType:continue
        values=vars(module);filename=values.get('__file__')
        if filename:
            try:
                if Path(filename).resolve() in (path.resolve(),root/'integrations/movio_booking_reliability.py'):namespace(values)
            except (OSError,ValueError,TypeError):pass
        known=values.get('ROOT');handler=values.get('Handler')
        if known is None or not isinstance(handler,type):continue
        try:
            if Path(known).resolve()==root:
                website_modules.append(module);visit(getattr(handler,'do_GET',None));visit(getattr(handler,'do_POST',None))
        except (OSError,ValueError,TypeError):continue
    for candidate in classes:
        for name in method_sources:
            function=candidate.__dict__[name]
            if signature(function.__code__) not in (signature(getattr(baseline['Forecast'],name).__code__),signature(getattr(fresh.Forecast,name).__code__)):
                raise RuntimeError('An active forecast method was separately modified: '+name+'. No files changed.')
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet') if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    if len(listeners)!=1:raise RuntimeError('Expected the existing website on port 8080. No files changed.')
    pid=listeners[0];external=None;external_started=None
    if pid==os.getpid():
        if not website_modules or not classes:raise RuntimeError('The in-kernel forecast service was not identified. No files changed.')
    else:
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        if not any((Path(cwd)/Path(arg)).resolve()==server for arg in command if arg.endswith('.py')):
            raise RuntimeError('A different service owns port 8080. No files changed or process stopped.')
        controller=process
        for value in list(globals().values()):
            if type(value) in weakref.ProxyTypes:continue
            try:
                if isinstance(value,subprocess.Popen) and value.poll() is None and isinstance(value.args,(list,tuple)) and list(value.args)==command:
                    controller=value;break
            except ReferenceError:continue
        external=(controller,command,cwd,process.environ())
        external_started=process.create_time()
    protected=('server.py','web/index.html','web/app.js','web/customer_booking_tools.js','web/weather_manual.js',
        'integrations/movio_customer_tools.py','integrations/movio_booking_reliability.py',
        'integrations/ai_call_page.html','integrations/movio_original_tts_adapter.py')
    protected_before={name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in protected if (root/name).is_file()}
    def identity(value):return (id(getattr(value,'__func__',value)),id(getattr(value,'__self__',None)))
    voice_before={name:identity(getattr(bridge,name,None)) for name in ('_wx_endpoint','speak','TTS','LLM')}
    backup=None;written=False;patched=[];launched=None;stopped=False
    session=requests.Session();session.trust_env=False
    def healthy():
        try:
            reply=session.get('http://127.0.0.1:8080/api/weather-booking/reliability-health',timeout=(1,2))
            return reply.status_code==200 and reply.json().get('version')=='booking-reliability-1.0'
        except (requests.RequestException,ValueError):return False
    def wait_health():
        for attempt in range(12):
            if healthy():return True
            time.sleep(.25)
        return False
    def start_website(details):
        _,command,cwd,environment=details
        with (root/'journey_forecast_website.log').open('a',encoding='utf-8') as output:
            process=subprocess.Popen(command,cwd=cwd,env=environment,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
        globals()['WX_WEBSITE_PROCESS']=process
        return process
    try:
        if not healthy():raise RuntimeError('Original website health check failed. No files changed.')
        if path.read_text(encoding='utf-8')!=original:raise RuntimeError('Forecast source changed during validation. No files changed.')
        if source!=original:
            backup=root/('journey_forecast_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));saved=backup/path.relative_to(root)
            saved.parent.mkdir(parents=True);shutil.copy2(path,saved)
            temporary=path.with_name(path.name+'.independent.tmp')
            try:temporary.write_text(source,encoding='utf-8');os.replace(temporary,path)
            finally:temporary.unlink(missing_ok=True)
            written=True
        # Update the ORIGINAL methods in every live manual/voice provider copy.
        # Keep class identities, proxy closures, consent state and caches intact.
        for candidate in classes:
            for name in method_sources:
                function=candidate.__dict__[name];replacement=getattr(fresh.Forecast,name)
                patched.append((function,function.__code__,function.__doc__))
                function.__code__=replacement.__code__;function.__doc__=replacement.__doc__
        # A cached provider in a process older than the source must be reloaded,
        # even when a previous attempt already wrote the new source to disk.
        if external and (written or external_started<path.stat().st_mtime-1):
            external[0].terminate();stopped=True;external[0].wait(timeout=8);launched=start_website(external)
        if not wait_health():raise RuntimeError('Updated website health check failed.')
        for candidate in classes:
            for name in method_sources:
                assert signature(candidate.__dict__[name].__code__)==signature(getattr(fresh.Forecast,name).__code__)
        assert all(hashlib.sha256((root/name).read_bytes()).hexdigest()==value for name,value in protected_before.items())
        assert all(identity(getattr(bridge,name,None))==value for name,value in voice_before.items())
    except Exception as error:
        if written:shutil.copy2(backup/path.relative_to(root),path)
        for function,code,doc in reversed(patched):function.__code__=code;function.__doc__=doc
        recovery='not needed'
        if stopped:
            try:
                if launched is not None and launched.poll() is None:launched.terminate();launched.wait(timeout=8)
                start_website(external);recovery='PASS' if wait_health() else 'FAILED'
            except Exception as exc:recovery='FAILED: '+type(exc).__name__
        raise RuntimeError(str(error)+' | Source rollback: '+('done' if written else 'not needed')+' | Website recovery: '+recovery+' | Backup: '+str(backup)) from error
    finally:session.close()

    # Two real provider calls, using labelled diagnostic points rather than
    # customer records. No booking/preview request or model generation occurs.
    live=dict(status='SKIPPED',reason='JW_WEATHER_LIVE_CHECK=False')
    if globals().get('JW_WEATHER_LIVE_CHECK',True):
        try:
            zone=ZoneInfo('Asia/Kolkata');now=datetime.now(zone)
            when=globals().get('JW_WEATHER_TEST_AT') or (now+timedelta(days=1)).replace(hour=19,minute=37,second=0,microsecond=0)
            if isinstance(when,str):when=datetime.fromisoformat(when)
            when=when.replace(tzinfo=zone) if when.tzinfo is None else when.astimezone(zone)
            points=copy.deepcopy(globals().get('JW_WEATHER_TEST_POINTS') or [dict(lat=13.0827,lng=80.2707),dict(lat=12.9249,lng=80.1000)])
            actual_requests=[]
            def getter(url,params,timeout):
                actual_requests.append(dict(latitude=params['latitude'],longitude=params['longitude'],date=params['start_date'],timezone=params['timezone']))
                return requests.get(url,params=params,timeout=timeout)
            client=fresh.Forecast.__new__(fresh.Forecast)
            client.base=types.SimpleNamespace(getter=getter,ttl=900,rain_threshold=60,heat_c=38,feels_like_c=40)
            client.engine=dict(TZ=zone,RAIN_CODES={51,53,55,56,57,61,63,65,66,67,80,81,82,95,96,99},
                valid_geo=lambda p:isinstance(p,dict) and fresh.number(p.get('lat'),-90,90) is not None and fresh.number(p.get('lng'),-180,180) is not None)
            client.logger=logging.Logger('isolated-live-weather-check',level=logging.CRITICAL);client.logger.addHandler(logging.NullHandler())
            client.cache={};client.lock=threading.RLock();client.blocked_until=0;client.last_error=None;client.requests=0
            weather=client.forecast(points,when,now=now)
            live=dict(status='PASS' if weather.get('status')=='ok' else 'PENDING',diagnostic_points_only=True,
                requested_pickup_time=when.isoformat(),provider_requests=actual_requests,weather_status=weather.get('status'),
                reason=weather.get('reason'),http_status=weather.get('http_status'),points=weather.get('points',[]))
            if weather.get('status')=='ok':
                a,b=weather['points'];fields=('weather_code','precipitation_probability','temperature_2m')
                live['matching_weather_values']=all(a.get(key)==b.get(key) for key in fields)
                live['matching_values_are_allowed']=True
                assert len(actual_requests)==2
        except Exception as exc:live=dict(status='ERROR',error_type=type(exc).__name__,message=str(exc)[:240])
    report=dict(step=3,version='independent-forecast-1.0',installation='PASS',offline_checks=verification['checks'],
        independent_requests=True,cache_key='exact ordered coordinates + Kolkata forecast hour + adapter version',
        active_forecast_classes_verified=len(classes),website_health='PASS',ui_and_locality_sources_preserved=True,
        voice_handler_identities_preserved=True,changed_files=[str(path.relative_to(root))] if written else [],
        backup=str(backup) if backup else None,live_provider_probe=live,
        browser_verification='REQUIRED: refresh, select two fresh pins and a pickup time, then Review ride')
    print(json.dumps(report,ensure_ascii=False,indent=2))
    print('Hourly selection is explicit: 19:37 uses the 19:00 forecast sample, in Asia/Kolkata.')
    print('Identical provider forecasts remain valid; values are never changed just to look different.')
    print('Refresh the dashboard. Select pickup, drop-off and pickup time, then Review ride.')
    print('Share this output and check that both displayed locality names match the selected pins.')
    return report

JW_INDEPENDENT_FORECAST=movio_fix_independent_journey_forecasts()


Independent forecast regression checks: 32 | PASS
{
  "step": 3,
  "version": "independent-forecast-1.0",
  "installation": "PASS",
  "offline_checks": 32,
  "independent_requests": true,
  "cache_key": "exact ordered coordinates + Kolkata forecast hour + adapter version",
  "active_forecast_classes_verified": 3,
  "website_health": "PASS",
  "ui_and_locality_sources_preserved": true,
  "voice_handler_identities_preserved": true,
  "changed_files": [
    "integrations/movio_required_weather.py"
  ],
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/journey_forecast_backup_20261028_070733_979570",
  "live_provider_probe": {
    "status": "PASS",
    "diagnostic_points_only": true,
    "requested_pickup_time": "2026-10-29T19:37:00+05:30",
    "provider_requests": [
      {
        "latitude": "13.0827",
        "longitude": "80.2707",
        "date": "2026-10-29",
        "timezone": "Asia/Kolkata"
      },
      {
        "latitude": "12.9249",
        "longitude": "80.1",
        "date":

In [124]:
# JOURNEY WEATHER — STEP 2B: individual area names instead of zone/ward labels.
# Run this ONE cell in the current Kaggle kernel BEFORE evaluation.
# It edits the original handlers in place, backs up the changed source files and
# restarts only the website process when required. Forecast retrieval is unchanged.
def movio_fix_individual_area_names():
    import ast
    import copy
    import hashlib
    import json
    import logging
    import os
    import re
    import shutil
    import subprocess
    import sys
    import time
    import types
    import weakref
    from datetime import datetime
    from pathlib import Path
    from urllib.parse import parse_qsl,urlencode,urlsplit,urlunsplit
    import psutil
    import requests

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    revision_name='main-locality-1.1'
    reverse_source='def reverse_address(lat,lng):\n    """MOVIO_MAIN_LOCALITY_V1_1: real area labels; exact pin/GPS coordinates."""\n    import logging\n    lat,lng=coords(lat,lng)\n    fields=(\'suburb\',\'quarter\',\'neighbourhood\',\'neighborhood\',\'locality\',\'village\',\'hamlet\',\'residential\',\'town\')\n    admin=re.compile(r\'(?<!\\w)(?:sub[- ]?zone|zone|ward|division|taluk|tehsil|corporation|municipality|panchayat)(?:\\b|(?=\\d))|மண்டலம்|வார்டு|மாநகராட்சி|நகராட்சி|ஊராட்சி\',re.I)\n    road=re.compile(r\'\\b(?:road|street|highway|expressway|lane|avenue)\\b|சாலை|தெரு|நெடுஞ்சாலை\',re.I)\n    def clean(value):return re.sub(r\'\\s+\',\' \',value).strip() if isinstance(value,str) else \'\'\n    def acceptable(value,broad=()):\n        value=clean(value)\n        return bool(value and any(c.isalpha() for c in value) and not admin.search(value)\n            and not road.search(value) and value.casefold() not in broad)\n    def select(row,format_name):\n        if not isinstance(row,dict):return None\n        if format_name==\'geocodejson\':\n            features=row.get(\'features\')\n            if not isinstance(features,list) or len(features)!=1:return None\n            feature=features[0]\n            if not isinstance(feature,dict):return None\n            properties=feature.get(\'properties\')\n            if not isinstance(properties,dict):return None\n            data=properties.get(\'geocoding\')\n            if not isinstance(data,dict):return None\n            broad={clean(data.get(key)).casefold() for key in (\'city\',\'county\',\'state\',\'country\') if clean(data.get(key))}\n            for key in (\'locality\',\'district\'):\n                if acceptable(data.get(key),broad):return clean(data[key]),clean(data.get(\'label\')),\'geocodejson.\'+key\n            if data.get(\'type\') in (\'locality\',\'district\') and acceptable(data.get(\'name\'),broad):\n                return clean(data[\'name\']),clean(data.get(\'label\')),\'geocodejson.name\'\n            return None\n        address=row.get(\'address\')\n        if not isinstance(address,dict):address={}\n        broad={clean(address.get(key)).casefold() for key in (\'city\',\'municipality\',\'county\',\'state_district\',\'state\',\'country\') if clean(address.get(key))}\n        for key in fields:\n            # A real town/village may legitimately also be the city name.\n            excluded=() if key in (\'town\',\'village\',\'hamlet\') else broad\n            if acceptable(address.get(key),excluded):\n                return clean(address[key]),clean(row.get(\'display_name\')),\'address.\'+key\n        if row.get(\'addresstype\',row.get(\'type\')) in fields and acceptable(row.get(\'name\'),broad):\n            return clean(row[\'name\']),clean(row.get(\'display_name\')),\'place.name\'\n        return None\n    # A building/street response can omit the neighbourhood. Only in that case\n    # request the provider\'s neighbourhood-level structured result at this pin.\n    for format_name,zoom in ((\'json\',\'18\'),(\'geocodejson\',\'14\')):\n        try:\n            row=geo_request(\'reverse\',dict(lat=str(lat),lon=str(lng),format=format_name,\n                zoom=zoom,addressdetails=\'1\',layer=\'address\'))\n            chosen=select(row,format_name)\n            if chosen:\n                locality,full,source=chosen\n                return dict(name=locality[:170],locality=locality[:170],full_address=(full or locality)[:1000],\n                    lat=lat,lng=lng,locality_status=\'ok\',locality_source=source,locality_policy=\'main-locality-1.1\')\n        except (LookupError,KeyError,ValueError,TypeError) as exc:\n            logging.getLogger(\'movio.customer_tools\').warning(\'reverse_area_lookup_failed: %s\',type(exc).__name__)\n            break\n    logging.getLogger(\'movio.customer_tools\').warning(\'reverse_area_unresolved: no non-administrative locality returned\')\n    # Do not relabel a whole zone as a neighbourhood or snap a GPS coordinate.\n    return dict(name=\'Selected location\',locality=\'\',full_address=\'\',lat=lat,lng=lng,\n        locality_status=\'unavailable\',locality_source=None,locality_policy=\'main-locality-1.1\')\n'
    replacements=[('const controller=new AbortController(),timer=setTimeout(()=>controller.abort(),16000);', "const controller=new AbortController(),timer=setTimeout(()=>controller.abort(),path.startsWith('reverse-address?')?30000:16000);"), ("localityVersion:'main-locality-1.0'", "localityVersion:'main-locality-1.1'")]
    node_test='\nconst vm=require(\'node:vm\'),assert=require(\'node:assert/strict\');\nconst sources=JSON.parse(require(\'node:fs\').readFileSync(0,\'utf8\'));\nconst elements=new Map(),calls=[],reverse=[],previews=[];\nfunction element(id){if(!elements.has(id))elements.set(id,{id,value:\'\',disabled:false,hidden:false,title:\'\',textContent:\'\',onclick:null,listeners:{},\n addEventListener(kind,fn){(this.listeners[kind]??=[]).push(fn)},setAttribute(){},replaceChildren(){},append(){},focus(){},\n querySelector(){return null}});return elements.get(id)}\nelement(\'wxManualLanguage\').value=\'English\';element(\'wxManualDate\').value=\'2026-10-20T19:37\';element(\'wxManualCount\').value=\'2\';\nelement(\'bookRide\').onclick=async()=>previews.push(JSON.parse(JSON.stringify({pickup:c.pickup,destination:c.destination})));\nconst c={window:{},document:{getElementById:element},console:{warn(){}},AbortController,URLSearchParams,Promise,Number,JSON,Object,Array,String,\n role:\'customer\',pickup:null,destination:null,selecting:\'pickup\',vehicle:\'Mini\',km:0,minutes:0,routeData:null,routeLine:null,searchTimer:null,\n setTimeout:()=>1,clearTimeout(){},fares(){},map:{panTo(){},removeLayer(){},fitBounds(){}},L:{},\n updateRoute(){},locate:async()=>({lat:12.95555555555,lng:80.24444444444}),\n fetch(url,options){calls.push({url,options});\n  if(url.includes(\'/reverse-address?\'))return new Promise((resolve,reject)=>reverse.push({url,options,resolve:p=>resolve({ok:true,json:async()=>({ok:true,place:p})}),reject}));\n  if(url.includes(\'/route?\'))return Promise.resolve({ok:true,json:async()=>({ok:true,route:{km:12.7,minutes:24,geometry:[]}})});\n  throw Error(\'Unexpected request: \'+url);\n },\n setPlace(id,p){c[id]={lat:p.lat,lng:p.lng,name:p.name};element(id).value=p.name;if(id===\'pickup\')c.selecting=\'destination\';c.updateRoute();}\n};\nvm.createContext(c);\nconst boundary=sources.tools.indexOf(\' // Details use the authenticated single-booking API\');\nassert.ok(boundary>0,\'Existing UI test boundary missing\');\nvm.runInContext(sources.tools.slice(0,boundary)+"window.MOVIOCustomerBookingUI={pinPlace,preview,currentPickup,state,applyPlace};})();",c);\nconst ui=c.window.MOVIOCustomerBookingUI;\nconst point=(name,lat=0,lng=0)=>({name,locality:name,full_address:name+\' street address\',lat,lng,locality_status:\'ok\'});\nconst turn=()=>new Promise(resolve=>setImmediate(resolve));\nlet checks=0;\nasync function main(){\n const a={lat:12.97123456789,lng:80.26123456789},b={lat:12.92123456789,lng:80.21123456789};\n const pa=ui.pinPlace(\'pickup\',a),pb=ui.pinPlace(\'destination\',b);\n assert.equal(c.pickup.lat,a.lat);assert.equal(c.destination.lng,b.lng);checks++;\n assert.equal(calls.filter(x=>x.url.includes(\'/route?\')).length,0,\'Route must await labels\');checks++;\n reverse[1].resolve(point(\'Tambaram\',9,8));await pb;\n assert.equal(c.destination.name,\'Tambaram\');assert.equal(c.destination.lat,b.lat);assert.equal(c.destination.lng,b.lng);checks++;\n assert.equal(calls.filter(x=>x.url.includes(\'/route?\')).length,0);checks++;\n reverse[0].resolve(point(\'Kottivakkam\',7,6));await pa;await turn();\n assert.equal(c.pickup.name,\'Kottivakkam\');assert.equal(c.pickup.lat,a.lat);assert.equal(c.pickup.lng,a.lng);checks++;\n const routes=calls.filter(x=>x.url.includes(\'/route?\'));assert.equal(routes.length,1);\n const query=new URL(routes[0].url,\'http://localhost\').searchParams;\n assert.equal(query.get(\'plat\'),String(a.lat));assert.equal(query.get(\'dlng\'),String(b.lng));checks++;\n const old=ui.pinPlace(\'pickup\',{lat:13.001,lng:80.001}),newer=ui.pinPlace(\'pickup\',{lat:13.002,lng:80.002});\n reverse[3].resolve(point(\'New locality\'));await newer;reverse[2].resolve(point(\'Stale locality\'));await old;await turn();\n assert.equal(c.pickup.name,\'New locality\');assert.equal(c.pickup.lat,13.002);checks++;\n const pending=ui.pinPlace(\'pickup\',{lat:13.003,lng:80.003});\n element(\'pickup\').value=\'Typed correction\';for(const fn of element(\'pickup\').listeners.input)fn();\n reverse[4].resolve(point(\'Late locality\'));await pending;\n assert.equal(element(\'pickup\').value,\'Typed correction\');checks++;\n const failed=ui.pinPlace(\'pickup\',a);reverse[5].reject(Error(\'provider unavailable\'));await failed;await turn();\n assert.equal(c.pickup.name,\'Selected pickup\');assert.equal(c.pickup.lat,a.lat);assert.equal(c.pickup.lng,a.lng);checks++;\n const gps=ui.currentPickup();await turn();reverse[6].resolve(point(\'GPS locality\',1,2));await gps;await turn();\n assert.equal(c.pickup.name,\'GPS locality\');assert.equal(c.pickup.lat,12.95555555555);assert.equal(c.pickup.lng,80.24444444444);checks++;\n const last=ui.pinPlace(\'pickup\',a);const before=previews.length,review=ui.preview();await turn();\n assert.equal(previews.length,before,\'Review must await the selected pin label\');checks++;\n reverse[7].resolve(point(\'Kottivakkam\'));await last;await review;await turn();\n assert.equal(previews.length,before+1);assert.equal(previews.at(-1).pickup.lat,a.lat);assert.equal(previews.at(-1).destination.lng,b.lng);checks++;\n const count=reverse.length;element(\'wxManualDate\').disabled=true;await ui.pinPlace(\'pickup\',{lat:1,lng:2});assert.equal(reverse.length,count);checks++;\n element(\'wxManualDate\').disabled=false;c.role=\'driver\';await ui.pinPlace(\'pickup\',{lat:1,lng:2});assert.equal(reverse.length,count);checks++;\n const handlers=[];c.role=\'customer\';c.map.on=(event,fn)=>handlers.push(fn);c.toast=()=>{};\n vm.runInContext(sources.map,c);assert.equal(handlers.length,1);checks++;\n handlers[0]({latlng:{lat:12.88,lng:80.22}});assert.equal(reverse.length,count+1);checks++;\n reverse.at(-1).resolve(point(\'Pinned locality\'));await turn();\n console.log(JSON.stringify({javascript_checks:checks,passed:true}));\n}\nmain().catch(error=>{console.error(error.stack);process.exitCode=1});\n'
    old_map="map.on('click',e=>{if(role==='driver')return;const which=selecting;if(window.MOVIOCustomerBookingUI?.pinPlace){void window.MOVIOCustomerBookingUI.pinPlace(which,e.latlng);return;}setPlace(which,{lat:e.latlng.lat,lng:e.latlng.lng,name:`Map pin (${e.latlng.lat.toFixed(4)}, ${e.latlng.lng.toFixed(4)})`});toast(which==='pickup'?'Pickup pinned. Tap map again for destination.':'Destination pinned.');});"
    new_map="map.on('click',e=>{if(role==='driver')return;const which=selecting;if(window.MOVIOCustomerBookingUI?.pinPlace){void window.MOVIOCustomerBookingUI.pinPlace(which,e.latlng);return;}setPlace(which,{lat:e.latlng.lat,lng:e.latlng.lng,name:`Map pin (${e.latlng.lat.toFixed(4)}, ${e.latlng.lng.toFixed(4)})`});toast(which==='pickup'?'Pickup pinned. Tap map again for destination.':'Destination pinned.');});"
    allowed={'integrations/movio_customer_tools.py': ('408a7d188816efc7ce5656b692387936be5e49571d1a1796573c961b7a4c460e', 'f2d87dfd8e624b2d487d78f1786f4afc91741da0f9a83f0b5b919d15c0dee73d'), 'web/customer_booking_tools.js': ('60b65a38750871fe708fa96d41169789bc3a7f4fef88cc17a2fcc3649b932a9a', 'd2cd1b9e878867e848afbc32cd77027bd5ff645dad3483809ec76bbd38224740')}
    names=('integrations/movio_customer_tools.py','web/customer_booking_tools.js','web/app.js','web/index.html')
    paths={name:root/name for name in names}
    originals={name:paths[name].read_text(encoding='utf-8') for name in names}
    sources=dict(originals)
    sha=lambda value:hashlib.sha256(value.encode()).hexdigest()
    for name,hashes in allowed.items():
        if sha(sources[name]) not in hashes:
            raise RuntimeError('The inspected source changed: '+name+'. No files changed. Share a fresh Step 1 output.')

    backend=names[0];tools=names[1]
    tree=ast.parse(sources[backend]);old_reverse=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name=='reverse_address')
    existing=ast.get_source_segment(sources[backend],old_reverse)
    sources[backend]=sources[backend].replace(existing,reverse_source.rstrip(),1)
    for before,after in replacements:
        if sources[tools].count(before)==1:sources[tools]=sources[tools].replace(before,after,1)
        elif sources[tools].count(after)!=1:
            raise RuntimeError('The customer selection handler differs from Step 2. No files changed.')
    if sources['web/app.js'].count(old_map)==1:
        sources['web/app.js']=sources['web/app.js'].replace(old_map,new_map,1)
    elif sources['web/app.js'].count(new_map)!=1:
        raise RuntimeError('The original map-click handler differs. No files changed.')
    if sources['web/app.js'].count("map.on('click'")!=originals['web/app.js'].count("map.on('click'"):
        raise AssertionError('Map handlers must not be duplicated.')

    revision=sha(sources[backend]+sources[tools]+sources['web/app.js'])[:12]
    counts={name:0 for name in ('app.js','customer_booking_tools.js')}
    def cache_bust(match):
        value=match.group(2);parsed=urlsplit(value);name=Path(parsed.path).name
        if name not in counts:return match.group(0)
        counts[name]+=1
        query=[(k,v) for k,v in parse_qsl(parsed.query,keep_blank_values=True) if k!='journey_locality']
        query.append(('journey_locality',revision))
        src=urlunsplit((parsed.scheme,parsed.netloc,parsed.path,urlencode(query),parsed.fragment))
        return match.group(1)+src+match.group(3)
    sources['web/index.html']=re.sub(r'(<script\b[^>]*\bsrc=["\x27])([^"\x27]+)(["\x27])',cache_bust,sources['web/index.html'],flags=re.I)
    if any(count!=1 for count in counts.values()):raise RuntimeError('Expected one existing app/customer-tools script. No files changed.')

    # Verify that reverse_address is the only changed Python definition.
    def other_python(source):
        tree=ast.parse(source);tree.body=[n for n in tree.body if not isinstance(n,ast.FunctionDef) or n.name!='reverse_address']
        return ast.dump(tree,include_attributes=False)
    if other_python(sources[backend])!=other_python(originals[backend]):raise AssertionError('Unrelated backend change detected.')
    namespace=dict(__name__='isolated_locality_fixture',__file__=str(paths[backend]))
    exec(compile(sources[backend],str(paths[backend]),'exec'),namespace)
    code=namespace['reverse_address'].__code__
    fixture_scope={};exec("def check_area_selector(namespace):\n    import copy\n    import logging\n    pin=(12.97123456789,80.26123456789);checks=0\n    cases=[\n        ({'suburb':'Zone 5 Royapuram','neighbourhood':'Old Washermanpet','city':'Chennai'},'Old Washermanpet'),\n        ({'suburb':'Zone 1 Tiruvottiyur','quarter':'Tondiarpet','city':'Chennai'},'Tondiarpet'),\n        ({'suburb':'Zone 13 Adyar','neighbourhood':'Kottivakkam','road':'Kottivakkam Kuppam Road'},'Kottivakkam'),\n        ({'suburb':'Ward 173','quarter':'Division 174','neighbourhood':'Kottivakkam'},'Kottivakkam'),\n        ({'suburb':'Zone5 Royapuram','neighbourhood':'Royapuram'},'Royapuram'),\n        ({'suburb':'ZONE-13 ADYAR','neighbourhood':'Palavakkam'},'Palavakkam'),\n        ({'suburb':'மண்டலம் 13 அடையாறு','neighbourhood':'கொட்டிவாக்கம்'},'கொட்டிவாக்கம்'),\n        ({'suburb':'Ward No. 10','locality':'Anna Nagar'},'Anna Nagar'),\n        ({'suburb':'Kottivakkam','neighbourhood':'Kuppam','road':'Kuppam Road'},'Kottivakkam'),\n        ({'town':'Tambaram','city':'Tambaram'},'Tambaram'),\n        ({'village':'Vandalur','city':'Vandalur'},'Vandalur'),\n        ({'suburb':'Chennai','city':'Chennai','neighbourhood':'Adyar'},'Adyar'),\n        ({'suburb':'Kottivakkam Kuppam Road','neighbourhood':'Kottivakkam'},'Kottivakkam'),\n        ({'quarter':'  Anna   Nagar '},'Anna Nagar'),\n    ]\n    original=namespace['geo_request'];logger=logging.getLogger('movio.customer_tools');disabled=logger.disabled\n    try:\n        logger.disabled=True\n        for address,expected in cases:\n            row=dict(lat='13.99',lon='80.99',address=address,display_name='Test Road, '+expected+', Chennai');before=copy.deepcopy(row);calls=[]\n            def fake(kind,params):calls.append(dict(params));return row\n            namespace['geo_request']=fake;result=namespace['reverse_address'](*pin)\n            assert result['name']==expected and (result['lat'],result['lng'])==pin\n            assert row==before and len(calls)==1 and calls[0]['lat']==str(pin[0]) and calls[0]['lon']==str(pin[1])\n            checks+=1\n        def response(area='Kottivakkam',kind='locality'):\n            return dict(features=[dict(geometry=dict(type='Point',coordinates=[80.999,13.999]),\n                properties=dict(geocoding=dict(type=kind,locality=area,city='Chennai',label=area+', Chennai')))])\n        for rejected in ('Zone 5 Royapuram','Zone 1 Tiruvottiyur','Ward 27','Division 13','Chennai'):\n            calls=[]\n            def fallback(kind,params):\n                calls.append(dict(params))\n                return dict(address=dict(suburb=rejected,city='Chennai')) if params['format']=='json' else response()\n            namespace['geo_request']=fallback;result=namespace['reverse_address'](*pin)\n            assert result['name']=='Kottivakkam' and result['locality_source']=='geocodejson.locality'\n            assert len(calls)==2 and calls[1]['zoom']=='14' and all(p['lat']==str(pin[0]) and p['lon']==str(pin[1]) for p in calls)\n            assert (result['lat'],result['lng'])==pin;checks+=1\n        for area in ('Zone 5 Royapuram','Ward 3','Division 12','Chennai','Test Road','12345'):\n            def unresolved(kind,params):\n                return dict(address=dict(suburb='Zone 5 Royapuram',city='Chennai')) if params['format']=='json' else response(area)\n            namespace['geo_request']=unresolved;result=namespace['reverse_address'](*pin)\n            assert result['locality_status']=='unavailable' and result['name']=='Selected location' and (result['lat'],result['lng'])==pin\n            checks+=1\n        namespace['geo_request']=lambda *_:dict(features=None)\n        result=namespace['reverse_address'](*pin);assert result['locality_status']=='unavailable';checks+=1\n        def failed(*_):raise namespace['LookupError']('fixture network failure')\n        namespace['geo_request']=failed;result=namespace['reverse_address'](*pin)\n        assert result['locality_status']=='unavailable' and (result['lat'],result['lng'])==pin;checks+=1\n        try:namespace['reverse_address'](91,80)\n        except ValueError:checks+=1\n        else:raise AssertionError('Invalid coordinates accepted')\n        return checks\n    finally:namespace['geo_request']=original;logger.disabled=disabled\n",fixture_scope)
    checks=fixture_scope['check_area_selector'](namespace)
    node=shutil.which('node')
    if not node:raise RuntimeError('Node.js is required for this cell’s JavaScript checks. No files changed.')
    for name in ('web/app.js','web/customer_booking_tools.js'):
        result=subprocess.run([node,'--check'],input=sources[name],capture_output=True,text=True,timeout=15)
        if result.returncode:raise RuntimeError('JavaScript syntax check failed for '+name+': '+result.stderr[:400])
    result=subprocess.run([node,'-e',node_test],input=json.dumps(dict(tools=sources[tools],map=new_map)),capture_output=True,text=True,timeout=20)
    if result.returncode:raise RuntimeError('Selection regression check failed; no files changed.\n'+result.stderr[-1500:])
    js_checks=json.loads(result.stdout.strip().splitlines()[-1])
    print('Offline locality/coordinate checks:',checks,'| JavaScript selection checks:',js_checks['javascript_checks'],'| PASS')

    protected=('server.py','integrations/movio_required_weather.py','integrations/movio_booking_reliability.py',
        'integrations/ai_call_page.html','integrations/movio_original_tts_adapter.py')
    protected_before={name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in protected if (root/name).is_file()}
    server=(root/'server.py').resolve()
    listeners=sorted({c.pid for c in psutil.net_connections(kind='inet') if c.status=='LISTEN' and c.laddr.port==8080 and c.pid})
    if len(listeners)!=1:raise RuntimeError('Expected one existing website on port 8080. No files changed.')
    pid=listeners[0];runtime_namespaces=[];external=None
    if pid==os.getpid():
        seen=set()
        def visit(fn):
            if not isinstance(fn,types.FunctionType) or id(fn) in seen:return
            seen.add(id(fn));values=fn.__globals__;filename=values.get('__file__')
            if filename and Path(filename).resolve()==paths[backend].resolve() and callable(values.get('reverse_address')):
                if not any(values is old for old in runtime_namespaces):runtime_namespaces.append(values)
            for cell in fn.__closure__ or ():
                try:visit(cell.cell_contents)
                except ValueError:pass
            visit(getattr(fn,'__wrapped__',None))
        for module in list(sys.modules.values()):
            if type(module) is not types.ModuleType:continue
            values=vars(module);known=values.get('ROOT');handler=values.get('Handler')
            if known is None or not isinstance(handler,type):continue
            try:
                if Path(known).resolve()==root:visit(getattr(handler,'do_GET',None))
            except (OSError,ValueError,TypeError):continue
        if not runtime_namespaces:raise RuntimeError('Could not identify the existing in-kernel customer handler. No files changed.')
    else:
        process=psutil.Process(pid);command=process.cmdline();cwd=process.cwd()
        scripts=[Path(arg) for arg in command if arg.endswith('.py')]
        if not any((Path(cwd)/path).resolve()==server for path in scripts):
            raise RuntimeError('A different service owns port 8080. No files changed or process stopped.')
        controller=process
        for value in list(globals().values()):
            if type(value) in weakref.ProxyTypes:continue
            try:
                if isinstance(value,subprocess.Popen) and value.poll() is None and isinstance(value.args,(list,tuple)) and list(value.args)==command:
                    controller=value;break
            except ReferenceError:continue
        external=(controller,command,cwd,process.environ())

    changes={name:value for name,value in sources.items() if value!=originals[name]}
    backup=None;written=[];active_old=[];launched=None;stopped=False
    session=requests.Session();session.trust_env=False
    def healthy():
        try:
            response=session.get('http://127.0.0.1:8080/api/customer-tools/health',timeout=(1,2))
            return response.status_code==200 and response.json().get('version')=='customer-booking-tools-1.0'
        except (requests.RequestException,ValueError):return False
    def wait_health():
        for attempt in range(12):
            if healthy():return True
            time.sleep(.25)
        return False
    def start_website(details):
        _,command,cwd,environment=details
        with (root/'journey_area_website.log').open('a',encoding='utf-8') as output:
            proc=subprocess.Popen(command,cwd=cwd,env=environment,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
        globals()['WX_WEBSITE_PROCESS']=proc
        return proc
    try:
        if not healthy():raise RuntimeError('The original customer-tools health check failed. No files changed.')
        if changes:
            if any(paths[name].read_text(encoding='utf-8')!=originals[name] for name in names):
                raise RuntimeError('A source file changed during validation. No files changed.')
            backup=root/('journey_area_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
            for name in changes:
                target=backup/name;target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(paths[name],target)
            for name,value in changes.items():
                temporary=paths[name].with_name(paths[name].name+'.locality.tmp')
                try:temporary.write_text(value,encoding='utf-8');os.replace(temporary,paths[name])
                finally:temporary.unlink(missing_ok=True)
                written.append(name)
        for values in runtime_namespaces:
            old=values['reverse_address'];active_old.append((values,old))
            values['reverse_address']=types.FunctionType(code,values,'reverse_address')
            assert 'MOVIO_MAIN_LOCALITY_V1' in (values['reverse_address'].__doc__ or '')
        if external and (backend in changes):
            controller=external[0];controller.terminate();stopped=True;controller.wait(timeout=8)
            launched=start_website(external)
        if not wait_health():raise RuntimeError('The updated website did not pass its health check.')
        for name in ('web/app.js','web/customer_booking_tools.js'):
            response=session.get('http://127.0.0.1:8080/'+Path(name).name+'?journey_locality='+revision,timeout=(1,3))
            if response.status_code!=200 or hashlib.sha256(response.content).hexdigest()!=sha(sources[name]):
                raise RuntimeError('The website is not serving the validated '+Path(name).name)
        denied=session.get('http://127.0.0.1:8080/api/customer-tools/reverse-address?lat=13&lng=80',timeout=(1,2))
        if denied.status_code not in (401,403):raise RuntimeError('Customer authentication check failed.')
        assert all(hashlib.sha256((root/name).read_bytes()).hexdigest()==value for name,value in protected_before.items())
    except Exception as error:
        for name in reversed(written):shutil.copy2(backup/name,paths[name])
        for values,old in active_old:values['reverse_address']=old
        recovery='not needed'
        if stopped:
            try:
                if launched is not None and launched.poll() is None:launched.terminate();launched.wait(timeout=8)
                start_website(external);recovery='PASS' if wait_health() else 'FAILED'
            except Exception as exc:recovery='FAILED: '+type(exc).__name__
        raise RuntimeError(str(error)+' | Source rollback: '+('done' if written else 'not needed')+' | Website recovery: '+recovery+' | Backup: '+str(backup)) from error
    finally:session.close()
    result=dict(step='2B',version=revision_name,passed=True,python_checks=checks,javascript_checks=js_checks['javascript_checks'],
        website_health='PASS',served_javascript='PASS',customer_authentication='PASS',
        forecast_and_voice_sources_unchanged=True,changed_files=list(changes),backup=str(backup) if backup else None,
        browser_map_verification='REQUIRED: refresh, test a new map pin AND Use current location; check the actual area names')
    print(json.dumps(result,ensure_ascii=False,indent=2))
    print('Reload the CUSTOMER DASHBOARD with Ctrl+Shift+R. On mobile, reload the page.')
    print('Test a NEW map pin, then Use current location. The name must be a real area, not a zone/ward.')
    print('Existing form values can retain old labels until you reselect a pin or current location. Share the new names and this output.')
    return result

JW_AREA_NAME_FIX=movio_fix_individual_area_names()


Offline locality/coordinate checks: 28 | JavaScript selection checks: 16 | PASS
{
  "step": "2B",
  "version": "main-locality-1.1",
  "passed": true,
  "python_checks": 28,
  "javascript_checks": 16,
  "website_health": "PASS",
  "served_javascript": "PASS",
  "customer_authentication": "PASS",
  "forecast_and_voice_sources_unchanged": true,
  "changed_files": [
    "integrations/movio_customer_tools.py",
    "web/customer_booking_tools.js",
    "web/index.html"
  ],
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/journey_area_backup_20261028_070735_319248",
  "browser_map_verification": "REQUIRED: refresh, test a new map pin AND Use current location; check the actual area names"
}
Reload the CUSTOMER DASHBOARD with Ctrl+Shift+R. On mobile, reload the page.
Test a NEW map pin, then Use current location. The name must be a real area, not a zone/ward.
Existing form values can retain old labels until you reselect a pin or current location. Share the new names and this output.


In [125]:
# AI CALL WEATHER POPUP — STEP 1: READ-ONLY INSPECTION.
# Paste this entire file into ONE Kaggle cell before evaluation.
# No patches, model/audio generation, weather queries or booking actions.
def movio_inspect_call_weather_popup():
    import hashlib
    import inspect
    import json
    import re
    import shutil
    import subprocess
    import types
    from collections import Counter
    from html.parser import HTMLParser
    from pathlib import Path
    from urllib.parse import urlsplit

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    bridge=globals().get('bridge')
    active=getattr(bridge,'HTML',None)
    report=dict(step='Call weather popup 1 — INSPECTION ONLY',root=str(root),files=[],runtime={},
        served_call_page={},html={},weather_blocks={},call_handlers={},backend_sources={})
    def redact(value):
        value=re.sub(r'(?i)(Bearer\s+)[A-Za-z0-9._-]+',r'\1[REDACTED]',str(value))
        return re.sub(r'(?i)((?:api[_-]?(?:key|token)|auth[_-]?token|access[_-]?token|subscription[_-]?key|password|secret|token)\s*[:=]\s*)["\x27][^"\x27]*["\x27]',r'\1"[REDACTED]"',value)
    def digest(value):return hashlib.sha256(value.encode()).hexdigest()
    sources={}
    for name in ('integrations/ai_call_page.html','integrations/weather_call_page.html',
        'integrations/movio_confirmation_flow.py','integrations/movio_required_weather.py',
        'integrations/movio_friendly_weather.py'):
        path=root/name;item=dict(path=name,exists=path.is_file())
        if path.is_file():
            source=path.read_text(encoding='utf-8');sources[name]=source
            item.update(bytes=path.stat().st_size,sha256=digest(source))
        report['files'].append(item)
    if isinstance(active,str):
        report['runtime']['bridge_html']=dict(chars=len(active),sha256=digest(active),
            matches_files={name:source==active for name,source in sources.items() if name.endswith('.html')})
    else:report['runtime']['bridge_html']=dict(available=False)

    # Read the local call-page route only; never open a session or call a model.
    served=None
    try:
        import requests
        session=requests.Session();session.trust_env=False
        try:
            response=session.get('http://127.0.0.1:8000/call-page',timeout=(1,3),allow_redirects=False)
            report['served_call_page']['http_status']=response.status_code
            if response.status_code==200 and '<html' in response.text.lower():
                served=response.text
                report['served_call_page'].update(chars=len(served),sha256=digest(served),matches_bridge_html=served==active)
        finally:session.close()
    except ImportError:report['served_call_page']['note']='requests unavailable; served-page read skipped'
    except Exception as error:report['served_call_page']['error']=type(error).__name__
    html=served if isinstance(served,str) else active if isinstance(active,str) else sources.get('integrations/ai_call_page.html','')
    report['html']['inspected_source']='local /call-page' if served is not None else 'bridge.HTML' if isinstance(active,str) else 'saved ai_call_page.html'
    if not html:
        report['html']['note']='No call page found. Run the existing voice setup first.'
        print(json.dumps(report,ensure_ascii=False,indent=2));return report

    class Parts(HTMLParser):
        def __init__(self):super().__init__();self.ids=Counter();self.parts=[];self.current=None
        def handle_starttag(self,tag,attrs):
            values=dict(attrs)
            if values.get('id'):self.ids[values['id']]+=1
            if tag in ('style','script'):
                self.current=dict(tag=tag,id=values.get('id'),type=values.get('type'),src=values.get('src'),text='')
                self.parts.append(self.current)
        def handle_data(self,text):
            if self.current is not None:self.current['text']+=text
        def handle_endtag(self,tag):
            if self.current is not None and tag==self.current['tag']:self.current=None
    parsed=Parts();parsed.feed(html)
    required=('log','lang','call','hang','interrupt','bookMode','cancelMode','send','readSummary')
    report['html']['control_counts']={key:parsed.ids[key] for key in required}
    report['html']['duplicate_ids']={key:value for key,value in parsed.ids.items() if value>1}
    report['html']['weather_ids']=[key for key in parsed.ids if re.search(r'weather|movioWx|wxCall',key,re.I)]
    scripts=[part for part in parsed.parts if part['tag']=='script']
    styles=[part for part in parsed.parts if part['tag']=='style']
    report['html']['external_scripts']=[urlsplit(part['src']).path for part in scripts if part.get('src')]
    marker=re.compile(r'movioCallWeather|movioRenderWeatherCard|MOVIO_CALL_WEATHER|weatherPopup|callWeatherPopup',re.I)
    for group,parts,limit in (('styles',styles,8000),('scripts',scripts,15000)):
        found=[part for part in parts if marker.search(part['text']) or marker.search(part.get('id') or '')]
        report['weather_blocks'][group]=[dict(id=part['id'],chars=len(part['text']),complete=len(part['text'])<=limit,
            source=redact(part['text'][:limit])) for part in found]
    core='\n'.join(part['text'] for part in scripts if not part.get('src') and
        re.search(r'function\s+(?:drainSpeech|queueSpeech|play|log|send)\s*\(|socket\.onmessage',part['text']))

    # A quote/comment-aware brace scan captures existing JS functions exactly.
    def block(source,start,limit):
        opening=source.find('{',start)
        if opening<0:return None
        depth=0;quote=None;comment=None;escaped=False;i=opening
        while i<len(source):
            c=source[i];following=source[i:i+2]
            if comment=='line':
                if c=='\n':comment=None
            elif comment=='block':
                if following=='*/':comment=None;i+=1
            elif quote:
                if escaped:escaped=False
                elif c=='\\':escaped=True
                elif c==quote:quote=None
            elif following=='//':comment='line';i+=1
            elif following=='/*':comment='block';i+=1
            elif c in ('"',"'",'`'):quote=c
            elif c=='{':depth+=1
            elif c=='}':
                depth-=1
                if depth==0:
                    value=source[start:i+1]
                    return dict(chars=len(value),complete=len(value)<=limit,source=redact(value[:limit]))
            i+=1
        return dict(complete=False,source=redact(source[start:start+limit]),note='Boundary could not be resolved; inspect before patching.')
    for name in ('log','send','stopAudio','play','queueSpeech','drainSpeech','offerReview','typed'):
        patterns=(r'(?:async\s+)?function\s+'+name+r'\s*\(',r'\b'+name+r'\s*=\s*(?:async\s+)?function\s*\(')
        matches=[]
        for pattern in patterns:matches.extend(re.finditer(pattern,core))
        snippets=[block(core,match.start(),5500) for match in sorted(matches,key=lambda m:m.start())[:3]]
        report['call_handlers'][name]=[item for item in snippets if item]
    handler=re.search(r'\bsocket\.onmessage\s*=',core)
    report['call_handlers']['socket_onmessage']=block(core,handler.start(),7500) if handler else None
    report['html']['weather_layout_flags']=dict(
        card_inserted_before_log='log.before(card)' in html,
        mobile_single_column=bool(re.search(r'@media[^}]*max-width\s*:\s*420px[\s\S]{0,350}grid-template-columns\s*:\s*minmax\(0,1fr\)',html)),
        playback_ended_event='playback_ended' in html,
        weather_summary_kind='summary_kind' in html and "'weather'" in html)
    node=shutil.which('node');syntax=[]
    if node:
        for index,part in enumerate(scripts):
            if part.get('src') or not part['text'].strip():continue
            result=subprocess.run([node,'--check'],input=part['text'],text=True,capture_output=True,timeout=12)
            error=next((line.strip() for line in result.stderr.splitlines() if line.strip().startswith(('SyntaxError:','Error:'))),'FAILED')
            syntax.append(dict(script=index,id=part['id'],syntax='PASS' if result.returncode==0 else error))
    report['html']['javascript_syntax']=syntax or [dict(note='Node.js unavailable or no inline scripts')]

    endpoint=getattr(bridge,'_wx_endpoint',None)
    engine=getattr(endpoint,'__globals__',{})
    for key,value in list(engine.items()):
        if key.startswith('_MOVIO_') and key.endswith('_VERSION') and isinstance(value,(str,int,float,bool)):
            report['runtime'][key]=value
    def describe(function,max_chars=4000):
        if not isinstance(function,(types.FunctionType,types.MethodType)):return dict(available=False)
        item=dict(module=getattr(function,'__module__',''),name=getattr(function,'__qualname__',''),
            code_file=getattr(getattr(function,'__code__',None),'co_filename',None))
        try:
            value=inspect.getsource(function);item.update(chars=len(value),complete=len(value)<=max_chars,source=redact(value[:max_chars]))
        except (OSError,TypeError,IOError):item['note']='Dynamic function source unavailable; no function was executed.'
        return item
    socket_class=engine.get('ConfirmationSocket');sender=getattr(socket_class,'send_json',None)
    report['backend_sources']['ConfirmationSocket.send_json']=describe(sender,5000)
    sender_space=getattr(sender,'__globals__',{})
    report['backend_sources']['call_weather_card']=describe(sender_space.get('call_weather_card'),4500)
    for name in ('display_reply','advisory','weather_question','speech_segments','prepare_turn'):
        report['backend_sources'][name]=describe(engine.get(name),4500)
    report['backend_sources']['tanglish_speech_text']=describe(getattr(bridge,'tanglish_speech_text',None),2500)
    print('AI CALL WEATHER POPUP — READ-ONLY DIAGNOSTICS')
    print(json.dumps(report,ensure_ascii=False,indent=2))
    print('No files, active handlers, caches, conversations or booking records were changed.')
    print('No weather/STT/LLM/TTS generation was requested. Share this output before the popup modification.')
    return report

MOVIO_POPUP_AUDIT=movio_inspect_call_weather_popup()


AI CALL WEATHER POPUP — READ-ONLY DIAGNOSTICS
{
  "step": "Call weather popup 1 — INSPECTION ONLY",
  "root": "/kaggle/working/MOVIO_MAIN_WEBSITE",
  "files": [
    {
      "path": "integrations/ai_call_page.html",
      "exists": true,
      "bytes": 49718,
      "sha256": "7398015d697857b9387340ca149266cef1ad73684032d5db67d2a5e418c19c40"
    },
    {
      "path": "integrations/weather_call_page.html",
      "exists": true,
      "bytes": 49718,
      "sha256": "7398015d697857b9387340ca149266cef1ad73684032d5db67d2a5e418c19c40"
    },
    {
      "path": "integrations/movio_confirmation_flow.py",
      "exists": true,
      "bytes": 22599,
      "sha256": "1b6827aa18c411f715c96f448c3026dff23bc5b766cf64fa4e3ddaaa3e552589"
    },
    {
      "path": "integrations/movio_required_weather.py",
      "exists": true,
      "bytes": 16308,
      "sha256": "0506b8f43f455d57def44309aa6c6e289bc594eccff712a3f9b3759c1883071a"
    },
    {
      "path": "integrations/movio_friendly_weather.py",
   

In [126]:
# MOVIO AI CALL WEATHER POPUP — STEP 2.
# End the current AI Call. Paste this ENTIRE file into ONE Kaggle code cell
# after the existing voice/weather setup and before evaluation.
# Uses only the existing authenticated weather payload. No package installs.
def movio_enable_call_weather_popup():
    import ast
    import hashlib
    import json
    import os
    import re
    import shutil
    import subprocess
    import tempfile
    import textwrap
    import types
    from collections import Counter
    from datetime import datetime
    from html.parser import HTMLParser
    from pathlib import Path

    version='call-weather-popup-1.0'
    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    b=globals().get('bridge')
    engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',{})
    active=getattr(b,'HTML',None)
    display=engine.get('display_reply')
    if not isinstance(active,str) or not isinstance(display,types.FunctionType):
        raise RuntimeError('Run the existing MOVIO voice/weather setup first. No changes were made.')
    node=shutil.which('node')
    if not node:
        raise RuntimeError('Node.js is required for offline JavaScript checks. No changes were made.')
    css=r'''/* MOVIO_CALL_WEATHER_POPUP_V1: non-modal; no backdrop or microphone changes. */
.phone #movioCallWeather[hidden],.phone #movioWeatherTab[hidden]{display:none!important}
.phone #movioCallWeather{position:fixed;z-index:90;inset:auto 14px auto auto;top:max(12px,env(safe-area-inset-top));width:min(352px,calc(100vw - 28px));max-height:min(310px,42vh);max-height:min(310px,42dvh);display:flex;flex-direction:column;margin:0;padding:0;background:#fff;color:#263443;border:1px solid #e1e6ec;border-radius:18px;box-shadow:0 14px 44px #24364a24;white-space:normal;font-family:inherit;overflow:hidden;isolation:isolate}
#movioCallWeather *{box-sizing:border-box;min-width:0}
#movioCallWeather .movio-wx-header{display:flex;align-items:center;gap:9px;padding:11px 13px 9px;border-bottom:1px solid #edf0f4;flex:none;background:#fffdf6}
#movioCallWeather .movio-wx-heading{margin:0;font-size:14px;font-weight:750;line-height:1.4;flex:1}
#movioCallWeather .movio-wx-dot{width:8px;height:8px;border-radius:50%;background:#e2ad1c;box-shadow:0 0 0 4px #fff0bc;flex:none;margin:0 3px}
.phone #movioCallWeather .movio-wx-minimize{display:grid;place-items:center;flex:none;width:34px;height:34px;min-height:34px;padding:0;border:1px solid #e2e7ec;border-radius:10px;background:#fff;color:#526171;font:23px/1 system-ui;cursor:pointer}
#movioCallWeather .movio-wx-content{overflow:auto;overscroll-behavior:contain;padding:10px 12px 12px;min-height:0}
#movioCallWeather .movio-wx-subtitle{margin:0 0 8px;color:#7b8794;font-size:10px;line-height:1.45}
#movioCallWeather .movio-wx-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px}
#movioCallWeather .movio-wx-point{padding:9px 10px;background:#f6f8fb;border:1px solid #e8edf3;border-radius:12px;min-width:0}
#movioCallWeather .movio-wx-role{font-size:9px;line-height:1.5;font-weight:750;color:#748292;text-transform:uppercase;letter-spacing:.035em}
#movioCallWeather .movio-wx-place{font-size:12px;line-height:1.4;font-weight:700;margin:2px 0 7px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
#movioCallWeather .movio-wx-main{display:flex;align-items:center;justify-content:space-between;gap:4px}
#movioCallWeather .movio-wx-icon{display:flex;flex:none;width:30px;height:30px;color:#68819b}
#movioCallWeather .movio-wx-icon svg{width:30px;height:30px;fill:none;stroke:currentColor;stroke-width:1.65;stroke-linecap:round;stroke-linejoin:round}
#movioCallWeather .movio-wx-temperature{font-size:24px;line-height:1.25;font-weight:750;letter-spacing:-.7px;white-space:nowrap}
#movioCallWeather .movio-wx-temperature small{font-size:11px;letter-spacing:0;font-weight:500;color:#73808f;margin-left:2px}
#movioCallWeather .movio-wx-condition{font-size:10px;line-height:1.4;font-weight:650;color:#536b83;margin:5px 0 3px;overflow-wrap:anywhere}
#movioCallWeather .movio-wx-rain{font-size:10px;line-height:1.5;color:#657989;overflow-wrap:anywhere}
#movioCallWeather [data-condition=clear] .movio-wx-icon{color:#bc8d13}
#movioCallWeather [data-condition=rain] .movio-wx-icon{color:#3282ad}
#movioCallWeather [data-condition=storm] .movio-wx-icon{color:#776195}
#movioCallWeather .movio-wx-tip{margin:9px 0 0;padding:8px 9px;background:#fff7dc;border-radius:9px;color:#705b26;font-size:11px;line-height:1.5;overflow-wrap:anywhere}
#movioCallWeather .movio-wx-footer{margin:7px 0 0;color:#84909e;font-size:9px;line-height:1.5;overflow-wrap:anywhere}
.phone #movioWeatherTab{position:fixed;z-index:89;right:14px;top:max(12px,env(safe-area-inset-top));display:flex;align-items:center;gap:7px;width:auto;max-width:calc(100vw - 28px);min-height:40px;padding:9px 12px;margin:0;border:1px solid #ecd898;border-radius:999px;background:#fff7db;color:#6b5420;box-shadow:0 5px 18px #21334a14;font:700 12px/1.4 system-ui;cursor:pointer;white-space:nowrap}
#movioWeatherTab svg{width:19px;height:19px;fill:none;stroke:currentColor;stroke-width:1.7;stroke-linecap:round;stroke-linejoin:round;flex:none}
.phone #movioCallWeather button:focus-visible,.phone #movioWeatherTab:focus-visible{outline:3px solid #d59d00;outline-offset:-3px}
@media(min-width:1150px){.phone #movioCallWeather,.phone #movioWeatherTab{right:20px;top:24px}}
@media(max-width:360px){.phone #movioCallWeather{right:10px;width:calc(100vw - 20px)}#movioCallWeather .movio-wx-content{padding:8px 10px 10px}#movioCallWeather .movio-wx-point{padding:8px}#movioCallWeather .movio-wx-temperature{font-size:22px}}
@media(max-height:460px){.phone #movioCallWeather{max-height:45dvh}#movioCallWeather .movio-wx-content{padding-top:7px}}
'''
    javascript=r'''/* MOVIO_CALL_WEATHER_POPUP_V1: one renderer, existing response and playback only. */
(()=>{
 'use strict';
 const transcript=document.getElementById('log'),language=document.getElementById('lang');
 if(!transcript||!language||window.movioCallWeatherUI)return;
 const panel=transcript.closest('.phone');if(!panel)return;
 const card=document.createElement('section');card.id='movioCallWeather';card.hidden=true;
 card.setAttribute('role','region');card.setAttribute('aria-labelledby','movioWeatherTitle');
 const tab=document.createElement('button');tab.id='movioWeatherTab';tab.type='button';tab.hidden=true;
 tab.setAttribute('aria-controls',card.id);tab.setAttribute('aria-expanded','false');
 panel.append(card,tab);
 let current=null,currentKey='',session=null,pending=null,expanded=false;
 const opened=new Set(),printed=new Set();
 // EXISTING_WEATHER_HELPERS: inserted here at build time, not loaded externally.
 const copy={
  English:{title:'Your trip weather',subtitle:'Both locations at your pickup time',pickup:'Pickup',destination:'Destination',rain:'rain chance',missing:'Not reported',forecast:'Forecast',
   clear:'Sunny',mainlyClear:'Mostly clear',partlyCloudy:'Partly cloudy',cloudy:'Cloudy',fog:'Foggy',drizzle:'Drizzle',rainy:'Rainy',heavyRain:'Heavy rain',freezingRain:'Freezing rain',snow:'Snow',showers:'Rain showers',storm:'Thunderstorms',hail:'Thunderstorms with hail',unknown:'Condition not reported'},
  Tamil:{title:'உங்கள் பயண வானிலை',subtitle:'புறப்படும் நேரத்தில் இரு இடங்களின் முன்னறிவிப்பு',pickup:'புறப்படும் இடம்',destination:'சேரும் இடம்',rain:'மழை வாய்ப்பு',missing:'தகவல் இல்லை',forecast:'முன்னறிவிப்பு',
   clear:'வெயில்',mainlyClear:'தெளிவான வானம்',partlyCloudy:'ஓரளவு மேகமூட்டம்',cloudy:'மேகமூட்டம்',fog:'மூடுபனி',drizzle:'தூறல்',rainy:'மழை',heavyRain:'கனமழை',freezingRain:'உறையும் மழை',snow:'பனிப்பொழிவு',showers:'மழைச் சாரல்',storm:'இடியுடன் மழை',hail:'இடியுடன் ஆலங்கட்டி மழை',unknown:'வானிலை தகவல் இல்லை'},
  Tanglish:{title:'Unga trip weather',subtitle:'Pickup time-la rendu places-oda forecast',pickup:'Pickup',destination:'Destination',rain:'rain chance',missing:'Not reported',forecast:'Forecast',
   clear:'Sunny',mainlyClear:'Mostly clear',partlyCloudy:'Partly cloudy',cloudy:'Cloudy',fog:'Foggy',drizzle:'Drizzle',rainy:'Rainy',heavyRain:'Heavy rain',freezingRain:'Freezing rain',snow:'Snow',showers:'Rain showers',storm:'Thunderstorms',hail:'Thunderstorms with hail',unknown:'Condition not reported'}
 };
 const drawings={
  clear:'<circle cx="20" cy="20" r="6"/><path d="M20 4v4m0 24v4M4 20h4m24 0h4M9 9l3 3m16 16l3 3M9 31l3-3m16-16l3-3"/>',
  cloudy:'<path d="M11 29h18a7 7 0 0 0 .2-14 10 10 0 0 0-19-1A7.6 7.6 0 0 0 11 29Z"/>',
  rain:'<path d="M10 24h20a6 6 0 0 0-1-12 9 9 0 0 0-17-1 6.6 6.6 0 0 0-2 13ZM13 29l-2 5m10-5-2 5m10-5-2 5"/>',
  storm:'<path d="M10 24a6.6 6.6 0 0 1 2-13 9 9 0 0 1 17 1 6 6 0 0 1 1 12M22 21l-7 9h7l-4 7"/>',
  fog:'<path d="M10 23h20a6 6 0 0 0-1-12 9 9 0 0 0-17-1 6.6 6.6 0 0 0-2 13M7 29h26M11 34h18"/>',
  snow:'<path d="M10 24h20a6 6 0 0 0-1-12 9 9 0 0 0-17-1 6.6 6.6 0 0 0-2 13M13 30h.1M20 34h.1M28 30h.1"/>'
 };
 function condition(code){
  if(code===0)return ['clear','clear'];if(code===1)return ['mainlyClear','clear'];
  if(code===2)return ['partlyCloudy','cloudy'];if(code===3)return ['cloudy','cloudy'];
  if([45,48].includes(code))return ['fog','fog'];if([51,53,55,56,57].includes(code))return ['drizzle','rain'];
  if([61,63].includes(code))return ['rainy','rain'];if([65,82].includes(code))return ['heavyRain','rain'];
  if([66,67].includes(code))return ['freezingRain','rain'];if([71,73,75,77,85,86].includes(code))return ['snow','snow'];
  if([80,81].includes(code))return ['showers','rain'];if(code===95)return ['storm','storm'];
  if([96,99].includes(code))return ['hail','storm'];return ['unknown','cloudy'];
 }
 function node(tag,cls,text){const element=document.createElement(tag);if(cls)element.className=cls;if(text!==undefined)element.textContent=text;return element;}
 function numeric(value,lower,upper){return typeof value==='number'&&Number.isFinite(value)&&value>=lower&&value<=upper;}
 function format(value){return new Intl.NumberFormat('en-IN',{maximumFractionDigits:1}).format(value);}
 function date(value,lang){try{const instant=new Date(value);if(Number.isNaN(instant.getTime()))return '';return new Intl.DateTimeFormat(lang==='Tamil'?'ta-IN':'en-IN',{timeZone:'Asia/Kolkata',day:'numeric',month:'short',hour:'numeric',minute:'2-digit',hour12:true}).format(instant);}catch{return '';}}

 const extra={
  English:{tab:'Trip weather',minimize:'Minimize weather',tip:'Travel tip',
   rainTip:'Carry an umbrella and protect your belongings from rain.',
   stormTip:'Check local warnings and take shelter if thunder is nearby.',
   heatTip:'Carry drinking water and use sun protection.',
   clearTip:'Keep water handy and use sun protection outdoors.',
   fogTip:'Take care around the pickup point when visibility is low.',
   snowTip:'Wear suitable layers and take care on slippery surfaces.',
   neutralTip:'Keep your travel essentials handy before leaving.'},
  Tamil:{tab:'பயண வானிலை',minimize:'வானிலையைச் சுருக்கு',tip:'பயணக் குறிப்பு',
   rainTip:'குடை எடுத்துக்கொண்டு உங்கள் பொருட்களை மழையிலிருந்து பாதுகாத்துக்கொள்ளுங்கள்.',
   stormTip:'உள்ளூர் எச்சரிக்கைகளைக் கவனியுங்கள். இடி அருகில் கேட்டால் பாதுகாப்பான இடத்தில் இருங்கள்.',
   heatTip:'குடிநீர் எடுத்துக்கொள்ளுங்கள். வெயிலிலிருந்து பாதுகாத்துக்கொள்ளுங்கள்.',
   clearTip:'குடிநீர் வைத்துக்கொள்ளுங்கள். வெளியே செல்லும்போது வெயிலிலிருந்து பாதுகாத்துக்கொள்ளுங்கள்.',
   fogTip:'பார்வை தெளிவாக இல்லாவிட்டால் வாகனத்தில் ஏறுமிடத்தில் கவனமாக இருங்கள்.',
   snowTip:'பொருத்தமான உடை அணியுங்கள். வழுக்கும் இடங்களில் கவனமாக இருங்கள்.',
   neutralTip:'கிளம்பும் முன் பயணத்திற்குத் தேவையான பொருட்களை எடுத்துக்கொள்ளுங்கள்.'},
  Tanglish:{tab:'Trip weather',minimize:'Weather-a minimize pannunga',tip:'Travel tip',
   rainTip:'Umbrella eduthukkonga. Unga belongings-a rain-la irundhu protect pannikkonga.',
   stormTip:'Local warnings-a check pannunga. Idi pakkathula ketta safe place-la irunga.',
   heatTip:'Drinking water eduthukkonga. Sun protection use pannunga.',
   clearTip:'Water vachukkonga. Velila pogumbodhu sun protection use pannunga.',
   fogTip:'Visibility kammiya irundha pickup place-la careful-ah irunga.',
   snowTip:'Suitable clothes pottukkonga. Slippery places-la careful-ah irunga.',
   neutralTip:'Kelambura munnaadi travel essentials-a eduthukkonga.'}
 };
 function bounded(set,key){set.add(key);if(set.size>192)set.delete(set.values().next().value);}
 function valid(data){
  return !!data&&data.version==='call-weather-card-v1'&&Array.isArray(data.points)&&data.points.some(p=>
   p&&['pickup','destination'].includes(p.role)&&(numeric(p.temperature_c,-100,70)||numeric(p.rain_probability_pct,0,100)||numeric(p.weather_code,0,99)));
 }
 function identity(data){return JSON.stringify([data.pickup_at,data.points.filter(p=>p&&['pickup','destination'].includes(p.role)).map(p=>
  [p.role,p.place,p.valid_at,p.weather_code,p.temperature_c,p.rain_probability_pct])]);}
 function labels(){return {...(copy[language.value]||copy.English),...(extra[language.value]||extra.English)};}
 function show(value){
  expanded=!!value&&!!current;card.hidden=!expanded;tab.hidden=!current||expanded;
  tab.setAttribute('aria-expanded',String(expanded));
 }
 function hide(){current=null;currentKey='';pending=null;show(false);card.replaceChildren();}
 function reset(id){hide();session=id;opened.clear();printed.clear();}
 function minimize(){show(false);}
 function tip(points,text){
  const icons=points.map(p=>condition(p.weather_code)[1]);
  const rain=points.some(p=>numeric(p.rain_probability_pct,0,100)&&p.rain_probability_pct>=50);
  if(icons.includes('storm'))return text.stormTip;
  if(icons.includes('rain')||rain)return text.rainTip;
  if(points.some(p=>numeric(p.temperature_c,-100,70)&&p.temperature_c>=32))return text.heatTip;
  if(icons.includes('snow'))return text.snowTip;
  if(icons.includes('fog'))return text.fogTip;
  if(icons.includes('clear'))return text.clearTip;
  return text.neutralTip;
 }
 function render(){
  if(!current)return;
  const text=labels(),data=current;
  const focusClose=card.contains(document.activeElement);card.replaceChildren();
  const header=node('div','movio-wx-header'),dot=node('span','movio-wx-dot');dot.setAttribute('aria-hidden','true');
  const heading=node('h3','movio-wx-heading',text.title);heading.id='movioWeatherTitle';
  const close=node('button','movio-wx-minimize','−');close.type='button';close.setAttribute('aria-label',text.minimize);close.title=text.minimize;
  close.addEventListener('click',()=>{minimize();tab.focus({preventScroll:true});});
  header.append(dot,heading,close);card.append(header);
  const content=node('div','movio-wx-content');content.tabIndex=0;
  content.append(node('p','movio-wx-subtitle',text.subtitle));
  const grid=node('div','movio-wx-grid');
  const points=['pickup','destination'].map(role=>data.points.find(p=>p&&p.role===role)).filter(Boolean);
  for(const point of points){
   const key=condition(point.weather_code),box=node('article','movio-wx-point');box.dataset.condition=key[1];
   const place=node('div','movio-wx-place',String(point.place||text[point.role]).slice(0,120));place.title=place.textContent;
   box.append(node('div','movio-wx-role',text[point.role]),place);
   const main=node('div','movio-wx-main'),icon=node('span','movio-wx-icon');icon.setAttribute('aria-hidden','true');
   icon.innerHTML='<svg viewBox="0 0 40 40" aria-hidden="true">'+(drawings[key[1]]||drawings.cloudy)+'</svg>';
   const temperature=node('div','movio-wx-temperature');
   temperature.append(node('span','',numeric(point.temperature_c,-100,70)?format(point.temperature_c):'—'),node('small','','°C'));
   main.append(icon,temperature);box.append(main,node('div','movio-wx-condition',text[key[0]]));
   if(numeric(point.rain_probability_pct,0,100))box.append(node('div','movio-wx-rain',format(point.rain_probability_pct)+'% '+text.rain));
   grid.append(box);
  }
  content.append(grid);
  const suggestion=node('p','movio-wx-tip',tip(points,text));suggestion.setAttribute('aria-label',text.tip);content.append(suggestion);
  content.append(node('p','movio-wx-footer',[String(data.source||text.forecast).slice(0,60),date(data.pickup_at,language.value)].filter(Boolean).join(' · ')));
  card.append(content);
  tab.replaceChildren();const symbol=node('span','');symbol.setAttribute('aria-hidden','true');
  symbol.innerHTML='<svg viewBox="0 0 40 40">'+drawings.cloudy+'</svg>';
  tab.append(symbol,node('span','',text.tab));tab.setAttribute('aria-label',text.tab);
  if(focusClose)close.focus({preventScroll:true});
 }
 function receive(data){
  if(!valid(data)){hide();return;}
  const key=identity(data),changed=key!==currentKey;
  current=data;currentKey=key;render();
  if(!opened.has(key)){bounded(opened,key);show(true);}
  else if(changed)show(false);
  else show(expanded);
 }
 function reply(message,id){
  if(session!==id)reset(id);
  receive(message.weather_card||null);
  pending=current&&(message.streaming||message.payload)?{id,epoch:message.epoch,nonce:message.summary_nonce||null,key:currentKey}:null;
 }
 function shouldLog(message,id){
  // De-duplicate presentation only. Never consume a reply/ack/booking action.
  if(!valid(message.weather_card)||!message.text)return true;
  const weatherOnly=message.summary_kind==='weather'||message.stage==='weather_advisory';
  const key=JSON.stringify([id,language.value,identity(message.weather_card),weatherOnly?'weather':message.epoch,
   weatherOnly?'':message.summary_nonce||'',message.text]);
  if(printed.has(key))return false;
  bounded(printed,key);return true;
 }
 function playbackFinished(message,id,ended){
  if(!ended||!pending||pending.id!==id||pending.epoch!==message.epoch||
   pending.nonce!==(message.summary_nonce||null)||pending.key!==currentKey)return;
  pending=null;minimize();
 }
 function event(message,id){
  if(session!==null&&id!==session)return;
  if(message.event==='interrupted'||message.event==='error')hide();
 }
 window.movioCallWeatherUI=Object.freeze({version:'call-weather-popup-1.0',reply,shouldLog,playbackFinished,event,hide,
  state:()=>({available:!!current,expanded,key:currentKey,pending:!!pending})});
 // Backward-compatible display entry point; uses the same sole renderer.
 window.movioRenderWeatherCard=data=>receive(data);
 tab.addEventListener('click',()=>show(true));
 card.addEventListener('keydown',e=>{if(e.key==='Escape'){minimize();tab.focus({preventScroll:true});}});
 language.addEventListener('change',()=>{if(current)render();});
 for(const id of ['call','hang','interrupt','bookMode','cancelMode'])document.getElementById(id)?.addEventListener('click',hide);
 for(const id of ['manual','send','readSummary'])document.getElementById(id)?.addEventListener('focus',minimize);
 // A programmatic interrupt changes the existing status; no socket wrapper.
 const status=document.getElementById('status'),observer=new MutationObserver(()=>{
  if(typeof awaitInterrupt==='boolean'&&awaitInterrupt)hide();
  else if(typeof connected==='boolean'&&!connected)hide();
 });
 if(status)observer.observe(status,{childList:true,subtree:true,characterData:true});
 function onScroll(){if(expanded)minimize();}
 window.addEventListener('scroll',onScroll,{passive:true});
 window.addEventListener('beforeunload',()=>{observer.disconnect();window.removeEventListener('scroll',onScroll);},{once:true});
})();
'''
    old_display_source='def display_reply(state, text):\n    # Forecast details are spoken only; the app shows the booking dialogue.\n    report = advisory(state)\n    return text.replace(report, "", 1).strip()\n'
    new_display_source='def display_reply(state, text):\n    """MOVIO call weather popup v1: retain full display text before speech conversion."""\n    return str(text).strip()\n'
    state_tests=r'''const assert=require('node:assert/strict'),vm=require('node:vm'),fs=require('node:fs');
let checks=0;const check=(v,label)=>{assert.ok(v,label);checks++;};
class Element{
 constructor(tag,doc){this.tagName=tag;this.doc=doc;this.children=[];this.attrs={};this.events={};this.dataset={};this.hidden=false;this._text='';}
 set id(value){this._id=value;this.doc.ids[value]=this;}get id(){return this._id;}
 set textContent(value){this._text=String(value);this.children=[];}get textContent(){return this._text+this.children.map(c=>c.textContent||'').join(' ');}
 set innerHTML(value){this._html=value;}get innerHTML(){return this._html||'';}
 append(...values){for(const value of values){value.parentElement=this;this.children.push(value);}}
 replaceChildren(...values){this.children=[];this._text='';this.append(...values);}
 setAttribute(name,value){this.attrs[name]=String(value);}
 addEventListener(name,listener){(this.events[name]||=[]).push(listener);}
 focus(){this.doc.activeElement=this;for(const f of this.events.focus||[])f({});}
 contains(node){return this===node||this.children.some(child=>child.contains?.(node));}
 closest(){return this.doc.panel;}
 click(){for(const f of this.events.click||[])f({});}
}
const document={ids:{},activeElement:null,createElement(tag){return new Element(tag,this);},getElementById(id){return this.ids[id]||null;}};
document.panel=new Element('main',document);
for(const id of ['log','lang','call','hang','interrupt','bookMode','cancelMode','manual','send','readSummary','status']){
 const el=new Element('div',document);el.id=id;document.panel.append(el);
}
document.ids.lang.value='English';
const window={document,connected:true,awaitInterrupt:false,events:{},addEventListener(name,fn){(this.events[name]||=[]).push(fn);},removeEventListener(){},
 MutationObserver:class{observe(){}disconnect(){}}};
window.window=window;const context=vm.createContext(window);
vm.runInContext(fs.readFileSync(0,'utf8'),context);
const ui=window.movioCallWeatherUI,card=document.ids.movioCallWeather,tab=document.ids.movioWeatherTab;
const sample={version:'call-weather-card-v1',source:'Open-Meteo',pickup_at:'2030-10-19T19:00:00+05:30',points:[
 {role:'pickup',place:'Chennai Central',weather_code:3,temperature_c:27.3,rain_probability_pct:42,valid_at:'2030-10-19T19:00'},
 {role:'destination',place:'Tambaram',weather_code:61,temperature_c:26.5,rain_probability_pct:75,valid_at:'2030-10-19T19:00'}]};
const message={event:'reply',epoch:5,summary_nonce:'weather-5',summary_kind:'weather',streaming:true,text:'Actual full advisory.',weather_card:sample};
ui.reply(message,1);check(!card.hidden&&tab.hidden,'first forecast opens popup');
check(card.textContent.includes('27.3')&&card.textContent.includes('75%'),'actual independent point values retained');
check(card.textContent.includes('Carry an umbrella'),'practical rain suggestion');
check(ui.shouldLog(message,1),'first advisory appears in transcript');check(!ui.shouldLog(message,1),'duplicate advisory is suppressed');
ui.playbackFinished(message,1,false);check(!card.hidden,'audio failure cannot claim completion');
ui.playbackFinished({...message,epoch:4},1,true);check(!card.hidden,'stale audio cannot minimize a current forecast');
ui.playbackFinished(message,1,true);check(card.hidden&&!tab.hidden,'actual completion minimizes');
ui.reply({...message,epoch:6,summary_nonce:'next',summary_kind:'final',text:'Booking summary.'},1);
check(card.hidden&&!tab.hidden,'same forecast in confirmation does not reopen');
check(ui.shouldLog({...message,epoch:6,summary_kind:'final',text:'Booking summary.'},1),'booking summary is not suppressed');
tab.click();check(!card.hidden,'tab reopens');
const close=card.children[0].children[2];close.click();check(card.hidden&&!tab.hidden,'manual minimize');
ui.reply({...message,epoch:7,weather_card:{...sample,pickup_at:'2030-10-20T19:00:00+05:30'}},1);
check(!card.hidden,'new travel time opens a new forecast');
for(const lang of ['Tamil','Tanglish','English']){
 document.ids.lang.value=lang;for(const f of document.ids.lang.events.change||[])f({});
 check(card.textContent.includes(lang==='Tamil'?'மழை வாய்ப்பு':'rain chance'),lang+' values are localized');
 if(lang==='Tanglish')check(!/[\u0B80-\u0BFF]/u.test(card.textContent),'Tanglish popup stays Romanized');
}
for(const [code,tip] of [[0,'sun protection'],[3,'essentials'],[45,'visibility'],[61,'umbrella'],[95,'warnings'],[71,'layers']]){
 const data={...sample,points:[{...sample.points[0],weather_code:code,rain_probability_pct:0,temperature_c:20}]};
 window.movioRenderWeatherCard(data);check(card.textContent.includes(tip),'condition '+code+' suggestion');
}
window.movioRenderWeatherCard({...sample,points:[{...sample.points[0],weather_code:3,temperature_c:35,rain_probability_pct:0}]});
check(card.textContent.includes('drinking water'),'heat precautions use actual temperature');
window.movioRenderWeatherCard({...sample,points:[{...sample.points[0],temperature_c:0,rain_probability_pct:0}]});
check(card.textContent.includes('0%'),'zero rain chance retained');
for(const data of [null,{}, {...sample,points:[]},{...sample,points:[{role:'pickup',weather_code:null,temperature_c:null,rain_probability_pct:null}]}]){
 window.movioRenderWeatherCard(data);check(card.hidden&&tab.hidden,'no valid forecast hides popup and tab');
}
ui.reply(message,2);check(!card.hidden,'new call resets automatic opening');
ui.event({event:'interrupted'},2);check(card.hidden&&tab.hidden,'interrupt hides stale forecast');
ui.reply(message,2);check(card.hidden&&!tab.hidden,'repeated forecast after interruption does not reopen');
document.ids.manual.focus();check(card.hidden,'typing remains accessible');
check(!('send' in window)&&!('fetch' in window)&&!('WebSocket' in window),'no backend request or playback ack is sent');
console.log(JSON.stringify({checks}));
'''
    checks=0
    def check(value,description):
        nonlocal checks
        if not value:raise RuntimeError('Check failed: '+description+'. No integration was activated.')
        checks+=1
    def sha(value):return hashlib.sha256(value.encode('utf-8')).hexdigest()
    def ast_shape(value):return ast.dump(ast.parse(textwrap.dedent(value)),include_attributes=False)
    def signature(code):
        constants=tuple(signature(c) if isinstance(c,types.CodeType) else c for c in code.co_consts)
        return (code.co_code,constants,code.co_names,code.co_varnames,code.co_freevars,
            code.co_cellvars,code.co_argcount,code.co_kwonlyargcount,code.co_posonlyargcount,
            code.co_flags,code.co_exceptiontable)
    def function_code(code,name):
        found=[c for c in code.co_consts if isinstance(c,types.CodeType) and c.co_name==name]
        check(len(found)==1,'one saved '+name+' function')
        return found[0]

    # The existing native Tanglish wrapper remains intact. Change its ORIGINAL
    # display filter in place, so the advisory is printed before speech conversion.
    base=display
    if base.__closure__:
        enclosed={name:cell.cell_contents for name,cell in zip(base.__code__.co_freevars,base.__closure__)}
        check(set(enclosed)=={'original_display','romanize'},'recognized Tanglish display wrapper')
        base=enclosed['original_display']
    check(isinstance(base,types.FunctionType) and not base.__closure__,'original display filter is available')
    display_path=Path(base.__code__.co_filename).resolve()
    check(display_path.is_relative_to(root) and display_path.is_file(),'display source belongs to this MOVIO project')
    before_python=display_path.read_text(encoding='utf-8')
    parsed=ast.parse(before_python)
    functions=[n for n in parsed.body if isinstance(n,ast.FunctionDef) and n.name=='display_reply']
    check(len(functions)==1,'one original display_reply definition')
    fn=functions[0];original=ast.get_source_segment(before_python,fn)
    check(ast_shape(original) in (ast_shape(old_display_source),ast_shape(new_display_source)),
        'recognized display-only weather filter')
    old_compiled=compile(before_python,str(display_path),'exec',dont_inherit=True)
    check(signature(function_code(old_compiled,'display_reply'))==signature(base.__code__),
        'live display filter matches its saved source')
    lines=before_python.splitlines(keepends=True)
    lines[fn.lineno-1:fn.end_lineno]=[new_display_source.strip()+'\n']
    after_python=''.join(lines)
    compiled=compile(after_python,str(display_path),'exec',dont_inherit=True)
    new_code=function_code(compiled,'display_reply')
    check(new_code.co_freevars==base.__code__.co_freevars,'display closure is preserved')
    candidate=types.FunctionType(new_code,base.__globals__,base.__name__,base.__defaults__,base.__closure__)
    for lang,text in (
        ('English','Weather update! Rain chance is 75%. Carry an umbrella. Shall we continue?'),
        ('Tamil','வானிலை தகவல்! 75% மழை வாய்ப்பு. குடை எடுத்துக்கொள்ளுங்கள். தொடரலாமா?'),
        ('Tanglish','Oru weather update! 75% rain chance. Umbrella eduthukkonga. Continue pannalama?')):
        check(candidate({'lang':lang},text)==text,'complete '+lang+' advisory retained for display')
    # No advisory(), romanizer or speech conversion executes during prechecks.
    check('advisory' not in candidate.__code__.co_names,'display cannot regenerate a weather advisory')

    class Parts(HTMLParser):
        def __init__(self):super().__init__();self.ids=Counter();self.scripts=[];self.current=None
        def handle_starttag(self,tag,attrs):
            attrs=dict(attrs)
            if attrs.get('id'):self.ids[attrs['id']]+=1
            if tag=='script':self.current={'src':attrs.get('src'),'text':''};self.scripts.append(self.current)
        def handle_data(self,data):
            if self.current is not None:self.current['text']+=data
        def handle_endtag(self,tag):
            if tag=='script':self.current=None
    def remove_hooks(value):
        return re.sub(r'/\* MOVIO_WX_(REPLY|EVENT|PLAY|STREAM)_BEGIN \*/.*?/\* MOVIO_WX_\1_END \*/','',value,flags=re.S)
    def strip_weather(value):
        for kind in ('STYLE','SCRIPT'):
            value=re.sub(r'<!-- MOVIO_CALL_WEATHER_'+kind+r'_BEGIN -->.*?<!-- MOVIO_CALL_WEATHER_'+kind+r'_END -->','',value,flags=re.S)
        return value
    def block(value,pattern):
        matches=list(re.finditer(pattern,value));check(len(matches)==1,'one existing '+pattern)
        start=matches[0].start();opening=value.find('{',matches[0].end())
        depth=0;quote=None;comment=None;escaped=False;i=opening
        while 0<=i<len(value):
            c=value[i];pair=value[i:i+2]
            if comment=='line':
                if c=='\n':comment=None
            elif comment=='block':
                if pair=='*/':comment=None;i+=1
            elif quote:
                if escaped:escaped=False
                elif c=='\\':escaped=True
                elif c==quote:quote=None
            elif pair=='//':comment='line';i+=1
            elif pair=='/*':comment='block';i+=1
            elif c in ('"',"'",'`'):quote=c
            elif c=='{':depth+=1
            elif c=='}':
                depth-=1
                if depth==0:return start,i+1,value[start:i+1]
            i+=1
        raise RuntimeError('An existing JavaScript boundary could not be verified. No changes were made.')
    def insert_once(value,pattern,addition,where='before'):
        matches=list(re.finditer(pattern,value));check(len(matches)==1,'one playback/reply insertion point')
        m=matches[0];offset=m.start() if where=='before' else m.end()
        return value[:offset]+addition+value[offset:]
    def hook(name,value):return '/* MOVIO_WX_'+name+'_BEGIN */'+value+'/* MOVIO_WX_'+name+'_END */'
    def patch_html(value):
        before=value;value=remove_hooks(value)
        for kind in ('STYLE','SCRIPT'):
            pattern=r'<!-- MOVIO_CALL_WEATHER_'+kind+r'_BEGIN -->.*?<!-- MOVIO_CALL_WEATHER_'+kind+r'_END -->'
            check(len(re.findall(pattern,value,re.S))==1,'one existing weather '+kind.lower()+' block')
            content=css if kind=='STYLE' else javascript
            tag='style' if kind=='STYLE' else 'script'
            block_id='movioCallWeatherStyle' if kind=='STYLE' else 'movioCallWeatherScript'
            replacement='<!-- MOVIO_CALL_WEATHER_'+kind+'_BEGIN --><'+tag+' id="'+block_id+'">'+content+'</'+tag+'><!-- MOVIO_CALL_WEATHER_'+kind+'_END -->'
            value=re.sub(pattern,lambda m:replacement,value,count=1,flags=re.S)
        # All original guards, audio operations and confirmation acks are intact.
        start,end,handler=block(value,r'\bsocket\.onmessage\s*=')
        handler=insert_once(handler,r'if\s*\(\s*m\.text\s*\)\s*log\(\s*[\x27\"]AI[\x27\"]\s*,\s*m\.text\s*\)\s*;',
            hook('REPLY','window.movioCallWeatherUI?.reply(m,id);if(window.movioCallWeatherUI?.shouldLog(m,id)!==false)'))
        handler=insert_once(handler,r'if\s*\(\s*m\.event\s*===\s*[\x27\"]ready[\x27\"]\s*\)',
            hook('EVENT','window.movioCallWeatherUI?.event(m,id);'))
        value=value[:start]+handler+value[end:]
        start,end,handler=block(value,r'async\s+function\s+play\s*\(')
        handler=insert_once(handler,r'busy\s*=\s*false\s*;\s*playing\s*=\s*false\s*;',
            hook('PLAY','window.movioCallWeatherUI?.playbackFinished(m,id,ended);'),where='after')
        value=value[:start]+handler+value[end:]
        start,end,handler=block(value,r'async\s+function\s+drainSpeech\s*\(')
        handler=insert_once(handler,r'if\s*\(\s*current\.next\s*===\s*current\.total\s*\)\s*\{\s*speech\s*=\s*null\s*;\s*playing\s*=\s*false\s*;\s*busy\s*=\s*false\s*;',
            hook('STREAM','window.movioCallWeatherUI?.playbackFinished(current.meta,current.id,true);'),where='after')
        value=value[:start]+handler+value[end:]
        check(strip_weather(remove_hooks(value))==strip_weather(remove_hooks(before)),
            'all original UI, booking, cancellation and playback source remains intact')
        part=Parts();part.feed(value)
        for control in ('log','lang','call','hang','interrupt','bookMode','cancelMode','send','readSummary','manual'):
            check(part.ids[control]==1,'one original '+control+' control')
        check(not any(n>1 for n in part.ids.values()),'no duplicate HTML IDs')
        check(value.count('window.movioCallWeatherUI=Object.freeze')==1,'one weather UI controller')
        check('candidate.addEventListener(\'message\',receiveForecast)' not in value,'old weather socket observer removed')
        return value

    html_paths=[root/'integrations'/name for name in ('ai_call_page.html','weather_call_page.html')]
    for path in html_paths:check(path.is_file(),'existing '+path.name)
    html_before={path:path.read_text(encoding='utf-8') for path in html_paths}
    check(all(value==active for value in html_before.values()),'live and both saved call pages match')
    patched=patch_html(active)
    check(patch_html(patched)==patched,'rerunning this cell is idempotent')
    writes={display_path:(before_python,after_python),**{path:(value,patched) for path,value in html_before.items()}}
    scripts=Parts();scripts.feed(patched)
    for part in scripts.scripts:
        if part['src'] or not part['text'].strip():continue
        result=subprocess.run([node,'--check'],input=part['text'],text=True,capture_output=True,timeout=15)
        check(result.returncode==0,'JavaScript syntax: '+result.stderr[:180])
    tests=subprocess.run([node,'-e',state_tests],input=javascript,text=True,capture_output=True,timeout=20)
    check(tests.returncode==0,'offline popup lifecycle tests: '+tests.stderr[:600])
    js_checks=json.loads(tests.stdout)['checks']

    def fetch_call_page():
        import requests
        session=requests.Session();session.trust_env=False
        try:
            response=session.get('http://127.0.0.1:8000/call-page',timeout=(1,4),allow_redirects=False)
            check(response.status_code==200,'local call-page HTTP 200')
            return response.text
        finally:session.close()
    check(fetch_call_page()==active,'local call page serves the inspected live HTML')
    protected={name:(value,getattr(value,'__code__',None)) for name,value in (
        ('voice_endpoint',getattr(b,'_wx_endpoint',None)),('speak',getattr(b,'speak',None)),
        ('tts_synthesize',getattr(getattr(b,'TTS',None),'synthesize',None)),
        ('tanglish_speech',getattr(b,'tanglish_speech_text',None)),('prepare_turn',engine.get('prepare_turn')),
        ('speech_segments',engine.get('speech_segments')),('weather_sender',getattr(engine.get('ConfirmationSocket'),'send_json',None)))}
    # Back up everything BEFORE the first write. Only three existing files change.
    changes={path:(before,after) for path,(before,after) in writes.items() if before!=after}
    backup=None
    if changes:
        backup=root/('call_weather_popup_backup_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));backup.mkdir()
        for path,(before,_) in changes.items():
            destination=backup/path.relative_to(root);destination.parent.mkdir(parents=True,exist_ok=True)
            destination.write_text(before,encoding='utf-8')
    old_code,old_doc=base.__code__,base.__doc__;written=[]
    def atomic_write(path,value):
        fd,name=tempfile.mkstemp(prefix='.'+path.name+'.',suffix='.tmp',dir=path.parent)
        try:
            with os.fdopen(fd,'w',encoding='utf-8') as stream:stream.write(value)
            shutil.copymode(path,name);os.replace(name,path)
        finally:Path(name).unlink(missing_ok=True)
    try:
        for path,(_,after) in changes.items():atomic_write(path,after);written.append(path)
        base.__code__=new_code;base.__doc__=candidate.__doc__;b.HTML=patched
        check(fetch_call_page()==patched,'updated popup HTML is served by the existing voice bridge')
        for name,(function,code) in protected.items():
            check(getattr(function,'__code__',None) is code,name+' function and implementation are preserved')
    except BaseException:
        b.HTML=active;base.__code__=old_code;base.__doc__=old_doc
        for path in reversed(written):atomic_write(path,writes[path][0])
        print('Popup installation rolled back. Existing voice and booking code restored.')
        raise
    result=dict(step=2,version=version,passed=True,python_and_source_checks=checks,javascript_lifecycle_checks=js_checks,
        served_call_page='PASS',complete_advisory_transcript=True,tts_and_booking_logic_unchanged=True,
        changed_files=[str(path.relative_to(root)) for path in changes],backup=str(backup) if backup else None,
        browser_voice_verification='REQUIRED: new AI Call; popup, full transcript, speech, automatic minimize and reopen')
    print(json.dumps(result,ensure_ascii=False,indent=2))
    print('Close the OLD AI Call, refresh your signed-in dashboard, then open a NEW AI Call.')
    print('Test one ride. Weather should open once, print the complete advisory, then minimize after playback.')
    print('Tap Trip weather to reopen. The minus button minimizes it. Test Tamil and Tanglish too.')
    print('Share this output before the next step. No evaluation code or booking records were changed.')
    return result

MOVIO_CALL_WEATHER_POPUP=movio_enable_call_weather_popup()


{
  "step": 2,
  "version": "call-weather-popup-1.0",
  "passed": true,
  "python_and_source_checks": 79,
  "javascript_lifecycle_checks": 34,
  "served_call_page": "PASS",
  "complete_advisory_transcript": true,
  "tts_and_booking_logic_unchanged": true,
  "changed_files": [
    "integrations/movio_weather_guard.py",
    "integrations/ai_call_page.html",
    "integrations/weather_call_page.html"
  ],
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/call_weather_popup_backup_20261028_070736_439792",
  "browser_voice_verification": "REQUIRED: new AI Call; popup, full transcript, speech, automatic minimize and reopen"
}
Close the OLD AI Call, refresh your signed-in dashboard, then open a NEW AI Call.
Test one ride. Weather should open once, print the complete advisory, then minimize after playback.
Tap Trip weather to reopen. The minus button minimizes it. Test Tamil and Tanglish too.
Share this output before the next step. No evaluation code or booking records were changed.


In [127]:

from pathlib import Path
import ast
import json
import textwrap

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
assert (ROOT / "server.py").is_file(), "MOVIO project not found"

TARGET = ROOT / "integrations/movio_customer_care.py"

if TARGET.exists():
    print("Existing recommendation module found.")
    print("No overwrite performed:", TARGET)
else:
    SOURCE = r'''
from datetime import datetime
from zoneinfo import ZoneInfo

VERSION = "customer-care-0.1"
TZ = ZoneInfo("Asia/Kolkata")

MESSAGES = {
    "rain": (
        "Rain is possible. Carry an umbrella and keep your belongings dry.",
        "மழை பெய்ய வாய்ப்புள்ளது. குடை எடுத்துச் செல்லுங்கள்; உடைமைகளை நனையாமல் பாதுகாத்துக்கொள்ளுங்கள்.",
        "Mazhai peiya vaaippu irukku. Umbrella eduthuttu ponga; belongings nanaiyaama paathukkonga."
    ),
    "heat": (
        "It's quite warm. Carry drinking water and stay hydrated.",
        "வெப்பம் அதிகமாக உள்ளது. குடிநீர் எடுத்துச் சென்று உடலில் நீர்ச்சத்து குறையாமல் பார்த்துக்கொள்ளுங்கள்.",
        "Veyil adhigama irukku. Thanni eduthuttu ponga; hydrated ah irunga."
    ),
    "night": (
        "For late-night travel, wait in a well-lit area and verify your driver before boarding.",
        "இரவு நேரப் பயணத்தில் வெளிச்சமான இடத்தில் காத்திருந்து, ஏறும் முன் ஓட்டுநர் விவரங்களைச் சரிபாருங்கள்.",
        "Night travel ku velichamaana place la wait pannunga. Erurathukku munnaadi driver details verify pannunga."
    ),
    "otp": (
        "Share your pickup OTP only with your assigned driver when boarding.",
        "வாகனத்தில் ஏறும்போது மட்டும் ஒதுக்கப்பட்ட ஓட்டுநரிடம் உங்கள் OTP-ஐ பகிருங்கள்.",
        "Board pannumbodhu mattum assigned driver kitta pickup OTP share pannunga."
    ),
    "capacity": (
        "Please select a vehicle that comfortably accommodates everyone travelling.",
        "பயணம் செய்யும் அனைவருக்கும் வசதியான வாகனத்தைத் தேர்ந்தெடுக்கவும்.",
        "Travel panra ellaarukkum comfortable ah irukkura vehicle select pannunga."
    ),
    "airport": (
        "Before leaving for the airport, check your travel documents and luggage.",
        "விமான நிலையத்திற்குச் செல்லும் முன் பயண ஆவணங்களையும் உடைமைகளையும் சரிபாருங்கள்.",
        "Airport ku kilamburathukku munnaadi documents um luggage um check pannunga."
    ),
    "railway": (
        "Keep your train ticket and travel essentials ready.",
        "ரயில் பயணச்சீட்டையும் தேவையான பொருட்களையும் தயாராக வைத்துக்கொள்ளுங்கள்.",
        "Train ticket um travel essentials um ready ah vechukkonga."
    ),
    "hospital": (
        "Remember any medical documents and essential items you need.",
        "தேவையான மருத்துவ ஆவணங்களையும் அத்தியாவசியப் பொருட்களையும் எடுத்துக்கொள்ளுங்கள்.",
        "Thevaiyaana medical documents um essentials um eduthukkonga."
    ),
    "phone": (
        "Keep your phone charged so you can contact and track your driver.",
        "ஓட்டுநரைத் தொடர்புகொள்ளவும் கண்காணிக்கவும் கைப்பேசியில் போதுமான சார்ஜ் வைத்திருங்கள்.",
        "Driver ah contact pannavum track pannavum phone charge vechukkonga."
    ),
    "arrival": (
        "Before leaving the vehicle, please check that you have all your belongings.",
        "வாகனத்திலிருந்து இறங்கும் முன் உங்கள் உடைமைகள் அனைத்தும் இருப்பதைச் சரிபாருங்கள்.",
        "Vehicle la irundhu erangurathukku munnaadi belongings ellaam irukka nu check pannunga."
    ),
    "pickup": (
        "Choose a safe, accessible pickup point where your driver can stop.",
        "ஓட்டுநர் நிறுத்தக்கூடிய பாதுகாப்பான, எளிதில் அணுகக்கூடிய இடத்தைத் தேர்ந்தெடுக்கவும்.",
        "Driver stop panna mudiyura safe ah accessible ah irukkura pickup point choose pannunga."
    ),
    "accessibility": (
        "If you need boarding assistance, please coordinate with your assigned driver.",
        "ஏறுவதற்கு உதவி தேவைப்பட்டால் ஒதுக்கப்பட்ட ஓட்டுநருடன் தொடர்புகொள்ளுங்கள்.",
        "Boarding assistance venumna assigned driver kitta coordinate pannunga."
    ),
    "ready": (
        "Your driver is approaching. Please get ready at your pickup point.",
        "உங்கள் ஓட்டுநர் நெருங்கிக்கொண்டிருக்கிறார். ஏறும் இடத்தில் தயாராக இருங்கள்.",
        "Unga driver nerungittu irukkaaru. Pickup point la ready ah irunga."
    ),
}

CAPACITY = {"bike": 1, "auto": 3, "mini": 4, "sedan": 4, "suv": 6}

def evaluate(context):
    """Return up to two evidence-based recommendations, without side effects."""
    c = dict(context or {})
    language = c.get("language", "English")
    index = {"English": 0, "Tamil": 1, "Tanglish": 2}.get(language, 0)
    stage = c.get("stage", "dashboard")
    acknowledged = set(c.get("acknowledged") or [])
    candidates = []

    def add(key, score, evidence):
        if key not in acknowledged:
            candidates.append({
                "id": key,
                "score": score,
                "text": MESSAGES[key][index],
                "evidence": evidence
            })

    weather = c.get("weather") or {}
    if weather.get("status") == "ok":
        probability = weather.get("precipitation_probability")
        code = weather.get("weather_code")
        temperature = weather.get("temperature_2m")
        feels = weather.get("apparent_temperature")

        rain_codes = {51,53,55,56,57,61,63,65,66,67,80,81,82,95,96,99}
        if stage in ("dashboard", "details", "pre_confirmation"):
            if (isinstance(code, int) and code in rain_codes) or (
                isinstance(probability, (int,float)) and
                not isinstance(probability, bool) and probability >= 60
            ):
                add("rain", 95, "verified_weather")

            if (
                isinstance(temperature, (int,float)) and
                not isinstance(temperature, bool) and temperature >= 38
            ) or (
                isinstance(feels, (int,float)) and
                not isinstance(feels, bool) and feels >= 40
            ):
                add("heat", 90, "verified_weather")

    pickup_time = c.get("pickup_time")
    if stage in ("details", "pre_confirmation") and pickup_time:
        try:
            dt = datetime.fromisoformat(str(pickup_time))
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=TZ)
            hour = dt.astimezone(TZ).hour
            if hour >= 22 or hour < 5:
                add("night", 85, "pickup_time")
        except (ValueError, TypeError, OverflowError):
            pass

    vehicle = str(c.get("vehicle") or "").lower()
    count = c.get("passenger_count")
    if stage == "details" and vehicle in CAPACITY:
        if isinstance(count, int) and not isinstance(count, bool) and count > CAPACITY[vehicle]:
            add("capacity", 100, "passenger_count_and_vehicle")

    if stage == "pre_confirmation":
        destination = str(c.get("destination") or "").lower()
        if any(x in destination for x in ("airport", "விமான நிலைய")):
            add("airport", 75, "destination")
        elif any(x in destination for x in ("railway", "railway station", "train station", "ரயில் நிலைய")):
            add("railway", 75, "destination")
        elif any(x in destination for x in ("hospital", "மருத்துவமனை")):
            add("hospital", 75, "destination")

    if stage == "driver_assigned" and c.get("driver_assigned"):
        add("otp", 95, "driver_assignment")

    if stage == "driver_approaching" and c.get("driver_assigned"):
        eta = c.get("driver_eta_minutes")
        if isinstance(eta, (int,float)) and not isinstance(eta, bool) and 0 <= eta <= 10:
            add("ready", 95, "verified_driver_eta")

    if stage == "ride_completed" and c.get("ride_status") == "completed":
        add("arrival", 100, "completed_ride")

    if stage == "pre_confirmation" and c.get("pickup"):
        add("pickup", 45, "pickup_location")

    if stage == "driver_assigned" and c.get("accessibility_requested"):
        add("accessibility", 85, "explicit_customer_request")

    if stage == "driver_assigned" and c.get("driver_assigned"):
        add("phone", 40, "driver_assignment")

    candidates.sort(key=lambda x: (-x["score"], x["id"]))
    return candidates[:2]
'''
    SOURCE = textwrap.dedent(SOURCE)
    ast.parse(SOURCE)
    TARGET.write_text(SOURCE, encoding="utf-8")
    print("Created:", TARGET)

import importlib.util

spec = importlib.util.spec_from_file_location(
    "movio_customer_care", TARGET
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

tests = [
    {
        "stage": "dashboard",
        "language": "English",
        "weather": {
            "status": "ok",
            "weather_code": 61,
            "precipitation_probability": 75
        }
    },
    {
        "stage": "pre_confirmation",
        "language": "Tamil",
        "pickup_time": "2026-10-19T23:55:00",
        "destination": "Chennai Airport"
    },
    {
        "stage": "driver_assigned",
        "language": "Tanglish",
        "driver_assigned": True
    },
    {
        "stage": "dashboard",
        "weather": {"status": "unavailable"}
    },
    {
        "stage": "ride_completed",
        "ride_status": "completed"
    }
]

for i, case in enumerate(tests, 1):
    result = module.evaluate(case)
    assert len(result) <= 2
    print(f"\nTEST {i}:")
    print(json.dumps(result, ensure_ascii=False, indent=2))

print("\nCUSTOMER CARE ENGINE INITIAL TESTS PASSED")
print("Existing booking, dashboard and TTS files unchanged.")


Created: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_customer_care.py

TEST 1:
[
  {
    "id": "rain",
    "score": 95,
    "text": "Rain is possible. Carry an umbrella and keep your belongings dry.",
    "evidence": "verified_weather"
  }
]

TEST 2:
[
  {
    "id": "night",
    "score": 85,
    "text": "இரவு நேரப் பயணத்தில் வெளிச்சமான இடத்தில் காத்திருந்து, ஏறும் முன் ஓட்டுநர் விவரங்களைச் சரிபாருங்கள்.",
    "evidence": "pickup_time"
  },
  {
    "id": "airport",
    "score": 75,
    "text": "விமான நிலையத்திற்குச் செல்லும் முன் பயண ஆவணங்களையும் உடைமைகளையும் சரிபாருங்கள்.",
    "evidence": "destination"
  }
]

TEST 3:
[
  {
    "id": "otp",
    "score": 95,
    "text": "Board pannumbodhu mattum assigned driver kitta pickup OTP share pannunga.",
    "evidence": "driver_assignment"
  },
  {
    "id": "phone",
    "score": 40,
    "text": "Driver ah contact pannavum track pannavum phone charge vechukkonga.",
    "evidence": "driver_assignment"
  }
]

TEST 4:
[]

TEST 5:
[
  {
  

In [128]:

from pathlib import Path
import ast
import importlib.util

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
CARE = ROOT / "integrations/movio_customer_care.py"
BRIDGE = ROOT / "integrations/movio_customer_care_api.py"

assert CARE.is_file(), "Run Step 4 first."
assert (ROOT / "server.py").is_file()

source = r'''
"""Read-only customer-care recommendation endpoint."""

import importlib.util
import json
from pathlib import Path
from urllib.parse import urlparse

VERSION = "customer-care-api-0.1"

def install_backend(server):
    if getattr(server.Handler, "_customer_care_api_version", None) == VERSION:
        return {"version": VERSION, "already_installed": True}

    root = Path(server.ROOT)
    path = root / "integrations/movio_customer_care.py"

    spec = importlib.util.spec_from_file_location(
        "movio_customer_care", path
    )
    care = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(care)

    previous_post = server.Handler.do_POST

    def do_post(handler):
        if urlparse(handler.path).path != "/api/customer-care/recommend":
            return previous_post(handler)

        try:
            with server.conn() as db:
                user = handler.user(db)
                if user["role"] != "customer":
                    return handler.json(
                        403, {"error": "Customer account required"}
                    )

            payload = handler.payload()
            if not isinstance(payload, dict):
                return handler.json(400, {"error": "Invalid request"})

            context = payload.get("context", {})
            if not isinstance(context, dict):
                return handler.json(400, {"error": "Invalid context"})

            # Never trust browser-submitted weather or assignment claims.
            # Until verified server-side context is connected, only
            # non-sensitive, general advice can be returned.
            safe = {
                "language": context.get("language", "English"),
                "stage": "dashboard",
                "weather": {},
                "acknowledged": context.get("acknowledged", []),
            }

            recommendations = care.evaluate(safe)

            return handler.json(200, {
                "ok": True,
                "version": VERSION,
                "recommendations": recommendations
            })

        except PermissionError:
            return handler.json(
                401, {"error": "Please sign in"}
            )
        except Exception:
            return handler.json(
                500, {"error": "Customer care unavailable"}
            )

    server.Handler.do_POST = do_post
    server.Handler._customer_care_api_version = VERSION

    return {"version": VERSION, "installed": True}
'''

ast.parse(source)

if BRIDGE.exists():
    print("Bridge already exists; not overwriting:", BRIDGE)
else:
    BRIDGE.write_text(source, encoding="utf-8")
    print("Created:", BRIDGE)

print("Syntax validation: PASS")
print("Existing server.py unchanged")
print("Existing booking and TTS unchanged")
print("API bridge created but NOT YET ACTIVATED")


Created: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_customer_care_api.py
Syntax validation: PASS
Existing server.py unchanged
Existing booking and TTS unchanged
API bridge created but NOT YET ACTIVATED


In [129]:

from pathlib import Path
import ast
import importlib.util
import textwrap

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
TARGET = ROOT / "integrations/movio_customer_care_context.py"

assert (ROOT / "integrations/movio_customer_care.py").is_file()
assert (ROOT / "integrations/movio_customer_tools.py").is_file()

SOURCE = r'''
"""Current-location context for MOVIO customer care."""

import math
import time
import requests
from datetime import datetime
from zoneinfo import ZoneInfo

VERSION = "customer-care-context-0.1"
TZ = ZoneInfo("Asia/Kolkata")

def valid_coordinates(lat, lng):
    if isinstance(lat, bool) or isinstance(lng, bool):
        raise ValueError("Invalid coordinates")
    lat, lng = float(lat), float(lng)
    if not (
        math.isfinite(lat) and math.isfinite(lng)
        and -90 <= lat <= 90
        and -180 <= lng <= 180
    ):
        raise ValueError("Invalid coordinates")
    return lat, lng

def current_weather(lat, lng, getter=None):
    lat, lng = valid_coordinates(lat, lng)
    getter = getter or requests.get

    try:
        response = getter(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lng,
                "current": (
                    "temperature_2m,apparent_temperature,"
                    "weather_code,precipitation"
                ),
                "hourly": "precipitation_probability",
                "forecast_days": 1,
                "timezone": "Asia/Kolkata",
            },
            timeout=(3, 6),
        )
        response.raise_for_status()
        payload = response.json()

        current = payload.get("current")
        hourly = payload.get("hourly")

        if not isinstance(current, dict):
            raise ValueError("Missing current weather")

        def number(value, low, high):
            if isinstance(value, bool):
                return None
            try:
                value = float(value)
            except (TypeError, ValueError):
                return None
            return value if math.isfinite(value) and low <= value <= high else None

        code = number(current.get("weather_code"), 0, 99)
        temp = number(current.get("temperature_2m"), -90, 65)
        feels = number(current.get("apparent_temperature"), -100, 80)

        if code is None or not code.is_integer() or temp is None:
            raise ValueError("Incomplete current weather")

        probability = None
        observed_time = current.get("time")

        if isinstance(hourly, dict) and isinstance(observed_time, str):
            times = hourly.get("time")
            probabilities = hourly.get("precipitation_probability")

            if isinstance(times, list) and isinstance(probabilities, list):
                hour = observed_time[:13]
                matches = [
                    i for i, value in enumerate(times)
                    if isinstance(value, str) and value[:13] == hour
                ]
                if len(matches) == 1 and matches[0] < len(probabilities):
                    probability = number(
                        probabilities[matches[0]], 0, 100
                    )

        return {
            "status": "ok",
            "source": "Open-Meteo",
            "weather_code": int(code),
            "temperature_2m": temp,
            "apparent_temperature": feels,
            "precipitation_probability": probability,
            "observed_at": observed_time,
            "fetched_at": time.time(),
        }

    except (requests.RequestException, ValueError, TypeError, KeyError):
        return {
            "status": "unavailable",
            "source": "Open-Meteo",
        }

def dashboard_context(lat, lng, reverse_address, getter=None):
    lat, lng = valid_coordinates(lat, lng)

    locality = None
    try:
        result = reverse_address(lat, lng)
        if isinstance(result, dict):
            locality = (
                result.get("locality")
                or result.get("area")
                or result.get("name")
            )
        elif isinstance(result, (tuple, list)) and result:
            locality = result[0]
        elif isinstance(result, str):
            locality = result

        if not isinstance(locality, str) or not locality.strip():
            locality = None
    except Exception:
        locality = None

    return {
        "locality": locality,
        "weather": current_weather(lat, lng, getter=getter),
        "time_of_day": datetime.now(TZ).hour,
    }
'''

SOURCE = textwrap.dedent(SOURCE)
ast.parse(SOURCE)

if TARGET.exists():
    print("Existing context module found; not overwriting.")
else:
    TARGET.write_text(SOURCE, encoding="utf-8")
    print("Created:", TARGET)

spec = importlib.util.spec_from_file_location(
    "movio_customer_care_context", TARGET
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

class MockResponse:
    def raise_for_status(self):
        pass

    def json(self):
        return {
            "current": {
                "time": "2026-10-19T14:00",
                "temperature_2m": 29.5,
                "apparent_temperature": 34.0,
                "weather_code": 61,
                "precipitation": 1.2,
            },
            "hourly": {
                "time": ["2026-10-19T14:00"],
                "precipitation_probability": [80],
            },
        }

def mock_get(*args, **kwargs):
    return MockResponse()

result = module.dashboard_context(
    12.9249,
    80.1000,
    reverse_address=lambda lat, lng: {"locality": "Tambaram"},
    getter=mock_get,
)

assert result["locality"] == "Tambaram"
assert result["weather"]["status"] == "ok"
assert result["weather"]["precipitation_probability"] == 80

print("Mock weather and locality test: PASS")
print("Result:", result)
print("Existing server, dashboard, booking and TTS unchanged.")


Created: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_customer_care_context.py
Mock weather and locality test: PASS
Result: {'locality': 'Tambaram', 'weather': {'status': 'ok', 'source': 'Open-Meteo', 'weather_code': 61, 'temperature_2m': 29.5, 'apparent_temperature': 34.0, 'precipitation_probability': 80.0, 'observed_at': '2026-10-19T14:00', 'fetched_at': 1793171256.507511}, 'time_of_day': 12}
Existing server, dashboard, booking and TTS unchanged.


In [130]:

from pathlib import Path
import importlib.util
import json

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")

def load_module(name, filename):
    path = ROOT / "integrations" / filename
    assert path.is_file(), f"Missing: {path}"
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

care = load_module(
    "movio_customer_care_context",
    "movio_customer_care_context.py"
)

tools = load_module(
    "movio_customer_tools",
    "movio_customer_tools.py"
)

# Test coordinates only — not the customer's GPS location.
TEST_LAT = 12.9249
TEST_LNG = 80.1000

print("=" * 60)
print("TEST 1 — LIVE OPEN-METEO")
print("=" * 60)

weather = care.current_weather(TEST_LAT, TEST_LNG)
print(json.dumps(weather, ensure_ascii=False, indent=2))

print("\n" + "=" * 60)
print("TEST 2 — EXISTING REVERSE GEOCODER")
print("=" * 60)

try:
    locality = tools.reverse_address(TEST_LAT, TEST_LNG)
    print("Return type:", type(locality).__name__)
    print(json.dumps(
        locality,
        ensure_ascii=False,
        indent=2,
        default=str
    ))
except Exception as error:
    print("Reverse geocoding failed:")
    print(type(error).__name__, str(error)[:300])

print("\n" + "=" * 60)
print("TEST 3 — RECOMMENDATION ENGINE")
print("=" * 60)

engine = load_module(
    "movio_customer_care",
    "movio_customer_care.py"
)

recommendations = engine.evaluate({
    "stage": "dashboard",
    "language": "English",
    "weather": weather
})

print(json.dumps(
    recommendations,
    ensure_ascii=False,
    indent=2
))

print("\nREAD-ONLY LIVE SERVICE TEST COMPLETE")
print("No files modified.")


TEST 1 — LIVE OPEN-METEO
{
  "status": "ok",
  "source": "Open-Meteo",
  "weather_code": 3,
  "temperature_2m": 32.3,
  "apparent_temperature": 35.6,
  "precipitation_probability": 0.0,
  "observed_at": "2026-10-28T12:30",
  "fetched_at": 1793171256.7202806
}

TEST 2 — EXISTING REVERSE GEOCODER
Return type: dict
{
  "name": "Kannadapalayam",
  "locality": "Kannadapalayam",
  "full_address": "Tambaram - Mudichur - Sriperumbudur Road, Kannadapalayam, Tambaram, Chennai Metropolitan Area, Tambaram, Chengalpattu, Tamil Nadu, 600045, India",
  "lat": 12.9249,
  "lng": 80.1,
  "locality_status": "ok",
  "locality_source": "address.suburb",
  "locality_policy": "main-locality-1.1"
}

TEST 3 — RECOMMENDATION ENGINE
[]

READ-ONLY LIVE SERVICE TEST COMPLETE
No files modified.


In [131]:

from pathlib import Path
import ast
import shutil
from datetime import datetime

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
TARGET = ROOT / "integrations/movio_customer_care_context.py"

source = TARGET.read_text(encoding="utf-8")

old = '''                hour = observed_time[:13]
                matches = [
                    i for i, value in enumerate(times)
                    if isinstance(value, str) and value[:13] == hour
                ]
                if len(matches) == 1 and matches[0] < len(probabilities):
                    probability = number(
                        probabilities[matches[0]], 0, 100
                    )'''

new = '''                try:
                    observation = datetime.fromisoformat(observed_time)
                    candidates = []
                    for i, value in enumerate(times):
                        if not isinstance(value, str) or i >= len(probabilities):
                            continue
                        forecast_time = datetime.fromisoformat(value)
                        delta = (forecast_time - observation).total_seconds()
                        if 0 <= delta <= 7200:
                            candidates.append((delta, i))
                    if candidates:
                        _, index = min(candidates)
                        probability = number(probabilities[index], 0, 100)
                except (ValueError, TypeError, OverflowError):
                    probability = None'''

if old not in source:
    print("Expected code not found. No files modified.")
else:
    updated = source.replace(old, new, 1)
    ast.parse(updated)

    backup = TARGET.with_suffix(".py.step8_backup")
    shutil.copy2(TARGET, backup)
    TARGET.write_text(updated, encoding="utf-8")

    print("Updated:", TARGET)
    print("Backup:", backup)
    print("Syntax validation: PASS")
    print("Existing booking, server, dashboard and TTS unchanged.")


Updated: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_customer_care_context.py
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_customer_care_context.py.step8_backup
Syntax validation: PASS
Existing booking, server, dashboard and TTS unchanged.


In [132]:

from pathlib import Path
import importlib.util
import json

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")

def load_module(name, filename):
    path = ROOT / "integrations" / filename
    assert path.is_file(), f"Missing: {path}"

    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

context = load_module(
    "movio_customer_care_context_step9",
    "movio_customer_care_context.py"
)

engine = load_module(
    "movio_customer_care_step9",
    "movio_customer_care.py"
)

# Fixed test coordinates, NOT the customer's live GPS.
LAT, LNG = 12.9249, 80.1000

print("=" * 55)
print("TEST 1 — LIVE WEATHER AFTER PATCH")
print("=" * 55)

weather = context.current_weather(LAT, LNG)
print(json.dumps(weather, indent=2, ensure_ascii=False))

assert weather["status"] in ("ok", "unavailable")

if weather["status"] == "ok":
    probability = weather.get("precipitation_probability")
    assert probability is None or 0 <= probability <= 100
    print("Weather validation: PASS")

print("\n" + "=" * 55)
print("TEST 2 — ENGLISH RECOMMENDATIONS")
print("=" * 55)

english = engine.evaluate({
    "stage": "dashboard",
    "language": "English",
    "weather": weather
})

print(json.dumps(english, indent=2, ensure_ascii=False))

print("\n" + "=" * 55)
print("TEST 3 — TAMIL RECOMMENDATIONS")
print("=" * 55)

tamil = engine.evaluate({
    "stage": "dashboard",
    "language": "Tamil",
    "weather": weather
})

print(json.dumps(tamil, indent=2, ensure_ascii=False))

print("\n" + "=" * 55)
print("TEST 4 — UNAVAILABLE WEATHER")
print("=" * 55)

unavailable = engine.evaluate({
    "stage": "dashboard",
    "language": "English",
    "weather": {"status": "unavailable"}
})

assert unavailable == []
print("No fabricated weather advice: PASS")

print("\n" + "=" * 55)
print("STEP 9 COMPLETE")
print("No files modified.")


TEST 1 — LIVE WEATHER AFTER PATCH
{
  "status": "ok",
  "source": "Open-Meteo",
  "weather_code": 3,
  "temperature_2m": 32.3,
  "apparent_temperature": 35.6,
  "precipitation_probability": 0.0,
  "observed_at": "2026-10-28T12:30",
  "fetched_at": 1793171257.6203732
}
Weather validation: PASS

TEST 2 — ENGLISH RECOMMENDATIONS
[]

TEST 3 — TAMIL RECOMMENDATIONS
[]

TEST 4 — UNAVAILABLE WEATHER
No fabricated weather advice: PASS

STEP 9 COMPLETE
No files modified.


In [133]:

from pathlib import Path
import ast
import shutil
import importlib.util

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
TARGET = ROOT / "integrations/movio_customer_care.py"

assert TARGET.is_file(), "Customer-care engine not found"

MARKER = "# MOVIO_CUSTOMER_CARE_GREETING_V1"

ADDITION = r'''

# MOVIO_CUSTOMER_CARE_GREETING_V1
def dashboard_greeting(context):
    """Generate a greeting without inventing location or weather."""
    c = dict(context or {})
    language = c.get("language", "English")
    if language not in ("English", "Tamil", "Tanglish"):
        language = "English"

    locality = c.get("locality")
    if not isinstance(locality, str):
        locality = None
    elif not locality.strip() or len(locality) > 100:
        locality = None
    else:
        locality = locality.strip()

    weather = c.get("weather") or {}
    valid_weather = weather.get("status") == "ok"

    greetings = {
        "English": "Welcome back! Wishing you a comfortable journey.",
        "Tamil": "மீண்டும் வரவேற்கிறோம்! உங்கள் பயணம் இனிதாக அமையட்டும்.",
        "Tanglish": "Welcome back! Unga journey comfortable ah irukkattum."
    }

    messages = {
        "English": {
            "normal": "Hope you have a pleasant journey.",
            "rain": "Rain is possible. Carry an umbrella.",
            "heat": "It's quite warm. Remember to stay hydrated.",
            "pleasant": "The weather looks pleasant for your journey."
        },
        "Tamil": {
            "normal": "உங்கள் பயணம் இனிதாக அமையட்டும்.",
            "rain": "மழை பெய்ய வாய்ப்புள்ளது. குடை எடுத்துச் செல்லுங்கள்.",
            "heat": "வெப்பம் அதிகமாக உள்ளது. போதுமான தண்ணீர் குடியுங்கள்.",
            "pleasant": "பயணத்திற்கு வானிலை இனிமையாக உள்ளது."
        },
        "Tanglish": {
            "normal": "Unga journey nalla irukkattum.",
            "rain": "Mazhai peiya vaaippu irukku. Umbrella eduthuttu ponga.",
            "heat": "Veyil adhigama irukku. Thanni kudinga.",
            "pleasant": "Journey ku weather pleasant ah irukku."
        }
    }

    if not valid_weather:
        return {
            "text": greetings[language],
            "weather_verified": False,
            "locality_verified": False
        }

    code = weather.get("weather_code")
    temp = weather.get("temperature_2m")
    feels = weather.get("apparent_temperature")
    probability = weather.get("precipitation_probability")

    def numeric(value):
        return (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
        )

    rain_codes = {
        51, 53, 55, 56, 57, 61, 63, 65, 66, 67,
        80, 81, 82, 95, 96, 99
    }

    rain = (
        isinstance(code, int) and code in rain_codes
    ) or (
        numeric(probability) and probability >= 60
    )

    heat = (
        numeric(temp) and temp >= 38
    ) or (
        numeric(feels) and feels >= 40
    )

    pleasant = (
        isinstance(code, int)
        and code in (0, 1, 2)
        and numeric(temp)
        and 18 <= temp <= 32
        and not heat
    )

    condition = (
        "rain" if rain else
        "heat" if heat else
        "pleasant" if pleasant else
        "normal"
    )

    location_prefix = ""
    if locality:
        if language == "English":
            location_prefix = f"In {locality}, "
        elif language == "Tamil":
            location_prefix = f"{locality} பகுதியில், "
        else:
            location_prefix = f"{locality} la, "

    return {
        "text": location_prefix + messages[language][condition],
        "weather_verified": True,
        "locality_verified": bool(locality),
        "condition": condition
    }
'''

source = TARGET.read_text(encoding="utf-8")

if MARKER in source:
    print("Greeting function already exists. No changes made.")
else:
    updated = "import math\n" + source + ADDITION
    ast.parse(updated)

    backup = TARGET.with_suffix(".py.step10_backup")
    shutil.copy2(TARGET, backup)
    TARGET.write_text(updated, encoding="utf-8")

    print("Greeting function added.")
    print("Backup:", backup)

spec = importlib.util.spec_from_file_location(
    "movio_customer_care_step10", TARGET
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

cases = [
    {
        "language": "English",
        "locality": "Kannadapalayam",
        "weather": {
            "status": "ok",
            "weather_code": 3,
            "temperature_2m": 25.5,
            "apparent_temperature": 31.6,
            "precipitation_probability": None
        }
    },
    {
        "language": "Tamil",
        "weather": {"status": "unavailable"}
    },
    {
        "language": "Tanglish",
        "locality": "Tambaram",
        "weather": {
            "status": "ok",
            "weather_code": 61,
            "temperature_2m": 27,
            "precipitation_probability": 80
        }
    }
]

for i, case in enumerate(cases, 1):
    result = module.dashboard_greeting(case)
    print(f"\nTEST {i}:", result)
    assert result["text"]

assert module.dashboard_greeting(cases[1])["weather_verified"] is False

print("\nSTEP 10 GREETING TESTS PASSED")
print("Existing booking, weather API, server and TTS unchanged.")


Greeting function added.
Backup: /kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_customer_care.py.step10_backup

TEST 1: {'text': 'In Kannadapalayam, Hope you have a pleasant journey.', 'weather_verified': True, 'locality_verified': True, 'condition': 'normal'}

TEST 2: {'text': 'மீண்டும் வரவேற்கிறோம்! உங்கள் பயணம் இனிதாக அமையட்டும்.', 'weather_verified': False, 'locality_verified': False}

TEST 3: {'text': 'Tambaram la, Mazhai peiya vaaippu irukku. Umbrella eduthuttu ponga.', 'weather_verified': True, 'locality_verified': True, 'condition': 'rain'}

STEP 10 GREETING TESTS PASSED
Existing booking, weather API, server and TTS unchanged.


In [134]:
# MOVIO CUSTOMER CARE — STEP 4: COMPACT CUSTOMER DASHBOARD CARD.
# Paste this ENTIRE file into ONE new Kaggle cell BEFORE evaluation,
# after the verified Step 3B backend has passed. Then reload the website.
# Uses the existing customer-login function and one-shot locate() utility.
# No website/model restart, booking-pickup change, or browser GPS tracking loop.
#
# The weather backend's version cannot be read without a customer sign-in when its
# health route is protected. In that case this cell warns and still installs the card.
# Optional: run `MOVIO_COMPANION_ALLOW_WITHOUT_WEATHER = True` in an earlier cell
# to install even when no customer-care route can be found at all.
def movio_add_travel_companion_card():
    import hashlib
    import json
    import os
    import re
    import shutil
    import stat
    import subprocess
    import tempfile
    import types
    from datetime import datetime,timezone
    from html.parser import HTMLParser
    from pathlib import Path
    import requests

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    web=root/'web';index=web/'index.html';auth=web/'auth.js'
    js_path=web/'customer_care_companion.js';css_path=web/'customer_care_companion.css'
    digest=lambda value:hashlib.sha256(value).hexdigest()
    js=r'''/* MOVIO travel companion 1.0. Existing booking, GPS-sharing and voice handlers stay in control. */
(() => {
  'use strict';
  if (window.MOVIOTravelCompanion) return;
  const VERSION = 'travel-companion-1.0';
  const byId = id => document.getElementById(id);
  const state = { userId: null, sequence: 0, busy: false, attempted: false, pending: false,
    controller: null, position: null, response: null, gps: 'unavailable', phase: 'idle', languageDirty: false };
  const labels = {
    English: { title: 'Your Travel Companion', refresh: 'Refresh travel advice',
      greeting: 'Welcome back! Wishing you a comfortable journey.', loading: 'Checking local conditions…',
      rain: 'Rain chance', hourly: 'Forecast for this hour', near: 'Near ',
      clear: 'Clear skies', cloudy: 'Cloudy', rainCondition: 'Rain possible', heavy_rain: 'Heavy rain',
      storm: 'Thunderstorms', fog: 'Foggy', other: 'Local weather', fallback: 'Have a wonderful journey!' },
    Tamil: { title: 'உங்கள் பயணத் துணை', refresh: 'பயணத் தகவலைப் புதுப்பிக்கவும்',
      greeting: 'மீண்டும் வரவேற்கிறோம்! உங்கள் பயணம் இனிதாக அமையட்டும்.', loading: 'வானிலை தகவல் பெறப்படுகிறது…',
      rain: 'மழை வாய்ப்பு', hourly: 'இந்த நேரத்திற்கான முன்னறிவிப்பு', near: '',
      clear: 'தெளிவான வானம்', cloudy: 'மேகமூட்டம்', rainCondition: 'மழை வாய்ப்பு', heavy_rain: 'கனமழை',
      storm: 'இடியுடன் மழை', fog: 'மூடுபனி', other: 'வானிலை', fallback: 'உங்கள் பயணம் இனிமையாக அமையட்டும்!' },
    Tanglish: { title: 'Unga Travel Companion', refresh: 'Travel advice refresh pannunga',
      greeting: 'Welcome back! Unga payanam inimaiyaaga irukkattum.', loading: 'Local weather check pannrom…',
      rain: 'Rain chance', hourly: 'Indha hour forecast', near: '',
      clear: 'Clear skies', cloudy: 'Cloudy', rainCondition: 'Rain chance irukku', heavy_rain: 'Heavy rain',
      storm: 'Thunderstorms', fog: 'Foggy', other: 'Local weather', fallback: 'Unga payanam inimaiyaaga amaiyattum!' }
  };
  function language() {
    const select = byId('language');
    const option = select?.selectedOptions?.[0];
    const value = String(option?.textContent || select?.value || '').toLowerCase();
    if (value.includes('tanglish')) return 'Tanglish';
    if (/tamil|தமிழ்/.test(value) || /^ta(?:-|$)/i.test(select?.value || '')) return 'Tamil';
    return 'English';
  }
  function text(node, value) { const next = String(value ?? ''); if (node.textContent !== next) node.textContent = next; }
  function el(tag, className, content) {
    const node = document.createElement(tag);
    if (className) node.className = className;
    if (content !== undefined) node.textContent = content;
    return node;
  }
  function mounted() { return byId('movioTravelCompanion'); }
  function mount() {
    const existing = mounted(); if (existing) return existing;
    const panel = byId('riderPanel'); if (!panel) return null;
    const card = el('section', 'movio-companion'); card.id = 'movioTravelCompanion';
    card.setAttribute('aria-labelledby', 'movioCompanionTitle');
    const head = el('div', 'movio-companion-head');
    const title = el('h2'); title.id = 'movioCompanionTitle';
    const refresh = el('button', 'movio-companion-refresh', '↻');
    refresh.id = 'movioCompanionRefresh'; refresh.type = 'button';
    refresh.addEventListener('click', () => void refreshAdvice(true));
    head.append(title, refresh);
    const greeting = el('p', 'movio-companion-greeting'); greeting.id = 'movioCompanionGreeting';
    const weather = el('div', 'movio-companion-weather'); weather.id = 'movioCompanionWeather'; weather.hidden = true;
    const primary = el('div', 'movio-companion-primary');
    const temperature = el('strong'); temperature.id = 'movioCompanionTemperature';
    const summary = el('div', 'movio-companion-summary');
    const condition = el('b'); condition.id = 'movioCompanionCondition';
    const locality = el('span'); locality.id = 'movioCompanionLocality'; summary.append(condition, locality);
    primary.append(temperature, summary);
    const chance = el('div', 'movio-companion-rain');
    const probability = el('b'); probability.id = 'movioCompanionProbability';
    const caption = el('span'); caption.id = 'movioCompanionRainLabel'; chance.append(probability, caption);
    weather.append(primary, chance);
    const tips = el('ul', 'movio-companion-tips'); tips.id = 'movioCompanionTips';
    const note = el('p', 'movio-companion-note'); note.id = 'movioCompanionNote'; note.hidden = true;
    card.append(head, greeting, weather, tips, note);
    const heading = Array.from(panel.children).find(child => child.classList.contains('sheet-head'));
    const anchor = heading?.nextElementSibling?.matches('p.muted') ? heading.nextElementSibling : heading;
    if (anchor) anchor.after(card); else panel.prepend(card);
    return card;
  }
  function display(response = null) {
    const card = mount(); if (!card) return;
    const lang = language(), words = labels[lang];
    text(byId('movioCompanionTitle'), words.title);
    const refresh = byId('movioCompanionRefresh');
    refresh.disabled = state.busy; refresh.title = words.refresh; refresh.setAttribute('aria-label', words.refresh);
    card.setAttribute('aria-busy', String(state.busy));
    text(byId('movioCompanionGreeting'), response?.language === lang ? response.greeting : words.greeting);
    const weather = response?.language === lang ? response.weather : null;
    const available = weather?.status === 'ok';
    byId('movioCompanionWeather').hidden = !available;
    const number = value => typeof value === 'number' && Number.isFinite(value) ? value : null;
    if (available) {
      const temp = number(weather.temperature_c), rain = number(weather.rain_probability_pct);
      text(byId('movioCompanionTemperature'), temp === null ? '—' : `${Number(temp.toFixed(1))}°C`);
      text(byId('movioCompanionCondition'), words[weather.condition === 'rain' ? 'rainCondition' : weather.condition] || words.other);
      const locality = typeof response.locality === 'string' ? response.locality : '';
      text(byId('movioCompanionLocality'), locality ? (lang === 'Tamil' ? locality + ' அருகில்' : lang === 'Tanglish' ? locality + ' arugil' : words.near + locality) : '');
      text(byId('movioCompanionProbability'), rain === null ? '—' : `${Number(rain.toFixed(1))}%`);
      text(byId('movioCompanionRainLabel'), words.rain);
    }
    const tips = byId('movioCompanionTips');
    const rows = response?.language === lang && Array.isArray(response.recommendations)
      ? response.recommendations.filter(row => row && typeof row.text === 'string' && row.text.trim()).slice(0, 2) : [];
    // Weather values already appear above. Keep the engine-selected practical
    // precaution here rather than reading the same percentage twice.
    const messages = rows.length ? rows.map(row => {
      const full = row.text.trim();
      const sentences = full.split(/(?<=[.!?])\s+/);
      return row.group === 'weather' && sentences.length > 1 ? sentences.at(-1) : full;
    }) : [words.fallback];
    const signature = JSON.stringify(messages);
    if (tips.dataset.signature !== signature) {
      tips.replaceChildren(...messages.map(message => el('li', '', message)));
      tips.dataset.signature = signature;
    }
    const note = byId('movioCompanionNote');
    note.hidden = !state.busy && !available;
    text(note, state.busy ? words.loading : available ? words.hourly : '');
  }
  function invalidate() {
    state.sequence++; state.controller?.abort(); state.controller = null; state.busy = false;
  }
  function clear() {
    invalidate(); state.userId = null; state.attempted = false; state.pending = false;
    state.position = null; state.response = null; state.gps = 'unavailable'; state.phase = 'idle'; state.languageDirty = false;
    mounted()?.remove();
  }
  function shown() {
    return byId('appShell') && !byId('appShell').classList.contains('hidden') &&
      byId('riderPanel') && !byId('riderPanel').classList.contains('hidden');
  }
  async function jsonFetch(url, options, token) {
    const controller = new AbortController(); state.controller = controller;
    const timer = setTimeout(() => controller.abort(), 25000);
    try {
      const response = await fetch(url, { ...options, credentials: 'same-origin', cache: 'no-store', signal: controller.signal });
      if (token !== state.sequence) throw new Error('stale_response');
      if (response.status === 401 || response.status === 403) { clear(); throw new Error('signed_out'); }
      if (!response.ok) throw new Error('optional_service_unavailable');
      return await response.json();
    } finally { clearTimeout(timer); if (state.controller === controller) state.controller = null; }
  }
  async function position() {
    if (!navigator.geolocation) throw { code: 0, unsupported: true };
    // Reuse the inspected one-shot utility. locate() reads GPS only; unlike
    // the map's current-location button, it does not select a booking pickup.
    if (typeof locate === 'function') {
      const point = await locate();
      return { lat: point.lat, lng: point.lng, observed_at: Date.now() };
    }
    return await new Promise((resolve, reject) => navigator.geolocation.getCurrentPosition(
      point => resolve({ lat: point.coords.latitude, lng: point.coords.longitude, observed_at: point.timestamp }),
      reject, { enableHighAccuracy: true, timeout: 12000, maximumAge: 3000 }));
  }
  async function refreshAdvice(readPosition = true) {
    if (state.busy || state.userId === null || !shown()) return;
    const token = ++state.sequence; state.busy = true; state.phase = 'checking'; display(state.response);
    try {
      const me = await jsonFetch('/api/me', {}, token);
      if (token !== state.sequence) return;
      if (me.role !== 'customer' || String(me.id) !== state.userId) { clear(); return; }
      if (readPosition) {
        state.phase = 'locating'; state.position = null;
        try {
          const point = await position(); if (token !== state.sequence) return;
          if (!Number.isFinite(point.lat) || !Number.isFinite(point.lng)) throw new Error('invalid_position');
          // Only a temporary approximate area stays in this tab's memory.
          state.position = { status: 'granted', lat: Number(point.lat.toFixed(3)), lng: Number(point.lng.toFixed(3)), observed_at: point.observed_at };
          state.gps = 'granted';
        } catch (error) {
          if (token !== state.sequence) return;
          state.gps = error?.unsupported ? 'unsupported' : error?.code === 1 ? 'denied' : error?.code === 3 ? 'timeout' : 'unavailable';
        }
      }
      state.phase = 'weather';
      const location = state.position || { status: state.gps };
      const response = await jsonFetch('/api/customer-care/recommend', {
        method: 'POST', headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ context: { language: language(), current_location: location } })
      }, token);
      if (token !== state.sequence) return;
      state.response = response; state.phase = response.weather?.status === 'ok' ? 'ready' : 'neutral';
    } catch (error) {
      if (token === state.sequence) { state.response = null; state.phase = 'neutral'; }
    } finally {
      if (token === state.sequence) {
        state.busy = false; display(state.response);
        if (state.languageDirty && state.userId !== null && shown()) {
          state.languageDirty = false;
          if (!state.response || state.response.language !== language()) void refreshAdvice(false);
        }
      }
    }
  }
  function login(me) {
    if (!me || me.role !== 'customer' || me.id === undefined || me.id === null) { clear(); return; }
    const id = String(me.id);
    if (state.userId !== id) { clear(); state.userId = id; }
    if (!shown()) return;
    display(state.response);
    if (state.attempted) return;
    if (document.visibilityState === 'hidden') { state.pending = true; return; }
    state.pending = false; state.attempted = true; void refreshAdvice(true);
  }
  const publicAPI = Object.freeze({ version: VERSION, login,
    refresh: () => refreshAdvice(true), dispose: clear,
    getStatus: () => ({ phase: state.phase, gps: state.gps, busy: state.busy, mounted: !!mounted() }) });
  window.MOVIOTravelCompanion = publicAPI;
  byId('language')?.addEventListener('change', () => {
    if (state.userId === null) return;
    if (state.busy) { state.languageDirty = true; display(state.response); return; }
    const hadResponse = state.response;
    invalidate(); state.response = null; display();
    if (hadResponse && shown()) void refreshAdvice(false);
  });
  byId('logout')?.addEventListener('click', clear, { capture: true });
  window.addEventListener('pagehide', clear);
  document.addEventListener('visibilitychange', () => {
    if (state.pending && document.visibilityState !== 'hidden' && shown()) {
      state.pending = false; state.attempted = true; void refreshAdvice(true);
    }
  });
  function syncExistingLogin() {
    if (typeof movioMe !== 'undefined' && movioMe && shown()) login(movioMe);
  }
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', syncExistingLogin, { once: true });
  else syncExistingLogin();
  window.addEventListener('pageshow', event => { if (event.persisted) syncExistingLogin(); });
})();
'''
    css=r'''/* Scoped to the companion card; existing dashboard controls retain their styles. */
#movioTravelCompanion{background:linear-gradient(120deg,#fff9e8 0%,var(--paper,#fff) 78%);border:1px solid #f2df9c;border-radius:17px;padding:12px 14px;margin:14px 0 16px;color:var(--ink,#17191d);min-width:0}
#movioTravelCompanion .movio-companion-head{display:flex;align-items:center;justify-content:space-between;gap:10px}
#movioTravelCompanion h2{font-family:inherit;font-size:13px;letter-spacing:0;line-height:1.35;margin:0;font-weight:800}
#movioTravelCompanion .movio-companion-refresh{flex:none;min-width:36px;min-height:36px;border:1px solid #eadca8;border-radius:11px;background:var(--paper,#fff);color:var(--ink,#17191d);font-size:22px;line-height:1;padding:4px;cursor:pointer}
#movioTravelCompanion .movio-companion-refresh:focus-visible{outline:2px solid #9e7300;outline-offset:2px}
#movioTravelCompanion .movio-companion-refresh:disabled{opacity:.45;cursor:wait}
#movioTravelCompanion .movio-companion-greeting{font-size:11px;color:#646a72;line-height:1.45;margin:2px 0 8px;overflow-wrap:anywhere}
#movioTravelCompanion .movio-companion-weather{display:flex;align-items:center;justify-content:space-between;gap:10px;margin:5px 0 9px;min-width:0}
#movioTravelCompanion [hidden]{display:none!important}
#movioTravelCompanion .movio-companion-primary{display:flex;align-items:center;gap:10px;min-width:0}
#movioTravelCompanion .movio-companion-primary>strong{font-family:Manrope,'DM Sans',Arial,sans-serif;font-size:29px;line-height:1.1;letter-spacing:-1.1px;font-weight:800;white-space:nowrap}
#movioTravelCompanion .movio-companion-summary{display:grid;gap:3px;min-width:0}
#movioTravelCompanion .movio-companion-summary>b{font-size:12px;line-height:1.25}
#movioTravelCompanion .movio-companion-summary>span{font-size:10px;color:#646a72;overflow-wrap:anywhere;line-height:1.3}
#movioTravelCompanion .movio-companion-rain{display:grid;gap:3px;text-align:right;flex:none;padding-left:9px;border-left:1px solid #eadca8}
#movioTravelCompanion .movio-companion-rain>b{font-size:19px;line-height:1.15}
#movioTravelCompanion .movio-companion-rain>span{font-size:9px;color:#646a72;max-width:70px;line-height:1.3}
#movioTravelCompanion .movio-companion-tips{list-style:none;padding:0;margin:7px 0 0;display:grid;gap:5px}
#movioTravelCompanion .movio-companion-tips>li{font-size:12px;line-height:1.45;margin:0;padding-left:10px;position:relative;overflow-wrap:anywhere}
#movioTravelCompanion .movio-companion-tips>li:before{content:'';position:absolute;left:0;top:.57em;width:4px;height:4px;border-radius:50%;background:#b0881b}
#movioTravelCompanion .movio-companion-note{font-size:9px;line-height:1.35;color:#76736a;margin:8px 0 0}
@media(max-width:600px){#movioTravelCompanion{padding:10px 12px;margin:11px 0 13px}#movioTravelCompanion .movio-companion-refresh{min-width:44px;min-height:44px}#movioTravelCompanion .movio-companion-primary>strong{font-size:26px}}
'''
    js_bytes,css_bytes=js.encode('utf-8'),css.encode('utf-8')
    for path in (index,auth):
        if path.is_symlink() or not path.is_file():raise RuntimeError('A dashboard source is missing or is a symlink. No files changed.')
    for path,expected in ((js_path,js_bytes),(css_path,css_bytes)):
        if path.is_symlink() or path.exists() and path.read_bytes()!=expected:
            raise RuntimeError('A different companion asset exists: '+path.name+'. No files were changed.')
    for path in web.glob('*.js'):
        if path!=js_path and re.search(r'customer.?care|companion',path.name,re.I):
            raise RuntimeError('Another companion client exists: '+path.name+'. Inspect it before installing overlapping UI.')
    old_index=index.read_bytes();old_auth=auth.read_bytes()
    old_tail="if(movioMe.role==='driver'&&(!movioMe.vehicle_type||!movioMe.vehicle_number))movioEditProfile()}"
    new_tail=old_tail[:-1]+";/* MOVIO_TRAVEL_COMPANION_LOGIN_V1 */window.MOVIOTravelCompanion?.login(movioMe)}"
    auth_text=old_auth.decode('utf-8')
    base_auth=auth_text.replace(new_tail,old_tail)
    if digest(base_auth.encode('utf-8'))!='f4d9fbb77b784c2f2e8cdddea80182a76a84f6ddd2d43d92b2974a86271177f6' or base_auth.count(old_tail)!=1:
        raise RuntimeError('The login function changed after inspection. No files or handlers were changed.')
    new_auth=base_auth.replace(old_tail,new_tail,1).encode('utf-8')
    if new_auth.decode().replace(new_tail,old_tail).encode()!=base_auth.encode():raise RuntimeError('Login-hook isolation failed.')
    style_block=('<!-- MOVIO_TRAVEL_COMPANION_STYLE_V1 -->\n'
        '<link rel="stylesheet" href="/customer_care_companion.css?v='+digest(css_bytes)[:12]+'">\n'
        '<!-- MOVIO_TRAVEL_COMPANION_STYLE_END_V1 -->\n')
    script_block=('<!-- MOVIO_TRAVEL_COMPANION_SCRIPT_V1 -->\n'
        '<script defer src="/customer_care_companion.js?v='+digest(js_bytes)[:12]+'"></script>\n'
        '<!-- MOVIO_TRAVEL_COMPANION_SCRIPT_END_V1 -->\n')
    html=old_index.decode('utf-8')
    for marker,block in (('STYLE',style_block),('SCRIPT',script_block)):
        opening='<!-- MOVIO_TRAVEL_COMPANION_'+marker+'_V1 -->'
        if opening in html:
            if html.count(block)!=1:raise RuntimeError('A different companion HTML block exists; no changes made.')
            html=html.replace(block,'',1)
    # Remove only our own auth cache token before comparing to the inspected page.
    html=re.sub(r"(src\s*=\s*['\"]auth\.js)\?movio_companion=[0-9a-f]{12}(['\"])",r'\1\2',html)
    if digest(html.encode('utf-8'))!='6a0e3baa973d2ad0b299ae1880bbc96ff7a1e31cbd15cacd24b12a88a783e86c':
        raise RuntimeError('The customer dashboard changed after inspection. No files were changed.')
    base_html=html
    if len(re.findall(r'</head\s*>',html,re.I))!=1 or len(re.findall(r'</body\s*>',html,re.I))!=1:
        raise RuntimeError('The dashboard head/body layout is unexpected; no changes made.')
    html=re.sub(r'</head\s*>',lambda match:style_block+match[0],html,count=1,flags=re.I)
    html=re.sub(r'</body\s*>',lambda match:script_block+match[0],html,count=1,flags=re.I)
    auth_pattern=r"(src\s*=\s*['\"]auth\.js)(['\"])"
    if len(re.findall(auth_pattern,html))!=1:raise RuntimeError('A unique auth script reference was not found.')
    html=re.sub(auth_pattern,lambda match:match[1]+'?movio_companion='+digest(new_auth)[:12]+match[2],html,count=1)
    isolated=html.replace(style_block,'',1).replace(script_block,'',1)
    isolated=re.sub(r"(src\s*=\s*['\"]auth\.js)\?movio_companion=[0-9a-f]{12}(['\"])",r'\1\2',isolated)
    if isolated!=base_html:raise RuntimeError('Dashboard-source isolation failed; no changes made.')
    class Page(HTMLParser):
        def __init__(self):super().__init__();self.scripts=[];self.links=[];self.ids=[]
        def handle_starttag(self,tag,attrs):
            fields=dict(attrs)
            if tag=='script' and fields.get('src'):self.scripts.append(fields['src'])
            if tag=='link' and fields.get('href'):self.links.append(fields['href'])
            if fields.get('id'):self.ids.append(fields['id'])
    before,after=Page(),Page();before.feed(base_html);after.feed(html)
    if before.ids!=after.ids or len(after.scripts)!=len(before.scripts)+1:
        raise RuntimeError('The original dashboard controls/script count changed unexpectedly.')
    if len([s for s in after.scripts if s.split('?')[0].endswith('customer_care_companion.js')])!=1:
        raise RuntimeError('Companion script is not unique.')
    node=shutil.which('node');syntax={}
    if node:
        with tempfile.TemporaryDirectory(prefix='movio_companion_syntax_') as folder:
            for name,value in (('auth.js',new_auth),('customer_care_companion.js',js_bytes)):
                path=Path(folder)/name;path.write_bytes(value)
                check=subprocess.run([node,'--check',str(path)],capture_output=True,text=True,timeout=15)
                if check.returncode:raise RuntimeError('JavaScript syntax check failed for '+name+': '+check.stderr[:500])
                syntax[name]='PASS'
    else:syntax['note']='Node.js unavailable in this kernel; JavaScript was browser-tested during preparation.'
    session=requests.Session();session.trust_env=False
    def fetch(path):return session.get('http://127.0.0.1:8080'+path,timeout=(1,4),allow_redirects=False)

    def probe_backend():
        """Read-only, unauthenticated probe. The version is readable only when the health route is public.
        A 401/403 means the route exists but wants a customer sign-in, so the version cannot be checked here."""
        info=dict(state='unreachable',version=None,health_status=None,route_status=None,proceed=False,message='')
        try:
            health=fetch('/api/customer-care/health')
        except requests.RequestException as error:
            info['message']='The website did not answer on 127.0.0.1:8080 ('+type(error).__name__+'). Start the website first. No UI files were changed.'
            return info
        info['health_status']=health.status_code
        try:body=health.json()
        except ValueError:body=None
        version=str(body.get('version','')) if isinstance(body,dict) else ''
        info['version']=version or None
        found=re.fullmatch(r'customer-care-api-(\d+)\.(\d+)',version)
        if health.status_code==200 and found and (int(found[1]),int(found[2]))>=(1,0):
            info.update(state='ready',proceed=True)
            return info
        if health.status_code in (401,403):
            info.update(state='sign_in_required',proceed=True)
            return info
        try:
            route=session.post('http://127.0.0.1:8080/api/customer-care/recommend',json={},timeout=(1,4),allow_redirects=False)
            info['route_status']=route.status_code
        except requests.RequestException:
            info['route_status']=None
        tail=(' To install the card anyway with the greeting and general tips only, run '
              'MOVIO_COMPANION_ALLOW_WITHOUT_WEATHER = True in an earlier cell, then run this cell again.')
        if health.status_code==200:
            info['state']='unexpected_version'
            info['message']=('The customer-care health route reports version '+repr(version or None)+', not 1.x or newer. No UI files were changed. '
                'Re-run the Step 3B backend cell, then this cell.'+tail)
        elif info['route_status'] in (401,403):
            info['state']='older_route'
            info['message']=('There is no customer-care health route (HTTP '+str(health.status_code)+'), but a recommendation route exists (HTTP '+str(info['route_status'])+'). '
                'That matches the older bridge, which returns no weather. No UI files were changed. Re-run the Step 3B backend cell, then this cell.'+tail)
        else:
            info['state']='missing'
            info['message']=('No customer-care routes were found (health HTTP '+str(health.status_code)+', recommend HTTP '+str(info['route_status'])+'). '
                'No UI files were changed. Re-run the customer-care backend cells, then this cell.'+tail)
        return info

    backend=probe_backend()
    allow_without_weather=bool(globals().get('MOVIO_COMPANION_ALLOW_WITHOUT_WEATHER'))
    if not backend['proceed'] and (not allow_without_weather or backend['state']=='unreachable'):
        session.close();raise RuntimeError(backend['message'])

    # Non-blocking evidence: which customer-care API versions are written in the project files.
    disk_versions={}
    for source_path in [root/'server.py']+sorted((root/'integrations').glob('*.py')):
        if not source_path.is_file() or source_path.is_symlink():continue
        try:source_text=source_path.read_text(encoding='utf-8',errors='ignore')
        except OSError:continue
        for found_version in sorted(set(re.findall(r'customer-care-api-\d+\.\d+',source_text))):
            disk_versions.setdefault(found_version,[]).append(str(source_path.relative_to(root)))

    protected={}
    targets={index,auth,js_path,css_path}
    paths={root/'server.py'}
    for folder in (web,root/'integrations'):
        paths.update(path for path in folder.rglob('*') if path.suffix in {'.py','.js','.html','.css'})
    for path in paths:
        if path in targets or not path.is_file() or path.is_symlink():continue
        if any('backup' in part.casefold() or part=='__pycache__' for part in path.relative_to(root).parts):continue
        protected[path]=digest(path.read_bytes())
    b=globals().get('bridge');engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',{})
    functions=[getattr(b,name,None) for name in ('speak','tanglish_speech_text','_wx_endpoint')]
    functions+=[engine.get(name) for name in ('prepare_turn','advisory','display_reply','tts_reply')]
    codes=[(function,function.__code__) for function in functions if isinstance(function,types.FunctionType)]
    def verify():
        for path,value in protected.items():
            if not path.is_file() or digest(path.read_bytes())!=value:raise RuntimeError('A protected source changed: '+str(path.relative_to(root)))
        if any(function.__code__ is not code for function,code in codes):raise RuntimeError('A protected voice handler changed.')
    plans=[(css_path,css_path.read_bytes() if css_path.is_file() else None,css_bytes),
           (js_path,js_path.read_bytes() if js_path.is_file() else None,js_bytes),
           (auth,old_auth,new_auth),(index,old_index,html.encode('utf-8'))]
    changes=[row for row in plans if row[1]!=row[2]]
    backup=None
    if changes:
        backup=root/('travel_companion_ui_backup_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
        backup.mkdir(exist_ok=False)
        manifest={}
        for path,old,new in changes:
            relative=str(path.relative_to(root));manifest[relative]=dict(existed=old is not None)
            if old is not None:
                saved=backup/relative;saved.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(path,saved)
                if saved.read_bytes()!=old:raise RuntimeError('Backup verification failed; no dashboard sources were replaced.')
        (backup/'restore_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    def atomic(path,value):
        mode=stat.S_IMODE(path.stat().st_mode) if path.is_file() else 0o644
        descriptor,temporary=tempfile.mkstemp(prefix='.movio_companion_',dir=path.parent)
        try:
            with os.fdopen(descriptor,'wb') as output:
                output.write(value);output.flush();os.fsync(output.fileno());os.fchmod(output.fileno(),mode)
            os.replace(temporary,path)
        finally:Path(temporary).unlink(missing_ok=True)
    applied=[]
    try:
        verify()
        for path,old,new in changes:
            if path.is_symlink() or (path.read_bytes() if path.is_file() else None)!=old:
                raise RuntimeError('A UI source changed during installation: '+path.name)
            atomic(path,new);applied.append((path,old,new))
        served={}
        for path,old,new in plans:
            url='/' if path==index else '/'+path.name+'?companion_check='+digest(new)[:12]
            response=fetch(url)
            if response.status_code!=200 or response.content!=new:raise RuntimeError('The website did not serve the updated '+path.name)
            served[path.name]='source bytes match'
        after_backend=probe_backend()
        if after_backend['state']!=backend['state'] or after_backend['version']!=backend['version']:
            raise RuntimeError('The customer-care API changed during UI installation (was '+backend['state']+' '+str(backend['version'])+', now '+after_backend['state']+' '+str(after_backend['version'])+').')
        verify()
    except BaseException:
        for path,old,new in reversed(applied):
            if path.is_file() and not path.is_symlink() and path.read_bytes()==new:
                if old is None:path.unlink()
                else:atomic(path,old)
        session.close();raise
    session.close()
    result=dict(step=4,version='travel-companion-1.0',passed=True,
        changed_files=[str(path.relative_to(root)) for path,_,_ in changes],
        backup=str(backup) if backup else None,javascript_syntax=syntax,served_assets=served,
        backend_state=backend['state'],backend_version=backend['version'],
        weather_backend_verified=backend['state']=='ready',
        on_disk_api_versions=disk_versions,
        source_isolation='PASS: one login hook and companion asset links; original controls retained',
        website_restarted=False,voice_handlers_preserved=True,
        automatic_location='One request after customer sign-in; no tracking watch or automatic booking-pickup selection.',
        browser_verification='REQUIRED: reload the customer dashboard, allow or deny location, check the card and an unchanged booking pickup.')
    print(json.dumps(result,ensure_ascii=False,indent=2))
    if backend['state']=='sign_in_required':
        print('NOTE: the weather backend version could not be verified without a customer sign-in (its health route is protected).')
        print('Sign in as a customer after reloading. If the weather row and locality appear, Step 3B is active.')
        print('If only the greeting and a general tip appear, re-run the Step 3B backend cell and reload.')
    elif backend['state']!='ready':
        print('NOTE: installed WITHOUT a verified weather backend (state: '+backend['state']+'). The card shows the greeting and general tips only until Step 3B is active.')
    print('Reload the customer dashboard with Ctrl+Shift+R; on mobile, reload the page.')
    print('Allow location when the browser asks, or deny it to check the neutral greeting.')
    print('Share the output and whether locality, temperature and rain chance appear. Do not share precise GPS coordinates.')
    return result

print('Step 4 function defined (installer NOT run). Now run the conflict-fix cell.')

Step 4 function defined (installer NOT run). Now run the conflict-fix cell.


###### --------------------------------

In [135]:
# ============================================================
# MOVIO CUSTOMER CARE — STEP 4 CONFLICT FIX + INSTALLATION
# ============================================================

from pathlib import Path
from datetime import datetime
import re
import json
import shutil
import traceback

ROOT = Path(
    globals().get(
        "WX_ROOT",
        "/kaggle/working/MOVIO_MAIN_WEBSITE"
    )
).resolve()

WEB = ROOT / "web"

print("=" * 65)
print("MOVIO CUSTOMER CARE — STEP 4")
print("=" * 65)

# ------------------------------------------------------------
# 1. VERIFY PROJECT AND INSTALLER
# ------------------------------------------------------------

if not WEB.is_dir():
    raise FileNotFoundError(
        f"MOVIO web directory not found: {WEB}"
    )

if not callable(globals().get("movio_add_travel_companion_card")):
    raise RuntimeError(
        "Step 4 function is not loaded. "
        "Run the original Step 4 definition cell first."
    )

print("PASS: Project directory found")
print("PASS: Step 4 installer available")

# ------------------------------------------------------------
# 2. IDENTIFY CONFLICTING JAVASCRIPT FILES
# ------------------------------------------------------------

expected_client = WEB / "customer_care_companion.js"

conflicts = [
    path
    for path in WEB.glob("*.js")
    if path != expected_client
    and re.search(
        r"customer.?care|companion",
        path.name,
        re.IGNORECASE
    )
]

print("\nChecking conflicting JavaScript files...")

if conflicts:
    for path in conflicts:
        print("FOUND:", path.name)
else:
    print("PASS: No conflicting JavaScript files")

# ------------------------------------------------------------
# 3. CHECK WHETHER CONFLICTING FILES ARE IN USE
# ------------------------------------------------------------

def find_references(filename):

    references = []

    for path in WEB.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".html", ".js", ".css"
        }:
            continue

        if path.name == filename:
            continue

        if path.is_symlink():
            continue

        try:
            content = path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            if filename in content:
                references.append(
                    str(path.relative_to(ROOT))
                )

        except OSError:
            pass

    return references


# ------------------------------------------------------------
# 4. BACK UP AND DISABLE ONLY UNUSED DEBUG FILES
# ------------------------------------------------------------

disabled_files = []
blocked_files = []

for path in conflicts:

    references = find_references(path.name)

    if references:
        print(
            f"\nBLOCKED: {path.name} is referenced by:"
        )

        for reference in references:
            print("  ", reference)

        blocked_files.append(path.name)
        continue

    # Only automatically disable the known debug script.
    if path.name != "movio_companion_debug.js":
        print(
            f"\nBLOCKED: {path.name} is not an approved debug file."
        )
        blocked_files.append(path.name)
        continue

    if path.is_symlink():
        print("BLOCKED: Debug file is a symbolic link")
        blocked_files.append(path.name)
        continue

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )

    backup_directory = (
        ROOT / "companion_conflict_backups" / timestamp
    )

    backup_directory.mkdir(
        parents=True,
        exist_ok=False
    )

    backup_file = backup_directory / path.name

    shutil.copy2(path, backup_file)

    if backup_file.read_bytes() != path.read_bytes():
        raise RuntimeError(
            "Backup verification failed. "
            "Original file was not modified."
        )

    disabled_path = path.with_name(
        path.name + ".disabled"
    )

    if disabled_path.exists():
        raise RuntimeError(
            f"Disabled file already exists: {disabled_path}"
        )

    path.rename(disabled_path)

    disabled_files.append(
        {
            "original": str(path),
            "disabled": str(disabled_path),
            "backup": str(backup_file)
        }
    )

    print("\nPASS: Unused debug file disabled")
    print("Backup:", backup_file)

# ------------------------------------------------------------
# 5. STOP IF AN ACTIVE CONFLICT STILL EXISTS
# ------------------------------------------------------------

if blocked_files:

    print("\nINSTALLATION BLOCKED")
    print("The following files require manual inspection:")

    for filename in blocked_files:
        print(" -", filename)

    print(
        "\nNo Step 4 installation attempted."
    )

else:

    # --------------------------------------------------------
    # 6. RUN ORIGINAL STEP 4 INSTALLER
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("INSTALLING CUSTOMER TRAVEL COMPANION")
    print("=" * 65)

    try:

        MOVIO_CUSTOMER_CARE_STEP_4 = (
            movio_add_travel_companion_card()
        )

        print("\n" + "=" * 65)
        print("STEP 4 INSTALLATION COMPLETED")
        print("=" * 65)

        print(
            json.dumps(
                MOVIO_CUSTOMER_CARE_STEP_4,
                indent=2,
                ensure_ascii=False,
                default=str
            )
        )

        print("\nNEXT STEPS:")
        print("1. Open the MOVIO website.")
        print("2. Refresh using Ctrl + Shift + R.")
        print("3. Log in as a customer.")
        print("4. Allow browser location permission.")
        print("5. Check the Travel Companion card.")
        print("6. Verify weather and travel suggestions.")

    except Exception as error:

        print("\nSTEP 4 INSTALLATION FAILED")
        print("Reason:", str(error))

        traceback.print_exc()

        print(
            "\nThe original Step 4 installer performs "
            "its own validation and rollback."
        )

        print(
            "Do not bypass HTML, authentication, "
            "or backend compatibility checks."
        )

        raise

print("\nMOVIO Step 4 conflict check finished.")

MOVIO CUSTOMER CARE — STEP 4
PASS: Project directory found
PASS: Step 4 installer available

Checking conflicting JavaScript files...
PASS: No conflicting JavaScript files

INSTALLING CUSTOMER TRAVEL COMPANION
{
  "step": 4,
  "version": "travel-companion-1.0",
  "passed": true,
  "changed_files": [
    "web/customer_care_companion.css",
    "web/customer_care_companion.js",
    "web/auth.js",
    "web/index.html"
  ],
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/travel_companion_ui_backup_20261028_070737_791625",
  "javascript_syntax": {
    "auth.js": "PASS",
    "customer_care_companion.js": "PASS"
  },
  "served_assets": {
    "customer_care_companion.css": "source bytes match",
    "customer_care_companion.js": "source bytes match",
    "auth.js": "source bytes match",
    "index.html": "source bytes match"
  },
  "backend_state": "sign_in_required",
  "backend_version": null,
  "weather_backend_verified": false,
  "on_disk_api_versions": {
    "customer-care-api-0.1": [
      

In [136]:
# MOVIO CUSTOMER CARE — STEP 4: COMPACT CUSTOMER DASHBOARD CARD.
# Paste this ENTIRE file into ONE new Kaggle cell BEFORE evaluation,
# after the verified Step 3B backend has passed. Then reload the website.
# Uses the existing customer-login function and one-shot locate() utility.
# No website/model restart, booking-pickup change, or browser GPS tracking loop.
#
# The weather backend's version cannot be read without a customer sign-in when its
# health route is protected. In that case this cell warns and still installs the card.
# Optional: run `MOVIO_COMPANION_ALLOW_WITHOUT_WEATHER = True` in an earlier cell
# to install even when no customer-care route can be found at all.
def movio_add_travel_companion_card():
    import hashlib
    import json
    import os
    import re
    import shutil
    import stat
    import subprocess
    import tempfile
    import types
    from datetime import datetime,timezone
    from html.parser import HTMLParser
    from pathlib import Path
    import requests

    root=Path(globals().get('WX_ROOT','/kaggle/working/MOVIO_MAIN_WEBSITE')).resolve()
    web=root/'web';index=web/'index.html';auth=web/'auth.js'
    js_path=web/'customer_care_companion.js';css_path=web/'customer_care_companion.css'
    digest=lambda value:hashlib.sha256(value).hexdigest()
    js=r'''/* MOVIO travel companion 1.0. Existing booking, GPS-sharing and voice handlers stay in control. */
(() => {
  'use strict';
  if (window.MOVIOTravelCompanion) return;
  const VERSION = 'travel-companion-1.0';
  const byId = id => document.getElementById(id);
  const state = { userId: null, sequence: 0, busy: false, attempted: false, pending: false,
    controller: null, position: null, response: null, gps: 'unavailable', phase: 'idle', languageDirty: false };
  const labels = {
    English: { title: 'Your Travel Companion', refresh: 'Refresh travel advice',
      greeting: 'Welcome back! Wishing you a comfortable journey.', loading: 'Checking local conditions…',
      rain: 'Rain chance', hourly: 'Forecast for this hour', near: 'Near ',
      clear: 'Clear skies', cloudy: 'Cloudy', rainCondition: 'Rain possible', heavy_rain: 'Heavy rain',
      storm: 'Thunderstorms', fog: 'Foggy', other: 'Local weather', fallback: 'Have a wonderful journey!' },
    Tamil: { title: 'உங்கள் பயணத் துணை', refresh: 'பயணத் தகவலைப் புதுப்பிக்கவும்',
      greeting: 'மீண்டும் வரவேற்கிறோம்! உங்கள் பயணம் இனிதாக அமையட்டும்.', loading: 'வானிலை தகவல் பெறப்படுகிறது…',
      rain: 'மழை வாய்ப்பு', hourly: 'இந்த நேரத்திற்கான முன்னறிவிப்பு', near: '',
      clear: 'தெளிவான வானம்', cloudy: 'மேகமூட்டம்', rainCondition: 'மழை வாய்ப்பு', heavy_rain: 'கனமழை',
      storm: 'இடியுடன் மழை', fog: 'மூடுபனி', other: 'வானிலை', fallback: 'உங்கள் பயணம் இனிமையாக அமையட்டும்!' },
    Tanglish: { title: 'Unga Travel Companion', refresh: 'Travel advice refresh pannunga',
      greeting: 'Welcome back! Unga payanam inimaiyaaga irukkattum.', loading: 'Local weather check pannrom…',
      rain: 'Rain chance', hourly: 'Indha hour forecast', near: '',
      clear: 'Clear skies', cloudy: 'Cloudy', rainCondition: 'Rain chance irukku', heavy_rain: 'Heavy rain',
      storm: 'Thunderstorms', fog: 'Foggy', other: 'Local weather', fallback: 'Unga payanam inimaiyaaga amaiyattum!' }
  };
  function language() {
    const select = byId('language');
    const option = select?.selectedOptions?.[0];
    const value = String(option?.textContent || select?.value || '').toLowerCase();
    if (value.includes('tanglish')) return 'Tanglish';
    if (/tamil|தமிழ்/.test(value) || /^ta(?:-|$)/i.test(select?.value || '')) return 'Tamil';
    return 'English';
  }
  function text(node, value) { const next = String(value ?? ''); if (node.textContent !== next) node.textContent = next; }
  function el(tag, className, content) {
    const node = document.createElement(tag);
    if (className) node.className = className;
    if (content !== undefined) node.textContent = content;
    return node;
  }
  function mounted() { return byId('movioTravelCompanion'); }
  function mount() {
    const existing = mounted(); if (existing) return existing;
    const panel = byId('riderPanel'); if (!panel) return null;
    const card = el('section', 'movio-companion'); card.id = 'movioTravelCompanion';
    card.setAttribute('aria-labelledby', 'movioCompanionTitle');
    const head = el('div', 'movio-companion-head');
    const title = el('h2'); title.id = 'movioCompanionTitle';
    const refresh = el('button', 'movio-companion-refresh', '↻');
    refresh.id = 'movioCompanionRefresh'; refresh.type = 'button';
    refresh.addEventListener('click', () => void refreshAdvice(true));
    head.append(title, refresh);
    const greeting = el('p', 'movio-companion-greeting'); greeting.id = 'movioCompanionGreeting';
    const weather = el('div', 'movio-companion-weather'); weather.id = 'movioCompanionWeather'; weather.hidden = true;
    const primary = el('div', 'movio-companion-primary');
    const temperature = el('strong'); temperature.id = 'movioCompanionTemperature';
    const summary = el('div', 'movio-companion-summary');
    const condition = el('b'); condition.id = 'movioCompanionCondition';
    const locality = el('span'); locality.id = 'movioCompanionLocality'; summary.append(condition, locality);
    primary.append(temperature, summary);
    const chance = el('div', 'movio-companion-rain');
    const probability = el('b'); probability.id = 'movioCompanionProbability';
    const caption = el('span'); caption.id = 'movioCompanionRainLabel'; chance.append(probability, caption);
    weather.append(primary, chance);
    const tips = el('ul', 'movio-companion-tips'); tips.id = 'movioCompanionTips';
    const note = el('p', 'movio-companion-note'); note.id = 'movioCompanionNote'; note.hidden = true;
    card.append(head, greeting, weather, tips, note);
    const heading = Array.from(panel.children).find(child => child.classList.contains('sheet-head'));
    const anchor = heading?.nextElementSibling?.matches('p.muted') ? heading.nextElementSibling : heading;
    if (anchor) anchor.after(card); else panel.prepend(card);
    return card;
  }
  function display(response = null) {
    const card = mount(); if (!card) return;
    const lang = language(), words = labels[lang];
    text(byId('movioCompanionTitle'), words.title);
    const refresh = byId('movioCompanionRefresh');
    refresh.disabled = state.busy; refresh.title = words.refresh; refresh.setAttribute('aria-label', words.refresh);
    card.setAttribute('aria-busy', String(state.busy));
    text(byId('movioCompanionGreeting'), response?.language === lang ? response.greeting : words.greeting);
    const weather = response?.language === lang ? response.weather : null;
    const available = weather?.status === 'ok';
    byId('movioCompanionWeather').hidden = !available;
    const number = value => typeof value === 'number' && Number.isFinite(value) ? value : null;
    if (available) {
      const temp = number(weather.temperature_c), rain = number(weather.rain_probability_pct);
      text(byId('movioCompanionTemperature'), temp === null ? '—' : `${Number(temp.toFixed(1))}°C`);
      text(byId('movioCompanionCondition'), words[weather.condition === 'rain' ? 'rainCondition' : weather.condition] || words.other);
      const locality = typeof response.locality === 'string' ? response.locality : '';
      text(byId('movioCompanionLocality'), locality ? (lang === 'Tamil' ? locality + ' அருகில்' : lang === 'Tanglish' ? locality + ' arugil' : words.near + locality) : '');
      text(byId('movioCompanionProbability'), rain === null ? '—' : `${Number(rain.toFixed(1))}%`);
      text(byId('movioCompanionRainLabel'), words.rain);
    }
    const tips = byId('movioCompanionTips');
    const rows = response?.language === lang && Array.isArray(response.recommendations)
      ? response.recommendations.filter(row => row && typeof row.text === 'string' && row.text.trim()).slice(0, 2) : [];
    // Weather values already appear above. Keep the engine-selected practical
    // precaution here rather than reading the same percentage twice.
    const messages = rows.length ? rows.map(row => {
      const full = row.text.trim();
      const sentences = full.split(/(?<=[.!?])\s+/);
      return row.group === 'weather' && sentences.length > 1 ? sentences.at(-1) : full;
    }) : [words.fallback];
    const signature = JSON.stringify(messages);
    if (tips.dataset.signature !== signature) {
      tips.replaceChildren(...messages.map(message => el('li', '', message)));
      tips.dataset.signature = signature;
    }
    const note = byId('movioCompanionNote');
    note.hidden = !state.busy && !available;
    text(note, state.busy ? words.loading : available ? words.hourly : '');
  }
  function invalidate() {
    state.sequence++; state.controller?.abort(); state.controller = null; state.busy = false;
  }
  function clear() {
    invalidate(); state.userId = null; state.attempted = false; state.pending = false;
    state.position = null; state.response = null; state.gps = 'unavailable'; state.phase = 'idle'; state.languageDirty = false;
    mounted()?.remove();
  }
  function shown() {
    return byId('appShell') && !byId('appShell').classList.contains('hidden') &&
      byId('riderPanel') && !byId('riderPanel').classList.contains('hidden');
  }
  async function jsonFetch(url, options, token) {
    const controller = new AbortController(); state.controller = controller;
    const timer = setTimeout(() => controller.abort(), 25000);
    try {
      const response = await fetch(url, { ...options, credentials: 'same-origin', cache: 'no-store', signal: controller.signal });
      if (token !== state.sequence) throw new Error('stale_response');
      if (response.status === 401 || response.status === 403) { clear(); throw new Error('signed_out'); }
      if (!response.ok) throw new Error('optional_service_unavailable');
      return await response.json();
    } finally { clearTimeout(timer); if (state.controller === controller) state.controller = null; }
  }
  async function position() {
    if (!navigator.geolocation) throw { code: 0, unsupported: true };
    // Reuse the inspected one-shot utility. locate() reads GPS only; unlike
    // the map's current-location button, it does not select a booking pickup.
    if (typeof locate === 'function') {
      const point = await locate();
      return { lat: point.lat, lng: point.lng, observed_at: Date.now() };
    }
    return await new Promise((resolve, reject) => navigator.geolocation.getCurrentPosition(
      point => resolve({ lat: point.coords.latitude, lng: point.coords.longitude, observed_at: point.timestamp }),
      reject, { enableHighAccuracy: true, timeout: 12000, maximumAge: 3000 }));
  }
  async function refreshAdvice(readPosition = true) {
    if (state.busy || state.userId === null || !shown()) return;
    const token = ++state.sequence; state.busy = true; state.phase = 'checking'; display(state.response);
    try {
      const me = await jsonFetch('/api/me', {}, token);
      if (token !== state.sequence) return;
      if (me.role !== 'customer' || String(me.id) !== state.userId) { clear(); return; }
      if (readPosition) {
        state.phase = 'locating'; state.position = null;
        try {
          const point = await position(); if (token !== state.sequence) return;
          if (!Number.isFinite(point.lat) || !Number.isFinite(point.lng)) throw new Error('invalid_position');
          // Only a temporary approximate area stays in this tab's memory.
          state.position = { status: 'granted', lat: Number(point.lat.toFixed(3)), lng: Number(point.lng.toFixed(3)), observed_at: point.observed_at };
          state.gps = 'granted';
        } catch (error) {
          if (token !== state.sequence) return;
          state.gps = error?.unsupported ? 'unsupported' : error?.code === 1 ? 'denied' : error?.code === 3 ? 'timeout' : 'unavailable';
        }
      }
      state.phase = 'weather';
      const location = state.position || { status: state.gps };
      const response = await jsonFetch('/api/customer-care/recommend', {
        method: 'POST', headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ context: { language: language(), current_location: location } })
      }, token);
      if (token !== state.sequence) return;
      state.response = response; state.phase = response.weather?.status === 'ok' ? 'ready' : 'neutral';
    } catch (error) {
      if (token === state.sequence) { state.response = null; state.phase = 'neutral'; }
    } finally {
      if (token === state.sequence) {
        state.busy = false; display(state.response);
        if (state.languageDirty && state.userId !== null && shown()) {
          state.languageDirty = false;
          if (!state.response || state.response.language !== language()) void refreshAdvice(false);
        }
      }
    }
  }
  function login(me) {
    if (!me || me.role !== 'customer' || me.id === undefined || me.id === null) { clear(); return; }
    const id = String(me.id);
    if (state.userId !== id) { clear(); state.userId = id; }
    if (!shown()) return;
    display(state.response);
    if (state.attempted) return;
    if (document.visibilityState === 'hidden') { state.pending = true; return; }
    state.pending = false; state.attempted = true; void refreshAdvice(true);
  }
  const publicAPI = Object.freeze({ version: VERSION, login,
    refresh: () => refreshAdvice(true), dispose: clear,
    getStatus: () => ({ phase: state.phase, gps: state.gps, busy: state.busy, mounted: !!mounted() }) });
  window.MOVIOTravelCompanion = publicAPI;
  byId('language')?.addEventListener('change', () => {
    if (state.userId === null) return;
    if (state.busy) { state.languageDirty = true; display(state.response); return; }
    const hadResponse = state.response;
    invalidate(); state.response = null; display();
    if (hadResponse && shown()) void refreshAdvice(false);
  });
  byId('logout')?.addEventListener('click', clear, { capture: true });
  window.addEventListener('pagehide', clear);
  document.addEventListener('visibilitychange', () => {
    if (state.pending && document.visibilityState !== 'hidden' && shown()) {
      state.pending = false; state.attempted = true; void refreshAdvice(true);
    }
  });
  function syncExistingLogin() {
    if (typeof movioMe !== 'undefined' && movioMe && shown()) login(movioMe);
  }
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', syncExistingLogin, { once: true });
  else syncExistingLogin();
  window.addEventListener('pageshow', event => { if (event.persisted) syncExistingLogin(); });
})();
'''
    css=r'''/* Scoped to the companion card; existing dashboard controls retain their styles. */
#movioTravelCompanion{background:linear-gradient(120deg,#fff9e8 0%,var(--paper,#fff) 78%);border:1px solid #f2df9c;border-radius:17px;padding:12px 14px;margin:14px 0 16px;color:var(--ink,#17191d);min-width:0}
#movioTravelCompanion .movio-companion-head{display:flex;align-items:center;justify-content:space-between;gap:10px}
#movioTravelCompanion h2{font-family:inherit;font-size:13px;letter-spacing:0;line-height:1.35;margin:0;font-weight:800}
#movioTravelCompanion .movio-companion-refresh{flex:none;min-width:36px;min-height:36px;border:1px solid #eadca8;border-radius:11px;background:var(--paper,#fff);color:var(--ink,#17191d);font-size:22px;line-height:1;padding:4px;cursor:pointer}
#movioTravelCompanion .movio-companion-refresh:focus-visible{outline:2px solid #9e7300;outline-offset:2px}
#movioTravelCompanion .movio-companion-refresh:disabled{opacity:.45;cursor:wait}
#movioTravelCompanion .movio-companion-greeting{font-size:11px;color:#646a72;line-height:1.45;margin:2px 0 8px;overflow-wrap:anywhere}
#movioTravelCompanion .movio-companion-weather{display:flex;align-items:center;justify-content:space-between;gap:10px;margin:5px 0 9px;min-width:0}
#movioTravelCompanion [hidden]{display:none!important}
#movioTravelCompanion .movio-companion-primary{display:flex;align-items:center;gap:10px;min-width:0}
#movioTravelCompanion .movio-companion-primary>strong{font-family:Manrope,'DM Sans',Arial,sans-serif;font-size:29px;line-height:1.1;letter-spacing:-1.1px;font-weight:800;white-space:nowrap}
#movioTravelCompanion .movio-companion-summary{display:grid;gap:3px;min-width:0}
#movioTravelCompanion .movio-companion-summary>b{font-size:12px;line-height:1.25}
#movioTravelCompanion .movio-companion-summary>span{font-size:10px;color:#646a72;overflow-wrap:anywhere;line-height:1.3}
#movioTravelCompanion .movio-companion-rain{display:grid;gap:3px;text-align:right;flex:none;padding-left:9px;border-left:1px solid #eadca8}
#movioTravelCompanion .movio-companion-rain>b{font-size:19px;line-height:1.15}
#movioTravelCompanion .movio-companion-rain>span{font-size:9px;color:#646a72;max-width:70px;line-height:1.3}
#movioTravelCompanion .movio-companion-tips{list-style:none;padding:0;margin:7px 0 0;display:grid;gap:5px}
#movioTravelCompanion .movio-companion-tips>li{font-size:12px;line-height:1.45;margin:0;padding-left:10px;position:relative;overflow-wrap:anywhere}
#movioTravelCompanion .movio-companion-tips>li:before{content:'';position:absolute;left:0;top:.57em;width:4px;height:4px;border-radius:50%;background:#b0881b}
#movioTravelCompanion .movio-companion-note{font-size:9px;line-height:1.35;color:#76736a;margin:8px 0 0}
@media(max-width:600px){#movioTravelCompanion{padding:10px 12px;margin:11px 0 13px}#movioTravelCompanion .movio-companion-refresh{min-width:44px;min-height:44px}#movioTravelCompanion .movio-companion-primary>strong{font-size:26px}}
'''
    js_bytes,css_bytes=js.encode('utf-8'),css.encode('utf-8')
    for path in (index,auth):
        if path.is_symlink() or not path.is_file():raise RuntimeError('A dashboard source is missing or is a symlink. No files changed.')
    for path,expected in ((js_path,js_bytes),(css_path,css_bytes)):
        if path.is_symlink() or path.exists() and path.read_bytes()!=expected:
            raise RuntimeError('A different companion asset exists: '+path.name+'. No files were changed.')
    for path in web.glob('*.js'):
        if path!=js_path and re.search(r'customer.?care|companion',path.name,re.I):
            raise RuntimeError('Another companion client exists: '+path.name+'. Inspect it before installing overlapping UI.')
    old_index=index.read_bytes();old_auth=auth.read_bytes()
    old_tail="if(movioMe.role==='driver'&&(!movioMe.vehicle_type||!movioMe.vehicle_number))movioEditProfile()}"
    new_tail=old_tail[:-1]+";/* MOVIO_TRAVEL_COMPANION_LOGIN_V1 */window.MOVIOTravelCompanion?.login(movioMe)}"
    auth_text=old_auth.decode('utf-8')
    base_auth=auth_text.replace(new_tail,old_tail)
    if digest(base_auth.encode('utf-8'))!='f4d9fbb77b784c2f2e8cdddea80182a76a84f6ddd2d43d92b2974a86271177f6' or base_auth.count(old_tail)!=1:
        raise RuntimeError('The login function changed after inspection. No files or handlers were changed.')
    new_auth=base_auth.replace(old_tail,new_tail,1).encode('utf-8')
    if new_auth.decode().replace(new_tail,old_tail).encode()!=base_auth.encode():raise RuntimeError('Login-hook isolation failed.')
    style_block=('<!-- MOVIO_TRAVEL_COMPANION_STYLE_V1 -->\n'
        '<link rel="stylesheet" href="/customer_care_companion.css?v='+digest(css_bytes)[:12]+'">\n'
        '<!-- MOVIO_TRAVEL_COMPANION_STYLE_END_V1 -->\n')
    script_block=('<!-- MOVIO_TRAVEL_COMPANION_SCRIPT_V1 -->\n'
        '<script defer src="/customer_care_companion.js?v='+digest(js_bytes)[:12]+'"></script>\n'
        '<!-- MOVIO_TRAVEL_COMPANION_SCRIPT_END_V1 -->\n')
    html=old_index.decode('utf-8')
    for marker,block in (('STYLE',style_block),('SCRIPT',script_block)):
        opening='<!-- MOVIO_TRAVEL_COMPANION_'+marker+'_V1 -->'
        if opening in html:
            if html.count(block)!=1:raise RuntimeError('A different companion HTML block exists; no changes made.')
            html=html.replace(block,'',1)
    # Remove only our own auth cache token before comparing to the inspected page.
    html=re.sub(r"(src\s*=\s*['\"]auth\.js)\?movio_companion=[0-9a-f]{12}(['\"])",r'\1\2',html)
    if digest(html.encode('utf-8'))!='6a0e3baa973d2ad0b299ae1880bbc96ff7a1e31cbd15cacd24b12a88a783e86c':
        raise RuntimeError('The customer dashboard changed after inspection. No files were changed.')
    base_html=html
    if len(re.findall(r'</head\s*>',html,re.I))!=1 or len(re.findall(r'</body\s*>',html,re.I))!=1:
        raise RuntimeError('The dashboard head/body layout is unexpected; no changes made.')
    html=re.sub(r'</head\s*>',lambda match:style_block+match[0],html,count=1,flags=re.I)
    html=re.sub(r'</body\s*>',lambda match:script_block+match[0],html,count=1,flags=re.I)
    auth_pattern=r"(src\s*=\s*['\"]auth\.js)(['\"])"
    if len(re.findall(auth_pattern,html))!=1:raise RuntimeError('A unique auth script reference was not found.')
    html=re.sub(auth_pattern,lambda match:match[1]+'?movio_companion='+digest(new_auth)[:12]+match[2],html,count=1)
    isolated=html.replace(style_block,'',1).replace(script_block,'',1)
    isolated=re.sub(r"(src\s*=\s*['\"]auth\.js)\?movio_companion=[0-9a-f]{12}(['\"])",r'\1\2',isolated)
    if isolated!=base_html:raise RuntimeError('Dashboard-source isolation failed; no changes made.')
    class Page(HTMLParser):
        def __init__(self):super().__init__();self.scripts=[];self.links=[];self.ids=[]
        def handle_starttag(self,tag,attrs):
            fields=dict(attrs)
            if tag=='script' and fields.get('src'):self.scripts.append(fields['src'])
            if tag=='link' and fields.get('href'):self.links.append(fields['href'])
            if fields.get('id'):self.ids.append(fields['id'])
    before,after=Page(),Page();before.feed(base_html);after.feed(html)
    if before.ids!=after.ids or len(after.scripts)!=len(before.scripts)+1:
        raise RuntimeError('The original dashboard controls/script count changed unexpectedly.')
    if len([s for s in after.scripts if s.split('?')[0].endswith('customer_care_companion.js')])!=1:
        raise RuntimeError('Companion script is not unique.')
    node=shutil.which('node');syntax={}
    if node:
        with tempfile.TemporaryDirectory(prefix='movio_companion_syntax_') as folder:
            for name,value in (('auth.js',new_auth),('customer_care_companion.js',js_bytes)):
                path=Path(folder)/name;path.write_bytes(value)
                check=subprocess.run([node,'--check',str(path)],capture_output=True,text=True,timeout=15)
                if check.returncode:raise RuntimeError('JavaScript syntax check failed for '+name+': '+check.stderr[:500])
                syntax[name]='PASS'
    else:syntax['note']='Node.js unavailable in this kernel; JavaScript was browser-tested during preparation.'
    session=requests.Session();session.trust_env=False
    def fetch(path):return session.get('http://127.0.0.1:8080'+path,timeout=(1,4),allow_redirects=False)

    def probe_backend():
        """Read-only, unauthenticated probe. The version is readable only when the health route is public.
        A 401/403 means the route exists but wants a customer sign-in, so the version cannot be checked here."""
        info=dict(state='unreachable',version=None,health_status=None,route_status=None,proceed=False,message='')
        try:
            health=fetch('/api/customer-care/health')
        except requests.RequestException as error:
            info['message']='The website did not answer on 127.0.0.1:8080 ('+type(error).__name__+'). Start the website first. No UI files were changed.'
            return info
        info['health_status']=health.status_code
        try:body=health.json()
        except ValueError:body=None
        version=str(body.get('version','')) if isinstance(body,dict) else ''
        info['version']=version or None
        found=re.fullmatch(r'customer-care-api-(\d+)\.(\d+)',version)
        if health.status_code==200 and found and (int(found[1]),int(found[2]))>=(1,0):
            info.update(state='ready',proceed=True)
            return info
        if health.status_code in (401,403):
            info.update(state='sign_in_required',proceed=True)
            return info
        try:
            route=session.post('http://127.0.0.1:8080/api/customer-care/recommend',json={},timeout=(1,4),allow_redirects=False)
            info['route_status']=route.status_code
        except requests.RequestException:
            info['route_status']=None
        tail=(' To install the card anyway with the greeting and general tips only, run '
              'MOVIO_COMPANION_ALLOW_WITHOUT_WEATHER = True in an earlier cell, then run this cell again.')
        if health.status_code==200:
            info['state']='unexpected_version'
            info['message']=('The customer-care health route reports version '+repr(version or None)+', not 1.x or newer. No UI files were changed. '
                'Re-run the Step 3B backend cell, then this cell.'+tail)
        elif info['route_status'] in (401,403):
            info['state']='older_route'
            info['message']=('There is no customer-care health route (HTTP '+str(health.status_code)+'), but a recommendation route exists (HTTP '+str(info['route_status'])+'). '
                'That matches the older bridge, which returns no weather. No UI files were changed. Re-run the Step 3B backend cell, then this cell.'+tail)
        else:
            info['state']='missing'
            info['message']=('No customer-care routes were found (health HTTP '+str(health.status_code)+', recommend HTTP '+str(info['route_status'])+'). '
                'No UI files were changed. Re-run the customer-care backend cells, then this cell.'+tail)
        return info

    backend=probe_backend()
    allow_without_weather=bool(globals().get('MOVIO_COMPANION_ALLOW_WITHOUT_WEATHER'))
    if not backend['proceed'] and (not allow_without_weather or backend['state']=='unreachable'):
        session.close();raise RuntimeError(backend['message'])

    # Non-blocking evidence: which customer-care API versions are written in the project files.
    disk_versions={}
    for source_path in [root/'server.py']+sorted((root/'integrations').glob('*.py')):
        if not source_path.is_file() or source_path.is_symlink():continue
        try:source_text=source_path.read_text(encoding='utf-8',errors='ignore')
        except OSError:continue
        for found_version in sorted(set(re.findall(r'customer-care-api-\d+\.\d+',source_text))):
            disk_versions.setdefault(found_version,[]).append(str(source_path.relative_to(root)))

    protected={}
    targets={index,auth,js_path,css_path}
    paths={root/'server.py'}
    for folder in (web,root/'integrations'):
        paths.update(path for path in folder.rglob('*') if path.suffix in {'.py','.js','.html','.css'})
    for path in paths:
        if path in targets or not path.is_file() or path.is_symlink():continue
        if any('backup' in part.casefold() or part=='__pycache__' for part in path.relative_to(root).parts):continue
        protected[path]=digest(path.read_bytes())
    b=globals().get('bridge');engine=getattr(getattr(b,'_wx_endpoint',None),'__globals__',{})
    functions=[getattr(b,name,None) for name in ('speak','tanglish_speech_text','_wx_endpoint')]
    functions+=[engine.get(name) for name in ('prepare_turn','advisory','display_reply','tts_reply')]
    codes=[(function,function.__code__) for function in functions if isinstance(function,types.FunctionType)]
    def verify():
        for path,value in protected.items():
            if not path.is_file() or digest(path.read_bytes())!=value:raise RuntimeError('A protected source changed: '+str(path.relative_to(root)))
        if any(function.__code__ is not code for function,code in codes):raise RuntimeError('A protected voice handler changed.')
    plans=[(css_path,css_path.read_bytes() if css_path.is_file() else None,css_bytes),
           (js_path,js_path.read_bytes() if js_path.is_file() else None,js_bytes),
           (auth,old_auth,new_auth),(index,old_index,html.encode('utf-8'))]
    changes=[row for row in plans if row[1]!=row[2]]
    backup=None
    if changes:
        backup=root/('travel_companion_ui_backup_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
        backup.mkdir(exist_ok=False)
        manifest={}
        for path,old,new in changes:
            relative=str(path.relative_to(root));manifest[relative]=dict(existed=old is not None)
            if old is not None:
                saved=backup/relative;saved.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(path,saved)
                if saved.read_bytes()!=old:raise RuntimeError('Backup verification failed; no dashboard sources were replaced.')
        (backup/'restore_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    def atomic(path,value):
        mode=stat.S_IMODE(path.stat().st_mode) if path.is_file() else 0o644
        descriptor,temporary=tempfile.mkstemp(prefix='.movio_companion_',dir=path.parent)
        try:
            with os.fdopen(descriptor,'wb') as output:
                output.write(value);output.flush();os.fsync(output.fileno());os.fchmod(output.fileno(),mode)
            os.replace(temporary,path)
        finally:Path(temporary).unlink(missing_ok=True)
    applied=[]
    try:
        verify()
        for path,old,new in changes:
            if path.is_symlink() or (path.read_bytes() if path.is_file() else None)!=old:
                raise RuntimeError('A UI source changed during installation: '+path.name)
            atomic(path,new);applied.append((path,old,new))
        served={}
        for path,old,new in plans:
            url='/' if path==index else '/'+path.name+'?companion_check='+digest(new)[:12]
            response=fetch(url)
            if response.status_code!=200 or response.content!=new:raise RuntimeError('The website did not serve the updated '+path.name)
            served[path.name]='source bytes match'
        after_backend=probe_backend()
        if after_backend['state']!=backend['state'] or after_backend['version']!=backend['version']:
            raise RuntimeError('The customer-care API changed during UI installation (was '+backend['state']+' '+str(backend['version'])+', now '+after_backend['state']+' '+str(after_backend['version'])+').')
        verify()
    except BaseException:
        for path,old,new in reversed(applied):
            if path.is_file() and not path.is_symlink() and path.read_bytes()==new:
                if old is None:path.unlink()
                else:atomic(path,old)
        session.close();raise
    session.close()
    result=dict(step=4,version='travel-companion-1.0',passed=True,
        changed_files=[str(path.relative_to(root)) for path,_,_ in changes],
        backup=str(backup) if backup else None,javascript_syntax=syntax,served_assets=served,
        backend_state=backend['state'],backend_version=backend['version'],
        weather_backend_verified=backend['state']=='ready',
        on_disk_api_versions=disk_versions,
        source_isolation='PASS: one login hook and companion asset links; original controls retained',
        website_restarted=False,voice_handlers_preserved=True,
        automatic_location='One request after customer sign-in; no tracking watch or automatic booking-pickup selection.',
        browser_verification='REQUIRED: reload the customer dashboard, allow or deny location, check the card and an unchanged booking pickup.')
    print(json.dumps(result,ensure_ascii=False,indent=2))
    if backend['state']=='sign_in_required':
        print('NOTE: the weather backend version could not be verified without a customer sign-in (its health route is protected).')
        print('Sign in as a customer after reloading. If the weather row and locality appear, Step 3B is active.')
        print('If only the greeting and a general tip appear, re-run the Step 3B backend cell and reload.')
    elif backend['state']!='ready':
        print('NOTE: installed WITHOUT a verified weather backend (state: '+backend['state']+'). The card shows the greeting and general tips only until Step 3B is active.')
    print('Reload the customer dashboard with Ctrl+Shift+R; on mobile, reload the page.')
    print('Allow location when the browser asks, or deny it to check the neutral greeting.')
    print('Share the output and whether locality, temperature and rain chance appear. Do not share precise GPS coordinates.')
    return result


MOVIO_CUSTOMER_CARE_STEP_4=movio_add_travel_companion_card()

{
  "step": 4,
  "version": "travel-companion-1.0",
  "passed": true,
  "changed_files": [],
  "backup": null,
  "javascript_syntax": {
    "auth.js": "PASS",
    "customer_care_companion.js": "PASS"
  },
  "served_assets": {
    "customer_care_companion.css": "source bytes match",
    "customer_care_companion.js": "source bytes match",
    "auth.js": "source bytes match",
    "index.html": "source bytes match"
  },
  "backend_state": "sign_in_required",
  "backend_version": null,
  "weather_backend_verified": false,
  "on_disk_api_versions": {
    "customer-care-api-0.1": [
      "integrations/movio_customer_care_api.py"
    ]
  },
  "source_isolation": "PASS: one login hook and companion asset links; original controls retained",
  "website_restarted": false,
  "voice_handlers_preserved": true,
  "automatic_location": "One request after customer sign-in; no tracking watch or automatic booking-pickup selection.",
  "browser_verification": "REQUIRED: reload the customer dashboard, allow

In [137]:
# Run this ENTIRE file in ONE Kaggle cell after your working voice setup,
# before evaluation. Inspection only: no patches, bookings, API calls or TTS.
def movio_inspect_speech_and_slots():
    import ast
    import hashlib
    import json
    import linecache
    import re
    import types
    from pathlib import Path

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    b = globals().get('bridge')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    if not b or not callable(engine.get('collect_details')):
        raise RuntimeError('Run in the original, currently working MOVIO voice kernel.')
    report = dict(step='speech-and-slots-1-inspection', changed_files=[],
                  parser_tests=[], files={}, active_functions={}, lookup_events={})
    snapshots = {}
    parsed = {}

    def clean(value):
        value = re.sub(r'(?i)(Bearer\s+)[A-Za-z0-9._-]+', r'\1[REDACTED]', str(value))
        value = re.sub(r'''(?i)((?:api[_-]?(?:key|token)|auth[_-]?token|subscription[_-]?key|password|secret)\s*[:=]\s*)["'][^"']*["']''', r'\1"[REDACTED]"', value)
        return re.sub(r'''(?i)([?&](?:token|key|api_key|access_token)=)[^&\s"']+''', r'\1[REDACTED]', value)

    def read(path):
        path = Path(path)
        if not path.is_file():
            return ''.join(linecache.getlines(str(path)))
        data = path.read_bytes()
        if path.is_relative_to(root):
            snapshots[str(path)] = hashlib.sha256(data).hexdigest()
        return data.decode('utf-8-sig')

    def compiled(source, path):
        key = (str(path), hashlib.sha256(source.encode()).hexdigest())
        if key not in parsed:
            parsed[key] = (ast.parse(source), compile(source, str(path), 'exec', dont_inherit=True))
        return parsed[key]

    def signature(code):
        constants = tuple(signature(v) if isinstance(v, types.CodeType) else v for v in code.co_consts)
        return (code.co_code, constants, code.co_names, code.co_varnames, code.co_freevars,
                code.co_cellvars, code.co_argcount, code.co_kwonlyargcount,
                code.co_posonlyargcount, code.co_flags, getattr(code, 'co_exceptiontable', b''))

    def children(code):
        for value in code.co_consts:
            if isinstance(value, types.CodeType):
                yield value
                yield from children(value)

    def describe(fn):
        if isinstance(fn, types.MethodType):
            fn = fn.__func__
        if not isinstance(fn, types.FunctionType):
            return dict(available=False)
        code = fn.__code__
        item = dict(name=fn.__qualname__, file=code.co_filename,
                    free_variables=list(code.co_freevars), source_verified=False)
        try:
            source = read(code.co_filename)
            tree, module_code = compiled(source, code.co_filename)
            matches = [c for c in children(module_code)
                       if c.co_name == code.co_name and signature(c) == signature(code)]
            if len(matches) == 1:
                nodes = [n for n in ast.walk(tree)
                         if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
                         and n.name == matches[0].co_name
                         and min([n.lineno] + [d.lineno for d in n.decorator_list]) == matches[0].co_firstlineno]
                if len(nodes) == 1:
                    item.update(source_verified=True, source=clean(ast.get_source_segment(source, nodes[0])))
        except (OSError, SyntaxError, UnicodeError, ValueError, TypeError, RecursionError) as exc:
            item['source_error'] = type(exc).__name__
        return item

    slots = getattr(b, '_wx_group_slots_module', None)
    if slots is None:
        collector = engine['collect_details']
        for name, cell in zip(getattr(collector, '__code__', types.SimpleNamespace(co_freevars=())).co_freevars,
                              getattr(collector, '__closure__', None) or ()):
            if name == 'slots':
                slots = cell.cell_contents
    ns = slots if isinstance(slots, dict) else vars(slots) if slots is not None else {}
    extract = ns.get('extract_fields')
    report['parser_version'] = ns.get('VERSION')
    report['active_patterns'] = {name: getattr(ns.get(name), 'pattern', None)
                                 for name in ('_TA_FROM', '_ROMAN_FROM', '_CARRIER', '_TIME')}

    # Detached rule-only cases. Never call collect_details/prepare_turn here:
    # those can invoke a model, lookups or the real dialogue state machine.
    samples = [
        ('airport-first-turn', 'நான் Chennai airport-ல இருந்து Tambaram போகணும். நாங்க 3 பேர் இருக்கோம். எனக்கு SUV வேணும். இன்னைக்கு 11 pm-க்கு வண்டி வேணும்.'),
        ('central-inflected-count', 'நான் Chennai Central-ல இருந்து Tambaram போகணும். நாங்க 3 பேருக்கும் SUV வேணும். இன்னைக்கு 01:00 மணிக்கு book பண்ணு.'),
        ('central-plain-count', 'நான் Chennai Central-ல இருந்து Tambaram போகணும். நாங்கள் 3 பேர் இருக்கோம். SUV வேணும்.'),
        ('romanized-route', 'Chennai airport la irundhu Tambaram poganum. Naanga 3 peru. SUV venum. Innaiku 11 PM.'),
        ('english-control', 'From Chennai Airport to Tambaram. We are 3 people. SUV today at 11 PM.'),
        ('route-only-repetition', 'Chennai airport to Tambaram'),
    ]
    expected = dict(count=3, vehicle='SUV', destination='Tambaram')
    for case, text in samples:
        for language in ('Tamil', 'Tanglish'):
            row = dict(case=case, language=language, input=text)
            state = dict(mode='book', stage='count', lang=language)
            before = dict(state)
            try:
                if not callable(extract):
                    raise RuntimeError('Active shared parser was not found')
                result = extract(b, text, state, use_qwen=False, engine=engine)
                updates = result.get('updates', {})
                row.update(updates=updates, ambiguities=result.get('ambiguities', {}),
                           evidence=result.get('evidence', {}), test_state_unchanged=state == before)
                if case != 'route-only-repetition':
                    row['missing_or_different'] = {k: dict(expected=v, actual=updates.get(k))
                                                   for k, v in expected.items() if updates.get(k) != v}
                row['pickup_found'] = bool(updates.get('pickup'))
            except Exception as exc:
                row['error_type'] = type(exc).__name__
            report['parser_tests'].append(row)

    # Save only relevant definitions, not credentials, live sessions or bookings.
    selected = {
        'movio_group_booking_slots.py': {'count', 'clean_place', 'add_place', 'route', 'parse', 'collect_details', 'extract_fields'},
        'movio_friendly_weather.py': {'report', 'question', 'advisory', 'speech_segments'},
        'movio_tanglish_speech_quality.py': {'speech_text'},
        'movio_time_route.py': {'collect_details', 'geocode', 'road_fare'},
    }
    for filename, wanted in selected.items():
        path = root / 'integrations' / filename
        if not path.is_file():
            report['files'][filename] = dict(exists=False)
            continue
        source = read(path)
        try:
            tree, _ = compiled(source, path)
            functions = [dict(name=n.name, line=n.lineno, source=clean(ast.get_source_segment(source, n)))
                         for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name in wanted]
            report['files'][filename] = dict(sha256=snapshots[str(path)], functions=functions)
        except (SyntaxError, ValueError) as exc:
            report['files'][filename] = dict(error_type=type(exc).__name__)

    for name in ('collect_details', 'speech_segments', 'tts_reply', 'advisory', 'display_reply'):
        report['active_functions'][name] = describe(engine.get(name))
    report['active_functions']['bridge.tanglish_speech_text'] = describe(getattr(b, 'tanglish_speech_text', None))
    for name in ('count', 'clean_place', 'add_place', 'route', 'parse'):
        report['active_functions']['Parser.' + name] = describe(getattr(ns.get('Parser'), name, None))
    # Inspect current lookup wrappers and their underlying original functions.
    # Never invoke them: no network requests, cache resets or provider calls.
    seen = set()
    for prefix in ('geocode', 'road_fare'):
        queue = [(prefix, getattr(b, prefix, None), 0)]
        while queue:
            label, fn, depth = queue.pop(0)
            if not callable(fn) or id(fn) in seen or depth > 4:
                continue
            seen.add(id(fn))
            report['active_functions']['bridge.' + label] = describe(fn)
            original = getattr(fn, '_original_geocode', None)
            if callable(original):
                queue.append((label + '.original', original, depth + 1))
            for key, cell in zip(getattr(getattr(fn, '__code__', None), 'co_freevars', ()),
                                 getattr(fn, '__closure__', None) or ()):
                if key in ('original_geo', 'original_route', 'previous', 'raw'):
                    value = cell.cell_contents
                    if callable(value):
                        queue.append((label + '.' + key, value, depth + 1))

    for filename in ('booking_lookup.log', 'required_weather.log'):
        path = root / filename
        if not path.is_file():
            continue
        with path.open('rb') as handle:
            handle.seek(max(0, path.stat().st_size - 24000))
            lines = handle.read().decode('utf-8', errors='replace').splitlines()
        events = []
        for line in lines:
            try:
                item = json.loads(line[line.index('{'):])
                events.append({k: item[k] for k in ('event', 'reason', 'error_type', 'http_status', 'location', 'pickup', 'destination') if k in item})
            except (ValueError, TypeError):
                continue
        report['lookup_events'][filename] = events[-10:]
    report['source_unchanged'] = all(Path(p).is_file() and hashlib.sha256(Path(p).read_bytes()).hexdigest() == digest
                                     for p, digest in snapshots.items())
    report['scope'] = 'Rules and source inspection only; no API calls, model generation, booking changes or speech.'
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('Share the complete output. The next cell will target the verified active handlers.')
    return report


MOVIO_SPEECH_SLOT_AUDIT = movio_inspect_speech_and_slots()


{
  "step": "speech-and-slots-1-inspection",
  "changed_files": [],
  "parser_tests": [
    {
      "case": "airport-first-turn",
      "language": "Tamil",
      "input": "நான் Chennai airport-ல இருந்து Tambaram போகணும். நாங்க 3 பேர் இருக்கோம். எனக்கு SUV வேணும். இன்னைக்கு 11 pm-க்கு வண்டி வேணும்.",
      "updates": {
        "count": 3,
        "vehicle": "SUV",
        "pickup": "Chennai Airport",
        "destination": "க்கு"
      },
      "ambiguities": {},
      "evidence": {
        "count": {
          "text": "3 பேர்",
          "source": "rule",
          "confidence": 1.0
        },
        "vehicle": {
          "text": "SUV",
          "source": "rule",
          "confidence": 1.0
        },
        "pickup": {
          "text": "Chennai airport-ல இருந்து",
          "source": "rule",
          "confidence": 1.0
        },
        "destination": {
          "text": "-க்கு",
          "source": "rule",
          "confidence": 1.0
        }
      },
      "test_state_unchan

In [138]:
# MOVIO STEP 2 — fix passenger inflections and time-suffix location overwrites.
# Close the current AI Call. Paste this ENTIRE file into ONE Kaggle cell
# after the working voice setup and BEFORE evaluation. No model restart needed.
def movio_fix_passenger_and_route_suffixes():
    import ast
    import copy
    import hashlib
    import itertools
    import json
    import os
    import re
    import shutil
    import sys
    import tempfile
    import types
    from datetime import datetime
    from pathlib import Path

    replacements = {'old': {'count': "def count(self):\n        words=number_words(self.engine)\n        token=r'\\d{1,2}|'+'|'.join(re.escape(k) for k in sorted(words,key=len,reverse=True) if not k.isdigit())\n        unit=r'(?:பயணிகள்|நபர்கள்|பேர்கள்|நபர்|பேரு|பேர்|passengers?|people|persons?|pax|peru|paeru|per)'\n        pattern=r'(?<!['+_LETTERS+r'\\d])(?P<n>'+token+r')\\s*'+unit+r'(?!['+_LETTERS+r'])'\n        found=list(re.finditer(pattern,self.scan,re.I))\n        if not found:\n            m=re.search(r'(?<![a-z])(?:we\\s+are|there\\s+are|for)\\s+(?P<n>'+token+r')(?=\\s*(?:of\\s+us\\b|$))',self.scan,re.I)\n            if m:found=[m]\n        if not found and self.state.get('stage')=='count':\n            m=re.fullmatch(r'\\s*(?P<n>'+token+r')\\s*[.!]?\\s*',self.scan,re.I)\n            if m:found=[m]\n        if not found:return\n        choices=[]\n        for m in found:\n            value=normalize(m['n']);n=int(value) if value.isdigit() else words[value]\n            self.noise.append(m.span())\n            if not re.search(r'\\bnot\\s*$',self.scan[max(0,m.start()-12):m.start()],re.I):choices.append((n,m))\n        if not choices:return\n        if len({n for n,_ in choices})>1 and not _CORRECTION.search(self.scan):\n            self.conflicts['count']='More than one passenger count stated';return\n        n,m=choices[-1];self.accept('count',n,self.raw[m.start():m.end()])", 'clean_place': "def clean_place(self,value,role):\n        text=str(value or '').strip(' \\t\\r\\n.,;:!?-–—')\n        text=re.sub(r'(?i)^(?:i\\s+(?:want|need)(?:\\s+to\\s+(?:go|travel))?|i\\s+am|my\\s+pickup\\s+is|my\\s+destination\\s+is|please|naan|nan|enakku|enaku|நான்|எனக்கு)\\s+','',text).strip()\n        text=re.sub(r'(?i)\\s+(?:for|with|and|at|from|to)\\s*$','',text).strip(' ,-')\n        # Match complete known names/case forms, not a place hidden inside a sentence.\n        if place_key(text) in self.places:return self.places[place_key(text)]\n        # Separate case markers can be removed without changing the spoken name.\n        text=re.sub(r'(?i)(?:[-\\s]+(?:இல்\\s*இருந்து|இலிருந்து|லிருந்து|இருந்து|க்கு|வரை|la\\s*irun[dt]hu|il\\s*irun[dt]hu|irun[dt]hu|la|ku|kku|varai|varaikkum))$','',text).strip(' ,-')\n        if place_key(text) in self.places:return self.places[place_key(text)]\n        if not text or len(text)>180 or not re.search(r'[A-Za-z\\u0B80-\\u0BFF]',text):return None\n        if normalize(text) in {'time','date','fare','vehicle','booking','car','cab','taxi','நேரம்','தேதி','இடம்'}:return None\n        if _FILLER.match(text) or re.fullmatch(r'(?:இரண்டு|ரெண்டு|\\d+)',text):return None\n        if any(re.fullmatch(pattern,text,re.I) for _,pattern in _VEHICLES):return None\n        if _TIME.search(text) or re.search(r'(?<![a-z])(?:poganum|venum|irukkom|we are|passengers?|people)(?![a-z])|வேணும்|வேண்டும்|இருக்கோம்|பேர்',text,re.I):return None\n        canonicalizer=getattr(self.bridge,'canonical_place',None)\n        canonical=canonicalizer(text) if callable(canonicalizer) else text\n        return str(canonical).strip()[:120] if canonical else None"}, 'new': {'count': 'def count(self):\n        """MOVIO slot-suffix-1.0: recognize explicit inflected passenger nouns."""\n        words=number_words(self.engine)\n        token=r\'\\d{1,2}|\'+\'|\'.join(re.escape(k) for k in sorted(words,key=len,reverse=True) if not k.isdigit())\n        unit=r\'(?:பேருக்காக|பேருக்கும்|பேருக்கு|பேருடன்|பேரோடு|பேரும்|பேரை|பயணிகள்|நபர்கள்|பேர்கள்|நபர்|பேரு|பேர்|passengers?|people|persons?|pax|peru|paeru|per)\'\n        pattern=r\'(?<![\'+_LETTERS+r\'\\d])(?P<n>\'+token+r\')\\s*\'+unit+r\'(?![\'+_LETTERS+r\'])\'\n        found=list(re.finditer(pattern,self.scan,re.I))\n        if not found:\n            m=re.search(r\'(?<![a-z])(?:we\\s+are|there\\s+are|for)\\s+(?P<n>\'+token+r\')(?=\\s*(?:of\\s+us\\b|$))\',self.scan,re.I)\n            if m:found=[m]\n        if not found and self.state.get(\'stage\')==\'count\':\n            m=re.fullmatch(r\'\\s*(?P<n>\'+token+r\')\\s*[.!]?\\s*\',self.scan,re.I)\n            if m:found=[m]\n        if not found:return\n        choices=[]\n        for m in found:\n            value=normalize(m[\'n\']);n=int(value) if value.isdigit() else words[value]\n            self.noise.append(m.span())\n            if not re.search(r\'\\bnot\\s*$\',self.scan[max(0,m.start()-12):m.start()],re.I):choices.append((n,m))\n        if not choices:return\n        if len({n for n,_ in choices})>1 and not _CORRECTION.search(self.scan):\n            self.conflicts[\'count\']=\'More than one passenger count stated\';return\n        n,m=choices[-1];self.accept(\'count\',n,self.raw[m.start():m.end()])', 'clean_place': 'def clean_place(self,value,role):\n        """MOVIO slot-suffix-1.0: a grammatical suffix is never a location."""\n        text=str(value or \'\').strip(\' \\t\\r\\n.,;:!?-–—\')\n        if normalize(text) in {\'க்கு\',\'கு\',\'க்கே\',\'க்கும்\',\'க்குப்\',\'ல\',\'இல்\',\'இருந்து\',\'வரை\'}:return None\n        text=re.sub(r\'(?i)^(?:i\\s+(?:want|need)(?:\\s+to\\s+(?:go|travel))?|i\\s+am|my\\s+pickup\\s+is|my\\s+destination\\s+is|please|naan|nan|enakku|enaku|நான்|எனக்கு)\\s+\',\'\',text).strip()\n        text=re.sub(r\'(?i)\\s+(?:for|with|and|at|from|to)\\s*$\',\'\',text).strip(\' ,-\')\n        # Match complete known names/case forms, not a place hidden inside a sentence.\n        if place_key(text) in self.places:return self.places[place_key(text)]\n        # Separate case markers can be removed without changing the spoken name.\n        text=re.sub(r\'(?i)(?:[-\\s]+(?:இல்\\s*இருந்து|இலிருந்து|லிருந்து|இருந்து|க்கு|வரை|la\\s*irun[dt]hu|il\\s*irun[dt]hu|irun[dt]hu|la|ku|kku|varai|varaikkum))$\',\'\',text).strip(\' ,-\')\n        if place_key(text) in self.places:return self.places[place_key(text)]\n        if not text or len(text)>180 or not re.search(r\'[A-Za-z\\u0B80-\\u0BFF]\',text):return None\n        if normalize(text) in {\'time\',\'date\',\'fare\',\'vehicle\',\'booking\',\'car\',\'cab\',\'taxi\',\'நேரம்\',\'தேதி\',\'இடம்\'}:return None\n        if _FILLER.match(text) or re.fullmatch(r\'(?:இரண்டு|ரெண்டு|\\d+)\',text):return None\n        if any(re.fullmatch(pattern,text,re.I) for _,pattern in _VEHICLES):return None\n        if _TIME.search(text) or re.search(r\'(?<![a-z])(?:poganum|venum|irukkom|we are|passengers?|people)(?![a-z])|வேணும்|வேண்டும்|இருக்கோம்|பேர்\',text,re.I):return None\n        canonicalizer=getattr(self.bridge,\'canonical_place\',None)\n        canonical=canonicalizer(text) if callable(canonicalizer) else text\n        return str(canonical).strip()[:120] if canonical else None'}, 'original_sha256': '3b23c66f0047365ba1a60abd29154c06aec599a7f05c856b431fb1b4673f9478'}
    version = 'slot-suffix-1.0'
    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    target = root / 'integrations/movio_group_booking_slots.py'
    if target.is_symlink() or not target.is_file():
        raise RuntimeError('Expected the existing, regular shared booking-slot file.')
    before = target.read_bytes()
    source = before.decode('utf-8-sig')
    b = globals().get('bridge')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    if not b or not callable(engine.get('collect_details')):
        raise RuntimeError('Run in the original working MOVIO voice kernel.')

    def signature(code):
        return (code.co_code, tuple(signature(x) if isinstance(x, types.CodeType) else x for x in code.co_consts),
                code.co_names, code.co_varnames, code.co_freevars, code.co_cellvars,
                code.co_argcount, code.co_posonlyargcount, code.co_kwonlyargcount, code.co_flags,
                getattr(code, 'co_exceptiontable', b''))

    def nested(code):
        for value in code.co_consts:
            if isinstance(value, types.CodeType):
                yield value
                yield from nested(value)

    def method_code(module_code, name):
        matches = [x for x in nested(module_code) if x.co_name == name
                   and getattr(x, 'co_qualname', 'Parser.' + name) == 'Parser.' + name]
        if len(matches) != 1:
            raise RuntimeError('Could not identify the original Parser.' + name)
        return matches[0]

    tree = ast.parse(source, filename=str(target))
    classes = [n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == 'Parser']
    if len(classes) != 1:
        raise RuntimeError('The shared Parser layout changed; nothing was modified.')
    edits = []
    for name in ('count', 'clean_place'):
        nodes = [n for n in classes[0].body if isinstance(n, ast.FunctionDef) and n.name == name]
        if len(nodes) != 1:
            raise RuntimeError('Parser.' + name + ' is missing or duplicated.')
        node = nodes[0]
        text = ast.get_source_segment(source, node)
        if text == replacements['new'][name]:
            continue
        if text != replacements['old'][name]:
            raise RuntimeError('Parser.' + name + ' differs from the inspected version; no changes were made.')
        edits.append((node.lineno - 1, node.end_lineno, replacements['new'][name]))
    lines = source.splitlines(keepends=True)
    for start, stop, text in sorted(edits, reverse=True):
        # ast.get_source_segment omits only the initial class indentation.
        lines[start:stop] = ['    ' + text + '\n']
    after_source = ''.join(lines)
    original_code = compile(source, str(target), 'exec', dont_inherit=True)
    candidate_code = compile(after_source, str(target), 'exec', dont_inherit=True)
    # Everything except these two original method bodies must be identical.
    def isolated_ast(value):
        value = ast.parse(value)
        parser = next(n for n in value.body if isinstance(n, ast.ClassDef) and n.name == 'Parser')
        for n in parser.body:
            if isinstance(n, ast.FunctionDef) and n.name in ('count', 'clean_place'):
                n.body = [ast.Pass()]
        return ast.dump(value, include_attributes=False)
    if isolated_ast(source) != isolated_ast(after_source):
        raise RuntimeError('Source isolation check failed; nothing was modified.')

    slots = getattr(b, '_wx_group_slots_module', None)
    if slots is None:
        fn = engine['collect_details']
        for key, cell in zip(getattr(fn.__code__, 'co_freevars', ()), fn.__closure__ or ()):
            if key == 'slots':
                slots = cell.cell_contents
    active = slots if isinstance(slots, dict) else vars(slots) if slots is not None else {}
    if not isinstance(active.get('Parser'), type) or not callable(active.get('extract_fields')):
        raise RuntimeError('The active shared slot parser was not found; no changes were made.')

    # Keep all existing collectors, time validation and wrapper identities.
    # Updating the base function object also updates the reference captured
    # by the existing time guard. No additional parser wrappers are installed.
    namespaces = [active]
    possible = [getattr(x, '__dict__', {}) for x in list(sys.modules.values()) if x is not None]
    old_collector = engine.get('_movio_optional_weather_original', {}).get('collect_details')
    if isinstance(old_collector, types.FunctionType):
        possible.append(old_collector.__globals__)
    for ns in possible:
        path = ns.get('__file__')
        if not path or not isinstance(ns.get('Parser'), type):
            continue
        try:
            same = Path(path).resolve() == target
        except (OSError, TypeError, ValueError):
            same = False
        if same and all(ns is not v for v in namespaces):
            namespaces.append(ns)

    plans = []
    base_map = {}
    for ns in namespaces:
        for name in ('count', 'clean_place'):
            fn = getattr(ns['Parser'], name, None)
            queue, found, seen = [fn], [], set()
            while queue:
                fn = queue.pop()
                if not isinstance(fn, types.FunctionType) or id(fn) in seen:
                    continue
                seen.add(id(fn))
                if fn.__globals__ is ns and fn.__code__.co_name == name:
                    found.append(fn)
                for cell in fn.__closure__ or ():
                    try:
                        value = cell.cell_contents
                    except ValueError:
                        continue
                    if isinstance(value, types.FunctionType):
                        queue.append(value)
            if len(found) != 1:
                raise RuntimeError('Cannot safely identify the active base Parser.' + name)
            fn = found[0]
            old_code = method_code(original_code, name)
            new_code = method_code(candidate_code, name)
            if signature(fn.__code__) not in (signature(old_code), signature(new_code)):
                raise RuntimeError('The live base Parser.' + name + ' differs from the inspected file.')
            base_map[fn] = name
            if all(fn is not p[0] for p in plans):
                plans.append((fn, fn.__code__, new_code))

    candidate = dict(__name__='_movio_suffix_fixture', __file__=str(target))
    exec(candidate_code, candidate)
    candidate['_ENGINE'] = engine
    # Use the actual patched clock/carrier patterns in isolated fixtures.
    for key in ('_TIME', '_CARRIER'):
        candidate[key] = active[key]
    original_parser = active['Parser']
    candidate_parser = candidate['Parser']
    clones = {}
    def cell_for(value):
        return (lambda: value).__closure__[0]
    def clone(fn):
        if not isinstance(fn, types.FunctionType):
            return fn
        if fn in base_map:
            return getattr(candidate_parser, base_map[fn])
        if fn in clones:
            return clones[fn]
        if fn.__globals__ is active:
            value = candidate.get(fn.__name__)
            if callable(value):
                return value
        cells = []
        for cell in fn.__closure__ or ():
            value = cell.cell_contents
            if value is active:
                value = candidate
            elif isinstance(value, types.FunctionType):
                value = clone(value)
            cells.append(cell_for(value))
        defaults = tuple(clone(v) for v in fn.__defaults__) if fn.__defaults__ else None
        result = types.FunctionType(fn.__code__, fn.__globals__, fn.__name__, defaults,
                                    tuple(cells) if cells else None)
        result.__kwdefaults__ = {k: clone(v) for k, v in (fn.__kwdefaults__ or {}).items()} or None
        clones[fn] = result
        return result
    candidate_parser.clean_place = clone(original_parser.clean_place)
    candidate_parser.add_place = clone(original_parser.add_place)

    # A detached bridge prevents any Qwen, weather, geocoder or TTS calls.
    fixture_bridge = types.SimpleNamespace(
        VOICE_PLACE_ALIASES=copy.deepcopy(getattr(b, 'VOICE_PLACE_ALIASES', {})),
        GROUP_PLACE_ALIASES=copy.deepcopy(getattr(b, 'GROUP_PLACE_ALIASES', {})),
        canonical_place=lambda value: value,
        _wx_context_extract=None)
    def fixture_canonical(value):
        # Mirror the existing pure alias lookup without invoking any bridge
        # callbacks. This also preserves known names with spoken case endings.
        key = candidate['place_key'](value)
        matches = []
        for canonical, aliases in fixture_bridge.VOICE_PLACE_ALIASES.items():
            variants = (canonical,) + tuple(aliases)
            if any(re.search(r'(?<![A-Za-z])' + re.escape(candidate['place_key'](alias)) + r'(?![A-Za-z])', key)
                   for alias in variants if str(alias).strip()):
                matches.append(canonical)
        return matches[0] if len(matches) == 1 else value
    fixture_bridge.canonical_place = fixture_canonical
    checks = 0
    def check(condition, label):
        nonlocal checks
        if not condition:
            raise RuntimeError('Detached parser regression failed: ' + label)
        checks += 1
    airport = 'நான் Chennai airport-ல இருந்து Tambaram போகணும். நாங்க 3 பேர் இருக்கோம். எனக்கு SUV வேணும். இன்னைக்கு 11 pm-க்கு வண்டி வேணும்.'
    central = 'நான் Chennai Central-ல இருந்து Tambaram போகணும். நாங்க 3 பேருக்கும் SUV வேணும். இன்னைக்கு 01:00 மணிக்கு book பண்ணு.'
    def run_checks(ns):
        extract = ns['extract_fields']
        result_samples = []
        for language in ('English', 'Tamil', 'Tanglish'):
            for label, text, pickup in (('airport', airport, 'Chennai Airport'), ('central', central, 'Chennai Central')):
                state = dict(mode='book', stage='count', lang=language)
                before_state = copy.deepcopy(state)
                updates = extract(fixture_bridge, text, state, use_qwen=False, engine=engine)['updates']
                want = dict(count=3, vehicle='SUV', pickup=pickup, destination='Tambaram')
                check(all(updates.get(k) == v for k, v in want.items()), language + ':' + label + ':' + str(updates))
                check(state == before_state, 'read-only extraction preserves test state')
                result_samples.append(dict(language=language, case=label, updates=updates))
            clauses = ('நான் Chennai airport-ல இருந்து Tambaram போகணும்.', 'நாங்க 3 பேருக்கும்.', 'SUV வேணும்.', 'இன்னைக்கு 11 PM-க்கு வண்டி வேணும்.')
            for order in itertools.permutations(clauses):
                updates = extract(fixture_bridge, ' '.join(order), {'mode':'book','stage':'count','lang':language}, use_qwen=False, engine=engine)['updates']
                check(all(updates.get(k) == v for k, v in dict(count=3, vehicle='SUV', pickup='Chennai Airport', destination='Tambaram').items()), 'slot order:' + str(updates))
        for unit in ('பேருக்கும்', 'பேருக்கு', 'பேருடன்', 'பேரோடு', 'பேரும்', 'பேரை', 'பேருக்காக', 'பேர்', 'பேரு', 'people', 'passengers', 'peru'):
            updates = extract(fixture_bridge, '3 ' + unit, {'mode':'book','stage':'count'}, use_qwen=False, engine=engine)['updates']
            check(updates.get('count') == 3, 'passenger unit:' + unit)
        for suffix in ('க்கு', '-க்கு', 'கு', 'க்கே', 'க்கும்', 'க்குப்', 'ல', 'இல்', 'இருந்து', 'வரை'):
            parser = ns['Parser'](fixture_bridge, {}, '', engine)
            check(parser.clean_place(suffix, 'destination') is None, 'reject orphan suffix:' + suffix)
        routes = (
            ('From Chennai Airport to Tambaram', 'Chennai Airport', 'Tambaram'),
            ('Chennai airport la irundhu Tambaram poganum', 'Chennai Airport', 'Tambaram'),
            ('தாம்பரத்திலிருந்து வண்டலூர் போகணும்', 'Tambaram', 'Vandalur'),
            ('Tambaram to Vandalur', 'Tambaram', 'Vandalur'),
        )
        for text, pickup, destination in routes:
            updates = extract(fixture_bridge, text, {'stage':'count'}, use_qwen=False, engine=engine)['updates']
            check(updates.get('pickup') == pickup and updates.get('destination') == destination, 'existing route:' + str(updates))
        for text in ('11 PM-க்கு வண்டி வேணும்', 'இன்னைக்கு 01:00 மணிக்கு book பண்ணு', 'Today at 3 PM'):
            updates = extract(fixture_bridge, text, {'stage':'vehicle'}, use_qwen=False, engine=engine)['updates']
            check(not any(k in updates for k in ('count','pickup','destination')), 'time does not become another slot:' + str(updates))
        conflict = extract(fixture_bridge, '3 பேருக்கும். 4 பேருக்கும்.', {}, use_qwen=False, engine=engine)
        check('count' in conflict['ambiguities'], 'conflicting counts still require clarification')
        conflict = extract(fixture_bridge, 'SUV வேணும். Bike வேணும்.', {}, use_qwen=False, engine=engine)
        check('vehicle' in conflict['ambiguities'], 'conflicting vehicles preserved')
        state = dict(mode='book', stage='vehicle', lang='Tamil', pickup='Chennai Airport', destination='Tambaram',
                     count=3, vehicle='Bike', pickup_at='2030-01-01T23:00:00+05:30')
        ns['collect_details'](fixture_bridge, state, 'Change to Mini')
        check(state.get('vehicle') == 'Mini', 'vehicle correction')
        check(all(state.get(k) == v for k,v in dict(count=3,pickup='Chennai Airport',destination='Tambaram',pickup_at='2030-01-01T23:00:00+05:30').items()), 'correction retains unrelated slots')
        cancel = dict(mode='cancel', stage='cancel_confirm', count=3, vehicle='SUV')
        ns['collect_details'](fixture_bridge, cancel, '2 people, Mini')
        check(cancel == dict(mode='cancel', stage='cancel_confirm', count=3, vehicle='SUV'), 'cancellation collector unaffected')
        return result_samples
    run_checks(candidate)
    offline_checks = checks

    protected = {name: engine.get(name) for name in ('collect_details', 'prepare_turn', 'advisory', 'display_reply', 'speech_segments', 'tts_reply')}
    wrappers = (original_parser.clean_place, original_parser.add_place)
    speech = getattr(b, 'tanglish_speech_text', None)
    speech_code = getattr(speech, '__code__', None)
    backup = None
    after = after_source.encode('utf-8')
    wrote = False
    def atomic_write(data):
        fd, name = tempfile.mkstemp(prefix='.movio_slots_', suffix='.tmp', dir=str(target.parent))
        try:
            with os.fdopen(fd, 'wb') as handle:
                handle.write(data)
                handle.flush()
                os.fsync(handle.fileno())
            os.chmod(name, target.stat().st_mode & 0o777)
            os.replace(name, target)
        finally:
            if os.path.exists(name):
                os.unlink(name)
    try:
        if target.read_bytes() != before:
            raise RuntimeError('The source changed during inspection; rerun the diagnostic.')
        if after != before:
            backup = root / ('slot_suffix_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
            destination = backup / 'integrations' / target.name
            destination.parent.mkdir(parents=True, exist_ok=False)
            shutil.copy2(target, destination)
            atomic_write(after)
            wrote = True
        for fn, old, new in plans:
            fn.__code__ = new
        samples = run_checks(active)
        check(all(engine.get(name) is value for name,value in protected.items()), 'voice/dialogue handler identities')
        check(original_parser.clean_place is wrappers[0] and original_parser.add_place is wrappers[1], 'existing guard wrappers')
        check(getattr(b, 'tanglish_speech_text', None) is speech and getattr(speech, '__code__', None) is speech_code, 'speech unchanged')
    except Exception:
        for fn, old, new in plans:
            fn.__code__ = old
        if wrote and target.read_bytes() == after:
            atomic_write(before)
        raise
    result = dict(step=2, version=version, passed=True, offline_checks=offline_checks,
                  live_parser_fixture_checks=checks-offline_checks,
                  changed_files=['integrations/movio_group_booking_slots.py'] if wrote else [],
                  backup=str(backup) if backup else None, updated_base_functions=len(plans),
                  existing_wrapper_identities_preserved=True, speech_and_confirmation_unchanged=True,
                  examples=samples,
                  scope='Detached text-only fixtures against the active parser; no live booking, API or TTS test.')
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print('Open a NEW dashboard AI Call and repeat the two reported sentences. Share the output and conversation.')
    print('Pronunciation and Tamil weather pauses are the next separate step.')
    return result


MOVIO_SLOT_SUFFIX_STEP_2 = movio_fix_passenger_and_route_suffixes()


{
  "step": 2,
  "version": "slot-suffix-1.0",
  "passed": true,
  "offline_checks": 118,
  "live_parser_fixture_checks": 121,
  "changed_files": [
    "integrations/movio_group_booking_slots.py"
  ],
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/slot_suffix_backup_20261028_070739_567546",
  "updated_base_functions": 2,
  "existing_wrapper_identities_preserved": true,
  "speech_and_confirmation_unchanged": true,
  "examples": [
    {
      "language": "English",
      "case": "airport",
      "updates": {
        "count": 3,
        "vehicle": "SUV",
        "pickup": "Chennai Airport",
        "destination": "Tambaram"
      }
    },
    {
      "language": "English",
      "case": "central",
      "updates": {
        "count": 3,
        "vehicle": "SUV",
        "pickup": "Chennai Central",
        "destination": "Tambaram"
      }
    },
    {
      "language": "Tamil",
      "case": "airport",
      "updates": {
        "count": 3,
        "vehicle": "SUV",
        "pickup": "Ch

In [139]:
# MOVIO — GATEWAY FIX: forward /weather-speech to the AI service.
# Paste this ENTIRE file into ONE Kaggle cell, then re-run the "Listen to Forecast" cell.
# Changes ONE line of the gateway file and restarts ONLY the gateway process (port 8090).
# Models, the website (8080), the AI service (8000), the ngrok tunnel and the database
# are not touched. Close any open AI Call first: a call in progress will be dropped.
def movio_route_weather_speech_through_gateway():
    import os
    import sys
    import subprocess
    import tempfile
    import time
    from datetime import datetime
    from pathlib import Path
    import psutil
    import requests

    marker = '# MOVIO_WEATHER_SPEECH_ROUTE_V1'
    old_line = "    backend = AI_HTTP if path == 'health' else WEB_HTTP"
    new_line = "    backend = AI_HTTP if path in ('health', 'weather-speech') else WEB_HTTP  " + marker
    gateway_file = Path(globals().get('GATEWAY_FILE') or '/kaggle/working/movio_gateway.py').resolve()
    site_root = Path(globals().get('TARGET') or globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    if gateway_file.is_symlink() or not gateway_file.is_file():
        raise RuntimeError('Gateway file not found: ' + str(gateway_file) + '. Nothing was changed.')

    session = requests.Session()
    session.trust_env = False  # loopback checks go straight to this notebook
    def get(port, path):
        return session.get('http://127.0.0.1:' + str(port) + path, timeout=(2, 10), allow_redirects=False)

    def listeners(port):
        helper = globals().get('listener_pids')
        if callable(helper):
            try:
                return sorted(set(int(pid) for pid in helper(port)))
            except Exception:
                pass
        found = set()
        for connection in psutil.net_connections(kind='tcp'):
            if connection.status == psutil.CONN_LISTEN and connection.laddr and connection.laddr.port == port and connection.pid:
                found.add(connection.pid)
        return sorted(found)

    def script_paths(process):
        paths = []
        for argument in process.cmdline():
            if argument.endswith('.py'):
                candidate = Path(argument)
                if not candidate.is_absolute():
                    candidate = Path(process.cwd()) / candidate
                paths.append(candidate.resolve())
        return paths

    def same_response(a, b):
        return a.status_code == b.status_code and a.content == b.content

    try:
        original = gateway_file.read_text(encoding='utf-8')
        patched_already = new_line in original and original.count(new_line) == 1
        if patched_already:
            patched = original
        else:
            if original.count(old_line) != 1:
                shown = [line for line in original.splitlines() if 'backend' in line and 'AI_HTTP' in line]
                raise RuntimeError('The expected routing line was not found exactly once, so the gateway was not changed. '
                                   'Lines mentioning the backend: ' + repr(shown[:4]) + '. Share this output.')
            patched = original.replace(old_line, new_line, 1)
        compile(patched, str(gateway_file), 'exec')
        if patched.replace(new_line, old_line, 1) != original and not patched_already:
            raise RuntimeError('Single-line isolation check failed; nothing was changed.')

        # The process on 8090 must be THIS gateway file; never touch another application.
        pids = listeners(8090)
        if len(pids) != 1:
            raise RuntimeError('Expected exactly one process on port 8090, found ' + str(pids) + '. Nothing was changed.')
        if pids[0] == os.getpid():
            raise RuntimeError('Port 8090 runs inside this notebook; restarting it would unload the models. Nothing was changed.')
        old_process = psutil.Process(pids[0])
        if gateway_file not in script_paths(old_process):
            raise RuntimeError('Port 8090 is served by a different script (' + ' '.join(old_process.cmdline())[:200] + '), '
                               'not ' + str(gateway_file) + '. It was left running and unchanged. '
                               'Share that script so the same one-line route can be added to it.')

        # Baselines taken before any change.
        if get(8090, '/gateway-health').json().get('service') != 'movio-gateway':
            raise RuntimeError('The process on 8090 is not the MOVIO gateway. Nothing was changed.')
        before = {path: get(8090, path).status_code for path in ('/', '/api/health', '/call-page')}
        ai_health = get(8000, '/health')
        if ai_health.status_code != 200:
            raise RuntimeError('The AI service on 8000 is not healthy (HTTP ' + str(ai_health.status_code) + '). Nothing was changed.')
        speech_before = get(8090, '/weather-speech').status_code

        try:
            old_env = old_process.environ()
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            old_env = {}
        env = {**os.environ, **old_env}
        env['MOVIO_SITE_DIR'] = old_env.get('MOVIO_SITE_DIR') or str(site_root)
        for key in ('NO_PROXY', 'no_proxy'):
            env[key] = ','.join(filter(None, (env.get(key, ''), '127.0.0.1', 'localhost')))
        log_path = gateway_file.with_suffix('.log')

        def restart(source_text):
            """Stop only the verified gateway process, write the source atomically, start it, wait for health."""
            current = listeners(8090)
            for pid in current:
                if pid == os.getpid():
                    raise RuntimeError('Port 8090 runs inside this notebook.')
                process = psutil.Process(pid)
                if gateway_file not in script_paths(process):
                    raise RuntimeError('Port 8090 changed owner during the update; it was left running.')
                process.terminate()
                try:
                    process.wait(timeout=8)
                except psutil.TimeoutExpired:
                    raise RuntimeError('The old gateway did not stop; no forced kill was performed.')
            descriptor, temporary = tempfile.mkstemp(prefix='.movio_gateway_', suffix='.tmp', dir=str(gateway_file.parent))
            try:
                with os.fdopen(descriptor, 'w', encoding='utf-8') as handle:
                    handle.write(source_text)
                    handle.flush()
                    os.fsync(handle.fileno())
                os.chmod(temporary, gateway_file.stat().st_mode & 0o777)
                os.replace(temporary, gateway_file)
            finally:
                if os.path.exists(temporary):
                    os.unlink(temporary)
            offset = log_path.stat().st_size if log_path.exists() else 0
            with log_path.open('a') as log:
                process = subprocess.Popen([sys.executable, '-u', str(gateway_file)], cwd=str(gateway_file.parent), env=env,
                                           stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
            for _ in range(60):
                if process.poll() is not None:
                    break
                try:
                    if get(8090, '/gateway-health').status_code == 200:
                        globals()['movio_gateway_process'] = process
                        return process
                except requests.RequestException:
                    pass
                time.sleep(0.5)
            with log_path.open() as log:
                log.seek(offset)
                print(log.read()[-3000:])
            raise RuntimeError('The gateway did not come back up. See ' + str(log_path))

        def verify():
            problems = []
            if get(8090, '/gateway-health').json().get('service') != 'movio-gateway':
                problems.append('gateway health')
            after = {path: get(8090, path).status_code for path in ('/', '/api/health', '/call-page')}
            if after != before:
                problems.append('existing routes changed: ' + str(before) + ' -> ' + str(after))
            if get(8090, '/health').status_code != 200:
                problems.append('/health no longer reaches the AI service')
            direct, through = get(8000, '/weather-speech'), get(8090, '/weather-speech')
            if not same_response(direct, through):
                problems.append('/weather-speech through the gateway (HTTP ' + str(through.status_code) +
                                ') is not the AI service answer (HTTP ' + str(direct.status_code) + ')')
            return problems, direct.status_code, through.status_code

        backup = None
        if not patched_already:
            backup = gateway_file.with_name(gateway_file.name + '.before_weather_speech_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
            backup.write_text(original, encoding='utf-8')
        restarted = False
        if patched_already:
            problems, direct_status, through_status = verify()
        else:
            restart(patched)
            restarted = True
            problems, direct_status, through_status = verify()
        if problems and restarted:
            try:
                restart(original)
            except Exception as rollback_error:
                raise RuntimeError('Verification failed (' + '; '.join(problems) + ') AND the rollback failed: ' + str(rollback_error))
            raise RuntimeError('Verification failed, so the gateway was restored to its previous file: ' + '; '.join(problems))
        if problems:
            # File already patched but the running process is older: restart it once.
            restart(patched)
            restarted = True
            problems, direct_status, through_status = verify()
            if problems:
                raise RuntimeError('Gateway still not forwarding correctly: ' + '; '.join(problems))
        globals()['gateway_source'] = patched  # later runs of the routing-repair cell keep the fix
        print('GATEWAY FIX PASS')
        print('  /weather-speech before the fix, through the gateway: HTTP', speech_before)
        print('  /weather-speech now, AI service direct / through gateway: HTTP', direct_status, '/', through_status)
        print('  Website, health and call-page routes unchanged:', before)
        print('  Gateway restarted:', restarted, '| backup:', str(backup) if backup else 'none needed')
        print('Now re-run the "Listen to Forecast" cell. It should pass its gateway check.')
        print('To keep the fix after a full re-run, also change this one line inside gateway_source in the original gateway cell:')
        print('  ' + old_line.strip())
        print('  -> ' + new_line.strip())
        return dict(passed=True, restarted=restarted, backup=str(backup) if backup else None,
                    weather_speech_before=speech_before, weather_speech_now=through_status)
    finally:
        session.close()


MOVIO_GATEWAY_WEATHER_SPEECH_FIX = movio_route_weather_speech_through_gateway()

GATEWAY FIX PASS
  /weather-speech before the fix, through the gateway: HTTP 404
  /weather-speech now, AI service direct / through gateway: HTTP 404 / 404
  Website, health and call-page routes unchanged: {'/': 200, '/api/health': 200, '/call-page': 200}
  Gateway restarted: True | backup: /kaggle/working/movio_gateway.py.before_weather_speech_20261028_070740
Now re-run the "Listen to Forecast" cell. It should pass its gateway check.
To keep the fix after a full re-run, also change this one line inside gateway_source in the original gateway cell:
  backend = AI_HTTP if path == 'health' else WEB_HTTP
  -> backend = AI_HTTP if path in ('health', 'weather-speech') else WEB_HTTP  # MOVIO_WEATHER_SPEECH_ROUTE_V1


In [140]:
# MOVIO \u2014 "Listen to Forecast" button (standalone, no checksum dependency).
# Run this ENTIRE cell once in Kaggle, then hard-refresh the dashboard.
# Adds ONE new file (web/movio_forecast_listen.js) + ONE <script> tag in index.html.
# Nothing in weather_manual.js, booking, or voice handlers is modified.
def movio_add_listen_to_forecast():
    from pathlib import Path
    from datetime import datetime
    import hashlib, re, shutil, subprocess, requests

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    web = root / 'web'
    index = web / 'index.html'
    js_path = web / 'movio_forecast_listen.js'
    if not index.is_file():
        raise RuntimeError('web/index.html not found at ' + str(index))

    JS = r'''/* MOVIO_FORECAST_LISTEN_V2 - optional, tap-to-play, never autoplays, never touches booking. */
(function(){
 'use strict';
 var ROW_ID='wxForecastListenRow', synth=window.speechSynthesis, speaking=false, ticket=0;
 var COPY={
  en:{listen:'\uD83D\uDD0A Listen to Forecast',stop:'\u23F9 Stop forecast',unsupported:'Voice is not supported on this device.'},
  ta:{listen:'\uD83D\uDD0A \u0BB5\u0BBE\u0BA9\u0BBF\u0BB2\u0BC8 \u0BAE\u0BC1\u0BA9\u0BCD\u0BA9\u0BB1\u0BBF\u0BB5\u0BBF\u0BAA\u0BCD\u0BAA\u0BC8\u0B95\u0BCD \u0B95\u0BC7\u0BB3\u0BC1\u0B99\u0BCD\u0B95\u0BB3\u0BCD',stop:'\u23F9 \u0BA8\u0BBF\u0BB1\u0BC1\u0BA4\u0BCD\u0BA4\u0BC1',unsupported:'\u0B87\u0BA8\u0BCD\u0BA4 \u0B9A\u0BBE\u0BA4\u0BA9\u0BA4\u0BCD\u0BA4\u0BBF\u0BB2\u0BCD \u0B95\u0BC1\u0BB0\u0BB2\u0BCD \u0B86\u0BA4\u0BB0\u0BB5\u0BC1 \u0B87\u0BB2\u0BCD\u0BB2\u0BC8.'}
 };
 function $(id){return document.getElementById(id);}
 function isTamil(t){return /[\u0B80-\u0BFF]/.test(t);}
 function forecastText(){
  var a=$('wxManualAdvice'); var t=a?(a.textContent||'').trim():'';
  if(!t){var d=document.querySelector('#wxManualReview .wx-details');t=d?(d.textContent||'').trim():'';}
  return t;
 }
 function reviewVisible(){var r=$('wxManualReview');return !!r && !r.hidden && r.offsetParent!==null;}
 function setLabel(btn){
  var c=COPY[isTamil(forecastText())?'ta':'en'];
  btn.textContent=speaking?c.stop:c.listen;
  btn.setAttribute('aria-pressed',String(speaking));
 }
 function stop(){
  ticket++;speaking=false;
  try{if(synth)synth.cancel();}catch(e){}
  var b=$('wxForecastListenBtn');if(b)setLabel(b);
 }
 function pickVoice(lang){
  var vs=synth.getVoices()||[],base=lang.split('-')[0].toLowerCase();
  return vs.find(function(v){return v.lang&&v.lang.toLowerCase().replace('_','-')===lang.toLowerCase();})
      || vs.find(function(v){return v.lang&&v.lang.toLowerCase().indexOf(base)===0;})||null;
 }
 function play(){
  var note=$('wxForecastListenNote');
  if(!synth||typeof SpeechSynthesisUtterance==='undefined'){if(note)note.textContent=COPY.en.unsupported;return;}
  if(speaking){stop();return;}
  var text=forecastText();if(!text)return;
  var tamil=isTamil(text),lang=tamil?'ta-IN':'en-IN';
  var mine=++ticket;
  try{synth.cancel();}catch(e){}
  var u=new SpeechSynthesisUtterance(text);
  u.lang=lang;var v=pickVoice(lang);if(v)u.voice=v;u.rate=0.95;
  u.onend=u.onerror=function(){if(mine!==ticket)return;speaking=false;var b=$('wxForecastListenBtn');if(b)setLabel(b);};
  speaking=true;if(note)note.textContent='';
  var b=$('wxForecastListenBtn');if(b)setLabel(b);
  synth.speak(u); /* called directly from the tap, so mobile browsers allow it */
 }
 function ensureRow(){
  var review=$('wxManualReview');
  if(!review)return;
  var row=$(ROW_ID);
  if(!reviewVisible()||!forecastText()){ if(row)row.hidden=true; if(speaking)stop(); return; }
  var actions=review.querySelector('.wx-review-actions');
  if(!actions)return;
  if(!row){
   row=document.createElement('div');row.id=ROW_ID;
   row.innerHTML='<button type="button" id="wxForecastListenBtn" aria-pressed="false"></button><span id="wxForecastListenNote" role="status" aria-live="polite"></span>';
   row.querySelector('button').addEventListener('click',play);
  }
  /* sit directly above Confirm booking / Edit details = below Full details */
  if(row.nextElementSibling!==actions||row.parentNode!==actions.parentNode)actions.parentNode.insertBefore(row,actions);
  row.hidden=false;
  setLabel(row.querySelector('button'));
 }
 var st=document.createElement('style');
 st.textContent='#'+ROW_ID+'{margin:12px 0;display:flex;align-items:center;gap:8px;flex-wrap:wrap}#'+ROW_ID+'[hidden]{display:none!important}'
  +'#wxForecastListenBtn{min-height:44px;width:100%;padding:10px 14px;border:1px solid var(--line,#d9d9d9);border-radius:12px;background:var(--yellow,#ffcc29);color:var(--ink,#17191d);font:inherit;font-weight:600;cursor:pointer}'
  +'#wxForecastListenBtn[aria-pressed="true"]{background:var(--paper,#fff)}#wxForecastListenNote{font-size:12px;color:var(--muted,#667085)}';
 document.head.appendChild(st);
 var queued=false;
 function schedule(){if(queued)return;queued=true;setTimeout(function(){queued=false;ensureRow();},60);}
 new MutationObserver(schedule).observe(document.body,{subtree:true,childList:true,attributes:true,attributeFilter:['hidden'],characterData:true});
 /* Booking actions or leaving the page silence the voice; they are otherwise untouched. */
 document.addEventListener('click',function(e){if(e.target.closest&&e.target.closest('#wxManualContinue,#wxManualChange,#logout,#aiCall'))stop();},true);
 window.addEventListener('pagehide',stop);
 document.addEventListener('visibilitychange',function(){if(document.hidden)stop();});
 if(synth&&synth.addEventListener)synth.addEventListener('voiceschanged',function(){});
 schedule();
})();
'''
    # 1) syntax check
    import tempfile
    node = shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory() as d:
            p = Path(d) / 't.js'; p.write_text(JS, encoding='utf-8')
            r = subprocess.run([node, '--check', str(p)], capture_output=True, text=True, timeout=20)
            if r.returncode: raise RuntimeError('JS syntax error: ' + r.stderr[:400])

    # 2) backup + write
    backup = root / ('listen_forecast_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    backup.mkdir()
    shutil.copy2(index, backup / 'index.html')
    js_path.write_text(JS, encoding='utf-8')
    token = hashlib.sha256(JS.encode()).hexdigest()[:12]

    # 3) idempotent script tag (replace old tag if re-run)
    html = index.read_text(encoding='utf-8')
    html = re.sub(r'\s*<script[^>]*movio_forecast_listen\.js[^>]*></script>', '', html)
    tag = '\n<script src="/movio_forecast_listen.js?v=%s" defer></script>' % token
    if '</body>' in html:
        html = html.replace('</body>', tag + '\n</body>', 1)
    else:
        html += tag
    index.write_text(html, encoding='utf-8')

    # 4) verify it is actually being served
    served = requests.get('http://127.0.0.1:8080/movio_forecast_listen.js?v=' + token, timeout=8)
    page = requests.get('http://127.0.0.1:8080/', timeout=8)
    ok_js = served.status_code == 200 and 'MOVIO_FORECAST_LISTEN_V2' in served.text
    ok_page = 'movio_forecast_listen.js' in page.text
    print({'script_served': ok_js, 'tag_in_page': ok_page, 'backup': str(backup.name)})
    if not (ok_js and ok_page):
        shutil.copy2(backup / 'index.html', index)
        raise RuntimeError('Not served by the website (status %s). index.html restored. Share this output.' % served.status_code)
    print('Done. Hard-refresh the customer dashboard (close the tab and reopen), review a ride, '
          'and the "Listen to Forecast" button appears under Full details, above Confirm booking.')

movio_add_listen_to_forecast()

{'script_served': True, 'tag_in_page': True, 'backup': 'listen_forecast_backup_20261028_070742'}
Done. Hard-refresh the customer dashboard (close the tab and reopen), review a ride, and the "Listen to Forecast" button appears under Full details, above Confirm booking.


In [141]:
def register_forecast_audio(app, bridge_object, engine, public_origin):
    """Reuse the existing endpoint, or add one authenticated TTS-only route."""
    import asyncio
    import base64
    import inspect
    import json
    import threading
    import requests
    from fastapi import HTTPException, Request

    version='manual-forecast-voice-1.0'
    existing=[r for r in app.routes if getattr(r,'path',None)=='/weather-speech']
    if existing:
        if len(existing)!=1 or 'POST' not in (getattr(existing[0],'methods',set()) or set()):
            raise RuntimeError('Conflicting weather-speech routes; no changes were made.')
        route=existing[0]
        fields=getattr(getattr(route,'body_field',None),'type_',None)
        fields=getattr(fields,'model_fields',None) or getattr(fields,'__fields__',{})
        if getattr(route.endpoint,'_movio_forecast_voice',None)!=version and not {'text','language'}<=set(fields):
            raise RuntimeError('The current weather-speech endpoint has a different request format. Share its source before modifying it.')
        return dict(added=False,route=route,endpoint=route.endpoint)
    tts=engine.get('tts_reply')
    language_scope=getattr(bridge_object,'CURRENT_LANGUAGE',None)
    if not callable(tts) or not callable(getattr(language_scope,'set',None)):
        raise RuntimeError('The existing optimized DhVaani reply handler is missing.')
    register=None
    advisory=engine.get('advisory')
    if callable(advisory):
        try:register=inspect.getclosurevars(advisory).nonlocals.get('register')
        except (TypeError,ValueError):pass
    busy=threading.BoundedSemaphore(1)
    class Busy(Exception):pass
    def render(text,language):
        if not busy.acquire(blocking=False):raise Busy()
        token=language_scope.set(language)
        try:
            if callable(register):register(text,'weather')
            audio=tts(bridge_object,text)
            if not isinstance(audio,str):raise ValueError('Invalid speech response')
            raw=base64.b64decode(audio,validate=True)
            if len(raw)<44 or len(raw)>9000000 or raw[:4]!=b'RIFF' or raw[8:12]!=b'WAVE':
                raise ValueError('Invalid WAV response')
            return audio
        finally:
            language_scope.reset(token);busy.release()
    def authenticate(cookie):
        # Reuse website authentication without exposing account data or keys.
        with requests.Session() as session:
            session.trust_env=False
            response=session.get('http://127.0.0.1:8080/api/me',headers={'Cookie':cookie},
                                 timeout=(2,5),allow_redirects=False)
            if response.status_code!=200:raise PermissionError()
            account=response.json()
            if account.get('role')!='customer':raise PermissionError()

    async def manual_forecast_voice(request:Request):
        origin=request.headers.get('origin','').rstrip('/')
        if origin and origin!=public_origin.rstrip('/'):
            raise HTTPException(403,'Origin not allowed')
        cookie=request.headers.get('cookie','')
        if not cookie:raise HTTPException(401,'Sign in to listen to this forecast')
        try:await asyncio.to_thread(authenticate,cookie)
        except PermissionError:raise HTTPException(403,'Customer login required')
        except Exception:raise HTTPException(503,'Account verification is temporarily unavailable')
        try:
            body=await request.body()
            if len(body)>16000:raise ValueError()
            data=json.loads(body)
            if not isinstance(data,dict):raise ValueError()
            text=data.get('text');language=data.get('language')
            if not isinstance(text,str) or not text.strip() or len(text)>1500:raise ValueError()
            if language not in ('English','Tamil','Tanglish'):raise ValueError()
        except (ValueError,TypeError,UnicodeError):raise HTTPException(400,'Invalid forecast voice request')
        try:
            audio=await asyncio.to_thread(render,text.strip(),language)
            return {'audio':audio,'format':'wav'}
        except Busy:raise HTTPException(429,'Forecast voice is busy; please retry')
        except Exception:raise HTTPException(503,'Forecast voice could not be generated')
    manual_forecast_voice._movio_forecast_voice=version
    manual_forecast_voice.__annotations__['request']=Request
    app.add_api_route('/weather-speech',manual_forecast_voice,methods=['POST'])
    route=next(r for r in app.routes if getattr(r,'endpoint',None) is manual_forecast_voice)
    app.openapi_schema=None
    return dict(added=True,route=route,endpoint=manual_forecast_voice)


def movio_add_listen_to_forecast():
    """Show 'Listen to Forecast' and play it through the EXISTING MOVIO TTS (/weather-speech)."""
    from pathlib import Path
    from datetime import datetime
    from urllib.parse import urlsplit
    import hashlib, re, shutil, subprocess, tempfile
    import requests

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    web = root / 'web'; index = web / 'index.html'; js_path = web / 'movio_forecast_listen.js'
    if not index.is_file():
        raise RuntimeError('web/index.html not found at ' + str(index))

    JS = r'''/* MOVIO_FORECAST_LISTEN_V3 - plays the forecast with the existing MOVIO TTS (/weather-speech). Tap only; never autoplays; never touches booking. */
(function(){
 'use strict';
 var ROW_ID='wxForecastListenRow';
 var COPY={
  English:{listen:'\uD83D\uDD0A Listen to Forecast',stop:'\u23F9 Stop forecast',cancel:'Cancel voice',loading:'Preparing forecast voice\u2026',error:'Voice could not play. Tap to retry.',replay:'\uD83D\uDD0A Replay forecast'},
  Tamil:{listen:'\uD83D\uDD0A \u0BB5\u0BBE\u0BA9\u0BBF\u0BB2\u0BC8 \u0BAE\u0BC1\u0BA9\u0BCD\u0BA9\u0BB1\u0BBF\u0BB5\u0BBF\u0BAA\u0BCD\u0BAA\u0BC8\u0B95\u0BCD \u0B95\u0BC7\u0BB3\u0BC1\u0B99\u0BCD\u0B95\u0BB3\u0BCD',stop:'\u23F9 \u0B92\u0BB2\u0BBF\u0BAF\u0BC8 \u0BA8\u0BBF\u0BB1\u0BC1\u0BA4\u0BCD\u0BA4\u0BC1',cancel:'\u0BB0\u0BA4\u0BCD\u0BA4\u0BC1\u0B9A\u0BC6\u0BAF\u0BCD',loading:'\u0BB5\u0BBE\u0BA9\u0BBF\u0BB2\u0BC8 \u0B95\u0BC1\u0BB0\u0BB2\u0BC8\u0BA4\u0BCD \u0BA4\u0BAF\u0BBE\u0BB0\u0BBF\u0B95\u0BCD\u0B95\u0BBF\u0BB1\u0BCB\u0BAE\u0BCD\u2026',error:'\u0B95\u0BC1\u0BB0\u0BB2\u0BC8 \u0B87\u0BAF\u0B95\u0BCD\u0B95 \u0BAE\u0BC1\u0B9F\u0BBF\u0BAF\u0BB5\u0BBF\u0BB2\u0BCD\u0BB2\u0BC8. \u0BAE\u0BC0\u0BA3\u0BCD\u0B9F\u0BC1\u0BAE\u0BCD \u0BAE\u0BC1\u0BAF\u0BB1\u0BCD\u0B9A\u0BBF\u0B95\u0BCD\u0B95\u0BB5\u0BC1\u0BAE\u0BCD.',replay:'\uD83D\uDD0A \u0BAE\u0BC0\u0BA3\u0BCD\u0B9F\u0BC1\u0BAE\u0BCD \u0B95\u0BC7\u0BB3\u0BC1\u0B99\u0BCD\u0B95\u0BB3\u0BCD'},
  Tanglish:{listen:'\uD83D\uDD0A Listen to Forecast',stop:'\u23F9 Forecast niruthunga',cancel:'Voice cancel pannunga',loading:'Forecast voice prepare pannrom\u2026',error:'Voice play aagala. Thirumba tap pannunga.',replay:'\uD83D\uDD0A Forecast thirumba kelunga'}
 };
 var state='idle',epoch=0,controller=null,audio=null,url=null,cacheKey='',cacheBlob=null,played=false;
 function $(id){return document.getElementById(id);}
 function isTamil(t){return /[\u0B80-\u0BFF]/.test(t);}
 function forecastText(){var a=$('wxManualAdvice');return a?(a.textContent||'').trim():'';}
 function language(){
  var sel=$('wxManualLanguage'),v=sel&&sel.value;
  if(v==='Tamil'||v==='Tanglish'||v==='English')return v;
  return isTamil(forecastText())?'Tamil':'English';
 }
 function words(){return COPY[language()]||COPY.English;}
 function reviewVisible(){var r=$('wxManualReview');return !!r&&!r.hidden&&r.offsetParent!==null;}
 function setLabel(){
  var b=$('wxForecastListenBtn');if(!b)return;
  b.textContent=state==='loading'?words().cancel:state==='playing'?words().stop:played?words().replay:words().listen;
  b.setAttribute('aria-pressed',String(state==='playing'));b.setAttribute('aria-busy',String(state==='loading'));
 }
 function note(t){var n=$('wxForecastListenNote');if(n)n.textContent=t||'';}
 function stop(clearCache){
  epoch++;if(controller){try{controller.abort();}catch(e){}controller=null;}
  if(audio){try{audio.pause();audio.removeAttribute('src');audio.load();}catch(e){}audio=null;}
  if(url){URL.revokeObjectURL(url);url=null;}
  if(clearCache){cacheBlob=null;cacheKey='';played=false;}
  state='idle';note('');setLabel();
 }
 async function play(){
  if(!reviewVisible()||!forecastText())return;
  if(state!=='idle'){stop(false);return;}
  var text=forecastText(),lang=language(),key=lang+'\n'+text,ticket=++epoch;
  try{
   var blob=cacheKey===key?cacheBlob:null;
   if(!blob){
    state='loading';setLabel();note(words().loading);
    controller=new AbortController();var ctl=controller,timer=setTimeout(function(){ctl.abort();},120000);
    try{
     var res=await fetch('/weather-speech',{method:'POST',credentials:'same-origin',cache:'no-store',signal:ctl.signal,headers:{'Content-Type':'application/json'},body:JSON.stringify({text:text,language:lang})});
     if(!res.ok)throw new Error('voice '+res.status);
     var data=await res.json();
     if(ticket!==epoch)return;
     if(typeof data.audio!=='string'||!data.audio)throw new Error('bad audio');
     var bin=atob(data.audio),bytes=new Uint8Array(bin.length);
     for(var i=0;i<bin.length;i++)bytes[i]=bin.charCodeAt(i);
     blob=new Blob([bytes],{type:'audio/wav'});cacheKey=key;cacheBlob=blob;
    }finally{clearTimeout(timer);}
   }
   if(ticket!==epoch)return;
   controller=null;note('');
   url=URL.createObjectURL(blob);audio=new Audio(url);audio.setAttribute('playsinline','');
   var cur=audio;
   cur.addEventListener('ended',function(){if(ticket!==epoch)return;played=true;stop(false);},{once:true});
   cur.addEventListener('error',function(){if(ticket!==epoch)return;stop(true);note(words().error);},{once:true});
   state='playing';setLabel();
   try{await cur.play();}
   catch(e){if(ticket!==epoch)return;stop(false);note(words().error);}
  }catch(e){
   if(ticket!==epoch)return;
   stop(true);note(words().error);
  }
 }
 function ensureRow(){
  var review=$('wxManualReview');if(!review)return;
  var row=$(ROW_ID);
  if(!reviewVisible()||!forecastText()){if(row)row.hidden=true;if(state!=='idle')stop(false);return;}
  var actions=review.querySelector('.wx-review-actions');if(!actions)return;
  if(!row){
   row=document.createElement('div');row.id=ROW_ID;
   row.innerHTML='<button type="button" id="wxForecastListenBtn" aria-pressed="false"></button><span id="wxForecastListenNote" role="status" aria-live="polite"></span>';
   row.querySelector('button').addEventListener('click',play);
  }
  if(row.nextElementSibling!==actions||row.parentNode!==actions.parentNode)actions.parentNode.insertBefore(row,actions);
  row.hidden=false;
  /* new forecast text or language => drop old audio */
  if(cacheKey&&cacheKey!==language()+'\n'+forecastText()&&state==='idle'){cacheBlob=null;cacheKey='';played=false;}
  setLabel();
 }
 var st=document.createElement('style');
 st.textContent='#'+ROW_ID+'{margin:12px 0;display:flex;align-items:center;gap:8px;flex-wrap:wrap}#'+ROW_ID+'[hidden]{display:none!important}'
  +'#wxForecastListenBtn{min-height:44px;width:100%;padding:10px 14px;border:1px solid var(--line,#d9d9d9);border-radius:12px;background:var(--yellow,#ffcc29);color:var(--ink,#17191d);font:inherit;font-weight:600;cursor:pointer;white-space:normal}'
  +'#wxForecastListenBtn[aria-pressed="true"]{background:var(--paper,#fff)}#wxForecastListenNote{font-size:12px;color:var(--muted,#667085);overflow-wrap:anywhere}';
 document.head.appendChild(st);
 var queued=false;
 function schedule(){if(queued)return;queued=true;setTimeout(function(){queued=false;ensureRow();},60);}
 new MutationObserver(schedule).observe(document.body,{subtree:true,childList:true,attributes:true,attributeFilter:['hidden'],characterData:true});
 document.addEventListener('change',function(e){if(e.target&&e.target.id==='wxManualLanguage'){stop(true);schedule();}},true);
 document.addEventListener('click',function(e){if(e.target.closest&&e.target.closest('#wxManualContinue,#wxManualChange,#logout,#driverMode,#aiCall'))stop(false);},true);
 window.addEventListener('pagehide',function(){stop(true);});
 document.addEventListener('visibilitychange',function(){if(document.hidden)stop(false);});
 schedule();
})();
'''

    node = shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory() as d:
            p = Path(d) / 't.js'; p.write_text(JS, encoding='utf-8')
            r = subprocess.run([node, '--check', str(p)], capture_output=True, text=True, timeout=20)
            if r.returncode: raise RuntimeError('JS syntax error: ' + r.stderr[:400])

    # --- 1. Make sure the existing-TTS audio route exists (reuses the DhVaani handler) ---
    b = globals().get('bridge')
    if not b: raise RuntimeError('Run the existing MOVIO voice setup first (bridge is missing).')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    current = b; seen = set(); app = None; actual = None
    while current is not None and id(current) not in seen:
        seen.add(id(current)); cand = getattr(current, 'app', None)
        if callable(getattr(cand, 'add_api_route', None)):
            app = cand; actual = current; break
        current = getattr(current, '__dict__', {}).get('_original')
    if app is None: raise RuntimeError('The running AI application was not found behind the bridge.')
    public = str(globals().get('MOVIO_WEBSITE_URL') or getattr(actual, 'PUBLIC_URL', '')).rstrip('/')
    if not urlsplit(public).scheme or not urlsplit(public).netloc:
        raise RuntimeError('The public website URL is missing.')
    audio = register_forecast_audio(app, b, engine, public)

    s = requests.Session(); s.trust_env = False
    try:
        for port in (8000, 8090):
            code = s.get('http://127.0.0.1:%d/weather-speech' % port, timeout=(2, 8), allow_redirects=False).status_code
            if code != 405:
                raise RuntimeError('Port %d does not expose POST /weather-speech (GET returned %s). Share this output.' % (port, code))
    except Exception:
        if audio['added']:
            app.routes[:] = [r for r in app.routes if r is not audio['route']]
        raise

    # --- 2. Install the button script ---
    backup = Path(tempfile.mkdtemp(
        prefix='listen_forecast_backup_',
        dir=str(root)
    ))
    shutil.copy2(index, backup / 'index.html')
    js_path.write_text(JS, encoding='utf-8')
    token = hashlib.sha256(JS.encode()).hexdigest()[:12]
    html = index.read_text(encoding='utf-8')
    html = re.sub(r'\s*<script[^>]*movio_forecast_listen\.js[^>]*></script>', '', html)
    tag = '\n<script src="/movio_forecast_listen.js?v=%s" defer></script>' % token
    html = html.replace('</body>', tag + '\n</body>', 1) if '</body>' in html else html + tag
    index.write_text(html, encoding='utf-8')

    served = s.get('http://127.0.0.1:8080/movio_forecast_listen.js?v=' + token, timeout=8)
    page = s.get('http://127.0.0.1:8080/', timeout=8)
    gw = s.get('http://127.0.0.1:8080/weather-speech', timeout=8, allow_redirects=False).status_code
    ok = served.status_code == 200 and 'MOVIO_FORECAST_LISTEN_V3' in served.text and 'movio_forecast_listen.js' in page.text
    print({'script_served': ok, 'audio_route_added': audio['added'], 'website_gateway_GET_status': gw, 'backup': backup.name})
    s.close()
    if not ok:
        shutil.copy2(backup / 'index.html', index)
        raise RuntimeError('Website did not serve the new script. index.html restored.')
    print('Done. Close and reopen the dashboard tab, review a ride, tap "Listen to Forecast". It now uses your existing MOVIO voice.')

movio_add_listen_to_forecast()

{'script_served': True, 'audio_route_added': True, 'website_gateway_GET_status': 404, 'backup': 'listen_forecast_backup_9zweju45'}
Done. Close and reopen the dashboard tab, review a ride, tap "Listen to Forecast". It now uses your existing MOVIO voice.


In [142]:
# ---------------------------------------------------------------------------
# MOVIO forecast speech preparation (used only by the "Listen to Forecast" button).
# Pure text/WAV helpers: they never touch the booking, call or confirmation voice.
# ---------------------------------------------------------------------------
import re as _re

_MOVIO_LETTERS = {
    'A': 'ஏ', 'B': 'பி', 'C': 'சி', 'D': 'டி', 'E': 'ஈ', 'F': 'எஃப்', 'G': 'ஜி', 'H': 'ஹெச்',
    'I': 'ஐ', 'J': 'ஜே', 'K': 'கே', 'L': 'எல்', 'M': 'எம்', 'N': 'என்', 'O': 'ஓ', 'P': 'பி',
    'Q': 'கியூ', 'R': 'ஆர்', 'S': 'எஸ்', 'T': 'டி', 'U': 'யூ', 'V': 'வி', 'W': 'டபிள்யூ',
    'X': 'எக்ஸ்', 'Y': 'வை', 'Z': 'ஸெட்',
}
# Real words that happen to be written in capitals: never spell these out.
_MOVIO_CAPS_WORDS = {'AUTO', 'MINI', 'BIKE', 'TAXI', 'SEDAN', 'RAIN', 'HEAT', 'NOTE', 'HIGH',
                     'LOW', 'YES', 'NO', 'OK', 'AND', 'THE', 'FOR'}
_MOVIO_WORDS = (  # spoken with Tamil pronunciation in English, Tanglish and Tamil
    (r'(?<![A-Za-z])temperatures(?![A-Za-z])', 'டெம்பரேச்சர்ஸ்'),
    (r'(?<![A-Za-z])temperature(?![A-Za-z])', 'டெம்பரேச்சர்'),
    (r'(?<![A-Za-z])celsius(?![A-Za-z])', 'செல்சியஸ்'),
    (r'(?<![A-Za-z])autos(?![A-Za-z])', 'ஆட்டோக்கள்'),
    (r'(?<![A-Za-z])auto(?![A-Za-z])', 'ஆட்டோ'),
)


def movio_spell_acronym(letters):
    """ICF -> ஐசிஎஃப்: letter names joined into ONE word, so it is read together."""
    return ''.join(_MOVIO_LETTERS[c] for c in letters if c in _MOVIO_LETTERS)


def movio_prepare_forecast_text(text, language):
    value = _re.sub(r'\s+', ' ', str(text)).strip()
    # 28.6°C / 28.6 °C / "degrees Celsius" -> Tamil pronunciation of Celsius
    value = _re.sub(r'\s*°\s*C(?![A-Za-z])',
                    ' டிகிரி செல்சியஸ்' if language == 'Tamil' else ' degrees செல்சியஸ்', value)
    for pattern, spoken in _MOVIO_WORDS:
        value = _re.sub(pattern, spoken, value, flags=_re.I)

    def dotted(match):
        end = match.end()
        tail = value[end:end + 3]
        keep_stop = end >= len(value) or _re.match(r'\s+[A-Z஀-௿]', tail + ' ')
        return movio_spell_acronym(_re.sub(r'[^A-Z]', '', match.group(0))) + ('.' if keep_stop else '')
    value = _re.sub(r'(?<![A-Za-z])(?:[A-Z]\.){2,6}', dotted, value)
    value = _re.sub(r'(?<![A-Za-z0-9])[A-Z]{2,6}(?![A-Za-z0-9])',
                    lambda m: m.group(0) if m.group(0) in _MOVIO_CAPS_WORDS else movio_spell_acronym(m.group(0)),
                    value)
    # A colon inside a sentence must not break it (time such as 4:02 is kept).
    value = _re.sub(r'(?<!\d):|:(?!\d)', ' ', value)
    return _re.sub(r'\s+', ' ', value).strip()


def movio_forecast_parts(text, language):
    """[(sentence, pause_seconds_after)]: pause after every full stop and semicolon."""
    prepared = movio_prepare_forecast_text(text, language)
    parts = []
    for piece in _re.split(r'(?<=[.;!?।])\s+', prepared):
        piece = piece.strip()
        if not piece:
            continue
        stop = piece[-1]
        body = piece.rstrip('.;!?। ').strip()
        if not _re.search(r'[A-Za-z0-9஀-௿]', body):
            continue
        parts.append((body + ('?' if stop == '?' else '.'), 0.5 if stop == ';' else 0.7))
    if parts:
        parts[-1] = (parts[-1][0], 0.0)
    return parts


def movio_join_wavs(wav_pauses):
    """Join WAV byte strings into one WAV, inserting the given silence after each."""
    import io, wave
    out = io.BytesIO(); writer = None; params = None
    for raw, pause in wav_pauses:
        with wave.open(io.BytesIO(raw), 'rb') as part:
            current = (part.getnchannels(), part.getsampwidth(), part.getframerate())
            frames = part.readframes(part.getnframes())
        if writer is None:
            params = current
            writer = wave.open(out, 'wb')
            writer.setnchannels(current[0]); writer.setsampwidth(current[1]); writer.setframerate(current[2])
        elif current != params:
            raise ValueError('Speech chunks have different audio formats')
        writer.writeframes(frames)
        if pause > 0:
            writer.writeframes(b'\x00' * (int(params[2] * pause) * params[0] * params[1]))
    if writer is None:
        raise ValueError('No speech was generated')
    writer.close()
    return out.getvalue()


def register_forecast_audio(app, bridge_object, engine, public_origin):
    """Reuse the existing endpoint, or add one authenticated TTS-only route."""
    import asyncio
    import base64
    import inspect
    import json
    import threading
    import requests
    from fastapi import HTTPException, Request

    version='forecast-voice-2.0'
    existing=[r for r in app.routes if getattr(r,'path',None)=='/weather-speech']
    replaced=[]
    for route in existing:
        if getattr(route.endpoint,'_movio_forecast_voice',None) is None:
            raise RuntimeError('A different /weather-speech endpoint exists. Share its source before modifying it.')
    if len(existing)==1 and existing[0].endpoint._movio_forecast_voice==version and 'POST' in (getattr(existing[0],'methods',set()) or set()):
        return dict(added=False,replaced=[],route=existing[0],endpoint=existing[0].endpoint)
    replaced=list(existing)  # older MOVIO forecast-voice route: replaced by this version
    if replaced:app.routes[:]=[r for r in app.routes if r not in replaced]
    tts=engine.get('tts_reply')
    language_scope=getattr(bridge_object,'CURRENT_LANGUAGE',None)
    if not callable(tts) or not callable(getattr(language_scope,'set',None)):
        raise RuntimeError('The existing optimized DhVaani reply handler is missing.')
    register=None
    advisory=engine.get('advisory')
    if callable(advisory):
        try:register=inspect.getclosurevars(advisory).nonlocals.get('register')
        except (TypeError,ValueError):pass
    busy=threading.BoundedSemaphore(1)
    class Busy(Exception):pass
    def render(text,language):
        parts=movio_forecast_parts(text,language)
        if not parts or len(parts)>14:raise ValueError('Forecast text cannot be spoken')
        if not busy.acquire(blocking=False):raise Busy()
        token=language_scope.set(language)
        try:
            pieces=[]
            for sentence,pause in parts:
                if callable(register):register(sentence,'weather')
                audio=tts(bridge_object,sentence)
                if not isinstance(audio,str):raise ValueError('Invalid speech response')
                raw=base64.b64decode(audio,validate=True)
                if len(raw)<44 or raw[:4]!=b'RIFF' or raw[8:12]!=b'WAVE':raise ValueError('Invalid WAV response')
                pieces.append((raw,pause))
            joined=movio_join_wavs(pieces)
            if len(joined)>9000000:raise ValueError('Speech response too large')
            return base64.b64encode(joined).decode()
        finally:
            language_scope.reset(token);busy.release()
    def authenticate(cookie):
        # Reuse website authentication without exposing account data or keys.
        with requests.Session() as session:
            session.trust_env=False
            response=session.get('http://127.0.0.1:8080/api/me',headers={'Cookie':cookie},
                                 timeout=(2,5),allow_redirects=False)
            if response.status_code!=200:raise PermissionError()
            account=response.json()
            if account.get('role')!='customer':raise PermissionError()

    async def manual_forecast_voice(request:Request):
        origin=request.headers.get('origin','').rstrip('/')
        if origin and origin!=public_origin.rstrip('/'):
            raise HTTPException(403,'Origin not allowed')
        cookie=request.headers.get('cookie','')
        if not cookie:raise HTTPException(401,'Sign in to listen to this forecast')
        try:await asyncio.to_thread(authenticate,cookie)
        except PermissionError:raise HTTPException(403,'Customer login required')
        except Exception:raise HTTPException(503,'Account verification is temporarily unavailable')
        try:
            body=await request.body()
            if len(body)>16000:raise ValueError()
            data=json.loads(body)
            if not isinstance(data,dict):raise ValueError()
            text=data.get('text');language=data.get('language')
            if not isinstance(text,str) or not text.strip() or len(text)>1500:raise ValueError()
            if language not in ('English','Tamil','Tanglish'):raise ValueError()
        except (ValueError,TypeError,UnicodeError):raise HTTPException(400,'Invalid forecast voice request')
        try:
            audio=await asyncio.to_thread(render,text.strip(),language)
            return {'audio':audio,'format':'wav'}
        except Busy:raise HTTPException(429,'Forecast voice is busy; please retry')
        except Exception:raise HTTPException(503,'Forecast voice could not be generated')
    manual_forecast_voice._movio_forecast_voice=version
    manual_forecast_voice.__annotations__['request']=Request
    app.add_api_route('/weather-speech',manual_forecast_voice,methods=['POST'])
    route=next(r for r in app.routes if getattr(r,'endpoint',None) is manual_forecast_voice)
    app.openapi_schema=None
    return dict(added=True,replaced=replaced,route=route,endpoint=manual_forecast_voice)


def movio_add_listen_to_forecast():
    """Show 'Listen to Forecast' and play it through the EXISTING MOVIO TTS (/weather-speech)."""
    from pathlib import Path
    from datetime import datetime
    from urllib.parse import urlsplit
    import hashlib, re, shutil, subprocess, tempfile
    import requests

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    web = root / 'web'; index = web / 'index.html'; js_path = web / 'movio_forecast_listen.js'
    if not index.is_file():
        raise RuntimeError('web/index.html not found at ' + str(index))

    JS = r'''/* MOVIO_FORECAST_LISTEN_V3 - plays the forecast with the existing MOVIO TTS (/weather-speech). Tap only; never autoplays; never touches booking. */
(function(){
 'use strict';
 var ROW_ID='wxForecastListenRow';
 var COPY={
  English:{listen:'\uD83D\uDD0A Listen to Forecast',stop:'\u23F9 Stop forecast',cancel:'Cancel voice',loading:'Preparing forecast voice\u2026',error:'Voice could not play. Tap to retry.',replay:'\uD83D\uDD0A Replay forecast'},
  Tamil:{listen:'\uD83D\uDD0A \u0BB5\u0BBE\u0BA9\u0BBF\u0BB2\u0BC8 \u0BAE\u0BC1\u0BA9\u0BCD\u0BA9\u0BB1\u0BBF\u0BB5\u0BBF\u0BAA\u0BCD\u0BAA\u0BC8\u0B95\u0BCD \u0B95\u0BC7\u0BB3\u0BC1\u0B99\u0BCD\u0B95\u0BB3\u0BCD',stop:'\u23F9 \u0B92\u0BB2\u0BBF\u0BAF\u0BC8 \u0BA8\u0BBF\u0BB1\u0BC1\u0BA4\u0BCD\u0BA4\u0BC1',cancel:'\u0BB0\u0BA4\u0BCD\u0BA4\u0BC1\u0B9A\u0BC6\u0BAF\u0BCD',loading:'\u0BB5\u0BBE\u0BA9\u0BBF\u0BB2\u0BC8 \u0B95\u0BC1\u0BB0\u0BB2\u0BC8\u0BA4\u0BCD \u0BA4\u0BAF\u0BBE\u0BB0\u0BBF\u0B95\u0BCD\u0B95\u0BBF\u0BB1\u0BCB\u0BAE\u0BCD\u2026',error:'\u0B95\u0BC1\u0BB0\u0BB2\u0BC8 \u0B87\u0BAF\u0B95\u0BCD\u0B95 \u0BAE\u0BC1\u0B9F\u0BBF\u0BAF\u0BB5\u0BBF\u0BB2\u0BCD\u0BB2\u0BC8. \u0BAE\u0BC0\u0BA3\u0BCD\u0B9F\u0BC1\u0BAE\u0BCD \u0BAE\u0BC1\u0BAF\u0BB1\u0BCD\u0B9A\u0BBF\u0B95\u0BCD\u0B95\u0BB5\u0BC1\u0BAE\u0BCD.',replay:'\uD83D\uDD0A \u0BAE\u0BC0\u0BA3\u0BCD\u0B9F\u0BC1\u0BAE\u0BCD \u0B95\u0BC7\u0BB3\u0BC1\u0B99\u0BCD\u0B95\u0BB3\u0BCD'},
  Tanglish:{listen:'\uD83D\uDD0A Listen to Forecast',stop:'\u23F9 Forecast niruthunga',cancel:'Voice cancel pannunga',loading:'Forecast voice prepare pannrom\u2026',error:'Voice play aagala. Thirumba tap pannunga.',replay:'\uD83D\uDD0A Forecast thirumba kelunga'}
 };
 var state='idle',epoch=0,controller=null,audio=null,url=null,cacheKey='',cacheBlob=null,played=false;
 function $(id){return document.getElementById(id);}
 function isTamil(t){return /[\u0B80-\u0BFF]/.test(t);}
 function forecastText(){var a=$('wxManualAdvice');return a?(a.textContent||'').trim():'';}
 function language(){
  var sel=$('wxManualLanguage'),v=sel&&sel.value;
  if(v==='Tamil'||v==='Tanglish'||v==='English')return v;
  return isTamil(forecastText())?'Tamil':'English';
 }
 function words(){return COPY[language()]||COPY.English;}
 function reviewVisible(){var r=$('wxManualReview');return !!r&&!r.hidden&&r.offsetParent!==null;}
 function setLabel(){
  var b=$('wxForecastListenBtn');if(!b)return;
  b.textContent=state==='loading'?words().cancel:state==='playing'?words().stop:played?words().replay:words().listen;
  b.setAttribute('aria-pressed',String(state==='playing'));b.setAttribute('aria-busy',String(state==='loading'));
 }
 function note(t){var n=$('wxForecastListenNote');if(n)n.textContent=t||'';}
 function stop(clearCache){
  epoch++;if(controller){try{controller.abort();}catch(e){}controller=null;}
  if(audio){try{audio.pause();audio.removeAttribute('src');audio.load();}catch(e){}audio=null;}
  if(url){URL.revokeObjectURL(url);url=null;}
  if(clearCache){cacheBlob=null;cacheKey='';played=false;}
  state='idle';note('');setLabel();
 }
 async function play(){
  if(!reviewVisible()||!forecastText())return;
  if(state!=='idle'){stop(false);return;}
  var text=forecastText(),lang=language(),key=lang+'\n'+text,ticket=++epoch;
  try{
   var blob=cacheKey===key?cacheBlob:null;
   if(!blob){
    state='loading';setLabel();note(words().loading);
    controller=new AbortController();var ctl=controller,timer=setTimeout(function(){ctl.abort();},120000);
    try{
     var res=await fetch('/weather-speech',{method:'POST',credentials:'same-origin',cache:'no-store',signal:ctl.signal,headers:{'Content-Type':'application/json'},body:JSON.stringify({text:text,language:lang})});
     if(!res.ok)throw new Error('voice '+res.status);
     var data=await res.json();
     if(ticket!==epoch)return;
     if(typeof data.audio!=='string'||!data.audio)throw new Error('bad audio');
     var bin=atob(data.audio),bytes=new Uint8Array(bin.length);
     for(var i=0;i<bin.length;i++)bytes[i]=bin.charCodeAt(i);
     blob=new Blob([bytes],{type:'audio/wav'});cacheKey=key;cacheBlob=blob;
    }finally{clearTimeout(timer);}
   }
   if(ticket!==epoch)return;
   controller=null;note('');
   url=URL.createObjectURL(blob);audio=new Audio(url);audio.setAttribute('playsinline','');
   var cur=audio;
   cur.addEventListener('ended',function(){if(ticket!==epoch)return;played=true;stop(false);},{once:true});
   cur.addEventListener('error',function(){if(ticket!==epoch)return;stop(true);note(words().error);},{once:true});
   state='playing';setLabel();
   try{await cur.play();}
   catch(e){if(ticket!==epoch)return;stop(false);note(words().error);}
  }catch(e){
   if(ticket!==epoch)return;
   stop(true);note(words().error);
  }
 }
 function ensureRow(){
  var review=$('wxManualReview');if(!review)return;
  var row=$(ROW_ID);
  if(!reviewVisible()||!forecastText()){if(row)row.hidden=true;if(state!=='idle')stop(false);return;}
  var actions=review.querySelector('.wx-review-actions');if(!actions)return;
  if(!row){
   row=document.createElement('div');row.id=ROW_ID;
   row.innerHTML='<button type="button" id="wxForecastListenBtn" aria-pressed="false"></button><span id="wxForecastListenNote" role="status" aria-live="polite"></span>';
   row.querySelector('button').addEventListener('click',play);
  }
  if(row.nextElementSibling!==actions||row.parentNode!==actions.parentNode)actions.parentNode.insertBefore(row,actions);
  row.hidden=false;
  /* new forecast text or language => drop old audio */
  if(cacheKey&&cacheKey!==language()+'\n'+forecastText()&&state==='idle'){cacheBlob=null;cacheKey='';played=false;}
  setLabel();
 }
 var st=document.createElement('style');
 st.textContent='#'+ROW_ID+'{margin:12px 0;display:flex;align-items:center;gap:8px;flex-wrap:wrap}#'+ROW_ID+'[hidden]{display:none!important}'
  +'#wxForecastListenBtn{min-height:44px;width:100%;padding:10px 14px;border:1px solid var(--line,#d9d9d9);border-radius:12px;background:var(--yellow,#ffcc29);color:var(--ink,#17191d);font:inherit;font-weight:600;cursor:pointer;white-space:normal}'
  +'#wxForecastListenBtn[aria-pressed="true"]{background:var(--paper,#fff)}#wxForecastListenNote{font-size:12px;color:var(--muted,#667085);overflow-wrap:anywhere}';
 document.head.appendChild(st);
 var queued=false;
 function schedule(){if(queued)return;queued=true;setTimeout(function(){queued=false;ensureRow();},60);}
 new MutationObserver(schedule).observe(document.body,{subtree:true,childList:true,attributes:true,attributeFilter:['hidden'],characterData:true});
 document.addEventListener('change',function(e){if(e.target&&e.target.id==='wxManualLanguage'){stop(true);schedule();}},true);
 document.addEventListener('click',function(e){if(e.target.closest&&e.target.closest('#wxManualContinue,#wxManualChange,#logout,#driverMode,#aiCall'))stop(false);},true);
 window.addEventListener('pagehide',function(){stop(true);});
 document.addEventListener('visibilitychange',function(){if(document.hidden)stop(false);});
 schedule();
})();
'''

    node = shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory() as d:
            p = Path(d) / 't.js'; p.write_text(JS, encoding='utf-8')
            r = subprocess.run([node, '--check', str(p)], capture_output=True, text=True, timeout=20)
            if r.returncode: raise RuntimeError('JS syntax error: ' + r.stderr[:400])

    # --- 1. Make sure the existing-TTS audio route exists (reuses the DhVaani handler) ---
    b = globals().get('bridge')
    if not b: raise RuntimeError('Run the existing MOVIO voice setup first (bridge is missing).')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    current = b; seen = set(); app = None; actual = None
    while current is not None and id(current) not in seen:
        seen.add(id(current)); cand = getattr(current, 'app', None)
        if callable(getattr(cand, 'add_api_route', None)):
            app = cand; actual = current; break
        current = getattr(current, '__dict__', {}).get('_original')
    if app is None: raise RuntimeError('The running AI application was not found behind the bridge.')
    public = str(globals().get('MOVIO_WEBSITE_URL') or getattr(actual, 'PUBLIC_URL', '')).rstrip('/')
    if not urlsplit(public).scheme or not urlsplit(public).netloc:
        raise RuntimeError('The public website URL is missing.')
    audio = register_forecast_audio(app, b, engine, public)
    audio_ok = False

    s = requests.Session(); s.trust_env = False
    try:
        for port in (8000, 8090):
            code = s.get('http://127.0.0.1:%d/weather-speech' % port, timeout=(2, 8), allow_redirects=False).status_code
            if code != 405:
                raise RuntimeError('Port %d does not expose POST /weather-speech (GET returned %s). Share this output.' % (port, code))
    except Exception:
        if audio['added']:
            app.routes[:] = [r for r in app.routes if r is not audio['route']] + audio['replaced']
        raise

    # --- 2. Install the button script ---
    # tempfile.mkdtemp guarantees a unique backup directory on every run.
    backup = Path(tempfile.mkdtemp(prefix='listen_forecast_backup_', dir=str(root)))
    shutil.copy2(index, backup / 'index.html')
    js_path.write_text(JS, encoding='utf-8')
    token = hashlib.sha256(JS.encode()).hexdigest()[:12]
    html = index.read_text(encoding='utf-8')
    html = re.sub(r'\s*<script[^>]*movio_forecast_listen\.js[^>]*></script>', '', html)
    tag = '\n<script src="/movio_forecast_listen.js?v=%s" defer></script>' % token
    html = html.replace('</body>', tag + '\n</body>', 1) if '</body>' in html else html + tag
    index.write_text(html, encoding='utf-8')

    served = s.get('http://127.0.0.1:8080/movio_forecast_listen.js?v=' + token, timeout=8)
    page = s.get('http://127.0.0.1:8080/', timeout=8)
    gw = s.get('http://127.0.0.1:8080/weather-speech', timeout=8, allow_redirects=False).status_code
    ok = served.status_code == 200 and 'MOVIO_FORECAST_LISTEN_V3' in served.text and 'movio_forecast_listen.js' in page.text
    print({'script_served': ok, 'speech_route': 'updated' if audio['added'] else 'already current', 'website_gateway_GET_status': gw, 'backup': backup.name})
    s.close()
    if not ok:
        shutil.copy2(backup / 'index.html', index)
        raise RuntimeError('Website did not serve the new script. index.html restored.')
    print('Done. Close and reopen the dashboard tab, review a ride, tap "Listen to Forecast". It uses your existing MOVIO voice, with Tamil pronunciation for temperature/Celsius/auto, joined acronyms, and pauses after full stops and semicolons.')

movio_add_listen_to_forecast()

{'script_served': True, 'speech_route': 'updated', 'website_gateway_GET_status': 404, 'backup': 'listen_forecast_backup_ibu6n76e'}
Done. Close and reopen the dashboard tab, review a ride, tap "Listen to Forecast". It uses your existing MOVIO voice, with Tamil pronunciation for temperature/Celsius/auto, joined acronyms, and pauses after full stops and semicolons.


In [143]:
# ============================================================
# MOVIO — LISTEN TO FORECAST BACKUP ERROR FIX
# Fixes FileExistsError without changing existing functionality
# ============================================================

from pathlib import Path
from datetime import datetime
from uuid import uuid4
import shutil
import inspect

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
INDEX = ROOT / "web" / "index.html"

# 1. Verify existing website
if not INDEX.is_file():
    raise FileNotFoundError(f"Website index not found: {INDEX}")

# 2. Create a unique backup folder
BACKUP = ROOT / (
    "listen_forecast_backup_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + "_"
    + uuid4().hex[:8]
)

BACKUP.mkdir(parents=True, exist_ok=False)
shutil.copy2(INDEX, BACKUP / "index.html")

print("Backup created:", BACKUP)

# 3. Locate the existing Listen to Forecast function
FUNCTION_NAME = "movio_add_listen_to_forecast"

if FUNCTION_NAME not in globals():
    raise RuntimeError(
        "Original Listen to Forecast function not found. "
        "Run the original code cell first, after correcting its "
        "backup.mkdir() line."
    )

# 4. Replace the original function's backup creation code
original_function = globals()[FUNCTION_NAME]

try:
    source = inspect.getsource(original_function)
except (OSError, TypeError):
    source = None

if source is None:
    print(
        "\nYour notebook function source cannot be retrieved automatically."
        "\nIn the original Listen to Forecast cell, replace:"
        "\n"
        "\nbackup = root / ('listen_forecast_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S'))"
        "\nbackup.mkdir(); shutil.copy2(index, backup / 'index.html')"
        "\n"
        "\nwith:"
        "\n"
        "\nbackup = root / ('listen_forecast_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '_' + __import__('uuid').uuid4().hex[:8])"
        "\nbackup.mkdir(parents=True, exist_ok=False)"
        "\nshutil.copy2(index, backup / 'index.html')"
        "\n"
        "\nThen rerun the original cell."
    )
else:
    old = """backup = root / ('listen_forecast_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    backup.mkdir(); shutil.copy2(index, backup / 'index.html')"""

    new = """backup = root / (
        'listen_forecast_backup_'
        + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        + '_'
        + __import__('uuid').uuid4().hex[:8]
    )
    backup.mkdir(parents=True, exist_ok=False)
    shutil.copy2(index, backup / 'index.html')"""

    if old not in source:
        print("Original backup code not found; no function changes made.")
    else:
        import textwrap

        updated_source = textwrap.dedent(source).replace(old, new)

        # Compile before replacing the function
        compiled = compile(
            updated_source,
            "<movio_forecast_backup_fix>",
            "exec"
        )

        namespace = original_function.__globals__
        exec(compiled, namespace)

        print("Forecast function corrected successfully.")

        # 5. Execute corrected function
        namespace[FUNCTION_NAME]()

        print("\nSUCCESS: Listen to Forecast setup completed.")
        print("Existing booking, weather, voice and OTP logic preserved.")

Backup created: /kaggle/working/MOVIO_MAIN_WEBSITE/listen_forecast_backup_20261028_070742_454057_b999204a
Original backup code not found; no function changes made.


In [144]:
# MOVIO - Travel Companion DEBUG. Paste into ONE Kaggle cell and run.
# 1) prints what is on disk (which customer-care backend version is saved),
# 2) adds a small grey debug line at the bottom of the dashboard showing exactly
#    what the card received (HTTP status, backend version, weather status/reason,
#    and whether your phone sent a location).
# To remove the debug line later, run it again after setting  MOVIO_DEBUG_REMOVE = True  in an earlier cell.
import re, json, hashlib, shutil, requests
from pathlib import Path

def movio_companion_debug():
    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    web = root / 'web'; index = web / 'index.html'; js = web / 'movio_companion_debug.js'

    # ---- disk report ----
    rep = {}
    for p in sorted((root / 'integrations').glob('movio_customer_care*.py')) + [root / 'server.py']:
        if p.is_file():
            t = p.read_text(encoding='utf-8', errors='ignore')
            rep[p.name] = dict(
                saved=__import__('datetime').datetime.fromtimestamp(p.stat().st_mtime).strftime('%H:%M:%S'),
                versions=sorted(set(re.findall(r'customer-care-(?:api|context)-\d+\.\d+', t))),
                mentions_care_api='movio_customer_care_api' in t,
                returns_weather_to_dashboard=('"weather"' in t and 'locality' in t))
    print(json.dumps(rep, indent=2))
    try:
        import subprocess
        ps = subprocess.run(['ps', '-eo', 'lstart,cmd'], capture_output=True, text=True).stdout
        print('\nWebsite process(es):\n' + '\n'.join(l for l in ps.splitlines() if 'server.py' in l and 'grep' not in l))
    except Exception:
        pass

    html = index.read_text(encoding='utf-8')
    html = re.sub(r'\s*<script[^>]*movio_companion_debug\.js[^>]*></script>', '', html)
    if globals().get('MOVIO_DEBUG_REMOVE'):
        index.write_text(html, encoding='utf-8'); print('\nDebug line removed.'); return

    JS = r"""(function(){
 var box=document.createElement('div');
 box.style.cssText='position:fixed;left:0;right:0;bottom:0;z-index:99999;background:#111;color:#9f9;font:11px/1.35 monospace;padding:4px 8px;white-space:pre-wrap;word-break:break-all;max-height:30vh;overflow:auto';
 box.textContent='companion debug: waiting for the card to load...';
 document.addEventListener('DOMContentLoaded',function(){document.body.appendChild(box);});
 if(document.body)document.body.appendChild(box);
 var f=window.fetch;
 window.fetch=function(u,o){
  var url=String(u&&u.url||u),p=f.apply(this,arguments);
  if(url.indexOf('/api/customer-care/recommend')>-1){
   var sent='';try{var b=JSON.parse(o&&o.body||'{}');var l=b.context&&b.context.current_location||{};
    sent=('lat' in l)?'location SENT ('+l.lat+','+l.lng+')':'location NOT sent, status='+l.status;}catch(e){}
   p.then(function(r){return r.clone().json().then(function(j){
    var w=j.weather||{};
    box.textContent='companion debug: HTTP '+r.status+' | backend '+(j.version||'?')+' | weather.status='+(w.status||'none')+(w.reason?(' reason='+w.reason):'')+' | '+sent+(j.error?(' | error='+j.error):'');
   });}).catch(function(e){box.textContent='companion debug: request failed '+e;});
  }
  return p;
 };
})();
"""
    js.write_text(JS, encoding='utf-8')
    tag = '\n<script src="/movio_companion_debug.js?v=%s"></script>' % hashlib.sha256(JS.encode()).hexdigest()[:10]
    # must load BEFORE the dashboard scripts so it can wrap fetch
    html = html.replace('<head>', '<head>' + tag, 1) if '<head>' in html else tag + html
    shutil.copy2(index, index.with_name('index.before_companion_debug.bak'))
    index.write_text(html, encoding='utf-8')
    s = requests.Session(); s.trust_env = False
    ok = s.get('http://127.0.0.1:8080/movio_companion_debug.js', timeout=8).status_code == 200
    print('\nDebug script served:', ok)
    print('Now close and reopen the dashboard on the phone, sign in, wait for the card, and send me a screenshot of the dark line at the bottom.')

movio_companion_debug()

{
  "movio_customer_care.py": {
    "saved": "07:07:37",
    "versions": [],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": true
  },
  "movio_customer_care_api.py": {
    "saved": "07:07:36",
    "versions": [
      "customer-care-api-0.1"
    ],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": false
  },
  "movio_customer_care_context.py": {
    "saved": "07:07:37",
    "versions": [
      "customer-care-context-0.1"
    ],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": true
  },
  "server.py": {
    "saved": "07:07:27",
    "versions": [],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": false
  }
}

Website process(es):
Wed Oct 28 07:07:35 2026 /usr/bin/python3 /kaggle/working/MOVIO_MAIN_WEBSITE/server.py

Debug script served: True
Now close and reopen the dashboard on the phone, sign in, wait for the card, and send me a screenshot of the dark line at the bottom.


In [145]:
# MOVIO - Travel Companion DEBUG. Paste into ONE Kaggle cell and run.
# 1) prints what is on disk (which customer-care backend version is saved),
# 2) adds a small grey debug line at the bottom of the dashboard showing exactly
#    what the card received (HTTP status, backend version, weather status/reason,
#    and whether your phone sent a location).
# To remove the debug line later, run it again after setting  MOVIO_DEBUG_REMOVE = True  in an earlier cell.
import re, json, hashlib, shutil, requests
from pathlib import Path

def movio_companion_debug():
    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    web = root / 'web'; index = web / 'index.html'; js = web / 'movio_companion_debug.js'

    # ---- disk report ----
    rep = {}
    for p in sorted((root / 'integrations').glob('movio_customer_care*.py')) + [root / 'server.py']:
        if p.is_file():
            t = p.read_text(encoding='utf-8', errors='ignore')
            rep[p.name] = dict(
                saved=__import__('datetime').datetime.fromtimestamp(p.stat().st_mtime).strftime('%H:%M:%S'),
                versions=sorted(set(re.findall(r'customer-care-(?:api|context)-\d+\.\d+', t))),
                mentions_care_api='movio_customer_care_api' in t,
                returns_weather_to_dashboard=('"weather"' in t and 'locality' in t))
    print(json.dumps(rep, indent=2))
    try:
        import subprocess
        ps = subprocess.run(['ps', '-eo', 'lstart,cmd'], capture_output=True, text=True).stdout
        print('\nWebsite process(es):\n' + '\n'.join(l for l in ps.splitlines() if 'server.py' in l and 'grep' not in l))
    except Exception:
        pass

    html = index.read_text(encoding='utf-8')
    html = re.sub(r'\s*<script[^>]*movio_companion_debug\.js[^>]*></script>', '', html)
    if globals().get('MOVIO_DEBUG_REMOVE'):
        index.write_text(html, encoding='utf-8'); print('\nDebug line removed.'); return

    JS = r"""(function(){
 var box=document.createElement('div');
 box.style.cssText='position:fixed;left:0;right:0;bottom:0;z-index:99999;background:#111;color:#9f9;font:11px/1.35 monospace;padding:4px 8px;white-space:pre-wrap;word-break:break-all;max-height:30vh;overflow:auto';
 box.textContent='companion debug: waiting for the card to load...';
 document.addEventListener('DOMContentLoaded',function(){document.body.appendChild(box);});
 if(document.body)document.body.appendChild(box);
 var f=window.fetch;
 window.fetch=function(u,o){
  var url=String(u&&u.url||u),p=f.apply(this,arguments);
  if(url.indexOf('/api/customer-care/recommend')>-1){
   var sent='';try{var b=JSON.parse(o&&o.body||'{}');var l=b.context&&b.context.current_location||{};
    sent=('lat' in l)?'location SENT ('+l.lat+','+l.lng+')':'location NOT sent, status='+l.status;}catch(e){}
   p.then(function(r){return r.clone().json().then(function(j){
    var w=j.weather||{};
    box.textContent='companion debug: HTTP '+r.status+' | backend '+(j.version||'?')+' | weather.status='+(w.status||'none')+(w.reason?(' reason='+w.reason):'')+' | '+sent+(j.error?(' | error='+j.error):'');
   });}).catch(function(e){box.textContent='companion debug: request failed '+e;});
  }
  return p;
 };
})();
"""
    js.write_text(JS, encoding='utf-8')
    tag = '\n<script src="/movio_companion_debug.js?v=%s"></script>' % hashlib.sha256(JS.encode()).hexdigest()[:10]
    # must load BEFORE the dashboard scripts so it can wrap fetch
    html = html.replace('<head>', '<head>' + tag, 1) if '<head>' in html else tag + html
    shutil.copy2(index, index.with_name('index.before_companion_debug.bak'))
    index.write_text(html, encoding='utf-8')
    s = requests.Session(); s.trust_env = False
    ok = s.get('http://127.0.0.1:8080/movio_companion_debug.js', timeout=8).status_code == 200
    print('\nDebug script served:', ok)
    print('Now close and reopen the dashboard on the phone, sign in, wait for the card, and send me a screenshot of the dark line at the bottom.')

movio_companion_debug()

{
  "movio_customer_care.py": {
    "saved": "07:07:37",
    "versions": [],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": true
  },
  "movio_customer_care_api.py": {
    "saved": "07:07:36",
    "versions": [
      "customer-care-api-0.1"
    ],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": false
  },
  "movio_customer_care_context.py": {
    "saved": "07:07:37",
    "versions": [
      "customer-care-context-0.1"
    ],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": true
  },
  "server.py": {
    "saved": "07:07:27",
    "versions": [],
    "mentions_care_api": false,
    "returns_weather_to_dashboard": false
  }
}

Website process(es):
Wed Oct 28 07:07:35 2026 /usr/bin/python3 /kaggle/working/MOVIO_MAIN_WEBSITE/server.py

Debug script served: True
Now close and reopen the dashboard on the phone, sign in, wait for the card, and send me a screenshot of the dark line at the bottom.


In [146]:
# MOVIO - FIX "Your Travel Companion" weather card after the website restart.
# Re-adds the customer-care weather backend (integrations/movio_customer_care_api.py, v1.0)
# to server.py and restarts ONLY the website process (port 8080). AI call / tunnel are untouched.
# server.py is backed up first and restored automatically if the website does not come back.
def movio_fix_companion():
    import ast, os, re, shutil, subprocess, sys, time
    import importlib.util
    from datetime import datetime
    from pathlib import Path
    import psutil, requests

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    server_path = root / 'server.py'
    api_path = root / 'integrations/movio_customer_care_api.py'
    for p in (server_path, api_path, root / 'integrations/movio_customer_care.py', root / 'integrations/movio_customer_care_context.py'):
        if not p.is_file(): raise RuntimeError('Missing file: ' + str(p) + ' - re-run the customer-care setup cells first.')

    api_src = api_path.read_text(encoding='utf-8')
    ast.parse(api_src)
    if 'def install_backend' not in api_src: raise RuntimeError('The care API file has no install_backend().')
    print('Care API file version:', re.findall(r'customer-care-api-\d+\.\d+', api_src))
    print('Care API uses weather context module:', 'movio_customer_care_context' in api_src or 'context' in api_src)

    marker = '# MOVIO_CARE_API_BOOTSTRAP_V1'; end_marker = '# MOVIO_CARE_API_BOOTSTRAP_END_V1'
    text = server_path.read_text(encoding='utf-8')
    if marker in text:
        s = text.index(marker); e = text.find(end_marker, s)
        text = text[:s] + text[e + len(end_marker):] if e > 0 else text[:s]
    tree = ast.parse(text)
    launchers = [n for n in tree.body if isinstance(n, ast.If)
                 and any(isinstance(x, ast.Name) and x.id == '__name__' for x in ast.walk(n.test))
                 and any(isinstance(c, ast.Call) and isinstance(c.func, ast.Attribute) and c.func.attr == 'serve_forever'
                         for st in n.body for c in ast.walk(st))]
    if len(launchers) != 1: raise RuntimeError('Could not identify the single website launch block; nothing changed.')
    bootstrap = marker + '''
if __name__ == '__main__':
    import importlib.util as _cc_import
    import sys as _cc_sys
    _cc_path = ROOT / 'integrations/movio_customer_care_api.py'
    _cc_spec = _cc_import.spec_from_file_location('movio_customer_care_api', _cc_path)
    _cc_module = _cc_import.module_from_spec(_cc_spec)
    _cc_sys.modules[_cc_spec.name] = _cc_module
    _cc_spec.loader.exec_module(_cc_module)
    _cc_module.install_backend(_cc_sys.modules[__name__])
''' + end_marker + '\n\n'
    lines = text.splitlines(keepends=True); at = launchers[0].lineno - 1
    # keep any decorators/comments directly above the launcher untouched
    new_text = ''.join(lines[:at]) + bootstrap + ''.join(lines[at:])
    compile(new_text, str(server_path), 'exec')

    backup = server_path.with_name('server.py.before_care_api_' + datetime.now().strftime('%H%M%S'))
    shutil.copy2(server_path, backup)

    def website_pids():
        return sorted({c.pid for c in psutil.net_connections(kind='inet')
                       if c.status == 'LISTEN' and c.laddr.port == 8080 and c.pid})
    def restart():
        env = dict(os.environ)
        for pid in website_pids():
            if pid == os.getpid(): raise RuntimeError('An in-kernel service owns port 8080; use your original website launcher.')
            proc = psutil.Process(pid)
            if not any((Path(proc.cwd()) / a).resolve() == server_path for a in proc.cmdline() if a.endswith('.py')):
                raise RuntimeError('A different application owns port 8080; nothing was stopped.')
            env.update(proc.environ()); proc.terminate(); proc.wait(timeout=10)
        if globals().get('bridge') is not None and getattr(globals()['bridge'], 'PUBLIC_URL', None):
            env['MOVIO_PUBLIC_URL'] = bridge.PUBLIC_URL
        env['MOVIO_PORT'] = '8080'
        with (root / 'weather_website.log').open('a') as log:
            subprocess.Popen([sys.executable, str(server_path)], cwd=root, env=env, stdout=log,
                             stderr=subprocess.STDOUT, start_new_session=True)
        s = requests.Session(); s.trust_env = False
        for _ in range(40):
            time.sleep(0.5)
            try:
                if s.get('http://127.0.0.1:8080/api/health', timeout=2).status_code == 200:
                    return s.get('http://127.0.0.1:8080/api/customer-care/health', timeout=3).status_code
            except Exception: pass
        return None

    server_path.write_text(new_text, encoding='utf-8')
    status = restart()
    if status not in (200, 401, 403):
        shutil.copy2(backup, server_path); restart()
        raise RuntimeError('Website did not return healthy (customer-care health=%s). server.py restored from %s.' % (status, backup.name))
    print('Website restarted. customer-care health route:', status, '(401 = present, needs sign-in)')
    print('Backup of previous server.py:', backup.name)
    print('Now: close the dashboard tab, reopen it, sign in as a customer, allow Location, and tap the refresh button on the Travel Companion card.')

movio_fix_companion()

Care API file version: ['customer-care-api-0.1']
Care API uses weather context module: True
Website restarted. customer-care health route: 401 (401 = present, needs sign-in)
Backup of previous server.py: server.py.before_care_api_070742
Now: close the dashboard tab, reopen it, sign in as a customer, allow Location, and tap the refresh button on the Travel Companion card.


In [147]:
# UPDATED FORECAST LISTENING CELL — run after the existing voice setup, before evaluation.
# Replace the previous Listen to Forecast cell with this complete cell, then hard-refresh
# the dashboard. No model restart, booking changes or sampling-quality reduction.
# Starts after the first sentence rather than waiting for the entire forecast.
# Later sentences can still wait for GPU time; actual timings depend on Kaggle load.

# ---------------------------------------------------------------------------
# MOVIO forecast speech preparation (used only by the "Listen to Forecast" button).
# Pure text/WAV helpers: they never touch the booking, call or confirmation voice.
# ---------------------------------------------------------------------------
import re as _re

_MOVIO_LETTERS = {
    'A': 'ஏ', 'B': 'பி', 'C': 'சி', 'D': 'டி', 'E': 'ஈ', 'F': 'எஃப்', 'G': 'ஜி', 'H': 'ஹெச்',
    'I': 'ஐ', 'J': 'ஜே', 'K': 'கே', 'L': 'எல்', 'M': 'எம்', 'N': 'என்', 'O': 'ஓ', 'P': 'பி',
    'Q': 'கியூ', 'R': 'ஆர்', 'S': 'எஸ்', 'T': 'டி', 'U': 'யூ', 'V': 'வி', 'W': 'டபிள்யூ',
    'X': 'எக்ஸ்', 'Y': 'வை', 'Z': 'ஸெட்',
}
# Real words that happen to be written in capitals: never spell these out.
_MOVIO_CAPS_WORDS = {'AUTO', 'MINI', 'BIKE', 'TAXI', 'SEDAN', 'RAIN', 'HEAT', 'NOTE', 'HIGH',
                     'LOW', 'YES', 'NO', 'OK', 'AND', 'THE', 'FOR'}
_MOVIO_WORDS = (  # spoken with Tamil pronunciation in English, Tanglish and Tamil
    (r'(?<![A-Za-z])temperatures(?![A-Za-z])', 'டெம்பரேச்சர்ஸ்'),
    (r'(?<![A-Za-z])temperature(?![A-Za-z])', 'டெம்பரேச்சர்'),
    (r'(?<![A-Za-z])celsius(?![A-Za-z])', 'செல்சியஸ்'),
    (r'(?<![A-Za-z])autos(?![A-Za-z])', 'ஆட்டோக்கள்'),
    (r'(?<![A-Za-z])auto(?![A-Za-z])', 'ஆட்டோ'),
)


def movio_spell_acronym(letters):
    """ICF -> ஐசிஎஃப்: letter names joined into ONE word, so it is read together."""
    return ''.join(_MOVIO_LETTERS[c] for c in letters if c in _MOVIO_LETTERS)


def movio_prepare_forecast_text(text, language, vocab=None):
    value = _re.sub(r'\s+', ' ', str(text)).strip()
    if language == 'Tamil':   # spoken wording of the opening phrase
        value = _re.sub(
            r'^(?:ஒரு\s+சின்ன\s+வானிலை\s+தகவல்|ஒரு\s+வானிலை\s+அறிவிப்பு|ஒரு\s+அறிவிப்பு)(?![\w\u0B80-\u0BFF])\s*[:;.!?।]?\s*',
            'ஒரு அறிவிப்பு. ',
            value
        )
    elif language == 'Tanglish':
        # These boundaries affect the speech copy only, never the transcript.
        value = _re.sub(
            r'^oru\s+weather\s+update(?![A-Za-z])\s*[:;.!?।]?\s*',
            'Oru weather update. ', value, flags=_re.I
        )
        value = _re.sub(
            r'(?<![A-Za-z])(umbrella\s+eduthukkondu)(?![A-Za-z])\s*[.,;:!?।]?\s*',
            lambda match: match.group(1) + '; ', value, flags=_re.I
        )
        # Phonetic spelling of English "ninety", not a translation of the value.
        # Match only the complete 90% / 90 percent entity; leave 190%, 9.0%,
        # temperatures, times, dates and identifiers unchanged.
        value = _re.sub(
            r'(?<![\w.])90\s*(?:%|percent(?![A-Za-z]))',
            'நைன்டி percent', value, flags=_re.I
        )
    # 28.6°C / 28.6 °C / "degrees Celsius" -> Tamil pronunciation of Celsius
    value = _re.sub(r'\s*°\s*C(?![A-Za-z])',
                    ' டிகிரி செல்சியஸ்' if language == 'Tamil' else ' degrees செல்சியஸ்', value)
    for pattern, spoken in _MOVIO_WORDS:
        value = _re.sub(pattern, spoken, value, flags=_re.I)

    def dotted(match):
        end = match.end()
        tail = value[end:end + 3]
        keep_stop = end >= len(value) or _re.match(r'\s+[A-Z஀-௿]', tail + ' ')
        return movio_spell_acronym(_re.sub(r'[^A-Z]', '', match.group(0))) + ('.' if keep_stop else '')
    value = _re.sub(r'(?<![A-Za-z])(?:[A-Z]\.){2,6}', dotted, value)
    value = _re.sub(r'(?<![A-Za-z0-9])[A-Z]{2,6}(?![A-Za-z0-9])',
                    lambda m: m.group(0) if m.group(0) in _MOVIO_CAPS_WORDS else movio_spell_acronym(m.group(0)),
                    value)
    if language == 'Tanglish':
        value = movio_tanglish_script(value, vocab)
    else:   # English and Tamil: same long-ee spelling so it is never read as 'mani'
        value = _re.sub(r'(?<![A-Za-z])mini(?![A-Za-z])', 'மினீ', value, flags=_re.I)
    # A colon inside a sentence must not break it (time such as 4:02 is kept).
    value = _re.sub(r'(?<!\d):|:(?!\d)', ' ', value)
    return _re.sub(r'\s+', ' ', value).strip()


def movio_forecast_parts(text, language, vocab=None):
    """[(sentence, pause_seconds_after)]: pause after every full stop and semicolon."""
    prepared = movio_prepare_forecast_text(text, language, vocab)
    parts = []
    for piece in _re.split(r'(?<=[.;!?।])\s+', prepared):
        piece = piece.strip()
        if not piece:
            continue
        stop = piece[-1]
        body = piece.rstrip('.;!?। ').strip()
        if not _re.search(r'[A-Za-z0-9஀-௿]', body):
            continue
        parts.append((body + ('?' if stop == '?' else '.'), 0.5 if stop == ';' else 0.7))
    if parts:
        parts[-1] = (parts[-1][0], 0.0)
    return parts


def movio_join_wavs(wav_pauses):
    """Join WAV byte strings into one WAV, inserting the given silence after each."""
    import io, wave
    out = io.BytesIO(); writer = None; params = None
    for raw, pause in wav_pauses:
        with wave.open(io.BytesIO(raw), 'rb') as part:
            current = (part.getnchannels(), part.getsampwidth(), part.getframerate())
            frames = part.readframes(part.getnframes())
        if writer is None:
            params = current
            writer = wave.open(out, 'wb')
            writer.setnchannels(current[0]); writer.setsampwidth(current[1]); writer.setframerate(current[2])
        elif current != params:
            raise ValueError('Speech chunks have different audio formats')
        writer.writeframes(frames)
        if pause > 0:
            writer.writeframes(b'\x00' * (int(params[2] * pause) * params[0] * params[1]))
    if writer is None:
        raise ValueError('No speech was generated')
    writer.close()
    return out.getvalue()


# ---------------------------------------------------------------------------
# Tanglish speech copy: Tamil words -> Tamil script (Tamil pronunciation);
# English words stay in English letters (English pronunciation).
# ---------------------------------------------------------------------------
_TG_ENGLISH = set(['a', 'ai', 'am', 'an', 'and', 'approximate', 'april', 'are', 'around', 'at', 'august', 'auto', 'bags', 'be', 'belongings', 'bike', 'billion', 'book', 'booking', 'by', 'can', 'cancel', 'cancellation', 'car', 'care', 'celsius', 'chance', 'change', 'check', 'cloudy', 'comfortable', 'confirm', 'confirmation', 'consider', 'continue', 'correct', 'cover', 'covered', 'crore', 'crores', 'december', 'degree', 'degrees', 'destination', 'detail', 'details', 'distance', 'driver', 'drop', 'eight', 'eighteen', 'eighty', 'eleven', 'estimated', 'evening', 'existing', 'fare', 'february', 'fifteen', 'fifty', 'five', 'for', 'forecast', 'forty', 'four', 'fourteen', 'friday', 'from', 'have', 'heat', 'heavy', 'help', 'hundred', 'id', 'in', 'is', 'it', 'january', 'journey', 'july', 'june', 'keep', 'kilometer', 'kilometers', 'kilometres', 'lakh', 'lakhs', 'map', 'march', 'may', 'me', 'million', 'mini', 'monday', 'morning', 'my', 'night', 'nine', 'nineteen', 'ninety', 'no', 'november', 'number', 'october', 'of', 'oh', 'on', 'one', 'open', 'otp', 'our', 'passenger', 'passengers', 'pickup', 'place', 'pleasant', 'please', 'pm', 'point', 'prepare', 'protection', 'rain', 'record', 'request', 'retain', 'retry', 'ride', 'road', 'route', 'rupee', 'rupees', 'same', 'saturday', 'save', 'saved', 'screen', 'sedan', 'select', 'selected', 'september', 'service', 'seven', 'seventeen', 'seventy', 'six', 'sixteen', 'sixty', 'strong', 'sun', 'sunday', 'sunny', 'suv', 'switch', 'taxi', 'temperature', 'ten', 'thank', 'thanks', 'the', 'thirteen', 'thirty', 'thousand', 'three', 'thunderstorm', 'thunderstorms', 'thursday', 'time', 'to', 'today', 'tomorrow', 'travel', 'tuesday', 'twelve', 'twenty', 'two', 'umbrella', 'update', 'vehicle', 'verify', 'warnings', 'weather', 'wednesday', 'welcome', 'will', 'wind', 'windy', 'with', 'wonderful', 'yes', 'you', 'your', 'zero']) | set('''quick possible about should provides provide carry protect side drop off dropoff
selected pickup ride passengers passenger vehicle mini auto sedan suv bike car cover umbrella bags bag
weather update rain chance degrees degree temperature forecast shall confirm booking please'''.split())
_TG_ALIASES = {'taxi': 'டெக்ஸி', 'auto': 'ஆட்டோ', 'booking': 'புக்கிங்', 'celsius': 'செல்சியஸ்', 'temperature': 'டெம்பரேச்சர்', 'central': 'சென்ட்ரல்', 'belongings': 'பிலாங்கிங்ஸ்', 'belonging': 'பிலாங்கிங்', 'iruki': 'இருக்கு', 'irukki': 'இருக்கு', 'iruku': 'இருக்கு', 'solluren': 'சொல்லுறேன்', 'solluraen': 'சொல்லுறேன்', 'vanakkam': 'வணக்கம்', 'nandri': 'நன்றி', 'unga': 'உங்க', 'ungal': 'உங்கள்', 'ungalukku': 'உங்களுக்கு', 'neenga': 'நீங்க', 'eppadi': 'எப்படி', 'pannalama': 'பண்ணலாமா', 'pannalaama': 'பண்ணலாமா', 'pannalaam': 'பண்ணலாம்', 'pannunga': 'பண்ணுங்க', 'pannikkonga': 'பண்ணிக்கோங்க', 'pannrom': 'பண்றோம்', 'sollunga': 'சொல்லுங்க', 'sollalaam': 'சொல்லலாம்', 'illa': 'இல்ல', 'illai': 'இல்லை', 'vera': 'வேற', 'veru': 'வேறு', 'aiye': 'ஐயே', 'aiyum': 'ஐயும்', 'um': 'உம்', 'adhe': 'அதே', 'athe': 'அதே', 'idhe': 'இதே', 'ithae': 'இதே', 'indha': 'இந்த', 'andha': 'அந்த', 'naal': 'நாள்', 'naalaiku': 'நாளைக்கு', 'innikku': 'இன்னைக்கு', 'innaiku': 'இன்னைக்கு', 'neram': 'நேரம்', 'nerathil': 'நேரத்தில்', 'nerathula': 'நேரத்துல', 'kelambura': 'கிளம்புற', 'kelambumbodhu': 'கிளம்பும்போது', 'purappadum': 'புறப்படும்', 'purappadumbodhu': 'புறப்படும்போது', 'irukku': 'இருக்கு', 'irukkum': 'இருக்கும்', 'irukkura': 'இருக்குற', 'irukkalaam': 'இருக்கலாம்', 'irundhu': 'இருந்து', 'ilirundhu': 'இலிருந்து', 'peru': 'பேர்', 'ku': 'க்கு', 'la': 'ல', 'ah': 'ஆ', 'mazhai': 'மழை', 'peiya': 'பெய்ய', 'vaaippu': 'வாய்ப்பு', 'vaanilai': 'வானிலை', 'veppanilai': 'வெப்பநிலை', 'sumaar': 'சுமார்', 'adhigamaaga': 'அதிகமாக', 'adhigamaa': 'அதிகமா', 'konjam': 'கொஞ்சம்', 'megamoottam': 'மேகமூட்டம்', 'kanamazhai': 'கனமழை', 'kudai': 'குடை', 'kudineer': 'குடிநீர்', 'eduthuttu': 'எடுத்துட்டு', 'eduthukkonga': 'எடுத்துக்கோங்க', 'eduthukkondu': 'எடுத்துக்கொண்டு', 'nanaiyaamal': 'நனையாமல்', 'nanaiyama': 'நனையாம', 'paadhugaathukkollunga': 'பாதுகாத்துக்கொள்ளுங்க', 'paadhugaappa': 'பாதுகாப்பா', 'vachukkonga': 'வச்சுக்கோங்க', 'erumboadhum': 'ஏறும்போதும்', 'irangumboadhum': 'இறங்கும்போதும்', 'eravum': 'ஏறவும்', 'irangavum': 'இறங்கவும்', 'pakkavaatil': 'பக்கவாட்டில்', 'padalaam': 'படலாம்', 'purappadumun': 'புறப்படும் முன்', 'mun': 'முன்', 'ulloor': 'உள்ளூர்', 'paarunga': 'பாருங்க', 'thevaiyaana': 'தேவையான', 'munnerpaadugal': 'முன்னேற்பாடுகள்', 'payanathirku': 'பயணத்திற்கு', 'payanam': 'பயணம்', 'inimaiyaaga': 'இனிமையாக', 'amaiyattum': 'அமையட்டும்', 'nandraagave': 'நன்றாகவே'}
# Edit this table to fix how any Tanglish word is spoken (checked before everything else).
_TG_OVERRIDES = dict(peiya='பெய்ய', peyya='பெய்ய', umbrella='அம்ப்ரெல்லா', umbrellas='அம்ப்ரெல்லாக்கள்', mini='மினீ',
                     ninety='நைன்டி', forecast='ஃபோர்காஸ்ட்')
_TG_ALIASES.update(virumbina='விரும்பினா', oru='ஒரு', ana='ஆனா', aana='ஆனா', thaan='தான்', than='தான்', enna='என்ன', mela='மேல',
                   kelunga='கேளுங்க', irukkunga='இருக்குங்க', irukkumnu='இருக்கும்னு')
_TG_SIGNS = ('zh', 'dh', 'aa', 'oa', 'ae', 'nga', 'ngu', 'kk', 'rr', 'nj', 'ai', 'ei', 'au', 'ee', 'oo')
_TG_STRONG = ('zh', 'dh', 'aa', 'oa', 'ae', 'kk', 'rr', 'nj', 'ngu', 'nga')
_TG_SUFFIX = ('kku', 'aal', 'udan', 'nga', 'ngal', 'ndu', 'adhu', 'athu', 'aama', 'aamal', 'aaga', 'irukku', 'irukkum', 'kollunga')

_TG_C = [('ng', 'ங்'), ('nj', 'ஞ்'), ('zh', 'ழ்'), ('th', 'த்'), ('dh', 'த்'), ('ch', 'ச்'), ('sh', 'ஷ்'),
         ('kk', 'க்க்'), ('pp', 'ப்ப்'), ('tt', 'ட்ட்'), ('mm', 'ம்ம்'), ('nn', 'ன்ன்'), ('ll', 'ல்ல்'), ('rr', 'ற்ற்'),
         ('vv', 'வ்வ்'), ('yy', 'ய்ய்'), ('ss', 'ஸ்ஸ்'),
         ('k', 'க்'), ('g', 'க்'), ('p', 'ப்'), ('b', 'ப்'), ('m', 'ம்'), ('y', 'ய்'), ('r', 'ர்'), ('l', 'ல்'),
         ('v', 'வ்'), ('w', 'வ்'), ('s', 'ச்'), ('j', 'ஜ்'), ('h', 'ஹ்'), ('n', 'ன்'), ('t', 'ட்'), ('d', 'ட்'),
         ('f', 'ஃப்'), ('x', 'க்ஸ்'), ('q', 'க்'), ('z', 'ஸ்')]
_TG_V = [('aa', 'ஆ', 'ா'), ('ae', 'ஏ', 'ே'), ('ai', 'ஐ', 'ை'), ('au', 'ஔ', 'ௌ'), ('ee', 'ஈ', 'ீ'), ('ii', 'ஈ', 'ீ'),
         ('oo', 'ஊ', 'ூ'), ('uu', 'ஊ', 'ூ'), ('oa', 'ஓ', 'ோ'), ('oe', 'ஓ', 'ோ'),
         ('a', 'அ', ''), ('i', 'இ', 'ி'), ('u', 'உ', 'ு'), ('e', 'எ', 'ெ'), ('o', 'ஒ', 'ொ')]


def movio_roman_to_tamil(word):
    """Rule-based Tanglish -> Tamil script for words missing from the dictionary."""
    w = word.lower().replace("'", '')
    out, i = [], 0                                       # out[-1] ends with a virama while a consonant is open
    def open_consonant():
        return bool(out) and out[-1].endswith('\u0BCD')
    while i < len(w):
        if w.startswith('ei', i):                       # peiya -> பெய்ய
            if open_consonant():out[-1] = out[-1][:-1] + '\u0BC6'
            else:out.append('\u0B8E')
            out.append('\u0BAF\u0BCD'); i += 2; continue
        vowel = next((v for v in _TG_V if w.startswith(v[0], i)), None)
        if vowel:
            if open_consonant():out[-1] = out[-1][:-1] + vowel[2]
            else:out.append(vowel[1])
            i += len(vowel[0]); continue
        cons = next((c for c in _TG_C if w.startswith(c[0], i)), None)
        if not cons:
            return word                                  # unknown letter: leave untouched
        text = cons[1]
        nxt = w[i + len(cons[0]):i + len(cons[0]) + 1]
        if cons[0] in ('ng', 'nj') and nxt and nxt in 'aeiou':
            text += '\u0B95\u0BCD' if cons[0] == 'ng' else '\u0B9A\u0BCD'   # nga -> ங்க, nja -> ஞ்ச
        elif cons[0] == 'n' and i == 0:
            text = '\u0BA8\u0BCD'
        out.append(text); i += len(cons[0])
    result = ''.join(out)
    return result if _re.search(r'[\u0B80-\u0BFF]', result) else word


def movio_runtime_vocab(b):
    """Tamil-script spellings already used by the running Tanglish voice (optional)."""
    try:
        speech = getattr(b, 'tanglish_speech_text', None)
        cells = dict(zip(speech.__code__.co_freevars, speech.__closure__ or ()))
        return {k: v for k, v in cells['words'].cell_contents.items()
                if isinstance(v, str) and _re.search(r'[\u0B80-\u0BFF]', v)}
    except Exception:
        return {}


def movio_tanglish_script(value, vocab=None):
    """Tamil words -> Tamil script. English words, names and numbers stay as they are."""
    vocab = vocab or {}

    def token(match):
        raw = match.group(0); low = raw.lower()
        before = value[:match.start()].rstrip()
        sentence_start = not before or before[-1] in '.;!?'
        if low in _TG_OVERRIDES:
            return _TG_OVERRIDES[low]
        if low in _TG_ALIASES:
            return _TG_ALIASES[low]
        if low in vocab:
            return vocab[low]
        if low in _TG_ENGLISH:
            return raw
        if raw[0].isupper() and not sentence_start:
            return raw                                   # place names such as Tondiarpet
        if any(sign in low for sign in _TG_STRONG) or (len(low) > 4 and low.endswith(_TG_SUFFIX)):
            return movio_roman_to_tamil(raw)
        return raw
    return _re.sub(r"[A-Za-z][A-Za-z']*", token, value)



import contextvars as _cv
_MOVIO_ONE_CHUNK = _cv.ContextVar('movio_forecast_one_chunk', default=False)


def movio_install_single_chunk(b):
    """While a forecast sentence is spoken, give the voice engine ONE speech chunk per sentence
    (up to 28 words) so it never breaks inside e.g. 'ஒரு தகவல்'. Other speech is unaffected."""
    adapter = getattr(b, 'TTS', None)
    ns = getattr(adapter, 'ns', None)
    targets = []
    if isinstance(ns, dict):
        targets.append(ns)
        stream = ns.get('movio_stream_response')
        stream_globals = getattr(stream, '__globals__', None)
        if isinstance(stream_globals, dict) and stream_globals is not ns:
            targets.append(stream_globals)
    wrapped = 0
    for target in targets:
        previous = target.get('prepare_movio_request')
        if not callable(previous):
            continue
        if getattr(previous, '_movio_one_chunk', False):
            wrapped += 1
            continue
        def make(prev):
            def prepare(*args, **kwargs):
                plan = prev(*args, **kwargs)
                if not _MOVIO_ONE_CHUNK.get() or not isinstance(plan, dict):
                    return plan
                chunks = [str(c).strip() for c in plan.get('chunks', []) if str(c).strip()]
                groups, current = [], ''
                for chunk in chunks:
                    if current and len((current + ' ' + chunk).split()) > 28:
                        groups.append(current); current = chunk
                    else:
                        current = (current + ' ' + chunk).strip()
                if current:
                    groups.append(current)
                return {**plan, 'chunks': groups or chunks}
            prepare._movio_one_chunk = True
            return prepare
        target['prepare_movio_request'] = make(previous)
        wrapped += 1
    return wrapped


def register_forecast_audio(app, bridge_object, engine, public_origin):
    """Reuse the existing endpoint, or add one authenticated TTS-only route."""
    import asyncio
    import base64
    import inspect
    import json
    import threading
    import time
    import hashlib
    from collections import OrderedDict
    import requests
    from fastapi import HTTPException, Request

    version='forecast-voice-3.0-progressive'
    existing=[r for r in app.routes if getattr(r,'path',None)=='/weather-speech']
    replaced=[]
    for route in existing:
        if getattr(route.endpoint,'_movio_forecast_voice',None) is None:
            raise RuntimeError('A different /weather-speech endpoint exists. Share its source before modifying it.')
    if len(existing)==1 and existing[0].endpoint._movio_forecast_voice==version and 'POST' in (getattr(existing[0],'methods',set()) or set()):
        return dict(added=False,replaced=[],route=existing[0],endpoint=existing[0].endpoint)
    replaced=list(existing)  # older MOVIO forecast-voice route: replaced by this version
    if replaced:app.routes[:]=[r for r in app.routes if r not in replaced]
    tts=engine.get('tts_reply')
    language_scope=getattr(bridge_object,'CURRENT_LANGUAGE',None)
    if not callable(tts) or not callable(getattr(language_scope,'set',None)):
        raise RuntimeError('The existing optimized DhVaani reply handler is missing.')
    register=None
    advisory=engine.get('advisory')
    if callable(advisory):
        try:register=inspect.getclosurevars(advisory).nonlocals.get('register')
        except (TypeError,ValueError):pass
    if not callable(register):
        # Fallback: write straight into the weather-profile registry used by tts_reply.
        try:
            profile_for=inspect.getclosurevars(tts).nonlocals.get('profile_for')
            profiles=inspect.getclosurevars(profile_for).nonlocals.get('profiles')
            guard=inspect.getclosurevars(profile_for).nonlocals.get('registry_lock')
            if profiles is not None and guard is not None:
                def register(text,kind):
                    with guard:
                        profiles[str(text).strip()]=kind;profiles.move_to_end(str(text).strip())
        except (TypeError,ValueError,AttributeError):pass
    busy=threading.BoundedSemaphore(1)
    cache=OrderedDict()
    cache_lock=threading.Lock()
    cache_bytes=0
    CACHE_LIMIT=32*1024*1024
    CACHE_TTL=900  # In-memory only; never write forecast audio or location data to disk.
    class Busy(Exception):pass
    def clear_cache():
        nonlocal cache_bytes
        with cache_lock:cache.clear();cache_bytes=0
    def render(text,language,part=None):
        nonlocal cache_bytes
        started=time.perf_counter()
        parts=movio_forecast_parts(text,language,movio_runtime_vocab(bridge_object) if language=='Tanglish' else None)
        if not parts or len(parts)>14:raise ValueError('Forecast text cannot be spoken')
        if part is not None and (type(part) is not int or not 0<=part<len(parts)):
            raise ValueError('Invalid forecast part')
        selected=parts if part is None else [parts[part]]
        if not busy.acquire(blocking=False):raise Busy()
        token=language_scope.set(language)
        quality=getattr(bridge_object,'WX_SPEECH_QUALITY',None)
        old_min=quality.get('min_words') if isinstance(quality,dict) else None
        had_min=isinstance(quality,dict) and 'min_words' in quality
        if isinstance(quality,dict):quality['min_words']=max(int(old_min or 8),16)
        one=None
        try:
            pieces=[];hits=0
            movio_install_single_chunk(bridge_object)
            one=_MOVIO_ONE_CHUNK.set(True)
            ns=getattr(getattr(bridge_object,'TTS',None),'ns',{})
            voice_settings={k:ns.get(k) for k in ('FIRST_CHUNK_NUM_STEP','REST_CHUNK_NUM_STEP',
                'ENGLISH_SPEED','TAMIL_SPEED','TANGLISH_SPEED','USE_FP16_SAMPLE','SEED','MODEL_ID','REVISION')} if isinstance(ns,dict) else {}
            signature=repr((version,id(tts),quality,voice_settings,globals().get('MOVIO_FORECAST_CACHE_REVISION','1')))
            for sentence,pause in selected:
                key=hashlib.sha256((signature+'\n'+language+'\n'+sentence).encode()).hexdigest()
                raw=None;now=time.monotonic()
                with cache_lock:
                    # Expire inactive entries too, so retained speech stays bounded by age.
                    for old_key,(created,old_raw) in list(cache.items()):
                        if now-created>CACHE_TTL:
                            cache_bytes-=len(old_raw);del cache[old_key]
                    entry=cache.get(key)
                    if entry is not None:raw=entry[1];cache.move_to_end(key);hits+=1
                if raw is None:
                    if callable(register):register(sentence,'weather')
                    audio=tts(bridge_object,sentence)
                    if not isinstance(audio,str):raise ValueError('Invalid speech response')
                    raw=base64.b64decode(audio,validate=True)
                    if len(raw)<44 or raw[:4]!=b'RIFF' or raw[8:12]!=b'WAVE':raise ValueError('Invalid WAV response')
                    # Validate before caching; the same working pronunciation/TTS path is used.
                    movio_join_wavs([(raw,0)])
                    if len(raw)>9000000:raise ValueError('Speech response too large')
                    with cache_lock:
                        cache[key]=(time.monotonic(),raw);cache_bytes+=len(raw)
                        while cache_bytes>CACHE_LIMIT or len(cache)>256:
                            _,(_,removed)=cache.popitem(last=False);cache_bytes-=len(removed)
                pieces.append((raw,pause))
            joined=movio_join_wavs(pieces)
            if len(joined)>9000000:raise ValueError('Speech response too large')
            return dict(audio=base64.b64encode(joined).decode(),format='wav',
                part=part,total_parts=len(parts),cached_parts=hits,
                generation_ms=round((time.perf_counter()-started)*1000,1),protocol='parts-v1')
        finally:
            if one is not None:_MOVIO_ONE_CHUNK.reset(one)
            if isinstance(quality,dict):
                if had_min:quality['min_words']=old_min
                else:quality.pop('min_words',None)
            language_scope.reset(token);busy.release()
    def authenticate(cookie):
        # Reuse website authentication without exposing account data or keys.
        with requests.Session() as session:
            session.trust_env=False
            response=session.get('http://127.0.0.1:8080/api/me',headers={'Cookie':cookie},
                                 timeout=(2,5),allow_redirects=False)
            if response.status_code!=200:raise PermissionError()
            account=response.json()
            if account.get('role')!='customer':raise PermissionError()

    async def manual_forecast_voice(request:Request):
        origin=request.headers.get('origin','').rstrip('/')
        if origin and origin!=public_origin.rstrip('/'):
            raise HTTPException(403,'Origin not allowed')
        cookie=request.headers.get('cookie','')
        if not cookie:raise HTTPException(401,'Sign in to listen to this forecast')
        try:await asyncio.to_thread(authenticate,cookie)
        except PermissionError:raise HTTPException(403,'Customer login required')
        except Exception:raise HTTPException(503,'Account verification is temporarily unavailable')
        try:
            body=await request.body()
            if len(body)>16000:raise ValueError()
            data=json.loads(body)
            if not isinstance(data,dict):raise ValueError()
            text=data.get('text');language=data.get('language')
            if not isinstance(text,str) or not text.strip() or len(text)>1500:raise ValueError()
            if language not in ('English','Tamil','Tanglish'):raise ValueError()
            part=data.get('part')
            if part is not None and (type(part) is not int or not 0<=part<14):raise ValueError()
        except (ValueError,TypeError,UnicodeError):raise HTTPException(400,'Invalid forecast voice request')
        try:
            return await asyncio.to_thread(render,text.strip(),language,part)
        except Busy:raise HTTPException(429,'Forecast voice is busy; please retry')
        except ValueError:raise HTTPException(400,'Invalid forecast speech part or audio')
        except Exception:raise HTTPException(503,'Forecast voice could not be generated')
    manual_forecast_voice._movio_forecast_clear_cache=clear_cache
    manual_forecast_voice._movio_forecast_voice=version
    manual_forecast_voice.__annotations__['request']=Request
    app.add_api_route('/weather-speech',manual_forecast_voice,methods=['POST'])
    route=next(r for r in app.routes if getattr(r,'endpoint',None) is manual_forecast_voice)
    app.openapi_schema=None
    return dict(added=True,replaced=replaced,route=route,endpoint=manual_forecast_voice)


def movio_add_listen_to_forecast():
    """Show 'Listen to Forecast' and play it through the EXISTING MOVIO TTS (/weather-speech)."""
    from pathlib import Path
    from datetime import datetime
    from urllib.parse import urlsplit
    import hashlib, re, shutil, subprocess, tempfile
    import requests

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    web = root / 'web'; index = web / 'index.html'; js_path = web / 'movio_forecast_listen.js'
    if not index.is_file():
        raise RuntimeError('web/index.html not found at ' + str(index))

    JS = r'''/* MOVIO_FORECAST_LISTEN_V3 - plays the forecast with the existing MOVIO TTS (/weather-speech). Tap only; never autoplays; never touches booking. */
(function(){
 'use strict';
 var ROW_ID='wxForecastListenRow';
 var COPY={
  English:{listen:'\uD83D\uDD0A Listen to Forecast',stop:'\u23F9 Stop forecast',cancel:'Cancel voice',loading:'Preparing forecast voice\u2026',error:'Voice could not play. Tap to retry.',replay:'\uD83D\uDD0A Replay forecast'},
  Tamil:{listen:'\uD83D\uDD0A \u0BB5\u0BBE\u0BA9\u0BBF\u0BB2\u0BC8 \u0BAE\u0BC1\u0BA9\u0BCD\u0BA9\u0BB1\u0BBF\u0BB5\u0BBF\u0BAA\u0BCD\u0BAA\u0BC8\u0B95\u0BCD \u0B95\u0BC7\u0BB3\u0BC1\u0B99\u0BCD\u0B95\u0BB3\u0BCD',stop:'\u23F9 \u0B92\u0BB2\u0BBF\u0BAF\u0BC8 \u0BA8\u0BBF\u0BB1\u0BC1\u0BA4\u0BCD\u0BA4\u0BC1',cancel:'\u0BB0\u0BA4\u0BCD\u0BA4\u0BC1\u0B9A\u0BC6\u0BAF\u0BCD',loading:'\u0BB5\u0BBE\u0BA9\u0BBF\u0BB2\u0BC8 \u0B95\u0BC1\u0BB0\u0BB2\u0BC8\u0BA4\u0BCD \u0BA4\u0BAF\u0BBE\u0BB0\u0BBF\u0B95\u0BCD\u0B95\u0BBF\u0BB1\u0BCB\u0BAE\u0BCD\u2026',error:'\u0B95\u0BC1\u0BB0\u0BB2\u0BC8 \u0B87\u0BAF\u0B95\u0BCD\u0B95 \u0BAE\u0BC1\u0B9F\u0BBF\u0BAF\u0BB5\u0BBF\u0BB2\u0BCD\u0BB2\u0BC8. \u0BAE\u0BC0\u0BA3\u0BCD\u0B9F\u0BC1\u0BAE\u0BCD \u0BAE\u0BC1\u0BAF\u0BB1\u0BCD\u0B9A\u0BBF\u0B95\u0BCD\u0B95\u0BB5\u0BC1\u0BAE\u0BCD.',replay:'\uD83D\uDD0A \u0BAE\u0BC0\u0BA3\u0BCD\u0B9F\u0BC1\u0BAE\u0BCD \u0B95\u0BC7\u0BB3\u0BC1\u0B99\u0BCD\u0B95\u0BB3\u0BCD'},
  Tanglish:{listen:'\uD83D\uDD0A Listen to Forecast',stop:'\u23F9 Forecast niruthunga',cancel:'Voice cancel pannunga',loading:'Forecast voice prepare pannrom\u2026',error:'Voice play aagala. Thirumba tap pannunga.',replay:'\uD83D\uDD0A Forecast thirumba kelunga'}
 };
 var state='idle',epoch=0,controller=null,context=null,sources=[],cacheKey='',cacheParts=[],cacheTotal=0,played=false,activeKey='';
 function $(id){return document.getElementById(id);}
 function isTamil(t){return /[\u0B80-\u0BFF]/.test(t);}
 // MOVIO_FORECAST_BIKE_CHOICE_V1: advisory only; never updates a booking slot.
 var BIKE_CHOICE={
  English:'If you prefer, you can choose another vehicle, or continue with your bike.',
  Tamil:'நீங்கள் விரும்பினால் வேறு வாகனத்தைத் தேர்வு செய்யலாம், அல்லது பைக்கிலேயே பயணத்தைத் தொடரலாம்.',
  Tanglish:'Neenga virumbina vera vehicle choose pannalaam, illa unga bike-la continue pannalaam.'
 };
 function rawForecastText(){var a=$('wxManualAdvice');return a?(a.textContent||'').trim():'';}
 function selectedBike(){
  var selected=typeof vehicle==='string'?vehicle:'';
  if(!selected){var active=document.querySelector('#vehicles .vehicle.active');selected=active&&active.dataset.type||'';}
  return String(selected).trim().toLowerCase()==='bike';
 }
 function forecastText(){
  var text=rawForecastText();
  if(!text||!selectedBike())return text;
  // Do not repeat an equivalent choice already included in the real advisory.
  var already=/keep\s+(?:your\s+|the\s+)?bike|continue\s+with\s+(?:your\s+|the\s+)?bike|bike[- ]la\s+continue|பைக்கிலேயே\s+(?:பயணத்தைத்\s+)?தொடர|முடிவு\s+உங்களுடையது|mudivu\s+unga\s+choice/i.test(text);
  if(already)return text;
  return text+' '+(BIKE_CHOICE[language()]||BIKE_CHOICE.English);
 }
 function language(){
  var sel=$('wxManualLanguage'),v=sel&&sel.value;
  if(v==='Tamil'||v==='Tanglish'||v==='English')return v;
  return isTamil(rawForecastText())?'Tamil':'English';
 }
 function words(){return COPY[language()]||COPY.English;}
 function reviewVisible(){var r=$('wxManualReview');return !!r&&!r.hidden&&r.offsetParent!==null;}
 function setLabel(){
  var b=$('wxForecastListenBtn');if(!b)return;
  var label=state==='loading'?words().cancel:state==='playing'?words().stop:played?words().replay:words().listen;
  if(b.textContent!==label)b.textContent=label;
  b.setAttribute('aria-pressed',String(state==='playing'));b.setAttribute('aria-busy',String(state==='loading'));
 }
 function note(t){var n=$('wxForecastListenNote');if(n&&n.textContent!==(t||''))n.textContent=t||'';}
 function stop(clearCache){
  epoch++;if(controller){controller.abort();controller=null;}
  sources.forEach(function(source){try{source.stop();}catch(e){}});sources=[];
  if(context){var old=context;context=null;old.close().catch(function(){});}
  if(clearCache){cacheParts=[];cacheTotal=0;cacheKey='';played=false;}
  activeKey='';state='idle';note('');setLabel();
 }
 async function play(){
  if(!reviewVisible()||!forecastText())return;
  if(state!=='idle'){stop(false);return;}
  var text=forecastText(),lang=language(),key=lang+'\n'+text,ticket=++epoch;
  var started=performance.now(),firstPlayed=false,nextTime=0,total=0,totalBytes=0;
  try{
   var AudioCtx=window.AudioContext||window.webkitAudioContext;
   if(!AudioCtx)throw new Error('Web Audio is unavailable');
   context=new AudioCtx();var ctx=context;
   // Unlock audio on the customer's tap, before waiting for any network response.
   var resume=ctx.resume();
   activeKey=key;state='loading';setLabel();note(words().loading);
   if(cacheKey!==key){cacheKey=key;cacheParts=[];cacheTotal=0;played=false;}
   controller=new AbortController();var ctl=controller;
   await resume;
   for(var part=0;part<(total||1);part++){
    if(ticket!==epoch)return;
    var raw=cacheParts[part];
    if(!raw){
     var timer=setTimeout(function(){ctl.abort();},120000);
     try{
      var res=await fetch('/weather-speech',{method:'POST',credentials:'same-origin',cache:'no-store',signal:ctl.signal,
       headers:{'Content-Type':'application/json'},body:JSON.stringify({text:text,language:lang,part:part})});
      if(!res.ok)throw new Error('voice '+res.status);
      var data=await res.json();
      if(ticket!==epoch)return;
      if(data.protocol!=='parts-v1'||data.part!==part||!Number.isInteger(data.total_parts)||data.total_parts<1||data.total_parts>14||typeof data.audio!=='string')throw new Error('bad audio protocol');
      if(total&&total!==data.total_parts)throw new Error('changed forecast');
      cacheTotal=data.total_parts;
      var bin=atob(data.audio),bytes=new Uint8Array(bin.length);
      for(var i=0;i<bin.length;i++)bytes[i]=bin.charCodeAt(i);
      raw=bytes.buffer;cacheParts[part]=raw;
     }finally{clearTimeout(timer);}
    }
    total=cacheTotal;totalBytes+=raw.byteLength;
    if(totalBytes>9000000)throw new Error('forecast audio too large');
    var buffer=await ctx.decodeAudioData(raw.slice(0));
    if(ticket!==epoch)return;
    var source=ctx.createBufferSource();source.buffer=buffer;source.connect(ctx.destination);sources.push(source);
    nextTime=Math.max(nextTime,ctx.currentTime+0.02);var scheduled=nextTime;
    nextTime+=buffer.duration;
    if(part===total-1){source.onended=function(){if(ticket!==epoch)return;played=true;stop(false);};}
    source.start(scheduled);
    if(!firstPlayed){
     firstPlayed=true;state='playing';note('');setLabel();
     window.MOVIO_FORECAST_VOICE_TIMING={first_audio_ready_ms:Math.round(performance.now()-started),parts:total,complete:false};
    }
    // Prepare the next sentence while the already scheduled sentence is playing.
    // Deliberate pauses are included in each WAV by the existing pause helper.
   }
   if(ticket!==epoch)return;
   controller=null;
   window.MOVIO_FORECAST_VOICE_TIMING.prepared_ms=Math.round(performance.now()-started);
   window.MOVIO_FORECAST_VOICE_TIMING.complete=true;
   console.info('MOVIO forecast voice timing',window.MOVIO_FORECAST_VOICE_TIMING);
  }catch(e){if(ticket!==epoch)return;stop(true);note(words().error);console.warn('MOVIO forecast playback:',e.message);}
 }
 function ensureRow(){
  var review=$('wxManualReview');if(!review)return;
  var row=$(ROW_ID);
  if(!reviewVisible()||!forecastText()){if(row&&!row.hidden)row.hidden=true;if(state!=='idle')stop(false);return;}
  var actions=review.querySelector('.wx-review-actions');if(!actions)return;
  if(!row){
   row=document.createElement('div');row.id=ROW_ID;
   row.innerHTML='<button type="button" id="wxForecastListenBtn" aria-pressed="false"></button><span id="wxForecastListenNote" role="status" aria-live="polite"></span>';
   row.querySelector('button').addEventListener('click',play);
  }
  if(row.nextElementSibling!==actions||row.parentNode!==actions.parentNode)actions.parentNode.insertBefore(row,actions);
  if(row.hidden)row.hidden=false;
  /* new forecast text or language => drop old audio */
  var currentKey=language()+'\n'+forecastText();
  if(activeKey&&activeKey!==currentKey)stop(true);
  if(cacheKey&&cacheKey!==currentKey&&state==='idle'){cacheParts=[];cacheTotal=0;cacheKey='';played=false;}
  setLabel();
 }
 var st=document.createElement('style');
 st.textContent='#'+ROW_ID+'{margin:12px 0;display:flex;align-items:center;gap:8px;flex-wrap:wrap}#'+ROW_ID+'[hidden]{display:none!important}'
  +'#wxForecastListenBtn{min-height:44px;width:100%;padding:10px 14px;border:1px solid var(--line,#d9d9d9);border-radius:12px;background:var(--yellow,#ffcc29);color:var(--ink,#17191d);font:inherit;font-weight:600;cursor:pointer;white-space:normal}'
  +'#wxForecastListenBtn[aria-pressed="true"]{background:var(--paper,#fff)}#wxForecastListenNote{font-size:12px;color:var(--muted,#667085);overflow-wrap:anywhere}';
 document.head.appendChild(st);
 var queued=false;
 function schedule(){if(queued)return;queued=true;setTimeout(function(){queued=false;ensureRow();},60);}
 new MutationObserver(schedule).observe(document.body,{subtree:true,childList:true,attributes:true,attributeFilter:['hidden'],characterData:true});
 document.addEventListener('change',function(e){if(e.target&&e.target.id==='wxManualLanguage'){stop(true);schedule();}},true);
 document.addEventListener('click',function(e){if(e.target.closest&&e.target.closest('#wxManualContinue,#wxManualChange,#logout,#driverMode,#aiCall,#vehicles,#bookRide'))stop(false);},true);
 window.addEventListener('pagehide',function(){stop(true);});
 document.addEventListener('visibilitychange',function(){if(document.hidden)stop(false);});
 schedule();
})();
'''

    node = shutil.which('node')
    if node:
        with tempfile.TemporaryDirectory() as d:
            p = Path(d) / 't.js'; p.write_text(JS, encoding='utf-8')
            r = subprocess.run([node, '--check', str(p)], capture_output=True, text=True, timeout=20)
            if r.returncode: raise RuntimeError('JS syntax error: ' + r.stderr[:400])

    # --- 1. Make sure the existing-TTS audio route exists (reuses the DhVaani handler) ---
    b = globals().get('bridge')
    if not b: raise RuntimeError('Run the existing MOVIO voice setup first (bridge is missing).')
    engine = getattr(getattr(b, '_wx_endpoint', None), '__globals__', {})
    current = b; seen = set(); app = None; actual = None
    while current is not None and id(current) not in seen:
        seen.add(id(current)); cand = getattr(current, 'app', None)
        if callable(getattr(cand, 'add_api_route', None)):
            app = cand; actual = current; break
        current = getattr(current, '__dict__', {}).get('_original')
    if app is None: raise RuntimeError('The running AI application was not found behind the bridge.')
    public = str(globals().get('MOVIO_WEBSITE_URL') or getattr(actual, 'PUBLIC_URL', '')).rstrip('/')
    if not urlsplit(public).scheme or not urlsplit(public).netloc:
        raise RuntimeError('The public website URL is missing.')
    audio = register_forecast_audio(app, b, engine, public)
    globals()['MOVIO_FORECAST_VOICE_CONTROL']=audio['endpoint']
    audio_ok = False

    s = requests.Session(); s.trust_env = False
    try:
        for port in (8000, 8090):
            code = s.get('http://127.0.0.1:%d/weather-speech' % port, timeout=(2, 8), allow_redirects=False).status_code
            if code != 405:
                raise RuntimeError('Port %d does not expose POST /weather-speech (GET returned %s). Share this output.' % (port, code))
    except Exception:
        if audio['added']:
            app.routes[:] = [r for r in app.routes if r is not audio['route']] + audio['replaced']
        raise

    # --- 2. Install the button script ---
    backup = Path(tempfile.mkdtemp(
        prefix='listen_forecast_backup_',
        dir=str(root)
    ))
    shutil.copy2(index, backup / 'index.html')
    if js_path.is_file():shutil.copy2(js_path,backup/'movio_forecast_listen.js')
    js_path.write_text(JS, encoding='utf-8')
    token = hashlib.sha256(JS.encode()).hexdigest()[:12]
    html = index.read_text(encoding='utf-8')
    html = re.sub(r'\s*<script[^>]*movio_forecast_listen\.js[^>]*></script>', '', html)
    tag = '\n<script src="/movio_forecast_listen.js?v=%s" defer></script>' % token
    html = html.replace('</body>', tag + '\n</body>', 1) if '</body>' in html else html + tag
    index.write_text(html, encoding='utf-8')

    served = s.get('http://127.0.0.1:8080/movio_forecast_listen.js?v=' + token, timeout=8)
    page = s.get('http://127.0.0.1:8080/', timeout=8)
    gw = s.get('http://127.0.0.1:8080/weather-speech', timeout=8, allow_redirects=False).status_code
    ok = served.status_code == 200 and 'MOVIO_FORECAST_LISTEN_V3' in served.text and 'movio_forecast_listen.js' in page.text
    print({'script_served': ok, 'speech_route': 'updated' if audio['added'] else 'already current', 'website_gateway_GET_status': gw, 'backup': backup.name})
    s.close()
    if not ok:
        shutil.copy2(backup / 'index.html', index)
        raise RuntimeError('Website did not serve the new script. index.html restored.')
    print('Installed progressive forecast playback. Refresh the dashboard with Ctrl+Shift+R, then tap Listen to Forecast. Audio starts after the first sentence; later sentences prepare while it plays. Existing pronunciation, sampling quality and pauses are retained.')
    print('No audio is generated before you tap. Repeated phrases are cached in RAM for 15 minutes (32 MiB limit). Browser replays reuse the same audio.')
    print('To inspect timing, open browser Console and enter: window.MOVIO_FORECAST_VOICE_TIMING')
    print('To clear server voice cache after changing a model/voice: MOVIO_FORECAST_VOICE_CONTROL._movio_forecast_clear_cache()')

movio_add_listen_to_forecast()

{'script_served': True, 'speech_route': 'updated', 'website_gateway_GET_status': 404, 'backup': 'listen_forecast_backup_mvfwddm4'}
Installed progressive forecast playback. Refresh the dashboard with Ctrl+Shift+R, then tap Listen to Forecast. Audio starts after the first sentence; later sentences prepare while it plays. Existing pronunciation, sampling quality and pauses are retained.
No audio is generated before you tap. Repeated phrases are cached in RAM for 15 minutes (32 MiB limit). Browser replays reuse the same audio.
To inspect timing, open browser Console and enter: window.MOVIO_FORECAST_VOICE_TIMING
To clear server voice cache after changing a model/voice: MOVIO_FORECAST_VOICE_CONTROL._movio_forecast_clear_cache()


In [148]:
# STEP 1 — Inspect AI Call latency handlers.
# No bookings, speech generation, installations or source changes.

import inspect
import json
import re

def movio_inspect_call_latency():
    b = globals().get("bridge")
    if b is None:
        raise RuntimeError("Run your existing MOVIO voice setup first.")

    report = {"settings": {}, "handlers": {}, "playback_code": []}

    def source(fn, limit=22000):
        if not callable(fn):
            return {"available": False}
        try:
            text = inspect.getsource(fn)
            # Hide credentials if a source contains a hardcoded assignment.
            text = re.sub(
                r"""(?i)((?:api_key|access_token|password|secret)\s*=\s*)(['"])[^'"\n]+\2""",
                r"\1'<REDACTED>'",
                text,
            )
            return {
                "available": True,
                "function": getattr(fn, "__qualname__", str(type(fn))),
                "file": inspect.getsourcefile(fn),
                "truncated": len(text) > limit,
                "source": text[:limit],
            }
        except (OSError, TypeError) as exc:
            return {"available": False, "reason": type(exc).__name__}

    # Find the actual FastAPI application behind any existing bridge proxy.
    current, seen, app = b, set(), None
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        candidate = getattr(current, "app", None)
        if callable(getattr(candidate, "add_api_route", None)):
            app = candidate
            break
        current = getattr(current, "__dict__", {}).get("_original")

    endpoint = getattr(b, "_wx_endpoint", None)
    engine = getattr(endpoint, "__globals__", {})

    if app is not None:
        routes = [
            r for r in app.routes
            if getattr(r, "path", "") == "/browser-call"
        ]
        report["browser_call_route_count"] = len(routes)
        for i, route in enumerate(routes):
            report["handlers"][f"browser_call_{i}"] = source(route.endpoint)
        if not engine and routes:
            engine = getattr(routes[0].endpoint, "__globals__", {})

    for name in ("tts_reply", "speech_segments", "display_reply"):
        fn = engine.get(name)
        report["handlers"][name] = source(fn)

        # Inspect the existing TTS wrapper's immediate dependencies.
        if name == "tts_reply" and callable(fn):
            try:
                closed = inspect.getclosurevars(fn).nonlocals
                for dependency in ("previous_tts", "profile_for"):
                    if callable(closed.get(dependency)):
                        report["handlers"][dependency] = source(
                            closed[dependency], limit=14000
                        )
            except (TypeError, ValueError):
                pass

    report["handlers"]["bridge_speak"] = source(getattr(b, "speak", None))

    adapter = getattr(b, "TTS", None)
    report["handlers"]["adapter_synthesize"] = source(
        getattr(adapter, "synthesize", None)
    )

    ns = getattr(adapter, "ns", {})
    if not isinstance(ns, dict):
        ns = {}

    for name in (
        "FIRST_CHUNK_NUM_STEP", "REST_CHUNK_NUM_STEP",
        "ENGLISH_SPEED", "TAMIL_SPEED", "TANGLISH_SPEED",
        "FIRST_AUDIO_DEADLINE_MS",
    ):
        value = ns.get(name, globals().get(name))
        if value is None or isinstance(value, (str, int, float, bool)):
            report["settings"][name] = value

    quality = getattr(b, "WX_SPEECH_QUALITY", None)
    if isinstance(quality, dict):
        report["settings"]["speech_quality"] = {
            k: v for k, v in quality.items()
            if isinstance(v, (str, int, float, bool))
        }

    scheduler = ns.get("SCHEDULER", globals().get("SCHEDULER"))
    if scheduler is not None:
        report["scheduler"] = {}
        for name in ("running", "batch_window_ms", "max_batch_size"):
            value = getattr(scheduler, name, None)
            if value is None or isinstance(value, (str, int, float, bool)):
                report["scheduler"][name] = value
        worker = getattr(scheduler, "worker", None)
        report["scheduler"]["worker_alive"] = (
            worker.is_alive() if callable(getattr(worker, "is_alive", None))
            else None
        )
        queue = getattr(scheduler, "heap", None)
        report["scheduler"]["queued_items"] = (
            len(queue) if queue is not None else None
        )

    # Inspect playback logic from the active call page.
    html = getattr(b, "HTML", "") or ""
    scripts = re.findall(
        r"<script\b[^>]*>(.*?)</script>", html,
        flags=re.I | re.S,
    )
    for script in scripts:
        if re.search(
            r"onmessage|decodeAudioData|new Audio|MediaRecorder|audioQueue",
            script,
        ):
            report["playback_code"].append({
                "truncated": len(script) > 22000,
                "source": script[:22000],
            })
            if len(report["playback_code"]) >= 2:
                break

    report["status"] = "Inspection only; existing voice and booking unchanged."
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report

MOVIO_CALL_LATENCY_INSPECTION = movio_inspect_call_latency()

{
  "settings": {
    "FIRST_CHUNK_NUM_STEP": 3,
    "REST_CHUNK_NUM_STEP": 4,
    "ENGLISH_SPEED": 1.12,
    "TAMIL_SPEED": 1.12,
    "TANGLISH_SPEED": 1.12,
    "FIRST_AUDIO_DEADLINE_MS": 500.0,
    "speech_quality": {
      "weather_steps": 8,
      "closing_steps": 8,
      "min_words": 8
    }
  },
  "handlers": {
    "browser_call_0": {
      "available": true,
      "function": "install_socket.<locals>.wrapped",
      "file": "/kaggle/working/MOVIO_MAIN_WEBSITE/integrations/movio_friendly_weather.py",
      "truncated": false,
      "source": "    def __init__(self,ws,bridge,engine):self.ws,self.bridge,self.engine,self.sid=ws,bridge,engine,None;self.pending=None\n"
    },
    "tts_reply": {
      "available": true,
      "function": "movio_weather_closing_audio_fix.<locals>.tts_reply",
      "file": "/tmp/ipykernel_49/1718067646.py",
      "truncated": false,
      "source": "    def tts_reply(bridge_object, text):\n        kind = profile_for(text)\n\n        # Final confirmatio

In [149]:
# Run this ONE cell after the existing MOVIO voice setup, before evaluation.
# Close an active AI Call before running, then open a NEW AI Call afterwards.
# Reduces browser end-of-speech silence from 1900 ms to 1100 ms.
# Does not change STT/TTS models, speech pronunciation, pauses or booking logic.

MOVIO_CALL_SILENCE_MS = 1100  # Use 1400 if your normal mid-sentence pauses need more time.


def movio_call_latency_patch_html(html, silence_ms):
    import hashlib
    import re
    if type(silence_ms) is not int or not 900 <= silence_ms <= 1900:
        raise ValueError('MOVIO_CALL_SILENCE_MS must be an integer from 900 to 1900.')
    if not isinstance(html, str):
        raise TypeError('Call page HTML is missing.')
    scripts = list(re.finditer(r'(<script\b[^>]*>)(.*?)(</script\s*>)', html, re.I | re.S))
    candidates = [m for m in scripts if 'function listen(id)' in m.group(2) and 'lastVoice' in m.group(2)]
    if len(candidates) != 1:
        raise RuntimeError('The call microphone handler differs from the inspected page. No changes made.')
    match = candidates[0]
    js = match.group(2)
    pattern = r'(now-first\s*>\s*450\s*&&\s*now-lastVoice\s*>\s*)(\d+)(\s*\|\|\s*now-started\s*>\s*27000)'
    thresholds = list(re.finditer(pattern, js))
    if len(thresholds) != 1:
        raise RuntimeError('The inspected end-of-speech condition was not found exactly once.')
    t = thresholds[0]
    # Confirm every other byte of the call script matches your supplied inspection.
    original_js = js[:t.start(2)] + '1900' + js[t.end(2):]
    expected = 'c72ebfa3e7d3d4336b187f7b2d008fcada2552e944dd78975552ed4775159bc8'
    if hashlib.sha256(original_js.encode('utf-8')).hexdigest() != expected:
        raise RuntimeError('The AI Call script changed after inspection. Share a fresh diagnostic output; no changes made.')
    updated_js = js[:t.start(2)] + str(silence_ms) + js[t.end(2):]
    updated_html = html[:match.start(2)] + updated_js + html[match.end(2):]
    return updated_html, updated_js, int(t.group(2))


def movio_reduce_call_input_wait():
    from pathlib import Path
    import json
    import re
    import shutil
    import subprocess
    import tempfile
    import requests

    silence_ms = globals().get('MOVIO_CALL_SILENCE_MS', 1100)
    b = globals().get('bridge')
    if b is None:
        raise RuntimeError('Run the existing MOVIO voice setup first.')
    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    if not root.is_dir():
        raise FileNotFoundError(str(root))
    active = getattr(b, 'HTML', None)
    updated, js, old_ms = movio_call_latency_patch_html(active, silence_ms)

    # Validate the complete existing call script before touching runtime or files.
    node = shutil.which('node')
    if not node:
        raise RuntimeError('Node.js is required for the call-page syntax check. No changes made.')
    with tempfile.TemporaryDirectory() as td:
        probe = Path(td) / 'call.js'
        probe.write_text(js, encoding='utf-8')
        checked = subprocess.run([node, '--check', str(probe)], capture_output=True, text=True, timeout=20)
        if checked.returncode:
            raise RuntimeError('JavaScript syntax check failed: ' + checked.stderr[:600])

    # Locate the existing bridge and the GET /call-page module's HTML binding.
    objects, seen, current, app = [], set(), b, None
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        if isinstance(getattr(current, 'HTML', None), str):
            objects.append(current)
        candidate = getattr(current, 'app', None)
        if callable(getattr(candidate, 'add_api_route', None)):
            app = candidate
        current = getattr(current, '__dict__', {}).get('_original')
    if app is None:
        raise RuntimeError('The existing AI application could not be identified. No changes made.')
    original_routes = list(app.routes)
    runtime = []
    for obj in objects:
        before = getattr(obj, 'HTML')
        after, _, _ = movio_call_latency_patch_html(before, silence_ms)
        runtime.append(('attribute', obj, before, after))
    seen_globals = set()
    for route in original_routes:
        if getattr(route, 'path', '') != '/call-page':
            continue
        namespace = getattr(getattr(route, 'endpoint', None), '__globals__', {})
        if id(namespace) in seen_globals or not isinstance(namespace.get('HTML'), str):
            continue
        seen_globals.add(id(namespace))
        before = namespace['HTML']
        after, _, _ = movio_call_latency_patch_html(before, silence_ms)
        runtime.append(('namespace', namespace, before, after))

    # Keep on-disk call templates consistent where they contain this microphone handler.
    files = []
    skipped = []
    for relative in ('integrations/ai_call_page.html', 'integrations/weather_call_page.html'):
        path = root / relative
        if not path.is_file():
            continue
        before_bytes = path.read_bytes()
        before = before_bytes.decode('utf-8')
        if 'function listen(id)' not in before:
            skipped.append(relative)
            continue
        try:
            after, _, _ = movio_call_latency_patch_html(before, silence_ms)
        except RuntimeError:
            # Some notebooks retain an older template alongside the current active page.
            # Preserve it rather than replacing unrelated code; the served page is verified below.
            skipped.append(relative + ' (different retained template; preserved)')
            continue
        if after != before:
            files.append((path, before_bytes, after))

    session = requests.Session()
    session.trust_env = False
    backup = None
    applied_files, applied_runtime = [], []
    try:
        served_before = session.get('http://127.0.0.1:8000/call-page', timeout=(2, 8), allow_redirects=False)
        if served_before.status_code != 200:
            raise RuntimeError('Local AI call page returned HTTP %s. No changes made.' % served_before.status_code)
        _, expected_served_js, served_old_ms = movio_call_latency_patch_html(served_before.text, silence_ms)
        if expected_served_js != js:
            raise RuntimeError('The served and active call scripts differ. No changes made.')
        changed_runtime = any(before != after for _, _, before, after in runtime)
        if files or changed_runtime:
            backup = Path(tempfile.mkdtemp(prefix='call_input_latency_backup_', dir=str(root)))
            (backup / 'active_call_page.html').write_text(active, encoding='utf-8')
            for path, before, _ in files:
                destination = backup / path.relative_to(root)
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(before)
            for path, before, after in files:
                applied_files.append((path, before))
                path.write_text(after, encoding='utf-8')
            for kind, target, before, after in runtime:
                if before == after:
                    continue
                applied_runtime.append((kind, target, before))
                if kind == 'attribute':
                    setattr(target, 'HTML', after)
                else:
                    target['HTML'] = after

        served_after = session.get('http://127.0.0.1:8000/call-page', timeout=(2, 8), allow_redirects=False)
        if served_after.status_code != 200:
            raise RuntimeError('The updated call page could not be verified.')
        _, actual_js, actual_ms = movio_call_latency_patch_html(served_after.text, silence_ms)
        # actual_js normalizes to the requested threshold; check the actual original value too.
        if actual_ms != silence_ms or actual_js != js:
            raise RuntimeError('The server is still serving a different microphone threshold.')
        if len(app.routes) != len(original_routes) or any(x is not y for x, y in zip(app.routes, original_routes)):
            raise RuntimeError('Unexpected route changes detected.')
        result = dict(
            step='AI Call latency — input silence', passed=True,
            previous_silence_ms=served_old_ms, end_of_speech_silence_ms=actual_ms,
            configured_silence_reduction_ms=1900-actual_ms,
            javascript_syntax='PASS', served_call_page='PASS',
            changed_files=[str(p.relative_to(root)) for p, _, _ in files],
            skipped_older_templates=skipped,
            backup=str(backup) if backup else None,
            booking_handlers_preserved=True, speech_quality_preserved=True,
            scope='Browser input timing only; actual STT/LLM/weather/TTS latency is not measured here.'
        )
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print('Close the old AI Call popup and start a NEW AI Call from the dashboard.')
        print('Test one English, Tamil and Tanglish reply. Share this output and whether the wait improves.')
        return result
    except Exception:
        for kind, target, before in reversed(applied_runtime):
            if kind == 'attribute':
                setattr(target, 'HTML', before)
            else:
                target['HTML'] = before
        for path, before in reversed(applied_files):
            path.write_bytes(before)
        raise
    finally:
        session.close()


MOVIO_AI_CALL_INPUT_LATENCY = movio_reduce_call_input_wait()


{
  "step": "AI Call latency — input silence",
  "passed": true,
  "previous_silence_ms": 1900,
  "end_of_speech_silence_ms": 1100,
  "configured_silence_reduction_ms": 800,
  "javascript_syntax": "PASS",
  "served_call_page": "PASS",
  "changed_files": [
    "integrations/ai_call_page.html",
    "integrations/weather_call_page.html"
  ],
  "skipped_older_templates": [],
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/call_input_latency_backup_kqnnuu0e",
  "booking_handlers_preserved": true,
  "speech_quality_preserved": true,
  "scope": "Browser input timing only; actual STT/LLM/weather/TTS latency is not measured here."
}
Close the old AI Call popup and start a NEW AI Call from the dashboard.
Test one English, Tamil and Tanglish reply. Share this output and whether the wait improves.


In [150]:
# CORRECTED PYTHON-COMPATIBILITY CHECK — replace the previous upgrade cell.
# ONE KAGGLE CELL — run AFTER the LAST website-launch/restart cell, before evaluation.
# Upgrades the inspected legacy care API v0.1 to the verified weather API v1.0.
# Restores its website startup hook. Restores the known compatible care engine only if needed.
# Restarts only the verified website process on port 8080. Existing UI/voice/booking handlers remain in control.
# Close any in-progress manual booking before running this website restart.


def movio_companion_startup_source(source):
    import ast
    import re
    pairs = (
        ('# MOVIO_CARE_API_BOOTSTRAP_V1', '# MOVIO_CARE_API_BOOTSTRAP_END_V1'),
        ('# MOVIO_CUSTOMER_CARE_BOOTSTRAP_V1', '# MOVIO_CUSTOMER_CARE_BOOTSTRAP_END_V1'),
    )
    base = source
    for start, end in pairs:
        starts = list(re.finditer(r'^' + re.escape(start) + r'[ \t]*$', base, re.M))
        ends = list(re.finditer(r'^' + re.escape(end) + r'[ \t]*$', base, re.M))
        if len(starts) != len(ends) or len(starts) > 1:
            raise RuntimeError('Incomplete or duplicate care startup markers. No files changed; share the server startup section.')
        if starts:
            if starts[0].end() >= ends[0].start():
                raise RuntimeError('Care startup markers are out of order. No files changed.')
            block = base[starts[0].end():ends[0].start()]
            if 'movio_customer_care_api.py' not in block or 'install_backend' not in block:
                raise RuntimeError('The existing marked block has unexpected contents. No files changed.')
            ast.parse(block)
            after = ends[0].end()
            if base[after:].startswith('\n\n'):after += 2
            elif base[after:].startswith('\n'):after += 1
            base = base[:starts[0].start()] + base[after:]
    tree = ast.parse(base)
    launchers = [n for n in tree.body if isinstance(n, ast.If)
        and any(isinstance(x, ast.Name) and x.id == '__name__' for x in ast.walk(n.test))
        and any(isinstance(x, ast.Constant) and x.value == '__main__' for x in ast.walk(n.test))
        and any(isinstance(x, ast.Call) and isinstance(x.func, ast.Attribute)
                and x.func.attr == 'serve_forever' for x in ast.walk(n))]
    if len(launchers) != 1:
        raise RuntimeError('The unique website serve_forever launch block was not found. No files changed.')
    bootstrap = """# MOVIO_CARE_API_BOOTSTRAP_V1
if __name__ == '__main__':
    import importlib.util as _cc_import
    import sys as _cc_sys
    _cc_path = ROOT / 'integrations/movio_customer_care_api.py'
    _cc_spec = _cc_import.spec_from_file_location('movio_customer_care_api', _cc_path)
    _cc_module = _cc_import.module_from_spec(_cc_spec)
    _cc_sys.modules[_cc_spec.name] = _cc_module
    exec(compile(_cc_path.read_text(encoding='utf-8-sig'), str(_cc_path), 'exec'), _cc_module.__dict__)
    _cc_module.install_backend(_cc_sys.modules[__name__])
    if getattr(Handler, '_customer_care_api_version', None) != 'customer-care-api-1.0':
        raise RuntimeError('Customer-care API v1.0 was not installed; website startup stopped.')
# MOVIO_CARE_API_BOOTSTRAP_END_V1

"""
    at = sum(len(line) for line in base.splitlines(keepends=True)[:launchers[0].lineno - 1])
    updated = base[:at] + bootstrap + base[at:]
    compile(updated, 'server.py', 'exec', dont_inherit=True)
    isolated = updated[:at] + updated[at + len(bootstrap):]
    if ast.dump(ast.parse(isolated), include_attributes=False) != ast.dump(tree, include_attributes=False):
        raise RuntimeError('Website source isolation check failed. No files changed.')
    return updated


def movio_upgrade_companion_backend():
    import ast
    import hashlib
    import json
    import os
    from pathlib import Path
    import shutil
    import subprocess
    import tempfile
    import time
    import types
    import psutil
    import requests

    root = Path(globals().get('WX_ROOT') or '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    server = root / 'server.py'
    api = root / 'integrations/movio_customer_care_api.py'
    engine = root / 'integrations/movio_customer_care.py'
    for path in (server, api, engine):
        if path.is_symlink() or not path.is_file():
            raise RuntimeError('Required project file missing or symlink: ' + str(path))
    # Reviewed sources from the previously tested compatible care setup.
    api_candidate = r'''"""Authenticated dashboard care using the existing geocoder/weather client.

Browser coordinates are user-supplied location hints, not independently verified
GPS evidence. Weather/locality values are retrieved on the server. This module
does not write bookings, tracking grants, coordinates, or recommendation history.
"""
import copy
import importlib.util
import json
import logging
import math
import sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

VERSION='customer-care-api-1.0'
ENGINE_VERSION='customer-care-engine-1.1-compatible'
LOG=logging.getLogger('movio.customer_care')


def _load_care(root):
    path=(root/'integrations/movio_customer_care.py').resolve()
    current=sys.modules.get('movio_customer_care')
    if (getattr(current,'VERSION',None)==ENGINE_VERSION
            and Path(getattr(current,'__file__','')).resolve()==path):return current
    spec=importlib.util.spec_from_file_location('movio_customer_care',path)
    care=importlib.util.module_from_spec(spec)
    exec(compile(path.read_text(encoding='utf-8'),str(path),'exec'),care.__dict__)
    if care.VERSION!=ENGINE_VERSION:raise RuntimeError('Run the compatible customer-care engine setup first.')
    sys.modules['movio_customer_care']=care
    return care


def _number(value,low,high):
    if isinstance(value,bool) or value is None or value=='':return None
    try:number=float(value)
    except (TypeError,ValueError,OverflowError):return None
    return number if math.isfinite(number) and low<=number<=high else None


def _instant(value):
    try:
        if isinstance(value,(int,float)) and not isinstance(value,bool):
            # Browser GeolocationPosition.timestamp uses milliseconds.
            if value>100000000000:value=value/1000
            return datetime.fromtimestamp(value,timezone.utc)
        if isinstance(value,str):
            parsed=datetime.fromisoformat(value.strip().replace('Z','+00:00'))
            return parsed.astimezone(timezone.utc) if parsed.tzinfo else None
    except (ValueError,TypeError,OverflowError,OSError):pass
    return None


def _delivered(value):
    allowed={'delivered','acknowledged','dismissed'}
    if not isinstance(value,dict):return {}
    return {key:status for key,status in list(value.items())[-128:]
            if isinstance(key,str) and len(key)<=180 and isinstance(status,str) and status in allowed}


def _weather_view(care,result):
    weather=result['weather']
    if weather is None:return dict(status='unavailable')
    codes={point['weather_code'] for point in weather['points']}
    condition=('storm' if weather['storm'] else 'heavy_rain' if weather['heavy_rain'] else
               'rain' if weather['rain'] else 'cloudy' if codes&{2,3} else
               'clear' if codes and codes<={0,1} else 'fog' if codes&{45,48} else 'other')
    return dict(status='ok',source=weather['source'],condition=condition,
                temperature_c=weather['max_temperature_c'],
                rain_probability_pct=weather['max_rain_probability_pct'],
                valid_at=weather['points'][0]['valid_at'],basis='current_hour_forecast')


def dashboard_response(care,context,user,reverse,forecast,now):
    """Pure orchestration except for the TWO injected, existing provider calls."""
    lang=care.LANGUAGES.get(str(context.get('language',context.get('lang','English'))).casefold(),'English')
    safe=dict(lang=lang,care_stage='dashboard',now=now.isoformat(),
              current_location={'status':'unavailable'},current_weather=None,
              care_scope_id='dashboard:'+str(user['id'])+':'+now.date().isoformat())
    location=context.get('current_location')
    if not isinstance(location,dict):location={}
    status=location.get('status')
    status=status if status in {'granted','denied','unavailable','unsupported','timeout'} else 'unavailable'
    safe['current_location']['status']=status
    if status=='granted':
        lat=_number(location.get('lat',location.get('latitude')),-90,90)
        lng=_number(location.get('lng',location.get('longitude')),-180,180)
        observed=_instant(location.get('observed_at',location.get('timestamp')))
        age=(now.astimezone(timezone.utc)-observed).total_seconds() if observed else None
        if lat is None or lng is None or age is None or not -60<=age<=900:
            safe['current_location']['status']='unavailable'
        else:
            # Only the approximate area is needed for companion advice. Never
            # change booking/map pins, and do not introduce a precise-GPS cache.
            area_lat,area_lng=round(lat,3),round(lng,3)
            safe['care_scope_id']+=':'+str(area_lat)+','+str(area_lng)
            safe['current_location']=dict(status='granted',observed_at=observed.isoformat(),
                locality='',locality_status='unavailable')
            try:
                place=reverse(area_lat,area_lng)
                if isinstance(place,dict) and place.get('locality_status')=='ok':
                    safe['current_location'].update(locality=place.get('locality') or '',locality_status='ok')
            except Exception as exc:
                LOG.warning('companion_locality_failed kind=%s',type(exc).__name__)
            try:
                # The existing optional-weather adapter requires exactly TWO
                # points. This dashboard has one current area, so query that
                # same area in both protocol slots and retain only its first
                # point. These are not the booking's pickup/drop-off pins.
                point=dict(lat=area_lat,lng=area_lng)
                value=forecast([dict(point),dict(point)],now,now=now)
                if isinstance(value,dict):
                    value=copy.deepcopy(value)
                    if isinstance(value.get('points'),list):value['points']=value['points'][:1]
                    value['scope']='dashboard_current_area'
                    safe['current_weather']=value
                if not isinstance(value,dict) or value.get('status')!='ok':LOG.warning('companion_weather_unavailable')
            except Exception as exc:
                LOG.warning('companion_weather_failed kind=%s',type(exc).__name__)
    history=_delivered(context.get('delivered'))
    safe['delivered']=history
    acknowledged=context.get('acknowledged')
    safe['acknowledged']=[v for v in acknowledged[:32] if isinstance(v,str) and len(v)<100] if isinstance(acknowledged,list) else []
    # Ignore all client weather, locality, date, fare, driver and ride-status claims.
    # Both entry points use the same shared engine and delivery history.
    result=care.recommend(safe,delivered=history)
    recommendations=care.evaluate(safe)
    view=_weather_view(care,result)
    raw=safe.get('current_weather') or {}
    if view['status']=='ok':view['fetched_at']=raw.get('fetched_at')
    return dict(ok=True,version=VERSION,engine_version=care.VERSION,language=lang,
                greeting=result['greeting'],locality=result['locality'],
                location_status=safe['current_location']['status'],weather=view,
                recommendations=recommendations,optional=True)


def install_backend(server,providers=None,clock=None,care=None):
    existing=getattr(server.Handler,'_customer_care_api_version',None)
    if existing==VERSION:return dict(version=VERSION,already_installed=True)
    if existing is not None:
        raise RuntimeError('Restart the website service to replace the older care endpoint without overlapping handlers.')
    root=Path(server.ROOT).resolve()
    care=_load_care(root) if care is None else care
    if providers is None:
        tools=getattr(server,'_ct_module',None)
        weather=getattr(server,'_wx_weather_client',None)
        if tools is None or not callable(getattr(tools,'reverse_address',None)):
            raise RuntimeError('The original customer-tools bootstrap must run before customer care.')
        if weather is None or not callable(getattr(weather,'forecast',None)):
            raise RuntimeError('The original weather bootstrap must run before customer care.')
        reverse,forecast=tools.reverse_address,weather.forecast
    else:reverse,forecast=providers
    now_fn=clock or (lambda:datetime.now(care.TZ))
    previous_post,previous_get=server.Handler.do_POST,server.Handler.do_GET
    def do_get(handler):
        if urlparse(handler.path).path!='/api/customer-care/health':return previous_get(handler)
        return handler.json(200,dict(ok=True,version=VERSION,engine_version=care.VERSION,
                                    verified_weather_connected=True))
    def do_post(handler):
        if urlparse(handler.path).path!='/api/customer-care/recommend':return previous_post(handler)
        try:
            with server.conn() as db:
                user=handler.user(db)
                if user['role']!='customer':return handler.json(403,dict(error='Customer account required'))
                user=dict(user)
            try:
                payload=handler.payload(limit=32768)
                if isinstance(payload,(bytes,bytearray)):payload=bytes(payload).decode('utf-8')
                if isinstance(payload,str):
                    def invalid_constant(value):raise ValueError('Invalid JSON number')
                    payload=json.loads(payload,parse_constant=invalid_constant)
                if not isinstance(payload,dict):raise ValueError('Invalid request')
                context=payload.get('context',{})
                if not isinstance(context,dict):raise ValueError('Invalid context')
            except (ValueError,TypeError,UnicodeError):return handler.json(400,dict(error='Invalid customer-care request'))
            answer=dashboard_response(care,context,user,reverse,forecast,now_fn())
            return handler.json(200,answer)
        except PermissionError:return handler.json(401,dict(error='Please sign in'))
        except Exception as exc:
            LOG.warning('companion_request_failed kind=%s',type(exc).__name__)
            return handler.json(503,dict(error='Travel companion temporarily unavailable',optional=True))
    server.Handler.do_POST=do_post;server.Handler.do_GET=do_get
    server.Handler._customer_care_api_version=VERSION
    return dict(version=VERSION,installed=True,engine_version=care.VERSION)
'''
    engine_candidate = r'''"""Shared MOVIO customer-care rules. Pure functions: no network, GPS, DB or TTS.

Call recommend() with a verified context; call remember_delivered() only after
actual display/playback/dismissal. Generation alone never acknowledges advice.
"""
import copy
import hashlib
import json
import math
import re
from datetime import datetime, timedelta, timezone
try:
    from zoneinfo import ZoneInfo
    TZ=ZoneInfo('Asia/Kolkata')
except (ImportError,KeyError):
    TZ=timezone(timedelta(hours=5,minutes=30))

VERSION='customer-care-engine-1.0'
RULE_IDS=('rain_preparation','heat_protection','late_night_safety','pickup_guidance',
    'otp_safety','passenger_comfort','destination_reminder','departure_readiness',
    'traffic_awareness','accessibility_assistance','phone_readiness','arrival_care')
DEFAULT_POLICY=dict(rain_probability_pct=60,warm_c=32,hot_c=38,
    weather_max_age_s=1800,weather_hour_tolerance_s=3600,
    location_max_age_s=900,eta_max_age_s=180,traffic_max_age_s=300,
    departure_window_minutes=20,night_start_hour=22,night_end_hour=5,
    trusted_traffic_sources=[],
    capacities={'Bike':1,'Auto':3,'Mini':4,'Sedan':4,'SUV':6})
SCORE_WEIGHTS=dict(urgency=4,safety=3,relevance=2,evidence=1)
RAIN_CODES={51,53,55,56,57,61,63,65,66,67,80,81,82}
STORM_CODES={95,96,99}
STATUS_ACTIVE={'scheduled','requested','accepted','arriving','in_progress'}
LANGUAGES={'english':'English','en':'English','en-in':'English','tamil':'Tamil',
    'ta':'Tamil','ta-in':'Tamil','tanglish':'Tanglish'}


def _text(value,limit=160):
    return re.sub(r'\s+',' ',str(value or '')).strip()[:limit]


def _number(value,low=None,high=None):
    if isinstance(value,bool) or value is None or value=='':return None
    try:result=float(value)
    except (ValueError,TypeError,OverflowError):return None
    return result if math.isfinite(result) and (low is None or result>=low) and (high is None or result<=high) else None


def _time(value):
    try:
        if isinstance(value,datetime):result=value
        elif isinstance(value,(int,float)) and not isinstance(value,bool):result=datetime.fromtimestamp(value,TZ)
        elif isinstance(value,str) and value.strip():result=datetime.fromisoformat(value.strip().replace('Z','+00:00'))
        else:return None
        return (result.replace(tzinfo=TZ) if result.tzinfo is None else result.astimezone(TZ))
    except (ValueError,TypeError,OverflowError,OSError):return None


def _fresh(value,now,seconds):
    instant=_time(value)
    return instant is not None and -60<=(now-instant).total_seconds()<=seconds


def _pick(lang,en,ta,tanglish):return {'English':en,'Tamil':ta,'Tanglish':tanglish}[lang]


def _fmt(value):return format(float(value),'.1f').rstrip('0').rstrip('.')


def _policy(value):
    result=copy.deepcopy(DEFAULT_POLICY)
    if value is not None and not isinstance(value,dict):raise ValueError('policy must be a dictionary')
    for key,value in (value or {}).items():
        if key not in result:raise ValueError('Unknown policy key: '+str(key))
        if key=='capacities':
            if not isinstance(value,dict):raise ValueError('capacities must be a dictionary')
            for vehicle,n in value.items():
                v=_number(n,1,16)
                if vehicle not in result[key] or v is None or not v.is_integer():raise ValueError('Invalid vehicle capacity')
                result[key][vehicle]=int(v)
        elif key=='trusted_traffic_sources':
            if not isinstance(value,(list,tuple)) or not all(isinstance(x,str) and x.strip() for x in value):raise ValueError('Traffic sources must be configured explicitly')
            result[key]=[_text(x,80).casefold() for x in value]
        else:
            number=_number(value,0)
            if number is None:raise ValueError('Invalid policy value: '+key)
            result[key]=number
    if not 0<=result['rain_probability_pct']<=100:raise ValueError('Invalid rain threshold')
    if not 0<=result['night_start_hour']<=23 or not 0<=result['night_end_hour']<=23:raise ValueError('Invalid night hours')
    if result['warm_c']>result['hot_c']:raise ValueError('warm_c must not exceed hot_c')
    return result


def _weather(raw,target,now,policy):
    """Use real, fresh Open-Meteo points for the target hour; never guess values."""
    if not isinstance(raw,dict) or raw.get('status')!='ok' or target is None:return None
    if _text(raw.get('source')).casefold().replace(' ','-')!='open-meteo':return None
    if not _fresh(raw.get('fetched_at'),now,policy['weather_max_age_s']):return None
    rows=raw.get('points');points=[]
    if not isinstance(rows,list):return None
    for row in rows:
        if not isinstance(row,dict):continue
        when=_time(row.get('valid_at'))
        if when is None or abs((when-target).total_seconds())>policy['weather_hour_tolerance_s']:continue
        code=_number(row.get('weather_code'),0,99)
        code=int(code) if code is not None and code.is_integer() else None
        temp=_number(row.get('temperature_2m'),-100,70)
        chance=_number(row.get('precipitation_probability'),0,100)
        wind=_number(row.get('wind_speed_10m'),0,400)
        if code is None and temp is None and chance is None:continue
        points.append(dict(role=_text(row.get('location'),20),weather_code=code,
            temperature_c=temp,rain_probability_pct=chance,wind_kmh=wind,valid_at=when.isoformat()))
    if not points:return None
    codes={p['weather_code'] for p in points}
    temperatures=[p['temperature_c'] for p in points if p['temperature_c'] is not None]
    chances=[p['rain_probability_pct'] for p in points if p['rain_probability_pct'] is not None]
    return dict(source='Open-Meteo',points=points,
        rain=bool(codes&(RAIN_CODES|STORM_CODES)) or any(p>=policy['rain_probability_pct'] for p in chances),
        storm=bool(codes&STORM_CODES),heavy_rain=bool(codes&{65,67,82}),max_temperature_c=max(temperatures) if temperatures else None,
        max_rain_probability_pct=max(chances) if chances else None)


def _locality(context,now,policy):
    location=context.get('current_location') or {}
    if not isinstance(location,dict) or location.get('status')!='granted':return ''
    if location.get('locality_status')!='ok' or not _fresh(location.get('observed_at'),now,policy['location_max_age_s']):return ''
    name=_text(location.get('locality'),100)
    # The existing reverse geocoder performs the real lookup. Refuse fallback
    # labels and administrative zones here as an additional presentation guard.
    if re.search(r'\b(?:zone|ward|division|road|street|selected location|map pin)\b|மண்டலம்|வார்டு',name,re.I):return ''
    return name if any(c.isalpha() for c in name) else ''


def _destination_kind(context):
    category=_text(context.get('destination_category')).casefold()
    if category in {'airport','railway','hospital'} and context.get('destination_category_verified') is True:return category
    name=_text(context.get('destination')).casefold()
    if re.search(r'\b(?:road|street|metro|bus stop)\b|சாலை|தெரு|மெட்ரோ',name):return None
    if name in {'chennai airport','chennai international airport','சென்னை விமான நிலையம்'}:return 'airport'
    if name in {'chennai central','chennai central railway station','tambaram railway station',
            'சென்னை சென்ட்ரல்','தாம்பரம் ரயில் நிலையம்'}:return 'railway'
    # A hospital name identifies a destination, never a diagnosis or emergency.
    if re.search(r'\bhospital\b|மருத்துவமனை',name):return 'hospital'
    return None


def _scope(context,stage,now,locality):
    opaque=_text(context.get('care_scope_id'),120)
    if opaque:return hashlib.sha256(('care:'+opaque).encode()).hexdigest()[:24]
    ride=context.get('ride_id',context.get('id'))
    ride=_number(ride,1)
    if ride is not None and ride.is_integer():return 'ride:'+str(int(ride))
    if stage=='dashboard':parts=['dashboard',now.date().isoformat(),locality]
    else:parts=['journey',context.get('session_id',''),context.get('pickup',''),context.get('destination',''),context.get('pickup_at','')]
    # Scope hashes exclude GPS coordinates and pronunciation text.
    return hashlib.sha256(json.dumps(parts,ensure_ascii=False,sort_keys=True,default=str).encode()).hexdigest()[:24]


def _seen(delivered):
    if isinstance(delivered,dict):
        return {key for key,value in delivered.items() if value in ('delivered','acknowledged','dismissed')}
    if isinstance(delivered,(str,bytes)):raise ValueError('delivered must contain complete delivery keys')
    result=set(delivered or ())
    if not all(isinstance(key,str) for key in result):raise ValueError('Delivery keys must be strings')
    return result


def recommend(context,delivered=(),limit=2,policy=None):
    """Return at most two suggestions. Never mutate context or delivery history.

    Accepted stages: dashboard, booking_details, before_confirmation,
    driver_assigned, driver_approaching, before_pickup, ride_completed. Existing confirm and
    weather_advisory stages are accepted as aliases. Booking stage done does
    NOT mean the ride is completed. All optional advice is declarative.
    """
    if not isinstance(context,dict):raise ValueError('context must be a dictionary')
    policy=_policy(policy);now=_time(context.get('now')) if context.get('now') is not None else datetime.now(TZ)
    if now is None:raise ValueError('Invalid context time')
    lang=LANGUAGES.get(_text(context.get('lang',context.get('language','English'))).casefold())
    if lang is None:raise ValueError('Choose English, Tamil or Tanglish')
    stage=_text(context.get('care_stage',context.get('stage','dashboard')))
    stage={'confirm':'before_confirmation','weather_advisory':'booking_details'}.get(stage,stage)
    allowed={'dashboard','booking_details','before_confirmation','driver_assigned','driver_approaching','before_pickup','ride_completed'}
    if stage not in allowed:stage='inactive'
    status=_text(context.get('ride_status',context.get('status',''))).casefold()
    locality=_locality(context,now,policy) if stage=='dashboard' else ''
    location=context.get('current_location') or {}
    # Permission denial produces a neutral dashboard greeting. Existing booked
    # journey reminders remain eligible through their OWN verified ride events.
    dashboard_location=(stage=='dashboard' and isinstance(location,dict) and location.get('status')=='granted'
        and _fresh(location.get('observed_at'),now,policy['location_max_age_s']))
    target=now if stage=='dashboard' else _time(context.get('pickup_at',context.get('pickup_time')))
    weather=_weather(context.get('current_weather') if stage=='dashboard' else context.get('weather'),target,now,policy)
    if stage=='dashboard' and not dashboard_location:weather=None
    hour=now.hour if stage=='dashboard' else target.hour if target is not None else None
    night=hour is not None and (hour>=policy['night_start_hour'] or hour<policy['night_end_hour'])
    period='morning' if 5<=now.hour<12 else 'afternoon' if 12<=now.hour<17 else 'evening'
    greeting=_pick(lang,{'morning':'Good morning!','afternoon':'Good afternoon!','evening':'Hello!'}[period],
        {'morning':'காலை வணக்கம்!','afternoon':'வணக்கம்!','evening':'வணக்கம்!'}[period],
        {'morning':'Kaalai vanakkam!','afternoon':'Vanakkam!','evening':'Vanakkam!'}[period])
    if locality:greeting+=' '+_pick(lang,'Travel tips for '+locality+'.',locality+' பகுதியில் உங்கள் பயணத்திற்கான குறிப்புகள்.',locality+'-la unga travel tips.')
    scope=_scope(context,stage,now,locality);seen=_seen(delivered);candidates=[]
    stage_prefix=scope+':stage:'+stage+':'
    remaining=max(0,2-sum(key.startswith(stage_prefix) for key in seen))
    def add(rule,en,ta,tg,evidence,urgency=2,safety=2,relevance=4,group='preparation'):
        # One rule may be relevant at several stages; its stable journey key
        # prevents repeated announcements after actual delivery/acknowledgment.
        key=scope+':'+rule
        if key in seen:return
        strength=5 if evidence else 3
        score=sum(SCORE_WEIGHTS[k]*v for k,v in dict(urgency=urgency,safety=safety,relevance=relevance,evidence=strength).items())
        candidates.append(dict(id=rule,delivery_key=key,stage_delivery_key=stage_prefix+rule,text=_pick(lang,en,ta,tg),language=lang,
            score=score,group=group,evidence=evidence,question=False))
    preparation=stage in {'booking_details','before_confirmation'} or dashboard_location
    vehicle=_text(context.get('vehicle'));count=_number(context.get('count',context.get('passengers')),1,16)
    if count is not None and not count.is_integer():count=None
    driver_id=_number(context.get('driver_id'),1)
    assigned=driver_id is not None and driver_id.is_integer() and status in {'accepted','arriving','in_progress'}

    if preparation and weather and weather['rain']:
        chance=weather['max_rain_probability_pct'];chance_en=(' Rain chance is '+_fmt(chance)+'%.') if chance is not None else ''
        chance_ta=(' மழை வாய்ப்பு '+_fmt(chance)+'%.') if chance is not None else ''
        chance_tg=(' Rain chance '+_fmt(chance)+'%.') if chance is not None else ''
        if weather['storm']:
            messages=('Thunderstorms are forecast around this time.'+chance_en+' Wait in a sheltered pickup area and check local warnings.',
                'அந்த நேரத்தில் இடியுடன் மழை எதிர்பார்க்கப்படுகிறது.'+chance_ta+' பாதுகாப்பான நிழற்குடை உள்ள இடத்தில் காத்திருந்து உள்ளூர் எச்சரிக்கைகளைச் சரிபாருங்கள்.',
                'Andha time-la thunderstorms expected.'+chance_tg+' Sheltered pickup place-la wait pannunga; local warnings-a check pannunga.')
        else:
            en='Heavy rain is forecast around this time.' if weather['heavy_rain'] else 'Rain is forecast around this time.'
            ta='அந்த நேரத்தில் கனமழை எதிர்பார்க்கப்படுகிறது.' if weather['heavy_rain'] else 'அந்த நேரத்தில் மழை எதிர்பார்க்கப்படுகிறது.'
            tg='Andha time-la heavy rain expected.' if weather['heavy_rain'] else 'Andha time-la rain expected.'
            messages=(en+chance_en+' Carry an umbrella and protect your belongings.',
                ta+chance_ta+' குடை எடுத்துக்கொண்டு உங்கள் பொருட்களைப் பாதுகாத்துக்கொள்ளுங்கள்.',
                tg+chance_tg+' Umbrella eduthukkonga; unga belongings-a protect pannikkonga.')
        if vehicle=='Bike':
            suffix=(' A covered vehicle can provide rain protection; the choice is yours.',
                ' மூடிய வாகனம் மழையிலிருந்து பாதுகாப்பளிக்கலாம்; முடிவு உங்களுடையது.',
                ' Covered vehicle rain protection kudukkum; unga choice.')
            messages=tuple(a+b for a,b in zip(messages,suffix))
        elif vehicle=='Auto':
            suffix=(' Protect yourself and your bags from rain entering through open sides.',
                ' திறந்த பக்கங்கள் வழியே வரும் மழையிலிருந்து உங்களையும் உடைமைகளையும் பாதுகாத்துக்கொள்ளுங்கள்.',
                ' Open sides-la varra rain-la irundhu ungalaiyum bags-aiyum protect pannikkonga.')
            messages=tuple(a+b for a,b in zip(messages,suffix))
        severity=5 if weather['storm'] else 4 if weather['heavy_rain'] else 3
        add('rain_preparation',*messages,['Open-Meteo target-hour points'],urgency=severity,safety=severity,group='weather')
    if preparation and weather and weather['max_temperature_c'] is not None and weather['max_temperature_c']>=policy['warm_c']:
        temp=_fmt(weather['max_temperature_c'])
        add('heat_protection','The forecast temperature reaches '+temp+'°C. Carry drinking water and use sun protection.',
            'முன்னறிவிப்பில் வெப்பநிலை '+temp+'°C வரை உள்ளது. குடிநீர் எடுத்துக்கொண்டு வெயிலிலிருந்து பாதுகாத்துக்கொள்ளுங்கள்.',
            'Forecast temperature '+temp+'°C varaikkum irukku. Drinking water eduthukkonga; sun protection use pannunga.',
            ['Open-Meteo temperature', 'application preparation threshold'],urgency=4 if float(temp)>=policy['hot_c'] else 2,safety=4,group='weather')
    if preparation and night:
        add('late_night_safety','For late-night travel, wait in a well-lit pickup area and verify the driver and vehicle before boarding.',
            'இரவு நேரப் பயணத்தில் வெளிச்சமுள்ள இடத்தில் காத்திருங்கள். ஏறும் முன் ஓட்டுநரையும் வாகனத்தையும் சரிபாருங்கள்.',
            'Late-night travel-ku well-lit pickup place-la wait pannunga. Era munnaadi driver-um vehicle-um verify pannunga.',
            ['Asia/Kolkata travel hour'],urgency=4,safety=5,group='safety')
    if stage in {'before_confirmation','driver_assigned','driver_approaching','before_pickup'} and status!='completed':
        add('pickup_guidance','Wait where your assigned vehicle can safely stop, and keep the pickup point easy to find.',
            'வாகனம் பாதுகாப்பாக நிற்கக்கூடிய இடத்தில் காத்திருங்கள். உங்களை எளிதில் கண்டுபிடிக்கக்கூடிய இடத்தைத் தேர்வு செய்யுங்கள்.',
            'Vehicle safe-ah stop panna mudiyura pickup place-la wait pannunga. Easy-ah kandupidikka mudiyura idathula irunga.',
            ['journey preparation stage'],urgency=2,safety=3)
    if stage in {'driver_assigned','driver_approaching','before_pickup'} and assigned and status in {'accepted','arriving'} and context.get('pickup_verified') is False:
        add('otp_safety','Verify the assigned driver and vehicle. Share your pickup OTP only with that driver when you board.',
            'ஒதுக்கப்பட்ட ஓட்டுநரையும் வாகனத்தையும் சரிபாருங்கள். வாகனத்தில் ஏறும்போது அந்த ஓட்டுநரிடம் மட்டும் பயணக் குறியீட்டைப் பகிருங்கள்.',
            'Assigned driver-um vehicle-um verify pannunga. Board pannumbodhu andha driver-kitta mattum pickup OTP share pannunga.',
            ['assigned driver', 'active ride', 'pickup not verified'],urgency=4,safety=5,group='safety')
    if stage in {'booking_details','before_confirmation'} and count is not None and vehicle in policy['capacities']:
        capacity=policy['capacities'][vehicle];n=int(count)
        if n>capacity:
            tamil_vehicle={'Bike':'பைக்','Auto':'ஆட்டோ','Mini':'மினி கார்','Sedan':'செடான்','SUV':'எஸ் யூ வி'}[vehicle]
            add('passenger_comfort',f'The configured {vehicle} passenger capacity is {capacity}; your group has {n}. Please review the vehicle choice.',
                f'{tamil_vehicle} வாகனத்தின் பயணிகள் கொள்ளளவு {capacity}; உங்கள் குழுவில் {n} பேர் உள்ளனர். வாகனத் தேர்வைச் சரிபாருங்கள்.',
                f'{vehicle} passenger capacity {capacity}; unga group-la {n} peru. Vehicle choice-a review pannunga.',
                ['passenger count','configured vehicle capacity'],urgency=5,safety=5)
        elif n>=3:
            add('passenger_comfort',f'For your group of {n}, keep luggage within the vehicle\'s available space and leave comfortable seating room.',
                f'உங்கள் குழுவில் {n} பேர் இருப்பதால், வாகனத்தில் உள்ள இடத்திற்கேற்ப உடைமைகளை வைத்துக்கொண்டு வசதியாக அமருங்கள்.',
                f'Unga group-la {n} peru; available space-ku etha maari luggage-a vachuttu comfortable-ah ukkarunga.',
                ['passenger count','chosen vehicle capacity'],urgency=1,safety=2)
    kind=_destination_kind(context)
    if stage in {'booking_details','before_confirmation'} and kind:
        messages={
            'airport':('Before leaving for the airport, check your luggage, travel documents and booking details.',
                'விமான நிலையத்திற்குக் கிளம்பும் முன் உடைமைகள், பயண ஆவணங்கள், முன்பதிவு விவரங்களைச் சரிபாருங்கள்.',
                'Airport-ku kelambura munnaadi luggage, travel documents, booking details-a check pannunga.'),
            'railway':('Before leaving for the railway station, keep your ticket and luggage ready.',
                'ரயில் நிலையத்திற்குக் கிளம்பும் முன் பயணச்சீட்டையும் உடைமைகளையும் தயாராக வைத்துக்கொள்ளுங்கள்.',
                'Railway station-ku kelambura munnaadi ticket-um luggage-um ready-ah vachukkonga.'),
            'hospital':('If needed for your hospital visit, keep appointment details, relevant documents and essentials ready.',
                'மருத்துவமனைப் பயணத்திற்குத் தேவைப்பட்டால் சந்திப்பு விவரங்கள், தொடர்புடைய ஆவணங்கள், அத்தியாவசியப் பொருட்களைத் தயாராக வைத்துக்கொள்ளுங்கள்.',
                'Hospital visit-ku thevaippatta appointment details, relevant documents, essentials-a ready-ah vachukkonga.')}
        add('destination_reminder',*messages[kind],['confirmed destination: '+kind],urgency=3,safety=2)
    if stage in {'driver_assigned','driver_approaching','before_pickup'} and status in STATUS_ACTIVE:
        eta=context.get('driver_eta') or {};eta_minutes=_number(eta.get('minutes'),0,180) if isinstance(eta,dict) else None
        has_eta=(assigned and eta_minutes is not None and eta.get('verified') is True and eta.get('live_driver_position') is True
            and _text(eta.get('source')) and _fresh(eta.get('updated_at'),now,policy['eta_max_age_s']))
        until=(target-now).total_seconds()/60 if target is not None else None
        if has_eta and eta_minutes<=10:
            value=_fmt(eta_minutes)
            add('departure_readiness',f'The latest driver ETA is about {value} minutes. Please be ready at your pickup point.',
                f'சமீபத்திய கணிப்பின்படி ஓட்டுநர் வர சுமார் {value} நிமிடங்கள் ஆகலாம். புறப்படும் இடத்தில் தயாராக இருங்கள்.',
                f'Latest driver ETA around {value} minutes. Pickup point-la ready-ah irunga.',
                ['verified driver ETA',_text(eta['source'])],urgency=5,safety=3)
        elif until is not None and 0<=until<=policy['departure_window_minutes']:
            add('departure_readiness','Your scheduled pickup time is approaching. Please have your essentials ready.',
                'நீங்கள் முன்பதிவு செய்த புறப்படும் நேரம் நெருங்குகிறது. தேவையான பொருட்களைத் தயாராக வைத்துக்கொள்ளுங்கள்.',
                'Scheduled pickup time nerungudhu. Unga essentials-a ready-ah vachukkonga.',
                ['scheduled pickup time'],urgency=4,safety=2)
    traffic=context.get('traffic') or {};delay=_number(traffic.get('extra_minutes'),1,240) if isinstance(traffic,dict) else None
    traffic_source=_text(traffic.get('source')).casefold() if isinstance(traffic,dict) else ''
    trusted_traffic=traffic_source in policy['trusted_traffic_sources'] and not re.search(r'osrm|straight.?line|static.?route',traffic_source)
    if stage in {'booking_details','before_confirmation','driver_approaching'} and delay is not None and traffic.get('live') is True and traffic.get('verified') is True and trusted_traffic and _fresh(traffic.get('updated_at'),now,policy['traffic_max_age_s']):
        value=_fmt(delay)
        add('traffic_awareness',f'The live traffic estimate indicates about {value} extra minutes. Allow time for your journey.',
            f'நேரடி போக்குவரத்துக் கணிப்பில் சுமார் {value} கூடுதல் நிமிடங்கள் காட்டப்படுகின்றன. பயணத்திற்கு நேரம் ஒதுக்குங்கள்.',
            f'Live traffic estimate-la around {value} extra minutes kaattudhu. Journey-ku time allow pannikkonga.',
            ['verified live traffic',_text(traffic['source'])],urgency=4,safety=2)
    if stage in {'before_confirmation','driver_assigned'} and context.get('accessibility_requested') is True:
        add('accessibility_assistance','If you need boarding assistance, contact the service or assigned driver in advance to check what support is available.',
            'வாகனத்தில் ஏற உதவி தேவைப்பட்டால், கிடைக்கக்கூடிய உதவியை முன்கூட்டியே சேவை மையம் அல்லது ஒதுக்கப்பட்ட ஓட்டுநரிடம் சரிபாருங்கள்.',
            'Boarding assistance thevaippatta service illa assigned driver-kitta advance-ah available support-a check pannunga.',
            ['customer requested assistance'],urgency=3,safety=4)
    if dashboard_location or stage in {'before_confirmation','driver_assigned'}:
        add('phone_readiness','Keep your phone charged for driver calls and trip tracking.',
            'ஓட்டுநருடன் பேசவும் பயணத்தைக் கண்காணிக்கவும் உங்கள் கைப்பேசியில் போதுமான மின்சக்தி இருக்குமாறு பார்த்துக்கொள்ளுங்கள்.',
            'Driver calls-ukkum trip tracking-ukkum phone charge-a vachukkonga.',
            ['travel preparation stage'],urgency=1,safety=1)
    if stage=='ride_completed' and status=='completed':
        add('arrival_care','Before leaving the vehicle, check that you have your phone, bags and other belongings. Thank you, and take care!',
            'வாகனத்திலிருந்து இறங்கும் முன் கைப்பேசி, பைகள், மற்ற உடைமைகளை எடுத்துக்கொண்டீர்களா என்று சரிபாருங்கள். நன்றி! நலமாகப் பயணியுங்கள்!',
            'Vehicle-la irundhu erangura munnaadi phone, bags, matha belongings-a check pannunga. Nandri! Take care!',
            ['ride status is completed'],urgency=4,safety=3,group='arrival')
    if status=='cancelled' or stage=='inactive':candidates=[]
    # Rank policy: 4*urgency + 3*safety + 2*relevance + evidence.
    candidates.sort(key=lambda r:(-r['score'],RULE_IDS.index(r['id'])))
    selected=[];weather_count=0
    maximum=_number(limit,0,2)
    if maximum is None or not maximum.is_integer():raise ValueError('limit must be 0, 1 or 2')
    for item in candidates:
        if len(selected)>=min(int(maximum),remaining):break
        if item['group']=='weather' and weather_count:continue
        selected.append(item);weather_count+=item['group']=='weather'
    # Only minimal presentation data is exposed; GPS/OTP/phone/driver IDs remain
    # in the caller's existing protected booking/location architecture.
    return dict(version=VERSION,stage=stage,language=lang,greeting=greeting,locality=locality or None,
        weather=weather,recommendations=selected,
        candidate_ids=[item['id'] for item in candidates],remaining_stage_budget=remaining,optional=True)


def remember_delivered(history,recommendations,status='delivered',max_items=256):
    """Return a bounded copy; caller invokes this after real delivery/ack only."""
    if status not in {'delivered','acknowledged','dismissed'}:raise ValueError('Invalid delivery status')
    if not isinstance(history,dict):raise ValueError('history must be a dictionary')
    result=dict(history)
    for item in recommendations:
        key=item.get('delivery_key') if isinstance(item,dict) else None
        if not isinstance(key,str) or not key:raise ValueError('Missing recommendation delivery key')
        result[key]=status
        stage_key=item.get('stage_delivery_key')
        if isinstance(stage_key,str) and stage_key:result[stage_key]=status
    while len(result)>max(1,int(max_items)):result.pop(next(iter(result)))
    return result


MESSAGES = {
    "rain": (
        "Rain is possible. Carry an umbrella and keep your belongings dry.",
        "மழை பெய்ய வாய்ப்புள்ளது. குடை எடுத்துச் செல்லுங்கள்; உடைமைகளை நனையாமல் பாதுகாத்துக்கொள்ளுங்கள்.",
        "Mazhai peiya vaaippu irukku. Umbrella eduthuttu ponga; belongings nanaiyaama paathukkonga."
    ),
    "heat": (
        "It's quite warm. Carry drinking water and stay hydrated.",
        "வெப்பம் அதிகமாக உள்ளது. குடிநீர் எடுத்துச் சென்று உடலில் நீர்ச்சத்து குறையாமல் பார்த்துக்கொள்ளுங்கள்.",
        "Veyil adhigama irukku. Thanni eduthuttu ponga; hydrated ah irunga."
    ),
    "night": (
        "For late-night travel, wait in a well-lit area and verify your driver before boarding.",
        "இரவு நேரப் பயணத்தில் வெளிச்சமான இடத்தில் காத்திருந்து, ஏறும் முன் ஓட்டுநர் விவரங்களைச் சரிபாருங்கள்.",
        "Night travel ku velichamaana place la wait pannunga. Erurathukku munnaadi driver details verify pannunga."
    ),
    "otp": (
        "Share your pickup OTP only with your assigned driver when boarding.",
        "வாகனத்தில் ஏறும்போது மட்டும் ஒதுக்கப்பட்ட ஓட்டுநரிடம் உங்கள் OTP-ஐ பகிருங்கள்.",
        "Board pannumbodhu mattum assigned driver kitta pickup OTP share pannunga."
    ),
    "capacity": (
        "Please select a vehicle that comfortably accommodates everyone travelling.",
        "பயணம் செய்யும் அனைவருக்கும் வசதியான வாகனத்தைத் தேர்ந்தெடுக்கவும்.",
        "Travel panra ellaarukkum comfortable ah irukkura vehicle select pannunga."
    ),
    "airport": (
        "Before leaving for the airport, check your travel documents and luggage.",
        "விமான நிலையத்திற்குச் செல்லும் முன் பயண ஆவணங்களையும் உடைமைகளையும் சரிபாருங்கள்.",
        "Airport ku kilamburathukku munnaadi documents um luggage um check pannunga."
    ),
    "railway": (
        "Keep your train ticket and travel essentials ready.",
        "ரயில் பயணச்சீட்டையும் தேவையான பொருட்களையும் தயாராக வைத்துக்கொள்ளுங்கள்.",
        "Train ticket um travel essentials um ready ah vechukkonga."
    ),
    "hospital": (
        "Remember any medical documents and essential items you need.",
        "தேவையான மருத்துவ ஆவணங்களையும் அத்தியாவசியப் பொருட்களையும் எடுத்துக்கொள்ளுங்கள்.",
        "Thevaiyaana medical documents um essentials um eduthukkonga."
    ),
    "phone": (
        "Keep your phone charged so you can contact and track your driver.",
        "ஓட்டுநரைத் தொடர்புகொள்ளவும் கண்காணிக்கவும் கைப்பேசியில் போதுமான சார்ஜ் வைத்திருங்கள்.",
        "Driver ah contact pannavum track pannavum phone charge vechukkonga."
    ),
    "arrival": (
        "Before leaving the vehicle, please check that you have all your belongings.",
        "வாகனத்திலிருந்து இறங்கும் முன் உங்கள் உடைமைகள் அனைத்தும் இருப்பதைச் சரிபாருங்கள்.",
        "Vehicle la irundhu erangurathukku munnaadi belongings ellaam irukka nu check pannunga."
    ),
    "pickup": (
        "Choose a safe, accessible pickup point where your driver can stop.",
        "ஓட்டுநர் நிறுத்தக்கூடிய பாதுகாப்பான, எளிதில் அணுகக்கூடிய இடத்தைத் தேர்ந்தெடுக்கவும்.",
        "Driver stop panna mudiyura safe ah accessible ah irukkura pickup point choose pannunga."
    ),
    "accessibility": (
        "If you need boarding assistance, please coordinate with your assigned driver.",
        "ஏறுவதற்கு உதவி தேவைப்பட்டால் ஒதுக்கப்பட்ட ஓட்டுநருடன் தொடர்புகொள்ளுங்கள்.",
        "Boarding assistance venumna assigned driver kitta coordinate pannunga."
    ),
    "ready": (
        "Your driver is approaching. Please get ready at your pickup point.",
        "உங்கள் ஓட்டுநர் நெருங்கிக்கொண்டிருக்கிறார். ஏறும் இடத்தில் தயாராக இருங்கள்.",
        "Unga driver nerungittu irukkaaru. Pickup point la ready ah irunga."
    ),
}

CAPACITY = {"bike": 1, "auto": 3, "mini": 4, "sedan": 4, "suv": 6}

# MOVIO_CUSTOMER_CARE_COMPAT_V1
# The legacy public API delegates to the SAME pure recommendation engine.
VERSION='customer-care-engine-1.1-compatible'
_CARE_LEGACY_STAGES={'details':'booking_details','pre_confirmation':'before_confirmation'}
_CARE_LEGACY_IDS={'rain_preparation':'rain','heat_protection':'heat',
    'late_night_safety':'night','pickup_guidance':'pickup','otp_safety':'otp',
    'passenger_comfort':'capacity','departure_readiness':'ready',
    'traffic_awareness':'traffic','accessibility_assistance':'accessibility',
    'phone_readiness':'phone','arrival_care':'arrival'}


def _care_legacy_context(context,stage=None):
    c=copy.deepcopy(context) if isinstance(context,dict) else {}
    lang=LANGUAGES.get(_text(c.get('lang',c.get('language','English'))).casefold(),'English')
    c['lang']=lang
    chosen=stage or c.get('care_stage',c.get('stage','dashboard'))
    c['care_stage']=_CARE_LEGACY_STAGES.get(chosen,chosen)
    if 'count' not in c and 'passenger_count' in c:c['count']=c['passenger_count']
    vehicle=_text(c.get('vehicle')).casefold()
    canonical={'bike':'Bike','auto':'Auto','mini':'Mini','mini car':'Mini','sedan':'Sedan','suv':'SUV'}
    if vehicle in canonical:c['vehicle']=canonical[vehicle]
    if 'pickup_at' not in c and c.get('pickup_time'):c['pickup_at']=c['pickup_time']
    # A bare locality or status='ok' is NOT proof of GPS permission or freshness.
    # Current dashboard callers must provide the existing verified-location
    # object and timestamp. Never infer permission from a string or coordinates.
    if not isinstance(c.get('current_location'),dict):c['current_location']={}
    raw=c.get('current_weather') if c['care_stage']=='dashboard' else c.get('weather')
    if c['care_stage']=='dashboard' and raw is None:raw=c.get('weather')
    if isinstance(raw,dict) and 'points' not in raw:
        # Accept the old flat shape ONLY when it already carries provider,
        # fetch timestamp and forecast-valid timestamp. Add no invented data.
        keys=('weather_code','precipitation_probability','temperature_2m','wind_speed_10m')
        raw=dict(raw,points=[dict(location='pickup',valid_at=raw.get('valid_at'),
                                 **{key:raw.get(key) for key in keys})])
    if c['care_stage']=='dashboard':c['current_weather']=raw
    else:c['weather']=raw
    return c


def _care_legacy_id(rule,context):
    if rule=='destination_reminder':return _destination_kind(context) or 'destination'
    return _CARE_LEGACY_IDS.get(rule,rule)


def _care_legacy_result(context,stage=None):
    c=_care_legacy_context(context,stage)
    history=c.get('delivered',())
    if not isinstance(history,(dict,list,tuple,set)):history=()
    # Legacy acknowledged rule IDs remain understood; generation alone never
    # creates a delivery record. New clients can retain complete delivery keys.
    acknowledged=c.get('acknowledged') or []
    if not isinstance(acknowledged,(list,tuple,set)):acknowledged=[]
    acknowledged={item for item in acknowledged if isinstance(item,str)}
    initial=recommend(c,delivered=history,limit=0)
    now=_time(c.get('now')) if c.get('now') is not None else datetime.now(TZ)
    scope=_scope(c,initial['stage'],now,initial['locality'] or '')
    seen=_seen(history)
    for rule in RULE_IDS:
        if rule in acknowledged or _care_legacy_id(rule,c) in acknowledged:
            seen.add(scope+':'+rule)
    return c,recommend(c,delivered=seen,limit=2)


def evaluate(context):
    """Legacy API: a list of at most two recommendations, with stable IDs."""
    c,result=_care_legacy_result(context)
    return [dict(row,id=_care_legacy_id(row['id'],c),rule_id=row['id'],
                 evidence='; '.join(row['evidence'])) for row in result['recommendations']]


def dashboard_greeting(context):
    """Legacy greeting dictionary, using only verified current context."""
    c,result=_care_legacy_result(context,'dashboard')
    weather=result['weather'];locality=result['locality'];lang=result['language']
    if weather is None:
        return dict(text=result['greeting'],weather_verified=False,
                    locality_verified=bool(locality))
    codes={point['weather_code'] for point in weather['points']}
    temp=weather['max_temperature_c'];chance=weather['max_rain_probability_pct']
    hot=temp is not None and temp>=DEFAULT_POLICY['warm_c']
    pleasant=bool(codes&{0,1,2}) and temp is not None and 18<=temp<=32 and not weather['rain']
    condition=('rain' if weather['rain'] else 'heat' if hot else
               'cloudy' if 3 in codes else 'pleasant' if pleasant else 'normal')
    messages={
        'rain':('Rain is forecast around this time.','அந்த நேரத்தில் மழை எதிர்பார்க்கப்படுகிறது.',
                'Andha time-la rain expected.'),
        'heat':('It is warm around your area.','உங்கள் பகுதியில் வெப்பம் அதிகமாக உள்ளது.',
                'Unga area-la veyil adhigama irukku.'),
        'cloudy':('Cloudy weather is expected around your area.','உங்கள் பகுதியில் மேகமூட்டம் எதிர்பார்க்கப்படுகிறது.',
                  'Unga area-la cloudy weather expected.'),
        'pleasant':('The weather looks comfortable for your journey.','பயணத்திற்கு வானிலை இனிமையாக உள்ளது.',
                    'Unga journey-ku weather comfortable-ah irukku.'),
        'normal':('Here is your local weather update.','உங்கள் பகுதியின் வானிலை தகவல்.',
                  'Unga area weather update.')}
    text=result['greeting']+' '+_pick(lang,*messages[condition])
    if temp is not None:
        value=_fmt(temp)
        text+=' '+_pick(lang,f'Temperature is around {value}°C.',f'வெப்பநிலை சுமார் {value}°C.',
                       f'Temperature around {value}°C.')
    if chance is not None:
        value=_fmt(chance)
        text+=' '+_pick(lang,f'Rain chance is {value}%.',f'மழை வாய்ப்பு {value}%.',f'Rain chance {value}%.')
    return dict(text=text,weather_verified=True,locality_verified=bool(locality),
                condition=condition,temperature_c=temp,rain_probability_pct=chance)
'''
    api_checks_source = r'''def run_backend_checks(api,care):
    import copy
    import json
    import types
    from datetime import datetime,timedelta
    from contextlib import nullcontext
    from pathlib import Path
    checks=[];calls=[]
    def check(value,label):
        if not value:raise AssertionError(label)
        checks.append(label)
    now=datetime(2031,10,19,14,25,tzinfo=care.TZ)
    location=dict(status='granted',lat=12.9123456,lng=80.2123456,observed_at=now.isoformat())
    def reverse(lat,lng):
        calls.append(('reverse',lat,lng))
        return dict(locality='Tambaram',locality_status='ok')
    def forecast(points,pickup,now=None):
        check(len(points)==2 and points[0]==points[1],'existing two-point provider protocol')
        calls.append(('forecast',points,pickup))
        hour=pickup.replace(minute=0,second=0,microsecond=0).isoformat()
        return dict(status='ok',source='Open-Meteo',fetched_at=now.timestamp(),points=[
            dict(location=role,valid_at=hour,weather_code=61,temperature_2m=27,precipitation_probability=75)
            for role in ('pickup','destination')])
    class Handler:
        path='/api/customer-care/recommend'
        role='customer'
        denied=False
        body={}
        def user(self,db):
            if self.denied:raise PermissionError('fixture authentication')
            return dict(id=7,role=self.role)
        def payload(self,limit=0):
            check(limit==32768,'request body has a bounded size')
            return self.body
        def json(self,status,payload):return status,payload
        def do_POST(self):return ('original_post',self.path)
        def do_GET(self):return ('original_get',self.path)
    server=types.SimpleNamespace(ROOT=Path('/not/a/real/project'),Handler=Handler,conn=lambda:nullcontext(object()))
    original_log=api.LOG;api.LOG=types.SimpleNamespace(warning=lambda *args,**kwargs:None)
    try:
        api.install_backend(server,providers=(reverse,forecast),clock=lambda:now,care=care)
        post=Handler.do_POST;get=Handler.do_GET
        api.install_backend(server,providers=(reverse,forecast),clock=lambda:now,care=care)
        check(Handler.do_POST is post and Handler.do_GET is get,'endpoint installation is idempotent')
        h=Handler()
        for language in ('English','Tamil','Tanglish'):
            context=dict(language=language,current_location=location,weather={'status':'ok','temperature_2m':99},
                         locality='Invented area',stage='driver_assigned',driver_id=1,ride_status='accepted')
            h.body=json.dumps({'context':context},ensure_ascii=False).encode('utf-8')
            status,result=h.do_POST()
            check(status==200 and result['language']==language,language+' raw JSON bytes accepted')
            check(result['locality']=='Tambaram' and result['weather']['temperature_c']==27,language+' server lookup overrides browser claims')
            check(result['weather']['rain_probability_pct']==75 and result['weather']['condition']=='rain',language+' actual probability and weather')
            check(len(result['recommendations'])<=2 and all(r['id']!='otp' for r in result['recommendations']),language+' dashboard scope cannot become assigned-driver context')
            check('12.9123456' not in json.dumps(result) and '80.2123456' not in json.dumps(result),language+' precise GPS is not exposed')
            before=copy.deepcopy(context);api.dashboard_response(care,context,{'id':7},reverse,forecast,now)
            check(before==context,language+' browser booking/context data remains unchanged')
        check(calls[0]==('reverse',12.912,80.212),'only approximate area is sent to the existing locality cache')
        for state in ('denied','unavailable','unsupported','timeout'):
            calls.clear();h.body={'context':dict(current_location=dict(location,status=state),weather={'status':'ok','weather_code':61})}
            status,result=h.do_POST()
            check(status==200 and result['weather']['status']=='unavailable' and result['locality'] is None,'neutral response for '+state)
            check(not calls,'no provider calls for '+state)
        for modified in ({'lat':True},{'lng':181},{'observed_at':(now-timedelta(hours=1)).isoformat()},
                         {'observed_at':(now+timedelta(minutes=10)).isoformat()}):
            calls.clear();h.body={'context':dict(current_location=dict(location,**modified))}
            status,result=h.do_POST();check(status==200 and not calls,'invalid/stale position is ignored')
        for body in (b'{',b'[]',b'{"context":[]}',b'{"context":{"n":NaN}}'):
            h.body=body;check(h.do_POST()[0]==400,'malformed request is rejected')
        h.body={'context':{}};h.denied=True;calls.clear()
        check(h.do_POST()[0]==401 and not calls,'signed-out users cannot trigger lookups')
        h.denied=False;h.role='driver'
        check(h.do_POST()[0]==403 and not calls,'driver accounts cannot use the customer endpoint')
        h.role='customer';h.path='/api/rides';check(h.do_POST()==('original_post','/api/rides'),'booking POST handler remains in control')
        h.path='/api/customer-tools/reverse-address';check(h.do_GET()==('original_get',h.path),'existing geocoding GET handler remains in control')
        h.path='/api/customer-care/health';check(h.do_GET()[1]['version']==api.VERSION,'new health endpoint reports the installed version')
        def unavailable(*args,**kwargs):return dict(status='unavailable',source='Open-Meteo')
        result=api.dashboard_response(care,dict(current_location=location),{'id':7},reverse,unavailable,now)
        check(result['ok'] and result['weather']['status']=='unavailable' and result['optional'],'forecast failure is optional')
        def failed(*args,**kwargs):raise LookupError('fixture provider failure')
        result=api.dashboard_response(care,dict(current_location=location),{'id':7},failed,forecast,now)
        check(result['locality'] is None and result['weather']['status']=='ok','locality failure does not discard valid weather')
        result=api.dashboard_response(care,dict(current_location=dict(location,observed_at=now.timestamp()*1000)),{'id':7},reverse,forecast,now)
        check(result['weather']['status']=='ok','browser millisecond timestamp is understood')
        history=care.remember_delivered({},result['recommendations'])
        again=api.dashboard_response(care,dict(current_location=location,delivered=history),{'id':7},reverse,forecast,now)
        check(not again['recommendations'],'displayed recommendations do not rotate into repeated extra tips')
        # Each request uses its own point list; inputs from the provider are not mutated.
        check(care.VERSION=='customer-care-engine-1.1-compatible','existing engine version preserved')
        return dict(passed=True,checks=len(checks),scope='Offline provider and authenticated-handler fixtures; no real user, live GPS, database writes or TTS.')
    finally:api.LOG=original_log
'''
    engine_checks_source = r'''def run_care_checks(care):
    import copy
    import json
    import re
    from datetime import datetime,timedelta
    now=datetime(2031,10,19,14,0,tzinfo=care.TZ)
    checks=[];covered=set()
    def check(condition,name):
        if not condition:raise AssertionError(name)
        checks.append(name)
    def weather(code=3,chance=10,temp=27,target=None):
        return dict(status='ok',source='Open-Meteo',fetched_at=now.timestamp(),points=[
            dict(location='pickup',weather_code=code,precipitation_probability=chance,temperature_2m=temp,valid_at=(target or now).isoformat())])
    def context(stage='booking_details',**values):
        result=dict(stage=stage,lang='English',now=now.isoformat(),session_id='fixture-session',
            pickup='Tambaram',destination='Vandalur',pickup_at=now.isoformat(),vehicle='Mini',count=1)
        result.update(values);return result
    def evaluate(value,**kwargs):
        result=care.recommend(value,**kwargs);covered.update(result['candidate_ids'])
        check(len(result['recommendations'])<=2,'at most two recommendations')
        check(all(not row['question'] and '?' not in row['text'] for row in result['recommendations']),'optional advice asks no follow-up question')
        return result
    def selected(result):return {r['id'] for r in result['recommendations']}
    def candidates(result):return set(result['candidate_ids'])
    def location(status='granted',name='Tambaram'):
        return dict(status=status,locality=name,locality_status='ok',observed_at=now.isoformat())

    for lang in ('English','Tamil','Tanglish'):
        rainy=context(lang=lang,weather=weather(61,75))
        before=copy.deepcopy(rainy);r=evaluate(rainy)
        check('rain_preparation' in selected(r),lang+' rain preparation selected')
        check('75%' in r['recommendations'][0]['text'],lang+' actual rain percentage retained')
        check(rainy==before,lang+' booking slots and weather remain unchanged')
        texts=r['greeting']+' '+''.join(x['text'] for x in r['recommendations'])
        check(bool(re.search(r'[\u0B80-\u0BFF]',texts))==(lang=='Tamil'),lang+' display script is correct')
        for failed in ('unavailable','pending','outside_forecast_range'):
            r=evaluate(context('before_confirmation',lang=lang,weather=dict(status=failed)))
            check(r['weather'] is None and 'rain_preparation' not in candidates(r),lang+' '+failed+' cannot invent a forecast')
            check(r['optional'] is True,lang+' weather failure does not require consent')
        for state in ('denied','unavailable','unsupported','timeout'):
            r=evaluate(context('dashboard',lang=lang,current_location=location(state),current_weather=weather(61,95)))
            check(r['locality'] is None and r['weather'] is None and not r['recommendations'],lang+' '+state+' has a neutral greeting')
    r=evaluate(context('dashboard',current_location=location(),current_weather=weather(0,0,27)))
    check(r['locality']=='Tambaram' and 'Tambaram' in r['greeting'],'actual locality in dashboard greeting')
    check('phone_readiness' in selected(r),'phone readiness is available for dashboard travel preparation')
    for name in ('Zone 14','Ward 8','Map pin (12.9,80.2)','Selected location','Kottivakkam Kuppam Road'):
        r=evaluate(context('dashboard',current_location=location(name=name),current_weather=weather()))
        check(r['locality'] is None,'administrative/road/fallback label is not a locality: '+name)
    old_location=location();old_location['observed_at']=(now-timedelta(hours=2)).isoformat()
    r=evaluate(context('dashboard',current_location=old_location,current_weather=weather(61,95)))
    check(r['locality'] is None and r['weather'] is None and not r['recommendations'],'stale GPS does not become current-locality evidence')

    r=evaluate(context(weather=weather(0,0,35)));check('heat_protection' in selected(r),'heat protection uses measured forecast temperature')
    for h in (0,1,23):
        target=now.replace(hour=h)+timedelta(days=1)
        r=evaluate(context(pickup_at=target.isoformat(),weather=weather(61,80,target=target)))
        check('late_night_safety' in selected(r),'late-night advice for hour '+str(h))
    r=evaluate(context(pickup_at=now.astimezone(care.timezone.utc).replace(hour=18).isoformat()))
    check('late_night_safety' in candidates(r),'UTC input is converted to Asia/Kolkata')
    r=evaluate(context('before_confirmation'))
    check('pickup_guidance' in selected(r),'accessible pickup guidance before confirmation')
    assigned=context('driver_assigned',ride_id=7,ride_status='accepted',driver_id=9,pickup_verified=False)
    r=evaluate(assigned);check('otp_safety' in selected(r),'OTP safety requires an assigned driver')
    for driver,status,verified in ((None,'accepted',False),(9,'requested',False),(9,'in_progress',True),(9,'accepted',True)):
        r=evaluate(context('driver_assigned',ride_status=status,driver_id=driver,pickup_verified=verified))
        check('otp_safety' not in candidates(r),'OTP stage and driver association are respected')
    r=evaluate(context(count=4,vehicle='Mini'));check('passenger_comfort' in selected(r),'group comfort uses passenger count')
    r=evaluate(context(count=4,vehicle='Auto'));check('passenger_comfort' in selected(r),'over-capacity vehicle gets preparation guidance')
    for place in ('Chennai Airport','Chennai Central','Apollo Hospital'):
        r=evaluate(context(destination=place));check('destination_reminder' in selected(r),'destination reminder: '+place)
    for place in ('Airport Road','Chennai Airport Metro','Hospital Street'):
        r=evaluate(context(destination=place));check('destination_reminder' not in candidates(r),'road/metro is not assumed to be a terminal or hospital visit')
    due=now+timedelta(minutes=10)
    r=evaluate(context('before_pickup',ride_status='scheduled',pickup_at=due.isoformat()))
    check('departure_readiness' in selected(r),'scheduled pickup readiness without invented driver ETA')
    check(all('ETA' not in row['text'] for row in r['recommendations']),'schedule reminder does not claim driver arrival')
    eta=dict(minutes=4,updated_at=now.isoformat(),source='tracking-route-fixture',verified=True,live_driver_position=True)
    approach=context('driver_approaching',ride_id=7,ride_status='arriving',driver_id=9,pickup_verified=False,driver_eta=eta)
    r=evaluate(approach);check('departure_readiness' in selected(r),'fresh ETA backed by live driver position')
    for change in ({'updated_at':(now-timedelta(minutes=10)).isoformat()},{'live_driver_position':False},{'verified':False}):
        bad=context('driver_approaching',ride_status='arriving',driver_id=9,driver_eta={**eta,**change},pickup_at=(now+timedelta(hours=2)).isoformat())
        r=evaluate(bad);check('departure_readiness' not in candidates(r),'stale/unverified/static route ETA is not announced')

    traffic=dict(extra_minutes=8,live=True,verified=True,source='fixture-live-provider',updated_at=now.isoformat())
    r=evaluate(context(traffic=traffic));check('traffic_awareness' not in candidates(r),'traffic disabled without configured reliable source')
    r=evaluate(context(traffic=traffic),policy={'trusted_traffic_sources':['fixture-live-provider']})
    check('traffic_awareness' in selected(r),'traffic advice with explicitly configured live fixture evidence')
    for name in ('OSRM','straight-line route','static_route'):
        r=evaluate(context(traffic={**traffic,'source':name}),policy={'trusted_traffic_sources':[name]})
        check('traffic_awareness' not in candidates(r),'static route data is not live traffic: '+name)
    r=evaluate(context('before_confirmation',accessibility_requested=True));check('accessibility_assistance' in selected(r),'optional requested boarding support')
    r=evaluate(context('before_confirmation'));check('accessibility_assistance' not in candidates(r),'no disability or accessibility need is inferred')
    r=evaluate(context('ride_completed',ride_id=7,ride_status='completed'))
    check(selected(r)=={'arrival_care'},'belongings reminder after actual ride completion')
    for stage,status in (('done','requested'),('ride_completed','requested'),('confirm','cancelled')):
        r=evaluate(context(stage,ride_status=status));check('arrival_care' not in candidates(r),'booking completion is not journey completion')
    r=evaluate(context('before_confirmation',ride_status='cancelled',weather=weather(61,90)))
    check(not r['recommendations'],'cancelled booking has no preparation advice')

    # Delivery is explicit: interrupted/generated-only speech cannot consume it.
    request=context('before_confirmation',weather=weather(61,80),pickup_at=now.replace(hour=23).isoformat())
    request['weather']=weather(61,80,target=now.replace(hour=23));prior=copy.deepcopy(request)
    first=care.recommend(request);again=care.recommend(request)
    check(first==again,'recommendation generation alone never marks advice as heard')
    history={};updated=care.remember_delivered(history,first['recommendations'],'acknowledged')
    check(history=={} and request==prior,'history and context inputs are not mutated')
    check(not care.recommend(request,delivered=updated)['recommendations'],'at most two delivered suggestions per stage; polling cannot drip all rules')
    next_stage={**request,'stage':'driver_assigned','ride_status':'accepted','driver_id':9,'pickup_verified':False}
    r=evaluate(next_stage,delivered=updated)
    check('otp_safety' in selected(r),'new ride stage enables its own relevant safety advice')
    check('rain_preparation' not in candidates(r),'acknowledged weather advice is not repeated at another stage')
    check(care.remember_delivered(updated,first['recommendations'],'acknowledged')==updated,'repeated delivery does not append duplicate records')
    preparation=context('before_confirmation',care_scope_id='stable-fixture-journey')
    shown=care.recommend(preparation)
    ledger=care.remember_delivered({},shown['recommendations'])
    later={**preparation,'stage':'driver_assigned','ride_id':99,'driver_id':9,'ride_status':'accepted','pickup_verified':False}
    r=evaluate(later,delivered=ledger)
    check('phone_readiness' not in candidates(r),'stable care scope survives allocation of a database booking ID')
    for v,phrase in (('Bike','covered vehicle'),('Auto','open sides')):
        r=evaluate(context(vehicle=v,weather=weather(61,80)))
        check(any(phrase in row['text'].casefold() for row in r['recommendations']),'vehicle-specific rain preparation for '+v)
    limited=care.remember_delivered({str(i):'delivered' for i in range(300)},first['recommendations'],max_items=32)
    check(len(limited)==32,'bounded delivery history')

    base_weather=weather(3,0,0);r=evaluate(context(weather=base_weather))
    check(r['weather']['max_temperature_c']==0 and r['weather']['max_rain_probability_pct']==0,'zero is preserved as a real weather value')
    for raw in (
        {**weather(),'fetched_at':(now-timedelta(hours=2)).timestamp()},
        {**weather(),'fetched_at':(now+timedelta(hours=1)).timestamp()},
        {**weather(),'source':'Unverified guess'},
        weather(target=now+timedelta(hours=3))):
        check(evaluate(context(weather=raw))['weather'] is None,'stale, future-fetched, wrong-source or wrong-hour forecast rejected')
    r=evaluate(context(weather=weather(temp=float('nan'),chance=float('nan'),code=None)))
    check(r['weather'] is None,'non-finite weather does not become a forecast')
    storm_time=now.replace(hour=23)
    r=evaluate(context(pickup_at=storm_time.isoformat(),weather=weather(95,95,40,target=storm_time)))
    check('rain_preparation' in selected(r) and 'late_night_safety' in selected(r),'severe weather and late-night safety outrank general tips')
    check(len([row for row in r['recommendations'] if row['group']=='weather'])<=1,'weather recommendations do not crowd the card')
    secret=context('before_confirmation',pickup_geo={'lat':12.9,'lng':80.2},destination_geo={'lat':13.0,'lng':80.3},pickup_otp='SECRET_OTP',driver_phone='SECRET_PHONE')
    encoded=json.dumps(evaluate(secret),ensure_ascii=False)
    check('SECRET_OTP' not in encoded and 'SECRET_PHONE' not in encoded and '12.9' not in encoded,'precise GPS, OTP and phone are not emitted in recommendation output')
    for limit in (0,1,2):check(len(care.recommend(context('before_confirmation'),limit=limit)['recommendations'])<=limit,'explicit output limit '+str(limit))
    for bad in ({'lang':'unsupported'},{'now':'bad timestamp'}):
        try:care.recommend(context(**bad))
        except ValueError:pass
        else:raise AssertionError('invalid context must be rejected')
        checks.append('invalid context is rejected')
    check(covered==set(care.RULE_IDS),'all twelve rule families exercised')
    return dict(passed=True,checks=len(checks),rule_families=sorted(covered),
        scope='Offline deterministic fixtures; no real GPS, external APIs, speech generation or live booking actions.')
'''

    # Compare both reviewed sources in the CURRENT Python runtime.
    # AST dump formatting changes across Python versions; a precomputed dump hash
    # must never be used to reject an otherwise identical source file.
    reviewed_legacy_api = r'''
"""Read-only customer-care recommendation endpoint."""

import importlib.util
import json
from pathlib import Path
from urllib.parse import urlparse

VERSION = "customer-care-api-0.1"

def install_backend(server):
    if getattr(server.Handler, "_customer_care_api_version", None) == VERSION:
        return {"version": VERSION, "already_installed": True}

    root = Path(server.ROOT)
    path = root / "integrations/movio_customer_care.py"

    spec = importlib.util.spec_from_file_location(
        "movio_customer_care", path
    )
    care = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(care)

    previous_post = server.Handler.do_POST

    def do_post(handler):
        if urlparse(handler.path).path != "/api/customer-care/recommend":
            return previous_post(handler)

        try:
            with server.conn() as db:
                user = handler.user(db)
                if user["role"] != "customer":
                    return handler.json(
                        403, {"error": "Customer account required"}
                    )

            payload = handler.payload()
            if not isinstance(payload, dict):
                return handler.json(400, {"error": "Invalid request"})

            context = payload.get("context", {})
            if not isinstance(context, dict):
                return handler.json(400, {"error": "Invalid context"})

            # Never trust browser-submitted weather or assignment claims.
            # Until verified server-side context is connected, only
            # non-sensitive, general advice can be returned.
            safe = {
                "language": context.get("language", "English"),
                "stage": "dashboard",
                "weather": {},
                "acknowledged": context.get("acknowledged", []),
            }

            recommendations = care.evaluate(safe)

            return handler.json(200, {
                "ok": True,
                "version": VERSION,
                "recommendations": recommendations
            })

        except PermissionError:
            return handler.json(
                401, {"error": "Please sign in"}
            )
        except Exception:
            return handler.json(
                500, {"error": "Customer care unavailable"}
            )

    server.Handler.do_POST = do_post
    server.Handler._customer_care_api_version = VERSION

    return {"version": VERSION, "installed": True}
'''
    reviewed_legacy_engine = r'''import math

from datetime import datetime
from zoneinfo import ZoneInfo

VERSION = "customer-care-0.1"
TZ = ZoneInfo("Asia/Kolkata")

MESSAGES = {
    "rain": (
        "Rain is possible. Carry an umbrella and keep your belongings dry.",
        "மழை பெய்ய வாய்ப்புள்ளது. குடை எடுத்துச் செல்லுங்கள்; உடைமைகளை நனையாமல் பாதுகாத்துக்கொள்ளுங்கள்.",
        "Mazhai peiya vaaippu irukku. Umbrella eduthuttu ponga; belongings nanaiyaama paathukkonga."
    ),
    "heat": (
        "It's quite warm. Carry drinking water and stay hydrated.",
        "வெப்பம் அதிகமாக உள்ளது. குடிநீர் எடுத்துச் சென்று உடலில் நீர்ச்சத்து குறையாமல் பார்த்துக்கொள்ளுங்கள்.",
        "Veyil adhigama irukku. Thanni eduthuttu ponga; hydrated ah irunga."
    ),
    "night": (
        "For late-night travel, wait in a well-lit area and verify your driver before boarding.",
        "இரவு நேரப் பயணத்தில் வெளிச்சமான இடத்தில் காத்திருந்து, ஏறும் முன் ஓட்டுநர் விவரங்களைச் சரிபாருங்கள்.",
        "Night travel ku velichamaana place la wait pannunga. Erurathukku munnaadi driver details verify pannunga."
    ),
    "otp": (
        "Share your pickup OTP only with your assigned driver when boarding.",
        "வாகனத்தில் ஏறும்போது மட்டும் ஒதுக்கப்பட்ட ஓட்டுநரிடம் உங்கள் OTP-ஐ பகிருங்கள்.",
        "Board pannumbodhu mattum assigned driver kitta pickup OTP share pannunga."
    ),
    "capacity": (
        "Please select a vehicle that comfortably accommodates everyone travelling.",
        "பயணம் செய்யும் அனைவருக்கும் வசதியான வாகனத்தைத் தேர்ந்தெடுக்கவும்.",
        "Travel panra ellaarukkum comfortable ah irukkura vehicle select pannunga."
    ),
    "airport": (
        "Before leaving for the airport, check your travel documents and luggage.",
        "விமான நிலையத்திற்குச் செல்லும் முன் பயண ஆவணங்களையும் உடைமைகளையும் சரிபாருங்கள்.",
        "Airport ku kilamburathukku munnaadi documents um luggage um check pannunga."
    ),
    "railway": (
        "Keep your train ticket and travel essentials ready.",
        "ரயில் பயணச்சீட்டையும் தேவையான பொருட்களையும் தயாராக வைத்துக்கொள்ளுங்கள்.",
        "Train ticket um travel essentials um ready ah vechukkonga."
    ),
    "hospital": (
        "Remember any medical documents and essential items you need.",
        "தேவையான மருத்துவ ஆவணங்களையும் அத்தியாவசியப் பொருட்களையும் எடுத்துக்கொள்ளுங்கள்.",
        "Thevaiyaana medical documents um essentials um eduthukkonga."
    ),
    "phone": (
        "Keep your phone charged so you can contact and track your driver.",
        "ஓட்டுநரைத் தொடர்புகொள்ளவும் கண்காணிக்கவும் கைப்பேசியில் போதுமான சார்ஜ் வைத்திருங்கள்.",
        "Driver ah contact pannavum track pannavum phone charge vechukkonga."
    ),
    "arrival": (
        "Before leaving the vehicle, please check that you have all your belongings.",
        "வாகனத்திலிருந்து இறங்கும் முன் உங்கள் உடைமைகள் அனைத்தும் இருப்பதைச் சரிபாருங்கள்.",
        "Vehicle la irundhu erangurathukku munnaadi belongings ellaam irukka nu check pannunga."
    ),
    "pickup": (
        "Choose a safe, accessible pickup point where your driver can stop.",
        "ஓட்டுநர் நிறுத்தக்கூடிய பாதுகாப்பான, எளிதில் அணுகக்கூடிய இடத்தைத் தேர்ந்தெடுக்கவும்.",
        "Driver stop panna mudiyura safe ah accessible ah irukkura pickup point choose pannunga."
    ),
    "accessibility": (
        "If you need boarding assistance, please coordinate with your assigned driver.",
        "ஏறுவதற்கு உதவி தேவைப்பட்டால் ஒதுக்கப்பட்ட ஓட்டுநருடன் தொடர்புகொள்ளுங்கள்.",
        "Boarding assistance venumna assigned driver kitta coordinate pannunga."
    ),
    "ready": (
        "Your driver is approaching. Please get ready at your pickup point.",
        "உங்கள் ஓட்டுநர் நெருங்கிக்கொண்டிருக்கிறார். ஏறும் இடத்தில் தயாராக இருங்கள்.",
        "Unga driver nerungittu irukkaaru. Pickup point la ready ah irunga."
    ),
}

CAPACITY = {"bike": 1, "auto": 3, "mini": 4, "sedan": 4, "suv": 6}

def evaluate(context):
    """Return up to two evidence-based recommendations, without side effects."""
    c = dict(context or {})
    language = c.get("language", "English")
    index = {"English": 0, "Tamil": 1, "Tanglish": 2}.get(language, 0)
    stage = c.get("stage", "dashboard")
    acknowledged = set(c.get("acknowledged") or [])
    candidates = []

    def add(key, score, evidence):
        if key not in acknowledged:
            candidates.append({
                "id": key,
                "score": score,
                "text": MESSAGES[key][index],
                "evidence": evidence
            })

    weather = c.get("weather") or {}
    if weather.get("status") == "ok":
        probability = weather.get("precipitation_probability")
        code = weather.get("weather_code")
        temperature = weather.get("temperature_2m")
        feels = weather.get("apparent_temperature")

        rain_codes = {51,53,55,56,57,61,63,65,66,67,80,81,82,95,96,99}
        if stage in ("dashboard", "details", "pre_confirmation"):
            if (isinstance(code, int) and code in rain_codes) or (
                isinstance(probability, (int,float)) and
                not isinstance(probability, bool) and probability >= 60
            ):
                add("rain", 95, "verified_weather")

            if (
                isinstance(temperature, (int,float)) and
                not isinstance(temperature, bool) and temperature >= 38
            ) or (
                isinstance(feels, (int,float)) and
                not isinstance(feels, bool) and feels >= 40
            ):
                add("heat", 90, "verified_weather")

    pickup_time = c.get("pickup_time")
    if stage in ("details", "pre_confirmation") and pickup_time:
        try:
            dt = datetime.fromisoformat(str(pickup_time))
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=TZ)
            hour = dt.astimezone(TZ).hour
            if hour >= 22 or hour < 5:
                add("night", 85, "pickup_time")
        except (ValueError, TypeError, OverflowError):
            pass

    vehicle = str(c.get("vehicle") or "").lower()
    count = c.get("passenger_count")
    if stage == "details" and vehicle in CAPACITY:
        if isinstance(count, int) and not isinstance(count, bool) and count > CAPACITY[vehicle]:
            add("capacity", 100, "passenger_count_and_vehicle")

    if stage == "pre_confirmation":
        destination = str(c.get("destination") or "").lower()
        if any(x in destination for x in ("airport", "விமான நிலைய")):
            add("airport", 75, "destination")
        elif any(x in destination for x in ("railway", "railway station", "train station", "ரயில் நிலைய")):
            add("railway", 75, "destination")
        elif any(x in destination for x in ("hospital", "மருத்துவமனை")):
            add("hospital", 75, "destination")

    if stage == "driver_assigned" and c.get("driver_assigned"):
        add("otp", 95, "driver_assignment")

    if stage == "driver_approaching" and c.get("driver_assigned"):
        eta = c.get("driver_eta_minutes")
        if isinstance(eta, (int,float)) and not isinstance(eta, bool) and 0 <= eta <= 10:
            add("ready", 95, "verified_driver_eta")

    if stage == "ride_completed" and c.get("ride_status") == "completed":
        add("arrival", 100, "completed_ride")

    if stage == "pre_confirmation" and c.get("pickup"):
        add("pickup", 45, "pickup_location")

    if stage == "driver_assigned" and c.get("accessibility_requested"):
        add("accessibility", 85, "explicit_customer_request")

    if stage == "driver_assigned" and c.get("driver_assigned"):
        add("phone", 40, "driver_assignment")

    candidates.sort(key=lambda x: (-x["score"], x["id"]))
    return candidates[:2]


# MOVIO_CUSTOMER_CARE_GREETING_V1
def dashboard_greeting(context):
    """Generate a greeting without inventing location or weather."""
    c = dict(context or {})
    language = c.get("language", "English")
    if language not in ("English", "Tamil", "Tanglish"):
        language = "English"

    locality = c.get("locality")
    if not isinstance(locality, str):
        locality = None
    elif not locality.strip() or len(locality) > 100:
        locality = None
    else:
        locality = locality.strip()

    weather = c.get("weather") or {}
    valid_weather = weather.get("status") == "ok"

    greetings = {
        "English": "Welcome back! Wishing you a comfortable journey.",
        "Tamil": "மீண்டும் வரவேற்கிறோம்! உங்கள் பயணம் இனிதாக அமையட்டும்.",
        "Tanglish": "Welcome back! Unga journey comfortable ah irukkattum."
    }

    messages = {
        "English": {
            "normal": "Hope you have a pleasant journey.",
            "rain": "Rain is possible. Carry an umbrella.",
            "heat": "It's quite warm. Remember to stay hydrated.",
            "pleasant": "The weather looks pleasant for your journey."
        },
        "Tamil": {
            "normal": "உங்கள் பயணம் இனிதாக அமையட்டும்.",
            "rain": "மழை பெய்ய வாய்ப்புள்ளது. குடை எடுத்துச் செல்லுங்கள்.",
            "heat": "வெப்பம் அதிகமாக உள்ளது. போதுமான தண்ணீர் குடியுங்கள்.",
            "pleasant": "பயணத்திற்கு வானிலை இனிமையாக உள்ளது."
        },
        "Tanglish": {
            "normal": "Unga journey nalla irukkattum.",
            "rain": "Mazhai peiya vaaippu irukku. Umbrella eduthuttu ponga.",
            "heat": "Veyil adhigama irukku. Thanni kudinga.",
            "pleasant": "Journey ku weather pleasant ah irukku."
        }
    }

    if not valid_weather:
        return {
            "text": greetings[language],
            "weather_verified": False,
            "locality_verified": False
        }

    code = weather.get("weather_code")
    temp = weather.get("temperature_2m")
    feels = weather.get("apparent_temperature")
    probability = weather.get("precipitation_probability")

    def numeric(value):
        return (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
        )

    rain_codes = {
        51, 53, 55, 56, 57, 61, 63, 65, 66, 67,
        80, 81, 82, 95, 96, 99
    }

    rain = (
        isinstance(code, int) and code in rain_codes
    ) or (
        numeric(probability) and probability >= 60
    )

    heat = (
        numeric(temp) and temp >= 38
    ) or (
        numeric(feels) and feels >= 40
    )

    pleasant = (
        isinstance(code, int)
        and code in (0, 1, 2)
        and numeric(temp)
        and 18 <= temp <= 32
        and not heat
    )

    condition = (
        "rain" if rain else
        "heat" if heat else
        "pleasant" if pleasant else
        "normal"
    )

    location_prefix = ""
    if locality:
        if language == "English":
            location_prefix = f"In {locality}, "
        elif language == "Tamil":
            location_prefix = f"{locality} பகுதியில், "
        else:
            location_prefix = f"{locality} la, "

    return {
        "text": location_prefix + messages[language][condition],
        "weather_verified": True,
        "locality_verified": bool(locality),
        "condition": condition
    }
'''

    def read_version(source):
        result=None
        for node in ast.parse(source).body:
            if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='VERSION' for t in node.targets):
                if isinstance(node.value,ast.Constant):result=node.value.value
        return result
    def ast_sha(source):
        return hashlib.sha256(ast.dump(ast.parse(source),include_attributes=False).encode()).hexdigest()
    old_api=api.read_bytes();old_engine=engine.read_bytes();old_bytes=server.read_bytes()
    actual_api=old_api.decode('utf-8-sig');actual_engine=old_engine.decode('utf-8-sig')
    old_api_version=read_version(actual_api);old_engine_version=read_version(actual_engine)
    expected_api='customer-care-api-1.0';expected_engine='customer-care-engine-1.1-compatible'
    if old_api_version=='customer-care-api-0.1':
        if ast_sha(actual_api)!=ast_sha(reviewed_legacy_api):
            raise RuntimeError('The API is a modified v0.1. Its source needs inspection before replacement. No files changed.\n'+actual_api)
        chosen_api=api_candidate
    elif old_api_version==expected_api:
        # Preserve compatible v1.0 customizations if the offline API tests pass.
        chosen_api=actual_api
    else:
        raise RuntimeError('Unexpected care API version %r. No files changed.' % old_api_version)
    if old_engine_version=='customer-care-0.1':
        if ast_sha(actual_engine)!=ast_sha(reviewed_legacy_engine):
            raise RuntimeError('The legacy care engine has custom changes. Share its source before upgrading. No files changed.')
        chosen_engine=engine_candidate
    elif old_engine_version==expected_engine:
        chosen_engine=actual_engine
    else:
        raise RuntimeError('Unexpected care engine version %r. No files changed.' % old_engine_version)
    care=types.ModuleType('movio_companion_upgrade_engine_fixture');care.__file__=str(engine)
    exec(compile(chosen_engine,str(engine),'exec',dont_inherit=True),care.__dict__)
    candidate=types.ModuleType('movio_companion_upgrade_api_fixture');candidate.__file__=str(api)
    exec(compile(chosen_api,str(api),'exec',dont_inherit=True),candidate.__dict__)
    tests={}
    exec(compile(engine_checks_source,'<care engine checks>','exec'),tests)
    engine_tests=tests['run_care_checks'](care)
    exec(compile(api_checks_source,'<care API checks>','exec'),tests)
    api_tests=tests['run_backend_checks'](candidate,care)
    print('Offline checks: engine %s, API %s — PASS.' % (engine_tests['checks'],api_tests['checks']),flush=True)
    new_text=movio_companion_startup_source(old_bytes.decode('utf-8-sig'))
    desired=[(api,old_api,chosen_api.encode('utf-8')),
             (engine,old_engine,chosen_engine.encode('utf-8')),
             (server,old_bytes,new_text.encode('utf-8'))]
    changes=[item for item in desired if item[1]!=item[2]]
    version=expected_api

    def listeners(port):
        return sorted({c.pid for c in psutil.net_connections(kind='inet')
            if c.status == 'LISTEN' and c.laddr.port == port and c.pid})

    def verified_owner(pid):
        if pid == os.getpid():
            raise RuntimeError('Port 8080 belongs to the notebook kernel. Use its original website-only launcher; no process stopped.')
        process = psutil.Process(pid)
        cwd = Path(process.cwd())
        command = process.cmdline()
        scripts = [Path(arg) for arg in command[1:] if arg.endswith('.py')]
        if not any((cwd / path).resolve() == server for path in scripts):
            raise RuntimeError('A different application owns port 8080; no process stopped.')
        return process, cwd, command

    owners = listeners(8080)
    if len(owners) != 1:
        raise RuntimeError('Expected one running website process on port 8080; found %s. No files changed.' % owners)
    process, cwd, command = verified_owner(owners[0])
    command = [process.exe()] + command[1:]
    environment = process.environ()  # Preserve existing website/AI URLs and all existing configuration.
    protected_before = {str(port): listeners(port) for port in (8000, 8090)}
    modes={path:path.stat().st_mode & 0o777 for path,_,_ in desired}
    session = requests.Session()
    session.trust_env = False
    backup = None
    new_process = None

    def fetch(path, method='get', **kwargs):
        return getattr(session, method)('http://127.0.0.1:8080' + path,
            timeout=(1, 3), allow_redirects=False, **kwargs)

    def atomic_write(path,value):
        fd, name = tempfile.mkstemp(prefix='.companion_server_', suffix='.tmp', dir=str(path.parent))
        try:
            with os.fdopen(fd, 'wb') as out:out.write(value)
            os.chmod(name,modes[path])
            os.replace(name,path)
        finally:
            if os.path.exists(name):os.unlink(name)

    def restart_website():
        nonlocal new_process
        current = [verified_owner(pid)[0] for pid in listeners(8080)]
        for owner in current:
            owner.terminate()
            owner.wait(timeout=8)
        if listeners(8080):raise RuntimeError('Website port is still occupied; no second server started.')
        with (root / 'weather_website.log').open('a') as log:
            new_process = subprocess.Popen(command, cwd=str(cwd), env=environment,
                stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

    def wait_website(check_care):
        deadline = time.monotonic() + 22
        last = 'starting'
        while time.monotonic() < deadline:
            if new_process is not None and new_process.poll() is not None:
                raise RuntimeError('The website process exited during startup; see weather_website.log.')
            try:
                health = fetch('/api/health')
                if health.status_code == 200:
                    if not check_care:return {'website': 200}
                    care = fetch('/api/customer-care/health')
                    if care.status_code not in (200, 401, 403):
                        raise RuntimeError('Care health HTTP %s' % care.status_code)
                    if care.status_code == 200:
                        view = care.json()
                        if not isinstance(view, dict) or view.get('ok') is not True:
                            raise RuntimeError('Care health did not return a valid health response.')
                        if view.get('version', version) != version:
                            raise RuntimeError('A different care API version is active.')
                    profile = fetch('/api/me')
                    signed_out = fetch('/api/customer-care/recommend', 'post', json={'context': {}})
                    if profile.status_code not in (401, 403) or signed_out.status_code not in (401, 403):
                        raise RuntimeError('Customer authentication verification failed.')
                    return dict(website=200, care_health=care.status_code,
                        signed_out_profile=profile.status_code, signed_out_care=signed_out.status_code)
            except (requests.RequestException, RuntimeError, ValueError) as exc:
                last = str(exc)
            time.sleep(0.35)
        raise RuntimeError('Website checks timed out: ' + last)

    try:
        if fetch('/api/health').status_code != 200:
            raise RuntimeError('The current website is not healthy. No files changed.')
        backup=Path(tempfile.mkdtemp(prefix='companion_api_upgrade_backup_',dir=str(root)))
        for path,before,_ in desired:
            if path.read_bytes()!=before:raise RuntimeError('A target changed during inspection. Rerun this cell.')
            saved=backup/path.relative_to(root);saved.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(path,saved)
            if saved.read_bytes()!=before:raise RuntimeError('Backup verification failed; no files changed.')
        print('Upgrading companion backend and restarting only the verified website process…',flush=True)
        written=[]
        try:
            for path,before,after in changes:
                if path.read_bytes()!=before:raise RuntimeError('A target changed before replacement.')
                atomic_write(path,after);written.append((path,before,after))
            restart_website()
            checks=wait_website(True)
            for path,_,after in desired:
                if path.read_bytes()!=after:raise RuntimeError('A target changed during the restart.')
        except Exception as original:
            for path,before,after in reversed(written):
                if path.read_bytes()==after:atomic_write(path,before)
            try:
                restart_website()
                wait_website(False)
                recovery='Previous sources restored and website restarted.'
            except Exception as recovery_error:
                recovery='Previous sources restored. Website recovery needs the original website-only launcher: '+str(recovery_error)
            raise RuntimeError(str(original)+'\n'+recovery+'\nBackup: '+str(backup)) from original
        result=dict(passed=True,previous_api_version=old_api_version,api_version=version,
            previous_engine_version=old_engine_version,engine_version=care.VERSION,
            offline_engine_checks=engine_tests['checks'],offline_api_checks=api_tests['checks'],
            changed_files=[str(path.relative_to(root)) for path,_,_ in changes],
            website_restarted=True,live_checks=checks,backup=str(backup),
            ai_and_gateway_listener_pids_unchanged=protected_before=={str(port):listeners(port) for port in (8000,8090)},
            ui_and_voice_sources_preserved=True,
            browser_weather_verification='Required: refresh the companion card in your signed-in dashboard.')
        print(json.dumps(result, indent=2))
        print('Refresh the customer dashboard, then tap the companion refresh button. Share the output and card result.')
        print('Keep this cell AFTER the last website restart/launch cell. Replacing server.py later can remove its startup hook again.')
        return result
    finally:
        session.close()


MOVIO_COMPANION_API_UPGRADE = movio_upgrade_companion_backend()


Offline checks: engine 270, API 70 — PASS.
Upgrading companion backend and restarting only the verified website process…
{
  "passed": true,
  "previous_api_version": "customer-care-api-0.1",
  "api_version": "customer-care-api-1.0",
  "previous_engine_version": "customer-care-0.1",
  "engine_version": "customer-care-engine-1.1-compatible",
  "offline_engine_checks": 270,
  "offline_api_checks": 70,
  "changed_files": [
    "integrations/movio_customer_care_api.py",
    "integrations/movio_customer_care.py",
    "server.py"
  ],
  "website_restarted": true,
  "live_checks": {
    "website": 200,
    "care_health": 200,
    "signed_out_profile": 401,
    "signed_out_care": 401
  },
  "backup": "/kaggle/working/MOVIO_MAIN_WEBSITE/companion_api_upgrade_backup_7dmukq6v",
  "ai_and_gateway_listener_pids_unchanged": true,
  "ui_and_voice_sources_preserved": true,
  "browser_weather_verification": "Required: refresh the companion card in your signed-in dashboard."
}
Refresh the customer dashb

In [151]:

from pathlib import Path
import sqlite3
import requests

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")

print("=== MOVIO DRIVER BOOKING DIAGNOSTIC ===")

# Check services
session = requests.Session()
session.trust_env = False

for port in (8080, 8000, 8090):
    try:
        r = session.get(
            f"http://127.0.0.1:{port}/api/health",
            timeout=4
        )
        print(f"Port {port}: HTTP {r.status_code}")
    except requests.RequestException as e:
        print(f"Port {port}: Unreachable ({type(e).__name__})")

# Locate booking databases
databases = sorted(root.rglob("*.db")) + sorted(root.rglob("*.sqlite"))

if not databases:
    print("No SQLite database found under project folder.")

for db in dict.fromkeys(databases):
    print("\nDATABASE:", db)

    try:
        with sqlite3.connect(
            f"file:{db}?mode=ro", uri=True
        ) as con:
            con.row_factory = sqlite3.Row

            tables = [
                row[0] for row in con.execute(
                    "SELECT name FROM sqlite_master WHERE type='table'"
                )
            ]

            if "rides" not in tables:
                print("No rides table")
                continue

            columns = [
                row[1] for row in con.execute(
                    "PRAGMA table_info(rides)"
                )
            ]

            print("Ride columns:", columns)

            print("Bookings by status:")
            for row in con.execute(
                "SELECT status, COUNT(*) AS total "
                "FROM rides GROUP BY status"
            ):
                print(dict(row))

            selected = [
                c for c in [
                    "id", "status", "vehicle",
                    "driver_id", "pickup_time",
                    "created"
                ] if c in columns
            ]

            if selected:
                query = (
                    "SELECT " + ", ".join(selected) +
                    " FROM rides ORDER BY id DESC LIMIT 15"
                )

                print("\nRecent bookings:")
                for row in con.execute(query):
                    print(dict(row))

    except sqlite3.Error as e:
        print("Database error:", e)

session.close()


=== MOVIO DRIVER BOOKING DIAGNOSTIC ===
Port 8080: HTTP 200
Port 8000: HTTP 404
Port 8090: HTTP 200

DATABASE: /kaggle/working/MOVIO_MAIN_WEBSITE/movio_demo.db
Ride columns: ['id', 'customer_id', 'pickup', 'destination', 'plat', 'plng', 'dlat', 'dlng', 'vehicle', 'fare', 'status', 'driver_id', 'source', 'created', 'updated', 'passengers', 'pickup_time', 'weather_json', 'journey_version', 'voice_call_id']
Bookings by status:

Recent bookings:


In [152]:
from pathlib import Path
import zipfile

root = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")
output = Path("/kaggle/working/MOVIO_DRIVER_FIX_SOURCE.zip")

files = [
    "server.py",
    "web/index.html",
    "web/app.js",
    "web/dashboard_priority.js",
    "web/styles.css",
    "integrations/movio_weather_guard.py"
]

with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
    for name in files:
        path = root / name
        if path.is_file():
            z.write(path, arcname=name)
            print("Included:", name)
        else:
            print("Not found:", name)

print("\nZIP created:", output)

Included: server.py
Included: web/index.html
Included: web/app.js
Included: web/dashboard_priority.js
Included: web/styles.css
Included: integrations/movio_weather_guard.py

ZIP created: /kaggle/working/MOVIO_DRIVER_FIX_SOURCE.zip


In [153]:
from pathlib import Path
from datetime import datetime
from uuid import uuid4
import zipfile
import shutil
import ast

ROOT = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")

FILES = [
    "server.py",
    "web/index.html",
    "web/movio_driver_upcoming.js"
]

# 1. Search all ZIP files in Kaggle
zip_files = []

for folder in [Path("/kaggle/input"), Path("/kaggle/working")]:
    if folder.exists():
        zip_files.extend(folder.rglob("*.zip"))

print("ZIP files found:")
for path in zip_files:
    print(" -", path)

# 2. Identify patch by its contents
PATCH = None

for path in zip_files:
    try:
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            if all(name in names for name in FILES):
                PATCH = path
                break
    except (zipfile.BadZipFile, OSError):
        continue

if PATCH is None:
    print("\nPATCH NOT FOUND")
    print("Upload MOVIO_DRIVER_UPCOMING_PATCH.zip to Kaggle using Add Input.")
    print("Then rerun this cell.")
else:
    print("\nPATCH FOUND:", PATCH)

    if not ROOT.is_dir():
        raise FileNotFoundError(f"Website folder missing: {ROOT}")

    # 3. Validate patch
    with zipfile.ZipFile(PATCH) as z:
        ast.parse(z.read("server.py").decode("utf-8"))

        # 4. Create unique backup
        BACKUP = ROOT / (
            "driver_fix_backup_"
            + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            + "_"
            + uuid4().hex[:8]
        )

        BACKUP.mkdir(parents=True, exist_ok=False)

        # 5. Backup existing files
        for name in FILES:
            source = ROOT / name

            if source.exists():
                destination = BACKUP / name
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)

        # 6. Install patch
        for name in FILES:
            destination = ROOT / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(z.read(name))
            print("UPDATED:", name)

    print("\nPATCH INSTALLED SUCCESSFULLY")
    print("Backup:", BACKUP)
    print("Database and existing bookings unchanged.")

ZIP files found:
 - /kaggle/working/MOVIO_DRIVER_FIX_SOURCE.zip
 - /kaggle/working/all_kaggle_inputs.zip
 - /kaggle/working/MOVIO_CURRENT_WEBSITE.zip

PATCH NOT FOUND
Upload MOVIO_DRIVER_UPCOMING_PATCH.zip to Kaggle using Add Input.
Then rerun this cell.


In [154]:
# Run as ONE Kaggle cell after the last website startup cell.
# Read-only: no bookings, files, profiles or services are changed.
def movio_check_driver_matching():
    import ast
    import json
    import os
    import sqlite3
    from datetime import datetime
    from pathlib import Path
    from zoneinfo import ZoneInfo
    from urllib.request import build_opener, ProxyHandler
    from urllib.error import HTTPError

    root = Path(globals().get('WX_ROOT') or
                '/kaggle/working/MOVIO_MAIN_WEBSITE').resolve()
    server = root / 'server.py'
    if not server.is_file():
        raise FileNotFoundError(str(server))
    now = datetime.now(ZoneInfo('Asia/Kolkata'))
    report = {'local_now': now.isoformat(), 'changes_made': False}
    source = server.read_text(encoding='utf-8')
    tree = ast.parse(source)
    lines = source.splitlines()

    # Inspect the running website without printing its environment or credentials.
    report['website_processes'] = []
    try:
        import psutil
        pids = {c.pid for c in psutil.net_connections(kind='inet')
                if c.status == 'LISTEN' and c.laddr.port == 8080 and c.pid}
        for pid in sorted(pids):
            p = psutil.Process(pid)
            scripts = [str((Path(p.cwd()) / a).resolve())
                       for a in p.cmdline() if a.endswith('.py')]
            report['website_processes'].append({
                'pid': pid, 'in_notebook_kernel': pid == os.getpid(),
                'expected_server': str(server) in scripts})
    except Exception as exc:
        report['process_check'] = type(exc).__name__

    client = build_opener(ProxyHandler({}))
    report['health'] = {}
    for endpoint in ('/api/health', '/api/weather-booking/health'):
        try:
            with client.open('http://127.0.0.1:8080' + endpoint,
                             timeout=5) as response:
                report['health'][endpoint] = response.status
        except HTTPError as exc:
            report['health'][endpoint] = exc.code
        except Exception as exc:
            report['health'][endpoint] = type(exc).__name__

    # Never create a missing database. Use SQLite's read-only connection.
    database = root / 'movio_demo.db'
    report['expected_database_exists'] = database.is_file()
    voice_db = getattr(globals().get('bridge'), '_wx_db', None)
    voice_path = getattr(voice_db, 'path', None)
    report['voice_database_matches_expected'] = (
        Path(voice_path).resolve() == database if voice_path else 'not loaded')
    report['findings'] = []
    if database.is_file():
        with sqlite3.connect(database.as_uri() + '?mode=ro', uri=True,
                             timeout=5) as db:
            db.row_factory = sqlite3.Row
            tables = {r[0] for r in db.execute(
                "SELECT name FROM sqlite_master WHERE type='table'")}
            report['schema'] = {}
            for table in ('rides', 'driver_profiles'):
                if table in tables:
                    report['schema'][table] = [r[1] for r in db.execute(
                        'PRAGMA table_info(' + table + ')')]
            cols = set(report['schema'].get('rides', []))
            if {'id', 'vehicle', 'status'} <= cols:
                selected = [k for k in ('id', 'vehicle', 'status',
                            'pickup_time', 'source') if k in cols]
                sql = 'SELECT ' + ','.join(selected)
                if 'driver_id' in cols:
                    sql += ',(driver_id IS NOT NULL) AS has_driver'
                rows = [dict(r) for r in db.execute(
                    sql + ' FROM rides ORDER BY id DESC LIMIT 15')]
                for row in rows:
                    raw = row.get('pickup_time')
                    if raw:
                        try:
                            when = datetime.fromisoformat(str(raw).replace('Z', '+00:00'))
                            if when.tzinfo is None:
                                row['pickup_time_issue'] = 'missing timezone'
                            else:
                                minutes = (when - now).total_seconds() / 60
                                row['minutes_to_pickup'] = round(minutes, 1)
                                if row['status'] == 'scheduled':
                                    row['dispatch_check'] = (
                                        'not due: outside 20-minute window' if minutes > 20
                                        else 'due: check release worker' if minutes >= 0
                                        else 'past pickup: original release skips this ride')
                        except (ValueError, TypeError):
                            row['pickup_time_issue'] = 'invalid datetime'
                report['recent_bookings'] = rows
                report['vehicle_status_counts'] = [dict(r) for r in db.execute(
                    'SELECT vehicle,status,COUNT(*) AS n FROM rides '
                    'GROUP BY vehicle,status ORDER BY vehicle,status')]
            profile_cols = set(report['schema'].get('driver_profiles', []))
            if 'vehicle_type' in profile_cols:
                report['driver_vehicle_counts'] = [dict(r) for r in db.execute(
                    'SELECT vehicle_type,COUNT(*) AS n FROM driver_profiles '
                    'GROUP BY vehicle_type')]

    # Show only matching-related source: no imports or server execution.
    def excerpt(text, words, limit=90):
        ls = text.splitlines()
        indexes = set()
        for i, line in enumerate(ls):
            if any(word in line for word in words):
                indexes.update(range(max(0, i-3), min(len(ls), i+5)))
        result = []
        for i in sorted(indexes)[:limit]:
            line = ls[i]
            if any(w in line.lower() for w in (
                    'password', 'secret', 'api_key', 'authorization', 'authtoken')):
                line = '[credential-related source omitted]'
            result.append(f'{i+1}: {line}')
        return result

    words = ('driver_profiles', 'vehicle_type', "'requested'", '"requested"',
             "'scheduled'", '"scheduled"', '/api/rides', '/accept',
             'driver_id', 'available', 'online', 'busy')
    report['server_matching_source'] = excerpt(source, words, 150)
    report['database_configuration_source'] = [
        ast.get_source_segment(source, n) for n in tree.body
        if isinstance(n, (ast.Assign, ast.AnnAssign)) and any(
            isinstance(x, ast.Name) and x.id in ('DB', 'ROOT')
            for target in (n.targets if isinstance(n, ast.Assign) else [n.target])
            for x in ast.walk(target))]
    report['source_checks'] = {}
    for rel in ('integrations/movio_weather_guard.py', 'web/driver_dashboard.js',
                'web/auth.js', 'web/movio_enhancements.js', 'web/app.js'):
        path = root / rel
        if path.is_file():
            report['source_checks'][rel] = excerpt(
                path.read_text(encoding='utf-8'),
                ('release_due', '_wx_dispatch_thread', 'install_backend',
                 '/api/rides', 'refreshRides', 'setInterval',
                 'vehicle_type', 'incoming=', 'status===', 'MOVIO_PORT'), 45)
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('\nShare this output. No repair has been applied yet.')
    return report

MOVIO_DRIVER_MATCHING_CHECK = movio_check_driver_matching()


{
  "local_now": "2026-10-28T12:37:44.136467+05:30",
  "changes_made": false,
  "website_processes": [
    {
      "pid": 947,
      "in_notebook_kernel": false,
      "expected_server": true
    }
  ],
  "health": {
    "/api/health": 200,
    "/api/weather-booking/health": 200
  },
  "expected_database_exists": true,
  "voice_database_matches_expected": true,
  "findings": [],
  "schema": {
    "rides": [
      "id",
      "customer_id",
      "pickup",
      "destination",
      "plat",
      "plng",
      "dlat",
      "dlng",
      "vehicle",
      "fare",
      "status",
      "driver_id",
      "source",
      "created",
      "updated",
      "passengers",
      "pickup_time",
      "weather_json",
      "journey_version",
      "voice_call_id"
    ],
    "driver_profiles": [
      "user_id",
      "vehicle_type",
      "vehicle_number"
    ]
  },
  "recent_bookings": [],
  "vehicle_status_counts": [],
  "driver_vehicle_counts": [],
  "server_matching_source": [
    "21:  with 

In [155]:
# Reuse the notebook's existing routing repair.
# No accounts or bookings are deleted.
from pathlib import Path
from html import escape
import sqlite3
from IPython.display import display, HTML

repair = globals().get("movio_repair_public_routing")
if not callable(repair):
    raise RuntimeError(
        "Run the original public-routing cell defining "
        "movio_repair_public_routing, then rerun this cell."
    )

MOVIO_ROUTING_STATUS = repair()

url = str(MOVIO_ROUTING_STATUS.get("website_url") or "").rstrip("/")
if not MOVIO_ROUTING_STATUS.get("ok") or not url.startswith("https://"):
    raise RuntimeError("The website connection could not be verified.")

root = Path(
    globals().get("WX_ROOT")
    or "/kaggle/working/MOVIO_MAIN_WEBSITE"
).resolve()

database = root / "movio_demo.db"
if not database.is_file():
    raise FileNotFoundError(str(database))

with sqlite3.connect(
    database.as_uri() + "?mode=ro", uri=True, timeout=5
) as db:
    print("Accounts by role:", dict(db.execute(
        "SELECT role, COUNT(*) FROM users GROUP BY role"
    )))
    print("Saved bookings:", db.execute(
        "SELECT COUNT(*) FROM rides"
    ).fetchone()[0])
    print("Driver profiles:", db.execute(
        "SELECT COUNT(*) FROM driver_profiles"
    ).fetchone()[0])

display(HTML(
    '<p>Use this same website for both accounts:</p>'
    f'<a href="{escape(url, quote=True)}" '
    f'target="_blank" rel="noopener noreferrer">{escape(url)}</a>'
))

Public gateway: HTTP 200
Public website: HTTP 200
Public website health: HTTP 200
Public AI health: HTTP 200
Public call page: HTTP 200
Protected voice configuration: HTTP 401
Website: https://army-grandson-backside.ngrok-free.dev
Routing checks passed. Rerun the original verification cell, then continue below it.
Accounts by role: {}
Saved bookings: 0
Driver profiles: 0


In [156]:
# Shared pronunciation correction — English, Tamil and Tanglish
# Run after your existing voice/pronunciation setup.

import re
import functools

def movio_install_common_pronunciation():
    namespace = globals()
    current = namespace.get("normalize_taxi_text")

    if not callable(current):
        raise RuntimeError(
            "Run the original TTS setup first: "
            "normalize_taxi_text is not available."
        )

    # Re-running this cell does not add another wrapper.
    if getattr(current, "_movio_place_celsius_v1", False):
        print("Pronunciation correction is already installed.")
        return

    replacements = (
        (
            re.compile(
                r"(?<![A-Za-z])(?:vandaloor|vandalur)(?![A-Za-z])",
                re.IGNORECASE,
            ),
            "வண்டலூர்",
        ),
        (
            re.compile(
                r"(?<![A-Za-z])celsius(?![A-Za-z])",
                re.IGNORECASE,
            ),
            "செல்சியஸ்",
        ),
    )

    def correct_pronunciation(text):
        for pattern, spoken_form in replacements:
            text = pattern.sub(spoken_form, text)
        return text

    @functools.wraps(current)
    def corrected_normalizer(*args, **kwargs):
        # Preserve all existing normalization first.
        result = current(*args, **kwargs)

        # Apply independently of the selected language.
        if isinstance(result, str):
            return correct_pronunciation(result)

        return result

    corrected_normalizer._movio_place_celsius_v1 = True
    corrected_normalizer._movio_original = current

    # Verify the replacement rules without generating speech.
    assert correct_pronunciation("Vandaloor") == "வண்டலூர்"
    assert correct_pronunciation("Vandalur") == "வண்டலூர்"
    
    assert correct_pronunciation("degrees Celsius") == "degrees செல்சியஸ்"
    assert correct_pronunciation("28.5 Celsius") == "28.5 செல்சியஸ்"
    assert correct_pronunciation("வண்டலூர் செல்சியஸ்") == "வண்டலூர் செல்சியஸ்"

    namespace["normalize_taxi_text"] = corrected_normalizer

    print("Pronunciation rules installed; replacement checks passed.")
    print("Vandaloor / Vandalur → வண்டலூர்")
    print("Celsius → செல்சியஸ்")
    print("Rules apply regardless of the selected language.")
    print("Close the old AI Call popup and start a new call.")

movio_install_common_pronunciation()

Pronunciation rules installed; replacement checks passed.
Vandaloor / Vandalur → வண்டலூர்
Celsius → செல்சியஸ்
Rules apply regardless of the selected language.
Close the old AI Call popup and start a new call.


In [157]:
from pathlib import Path
import shutil
from IPython.display import display, FileLink

project = Path("/kaggle/working/MOVIO_MAIN_WEBSITE")

if not project.is_dir():
    raise FileNotFoundError(f"Project folder not found: {project}")

archive = shutil.make_archive(
    "/kaggle/working/MOVIO_CURRENT_PROJECT",
    "zip",
    root_dir=project.parent,
    base_dir=project.name,
)

print("Current project exported.")
display(FileLink(archive))

Current project exported.


/kaggle/working/MOVIO_CURRENT_PROJECT.zip